# Tools and Toolchains

This notebook demonstrates using tools and toolchains when ordering with the orders api. Specifically, this notebook demonstrates the following toolchains:
 - [clip](#clip)
 - [bandmath](#bandmath)
 - [toar](#toar)
 - [composite](#composite)
 - [clip -> bandmath](#clip_bandmath)
 - [toar -> reproject -> tile](#toar_reproject_tile)

For background on ordering and downloading with the orders api, see the [Ordering and Delivery](https://github.com/planetlabs/notebooks/blob/master/jupyter-notebooks/orders_api_tutorials/ordering_and_delivery.ipynb) notebook.

Reference information can be found at [Tools & toolchains](https://developers.planet.com/apis/orders/tools/).

## Setup

In [23]:
import json
import os
import pathlib
import time

import numpy as np
from planet import Auth, reporting
from planet import Session, DataClient, OrdersClient
import rasterio
from rasterio.plot import show
import requests
import glob

In [2]:
# API Key stored as an env variable
# if your Planet API Key is not set as an environment variable, you can paste it below
if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = 'PASTE_API_KEY_HERE'

# construct auth tuple for use in the requests library
BASIC_AUTH = (API_KEY, '')

# Setup the session
session = requests.Session()

# Authenticate
session.auth = (API_KEY, "")

# Establish headers & Orders URL
headers = {'content-type': 'application/json'}
orders_url = 'https://api.planet.com/compute/ops/orders/v2'

In [3]:
# define products part of order
single_product = [
    {
      "item_ids": ["20151119_025740_0c74"],
      "item_type": "PSScene",
      "product_bundle": "analytic_udm2"
    }
]

same_src_products = [
    {
      "item_ids": ["20151119_025740_0c74",
                   "20151119_025739_0c74"],
      "item_type": "PSScene",
      "product_bundle": "analytic_udm2"
    }
]

multi_src_products = [
    {
      "item_ids": ["20151119_025740_0c74"],
      "item_type": "PSScene",
      "product_bundle": "analytic_udm2"
    },
    {
      "item_ids": ["LC81330492015320LGN01"],
      "item_type": "Landsat8L1G",
      "product_bundle": "analytic"
    },
    
]

In [16]:
# to get this list I used the list on the planet website from my previous downloads
# and used cot editor find and replace regex to insert quote marks etc.
# find: \R replace with: ",\r
images_21 = ["20211128_112201_58_2274",
            "20211126_112140_93_2413", 
            "20211126_112138_63_2413", 
            "20211121_104533_42_225a", 
            "20211121_103426_25_2449", 
            "20211121_103423_95_2449", 
            "20211115_104512_70_2233", 
            "20211115_103642_53_2458", 
            "20211115_103640_06_2458", 
            "20211022_112631_93_2406", 
            "20211021_112311_88_227a", 
            "20210901_103602_27_2460", 
            "20210901_103559_97_2460", 
            "20210818_104349_95_227e", 
            "20210818_104347_66_227e", 
            "20210818_103124_15_241b", 
            "20210818_103121_67_241b", 
            "20210810_103630_73_220b", 
            "20210726_103643_58_245d", 
            "20210715_104637_80_222f", 
            "20210715_104635_59_222f", 
            "20210715_103425_00_2449", 
            "20210715_103422_70_2449", 
            "20210701_112856_26_2405", 
            "20210627_103640_07_2449", 
            "20210627_103252_52_245a", 
            "20210531_112802_19_227c", 
            "20210531_112759_70_227c", 
            "20210531_104849_25_2259", 
            "20210530_103838_88_242a", 
            "20210530_103836_57_242a", 
            "20210530_103659_28_2453", 
            "20210530_103656_98_2453", 
            "20210505_103609_03_2445", 
            "20210425_104201_66_2441", 
            "20210425_103321_12_2456", 
            "20210423_103443_39_245e", 
            "20210422_112648_84_2416", 
            "20210422_112646_49_2416", 
            "20210422_103413_14_2212", 
            "20210419_112830_31_2426", 
            "20210416_103945_82_2465", 
            "20210412_104349_05_2259", 
            "20210405_103433_50_245e", 
            "20210403_112730_08_2412", 
            "20210327_103856_65_2463", 
            "20210327_103854_29_2463", 
            "20210321_112622_77_2401", 
            "20210321_112620_51_2401", 
            "20210319_103335_25_245a", 
            "20210315_103455_16_2421", 
            "20210315_103452_81_2421", 
            "20210301_112635_82_2413", 
            "20210301_112633_55_2413", 
            "20210225_104655_34_2264", 
            "20210222_112512_89_2408", 
            "20210211_113113_80_2403", 
            "20210208_113054_06_2407", 
            "20210208_112813_70_2424", 
            "20210208_112811_36_2424", 
            "20210130_112944_25_241c", 
            "20210125_112739_86_2414", 
            "20210125_112737_58_2414", 
            "20210125_104635_84_2441", 
            "20210125_104633_56_2441", 
            "20210122_113115_67_240a", 
            "20210122_113113_32_240a", 
            "20210112_112441_70_2414", 
            "20210106_112858_23_240c", 
            "20210106_112446_22_2416", 
            "20201203_113143_06_227b", 
            "20201203_113140_69_227b", 
            "20201125_104259_22_2278", 
            "20201125_104257_01_2278", 
            "20201119_103352_21_2235", 
            "20200926_113129_67_2412", 
            "20200926_113127_32_2412", 
            "20200918_104047_72_2223", 
            "20200918_104045_51_2223", 
            "20200407_103546_31_2275"]

In [18]:
src_21 = [
    {
      "item_ids": images_21,
      "item_type": "PSScene",
      "product_bundle": "analytic_8b_sr_udm2"
    }
]

In [25]:
# get all planet IDs that I've ever downloaded?
data_path = '/Users/daa5@stir.ac.uk/Project/DALEC_processing/planetData/'
globos = glob.glob(data_path + '*8b_udm2/PSScene/*_3B_AnalyticMS_8b_clip.tif')
print(len(globos))
planet_ids = [globi[-49:-26] for globi in globos]
planet_ids_no_dups = []
[planet_ids_no_dups.append(x) for x in planet_ids if x not in planet_ids_no_dups] 
print(len(planet_ids_no_dups))
planet_ids_no_dups

361
337


['20221125_111318_95_2413',
 '20230209_111628_64_240c',
 '20220305_105837_49_248e',
 '20221128_110618_08_227a',
 '20220920_105809_51_2495',
 '20220215_102554_23_241b',
 '20220321_102740_31_242b',
 '20230126_110351_81_2485',
 '20220227_111348_12_2274',
 '20220227_103256_21_2430',
 '20221205_110047_16_2484',
 '20230125_104939_27_2276',
 '20230126_110127_66_24a5',
 '20220910_110421_57_249c',
 '20230128_110445_35_2480',
 '20221016_110338_23_2461',
 '20220327_111309_86_2416',
 '20220924_104649_13_2251',
 '20221206_111226_60_227b',
 '20230126_110354_09_2485',
 '20220324_111806_41_2403',
 '20230224_102308_97_241d',
 '20220331_102509_46_2451',
 '20230128_102801_66_2464',
 '20230117_105702_44_2475',
 '20220308_103324_53_241e',
 '20220127_110501_81_2475',
 '20230201_110354_54_2475',
 '20221018_102515_23_2451',
 '20221002_105839_86_2482',
 '20220217_102736_74_2428',
 '20220208_111800_46_2254',
 '20230130_101429_64_2445',
 '20221119_110126_09_249d',
 '20220920_110455_06_247b',
 '20220914_102314_67

In [26]:
# 8 band SR products for the entire series that I've worked with so far
src_full_IDS = [
    {
      "item_ids": planet_ids_no_dups,
      "item_type": "PSScene",
      "product_bundle": "analytic_8b_sr_udm2"
    }
]

In [15]:
async def poll_and_download(order):
    async with Session() as sess:
        cl = OrdersClient(sess)

        # Use "reporting" to manage polling for order status
        with reporting.StateBar(state='creating') as bar:
            # Grab the order ID
            bar.update(state='created', order_id=order['id'])

            # poll...poll...poll...
            await cl.wait(order['id'], callback=bar.update_state)

        # if we get here that means the order completed. Yay! Download the files.
        filenames = await cl.download_order(order['id'])

In [5]:
# define helpful functions for visualizing downloaded imagery
def show_rgb(img_file):
    with rasterio.open(img_file) as src:
        b,g,r,n = src.read()

    rgb = np.stack((r,g,b), axis=0)
    show(rgb/rgb.max())
    
def show_gray(img_file):
    with rasterio.open(img_file) as src:
        g = src.read(1)
    show(g/g.max())

## Tool Demos

### No Processing (reference)

We will order and download the unprocessed image for comparison with the output of the toolchains defined below.

In [6]:
request = {
  "name": "no processing",
  "products": single_product,
}

In [7]:
# allow for caching, replace this with your image file
img_file = 'data/50df6201-ea94-48f1-bec9-65dd9cd8354b/1/files/PSScene/20151119_025740_0c74/analytic_udm2/20151119_025740_0c74_3B_AnalyticMS.tif'
img_file

'data/50df6201-ea94-48f1-bec9-65dd9cd8354b/1/files/PSScene/20151119_025740_0c74/analytic_udm2/20151119_025740_0c74_3B_AnalyticMS.tif'

In [13]:
if not os.path.isfile(img_file):
    async with Session() as sess:
        cl = OrdersClient(sess)
        print('ordering')
        order = await cl.create_order(request)
        print(order)
        print('downloading')
        download = await poll_and_download(order)
        print(download)
    # this last line doesn't work because poll_and_download doesn't return anything...
    # but the file does appear in planet account available to download!
    img_file = next(download[d] for d in download
                    if d.endswith('_3B_AnalyticMS.tif'))

ordering
{'_links': {'_self': 'https://api.planet.com/compute/ops/orders/v2/aafeef05-f9d7-47c0-9d74-5d675c59bf8e'}, 'created_on': '2024-05-30T10:20:06.211Z', 'error_hints': [], 'id': 'aafeef05-f9d7-47c0-9d74-5d675c59bf8e', 'last_message': 'Preparing order', 'last_modified': '2024-05-30T10:20:06.211Z', 'name': 'no processing', 'products': [{'item_ids': ['20151119_025740_0c74'], 'item_type': 'PSScene', 'product_bundle': 'analytic_udm2'}], 'state': 'queued'}
downloading


02:55 - order aafeef05-f9d7-47c0-9d74-5d675c59bf8e - state: success


[PosixPath('aafeef05-f9d7-47c0-9d74-5d675c59bf8e/PSScene/20151119_025740_0c74_3B_udm2.tif'), PosixPath('aafeef05-f9d7-47c0-9d74-5d675c59bf8e/PSScene/20151119_025740_0c74_3B_AnalyticMS_metadata.xml'), PosixPath('aafeef05-f9d7-47c0-9d74-5d675c59bf8e/PSScene/20151119_025740_0c74_3B_AnalyticMS.tif'), PosixPath('aafeef05-f9d7-47c0-9d74-5d675c59bf8e/PSScene/20151119_025740_0c74_metadata.json'), PosixPath('aafeef05-f9d7-47c0-9d74-5d675c59bf8e/manifest.json')]


AttributeError: 'PosixPath' object has no attribute 'endswith'

In [14]:
img_file

'data/50df6201-ea94-48f1-bec9-65dd9cd8354b/1/files/PSScene/20151119_025740_0c74/analytic_udm2/20151119_025740_0c74_3B_AnalyticMS.tif'

### Clip
<a id='clip'></a>

Clipping is likely the most common tool that will be used. It allows us to only download the pixels we are interested in.

In [19]:
clip_aoi = {"coordinates":[[[-3.925805,56.143895],
                            [-3.911592,56.143895],
                            [-3.911592,56.149272],
                            [-3.925805,56.149272],
                            [-3.925805,56.143895]]],
            "type":"Polygon"}

In [20]:
# define the clip tool
clip = {
    "clip": {
        "aoi": clip_aoi
    }
}

In [29]:
# create an order request with the clipping tool
request_clip = {
  "name": "full_airth_SR",
  "products": src_full_IDS,
  "tools": [clip]
}

request_clip

{'name': 'full_airth_SR',
 'products': [{'item_ids': ['20221125_111318_95_2413',
    '20230209_111628_64_240c',
    '20220305_105837_49_248e',
    '20221128_110618_08_227a',
    '20220920_105809_51_2495',
    '20220215_102554_23_241b',
    '20220321_102740_31_242b',
    '20230126_110351_81_2485',
    '20220227_111348_12_2274',
    '20220227_103256_21_2430',
    '20221205_110047_16_2484',
    '20230125_104939_27_2276',
    '20230126_110127_66_24a5',
    '20220910_110421_57_249c',
    '20230128_110445_35_2480',
    '20221016_110338_23_2461',
    '20220327_111309_86_2416',
    '20220924_104649_13_2251',
    '20221206_111226_60_227b',
    '20230126_110354_09_2485',
    '20220324_111806_41_2403',
    '20230224_102308_97_241d',
    '20220331_102509_46_2451',
    '20230128_102801_66_2464',
    '20230117_105702_44_2475',
    '20220308_103324_53_241e',
    '20220127_110501_81_2475',
    '20230201_110354_54_2475',
    '20221018_102515_23_2451',
    '20221002_105839_86_2482',
    '20220217_102736

In [ ]:
# allow for caching so we don't always run clip
run_clip = True

clip_img_file = 'data/6c23f9e1-d86a-47fe-9eb2-2693196168e0/1/files/20151119_025740_0c74_3B_AnalyticMS_clip.tif'
if os.path.isfile(clip_img_file): run_clip = False

In [30]:
# if run_clip:
async with Session() as sess:
    cl = OrdersClient(sess)
    order = await cl.create_order(request_clip)
    download = await poll_and_download(order)
#     clip_img_file = next(downloaded_clip_files[d] for d in downloaded_clip_files
#                      if d.endswith('_3B_AnalyticMS_clip.tif'))
# clip_img_file

16:50 - order 9c146ad9-20eb-4407-83b5-7ad963549f7c - state: running


ClientError: Maximum number of attempts (200) reached.

In [33]:
order['id']

'9c146ad9-20eb-4407-83b5-7ad963549f7c'

In [36]:
order

{'_links': {'_self': 'https://api.planet.com/compute/ops/orders/v2/9c146ad9-20eb-4407-83b5-7ad963549f7c'},
 'created_on': '2024-05-30T11:55:21.873Z',
 'error_hints': [],
 'id': '9c146ad9-20eb-4407-83b5-7ad963549f7c',
 'last_message': 'Preparing order',
 'last_modified': '2024-05-30T11:55:21.873Z',
 'name': 'full_airth_SR',
 'products': [{'item_ids': ['20221125_111318_95_2413',
    '20230209_111628_64_240c',
    '20220305_105837_49_248e',
    '20221128_110618_08_227a',
    '20220920_105809_51_2495',
    '20220215_102554_23_241b',
    '20220321_102740_31_242b',
    '20230126_110351_81_2485',
    '20220227_111348_12_2274',
    '20220227_103256_21_2430',
    '20221205_110047_16_2484',
    '20230125_104939_27_2276',
    '20230126_110127_66_24a5',
    '20220910_110421_57_249c',
    '20230128_110445_35_2480',
    '20221016_110338_23_2461',
    '20220327_111309_86_2416',
    '20220924_104649_13_2251',
    '20221206_111226_60_227b',
    '20230126_110354_09_2485',
    '20220324_111806_41_2403',


In [37]:
order_dict = {"_links":{"_self":"https://api.planet.com/compute/ops/orders/v2/9c146ad9-20eb-4407-83b5-7ad963549f7c", 
"results":[{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.367Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImN4YjcxTjRwc3R1OUtsaHprMlA2R281VWt1a2FKMlQ5UGhxTUN2QmlsUWlQOFplRXUvSnB0cGlIK0hRcEtKYk9Ya055dGdJMVAvMHoyZUZKTGhiVllRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDQ2NDlfMTNfMjI1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjQ4OWEyOTYxMDNkOTE2Y2VmOWQ5NzVkYzRjYzIxYWE5ZWFiYmU5M2U0MjlhODM1MGQyYTMwZmRlMzQ1NjdlYjMxOWNmODVlODY2Y2U4NTI3YWM4NGQ5ZWM2NGRiZjczZjA2YjAwMzY2MjI4N2FiYzRmYWFmOGZlYjgwNmI1ZTU1ZTA1ZTUzM2YzZGI3N2M5N2Y4YzBkNjJlYTNkMDI4NTk5MmQ1YzkyNmExZTg2NTA5YWM3NWNiMTgxOWMyMzlmZTUxMzRmMzgzZmU0YmIzNGU1MmJkZWVkZTZiZmYzNmY2MjRlYWNkY2EwMzUwNWM0NDY5MDlkMTQxMzdhMGQxNTZhYjI5NTEyYTVhYzU0NDkxYmE1ZDM3OGU2YTI4M2ZhOGI4YzhmZjVlOGM5NDE3NmQxOWE5ZTNjZWUxYTNmNGQxM2JkNTY4OTExMDU5OTNjZGM2MTg3ODJkODkwZDI5ODUxZWYyOTA2ZmEzZmIyM2M4YzM0NjQ3ODAwNzU1YTc0ODYwYTRiMTlmYzA4NTFlNGU1MWNlZjUzNmM1YjFlMThlODBkMDAzMzg2MWQxOGQyYzJlYjI1YzVhNjgyYzhjNmU1YzViMjU5YTFmYjg1Nzc4NmQxYTgyNmU4MWE4ZGMzNDFjMTFiM2M2NTU1Nzk1ZmQ2ZDYxMTBmZDg1YzM0NDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TmmPwIcZG7ehMHM-kOPEzm19xqtSxNZDTb5Y1QQTVIdI-FaqpXiJnL8fvbZWK9lA-hhzcj41uDEuhR7v_JFRZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.369Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkFzTjcrMHhwbWJJR3ozRXNickN3NjM2SGVVWlFjcWhNckVaQ0lheW1oUWpmSWpIR1JrMlVKeVRSSE1ZL3BIVko3NzkrQkR6bTc5SkVwcU94RHM4clNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDQ2NDlfMTNfMjI1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTE5OWMxNjY2ZTBmN2Y5ZDczZDU0YTdlNjc2NDVkYzYyMjliMDc3ZmVhNDEzYjRlNDE2YmE0ZGNmZTU0NzFjMDkyNjdjMGM4YTQwODc5NTk4MDdiZmVmYzVmZTdmM2UyMDllODY3ZmE4YWIyMDNiNTAxN2Q3OGJhNTg5NTYyNzM3YjUyY2IwNDdlNWYyNWFiMjZhMzcxZjAwZmVmZWY4OWQ2YjZhZGU0MzJjYTI2NTE3YjBhNzQ3OTk2OGVkYWVmOGM3YjgxMjQ2MDYwMzI3MjI4YzNmZDMzMzdkYzY5ODUxMTJjNDFlZjE0ZjQyNjliNTBhZjc3MTgzYjY3ZTYxYmU0YzZiM2U1YjAzNzFlOWI1ZWYyNDUwOTVjYmNhMDY3NzRkZTZmNDQwNzQyYzliNDgxMzA5ZGUzYjQ3NWYzOGNlZTRkNGRkZTIwNTBkZWRjOGQ3M2FiYzc1NmRlMTI2NGJkYTQ0YTdlNWUwMzZiZDgxODc2ZmE5YzA3NDRkNTNkNTViOGU0ZDhhYjc2MGFhYzYxM2Q2OWViYTdjOGIwYTIwN2Q2MzBkYjVjZDA3OWIxN2I3NTI1MjExZjgyZDZhNGJlNWFiYjhmYjczYTQ5NWE3MDIwYzg4Y2I4ODE3MWNlZmQ1YTEzZmJiNWRkOTI5NjNmNzAwOWI1NzdiY2Y0MTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cTcDxqRj-WobI_rBkxpEVhl8UNqdzYGFoXW65ZyPk99iPjpW_PEVjESJqSdPsuvq1JdsL1KNi7Qf0mkkxgGnRw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.372Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InFMWURqbjd3MXZpemlIMGQ5K01QaEwwaGx3aE8rOHlGUWpCVjNQb2tPZy9VZERzVlJZVnZHb2szWjN6UUVVTVNMbzB3cGFKRll2VDVSN0ZEZ0p2UUhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDQ2NDlfMTNfMjI1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQyZjIwMDViOWNiNmY4MTVkYTUzNDY2OWI4YzYzNGJkYWM3MmZkYTM0OWU3ZWMzYzEzNjkzOTNhMzRlNzI1NTkzODVjMjFkOGY1YmExZTM0NmU1MGEyYzBjZTAxMWFiOTRmYmFlMDA2NWI0NTk0NjY3OWE2ZmU1NzE0NWJhNGUzNzkwZjNiZWVmNWY5ODk1N2RkMTljZjQ5NzM3ZTMzMmY4MmFkMDBlYWQwMTE3MWQwZjg2NjBjMjRkODkwOTRlZDU0MWUyYmI5YzE5M2YzNmMzNWRhZjM5MmQ4ODBjYzg4N2Y2NGFmNThiMTRlYWM4YjE3YmQ1YWYwMTc1Njc3ZDc4MTEzYTJhZjEwMzQ2ZTUzYjNlMDA0NDNmNGNiNTU3NDQxNzc2ODQxMzA2MTIwZWM1N2Y1NDIxODI5YWVlNmRiODQ3YjAxNmQ3OGE0ZjhkNmNhMjNiYzY0NTQzYWFmNDhmOGI4ZDRhMzM0NTJkNWExOTA0ODFhYTlhZTgyMzdkODZlOGE2MGFjYzU3NjM0YjI4Mzk5NWJjZTEzYmFhZTA1YjZmODFhMTJkN2Y0ZjQ2ZDhjMmQyN2MxZTg5OWY1MjNhMGE2M2RlMDI2YmU4MzQxYjBhOTFhOWYyYjVmZDA3NjI4Yjk4M2U1ODQ5ZWRjYmEwN2JiMWY0YTQyNDJhYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._Vzbcib-8GGI_UpOqM-z06FdEEYvUq-NnaomXc6nNWDTTBTRdGoPTLm4WOeRLhkSS47CCfZ8cpzsoP5RWGtmlw", 
 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.375Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InRQVEpRbG1Qb0xWTGZNQkl5ZWdhNFVoQ1hocDRoR1U4N0U5RVB0Sjd3VTNiOFRZeUZhcFF4ZnpuVEhDMjA1Y1RTSWUyR0ZTbjhLODBRWmVGdVdkWERnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDQ2NDlfMTNfMjI1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjYzMjAxZTY2M2RlYjc4Yjc4NTkwYjRmOWZjNWRlNGI3ZmRiM2U1MzRhMmI3ODI3Y2RkOWI4NmViMDE2NWFkN2I2M2ViZDBjNWUyNTdhNmZkMTZmOWU3MWJiY2U2N2VjNDdmZGYyZDhjYTc3ZmFiNjEwZDdiOTliYmQyMjBkODRiZmUxNjVhOTJkZDc1ZDAzNGMxYTdmMGI1MzZlYzkyOWE3NmQ4MGYzMWU5MWFkODBjZWM4MzFhM2Y2YjI5YThkYjEzYjk1Yjg3YmNmY2JlZjI0ZjI3YTk3NTBjMDgwY2MyOWY0YTAwYWExMDAzZDY3ZTE2MzU5MDU2NGVhM2YwYjJjYjUxYzhlZWQxYWIzN2M1NGQ3ZmM2OTI2MmJlOGE3OTYzNDAxNGQ5YmQ4NmJkZmUyNGRiZTM3YzViYzZiMTcxODBlZjZmZWI5MzgzYWVhMWVmYzE2YTQ2NjNhZWM1YTIxOTRjNWI0YzJkZmJmMjM4MWY3NjdlZDc2NmYyMGEwYmNjZDMwOTg0ZTY2ZmM2ZjUwM2E5MTY3M2YyYmEwYjFiMWFmZDdhYjQ1MmI3MWNmOGVhMDhjODVkZDMwYmU4NWUyMzkwNjgzYjVkZDM5ZDk4ODY1Njc4OTdlMmQwY2Q0ZmU3OWE3M2JhMTY5MTMwYmI5ZjRiYjg2NDE5OGY2MjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vDZ-o0REMdTt2svjmJdopV88IAiSW25n33JRVi4QcC1WQqO1MeLHHz18Nj_ds6UzV6X-8zA3qZIruPeBGFfA9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.377Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjZaV1NBazlQdlNZUFp0WWg4M2tSdDUvS2ZkT1U3anZVSW1oQXRvYmREaXAyZm1Zc1ZxejJzMEljZHZyT3NLRDJPRk5lRHhTbEQrSTNVMnFIV2tkcm5RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjZfNjBfMjI3Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY0Y2VlMDZjYzVjN2E5NDM2YTFkMzZlMmZjODAyOTNjNTgwYjdjOWYzMzdiOGYyNDkzNWVkY2Y4ZjhmMjM0Njc3NzA1Y2RhM2FlNGMwNmE0MThjMGZiODZhNzBkYWY3MTYwYjFkYmEwZWM3ZjZjMWUyODU2NWQ1ZjcwOTY5ZTg0M2JjMDQ3MjRjNDc1ZGVhNDBjMDE4NzAwOGE0ZTA1YmY5Y2NiNGY1NGNjOGIxNTIxZjdmYzZjNWFiNzUxMjRhODg5YTE4ZDVhNmU4YWUyNTE4NDRmN2Y0MGIxZTBhMzY5YWRiMzRhMGQwMDAzM2UzOTRjMjczMTFhZTFhNmU4OGEyMjlkOTBhZjdjYjRjODVkYzkzMmZiMTk0ZTM1NTJiOTdkZDI2ZGQxMDQzZTkwMjY3ODRiMTJiNTJkZDY4NzNmNDIxYTQzYWU1ZDQzZmRkZGQ5OGU0NGI5M2QyY2YxMjBkZjllZGRkMGFhYzAxYzM0YzMyZWRjNDNiYzI2YzVlMzNlYzMwMjcyY2YzOGJiYTllYzI3OWRmZmJjMjdmNGIzOTZhOTA0NjY0N2M5OTE4MmFjZTYyZGU3MzZhYzhlZDAzZWQ5ZGM3NWM5Njk3ZDMyMzFhZDFjYjQ3ZTIzOGFjODZlYmQxZThiZmYzYzdiZGY1YzRiM2ZiYzYxMzczYTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PyhNQu7Rshik18zYmeuL4C02DSSnVmGFo-aJZbXn1pGFGH2cM57g6jDnKtll3IUX64d13EJAY0wKMWsEFQFSQA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.380Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InoxMHZ1cUd5RUI1TjlzY2gvbHpNdTQrNkZQcVdKbTlkZ2hKTXB0OGRiblFrMkppWUdtRmt2U1AweFdxeWtrNVpTcno1dUY5eSsvUFF3TXF0Z0NyeGVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjZfNjBfMjI3Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjU0NzI0MTFlMjAzOGIwNjZmZWI0OWI0NzYwZjJjNjM5MWUwMmRhYTFjMzE5NWUwZTczNmYyNjEyNzc1ODQ5YTRjNjEzZTgxYzYwMzczMzkxN2Y2NzdkYmJjYzEwZjUyODJkY2VhZjIwOTc3NmMzMmVjY2ExYjI2NWI0MzBkYzRlODhiZmUzNTEyY2M3NTU0MTlmZmM1M2Q4MmYwOWZiMjFmOWE4OTczMzQ3YzYyY2U1NzBhMzMwZGE5ZjRmMmI0NDFjNDRjMmY4YTBhMmM5NjA0YTE5YTU2YmNhOTUwMDE0MTRlNjRiYWJjYjg3ZTJkZDExOGE0NTdiOGM5MjdhZjgwZDE4NWE4MDI0NDMxZjE0YThhOGY4MjcyYjUyZTliNjNlY2IzZDMwZWM4NzRjNTQxOTVjOTdmODExMDIwMjg3ZWRmZjhlZWRhOTBiNjViZmM1NTM1MTlmY2QzOTkwMDAyYmVhMDBiNDQyZjU4M2NhYmNlY2ZlMDhmZDQwODE1NmEwMDE0ZTFhNWI1M2U5MjI0YzFhMDFiOWE2MDIzMTRmNTlkOTE0NGYzZDJmMzFiNWY2MDU0NTdiMjU2Nzg2ZDQwZTc2NTU2MmIwYTVmOWYxNTFhMGFkMjA5M2JlYmQ1Y2MwNjdhYWM3YTAwYTA5ZmRjNGY1NDc0ZmYwNTEwZGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jvImoEZRdUq5ddA7J4C2M9fQNdXhNp6ze_u3tJk6v3SVapXVVFuxNflt4NjXiJlMKYpWSf1hWfr89_ZrfV9lmw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.383Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ijg1TWxTbjB6OVQ1NnpXK3k0dEdFeHROdzBQZ05JYmFBZWFvRElBZGh0c1JkdWJldUd0SjQyMERVQ0swcHkyMFZvUGlCemxxbi90OWtkMUxGajh3ZGpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjZfNjBfMjI3Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWU0ODkzMzYyZGFjYWM0YjEwNWE2YWJlNmZjNTQwN2FjOTMxZjRlNzU4MGY2YTliMjI2NTlmNGEyMTgxNjFiNTk4YWU1NmZhNmIwY2I4YmE3MjIxY2JhNTczNDEzMGNiNjY4MmYxN2FmMWU1OWE4Yzg4NTNmNjcyMzdiMWIzZTAwNDJiZTQ1NTcwNTdkYzk3NzQzNzY0MmQ0ZTc3MjM1MzRlNmMzODFkMDMxZGQ0ZjFmNjJlMGE5MzhiOTUzODljMzIwY2IwNTI0NDkzMDY3OTllMGExMTMyZWVmOTVkZjE5MjdlMWJjZjg4M2QwNjQwYmJkMDBmMjNhM2NhMjkzOTMyMWRmNzMzYmViNDFhOTY1NmVlMzVjNWQzODJlN2JhODJlNGM2MDExNzQ3ZWU4M2E4MzE2ZDljNWIzOTcyZjhjNmMwZjVlMWM4YmU3NGZhNDg3MTU1MzM1NTMzYjZjMTU2MWRmOWRjOTllZmM4MDQ4Y2VjMjg3YWI0N2RjZDdkOTI5ZjFhOWJlZWUxYzNiMGUyYzgwZjQyOTRkNjUyM2FkMTAyNDlmNmUxYjVjYTk4ZjY4NmUyNjhlNTI3ZTI5NjE1OTFkMjJhMTJhMjVmNTZjNjI4MjQ3OGU1OWVhZGMwNjYyMmE0NGRiMmRmMjFkMWM2MjU3Y2IzYzMxMTI1MjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8CtzKlDqCqJF4xOD3UZf4gXsqPHwqxx1Y_GGvtOEDfayATvap1Wv02DOXbUUlWnpqLgQuOqUK1BETLExkD_OwQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.385Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkIvU1ZVcnc2K1E0VmlycmQ0elQ0bElQM0xDL2t2Si8rbzlHRDJDMkJ0QW1ZNDZ5MldVYlBRdWxKVWJqT0NoaDM5ZUdlOVJNVnljU0RGbEhoWG4vaHZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjZfNjBfMjI3Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjU3MjQxZjRkN2M1NTdkNGU4MGVmODk0Y2JhYjY4ZGYzZjFlNjc3ODdjM2Y5OTVkYjE1YjE3NDFhNTk5MTM3NWE5NmFiMWFjNWUyODQwMGRjNmNmNmVkMzg3NTA0MDAzYmYwNDUzNjc2YjBhYzFiZDU1N2NmZDZiZDVmOTEwMWMxMWIzMTU3YzlkOWJlMzJlZGRmOWRkNzZmMzlhZGU0OGYxN2FjYjI2NzBkM2VhYjJiMTEyMTI2MTlhY2VjZDE3YWI5OWViY2UyYzJhNjJiMmExMWI1ZjY1Y2U0YWFkZmNlZGI4NDhlNzBiYjA5YzQ5YTlmYTlhYTFlOWEzMWVhNmIwMzEwNmVkOTMyOTBmY2FjZDhjZjJjMjA4OGVhZjY2NWYzODNjOGY2MGQ5MzViZTI2OWYxOTJkYWRlZTM1MmU5YjczNjM4N2M1YzAxY2NlNThkNDEwMTk1MjhkOTEyMzgyNWFjMWE1ZTAwYmIzNjZhZmMzYTI3NDFkY2JjMzA0OTRiMDVhMDdlNWNkMzRhOTBlMjQ5YWU4OWVlYTAyMzI4MWI2OWM5YWM5MDA2NGFlZTAwNWM5MzU4YmZhYjVhMjY4NTJmNGExMWVlZWI0NjMyYmM2YzkzZDYzMWJiZDUxMjMyNzI0YWJjNDY3MzQ2NzEzODkyZjA5ZGUyZGY3OGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tANhtY7EahWaABSzX4CqSsDvmokqT7b8AK0WwWwFLfDuMvnCY5Ct3SWxSm4-4kv12r90TdT8ItQWNxiuM-AGTg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.390Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ii9jSllCTkRkN0pOR1lGTzdFUmZwelRkYlF4N2xJRThqRTF6Ny9YTHVzdXNPbGd6WFRrZlgwdFo3MndOdURzZkdheUt2WVhTYVRzY0dJUEdtNERrMU1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDNfNjdfMjQ1Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzczOTU4MWVjZGUxYjU1ZmI0OTAxMzBhN2RjNjUxODYyMjhhNDZmYzk2ODkyNWI4MjIzYzhkZjA2MGYxNjVlMDIwM2FhMjZhYTZmZjRlZDQ0NDZjNGZkNTdmMjc3M2VmMzY2YTU4NzE0NmRjN2UwYTZhMzM1YWZmZTVjM2ZhZWE1ZjIyNzBmYmVmMDZjMzJjM2I1YzVmNGQ3ZDIxODFiNjQ2MWUxZDMyMjY4ZGNkYmJlNGJmMjE0MDhjMzIwNWMxYzlmYTAxMWQ3MTAxZTkyMTlkYjA3MGI0YWM3ZGFiODcxZTdmMDcyNzIxNTY2NTJlNjJmZTg1NzEwMTczOTg3NjBlM2UyNzgwN2NjMjA0MWRiNWY3OWEwZDUxN2VlYTk0MmU0ZjVkOWU2ZjU4N2MzMjYwZTg5YWJhMWZiZmIwY2Y3MzUwYjhjYmRjYzIyYWQ2YjA0NjBiMTcwYjk4YjliZDZhMTAwMThlYzhlMGExNjljMzg0NWRiOTIxMjNiYTFkYWM2MzY0YzIwN2JiYWY5Y2RiM2EwNmQ0OWZiYjg5Yjk2YjNmNWFlYTFjNTZlNGQ3ODkzMzVlMjAyNDdlNDM3YzlmZDY1YTE1OGI3ZDZlMzE3OWE0ODUyZWY1Njk3M2Y0YmMwZDQ5NDVhMGQyYTY4OWEwZmZmNTM3YTQ1ZDNlZTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yHQofCFsdb_dXJSIsTe-eLOgZ4UsYAcwsoSIUbgucC8FjZ-cLY-tFx91GgQ8UayalJ8VGRywGsCQi1_qfkFObA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103503_67_245c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.392Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImdhZDdmNzBOVExTdzhvbDNZV01rOVR6eDkrQi9vUlE2WlpnVS9JemZQdWowZ09sUGV3M0dPR2dwMTNNTG91YWp6ekM4OGlvT211b1ZOK09HZ1RhWDlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDNfNjdfMjQ1Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTVkNGE5YzczNDhhMGE3MzM1YmFlMzQxMGIyNWNmODk4ODYzOGM0YjFiYTIxY2VhMjE4MWNlMGE5Mjc4Y2M5NDAzYTQ4NWZlNTJkNTc3YzRjZTU3YzNmZmNiOWQ4ZGE4ZGJlNzRmZTAyMjg4M2UwZDM2OTZjZDAzYzYzM2RmMTA2OWFhZjY1NjVjMGQzOWIwNTIwNzM2MTE4Y2ZjMTUxZGY1ZWVhMmI1Yzg0MTU3YTQzY2E5NGNmNWQ5NjMwMmM4NThmOWYwMmI1OWVjZDZmZDBkYzM2ZTdhZDY2NjJjMjExNTVjNTcwZjA0YTNlZmZkM2U2ZDJmNzRhMzU1YzM1YTY1NWE5NjQ1MDA3NDk5NDdlNjBlZmZkNWQ3NjUwMmUwYjBjYjRmOGUwMDYwYmEwNTdiMzZiODhkZWNkOTYxNTZiZDZiZmEyMzdjZWE2NDBkMzlkMGIyYzY1N2E1MTc3NmZkOTAyZWMzOGY1YTBmOWU2ODNmZGQ4YTJkZDdlYWExNTkxYzQ1OTM2YTBiNTE2MjQyMjhhNWM3ZDNiNTVhNzJmYjkxNTQ3MDdhY2QyM2MxN2NmNzhhOWU1Y2EwMmFiZGMzNDk4ZGI2MmVhNjc3OGUwZDVmZDQ2MTRmNDNmY2MzZmFlY2U5ODFmODIzYzFhMGVjM2NkM2M0MjMxYThlOGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Wgb5PYkj9Us11mZEr8HLxQ-wktHIcbskcCMNKbZ6n-_ZxAHn1wr9HzA6HGtOWAtAT49Aqy1N7Of7EhnV-8u7AA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103503_67_245c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.395Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ii9TUElFNHk0UW9KMHIzN21GRHJpL3RLQjV3UDRXVWhKVlVSMklMNVpQVDhwWEkrVDNURVp6MGt4U2VQdjFJWktXRjRMdU8yVWJFcm5Jd0h0T1c1TjVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDNfNjdfMjQ1Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjUzZTNiMDIzNGY2M2FjM2FlOWQ5NmRmN2JmNTJlY2JmNTk0MTZlNWIwZjBjYTRhZjAxNjIxZjllYmJjZWFlYTg1NTQwOTM1ZjJkNjUwMDRmMjkyZmU2YjhjMDY3YjllOGI1OTg1NTQ3ZTM3ZWU3MzlkYTgyZmVlZDJmMDAwNGQ2Y2E0Yzg3NzY1MDU1NTRhZTA0YWRkNTA1NTJlZTAwYzBmMjY1ZjY4YTQ0NDdkMzFlZTQyMGJkMzYyYjM5OTMwMWVlZGNmZDkxNzNhNzlkOTNlYWQwNDJmNDVhNjBlMTFmOTdiZWUwMzhkMzhiZTI5OTUzOTM1ZDJiMjIyMWExYzcwYzhkOGNjZGU4MDBiYjJmYjdiZjNiYzQyMzQyMWMxMTliNjZhNmQ2ZGJhNTY2NWYzNzU4NWE3NjU5NWM2ZTI2NjBhMmVmMjA5OTM4NDQ4MmNlZjBjMmJkMWY5NDJlMTA5MjdmYmM5MWNjODc2MjViYjMxYWNhMTlkMWY1NGFlNTVkZGQ3ZDM2OTJmODljNTdlYjFhNTAzOTY1MGU3NzdkOGFjMjJlZWMwZjU3YzM1N2E2NTY2Y2ViYmMwOWQ2NmVmYzU0ZDdhNWM2NDZlNGRhYzI5NWM4Y2VjZDIyMzY3ZmU0YThiM2E0NjVjYTY3MGVjZjNjOTcwMTNmZWFiNWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mH8-gWuppHp-WSIe7N-ksPQuvgute2sbyfpxAjSVjetmWz2F9vIYvmT9n3hzth24O-yhwx7K4NB59odK4l7qpA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103503_67_245c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.398Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im1lYXREdGk3TjJQa3BaNFBSb1dRNFh3ay9SWm41aEtzUHQwYzd4YTkwUENQUStyZnhDZlptUXZzYjEra2ExdWlua281dEY4YUh2Zmg0cjBOUUpUa01nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDNfNjdfMjQ1Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjJmYTkwNThmOTNmOWQ2ZDBiNjk2N2VmOTcyOGU2ZTEyNjY1OWEyNmY0ZWZjZjAxNzEwY2ZlYWJmMzBhYTllYjVlZjhlYWVkZDc2MDlmZTQzNTVkNDI0OWI1NGEyZDI0MGM1NzNlZjFmMTYyOThiOWVjMTEwNzBjNWY1YzhhNGJmMWFhYjk4ODFiYTYyNjVjZTNhNTc5NDg3YTRlMTZmYjY0OWFlZTNmYTYxOTgwMTExNGFmZDU4NGVjMjAwM2U0MjYwY2NjYTYyZjM4Mjg4YmRhYWQzYTJkMWU0NWMzNTMxYTRkZmI1NGM3MjcxMGRjN2Q5ODVjMWZmYjBjZTM3YTNhOTU0MjgyODcyMmI0ZDljOGM4YzIxMmYxNTI3NzM2M2RmNWZmNjkyMjIwNjQxNzEzM2M3MTA5MzkxZjM2ZWUyMDg2YjNiYWZmMzlmNTljYTRhMDcwYjYwNzgwNDI4NjgyNjg3OTE0NjY2ZWMyMTY5MDRiMzZhNmMwZjY0Y2ViMGVmYjQ5NjBhMjBjNWFkYjA1ZWQ0YTI2YzA3ZjkzZjRmNTA3NTZiMTcyM2RjOWI2YWY4N2IzOGFkMzJiYjhkNDUyYzlmOTJlNzlkOWFmOTExZGE3OWNhMDg2YWQzMDQ5MWQ3MTY5N2Q3OGUzM2Y1OTZjMmZiMzcyYTc5ZTE1MGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3QHWAA72MbYRfAK7AqKRApoZZlVXRGyW5ogsFVex8ZGZGQlDIhOoEzE3ZQgD39SeicJYGjK2cgn5TxVjNlPTIQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103503_67_245c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.401Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InlWNWl4S1BiNkpsZ2NRV1JDU0NsRkFXTXlFb2xwVzBYRnU3RXZmL1NYUU1lNG96OGpOcDBtT3FzeTV2KzFjYW40bGJxRkg0c2czYTZmT1RrSTF1OUhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNV8xMDMzMTlfNjlfMjRhYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmI2OWEwZTgyNWQ0YjliYmZjM2MyODc1YjIwOWIwZTQ5ZDg1NzgwMzhiNzUwOGJhZmQ3YmFkYTFhOWQyYWE2NDgwZjFkMjk4MWRmYTFmZWNkMmU5NTY1ZDgwOWQ0M2U0NzIwYTEzNGNmZGQ0NTUzZWE0YzhmNjhkYTdlMDhlYTQ1MDc3NTFjZDA4MTZhNmRjNzM3MWU3YThlY2Q5MTNhZjdhOWMxMTY2M2VjZGVjMDE2ZGJmYzBjYzdiNThjOGJiM2U4OTE5ZTM1YmI1MGFkYjFmY2RjZDcyMzFhNDhjOTc2MmMxMDg2M2M2NTQwMTBjMDNjMjM2Y2M1M2VmZmY2YTkzNGFlZTFkMWExZTljODgyODVmZjUyZDRlYzEzOGZiNzlmOGU2MTU0ZDQxNzIyZDVlNTQwYmU5ZmY1NDIyNDNkZDJmMzE2NzAyZGNiNjczNWRiNTZjYzQyMzk5ZmNjMjA1Mzk4MmFkMWQ4N2I3YTIyNTM2ZmQ1YWYxZmQ0YmQwYWE0Y2M1YzQ2ZGY1Y2QxMmJjZmFiNDBkMGVjZjk0MzQyZmUzNTNlZTNlYTYzOTg0Mzk1NWY3NDg4MWY2ZTUyNWU5YzJkZmIyZDk3YTRjZmQ0OTg3ZWNkNDNhMzk4ZGE4MDU2MzFkYjJiNDFmMWZlNjk1OGU1OTM0MzUxZjYwMWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Pg9vUl1M_dn0Meg3nJ4CQOqmNcXVgQQthQ7yg-vOkMyg19SGz1NN6r-KznGCt36KwkyLza41xU5Ck5EirkjJLQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230225_103319_69_24ab_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.403Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InVWeFV0cnZ3aHRQcnhZZUNVcmk4ZjlzRmd0NS9UOVN3aUdlWVJIY2VMRk0wSXAvaHZVYngwbFViQ0FKd3ljQTh6Vlc3d1FkOXRyOVV4c040V2JqZHN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNV8xMDMzMTlfNjlfMjRhYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDRhZDFlMGViMTg5ZTU5MThkNmQ3ZDg3OTQyMzI1NWQ3ZWFmYTA5ZDIzNjlmMGM0MGUzNjJkMGM5YzA5ZjI4NjUzYWFjNjU1OTlhNjdlODVmMmExMTY0NmM4YjkxN2MxNmUwOGU0ZmFhM2FkMGMyYTgwNmJjNjU5NGFkNWY4MzkyNzI5YWFlZTE4MTg1MDAzOWE0MTZiOTRlOWE5MWM5MzFmOTRmNDVlODgzNDM2ZGEzMjFiMzUwYjNiMTFjNDAyODU0MzJiN2IwYzE1Y2Y5NmU5NTNjOGI3YmNkMzNmNmRhOTBjM2NiNGViYTZlNTYzMTVkMTMxNzQ4MDE0NDUwYzQ3ZGQ5MGU1MjhiZDU0YjM4YmVjN2I3NTdhMTBlNmUwNzcxYTUzOTM0ODlhZWIxMjk3MjQzMWYwYWUzNTFlODY5MGE2N2E2MGI5ZjI4M2E5NzdiOTJiMTQwZmFjNmE1Y2Y3MjcxNjcwNzUwZDQ2YWM4MDU2NWVkZmQ1NDY2ZTM5ZmEwMWFhOWViZjUzNjI0ODc5YjMzZWM0NWUwY2Q4ODljNDRjOTJmODc2ZWI1NGRlMGI3YjY3YTYyOGZmMTc0MTI1ODMxNjYyMWFkY2RhMjdjMjNkMmRiYWUyN2IxMjdjMzE0MWQ3MDMzMDBiZWQ2MjFiZjliNTdiM2RhY2E5ZmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.swbHNnMxiwc8sdGgzzscak-iBx4tB5qOlyjUpM0AD5SPUUiaOeNjvQWqRYtDPddJu2J1bo9sexZjVRvSikji6g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230225_103319_69_24ab_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.407Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImZKUURGdy9ZK0ttS3A3Q05xalVFb3Z3cGovRHVrWEpzQnBmRTBLbmloYldJc0xWcEpvQURlL28wRGFOZUI0NXRGVXRjaEtlQlZJRkZFUHkyZmZsVWx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNV8xMDMzMTlfNjlfMjRhYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTg1NzY4NDIwYzRiMGRmYzIzNzU5ZTc0YzRhNmRlMDAwY2U0YWM3MGFmZTYzYzQzNDQzNmZiZTIxZjhjM2YyMTIxYTFiNTY5OWYwYjQ5NzUxNTFjOTJjMGJhNWJmM2QyZGQwOGVkYzU0YTA1YzkyNmJhYzc0NzUxYzc3Zjk2YmVmYmJlM2M2MDY1ZGRjZjY1YzZiMTA1MjJmMmNhYzdkNDBhM2QxYTE0ZTNkNTlkOGRlODFiMjlkNTg0MjlkYzcyMjJkNmYzNDg1MzAxYzM1NTcyZWU2NzhhOWQ3ZjQ1NTExMzA0NTExNzQ4MDU3MGZjZjA0NDE3MzA2YmIwNWZmZmU4MmJkZGY2ZTEzMWMyZGYxNzEwMWFjNWM4MTg5MTY2MTIxMzY3NjAzMWM2MzVjZjFmMGE2ZWU5MDIwZTg5YmE1YTEwZjYyYTlkMjlhYThmNTY2ZjdlNzIzYzU0NTRkZWUwZTNkZGU4N2E2ODU4NjZhMmMzZGI2ZjYzMGUyNDhkMzM0YWRlMGNlN2NjZDk5NDQ2NDNiNzFmYjQwMWI2NGY0NDMwNWYxOGI3MjAzY2Q2ZjZjNDIzMzg3MmZkNmJmMDllMGI2NzVkY2FkZmM2ZWU0NDVjOTAzZmZmMmQ4NzM2ZDVhYTlkMTI4NDdlNGE2M2JhMzc1Y2ZmMTVlOWUxZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8x6aJb2Sw6RDCzte1Prvdh7AAGGNGPK3KXr9HWVATYkFZd_uvD8f7tO-s1ZbL-SufCauSaxIEBOXN1L93WKl9g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230225_103319_69_24ab_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.410Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ino2bnV3VTF6QlRwOWpLZEVTejJPL09EVXhmZWl4VVNHdzNZTDBLTy9GWWY0ZzJyOGJOT3pndld0MStXVGt2N1lSa3Vlc3dpYXJRY2MxU05sM3B2eDFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNV8xMDMzMTlfNjlfMjRhYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWU2NDY5NDIxNjBiZGNlMjhhZmFmODVjZDNiZTllMDZiOTM2YzdmMWMzMGVlN2EzMjhjNDNmZGVhODFlMWJiN2Y5MWZlYTYwMGU4NjI0MWJiNDMxMGIwZDAyZGM5YTJjY2Q4ZjNhZGY1ZjQ4NWNmN2IwMjE1ZTliZWQzZDkwZTFkYTI1M2RjOTFjYzNhZDdlOTM2M2MxYzVkYmM3N2IwYzUwYzBhYzZhOTg1ZDA4MWUxZTlmNDk4M2ZjZTQyZTJkZWViYjBjMDdjZDY2NjNlMGY2NWQzY2U1NTUyYTViOGJmMTc3N2JkYjIwYTczZTYzOWZjOWUwNmQyZTRlOTUzNzBkMzZhYzQ1OWEwZWZkNjVmMjEwYjFjODNiYWU0YjQ5YTMxZDQwZmIwYzI1MjY1YWEwYTljZWI1OTgxMDIyZmI1OTc5MmJjNjVmYWQwYzdkYTAzY2ExYTJkMjk3N2NmNzZiZGMzNDM1Njg0ZGVkMzM5OTM1ZGUyNDI0NWIyNzU0ZjI1ODQ3NTJhMzc5YTMwNWFlMWY3MjVkMzc4YThjYmFkNTc0N2UzZjgyOGIyYjI4MWRmMWY1YTQ5MzA4MjYxNjk3YjBjMGRmMWY0NTFiNWYyNDE0YjA3NmI2ZjU1MmVhNDA2MTRiYjE5YjA4ODNiNWRkYzI2YTRhYTIwOWQ2MjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BiWNUzjYMbs-BJmRXwzHwBgOEY_laBePoNQNnnHqH9xBDft--TDZp1G9eZkIW3B1jh3eWipfmi3aAuhcV6IVOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230225_103319_69_24ab_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.413Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Inp0V0hxSDVrYnB5T1FzQlNOcnRiSFlzeU9tbWw1K1EzRXF6S0hiV0h5cTNEWFFDaTZFWGk1bys2dnR5THQ0Nks2T3dBbEE1U3lPNldKN2dQTnBiQndBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjNfNzNfMjQ5ZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDM3ZjZkZWQ5YTM1YzlhZWNiNDcyNjY5ZGFhNWJlN2EwYzg0YTNiMWM2NjMxODg1NTk0ZGMxOWFkNzE5YjlkNDg2ZWQ4YmM0YTk0YjJjMjg5ZTliMDRhM2QxMTc0OWI0MTM5ZDgxZmM5YWUxOWQ4NGMzMGVjMWY3YTA4ODlmZGQyNzMwZGY5OGJlMmFlNjVhMzA5ZDA1OGI5NDcyYmFkZDQ5ZWVkZDRiZmEwNjA4ODgzNmY1NGMzZGRhOTRkMzRmNjBiZTc1NDI1YTZmMDc5ZWFmZTBiOWViZDk4MTA1NjkwYTQ3OGViY2E0MjBiMDQ2OGU5ZjI0NjQ5N2JmNmMyOGY3NWZlMjk3NWU2ZWRmNmFjMmI5YTQ5MTViOTNlMmIzNmUzODc1ODI5OWRhY2QzODdhOGUxYTkxNWE0NzMxZWE1ODZhNGM1NWZkMWEwODFmOGM3YWJiY2QwMjM5YjA4M2VlNGIzNTFlYWJhYjFhYTYyZWE0YTUyMzgzMjM4MWNhMTM5YmFkZGVkZThmOWRhZTk4MTdmYThkNzFlOGQyMjhmZDk0Y2ViNzJhMTVjYjU1NTExZWYzNjM3Y2E1MzQ1ZGI4Y2I5ZTMxNTQyZTJjNGJhOWRkMDczNjJmNjI4MzYyM2MwZTM4NmQ3ODY0Zjk4MGRkMGUzMzlhYzgyZGMyOTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GWF-qlDFe_z-GmpahvIgRROdoeAecRnp6xIFUZSzavnpvgxTKNMrOvt-GghSdzpTaMQuvgZKXPp3SHLV_aWw2A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110123_73_249d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.416Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjZwQk11aVlhYW0wQnpaVlVhNy9tSUJOT01nQ2NJVXk0LzF3allyZnlXSXFmaFczTXpNenloYzdqYlplbnBLemFqbTd0UDBVNG9ZM2k5VWUzYnpOTTZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjNfNzNfMjQ5ZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE4MmFiY2UwYmY4ZjUwZjk4MTQxMTZkZDlmN2MyZDAxNGY5NjFkYjJiYzY5ZDE1NjUzNDg5ZmY2ZGUxNzVkODQ1YjdkZTVhNDA3NDI2M2FiNzhiNzA5Mjk0MzA3YjMyYmNhZDNkMzQ3YjllMjUwZDA5NjNjYWFkOTRmOGNhZjBjNTRmOWFkZmIyMzE0OTI2MTA2Nzc0Yjg0MjI0ODY4ZGYxYmIxN2UwOTQyNGE1YTI3YjRhMGZkMTg4MzJmMGMzMTEzZmJlZTZiZDVmMTcyZTRkZDdhZTljZDI1MTZhM2ZjNzEwZTYyMWY2MjdiYTdlNWU1Yjg0NGQwZDY3NGQ0ZDYyOGQyNjIyMWMzY2Q5ODg5M2Y3ZTNjZmRmNjQ3OWVkNDAxMWRlMTdhOWFmM2YyMTVmMjNkYzYxZjg5MGVlMWU5NjA2YmRkM2IyMTg5Y2JhNTQ2Y2I5MzRiN2M5Njg4Nzc0ZGIxZTA5ZGI5NzQxZmFkY2FhMTM3YWM1OGUyMzAzOTExM2NjMjA0MzM1YjVkZTMzNmFhZmVjYjViNWM0MjA4ODBhNmY2ODgwZjJlNzFjZjY3MjdlNjJkMGRjYTc3NjNjODQ5YjAwYTQ1NDk1OWVhYzVmNzI0ZWNhNWQ0MzgwNmQzYjRiNDIyYjI3MjgxODA2YzVlNWJiYjBkM2VlYTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nGtl6uACyoYKtf2mH_SDmT-g3OUniL_UBcyIyDw-fYGg5O6J6rLCCkgyQaG2c4_i1WjnHZ_aD1Tl0OLmCU8nGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110123_73_249d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.418Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IldDUHRxdVIyaFg3Mys5R2tRVmpFejNJTkxyNGJDLytGVHc2alNhU2lFR3B6SmFTWlU2OXFEaE1NbDZEZmJURjRhWERrYklMYm1jMm1UbFBEb2laS01BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjNfNzNfMjQ5ZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2Y4NmI5ZTUyYTI3ZTIwOTJmODk1NTg1YTAwN2I3ZTZmODRiZTU2MGNmYTA0ODgwNmQ3MDI1NzY5ZGYxYTU0YThkMWNkYzRlZDc5OGZlODFlZmNhYmEwOTBmMzJjMWQ3ZDQyMTQwZjQ1ZWM0M2NkM2VhMGEwMDAwNGVhM2I4MWM0MDA2MjM5YThhNjg2YTNlY2ZhMGM0YmU3YmE3NGNkN2VkNjY3NWIwNDhjNzA1MjQ5OTQ5MjMxYTYwNGNkZGJmNDRkMTc1ZGI3ODliMzljYjhjMDBmZDk4Njg0N2Y4Yzc4OGRiM2U2OGU2YzE1YzcwMDJhODUxNGMwMThjNzAyOWM2YjE5ZDI4NTZlOWQxYjY0MmU1MGNkZGVjYjJjNDJhMWIzNWJiNzcyODNhOGIzNTVmMGMxY2I2Y2M4OWMyMGZlZDBhZDYzOWFhYmY2ZmJmZWJhNGQyOTNjYjFmNjQ1MGRlM2I5NTdjZjA3OTQ0OWM5MGQzM2ViZTVmNTdmODE2MzQxYWI2NmI0Njc4MTVlOWU1ZmI0Njk1ZjVjZmEwYTAxZjFmZDM1ZGQwZjg0NjM3NmJlM2YyZDQ2NWQ1ZGZlY2M4OGE1YjI5NjQ3YjgzNTczMTdkNmI3NzFhOWFmOWMxZTlkNjE2NzgwODgxYzhhZDM4NmI5NzI1MDFjYzExYzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PMDznal6fWey3uE7r2EQNbweLkAlUHxKoE8KKJA-q0aLQF_IeiEa09ODnPP7y19wMabRLJMLlH00PvbO-0GM9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110123_73_249d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.421Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikovd1k3Nk9VQnRCaHBhWDBzUDAzVlkxTU90N0ZIRzBVdHczY2Ezak9VeXRJczYvcTYrcmFXdUI1RXBVNTYzdXVZZzByY1VMTTBjWTJJVjQvdjlweVF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjNfNzNfMjQ5ZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGMyMDIwMjlkOTI4OTFhMDNkNzRhYjRjNDZlYWMxMzhkMzEwMjVmOTNjMzU4NzcwZDc2ZWM4MTg2Y2JiOGYzMmIxMjk2OWRkMWVmY2I1OTNhMWJjMDNmZjBmOWU3NWMzYmE2NDk0ODcwNDZiY2JlMjdlNWEyZmU5ODFlYTIxMmM4MzZjOWU5YjZmNThiNzMyNGQyYTI0MjlmNDE5YTVlMTIwYTkxZjZmYTllYjk4ZWI5NmM1YWIzNGUzMWRmNzQyNjUzMTg2NDJiYTMxYTBkNGNlNDRkODhjMGYwNWU2ZDZiNTEwYTdjYjM0N2QyYTIwNThlNzk0OTRkMDhmMmIwZGU1YTVlMDMwYWQyODk0YzY1ZWFhYTYzNjhmNjQ0YTA3ZDJkMjY2NjI4YjVlZmQ1YWJiMmU0OGY5ZDQyMTUwNjkyYWJjMWU0YzU5MTA3ZmYwMmNhOThkODY4NmMzNzZkMzMyZTFiMDcwOGNlNjIwNDNlZjExZmExMzU2NDAwMTRhOGFjNzY0MGIyMDU0YmZlMTQ4NzBlYzI3NTJhNWJkNGViMWMzYmViOGY5Mjc4N2RlNmQwMWViZDNlY2JmODgyMzVkMDVmNTQ1OGI0MmE2NTI0Zjc5M2E5NDMyNTlkNWQ0NTM0YmVkODlmMjk0OTY5ZjA2ZDAyYWE3NGQzYWJlNTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.raSEEy3xBNRYdYp0ZHGEQE9m_xFQLwXnlhw12K03CFY1wlELbZRQ0i2zoR_SXFCquyTWZozQF8hJHAxk7GF_PQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110123_73_249d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.424Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkQ4UDR1UHYrNDV4S2NiZjczSjFhdllVWkhvQ0hSTlZIT0Zab2tIdXduUExTTnhCVXFVUjFYOUxmSkxlUklqNXQ5Qk5CenZDZlFMTEZsTEo3QllRUDZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDIyMjdfMDBfMjQyYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWU3Y2IxZjM3N2U2ZmUyOGQxNDA3MTQ0MmZiODBjOWIxYWE5YWJhYjNjMzIxZWYxNDRkZWY3NTZjNTA0YTg0NmZlMmNhYjU0YTk4YzVmNTkzYWJhOTAwZDk2MTE3YWFjYzc1ZmZlZjBlOTg3Y2U3MGZmOTYzNzk1NTBmMWIwZDI5YWQxYjFiZjBkODk0MTI4ZWRhN2ExNTNmM2I0NjNjMTYwOTYzZTZiNGM1MjIxZTZiMDVjMjQ0NTdiMzE1MGJhNmJhMTYzODgwOWU4YzYxZGUxMDZhNTI1YWViMmVkNDZjYjYxNzFjMDhlMjY3MjA4ZmUxMjFlODhjYjVlMzlhZWMzMmQ5ODI5OTg4MDE0NWY1YmY1MmMyZThkOTFlOGRkMjVjYzk3YTViZmY1NTVhZjM1ZGUwMTYxODgyZGYwMDZjOGVlMTAwMzA5ZmUxMjZmODY4ODA2YmE3NGM4NTVhNzMyZTI1ZmMzYWRmOWZjMjdlYWQyNTQ1MTRiYTkzOTY1YjBmMmI5ZWVmYjVmZTBlOTI5NmQ5MmJlZDEwNDBkMzU1MmViNWFlOWQzNDM1MjkwZmU3NTVmM2Q1MjgyOTkxNzI5MTlkYmYxNTg0ZjFlZjdkZTczOWIwYjE3OWQ1NTNjYjU3NGRmZTk3ZTE1MTc4NzhkNGUzNjkwNTM5ZTA1MTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pbFVUkXNcgbBackbf-yYwSIIb5XBE1aRdDS1UDRvWoyO5nuZjnExlRWCDxp3e2JpcllzLZpfpBv3rniIREmORw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_102227_00_242b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.427Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InV2Tmd6VFpnTC9wZytPMHRWYUZBNjJSbXYwNEhFSVhJdUxwUzh0YUhsQlNCV1QyRUVJNS9UVWo3Rkl3S2FXaWFhaGM2UE8xcVBRWW5qN3g1SkN3clBRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDIyMjdfMDBfMjQyYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGMyNmFjOWE3NDQxOGFhYTJkZWVmMmRlMGE0N2UxMTNiNjU1MzE1ZGU1YWE5MTdiZTZlNjNmY2I5YmI3NDFjMzA4MzI0M2NjYWQ5NGMzNmUzMzRkMjVhYjRmY2MzODNiMGNmNDMwYmI0Y2JlMzUzZjJiMGRlYjFmN2E4MDk5YTZiYzZkZjQyNTI1ZmRhZmY5Nzk0NjU0YWJhOTNlZmNhYTI5MTYyZDcwMjU2MTJhODY0YjVlODQ2ZWM3ZTc1ZTNhZGM3ZjQ2YTRmY2VkNjEyYTFjNzRjYjJiYTlmYTRjYjlkMWZmYWY1ZjI0NGM1ZjY0OWRhYTJlZmY3YWQ0NWJlOGVjMTNmNmI5ZjI5OTFjMWQ4MTRjMGM0ZDcxOTI3ODFjMTRlYzcwZTFmNWEyMWIzOWUxYzIzYzJhYmU2NjEzY2NhZmU1ZDQ3ZTY2NTE5ODcxODM4MDA4OWIwMTVmMmRiYjBhY2VhYjg3ODY1YjIzODI1OWI3NDUyZGEwN2IwMGRiZDg2NjQ0YjFkMGNmYjk2NmQzZTQ2Yjg3M2RmOTY2ZDEyNzczODc2MmUxZDA0Mzc0NjZiNWZjM2JkZjQwMzQ4NjE2YjQyODFhOTkyMzFiNTJmZjAxZGY4N2RiZGNiMmZmYjM2MjI0ODRjOWI2YWExYjdmMzUwZjU0NDlmNzcyOTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ngkHoU3YxtBJ5sXBBZ19WSYl6AMQ7ww5aQCN7aHVv9vMWX_7OwMmODvJ5xMxpta_sR0Ai3up7xnldohj9cQNtw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_102227_00_242b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.430Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im56aGFEQkFxS3doVkhSVmZRaVMveFRvdnh5ZlJ0QzEwMW5xQ0pudXdwTVV1em9mWlIrSmt5a01KL2VjWXRYaDJFY0Q4a2NKZWphNEpiRFoxMm1tN1JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDIyMjdfMDBfMjQyYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWIyN2U5MjQ2ZjI3NDg1OTA5NGUzZDYzOGFhNDViY2NhNWU4MjZmMzhhM2U1NDY3MDZkOWQ3MjNjZGNkNmUyYjcwMTMwNzI1Yjk1ZGNjYjU0ZDMyOTBhZTFmZmFkY2JiNzA5YjVlN2FkMjhjYTAwZDRhODJlNmQ2ODM5NzlkM2NmYWU1Mzg0NzEzMjNjOWY1NTlmYmMxMmJmYjA4N2E1OTFhZTZhZTAyMDU0OGFlMjkyY2ZhM2Y0YTE0Mzk0YWJlNjE0YTFjYjExMzhlMGE4ZWFlZDMwYTVhMjBkNTEzYTlmYjlmN2IyN2FiNGIwODNlZmNhMzgxODA5OGFlMzEwMTAwMzgxMTczYTI4MTRlNzM5N2IzNjg3ZTI2OTUxYzhjZThjN2QwZWZiOTE1MmU4NmQyOGNmNDdmZDg0NzEzNTc0YmQ1NGM4ZGE2YTFmM2NlZGI5ZTBlYjAyZTQ0ZDA4OGIzMTY2OTRlYTg1N2EyMjMwYzgwNDIzZDk0YjUzYWQzN2ZiYTFkYjJjYzEwZjUyYzhjNmUyOTU3ZjlmODY3MGU0YzM2MzlkNzQyNTkzNDQxMzlmNWFjODg1ZDRmZWZhYThhZDkzY2FlYmM5ZTk3ZWFkMGE5OWExYzJhNDMwYjE0ZTI3NDZjMjJjMzQ2OTVhMDcxMGYxZDA1ODQ4NDQ0ZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Dn8LgSSc8lQLuk_wkeNPu8r4dfLJ6SqcfEuqUh3wjFowmnlIClAx9bQK23sDBqmhbKSyo7y31UZQLYvGD6R8_g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_102227_00_242b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.434Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjdINGc3bERFalBPSXZmbHA5L3RKNGpaZC9iVlpnZ0RUMFF3S21sSzR3dXFHRUp6cDd6Y1RHUmY4bzdFdnFmcHo5ZlBLaDB3Z3FCZmhPbHM4SzVrNEtRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDIyMjdfMDBfMjQyYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODFkZjViYzE2YjIwZDExMGU5YjE0OWZiMGEwNWQwZDgyMzcwNmYxNWFlYmU5YWIyMzI5MmE3ODdjODVjY2Y2MzMxMmM0NTE4ZDQ4OWExOTBmZTljZjM4MDY5ZDBmOTRjZjYwOTk5NGU2N2YwMjdmM2Y0ZjM4MWUyNjE1MTU0MjMxMTU2Y2MxOGVlMDZkNzVkNzYzY2JhMzY0NGUwMjEzMTEwOWVlZTUyYzNiNzNhNDJkNTgxMjljMjA2OTc0YTcyOWQ5YzFjOGMxMjI5MTBlMzM5MDI3NWNiYWQwMDU2MzY2MmU2MGNhODVlZDBkYWNjMWZjNTBkN2VlYmI2MmM4NjRkYjI3ZTRlMzRiZjFhZGZlNGFlOTMwOGM0ZDQ4ZDM0OGYyOGQ4MjViNzNhY2Q3YjcyNDNmODA2ZmZiMGMxNzI3YTcxNmFhYWZjMWU5YmRiOGEzYzUyNjRkMWJhNmRjN2MzYTc2MmE0ZjAyMWQxNTI5ZDEzMGI4OWUzOTYwM2U0NmEwMTljNzE3ZmIyYTU2YmZkZTAwZThlYTJhYTJhNjlhNWVjZDIwZjY2OGE1Njg3MWY1ZDUxM2Q0N2UzMjkzYmNhOWMxY2M1MjlhNWU4MzA5MDRhZDMwYjgxMGRjYzVhNDlkNjdkNzE2YzJiMjk5Njc4ODI2ZTA5MzYyZjlmYzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6Ndpz5_Q41P2eR74ICdXKp9fzKhSh7Vt7-ntM-61V9OkcrgPsk_dgT3vkWeFPERHV5pQFldqfn8uelLRPVRHEw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_102227_00_242b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.437Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImJZZEQ5a3d1bk1mWGxCRXlsSmpoNDdWUmhVbUNOOHV1TXhHemFTYTV5dmo5Wm5OV3BrR1UydnNGaWhHaDlubURoNEs4S2FSRXAxd3cyRXhFZWFBMEdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzBfNjZfMjQ0N19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjNhYzFhNWQyOTkwNjFkMWNmYjAxZjdkZjI5MzIxMWUzMWUxYjE2ODMxNmYzNGUzZWVhZTg2NTQ4NzVkNjUxYWNmOTFiZDZhMTZjZDI0OTdjNTg3YTQ4YmU1ODAyYmI2YWVkYjk0MmUwMTFmMGZjYjAyZGNlZDIzNTE1NWYzMTU4MmE5MzBkY2U4ZTk1M2MyZGM3YTc1ZWNjNzUzNmI0MmExMzFlZmI4NmE3MGRlMGExNGU0ZDVmNDk1YzQ4MzdjY2JmYjY0MTg3MzEyMjE4MThkZmYzYjgyMjE4MGRmNGUzODY1NjI4Y2QxN2JlNTI3NDBiMDE3MGRhMWQ0NWE4NDAwN2ZkNjI4YzE5NzE4ZGM2ZGIwYjc3MTZlMjdhNDgxZTAzNzhkNzhjMmE2NTJjNDk1MzVkM2RkMzgwOTQwOGY5NTQxNTcxMDE4OTcyYTdmZDYzNThkZGM5NTI0Yjc0ZGNmOTVmOTMwMzEzMjA4YjExZDBjZWM4NDFlMzJhNThmNDRjODg5MTE3ZDVkNGVjYzE1MDM3MjAwMDliMGRiOWJkYzFmNzhiOGY5NmQ2NDdlNDE2MWRjNjY4MTAzMWRhNzRlNTJmZTZmN2QzZTJmZjE1YzU4N2U3OWI0MzBhMzRkODI1MDAwZjUyNjllNWMwOWM0MGVjNWUxOTdiYTk0NTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OthEf6D7Bp_ML7lXHT50m1MPeziTtBu5ouXMuxRJRBYocuRLBaQtDvbzJ-3uHl9QD8LeIaCjCkOn_ajoRoICyA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103330_66_2447_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.439Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImxOUGVTcTBXRytoM05iaXZ3SDVFNTZkQVJLWmRic2lXc3Z3dWNrdDcrczBLVWcyelV0TDdhS1NDSHZwWWV2UUU3NnMzZUMrMHB1WGFUYW0ycnhuY1VBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzBfNjZfMjQ0N18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDI1OGQ3Y2VkZTQxZjYzNDc3NjE0NWNlNjEzYTgzMDg2YmM4MmRkZGNhNDk5NGM5MDU5YjgwNGM2OTZlZGI2MDAzMGFhNDY0NmExMjE3YWU3YjBjMzIwZDNmYTAwYTA0ZTM3YWZiY2UxOWJjYTFkNDRjYzI2MTMxMmNkODllZmVlNmI0NGYzZjMyNzA3YzhmNDU0NDc2MjkzOTg1YTE1M2MxNTMyNGM4MzViNTgwYmQ4Mjk1ZTQ2YWNmNjliOTg3ZDBlNDljNzQyNDdlZTY2MTIzYmEyYzY5YzkwYjdkYjI1MTdjNWEzODVlNjNiMmJiMmFiMzBiNTg3ZDhmMDI1NzY1NGZlMzkyM2YyMGI2NGIwZTY5YTY2NDE5NmQ4ODA1MDU3M2U1M2FiMzUxNjEzMGQ3Nzg2MWFlZWI5MjRlMWEyNjk4ZThhYjIwNmYzNjc5ZTEwMTkzZjY2OWFhNDQ0OTA4MzM3MmQzN2I0OGQ2ZDJmZmU1MDQ3NjNlOWUyNTRkZjFjM2U4ZTViZDZkY2JjMDg4NWQ0NzlkOGMxNTFiMmZiMzcxOGNjODAyYmNlZDljNmFiNzk2ZmQ1MjM1MzBkNWM3YzNkNDAyZjk4ZmY3MjdhMzMyMTUzZDJmZWIxNTM3MjE0MWVmNTE4ZTQzZThiYWRiMzNiOGUwMmFiMzc3NDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Wt8UaBW4ObuPXzCVeAaeLL5Rw-Xmwlh7USrpKhdihVce-SRrVGHVWo72uAmdoviuTlI0SyDMk83yO6oVrbrMpQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103330_66_2447_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.444Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlRQQThNZlN2d1l5YjBkKzh0dlV3NmJveTFXdEgrZE00YVFrTmM0YWdBdTlYOUdBSlVKb3ZGNlJNRG9Ub1ZwOHFzQWdSdm9jVVd1dGEvbnd0ZVV5Qkp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzBfNjZfMjQ0N18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWRjN2M2ODJjNGFlZTE5NDJmMThiOWY2ZmVhZGJjMGNiYjgxMzVjODJlZGE4NjAwMTgyMzFkNWJkZGY2MjU3ODlkODk4NzY0MGM4NDA1MmQwNmQyY2YwNmYzZDdhMzBhZmFhZDFlYjRiMmE2MDUwYWQ1MzAzOTNjZGE1NDdiMmI4M2I2NGY0MTlmYjA0YTNhNDkyNmFhNDIyODllMzdmOGE4YWQ3ZWFlMTE5NjRjMmY4YjI4MzJlYWYxYzkxZDFjODRiYzAxM2E2Y2M1OTk5MWJiZDIwMTkzMmUwMTEwZDE1ZDZiM2U4NWQyNjc5YWU3ZDlmODlkZTFjYTY5M2E0Yjg4YThkMDQ2ZjQ3MTQzZmZiYzg1ZThjYmQ3MzE4ZGYzZjNhMjNlZTcyYmZiZmYzNzE4NTAwOTNhNjk1OTkzMTE1NTc3YzgxMDJiZmM1ZjUxYjE4ZWYxNzA4ODdhMTlmNDFkY2Y4ZmM1MmY2MTA1MGRhMzk2ZjNjOTQ0NGZiNGQ4MjcwM2U5NGQzNDdkNjYwM2UwYmU5NThiMGEwMzc2MDEzOWU5Y2ZhNzhjYzhiYjg1MTI4MTVmZjY1NGMyZDQ1OWRlZTI0OWI0MTQ2ZThjNzFjNGU2MGJmNGM4NTJmNGMwMTA5YjNlMTgxNTI0ZTZjODVjZmUyMDllM2MxM2IzZTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lKScKQ5vkGkJnc5XFkzA2HKcHPA-o9HoSvsP20tIjufK0PAg1MYq9if0F9tlNDAoV5Ewm5Gchlevzp7acGvewA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103330_66_2447_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.447Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ii9scVdnWjZ0NE5SSGJPU2lmTDdwMjZZeFRNL2UyV2ZlekJXTDdmTSt5dHdncHMrVEtCdytUbXZ6RE93Z0FMWm1mVUU1d2RmZXlzSWtlRkw0RFJzaWxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzBfNjZfMjQ0N18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzA1MmM4YjdkMTUwY2IyODNiMGFjMjMwYjhjZjM3NDM3MTY3MTQyNWIyNjM2MTY1MzM0MTU3Yjg3ZGZkZmUwMDI2NmU3YmM4Y2M0ZjQ3NjY3MTQ3MzU4ZDVkNmY1Y2E1OGMwMTAxOWYwMTIwMTA3MmRiODc5YjMwZWY1YTI5NzE5ZDdjNTIyM2FmYTE5N2JiMmQ1ODgyN2Y2ZjJkY2Q4MTJlOTc5MzI1ZmJkZDg2Mzk2YWQ2Y2JmYzVkMzEyYzFmYjQ0ODJjYjk0ZjQxZDBhM2E3NzJiMTlhZDI2MTQ1OWJkMmRlZDA5MTRlNWIzMjVjYmMxOGFhMjhjZTY2NDM4YjE5MzdkMmQxNmQ5NGZjMWZhNjA3ZDhhMDQ4OTc1Njg0YTk1Yzc4ZGQwZTYyZWVlNGYzOWI1YjRlMDJjMWRiMjA1YmVhOTkzNmM1ZDNiMWEzOTZiMjEzZjAwM2Q1MWU4MzBmYWQ0ODQ5MWEyMDU2NDI1ZmU1NDFjMjVkZGI1Mjk3MTcwZWEzNTMxN2JlMTI2OTQ0MTM3YWU3ZmFmNTg3MTVjYTkzY2JkNjFjYzFiNDQ1NGEzZWIzNjdmYjkwNTBlNTUyNzQ5MmNmNThiM2E2OGVmZDJlYjQ2MWUyMDU2ZmViNmI5NDdmODZmNTM4NmFmZWM2OTJmMmVlNDJkMmM0ZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.VtMqY7a4XcBZa_i4_iBSy1FlIY4Gv7AnkVUn3TS0S0K_1E9GJpzY4hM4O3BCx-CxrKx7IYZEuWYtYc3zb2bAhw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103330_66_2447_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.449Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlR3OWZTM0ZSdlEzcFRwSWRsaDkrWVlMbGluVTU4OG9sVm1UUTJxWkZlTGd3OFRnT1JOWXFXblRQbFIwWCszN2s1RGlia0lnUHRLeWRXMTRmcnhaMjN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MDlfNTFfMjQ5NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzJmZmE5MjliMGRiNDMxN2M1M2Y4ZTNmYmIzZmEzMmZiZjEwNDI4MWNiNjVhN2YwYjJiZThhYWRmNThiYjVmNjliYzdhN2U2MWU2YmNmYjc1MTExMDE4MTY0ZjM1NTY1N2ZkMmM0MWRlYWFiOGY1YjYyM2JjMzMxYjhkYTU0MGE4M2QxM2M5NDczZDgzZDNjNWEwMzdhNDFkZmY2M2I5YTFjZTBkY2Q2OTc1NWNjMDBiN2NhNWQzYmFhM2JkMzkxOGMxZjkyODM4YWYwZjhhYmNlNTdmODY0YjU0NWI4OGZlYzA2MTFmY2MyNjMzMWI0ZWY1NThkYTU5NWY2ZTRiMTg1NDlhNTVkN2Y5MjQwMGQ5N2RlZTdmYzYzZTJhZjU5YWJmMmM4NGFiZjcwOGNmY2RiZmFlMzM0ZWY5ZWY4ZjQzZDYzYmYyNTIwMjBlMThlYWY2Y2I4MjNlNTM0NzFmYjhmODQ4Nzk5N2IwNjVmZDM4Mzg4MjQxY2ViNGNjYzE0MzUwMGM2MDJjYmVmZmQ1NzhlOGE0YTI2ODFhZDhkYjU4ZmNjMTcxZmE2NGJkNTBjYjdmNjA1OWY5ZDE1N2Q5NmIzNDI0MTBlYTM3YjcyYTUxMTc2MzY5ZDRhNWRiZTE5NzVjZmUxMDU5OTkwYTIxOWNiZGRmNjFlZDJlZDFmYWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GG7jTustT3b_d5o48HxeCpqi2cGqL8f_JPzl779fYwbaj7MUh0trqX1ny7JPyV87LOTrCbWWPz1bvmBTaf0CbA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105809_51_2495_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.452Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ii9qem0vUEF2eXYwZkpjMVpPZmwySStlcUhKbkxjQVp0czc5anltUjZRWEJSejZ4VENleUYwc1grTG1iR1VxMUNBVzRqMjdCOXZEcXhDYmJDQmRnM3pBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MDlfNTFfMjQ5NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzdmN2YyMmQ1ZDNhODUyMjg3YjQ2ZDdiZjU3ZmVlMTQ3ODY1N2JhZTllODk2ZDA5ZjAwNjc1ZDY4ZDM5MjMxNmYwZGM1MmI5NGFhNmIyODBkMWNlMjg5YmUyY2RlMTk2NTM3ZDBlOTZlNDU5YTU1YWNhOTkxZTM2MDZhNTM2NDA1YjNjOTA4OTY5Yjk5NGNmYTI2ZTVmZjM2ODAxZTJjYjVhYjQ0ZmE0ZDg2YzlhNjIxMmJlYjA3MmM2ODQ5MDcyZjg2ZmUzYWVmNjdhYjgwYzM3YmU3NmVkYzdmMzVkYjVhYTQyZGE0Y2E3NTdhNmYwYjA4NGFiMmIxNWJlZDRkYTcwNmZhNDMxZDdiOWJhZmQ3MzM5MGQ0M2Q0MjRhZWY0NGZmNTgwMWY2MDI0MTNmOTg2YzE0NmU0MmUwNTE2NWU5NTRmYzljYzI3ZjMxMWQyY2Q1M2M4NjhjNjIwODMxMjZhYmM4N2ZjMWViMTEzZmFjZTQ5Mjk0MzlmYjM0ZWIwYzY3NmVmZGQzMmFiZTdmNjlmZWVkYWZjODlkN2MxOWMyY2ViMTJiMzQ2NDAzNGY2NDEwN2ZhMzEyZDI0M2UwZmM1NTcxZWJhNmQ0ZDY2MTVkODY5ODE4OGRmZDBiNGQ3OTVmODBjMWEyZTIwNmRlOTJhZDZiMjM2YWFiYjBmMDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OtfMtLsCgKtu-Y5WGdTXvtx04TyJfRZ0PIJndDyBdFKNkVPsMCtqFsIgZljqB0kMSGfjq8DiW3OubmYZTistyw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105809_51_2495_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.456Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkNHTWxxZ1JGc0J4Kzd5ZFg2THArb1RyTzhOQlRJaHJ6TmxMWVpyaExsdVlBb2VVU0hCOEc2Vm5IM0NzMENwQ205YWJnYXEzZklWdzhuVmEzM25OS3pRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MDlfNTFfMjQ5NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGZhNzE1NjNhZDk1NTA1NjNlMTAzMGVhOTRiMTlkN2E0YWM4OGE3OTdmZTBhMzMyNGU1NTMzYWFiMDQ1ZmU0MjFjNDM3ZjU5MGMwZjhlZGI1Y2U3NTk2YTNlYzgwYTEwNDNlYmJlZDk5ZmJiMmNlOGJkZjEyYzE3ZTg4NzFlNzg4MGRiN2E4YzVhMzQ2ZTk5NjQ4NTI0MGQ3NGZjZTZhMzliNzQ5YmM4YmUzODRhZjc1ZDIyZjVlOGI4ZTE0ZGQyYzU5MjM1YmE3YTRkYWU3NjgxYjg0YmQ2ZjZmMDc0NGJhZTNjYzI1YzAxMjk2MWQwZGNlMDYyYmIwNTgzNjcwYWQ2YTFlMDlhMTgwOWM0ODBkOWIxMTFmODJjYjcwMDc3ZGQ0NmRiNmJkMjAxZTkyZjU4NDc2ODlhZjUwNDQyYjNiNWY3MjEzNjA4OGIyNjdkOTNjODRmYjliZTk4ZTM5YWE1Y2M4ZDYyYjczZThjMDUyZTVhOGM5ZmZhZTUzZTI5NzIwNzM2OWFmNGQ0Nzc5MDZkZDYzODkxYjc2NjEwMDIzZGZlMDRmNzE5OTc2MTk3ODUxMGMzNWYzNDY3OTZjNTZkOWRhNTFkOWY2NTIwYTYzOGQ2MmJlNDQ5ZDkwZGExZjI3OTdiNGVlYjYwNWU2NWUxMDI3YjM4Yjk3YzllMDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.azrMYh08WpGINELqu3K5CNzVtb4E2gxKUHRHS7_9Vemts2sZWxRoMNkLbydkOkMwNCj4Bs97ueAoZVhPUI0KVw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105809_51_2495_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.460Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkdTVk4rR0Yxa09lM0RudGpqemJNTWc5MUovRnp1NVQ0RUxCeVNQUkV0Rnc3MTQ5ZWx5VHVhbDVZTjNSSzc3UmVsODJyYXJQL29rZWZNM1I2ZHdKc0VRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MDlfNTFfMjQ5NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQxM2JhMjc0MDZkNTdkNTcwOGQ5OGE3MjMxYmNmMDM1NTFhMmVmNjBlNDk2MTdmNTkwZGJhYmZhZGI0ZTMxODAzYzIxYzFmMmIzYTEwOWRhMmNhNTcyOTQ0NmJiM2Q0MTYxYjJlMmVhMGQwMjk5NDQ2NjI5MjNkNTIzOWViZTI1YzU4ZjJlYWVjOTQ2M2E3OTViNGZhMWZiZjgwMGJkMTJjNDEwMDkzN2Q4MzBiMTIxYTc1ODk0YjM1NGIxYjAwNWM3ZTQ3MDJiODU3NGZmYzE4MDE1NWMzNmM0OGQ4MmI2MTc0OTg1YzQ2MTBjNTA1YzJjNDEwMDAyODRmNjdhZDU5NGQxNTRmNzE4YWYxMzJiM2I3NGMxZTFiNGU0NDM2MzMyY2VmODFhN2IxMzFmN2Q0OGFmYTA0ZWY0ZmM4NGJmMzQwY2E4OTgzNmMxOWM4OGVkZTYyZGRiNTA1MWIwN2U0ZWIwZmRhNGNjNzVmYzkxYzcxZWQ2MGYxM2E4OTgyOWQwNmNjYTE3MzFlNjNjMzJmZDgyMmNlODcyZGM2ZDEyMjRjYzAzNDExOWMwNjZjNGQ0OGE3NzZmYzA2MjRkNzRjNDk1OTQ5MDE3Mzk4MmYzYjAwMDdjOTQ1ODE5NjAzOTFjYTIxYWQ2OWZmM2JlMWVlZTlhN2UxZTY1N2Y4MWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bc-QPaFI3nwLQgz9MahJz5wAq7JPfCAyGTUnGx_DpPJgYeMXUOBjSmBjYTVbuyTHaYsEml-ANb0yBb4nQJAT7Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105809_51_2495_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.464Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InJUR1pRdi9uWFlQOGpxYmF2cVNoRDhnNFZEamtnSnAyZnRNOTZhcGZiRFZpVjRBeG9wY2haNnRyOXpSVWNGUDJDcWc0RlAwWEZkWFd6Nm9HOWc1cHZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDQ5MzlfMjdfMjI3Nl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzY1YmYxMzQ3YzA0NDIzZjllNWE2NmUxNTlhNmYzMjUwYzlkODU5YmNlZmUzOWQ5ODYxNjYxNmUyYWRmNWM4MTNiNDlhNGI5NDc5YTkyYTUyYWVjMmQzMDdkZTZhNjIyMzViNmY0ZDFjMDY5ZTJjYzA5YjAxYmQ3MzFjMDY5NzI3MGEyNDViYWY4Yjg1NTA4NGE4MjM3ZjRmNDMyZDk4MWM2Y2YwYjQzYWVlMTE4OTZhN2MzNWQ2YzA5MWY3OTk2ZGY1NjE1NjgzNWMzNDBkOTQzOGFkOGE0YjZlNzg2NmEyNzAwZTQwZTA4N2Q0ZWIxNGJiNzJiNWFiZDQyNDIyZjkzNDhlNDg5YWFiMzliMmI2ZDhlODNhNzY2M2RiZTM3MTkzYzVjNWY0MDNkNzU5ZjI2MTk3NjBlMzg3MzFiN2UzZjk4NzA5NDdlMmQzODk5Zjc3ZGUxZjA0Mjc5ZTlkMzY0MmE2ZDhlOTdlMzBjNDUwNTNjNTUxMjlhYmU3OWNlMTQwNDgwMWUxOGVkNDY4NWM2NGFkZTIwYjkzNDZlNTgzNjQ2MTJiYzkwNGNiZWU5ZTZmZTVhNzE4Yzg1YjMyZTQzYjJlMWFiOTIzNmI1M2M3Y2Q1YmEzZTJlNzI2OWRjOWUyNThiYjg0MjZkMjUzM2NhNDFiM2M1YmFhMTEyZDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bnemnQ3Z5J1pKCH24E5dloPgSQYYBCcD3e3m2pnv2hTcVfxP-cXHvBQ_zXMMnsUgY1rSzXEq0xnJb_gKYFcrwA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_104939_27_2276_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.467Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImJhUWFYaUFqbFRSMVZqSlBLSVV6bE9ZYWp0SnB4bWxNc3RHUGg4bTBkZXNXdGNkY2srREdQRnd5UUJScTllczJZVmJFbXRJZjZYcmhDUlpPdytBU0NRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDQ5MzlfMjdfMjI3Nl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTg2NGJkZmU4ODkzODM5OTk3ZGEyMGQ3NzdiODhiYzZkYWNkMjViZGZiMGMyZTNmNmYyNWU1YzZmMjVlMjMwYzBkNGNlMWU3MzAzNzk3MmEzZWQyNjllYmEyNzJlN2QwNGZkODE5MmUzMTM3MWRmOTdmZDk4ZTM2YjUxOTAwZWRjZGJjZjBkYTc3OTUxZDczMzExYTlkNzRkODQzZDEyOGVhNDg4MzlhNThjYTBhZGExMjI3OTI3OWQzOTlmZTg1MjllNTg1MmZmYjExZTU3MDk2MDg5ZmI1NzhmMDA2M2RmYWNjNzYwZTM3ZjI3OWUwNDFiNzhlMGZjZWQzNzI1MTEzZDcxNTYwZjdkM2RiMjNmZDU2YjUyNGUyNTMxNWU4OGMyNjE2ZWM2ZjEyMmU2YmZmYzlhMDljM2RlMDhlNjYyMTZmNjZjZGVlYjZlNDRhMDRmZDhlZDExN2I5MWFhMDU5NTBjNjhhZmE4NDZmM2I5Yzk4ZjdkOTM5NWM1ODZhYTJiZWJmODI5YjhjNGE0MTdkNmEyMTJhNjdmNzZjOGU3NDZkN2U4NzY3YTM1YWFkNTk4MGE3ZDhhZWVkYjkyY2Q5ZmU1YTllOGI3NmY4Mjk3YWE2OTQ1MjY4ZGIxMjA5NjEwZDk3Y2U0MDNlOTY2MTQ2ZmI1ZTIxYzBjN2QwYmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DH0zysoAM_XZsf_dYsRarexrIrMTotpHWj-aZUmrVOE7rIFx2wJjfAn22OpD8_weBVwOVpwK9N7yI01XtroYhA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_104939_27_2276_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.470Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImlUSmRjT1FwbWRtaHlFLzhhVW55QTZGWk80aE00dE1SWlhPUUlxTm1mNHk1KzNTUUZCZTlFUzJ1YjhLTVlXM2dUU3pVVTZQVUNMb0xnYk5JSm51TS93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDQ5MzlfMjdfMjI3Nl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjdmNTE0NjU5Y2E1YWViMGU3NWUyOTkyMjlkNjQ4MDZkZDM2ZGZmNDc2YTE0Y2QxMGY1ZWI3ODhlMWZiNjEzOGQ3MzJkYTFiYjEyOWU3YTEzMTIzOTUzOGM4ZDZiNDNmNDZmMzYwMWFjNzk1MzllYTBkOWNiODI5YjQ2MGIyMDI3ODZjNDU2Y2VmMTMxZWNkOTNhMWVkYjY2YTE2NGE5ZjVhNjg0ODUxMzg4ZWNmNjg1MTBjYWZiOGY1MmQwNzBkYzIwMTU5ZmY5Yzc0NDM5NDJjNzA4MzkwZDNmMmQ3MmUzZmZlMGQxNTQxZmJhZWI1Njc3OGU4MGQ0MjhiZTAyOGQwZTQ5MTM3YjlkZDNjMGJhZDIyNjZjODQ3YThjMDU1NzAwN2U4M2IyZDFkYzg0ZTdkMjZiMWE4Mjk4NmFmODE1MmQ1Yjc2NTA2MDE4MTAwZTRhNjYyZTc2ODYyYTE2MzllMDY1NjgyM2Q5OWZlNzYxMzk5NzkxZDk4MWE0YjJkM2NiYWI3OWY1ZWZhYjlhMWM2YmRjZWJkNTgxYTkzN2Q0ZTg3NjI4NDFhMzlkZTRmZWVkNDYwYmNjYjFkNjMxMzhjNGI1MzZkM2FiYmI2NjAxZmM0ZDVjZjZlM2M3NWYyOTdmNTFiNmFmZTU2MzMxN2RlYTg3MmU1NmU2M2Q5YmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zAalRCtBOjciBmqr9yCbL4RVylmYEFcW31ConIzHcdOV6W2IRR_eUK9QLi-toL0dm87POWoLYxXFujLMIKivdg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_104939_27_2276_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.473Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImZLckJMSEJNTnJnL1R3WjQ1YTVpS3phdlFoMm9nVHJSRGsxeVNzd2VwbVp4VDUvTmFNb1dBZ3NOeFYrQ2Q3QlRybmZuV1orUXBtU3hpNExCL1E3OTVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDQ5MzlfMjdfMjI3Nl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2FlZjBmM2Y0OTNlNDE5NWEwMzU5YTEyMzQzZjEzMWNiZmJhZjQwNTI5M2RmOGVlNGZmODcyZjZlN2JkNWNiOGEzNjI5MGRkNWFhZWVlMTcwMjlkMzY2NWE0MGUwMDI4NTc2YTkwZjMzMjliNDgzOWM3YmNkNzIwZTE5ZjIwYjMwZTQ4NjJkMGFkZDBlMzk5MDQ5NDc2NGViNWFmMTk3ZmY0NzlmMjg1MjBjMGI0YThlMzE2MjRkMWIxNWJkN2NkMTk0OTE3MzFlYTU1ZTVhNWQzMGNlNTU4YzAwODg4NGRjMDQwZTc3ZjAwNzY3Mzg4NjVkMTk1NmRkNjRjZTM5MzAyOTMxNGIyYzJmMzMxMTYyZTU2NDY5ZTk5MGExYTgzNTUxYjVmYzVhNTY5ZWNmOWQ4ODAyYjdiNjEwYTcxNDIyNTFjOTBjOGU4ZmI0NjVmY2Q0MTIzYjlkMjRmMjZiMDNmOWNkOWNmYzRjYThkMjA5OGFkOGNkMDQ5YjI0NjNjZTg3MWZiMmUwN2VkYjcxN2UzNDVhYzEzYTdmODhjZTVjNTc3NzRmMzY2NzkwMjkxMDhmOTk2NmU1YmRmZmY0MDI1N2QyNTRlNDEyMTBmMmU5MjE4YTMyMzhhNDRmNzFlYzRjY2Y4NTZiNGUyMzU5YWU0OTg2ZmU2ZTRiNGM5ZWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.m7CQf9jTP3PQFQOkQ8dAll3PyYAJ55E3kKe8m5mXvGeO5Ave0patjv32cN4iJv_8UZoKf4Ofk2oNlJFp51DSzw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_104939_27_2276_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.477Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjJXcmpoeUEvV0hXa0MvcW8vc1BWTkEzVEo3VmRmS0tjdElLMkNhcjEyY1lmWVB3RGJ4aWFIZnpKWi9ZaU82Q1QvMG5ZV21jU3NaOC9mNkZkT21nSjVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDI3MzNfODNfMjQyOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWNkZTY2MmNmNzI5ZDA0MjQyZGJhZTUzYWEzNGFmYzQ4NzYyNzAzYTdkZTUzNDJmZTAyMTI3MzcxM2VjMTVhMWM5ZjUxMTZmMDBlYjVlY2U5YmMyYmVkZmY4MDI3MTQwNjJhYWE0NDA4MzQxMmUwODkxYzEyMTRkMzZiZTU3MjhiYjkxMDM5MzcxMDhiNDAyZGNmNmMxOGM4MWJhNjc4NjBjYzc1ODM0NWZkYmMyMDIwZTM1ODc0N2ZkODkzMGU2MzE4ZjQ0OThmODZjZTExNTZkYTk4OGRkNWI1N2FkZWY2NWFlOGUzN2RmZjMzNzJlNzMxNzEwOWNhMmJjMmJiNzgxZjFjYTQ3ZmM4YmU0ZDM4NGE5MDRlNjdmZTQwNjQyMjEwZTI2NWNhNzY2ZWEyOTEzYjlhM2NkNTllMjNkZDkwZjE4YzkxMjhiZjYyZTkwNmNjY2IzYjE0ZGFmMDI2MWZiMjEyZmVmYzllMmFjMDUzN2Y2ZjY5NjE5NTM1Y2JhYjQ0ZDA3M2RlZmVjOGYwMGY4NDY2M2EyNWZmYjkxNGU0MGZmOGY5NTRkYzljNDRlYzkxNGE4OWEwZmJiOTI4YjllMWVmMmFkYzhhNTYzNDJiZTA2M2ViYTMzN2NjNzMzNTFjYjcxYzU5MTZiYmVlMDkzZTA3ZDkyZTRlNWQzNzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yr5_Y_UxLYUX5HdSy-uRlFWQ_n62iN4KwuGN2wddEtLiXvh7K-5b3gmr3udMu0JZmRgYZCVbGvUaa27PA2vzKQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_102733_83_2429_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.483Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImEyZk5BMExvS1c0UkhuZUZFRlgrbHMyTVpNbUJIM3FTeGVrQlJtZTRtVE1HaGx2VHlQMHNCdHh1dnpNNFlKRTM2dmZrbyswcVFGSHBraUhQaVZqRU9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDI3MzNfODNfMjQyOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTFkOTNjYjRlY2QyNDMwMzRkYWI4ZGZkNzUwNmNhOTQyN2Q5MmU4NWM2Y2Q0ZmUzNmYwZmEyNGNlZmQ3YTg1YjAyNjI2ZjUzYTFmMzUxNzE5ZDA2ZGFlZTRmYzQ0ZjUxMzgyOTUwNjJiNjdhMDZkMjBiYTMzODAwY2E4ZjA2MmRiNDg5MThhYTZhYTFlMWI3YmVhNzkyMGMzNGI4NWVhODNhNjg2OGRmNzU3ODNlODVkODI3ZGNjNjgwYmRlMzA0NGM0ODVmYmY2Y2IyMTRkNzljNzY0NmQ1OWIzMjU1Y2JjNzVlMzgyZDcyMjVkOGM0NTk0ODJmZDAwYmU2ZTM2NmFlYmQ3NGY5YzZjZDhlMzllOWVhZDA5OWY4MWI2YTIwYmRjZTJhYTFkYWJhMTlhMmQyMzczNjFkYWI4NmY5M2Y5NWMzYzA2M2EzMzVjNzE2ODI3NDU1ZjI5OGQ2ZDkxZmE1M2E0YTg3Y2NmNTYzMDE3YzBkM2M3ZThhZDIxZWQzZjAzMjgzMGQ1ODExOWQ5MjE4YjIzY2M3ODBiMzExNzVjZWNkZGNkZDk4OGI4ZjI3MjNhY2M2MWQ1M2IzNGY4NWFjODYwODM4NzZkYzBjOWNkMmMxZWQyMGQ1YTI1MmM0MGUzNjY2OGE4NDIxNmNiNjE4NjYzM2Q5ZjliMGUzNmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.H6Eao624gOxpQClaXrTETNIdDb1r5Imd8nU-bGcsQcO3tlcb4RN44K08kjSQVq0WRN4H8NgDenurXJXZLbFmcQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_102733_83_2429_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.486Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkhpOEEwcVRqLyt5dUg4U01lYnlZVzRCTkh6dU9vL2tSSEpQYmJrQWtvc1U1cGZMWTEzNU81SElGTFdOL2w2WkdSSk1WNEp1UHRSNWVLRUNtb3BBNnBRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDI3MzNfODNfMjQyOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWRhYjc2MGIzZTYzNmM3MmQxNWNiNGFiNzIwNjc0MTE2ZWYzOGRmZGNkZDM2MmJmZmI3OWU3YzVlZTRmZmI2Mzg4NjJiNGQxZTQ0N2I2YTdlZTk2ZmJmZTdiNDIwYThhMzY5NWIwYzk0YWRhMTBkZjk4ZmUxYTFhMWE3M2IyMjc1MzdkMzNjYmFhNGUyYjc1ZWQxNmUzYjc5MGI1YWFhZGU2ZTBkZDk2MDIxM2YxNWEzMDU5OGFkMWNhNGIyNzA5MWE0N2I1ZDIwZWFiNjBlNGQ1NjMyYThmZWI4NGU0N2JhMzkzYWMwYWExYjlhYzg0ZDY2ZWY2ZjE5ZWIzMjE4MWZmNWNkMjY5ODMxYjczOTYwNWEyMzMzZmU1OWQ0ZjgzMGEzMmVmYWU2Mjg0NWExNWI2Nzk2ZGIzNGE2MjRhYjAzMmQzYTJlNjZjNjdhYjFmNjNmZGUyZjNlNmFiMmJhZTY3YzgyNTljNmVhMTQ0OGFmYmRmNmQzMDBlMjAzZGUyYmZjMTAwMjU0NTU2ZGJmOGNjMWE4YzFiYThhOWQ3N2JmNGQyYzFiZDNmMjFkMjgxZjIyMWI4NjIxODE4YWJjNzQ2MDQ3MTA5ZDA2Y2VhMTQ2ZTQ0ZTVlYjRhNTY1ZDdjMGE3OTY2MDYyYmIwZjc3YTQwNThiNjAyNzFiM2E3MzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2oxe6AyXh2ngq5B5fWrlTfQh8OWsQxLK4Q7odeE93ciXgDkgnMvvC3j1KJgfT6Yt_Hw3PdVQAXP1TXf0NvV5mw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_102733_83_2429_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.488Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Imx4dGpFM2czMmc3eGI2SXdqS1lUYlI0NlB4REVLYTEwQldDcUNneEpMOXRRVnpDbk1jd1lCbFUrNzJGTUlOcURDclEzVHpCMGRiT2w2elJBaHdLWkFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDI3MzNfODNfMjQyOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTVmNDNhMTA5NmNlY2UyMzMwMGZkMmJkYjllOGIzZGMyZWU2ZDUxOTU1MmJjMzcwOGUwMTE2Nzc5MjQ0MDE3ZjQ0Y2U1ZTNlNTViNzEyODhhNjNmMTRiOGI1MWQ3NWYxYjMyMDlmMDM1MzVmYWJhZTI2NWM5ZTdlMjQzNTUzMTllNzNhMzk5NDJlOThjOTY5MWYzN2M4MWJkMmI5MDkzNzFhNjY4Y2Y0MWEyOGQyOTA3ZWNjOTEzNGI3M2ZiYmVhZWJjZGE4ZGQ1MjY5MWJkODEzZWZmNjQ4N2I3ZDk4MDA1NjFlOTI3NTgzMzlkNWVmM2MwYTI1NGFmMjY4MGU1NDhkYjlkYjI3MDgyN2M5MDhmMGYzYjdmNzliMjEwYTY0YzljNDEwOTNjMWFiMmQ4NGE3N2JiNjljZjU2NzM3ZjA2NDU0NjFjNmQ2YWFiN2MwZmRlYjljYjE4ZjIxM2JiMjBkZjM5N2NhZmY0NGI1YmMwMTkwOGNhZjNlN2EwNzBlMjk1NTNjZGUyMDM5MTQ1NzdkNGVkNmZhMDk0YTk2OTQ4ZTk4ZjQwMTg1YzJmYTg2NjVjMjYwNzQ0NTA2Zjc4NTYzOTc5MDkyZDNlZTczZmE0NDZiMDZmZmUyZDI5ZDE4ZDA0MTA3MGQwMzQyZDExNjNkZTBiYjk5OTUzNTYwNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YVSoFomq9FMtD1SuuGJ2s4bqqZUvqDLSotIXITRf-SmtQNWx_WfUpDniKjqRLlv5gATex9kAfgeRWU6PVvREog", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_102733_83_2429_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.492Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjFCdy9BZEdHVVRyUTdpRW1aLzk4bEdtSmc4QnAydS9TWkozT1AzS0FWU2pQVXpLY2crYWlOYVdrTDlxTFZvNUx0dGw4a29XSDB1cHFoTW8yK3hsckxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMTA1MjZfNjNfMjQ4Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDVkZWVmZWQ3MmUwMzRiOTY3NjIwMTgxZjQxYTA0MDhkZDM2NmQ5OWMzYmFkOGI1YzNmMWNiNzllZjc2ZTcyOWY2OGMyMmE4NGJlYjQ2YjdhMTQyYmY0ZGRjMzhhNzIzOGJiZDc2MWMzZjBiZWEyOGZkOTBiOTg2ODE4MzI4YzQ5ZWZlNWFiMjQ0YzllOTI1NDdkNzJjMWFjYWZlMzAxNzk5OGQ1NGEwNDQ0NzI3NWRjMzMyMGJmNDVmNjA2YzI5YTllY2E3MTc0NGIzNzBhYTBmOGMxMTM3NWIxYWU4NjEwOGVjMDhiYmIxZjZiZTFiOWFkOWNkMzU0NTc5ZjA5ZGZmYjUwZjkwNWEyOGUxYWMyNTkyODk4NzRiMDliYzhjNTUwNThiODU1NWRhZTBiNDVhODc0ZTgzZmYwOWM4MmVkNmUzNDkxZmI3YmI4ODliNDhjZTlhZjBhOTUwMzQzOTI5NTk2MWE3MGFmOGJkMjc5MjZlM2VlZjgxODg3NWIzMmExNWRjOGU0ZDUzZmViNWYzNDlkY2RmMWM3OWVlNWYxZGU3M2IxNDhjODQyZWVlNjdjMjRiN2YzMzkxYzEwZTIwNTFlMjRlOGI1M2ZjZjNiNTkzODkwMzkzZTYzN2E4OWRkYTIwZDgxZDEwMGViNjI1ZDA3MTA5YjE4YTZhY2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.P3jaSXsCMfl-t3tAZxWsIsannupjm8_DXLKSeMHOATsZdoUEGMBeF_9aLoSFGkb33muv_98sa7KPqNneNW07Pg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_110526_63_248c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.503Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Iko5L1lBWlBDWEg1MUU0RHZNZGloSUlaOWpqREhJekw1Si9wSnozdjBGQ1FpaWVlT0lOWVNHd2x3YjlOTmhSNEpxcEtSWEd2RWp3ZFFENVBJYXlBQ3l3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMTA1MjZfNjNfMjQ4Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDNiODY0MDE0MGMyNTIyNmI4NjI0Y2I5YjlmYTMzMjE3NGUwMmFmYTYzYjM3YmVkNWY1NWUwZjNkOWYxNjJlYTY1NDNlNTE0MjhkZmQwMjJlNzU3NTM5M2M3OTliNmMwYmY4MmY0MDk5YzY4YjFkZWNhMTU2MmQ0YzQ5MmVjZTAxNDdlNDE5ZWQyNzg5OTdiN2I0MDFhZGMzZjA0ZmY2ZTQ0MTNlNGY2NGY2NjFjZDMxNTJhNzkwMzQzYTkxNjEwYmU3MDYxZjRlNGIwMjQ3ZjNmN2Y4Mzc3NGRmYTQ4ZmZjNjI1MmM2MDM4ZWEzYWFjYzU0N2U2NWVlOGEwYjkzNzEyYjM5NmU0NWJjNzI1NjVlZjMyNDc3OTc5ZDAyYzY5ZjljYjVlYjQ1MDcwYWI2NDk5YWViNGNjNzE3M2FjMWU0YjdkOGQ5NzE3NmRkYzAxODUwYWJhNWIyYjMxMGNiOGQzYzgwNWQyNGI5ZjM3ODU3MDg1OTJkNTA3NDFjZjc3NjE0ZGUwYTQ5MzQwNzg2NzczM2ZkOTk5N2FlOGY4NmVmYzA4NTgyNjY0MWVmMDgyZjEzNDkxODE0OGQ5ZjNkMzM0YmViNjU4NTU0YzhmZmMwOGRjNjQ3OTIzYWMyMzc3MDczMjUxMzhjNzY5NzYyYTcwNzU3MWFkZGM1MjE4NWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WjEY60SE-93bYRd2k3fC3raYrEhQSy_xhad55mANkd1aWPBONj2imxse8u07ymicyyIOOyvWZB0e5-uwm9ku9A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_110526_63_248c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.506Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InJUaVppRytVNmR4Um94VHk1MzJId20wUDZicVorU2tLcVNnVk9VUitsdkFKc25CaDhHN25zNExZWmYrN1dod0hvS2VCR2xibmtlSkZENmRQWlpwNUlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMTA1MjZfNjNfMjQ4Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDZkMDYzZDdlNTg1MTYxMTZkMDA0MWExODJhMjNjYjJkNTE5OGQ5NmU2ZDMwZjQzZWFkY2U4ZGQyNGEzN2Y0ZjRjMGYwMDYzOGVlOGJhMGI4YmE4NTE2ZjA3OGI2ZjkxMDdhYWI1MmY5OGU2MDJmMmZhMWI3Y2Y4YzAwODBmZDE3M2UxM2E3Mjk5YTBkZmEzMGIwNDg2Yjg1ZWQ4MzcwN2JkNzVhNDE3ZGQ4ZDgzZTU0NGUwOWFiN2Y2ZTMxNjJjNTNjYjc3N2Y1NWFjZmU2ZGFkZGZlYmUyNzZiMTUwOTY5MTM1NzJhNWM5OGZkOTIwZTQwYTg5NzQ3MzNkZGY1MGJmM2I4YjY4NmQ3ZDQ2MzQ1NDA2ZjYwOTliMzczZGEwNmEzMzc3ZTgzMWFjYjVkZWE5ZjljZDQ4YzVlMWMwM2Q5MjIxOTUwZGRkZTRmNzkwM2EwNTRlNTBiYTEwYzkxMjBkODc2NDgxMjE1MTFiY2ZhNmNkZjU4YjM4ZDE0YTAxZWNmNzA2ODZjOTU5ZmE4YTc0N2RiZGM3M2RlOWQxNjBjMTExNmQzYjZmZWE2NGNlZWFlZWIzM2NkOWNlYWMwOGY3ZDhhM2JhM2M2YjBiYTdmYTI3NTJhNmM0MTNmOWMyMDJkMmI4N2MyOWJmOWI5ZTRjNjIxNzBmMmNiZWEzNjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wkyOVG4y_4Yn29LTuv5QBwaA1fEbznCeZRxeoSvTq2T9AhshKiQXRrbGZSBLcKH-rAVbv1gaGMRRBC2P8YIXhA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_110526_63_248c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.509Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im56RTFPVlNvM3VvbjVRdllia3c2T0RvbHcwV1IySmJMSGh1NFloRXhVYklURXZTQTNkMFFBcStmQ3krbng4YndhbkxKTnlLL3FUeXJWTFcyejFBSnVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMTA1MjZfNjNfMjQ4Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWFmY2ViOWVmNGZmM2IwMTM4OTJkM2E2MWQ5M2Y4ZTNkMzljZDc5ZjQxYzcxZWUzOGY1MWFlM2MwMDkzNmQzMDNiZmZhMDI4OGJkZDc1YWEwMWJhNjE2MWMxM2NjODgxMTcyZjgwM2RmYWY4NTA5MTVlNjljZDgyNWU2YzNkMzBlNDBmOWZlN2E0NDE4YjAxYTdkM2U2MGI3NmI1ZjM4OWVhMzk3ODc5ZjRhZDFiZWVkYzYxYWI3MzllNjNlN2M4ZDhmZWRlMTIyNDM5NTI1NjdiZWRkNWU4MGNiYmMwYTZjNTAwMzkzZmU4OGJhNWM5YzEwODA0NDY2ZWFjYjM5YzUwNDAzMDViOGFlNzNlMGMxMjVkMDkyOGRjODZiMzkwZTMyNWUyMTVjOTE4NjA3OTdlNTdkNDZkZDFlZmMxOTA4YWM1M2U3NDFjMmRmNTc5NmI4ZGQzOGNkNWY2YzExY2YwZjk4MzQwNjVmN2JjNjZjNDJmYzg2MjU0NjgyMjQzOWYxNjkzMzViOTc5MzZmOWYzODRlMmI0MDFjMGQ5ZjA5NjNjNjgxNGY3NjlkNjI0ODFmZjVjNzBhOGU2ODI5MjE2ZGFjMjhlZjY5ZGJiMzcyMmMwZWE0N2UxZjZmN2MzN2IyZTJmNmQyNDkwYzMzYjNmNGU3OWE3NzM4N2EyYjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.kuJ4hjr4TD0uqXCdWJg7saakAzaMN2aQ5kvQu-wXZlGU-YVB1K76nw0A4K2y_1fFeY8rceFxjMfbiJsN9KUz6g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_110526_63_248c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.512Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlBCUFFXYjdUM1hMaTFZbkI3bTR0RGE4dXg0OXNDU3hFNXpSMS91bWRYeW1aOFNUdkp2QmhtY2hlaWo4S09iSERlVkgrbTBUa1Z1MG5iRWlhRTNWU2l3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTA0NTZfNTJfMjQ3M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2FhYjMyOTU0M2RlZThjYTk3MzhlYTRjMDE3YWI0MTdhMTg0OTU2MmE4ZWM4MTFjYTY3OTczMTNmODI0YjY2NzAwZmY3MzEyMTJlODljNjk4OGQyYmFhMTEwYTA0YmZmOWNlZTgyM2I3YjEyYmEyMTQ0Nzk5MTYzZmNhZGRlMTBmOTE0ZTllOTkwNTQzOTBlMzQ3ZTA5NmZjOWViMDU2NGQzNzY2NjAyNjY3ODhkNDBjMDZmYWQyMTYyYzZhMzI2ZTcwNGNkODRmYzY2MzQ5NTQ1Y2Y0MmNjMGRiYzZmZDcxNDEwOWJiYTBhYmIwNTY0Zjc5MWQ3NmVhYmVkZmE1YzVjNTY0N2NjMjkyZTc1M2JkZmE1NzYwZmNmMGU3NDA2NzUwMzJiY2MyNTFiOTI1M2VjZGI1MmJmNWZiZGYzZTU4MWUxMGIwY2M1OTBlNDdlODA1Y2Q0NDlmODlmZmJjNWQxN2M2ZGZlNTc5ZWE2ZGU5ZTkyNmZjNTY5ZDkzNzc3YTRmZThjMWM1NTI2MWUwYzU1MGYxYTc2MWE0ZjVhNzMxNmQ1MzJhNzllZWQ5MjI3NTY0ZTk2MGVhNTMxMGMyZTNjNTBjMTMxMjRlODIwYTZkMzc4ZWNhZTJiMGQwNjYyM2JjNjI4YmQ1ZjFjODkwM2EzNjBkNjc5MDc0YWFlMzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OIndBag7sCpJFK9lDoJoVd2YND99Xkl9yI6EYcZk1V2R8a0quwBdBpwOxdBhHlVqep2v-qlniex1MyxrXjhIYw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110456_52_2473_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.514Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjdKR2JFWHFGanNDc01mMUx0YitpeXlHemkyQTVMYTR4SVNIRSs5UjI0V0s5R0QrZEhFRnhnOTYwKzZzMzZWRzgwZ21hQTB3UkxmYkExclNFa0x0SUVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTA0NTZfNTJfMjQ3M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTFkZGEwNGZlZTllOGUwYjJmNTk4NWIzY2Y3Y2RhM2ViMTU5NGI1NjY3YzA3NDk0NTFlMjE2Y2FhZTM0ZWY1ZGYyZjJjYTllNTFlYTk5YzNlOGY3ODY1YTcyZDhkZDMzZmE2OTAwNTE3ZjZiZGYxOTc0MDc1ZTdiZjlkZmM2ZmFjNTFmMmYwZGJiMGI4OTVlNzU2N2MwZjA4OTQ3NjI0ZjI1Mjk2OTdlNDRhYWMxYTIxNTZjNmQ2OGE1ZmEyYjM3ZjIyMzVkMGY4MmI4YWQxOTVmZTlmNjczMzE2MmVhZWJlOGI4ZWYxNDBmYjVhMGZjNWIzZWNiZGIxNjNlYzk4NDBlYTNlMmQyNjNiN2ViYjI0ZTE3NDJlZjUwYmRmZjlhMTBiZWEyODM0ZTVhYzNhNDY2MDJhNzgyODY5ZGJhZjA3ZGUyMTY5MjhiYjE2OTY3ODkyYWMyMjM2NzU4NGExNjA3Yzc4NTJlODk4MjczYzVlZjhhNTQ5YTJjNzZmYTBhMThkZWVlNzc2NTNmMmVlNDFjYjBhZDgwOGU0Mzc5NDVmOTczY2FmNDJmZjAzMzMxMTZhN2U3YmEwOTVkNTYwNWM1MWVjNDA0MzI4NjY3YjgxNDg2MjhlYzU5ZjFmYTRjMTJhNzk1MDNkNTFjYWQ1YzQyYWU4ODFlNzI0NjIyOTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-spHr14H4R7YhOaoClfPKDec3pGLT6HGMVCekVpRzto51rAk-rIFGlij8Co_24yoxQiJTQIZmM4bb9y74dZDBA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110456_52_2473_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.518Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlBkaFN2bG5XR01JR21EeUNYVnpUa21YL2EzNFZ0aWdOVFoxejlIT2JTWXk4UXVCWUtQUTJCQlFiNGpuekxmK0hkOS8wOTBqc3ZqS3VGVWczeG1MN1ZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTA0NTZfNTJfMjQ3M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTA4NGNmOWQ4ZTNmY2ExMzJjYzNhMTZhNTViOTQxOTNhMDEzZmZmOWUyMmNiNWQ2ZDJkZDJkMDNlMjM0N2YyZTZmMTA3YzhmYmUyNzkzZTQ0N2Q5N2NjYmM3MDBmM2NkYjM4YjYyMTE1ODI3MzBkNTQxOTU1MDlkYjcxNTUyOGIyMDkzYjRlZDAyNTFiNWJmZjZhMzk4ZjE5ZTk1NjU1MjUzMjI3ZTZlNGNmYjBjZTM1OGEwNmNhMGUxYmY2NGM4MWZlMTUxZTgyMDE2YTdiMTJhMWFlMjU0NzdmODFhY2RhMzk0YjQ2M2EwYzNjMDEzNjMzMTYzOTk5ZDEwMjg1OGFkMzA5NWNkZDQ1ZDJmYWE4NjdiZmYwYzY0NmVjNjZhMjQ1NzlhNzNhOTUyNmEzYTY5OGM1YjkwNDYwZmYyMmUyZjg2NjI2ZjYyODA1YjZlMWIyZTI0NTQ1YTM1ZWFhNTIxNzQ4MTIzNmNiNzA3NjcwZmM5MmExOWRiMGNhMmVmYTM0NzJmNjMzZDAwOWQwNzU0ZDg3MDE1OTNiZTRmYjU2ZjZhOGE0Yjc0MzVmMGM0NTI2YzA4MzkyYTFmZGU1YWE0YmIzZjgwZTYzYWE5NzViOWU1OTU2ZGY5MjFiNzYxMTcxODg2ZjJiMjY5NmJmODI1ODk0ZTQ1ZTE0YWU0ZWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.txXUVx7Rmi16dYmfW61DpQB6x57HgrHjootBPVuhhLXYyIjFB9lcvN1NPEG8GzbgldvhKJ7yEXYy9FUDpzczYg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110456_52_2473_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.521Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImRtcHVCWE04WWVMdFdGSFdPbGwwRFdYNFVuK0ZUd3JKcWtJU3Q0ZDZOV1dYcmtleXM2ZnBIZGZYQkJnZkM3SEdtdlprWWFrZWJzVE9zbUtaUFh0ZHRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTA0NTZfNTJfMjQ3M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTI3YTAzYzQ5ZmFkZDk0N2ViZmY4ZThmNzhkZTgwZGUyZDBmN2Y4OGZhMzA5NWIzNTAyOWI3MGIwYzE5YzhlMGNiMTA1MzQ5N2VkZTQ0ZWQ5YzgwNzExODFkZTEyOGRiNzNkYzNhODYwYzM1ZjAzNjFmOTMwZWZmMGRkNzAwYTA3ZTUyMWZkYzhmYjYyNjlmZmEyMzJiYzRmNzNjNjBjODY5MWZjN2I5ZDEyZDFjNzNmYTVmN2U4MTc3OTg2NjBmYzhkZGY0ZmMxNTU5OGEwNDg2MzYzOTQ1MTI5ZjljM2Y1MmQ5OGIzYmY4YTIxZDM1ZjdhOTE4NGI3YWIxZWUzYmMyZjk5OTkxZjRmNzE3MjljNDY2YWQ4ZGJiOWQ5ZDgzMzA1NTVmMTZjNTU0NzFiNzY3ZmYxNDYzNzNiNTY4NjRjYjczODc5ODA1NzdlMjU2YTEyZWRiNjhiZTExYzdkYmRhMzZmNmZjM2M2NGY1NjlhZDhkNTIyNzE5ODRiNmRlNDUwNTkyNGE5ZjhmYjJkZTYzZmM4NWRjMTJiMzFhODM3OWE4NzU3MGQxMDg3ODUwYjAwYjMyNjNiZWY5MjE2ODVkY2Y4ZmU4YjM4MmM3OTg4YTI3MmFjZTY5MzA5M2Q0ZTA0NjIyODkyMmVhYTZjNTc5MTM2YzMxN2Q1ZWVlZjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PK3HpMuFfYpnPYhIxBIAw9N5U4tFHzXR6COBOjNtfOiFpRUUoiM-1GwJnuGIYW60ROe8fua1PKUECMYpGMpccQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110456_52_2473_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.525Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im0wNkdFaWt3K3RpL3dOOWphMDVrL1cwV1o5WDIxU0xwQzJuME5rTXNOb1ZWNFNObnVnakhXeHNWSGc1ODQzRmxwOXp0T1RzSmNuVkxGVEFSNHUrTXF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNDhfMTJfMjI3NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzFkNjkzZmJhMTc2YTQ5M2YxMDM4MDliNTA4OTIwMDMwNWYwYzRkMDcyNGU1YjY2MmYyYmExM2ZhMmY3MDhjN2JlODRkZjk3MGNmYjc3ZWQxOWMyMDUyZTQ0YTI2ZjgzNTEzMGMwNTQ0OTBhMGY2NTAxODNhZWI4ZjZmMDcxZDEyMGU1MzY2NGE0NTI4YzUzMTEyOTFjMWY5NTJkMmU0NTAzZWEyYTBlNjQzMjFjZGFkMGVhNDIwZTRlODE5MWMyZmNkZDNjNDc5NWU4OGU2MTZjNTQxNWRmMTc5M2E5NDU3Yzg3NTA5YTM4ZWYyOGIyNThkYTY3ZjMzZmM2Y2MzMGYxYTFlMDJkNDU2NTA2YzdhNjI1ODYyMjlmYmI0YWYxY2JhNWVmYmFiMDI4ZmE1OWQ4M2U0ZTBjNGM3ODBkNjIxZWNhNDZhYTAzMzJiZWUxNmE2ZTQ4ZjNlMGZiOWQ3ZGVhZDY1YjBhMTk1Zjk4YWYwZjRkOTY4NjkyMWJkYjI2MzJjYjQxYWMxOTQwZjJiOTc2N2Q3MjkwZGYxYjRhYWU0NTMxYTExZGRhZDMzNDlkZTIzODBmNzk5MjAyNzc5YTdhMTJmOTRhNmE2ZDgxYjFjOGVjYWU5ZjQ3MGQ1ZWJiMzA3MzdhZGJkOGExZGY4YmIwOTVhMzg1NTJiZTAwNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GK5d_C5SUOCThGG-xi1hLu3hMLZbBRwV_8jpZLbaj_wj2ayY7WYvhjt1S8Wvu4bJi1AmjyB6W-bZE2MsK6rW3Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111348_12_2274_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.530Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkZxOTVmd25uRnF6QWpDam8wdzY2dGtYaWNIbVpGOWpienk4Z0M5UHlCZDdQNjB6ZkV1UGZqdjhMRWJmZjNzQnpxWFRXTmZJcUFvZC9KRmY4ODlkWDRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNDhfMTJfMjI3NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGJlZDRjZjQ0ODkwMWJhNjJmNzYyOGYyODRiMmJhOWRkZjQwODYxM2Q0NjI3ZmNlNzcwNDlkMDc4ZTlhN2U2YjA2OWVjMjk5OTQ0YWFkMDg3OTE2ZDA2OWY3YWFmMTVkMzFhNTUxZDIzNzRlMTk3ZTNmMzRjYWY3MTk3Y2NlY2M0ZWU4NGZkNWMyNjdmOWY5OTYwMGI0NzdmYTVlNzRkYjJiMzMxOGE5ZmZmZmQwODkzNTUzNTBmMThiN2NlYjk2NzNjYjUwMjFmMDgyNzg5NmFmN2JlNWU0NGZkMGIwZjVkYjJlY2IzZjRlMGUwNGY2ZTM2MDE0MGNjNmQ4MzJlYWJhZTBiOTkzM2Y4ZjY1YjEwMjZlNDY2MmY4YzllZWQ5MzZjZTAzMjdlZWFjYTQ2N2RmZGYwMWMwNmI5OGQzY2FlMzVjZDVhZjk4NTUyMmU0MjljNTgwYzc5YmM5OGZkY2M3MWZlMThkMmIyY2FkODViZDUyMTNlNTA2Njg1Y2Y5NmQ4YmIxNmU0OGI5YTBmN2Q4MDE0YmI2N2QwMWE5MzhmYWRjZDViNzYwMTBiYWVhYTQyNTRjMGJjZGYyNjE3MTc2NDBlZDdlOTdlOTJiNDM4MDE5ZWYxYmQ4MTdlMDJlYjAyNDJkMTU0MjMwYjY2YTQ1MjA2YmZmYWQ5YWRkZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZOGAQLBCGdnERok0l0RAseIPlleCI5tQVP4g4JZ8iTbKl-9FZ0FCsd9kslG2B1gG6ifKmx4r8VYek6mCfvawhA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111348_12_2274_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.535Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IndwQTRzY3pIMnVielVBMFYyd0ZlZXdYblcyVnZiZkhtVmxEem1tL1hLR1BjRlZiTGowN3FpNjM4bkxQOW4rSFhUNnlGYTE1ODRRazVUa25vOTBYZ1dRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNDhfMTJfMjI3NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2I2MDA1YjU5ZjgwYzQ1ZmMzNTkwYmVhNzliNWYwMjA3NWYzNDI4YTE3OTBlNzMzMTg1OGY2MjZlYzQxMDg4MTdkYTZlMGQ3MTU0MzE3ODRkYTY0NTM0MTdhN2M3MDc3NzJiYjY3YmM2ZWE2MjcxMWNhNGNkZmM3NGZmMWE5Yjk2OTUzYmJhMzc2ZjMxZmZjZDljMjk4YWFhNzZmZGFlNTQ1NzcwNGRlNDNmODIwMmI4N2ZmNjVhNGY5NWRiMjg2ZmU2MzlhZDVmNDgwMjBmMzEyNzhlZDRjOGQwYjFiNTFiMzJhMWU1OGI2MDJhODdiNDQ1ZmFjMmUwN2NmYzQ3MDcyM2NjODM5YTgwMDI1ZjEyNjdiYzI3ZjdiMDIxZTVmNzA1NTZiZjIxNDk3ZWQzNDhhYTc2MDMwYTg2MGE3NzNjZjM5ZWExNjFlMGNhMDVhMTc4N2E3NGE3NTU3OGNlZmQzNmZhNGMxNjEzOWM5ZjVjMjY5MzgyNzE4MmJlMjY4NzExYjEzNTQ3YzVjODA4Yjc0MTQ5OTFkNDU1OTFhNzExYmZlMDRkNjY5ZTY0ODdmOTVkMjBlZWVjNzczYmUyMGNmZjJjZDA1YzAwOWUwOGMxNGIwOGY1MjYwMGY2Nzc5N2Q2MDNlNGZlNmFkNmJhZWJjZThlNDBmMmQ0ZTA3NmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.emRbuNxOuCEOXRPrXe2v8fEJxETU_HB34h8PbVkp1sAI3L9d6d6ls-p5pQq5Sq-c-CG90zUiqC5s93szC74mZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111348_12_2274_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.544Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjFzc2FtNnJKWFZuZXZmOVUvOVpTeE9MYXo1dGI3M1BVUEVDRFdXVGtaYlhUNGlWT0pUeHFxZkI1M28yWHgybXdRV3VqcTFoSWp6eWZyS3RXcFkzRlFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNDhfMTJfMjI3NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDM0NzI1ZGI3NGQyNmY4NzliMGZiMTg1NzJlM2U0ZDI1ZDZiYzQ4MmFmMzgyMDJlNDRkZjEwOTBjZDY1N2M4YzI5MThjM2ZlNzAxMWUzNWI4ZjUyOTg5YTBkN2YxMTQ3MzExODFiODU5OGM1NmE0NTljMjBlNWQxMmQ3Y2Q4MDk0NDliMGU0YmMzNWMwMDg5ZGZmZDdiZDlkM2MxMzI5ZjZkYmE3YjgxMGYzODY0Y2IwYzA2YmYyODY4NmY0YThjMThhNmI2Yzc4MDY2MDdhY2U5MDg4MDNhN2ZmY2FkM2Q5MDRhNjQzMTcyYzJlOGRlMGViZjgyZjc2MTQyMjkyMWVmYzc3YmQ1MDVjYzBlOTllMzk5ZjE2NDRhMWVkMWFjMjQ3MTMxYzgzMDBlNzliMmExNDFlMmYxNmJkYTNlMzExODEzYWQyYmJlMTQxN2ZkNDZhZDU5MGI4NTY0MjRkMjMyOTg5MzFhNWM3N2JmMGIzMTI0NDVjODdmMmZiYTA0NjcxNDdjZTBjYjViMmU5ZTZmZDA2NTg3MDQ2ODllODY2ZWNkNzZhMmQ0NjhmMGI1MWM2Y2QwYjhiNjM3YmM0MDUzZDY4NzM1ZDNiZTg1OGI3NmRjNzZkNTg4ODJkMGI0NWYyYTc5OGUxMzA1NTg4OGQxZjJmNDI2Y2E1MjNkN2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Hu_9LA_NLq8XVuJ0TcOS9AOfPPTXG9GWNrMhXpW38XSzAxqo6udnYobB2yPWyFZ7e1AOSPFgPWfdV8Tm9Yw-Eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111348_12_2274_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.548Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IktBK0szbnV0WjVsc0RFeDNWYVgzK1pRUC9TRzQ4U005c2xDazY1VnpyR2d3ckZRb1dWbHNBSVJzdUYyQXNjSXBjNm1EZUVabXFHYU5CdkIrVDhvZk1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUwNV8xMDM2MDlfMDNfMjQ0NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI3MzE3YmI2ZmE4MTRiY2JmYTVjNjBjZDc0MmM2YjNjYjM1ZDJiZjM5YWY2OWU2Y2IzNDRhMGJiZjhmNDlhNmY2MWU2MzQ0ZDc3ZDQ3MDc1MDU1MjZlNWM1ZTg5MDczYzk2OThiYTI0ZDU1MDk1MzllOGRkM2M4NjdhMThkNzZiNWU2YzcyNWNjNzg5N2ZhMmI4N2U3NmU3NzIxYTM3MzI2ZWJhM2Y4MTIzNWQyYzY0YzViNGY4ZjU0MTI5Y2FiZWE5NzRiYWZhMzEzYmIxYjYzM2YyNjE2ZGQ0NWYxMDQ2NjhlOGZkZjNkM2Q1NzhkOGIyZmIxMmYyNDNhZTdiZGJmOGU0ZjQ5MDhlNGU5YWFlM2RmMTIyN2FmNzQyOWY5NTgyZGMwZWM1MDljN2RlMjUyMzFkYWNiYWE5MWQ1YWY0OTBlZDEyZmZiMzBjNTk0N2QwMmYxNWEzZDhhNTQxZWY3ZDM2MWFlZTExMDJhNzEwOWE0ODY0OThjYzE1MDljMDgzNTVmNTRlY2U3YWRjZTk0OWJiMWU0MTMxMzAyYmZjNTQ1OGFhNjQ1YWIyY2I5MjNhMTNlMGRjYTkwOWM3MjNmMThlNzBmYmIyZGJhYTdkNWJlNmM1OGU2Y2Q1ZDdmZjZjNGZhZTc4NGViMmZiMTE1NWEwMTk5OGU2YTA5YjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9ffEZM3taUEd2FxEtcP-pLjQWtYPU0nR0LNvMP7ER8TCg6PBPnKlCI8-lrljPm5mNpdYYh3AesmW9q82xgjEOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210505_103609_03_2445_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.552Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ijl5bFgrL2sxMmVLeTJFUXl2ODRFK1BYWHRLd205aW9kUFZMNDFXRTBHa1Rsc3RTbmJra1JHTk5wQ25LY01YN2QvTCtWRDlmWlhlaEVZTyt5ZFBNdUFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUwNV8xMDM2MDlfMDNfMjQ0NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjZjYWUxMGM3MWRmMWFjOWZhYmI2MDEzYjNmOTM3MWIzZjQzZThjYzc0Y2Y3ZjY0MWE3OWZjZTNkMWRhMjc1NjgyNjM5MzJiZDgzZWM4OTYwMjc5NGM4MWExZmQyM2I1NmY2OGM2ZDFmZTZkOGI3MGMwNTQwMDIzMmQ3ODBiYTZlYTNmY2QzYmE2NDAzOGJiMzUxYzU5NzA2YWM5MWEzZWM4MjQ5ZjZlMGYwYjM2MTRkYWU3MmE0MmU4NTE0OWE2ODQwY2Y2ZTU4ODU5OTdhZWVjNjc5NzlhMDAwY2YzYzc3YzAyZWQxNGZkN2FlZmRhZTFmMTI3OTAxNjg0NmJmYmExNzE4YWZhZmUxZTYxNTc2ZjY1MWM0NjUwMTFkZWM5MzJmYmE1MGU1YjMxZjRlZTA0ODJjOTRhNDUwYzI1MGE4ZjAzNTlhZTM1NzM5NDYxMTIxNDhmNzYyMjFmMmU3Nzg4NmU1N2EyNjUyODU4OWZjMThmY2RlZTVjNWZmNmZlODdjMjc4MzViOGVjNjg4OTMyODI2NmRmMDk2ZDlhYTU5YTVkZDcyNTdlZDhmOWY1NzQwMjc5YjQ0MmJmY2Q1NDk0ZGU1NzUzZTBmY2ZiMzFmNmIxYjUzMTE1NGUzNzdmYjhhY2NkZDM0OGQzN2FlNmEyZWIwMTdiZmFjMDMzMDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gU8IGO-aaveFGDzkqWCQboi3V19TNojziG8AkEPPa9uLC74ZVu2D0h-FsZSgDNYY9ZI5ScGKP2UAjeXMcuKUnQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210505_103609_03_2445_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.564Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikt5SUhkbzQ4ZHl2UmFxZ2Zpck55UnRBRDMxbEVGMmpiVUFJK3dZZnFJbXE0VW1JUWJjamlxZVJ3MDVRMDd4RmZOREN1cFV3K0Z0aUhBUTJ3R3NERVd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUwNV8xMDM2MDlfMDNfMjQ0NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzE3NGE1NzlmMTQ1MWIwODNiZWY2NjA4MGU2NjExMTM5ODAwNzdjMTlhN2FmMTkxMGZjZjU4MTY2MDhmZTFkMDQ0ZjRiYzViMTIyNTY5M2EzNDc3Yjc3NGFkYmFmOTA2ZWE5MWQ3YWE5MmNiMjBkNzA2OWZhYWI2OThlNjI4M2ZhYTA1OWJlY2JmYzI4OGQ3ZmFmZTc4YjhmNThkYmE1NzdmNWU4M2M0NTgyYTQyNWQyNDQ3Yzc3M2Q0NzdiZDEwMDkzNWZmNTM5MGM3Y2UzMWJhNDNmNzI4NDU1MWYxYzM0NjFjYjY1OGY2MDEyNDk4Y2QyMjkzZTliZDVmMjNjM2ZiMzVmM2RmNzM0MjdiNTA0OTY5YjJhZDU2MjNmY2MyMzExOGU1MGEwYjhiOGI3MWNkZmI3NWU4NTU5NWRkMDdjMTAyNGE4MjViMThmOGIwZGIwY2JlOGI4NzhlM2U3YTkyMzIwZDc3MGVhMmEzYTJjZDU5NGEyZmQxNDlhYjM4NzVjMzJlNjI0MzcyNjM1MTk1ZDJkNzFiMzcxZDQ4ZWE2Y2E1MDU4ODdkOTE0MDM5M2E4NjdiNjkzMGM3YThjNzFlMDY2OWZlMmRkNjBjNTUyNmY3MjgzYmNmZTlhYTVkMTdjZDI3MzY3YzI3ZDQ3MjVhMjgzZWFjOWFhY2E1NDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PZLgOxd6NlESugB9vGBIbUKGgWtqPenraiwWjbVEQY3Re9bx59j_B5YIBgOLYmqMc4uI3TH5qc22lpW6DPBoPQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210505_103609_03_2445_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.568Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InFpYXlzMExoMTZUbm93bUJsN2Y1Qmx5aGhsYmU2K1JLdkdWVTlIdVdhbWlnU1hZWEowQ3MvVjBHTmd3b3JLdWg0VVNDVHF3WU5EUjhyRVowVXprTWdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUwNV8xMDM2MDlfMDNfMjQ0NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjNkMTQ1NmRjMzdlY2I0YWM3MGExNGM3OWUxNjYwMTY5M2M5ZWNjZWYyM2ExYmY2YTY3NDZjY2RmZjNiZjAyM2IyYWQ0ZDg1ZTY4MjY1MzYyYTk4N2Q4ZTMwODQzMDIzMTc3MTUwNmZjOTY1OWU5M2E5ZGY2NTk3NDRhNjg3YjZkMDQ0MmVjNmJhNzgyMjI2ZDIxNmFiMzViNjMzOGY1MTkzNWMzNTliM2M3MTQ3Mjk0YjAwZTA0ZmNmYzA0OWJiMjI5YzlhNjk5YjYxOGQ0NTlkMjlkNTdjNjdlMmFhMGNjNWQwYzQwOTlkMzJjMDJkNjNiZGZiODVkNjc5ZDAwYjI2ZjM4MTY1MzBlY2JmMzQ1ZDQ5NjA2N2MwMDY1ZGQ1ZTczZjdkZDUwNTYxNGNkZTMzNzg3M2MyMTYwMzc2ZTMzNjYyMzY1NDU2ZDdhNjFmYmU4NzkwNDYzNzdkM2YxMjNmNTRhZGI4NWQxY2E1MjgyMWYyZWNjZTAzMjI1OTQxZTI4NDRjZGIzYTc3YmExOGYyOGU5NzRmNzhjZGJmZTc1ZTc3NjU1MjBhNjdkNzY0ZjU0YjExYTNiN2MxZjdlZmFmNTNkNzZiNGNmYjg2ZGVjMmQ5YWQyMmJhNjVkNjBhMzk4N2JhZjhiNTExYmFiNDZmMDI3YzQ3MzI2ZWU1OTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Nn7mOxTB4NAo3WssQaSMLa_QHw0SyyoDTzuwkRkaWsr4sjfEiTb5Rd0K_gjDCHlA6MlyaD0tdB20gtmoBYx54Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210505_103609_03_2445_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.573Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImNNZm16aGtLVG9kQVROT3BTTmRVUGs1TXh2cmYrSmtaaXU1ZDB0c3piaUpoMG01LzJVeXRJbTkyWENHWW95WXpTR0tmWi94YksyZkt5TzhEdWp5a0JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMTA0NDVfMzVfMjQ4MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGVhZTc4YWM0ODBiNWIwY2JkZWMyYzQ3OTc0NTAyODUxMWI4ZjQ3NWM2Y2UyYTZlYTBjZDRiNTBiYWU3MGVjMjVjZDdhNzJiYzVjNWRiNDc1YmUxZDY5OTE1ZTQ2MTQyMjc5Yzg0MzAwY2Y0NDY2ZWE4NDBkZjlhOTI0YjViZjY0ODkwZjU5NTY0NTVjMGM2MTBkZWZjOWU4N2E4ZGUxNjdhNDFmNDNkNWRiM2E5ZWJiOWU3YmQ0NmE5M2YyNmVlODYzZWNmNTU2MjBiY2M2NDhmNjFkZWZhYjBiMjRmYmE3NDc0NGNiYWEwYTExZTI3N2JmNGQyMGMzZThkYWQwNWQwNzNhYWE3ZGFlNDhiZTk1YzNmMWViZDMxNTZlZWU1NzhkYTk2Njg2NzA5MzUzZDY2NzA2NjYwMGJlZjFhZDFhNzhmYTRmMGZlM2IwNGJmZTA2NWZlYzZhYWY3ZTYwM2EwMjliYjYxOWRhYjA4MGIxNmY5OTFiYTVmN2IyODgzNDYyOWY3NWJiZjc5NzZjNWM1ZGZlNjRmZGU3YzZlYzgyOTRmYmRhNTBmYWExOTEyMGQ5Nzk3ZDFkZTZhM2Q4M2E0NGIyMjU5MTZmYmE3NWEzZDM0OGU2NTEyNzI3NDE2MWY1ODFmYjY2NmQ3NmEyNmIwNzkxMzkzOTU4ODczZWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qObMiwMUIbyjeidhAxjZ2-SIUcj80xs1QN0Yry-h3gcp4aumYqYXm5LoThQgS13T-kgOMTOBPRNfmDyaokveEA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_110445_35_2480_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.585Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkpjMnlFZm1lWklvcFBObDBUL296L0YzS2czSHhzcjNSc2xMT0ZXMENUa1ZuL2swYkZiZk1NdU1wQXB5TmFlYUNianFHaUsyNzhhbHc1SFdZcWNKNU53PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMTA0NDVfMzVfMjQ4MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzlhYzRiZTgwNDdkMzFiY2U2ODIwMTc4MmU1NTQzMjNkODJiMWQyOWYzODZmYmU1MGVmYjg5MmE3MGMyYzlhMDUyYjE0YzQ3Njk4Nzk1ZTc4YzAzZTg5YzdiNzg2NzgxZWQyMDBjYWU3M2E4Njc1MDM5MDc0NDUxMjAwMGE0YmMxNzcwZjk2NzIzODZlNzY2M2Y3N2NlMWE1MjRlZjM5NWE1NDdjZDA1ZjYyYWRmMzRmZmEwYmVjMzhlMTBmNjZmNDRlYjc2ZDMxMTU3MWVhNmJjOTYxY2QxNTllOGJlMzJlMjExNGRkMDNmMWY4Y2U1NzA5ZjZiZjhjMmIwOGY3MDgyOWMwOWZhNTQ5MTQ5Mzg4MDg5OTgwYWE1YzgwNTFmOTk0OTJmODkxZDIzZWQyYTY3ZGIyNjU2OGI5YWNkOTU0NWY4MzQyMWNkZmRhZTdjOGVjZjEzYTlhZjdjNDVkMmFmNjM4ZGVmODEwNjc3NTQ0M2UyNDMyOTNiOTAzMGIzOTZiNDc0NTNjNzFmMDgxMjRjODRmMzRhZDZhYzY2MzQwMWJmMmIyY2E5YjVlOTQ2NzFkOTU4OGNkM2QwOGM5Y2FkOTkwZmI2MjgzODI0NDBhZDJkOWJlODM3MWEyN2I2YzI2MWIyZDExOGY2YTZlN2RiMmM0M2ZiMmY4MTkwN2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5iBEcheGOsQggslTdYIaVJPWVQjiDMMkzGusKDl6Gosd9175AZENlniM014P54999M3GQNf-2sHtHyr5k1QzDw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_110445_35_2480_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.589Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJWdHdEM1RDMGs2c1VpRnREdjJQMFdUWWVRRWVkZHNaeUptV2J3cVJScGhTL2V1V29VVEVSUkY1VVdSbVhDeEdHYk54dHEwYTZLWFNnYmRIc3gweTlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMTA0NDVfMzVfMjQ4MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzY5Njg3YzQyMmYzMjEyMjQ2OGY5ZjM1MGYxOTRjYzZhNTkzNzk2OWI3ZmViYzU5ZjVkMDRiYjMyZTdkMDFhNTYwZWFhNzEzMDJiYzI2MjY3NDk3YmE2MGIwZGQwMGUwZGZiMTM2NjdhMDEwZjIxYWMxMzU5MjA2MWIwMmUwNDIxZDdiN2Q0OTNmMmJlN2I5YzY5MjEwMGQzMzAxMzYzYjIyM2VlZjY0NWI0MTM2YzQxYjhlNGJhZjU3ZThiZmIzZWE4OTcwZTIzZjZhMzhjNDU2NzliYTliZThjMGQyZmM4YjZkNTRlMmM0NmY2YTZiNmJhOGQ2ZDA1NDc3NDEwNmU3ZGU2NGI4M2JmODdjYTRhY2E1ZTZkNjljNGJiNmEwMjQ0MTUyZDdhMDgxNzdhZmU0Mzc5MmQyMTcwNThiYzAzZjNmYTIyNTFjN2E1OGRmM2I0YzBjNWE4NzAyMDVkYmJiY2NhMzkyNDA1ZmI1ZGU4OTNiNzE2NWJiMGFkODhkY2ZjYWQwZmI1ZDc0ZWVkNzJkMTM1NWFjZDUxNGM2ZWZlNWZmNmY1ZWRjMGZiODMyM2IyNzUzYTE1MjgzZDhhMTVmZjI0ZmU4MzBlOGQ5ZjA1NjJhMjEzNjVkZWUxYWI2OTI4ZDNiYTQwZGFiOWM5NjhjZmM0Yjg3NjMxOTNjZjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.E9L1uVQyzoRw_ZbeBvHiVhuHrZq7dM4jax_3o2hQgg4RXIwlC0pTqkszLFvtoksbuCKiwbDVGm7gwT8TGBzQvA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_110445_35_2480_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.592Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkFocnl5QUt1V0REWXJaZHR3MU5welR5UGExNzJnRUZJM1dZQitBenZDOURDQWdUUkE4blVib2N1M0V4N3BodCtFR0ZQRkR0eWJaR0ZYdXJmWTBvcW9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMTA0NDVfMzVfMjQ4MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmFmODljYTcxNzEyNTAyYzRmMGFjN2QyNTliN2NlNjMxNDczMGM1OGFkZjI4ZGViYmQ0ZmMzYTg2ZjlkYzdhZjQ1N2Q5Yzk3OWU4MjA3MTA3NzQ1ZTczY2RhNzNjZTEzZTgzYTdlMjU1MjU1NmQ3OGRiMWQ0M2FkOGEyYWQ4ZDFiYjFiZTE1YWJhZTExNWI2NzAyNDAyZjgzZmI2MWFkODE0MDJhMjBhNWIwMzllZWVkNjA0NzQ5ZjkzY2NmNTQxY2FhZjBmOWI1ZGU0Njk5ZTA0Yjg0YzNjNThmM2FmZWVlNjcwZDI1MjY2MjVkOWEwZjk0ZmU4NzZlMDg0YmU0MTBkNTQ1NGI5MTcxZjM2YTE3NDE4MDc5MWNmMTQyM2I4MjA5ZWMxMzI5MTcyZDFkMjYxNjRmNTc3ZmZlYjE2YTY0OGYxMDJkODhiOGI1Y2QzZTY3NzljZGUzNTM4NTc3MTJlZGUwZDg5ZTYyMWU5ODY4YTQ2ZDU5MmQxMzRjMTgwNzJmNTAyNTAzMGIzNDIxMDRmZGY1NmIxZGRiMDcwZTY2YzZiYmNkZmYxMmUyNDdlNzM0NGQyNzQ1Y2Q2N2VlNDRiMWJmYzc0NjY4NWQ1MGI0ZTVlOTQ1NDRhNDFkM2M5MWZiNmQ0Njk3ZGU5Y2NjMzM4MDdjMTgwMDkyZGE0MDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rwjdkeG28O7lxUQqJzUyTlsk0suWq11G6czn9J3irMJu1X5YA774VDQMfGlsNVOD_uOVv-X3lkur2KSAwd_HwQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_110445_35_2480_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.601Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ii9rYmJCVk9xUzJheDBkMGszMGxISDVicXF4VkpoYWFNR016S01sOFlITTZURENMR3lRb0c0K29najlQYWozQzFycE94K2pMeXE4NG1jaDdUV3RNMURBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDFfMzZfMjQ1Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzIwYTEyNjU2OGNiZWM5ZGNmN2FhZWEyZjQwODk1ZjQ5M2ZiN2I1NzJlMGZhYTdjYTQzNTMyMWNmNWVjODViNmNiMzI4NzlmZmQ1NDk3MzdjN2E1Yzk1Y2RkNDVkM2QzZTNhN2U3MzIzYzcwNWZhMWE4NmRiNTFlM2JhNGFmOTZlYTBiZTZlOTM0NDEzMzczYzFiN2UxNzkyYWQxOWY5NTM2ODMzNGM4YWU4ZWExYzBlMWNiNGJhZWFkZTNhNmVkNjQzMTk3NWUyNTU5OWQ2ZDRjNGJmMGIwNzU5NGNjMjBjZTU5MDkwZTUyMTdjYWY2NTIzYThlMWMwMjhhNWVmZmQ0NzBiNGU4YTEyYmJlY2FhODM1ZjI3ZDczNGMwYzFlMzM3ZTViNzVlZGJiMDFjYzIxNDk5MDBkYTBhODlmZGFmNmIxNzIyNjBjMmQ4ZjhjY2YxZWM5NzdiZDlmZTdhMjhlZDRlODY3M2YxZjEzNTYzMjI5MzAyYzRiMjUyMmU0YjA5N2IwOGI1Y2Y0ZTI3MmYzMGIyMjc2ZDU3NzA3MmQxMzI4OGY3YjdiZjZmMzgzZjUyM2IyNTg2MGQ4NDlkZDhkMzk3OTcwZDE4NTM1MDEwNDU4NDhmNGZjYmM1MWM0NDAxYjJlMTBmMWQwMTI4ZWU1YjhkZDJiMGY1YWQ2NmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GYmGfmmTJWXR9BOfJ1e88yMf4_rApWPE6ch4YjHkOXBQ74nHrFarUyDH3jmhOebv9SrKXnN53-lbC_DDOAYQgg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103501_36_245c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.607Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkFiUWFmOG4zSFZCQTZGLzlZTmdtMER6V2o3cnQxZW15RGZiak4wZVNadlQ5blpkNlpZOWx2K250ZUVxenlzS2hvbVRvUEJoMGZROFlsUEFJVDdQVENRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDFfMzZfMjQ1Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTU1YzQ3YjhkNGUzMzdlNjkxODc2YmE4NDc5MDA5ZGNhM2FkZWU3Y2U3OTRmMmFmYTExZTFiOWM2NzY1ZjkxN2ZiODU5M2ZlZWQ2MTA1ZjAxNzJjYzFlNzQ0MzhhY2JlMGJhZWVhYjRmMzJlYTM4Yzg2YjVjNWQ3OTJiZTgxMWIzZWU2NjUwYTZiOWQ1MGFiZThjZDUzOGQyNTg4NDZiOTc4YTY2MTlhZmNjN2U2MDk4Y2Q5OTAyMWNlN2JiMmRiYjM4OWY4YWI4ZDc3NGVjZTZiMmNiZWVhNGVjMjQ1MmI5OGEzMDdlNjdiY2Y0OGMyMzYxM2MzMmY5Y2E2ZjkwNTQxNDExNjI5NTRlMTllNjc5OTU0ZGU0NDQ4MDE3MjE4MDNhODE1ZDAyZTgxZTQ5ZmIwNTU2NTAyYzM3MzIyZmU5NTYwZWY4ZDJmMjA5NDI3ZDJiMzVkN2U4NDY4NGJlZWNiYjlmM2JiODUzNjc2NzZhY2ZjODg3NTA1NDMxNjkzZjkyZTljNDY4NTZlNGMzMDU5ZWIxODAzOTExNjY1ODQyNmZiNGE3NjhkNzc0NWUwZjQ3MjlhZGFiYTI1MmQ3NzljMjk5NjgwZjJlODQyMTY0N2YwZTAxNTUxMzc1NzQ2ZDVhMmI5YjYxMDM3MmIyN2FmYTIzYTkyMDJjYjUyNmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iPSbbT2UNrHUbppqcS110vLPkVsy1BN-pCJNkctZx3njBaYJ86Qv1APs_Dw1HLVwY_XJo7M8HnyZnrq-UG3Ukg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103501_36_245c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.613Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkZOSzRWUXp1K04zWHB4QWdTdmNPK3pQWEE5UnZocXdJK1E2NGFCNHVvTkFCZHl3S1c2Y3hqcTNzeGVwWW51KzRxblNzS0JjZTNYNExMY1hia25yeFJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDFfMzZfMjQ1Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODgyY2NmNDlmMGQ2YTg0ZTgwZGZmYzExMjQzOTljNTNiMmY4MTY0MWY2NmYxZGYzN2I4YmFiOWEwZGU4YTM2NzcxNmQxYTYxMTg3MmQyNzE5NDA3YTg2M2ZjNjI4ZDZkYWIxYzY0NGEyMDljOGEyOTMzM2U1MGMxZGY2YjU0NzFiMzEwOTExMTM3NDFkNGY2OGExOTAzNWY4NWE5ZGY3MzkwZDkzZWVmNmRhMjA5ZGY4MjdkMDY4NjMyN2EyNzZiNjZmOTY0NTE3OTMwMDBiOWZkZDU2NzIyZTRmNTczM2FkMjBkMzBhZmI2ZjEzNjRiOTBiNDAzODJiYjdkM2VkMThmMjk2OWQ3ODAyZjM0NzIyYzIyNzg3YWYyNDdhOWY5YTVhOGQzMjU1YTY2YWNjZDExYzk0NGMxM2U5ODRjMzAxMDgzYmIzMzE3NDYwMjZjYzUyNmVjN2E0N2U0N2Y0OGI4NGVkMDZkMDBlZWI3MjkwNzU0YTBlOTM4NmFlNDI1NzAzNGJhNjBlZmVlYjM2MzQyZDM1NDE1MjgxZDk2NWNiMGU5YjdiYjNiMjYwMDg5ZWE5MDRlZWYzMjgzNmIwODc2NGRlNDFhZWM5N2JkOGRjYTJjZmI1YmVjZjUyOTZiMzAwNzE2YjRmYTZhMWJlYmMzZDNkMjgxMzFhYzlhZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dJP6y9X-_WVU_Gq2eioWX_0_9SFgyxUU3tIhM6j2T1jMtzOt6xwGoRX9TjT06OTyaHirH8ltadbJ_MJcWcOP4A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103501_36_245c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.618Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im1ZemgreEVsTlVWbHJJTTJGd1ZMSldCN2ttbHdURlBNM2FTVk1PN3hjQzlpQXFsZGlsWEE3SUNVSE9TTUVnMFBlemlCQ0wvdGpkWjdOL2hmVEFvVEN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMV8xMDM1MDFfMzZfMjQ1Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDhkNjk2NWViNzk5M2E0YzIzOWIwNTQzNTQyYTk1ODExMWJmY2U4MjhhMTk5ZTdjYzE0NGJlZWNmYTk4ODAwZmM3NGY0YzcxMDliZDEwNTEyNmVhZDc1NGE4YTliMGJmZjViOWM1NTI3NmQ1MTZkNmE0N2RlZjdjNjE0YjBlNDhmOTA4NDhiM2Y5NDk4ZTE5NDY5YTk3MDYxZGU5YzNkNGM4MjdmNDBmOTBiOGM2MjZlM2I4ZDZmNDMxZjExYWU3MDZjNjNiNmEzYWM5NGNhYzVkNWFjZTk0YjVhMTJlNmMzYTZiMzUzOWUxZDQzZWE3YThkNzFkMDVhZGQwY2U0MzMxY2I0N2Y3ZjA2YmMzNDYwZGJlNGJlZmJkNTk2NGUwMjM0NzFhMmZlOWNhODNkOTYwMWY0Zjk5MTU3MGIwN2MyYWQ1Mzg3ZjVmNjBiMjI2M2FkN2I0ZTEwNjU2MWZiNGY1NDNiMmIxMGVlZTlmZjUyMzEyMjMzNzNhOGQxMmYwNjA5NDExYTYyN2NhMGEwNjNhYzRhMDNkN2U1MGFiNGUzYmNjNzUwZTkwY2JkNzBhZjgyZjg1MmIyZmYyOGExOGZjMzEwMWUxNWNiNGNkNWZmYTc1ZGZhNWE0MDVmM2IwMDg1NzVhYTAyZGM3ZjVjZDYxMTc4YWE4ZGQ0ZTAwZDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UpD6Hmkay0DyKyr9fP6dK4OpTk2XDf0Qcuqn00lIQ51Uf_qJY1sKGj7VKq-yno6qrhg6_AWMJh9-Y8nRooSOsA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_103501_36_245c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.621Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkwxaERoR3Q2bEQrZ3pNQTBWK01wa2JGR0MvNzJEaWp4QXFOR3IwWVlFeFpzNDlhMmJkRXRxL3o4NWRob240N0p3SXZscmRzOEI4NUdRMGlRNCtidE1RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwM18xMTI3MzBfMDhfMjQxMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjIyODRlOTY4N2Y3OWNkODVjOTc1NzA4Y2ZlZDM4ZjdjNGVmNWQ0NmFhMmIwMTAxODIzMjdhZjAwYzA3NWM2NWI1YTc3NjE4NzIzYWRhOTBmMzgwNWM2OGQ3YTRjZWUwNTZlYmEwYTcxY2IyNTVkODkwN2Y1ZTQ2NTBjZGI0OTgxOTQwN2Q5OTYwOGQ3M2NjNGU3NGNjYzg3MWI5MTU4MzE5MmI1OTdmODRkMzBlNTg5NjgzNTFhOTIxNzJjZWM5YmM3YzVhODAwZmU4ZGU1ZTg3YmRlNGE3ZTEzYWQ0MTM4ODAzMmEwODM4ZGExOTg1ZDY1YmY1OTEyMTEzOGE4MmQ0ZmFiNWVkOWI5OWY2YTg4MzkzZTkxNjQ3MDA4ZTcxMTY3ZGJkZThjYWZmYTRmYjRkY2MxOTM1OGE3OTkwNWI4NjMwN2M4MWEwYjBmMGExYjExM2Q1NjJjZGIxZDA4MzU4MmExOGY1MzE5ZGZjN2JiZTczYzkwZDg4ZDliMTdhYzk5YzMwYzFiMGQwNGZiMDgyMTA4Njc4MjkzNTVhZWU1ZDlkYmM4ZDMyMWE4NDNlMmIwOTY2ZmM0MDVhZjY3ZjE1YTIxNWZjZmQ0NWRlNzdjZGE5MzI4YTIzOTkyMzhjMTE5NWUzOWY0ZmIzMzhmMGNkN2JmZjQ0ODZhNmIxYzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ul_AXWXAZY5JgV-TZc4_NW2xRq6Ex7BqY8E0SRNLlT6Cppwf_xgmxkCkZ6yZzfVLde2xKPWUniqCa7kga4dzMQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210403_112730_08_2412_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.626Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IndpZStXRTl5YmVZQ254elphNlBDZFhPOGppK2RxY1hSRUpvTHYzM0pJRk5MZTZ5bGxSTXY4UjhHMmd3ZkIvNGg1YWhEenhKV25idERNOHdESnpiU1pRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwM18xMTI3MzBfMDhfMjQxMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDAxYjc5MDA4MzA1NjdhZjZlMWE0MDVhYzA1ZmZkYjg4Nzc2MzQ5NWRkMDEwMzE5YWRkMDA0ZDJiNTlmODkzOWM3MTVlYmQyYzNmY2RjNjE4ZjQ5ZTU0OTkzMzg2OThjNjk3YzY2MGUxODdmZGIwYjhjMzNjNzBkY2E3YTA1Y2YzYjEzODQxMjc0ODBhMWEwMmJjZjYxZDhlNDljMzIyMjE2ZDc3ZDQyNzU4NTAzMzIzZjdkYzZiMWE1NTZlMmRhM2IzMTAyZGVjZmMzY2RmZTNjODc1ZjU0MTEwN2I2ZTBkMTdiNjJjMDJjODhmMjY4NjNhNjJhNmMzNjE3ZGE0YTEyYjJhZGVmNTkxODZlMThjOWZlMjJkZGQwNjRmNzdkMzlkMDAwNGMzYjJmNzEwMjBlZThlNjhkZjRiZTdmMDY3NTM2YTY1YWVmNzI2NzEyNmMxYmFlNGRjNmZkZDY0MGQ4OWNiYTc0NjQ0NjA0MDE3ZjRkNTdjZTYyYzJhMGYzNjU0ZTcxMmQ2NzAzOGZlMmQwODQwNjA5ZTMzYzg4ZjUzZmFmMjVkMGUzNDE0ODlhNjRhNjRhZmFkODE1Y2Y2NDIzMjQxNDMwMGNjZjFlZmQzZmUzYWYzOGQxMzk1YmY0ZjMxYzU1ZjY1MThlY2MxMzAwY2UyY2M0ODE0NzM4ZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RLMyxAwWJpSCPgyzJqVFg82EBETM_qulODzYRIWlaCIt0TknSNE2i36gcBGAX4yFoJh36OyWqLZIw6RoSA7pQQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210403_112730_08_2412_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.632Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImN5ZVRpZG9rdnFOVjhpUmJRZzBCcTlEZ2EyWWZqREM5TmdOOGxCUGFKZzZMK3hJL2JHSWsramZCZkJvV2UwN3lka2ZueEdIR0VKVnU4Wi9BRjBIbm9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwM18xMTI3MzBfMDhfMjQxMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2U0NTgyMTUxOGNiOTk0NTc0MGMyNTk3ZmZmYTY5YWEzMDU4M2IzMzRhMWIyMjBhY2VkODU5OGM0OTJjMjViYmVkZmFhMjMxYTk4Y2QyNThlZWExNGFjNzA5MDcyYjJjZGJlN2FiMWNkZGU5ZWUzODc2YTcxYzk4ZjkzOGYzOTVkZDcwODU4MzkxMTY3NDFiNWQyOGY2Y2NlYzM5NGJkNjVmZWNhZTdkNjFhZTA5MTEyN2EwOGY0NDEyZmM0MGQwOGMwMWIyNTU0OWU4ODU2MTFhMmJiNDZkZWUxNWFiMjVkNjk5MGZlODY0NmQ1ZmNlYWFlNzYyNjJmM2RlYTY3Yzg5MzQ4MGVjODc2NTQzMzNjNzUzYzhiNGVlZjI1ZGMxMmE2ODkwN2U1YWU2MzVjZDliNDA0MzM0NzBjN2VhOTA1M2FjNzBmMWQ1YWM1MDBiNzA5NmMwMzkzM2IxNzVlY2ZlY2I0M2NiZjI3NjA2Y2FmZDE5NjViMDIwMWM1NTM5MTIxMzc2YWVhYzAwMjA2YmEyZTYyMTRlZjQyMWFjODAxMGJiMTgxYmUxMWZkNTRiZjk2N2VjYjFiOWI1NmExMjI2YzkzNzgwNTc2YzM3NmY0MzA5MWQwM2I2NDg3YjEzNmM1YzQxZDU5NTJmNTMxYzVhMTA4NDNjNWQ0YmI1N2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Yhmk7E1u069jYt3ttkMPbqDcds28VccnxqU5I7J1vjamB2WsJLbf8vEr4QLp_Fgo4mX1Vv1jltsP9qJWl2QfEQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210403_112730_08_2412_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.635Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IncrUHlBWWtaSE9OSGhDYUxTSkl0QkVzenB1aDFoYy9xWEwwelphb200dmM0RWRZOW5oRHpncW9kNllJeFhGYXFzZFNpSHRiUWVUWU4wbmJZWFN6NnRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwM18xMTI3MzBfMDhfMjQxMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGNhZGFjY2Y5ZDUxNzFlMDFjZjI0Y2ZiMjRiMTMxMTMyZTQ2YWM5MGEwMzU4ZGIwYzJmOTU4NmI3NzdjYzZhNjJhNGM5N2I5OTU3MWJmMGFiM2Y4NTE4NGIwZmRjOGM3ZGE2Y2U5Y2QyMTg0YWQzNWNlNGM1NDUwZmVmNzA1ZjY4ZDVhYzNlZDZlZjVmNjRiNTFmNWJkNzcyZTI2NmMzYmU4NWIxNTE4MzA4NzRhMDRiMzg5MzBhZGFmNmY2ZWE5OTk3ZDZmMjkxYWY2MzI0ZTU0YWMyOTg2YTY0MDMxODM2MjUwMjc3NGZjNjljOTQ3YWVhZGJjNGQ2NDlkNDQxY2YwMzk5NzRjMDhhOTM4YTZjYTU0YmRjMmQ5NDhlZGFmOTM4MDAxZDJkYTlhMjMzZDQxYzFlYWQ2N2Y4N2NkYTdmNDQxNzk2NjNmZDk4NmYyZTBhZDE2NWIwNGFkNTU5ZDMwOWUzYmVhNDc2Y2U4NDU0MTRhN2VkZjBlZGFiMjUzMjhiOTU4YmQ5MzRiOGZjMGJlZDViMDEyNTc4MzliOTM1MWRhNjJhMTZhZWM4MjQwNzE2YTQ5NmVjYjMwMjMwOWQyYzhmOGIyMjdjOTg5NDI5NzkwZjFlOTQ0MjZiYTZlZmJlN2JjYWIwMWYzN2JlYzJiMjBiOGNkZjhmOTEwNDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.P10fHZ35faVQzRZVIXGB618ON-eqAznc5VTQbFbcDrvCLe-mZAm8g-iqGONUcNYN0bm8Hjwkx6YNrG3tSU86KA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210403_112730_08_2412_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.639Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InZjaWJOWk81aFVtVkxVblB4czh0TFh2ZUlLZmJiQTNGUUtteWFJNEFFRzlVcTFLTXkySU56RFNBR1FXOW5sRmlORmRsSWhrQXFHb3Mvb3A5NWlZQ1F3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDdfNjZfMjI3ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2NhNmNhYThmZGQzNDlmZWMzNjg4ZTBmYTg5ZWE4N2IxNzk4OGNiMDljMDUyMWE0MmY2NGQ5MTkwNzBmODNlNjExNWJmYzViODBjYzc5NjJhZWIzNjgwOTkxZmJkZDU2YmE2Njk1Mzk0YWI3YzM3NzQxNWE4ZGYyMTNhNWJjY2JhYjUxNGQ0MGM5Njc1MDZkMTgxNzRkZmZmNmZlYjllMDExZjExZWQ4YTkwMTU5ZmJlOTNiMzJjYzE2OTAyN2Y1NzU5ZjY0YjkxNGEwYmQ0NDEwYWU5M2E4NDFkYzJmYjgzZDRlNzA5NjhmYTdhZTYyNzczZDM0MDZlN2ZlNTIzNjljOWE3MWVhYzZlODYwY2NkZTAwNjUzMjlkMTJmNGQ0ZWMzODAwNjljY2VjY2JmODk1YTNhMmM0ZmM1ZjA2MGI2ZWU3MzA0NDgxZmZiOTlmZjkzOTYwMTBjZTg3Y2M4NTY3MjJkNzk1ZWFiNDM4OWYwYTQ2NDBkZDhlMWUzMDEzNzM3NjMxY2EwYTNjMzQyNTNmY2E5MjY1OTUyYmM3NTIxYmRlMDE3N2Y0MTlhMWU4N2QwNWI3MTljZmI1MmMzZGMzY2E0NDFhMGNjMGE5YjQyNjFiNzhhNjIyYmRhNzIwOWQ1YTU4NTJjNThlNjVhNTY3OWEwYzIxNDlhZDQ4MWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HNV1zCEYjCrdR_1ejTwA1dZUEcLWGsOyGz6fT44YBBHZl0qlAT4G-6w4YvqpNb2tOx9cUsd7JESUplLDWHDSKA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104347_66_227e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.642Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik9KTmtyWlhpcWdua3A2dUpja3MyMmQzSC9Pa3hxdTFCTUZUSDdiQzFSbE01RktoOU9TNW41dE56Ti9GdXRhTmxFUTBYTnpkMGh3RHJpb0Z0dWtoMmx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDdfNjZfMjI3ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjAzYjBmOGJlNzJjNzg3ZjFiYjA4MzQyYWQ3OGE0YjkzZTg1NThiNTc5OGY5OTcwMzk3MTUwNDY5N2VhZjAxODQxODJkMTFkYmQ5ZGQxN2EyNzFmZmUxMTk0ZGQ5MjMxOTI5OWQxZDk0ODRhYTZhYTFjMTQ4YTFmYzZlMGExZDIxY2Q3YTdiNjg2YTVhOWM5OTFlMmNiMDc0ZWQwNzM4YTMzMzg0MmQzOThiYjZjM2I0MTk2YWNhZTZiZTg1ZTJjYTJlZmY5NjY5MzUxZTQ0NjQxZjI2MTg2MWYyZDM3ODU1ZjBlZWJhZmEwMGY2Y2JlMzJmODczMTM2Mzg3ZTI2Zjk4YzQzNWJkMDBlYTNiMDA3MGNhYWM5M2Y3ZjNkZThhZGJiNWRkOWEwNjk1ZTE3MjFkNjMxNDE5ZjI2NDEzNWEwMTA0YWY3Y2UyYzQ0ODMxNWIwZWRmNTMyMTNhNTZjZmI5YzM2ODEzZWZkOTYyZDQzZjgwYjYwOTVmYjRlZThkMGRjNjA5OWY4ZWNiMTM5YjczYTk5NDM0OWZiMmVhZDllNTU2MmY2YmEwNjVlZTY2YTFjOGQzMjdkYzQ4MzkyNmNkZThkMzU2ZTMzZDY4NzQ3ZWE2MTY1MDVlOTg5YTgwZmIwM2EyYzhkOGRlODhjY2JhY2JjMjhjMWFiM2EyZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vJjnXEUR10lrM1BZi3o0ex_2j3vQcc1DlAgqa7cY2sjaiQyXYUGgzxke8V_RSzKHmFAXdCOXj0nOuN-i6xwhGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104347_66_227e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.645Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Inc1aVY0M0E4NzF2ZE1yd3V4RTBTY2pwUmdUWTN6L2ZTa1lkQnpacVdDVE1JM3pETHM3eTJSWFYvTmZrNkoxaitYbFpNRnhtRWdYaGpsZWJXTEdkSU9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDdfNjZfMjI3ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjJmZmUyYjY4ZDNiYmUxNDhmZjJhYmE2MjAzOTdlMjYyYWM3Y2YwZjBkNGU0ZWRlZjAyYmE3NzgyMTM5OWJhYjc5MjVjNDVhYzdkNDZiMWEwMDgzMmQ3OWRmYmI5ZmQ4ZmFlZjUzMzlhOWZlNzgzZGExYjE2OTI5ZjE1N2ZjZjA2MzYyMzZhMjJhMjkwMzI0NmQwMDQ4YThmODgzZjllY2YxYzc1MDAzMzg4NjE1MzZkZmNlNTgyZTAyMTNiYWE2YTE5YmY0YjEyMjVlNTgwMjM0MWU3NDg0MzAwZDllOGQwNjNjMTI5MmMyYTBmOWU3MzdiZmY2ZDBlOGQ1YTYwOTgyMWY4YWMyODQzYjg2ZTg4MzE3ZjA2NTk5YTMxZWI2MzYzYzcyODU3YzQwZTA1MzA2Zjc0MTE2M2I4YjUwYzlmYjY2NDFmODJmNzI0NTQzMDIzMGZlMzJmMDY1YjFhM2I5MDI0MDQxYTMzMTUwODE2ZDZjMTE2MGRhMjhkNTg0YmVjZjJmMTAyY2IwNTNhMGZkYmM0ODk2MjgzNTQ0ZDgxNjQzMjkyNDI5ODY1NGE3ODRhNmQ2NmE0Yzg4MDBhZThiZDA4YzhkMzBhZTM3M2MyZWUyNGMwNTdlOGQyNGU4NmU3YWNiMGM2ZWZjMGNkOTQ0YjUyOGM1MGQyNzQxMDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2jhiRchUoQMb2CdEue24brbFSCQ7WpWYLPK7isvALwdnIiB_ZouOtBxmMFaUy7PzrXLW4vHlxng0_E5LM9xlzQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104347_66_227e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.648Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik50TkZIZmNKZnR4VUs0Yy9SZkRGYXRhVjBGUDlEY2g5MUF0SmNmS09VKzZhbDJlYVN4NWtRL0Q2TDU0Zmk4OGdCTm1zNXRURUZuWlFKbzM5UUVEazN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDdfNjZfMjI3ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWUyNTI1ZTdlYWExNjgxYTUzZjZmMDdlZGMzYmEyODk0OTM5MTk4Y2E5OTRlNGYxNjgxNmRlMDI0Y2M3MWU3N2RiODBiNTczMGIxOGIyY2M3YjIzYjg1N2E1OTIzODZlNjQ1MWFkNTI4ZDE2MDEwOTIzNjYwMjMyZTQwODhjNTcyZTU1YWFiMjFkZTgxMDUzOGYyNmQ1YzE3N2IwMmY3MWFlZjQzNWJhY2Y5YzQ1YmVmNzQ3MGMwZjZhZDUwNTNjNGViOTBmZTVjNjc4ZGI3OTg5NjJkMDVhNWU2MDQ5NjJhMDY0ZGQ1Mjg3ZDBmYjIzN2Q4YzI1MjA3ZjVjMzdjNjgwODNkZDI2OWM5NGY3MWZkYmJlMjMxOGZjYWRkNmRhZWVhZThjZTFjN2FkNTFkMzM2YzY1Mjc5NGExOTNkZDE2MjkxMTkyN2RiYjU4ODdiNTViZjk2Yjc3Y2Q2NGRlOTNjYmE5NjMzOWY0YmVkMWVhYzAxNWJkZmM5ZDljM2MzNjZmY2EwMjVmYjBmY2E4Yzc1YmE5MDg2YmM3OTFhZGY5NjdhMGMyNWM4ZDZlMWIxNDFlNzY3ODQ2ZGZiZDMzMDU3ODllZDc4NjY1MTk4ZDc4ZmVlNDc5M2I4Mjg1NmRkNmU4M2JjN2IyYzc2MmI1ODVhNDZiN2JkOTUzMjdjZTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BYoY1B8Al3stD7Oj1fS3h-GEjboLFqAtwkx0UH0SuSQS7zQvQ_QyhC_vQ_E8vuTBtzsqFfs97NHElUpoy0xeqw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104347_66_227e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.655Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkNNL2FVSnZyc1RteWdta0VXT2QvdVVLM2MyZE5aSHFWZmhtVlZjc3J0NTFUZWk1OGRsaW1tZTl5bEt0QWpwMHV5R0I0ejRWc05XeGxma3BKQjl1T3hBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDFfNDVfMjQ1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTc2MGJiNDIxM2ZiYmQyMzc0YmFkY2NlN2QxMTY2MGZkZTAxMzMzNTY5NTdjZTdmY2IzOTc1MWFlMzk1ODY3N2FkZDQ3ZmNiMmZiYjg5ODlhYTA1YWMzNTYyNTZjOTZkNGVjODQzNDUwNDBlNjBkZTI5ODNjYjM1ZTAyMGZiNDFmNWY2ODEyMzVkMjU0ZTY1YzQ4YmM3ODViYTQxODFmNmFmNWNhMWY2NzYyYWQ0ZGIzODI5YzRiODE0Y2U3YTE0MTNmOTI2NzdlOGQ3N2U2ZGRjNDUxOTJlOGQzOWFmYTAwMzA2M2JiNTkxZTdhMjIxNmFlZWQyNWE1YTQ1OTcwZDE3MTc3MTIwMzAyYjE0NWY5YzUzNDExZDVjNmU3MzQ2MWRlNzY4YTg2YzFiMmU0MmNjODgzM2Q4MDQzMTBiNTNiYmRlNmY5NGFlOTEwMjEzNzFhY2NjYjdkNTcxOWZkMjdiNDY0MDA2NTU5OGNmZWM0MGM5NzViNTkzOTg0MjBhNTE0ODljOWEzNWZhNDRjMTBkNWU5YTY3OWJmZGE0YzkyMjE0OTU3MjkyNzBlOWYwODRhZTFkMTIyNjQyYTA1YTdhMThkMDBhYzk1NGYyYTAxNTIxN2MzN2YzYjdmZGIwNDAwZjk1NTNhZjFiOTIxYWNmNjFiZjcwMjk4OTQ5NTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NY9T2FaRGCO-NgPfTTVeirl0Osa8bEptkxTZtEQxgHPsKU8X9Wv2lqL_1dhv3IUpSbNs10ZMYRg-9eSZirhPTg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.661Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik9vSGdRejVZQjgvTE1JZXhrc2I4UWFpb0QrbU93MHl2THVwZktDT2EyRlJSUTNPQjBpSXM3dmo3K0dFRURyQVVNUjg4V1ovaXBJTXVsWENQQWwvcFFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDFfNDVfMjQ1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTU0MWNmOWY5NDJiNzQyODc4NzIzMzRiZmEwY2U0N2E2ZmRkZDZkZjE5YWQ1ZGMzYTllNDEzZWEzYTljNjEyZjQ3NDJhYTM0MDU2NDM4NDUyZWNlZGIxNTc0MjM1ZGViZDY4NTI1YWE0NmU0NGM5NzkzYzI3Mjg4MzlmMmQ2ZWNlOTc1ZDYzNmIwNGU4YzI5YWRjNzBhOTllNjM3N2M5MDRkNmZjNjQ0OGUzZWRhY2U5NTlhYmNlMTBkZTc5ZjI3MTI4MGQxZWUzMTA2ZDBiMjIzNDAyZjJlYTQ0NDc1NDZhM2FhMTE2ODBjOTNlMzJlZmVhNDE3ODEwNzMzZjY3Y2RjNWM2NDlmOTZhNjIzNGE0NTQ0NDQ5NmY2ZmVkY2U2YzFlNDNhYzA0ZDU0YWU1ZmRkMTYwYTlhMzE2ZmY5MjY3MDhmZWZkMDVhZmZjZDdiYTk0OGVjOGQwNWEyODE5NjUzYmM0OTM4MTY4NGUzNmY3ZTBmYTAxMzYxYmI5N2RhYjhjNTJmMjg2MmNiYTc0YjMyMTg2ZDgxNmM2ODk2MDFlNjYxZDgyMDViOGNhN2U4ZDYzYTJmYzM2OWJjZjNjNjE0MTUxNGE2ODA5ZWVmMGU3NmViY2ZhNjQ3MjZlZGVhZmQ1MWM1NDlmMDVhZGUzZDI5MjUyNzdkODE5OTI0M2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6QFvQT5t-zOhWnwpEVfWIWbC-ocmN24-bVFWS7ESgc4S3Q2_FXs7st2-j2IJn8KLOb0CTRhlyYWZsdgA1XfZTA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.667Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImFEK2htL3BNK3oxdXBkQkw2aU10WkV5Zkg0eE5aOHkvaGkreW5oeEhRbzJ2WE0wSms3NUlEVGtIaVJsVXlTUWpTeVp5OE14WHJmUUtBbXlEK0xYVkRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDFfNDVfMjQ1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWQ0OTA3MmNkNzY3OWQwMTgzZjIwZmM4Y2FjNGY0NGJlOGRlNmI2ZTBlNTQ5MmZkYWRmNzI1ZWI2MGY2N2NkYmEzYjgzZTIxMTkxOGNlYjNkOWRkOWI1MDg3ZGU5NjhmY2Q3YzA5Yzk0M2MwZTQ4MjZjZDZmZjFiMTdhNjgyMzU3OTdjMzc2ZjM4ZWRmNmYyZGY0YjljOGQ0NjkxN2FmYzBlMmFlNTVlMmJlYzhkZmIwOGY1MmZlMmFiNDBjOWM1YzkxODg3NWQwMGRjODhiOWNmZGQ3YWUwOWM0NGRkZmY5YjM1OGI1NTJmOGEwN2IwMWFmZWM1ZTMwMWZlNGRlNWE0ZDRjZjYwYzk3ZDM5NTI1NzE2NGFiNDRhMjQ0MWU5ZjBhNjQ0YjQ5ZWNiOWFlYjIzMjk5ZDBkNGU2YzdmNmFiNzM2YTQxNjdiZGY0Mjg0MTM2ZTQ0MzA0YjgyZDM2Y2E2MGQ5OTQ5YWRhYzc0MDgzNDUxODIxZjBlOWZjZmVhMjdhOWQ3NjQ4ZjRhZmQ5YTQyYjY4Mzc5NmNlYTNhOTFjN2Q0MGRjYmFkMjRmNGVhZWY2ZDY3ZTFiNDExNGY1MDhhNmU0NTJjMGUzNmY5Y2FlZjU5MDcwNGEyOGM5N2RlODYyYWZmZDY0OTRlOGRhOTEwZjYwNzE3YjlkNjg3MGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.A-GjH1SVujwHW35Bcjld5FRWmcMJOlC8r7Z1b8m79DjJlWg00ycD0UisC7kSReBbQ69xxPdyPMF8hbMAXHjecw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.670Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Iis1cWlMNlJQZkdqUGxqZTFVdWo4NW1lcEF3Yk5sMWZKYkE3QlVvUy95UjMzZkxicFpHME1zY3hOL1VBK09pZWFINkQ5alMrb2FIK3Zzdk0xdjhXSVlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDFfNDVfMjQ1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzk3MDQyNTdlZTM4NDI4MTY2OTUwNWI0OGViNzU5OTM4NTk4NDRiMzk5MTUwYjM2YmMzMzM2OTEwYWQ5OTlkNjc2NmM1M2Y5ZGQ5NmYxYzk1MWJiYzkyNDAxMjZkNDFlMDc2ZDMwNDc5ODk2YjM3N2IzZDYzMWM3YzkyNWVjYzI4YzQ0YTc4MjdmNjFlNWE3MTQxMjUxZDdhNTAzMGViZTNiN2I2N2EzYmQ0YzQ2MTk1NGUzYzhjMTA5Zjg2N2M4MGRlMzEwZjE2ZjVlMmI2ZDg2MDI5NjI2YjExOTMxOTg2Zjk5MmMxYzJkNTAxMDYwM2Q5ZjA0YTdmNDU4OWJkZTdiNjhjNjg3MGRlYTc4ZjdkMDBmYmRmYTFlYzI5NzkxNTJlMGVlZDViMDY1NjQ0MTU0ZmVhMTgzZGFmNDUyNGE0ZmRkYTM4NzNkMjE2Yzg4ZWE5Yzg2MDAwZWE0NGQ5ODU3NWZiODY1MDdjNjgwMjMyMjg0MjgyZGQ4MGZjYTIxODgxNGY0MTBjZWY5ZmEzNGQ2YzBjNGY4MDI2NjllYjVjMGNiMGIxOWRkODU2NWE1ZDI3YmFmMDhiZjJmZWZhYTRiZTM0OWVkM2IyMjFkNTNjYmRjMDNiM2VhZmViOGM5MDVhNzc2NWI2ZWYwYzJkYjhmZTI0YjcwZTIyOWVhNzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZIOs_eOydXG656b2ads8zUkbW29muUDPp-2DnasvtWZpZ44vM-3Sg27hXUOrS3I4LrA8P_erO5yUHPJEUOaIgA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.673Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik13MVlBMG1hTDFQLzZLRFM1UU11T1JOSGtWM3Zpc0J6QW4rM3NqcEYyYjJ5RmJNS1pKT3NLRGFLL09LOEMvMVh2TVpMWmY3ZTU2SkJqemZscjJqZThBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDM2NDBfMDdfMjQ0OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQ1MjQ1ODBiZWFjY2FhYzZhNWEyMTQxNzc3NDMwODExNjIzMGEzYjhmNDFkZDJjOTYzNmVhNTQ2Yjk0M2ZhNzBjMmNjODVlYWM1ZjYxODU1ODk0Nzc1ZTk1MzQzYTZhZjM1MmVlZDlkODI0YWM1YmFhYzEyNmI0Mjg5N2ZjOTQ4NzIzNzMxYWExMjQzYjVkNzJlM2I5OTliNmNiZWJmZWVjOGQ5ZmJlZWVjOTVlNTQyNmQ2ZWQ2NjA2OWFkOWQ3ZDBiMTBjOTVlMzRmN2JhZDdjYWQxZDdkZjU3OTNhY2YwNDExNGMyZjdiY2ExOGJjNzRjNTQ1MjZjOWIzYzU3MTA3YWMyM2E4ZTFlMGRjMjQ5YjM5M2Y4ZmZmZTE3NDg1NjkzOGVkMGRlMjcxMWUwN2Q2YTc2MzVkMWI0YmMxMTViNWYwZjkyYmZiMjNjYmYxMmJlNDdjMTUzMWM3MjM4YjQ4MWNkOTg1ZTUwZTBlODg5MGU4NjQwNWNmNmUwMDk5NjA4N2ExYmNmOTVlYmIwMTFmOTdiYzgyODQ2ZGVmN2QwZjBhY2U4MjNmOTRjYmUwYzI1OTgwOTUwNmQzYjRmN2IyZjVlNjI1MDU5NzUxOWRhMDY2NGNiZTI1NmE1NWJhZmU5YTIzMzM0MGJhODQ0OTE2ZGVhMjhhMDc5ZmMzYjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JtFH1HoYoPM-1tnNVnTS6-7-lwh5jNHMVX9O7ujv-x1SzMCLqz81k4xk5PUB0VBuMjHxEFsrALpeXyZZMsLBuw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103640_07_2449_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.676Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkQwektsNGx5NW9FMDdkVDhTWHo5ZnBUVHp4Y0ZYQ2h3Y3orRHZBMm1pM3d3d2libmowc01vbXJmOG5kNjhzc2crRmdGaFZlZEVteStwNHhmZUdLcW9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDM2NDBfMDdfMjQ0OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjRjZGMxMzdhNDA2NTg0YThiZGZlM2JlYmEzNzc4N2RmMmEzZmMwMDk1NGYwZmY0YTY1M2NiYTIwNGRkZWU5YTZjOWViMDBhOTUyYzU2MGY4MDc5YjY1NGQ3ZjE3YjgzM2NkY2ViN2I1NzA3YzdjZmM3ZGQzZGVjMzk5MjRhZmUxYTU3NzM2YmY2YjBkNWQ3OTg5MTRmY2RjOTZhMzJjZDAxNjI3YzVhZWM0YzA4MDg2OTEwN2FjNDA4NTMyZmFjMzY2NDljNzM5ZGYzMjU1YTBiOWIyOTg5NTQ2ODRlYjI4N2JkZDU4OTRkOWIyMzdhMDFiZDllMTk3MzIxNzJiMmUzOWRjYjk3MDdlNWRkZmJlMzdhNDQ4NjU0MDQ5MTkzMmM1OGRhNWI2YjIxNDJkOTk2YWExMWIzMjlhMGNhYWIwMTVlM2I4YmQwYzRmNTIxOGExMDI4NzI5OWI2ZTNlMWVhODMyNGIzZDFkNDcwOWVhMGRhODBhN2U2NDI3MmI2YjFhZDllOGU4NTk1MDIyMDI1ZjNkYzc2OTgxM2M3MzFmNjc2ZGQ3NjBmYWMwZjk0MTllMGY1ODcyOGI1ZjY3Njk4ZjhjYzE4ZTEzODM1MGIyODdiMDMzZmJkZWU1OWM4NDcyMGY1YWNiMmY5MWEzMGFlNzBiYjFiZWY4MDJkNmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CZBTQfBCxTfQDGB8JeakNSqBo_LQM_Wj0_PgJzNDCRTMsvrHHtXDbt69g5GbI-NC8fGBV4Q1TTH0TBxJ4WMCug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103640_07_2449_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.679Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im9rVVdhM2Voa0J2anF1emgvbmIrZEVNVUNwSG1CaS93MVhKSmRWWUhidjBWV1VZS3NKM2hmanNEVDJoeDVzcGVLRWRadmtWVTczYmh1SWxEWmwrdEJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDM2NDBfMDdfMjQ0OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzUyYWJjODIyZTQ4OGZjZGI1MTZjNTYzOTdiZWQ3YThjYjFiYjJhMjdiZTQ4YmFlZDA0ZDdhN2YwN2ZlNjA0Y2EzYjQ2Zjg3NmYyZWI4MTEzMzI3NzM3Njg3NTUyN2E5YzkxZjM5MDQyM2NhZWM5YzlkZTcxZmFiYWQ5YTk4Y2IzMGNlODFjNGFlZjk4ZTVmYWI3ZDViYWNjMTViYmY2OTk2NjFmMTJjMTE4ZjhkMjQwOTgzOTkxMzU0N2YxNjAxOWU1ZmJiMWVjOWRlYjUyZWY5OWNjMzI2NDBmZjVmMjU5ZTZhMWQ3YWFiNTYxYTYwNWU5ZjdmYjNkMDBmZjc5ZWE0ODU2MzJhMzg1M2JjODJiYWJkN2ExMzlhZDg3ZTI1MGY2NzEzZDZiMTMyZjUyODRmOGQyNjg5OGRhNDU4NzI4ODI5MDA0YTJkODcyNTgwODJkMjQ1MDJiMmNkNWY0NWNhOTkzZjkxNGNjZWNiZDFjMGRkMGY0ZTUzNzZhMmZiZjI0MGYzZjFjZTIyY2Y4YWY2ZmRlYWFkNDMxZGEzMGI3OTRjYzhiMDhhOTkyYmI3ZjRhOGRjOWZkZGU4Y2M0Zjk1Y2NmMjU2NDIxZWE0NzQ2MjZiYWIyNTk3MjcxOTRiNjQ2N2VlYzllMDcxNTA4ZGJhZjNhNzJhNzc1MDdkZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CR5TIg-r9UYpPd6vE5c5cj9ESfMD33ex6Yn7hzXNQWqKDujloDCgM6MLheuo9qWPhnIfSJQ8xUWAIUqHV4KGAw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103640_07_2449_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.682Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlZVREpDRXFvaWt2ajV0Ry8rU3NkdjRHc1FXOEMwL3hiRzZRV3FKRGFjeTZwM3dEOEV3TlpDbjI0bnRuMmhCOUgzZ3hZeTBrVEJ0NVdyckkwRmZmQ21nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDM2NDBfMDdfMjQ0OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzExOTE0YTlmZGExN2Y0MWI0M2UyYzQ4MzY5NTE3ZTdhMjM5YmVhYzQ1NjRjMzE3MDM4ZDhlYTJmNjc2ODQ1ODQ4NzA2MGVkYTFlZWYyNDdlN2JlMWNlM2NmYTAwNmMxMWI3MDAwZTI1NDdmN2EzZTE2ZGNhYWViYjk3YWFlNTc3MjQ5NTk1YTgzY2JlOWM0NWIxMmE3ODQ4YTA2MmMyZDg5OGExMzU5Mzg0YmE2NTlkYjJiOTVjOGVkMjFiZjRkYzZhM2MwYzc0NTFiZjM4ZGM1YzE4YzZmOGY2OGQ2NWJkMmNhNDAyMWI3ZGJlMGE4N2VkZTc2MTA1Yzk5NzQ4YmRkZGEyZjE4ZTdjZjdkZDVhM2E5NTQxZWVkODk4Mjk5ZWE5ZGUyMGIxMTFhYjkyZjVlZDE2MzBjNDkyYTFkNGJiOTBiYWE2MTVjZDkwOTZkYjVjYTBkM2I2MTFjNGMxZjg1NjI5MTMxYTM4N2MwYTZmYjA1ZTkzMWI3NDBlZTNjMTYxNmNlY2Q4ZDMzNmViYmYyMDYwZmQ3ZWRkZmU0MjUzMmRkZWRlYmFjNjU2MWYwNjU2YzdhMjgwZmM3MGRlNTYzZWZlNDA2ZmY2M2JmMjdjZmU4MWVhYTQ1MjFlZmE5YjU3ZTA0NDA5ZDIzYTAwZjk1ZGY0YjViNjEyZTBjZWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vTktC7WwWNIQcQCNe_MXs9JsS15NQcU1HcnAavAairNZxyhdFcnMf0IFzCbeXDUwYaP5VswQuNj774PwWiC1uA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103640_07_2449_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.685Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjlvUVIrT2RBNlB4QzlXTHhvZmJhcWFaSXZJNjI5ZmtsQml4Nk52U3pQZnlQbWNua00yYml3ZVU3Sm94VFFPQ2svSGZhUkw2UE9VT25Bb0p3ek9LMkxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTA2MThfMDhfMjI3YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmNjODdlNzkyYzFlMDQzNDVhYWYxYjVhOWFlYTUxYWJlMzNhYmFjMDkzNjllZjcxMzE3MTRjNDQ4MjQ3NDhkNzhmYjhhZmJlNTQxYzAyYjI3OWY1OGQ4OTNiMWVhOGUxYzgxYzkzYTJmNjJkNjJiZTNkZTg2NDRhODE2YjRlNDBhMzliMDRlYzNjNWMzNzRjYmY3NDE0YTFiODg1ZjQ5YWJmZWEzODIwZGFkYzU3M2RjNDc0ZDQ4MWJkMDMyYTQ5ODE4MDNiMmRmZTc5NGU4Yjg4ZTliMjlmYjU3ZDE0NWE4ZjRjODRiNmQwODEzZGZhZTcyZDk5ZDk5NzVkOWI2MTA3N2U5MjY2NWQxZTYyMzRkYzM5ZTJkOTlhZTZhMDgyNGU2OWNiOTI4MDRhOGE5MzZhNjY0Mjc0MGM5MzliZTA0ODIwNDYzMWZlOGFiZGI4OWU4ZmNkMjQ3NmNjZTUyNzEwYmU3YTRiMjZjNzdkOGRlMjdkYTNmMzBjYjkwNzk3MjVjZjQwNDU2YzNmN2EyZmFiYzRhZDQzNzFmMjYxYTJiOWIyMmVlYTU5MGExMDc1NzQ5NjIwNDM0YmEzY2M4NjY3YzhjNGU1ZjliZTA0MzExZjViYmU3ZDc1NzU1NjI1MzYyNzRhMDU3YWM0YzZjZmQ4ZjIyZjM0YjhmMTRiN2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._8XNJer0AG7mgGJW1jEAehZlfWXzQUvnt-_BzMgEhqcVuPmoQ0JIcry9adlRS3zVN8LQ8xyy-kAF6x-ak2L-KA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110618_08_227a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.687Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IllSMUtVWnJsK0w1RXJ3MkttNXNvcTFFRk56YkxMdHZja2dPVmRDUE94a211bjc5ZWZQZm9za2VQZ2dnWnRpR29meEhnVjNMY01yeFZDTHJYOWNkQWhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTA2MThfMDhfMjI3YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDFhZjRmMjlhZjBhNGJhOTZhZTIwNmY4YWVjMWZiZjExNWVhODM4ZWY2NmJhNmM3MmExYTBhNzlkODY3NTc0NWNiNWM2Y2ZkMmQxZjI4YzJlM2M0Zjc2Nzg4ZDFlNThlYmIyMjI4YjBjZmQyNTAzZTAzNGZmZjYwNTczZjNkM2FmZDFiY2EwODNmMzFhMWE5MTQzNjY3MmJlZDFmZjkyYjY1Y2JkZjk4NmViMGRjMDZmMDM4NjIwMmJiODJkY2IzZWUwNzc2NTFhZDJhOWI3YTRmNWE0OTYxY2UwNTQ0OGYwY2I2MjYwNTIzMTZkNTg4NzY5ZGVhNGIzMWE1ZDk0ODk1YjdkYzRhMWVhMjFlNmU1NjQ5MDliYmJhNzdiYjM2NGYxNGZkYmQzZWUxOWZlZDkzNmI3MTU0NDViNjk0Y2Y4MWQzNDU0M2M5ODkxY2JhMzQzMGQ2MzE0YWQ0OTJlYzEwODBkODFlOTAzNDg2MTQxNzk4MDk0OGYzZWI0NGYzZWU5MjI5YzFmOTdhYzRmZDkwODY3YTEwZjA3ZDA1OTE0NzM3Mzc2Y2I4NmM5Y2U0NGYzN2MxYjBjNjIzNjQ0MDExMTVjNmI1NmRjOTg3MDFmYWRjNWQ4NmJlYmZhYWM1MTQ0NTkwM2VkYjIxNWFmMjE5YTdmODYxZWQ1ZTJkZDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GQfwj0jcsPe3AEdA36YbDjA-sjl5SMXG6kn2BbfTBOEqgjJ2EHacWdmgwb7hyJZVL3x2KPzYWs8ws3uG36RTxg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110618_08_227a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.692Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjJLaXJvT2VBWXQxTzhKcDBoNXcxdEx4ckJSd0s5QytBQmZBenpsZjZGd0tWUFNyL0ttVkJnZjZKVEo4MVNFeG5rT0s3YlJzWGhOcDhSa2o3Vyt4QURRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTA2MThfMDhfMjI3YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2FmODMzMTRmNjBhMjczYmJiMjlmNjI3OWZkMGQ5ZDAzM2Y2NzcxYWQ0ZTYyOTMzMWJkZWY5ZWUyNGQ2NDRiMGU0YmIyOGM0ZTBhMTIyM2MwNjdhZGI0MDIyNzM1YjY5NDg0ZjhlNWFhYjBiNmI2ODU1OWUwYjYyMzQ3ODg2MjhmODM0YTJiMzU4MDE1OGMwMDM2MGNjNDlkMGJmNzRjZWJkMTc5OWYyZjViNDk3YjVlZTE2ODg1NTUyZTIxNDQyZmIzY2NjM2E1ZDlmMTUwZTUwMGRiMjRkYTExNjA4N2RhYzBlOWMwM2MyYjAxNWI0YjM0MzYxOTZiOGFhMzIzMjRlNGMwODNhZTMyNTMwNzJkYjQ4Y2ExOWM5NzFmMTUwMTA5MTQ1ODJhN2EyNGY2NWRiNjFlZjQ3ODI1MTc5YTgzZWFlNTI0M2VhMTMwYzQyMDJhOWQ4ZWUzOWEyN2U4YTMwZDY3MWYzMzI2MTA5ZDRiMTRhMTFiYmQyNDdmMjE5OTE3NDU4OWE0YWUxMTJjYjdkZWE2ZWRkZTQ2ZTYxZWY2MjIzYzczOGNiZTljYzk1MDZkOTY5ZmZhMTkxMzRkNmVmMzU2Y2E1OGRiYjVhY2YzZTdiZTVkOTk4MWUxMzNiNmI1ODViNGVhNTY4ZTI1ODE3MTM0ZjgxNTIyOWU2Y2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pr2-_Tdr1JwYMEWluoXzpBfx5AOf35Ge69T3v9NfepP-bansYvpFn6jpvlmDihMVx1Jz_UmHXdfssKhoaPuUVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110618_08_227a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.695Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlVOYUhVNWIwUzhabnJZUy9oUFBvQjJZeG1lUDErRE9hdnVDQUFrZDhVWGFPb1piYUh2UmxsZ1krVFZkOFpQTjMzSU92elN5YisrVUFOMnhweEx3aGJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTA2MThfMDhfMjI3YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzNjMDZhYzhjYTQ5MzA4YmRmZGEzYjE5MTZmMGNhOGY5YTU4ZGNlYzA2ZDliNWM1NDgwZDFmMDEyNGM3NzU1MjdiOWIyZjhmNDYxODEyZmUyOWIxOGY2NDhhMWI4MDlmZmIxZjU3ODU0Y2U3ZmU2ZDIyYTY3YjhlODFmNjJkMjI4MzQyM2JmMDAwM2RkMjc0MDUzYjhkYmMwN2JmZDlmMTUzMGFjMGJkYTk5NmE0ZjEwZTNhZTZhM2ExMWU4ODRlYzFlY2JhNGIzNDAxNWZjOTAzMzBlYWNjMDllMjRiYzVmNTk1Yzk4ZWExOTkwMDc5MGViYmRiYTQyMjA5YWI5ZWY5ODllOTE1ZTJmNmNhOTg3YWRlOTg5ODkzODk2ZmMwZWRjNGNkZjllMGRlOTM2NWJjZTBiN2VlMmZmN2QxMTVkZGViYmViM2M2ODc4ZGNkOTZkMTRhM2E0ZjNiOGYyMmMyMjMxZWE0MmI2ZmYwZGIzN2QxNjBjNTNkNGZkYzZhYjIzMDlkNDBlNDRkM2I0ZDVlNDM5ODc4YWU3Yzg2YTVmZDMwMGYxMzVhNTBlMTEwMjg2ZmIxNDI2ZDc0ZGFmOTc3ZDVkOTk4MmM0OGNkZDM0NzFjOGMxMjdmN2FiNjZlN2M3Y2Q3NjdlMDA0ZGE0OGZmZWMyMWUyMTJmNDc4NjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0aKBtqv6dDx5jbPdMnhfjnaIb0qqWDiM431BzD3OjqMluKb75cTBGLE9NW_B8ElWOkIM7ZvNP3SyuUgutusmiw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110618_08_227a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.698Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IndGWTBsSEtqc0xWRFhld3J3RnltLzRSNGFiQmpRZXJqNkRlaGdsdU4zK1FSbkVtYmZtbWdhS2NKUUdKN2htSW1WTFpPYUpNSUZLOEFUU2gzeUhlUzZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM2MDJfMjdfMjQ2MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGZiMjNhMzQ3MmE2NDM2MWQ2Y2YwMTlkNjNkMDg0MTcxNjUxYjZiMGZkNTZhOWY1OTVmZmZkMGM0NTc1MDc1YzNiOGJmNWU4OTgwNzU2OGMzZWY5NzFiOTEwMTk3ZmRjZDlhM2JmZWFmZGY0NjBhMjYyMzc2ZTYxYjkwZTI0MDY0ZWY0MjhlNmY0ZTFhNmRiY2RjMWU5MGI0ZTlkZDFiMGVkMjE5NDJjN2VmM2M3YTQ0M2E0MjRkMzRmZGRmZWI4MmQwYjdkZWE4YjE4MWFkZjc0MjQ5NmViN2M4Nzc4ZGQyNjNmZGJiYzEyZmEyOGZhMDczZmQ1MGFiNDBjMTIxN2EwNzJmZTliNThkNjQ5NjY3ZGVmMzBjN2IxYjllZmVlN2M4MGU3MTY3OTExNDlmZWZhNTBiYzBmNTY2Zjk5MzZkOWE2NDBhZjMyZDdjZThhZmVjYzhiMWQ0NjUwZWZhNDI3YjIyZDg3ZGQ4Zjk5YjYwY2Q5YjJmZmM0YzE4NDU5Y2Y1Y2RmZWI0NGMzMTdiODAyMTllMjgwYmI3YWIxNWJhZTMwNTYzYWVmZDBjMjcxMGVkMmM4OTI1YmEzN2FjZGEyM2Y1ZDdlZDczNWM4NzM4OWExNDIzNmZiYzI1NDhmNTE0NTMzZjY4NDExZDNlYmQzNjY5ZmIwNDlhYTU2NDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zVyMvdSy31Byn9VBLSgVA9YcQiP_jrx0zHhwIUAew0DX5SvzNJ9pfvXyEtCKQYzecsotUu-35yV34PkWtGdhLw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103602_27_2460_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.705Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InAzcHlIanpxY1B3dVIrcEJhNlRYYXVKY21Qb2tSNE1MUmVSejc2dmFLR1RqblphZW5VN1JMQTVIMlg0MTJtVWRkZVhndEhWb2c2NTB2ZisyakZWaENBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM2MDJfMjdfMjQ2MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDQ5YzRlY2UyNTUyNzgxOThmNzM3OTljNGJmZGZmMjVjMjFiYTVlMzM3MzViZTVkMDgwMjc0N2U5N2E0ZTMyYzQxYTVkNWNlYWYzYjZiOTFmM2QzOTlkYmZmNTdjMTc1NjljZjM2YWIyNGQ5ZGEzMWJhNGUxODU0MTAwNDk5MTI3NTY0M2RmY2JhOWMxYWMxMDM5NmMxOWZiZGY2ZDU4OTdhMGYyOWM1ZWU0ZjJjMWRmMWEzYWQ4OTRhM2FkZTYzZWRkY2ZjM2ZkNWRkMTUzNWFiNjAzMjM3MzY3MDZjNmIzNTNhMTJhYmQwZjNlN2M1NTE1Yjc4MThkNmZhOTlkMTM5MWMwMDQ2NDQ3NzdhNjQzMWI5NDRkZTVmMGY5ODZiYTJiNWVhNTViZGZlZDJjNzE1NTVmNTAyODAxM2YxZTM5MjRiMzhjYTAxZWY4YmU4NWFjODA0OWZiMjJiYjRiZDFhOWJhZjhiMmIxZTk1OGIzY2IyYTQ5Nzg5ZDU3NzJhZTA3NDAwNjFhOTIyM2VmYWEzNjM4ZjQ2NjBlZjA4ZWM1NmY0MzZlMThhYzVkNzM1NWEzMWYwNGZmOGE1Nzc0MmRiYmVkYjc1NzcwNGU4NzlhNDA0MmFjMjEyODBjMDY4YTBhOTgyMGNkYjg4NWQ4NWU1MjdiMWE0YmE0ZThmZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Jw6jNhX2TMqSCQUmjM4jKrzRYg229cnyj3TU7tWe2br_nX65Ol3rVfa07Yq0oRv3LeQ0eHUqd1A7kUbeSXz1qw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103602_27_2460_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.708Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlB1Y1ppTlZoSFpXYmdkRVJSK1VreXhobzluQVFTWlhvYmgyeTM1cDV6aEtKRDAweUY4TDU3Yk1lTmtlYkhsWjZremZubTFreC9jRzVUcTdSTW1YR25nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM2MDJfMjdfMjQ2MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTM2ZTQxZGUwYmVhZDhlNTExNTU3NTM2ZGU2MTI4YzJkYmYzZmU2NDYzMWRhMzhjN2FjMmE2NmE4Njg5MjNhZTQxYmJhZmU3NmMxYmMyNjNmODg0NzNlMWYwODAwM2I3MjQyMjE5NTQ4OTU1ZWFiYWU2NzJmNzI5MGNhZjI4MDM2NjU2OWJlZDBlMmE2MWY2Zjk3NGRiZmJlNzAxMjFiMjdmZWRjZjljMDlmOGJmMzlmY2ZiMDFmYmIxNmY4MzIyZDc5Zjg1OGQwMDNmYzk2ZjA0OGM5MjBhMWMzZTljNThiZGU3MzA3NzM5MDBhYzU4ODhhY2EzYjQ5OTI1NjVmOThjMGExMmE4MzNkMmFhODhkN2M1M2MyODRmYzYwMzc5MDM0YTRkMDc0Y2E1NjNhY2UzN2MwNWE4ZjBjNzIyNGU4YzU1NTg1ZmQwMzAzZWZiNzdkMWQyMDRjYjk2YWM0Yjg1OTRjMDBmMDdiNmI2MDZkMTQyNWE5MDFmYjE1NDdmOGI2ZWEwMzkzNTIwY2IzZjU2MzEzODBmZWUyNDI1ZjM4NzQ5MjMzOWVkZGRlYzkyOTk3ZmFlNTEyNjdlOWNiYThmZDliMzlhNjNjNzdhOTQxYTJkODhmYWEwMWRkNTc2YmVkMzZmMDcxZjRlYzZhNTFkYTkzYTkxNzk0MzFiNzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GO_U5xaOT0A1uf5pgqyRBTGUIojR-2FKAgB9b1vW7gTmPwM5v04CEhZp4-jl20VMWL1OXk6sVmqhAbN-lr3gEA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103602_27_2460_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.711Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjBtM3FYNEZxRjl5Ty81Qm42VjZHb2U3OWVTdW5hamh3SXVVRy9GZlkrdGh3Sm5aUWxodm1RdHNDWUlMYlMxZkNIWDN0MkVUV3RKTldYbVlFcXBKR0xnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM2MDJfMjdfMjQ2MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQyNDgyOWIzZjc5ODdjOTAzYzFhNjNkZjAwOGRkZDA4ZGM1NWMxNjYyNGU1MGZlYjY1NmM0NWViODRjNDQ3OTExNjU1MzU4YzQwNzI5MzkyY2IxMzkzZjMzZGQ4ZGRmYTUyZDFjNjRhMzcxMTBlOTc1YTMyNjE1N2NkMjhkMmJkZjI5NmVmOTQxMGYzMTk3YjdiMjQzZTY1N2EwM2YyZDY1YjMyNmRiMGI1YjRjMjZlM2Q2MjQzMzIxNGZmOTc3MWFlNDdmMzg5M2NiMjAxZWI0Mjc5MzExYTRmNzQ5Yzc1NDliZTljYmUxNWRjNjZjMDU4OTRiYTE1ZDBmNTMwOTFlZmNjNjY2NTJlMjY2MzMwMTRkNTY4NGEwODA2MTAxZDgzMjMxZTlhZmY1ZTRiN2U5ZTQ0OTczNGI2MDQ1NzhhNTQwN2E1MmQ5NDY4YWZhODE3NmEwMDBjNjUxZjc5NDk1YzM4MDdmODZmNGUzOWZjMjhjMDZjNzI1NzYyNTJiMjY5ODNiOTJhNWFhN2Y1NjE1NGE2OTNmMTlkNDUxMmE4ZDRkOTU1ZGFkYmMyNmViYTdiNDAzZmE3ZThmYmY2NTdhOTYzYjk5MzNkYjBkY2ZmNjg3NmMzMmI2ODVkOGM3YmU4NzVjMzk0MmZlM2Q5YzM0NWZiNWM4ZmVkNjY3OTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OZio6NfmQQrmyKL8c2aRVw9xBkSO0Dtb4DdY949fHUKRHZddnwaIk3bUK_LVtAmwdBDUqkFuzxj1sNOWYx_dXA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103602_27_2460_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.716Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlUvL3crSmQxQWd4WDBxZGhDaFFCQnVsd3kvemVNYWs1NFpZMTNpajBhWTlSMlZuaGtUTXFNZm5jSGh0d2xOQkJXMnBGRmZFQ25zbGYzaXVJMFhMUlNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIxMV8xMTMxMTNfODBfMjQwM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODgzYzljYWJlZjMxN2Q3NTE0YWJjZTg4OTUzZjI4ZGZkZDhlNzQwMmUzY2FlODQyNmY1ZTRjODcxZWQ4MGQzOGVlYmRjNTk0OTQxYzc2NDY3ZWMwNzU0YWU2MjIyOWI0ZjJkZDY0YzdhOTA4OWRhMjcxNWRhZmRkY2YwMDRiNzg3ZjE1MThmNGMyYzg0OTJjOGM0ODkyMWVmZmNlZmUwZmU2MzQyNWU4NDE1NDkxMmQyODViMzY1YzcyOGFlMDRmYzdkNmMxZGMzZTVhNzUxZjdmNGU0MDA1YWI0NTQ5YmQxZDBmY2ZlMmFlYjQzMTE3NDYwNjU3MzI4M2RkZGVkNGE5NjY5Y2E5M2Y1YzExZmQ0ZmI4ZmU0MmEzMjA4YmUxMDA4NmQxNTdmMjFlMDhmMmE5ZjFlMGJjMGFiOTEwMjg0ZWNlMzNlZTJlNmQxODA2ZGViYmMyYmEyZGU1NDM5Y2UzOGM3ZTQ0OGYxYWQ5NDlkNzI0ZjMzYzI0NWI3ZWUyMDEzNmFkNTE3YTk5ZDQ3YzQ4MjA0MGJhYzc2MGUyMDIwZDBlY2FjN2U1OGJlZDE1NGI1OGM1N2I3OTBkNzFkMjA2MzgyYjQxOWYyZmI4MGI0MzVkOTJhN2VmMDRkMmYzMDgxNzVlMzJmYjE0NTRiZDJlNmJiMjQ2ZmY2YjljNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hccHyN9ilsucjEQRisf503NsE9ZSoPtxIfFsFSGfBbo5_Dl3qqXS_7oIEUoRrFqQlel6FOFrctOobYMti0wdqg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210211_113113_80_2403_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.721Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkdGeUt0QWorekpUWDFzSCtVOWVSUzYrbjhUemxrbHBJQ3ZhbkFHQ1lvdHBWTnVVNVpUOE9KcWhJd0V0MDBvQ3pWajc3RWFLeWtORzc2SDdtZWNhVmlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIxMV8xMTMxMTNfODBfMjQwM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY5ODU0MGFkOGNiMzkyMTQzNDI0NTQyOGNiMWMzNzA4ZDI3MzMxOTkwZjM3OWZkNjFhNjAyY2JmZjVlYjI3ZjRlODcwYTI3OWI4MDIzMjM3MjAxZWMyYmQxMGU1OTA4NWZlNjEyYWU4NDJlMjFhNzU2MjlkNWI5ZTU3YmI5M2Q5YjdmZDNjYzQ4YzQzYTkxZGNmNTRiZmVlZjhmYjRmMDczNjBiZTVlZGI0ZTk5YzYyMDAzY2ZhMGU4ZDUzODgxYjc4MjhiYTRmOWJmNGRkMzU1NWNhMzA3YWU3MGU3NzU0ZWE0NTVmY2YzYTg1ZWQzYmViOThiZTg4ZWViOWU0MzVhZTA5ZmQ4MGE2YmI5MDQyNDQxMTQzYmVhOWExYTczMDVkZjQxOThiOGY4MzljOTMxMWRmOGQ0YTBjNzY0YTBhNmI4YTczNjg5MDMzMWVkMTQ1ODhlODYxNjE3YWQzNGQ1NzFhMmMwZDhkMjQ3ZGY5ZTk2YTQxYTNhN2E5NTMwOTIzZmQxZjFlYzlkMmMxMWI5OTNkM2MyNTY0NmFkY2FmYjBlODc4MzJiYjVlMGZhYjg5YTk4MGMyYWNlZGU0M2I4ZGZkZjdkMTViNGEwYjBhODFjMzcxMzM1ZDVjOGY2NWQ2N2ZlOTVkODgyMzc0NTI3YjFiNzYxNWZlOTlkZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.K6ySUVzOZIDH_8EUfAtIFgs_is1omyH7NKLDH7tPTNBphA7CnIS4ykk95aHjKW6wjWTiUjJhQf0JIHP5ENVkbg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210211_113113_80_2403_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.724Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlFYNkdaSE8vS0NtcXZYNXhteDhzUWNtclJFV21HcWJvU1V5YXV1WC83ZXJReFhrRmtZMWtNRVlYeEtvYXl4M095U0NmblF4ODBmc21mN25xWUIwbG13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIxMV8xMTMxMTNfODBfMjQwM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2EzYzRiM2M2M2ZmMWMwNTQzOWVlZjU0MmM4OTk5YzgzMmNlZmU4MzI2MDdkZjRmNTVmODJmM2RiZTY4MDNjMjliY2E0NzYxNjJiNmNiYTAxNmVlM2E1MTc1MTgyNmMyMmYzZTg0ODVmYmM0MDI4MDgxZWFiZjliYTQxNDhhNTIzNTI3YmUwNmE0ZjViNTNiOGQ0YmEwYTkyYzAwYjBkODU2MDAyZGU4YTJkYzkwMjFhOWNkZjNmODFjODcxZDg2Y2YzMGRkN2ZiZjY3NWYyNjkzYWZiNGY4OTM3NjA1ZTkxZmUzZWIzZmQyODA4ZGVkZmFhNTU3MTZjMjE2ZDM2ZWUyMTVhZTQ2M2IzNWMyZGY3ZjQ4YTFmZjI0NjE5MDg2ZjVkOTEyYTIwZWU5NWY0OWQ4N2U2NzgyNDg2N2JhOWNiNDQ1OGZiMzExODkxMzQwYzZiNGQwZGU4NTMyYzUwZjhkMWE5NjA2MWEwMWRiMWE5ZTg0OWFjZjMyNGU1OWVjMzI3YzYxNjQxOGNmYzFhNjY5MGNhY2QxOTU2YTMyMGRiMmNmNjllMGRiOGI0YjdiMzI4MTcxYTQ3ZGViYzcyNzI4OTdmM2E4N2EzMWMxNWFmY2IzMjI0NmJkNWYwMTE5OGU5OTA3NTg4ODU1YzQ4YjlkYTM2YWQ0YWZlZDFiMGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hXQ6M4rMChxrEQRQNmd_wObzEPOhrFM7msOwMQ-38wv3_8LRsFvyHI2ebQq80DihRKiDDUXfsgiEjHaiSX9QSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210211_113113_80_2403_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.728Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkhhSDBrNHdjYkdBTWZjYWNITXBRdWlIeVpLNWh3eVp4NjZGcTNLekZ6QzM1UTg0aFlrYUVETUQyZ1p1QWFoSDhseFhibnVaZEhJSE44b1VJLzVpdGhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIxMV8xMTMxMTNfODBfMjQwM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGM1OTlmNzE4YzI3OGU2YzNkYTUxNGNmZjlmOTg4N2UzMjhjYTgzMGM3ODkzZjBmMjAyMWNiMGJiZGIwOTY5ZTFlNjM4NjZlMzRlNDMzNTM4ODBhNTFhNzY0YTdhYTc1ZWU5NmE1M2MzNzNmZDViZTViZWY4YmVmYmU4ZTk1OGRkN2UzNWI2NzVkOGNjNmM5ZWJlYmY4NjE4OTE2MTQyMDFhODE2ZjI1NWJiODIyMDhmMDQyMmM5Yzg0MjliN2ExYTIwM2JiOTY2NzRhMGI0MGQ0ZGI1NGI1MWU4NDZiODJmNDU3MDZmNDMwMzcwYWRkNDU2YTQwN2EzNDhkMjE3MmI4MjQxYjI0MjUyMjNhYzU0ZTBkMDJiNzYyMmRhZDk1ZjIwZmI5MzVmZmRlZjRhN2EwNDdiYzc1MWNkNzc2ODU3ODE5ODU3NGQ2ZDAyMTE2YzkyMDhjYTU2YWVjMzRiNDMyOTAwYzkxMjJjMDRkZGYwNmVmZmE4ZGM1OGJlYzE5NWNiY2QxNDA4YTdhNmM3ZmJjYmM3NGQ2MGRkNWJhZGI2ZGFkNzVlOTZmZjJmMWU3ZTAzOTM0YWMwYzcyODRjZjg0NzMzYTQ4M2YzMTc0ZDI5N2YxYWU4YTYzODcwZDU5OTExMWFjNmFhZTU2YWJlODQxNzY2Mzg4MGEyNThmNGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iA4kJs8gwiNjLdq0tt6XElm8o5or60qZI7ARrT1emDX_AOJGYfDGjca3vGaC_G1OKlHsW94SU1ZcMSGSTamoOg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210211_113113_80_2403_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.731Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InJZd2xTUkhoNUw2cm55K1BpLzRNUkdxRmZGUUhzc0lWQVNQa1YvTnJuNUtCcDIyNDcwSXBBRUNoTjVzL05GNG5URWhFcnZ0RUF1TWRYekFYTlRwUE9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTZfNjVfMjQ2M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDQwMzFhZmIzZWNkMjNjNGVkNWRhYWRlOGExNGQ2MDA5MzUyY2FmOWM1OWQ0ZTJhNGNjYjBkYjE0YjFkODY1NzhkOWM0YzE0MzI5YjNmODcxMjgyYTY3NTkzZGQ1MDM3NzYxNDI3ZTJkYjA4NzBjNmQ1OTAyNzAwNGZmM2MzYWNmODVlODFkNDNiYzlkMDVmYWRiNjk2MDU0Y2UwNTk4ZDNiMjQ1ZDhmMmNjZTk2NGVmNjlmY2FkODBkYTEzNmIzNTZmYzZjNTg3ZjE0YTYyNTQyMzRlNGI2YzNmZGNlODkyMTk1ZWNjMGQwNGQzYzRhYmRjMTRhZjc4MzBlMmNkYTgzMWNmZjcxY2U4NzA5Njg0MWYxMjVmOGEyZDNmZmEwOTc2ZTE2ZjA1YWY4Zjc5MWUwYTY3NjVhZDc3YTkwMWJkMWIwZDY3NzM0OTgyOTNhMzU1ZTk1YzMzODNiNDA3NzMyMmYyNTk4YjIzMTdhYzQzZWJjODRiM2ZlMTJjMTU0MGNmMDI0NDgyYmUyNWE0OGEyMjFlNmE3NmE3OWIxMmQzOWFmY2I4YTI4OGMwNjBiYjk1ODA5OGY1YTdlNzA1ZGU3YjIyNzI4ZDU0OWNkYmUyOGVmOTQ5ZmU2ODAxN2JmNDJlYTUyY2JmMmE0M2RiYjFmOTI5OTE5MmNkODc3YWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bLMORGA08boPQh4da0umm9wv_Rc9NH1tQBpo29grWYYyXM4HAIzGb72mHABYH-UEiruiXedOWg_GAm9HIA1EOw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103856_65_2463_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.734Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IklBeUw2b3I1RlpVdk5nT0ZXSldqdmZ1NklKSnBRdXZON0lLNS9WTFhTSkJSTllyYkYyNHlIN0VzZEpaMDFXS0R3UTVuSjdpR1pCdkt4VGhnQnpvMTRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTZfNjVfMjQ2M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTZkYWU3OTI2OTZmMGJlNDI5NmQ0M2YxMjA5MGNiZThmNWRlZWQ1MjE1NTVjY2NlMjkxYzE1ZDczZmY5YzFmNjc3N2M0NjRjN2E0Njc4Mzc2MTI5OTY3NmMxNzdkYjQwZjlmYjJiOTg1YThhNGVhZmZiY2U3YzUxYWJmNWNjN2NmYjU1NzllYjdkMTBkMDI1MmYyM2UxYTYyMjhkOGJhZWUyMWJiODdjODQ0YmQ2MTM0MjQxYzYzN2UyZWYyMzBkNDcwNjhiNmY2ZDIyMWU2ZTgzNGM0NTc4NTc2ZDdiZjU4OTU2YjI0NWZiNjkzM2Q1OTVlYmUxMjE5YWU4ZTM0OGFlMzk0OGE1Mjk3MjNhZTIzZDg1OGJhZjU4OTdjYWViNTY2NDhjMjY5NDUxNTg0M2M2MDE4NzJmOTQwNGZlZDY0YTQwMjU0MjAzNzJmZTNmMzg1ZWYyMTNkNjFhNjkwYWZiMWJlMDYxODJjNzUyYjBjMzY3MTA4NDBmMjMwMWZlZTMyMTA4MDlhNDg1NTBiZWNjYmQzZWIzY2E4OTU1N2U1ODVmMDQwMTIxYzRmNzY4NmRiOWNlZGE5NTQ0ZWZiZGUxMzcyMGI1MDhkMjRlOWU5OWRmYjlhODU3Njc5YWZhMzEyOGE5NjVkMzU3ZGYyM2QxMjU2NTI2ZjU5N2I5MmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.v1Rh3yxxaI7pA6WajY6jqaJTboJ9vUXYNpj0lCmTMPnnQgX03TyQWgenwvK9MqttSQZGDFxbBjXcAtbz76gM6w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103856_65_2463_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.738Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IklaUnZPd1JpZmZoci95UzFyRENRZm1La3ZuNTcrVTRCUklHRFNNZTlCakZPTndrMW5UM1RFOXZkaUZJR0NKbmsyRTA3ZzREcGFXcWROOHNDNXFqNWJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTZfNjVfMjQ2M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWMyNTVmN2Q3ZDdiMDYwMTU4YzkwYTBmOWUxNDdjYjAxOTM3MjUxNTU1YTA0YjA3ZDA2MThiNWY2YTgwNmI2NGFhMWNlMGRmMDQ3NjRmYzZhMmJiNTIyOThmMzA2OTVlZGJjMmY4M2RiMjBlYTNmMTI2MTk3NzFhNTI4MDA1MzdjMWUwZjVlOGQ5ODJhNjQ5NmEwNzI2MWVlZmI1YzljMTgyMzYyYzEwNDExZjVkZDVmYjU1MDI4ZmEyNjVhOTc3ZmZkMGNmODdhYWY4MWEzMmQxZTAzMTlhMzYxZWM2NDdmOTAyMjQwNjE1ZGQ1NzE5ZjQ0NGFkNmIwMWI2ODAxODk2M2E2OWFhMmU1YmExZTAwODE4YWRkOWRlNWYwMzE0MzZkZTk4MTEzMDRjNzQ1ZGFkMTkxY2VjZTVkMzllOTAxMTQ2MmMzN2I0ZjYxZjkxNDkxZjMwMmQ0YzBkOTkwNjkwNWYzMWRiNWM2MzgzOWI3MWFkOWM5NTE1YTcwNGNkYzk2Yzk0MDk4M2I2MTAxMmFkZDdjMmE3YmIzYmZmZjUzYjRjODYzZjdmNjhlNTk1NmI5NmM3Y2JmYzM3MjM1NWI1ZjQzYjY2NmFmMTFhNTE5MzU4Yzg2NzFiN2Q2NWMyZThkY2ZlMmVhZGUyZmYyZTg0ODA0OTA4ZTAxMjVmNjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7OVClXxZnw6cS5DrIhaE9lwS2gR5dpQZNQl9Xe-HTVXpGJQxo3rCZ1SY6tdRUikDSYX-UvL3xVDYJ_ICN3kEKg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103856_65_2463_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.742Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InpxSm00UGtBWGhMekROcnQwR0tmaUNZSjlvMFZYbDQzTGFJRjhUU1dIeWpYNkFjMVZCbjlIa3lLcWU2cEQvRG1yeVFpSXFSM215anFMSFVRRk9UQm9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTZfNjVfMjQ2M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MThjOWVkMTZhZDcxYjJiN2U3MjcyMWNlNmVlN2M4OWIwZjkyYWQ4MTgzZmI1MTBlMTA0OTVkNTVmYTM3ZTdiMDc2NTA4MGU5ODY2OGUxMDllN2Q4NGRlNWZhYmFlYzUyZmZhZjY4ZjU3NDUwNDU0MzU2NThkNGVjZWQ0YmM2Y2UwM2VkM2UyOTAwYzc0MzQ0YzdhYjY3MmFmNmZhNWI4OTY0MDUzMDkxYWEzM2ExNzc3ZjExOWY1MGYxNjkyMWM5MjAyNjIzYjVlZjkxYjMwNjcwNGMxNzJlNTY2MmU1MWZmZjAxNmNiMDcwYWIyY2E3YmMwYTY0MTRlOTRhYjcxMjAzNTZiMTk0YTBiODUwZGM5MGFiNzBhYzAyNDVlODM1MDhhYWEzZTFhYTAyYmZmOTRhYjI4NmRkOTBiYTE5NmY0YWYwY2NjNGYwMmM0OTZjZWQ5MGU5NDI1YWFhNjhiNGVjN2M4NzcxNjNkZDBmZjYwOWY2OTI1M2U2YzY2ZWM2NTM2ODRjYTYzZjlmNTcxMDg3ODdiYTBmY2JkNDA2YjVkY2VhZmRjMzYxZWE5ODBjNjY2ZTY0YmYzNjVlYTJlZTNiYTMyNmE0YmUxZmVkMjFmMmRkMjEyZjkyYWJmYTQ4ODU1NzZkZGM1NDk4Yzc1OTllYWU5MDZlYTVhYjFiZjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.G3GM3gDUvphscmEK5KTu9Ni79hKtwS1gwpMZa_-VpQcKxuvSndMUKzzZj7YC3KL2V5oC3RvJmIkUR7zO6NU7Aw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103856_65_2463_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.746Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik9WalE3OHVObW9Db0lyOGp4N2JaV3Vqa2FOQlNKc1ptaS9ySG82TFVSM0NRTmRZYWhaREFLMVg5MXV3eDdodk9ROGJJMmkxODMzeS9IWjV5bmFZOHp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxMl8xMDQzNDlfMDVfMjI1OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjdiY2VjMWUxODc0MGRjZWYzZjc3YTQ2M2Q0MDFlMzlhODEwOTA2ZTZmNzU3ZGFmOTQ2NDE5YTA5MWIyM2JiYWE0OTM1ZDZiZTYxOThhZmFjMDliMDQ1ZWMwYjViN2RhYjZhMTI3NTVhZGIxOTZjMTI3NzZlMDEyMGNjYTNjMzlmNzgyZTgxOGYwMzAyMzM0ZWJhZjM5OWM2OGJmOTFhNzdkNTQwOTQ2YzgxYWNiN2UwZTViNTRhNjU1MjQwYmFiMzdkZDdkMTFhODdiZmQ4MzliMWM1ZTQ4ZDFkNDQ0ZjE0YTBjOTBhZTc0MjE4ODg2OGU4ZTU3YjA5ZDNlZDFlMzM2YWQwZTM0MzRlYjY2OTZjYjE2ZWRiNzk5MzZkYzIzY2NlNTU3M2Y0YmY0MDQ4Yjk0ZDlhYWRiNDFkZjJjNDY3N2YwOWEyZDdiNWE4ODk4NmE4MTQwNWI2ZTJiYWU4ZGVmNzExYmE4M2M5NmI5ZjlhNDM1MGNkNjhjYTgwNGFiOTliM2FkMzVmODNlNmRjMDgyYjllNzFjN2VhMzI2ODYzZmNiYzI2MWFmMTgwM2FjNDdiYWRkMTRlODE2NDgzM2UzMDE3ZGFlODY2MjE5NDlhNjcwYjIwNTY5NDg0MzFmZGJiMzJlNzgwYTQzMDYyY2EzYTc4MGIxYjI2MTA3NTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.H58KuqQWxFR6t3uYO6TCmIDA3jp9mdj2qou18P_u-qeskUObaha6TuPIJQCtLOheOVwaBxBkHUSVZ0TRDaM1dg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210412_104349_05_2259_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.749Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InpVTzJpSmU3L0ZnVkdyZUhiS25WbnhGd2xhYnhSVzhwVVNYeENTNkdiOENkanFXZ0d0QW1wQ2JMMWFlbmhqcXZrMUo3QXJPSXZ4amxPNm1UWmxQYitRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxMl8xMDQzNDlfMDVfMjI1OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjFhZDllZWRmMGE2MTM5N2Y4YWY2ZGYyYjY2ZmNjNWZlZWQ4ODk0ZmM2MzJiMzYyMDA1ODcwZWE1MjljNjI3OGRmODZjMTc4NzlkN2ExN2QzNGVjNjEyMGI1MjdhOTQxOTIxNzk1MDg3MDg3ZjIzNTQ3YWMxMTM2Njg0ODdhZDMyYjEwNGYyZDBmYzE4YzI0Y2IxODNkNzdmOTQ0OTE1OWM2OTllNmE2OTg2NTJjNjkwOGY4ZjVhNWJmN2M0ZDE5OTViZDEwN2U4MzgwOTBmNzM1NmM2ODdiMWIwNGQxZmI1OTVjM2MzNjIxYzU5Nzc3ODViZTY5MmM3OTk4MjQ0ZDNmYmJiMjc2YjMxZmY0ODQ1OTQzMjZmNDA3YTBjYjcwOTBmOGNhMjY4MWFkZTFkMjYyNmViZGIwMTU2ZDQwOWY0YTgwYWUxN2M5Yjk5OTczZDBmMDIwY2Y1YTc3MmY4NGM5ZDJhMjAyYjJjNjUxYTg0NWZkOWJmMmExM2NhMzRkMDhhYzFlN2Y3ZTZjZDliMGZhNjcxMmYzMDA1NTg2Yjk5ZGIyOGZhNzRlNmE4OTIzZjE4ZDY0NzFhMzJjMjkwODQ0NzhiZjJjYTVmYzgxZjgzMGE3ZmEzMTAxODc4ZTNhODk0M2I4MmFmMzQ2ZWE5MTg4ZDIyNmQ0ZDZhM2Y3YzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4PJzcQvPAJuDwdlY-Q50dtXS9APRvJ37U2YreUzUhu_g1lA-ITXMUaPkGT3qBZWIG_SFo8CCnWNuueVn6y1OxQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210412_104349_05_2259_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.752Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkRjbkxIVW9HUEFHcWM5VFpZNVpFTE1DcmJsWllwWk9aQm5RSURTR0xVaHl2aHZNbklzOUVhbkRPdEVMQll6MHkvdFJBWllRbTZUOUZGSG9VTnVtV3ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxMl8xMDQzNDlfMDVfMjI1OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzM4NTk1YTM2N2EzM2E1NWU1NjY0MzU5YmIwNTdhYWVhNDFmMzExYTcwNjAzMmZlMWJhZmJmMzc0NjIwZjY2NmZhMGM1ODU3OGVkOGFjYjg5NjcyOTg5NTUyYzJmNjA5MmZkNjZkYjQ0YmMwZWIyMGMxZTc5YThhZjUxZDRlZjdhYjU5MGI5NmZiMmEzMzhiODUxMjQ1YzhhMTY0YjQ4NGFmMWVmN2FlNTNlNjQzYjIxZmM3ZGJjNTE4MjA0ZGY5ZTI0MTY1YmRlZDUzOTlmYmVlOGE2MDVhZjNjNWNhZTY4ZDFiYWU3MGU3YzBhYTU5ZmJlMWYxYzQ5NDIwNTM1MjI0ZjU0MjVjMzQyNDkyZTdiZDVlM2I3ZGEzMWExYmZlNzFiNDU2N2U0NWFmYzgyMmEyMjJkNzI4ZWYyMTMyNjQ5NTVhZWRiYzI0YmI1ZjUwM2ExOGY1YWFiNWU4M2ZmMjkxYWRkYTk3NDU2NzFkY2FlYzNlM2E0MzM1OGNjNjY4ZmMyN2Q0ZmU4YTI2MWQzMzE0MmU2MjU0NzE4MDNhNjljYjhjOGQyMTcyNDQzZjgxM2FiNDJjMWU4M2IxYTg5Y2M3ZDg5Y2VjM2Y5ODQ2MDAxMTk0ZjdmMTEwM2E0YzQyNGUwYjY4YzQ0MTI1OTA1NGQzNmJhOGEwMjdmZWMzNDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tkS3f_iIv-FixHCEMgiunAkaM0C37ohOzN_vuReTmwt-3YnBGjBPuM9oqp7R2gAdtmKEBM1W7-KiTwBOrFM8GQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210412_104349_05_2259_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.756Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ino4aEpndkZLckpUV2p4VHBTb2htanM0NUpPaHN5cFJKd3NHUVdtSnp5SjF0KzA2eHFadjRlelJBZXk2aDlORzJKSmJER1hDYWdyMFA1Y200bWlFOWlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxMl8xMDQzNDlfMDVfMjI1OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWJhYTA5NzFlMzNiMzVhOGYwNjEwZmJlMjM4MDJmOTQzOTc0YTY0MzhjMWI0MTFjZDY3MGRmODI0MjcwNDEzN2Q2MTZkNDE4MDA0MGJjNTdhOTQ0MTIxNWJlYWFhMGZmZmRkYzJiZDdlNTE2YTgxZTdiODVjMzIzM2YyNWU2OGY0YmU5YzhmNmMyNjY0YzhiNzZkMmQwN2Q4NThkOTAyYjhmMWJiMGE0ZmI1NjllM2Y0NGU5ZWQ4NDViYzIxMmI1MWFmMWQ1M2YzOTcwYjRjOGMwOTA4NDkwZGViZWJhN2M2NGY1MTg0NTNmMDcwODUwY2M5ZmU1MTE1MjBlZTA0MmEwMjcxOTE4M2E2MmJlYmE2Zjc5YzJjOGJmMDljZDg5NTk1MjQ4NWZlYTdlOTBiMWNlZDMyMTExNTI4ZWIyZWM1Y2E1MjdjNjE3ZDFjZTYzMTU1YWE1YTEyOTEwNTRlZWU4ODM1ODZlODhkNDQwNjVhNTY4OTBiNjI5YTYxMGE3MzdjNjRmZTk3ZGYwMzgyZTg5MmI1OWU2OGFlMGRkMjJhNWNmZWM1Y2RiOTYxYzVjNTFkYmE0MDk1MjVmZTNjYTUxNmI4N2U1OGNiMmI3MmYwMjQ0Zjc0YTU1MWFhOGU4ZTI1ZTVmYjBiZDM5NDgxMjdhNDVkZDYwN2YwM2QxMWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DRIaRkv_x2kq9e6lrQ53lMh-AixYr7Spvwi5-OK7YG9Ob6vuryV-Yrs6kWE8JVsYnqLL9oDk5nJK1RxexobqLQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210412_104349_05_2259_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.760Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlhGQWttYmZ0Q01DSmFDTDhMY3dJalBhbi81dGRtVzhvVFdFYmJVUW5wcnV3R014akZ3NmFaTlB3bU5JbzhzSWExVkJlcmlTUVV0a2lnUGpmclQwRU9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDlfOTVfMjI3ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTliMDEwMjc1MWY4Zjc5OGRiOTZlYjAwZDhhNThjMzllZThjMjk3ZTk4ZjFmMzg0OGIzYTNjN2U5ZDRkMzdlNDM0OTk2NmE4ZTJmYzljOGYzMDkzYTE4YWY5NmZkZWE4NzZkNjAwYzgwYjRkOGJiODA2YmU4MzVhMTg2ZTk1MDEwMTlmMTgwNzBkN2QxZDEzY2UwYWUxYjE5MTRmOGY3MzM1ZmFkZGNlMjA5NTg2OGRlZmU3OGUzMjkwNGZmOGM1ZDVhZTNmZGJmNzkwMGNlMzFhZGJjMTk2MWQwZTM0NTAyNTcwOWQ5YWM3Y2Q0NDZlYTE2MTgzZWFmZDJjYjNlMDMyNTUzMmMyZjZiODcyNTY5ZjNlOGY2YzdiM2RjNDk3ZmFmMjMwYzk1MzA1ZTI1OWEzYjM4MDNjZTA2Y2YyZDBlYTE1OGEwMjNiOGI3NDU4MDQxNGRjNTZiNTAyZTc1NDVmNTkxYTA3OWM2OTIzZmIzNTdjYjI2MjE4Y2M0NjU1YzFhYmU3MjdmMDdjMTAwOGFhOGZjYWI4ZjhiZmY4ODNlY2NkMjAwMjcyZDM5MzFkZjAzNWI3MDJjNDA5NjU1NDY1ZmI3MTI4ZWY3NjNlZWI5NTJkYTQwNjE5YWNmN2UzYWQyNzI2YzhlNGU1N2M5ODE4ODczZjExZmM5YTI4YzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rvcsjsdGuvjVe3m80aGYWQJSXHfeUGb3ORGj--nhyZ9xY1Mif7JeZstMQsNoOLUn8-DMSqZivD7R0apiCVV6bA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104349_95_227e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.763Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImtuQkZSU0dhdmV5UjltdzR5dStqOGlmUVBPTFcyUHFkT0JhYS9lR2IxM2NydGYzWVZFMUZPMVRBVFNRalRranFxeEN1Tm1ZQzlNc0ZTeDR3MFp3S2pnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDlfOTVfMjI3ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTM3NjhmNWFkYzY3NmFmOTFkMGZjNTkzOGI3ZjdhZTc3Y2EzNjkzMzU1OGM3YWIyZmQ1ZmQ0YzhmMDYxMGMzNjk4YjEyY2ZjOTM2MWEwMDVlMDNiMDI1Y2JhYWEyNmIyN2EyZjNjYjY2YTVhNTVmN2VhMzBlODM1YTczODQxZWVkOGY3MjU2ZmMzYTJhYjY1Y2E2NTVjM2UzOTI0M2I1ZGUyMWU0Nzk2YzBmMmY1Mjg2NjZiMzk1NjgyM2EzYjllZDA1Nzg1NWQwMGRiOTRkMzM5MjBhMGE1MGYyZmFiYWZhZTdmOTc4ZDJkZDZmNzVhODU1NTYzNWU2YmU0YjEwNDFiYTMxYzFiZTkwMDlkY2U4MDZlZjBiZmJlZmMwZmI3N2ZmOTA4ODI5MjZlYzdkODZiMDZkMmM3OGVkZTRkYjljYzM5MTJlMGViY2Y0YzZkZmEyOGMxZTVhMzBkMzZlZWY3MjRjOGZjMjE0NWIyYWM2YzM3MWM5ZTM4YjhlNmFjZTA1MDUyNDE0MjUwNjRhNDdjYjgxZWU1ZTc5Y2YzNjllYzQ5ODkyODhmMWI3YTU3OTZiNDFkMWEzOWY5NjU2ZjllYzE0N2UxYjQyYTI4MTllMjM4M2NhOTVhNDI1YjBlMDYxN2U0MGZlODBkZDhkNjc3MDI5ODBiZmQxZmU4NWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lTpmXH0CadRb4xlHlJQyYzCHbSTZVmA4WZSFpN-oGLNNSKAXkcLq0hT3PY-kTD3YUoNxHtpsQ8DPPG4Tg3mNaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104349_95_227e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.766Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InlIT0dSMVJUWktmdThzTy92aEpOZmlJb2JxdjZKbVJlcDFjY0tSUEJxNis3WEU3dFg3UUVHMVN5MEpBRVNVUFJHWjkyRXZlUWVaN05admtINW55R0tBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDlfOTVfMjI3ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWE1NDg3ZDUzMGZmOWYzZTQ1NGRlODczMDE3YzBlMmU4M2M3NTQyN2FhNjAzZWVjOGNhZTdhYzI5YzMwZTNiODgxYTNmNjI4N2I5M2MwMmU1M2Q4NWRmNTQzNDYzNzZmMGJjMDk2M2RmNDc1MWQ1MGJlYWRlMjQ3YTYxOTMxMDY2ZTI1MzQ3YzY4YzA2ZDkwYTM1Njk5NDNlOTA0NDlhZmVkM2ZhMDA3YWI4ZDk1ZDEwN2FiNTgzMTgyMWYzNmI1ZTFhNWJhY2JhMTkwMjFkN2ZiNDA1MzFlZDQyODEyNjM3ZjY1N2JmZGQxMDZjM2Q3NzQ1MGNkYzQ2NzczNmQzMmQyZTY3Y2YyY2U0OGZhZDMzM2JjYTcxMTU3YTRjODNhZWI0N2NjYTM1NzI3NjA0ZGMwOTgzNmE5NDdlOWQ3YmJjZjcwY2VjOGU5ZTM0Nzk1NWU4MGVkYmQ2YTIwNmY3YjcyY2Y1NjU2Yzg0MjEwZDEzZWE0ZWU2NDc0N2ExZDJlYzYxNTIxNGFmMjFmNjIzNGM0MGU2MjM3ZTFjZDFhNmM0YzQzMTAyMWIxZmQ5OGVmZWVjZjRmNDA1NDFmMjMxZmViMWFmNjQ4OGM1ODI4Y2JhMmQzYTQ0OWYzNmZkMjZiZGM5YWE2ZjMzMzVjODUyYmJmOGExYjNmNjVhNTRhMDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.kDuE0BAJ1deHRqAbIfrIvbDHY-DTKyLVyk4SM27_S0Otnjmtlg-cVwKjXOrTbfqTxa8HshNIdZ3hcJFmecCkXQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104349_95_227e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.769Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik1ZeWVhb0FldkZaMklNWEJON3c3c1hZMGFldmVXWW9sckdFaWhyeG9vY0ZZYjlEb0l3blQ3K0NBWFRkbUZuMFphQ21oUGhHWXdJUDYyQ21kUXFuT1NRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDQzNDlfOTVfMjI3ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjRiMjcwZmJlZTI2ZGI1MTUyZWYzZjVmMTI3MjE3M2E5NmQxOGQxZjIyYzYzNTFiODdkZGY0NzYxNGUyZDIzN2Q2M2YxNWQ3ZDExNzExNjZhYTMyNjA2N2MwOWRiN2E0MjhiM2Q4ZDA4ODM1Y2E5Nzc2MDU1MjRjNjM4OGY0N2ZiYjEyOGQwOWFhNzc4ZTk1ZGJlNjY3NTAyYzc1NDhhMDNjNzAzNzUyZGE5NDdjMzdkNWEzNDM3MDY3YTdlMDRlYTZjMjA0YTBmYzc0NGJiZDE3MTRiODJjNzQ2ZTI5NTUwNTFmNzU4MmU1ZDhkYmJmNTRhNGFjZDFkMGUyNWU4ZmE2MjhhZjc0NTY3NzIwMWRiMzg4NmE5YjdiMjUxYTBlYjJjOWQxYWIxMGY3MzRiMzdjMTdjY2MxZjg0ZTBjYmFlMWYwMTVhZmY2NWE3MTIzMTQxMGQ5NGU5ZWJmODM3ZjljNTdhMDk0MDNjN2VlZDYxYmZkMjY3MDcxZGIyNTA0Y2EzODgxZTkxYzIzOWM3NjUzMWM5NGE0YmJmYmFmMGFkNzkwMjMxMTQxMDA2NjllMmI1MmVhMTI4OWIzMDFhYTc0NjZjNGVjMThmZjA2MTZhY2Q1OWE0MjkxMDA3MjU4ZTQ1ZDk5Y2ZkZTA3ZTljZmY3MDBlM2VhY2NjNGU0ZTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AuR7QKpkQWBaPgbZSrgmpRheLKig3UchpzADcTIlTANe-Qprm7WaTvW3jNB83dRZ7urlOVH2Y6Z-Q79VGcLKnQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104349_95_227e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.772Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InBVUm9zODJ6Ry82ekMrL011aWF4QVo5ZHIyaGdoN05NL09CMDFlSzFzdThSbjVVUjVxVmxVU3YwY1IzdTFtN0dLdHpSZ0hCSHlLMHVhYjY4Nm5EUFJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTlfMjJfMjI3OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njc1ZjcyYzAxOTdiNGY5MDE1M2MzNDM1NGI0OTA2YmE0NzFhMWVkZWNlNThkOGM5OTVjYTYzMmZhNzhkYmFmMTcwNDhhNTRlOTQ2YjllYzJlNWZkOTQ4NmI4NjUzMzI3NGU4YTc1MjM3YjljYjA2ZTZiYTc4NGU0ZmE3ZTYyNjg0NDk4YTAxZmUxNzVjMWIzNGYwZmI3MDJhY2Q1NGFlYWExYmJiZDIxOGVlZjFkNTAxOWJjZjJmNmM4NTFhNTM0MDU5Y2I1MGZhMjNlNTFjNDU1NjFjNjEwZGM0MTljN2UwZmJkY2MxYjEwMWVmZDgzNjFmNzI4Mzk2NTQ0OWUwYzMxNTNlYjVhNmZlZjliYTUwMTk1ZGI3ZWM0NzczODc5Yzg3YjdhMzNkZTVkMTM3NDM0N2U4MjI2NjFkYzYyNDc4YzFhZjQ3YmZjMDY0NjYyMzI4ZDVjYjEzNDU5NGVlMTI5ODU0MTIyMTQwMTg3ZjM2YmJiMjBmMDYyN2Y2NmVhOGJiZDQ4ZjQzMTViNTY4MWM3ZGMzODBmZDk5NzU2ZTY4YzdhNzFmMDgzNzk2YzAzNGMzMmI4NWYyZjM4MWZjODRjMWVhNThhZTQ5NzY2NWVlZTEzNGNkNzk0NDllNGY4NWQ5NDRmZjNmNzZjNGNhM2E3NmUwMzhjZjJjYTZkNTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.m1rHdMens7_MqqTWnwsA8GGGYgoooJQaULw1cZ69uWTHW6hX1U3tl4mEE-laVb-rV4eAdt3OeLxGGqlKM19U-g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104259_22_2278_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.777Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImdMVzdJT25xMFR0OXZIdHJDbzhMdC9kVy82L0RxRis3SWJBTU9kOUhPYmN2QmN4aTZiVUNJREM3eFhMbURkb3NvTnFWUSs5QWF1OXFYVmdDRXRhWWhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTlfMjJfMjI3OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjVkNWE5OTM1NjdlNGMzZDU0M2U3OTE1OTY4OGVmODY0MzIyZjk0YzcwNjc2NjUwNmQ3YmIzNjdiMjk3YzliYTM5MzgyZmY5Nzk0MzU2MjI4OWIxNTJhMTgzZGFiNGFiN2RkNTNkY2Q0Mjc1MDlhMGIwMWM0ODE3MjJmM2QwMGJhYjFiOTUxZmQzYjA4OThiOGYzOTM5ZGFjOTdiYjVlYWI5OGRkMzk4MTVkZjQ0NjI5MTZkNTQ4OWQyMzkxYzUxZmMxYjgwMzA1MWY4MzYzNDhkNzA5M2VjZjdiNzY0MzUzYWZkNTUyNjQwNjI1ODFjYzRjNDU3NjBhMDFmNmY5ZDM0NjVhNjVlM2FlOWQ2OGY2NmIwNmI1ZTViMWU2NzdkMzQyYWRiODc1ODdiYjQ2MjY5ODNhYjMxN2RlODVmMzBkYjI3MmRmNWY5YTRjZjlmNTM3MDY2NTliZGU5YTkxMWI2N2I1M2MxNDNlNTIwMTdiYzhhOTIxZWJkNWFkNDgyMGYwYjU4M2FhNjJkODJlMTZkMjUwM2QzYTUwZWQ2YjZmOTc3ZmY3M2JkZjRjZjNjNzE0OTQ1ZWZiMWRmNzJhNmY5NjhlMTAzMGI5ZDRjNGY5MWU3MDBhYjNiMjQ0ZTcxOTEwZWJiNDcyZGRlZGVmNmU0ZWU0MmM4OGQ4MTMwMmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0mrZfIldgSz-FSure8ZRDyUMoSzqsJWZVAY69te1zTZjj_NpG1f9QV04tbOu-FgfKVDDkwLeEaABSrFLoPaA3Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104259_22_2278_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.784Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IllxV3pYWFQrR3piVUdOT3QxdlRZcDNNZzhmVDJEOXRBdUZmZFJENzlvKzZ2bHQxY3N3QWVxek83TmNHajZ4YU5kbDZzTVd5THNIb3U0cTFvY0syNXlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTlfMjJfMjI3OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzk0ZDczZTFiNjUzMzNmNWE2MWE5MmFkZDJjNjNmOTBiOGM5NTc2YWMyNDIzOWUwZDNiZGZiYzczMzEyZTQ3ODY1N2E5MDFiNmQ1ODlhZTY0MjFlMjlkOGY2NjlhOGZiNGRiNGJhODI2MmRhOWQxZGU0ZjA4Y2Q5NzU5OGU0ZGVhZDU1NjRlZDk5NDE5MjFhMGYxYWRjMTcxZmI4ODNiYjdlNmJlZTIwZjM3ZjNlMTExNmI0MWU2OTY5Zjc2ZDQxYjNkNTRmZDU0OTI4ZjUzNDM4ZTVhMDFlZjMyNmUxODNmYmUxNmUzMWZjZjczOTUwODUwNTk0MDhhNDM1MTQ2NmM1MzMwZDM5MDYxYTE2Y2RiOTQ2Y2JmNmI4ZTVlYTNmYjI4MzYwYmI4MWM1N2M2MmY5NzYwMTBmZjg3ZWVmNThjOGNmOTZjZmNlZTNiMTk4NjNlZjRjODg5YmRkM2FkMThiZTk1MmVhZWRlMThlOTYyN2Y3Y2UxZmQ3MTg3MWQzMDg5MWNmMzc4MmFiODQ0NjA3ZjQ3YjQ2NGI5NmM2ODQ4YzNhNDZjOWI0MTM5ZmJmZWJhMjMzZjY2MTFhZjYzOGRhNjFhMzY4Yjk4YTc4MmI2ZThkZmUyMWQ2MGJkYTkzMTc2NWFmMTRkOTcwMWNhZWJiYjY2ZDZkMjFhNmU1M2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6hGrhcyIy8Gdjityb3Ut_JdtZE9lNvWcAAkv5af_CTm7PFMUHVGbZ6aQyxhcCS5fNXvKjkg0jfLs7j0QdZKAGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104259_22_2278_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.787Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlpNT1MrdHdabjgxcUlGaXBVQmhOZWV4cTZ6NmU5eVlwekprQzR4VkJYS3VEL0IzWURteStHRjlTYVhqckkySlhLanZYMklla2U0NjJHRi96VHNhWElRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTlfMjJfMjI3OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWI1YTlmYjMyMzIwMjEyYTY4OWM3ZjY1NWNiNmMxZDc3ODRkZjU0YmJiODk2ZWViN2UxZjU2YzdiOTE2ZjgyMjAxMzNkYzY4ZDcwOTZmYjBkNmJjNDA1ZGFmNDBhNWU1ZTczMGY4ZmIxNmRkNzk4ZDJjMjA2YTNhMTIzZDY3YmZiYTJmZTFiODRiOTc3MDAzMmZiNzgzMzk3YWQyODNmYjY0ODExMDU1MGYzOGViZmQ4Y2YxNjhkZGQwZWZiODhhZTRlMzhjMTg2MWI4ZWE3OGY5MjAxYTNkNjRjOTlmYTAxMzRmZWMwY2VkYzk1NGRjODZlOGM5NjEwZmM4NDJkNzUzZTRjYjk3Y2Q5YjUxYjNkMTc0YmYzZWMyNWZhYjRmMzRlODkyMjljNjhhYmIzODQ0ZDZiZDMxYmI1OWJlOTZlZDhiMjVhZmRhNjhjZjBjMTAxMDEyZjcyZTBkOTYwYjQ2NjkwYTA5NDZiMDgyYmUzY2ZiMzhjYmUzNjI4N2QwNzEwNjNjNDNhMGRkYjgxOTBmNGI3MTI1M2NmMTQ5ODQ3MjIxNzE2NjYyZGNmZWFkMGE1NDg3NTQ4ZjMwNDc5NWEyMmEzMzE5MGQ4MjM3NzI5YzQwZWY5ZjliYmNiMmI0ZTc1YWRhYmFjNzJjNGMzYzA5MGZiMzBkYjhkMGI0YjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eGoknyyIG5lahrBgggAwRZZecDUd5vbmPQrLXV0RVv91EXPFfI5YTb2aP8xlyptFwethirdarrBmj50t-I07GQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104259_22_2278_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.791Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im5heWpoSjE0RG55WlpSdnY0UDl5NTgrWC9qZ3A3OWpqeUIzL1htN1UrMEVwK1c3NW9UNDN5VEpmWm9TYUxBb20yd0poTU55aG5hZEUyaERuUDM5UHZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTNfODZfMjRhNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGRiZjk4NWYwMzk3YzZkODU2MDhmZjBkYzkwZmRkZWNmYmIwMmIyNzgwNzJkNThkNmZkZTQ3OGYwOWQ1MTBhNzRjOTJkZTRlNzgyYzcxNzdkM2EyODZkMWMwNGQyNjE0YzZlMTVjNTFmNmIyNTNmODE1NTM2NTk5ZDQ5OGE2M2ZiNzRlYWVkMWJjZDcwNTI4MDFjYzAxZmJiMWI1OGEwYjI0OTY1MzM0ZmIzMTI0NWFlOTcxMTRkMjA4YzgyNzY0N2FhMTlhODg4ZjBjN2IzZjAwZTM5OGRiOWViYTdhZWZlYWY3MDMxM2JkOTY4ZjVmM2JkZjU4YjA1NmY2ZTM1MWNmNzVlZTdmYzlkMWU1MWQ5MTdmYjcxZjMxNTFlZmVjZjU5ZmEwZWE1OGE3MDFmZTg3YWU3NDdkZjBhMTA4YmU1Zjc3YzJhMTFmYmQzNDAwOTEzNjAyZTllNjM4NTY3NjU0ZTYzMjMzMTZiYmFiYjJkYzRkMjEwMTgxOTRlYzdhYjlkMjIyOWVmZjUxYWNhMDE5YTM1YzM5YjU4ZTY5NzQzMjZjZDU4N2I4ZDc4MTNhZDY1OWJmMWM2MTZlMjc0Y2UyZWVhYTk5OTdhZmE4MzQzMDE4MmMwNTg2MDhiYmQ0NWJjZjJlZjgxMzBmYTM5MmY5OTk2YjU3NWYyNzZkNTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.U8Dsmz57CMhtMEuGJNBWlyC5wArxLbTv2piTV9fn4FKJwHMhPUymSHkcAxS_C1w1XZ86adGx6GlJg5E2p-vNnQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112213_86_24a4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.795Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ijh3QjlXZFY1aDJKYXdMV1ZzNjcrR1ljUVc2M1NpQlRVWGRwc294czNRaG1uZVZwdlhpOXdxbWYwOU5EZ01LZVRBTENnZXJ5d2ZJYkl4djIvZjlJdzJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTNfODZfMjRhNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjlhM2EwMWY4MjliMmI3NjA1NDc3ZWUzOGE3ODcxNGM3ZGFmOTgxNDM3MzQxN2JlNWM1YzJlNDQ2NDhkZDIwYmM1YzE2YzAyMmVkMTI5ZGU5NmQ1ZmY5MDFmMWUzYzE2MjgyNWE0ZTA3OTY1ZmI2MmJmODI0MDMwZDAyMWI5NGY2NWRmZDdmY2I2N2QxNDEwNzVhMzYzMDRkODUyNjM0YjhlN2U1M2VmZGYzY2YwN2MzOGY4ZDcwYWI3NjM0MDkzOWIyNzM1M2YyNDg1OTdlZWY4ZWMwODUyY2M1YmUwOGRkNDM4NTRhYjlkMmRiNmZjZTZhY2RjMGU4ZDgyZWQ0MmU2MjkzOTJmNDE4ZmU0NTE4Y2Q2N2Y3MjYzOTU4N2YyZDU4YzdjZWJlZmQzMTUxNDU3YzI5MzE2OTI4MzQ3YzhjYjA2NzNjNjMyOTYzMGQ4NmMwMmFlMGZhODZjNzA4NDYyZGVlYjBmMjVhOGQ1N2ZkYWY0MWM2NjdiYzFjNDMzMDc3OTFhMjkzYjU4ODQ1MzE4YzBlMzc1Mjk2NDdiNTFkNzY3ZDEyZGJjM2VmOWM0NjFhY2Y4YzIyMzA2M2M1ZTRmNjk3YmE0MjYyMmI0YjZmMDVhNzU2NjNhOTQ4Y2VmN2MxZGUzNTlkM2MwYmEyNDgzZWU0Yjk5MzczOGY2YTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.85EZ7UA6cDKQNhPcWIJRzOfATv-uID-M8ysf_4jMSH0BjOWS1GwjZTeYYPGIaEPHE8-BLQZJRhTrsbW2kWpltQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112213_86_24a4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.802Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjhiUDJNMjd0ZEdJSWwwZzVTWTlYRWpHV3ZhK0RHaWwwMzR3ZkEzQjFYc3diWjJKM05weTdGeGF6ZDhLR0xhM1Z5Vmk2N052M1hoV2IrVW5FVXpjQWJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTNfODZfMjRhNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGRmYjExYjJmOTYwNDM1NzMxMzQzZWU5YTBhZDM3YWVlMmM4ZmExMWViNjU3NzE1NGE5OGY2MWRmOTY2YWYyZjIwM2VhMjA2YjFlNmY5ZmQwOWYwMTE5NjMxMTc2NzQwNTE4ZDdkMjk0YTBmYTZkY2YyNDc3MGVjODQyN2Y4ZTc2YTQ5YzQyYWIzOTY0Yjg4MmVhYjY2MGZhMDkwNDk4NjVhZWYyYzM1MzU3NDExOGYwZDY4ZDBhOGZiZmE2OGY5ODJkZmVjMTY1NzM3N2UzMTlkMWEzM2ZiMjIyZTI3MjRkOGQzZWM2M2FmN2Q2OTNiOGU1MDNhMTY0ZGFiOTU4YjgyMTY3YzlmZWJkMmViYjQyNmFjZWRkODRiMmE0MmEzY2ZmZDJiOWFhMDA4ZjdjZGY5ZDZlOTdjNzM2OGJjNGVjNjZiMTEwYjQ0Zjk3NTMzMjlmMzM1MDgyNTNkNzkwYjdiMzMzMGVkYjA2ZjIxMGNkYzZhMjRmMGUwM2FiNTQ5ZDUyOTIyMDdlYzBjYWQwMTQyYWRjMWU1OTQ3NzIxMjVhYzQxMzU2YzJhMDM3YmM0ZTFjNDhkYTIzY2E2OTg2ZTAzNTVmYmI1MDZjZjgwYThmZWUxNDU4ZjVkNzY4NzQ3ZTg1NTkzMDkyNDY3NTlmYWJhYTQxNTZmOTczYzIzYjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.d56MKFGY7rxujy8GeCoNyOXvvAGveuUCgT54T3cybXBR1lx2NSuUDS54Q5-qa0ftyXyF8alEOBiyA_Fx34MxJA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112213_86_24a4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.805Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IktYS056QVJmYlBTcTFMUmdRTmtrSHVhd2djREtOVzZMQ3F2OE9nTlNqVWp5eDlsWWpMMzVOWnd4T1FJb2dFbU5tWU03YXlOUUtMb3Q3ZUVlNFpWT25nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTNfODZfMjRhNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTdiYmM0NTVmNzIxZmU3NWRkYTJiMWE1ZDI2OGFjOTIyN2EwNmZjMzQyZTA0NWZkODBjNTBkMDc1ZTc3YWI5ZWM5YzFjMzdkOWIyMGE1MWI1NTk4MTEwYTRjNTRmMjk2YjEwMjQyMWFmZjE2NTU3YzQ4NGI3NDYwMGNiNDY5Mjg4MmRlNjY4Y2Y5YzMyZDY5NjdjZmUzZWEyMjY0ZWZkMjc2Y2VhMmUyMjY5MDNlNTAwZjczNDlhNDU2MDAzZDlkZjRiMDAwZWRhMjBjMjhlMjEyODY2Nzc1ZjYyNTIyNTBhZDljZTBjYTdhNzkxMTYzMDIzMWY1MzI2YmI4OGVkN2Q2Nzc2MTZlOTNiYzBlMmFlYzFjYTAyODdlOTBkNjdjZGUyMTI3YmY3ZWEzNWZiYThhOGYxODgyYWRkN2I1OGY3YzQwMjZkZGEwMGZmYzFmZGY2ZjRiNzkyYmJmZjg4ZWM0ZDMyM2Y2YWQwYTU3NjcxMWJiNzNmMGMyYTRmYjAyNjg1MTNjMGQ2OGNmZGYyMWVmYjNiNWM1OTJjNzZlN2QzMjAwMDU1MTJlMTZhNTJjMDk4NzcyZmNmOTI0YWFiYTcxMzVkZWZjZTUyODgyZmI0NTA4ZTEzOTQ4MGMzZTdhNzE1ZTJmYmI5YWE0YThhOGZhMGM5NTFlN2JmODY2ZjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.M3nj93jZsUkYamEqn-brbH5gqmGJw59li3bAwooDeLPqQNFep-_YB-2lFvPRa1SrPm1ikRwQbZrc_8RPZlHceQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112213_86_24a4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.808Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImFTYkxRM2w1UURucytCWGs4MUhYNkZvOENETEFtUmRsdXJYaHNMdTJmYzN5a0Q3eUYrcER3QWUxOElxOGN0V081VkRCUUh3bUNpQ3dXVGhrVUFWK2dnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyMl8xMTI1MTJfODlfMjQwOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGNhNGI3MWZiOTExNjI1N2JjYzhlYzE5NDNmZjI3ZTNhZWM3ZjkxNDE4YzVmMWFlMzQ1YjRkM2Y3ZDgxN2I2ZWZlYzE4Zjg1ZDUwYzUxY2MxZmNiYzU1MzEwZjlmMjllMzYyMmZiMDQ5Mzc2NWZiMjk1ZjJhNTlmMDc5MzhlMmNjNzY3ODA2ODhhMThhYzY3N2YwNzFkNGJmYWRhMWQ4YzJkNWQ3NDc2OTI1MGQ0YTQ1NzZkN2M2MDRhMzNmZGQyOTg2NzNiYmIwZTUxYWQyNjMwNWMyNTBlNWRjOTBiZDRmY2ZmYWNmNTYyZmU5MTgwNDA1MDk1YThjNDBhNGVhZTA2ZDJiMjgyMDJmMGM5NGE3YzExMzg3YzA5NzY5ZDU4OGQ2OGI0MjY5OTEwNDMwY2ViNjI5ZTI1ZDQ1ZDZlY2YwZTkzYzFjNjM3ZThiYzBkM2UwZjJhNTM5NTAwMDBiYzgyNGU2OGQ3OGNmNWQ4M2RkZDM0YzAzOGE4NzM4NDQ0YTRiNjMxYmE0YTI5YzVhOTVjNWRiMDM5YzQ1YzE0ZWNlY2UwNmQ0YjI3MWNmYWY0ZmFlM2YyMDUyYjE5NDhiY2ZmOTY3YjAyNWMyNDI5NzA5YjNjMzAwZjQwZjljNjUzODE1NjgzZjE3NWJlOTU4NzNkN2RjZDFiZTQxOTIyYTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0PuE-CPtdhaBaETuItyH5JQ5Agnrun2_0sbSNdGvuhoykpPae3fZ11dnVWKB5t2KAeC083jLjT7sF3fhTELh1w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210222_112512_89_2408_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.810Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImFzZ3hsNkEzaGlvWG1yUWpQdUc5cit6Skszb3o0TjJONDFuaU9SdStIQ1ZrSXRLeElIazlQOXJiNU5TaS84bzlYZHZyM0R2aUp2bXcrUUEvM0prTGJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyMl8xMTI1MTJfODlfMjQwOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzdlZGU2NjY3YWE2ODgzMGI2Mjg3ODBmNzM5N2YzOWJjZDE2ZGFiMGZmMDEwNGM3YTE1Njk5YjgyYmU1M2NmYjFiYTM3ZTFjZDI0NjkzZWYyN2JjMjBhZTA5ZjY0YTJlZTFhNDAwNzNhY2RmNjE2MzQ5MzcyZjNjY2QzYTNiZTgyYjYzZTQxZjQ0ZjU5ZTk5YzMxOWE4YjExMjE4OWVlMzUyNGViNDUxNzRjNzQxOGVlMzJhY2Y2NWI4OWUxYzAzNWEzZTQ0NDg1NGVjNzk2MjU1M2MzNjQxNWQ0NDA2ODgyNDhmZWFmYTVhNzdhMjllYjkzNjVhMzg0NTAxNjg3YTk4ODEzMWJlZmY2ZmNhZWVmNjRlNmVmNTYxZTg0OTJiNzUzNDc0ZDFiNDdiYzgzNTNjY2YwOGUxZmFhZjBiOGE5Yzk0ZDQ1YzA5YTJlM2MwODA0NjM5NzAyZTk3ZDBmYmZjNDg1Y2QyYzQwOWZjYWI5ZWIyZTc0MmNlZjFjOTcyZmQ5ZjllM2M3MDAyMmRlN2RjYzRjN2RkMDljMGE3YWVjZWYzNjRhOGFiNWNiNzBkYTgzZWEzMTI2OWI4ODNmMWFmZWIyYzdkNjExNGI4ZDA3N2VmMDI0YmM1NDMzZWI4ODQyMGRjNTBmNzJlOTVjZTdlY2IyMzBjNTY3NzdhNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.r_23qoDSKAjX-Lmo2qQ4ZhP5dN3BOAuLwwU8rMhMOGoexcvxRNKtBgysgmrnTnpIdiWEEJ3LSmkgfwwz1oMitQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210222_112512_89_2408_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.813Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlNoRHZ3cS9qMjQ5MmM4RjdmV0QzVlMyN0VjWFgzVHdtUHB0TVF2NHF1c1h5REdNbUlFdng2Qmp3UWVUcjNlY0VzbnF1VXYvOVVrT3NoVG1LK0RraHVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyMl8xMTI1MTJfODlfMjQwOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2ZkZTk4YzAwZDBlNmZmYWExOGI1N2JjMTllNTA0ZTJlOWZkOTY3NWRhZmM5MmU3MTUzZTA1NzViMzk4YmU2MDM4MzEyZGNjMTFhNmQ3ODk4ZjRjMWU1MDQ1MjVlYThkYzhkODU3MmNiM2I2NTc2MWE0YTBiMWE3ZTU3MjFmNmJlMjk1OTk1NjlhYzBjYjUxNjllZTAwZmNkYmU4NGVmMjZiYzc1ODQzOWJhNzE2ZGU1MTk2ZDg0MTVkYjE5NWZiZjU4OGI1ZDE0NjY1ZTZlYTU3NTQzMDg5MTJmNWYzNzcxMjhmOTQyNmIyZjYzMjNhYTFlMWRhNTFmMjY3NDk1ZmRjYzQ5ZGM5ZjExZGU4ZTY5OTFjNDExOTFhYjk4MWI1ZmRiY2JhZmRhZGNjOGVmMWQwYmFiM2U5NmIwNzMzY2E2NTlhNTQ5NzM1ODE5YTYzYzlkN2ZjYjE4ZmIwZTE5M2I3NGZlOGMzZjRiMDRiMGUwNzE2ZTc4MWIzYTVlM2FjYjgxOTk1NzVkOTgyN2IxNGE2M2FlMGE5ZGY3NTM1NTM4NGEyNzJkN2ZmODYzZTYyYzFmOGU4MWRmZWY1ZmUzNDczZjc4YTUzZDRlODU2Zjk2ODg2Nzg1ZjI0NGUyYzIwZDEwZjI4MjU5YmZiZmQwZjJmZjYzMjgxZTk4YmFmOWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.e9Jz2trrScQx9xSVUtpqNAOCpyfX0s9XD4132jUUfbrqJv5Bhmopgl4ykuhlr-fxbAXqKKtUjvKSIILHcmjg_A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210222_112512_89_2408_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.819Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InlKZmRDWi9iVlhxK3NadEhaNUJiMjNOY3JIVUdzYjYxR0x2VWNGNkY4VGkyNXJiZ1ltNm1oeGp2RG5qd3ZtcHlrZVkybWNCNzFZVjJ3YUxPYmZjMlpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyMl8xMTI1MTJfODlfMjQwOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTUzYTVmZWQ5NDM5NjlmNDgyM2I3M2I4MjI1N2MwZjBmNjQ2N2JhZWUwMjBmZGRmN2M4ODQ1YzkyZjdmOTUyMjFlNTQxMjg0OGM5NDA4ZDkyZGZkOTNmYjFmYTg2MzQ0NzY3M2RmOWMzOGRlYWUyZDFmODVlOGYyOTI5M2NlZDBlZGE3MTFmMTBkYjg3ZGY2MmMxMDIxMzI4MmQzM2YxOWU5MWViYjJkYmVlZTUzNGFjYjlhM2U0NTA1YTQ0M2NlODlhMDY5ZDBkZmEzZWVmNjU3MjkxMmVmMjc5ZTIwZjQ3YmNkYzFlZjlkMDg2M2QyNjdhNmJmZGQ5MzAyNWRhNWYyYzE4MWY3OGY2NGE4YWYxMGZlYjdkMDhiODZlMjEwMGQ5Y2ZjMjM4MmE5ZGNiNDc4ZTY1YjhlZmYyMTg1NzkxMDA2NjZkN2FiYjRmODY4YTI3NWNmMjU4MDdhMDdlMmNkODMwMjU4NDc1OTdiMjhlMDMzODU4ZDEzNDNiMGVlNDEzZjVjNjVmYzc3OWQ5YjhkMTNlZTI3MjA3YWM4ZTRjZDE0YmE5MWE4NWE3ZTA1ZTFkY2M2ZGQyYWYzNzg4YWVhZWZiMDgyYmVjZmZjYWIwNjg1MGE2OWJjYjFlNzNhMzkyOTI0NjEyODA3N2ZlZmM2MGFmMzg1MzBiZmNkMWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LraSXRydHDBBvyZ3Nzi1t2GoYfpQQkcdHeSWimVH5Fekhi3KdBlLWZkmBJaPSyZ_uQnH8RADkAL3OUa8R_tfAw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210222_112512_89_2408_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.824Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImFaSEFJSVBBN3pMeW12SGR2RzQ5eVRGL1o3VHd3THc2R0ljeUFlN3UyOUo0UGY1Q3JLZzlZNFIreDRObVJyU1RhYVd3UnlBT2svRWZCdjZudjFMY2tRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyN18xMTAwMDFfMzVfMjQ5OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzlhZWQ2ZjIxMmFmN2FlZTM5MGYyMzQ2YzMyNjdmMWEwOTZlNDcxNWRhZTBkYmQ2MWI5ODgzNjkxMDdkZjQwZDJiYjZkZjI5ZmMwNzlkZmI0NDllYWY4ZDIxODBjZDIxMDhkNjcxNTVjM2E5ZWU3MGVlYTM1NTBhYjNhOWZmMGU4OTViNTA0OWFkYzc5ZWM1ODYzNTllN2VkZTcxZWE5NTAxMWMzYjFhNzNjYTM4NTE2NzViZGYzZDk4MmE5NDJkMGUxOGMwOWMxODc5MzU1NTQ1ZjNmYWEyYjM1NTAzMjU1NDNjMWRjZTg5NmI1NzAwZmI0ZWM1MmZkMDc1OTZiMjFkZjRhOGFkZGMxMTdjNWJhNjVmY2NjMzYwNjdiZDI2OGE4MWE2NDYyZWRlNzMxOTJjYzdmZTk5NjY3ZmQ4OWZmYTk0YzUwMWMyMzI2ZjE0NzZhYWI3YzZmOWU2NTYxMzJmYjA5ZDRiODk3NzIzYWRlZmI0ZTM5M2ZmZWNmN2U0Zjg1MDM4OTY2MDllYjcwMzUwYWQ0NjgwNmM2NjdjYzZkMmE3N2QwMDU2N2NjNzQxNDM3ZDExMjU5ZDVjY2Q3ODQ2ZGFmMGVjZjYxYzA2MDdiY2YzOGU0YWFlZWI4NDdmZGM4NWU0MjNkYmZhMDk1NDNkMGFjOTUzMGZkMjA1M2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oB5KIvrcRsDsP0LVWedeLndyWjLAzcdNKoapkm2jzcHj_YuNGIh4cFEe49R700dHMfLxwVnKvzv6d0AUK-gpGw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220827_110001_35_2499_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.827Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Imh2a0djc2g0V2NkdzFobUI1by8vWW5tUVQvM3lmbVlvck1EVFBFcEQxR1FEd245SGpYbnMxQ0lKcEhrOWZBSnJlRlp5NmF0b1BPZkRJSjlxV3hIbG5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyN18xMTAwMDFfMzVfMjQ5OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjJjODBkZWU4ZmEzMjhkZTYxOGUyZTBiOTc2NjIwNGQyNDU4NjZhYTk2NDNmYzI0OGVlZjZiNjI1YzQ5ODczM2MwODBmNmE3MWM0YjcxYTE1OGZjMjkzYmQzZTI2OWQ5ZTY3ZTBiNzk5YWM3MjU5Y2E1MDQyYzIxMjlhNjAwZTM5ZDJlZWZkMzM0ZGFlMzhkY2M3ZDgwMGExOTU4MTcyM2UwYzlhNDEyYmQzNTU5NDY4NTdkNjA5Zjg0OGZkMmVmMmZjYWQxZGVhNGVmZTk0Yjc2YzA0ODQ3ZDE1ZjhiYjM4MWY2YzA0NDJiZTgyOTA1NDI3YzJhZGEzMjE5ZjIxOGU2OWQ1Njk4NDNjNTIwNDZmMDU3MjI4MzgwNDhiMGIxNjYxMjQzNWM0ZTIwMTRiZjE1NDI4ZmEwNGM3OTM3NjU3NDI2ZjMwZGIwZTgyNDYxYjljNjIwOGUyMTM1MWE2NTQ5MDY0ZGFmNWM1ZjM4NmMzNDJiOGU4OTI1NDkyY2YzMzE5MGU3NTc0MDcwOWVlZDA2NjMwOTcxNWNiZWRlNWU0NzZmN2RhNDVlNWU3ZmFkMjZhNTdkYjg2ZDk2YjNmZmU4MWNkYmI2YTc4M2Q0YzFiNzE0YmEyMjBlNTNhOGUwMjc3MTM0MDg5ODg3MTlkZjNiYTZmNmM5OWEyOTBjMzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4KflE3jWlUtq-3a_jpHOK1avFxuW5kT23JXvNgYMTq0-WgnOYuuK0uihPKfTI2fNpB271R_Q__Ok7lqjEFa7Yw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220827_110001_35_2499_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.829Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJLTEpndWUzbFBCcDhpYlhGM2FObm9Uc1AxRUp6UUwzcW5KZGZablBEUkhiWDZOSXozTWNHRStZdjlRK2svRm0zTmwvbkxYdU00a0tMbEUvUWNhRG1RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyN18xMTAwMDFfMzVfMjQ5OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzcxNTYzZTAwYzExMDA4NjQ0MjViY2YwNjYwMzhjNmYyOGE0YThhNmVjZGIzZjJhNjcxNjRmOGFlMWNiYWZhNzgyNGY0ZDdlZGFiZDJlN2Q4NzUzOTc1NmNhMDU2MDc1ZjJjNDI0ZjcwMWI5MWNkMzA0OWM4ZWMxZjg0OWRlNjczNDFlNDk5MjNmNzY4YTA2ZmEzMzYxMGEyNzgyODAxMWRkYTM0NTZlOTExM2JiYTZkZmIwM2M5NTI5NzhkODUyMTg0ODIxZmUxMmY1ZDA1OGY4ZDdiNTQ4ZTUzYmYwNGE5OWI5MzE2OTIyMzQ4ZWNiZTE5ODU5OGIxYjA2Mjg0YmU3ODc5ODgyOGVmOTFjN2VlMzE3NjE5YjFhMjFmZWU2ZjM1YjVkMTNkNTM0YjliNTFhNDFlMjUzODZlODNhY2ZjN2QyYzI2MzMyODk3MzllNGY1NDBmODJmZDU2ODk3NzllZjk0MzY3OGJlMzEzOGNkYWJjMGJiODQwNjRlYTMxYjQyMmM0ZjdhMDhmMjFjYmQzMzE1ODk2YWViMmI3YWVlMDFhOGVlZmQwM2E2NmUwZDNiMzExZTQzZTI3NWQ0NDViY2UwOGJjZjYwNTdiZjE1NzRjZTZiYjIxODhmNjUwYzFlM2U3ZjhlOWFkMDI2NTEzYmM0Y2IyZjhiMzE3ZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aHJ0vfL94VOZ5GxEBh0fbu8Olp6uKMi8fj0w0K3xFQNdSHTdNtMoRhPpQvannPfTmzvfX_BEG_VwaM1MveMzFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220827_110001_35_2499_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.832Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImpERGpCa3ZYYlZYcmZoSXp1VFZrOVM0Z2F2RlZiNVFEbGU3MFlCNll5aTNWbWhtRUllNHFWa1JpR2dqZ2dBNndtMHVubFJ0U2FOWTJGV3M5dVpLSlpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyN18xMTAwMDFfMzVfMjQ5OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWZhNjJjMGZjZTY0N2RhYzcxZDNkN2Y3MzgzYzM0NTE2NzMzZDc5OWY4MDZkOGZjODhmNDA0OWE0NmNkMTg3ZDRmZWE4YTRmNzZlYmFlNzUyYTE5M2U0Mjk4ODhjN2RhZDIwYzliZGYzYTM2ZDg3MzFlZGJjYTVkOTA0NjY2NmRjYzEzMDUxMjFiNTU2NTZlOGZiMDNkZGQ0MTVhZDQ5MzlhMjJiMjMzMGRiODY5ODM0ZGNjZWY1ZWEzZjY0YTg1ZjdmZDk3YTYzYWM0MzQ5MTFmMWM5NGMwMTA3ODFjYjFlNzMzNGY4NmQ1NTk5N2NlOTMzMjBmODg2ZDQ4YzQ4NzVjZDQwNGYxMmVhYWYzMDdkYThkODBmZDU1NjBjOTY3MjRmMmNlNjFkM2I0NDE3OWJhZWU5YWU3NzExMjZjYWQ1YmJiNDZkN2Q4YjRmZTIxMzRiNzY5ZTJlN2UwNTZjNjU5ZWFhMTUyY2JlNTY2MzBiZThlODA3MGQ4ZDU3M2MwNTVmNWFjMzgyOTYyOWQ0ODkyOGNmMTdiODY2ZTEwOGZhZWE5NGQ3NGZjYTNmYmNkZDAzNWU3ZmZhNWRiYTkzNjJkYTExMmZhYzQxOWE1Yjk4OWM1Mzg3ZTY0YTQ4MmE0YTAyMmNhZGExNjM4NmU0Y2ZjNDY2OWFkNDQ4ZTQ5OGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RZ8ahOxniRg5awW7E1N1sgWDVB5KjF42NU3yD3nzuVSw8xpklglwMRPi1c3pbqBADSIkrk7LhzRSDyYzzDjihg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220827_110001_35_2499_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.841Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImNHZGVYSzIrOGUzOXcvVnZYTUdFbjR2ZHkxbC8wa1JFVTZJWmxOQ0hJY3lONnNrS0NFSzVhTXZPeTl2cHR6ZmU0RjBTWFRVUHlUR3IvVW95SEZLSEpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDQwN18xMDM1NDZfMzFfMjI3NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmY2NTBkMDAxMzRiN2FjMDYxYTNkMjVjNDg2NmM5NThiNTRkYjZiYmM5NGU0MTgxMzMwMzAxNDY3M2E4YzQ2MDMyMDFlNDkyMjg0ODRiOGJlY2QzNzc5OThiOWQ5NzZlYzM5ZmY0MWMyYjdmNTU4OTkwNzcxMjIwYjZhZTJmZDliNTkyMjI2NmM2MTIxZWExNDdjMWJiMTliOTY5OWU5MGY1OTE1NTZkZjBmNTg4YzcxMTAxYjRlNWFmZThlNmNmYmRjM2VhYjYyOGIwMTg5M2JlM2QzZDk4NjAwNWU0YWYxZmYxM2FlNWQzN2FhMzI3NDgwMjFhNzFjYzJlOTZhOTNiNTAyZmU3ZWFiNzNlMGVlZTNhNDI1MTMzZDhlOTM5Nzc3NjRmYWRkMDA1MmE5ZTg2MjA1NDEzNTY4NWYwYTdlMWQzNDA5ZWZmYjY0NmE4MTk2MGRjNzU1OWIyZWE1Y2VhYjVhZWU1ZjlhMWZhN2EyNWRlZWM0ZDkyZWEwNjJmODhmZmY2ZDg2ZTVjNjk4ODBhZjc2NWRlZmMyYzBiYWRiZDVjMDQ1MjU4ZjZiODMyYTdkOTRlYTZlMzI3ZmQ5NTJmNzQ2ZDQ2YTAzYjc2MjM1N2I4MmUyNmU4Yzk2ZmZjZDQ2YjIxMWNhMjA5NWFjODkxOWMzNjFiNTRjNmJhYmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RC5_bg8cgiDOdCWibbSwyxrOZRfkV2xbxbvrolwvX048NQbz__KxYTRxHsfSnpxVJ-fDmlGu4xhVbxvCo-3Gdw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200407_103546_31_2275_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.846Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImlvSmRiTy94VDhRaS84ZGV0SisyU1k0NTR4SXBncXBCcDFsR3VWUGVaTWlOem4rMlUxWXdxZjRWVGRGWHhtVERQem53ckZqdTJOU095UnVXeXY1dmdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDQwN18xMDM1NDZfMzFfMjI3NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTg0MWJmOTNkN2ViODNiNGJhNmE5ZWU4OGYxZmVjZDdlYzNiOTczMzEzMWU5NWYwZDU3MDkwZDMyZTg4NmM1NjRiNDBhZjgyZTc2ZmQ5OWU0YzIyYzYzMTk4YTAwZmE2MGJmOWUyYTlmZTZmNWFkNmZlMWI5Y2QzZDBmM2ZkNTJkMmU3NjM5MzZiMDNmYjNjYzExODk3NDYwZjk1YTdiYTA1NzMzM2VhNDljZDg0MTljYmNkZjUwNTExOGU0YTYzNDBjNDM5OGRjMWJkMDViODBiNWE3MmU1NjNmZGU0NTk5ZDQwZDZiMzc2MjMzNDg3ZGQxNTUzMDI2NmNkNGQyZDZmOWRlNDJkZDY1MjY3NTc0MDI4OGNjYzhmZjg2YzQ0YzI2MjAyMDY3YjIwNjkwNTFkYzdlNmNjMTVkY2FlZDgzNmQwNmVlMDBlYmU0ZjljMGQzNjExNDM5MzM4ODJjZjNiYjg5M2I5ODU5NmRhNmFhYmMwNWU1YzdjYWE1NDU4OGRlN2JkMTg1NTRkODM5ODM1MGRlNDg2N2QyYWU1ZWVmOWJlMGE3MTk2ZmJkNGNkMDFhYTBhYzY4ZmY0YWJhOTQ4NjE5ZDFhZGM1NzNiMWRiYTg0M2M2OTc5YTBjMDE4YTQ2NTM4OTI1OTFmYzM5OWM3OGQ5NGNlMmQwMjgyYTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Rw1eUsASNnCjT7TPPSi8nATrwOV_io-W4E5uwnWw-gAu7yDRmyIc8ATB4hHKseBgXxSJH-NnMCKSshHc6x46Mg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200407_103546_31_2275_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.849Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IldPc1V6aytUcXY0eDVaY3AyS0ZqTVl5U1NuMmdWcVpzS3gwNWVMSHljK290TmhuOE05dEcvcjFUQmJlVWlHVHIrcHRuNzU0RTZhTlB5b2U0WHhmamRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDQwN18xMDM1NDZfMzFfMjI3NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWI3YjEzNTk5OGMzNDk2MDZlZmZhZGY3OTY1MGQxNjI2YjNmNGQ0ZjFlMjhhYjhlNzBkNDNkMWY2YTYwNTUxZThhMzdhNzJhNzFkMTI3NDU0NjJkNWFiOGQzMDc0NTczNjI3MmJlNGQ3MmFjYWUzNGRjYjU0NmU3NTYyNWE0NzhkNjg4ZWUwYWZlNDI4NWU3NmZlMjAwNzY3NmNjMTU0MDAwNTU1NGZiNzRlMjEyZmIwYzM5ZjliMmFmMjQwOWY4YjU2NjUzNmE4MzNhOTI4NDNhMDViN2Y3YTdiNTViNzczMDFjNjhmNDgwNWExNzU4M2YyMGY1MmMzNjQ2YThiZTE3MmU2MDVkMjAzZTg2NWJmZTA2NDRiYjY2ODFkNTc0YTMzMWM5N2Q0Zjk5YmUyZTVlODYzMTA0ZjFkOTQxYzkyNTU1MjViODdiYWRmMTgyNmNlNjYxYmU0NWViMDEzMzUyMTYyOWYwNTVjNjQ2ZDQzYzRiNDY4MmEyMmNiMWQwYmU0ODRjNDljNTY4MjJjYTNiMDE0Yjc0MDgwMjhhOGRlNWJiYmExN2FlZDdlZjc0YTI0MTZiYTEyMWZhZWUwNWU0Yjc2MDkwYjdjMjQ2YTkxOTRjOTYxZGI1YjVkNDkxYWU1NzRiMzhjOWI4OTI1ZDU4ODZlMDBkNDk4MDM5Y2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ohDWdUwpUgeEGApL2BV2ekvBPnPQUnZ5VQKkHS0pY00jL2Kje0ZNNgKYDcnWjqE71rNHUJiGdeeG5rZ1YJVsnw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200407_103546_31_2275_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.853Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InlZdnhHYXBzdSthRXBaR0prdlRDUG1paUxYY3pVTDZqV3M3UmVBa0FpMmVRSHhmOFpTVWVHZFpia0dIem5BTFc1dnNDRTlYbjFTeGZ6YkhtL3FCY0J3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDQwN18xMDM1NDZfMzFfMjI3NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmM4ZDI1NmE2ZGUxMzQwZDcxYTYyYWIwMzJhMmFjYjI1NmFiYmU5Y2EzNzg5YmMyNzU2ZWEzYzMxOGFmMzA3YTkyOGEzNTkzOWJiOTE0MTNjNTgzMTJmZTJhOTI1N2FmYzdkZGU1NzgzYjk3YjIzOGM5YzVlNmMyMWZjODM3MGZhMmVhZjBlZGYzNjg0ZGFhMGY3NjY2ODMwZTJhMDdiOWUxZDhiNmYzNDhiNzhiOTg4NzljMDk0ZTg4YTQxYjJmYTJlMmU3NjRiZmY1YjllY2QxMmNhNDk5NjI2ZWQwZGFiMzY3ODJlYTc2ZmZlYWQyYjE2Yjk4NWVjZDllMTkwOTNhNjdkYjM0NDRjMmJhMmJlYjYxMGJlMWQwMTZkYTVhOTQ0ZDQwZmI2YTkyNWVhNDE4YjlmZWE0NWI4ZDNlZWQ5MWY2NTk5ZTI2MWE3NGM0MWMyOWU4NDIwOWQ4M2I5MGUwOTEzM2EyZWIyODgxMWRlOTgyODQ5YTM5YTk1NDNmMjY1NGM1NDcwOTY1M2RmZWJhMTBkNzFkZjUxZGYwZWNlNTgxZDUzMWIzNDM4YzVkMDJmYzg3MzIxZjI4YzMyNTAxNmIzZTc2YzM5MDA4MjA2MmM3OWJiZmRhNWIxZmE1ZTYyODg4N2U0YWEwNGEzZDkxZTE0YjJjNWJlMmY4NGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qD-XCHUNm1maeo3wSyq4v6n59oJTxRkO_YSuJsg3uErby1OG-KTVOI1QNMU2AEXavngA1OG-wnjNE5EcC1JiRQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200407_103546_31_2275_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.858Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InBLM3liU1hCWk9MaWEvR2VXNU02SWZKZnpTaGhCQ1FMNWFpRW0rUWh3YUFMc29vNGFySVRoL09GcjFSMktZM2VtcWIzNmJjbHZpNjJsT3lpZHJuYkVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNF8xMTIyMTRfMjZfMjQwMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTYzMjE5NDRhMjBjM2NkYzFhYTQ2NTI3YTNkZWQwZDc5MzBlN2QzYWRmNzQ4YzkwZDM1N2UwNDA3MTE0NWZjOTdmYmNhNTlkMmVjN2JhMTQ2NGYxYzkwNTcxNzFmM2FkNmJiYWU5NGJkYThhOGRhYTI2YzEzODM4N2I2MzExMzQwODZlZmMzOTQ1MDVjNDc2NTU3ZjgxMzRlMjNhZTQ1NjZjNDI1NGI3MDQxNTAyY2Y5ZDgzZThkOGJkOTZkMWJiNjU4NWVmMjQ4YzAxZWZkOTA0MmM5YTBmZDk5MWM0MzdjZmNmZTI3NThjMjFiNjMxNGIxN2U1YWI1MmNmNGFmOTBlZmI0MzNlMmEwMjVhNWU3MGQ1YzU5YWUyMTQ4ZGY1ZmM1ZjgxZWNmOTkwODE1ZjhjNGNhZjBiMjU4NTYwMDdjZTczZjQzM2YyYTgxNjFhNjg4OGI1NzBhNWZhNmQ3MjkyMTA3MzhmMzk2ZWE3MzlhZDJiMWI2MTBlZDkzMzhmZjhiNzRiMGJmM2E0Nzg2ZjNkZGI1Y2NjYWE4ZTkyZDMyMTFiZDYxMzdlNjMwZGVjOGNiNjRhYjA2MDliZDJlNmEwMWRmNjc5NzRlZTMxZDgxZjcwYmYyOGE2ZGY0ZWFiZGFhZTA2MzIzNzA1YzJkMmQxMWViOGRjZjA1OGUyMTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OasZTpa8JReiZ2Grbz7yVxDOWEdRLWTCWB1XMHRmpj1V0dWtLjkSMBXY1f5AVdTBoIATm2OVUg-4VzXv6HmJRA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230614_112214_26_2402_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.863Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im9PZUZza1FKRERVdVM4N3FNK1hFa21rSFdPb3U1WldNaVV6ZkdiWXBFWk1aUjF1aGFQanh0RlNhS2Zoci9saDA4VUhjSktTbjRKcGFtQW5DejF2b2FBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNF8xMTIyMTRfMjZfMjQwMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTBkNmZlNTc0ZjVjMThmM2RhNTE3ZDNkMjgxNjkxNDVjZTJiZDU0Mjk3NTQxMWM0NTEyYmYyMjgyZjRhMDJjMzc5MDViNTk3NzBjNWNlYjVjNjQ4NjY2MWE1NTY3N2I1OTU4YzBmNjVjYTEyZGQxNGUzMTA1MTZjYjZkNGMwNDVhNjRhYmEwMmI3Y2I2YTgzYjE2ODkwYjQxMTdiZjQ3YTMxMzdiNzdmODM5YjZkMDA4OTQxOWRmZmYzZDkzYTQ4YjA5NTVhOWYwZTBiNzIxYWM2ZDM2MTRjMjQ3NjAwNzc0MzZhNWJhOWVjNGRlNDhhZjhkMzQ3MGVmZWFkYzE2ZTAyNWJmODQwZTNkNWU2YTVkZWMwZTQxNzRlNGZjMzJkNjQ4MTdlY2QzODdiM2FmMjhlNzEyMjBiM2IxOTM5MzgyZjgyODAwZWMwNWM0MDBjMjQ1YzQyMDI0ODk0MDcxYThkNDUzZWJiNjAwMGU0MDUwOTk3ODUyNTliYjZjMTc4OGIyM2UzOGJkOWM0MjYxYjQ2YTc5ZGMzOTFiZWM1Y2NiMjQ4MmVjYTcyNGJjY2E0MGUxMDI3ZjAyNTM4NTU0YTlhZjEyOTQ1MzZlM2M5YjUwZTA1YjFkYjE0YmQyNzU2NzQ4ZWYwZDI1YzA4NzFiMzFjNDA5YzcwMzM4NzVjODFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.y8g_muayYN6ZAc2y8TGuGerm6rznUkQ7NUs9DqKc3sSSNqD5fNTkt62yGgABU5vSkuVJP5NjtSIOAzWwCsam2A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230614_112214_26_2402_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.866Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InIwKzhyN2k2dXFpN3RnNHY0ZzYwbFBoVFN4QXQzOHNYWWp2a0N3ZWJZMmZubkVBRVU2RVhJZU9rZGFsazVvWTVnU1AvR0JScFhsNmozcVlPQXIya2RBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNF8xMTIyMTRfMjZfMjQwMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDJhYmEyZDBhZWRlMTYxNzFlMDU0M2FiNzJhZGRiMjk1MzJmZDNlODk4ZWMyZDdmZGRlZTg5ZDA1ZDc2ZjRjZTFjOWM1OGU4YjA2MmExMjJlZTViN2YxYzY5NGM1MjRhYjhjZjk3NzBmOTg0ODgwMTMwMDAyMjM4OThmZDdmZmVhZjczNjc2YzU4ZmY0MzY5ZjA2ZjMwYzE5NWFkNzkzOThhNzM5YjdiNTYyOGFiNGU5NDQyZjEwZDhjYTU0M2Q2ZTUxODJjMzE4MWJiZGU5MWZjNzUzZjFmYWUyMjU3YWU5MmU5MjRiZTc3NTk5MzY4Njg4MjQwZGI4OTZjNGUzMGQ1MDJmNGFjMDkwMjUwNDMwZjcyMTY1ODgwOTNiZDVkY2Q2ZTc5ZDQ0N2NhNTVlMDg2Y2YzNzY3MDI4MjM1MmQxNTRkZWVlMDhlZmY0YWFlYzhkNDRiMzNkMTNlYmU0ZjFkYjVmZTZmZjU0NWY2MzE0NjE3M2QzMzIzNzhiNTM0NzIyOGM3MWRkZGVhMzhjMjZmYzRlNTM1ZWIwMzZhOTEyNTY2ZTlmZjNhNDVjMTM2ZGI4YjNmNjZlZTY2NDMwNTA1NmUxYzNiYTJjNTIxODhjMjAzYWM0ZTE4MmZlYjk2NDUxMzRmZTQ5NTIwNmYxYjBkY2ZiYTJjNmRmM2U3ZDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._Iu0xQS2S2JwQeZEvfdrl4kPL6tf_emyAxHZF_iTE7tn1z1jJhMowXascLT1QbjgG9yValLL_TqWYDFO-qoJtg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230614_112214_26_2402_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.869Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkNUZkFRVkxGWTBCcENkQ05tRnE5ZDU1d3o5TThlaG9sS3B5NWY3aVE3cVU1Z2lpdTVETXRBZWVwajVPVVp2clB5eGduc1R4NmhERmFpMnczYmE3aURnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNF8xMTIyMTRfMjZfMjQwMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzQxMzQyODE4OTcwM2FkMzA2MTcxNDk5NTg3Zjg3OTg0N2ZiZjg2ZDhiYjhmMDEzZDViMTczMzgwOTUwYTZlYzZkOTE3YTRmYTc1ZmM4ZmIxZjBmMWYzNTUzNTZjMGQxOGJhNmJlOWZjZGI2NzY5YzIzZjJhNTBmOGRlNWZlN2QwYzNhYzU2M2UyMmY1MDY2MzdjOTY0ZGYzOWUyN2MxYmIzNjc0ZTc1MGFhNGM0YTcxZGQ5NDUwYTY5YTZlYThhOGMxY2UzYjg0MjY1YzUyY2Y5NTExYmQxYmJlOTA4YTI1ZTE5MTgyMTAyNDNkMWQ1ZTY3MzdhNDM4OWMxMDcxNDk2N2IxMmQzM2U1MDA4MmY0ZDUxY2FjM2U4N2IwZTIwZWJjMTg0NmQ0MmQzNDQyY2FkMmYwZjllNzQ1YzIyOWVkZDNlOGNhY2FlNjE0NTFhZGFjMjk0YzE1YzUwMWMwNmQ3MWMzMWUyZTcxNWYyOTc2MGM3MWNlNmY4MjY1ZDVkMzI0ZDdmNDFjZTc5MTc2NmZlNTk0YjU1MTgwODgzNWFlNzRmM2E4NzEzZWUwZmZhMWUyMzIyN2VlM2VjYmE4YTVmYjljYTliMTVhYjBiOTM5MzhhOGIwNDY1ODg0ZDUwYjg4ZTAzMTU1MzFmYTVhZmRiZWJlMjU1YWVmOTM1MzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lVDe78MWV6zbdepbcJ9RfLHFbrcfDDq2F-lEa3ZsQlhDIYtFkiz84HcvJgTRm8puSk-SRa8nGhfnopTSlxlKEQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230614_112214_26_2402_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.872Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IisrUHZPbnZVdnVFSDFJeE40V0xCQUttandJd3Z1QUszSTBFWFZ2cVhQMS9EbWtmM0hWenYzN1Z0US9xU1FQRXltM2JoaDBnRS9SVkk1Vm1UNWRINWJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDQ1MzNfNDJfMjI1YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTdmNjAxZDBhYTdhY2M3ZGMyNjhhOTIyYzk5ZGFlODAzMDBhZGZmY2MyYzkzMDk5ZWI3NzIxN2Q4N2Y4NWExZGEwNjM1YWFiNTY4MDZmMGRhOTMwZDU2YjlmMGI2Mzc0Njg0ZTMwNTAyYTEyOTg4YTUwODliYTY3ZmMwNjUxMzYzNTVmYmE4ZDIzNTEwNTdlODA3Y2I1ZmUzOGJkOTNiYjgwNmIxMWNjOTQ1OThjNDg4NmFkYTE3OGFjYWFhYTdhMTc0NDJjNGViODk3YTExN2FmOThmZTY2ZWVlMGE2ZjdiZmJmMTRmYjAyOWQxY2Q2YmY3YTlmNzc4MjNkMmY1OWFmNjQ0YjNlNzU1YTA0MmNjNTIxOTAwMWY2MmYxMDcyMjkwNmEwYWY3ZWFlNzlhYzUyY2M1MGU0YWYwOTczNzhiOWY4NGFhNGM0MmUyYzhmOTNhYjE2ZTUyMjk0YjJlZDFkODNlMmZhMmUxN2VjODhlYjYzODJmMTFhMGExNzdlOTk0N2Q0YzBjZDg3MmYzYTM1NDI0MTUyZWUwZTk0ZDY2OTFjNTVhYjg0YzdjNWUyNDI4MGI0N2Y2ZmRkYWUxMTg1MzQ0ODRkMmUzYTU2NmYwZTJlMTUyMzNmNzkzYWI0MWM3NjBkNGY1Mzg3YmYwYTgzNTg1YzdkNDZkZGNlYTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._3DRROjEQsfOYFe8WEpLAVkaBEDKXRedtVm0imcgZWiJ6PR61X6ScTwcufvIcXhYGpylz4nBEYNQ7-iF3UfRFg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_104533_42_225a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.876Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjArcUVpV211RG5SNWJJZ1J0K1dWdnpXb01oWm1IRjBLVVJRUHV4Ykh2bm9pdFVsdDc3ditGYTFGeGRaTVMrZzhpNHVvZ0ZsNmk1N0JFS2UwcVMvQUtnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDQ1MzNfNDJfMjI1YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTg1MTRmMzhiOWEwZWE5MTUxMjk2MzJiNzg4NWNlNGU2NmJmM2U5ZDE1NmJhNjZmZjY5NzcwNWU5Njg1MGQ4ODY0MDgwMjNjMzU0YjBhZGZmM2NhOWNkMDRhOTFiMTc1YjM5NmI5MjE0YjYxYjQ1NjM3NmRmZjRlMDQzZjIxNGVjN2UyOTQ5NTNmNzI2OWIzNjljN2Q0OThhNzljMGM5Mjg2MGYwNzYzMjkxN2JiMzRmODg4MTdjZmUwNWFiYjBkYjRhMGExYjdmODRmMmY1MDZlYThlOTM5MWYwMmJkODJiNjY1MjcxODBiNDE3OTAzNDA1NzI5ZmZkMjJmYTRlM2U2YWE1NWQ0ZGQxMTZiYTU2MjVkMDcwOTdkZjM2ZmNhYjRlZTUwMzk3YzdkOTg3MmI1ZjA0NjI2ZTY2YWZhZmZmNzRmNmJiY2Q3YjA0N2FlNWMwOWIxY2JkNjdjMDc3NjAyNDgzOWM4MDVlZGI2YmY4NTA1ZjY2N2YxOWE2NTcxYjQ3YWYwMWE1MGFiNjJhYjdhZGJlMzU2OTU1YmQ2YjRkZWE4N2ExYzM5MzFlNDljNDYyZmM3NTMxYzQxMGRjM2Y3ZGQ3ZTg5YzNhZDhkODk2ODA5YWZlNGNjMjI4NmUyYjQxZDUxN2NiN2YzYjE3NjFhN2Q4MjAzMWNlMzBlODNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._yYRl-Hal9DLoWfDX4JNc1jC-9tH5sqF7NG1iUpqa3X3PCt7YqiQ-CpEIM0m5d0kS8bJ3oo1tQ5y66FhpXJSmQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_104533_42_225a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.881Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImkxRTMzdm83OXVIVkc0bUJZdEpvc05ZZ0g0aFBaRGtGWmVwYlRXbExOTlNOVThsaEVMZWJIZkQ4RXVGS29MTmNMUlJoYjEvTkN6U0s5ak5yMnNsQ3h3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDQ1MzNfNDJfMjI1YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmZiNmJhMWUzMzFmZTU5NDViZjM1ZWQ1ZDRjYTM0MDYxYjMwZjZmNjQ1MzU4OTJlODU5NWJjMDU2NjdmM2IwZDJmN2M4NWZjN2Q2YTMyNDlmZmQ2YmZlZTY0ZTNhZWY1ODA5ZmI5N2E3ZDdiMTZiY2Y5ODllN2ViMzE5YmE5ZjFhMDAzYmU0ZWE0NDIwZTRkNmJiY2JiMTg5NTU0NjA1ZWE1YTlkMWQyNjNjYzM3OTE3MTIxYmFkM2E0YTE4YjU3NzBiYzVmNzA0MmZhODEzOGFlNzdmN2NjMDI0MjVjZjlkZGI1MGFiYzViZDUwMWNhMDNjNzkxMTI0ZDZlZDYwMzVmNjQzYzg1NGEzZTAxOTcwMGI3NTQzZGM0OTAyN2EzMjExNzBjMzZhZDA2NWQzYWJjMWI0MjI4MmVkNGEzMWJiYjFlMWRkZWUwZDBiYmEzY2I5ZGFlOTE3YWEzODAxYzE2YjMwYjU5MDFhZDFkYjkyYTUyNTU2ZGViNzk1MzJkOWExNWI3ZjlmZTFkMzM2ODY5ZTVkZDdjMGVmZTg2ZDRjYzA3OTlhNmRhNWI2NWEyODljOGViZDE4MTZiZDllZjdmN2I1MDQ5MWIyMjg0OWFkMDcyZWU5YjkyZWMxYWFlMmJjNjRjYjI0ZDk2M2ViMzc3MTVkMTk2NDllMjBlZmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mjiurZvV_vOS60wNlG3ODucQWxTI3no4Kplv0WfgScHqhzQBba0UfSTvv_MHealwG_Ho0q9-wK27VQXw_D3c_g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_104533_42_225a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.887Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkF3WGFhcjRKTTFEK3F2RDhieGNjd2ZMVHJDdWdqcndJSjdpUjJvZjI0bnJKZTdmejFFMGlTUnQ3NVJ2WVNJQTl0ZGtDZm9PRG9wSWNIbVpmMjM4Z1NnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDQ1MzNfNDJfMjI1YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDQ5MzkwNzJiNDRiMWVmNTE5N2UyN2UwNjc1M2RlMDkxOTdmZTBiNGNkYjQzNDQxZTdmMDE5ZjAwNzhhNjhiNDEwMjViMTYyYzY0OTJjNGQ2MTJiYjcxMmVlN2MyM2JhZGQyYWI5YzVlMWVhMDdiNGQwYTczNGY5NmE5NzVjMmNkNzA1ZTAyNGZlYjFjNmE2ZmIxODJmNDQ3OWJiMjg2ZWUyZDI4YmY3NGZjMGQwNDY3MTI2MmI2Zjg5Mjk2YzU4MGJiZGQ0ODJkOWUzMzUyMTBhMTA4OTM4MDY1NTJkZDA0YjM0NGJiZDc5OWNiNDg3MDQwNzNjODg4M2VmMDBjMDc2MTA3MDFiM2VkYmNhNTUzNDZjOGQ3ZWNjNzBlY2Y3ZDQ4MTMwNzcxMzYzODc3NDY1YmM0NDM5NGYyNDMwMWQ3ZTBjOWFlYzUyMzEzOTI1NDJkNGJiMWUzOWE1NDgzNGQyMjA4ZDE0MjMxZjE4MDczNzkyYTU3MTk4NDA1YjNmNzY2MmJjMDllNTAxMWQwZjJmNzc0MGIzMDY1NDYxNTA3NjY2MmQ1Njc4OTFkOTBmMTkyNzFhMDE3M2U0ZDQwODk5OWE4MTBkMzQwODIxZTExNzRhMjlmNjJlOTFkYzVlMjM1YWJkYzg2NjBlMzY1YzljMmFkZjUyNDM0NjQwNTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QWJAM9ZRty1A1ivnSw24CDKAI3HA0rsnu-TIDCNtGUf8KTI5z3I_TUI-P3atq8GLa2ziSy374TTD4HAOI8s-PA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_104533_42_225a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.890Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkpUeTdwMWoxV2Y2Tld2cDA3cWhOZmhkRUlOdlBmaUNXMDI0VmZLTWwyNDRFTzFRc3JxNGtNVE1USnVCc1dCNnhBUEplWkVMS3VuM1VZSHJsNmJJTXdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NTFfODlfMjQ5Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTQ2NzFmZmMyMjA1MzIyNmY4MzExMWU3MDcxMTFiNTM5ZWEzM2ZmOTc0ZWM1ZjU2MzQyYWYzNzM4MDYyOGI1NmU0MjU2ZDU5Yzk5MWEwMDFjYmEzZTRiOTBkNjRlNzkzMmU2MmZkYmZmNjNhOGE2MzkyODIxNDUxZjYzZjA0Y2Q2OWUwNmJiYWQxMzZhZTFlNjgzYTI1M2U2MTMxMDgzMWEwZGI2YjEzNmJmYThkMGI3MjBmYTU2N2NlNGY0NjZkZDY1ZjA3NjkzMTI3NmZmMDBiMjk0MzFmNTFmYzI1YTNkMGYwNjQ5OGViNzI5MWRmZDFhNTUyOTk2M2NkNzExNTc5ZDY0NjgyZmZmMDU2ZDJmZDM1MzM2NDRmNGFjMzE1NGNjZGViMjNiZGIxNDQ1ODA0YmEyYWY2M2EzNGM4NTY0ZDI4OGUyYTg2YjU2NWNjZWMzNWVlYmM5NTkyOTM5MjQxMjlkZGYyMTdjOTYwNDNkZGQ1M2EzNWJiMGUzMzk5MmUwZGI5Y2VjMzY0ZDNjYTcyYzhmMmRlMTMxYmM2MWEwZWFiNmZhMjkxZjljZTc2MWQ0Zjc0YjhlN2QxNjlhNzE0NDAyOTk3NThjZDYxN2IxNGFmYzIwOGFhMDA0MDNkZjQ3ODNiMTEwYzA4OGJkN2RjNjNjZTMzNjZlOTAzZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZKlUCqUpJUaEQE0qU-UxwPAgWlsySUcYjA44RQvpGr3ge9-FS0NDpFTanYTCmwIpLZb3-n9i4jpGsSzvl8d_CQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110451_89_249c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.892Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImNuSjl6UFA2QXF1TURMSmo4VlhHSmx1dU9KZHUySW43ZjNOV1E2L2dTMTRkalByOGo1R2xBang4eVVpVzN0UWVaOWZSUWtac0xla2dMNmhJbnNVRkRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NTFfODlfMjQ5Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWZlNTg0MzhhZjQzY2FjZTdlMjQ0Y2I5NDc4ZmY3OWM0NDkxMmIzZTA1MWNhYWU1ZjgxY2YyN2VjYWFmOWNjZWRkMGY0MzMwMGM3MzhhNmQ5M2I0NmJmMWRkYzNmZWFiZmI3YmI5NjJmZWU0ZDRhYTNlMGM1ZjE1ZmUyOTA1N2FjNmFjODA4NTAyYzY3Mzk3Y2Q0Yjc4MWRlMzNmYjgxYzRlMzMwZDNhZDZjYThjZDhjZWQ0ZmVkNTJlMzBmMTMxNTU5M2Q4NWFmYzNjYzA4MGQ1ZGZhNWRjM2FlOWU4NGQ3MTQzZDBiNjk2YTIxOGFiNGI0N2Y5MTU5NzNiNjk5ZmE2NDI0NzlmZmMwYWU2NDNmMmViOTViN2Y0MDk0Y2ZkNTY1MDBmMWJiMzE2ODczN2QyNjhiZjBjNjBiZTllZTcyNzgwYTM1Y2M2YTRkZWI4MTQzNWI5ZTkyYmRlNGZjMjBjOTU2ZmJkM2Y2Yjc1OGMyODRiNTM5YjYzYjRlNDhhYzZmNzk1ZTc1ODc3OGYzNjkyMjkxMDFlYTYyMTczM2IxZGI4NjYwNzdjNjBjNGEzMGNhOWIyNmZmNjk1MDI5MmI2ZTIwZjFmNzc1MjYzNTFhOGMzY2Q2ZDc5ZTU3YTRmNGZlYjllMGRhMjU0MzJiOWNlYTNiZTQyYjYwNDk3ZWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hMRHHpR6SIFzQ9IJpuNf42r6jvfpkxReHxOyRQEM-gXjxIXoujmxVy1Ga8RRca3hIrgrcmU-PwDv3eaNkGXJXA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110451_89_249c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.895Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjNkdmltWUxDZDN6SWNuY0lBSkJrSTNhSmFJZW9oY01WamlQQks2SjZ0UTdtdXNvb1VlbXVCblFPNTVWc0c4WUNtazhUakJNNEFKY3E3MTBNUHBKNnBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NTFfODlfMjQ5Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWYyMGI1YzhiOTFlZjljOGI5MDA2ZjUzMDgzMDVmOWNmYTA0ZGUxYTE1Mzk0MGNhMTM4ZjFlNmEwMjZmOTZlYWZiMzQyMDdiOTdjMDhkNGUxNzc2ZWMxZDMyMThiYzQxYjcxOGUzMDdhZmIzMmZkY2I2N2JkYjk2ZTBkMDA4MDllOWVhZGYzNDNlMTk2YzkzOGFmZDkxMTI5YmVkMGFhZTZlYzgzMDZhMDllMWI1ZmU5YzZkNTVlYTdlNDQxMmZjYjdjNTBhZTU0NmZlOTU3M2MzMWJkNTgwOTJjM2M1YzhlNGQ5Mjg3Njc4ZmMxZjk1NDllZDMyMTVhMGNlNWZkOTE4NzJjMWI0ZjJlY2ZhMTI4NjQxNDRlOTNhZGVjMWY0ZjJlOTcyYzhhNDNjOTE4ZjEwOTA0NTBhMTQzYTUzMmJmMjA1MWE1ZDU1Y2JmN2E4MjYwYmRlNzA5ZTVkNmE4MzZlMDRiYzUyMzAyYjdhNWIwOGMzYmRkZTAzYWNjNjBlM2Y4MDFkZTUzMDkwYmE4ZjkwNjMyODk1NTg0OGNjOGE4ZmMzOWQxNGEwODdhMTNmODUwYTQ3NTQwZGY1ZGUzMDFjM2QwNTgxYzhiN2YwZDdjOGNlMjc1MDBlYmY1NWYzYjdmMjhkY2U4MzdiODJhZjQxNGY5Y2JlOGRhZjM3NWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Saz4oJ4mi2egz4R-UhhcVywbCCV5sSALdvLk5H_WVfdQA-VY-YBVeSyx_N62-MrODwfap57ZalmQnYjyEZtJQQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110451_89_249c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.898Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikd1NlhzOFlBaVlJYWlTbWpNbE5wVjF2OVRoUnBjejlOZ1BGK3RLaGprWmcrbzV0TWJiZE9xenFjSkJtcS8raGMwcWlpVWZSNWMxRjFXTEtkVkVEOEhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NTFfODlfMjQ5Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDk3ODYxZjI4YjYyNWNiOWMxNWViMmMxNTNiYTM2ZDg2YjgyZjUwMjA4MWUxZDE5NWE3ZjdmZmZhOWNjYzAyZWY3MzRhYTkyYjliMThmYTIxZTEyNzkxNzA3MWQ2NjBkZmU3OTNiM2FlZWY4MTkxMjQ0ZDQwZjE2NmJiYjViYzNmNzgyMTlhMTYzMDdhNzViMDZjZjMzYmI0YTRiY2UzZDAzY2RiYjc5YTBjZWYxNjlkNzQ1Yjc5ODBkNDY2ZTZlNWMyY2VhNTNmNTkxOWNmYmVmYzI0MzFmZDg5N2E3M2U1YmI0NzQ4NDBlMzdiNDY5ODFiNzdmMDg0ODhlYjA2YjY3NTRhYTg0N2MyMzc1Zjc5YjZkNTFmY2M1YTUwZDY2NTdkZWM5Njc2Y2JjMDYyMzc2MDBhZmE0MGU3NjMxZmU3YmUzOTY0ZmY1YmFmMjkxZDNlNDM1MjQ4N2FmOWQ1ZGZlNzE2NjAxN2QxMDMxNmQzMWE2MTBiOTAxODQwM2Y3OGExY2QxMWM3MzJiM2YxMGQ1MzMzNzA3NGI3NGJjOTE0YTdmMGM0MGU0YjRhMzExZDJiYzMyMDZiNjcxN2RmYzNlOGNjYjdkOTdjYmNmMzNkNWZlYjBkNmZkOWUzMTQ5NWNlNmNiMjExOWNmYzc1OTkzZDZkZjZiZDUwZGMyODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.y35SDxN43KvT1dRPqgGGE1STdqeAvg0dQfYgSaAtI__lHFRkTNzrvaYcMYx18lOhZ537Jq2cv6hOHV6yLIqirg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110451_89_249c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.901Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImlhV2ZldmFXU2xLY1VvNGNZMHNoUXh3bGJJZnBMRDE2MmNnSWp1L0craHZsdDdDckhPQWhNbWZjbUttWXlIa1NvZE1adUlDQlB6ZFgvN1N1ZnpWYjhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTdfMjdfMjQ4MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzQ1ZTZmYTIxMmU0Y2VkNTVmM2I1OTQ4NjZjMDMyZTY3NWRjZDFjNmQ1ODk3ZTJkZTgwMWNlYzI2ODkyYTkyZmNhMGMzMDA3OGY5YzJlMDdlZjllODYxZjEzZGNlNjgyMmViZjIzNjE4MDRiZWFlOWQxMjMwNjc5MWYzZWUzYzQzY2E1ZGI3MmZjMmUzYzhjNDFjZDc0ZjI5NmYwNGQ5YjBjNTExZTc2ZTk2NmI3NDZiMDYyNjNlOGQ4MDNhMmY2MGIyZjc0Yzg4YzY2YTJhZWE0YmE1MzM5ZGZhNmQ3MmYyYzU1NTY4MjZlMDRkZGJiZTRhM2U3ZTE4YjUyNzE2ZjkxZjYxYmZhZWYxNTQwYjhkMzVkZmZhOTAxNzVhMmE1ODRjMDA0MGVlMDg2ZDAzNTUxZmQ5NGJiZjQwYTY1ZDNjNDViYWFkM2MyNmQwZDNjOTkzNDdmMGQ0NDc4NmZkZDA4NmU3MDg0NWU1MmY0NzA3NTBhYzM5OGZjM2RhMjUxMzY4NWVmODIzNDU1OTQxMzM2YmVlNGZkZDMzNDhmMTZkNGMxMjIzOGYzNTc4Y2Q1NWQyZGFmMDJlMGM1NjBlZTU3NzJjOGZiMDk0NDA3YjU5Mjc1ZjdkMDBiM2RlNjJjMjhjMWJkZDk3OGQwMTIxNzI2YzVmODA3NzJmNGJjZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Do744vTEm_gxynJAWcuHVNukA1Q8Tm2YwyAdbrqCS-BW7rHtQl0iGAgWB2e2pxm7P-gaaepMOS31TEMw9Tetww", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110417_27_2481_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.904Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Iks5QUh4a3pqMmphaVlmQ2Vrd1JuVWxsdlQyVCsxWTVHSVRKOEY5UUp6bW96T0FYc3QzT0g0TnV6NlEzbXJCbVNCNTNBSm5sQXcvUk9Mdnp3aFJJNGlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTdfMjdfMjQ4MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTZhZjQxNTY5MmQwMzI2ODY5MDFmMjAxZWQyYWRlMTBiZmNiNjlhMWVjMDhjOGRlZWZhMGIyN2MyOWIxMDZhMzZhOWYxOWRiOGY1YTVhNmQ1MWY1MzgxYzQ2MjdkZDg1YmMyM2Y5NzdhZmJlZmY3N2YyM2Y1NGY1OTViZDg5YWY3OTVhNTc0MWQxNGZmMzE2ZDhhYmQ4NDFhMzkxOGU4MGIzMzIwYTNjMDg5YWI0M2EzOTQ0ZGMwMTdjMzU2NjUxMGYxOTEzZDgxYzQyZjY3YmU1MjIxMmFhMWNhMzhmZWZkNTdhZTZiZDc0MzEzOWJlZDliYmNhNjJhNGMzMzAwYzY1YThhMDg2NDkwMGJiZTRlNDNmMGQzZDZjN2M3MjNhNmU0ZmM3NzUwZGUyMTI3NTM2MWNmNTYyODFhYmM1YzllNzE1YmI4ZjAyMmE1Zjc4ZDMwN2FmZjdjMWQyYjc0YWU4NGFiN2Q0MTNmNzkzYzhlZDNiNDk3OGVmYjExMmZiZmQ2MDFmNzY1MGRjZWNhZWI0YzVjMmUzNjA1MTMzMzRhMjQ5MDY5MmZkNzA3NTZhZDk1MjZlMGM3NTA4MWE5ZGM3NTc3NTYyYmY2MmIwZDFkOTRmOWVlNDBmZmVkNmEwMmE4ZWFkMzFiZTYzMjNkOGQxN2IxNzc0YTU0YWRmYjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fyp-2XSF44mpSzjnibHS0CPsKGDVKgFvscRpekxfsBA-ayqYr5SGUXjYgpk1mjEE-ceilOFySyCypY5FG4aiBA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110417_27_2481_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.907Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlNnUjRKdHlzUFRqai85ODBCZHFoUmJKMVRVaHplRklNMGlkVjhiSGdqeXZZRGdaN3RCU0d5RStqQzFiMHFqbVlLb1c3dmU5SmZncXp2RkorTy9qK2pBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTdfMjdfMjQ4MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWE3YjI0N2NjNDc3NTY5YzAxMzFhMGUxYmYwMGYwMmE4MmYwYTM5MTI1OThiNjY1YmFhMzQ4YmI0YWJjYTQ2NmM0NjM3YjdjOTI1YmYwMjlkY2YzZDAyMDkxODk1MTgzZjIxNjg1MWY5YzczYWZhMzc5MDljMTE0OTY1ZWJkMDI3MDgyOGY3YmMzNzIxM2U1MGQ5MTQzNGI4MzYwMWEzYzhlZTc4MDYyMzc5ODVlNjMzMzM4YWJkYTNhYThjMTJhMjEyMDdhYTRiMjY5MzNjOTVlNjc0MDEzMWUyMzI3MzBjODdjMzc3NGZjMzk0OGIzYzJkMTZhMzhhYzczNzY2YzU3OGM4N2YxMjYxN2MwYzlkNWRhODQwNjU5NzdhY2E5NGExZTY0YmU0MDFjOWZlMmJhNjZkNWJkYmMzNmRiMGE0NzJmZDJlZDk2OWVkYmIzM2MxMTNmZmUwNTgwMmI5OWMyM2M2ZmMxNTA5MmI0MjcyNTY5Njg5ZTVhMGE5M2VjYzlhYzA1ODk0Mjk5ODhhNTRjMjQ0ZWQ2YzI1NWVlOWNhMjhkOTg3MjVjNjFiMWNkNzJmNDcwODVjM2E1YTc3ZTE4MjQ4MWQzY2RjOWRjMmVjZWJkYzRkNjI0MWJkM2FiZDQwN2JmMjA3YzRkMDQyN2Y3YTY4ZWE3OWY1NTUyYjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.p8ZpTLn1fECfQrxevphnIpw5QkaaqMB16eg8-h8L4BVFn7ybjqzm2OzdPPMdk_UT6BvepirCVp48d6VWfrF0hg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110417_27_2481_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.910Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkYraUlzSXdMQWxyQktldkxFWFF5a1g4OXJRSUdTNmRRRzY3LzE2UFZMWGFzZFRxQlpEOHpqbTg3Q1BORW5DSlNnRzc4OG1VYUsxaDE3NFlJNksrb3N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTdfMjdfMjQ4MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTJhZjE5ZjRiYzhjYmEwNGFlMTAxNTY0NjgwZjQ2MmJiYmU5MDE1NjliZWFjNWRhMDUxNmQwYjFmNmJjYzYyZWUzOTBjZWExYjk5YzI4OWQ3ZmYxMjIzNDRiYTg4ZDJkN2JjNDM2YmYyYWU1ZDkzNjZmYWUwMjJjMWY2NTJmMzExODBhZDAyODk3OWIyY2IxZDdiMjQ5ZGQ5NDhiODI4YjhmMDAwMWVlYmM3ZmU0YzllOTliODg4MzU5ZGQxNDVmYjcxNmMyZTVlZTI2MGMyYTNlYzA3ODAxODNjZjJmYTZlYjg1ZWQ3NDFlY2ViMmNiNjYyYmRmYmJhZjNmMjMxY2I4MmI3MjgxMTQxMjNjYTc4YTVmZGRkNDdjMTYxZTE5NGZlMDQ0ZTg4NGVlOTA3ZGVhZjE5Y2QwYmUxMTRhZTAyMWU0MzUxZDA2OTE0NTI2MDdhN2EyOTRkOTI4Mjc2ZjIwYjAzMGY1ZDhkOGYyNTgwYzEzNDdkZjRjZGEzYjc5ZTJlZDJjNWJjNDk3OTE4MGU1NzljZmZmMjhkZDcyMzVkOGNlOGY0NzA2MTA1ZDk1ZGNmYjBhNjVhN2Q2MTQ2NGMyYTQ5MWM1YzEyMGY0MTliOWQyNDdiOTI5YjU0MzlhZmI0Njc1ZDg2NjdjMTUwYmY2Njc0Y2FjNjJkMjJiMzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JZZ-Xf9UriW1EpJ6V-MAcNBemTJAqOLvJXhzo4FepE58nT9psBYcjYddLJ3ogDBxJhrbch7xo8V-QDR7MxzsTg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110417_27_2481_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.913Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InI5Q1gwaWZqcm5PUzh0d2VsY2ZnY3A1L253UGdOWFlTbWl1QU9kaWJQeDdrVmh1NTBoakxXNFFkdk0zbXBueTdZWVRLMmp3cWFabzJMV3U0dUdOdkxnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwOV8xMTA4MDZfODhfMjQ3MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjcwYjM0ZjdmMWU1ODg5Zjc5ZGQwY2E3ZDAzOTdmNDE5OTE4MzhjYzgwMGVlNGI0NGQ3ZjI3NzUyNmIzM2Y4NjQyYTE5M2E4YjRhZWFkOWJmNjMwNzYwNzBhZjFiMDU4MDJmNTZkN2U2ODM1YjVhZTI4YmYzYjRmYjQxMWRlMTMzNGY0YzA5NzBiZjY3OTVhODRjMWIyNDMxOWViMzMwNzY3NDI3NmFjODQ4YmM4MGM3ZDJhODEzMDc0OWYwODQzYWVmODlkZjUwNzdhNDNkMzM4YWQyMTQzN2I5ZWI4MGFkNzg0ZmE2YWY5NGVkYzcxNTJlYjY5NmQxMzIwYTFiMmUxNjJhN2JiMmU5M2QzNDI2YWYyMjMzYTZlMTA1MGZmNWI4YjA1NjUyNTI3MjQ5OTI4YWQ3MmJjNzdlYTY4YzhiMmRiYjA0ZDQzMWQwNzFhZmM1M2IxNDhiODc1MzViNDI5MmU0MmE2YWI3NzZhYzFiODZiY2JjNDkxNTY2MDlhNDE2YTMzY2UzYTVhNzM2ZTkwMGUxMjIyZjAzMzBhNzExZjU5MjA5ZWJkNTI5NjYwNDAxNzg0MjZkYzZkMzZmMjIxMTRiNThiZjE3MTc2NGRhM2E1MmQyYjcyM2Y4Zjc0M2E4ODAyNjUxZWE2NzUxNWM2NDY5ZDM0NWRiNGM3OTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bpfxfMypAkaEziW393hSLwdOvs8ITC8fq-fCAfzjmOsIIVgXVbcQ-pD3_WUBR6FIeg5tmKJwK-IzrEi53ocvPw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.916Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImtDcVYvaWllL09GNU9mZHNaakdlRDdxQTB4UW90b0RwMGJZVmNYZUI5RXp6N0MzcGpyWWU5WVpGNHAxemZLYlh0Wm9wZ0Fib1VJd3E3UDdSNmNRK2hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwOV8xMTA4MDZfODhfMjQ3MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzViM2JhMzYwM2I5NjdjNzQ0ZGM5YjFkMTZiMzg4NWM3ZDE5ZWQ4Yzc2ODU3MmViN2EzYmQ0NmMyNzU1MmFkYWI4NGFmZjc1YWFlMzE3NTNmMjJmMDU3NzkxYmI5YzZjYWVmNjI4NTYyMTQ0ZDI2MTYwZmU5YjkyODQyMTYxOTZhNWZkM2ZlMjYyODA2NTY0NzU0ZTJhYWQwM2QzZjUyNmVhZjE5N2VmNjcwMzIxNzcwOWM4NjJjM2Y3ZjU3NjZhOTZiMmQxMTE3MjgxZjU0NmQ1YjUwNDUyZTE5YmUzM2UwYTU4YTM4MDRlZjVjNmI3YmQ0NTgxMDM3NzRiYjA0MDI2ZTY0OTkyYjk3ODcyM2EzM2VhZDRkMTk1NzM0NTY5ODY1MjZmZTE1ZDlmNzdmMzRiZmJhNTM2NjllMzc2NWY5ZjRhZjY2YzEyMWI4OGNjMjRmMDU5ODcwYjEwZjQ4NGY2MGI1MGY2YWE0NjI5MDJhNDFhN2Y4OTNkZjY5Nzg1ZTlhOGQ1MTJhMDMxOWNlOGZhODRkNmFkNDgxOGU0YzdkNDYzODAzYjljMWUzYzFkMDY1MjMxZDJjMWExYWRlM2VmZDBiYjc1NmMyZTYyMjAyNzBmMjQyOWMyNDc2MzZmNmJjMzU5ZWNkOGEwYTlhY2Y0Y2FkMTI1NTg4ZGNiNDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6PoPTFYXi8rVmYegazCa_W50Tu_obL8UdadXqRBfQqlEFGPYNtcdl-8yIE0SLk-hngyoKQkPtrq1bhZNJLCsGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.918Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InM3ckpyZ3hXRmRnaE4zVUhFTkNNQ3VBQ1plYmRQVk1zdXdRSjJyYXFtcm1OVUNMNGhkOVhTQTVqaFZwTXRyaVpBOEkvVy8zSElzVCtUOTV1ZDJTYTF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwOV8xMTA4MDZfODhfMjQ3MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDA0ZTA1ZjJhZGE5ZGFhNGFhM2NiMmYwYTc5YWQzYjA0NGQ2ZDAxMzViZjJjN2RjMTkxOTNiNjhmZmQ1ZDExNTk5MmFiY2Y4MjU2ZGQ3NDljYTk2YzVlZjc0OWFmN2UyNTZkYjdmM2JmMWYxNTYyODFlMDE2ZjYwNDQyMGY1MDcwZTM1MmJjMDY5OWZiNmE0NWExZjAzYzNiNTgzODU2ZjYxNWJjMTRmNGE1YWQ0NmI2OTUzZWYwZGUzZGYzODdhZDc4YmZmYzg2MjY2YzM5ODE3YTQ0YTcwZjY2MWNhM2QwYzA0YTIxYWI3NDlhZTI0MDIwZjM4NTI4MTRhMzMyMmIzZDAzNmU5YWQ3MDgzNTM5ZmUyZDI5ZGI2ZTllMTUxNzViN2MzMWM3NzM4Y2JmYzlhOGQyYTMyNmY3YzNmODY0ZjdkOTFkZWMwMmZhYzM3ZmQxMGU1YzhkNjg4YmFlNGE5NjE5NDdkMzRmYWIyZTJjMmYxMDlmYzhhZDZiMGM4ZDRhNGI1MTZhZDJmMGUxNjk4ZjM4YzdlYTFhMmEzYjM3MDVjOGJjNTBmY2MwYmFiY2QzMjEzNjJkYWIwNTcwZjgxYjQyZmY4ODAyZDQ5NTE1ZTY5MDM5OWQ4OTJiZmY5ZWMyY2NiMzRkMzk0MTk2MjczNTYxZDEyNmRlNjFjMGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RvZpQjvwyRTU4uFOLW1nCJL0Js6ndiGB-BTfiFrIkN7T2glyQ7e23GyExXTDKCj_cxWm7G4uMNlpsyE4R7mtQQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.921Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlNwZFN5YUdHQ2xxbzd5NDFaKzZLWHA3SnNCeUZVcUtjcjVCZmtqU0dvcG9aakQ4TUd4Tk5aamg2bCt5ZnVhbDhNbDhwSXFjNC9DVmpNeDQyRUUyVHpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwOV8xMTA4MDZfODhfMjQ3MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmM1MzU1NDJkNjNjNGQ0MDI0NThiMDM5MjQ1ZGU1MDdlZmMyNGNiN2I5NzM4OThmNTJiZTQ1YjcwNDA2MTEyYjE5YWI4NzNlODE1OThmYzQ5ZWJkN2Q0YjMzZGRkZDQzYmM5NmMyNGYxZWFmNjU1NDJiMGQwYmYyZjk1ZjVkNTA4MGFmNDFlNzQ5ZmRjYmQzMjc2YjY5OGJlMDBkOGUzMTQ0ZTNjMTMwNTZlNDc3N2I5MzRkY2U1ODQyM2Q3OGFiODZiZGQzMzU2Y2IzMzBkZWRlMzAxNjFmYmVlZGI5NWY3YTc5YzgxYmUyOTM0OTdmMjIzOTEwM2FjOGE1ZWNhYzYyMjQyYjI5YWEzOWM2ODU2NjY2MmQxZTQ1NWI3ZGQ4N2YwY2Y0OGIxMmYzMGZhZjBkYWM4OTZiNWUyMTg0ODA5OWY0Nzc3ZDlkYmFkM2MwMmQ0ZDNlMWYwYTVlOGFiNzc5NjdiY2YyYTRlOGFmOTcyNTU5NDcxN2UyZDc0NWRhNjE0YWI3MDM0ZTEzZTU5MDQ0NjdjMjhjMmVjNDYxMWE5ZmI4NmQ3OTIzYWE3ZmVlZDI4MThmZWY2Y2U1Zjg4ODcwOGE4ODdiNTRkMjAwYWNiM2FjZjA3YzNjOTI3ZmY0N2M5ZDdjZDdjZWM1YTM3OTVhMTcwOTNmMDM3YjczYWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3c7ZfOoM00sbvLIwwt3pZABRTp3Q2XM93_P7j4K7jF6ZzgEUdFvTe3DHEEnwWbe1rQdPn-ibFsRuThANyB6SSA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.923Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImV6c0RjazdaM3NaZ2lVcmtXRmlyWGZZbmxlQVdHaGV1WTErSGhUV3luR3gvUzVkOElSbGd6VTBQWGlpVnZMUHRua3RpTUh1VWNocW1ZOUZ1OGhUYUNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAzMF8xMDM2MjFfNjhfMjQzOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzA3YTg1NDg1M2Q5OWJkNjM1NmNmMWExNDU5OGNkYjliMWI3NDdlMDQxNjkwN2E5OWFiZWJlZWYwYjE5NGY1N2NkZDZlNGM5YTZjYzc3MWE4YzBiZTBjNzBlYTk5OWZlMWU3MGUzMzI1YzNkYzQ4ZjhiYjUyNmVkNWFkMTc2MWYyMWJlZDVkNTc5ZDUxNzJlYjBhN2NlMzkxZDIxZTg1MTYzNTlkODk5M2NkNzY5MjIyMzFhMGQzMmZiNDExMjg0NGMzYmFiYzQ0MDUyZmY0OTJiNzMzODdiYjkzMjIxMzY0ZTllYzNhOGYyYTQ2ZjhjYzM4NmRjOTM2NTgyYzhjOTkzNGE1ZTBlNzdkMTQxNzE1YzBiMjE4YmJkZThhZmYxMzE4NzgwMTFmYTRiMDgwMjI0NzllMjUwYzNiODBhNTkwOTg5N2YzMmU1MDNmN2M0MDFiZjk4NjQyMTI0NzNkNGM2ZWI0M2RmMGY5OTZjMDU1MDhmZmI3OTRhMjI3MmNjZmI3Mjg5NDVmMjY3MDM1NTA4MjM3M2ZhYzljYjc0ZjdjYzM5OWZkNmQ3OTJiZmM2MWUyNWE3ZjIzYmFhMzAwOTBjN2ZiZTYxNDFlMmYzODkwYWMxZGI3ODNhNzAxYWYzYTQ1MzFlOGNiYjJhMTY0MGYzMmYxNGY0YWZmNGVmOWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qIc_wnbibrv36H96dcZkOKPtcTUf-oZ5ibFLJAMqCLOD9UkDNcWsqHDSWWIFjfFEr21-Fz3ugcTUGqPJrIxlqw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.926Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlAzaWNqUktMMTF4bGZpdzRHd1ZGZ0NFaTdxaU5CQnVEQTdVeTBvRkJmbGJjaW1kREJ4VHp4a3dLR1lZVVlDYVlUY2l5amhJNzF5N0NyVzB5NnJpQ3JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAzMF8xMDM2MjFfNjhfMjQzOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDhlZDQxMzkxMjJjY2FmNjgwMTNmMTRkNGY1MWEzNWQ3YTQ0YjI2MzE1NjQyY2I4MmVlMmY2ZGVkNDc2MzAxNTkyNmM4ZGY5M2I0ZDg3YmI3ZmRhYThkYTllYjVhYzE5ZWE5MWY3MTFkYTc5NDQ5ZGI0NmQ5MmEyYmY4MjRmMjNhNTE5MmI3NWU0NTBkNmZiNTMyNDgyZjNjMzc4NGYzNWU2ZTk3NDM3MjJlZDE4MjU1YjNhNjAzYTFhMmNhYWNiNjI4YjJhZDllYjg1NzMyNjI4ODdhNGFhN2I2MzNhNzY1YzJiMjFkMzhhNjUwNTU3ZmRlZjkzYzM4NjZiZTdkYjBjZWIxMGJhMThkNjUwM2YxOWM5ODU5N2ViODI1ZTY0N2QwODQ2ZmNkMTdlNGFjMjAxOTU2MTRlNWRjOTcyY2E4NDg1MzcyYWE2NmU0ZGVhNDY4MDMwNDQxOWQ0OTk1ODY1OGFkMDNjOTlmMzdjY2NiYTU4OWRhYWZjZjMzYjU1YjIzYzYyZTcxMmQxYWNiMWQxNWVjNDM5OTVhYmEyNTlhZGI1ODkyNDk5M2JlZDM0MDVmYzBhNzY2ZDA5NWJlZWNiNzEyNDcwZDQ1YWU3NzgyZmJlNjExNmNjYTBkYjc1OWIwOTViMDQxMDE5MDcxZmQ4ZjY5MzgyOGVkYTMwMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ID6o_o2PM966AkdpjZ2sY_gcunJ7HXIkI29ppuwWh_dx_a2Rugr-FAYKddCLCqOJXts4Fpo9Kj5dStRJKHh6gw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.928Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImpMUkhFSHNqczZFZGFrbTdGQlBJdUxDdDVwRTU4Y2l4ejg0bGJVQ1NoVERqaWEzVFFURkZpVUExdnlVYS9NdFhkeDVLek1IU2tZdkhQR045SzZDSzl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAzMF8xMDM2MjFfNjhfMjQzOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDViNDgxMGZjNzE3MjVjN2JiNTYzOGZhMDg4YzQwMjRjZmY5YWJlMmYxMWU0MmExY2ZlYWViMjg1Zjg3OTY3MzViMzQyYTRlZDhjY2E5YmNmMGUwM2RkNWZmNWJjNzk0M2JiYTNmOGZiYmM5NzlhZDQwMmVkY2ZiNzMxYWFhZTZkYzRiOGRhYmZmZGM1OTc3YmNjZjQ5Njk5MjRiZjVhZjkzZDQ3M2ZiY2NhNjVjYTQ1MThiMGU5MzAxOWZlYTBlMmI2MWZlNjJiZTQyOGI3YmUxOWEzNWVmZTA2NmYyYjcwNWVhNjVkM2ViYWIxMDg5MzNkOWNhZWQyYjI5YTk5MTk3Y2Y3ZDZjNTg3ZjE4NzZkYjU1MjBkY2JkNmNmM2ZiYjk3OGRlOGFjMDQ3OGI4NTgzYjE3NjIyZjdjMDZhMjI2NTE1MzM4YzJjYjM3YjdmMTRjYzllZDRjYTA2NWUxMTdlMWYwNDg4OWEwY2NiZmRhM2RmMzY3ZDY1ZmJkMDhhM2IwODllMDhkZTRhZmRhOGQwNDAyNDY2N2UyNzg3MjkzZWMxMzQxMmRmYTQ5MWVhNGQ3NzRlNDY2MWQ2NzJjYTE2YzljYjg0NTRkODg3NDdkZDMzNjhmNzdjMmFhZGYxMDJjMmY1MDQyODI1OTVkZDQzNzYwMTAxMDk1YjE1YWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Pe5Lak1msmhnWEHIy3gkdj5P3pKhLlM8hRTnNftT-LJqJOh0MmGorUAMpWx8zHNzyZipiN0rurONjJqPQGgNAA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.932Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjE1QTRYeGV2QkxjbDhFWkgxcGEvOGhucXkyVU1ZVlNOamZ2dDVWRkM4YysvcFI2ZkVraDhiR0czdEpnOVJlTEEyNjVvb1I0TktYOFUrYkFuNG9PMXBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAzMF8xMDM2MjFfNjhfMjQzOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDMxMTNjOWYyM2MyZTViMTA2NzAzMTYwODEzODgxZDM5YmIwZTRiMmY3ZmM0OTJkYmZjMDFjYmNiOTdiZjRhOWYyZGQ4NTA4MDgyZDE3NzM0Mjg3ZmY2MDVlMzQ1N2VlMGRiMTk2MDlkYmI2ZmE5MWZmMWM4NWU2NGM1NDdjMjNjMmRlODE3NTlhYTE0MjUzZWRiOWM3NjBiMTljNmZmNzExNjhjOWI1YjE4M2IyZmU3MmNiYTI5YjE5MTNlOTYyM2Y0YjU5MDU5ZTc4MjQzOTdlNjUwNGNiY2NhMzI1ZTg1YzlmNDE1OTFhMzE2MDdjMjMzYmVjYTljMjRmZDU2NjNjMDE4ODM0N2ViN2I5YjNkOTI3NGIyNTI0YmQ0OTBlMTBmYWNjNzRhMWE3ZTY1NzM2MWU2OGJhN2JhYmQ1M2Y3MGQxNDc4Yjk5NDliNjlmZGRkZTVmNGIyOTE3OGEwM2QyODMzOGM1YTVkZjgyNzFkMzAxZjkyZDU3OGYxNTA0MmQ5MzFhYjczMGRkNDRjZDBhMjEyYTA5NzY2NjUyMTNlMGY1YTk1ZDAyNTdiMTIxNjIyZTA2YTI2N2Q4NjIxMjM2ODRiOWMxMjY0YzgwMTZkNDUwMDg5MTVkYWExNDEzNjY4ZDM2ZjUyNDNmMDE0OThlMDQ3NGZjMjQxNDNlOTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sOZJdAh1l-WiX4dirjTVsAVapB2i85yev1V94qTzoOedql68OlH_UeX0N57rv-UvDLyKRvssuQxvnNkkuzWkVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.935Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im5BdmlLdXNTdkFaa1dlQW5Cb2F1SlI2YjNDY0djMUtXY1RKcXNYb0tUMzVMZ1FaRE9MWExQUDFhQjdEd21wcnJlVzR6T0NhQVk2NmwzRnJVQTBjNkNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAxMjdfNjZfMjRhNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQ5NWQ5OWNkNjg0Yjg2N2M2MDU5NmRjZTE5OTRkZjU5YzdhNTk0NTBlNTQ5NTM0OTRkNDYyMTdmZTBmNWM0NWNhNmE0ZWY5N2Q4MzI2OTdkOWYwMGI5YTk0MGFjZTIzNTE3MDcwYmNjMzU4MTllM2Q5M2VmMWZjNmFiMmM3YTBlYWM3NWJlYTA2ZDllNGVmMDQ3OGM4MDcxZGY5YjUzOWZkYjQ1ODk2NTUzNWFlMmQ0ODU1ZmU3N2UyODk2ZGU5ZmZhN2FhNjI4NWFiOWNjOWU2ZDczZWY1YjhkMWQxNDc5YTVhMmJmMTM3ZDcxM2EzMjJiNmIyN2U5MzkyNmY3MGEzMzEwMTdiMjVmNTg0NzgxM2I1YzA3YzA3YmU3MGZiZjdmMmI2NzRhNWI1NjdiNTIzMDY2NGFlYTA5MmZjNGRlMjZjNWQyNjE5MzFhNTg5M2Y5YmFiY2IxZWVmZTkwZjA5Njg4NTBlYmNlZjBiNGU2OTc3OTRkYTJjYjE5NDI5ZjE1YmM3N2I1OTU0Yzk1NzJlZTE4OWQzZTVlNDc2ZmFmN2Q1NDI2NmQ1NjRlYjQyYTIzNjhmNDM5YzFmODE4ZjdjZmNjNmZkMzg1ZGUzYWEyMDMyODkzYWNiZTVjNGFhZDEzNzhjNmMyM2JjODczOTVmNDQwMWZiM2U4NGExYTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lcTa55admKytJLB6vp0C0jVGgjqESJ97r7efceRoLL3Fc0jV5_JFLAjxC7e0jMRlJ38VQvJ19zdNsfLMbrKVFg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110127_66_24a5_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.938Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkVsTVh6UTVaNjJmbUpQSTRZck5tcWt5QUV1RmlYQjBqRnVlU29sRDZTRTF4UExqM1NPNFNneXZ2T1ZKWlBWRDZ4V0NKN0Z2ZFlpVWJQa2N0UnhFU0hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAxMjdfNjZfMjRhNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGYwODBlY2RmZDRlY2FjODQzZGUzOTdlNjE0OWM0OTBjMGNjYTYxNWViODhjMGU3YWRiM2Q0N2Y4NTRhNTU2MWE3YmE4NWYxNjI5NTkwMWVkZTU4MTJlNzNiOWQyZjNlYjkxNjNkNjBjYjJmYjRkZTRmODc5YzYzNTU0YTkwNzM5OWYyYWQ0OWExZjQ4NmEyYTljOWQ2YzhjMGMxN2NkZDNmMzYwY2UxMDY3NmRjZDc4MThmZDExYTBkNjY4ZjE3ZGNmYjcwM2VjYjM4NzVhNjkzMDY3ODhhZTc5OTgyMjMwMjVhZTZhNDE4MjEyNWVlZTZjNTU0MTM5NWVjNWFhNTdjODE5ZjhiYmI4YmVjZjU3OTNlM2Q5Yzk4M2JkZTQ5OGEzZDcwYTkzZGFmODg1ZjBhOGIzNDczZjliMmY1YjA0ZDMyNWUwMjJiMDc5MWViOTc1NDNhNTI0NGUxNTE4MGY1M2I2YTYxMThkZDAyZGE5NGIwYTY0OTI5ODE5MGU0NzhiYTEzNDQzOTg2NGQ2MTQxYzNjNDU4YWNiNDM2YmJhZWFlMTQ3MDg1NTk0NTljNzk2MGU1ODExMmYyNjNjNDgwNzQ4NzAzMGQwYjE3ZGQ1ZDI5Y2VlNzVlZjYwNWY1MjcyNWY0ZWVlYmZkYTE0YThiNzc3NWEzNzUyM2M5YWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.j0W2sDkfa1O__XMLdn_6RbC__E6FjV1zwWOfIio23Re77RTBP2yfSIdoZTjGMdvafMOSl6s_VTEhvL1afpT_Fg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110127_66_24a5_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.941Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InlSV08vTHU5WWp3TzBWT3hadmdMam1pYkYzT2I5RlNCemNQS2hOUGtuRTNROWlSelFEUmozWlp4eFdMZmxNMFJJcjhoVFB2M0pYelZ0VWQzMGUzb1hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAxMjdfNjZfMjRhNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTE5ZTMyN2Q3ZDE4NzJhMGE0MWQxYzU4YmE3ZWYzY2YxZDE1MjRhNzNiOTE0YjdlNzAyZDhlMmZiN2M3Yjk2ZGRhNWNiNGQxZmExZjBjYTNlZGM5NzVjNTMxZWVhY2VjMGJlNTQ4MzcxNzNjYjI5ODJmNjhlNTU5NWY3YWJmY2I0ZDQ0MTJmZWJlOGU1NzQ1YjY2ZWRkZTk2ZGZiMTFhNzczY2RlODNkMWYwMDIxYTVjM2M0NzhhMmZmYTRiNmQyODBjMTNjOGE4Y2ZkODVlZDRlOGVkNTliYzhmZGJhZWMzMmM4MTdjYjM1ODJjNTZiMmZiZWI5YzllMjExNWJhOWFjOWZkYjNiNjE2ZjRmMzFkNzcxNzZkODBiYTc5YWY5Y2JlZWQxYjViODU5OWY3MjhkMmE4YjEyNDk3NTgzZDM0ZjdkYzE5MDcwOTVmNzNlMTZjZTZiZWY0MjA3MzRlYjdmNWYzMjQ1Mjk2ZTNkODBjYzlkYmEzYTc0MzJkM2FkZWJhNDU2MmM2NmRkYjA4NzY0MDkxMGM5ZTczY2Y2ODBmMmE1NzQwMDEwNjQ4YmZlNTU0ZGQzZjA4OGI5ZmMxZjQ2OGVkYTI3YWI2NmFmM2RkNjUwZTlkOTA0NDdkYmI4ODcxYzg3OWZjZTc5ZTAyZTA2YzAxODFjZmMwZDRiMWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xIooz5yv1fnA7y6xdT5GG-OTpwwLjGT9XqUqCI5yi-nY9H4SFi8a4AxC_H_zDvLDi_aCgSYmC-BznrwbuEHSCQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110127_66_24a5_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.944Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlNKcVU5bnZFSXpHQlQxak1QY2VHd0EzSEZLWkNXMnhLYVV4OGltNkVRUzdvalFHSW11ei9DenUzRU1JVFpBSnFrS1ErVEk2U2x1aW43ejNWdzlCZnl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAxMjdfNjZfMjRhNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmQ1ZWI1ZmMxNzc5ODA4YzUyZTFjYmVlZjk0ZDc3YjljOTJkMjdjZDY0MTRkMGIyOGM4NTg5YzUyODRhZTBmYjg1MWY3ZDZjOWNhYTRlMThhNTA4N2RhMGE4YjljYTFkNWQzY2I5Y2VjYWEzZDdlNTgwZjY1YmM5YzI2NzUyOTE2ODk2NmRjYTVmYTM3MDJkMDA0MDVkNTJiYWViOTE5ZWNlMWQ1MzM3ZTIxZGE5NDllNDE2MTg4MjU5MDQ1NDQ5YTQ0NThiOGFmMTVhMGE2NGU4ZmQ1OWMzNzJmMmRmZDBkZDdlZDU5Zjk4OGFhOTI2NjYxMDUyODQ1N2NmZDMxNjZhOGU2M2U4NDRiNGQ4NDg0MGIyOTM5OTU2NGE2NGQ3OGUxZjZkMDI2NWE5NzM2ZDUyYjkzZWM5ZGM1YmVmNTIzNWJkYTNmYWRhNWJkYTIxOGJhMTNmMjE5NzJiMWZjMDA0M2IwYWQ4ZDkxZWFkYWEwZWJmYTYzNDRkYTQ0YzRlNWUxNjE3OTVmZTM0MDMxZGRiY2RlZDc1ZWE4MWE0ZWExYjQyOWUyNjMyZWVmNjZiNTI0OWI1MzAyMjU4YTc4ZjI4MjliMjllNDlkNDA0OTlmYWQ4Mjc0ZjI3OGJkYzQ0NzJkYTE3ZDQxNDlhMTljMWM1ODY1MzNjOGE1YTg4ZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yyfXDbAph-G1tIRAqjSK2c8VPh8OiIT4f8Ar_w-nJMTzsFZuQrwYreY0FE5IhXATFtkTy-5BZHerz47hJvuusQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110127_66_24a5_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.948Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik00MWVKbVhvNHlwenc3WjFKaitzZi9iajlJWlJHcmFWTXowUThIR3Q0N05VSlVQVzA2NStDMW84SEkxMG12NXdYMmN0c0ZTSlhjbjZxTSt6YUFkNGZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDQxNTFfNDhfMjQ1OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGY3ZWUzNzE4OThhZWFmMWZiNTk5Yjg2ZGQxYjk3ZTAxNGNhNWJjZTYzMzVlZWE0YmJjOGQwNDNmYTgwZGU0ZTI4ZWJjYjAwMTZkMzJjYzA5NmYzMGRiMTBjNDUxNzRlM2UzNjZlNjMxNmQyYTcyMGUxYzY4ZTgwYWQzZTI4NWJlOTkzNmE2ZGRlOWM1YTc3ZWE2M2NjZjBjYjcyMjUwOTdkYjk4NmM0ZjNlZmRkOTQ5YzYwMjY1YmQxNmMxZjQ1M2JkNmFjYTVjNzVkY2UxZDk4ZDQzNGRhNWQ4NDY1OTcwOTUzMDU5ZTFjZDA2NTU1YmIzYjEwNTExYWM3MDI5NDA2MGQ4ZDA5ZDZlOTc0YWI1MTQ2ZGM4YTZkZGIwMWExMjNhMWI2ZTE5NTg3MWUyNmI3M2Y0NjNhMThlNjdiZWU0OGUwMDk0NTJmMTJhNjk2MWNjYmM2ZWNlMTRlMTVhZTc1NWYzOTJjM2RjMDExZGU0ZWFiMzIyZjcwNDFmMGYxMWUxZmM0ODJlZjEwYjc5OGRmYzFhODE2MDJjNzk2MjIyZThkNTE0N2ExY2IwMTUyMDE3MmI3YjRlNDVmMTkyYTQ3MGNmNzkzZWM1NmVkZTYxZjYwNzYzYzdmNGZmN2Q4NzAzYTM2ZjA2ZmIyZTFiMDNmOGQ3NjBiZDYyNTg0ZGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZFq7099BLeZcsLQXotX4wHCaZEIyMRy4sCg8_XGSASe74IR5oStU50EQMPlT37PtuUqbQlzjEdbhkj5Hh8kCJg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_104151_48_2458_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.953Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Imc4VHNYNWFDSVA5NzVKdlpHdnd6SEdZejBSRXFmWFAyMFNtMzFNN09KODBCYTBkRmwzeTMrSzdpemVxMmtDcnEvWVVSV3EyVGRFekVGQnd4ckxWbHBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDQxNTFfNDhfMjQ1OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWNjZDhhNDg1YWY4ZjY2NGUxNmM1NDNkNGVmMzQ5Zjc1N2JlYTQ2ZDA4MTA0ZmRkM2JmZDkxMjE1MDY5MjNlMDhkYzM5YTQ4MTQ4MTk4ODUxYzc5NTFmZGNkM2YyZGNkODUwMTE1MzYzNWM4NjM4NTlkOWQ4MzYzYzYyMjFmZjJlNDVkMjZmMmQwNzg2ZThmZmFhMDk0M2RjNzUxMzZjYmJmNTk3NjA5ZDE2ZmQ3MGU2NjhiZTQ4M2Y0ZDRhMzZjNWQ4OGQ1N2ExMzVkNDE4OWYyZmE5Yzg4NDQ4YzBmMzk5MDVjYTUyZjc0YWU1NDVjOTkzOWY1NzcxNGNjYjJjNzE2ODYyNzcyYmZkM2FjZDc1NGU4MGQ3NzgyNmU0OTljOTE4ZDIyZDUyZGE0ZmM1YTk0Y2I0NzQ3MjQ4YTE5MWM2MThlNjI2ODFmNDhhYjFhOTc5ZTA1ZjIyZGJjY2M1N2MwMTFmODIzNzEzNzdmNWY4MjdhM2QwNjY0ODFhZGM3NmZjYjUzNDExOTc0MGFhODQxM2Q3OTVhNDdiZjk3YWI0YTcyODNmZDY2NzI5M2UzNzRiYWEzOWFlMzI3NjNlNWYyNzI2NzVjNmIwYTRhZGVkZDJlYzVhNTM0MjQ2YTE1NzM2M2E1Y2E1N2VhNjUwM2NhODcxODFmMGU3OTQyMTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Q5BfF_kocPv28w2KOlzeMoz5fqsK55vOsKuerW9UqUPBeF1wraKqUNKCArGYNVGFB51uQ-n3xALG-Fv3rz47eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_104151_48_2458_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.956Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Indwb1N5UGNtajlCR2lUTmZYSjV0Y0xvb0pOQkVPeDBWT01VTFNiN1Mzd1dXaVIvNUdKNm15UDNET1ArT1FCRGpvUC8xZHk0M00rQ2J3Vmhka3R3RlBRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDQxNTFfNDhfMjQ1OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzllYzRjYTNjMmY3OWYxYzAzNWE0NGNiZTc1ZjNmYjM2MGJhYmNlYjdmOGQ5ZTVlMjM5NGRlNzFlMTM1NThlOTE1ODQ0YzU5NGRlNTI5MTk5Y2JkZjcwOTc3ZDcxNmRiMTNiNjRmNmI5MTIwZDcyMTkxNGY1OWZkMDlmNGYyZThlZDYzZjE1ODA5ZTBlNjkzMDQ1MzdhNjZlODFlNTZiZjkxMzAzMGI5OWY0NDUwZGIyZWIyOWI3ZTJmNjFlOTE4YzViYTk2MzI2YTM0YmUyNDYxYWVlMzVjZjIzMDJjZTk2ZTZhY2M2OTFlOGE1NjQ3YTc3ZWFiNWVlNjM2YTE5OGI0NDQ1NWIwY2ExODZkOWI5OGEzMDcyZDgxOWYwNzZlOTJhODE3MDUyZGRjODdkNWM5MmI0MGM0ODU5YzU2YWU1YzA4MzJiNjdjMTg3YWM3ODE3ODBiOTM1ZTdlYzE1NTEyZjcyOTIwYWUwYzI3NzZiZTY1ODQ5MjhjYzcyNzFmNGMyYWMyOGMwNWEwZTc0YmQyMTM3NTFhMWFiYmYxMDZhYmVlNDI3YjhhMjk2MTdmYzFmYmEyYWQyNzk3ZDM2NDMzNjRjMTdkMjgwOWRiNTkwNTA1ZTM2MTRiN2QzYTI4ZTQ3NTFiNGI2MGRlNGNmODUwYjk2ZDY5OGJjYjZlOWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QL-Dsg0C_J82o6PYAECnOJIwM_OsYm1wejgKPuoTOThZyGt_t0yiLMzcsi8m0QBfKSKwaMRLz2VLkwePo56WDQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_104151_48_2458_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.960Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik1Tb2dSNElETGVNb2l0VjdsZ0hjbkZKSlRRQStuWW5id2loWGx4L09XNkwwWDliUzVzRjBkR3RTdnRkeldsS1ZDdUo0b2xWRmFpMnlteUp3SERiUWhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDQxNTFfNDhfMjQ1OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTBjMmRjYzk1MWY1Y2UyZDdjOTU4MmZiODE2OWFhM2IxYzU5NWJmZTNmMWJjNjcwYmFmM2M5ZDlmN2IyNjVhNTQ4OGU4Y2VkNTUyMGE0NWM0Mjc5NGFkNDVlNWY2NTAzZWU0ZTE2ZDIzYzEwYzlkMDE1ZTZmZjE2ZDBhODM4NWJkYTg4MWUyYmJjZjQwNjFmODc3NTQ2ZDdlZDFiZTQwNTdlYmY0NGEzZjE4NmJiNTA4ODM4YjU2NzQwODQ3NzM0MjM3Mzg4Njk4NDg0MjdhMGVkMmI3YzZiMTA5M2ZkMTE0MmNhMDU4NzA4NzU2ZDcxMGNlNjZlMmFjYTJlMWFmN2RjMTQxMGVjM2M3NjhlOGZkMDFlNDcwZTlhMzFhOWJkZjA4YmI0MTY4NWJkNjYwZDhkNzEyZjVmY2M2YjA1MGY0NThlZTBhNjhlM2MzNjM0YjczNzIxOWYzOGViNjA1ZTk3YjRlYzJmNmI2NzFjMjVjMmRkMDcyMzkzZThlNTI2YjQxNWIwYmVkMWM4OWI3YjE2MzlmNmFhZTk1ZThkNjE3ODIzZTI3ZTc2NzI0YmNhOTEzMTQ2ZGY0NDFjNjBmMDM1Yzg4NzczYjc3YmI2ZWY5N2MyMTU1OWEwNjMzZGYzYjg2M2ZhMTA2YTcxMTE4N2UxMTY3ODA1MGQ1NzcxNzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sxF2uoZ3BMuZn93N_xqx2gOgxc5WhXjc4ANpJ4qh4WIN7U90vaOAyGqnc_1_gXmGH1e2TZNRm6Ttfzns5O8lrA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_104151_48_2458_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.963Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjhQV2taQ0tUZVBEYlZnR2VLVzIxbmZoRG5ubEsxSWh5d1FJZDBZUWEwUDJuOWFLMVBsUllkZENBaWkySExicEczVG9GZm9FSGdOempYUnEwRWRiZDBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTJfOThfMjRjM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGJiODc1YzQ4ZDk4ODU2MzZkNGEyZjgyZDgyMGRjYzZkZDQxODQyMGRhNjM1YmVkMDhhZjhlNmMwZDk1NmFkYjk0MzBhMjJiNjJlZDFkNGU2ZTUwMGE1ZDk1Yjk5OTYzMjlhODE4YmQwNGU5NzAwOGZiZTcxZGM0MWQ0ODlmYzM1ZjQzYmU4YTY3Y2FkMWZiYzhhZThkMjY0MDg3YjY4YzdmZDI4NWFmNTFkYWIxZTc1MTA0NjY1YjNhMWNiZThkYTUyZTIxYzRkYjg1ZDUzZGY4ZTE4YzM2YzZhNjg4Yzg5YWQwYzY2ODMxZDFmNmEzYzgxZDA5NDA5N2M3M2Y0Njk3YmYxMmFhYjUzMDc3ODgyNmFkMzc1NDgyMzg3MmU3ZTJmYzM1MWUwOGViMmVkNjYzNGI5NWI2ZTYyNWY2Mjg5ZjFjNWJkMjFmNGZlNTgwNDNkM2E5YzA2ZThjOTM1NDhhNTgwOWUzODBkYjU5NGY5Njk0YTE3YTMzMmExMTExZWRiZGM1MzM4M2YyNTNlOWNhM2ViOGU3MjAyYWIzZDJjYjExMzViMGIxMDE5NzEwMTdjMTFlOTEyY2VlYWEzOWFjYjQwNDQwMTYzZTI0OGMyN2U5ZjdkOTJmZDliOTlmOGZmNzEyNzYwYTFkM2VmOWFhMTQ5ZTY4ZGI4ZThhNGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.l-H9jfBKhA1IHIbZZA0Dqz67MI3a5TvM4SDujFuyk_ipNxQuGEWzYG3x8ik3FVth43mAQgfDEc8VG51GBg65eA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102952_98_24c3_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.966Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlBmY1RuY3NFZjNtaFovL2orV0hmckxXS0YyOUMyc3lFSzRZNnVWYjU3a2QxNENpNEVra2RTZnFuVzRFYmdzMU9uNXRTUm53QVZDSk83aTdjTlUxQzZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTJfOThfMjRjM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTRjODQxYTA4ODUzOWUwNjY2MTA0YjgxZGFkNGM4YTY3ODQ2NmJjN2M1MDlmYTg0YTFlMTJmYTUxNDAxYWU4ZWVhZDk4NGFiZjUyMWJjODA3YjU5N2NjMzYyMTI1MGNlNDUxMmE2OTljN2IzOTFjMTdhYmNkMDJhYzljZDhjODQyOTg3MzkwOTcyOTUwZGU0MTMyMjY5NGVkZjI2N2E3ODdkZmNlZjFkZDk2ZDJiNzFlZDc2ZWJhMmRhNmJhOTFjMGY4ZWY3Mzc4NDU4MzVlOTFlNmI3OTJjNDBlMTI5ZjQxNDI5Y2QwNzlmYWMyMjQ2NzY3YzgwZTQ3ZGQ0NTgzNGNlYTkzMTQ2MDdmMDAzZTRhOGFhYTRiMWU2MmU4MTlkNDVkNmNmOGIyNjNlNjIwODRmYjE3YmZiM2UwNmQ4Y2E0NDUwNTZiNGQzYmU1OTljMTVhM2ExODA2MWZiZTU2NGY5ZDg5MTIyNGE4MmMwOThiOGYxZWMxY2Y4N2M0OTI0MmI0NjI4MjAzYzEyMTVlYzkzNThlNWMwMDI5NTcwYmFkMTVmMjI0YzdiMDA2ZDY1MzY3NWFhZGJmMzM0YmQwNGI2ZjJjNGRjYjczYWViOTIzMTU4MjA2Y2U1MTBmODk0Y2QyYWRjMTJkNzM3YTQyNzI4ZDhhMjA2YmZlNzc1MGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BnUYx4LYJIoK5k2bVxKPyTDwvDkI9nTyyERsW2_9_VYFS7KZmYb6nETu79IZozGwGSFhHz7ZKVpeITcrgFSwrw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102952_98_24c3_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.969Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InZWQkZzZXdXODFkSmlscVBGbjhFWVlseHRJQVJPQWIrUU1tdDdoWkpvK1BsbWQvN1B5UGx4RU1nbG9YMURpVVNCUDVCWkM2OFJnc1VSWmRLQm9sQmt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTJfOThfMjRjM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTc0ZWJkNjJjNzc4NDY3YjkyYjRjMTk3YzgyZWE0ZTkxMjE4MTRiMmQ0ZjkxMDg1ZGMwZTQ3ZTAwM2ExMmNmODcxMTA0ZDczYTlkODNmZmJiZWExZDUyZTUzMTM5YjEwNTdlYTdhMzMxZDNkMTZhMzQ0NGRhZTIwM2I2ZDYyMjg2YmRhNGRhZmIwYzY4ZjAyYzdmZGU0Y2NiYjc2Y2FhZTE4ZjVlMzRiZTQ3ZTA0MDA2MjkwNDZmN2ViNzc4OTIwZjI0NjZiM2FlOGI5ZTMzNGI3YTIzMTJkYjg4ZWYzMDE1YWUwOTFiZTI4ZmE0NGU3ZjBmNjQ2NTQ2NmUyYzQyOWFiZTMyMDNiNTgyZDBkZDBiYTZkNzlmNTRlMzAyZmYxZTRlOTgxYzIxNDU0NDgyM2M5MTFlZjkxZjljYzhmZThjNTZhZjY1ZjMyMDRmMmE0OTgzOTg5Nzc0OTQxZjlhMzYxNWQyNGM3YjUwMDM1YjU3MTlhN2YwMDEyYzVkMjJlYjkxYWNkN2YzYWZkNDVhNjZjOWU5ZDMxNmJkMTcwYTU2ODk1YjBlNDExZmNjZGFjYzc5MTQxMDQzZWQ2NzE5YThkMTRmN2MyYTdjYTVhNTQ5YmQ4Y2IyN2FkYTA2OWRhZGU0MDNiZDRhNzkxOTE3ZDJiZTAwOTVkYjFkZThmMDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NEysEXylV8RHkI9MMWy2DaMfztRVvwwVT9STboLlswhf7ECJjH4Yal-WBAz46s3_nNJaYcKF0rT6VUjT7k_0wQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102952_98_24c3_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.972Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlRkSHdSeGxISTFjaG9WNXlWQkV5K081VHlLNU53allKR1J1TlN1WnNFcUJTZmptSzhkeHpCV0xjUUNJR1ZzMVJ5QXRYWkhyVDZRcDd0MTYzSG9TTFB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTJfOThfMjRjM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGU5M2RkYWJiMTQ3NjY1MjRiNjNjYjIxZDhiNWQ0MGVhNDgyMjNkZjg5MmM2ZjQ5ZTEzOGIxOWQ0OGVmNTMxNzE1NWY1NjU3MmJiOTNjN2Q0ODMwYjdiM2U5NGZiYjA2ZDlhZTU0NWRmZTNiNGQ3Yzg0NzRkNzE0ZjliOTE3NzZkZWY5MDkyZWUxNWJhZjFkM2QyNDkzM2UyZWQ2ZGU5MjM5MWI4OGVmMmViMjI0Y2Q3YTQxYTA2NGJkODZjODQ4YzI3YmI4YTEzNTVlZWY3MTQ1YzU3MTk0NjRlNDVkY2FjZjYyNDY3MTM0OWI3MjRiNmYyZTUzNmY4MjFjMTQyZGVmMDg3NTZiY2E4ZjZiZTIwZmJjNDU3MzBmMGY2MWY4NjBiZWIyMTU3MWQwY2YwOTJiMDFjZDI3ZmRlYzcyNGM3Y2NhZWNlMWUyNGE0YzBlMDBhMTYxNzkyY2JmZGQ3M2ZhNjRkMWFmMTUxN2IwYmMwNzY1ZTQ5M2ZmNWNmMTYwODgxODU0MmE1ZTQwYjdjZTRiZmUxYzMzNzAzYjFiY2EwYjhhZTM5Yjc1MmI4NGI2OGYxNzJhMjhiYjg0YmU5MDQ5OWIxYmY4MDdkYTY0MWNlMmU3NDMyZmMxNmExNDA4ZGRjNDA3MzBhNTFkZGNmNjlhYjkxMThlMDcwYjkxNjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TiwzEGDAKxaGcPxHnEoKJHOk1Evnwwo1kqtrYNuRwfvbEWXM6Q70Ulrrj7789cWNQimhZ9yKLMtphY5xBQDFdw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102952_98_24c3_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.975Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im10WFRKT3VXT2dGSzVqVXVEamo0TmlyS1Y4aXUyK1hUd0dreGJxa2ZIQVRwTXpqN2pWTURTM3N6Zkljdk1EOXRsaWlMYzFrQXB3T0VjMkxXTzNQTVlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDhfMzdfMjQwMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2Y1YjhkOTM0MzIyMzExNWQ4YzQyNWRlOTY2Nzc3Mzk1ZmQzOTJlMTcxMDUzZmE4YjQyZGE5NTI3ZmMyMWQ2ZTZmMGE1OGRiY2MyZTUzODdmZTJjZTM0NDkwNGE4MDU1NjdlMjkwY2IzZTFmNTQzMWUyYTQ0ZjdjMWJjMWJjNmVjMDE2ZTViMzQ1OTY2NDA4NDZmOWMwNWVkNjZkNjFiOWIyYjcyMDk2YzBlNmVmNGFmN2RlOTIyNzgzMGExMDNiNzU2YTRjODVjY2VjYWM5ZGIwOGUyN2E2MTA4MjgzNGZkYTA4NzMyNDExYWI4ZjE0OTZmMzk5ZTJlZWJlNzg5YjBhZDUwNjYzMGVhY2ZjY2E4ODYyNWJmZmJjMWIyYWM3NTM2OTk3MWM5Njg0NTI1ZTkzNWJhZTdmMGFhMGNjNzUyZmU0YzkxM2MzNzkxNDRjMjRmZDQ3YTExZGIxYWJkMjM0MjFmOWM1NmFhYTZmYWNiM2ZhMDM2MDIyZDQ1ZGE4Y2RjMDc3MTA1NDNkZWFiZjRiN2E4OTA3NGIzODQxOWI0OGUwYTNmMDhkZjExZGRjYjFkMDkwMzBjM2ZkYmYyYzkxMGVjNzBkYWYyZjNmOWU1YzU3ZDUzYjliNDY4MWM1MTAwMWU0YTgyY2VlMThhMWI3NjY4MTMzNTVhYzY5MzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Vm4_emOc-ffjeHO4vebsNTWELXCDfKoCBOeC_8cYIIpVserFhGAzuFfK9UtzN0FBIPWS5X6JmgOBRz_w_JRp3A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112108_37_2402_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.980Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImhzN3FtWDUrYndodFcyVGZIM2E2SVBPNjRoSm9wT0loM0pBV212TTFZaVRHSmhhbEE3a1dQdG5CRm9veWd3VDlqSWRTUGlwdno0WDJ2VmVFVUpqb1lnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDhfMzdfMjQwMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGMxZGQ5MGY0ZTcwNzI5MDg2Nzc5MGVjMGFhYmMyNGYyMzE3YTQxNzM0NTkwMzQxY2U2ZGFmYzQ2OGYzMGQ1MWJlZGRhMjQ1YWJmZDRhNjMwN2YxMDBmZjY0YzRiYzg4MzE3YjVjNmRkNjVkMTJjOTE2Yzk0ZTgxZTMxMGE0YjdjMjFhZjQ1YzUxMDIzOGI5MWNmMDQyNzJlNDQ0NDFiZmI2YjFkZmZhYTQyNGFkOTM1ZDg2ZGQxOGRkZjEzNTNjNTM0NDAyMzc4M2ZmNzcxM2NmZWIwYjRkODY0YjY3ZjM3N2MwN2JlMjQ5NTlhMTI4OWE1OWIwNGNhODEyZDVkZmQ3OWRhYjM0MjNjZjc0YTQxNjJhZWY3NDZjYTExNGE3MDQ4OWRkNjg1YmFkMTg1NTUzNTUwNWRkNWNlMTQzMTVkOWI3YmJkMzNlZjc5NGJhYTJlZWY0ZWQ4MTdlY2JmMzQ2N2M2ZDJhZGY3ZWMzYWUyYzRlMjIyOTg5NWVhMjQ4MTQyMDk3MDQ3ZmUwMzM1YmUyMWZmM2MzZTBlMDFlNDBkNjA1ZmZiOGFmMmI0NDdlMzFhZWEyZjMzYzgwZGE1NWNhZjhkOGE1NDM5NjIyY2E1MWQwMTczYTdiZDIyNjdhMjk2NDA4Y2JmNWU0NGY4MGI0NTI0MzU5NDgxMDAxNzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tFDDW5CJrkA4mbFesb25oDUoU4qhgzA31LLDVH-bCfRYb628lwGyylrtKliWSmTiK1xj-XNGmM4ZW4VdfYF_Mg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112108_37_2402_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.984Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlZJUEVjTGZmN2llcmZHdnJvNGRZbS9UL0RQci9oaTYzQTkwZHkwak85QWxWaUtDYnQwYjFEc2VMWS9odzRCTy83T01TRENZRGlYdzMvRTYxdFRBbWpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDhfMzdfMjQwMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGUzOWUzYzVmMjUwZTcwNDA2MGQyZjdmZTYzZTQ4ZGI4YjUwYmYzMmFkMmVkMTkxNmExNDJhNjE2NDE0MTJiODllZmEwMDA5NTYwOGY4NDQ1OTQxODMzN2ZkOGMwYzFhMzQ4NDJlM2IxNGUyM2I5NzNjZTEwOTA2NjgyODlmOGI2ZmIxNDRiMjAyNzYyMzU1Njg0MGFlNmM1NWE1OGY0ZjMxNjQ2NDIyZWZlM2JmZWFlYzllNDU3Y2ViOWQyNWZkNDI0YzBhMjA1OTY4ZWI1NzIwYzMwMTY3ZmM5ZGY3NDQwYjk0ZjE0YTUyM2UxNzNmNjE4ZDEyOGIyNTQ0ZjdlMTgxNzc5NmYwN2FiOWYwNzU0MTc1ZDAyNzMzMDI0YTQ4MjVlYThjMjBlNjllN2YyNGE1YjMwZWM0OTZkY2FkNDEwNzAyYWVmY2Y1OWM3MzFlYWZlMzdlMzkzNDJmMWQ4YTM5MDEyMjU4MWYzNzZkODRjNmJjZjliNGZlNjhkMGFiN2JkZjFjYzJhZWIxNTc2MGM0YTBmOGMzYWZmM2QyNGQzZmYyMjM5Mjg4ZjE4NTY4NzM4ZmI4YzczYWRmNTRjYWFhZDU1N2YwNGFkOTg0ZTZiMDVkYjM3MmI5MzVkY2E1ZDAxZDE4ZmExZWI4YWQyYjQ5ZTNjOThkODA4NDg5MmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ydeod2XLa2U4Q5V49Jd07Nh9EZLKp0UxRVOD0JqjCF2xOz5rO9ReApH84KG1UinIytGqtv-3ki8NxI_Qk7qeaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112108_37_2402_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.987Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik9obEI0NXlZRWVTQjlKbHc3NUN6NGdpaisxQ0MxaFV0ME52THlhNm82Smt2bjZYeTlnbUxEem5Fbm11bVVaRUY4UWYvWkxCM05xR1JBaDlnTWNKSzNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDhfMzdfMjQwMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDZjM2FjYjFiYzNmOTJjN2I2Yjg2N2ZiMWQzNzE1ODBjM2U3MTI1NGM5MzRhNGY4NGU0NDIzY2QyYzM3ZGQwMDIxZDU5NjM2ZTBhNjU5NWFiZmMyNzY1OWU1NzAzZmVhNjcwNzcyZDg1NmZhMzM1MTExMzEzZDczNTEyZjMyYzRmOTBmNjNlMTIzOGI0ZTUyYzZjNTIzMDMxN2M3Yjc0NWIzNGQwOTJlZDk3MGU4ZDEzYTc0YzJhZjRkNzczN2I3NzIwZDA5ZTU5YmQwZmZlMjJjMTQxMDNjYzI3YjBlZWIxMDM5ZjhmZmQ5OWYwYjNkZmVlYzJlZWQyZTVjZWUyYjVjYTZlN2EzMTYwODMwMGI2YjAwYmFiZGRmNWRmOTczMTNkMDU4N2ZkMTYyYmQ0MTU0ZjVhMTU1ODg3ZDhjNTdiN2E3ODRiNjEwYzNmZmZiZTBhM2U3YWFkNmEzYTY2OWE2ODcxYTMwNDc1YTEyYWQ1ZGVjZTVjZjEzYTQ5ODNmZDQyMjUyYTQwYzY1ZWJjMWQ0ZTY1NzU3OGEzYmUwYjQ4YTllYTRhYWUwMWM5NmEzZmYzMDlkMWU2ZGZhMDA1YjgxZmI1NDIzMTQxYzRmYWU0MTc4YTU3M2U5Mjc5MDFjNWVkNmVjYjQxMWE0NGM4ZWUyOTczYTc2ZjU5MTEwOGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.k8ds2z3WCBAoPe2PN4sW9TkawHZ1QOJ4RQiL5ax1SqJrJ0qh_EanLkS_abrcQkeTkHx3FBy3kNxdbi8YWaORkw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112108_37_2402_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.990Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjRKR1BxeWlwK1VDN3pTVHNQenJkejRWam44dExRMExyRnBVRWo5QmJBRVUxdWMzN3ZRYlZtZlJyNEs3eWx2Smg1YTVsZXFFRUFQWWZTRnRXSkxPazB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MTFfNzVfMjQ1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjUzNDIyNWViMTIzYmM5MDE2YjZiNWNlMmNkNjNlOGY2OWM5MmJlOWNjZTc5YTFlMmUzYTA5OWQ5NzA5ZDRhYWVhN2U2MDhhOWZlY2JlYjllYTQ2ZjM4NGI2OWE2MTRhMmEyYWQ2NTYyMjg5MDgwZTcxMzUyYWU2YzNiOTY3MjY3OWRlMWFlNWQyMjQ0NzdiZDVjNjg2NGUxM2ZkMjVjODk3MzRlY2NiYzgxM2RlZmNhMjMyN2Y4YWMxZWRlNzE0NzdhYmI2YTYzNTcyMjIyMjU0ZDY0MzRiOWMxYzFiOTExOTg2ZTdkNjg2M2JhM2QzMzg2MGQxYjM2OWY3NjllOWQ3OTdiNDZjYjMwNTc5MTFkY2NiNTM0MzA1YTNmMzU4YmEyMGI0ZTI5ODdhZjAyMzk3OGYwZDRiZDMyOTgzZjQ1MGEzYTgyODEwNzc0Zjk1ZjQ0MTgxMjgxOTgzOGFiYzkyYmYwNjNmZDFlNjhiYzNmNDk2ZDAyNzNmNDdjMGI2YjQ4ZDE2YjhiNWQxZmRiMWI1NWUwZjhmODdmYjBhNTMxOWRhMWU2OGU0NzI2ZjE2NzI0MzJkZDYwMTdlMDgyYWUyNzQ0YjM1NmM2MjBjZTczMjdiNDFjMjg3MzVkOTljMTY2OTk3NWNmOGJkMzNjZDIzOWNhYjhiNzQ0NjgzMjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.g1SJ7gmSqN9fw_NqF7kmP7DNLcAa8rb5RJJFJz69misUoqp8yk1OFcpSHIs2LRzjm_H0jC8o0Zwz9Hut_EiEYA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102511_75_2451_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.993Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlJVd0g2a1dJSGU5bFNvQ1F1aWhaUkltajRpMU05S0ZKa21reXJLSFJsMnZBWGUwTHRFQVhsdE82TFFKNzZoQUE2WjZnZWxxTDFXYll1R0dLbXlwVzVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MTFfNzVfMjQ1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTdhOTdkM2QwYjlmZmRhMWVjMzAxN2NiOTU2MDZlM2VkOTMwMGYzYjNkZjE0NjI3MWMyN2FlODNlYmE1NzFhMGE5YjA3MWNmOTE1ZWMxOTJkOGU1MjU0OGM0MDkwYWM1Y2ZjMWUzMGRkOTJhYTNkZTAwNjFhYWU3NjQ3YWRhYzNiODhlOWYzMWI3Njk5OTgyOGY2NThlZjU1ZGJiYzY0NjIzMjUxOGQwMzI3NTM4OTQxNTM2ZWZlOTdhYjcwZjU3N2VlMTM5OTc2NTdlZmQ0OTU0N2IzNmViOWY3ZGQ2MTM2M2E4OTUyZmU3ZjgzOTAxNGIyZDdiOTcyY2I5N2ExZjEwYWQ4YzYzZjJjZDlmYzk3MzJlMmFlZWYyN2EzMDQ1M2Q5YjQ5NzZlYzc0Y2VmOTg0MTJmYjgyYzRiYmFhZjZhOTlhNGZhNDUzMjZiMDQzMDU0NzAxOWM3OGU4OGE5YTk3Mjk4ODI5NWRlNjdlNmU4ZThmODY2MTc4Y2M2MDEyNzM2ZGZkYmQ5ZWI1OTg5MThlYWMxYzk5NGM2NDkxMGQ3ZmE3MDJiZTBhZjI1ZjAyY2U2Y2ViNTQyZmMwMTdjYTRmZjE1OTQ5M2FhNTU0NTUzMDcwMzZjYmNkNWExMmYzY2E5Yjg1OGUxMDIyYTYxNTE5MDc0OGI1NTc3NzQ4ZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fkgC-Goox353t2aZTes5boUb4IJdCeKi6U5mf3jy04wPehnCT-jvze68v8g-yZxCsx5HXoimjv4TnEhZYXxzkQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102511_75_2451_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.996Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImFncklhVzl4VWpXUVZKQURlVHJEVWR4T1d4YzlxZGlIK3dlQXFrOVRkR1BsQjJxTkpqSzlVYTZBS0hoT2xpcE5QY2xXNEl0anlhYjFMYzVMb1M1cTJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MTFfNzVfMjQ1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTBkNzM3NmFjNWQ5OTRlNWJmNzYwODkxZmFmYjcwOGJhMWYyYmI2MDlkNWZiMTUxZTVkYzNiYzJiZTJjZmQ0ZWQyMWQyZjVhOGMxODA2YzdkODA3ZmY2ZjEyOGNlNTAxNDE1MmJlMThhZTQ5Zjc4Njc4YmNjNmYzYWI5MzljMWFjNDJkYjQxYWMyMTMzMzU0MDdmNWU1NGI5ODA3YmNiZWRjNGEyNDMwZWE3ZDlkZGU1M2JiZDIyMGFhMDdmOTUzZjY5NjUxNzNjMTU0NTc3OTJkOTMwZDJjNWZiYWExNzUzNWZkYjI2ZjdkOWYzMzZhNjZkMWJkMzc1ZmE5YWQ5NTVjN2JlMmVhODVkOWUxODljZTBkOTU5MTE1OWM4MWUyNjE5ZmVkY2MxYWMwMjQ5MmM3ZGY0NmM5NzA4MGFhYjc5MmE5MWFjZjQ4ZmRlMTg2NjExOWMwYjBiZTU4ZTBiM2NkMjNmMzFmNTc0ZmZjM2ViZjVjYmY0NTE1ODg5Zjg1MmRjYWQxYmUyOWQ4OGZlNWQyOTI2ZGViMTM2ZmYxZDc0ZmJlNmM3ZjQyMDExOWY5YzE0NmZiZjNjM2I0ZTdlNGViYTJmY2E3NGQwZTE3ODljNGFlY2Q1MTFlYzFiOWQxZGY5ZWU0Zjg4NmEwMWRmMzE3NTU4ZTVkMGExZjUzZWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._kwIxLrCKWQVh6RTyxZYRLQyh1mHVUSWpF4pjVAPkCkN9QwpPMoWkp1M_yujafdRBg9BlVEfgYf8TMHK47qYbQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102511_75_2451_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.999Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImJGMDhhak9pYnhHTjdjRnZmbVBjUFZsQllqdDhnZ2JuUnVnYzJKRC82R0JXYlFaNDVadGJyejVoV25sN243dVZKS0E2MW9EandHSnE4Vm12ZG5jTnl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MTFfNzVfMjQ1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTExMDcyMzgzNTdmY2NiMmJjYjk2ZTEyMjk1MjkxOGNkOWYwODdlNTAzYzFjMjY1ZjIzZWExZWU4NjE4ZjYxNmY0MjFmYjMwN2M1Mjc4N2VlYTc2NGNhN2I3M2IxMWQyNDgwN2E0YzBlZTRmNDk0YzE1MTVmZmZmNzE0MWQ4NzQyNjA1MjlmNDZjZTFhOGIwY2E4YjI5Yjc1YWY3ZTFlY2Y0ODJiY2Q2MDIyN2ViZDgxMzBlYzg3ZTRhYTg3MGUyMWQyMjNiMjlhODg4NzdiNzBiNzgwNGY2NTFhNDNkMGIxMjc5OGQ3MzZiY2IzYjIyZmU4N2I0ODEzNjdjMjVhNWI5ZjkxMzUwMjA4MmQ4OWZhZWYwM2UyM2ViNTA4OWMyYzM2MWQzZjA4ZjA0NDE3ODMxNzY0Yzc1MWQ1YzMxMDE2NmJhMWUxYjBlNjdiNmUzZDk3MGM5NzI2NDA0Yzg5MDQ1ODVlYWVjMDAwMzIzOWQxMWNmYjcyMzE0OTcxNmNlNmNjNmNlODZhY2JlYzhmMDJjMzFjYjIxMTEzYzVjYzY2N2M0N2NlYWUzZGNiZjIxMWU5YmY5ODdkMmY5ZmM5YzA2MzUxMzZiNWQ3Mjc0YjI1NDRiZTIxOTYxYzhlMTk4ZTFjY2FkNjMwOGI3Y2U2ODY4OWQwZjM0NDgzZjc2YjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4Wef4kz2wagK4wzh6bQou6hkpiQdwwUEc-OzqEznwigO0Pb1QrekqFe0-E58qDR7REBnrgt48wKHpz2Jd5N6xA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102511_75_2451_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.002Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IitvWVZqL1NWS3Vvdm85Yi9ISkNpOWwyV0t5Q0dLQ09Lb2NVSTIvM0xyODRUbTZnbnZZNzJyVFhjUkt6UnFySUhjQkg0UmtVMU91WnM2U2NHeFZtQ3J3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDRfNDFfMjQ5NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2FkNTYyOGU1MDg1M2I2NzMzZDFjMWFmNjJiNTA4OGFkNjQ3NGJjYTkyNzhkY2Q1ZTYzMjU4YjJkMGEzOGI4ZjVmN2E3ZGMzZGY4NjBjZDk4MDdjNTJlOTk0NjhmMTM4Mzg5NTdlMTBjZTIyZmM0ZmY0ZmZhZTcyOGNkNWM4YmY4NDI2OTBiYzZmNzRlMGM5ZWIzZjA0NTEyMzE1NjdjMjNlNzQ0YjE0ZDVlMGQwNzljZGYxMDIzNTJhNmRkMDM3NzA3YTI3Mjc4MDQ4MjJhNjE1MTcxNzU1ZmRjZTcyZGI3YTkwNmJkYjFlY2M1ODVlM2Y4ZmYyMGY0NDAwZTIxYTc2OGNjZTAzOTY3Nzc5Y2FlZGM2YmJiNTRkMTA0ZGNlNGM0YzQyY2U2MTI4M2E1NmFjNmZmY2RhN2I4ZDRmODc1OTEyMmU4ZTM5NmNlM2IwZDI4YzViN2JhNzJkOTExNTIxMDc0N2EyM2Y1NmNlZWU5N2I4YWJiOWMxZGI5YmM4MjczNzcxZGNmZjQ1NTg0N2EyMTlhZTM5MjU4OWFhMDc1MjI2ZTJkOTQ2YzZkNDA4YWQ0MzYzYWI1NjI3ZTNjNmQ1NzRiMzk2ZTRjMjcyNDhlOWJiZDU3MDcwMjBiYjMxNzRiNjJjOTU2N2Q2MjkwZTJhNDBkYTRhZmU4OTg3NGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ruWIKMeqVFJZJ0_u3cLMintzbcZqDYUYR7LaQLHlnIcy_S3b1_tu5WCFUbYVFOnlPXlGV2jrSCUllmpz0IR8bg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105944_41_2495_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.004Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtnT004SkI1aGlubElsQ2FvOU5PL1pWR2RxLzluTzYwamRsMEVNVmVCeW1oSWZ1VzZiaGxDYjYyV2F2Tm9oZ3lqbG5NcGY4VHBvRUVnNmVRMFUxUkJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDRfNDFfMjQ5NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2RkZGZkOGU4Y2NjNjQwZDIwMTJlNDMwMWJiYjEzNDEyMTVkNzRjMmIyNTZkYzI2YTM5N2U4MmU3YjYwMjRjNjg2NTVkODE2ZjMzYmVkMzg5NGEzOTYwNDhjOGZhYmJhODU2OWQ3MjY4NzU1MzIyNWU2MzRmNDBmMDEwZDhjZDNmODcwNDJmN2FhZGQ5NDBkOGUzYjY1NWU5MDVhNTQ2NWQ0NjE4NjY5YWM4ZmJkNTA3N2NhYjNkZTJiNWQyMDZjNGM2NmE1MjQ2YTRjZjI5MGNmZGI0M2EzOTU2ZTM3ZDViMDQxODgyODVlNTg5ZGM2MWYwMjU5MzFlZjE0ZGM3YzVjNmZkMWJkODgxNzYwZTg1MGEyODE3M2EwZDVjYmFjN2NhZjE0MWI2YjIyMjI3MzQ3MzYzMTlhZmI5M2RmNWViMDI1YTNkYmU1MzI0YmE5Y2M5YjEwOTdjYTFmMGY2ODAzZjJhMDhlYzFhZWU5ZmQxNzJlYWMzNmJmNDVmOTFmOTVkOGY5ZDk0OTM2ZDljN2JjOGI2Nzk5YzgzODE0MDllOTdiMTJjMjc0YjFkYTJhOWJlNTgxMDBmNTQxZWQ2ZjllZDI2OGI3OWJjZTNjZmE0YmRiZDI4YTczNDM2NzU2MjRjZThhYzIyNDdlYTE2OWM5ZGQyMWU5NjU0ZTFjYTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qUvji2N6creXa75mohqWosHG4tmoBWFHOXIZsMlLXgbi-tXL4yGoOITsyop9wuKoyuHiMJgdlPiVTGcMMPRiIA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105944_41_2495_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.008Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlM0ZWJYWFpXM05QMGF6REYzNGxnOFZBTjRONStaMnRHU1VycWpLOUJtc0xmMitLdlBPUlRBTXI5eFdwRi83cldOVldGcU5ha2RzdFRWSWg1S2k4OUlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDRfNDFfMjQ5NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjU2M2ZhZWIyZTZmZDAyMjI3ZWM4NjlmNDkxN2MxMWQxMjdhNzRiMzA4MTJmYTUwNDBhOTFlN2Y3MjM5OGUxMjFlNDg3YTk4NDE1MDA0YmNhZTkxMDQ4OGVjMDY0NGJlODJlMGRhMDJkYmQ1N2FiYzY1YjFkOGYwN2M2NTVhMmE3ODQzNWFmMDA1MmVkOTg4MTE5MjcwMWE5ZTdhMTk4NGQ2ODdjMGZmZWE2YWZlMWM2OWQ1ZTA0NWEzYzEzYTViZWJlYjk4ZTU5YjYxMjk4OGQ3N2U3NDNhOTAxMTM5NjU0ZTY3NTdkNjY4NGQ3N2M1MjM2OTJmYmZlMTRkYTQyYWEyMWI2NzVmMjA5NjBiYjRjZWUyZDNlZDE1NmM3NzcyZjYwOTZlNWVmYWZjNmNhMGUyOTcwYjU5NGU4YjI5YTAwZWIwNWE3MzNjNDBmZjJiYmZlZDM3NjdkNzZlNTM1ODcyMjk1ZjQxYmU0NDI2NDJmZDBkMTNhZDNhNzUxMjM5MThkYWI2NTg3ZDcwMWM0ODAzMmU5NTVjZWMxNTc2NjBiM2FkYmJhMDU0MzRkNzMzNzllOWUxYzdhZmE4MTJkOTc5NThkNmFlMTNiOTEyNTU5MTZiYmU5YmUzNGQwMTQ3ZGNkZDRlY2EyZDdlYTc0ZGUwY2NkYjgyM2FhYTUxYmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hVqNwx8L3IN2roUqDUReP8uPFBnBVteflcwKe4LGgSJ-EArvDNKtrwi6cbau-bKx_1XkPmuX4TQKqNKaKh1Hpg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105944_41_2495_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.011Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBGd0cvSWgxTW5vZVVTOFBRVDE1NjdBMkhMcGx2T09oUHdKaGRIVUhpNFZ1V0pCbGxhdlNYbTVuaUxJV2hFaFY0UlBmRGJSdHR2L2NSMXBLY0U3WkdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDRfNDFfMjQ5NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDg0YTA1NmE2NzYxYzAzNzZjNWRlZGZkNmQxOThhZDBkYTM3ODkzNTk1ZGFhY2Y1MmY3N2NiZDg2MzFiNWFmYjc0ODI3NDEzNzQ0MGRkMWRkNDdjMzYyNWUwMWFiMTBlMDE1ZWFhMTI3YzY5NWVmNDQ5ODljNjEwZjZlZGZlZjAwZDc0ZDJkY2Q2MTAzYjhmN2RmZmRhYzk0ZTgwODY1MWE2MDEyNmZmYjU1OWViM2E0Mzk5NzU4MWQ2YTE5NDRmODIxMzFlOGVkYWE4YTZmYTU2ZjRlOTE3NjNmNDQxYjlhOGUzMzM1MjYzYzIzOWQ1NzBjNDRmOWIwYWFjODgzZDNlNmQ3Njk4NDQyZTYwNTY3NjFlN2IxMzJlMzk1OGE5ODc4ZWFlZmEyYWI3MWRlMTVmNWVkYTRmNTM5YTMzMzBhZjBlYjNmNjJlYzA0YTBlYTM3YThlYTI4YzY3ZTAwN2JmNTQ0M2VjMjc2NjZhYTk2ZjNlMjczM2JlNDA1Mjc0MTZiYTFjM2I0ODliOTc2NGY4ZmVjYWVmZmNiODk3NmU2MTYzOTJkOThmNTU0Y2I3MmFlYjIyMjE2OTcwZDk2MTc4MjRlMGNhMzMzZDY3ZTUwZmIwNWQ0OTZiZjk0YTg1Y2Q5NWIwZWZiNWIwM2ViNzZjY2RhZTc4MDJmZjJjMmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wPHkDDoAW_tRE1xchgggvmD9tp66uTa3Tp6nWYnosdsHbpvpvuV7jtdB2nk31XVFyAXlF7jkdk744IpIK0DP5w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105944_41_2495_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.015Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikd5RllEcVd2R3N6b3VVUEhhcmJOMEFsZ1RxTnJCcWJFWHkrMUg3ZHNIV2FEcEpvdW1FeU1uRjUyOWYwZXpmaGpiK2d0b2ptV0dDbkJlQVpYTDhtamxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA1MDFfODFfMjQ3NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWEzNzI4NWFjYjU0YTA3MmM3N2NmZjQ0ZTVlOGQzMWU1MDcwM2NlMTMxYjI3NThiZWMwOTcwYTBkN2FlNTQzYjc3MzgyZDUwYTE0MzkyMTEzYThjOWUxZjYxY2YzN2ZmMzA0ZWUzMmU2MTBmM2VjZTA3NTM5MjVhOWRhN2NmODQ0ZGFiMjYyODEwNWJmMmQ2OTJjNDQzMTY2NDQ1NTZiNGZhNThlZDk5Y2U3NDAyMzU5MmFhNGZiNWY0MTI5NGQ5MzBlM2ZiYjAzYmQ5YWY3Y2UxZDNjYzZlMWZkYWY4NjY4NGM4YmNjMzZmMGI3MDVlOWY3ZmMzNTIyNTcxNDE1YzAzOTBlMDY0Y2YxNWExZWFmMmZlM2QxNDM1ZDJjMWEyMjM4YmMyYjhhZmM3YzI3Mjk1Yjk0ZDViOGYzY2VlMmIyZGU3YWY1NzVlOWVkMDYxY2IzYmRlMWIzMDhlZDExMDE2NTY0Y2U0Y2U3ODJjOWQ1Yjg5NTUxZjIzZDgwMGVjNDVmOGMxOTVlZjY3YzRlMzdhZTY5YmQ3ZWQ0MDFjYWMyMTkwNzk2ODhkNDE2MDZhMGU0MDU5MGYxNDNlZWEwNjc5ZTcyMTc4OWZjODdhMTdkMWRkZmU5YzFkZjliM2EwMTcyMDQzYjk0YTk2N2NmZmIyOTRhNGVlOWJkMTliYzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TxZO1oqIk7iqGhkS7nbkXIgzity0bIfA6RH06R_7N8KGhuUh3ZJzvUc1wQJeJq76xpp8uyVffEv_AexNSLveqQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110501_81_2475_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.019Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkVmK0JJamFqUDV1RXhIVi8vaUJ5TlhDNk1TTCtXNnZreW1ZZGNXRDlqcUdlbGltaUF2ZFovRyt0ak5FMCs1VlFNdlBnL1FzTFIvWndyVUJ2eHExc1p3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA1MDFfODFfMjQ3NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2YyMTk4M2I4ZDE3OGFkNzI4YTUwMGFhNThkMjIxNDEyMjNkYTg3M2QzZDhkOGYzMTk0MTdjYmNkZmNkYzhkYzM2NmRkZGFkOTRiMGU3NzVmMWJiNTNkYzQ1Mzg1ZThhMTQ4NjUxZGIzZmFmOTEzYzNkNzJlNGZkNTcwNzAxNWUwYmY2MzhmZjI1NThjMmY4MzljMmU2OWMxOGM2ODVlZGUwMTQ2OTg3MzY3NzY5NDdjYTNjOTMyNzYyYWYxZDM5ZjNiYTNhYzc4Y2ExNjkyOGNjYmUxYzM2YzA0MGZlZDQ5YzNkYmNkOGM1ODA3MTg4MDM4MGI0ZWVlMDhjM2RmYWZjZjFjMDdkYzJhZWQ2NGZjZTI3OTNjZGNlYjFkOWEyOGQzMzQxZjQ5MzRmNjM4ZTdhYjc5NDA2ZDA5NjIwNGM1ZjA1NzQxMjY4OWU0ZWU2NTcwYzcxY2UzZTA2MjdlODE1YjYwNjA5NWQ1MDU3MTViY2I3NGZhZDUwMDJiOWJkOTRmOTM0MjgwMDgyNDFmNDNmMGJiMjgzM2ZjMTc1ZjJmZGM1NmM5ZWFiMzkzZDMwNWYxNDU4YjYxYWVkMzU5NDU4MzQxYjFiNGE2MjNiZjhmY2RmZTUzMmUwYjU4OGRlNTE4OTkxZWJmN2U5Y2NkYmI5ZmVmY2VjODYwMzNmMWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.z1moJoY3hmLFwnGizoBhzyHRu39-Y7AMjzRtUWhVJPPnUi9gkF_EUVL2E38B5q2TvRED0P5G-sMB7m9tEThDOg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110501_81_2475_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.021Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJaSWlwTUFBNUtxM0M2YVVRaGxodGRSa29YOTBpZk4zeC8vN3lLczRTQkc1eWRsUGpETjhGK241QTdReUpwenJKTWo4WE9jTHJRWVRXMmJVUC9uMjhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA1MDFfODFfMjQ3NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDU3YmQ2Y2JlMmU5ZjY3OGJmYzc3NjFjZjFkZThkODIwZGY5YTZmZDk4NDc4ZDM2YWYxZjE1YmVjZmYwNzM1MGRhYTdkOTAwOGE1ODY0ZmI4YWQyODllN2FjMmM4ZWY3Zjk0OGM0ZWQwNjJkN2YwYWI0OGQyODE5ZmIzMzNhMGJhMDBkOWNhYTg5NDQ0M2I2ZGMxNTMyMTg3NDM3MGIwYTUwMmRlMTBlMzJmOTk5NzMxMzVmYzg2ZjcyMTk1MWYxYjg5MTljODAzN2UwMzA0MTA3ZTFiZTU1OTUzYWMxY2M3YmU1YzQyYTE1ODE2ODkxOTFlZWZlNDU2N2MwODBhMjgyYmU3N2YwZWI0ZDM5NjE4ODlmNzA4MjAxMDM5MWFjNTBkMDhiOGNmMjg3NDE5NjZjMmE0NGJlNmRjZTAzMWFiMTUxMjFmNjI0NDRkMDczN2VlNzMyMjQyMTFkNzdlMjNlMjdhZDNiZjQ1OWRmODc5ZjAzNmIwNGYwNmUwYzE4ZjU4ZTFjYWYyODE5YTliZGM0ODQ4NjZlZDNhYTI3MTE4ODI1NjU5NmI4MTYyMTA4OGY4MzViMTBiNjZjYjg5YzlkMDZiMGI1Yjg0OTY0NzY0MjczZGRjYmY1ZDJmNTA5OGZjNTZiOTg0YTczNzczODMyOTYzODJkMzcxOWQxYTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UO_0-7D0jiiYX6qrPlbwj45VXcKnEbVVjep_MNBaSFTzXmIn6m6jZF607wtUM0G-jJD83krhP4EdbC2k0p2qFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110501_81_2475_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.025Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZpbFFGUzlCbzZUMmhsWXBtUk1YdXR6ZUl3Q0hwZUU4VExlKzNzZlFReWVUSzd2VnBuMDBsQ3JoeTdxUkJBQStkTnVyUjRNZTdHYUp5RTBsSDZGdVVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA1MDFfODFfMjQ3NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjRlYWY3YTE5ZjU0MmY1MGU1Nzc3N2ZmMzIzZGEwYjNhNWQzMGNkNTk2MzFiZTMzMjk3ODgwOTYxMzUxOTY0ZjViYmVmYjQ3NDBhNzQ3ZTU0YjY4YzkxOTA1YzgwMjRmMDExZmFmNThlY2RiMjcwODFlY2Y2NDQ2NmIyY2Q4MjM4ZDQ2OWNjZWI1MzIwMTc4ZGFhNzg2YzQwZDA2ZjMwM2ZmZThkZWY1ODUzYWExMWZjYWYyNzdjY2NhZmQzY2QzMDg3NWZmY2FlNjg1YWJmNzA4YjNhNDZiMDUxODg0ZWMwMDczNDViZjRjZDIyNDA5NjVjMGE3M2I0MmE5ZGQ0YTgwODgxYzY2MmZkMTg5YjY3NzkxMTJiYTZmZTQwZTljY2ZiYTNhN2U5OTY3NmU0NzcyYzlkOThjZTdjM2Y5OWQ5ZDQ3MDQ1MmU0MDQyYTc2MzE5OWFkM2I0YzI0YTUxZWMwOTVlODgxMzYzM2Q2M2QzOGE3OWQ0YzliNDlhZTkzNzE0MWVhMmQxODc5NWEyYzQ4MzE0MTY1OTA1MTZhM2VlZjc4NmRlZjczNjE2YzBiZmI4OWFmODcxYjc4ZTNiNGNkOTRkMzEzNTg0M2U3ODE1MjRmNTY0ZDU5ZjRjODQyY2U5ZWRhNTY2MTFkOGU1NWM5YzM2MWQ1NjgwZjNmOTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cAFzbUcM7ewPfYxRWch1rjLYlLRtrGPtYyHW0C7jWQvQ8_yfm_BHHSK6Eo59EDw7HhMFoZWCG_-j9KDQQrBS1A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110501_81_2475_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.028Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImdESUNGMVh1OXlhbjNOSmFDUldHdjlENGZuUEtCTUZjT0RLUElYVVgyZG5PVUo1MmNkaUNyRStPdmF2djg3ZkZlbTIvdUhIcjhtK3JPYUlod01VeWl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxN18xMDI3MzZfNzRfMjQyOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2RkODBjZTU1YWY5ZmU3ODdiNjUxYjViYmU1MmM2NWZhOTFiMTc4MDE1YzBmODM5YzhmMWE5ZGViY2QwNjMyNWE2ZTFmNzkyNjExZjk4ZWM3Mzc1MzEyNGU4NDg3OGZiYWZjOWU4MjIwZDAxMTMzODRjM2M4MDE3MDU1OTBmNGZmNWNmMjE4M2MwNTlhZDQzMzg1ZTRiZjZmMTkxOTJkNzZjMWQxMjAwMDhiNjVhMzc5Yjc3NDQwNzRlMjlmMGFjYjQ2OGUzYWQ4MDQ2NGVhMzE3NGZhNThkYzEzMDU2NjRjNTQ4N2M0ZjJlZmY0YWJhNTk5YzM1NThhYmFmZmEzOGE0ODk2NzU1ZjZmYjg1NDEyODNhMTA2OGEwYjkwZjFjM2MwOWI2Y2U4NTJlYzJhZTdkMWQzOTllYzhlOWFjZDllMGRhMGVkZmUwYmYyNTMxZTE1OWQ0NTE5MWE4OGM2NzhjMDg0MzA0OTkyYzQ0MzM1MzEyMmM0MDc0ZTFkZjQ0YmMyOGU2YzM2MzljYmVjMDNhYTBmNGJlZDUyNjcyZTgzZDQ1NTM2ZjY4OTA1ZTljZDI5YTRjNjVhZjIzNzRjOWY0ZjZlMDMzODIwNzM5NzMzOGQzMTUyOTI4NTQyNmFiZjU3ZDQ5ZmVlNzVhNWJmYTQ3YzcxMDBlNWNlNDgwOTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.59SGkPe4Wfe1MLx5_crKBsVsVa5fcbBX43MiHwSBeyI7IqXd93gVq9KFVTcGYQcSYQ4Ha7sBY9Dkh5cH38U-5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220217_102736_74_2428_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.032Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNJNHMzdFdMUGFWa05LMXh5Vnpyek03L3ArUC9lMStwWGFoWFk3MHJoZDVHYnBDeFBBaXRRbU9Tbm4zS2wvbG9aelpoMWJnOWR5UklCOXl0eFJmZDNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxN18xMDI3MzZfNzRfMjQyOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzIwOGQ3M2Y2M2VlZTA5YTJkODZlMDgzODc5ZDNmZjY4NTA3ZDYwYWI2Yjc1MjM5ZTU4MjExOGExZmFiNzQ0NWQ0ZmEzN2QxNGY1N2FkNjAyOTc1NDVhY2FhMjFhMWI3ZTZlOTc5MTczMTZkZGJjNTZkMjVkMjhhYWVmNDA2NzUxNjliYTg1NGQxODQ2MGFmMzRkM2FmNGE1OGFlYTgxZDFlYTU2YjY3ZjIyNTc1MmI0Zjk4OTk3NjJkMGMyMjgzMjkyOTU1MjY2OGNjYThlYTkwMGRhOTkyNmY2M2Y4N2I5MTQzMDUzYWZjNTUwODE3NWZkODMwOWNlM2NiYmYzMTMwM2ZmMGUxNDYxMDE0NDNkZDZiYjRjODNlYWIyNDA5OThiODc5MzEyM2M0Y2ExMDlkZTgzMWE2ZWQwMWY1ZDUxNWM5MjI1NWRlM2ViZmVkMWQ5YTdkN2VkZDk5NDliNWZkZGVlNWJkMmQwMDNkZGRiZDkxZjljOGVkNmQzOTI1YmU3OTZhNGVlNGM0OWJiMTVkNzJhMjRiN2Y1YzFjY2JmNjE4NjJjYzMyNWQyNzcxNzZhZTIwZjkwZjhlZDI1ODNlNzNkOGYxNzRiYTJkNGMxNWQxZWYzYTIyZTcxMzhhNjYyYjc5YWM5YjNjMzZmMWJkNjhhYTgzYWNjNjEwNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CFr6ocmhp_HihwuIfdvXF-yOj32Ep-Cgs6S6VG6W09wxIllcYeDGg5KxCcqwcLdCksGh1J5hWmUZR3R-NPMlbA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220217_102736_74_2428_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.035Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJkRWhQT2VUTWYrMDh5YWNmVldzRTBQR2Fod2VXM3NRdG9TdkpJUCtaZmtQMDVHbWhaa0YzV3N5SHQ1VDlvTkY0eGdLTTg1aU9hWUxBbzdpdWNHQk9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxN18xMDI3MzZfNzRfMjQyOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzE3YTFhMGZjMzkwODc0YTY5ZWNiYWM4NWZlNTVlZGE0NGQ1YjE1YjE2NzQyMTczODI4YzUzMmJlZGRiNjNiNTQzMDE4MDA2MDZiMjU3NTQwZDAzMjA5MGM0MWFjOTY3Y2QwY2U1ZDZlMDcyOGRkNGU5YjMzNmZmMGQ5ZTY3NzUxZDIwMGZmZjJlODkzMmQ5NzMyZDdlNjgyNDJiMjQ1ZDYxYTQ5Y2E1Zjg4ZjNlMzExNDJhNWVhNTM5YzBkOWQ1YjM3ZmExMWE2NzZjNWYzYjA3OWI0YmQ1YTk4YmQxYzdlMjcyMWE1MzE5YWM4ZTVlMjgwZDhjNTE5ZWIyMTIwNjEyMmNlNTE3ZmY4NTkyMDMxYjUwZjVjNGQzYTNkMmU4MDEzNjBiMjQ3Y2NkZGQ0NjRjZTU1OTVjYjI4MWE1YTdlZDZjMjJjZmM0Y2I3NmQxMGU0MTdlYmVhMTFmMTFlODUyZDFhNjMwYzFkOTJhYmFhNjdjMjc3NDg3ZjE2MGFkYTFlYzliNzBiNDYzZmJkZDE2ZDg2NDYyMzFkYjFiMDg1ZDBjMmUzOWVhMWMyYmY3Y2Y1MWQ2MDE2ZjUwMDUxZmIwMDdhODJmYjVlYjJmMzJhZDY5MWMzZDAyYzA3ODlkOGFlYmYzNjUxMzE0NTk1MzIxZDc5N2FkZTU5YWE5YjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.F6Z_YwvgqzRYVZ_iirsX5DvINCc0g7JcGE7GAMfxvVGTTaI1SOVuw-h7fzJSEXG6CpOvUtC5azKJhWRGvNupmw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220217_102736_74_2428_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.038Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkdPOUt4OWVnbktoSEc2YUVzbXhpb20rdW9kTlFjVWFVd3FJdjBJZzU2TTVRVmtyTlFNdE9iYy9VL1NSUFlRTllLNDA2OE5zVHc2VVl6VGZFN2laZ0NBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxN18xMDI3MzZfNzRfMjQyOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTAwMjM1OTg2NGNkYzRmMmVjN2M2NmFiNDI3YjNmNWM5NjBlYjgwNjU0NTEyMzIyYzFhOTg0ZGE1NzU2ZTU0YWUxMmI3MzRiODZmYmU5Yzc4OWI0Yzc0NzJjYWM5OTU5YzlmNjEwM2Y4ZTZjMDk1NjVkMjJkYjhhN2YyMGVlMWI1NWEzMTQ3MmQ0MjY5YjJhMjhkNzZiODVkYzJlYmUyNGFmN2NmZDljOGMyMTZkYzU5ODM2MjliODJjZmUyZTcwZWU5MGMyZWFjNGU4NmNjMThlMjU2MzU1NWRmMWZjZWIyODlkM2VlNjE5ZjRkOWE3YjhhOWZhOTJjZTNjNzA2NTk5ZTQ1ZDViNmU5ZDUxYTYxOTM4ZmM4MDcyN2VjNDkwMjRkNDY1NGQ0N2NmOTBiZTdhODdkZmRlOWM4MzA1ZTQzNjNiNTQyMjljNGYwODU1MjFmMmIwN2I5OTJiYmRhNjE1NzFkYWVkYTM0OGJiYWY0ODg2NDc1ZDlhMWNhMTM3ODViYTE1ZWQ5Njg2YTY4ZmQ3ZWRjZjVhMTA4MDFlM2YxZTIzY2E0OTAzZGQ3ODFiNDJiMWZjYWFiNjIwYjkzY2VhZWE5ZTE0MGVjNWEyYThhMGVmNjdkMjY0MDE2ZmJkYTU2MWI3OWRhOWNlYWJlZjljMTdjNzNlYmEzMzM5Y2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mwKP90TtsIbEGrT73ScWgtpkXUS7dife0xf7fYLZLsu-_-mfqkZz12S3LGhVcKCbLy71XkyQyhf8Cfu2jBD_zg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220217_102736_74_2428_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.042Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InhyT0ZmSlRTKzVOYmk4RzBRVEluTnRIVVAzYnJIZlFwMlBMc1lnMW8yYWhSdllGRzdGSC9YZXZmMm51UUY1YVgwK1NYL3dNby9pMXpCeEpjL2dOdHFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5MjBfNjZfMjQzOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWU4YjY2ZjUwODU4MGY1MDBjNGExMzU0NmYzZGJmODZlMjQ2NzVhNWRkMWMwN2FhNDE4MTUyZWMxODc2ZTQ3Yzk0ZTBjZDgyZTY2MmE5MmFlMDZlNWQ0NjQxMmE3MDRjMWE2MWY0NWNjYWY3MWRmODYxZGE4Yzk2MTNiMDkyNmRkY2E5NzVhODhhOTRlOGQ3NzUzOGY4NWZlMTE4ZGMzMGRhYzcwMWUwZmMxYWZlYmQ1ZjAwOGY2ZGVlMTQzZGE3NDk1YmNkMTY0ODFiMTVkNzY3OTFjYTY1ZGEwZDM0MWMxMTE0Y2M1MjZkYmI4YmYwNjFmYmZkZGZhOTNjZTQ3ZGNmNTQ5YTQzNWU2ZTU1NDc3MjAwMTAxMmMzMmJkN2IyZGI0OGU1YjgxYTc4NjRhNTczNzEzNGE1OTBjYTVlMWQ3NGQyYWM3Zjk3ZmY2NTIxZTZlYTMzMzU3NWM5YWUzNjYzZWUwMzM0NTVhZGE0OTQzMWUxZmQ0ZjY5NWM2OTQzMmYwMjI2M2U2Yjg1NmEzODhlNzVhMjllYTI0OGE3ZmM5MzNkNTNkZDQ3YmMyMzczZWNiOGE0OGQ0MmYxM2E0ZTE3ZTZmODg2ZDU2MjM5ZTFhYWU3ZjVmZTY5MDZlZGE0YWJkMDdmODVkZGZkNzczOTFhZTI2NTdmNTE3MmFjZTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8dRXuMFN4bbydGn_d1tJXF5DPhIQMy0pQTMl1g8fM0iCdiPOSjNYOSa9lrPIOHy8sFXsFfi_ff5Z8v-20TWBbA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105920_66_2438_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.046Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkRjTndvVWJqNVJwMFdTdjlCV0h2QXJDTlZPeGFHM0k2WXZmd3pPZWpKbkxrQTBiY3JPRFpiTWdaSFZ6SXRrekNiYkpoWk9OT292S3U5VFN6OWp5akJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5MjBfNjZfMjQzOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTczZTc2MDY1OTgyNmFjYjdjZTg3ZmQ4ZGUwNjViMjMwYjViZGY0ZTIyYzc2MjI4NDA3MmU5ODkwODc2OTljNzcyNmMwNzAyOWI4YjY1ZTM4MzBiZDY2Y2I0NmQ1MWQxZTVjOGQwYWZkMzlmZjMzMGY2OGQ1MGIyOWIxNGI1ZDUzY2I2ZTdlMWRlNmYzNDNkMjBlNTg5YWIyMDNmYjUzYzA5OWYxMGVmODlhMDk2NWM0OGUzMjdjNTE0MWQ4NmJkODc5ZDU5YzYzMTA4ZWE4MzMwM2Y0MzI4ZDE0YjFlMTYzY2YwMjIzODI3YzA3ZDkzMTllOWM1MzhiZWFiYjJmMmNjZWM3YmE5YjFjZjRmMThiZGIwZDA1NDVkMzk2NzY0NzcwMzJjNDQ1OGRhZDRjMGE4Mzc0NWI3ZjcyYjFhZmZmMDAwNjdhZDZmYWU1M2Q1ZDA0MWI3ZTY5M2JjZDcxMjMyZjEwZWM1OGNjZGQ5MmM4ZjM3OTBiNmE3MzRlZjA3YjU5ZjJkMzQ1MDEzYzJhNDU3NWQ4OTE5NWZkZmQyYmZiM2E2MDlhODUwN2JjNWQ5MzUyNWY1ZjA5MzJkMjc4NTJhYmIyZDhmZTQ3NjU2YjVmNzcyMmFmZmQwN2UyZDk1YzU5YjJmNjBlZTYxYTE3ODk3NThjMjFjOTQ1NGU3NjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4GuUvecteDkg-5NRQgAFfmEz1oa1WcpapvB_Y9RN6Ty6YM5Mr9z-MyWkac5RIoa2CrIxP87dSaHhmDXZ5Ciz3Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105920_66_2438_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.049Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InF6czlrSlk2ZHhJclhvUkpjWHRYSnRHT1VQL2ZOaE0yeHpabkphMWxIZVB0WmJjbGY0Mms0bmVsd2lqVmtoMEFIZkZOYlpmdjNNSEtCNTE2enBMd3RnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5MjBfNjZfMjQzOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjMyYTkzZmYyMzU1Y2I4MmZmNmJjZWZmZjg0ZDU4MmJmZDJiYmFkNzk3OWIwZjQzMDMwMDQ3Njg4ZDNhZDA3YmNkZDc2OWE0MmZjN2ZmMzZhZGNhNDIxZTU3ZTI3Zjc0M2M1OTM4Y2FiZDI3NjBlODkyMTkwZDdkNTQ4ZWMyNWM5ZGZiODk1YWZlZGE2NWVjODBiZGY2OTEzZWFkY2FlNTNjYzBkNmI5MjFlNmI5ZjA0NDIyODBjNjA1ZWFlMjIzYzRlODI0YzAwMGFlMjU2NDE1MjQ3ZjQ5ZTgxOTdmOTJlODQ1OTU2NWM4NjM1NjczMjdkYjg1NGQ3MmJkZDkxYWE2NjU3YTgyNGM5MjQwMjhjNGFiOWI4YjkzMjIxOWVkY2Q2Mjc0OTcwMjRkNzM4MTZjYjVlZmRkYTcwMGUyNmQzMDNiZGNhZTg2N2EyZmFiMzNjYjE3ZTc5MTJhYWYxOGQ2NzAxMzZhNTYxYjFjNTI5NjU3OTQ1NGI4ZmVkZDg2NmI1Y2Q4ZWI5MzVlNzVjNDc1Y2RjMDgxMDMzYWY5OGRiODE2MTRlNTM1ZTc4NWVjYWM5NjM1ODkwZDE1ZDg2MjVmOWU0MzM0ODM5NmRmYmE2MTNlNDU3ZWRiZDRkOWMxODBmNzM1OGM0ZmVkMzU1N2FhM2M2ZDQ0MzhjNTlhMjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OuoPVyCababbEXU9gfnuRm9kcngDQ9a3msMkRwQ17h_SWCx5rRX0T_0_AF4JbT4Gj0Dm1fKxFXlVRWmNwcJ6tw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105920_66_2438_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.052Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjR4aWZQWUliOExDTkd1Q05uK2RCRFhzK25qM1lJSkVyeWtBNzdVV0FnVm1UQkIrMlhicEN3WnhsSWtJT0c1UUdwZVNFVWRSMFNKMGg3aEZXV2pNb3R3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5MjBfNjZfMjQzOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGFmNmI5MjE5Zjk3MDhmYzQzZmI1NzEwYWI2M2QyNzdjMGI4NzFjYTAxY2FlZGY0OTdkYTVjMTVlOWU2NmFkNTZiZmZiODAwZWE2NmUyNGNlZjZiNzk4ZTUyNDgxMzBjOTU5ZGMwOGU2NDAzNGExNDk5YzA1YzU4YTVmOTk3YTM2NzAzOGM5YTBlMjk4OTMxNjM5NjIxM2RmOTM0OWIwMTJjMmMxODgwYzhjZTBmOWMxMmZmNDU4ZDc1NTY4NGNmZWNkYmJjZDQ3MTE1MTA3MzRmMjk5Mjk1MDE2NzNhOTM2Mjk2MGEzYmRhN2JiY2NlMjQ4NGExNGEwMWYwOTViZWFhY2EyZmUyODkxMTkzYWE4MmZjYmVmOTEwMTQ2YTEzNzk4NDRlY2ViYTliZTMwN2FiY2I2MDQwODM5NzIzYmQ5YTYzNjBiOTZlODFiNTdlMWI1ZmZhYzcyMjk2ODk4NmE3YjZkMDA2Nzc0YWYxMTRhNWU3MGYzYjRmMjAzMDg3M2Q4NDI2ZWY2ZTllMWQyZTI3OWZjY2FkMGIxNjY3ODAwNThlNDAwNmRiODg5M2Q1ZmQwNmI4ZjBlZTE0Y2ZlNmE0YWJlZjcyMTUwZTEzMjI3OWM3M2U5NmUzNGZlZDBiOTBkOWVmOGQwZTE0YzZhMjUxOGE4NzY5NDYyNGI0MGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aJhID5GW3KmIzI0LpTpgns8UGJswgvFvSJA1W8qs8MwOpYE-ck2STLqH9HmSIO6a6rAhlYiPx4_OfVYVwN5CIA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105920_66_2438_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.056Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlpZbFloVzdHbUduTE9VMVpSaTQ3VFBLdWVaa3FzY0h2STBnYnBUSlUzck9PdnIrWnFWRXQzOWo3Uy9vNldBSmp4TWh6VjBTSUFmMlZvNlYxLzJLdUpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NDlfNzNfMjQ5Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjY0MjQ1NGM0MTM3YmNkNmU4MjY0MDUyOGU0MjIyMjIwNjc2ZDkxODc2NDUwMDM1YjYyODlhZjc5YmMwMGIxNWQ4M2Y2M2QyMmFkOGRhOTY4NTNjOWExYzY0M2VmMDY3N2RlMzk0N2I4MGE5YjAzYzk3NGNiMzI1ZGQ4M2EwNjNlYTU4YTM1OTQ5ZGViMzZiZGY4ZmI5NTJlZTE3NzczNjE5ZmYyODc0NTdiOTIwMzhhYmM3ZDlkMDRiNzUxZGIxZTFmNTYzODM5YzNkMzFhN2RkZDEwYzI1MjlmODFlNDY0NjhmODRlN2I2NWIxMTcwOGM5NTIxY2VhZmJhMTQyOWI4ZTY3ZTVjY2YwOTk1MzFkZTM4ZjMyZTA4NDkwZTJkMmI4M2E2YTc4YTliZGYzOTBiNTkzNzI2OWU3ZjU2YmNmZTljMzIyNTU0NjFjYzQzZjU5NmIzMDhkOGE2NTk2MzhkY2E5YThhM2JlODgxMGZhNzBkMjBlZDQwMWVkMzk5MmY1Y2RmOGM5OGU4Mjg3Mjk2ZjcxM2QwNGIzZTdiZGY5ZTFlOWVjOTI3OTRkZDBhNzRkZmZiMzI0Yjk3ZDE2MjU5ZTAyYWJkYWQwZWY5NzBiOGRlNWRkY2ZmOWY0ZmQwM2E0NzlhMWJkZmFiZGI3NGQwNDI0MjNjZjk0MmJkM2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZB_a1D1OxxQMUlILpVzgLg7UJ3uV51HUZUAsbT-GtcAHPjJhj7HgF98eYiH8fUlB_Igw51DdG11O1bv7i64Byg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110449_73_249c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.059Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpCbTlQVzVSbFROWllDNEVPbytydkVRa2U1RkF4TnNRbGE4dmM5N0gvMDFtcGFibFZCQWdWVzFQOEVNeXRCTUZRcXpoZWhqekpOY1I4b0lmaW5iYVpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NDlfNzNfMjQ5Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTU1YzYyYzdiODU3OTQ4MTM0ZDY1ZjgwODkyYjZiYzc5MzY1MzUyN2I1YjljYmMwOWE5NTNiMzhhZjQyOThhYTMwMGRiYThkN2Q3MDc5ZThkYTc4YWRmM2MwYzdiOTEyNzdjNDEzYTllNzIwZjVhYjJmM2ZiOGY3N2FhZDdmZDI2YTZmNmMwYTU0MzRhMmM4YmZmNjc4MWQ3ODVkMWZhOWQyMjNkYWNlNmZlODVhMmU2MzIyYTZlZWVjNTkxNTg0YzBiZDIxNTIyY2I3Zjg2NDYxZjIzYzFlMWVlNDQyY2Q2Yzc2OThiY2RiMWMzYWM3MTI2ZmIxYjVmMjczZmQ0ODNiYzEzMWE1YTY3N2Q3MzA3NTkzZTk1NGQxMDczNThiNzk0M2UxNTFiMmQ2ZjBkMTY1NjA4OWYzYzVmN2I0NDFiYjI5YjljMmU0ZGE2NGRjYzYwZDJkYTY0Nzg1NDIxNmRhMmRhODBhYTI0NjJkMTFjN2I2ZmViMTEyNjNhNWYxZjM3MGE0MTczNGVmN2Q2MDc2NWIzMDFjNGQ3ZDQ5M2ZiZGI1Mzc2MzQ1MTQwNDdlOWUwMzUxZTk0M2M0MTU4N2VkM2IwY2M1ZGFjNDIzNjA3MTk4YWIxODhjZTE5ZWIxMTI2ODVjZDVlMTY4MWUzNTQ3ZDMxNjBhMmY3ODY5MGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lPdW8BKXWBi6dhlScI6hoVQwm6jcqxc_2rAwBDvNrz3lMuUf2ISsTNFiBoCZ8pxiPUyeak9kfk6Bb7IIrt0COQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110449_73_249c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.062Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImdZWUlKSmZIRlhOejhWWHRGckJzVTJ2bnBHSS9ZaXI3d0VOVnFyZHdBSDZrT0VqTjdsRlB4Y1NQalZxM1hhWWV3UG9tYlZaWDZQVTdLV1o5cEhvQkFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NDlfNzNfMjQ5Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTYyNDAxZGQ3ZWNlYWVlM2M4YmQzYTExNWJkNDAzMjVlNGRmNGJiMGYwYzYzODEwOThkNWM4MzY5YWY1YjI2MDQ4OWFkOTg1YWZmY2I0YTNkNTEzNDA1NzA1YTI4Y2Y5OThjYTJiY2M1YmEwMDJiYzY3MmMwYTVkNzI4ZTFmMGU0ZTU5YzBlYmJhZjRkM2IyMmNmZWIxNWNhNWExZWYyYmU1ZTNlY2E0MGI0NzliZWY5YzQ0OTI2NDNkYzNlNTczYzFhZjljNDU2NjRjNTM0OTY4MGM1OTNjNTcwYjc1Njg5OGQ1OWI2Yzc2YzE5Yjk3OTY5ZTI0ZjhkZTIzYTQ5ZmNjMTFjNmMxNzk5ZGMxNzM0M2FhZmNhN2YwMmEwZTZiYjZiMjI3NTJmZTIyNTVhYTg0NzFhMGI3MWNjZGY4YzhmMmE3MGQwNjlmNjE4MjU0ZjhiMDY1ODdmM2ViNmZjNGRlYWY1ODkyY2FkZmRiNjJjMmI2MjUwYTljOGU3NzJiZTdiMTY0NDM3NDI1NGEyODk2NmRkZDkwODY0MGY4MmYyNzI2OGM4Y2JhOWY1NDY0NTM5NjlkYTIzYzczNDVjOWE4OGJkNjRjM2U0ZmFkNjg4NjdiZjc2MmYxMjIxYjJiNWFiMDQzOWVmYzI5NTg1ZGZlZGRhOTlmNjIzZTMxYTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3RARkXJ6p-Ah6EFG2rb4KSKW4CDZB1kdvbx5Z1ieR75M0YE5P7TJJG1xNin7gLEMetv7ZGTE42HdvS1PXmIeIQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110449_73_249c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.065Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlZiYzhCZE9NcnljVUZ3SzBEcE82OC9ZcnpQcThOUU4zV3JiTXhzSmtxS3VJWW5RbmVSVTd3WjJIdnNQSHJVMjhZdk9US1hwYk52Rng1ZStBUHNPWHpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA0NDlfNzNfMjQ5Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTc1ZjU1YjVjZGRkZDg0YzUwODBmZDA0MmNkYjYzZWEzOGM0OWQ3NDdkNGEwOTZkNjNmOGYxZGNhMjIzMmE3YjljOTNhNGNmYmJlZTYzMjYxM2NjYTNlODhlODFlYTQzNDQ5YjVjZDU5YThkOTdhYmU4NmY1OGViODU2ZmViODA5ZDg3NzM1NjI2ODg5NGE1ODBiMWU3MDM1M2Y1NjQ5OWNlMjNjOGViMjExZWNiOGMyNmVkZjI4MjE4ZTMzMzkwMjcxMDAzYjFjYWU1OTc3NjcwMTU3YzRkNzYxZTRiNGJlZWI5YTc0ODg4YTk1NzEwNWNiZGU4Y2Y2ZjE4MjhhMWRmMmZiYTExNGRkYjY0YWI3YjFjNjM4YmMxMGFlNGU4OGRiNjU5NWM3ZjMzYzJjYzAxYTk0ZTRlMDZlOTBjZTUyZWVkYTc2ZDM4N2MyNzZlOTliNWUyMDZmNjVkNDc3NWVmNGUwOTgwYzNjOWVkOGEyMWZhMGRhNjk3NDQzMjIyYmM1OTUzZThlZGNiNGE1ZGM2M2M1YjhiMmY2MzkzYTMxYjE0ZmYzNWIwYjUxMzNkMTQ1M2JmZWVjY2M0MzcyYTVlNjViMWQyOTg3NmNhZTE0YzA3M2I4MTQzZDIwNWIzZGJmNGMyMTA3MjhlZjdmN2M0Y2Y0NTJiNGM3NjZiMjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UGNUyCu_Aj7k-_hrfurVEvnPUzhL7j3RdbUEChqTjQA0bGlcpEhEKYflrrD1SVVAjIt6mDpd_JVOS8WhiVRWDQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110449_73_249c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.068Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlKaFAwdHBweEdybmpCb1FMcmlibnBEVFdYZVZkYkh2UEk4eGp4SDU1N3NjbjI1MnozSjVoeWZoQ0V0aVNLNExDWkZsdlJtVTZ0dFQ2NW92cENxR3hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzNfMzBfMjQ3YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTM1OWZkM2MxZTA1M2IzYmJjMGQ2N2ZlOWMyY2Q1NzdjYzYzODhlNjE5NjViMjk2ODc4MmRmODRhYmMyYjUxZGE1YTYzODdiNjgxOWY4MGI3ODMyMGZjYmYzOTVkOGYzOGJmYzNhNWMzNGEwM2M4YWZjNmZjZjhiMTFlZjI0ZGNmYzIyNjlhMTYwYWFmNDNlOTE5ZDhkOTEwOWUzZTFjMTk2OGUwZmQyODg1NDI5MzQ4NzdiY2E0OTBlNDMxMmRjYWU5YTYwNzg2NjdlNGEyMTNkNDZlNDc3YTcwNDRiNmQ2ZDNkODE5OTRlZTQ4MTI2ZWRmOGY2ZjQ5OWU5NjM5NDExNjMxMzc2ZmE4NmM0MGIxNDVlMjQ5OTgwMGE0YjExNGZkNmM3NGYyNTgzMGVjNWU2NGVmNmM4ZmJlMGE4YmZmMzlmODk2Y2Y1YmRlODUzMjkxZDAwMTQ3N2EwNjViZDY2YjNiYjRiN2QzMzdlYmI3ZjYxMjY2YjAxNmVkODk0NTRjZTVjMjU3ZWU1YzYwOGRhYTE5MWM5Y2JiMGU4NzU3Yzc5YjI1M2IwZjZlZjdhMmNlZDhmMzAxZjczYTYyY2M4NjcyZjNjZjE3ZTI5MmNiMGRjYTQ3YjAwMWVmOTU0N2M0NzU5ZjhiNmMwYzExMDI2ZDc1NDdkYWZiYjIxMDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4t96_J5XVNpu7uohmUQTNT7Cs_J0TCY4vGTBZU1oYAzqQUZ3xJDMptrabN56KToMJjakECLDBfhQL8CQ4sLaHQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111233_30_247a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.073Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InRoMDlTbzlQZ090dGdwSUU3dGk5UzZYSDFtdG1ycEpoT3hLYjJMckRkMFF4elozUGVwTldzdlJwMmM1S0Q3bjJGVU9vMlFYM3JzUHc0UzBOeGhXeDZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzNfMzBfMjQ3YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDNiYjFjNTAxMDdmZWI1NDRlN2IzOWRkOGFkYWNjYjQzMjIwZTA3ZWQ5OTQ2MmVhZDg2ZGMxMzhjMmE0NzE2YzIxNmY0YjI1NzRlM2M2ZTA4ODAzNmNhOGY3YjRjMmU1M2Q0MjJiYmFiOTg0ZmFjYjU0YzY5ZDY2OTMzZjQzOWVkZWExNzg2NjA2ODYwMzcxZTdmNmU4ZDU3Y2I5NjdiNzhjMjQ5MWQyY2U2ZTIxMTlhZmRmYmFkMjQ0MzA2NzIzMGZhMzE3ZThlYzA1YjM0NTIxMDI4MzZiMjhkYTZhYmFiMmUyYWRkNGFkYzg1YzhiODY5MjdmMTZiMDc5YWZmYzE5NzgxNzc4NmQxOTkyNjRmNTY2YjEyMjM2NzMzZWJjOGUxMWY3NDJkN2JiMTBmOGE4MWFkMDFmZjkwZjEwZGRhODAzMmYwMDUzMzJlOGM0NzljYTBmNTdiZTFlZjJjMjdlNmYzZjAxOTAzNWYwZTc4Mzc3YTJlOTg4MWM2M2Q2NzkyODAyZWQ4NTU3MzRkMGU1ZjE1NDFmZDdlMGUyZjg5ZDhiNjMwYTA5YmViMzI2MTEyZTVhOTdjMTBkNzc2YTUwMGVhNzllNmI5MDIzZTVlN2ExNTdlZjBjOTI2ZmI1MzFkNzkyYTJlZGQyMzMwZGFiZmZhYTAxYTQzOGRhNGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.h7dqiViqU5hG2tXtLY-zqS8uDQoym_Uw3OLJSFf0Zbq-rxFEYe18xi1RYY3j7UYNjhdwFyJXo9ORBTO-SZeu2Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111233_30_247a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.077Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikt3dFVQUmZaNm5aUWdPSDlUMThhT2JCeGp3V2RhREdPUkIrQWJ4ZjMzSWQ0TThlVEpQbXJCQ1IvRmVZL2dUZHBVbnR4ZEpCenRkYlNmeUlhVlFENW5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzNfMzBfMjQ3YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2IyYjQxZmRmYjc4OThkMjUwMDZkYjViNzk4ZTNhYTEzZmMyYmZhMTMzOTVlMjdhOTYxYTljZDUxYzhkNDM0ZDFmZWYyYjczMjZhNTA5ZGU4YzExNWYwZjI1ZmNhNGQ0MWJmMmY2OWJkOTU1NjA4ZTg2MDdiNmQxNzFiNjhmMDhiY2NjYTBmNjUzODdlZWUwMGI5MzIzMjdjNDdiYTZiYWU4MWNlOTk1MzU2ZTVkYWIyNjVmMTkyMjUyODA1MmQ4MmIzMjQzOTVlMWE0ZTQyNDg3MTE1NjJhNDE3OGQyZDlkZGE2NjdiNzc4M2E1YzMwMzQ1NDQzODczZmRlNzMzMzZmY2M4ZWUzNWQ3ZDcwMzA2YTcyMDZmOTU3M2VlM2YzMzhkYWNkZjhlMWYwZjg2ODA1MGNiMDhhNjFiNDkwZmNhNjI4MjI5NzU5YjQxMTlkMTA1YzQzZTg3ZjRmNjlmMGE3MGIyZDA3NzlkOThkNTEzZTM4NWQxNmNlYmE5NmY5NzM0MTNmNGVhMTdjYmQ0NWVjMWU2MWNiOTMxNGEwYWMwYWQ3N2VhNmRlYTZkNDYzMWIyZmM2YjI2ZTQyMzM5YWM2M2I5NmY5Zjc5ZDg1OTkyZGE1MGY1Y2MyMzg4MDRjZWJjZDYyZjk1NmVhYjYwNzkwOTllNTg5ZTE2N2VlM2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1sMraQJi8P7eaE5tl8RGDeEE17rGqXytcU3Sjqme9dXYHWCA5ONS-Dr7F3lH4Bp2UT2I4wLF7Rj4jBqLRkdHVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111233_30_247a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.080Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InlteTVtc0tyM0JNTmpIbG91ZkxheklzeWxDc1gzZm9ZWE9DL0ZnRU1heUttNmdDMEwrRm8vVkFtcndRVXRVOUtHK2JROG5vV25vaGNHRHh4UithWU1nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzNfMzBfMjQ3YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjM3ZTdlMTI0ZTNjMTk4NDBkYTc1YmY1OWU0MzBlYjIxZTc5ODZmMTc5OThlOTNhN2MyNDBhZjQ5ZDZhZGEyMmQ0M2RiMGY3ZGVhOWIzYmZjZmU1YTg0ODEyNzZmZDEyYzQ3ZDVjN2QyM2FiYmUyODAzNmI5MjgzMjE2M2YyYjhhNzk5YjBhZmE4YzkyMWExOWI0NDkyMWEyZTQ5MmQ4YzFhYTg4ZTNkNzNlNWM0NWRmNWFmMDg0NGZiYmJmMjYzY2U4OWYwMjBjOTI1NTJiZDgzOWNiYWMyMzBhOWFlN2VlNWFiMzYzZWVkNmYyZjQzZTA3MTEwYzI4MjI5NDZhNmM5ZjQ3ZDEyMDUxNDFiMDRhNTRhNzEyMzQ4OGE0YmUzY2VlMTVjMTFhMTQ0OTE4NjY0MDRjM2NiMTk0YmI4OTExYmQxZGY2MzRkMDUxOGQ5NzJhMjVjODFlYTE2YjY2ZjBiOWZjNWUyODRmY2Q5Mzg3Y2Y3MTc0MjljMTJiNGE2YmJkNzE4MjJjMmViNDE0N2QzMTU2N2VjYTQxODFiOTU3NjgzNTY0OGE5MzljMTAzNzdkMjUzMzNkN2JkOGJlNTE5YTgyYTExYzdlY2YyMzc5M2VjYTBhMzA3NGQyZTU2NDFlNGNhODk3ZjhhOGZmZTYyMWQ1NzUxODYxMDJiMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.I4Ag7VxxTnrMQJcOlN9ou1pjua-7BG8S9Cfe7HzsJdfAJzKfZVKK16uzju50fiKlvRrW3zsGjdcfby75fg5Y8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111233_30_247a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.083Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtKR0k3dlQxOXFBb2hPNkVId09jL2ViamZvQVVmQjFUekFvdm9hQWt5MEt1QUpLVk9TdU1CdHFBWWl4dU91QjJNZ3ZNM2FUM2dUeFZtRzVpYllaMWVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwMl8xMDI4NTVfOTVfMjRjOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQ3NDUxZWM3NThkN2E5ODZmZTc0OGFiYzE2MmYwYWRmZWVkNWYxMDRiZWUwMDU5NWJlMGE4MDRjMWE2NjE1ZWMyZDcyZmI0NzFhM2JjNTRkNmUzMzg1MWZlMWE4MzAwZTY1ZGRkNTI2YTA4N2VmZTA1ZDRlMGQ4ZDlmNzgyMDMzMDAzMjJjOGE1NTBlNjg3ZDNiMmMyNjA2ZGMwNjhiYzBkM2MwNzcyYTcyNDIxYTA2ZTRiODlkNGQ5NmQ5NGFlZTE3Yjc4MWU0YWUyYmE3MWFiYmJiZWMyMWY4NjgyYWUyMzY2MjUwYTQxZmJkZjMxODExOWY2NGZhZTc0YWU4MTkzOWRkNGYyMmE2ODNiYjk4NTNmYTFkZDQ3YmIzNjYzZGMzN2M0MGEwYjVjY2E2Y2YyMzhkZGE4OTgyMTFhZjViMzYyZDlkZDlhNGFkMjBhNDliYTI4ZGRmMDU5MTI2NTQxZWE3YWFiYjE1MjM0N2U1MmU1OTIxMjNiNDljN2IxNjM0MDg2NDZkOTYxMDhjZWU0NDQ2ZjA3OWZjZWViODE1ZGVhYzY4ZjQ0ZDY0OTFjOWYxOWYyMTk1YWVlOGUzZTQ3MGFiNzU5ODQzNmM0YzAxNzQ4OTVmNjE1Zjg4OTNjOGM5Y2QzOTJiMjk4ODY3YjM3ZjhhZTM1MjIyNzFlMGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cOZC3mxdrIhbIcgGXljzQGKJhgQdPVKm_MbIcOopXcPfEdGf6Cf9k15yg2mHPxVj2KR1zWX-CDrExliaDxTERw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230602_102855_95_24c9_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.086Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkQ1bDNFZytHcUlpOWN3bjVHTjFweHFyZUNEL21MejhwS3BScVJJUzFYa204cWZFbGhkcG01WCtlaHA3Nkh4NWN5b25nY2pPaDN4ZnU0VXlhWmUrSEpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwMl8xMDI4NTVfOTVfMjRjOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTJkMjljNzE1MDc1YjEzYjU1MTQzMjNlODA2MmRiMDk5YWUwNTk1YzVlNzIwYThjYjJhNjQ5NjYxYmQxYWJmZGQ1N2YxMDFjMjkzYWMwOGM1NzgxOTQwNDI5ZGNiNDNhZGIwNGFkYmZhZTZiMDFhYTQwNjYxNDdiM2I3YjRiYzgxOGE1ODIzZGVmMzUxZTc0MTg1NDg3MmJmODA2ZTBiNmEwMGYzOTExZDJlZTVkMmUxYjg1ZGJlODk1NmRiZDhjZWEzNzQ4MzIwMDczNmVmZjBmOWJiYWZlOTQ0MTI2ZjM0MzVmZWI3YzRjMDcwZDBmOTM0NjI4NmI5MGM0NTBjNzJlY2RlOWNlNTRjMGEwYzE2NjFhZDk4MDc5NjVhNTEyNmRmNjk2OTE0NDhmODRmYjMzNzE2MDkxYWEzYjgzNzk0Y2Y1NGM1ZDA4ZTY1OGQ5YmYwMjQ1MGQxMDY0MjdlMTE3MmFlNzVmOGFhNzY0YTQyMThlZDIxZWQ3MzA5ZWFiNTNiYjdiM2I3ZDhjOWM2ZTNiNWU3ZmNlZjBlNzYyNThkODZjNDk2MzFjM2JjY2FkYmQzNWMwMTI2ZTFkMDI3ZTE4N2EwZTVkNzM5NjA3NDFiMmIyMGI5OGUxYzJiNDczZjE1MDc0MGVjYzgwNDA1YjBjYWY3Yzc4MTdiMTM5OWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sUciKh3E3y1hzEz7H7wwmouKDPNH7AQeGiQlOj9_BXpJUgc3uqydL1msVteUYfKfhYRgpAKMiDFf6IdU0Rjjyw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230602_102855_95_24c9_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.089Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRqV0pHazFoY0R3Qjlzc1B4ZkErZDExZUtVdXdXVnVYeEZsOVJCQUFCVnZZVnFJWmZJcHJXU2JwdU9EOTNKcFR3TUVNN1NGMGgyLzBGaVRjODJZTnVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwMl8xMDI4NTVfOTVfMjRjOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2YwN2U2YTA2NzI2ZTRlOTkzZTRlMTFkNmVlZjhjNGZiMmFkZjJkYmY3OGRiMzg0N2ZhMmZhYmI5ZmYzODhiNjAyMzc3ZWYzZjFhNWViMDk2ZWJlNDYzODllMmFhOWI1YTg3ZWJiNmZmODNjYTc5MGQ3YTZiOWZhYzI5YTg0OGM5MzRkMTg4MmI4ZDYzNmZiNWQ2ZmQ1YTMxNTU0ODRiNmFhZGI5ZDg0ZmFhN2Y3NTI0YTNlZTI5MDcxOTZjZjFmODUzNDk4Y2YzZjFmNWJkZDk0ZTVkZGM0MmY0ZGFkYjRiMTIwMTQwY2FiNGIzNjdlYWJhYmU2ZjE2ZjdhZjdhMTBiMDU2OTdjNDRmZTZlMjQ5NThiNTM5Nzc3ODQzZGVmNjI5ZTFmNWNjM2JlMGZhNWY3ZDI5OTdjZDBjM2Q2NjQwZGNjZDU5ZGQ4NmVjYTk2MWVjZTdjZGY4MGM4ZjA2ZGY5NWFmNjU0NGUxMTQzM2Q3ZDVmZDg2YWQ2ZDU3MjBiYTAyYjc4ZTE3OWEyNGNhNGJhNzUwOTBmYTAyZTBjNGJhYTQ1YTU2NzgwZTc3M2I4ODVkZmJmMDA0YmNiOGYyMjc3YjkwM2VhOTcwZDkxZmQ1MDZhMmRhOTViMDYwZDhjZGNhYzQ0ZjNlOGY4NWUwNDEzYzU5N2Y2NTE5ZGU4NDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OpwYkXcUtQspIR-JXOixSqQFXshPypB2SNNJnGkOIQVLgfd6p4i4pcIY7G8jIHKxXCKKRP0pG74P4JzoAt7k2g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230602_102855_95_24c9_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.093Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlhvbEY3ajMvWmVOUHFFRU5td2ZWaHYxOWJUKzNacDFHdTMrS0JmbVFoY2xPQjlzQy96bnN0dTZqT3QrMGdMbFN4akJ2eVdaenZCSldzanpNT0o1RkN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwMl8xMDI4NTVfOTVfMjRjOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWViNTdjNGY4ZTBkODJiZjEzYjY0N2IzOWEyNjhhMzM5ZWZmYzMxMWZiMTM2Yjc4NjVmY2M0OTYwZmYyZGEzMjg4ZTlkMGVhYTQ0MDJlM2ExMzY1ODhkNWJiZDA3OTM2OWFmYmFlNzYwOTlkN2NjM2QxYjI2MjNiMWU4ZjE5YjFlZTBjZDQyMWEzOTQ1MWQ5MGUzY2M5MTNkODZjZjI4OTZlOGEwYTYxMzg5NWUwNTBlMmEwYzk3ZDA3NTFmMTA1ZmQ4OWVkZmFmMTQxNDVjZjA4MzBiYWM1MWViNWY1MjgxODIzNTRkMjAxMTBlOTQ0NTZkY2ExYWUzMzA0NjkxMWNjNDMwYTFlYzY1YmUyYjM4YWRjYzJkZWZiOTllMDNjZTllY2ZlODcwODk1YmY1NGVjMmQ1MjkxNjQ2ZmRhM2YyMzVhZTc0Y2RkMGI4OWE2M2Q2NzQ2NDg1NWM3MzNmM2JjNGU5MzUyN2E1NTMxYWRmODk2YTdlOTg4Mzc5Y2ZhMmFlZTU0Y2VhODBjMzkxOGZhYzNjOGY0OTY4NGE1NDY2NTY4MWY0NTUyM2U2ZDQxODM1OTBiNjNlNTQ2ODkwMmZlMDYzNWJlMDgyYmEwMjU5MWZiYmRlOGVkMTE0MmRjOWViZDlmOWFhMmEyNTdiMzVjYTY4Njk2MjM2MWVmYjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gdP9cc3wKxe-adhEm1AeZtVJFm7uUUVzj6kJfFDb_Pr78dtEYrpKSQlrE571Boi9MwzO0kDZcrMXfOad0h6Olg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230602_102855_95_24c9_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.096Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJHenh6YWVlcitQR0Nsem1wZVFETmppS0ozSlVtckFoRTJBS0plZVNnN0JTdEVieHFGWUlOZWpsSmp3dHFKNVhaZnJBZ0o2dk9hY2pXREMxc2ZRdStBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyMF8xMTAwNDZfOTdfMjQ5YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDM2YTYwMjEzZmEzNDA1MjFlMTgzNThkOWU0N2I5MWEzNjMzODNhNDVmYTAxY2Y1ZmRlMDFmYWY4NDE4NmYwZDY1NzkxMjdkYzQ5ODUxMTQwZjRhMDM0NTlhMzU2M2I0NWQwNzdlNzQxZjA0NzI1NzdjMWFkZmVkZTdiMmI5OGEyNDc2ZGMzYTVkOTZhN2YzZWJjM2U3ZTE0N2I2MWY0MGJiZDdlYjNkNTQwN2FhMmYyMDY2NTQ0OTg3ODE0Njc2ODZiZGNiZTFkYjA1OGM3MmFhNmYyNDQ4YWJmZGZmNjA2ZWY0MDRiMGMxMzcyY2E2M2M5MzQ2NDc1OTBmNzRmYjk5NDQ5YTMwNTFjY2YwOGY2ZmZlNmMxNDc0ZWEzOTcwOTVjNzQ2YTEzNzAzNjZkYTQ4ZTkwMjczMjY1ZTA2NmYwNWE2ZWZmNmFlZjIwNjMzMWZiNjNhYjI0ZmU5NGY5OTk0ZTg5Yzc5MWU1N2MxZTRlMjExZjRjNTJmOWFkNmI0MjAzZTkxYTBlOWFkZDU4OGRhYWU1NWNhODJkYjkzMjk5ZGNmNTQ1Y2E0M2ZjNTQyMGU3ZDY1NWY4OWYzNTQwZTJjNDgxOTg4YTU5OGY0NmIyYTdhODg5ODdjMjM3NmQ4ODQzYTJiMDFhOWEyNjBkNzQ3NTM2MzU3OTA5ZGI4ZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sZ3GoPCKk2NUblrFgNs34ouAEKfcv6LL98RRPQS4oMRLyMQZ56fP2up8vHS5J0d2eM163NKk8_ppllF0_3RGBA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230120_110046_97_249a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.098Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InEyREZjcmtMKzVYQVE2NUgzYWsvdFlTTWRJYlV4bkNNa2FSYWpNU2NmZlUwb0g0WEtQcWpsZ2JXeit5WEJ5OHZSU1V6ZUlJbkxscUMvODV5N21xbGlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyMF8xMTAwNDZfOTdfMjQ5YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzk0ZWYwMjJmNDYxYmY2MjM0ZGZiOTAzMjllYjE4YzFiYjg1MGIwNGVlMDQ2NjJhZTdmNDc2NTUwZWY4Y2JmNzFlMzZiYjExMzNmMjRiZGU5M2UyM2Y3ZTMyZmExYzA0NTRiZGQ5ODAxNmUyOTQ4NmMwODI3OGE4N2ZiZjFmNmZiN2U1MzhiYzVhMmYzZGM2MzI0NDYzMmU0NDhkZTMwMGZmNzcyM2FmZDk1MmZhZjNlNDE3NWM2MjhkNGE0ZGZlOTBmODgwOTBkZmI1ZTQyYjJhMjdkMjQ5MmExZjI2MzNlMjM2MjAxOTkyMmZkYmI0MmI5MWE3ODNhNzcyMzJhNTYyY2EwMzM2NzhkNDhjMWUxZDQwODRlNDMzOTI4Zjc3YjRkYzZmN2ZjOGIzNzUyYzJhZGRlNDFjNDdjMWVmOWU4OGNhMDRkYzUzMWFhYTFhOTJjMGIwNWY1YWVjZjA3MjY1NDY5YTI2Njg4ZjM1MGVjZGYxMjFhYjM5NzJjNjAxMzVlNTBmMDljYmUzOTIyY2VjY2NlY2M5OGQ0NDcwMThiNjdiNWNlZTk1ZjgwNmQzOGQwOTZkMmVmNWM0ZDE0ZjI5YzQzOGMxOGFiMDhhNzA4NjljYzk2MGM5NGQzMDFjYjEzNTcyYTBkOTIyMTVhYTdiY2IxNmMxMzNlNGEwOGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HfKAlLsWgDo1be3pd7jPOuR-mfCDvhhd-dr09J_rEbTA27rt0nIii8iO4sfJpjhVCTdL0-S3ECEf-hfs6BM09g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230120_110046_97_249a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.101Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJMOXdCTTREcHEwRmk2OUFibXpXNncrdXZjWDBMeFZGVHp1Nzdlcy9TKzJGN04vTndpOElMRlIrdjN3QmUyUEZEaFV0S3hZYnJxbjAyOVVBOG5tNWR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyMF8xMTAwNDZfOTdfMjQ5YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGNlZjI1OTBiNjg1NGI0YTI1YTMzNjMzZGY0OWVhNWJkYzc5NGJkMmU1ZWUzNzU1OWJiNzQ3N2VmNDQ0NDE2NjM0ODlmNTg4NTI5MzgyYzNkMTMwMWRkMWQxNDJmN2Y0YTdkYzBmYjc3NzdlNDA3NmQ1MTViMjQ3YzU1ZGQ0YjdjNGIwNzBmYmMxZDE2ZTEwNjEzNzc2MmFiOTExMDVlNmQxNWY3MmU1ZGY5Y2ZhYWM0NWNiY2VhM2Q5MzY0YjI4YTE1MzIwZWY5NDIxNTU3YWVkZDE5OWRhMDU4MTkxNWQ3ZjUyNzY3Zjk4ZGZlYThhOTZiMDcxOTk5MjIwMzE4NjQ4ZmQ0MzE3MDU5Njk0ODA2MmU5YmUzYTJhZmY1ODY5N2U4NTVlZjU0ZDRjZTZlOWE0MDE3OWEwNTM5ZGIxZGU4YjMzNTkzNzIxMjYwNDdiMjkwNTdhN2U1OGFhYjM0YTg0Zjk0YjQ4NjJkOWM3N2U4MmNhZDc2NjM1NzIzNGFmYTVjNjU1ZDIyOTFlNGYzOWJjODZiYzQ2YzhlNTNkMGU1NmViNWIwMTBjZGYzYzhmZTBlNDhjMWNiNGFiZTA0NmRjZGZmYzFkM2MyNGNmZWU1Y2IzNWZhNDQ1YjJmNjk4NTQ0MmM1MzFkZDRhODUxZTYxZmUyMjUyOGQ0NmE2NjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0R6gtuC1o_zxxusnCvBq_jAvnmh-hHiOsDBoShcEwUa89pHx3qakFFyANb86WHrKQ32ZB8588aCqYYyhNhORzw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230120_110046_97_249a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.105Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlVyeDlIV0dvQTZ3K3hZSmVvYm9FS1drK2kxMVplbVJXeFpoZU51Wlp0di9sQmlQZHBxZkozWmYwQkh3aFZWcFhOSFh2Wkk1SmRvZjR0Q2Q1ekdJVjRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyMF8xMTAwNDZfOTdfMjQ5YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzhlMGMwNGIwOTAxMTU2NGRjMThhNjJkY2Q0Mjk3ODU0NDVjOTgxMTBkYjRkZTRlNDBiYTliZjlkZjVlNWExNGI1ZWQ5ZWFmY2RhYTFmYjE4NWY5ZGNlZDdiMWZmZmNmODdlZWIxNGI4YjEzZGFlZWY5Nzc3MjYxOTliNmE2NzMxZmM5OGQ0YjUzNTAyMjhlMThiOTFjYWFlMjk5ZjdhMDhiOWZhZDNlMTY4MDFjY2I2MjI1NjgyYWMzOTlmNDJjNjhiOWU5YjRhNzM2ZmZiZTMxNzg3OWY2ZjAyN2M0YjVjZjBmZWY4YzRiYmI4NTJkYTE5ZDFiYWFhNWZlY2VkOTM2N2VkYmUxNWZjZjI0MWYyODcwNWZjMmY0ZGFiMjFhNDU1MWU2YzQ5NDY3YTE2MGI2MTBlZjJkMGYyMDlmZTVjZjFjMzQzZGE4NmMxZTlhNzFkNzY4NjZkN2UxYmZlZDYwMzc0NmY0YzNkMjMxOGRhYzVjMzQ2YTU4NWIyNTdmODRjNzdmZGRlNmE5ZjVjNGU2MThlMWZhNTZlMmJmMmVmMjE1MGY4YjBiZTRkOGNjMjRkNGU2Y2FkZjNlYTU4ZWVhOTdhZDVhMTBjYzBmMjcwY2VhYjgzNWE4NzliN2ZkYmM2MDdmNDk3ZjEyNDcyNTA3OGEyODU4NmM2NmRiYmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YM-LalLoSI2vyY7KJp3hLxhsF_bjUH8QI0NGNoNVCkneh5yOYSHWVFfjCPuueOF2Zrx9898vr0JkhPaLvxDSZQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230120_110046_97_249a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.108Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik42MGU5ZTRRMDYyblRsaWVycThic0VnaHlVZGp3TFBOejArTEdOeWdSYVlHU3VKU1NGd3ZYS1NpcFNSbTNDMmhVVFQ2OVZjbi96cEd0aUZtZkR5eFpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxNF8xMTEwNTVfMDdfMjQxY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGM2MGY3ZjM4M2U3MWJhNzEwZDg2YWY0NmQ3NDg5NzkxYjljN2FiMzU1YzU3ZGIyYzk1MWQwOGIwY2E1ZTdiMGY4NDUyZDJmZTQ5ZDYzZjlkNjAzMjgyOWM3ZTY2MjVkMWFhNDQxNzVjMzRhOGZhOTJkYWYyNDg4ZTdkODkxZmRhM2I3M2I5ZmU1NDk3OTY2MDE2MjNjNTFlMDBkNTU2MTdmNzcxZjE2ZDA5MDlhY2E0M2VmZDU5NTQxNjBmYzA2NzE5MDdhNGNkZjc2NmYwYTYxOGNkMzk0MTJiOWE3NjEwYjMwZGM4YmIyMTk2Y2RjNWNkNDViMDE1NmU4MWQyOWEwYjg3YTc5YjY1MGY2YjkzM2MyOWM0MmI1MDNlNTRjNDRjYTQ3YTdjZjY2NWZkMzJjZTg2NDQyYmY2MGY3ZGM0NWNkMDAyZjMyOWUwOTRjNjUzZThiMDBjYmM1NTQ0NTBkY2NkMTFhNTZlYmFhOWIzMWY0YTgzM2M1NDc1YjkwN2IzOTRiMzE0ZTExZTM4NzRhYjhhZDU5ZDFhYTY5NTMwMTE3NWM5ZGY3ZGFiZWVhYjVjMGVkNmIxOTE2NGE0ZTg5MmZhYTI5NTFhNGI4ZTgwMTBmZGVjN2M0ODFkMzFjODZiYzU1MTRjNWMxODkzNmVkOWY4ZDI5M2MyYjYyMGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JwNjcNdnl92RH9cNTEvYAmudIuzXb_UIzoFIDuEP1kOXbNNQ8PVCNZBmwQmDRErrylC5Xp2TkDKLsMCLCTw6oA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231014_111055_07_241c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.111Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlYvZmZHZlJHaDg5U2djZnFqVmsxZHE0UCt6U0lIZHhsc25KTHlTdjJWckZaeFZwMjBvN2Vwdjd1dFhDYnNSY0tqUzVKZXYyQ0UrU0o4aGdJK1FPSVFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxNF8xMTEwNTVfMDdfMjQxY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzdiMTMxMzM5ZjE4N2NhZmEzNjgzNzJhZGY2ODhlNDk3YjhmNDMxZTNiMTc5YTRkMzYxZDg3ZDNlY2YxNTExNDdhMWJlMjc1NzAyM2ZiYmMzNmQ4M2FlMDdiYzExMmZkNDRhOWM2MGMwNWY5NzU0MTNmNDJjYjYxNzU1OWViY2E0OTU1N2JmNDg0ZjRlM2JmNjkwOTFiN2Y5M2ZkNWIwMDhkODg5NWNmYzRmYWM2YTdhMTc4NmMzOTNlNGVjZDk1NDZhNDkxMjBmZTkxNDU0Yjk4ZTA4M2JlODMyYzcwMGFlM2FiZDE1ZDQ3YWFkMDMwYzk3OGQzNTJkYzNhYTBlMGZkM2JlYTlhMDBhNThjM2IzZDM5NGI1NjIwOGM1NTQ5ZDkxNTc1NmFlZTE4N2M0Y2EyZTM4N2E0YmVkMzZlNWU5MGRjM2ExNzcyODVjNzQwMjk5MjllMmI5OWU1MjA3YjBkYzAyNWRhMDgyM2NmYWEzMmYwZDM1MGM4OGM5ZGEwMTJjNTA1NzRhODM5ZWE1NDNlMDFmODI4ZjM1ZGU0NjA2NGRmM2U2MmJkNjc5NDUwYmI1MTRhNTE0MjlhMjYxNDMzMzhjNjUxMTU2NWRmYmY2MWE1YTczNGY2NDM4Njg5MjQyODk5NzY3N2U5YjRmYTE3MjZjOGZiYTVkMmQzYjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.D5EAgz51JcNOCi-TYbm7TDvN53TZxX9uYBZFeod8gfKEhMYsiT1n8wCddBilIcu_GH6dYrw_RinJ0rLacM5lpQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231014_111055_07_241c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.113Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImhQTUhUblJKVFRLRjI3VG1yeXcrV0pUajVzeU5kbTFwNFdvU0V5UFk0Vml3YUlKVnlzZ2ZjRFl4dHVQSFFrQWZKLzZ0Qm1mRFl5MFhTN2I5bkhZS01nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxNF8xMTEwNTVfMDdfMjQxY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDFmZWMyYmI1NDg1NDQ4MDE5Yjk0YWNlMGNjMjdjMzhlNjBiNmJjMzRlNTkzNWZlYjc5ZTZkZjI0ZTJjMDczNjQ0YzJhZGU2YzIxNmQ4NjFjYjM0MTdkNDViM2UyOGYzZGEwNjYxZWIzYjkxYTc1MWFmMGJmOWM4ZDIwOTNhYzNmYjQ2ZTRiODk0OTA5ZWIxYjNlZjJhZWM1ZDc2ZjU5N2E2NjdlZTYyNTFiOTZhMGE4NTgzMGEyMzc5NzAzNzFlNGYwNWNhYTdkNjE3ODhiZGU5ZTJiNDRlNzA3YmU0YmFhNjY1MTQxZGNjMTU0ZTU3ZTFkMDRjOGNlNmNlMjg1YjQ2MWFhMzhkMTlhNmMxNmY1NGUxNzdlNGVlOWE4YWFmYWZjOWQ4NTE5OWNmMDY3YzIxZTRiMGVmMDQ5OGMwZTY1Nzc3MjdkODdmNDhjMDcyODYyN2MwNDdmOTIyYmYwNzdmZWRhNzI1ZjUwNDEzZDY5YWEwNDdiZWQxZmMyODI4ZTI0NGVkMzhlZThlNjBhZmQ4ZTFkMmM3NGZhMWE1ZThiOGZkMDY5NThjMTczNTU4ODgwNTE4NTQ1MmExYTc5MWFiZWM2M2I3M2RiOWY3MDdmZWE0YjNhZDcxOWNmNzJmNmU2MTFhODJhZjc4Mjc0MWVkYmE4OGI2MGIzMzRiODhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xqUaBkLhCpJMHL2-pLMYYixaJITbc01AfJDhrtJlHCTEK4DbYoAqOfLSItxfyucw1KRGe-w7Ug4PMjA5U2OBwQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231014_111055_07_241c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.116Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhxMlMwdG9QVDNSMGEyRUxsU1VtNkd6N3Zxd3E5bEUzbndENnQzNEhxQXJpRk9GRmUwSFJoMVBnSXRSd2pVdG8vS2U2YTNoSEludjdpTEh5SGc1cWl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxNF8xMTEwNTVfMDdfMjQxY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjBiZDI5OGQ4NmRhYzRjNmNiYjc0NGE4ZTllM2E1MWMzM2JjNTg4YzE5NDg4MDI5ZmFlYTc5MTIyOTMyMGFkOGUyNmJlMmQxNzlkNTE4ODZiMjZhMTc5NzEwYWZiMWMxOGE4MTNiMTIxMTRjYjJjNGYzNmIwZGNjZGMzN2ZmYWE1ODdkOGRkNGMwMzJiYmRmMWE0M2FiYjg1ZTk2NmE0ZDEwN2YyMTZlOWUyNjU4M2FmZWU4YTk5YTI5YzI3NWVjZTRjMDJiZjZlNzBmNjdmYzIxMzAzNDQxN2Q0OWZiOTBmYzRlZjA2YmJkYjJkMGNjNWQwZjI5NzZjYjg1OWQ3ZDAyYjUxN2U0M2U0MjUxZWQyODczMGEwZDVmN2FjZWI0M2UzOTAyNjgzZTNkNGJjOWRjZWFkYmM1ODYxN2ZlNDFhN2Y1NDgzM2QyYmU5YjMzZjVlZGE3NzAzNTY2ZjdlZjM2NjM4MmM1YWExY2NlM2YyM2M0OWNkOGRhMGQ1OTdmNTIyMjFjMzczOGVlZDRlMmE2ZjU5Y2E5NjljNTIyNTA3YmQxOGUzOThiMWQzOWU2MGQyOWY0NTEyYTEyNTUzMTUwYmM4YTE3YWNkNjYxMTc1ZTZkNzM5ZjQ4MDRkZDEyZmFmNDgxMGU5NmNlNTk1NTI1NWMxMzI4MTEwMmQ0ZDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._7cj3gQP5nBeXOCa4hsy3r77ujLj-opVaCSdBm9yOF4X5cF6cZfzqOfdNeW1tcz4tkNfjQW8achWIiL55DF14Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231014_111055_07_241c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.118Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZiajE2TUEwempNVWxmLzJQcC9SYUlMa1RzRFZPd0k0YjZiZHlBN1JlL0d4NFNEa21YN1k5OFZqdURtS0lGanBHaTNxeVRpdGlYZ1pJZnlLU3RveVpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTRfMDlfMjQ4NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjE1MzljODczYTA0NjNmNDNjZDZmODVjMTY2ZGUzNmQyOWE2NmFlMGE4ODA1YWU2OTFiYzljNmM1YzEzNTc3YTM1N2NhMDg1Y2U2MjJkNTVhNTVjODY2NWI1M2IyOGQxMTZkMWMyZTM3ZTFiNmM1OGI5N2ZiMTc0NmQxZjM1MDhhN2VmOWY2NzQzZmRlZmEyMWI2YjVkYTZjODkxZDRiOWI4YmY0ZDUzYWZhNjIzMjY1NTYzMzFlMzNkMjQ4NjYxZTBjMDQ5N2Y3NWVhZGFlNWNmMzQ0N2QzN2Y0MzkxNzVmYzRiYWM4YjQzOTg0NWU1MDQ2ZjEwN2E3ZmIyOGU5ZTM2ZTdkNDZmMTM1MTdmZDY4YTc1YWZmMWY2Mjg2YTUzZDBlNDliMDc3NTU0M2M0MDc4MjcwNmU4MGJlYTQ1Y2U2MmJmMmM1NDgzZGNkNmRjYjc5MTU4YjdlNTEwMDhlYzNhNzA3OGIwY2Q3YTYyYTUyOTdkOGYxMDI4YWU2ZjM5M2U2OTQ2NjI5NjliZWRhMWYyNzAzOGU3YjZhZDUxYjk0ZWMxMmEwZDkyZmY2NjI3MWRmNjUzNzYyZWNjODdhYmI5NWQ4YjQ1OTE4ZTdkZTFiN2NmOGJiOTZjOTA5MDQzOTY3ZmE2OTgyZDA0YTk2YzY0ZDkyMjQzMGE2MGQwM2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MgNOPlGXwASHtABQYRNqSLcweZjDoCe-NWT39CUudILLwFQK-bl75TvIffSX-Fa8ctCnMt7exNt4bqN3ZrosrA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110354_09_2485_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.121Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9QWWVsTTRFcFd1Q1VzWTE0M3NNMlpkcVZINnVMU1l3WlYzZ3U0QU82SzlicVNDZ3pIRHdxcjFlOWxYaU9tWnhQdFVRd3BOSkRFRHpHNXJkckJOanpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTRfMDlfMjQ4NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODMxZDYxMWU4YjY1MTEwMGNjZTk4NjA0MmFkNjljZWEzOTQ4ODkzM2UwZmUzMDExYTczZTMxZTFiNGJlODdhZDc2NzZjNWJjNDJmZDRhNjVkOWNmOTAxMDc2YTAwZmZmYTBiOGY2MTJlZTBjMTJiYzhjYmJmMjQzY2YzMjE5N2I1NTc0YzYwNWViODQ1Yzg1OGQ2NGU5ZDZkM2NhZDI5MjJkZDVhMGI4ZjZjNDk0OGRhMTllNjM4YmViZWY5MzkxYzljNDIyNjU4ZmViODc0NjhjMzBiOWM5MGYwMGUyNTBmNWE4ZGE5OWMxODRlNTJiYjgzZDFjN2FjYjc5OWFiYzVlM2I3MWY0ZGYyNDUxMGUzOTY3M2JhOGQyNTI5NmYyYTc3OWZjZTg1N2NmODRlNzgyZTA5MGU4NDI5OWVjYzY0NGY4NjkzZWUzYjFkMDVlNmIyMzczMTdkZmYwMjczZWQ4NjRjYmI4N2E3ZjMzYTRiZDAzOGUwOWI0ZWNhZTgyNzc5OTNjMmQwZmY5OWM1MTE5ZjBjMDRhYWM3ODkwMmNkZTAxOWE4MWZlNGE3MzhmZjQ4MTBlZDEzMWUxYWRhOWU3ZWM3YTg5YTYwNGMyYjc1NDhmMjQ0NzhjNjk0YjQ5YTUxMWRkYWE5NDVkMTQyNDdlODg2MThhZDU3NjZlODdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fo_7MwrmEdQVdVvHvxxmPIz2XvSk0gkeC9H6k8koj2z07DrY-lNva3TFlzTmGsxX0cn9cSp3L5PXWb1mVsshxg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110354_09_2485_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.123Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik53cDdEb2pkL0hsaGVlaFdMUDZYbzBvSjRIK1F1SXZ4MzdLeHc3ZW9GbkcxSHpac2dKWkNtdEp3WHNzQXNvYWJPRWVockhTTTlmTlhLQ2ovQldMMWx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTRfMDlfMjQ4NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDU1MTYwODJmNTA2MDhiMjAwZmQ4MDFjOTQyZTdkNTlhOWE1MWY5MjY3ZDUzZTQyYTUyZmQ2ODJlMjcyOGZlMGQxMzhiYTdiMWQ5NmM2YzZmNDdhNjZmMTc1YzRkODM4OWY0NzEyOGRkY2Q0MGQ3NjYzM2IzYWY1NzRjMjc4NDRkMzg1NjJmMzI4NDVmMDM3NTA3YjA4ODJiYjVhYzUzZDFhZDJmMDhjMzBhOWIyNDc1YTk3MGRiMDg1YTk2ZWQzNThlM2FiYjU3MzNlZDUzZDU1OTMzZmM5OWViNGQ4N2VhZDdmOGZjNWRiMGRjMjBhZTQwMDk3NThjMDM0NmQzYWZlOTRkMTlhY2E4NGUxNjA0MTUxYzlhNmMxMjQ5ODQ2MDM4YmI3NGRiMmM3NTY5NzNjMTJlZTFkNzU1NWE1ZjU0YzQ2MTY2Y2MxZmVlNzAwNjdmY2U1MjA0YjUyMDYyYjc4ZTAxYTlkZGM1Mzg0OGIxNDU1NDY0YjAxMjY1YzVjZTQxNzQ3NjliYzQ4ZTAxODZkNTdlZjg2YmRhNjFlOTdmNThhYTMwM2YxM2RhZjYzZmQwYzI4MjAwNDJjMjhkMGVlN2MxYTY3NTc3NWM0N2MyMzFjZGMzMDAyMjFhMzI5NThhY2E4Yzg5OGM0NzRjZDg4YTVkNjUxYjE3ZWQ0ODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Bnc--zkuNZg3QoMW7My5de8Fa0QzE-BKR4Aq45X8fGRSTE7nxnJWTYnBTvklsEImo06HmGjJehgpsojTaJsdlw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110354_09_2485_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.127Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkFjcndsZHpvN3JqaFA3VVlWRWI2ZTFFbnR3all5RjhjSThnQTR4dzZMUnhVYjZYTXJoZEpLV0J0K0lxeGNoZE8zejh4UjZFM0NmN1dUdVM3UGRVSllRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTRfMDlfMjQ4NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE4OTY0YmUxOTY2MWNiMTdjNWE3ZjExZTM2NjA4NjQ1Mjk2YzBlZDc1YjkzN2JlOWNjNDJjM2Y2N2JmNjlmOWZmYTk2ZmNjZjU2MjQ4NmE3MmUyODA4ZjhlZjU2YmFiNWM1ZWEzZDBjZjdhN2M5MzhiM2JhYjFmNWEyYmMxOGM3MGIzYTBiMTJlOWJhYTlkMjAzNzAzNTNkY2Q4NTVmYzFhYjJkNWQ5NmY3NzYzMzNmOGJjMjFmZmRjMzY4MmQ1OTRmOWM3OTBiNDczM2IzOGY0YWUyY2M1ZTgyNmUzZTAxMDJiYzQxZjExODlmNTkyOTcwZWMzY2I1NGI5ZWM0OTZmNzEwYzE0OTk1MmVjZmFmOTRhZDg4MGY4ZDdmNmM4MjBlY2E1NjEwOTI2NzVlNTA0N2U5MGY2YTlmMjQ1MzU4ZmJlMDk4MGJkN2JmYzk3NmZiYmE2NTliOTE5NDdkYmNiYWU3YWQzMTgzNTNkOTZmOGQ1M2QyZTQ1Y2NjNWQ4N2M3MzU1NzNmMWI4MGI1MDZlZDI5NGM3OThlYmM0Yzk5NDkwZGFiMzY2ODcyNTA1ZjUzYzdhNGY1ZDdhYmQ3NGEzYWQ3YjU2ZGJiZDdlOTU0ODQwYjRiODA1NDFiYzk4ZWFjOTk0YzQ1MDAxMjMxZjMzNDEyZDcyYTM5OTk0YmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6Nm9LECAo9haB8ce8lDs59MdzYTl76d3d_A7ApBBNYCtWBYU34Tiik0_6qY2d6RGr2oITqIqbnZNVXu45Onipg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110354_09_2485_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.130Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZVYWVseC9ibFhiaENRVnlGbzUvMzdLUFRVR1QwZmFxL2hpMm1Vd3RINnFTdEpQcGF1L1pSVVRad1JFMHNQZGZpYU1Ua3llcm5SeDczaXl5NVFqYzNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzJfOTdfMjRmNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjI2OTJmZTEwZWIxZmU3MmY4YmVmZTY5YTcyZWM4OTBiNjg3NWU0YzdmMGVmNmQxZTM5YmRlMjQ3ZDA5NGE4M2RmZmQ5ZWI5OTNjYWFlYzMxNjFlN2E5YTM4NTg5NzI2OTQ3ZGY5MjUzZTFhMGQ1ZGU2YTA5ODk3MzU4MThkN2YxZWZjYmJlZjc5NTVmMjIxMzg3NDQzYmFhYzU5OTkzYjNkODIyZDZlMjE1ZTkyMWU2NWQ0ZmVkZTZlYzMyMTc5MGNmMGU3YTEyMDA4OTg1M2JkY2JlNDc1OTY1ODBhZDdhY2I0OTZlNjc1NGM4MWYyYmEyMjQ2YTcxNjQ0ZWJmZWRkN2FiYWYyOWIyNGM0Y2E3YTk0NjQ2ZDhiY2E1MGUzNTBjZTEwNmNiNjM0YzJiYzY0ZWY2YWM2NTRiZjlmYzUyMWE3YjZmN2Q4ZTA3ODkzMTRmMmRmZDI1MjAwMTE2ZDFmZmQ1YmFjYWJhMzU2NTJhNTkwMzkxZmE2YzU1YjE0ZmNlN2Y2ZDI4ZjgzMDkwYTE3NWEyYTdmMjEzMjQzMzJiMmMzMjg1NmQ1MjFiZjBmNjc3NWZjYjdlMjY4ZDQwYThmMjdjMWFiYTkzYTE5M2E3N2M5MWEwOTIyZTdkZGE2OTBmZjE3MDgzMmQyM2RhNGY1MzhjMWQ0ZTZlNWUwMDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eOLKylerW6Aok8icMPT_wcnIHZJKiBY5O6-DkbuWwVV0PpyQAoeIhBGn8xRLniz1lxk8wvD43qNooFlBNm2m4A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113532_97_24f5_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.133Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im5PZ09HNDNjazFCc1FaODBHaHFFZTJ6WFk4RlkraXMwZlZUdWdDQzJWV3VKbGdGekxTUngyRkJHQkZjdE1PQlNyNVZZcjd2cDByeFEyVGQ1THhDTTZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzJfOTdfMjRmNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWM4NTgzNDdjY2FmYTc3YWYwYzg4Y2Y5NzRjNDYwMzNmZDlhOThmOWM3MTEzZjViYzM0MDE1ODc2MmEzNzQxY2NjOTliOTg2NjFmNTE5ODQ5MWY0YmEwZjM5MTQ2YzRlYjUxNjU2NTBjYmJjOTE1ZmEwZjUzZjc3OWYxYWFmMTljYTU5OWJmYmMzMWM4ZWE3MTk4YzliZjE0YjhmNzE4YmE0YTA2NGI1ZjQ1YzkxNTNmZjZiZWJhODliMmM0ZTA4MTI2MTMyZjM1NzdmYzI2NWNhYWM5ODBmNTk2ODIxYjI3NmJiMzIwMGJmNmMzYTZiM2VhM2JhMDEzMzhmMmI4ZmY4ZTUwNjlmZTBhZDlhNzNjOTNkODQ1MWQ3ZTJhOGI3NmI5NmUxOWU2NWIyNmQ5MjZkNTFjOTU3NzcyYTg1MDA2ODQyZTllYzliZmE0NzAwODIyMWJhYjFhNTdhYzhiZDlhOTFjMmE1ODNmMWQ4ZWQ2ODc3Zjk3ZDlmZGI1YmE3OGQwMjFmMTIwZmFlMDZjNWYzYTY5MjUyMjhlZmYyOTE5NTM1MTAyYWU5NWExZWY1OGI3MzI1ODZiYjNjMzFiNDBhMDFlYTA3YTkwMGE1MGQ1MzA2ZGVlYmQ2MzY4NDAwN2FiZjI5Yjc3N2UyOTcwNTdhMDg0Y2RiMDRmYzk5MTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fAa3I2MGMK6c2mf7UHx_UMXolE-6NpYwb6tUr5_7XbL-XL8y6IsOzi4Qe5iYhmOy6jSRyeXjpEIl7ax3VqEdZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113532_97_24f5_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.136Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxDOWwxV2tlcmszbTFlY3lwbHZKMTYwQlFOYzRnT1ZtaFNHTTRudndHNmUwVGpIYTQxUDEyOVhXUE5TM252ZjNqRE5aT0RRZXRGayt5b0IwWVgyNldnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzJfOTdfMjRmNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI0OTZiYTgxOGJhYWZlMTcwYjBmYzAxOTUwZmEzOTBhMDlmOGYzNjM1ZmYzM2I1MWYwODhkZWY4YTVlZTgzZjNjNTg3ODZlZjUzOWExY2JlMjQxNjI0MGVlNGI0ODEwYTkxZjY4ZmUxYmRlM2Q5YmRkYmRhZmFkNDFhYjc0YmE1ZGY4MDA3NTQ4NDVkNWM3MmZlYWYxZTMxNmU0NTk3MThlODA3ZmE3YjcyYzhlM2E2MjFjNTU0ZWMwOTg2Njk2ZDY3YWNkNTVmZDA0MTNiZTFkOWFkNTEyNzUyZTNkZjY1MWEyODczNzczMmZiMjFkNzIzM2VkNDM3MDE2ODliNTVhNjAwMjgwNjVhZDgyYWE1NDBiYmVjMTMwY2FmMjc0ZmJkNzgzZDdkMDg0NWNkYzhiMmE3ZDE5MzA1OGMyZTc0YWQ4ZDdjMzZlZDI1M2UxNDU3ZTMwNTkyMDQzNWJhMDRkMGViMGYyNTE1NmU4Njg4Yjk2YzM4YTI4NWM4MDRhNzI5OWM5MzdhMjNiODQ4MzhiMWFhNzgyMjk1M2FlOWE2ODVmMzBjNGE3OGRkNTAzMGU1OTNlOTM4N2JmNDcyMjcxMmU0ZmQzNDc2OWEzNTJmN2NlOGFmODg0ZWY4Y2U0ZjY3NWNmZDRmZDVlOWM4MWU2M2M2ZGE0MmM0MDYxZTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WpNyvRd9Z7KsCdx9-bHhMJVkV2y4r9pVu8Vhaz-F17l8Ea_5jmYZvouQA10flu4Q8cPv1WxSl2-pOtV7hFhD0Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113532_97_24f5_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.138Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Inphdkg5NHZqc3FBWWE5VC9OQTRwUG1QTFFjTEhyTHBVazdxWkloS1F3UkhmMisxMUd2em0xK2RYNmc3OG9NU0hhOXFhcmVyelIwSko4c2taZVVObDl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzJfOTdfMjRmNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzA2Nzk2MjdkMTk3ODQ1N2MwZjQyZTFiODhlODdkNGFhNDE3YjE1NDY0NTdhZmE3YzQwZjU0OGVhNjdlMTZiOTAxZGZhYTQ2YTE0N2VmZmNjNjljZDM2OWNiNGJjZWQ2ZWVhYTEyZDU2OTZmMjIwYzAxM2FmMWIyNDYxOWM0ZTVkOGViZjgxNTg0ZGNkMDY3MzQ0ZDUzMzA5MmVjZDNmODYwN2VkZmU0OGNjZTYxZGVmYzAwN2JlNDk2ODdhZjE4YWFkOGNkYzM5MTQ2MTIxYWRiMGY2MWM5MDQ3Yjk4ZTllZDYxY2FjOWZjNjc5MjM5NzEyYzc5ZWJhN2FjZDQ1YWNjOGUwODA3M2I2MDYwMmIzYzU2YWQxM2JlMTM3MDdiYmMxOWM1MGQxNDBlZjgzOGY0MTdmNWVkYWZiZTI5ZjViMjBkYjM4M2QzMTNhNzVjNGYxOWQ5Y2NiNTVhYzE2MDRkMzgxY2EyMjc2YjJhMzY3ZmRmMzU0ZjQ4NDA2YjM3MzI3OTQ0ZDM3MzhjZTM3ZjA1YjVjZTZhOTgyMjlmMWU2YjE0NTAxZmQ5Zjg1ODdlYTk3NGQwMGJkNGIzOWJhY2UxM2I3MjRmYWRjMmY5NmUyYmYxMWM2YzA4MDg2ODEwYzNkMmJmNjRkZGE2MjgwN2Y2YzVhNDNmOWRjOWYxYzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.VHfmaJzh18tiC4eib_69P9blrk3yA63hPloJxttMe_gLlna-zlIH3dX6wm57aIZvb7K9d9S5gsW0hvnS-O3Rag", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113532_97_24f5_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.141Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjVTa3ZhcEN1ZlpoNEtPRU05L1I0VFZiM09BNmtINS9YaG95Wml0QzBvZ2k1YkQrZFptSGpuZzBJV3EvclZaNDIyVVBhRlFneDBmL0tDSzEzRG85ZWVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDM4NTNfNDFfMjRjM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDkwNjk4NGYwZWI1Y2VjNmJmNTAwZTZjNzYzMGUzYzI5Nzk4OTBmOGE0NDFkMjU2MDIzYmY3OTE1MThhYmQ3YTAxOWEwNGFiN2MzN2Y1Y2JkMDg3YTlkZDI1NGEwMzNiOWEyM2QwNGRlMmU5OWJmMTJlYzZhMTExNTg3ZGRjZDMyN2M3OWFhYTA0NjI2ZTk2MTdlZmE2ZDYxZDAwNzIyNGJlYmE1N2VkNzEzYjQ5ZTE0OTc0NzMyNzljNTEwNWMxZGIwYzQxMDIzZmM0NDkzNmY1NDQzM2VlNDM0OGJhMTM4Y2ZlYzdiMTRiOGI4YzBkMDM5MDU5ZWQ3N2FhYzk0YjYwOGMzNmU4MjRiNjM5YmM1NTdlYjk0MDAyMDNhMTM1ZmNjYzYxZDdjZWNlYjVlZmNkYjI2NzQ5MmY0M2I1NjEwYjJkNjUyMmJiMTgzNWEwNTg4MmQ2OWFiMTY4NTFkY2JjNTc2NDJhZWU3OWU1NzM2YTc1ODk5MjQ1NmQwMjc5NTFlM2FjMDc0YjgzNDhlNTE5MGM4MzU2NTYyZTA1MzQxOGQ3NTkwMTAzZTlmMjJhN2RmODE3YzhiNWJlZGY5MzRlZDkyYmI2MDJhOTYyYjhmY2JiMjM3NjFmYTQxMDhkNmRjYWMwOGNiOWVkN2RhNGQ1MmE4Zjk1NGRiNTQwNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SiX60ua8sijbjKu_Mjy_XH0berjI76zYRSd5nXi634456b3_dsZLGqKJOZXnviLAgKQ8ztgasX09Vr6eE5b93Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.144Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imx0Y1ZSWmVSbXcwOUI4dng4Y3k2dnZRVkpiSFhZU0FxallCc09YN28zTFB6NWJFMUxpMDBkNGgvU1hiaW9VWXNvcTg0SkpsYUNSV1l6YXZScFZQZG1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDM4NTNfNDFfMjRjM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWU2ZTNkODRlNDc4ZTk4MTBhMGYwOWE0Yjg3OTg3YzBhNjM3YjMyMzdiMzU5NDg1YTU5YTA3Zjc3OWI4YTgyOGQ5ZDg2ODIzMWZmNmUxYmMxNDI5MmRjOTIxOWQwZDMwOGQyMzZmNTRkNTY0YmYzYTIxZGZmZWQ3YjNjMTdmMTExYjM2ZmE3OTEyZjY0N2U4M2RlMTM3YWRiNmM3MThmMmJmYjNhNjNjMTQ1ZTk3MWJlYjdlMTA4MTEzZTAzOTg4MTA4MmVhODE0YTdiNzU4NGZiNDU5MmFmNjViYmU1ZTMxYTA4NmUyOThhYWM2MDBjMWI4Y2QzOWNjZTdjN2UxMjQ4NzA3YmE3Nzc4YWFjYWExODhjODczMjExODMwNDZkNjczYmUyNGFhMDNlNzAyYTIxN2FkODdjZDJjNjQ0MTYyMjJhZTZkOTcwNmQ1YjhjMzFhZGVkZDFiY2VjODE5OTA2MjQyNTk5ODYzOTU4MDFjNzQyMmY3ODg5NmMwYTYwOGQzN2UzNTFhOWZiY2Y0NjM3OTljZmFkZjA2OTVhYzlhMDU2NjM5YWMzM2Q0MDgyODY1ZDdjYzcxYzYyZTI1M2YxNjViZmFhYWE4YTc0MTgwYTgzZDQ3NDhiNTA1YzE2ZTY5YWIyM2FmYmUxZmRhZTM5MDgwZDNhYmMwZDY3ZWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WOHQFUgyGEdldKXNHqAWNaNt_n5ZPYo_4nLSpKzcrA4LHRZco4ic6tf6F1el52O5IVGt9pI89KqoYPUFgh1ILA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.146Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJTVTh5UXhoSlk5Z3k5UXcrN0JKMEdNWmpOMWZoc21BdndLenNOeTNCZkdTd0hCVlJFRTIxaHl2b0VadkYvK1BaOXJZaUV5K2txTnZyaWtEMEdOcjhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDM4NTNfNDFfMjRjM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2M2MzE3ZGQyZWIyZGJiMWJjY2Q2NTRjZWFlODBiOWI5YTEyMzRlNTYyMTg1ZDM0ZWNjNjU2MGVhOTQ3MGUyZjg2OTg0OGU5YjlkNjY4OTg5ZjBhOGMxMzA0OTRlNzYyODE4MjA3YTVlMGY1MzE5NDU5MmRkNDU0YjNkMGUxOTI3Zjc4NWM4M2VhY2RjNDM3ZTgxM2E0ZTVlZGFmNjA2MmUxOTM5NjMyMDVhM2UyNjljMGI5MTdhZjFhNDUzZDFlZjQxMmNiNWNkOWU4N2QzOWQ4NWM5NjgxYTE3MjQ2ZTRlNjg2Zjk4NzBhYWRhZjRiZmMyYzdiY2FkY2IxMTg2OWNlZDVlNGZkYTllM2U5ZTRkODgyYWNkMWViYmFhZTUzMzVhNzM4MDc4MDE1ZGZjODExYjUzNDlhMmExYTExNzA5MTgyYjBiYTIzNWQ1MDcyYWExMzZkNjc3ZDZjN2MxMTRhMjA0ZTdlMDNjNzJhY2Y4MDNlNTZjMzAzMzI0YTEyOWZiZjFmNzVlMGIwZWZmNDA4OWQ2OTEwN2IwNTAxYTc3MDBmYmRjYTkzYWEzYTcxZmRiNTc2N2Q5ZjRlYjY4N2Y1NmEzMzRmZmRiNjRlMTFmZDY5OGNhZmIwNjE5MjdiMWRkMTA1MTM5MmVmNDlmNThlNTRkMTdmMDI3OGY2MTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lGP-7Kc6G8i9dPUysbrI82nFHEfX5bWOj_wMAXk0sVDweB2BnlzckVURowjiD-qWLVV7Y2PTnHv7cP0PcgVzQQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.149Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZGWnU1Z09QcHdFQVdwNFdPSkZRdlhaaXI5b29mTXRIa2JrSk9ZZmZTb1ROamU5em9QZjhaOXBTNFhIL2tyYnl5WWZ2Y284Ny92MTB0ZXkxTnNhMnhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwMV8xMDM4NTNfNDFfMjRjM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTI2OWMwOTZkZGNmNDU4NTJmM2FmNTkyYWJjNzgxZDNiZTcyNWFkODkzMjYwN2M3OGNjMTRmODI4YzBhYjg5ZDZiNGJiOTM1NmIzNjU2YjcyNmU1MDMzZDQ5ZmNiZjZiN2RkYTdkYTEzMGU2NzZjZjhmYTY2MDY3Yzg4MDQwNjFlZTc4MTg5OTgyOTk0OTliOGJiZTc1MjE4N2RjM2Q5YmNiZTY1ZGFlYWYzODk1ZTZiMzAwYmQyYTBlY2Q0N2ViYjhlODZmYzRiNjEzMDRiNGEzY2FmZTgwN2JlOGE5ZTFiMzgwNmY5MWNlYWU0YmM0MDVhZWRiODY2NzllNWM4MjA5YmIwMWI4ZjY4OGFkMmQ0NTY3ODg3NzNjMTUyNDNmYWFjZGI5NThiNWU1OWRhMWM3ZTgxN2VkYjE1NTE2N2NkNTU5MzIyODhjOGYyOWZhMzk4NWNiMjY2ODllM2UzYjYwZGIwMGJlYjMzM2U0YWRjMmY1M2RkZjIyOWY4NGFhYThiZjE2MjU2ZThmMTUxN2YyMzBkMzhmODcxMWRjNGUyMGZkNjkzNjY1M2E4ZGMwNGNkMTNkMmNlMjkyMDBhYTY4ZmNhZjgwOTY1NTllOGE2YzExODdiZjc3OTRhYWJjNWFhYmYwNjU5YzNmNTg5YWZiOTE1ZWQzZjZiYTA3ZDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9fsitvzPqPBr8FgKLwQiRwluZXzsgL7coIYE-rJBcAxi0_sNCGVERz3W0C3xnhvwT_NZQtdrgYRfVu6wYkdYFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.153Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBvU2xwYTNFK0dqV1BMblZ0cENJTXlUeXB6dG04NFo4Q1ZlNnJEdWc4dnRjbGJjWG9uR09TSEFORGJmR203WURHMjFjRDY5N1BWM3FZajJWdGtQcEdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI1MzNfNTBfMjQ1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmRhN2FhZTUyNzk5NjAxNTUxNjU1MTE1Y2I1MGY0MjY4ODk1ODdhOTcwNDRhYjZjMmE1MjcwYjhiNmY5ZWFjMTBhZmZiNGY5MzkwNGM4ZTlhOGE1YzliMGI2Y2UzMmZiNWNiMThjOGU3YzQyMTE5NDMxMmRiZjliMTk2ZThmNTc4MzJkMmRmMWI3MGE3MTNhZWNiMzI0OGE4MTdjNmM3ODBhZDkxYzI1NmI5NTEyYmMwN2U1MzYzNmE0N2M1OGNjNWU1ODk0MGY0NWNkMWM5MzZkMmJiNzY4ZjVlNmQ2NDZhZmZiMjc5NjU3OWQyMjc4YmU5YzEwNGNmMmY4NGIyYzUyNjA1MTZkMDY2ZWVhMWY1NWEzZTQyMmE4MDc0OTQ5ZDdiYjVmOGMwODhhNWU1YTU3YjE3N2E1MjNhZTIwMDg5ZWVjYTc2YWUxMDAwZWI4NDg2OTZhNjM3MTA5OTJmYzhiM2RkYWY3NWM3ZTVlYTlhNjYwYTkxZTViZWRiOTQxZTgyYjY1ZDI2OGU0Yzg4NDZlM2NlMDhkNDQ5NzZjNDRmYzMxOGU5MmMwYTk2ZTJkYTkwNjU2NTA5M2M5MWJlZDdlOTkwNmZiZWJhMzhjNTc3ZTBiY2MzNTE0OTczYjM4MGVlNDI1ODQ0YjFiYTk1MTIwOTg2MDlmZDdmN2YzZTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rSyiFNOYk-NAinTasPogiZjxNLAtDtxDCe6gZEeYh_XEuHXlOep8IlIBevxGq0AslC8NTJiBd8K7p1iImn_3FQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102533_50_2451_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.156Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJyT0JudngzRXJYUW9UcmNFMngvdC9HUXBZbGFFWGN2THQ3OVZLMDNVM2FGR1VoR3Q1cnVRd0s4TXpRZkJsOXluR29vRTZLbTg3bHp6UExxNEVGMUVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI1MzNfNTBfMjQ1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGY5MDQ2YmU4NjYzNTliMWEyYTU2NjY3MDhlODdiNDIzOThhMmEyMjUwNzBjMTU5ODUyNGE0YWViNGVlYWJmZjcxOWM0ZjU2MWIxZmQwYmZhYzM0ODUyMjhkZGIzMGJjYWQ3ZmE0YTQzMWYzODk3MTBjMDMwZjg3ZTY0NDY2NTNhOWY0NTNkM2JhNGQ4OTVlMTlkZGVlYzQ1MGFmZWFlZmZlZDk0MjdjNDliZDdjMDlmNDYzZjRiODA4MTBiYTViZGRhNjMxZWExOGRiNmFhMjZhNTM5YjNkMjhmODNmNGQyN2Y2YmRhZDUzNDY2NTc0OTg3NmRkMDhiZGNiMjU0MTQxZjNiZTVjNmJlMmRlOTAyODY2MGMyNmI3MjZjYzZiNWM0Zjc2ZjkyNTljYWZkMTc1NTcyMzhjMzY5OTY2MjZkMTI4MjEyYjAyZjc5NmYwMTVmOTQxNGViNTI4YzJjNjIwYjNiZTBhNjhmODlhZWE4M2I1NWJiMjkzMTI4NjdiN2Y4ODY0ZDU1MTFmMWIzY2UyYzcwNmFiMmU0YWVkNDQyYTY5ZGNmY2M2NTAyODAzY2Y3ODE3MTUyYjUzMGM2NjE3ZTZiOGNlZWYwYTRjZDRjMjc2NTMzOTVhZGRiNTU0ZDNkZGY1YjRkZTE5ZGM4MjU1MTg4ZjlhZWNkNmFkZjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Fi2JRxInbSAimQj6N8SNeckBKrQNRxV8x0cE5XsU6Plm-txunlG_LVekAx21z_yi1PaQknVZcAd8pd-OHqj17g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102533_50_2451_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.160Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InBOd1l5R1JPL2ZWTTZaTnpPK2NNVGZaWVljT2Y2amNoTTBTL291QjlNbi9xVzVrTTAybnZRZEpncUMyNEZ2Z1RuM1dIUzNqRUFFV0dURGVWUXFRbVpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI1MzNfNTBfMjQ1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjU4NWE1YWFkNzZmMmU2YzA1ZjFhYzViMzU1ZTk3MTQxMzI0ZWI0NjhhMTJkMmFkZWE1ZjE2ZTUzMzUxNWMxNjgzZTk1OTlhY2Y2MGQ4MzVjMWFhMTg5MjU1N2U3YjhkNTZkODliMTFkN2UyMDgxMTY0MjVlNmYwOGMwN2FiNWE2M2M2OWJkN2NhZDUzOGU5MWZlOWIzYTIzOTk1YzA4ZWI4NzgxMDYxYjY5MGNjZmI3YmQwMGJlZjRhNzgxZjAwYzM0OTZkZjdmNDQ3OTVlZTI1NDM0YjAyNzc2MDZkZDM2NjgyODAwYTMwMDNkNzczNjk3YjhmM2VhZjEyYzBkZjA0OGIyOTFmOTQyYjEwMDFjZDJlMDJiZDI2ZWQ5NDcwZGM5YTU3OTY1ZmYwOTI3NzU3NzkwZjA2NTRiNzYyZDQ2YjZlNGExZjE3MTBhZGQ5NTExNWU1NGE4ZTg1YTljODI2M2ZlMmYzZTY4NzY3NmM2YjUwNDdiY2JiNjBjOTJkMTYzZGUxOGQzNWNkNTBhMWNkNTQ5YzRhOTI2NzI4MDRiNTFhMTNjMWU1Mzg4ZmQzMjY5NjU3N2ZjODViNTRmZGJjZjIzYTY3ZGU5ZjliYzFhZWZmYWRkNjY3ODBmYzY0ZGM4ZGNlNDZkY2M2OTU3ZTQyZDhlN2FkODU1MWRmNDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3iTN8O8WNUFmn8FkUJh2fqzx37QMPgEjp4A0LqMdkJ6BSFjSxkKvRT9DxL-9ecG-blU5dW60svBer5rgQL0r7w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102533_50_2451_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.164Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJ5dmxmYWZ5K2ptdnU0UUxQcWJ4enh5Z3l3MENxZTVUellRLzJ3anNiVDFsTWlzRXA5dEZ5K000Q3VmVVNkdmk0MWpDVm9acTZtNitmL0NkYmN0dUp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI1MzNfNTBfMjQ1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODJiNWIzMjBjZmNmODZiYzFlNGZiNzFhNjZiOGMyOTk2MzQ2MjkxMGEyZWI5ZDM5NzQ4NWIwODM4YTAzM2JhYmQ1MjQyN2E0YmRlY2VlNDk4YzgzMjFmOGJmZTA5NDBmMDE2NjIwMGU1ZGE1ZTg4MjNhNzcyNTRkYTQzZDFjYWE2Mzg3ZTYyZTUxZDJkYTExODYxOGViZDUzMWQ0ZjBmZDExODExMTBlZmVkZTFkMDRmM2E3ZmRkY2FjNzhjNjcwMmUyMDI0NTA1MjNkN2MyNTgwOTdlZGQxZmU2ZjhlMTQwZTlmYmZiYWMzMGM3ZDAzYWU5MzdjOGMxMTZjZDJmMTZhYjc4YTkwNjRlOGIxNTMzM2UxMjlkM2Q1YWY4NmVlYmM4ODNhZmU4ZWVmMzlmZWEzNzcyZDU1OWUzYjQ3NGM2MDUwNWYwY2IwYTNhOGQ2Nzc5MjZlYTI1ODBmYjY5Y2U1OTZlYWU0YTVjZTk5MzBmN2FhZjM3OWJmYjU2MTE1MzE2MjM1MjdjYjBhYTcyOWY4NmEwZjNkYTYwMmVkZGYzYWJkMDhhNTU0Zjk2MzE5Y2QzZmFkMGJiZWJmMjBlZjQ5NzE0N2FlMjk2MmYwZjdiNWM5MDZmZDBmMTkxMjJmNWRiMzY2ZmM2NGY2MGY2NTE4ZmRhNzEzZmM2ZDJhMTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GLOcQj2aU2rxIuJT9ITQeF19voLnROkoeS0lGQmn436hWhP0gZ7pPObwkpAXXe5U6mzcNc6dsZJW5V9eGyMspQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102533_50_2451_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.169Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkNQU2M5R1JvNm9WR0ozTk94cGhlbUlSL0xXT1JIZDRtam44a3dlcno2eDJ4VjZGZkRrT3Azbm5KWXVUUkRPSFBtN2RLb1htOUZGSGl5WWtGSzBVS2NnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNTFfNDRfMjQ5MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWUwMDM0ODhmNzZkMzA3ZjdhOTkwZTU4NGFhZmVhZWU1MGYxMzMyOTk0OTQ1OGRkMzk4ZjE1YTdhYjA2N2Q2OGFkMWZlYWM3YWY3NjcyMGE5ZTJhNzRjNzM2YThlYTU3MDQ1ZGI2ZTQ2YzNhMzY2OWY2MmI4YmRjN2Y2NWVlMzExOWUxYWQ0ZDc5OGY3MWY2MWNkMGM1ZDgzMTBlZjA2MmEzZGVhMTgwYTJlOTgxMTRmNzQwMGQ2ZGFkNmU2ZTQxZjc4YTI0ZWU4ZjdiYTNhNDBhYmNjZDYyM2Q2MDU4MmNlMTIwNDU5ZWVmZmQ4YjVkNDkyMzYyMzc1MmE2ZWJlOTQ4OTExNjExNTg3ZDQ0MTIzYzZmYmY0NDU4ODExNTVjNTU0YjhmYTBiYWY0MmE2NjYxOTcyOTNkZWMyY2Y2YzI0NTYxYWJhNDA0MjYyMTM5MTRmZDAwYzM0Y2EyYzEyNTdkZWMxZDVkYTk0NzBiODcyOWVkYTI0MDhlNWVhM2FjMzMzZDE5YmFhMDk2NWJhYzM1MjgwYTFjOTU4ZDRmYzBhZjE5ZjU2MTg5OWFlZjg2YjE0YzQ4MDJiZDlkZGQ3NmYzNzNkOWFmNWFlYTE0MGUxNWY2ZWYzODk1Nzk3ZmFmNzVkYmFkYzNjNDQ3MDE1NDA0ZjViN2M4NGYzNTliNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LBu5zeYW5Rx5t9gd9VBfXK2OAFuExzkCxdMLhfuxSr6BHlpmI7nXkwjckOVBMmbMmwR0vqfYW9DB1XP-J0yY8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110351_44_2490_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.174Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkF1Q3JrM2hIWFYyd01VcXpvT0ZVY2s5TzcxRXRvVWhBQzVNQmVEZzBaa25YK3RSbVNMT2ttNlNBWWJQOVNTU2Z1aHQ1d283Rldta05XeTY4Q3JOTVFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNTFfNDRfMjQ5MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODNjMmQ5YjEyMzQ3NGQ2YzIzODQ2NTgyYTM2MDU0ZTVkYWNkYzViOGYzY2RkYmUyNzJhZjM1MTc5OWFhZjkyZDI5MzIwNjZhMjJhZDlhNWRhNDJhMTIzMmZlOWI4ODc2ZjU2MjNkNDA4MTk3ZDdlNDZmYjk0NTJiNjJmZDYyMzRjNTU3OTRjN2IzNjNkYTg2ZDVlZjAwOGE0YzUwYTM3YWRmY2RkMmEwYTI3MTFkOGQ2MjNmNWQyNTc5YjVhMDI1NzI5NDU1MGFlZjMxN2I5MTdiOWI0NTQ2YzgwOGYzY2YxZmI2ODhmM2JlMGFjYTY4YjQ2YThmODY4YmU4NjQzNjhkMzdmNzJiZDY3NDJkMDlmNjViYTRmNTE5MGU4ZGIzOGFiNDlkMzlhNmJkOTlkOTVkNzg2ZjNiZDA1NzNhZGNlZjZhNmRmMTJkMGYxZTJjNGNmNmQ1MDE0NzUwNGQ4MzlhYTQxODQ3ODM0M2Y2OWYwMWI5ODMwM2FlOGMxODUzYzkwNTI4Y2NlMDI2NWQ4NTRlMWJkNmRkZGZkZjEzMTJiZTAwN2E5ZTlkYzg4MjFlYWViZWVlNGRlYzcxZTQwYjVlOTgxYWJhNTY2NzE1MTBkMjA2ZjIyMWVmMTVmNWU0ZDkyMjgwNTdmYjZlNTdmMjQ0NzgzMmY5ZTRjMjVlNGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aUmxrqb_CSv1uPEqBcUTx8jvy5asB9LFzWbwATJ6X3V8RE7I7gp_GS2y3A-_YQLy_z6EzoArwHn7mzGPainkxg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110351_44_2490_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.177Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkdQQS9wMUgrdmJZMk1EQ3ZBRTdzZ00rM3FJZ3ZUTkFFOGhCdVEvZXJCVzFrN2ZHdlZUeHIwT2piVXp0L3NSa29NeWQxUFRSTEFmanhmdWwxd1AzRFdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNTFfNDRfMjQ5MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTJhZGUwNjYwMDUyNDZlYjVmMzllOWUzOWZiMDNjZTNhMmQ4NWE3OWY4N2UwNGEwOWMyMWY3ZTlmY2NjNTdiMzMyNzI3ODNhOWExZDk5ZThmMDNjOTY2NDk5ZDNkY2E0MTBhMTc5MGU3NjFjMWVkZjQyN2JhZTYzMTRlYjI4M2M1NzcyZWU1ZWM5ZmVlZGJhZjBkZWRlZTVlMzM4NTY4ZmZjZjNhZjhkNzExNTc1MmQ0YmE5NmQ0NGRmZmJmMDg2NTVkNzJiOGRjYWMxNmFiYjE1YTllN2UwOGM4YzUzMzZiYzdkY2NhMTE2NzkwZGFjNmNhYjM3ODQ4MmM3MTNmOWVkYTZlOTRiMjRmNDcyYjBiMmE2YWNlMzBhMTJkNTU5ZjM4MDY2YjBiOTViZWYwZGRlMDE5NTI4M2Y1ODk4M2UyNzAwYjQ5ZDMwNGEwMDJkMDc3MTBlMDViZTBkMTI3ZDZmZmU2ZGZjYWQ3MmZiNGI0OTQ0YmQ3YmVlMGYyNDM1YTFiOWE2MzhlNzU4ZjU2M2NjZjhmMGJjZjViOGJiMjZlMmZhZmQxY2JkYWRjY2RlYTRiZDQ5ZmJkZjg2NGM5ZjA4OTdkYjFhNGY3M2RiODExMjAxZDJjYzI5OTFkNzJhZGUyZjA1ZDE2MzNhY2MxNDNmNzNkN2ZjODY1OTdmZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4Dp09XmEYXSk9C6P0ZMC9c1x9tI-Gn07D79Pon6KeddOuh9MWOBjgNsLPlN6OuDRRKuQXTeO9U0bt_8vuWhnZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110351_44_2490_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.180Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlhaQ2NQMTFVeUkwdjloMXI0eG1PWGlKQzVVMSsyTVJKb2hUaFdiOFpJSERNV09IYjM4ZWVTcDhNVXB3MHV4UStHS1ZKc2hKMEQ4V1RwMWxUMno0N0tnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNTFfNDRfMjQ5MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTMxZmQ5ZDA3N2E2ZjM0YzIyMDUzNTZiZWFjYjM3N2FmNGNhYjQ3MTdjZmI3MjJkMDc0MmYzZTExODA4ZmQ3MWMxNGRmYzJkZTI4ZjhjMmVlMGQ0YWRiMmViMmY1ZjRjMTQzNzU5YzdkZTU5ODRlNjhiNTAwNmFmNWY0YjFmNmZmYzc1ZDExNGQ0MmI3ODI5MDY2ZDAzMmQwZGUxYTU0MTgyYTViMmJlZDAwOTEwYTBlNTYzNGI5MjZjNjYwYTM3Y2JiZDA0Yjk4NzdiNDZkNzZkNmMzNDlkZjBkZDdkYzZmM2JmNjA1YTAzMzQ0ODFjMTdkYmQwYzQ1NjlmNDI1YTIzMjRhNWQxZDFiNjQ3MWE3ZjU5YTEzZTU3NzdlZGY3NThjNzVjNTRiNjE3YWQ2NjZkNDgyYWM3Mjg0Mjk1ZDhhODIyNzVlNGQxNTUzMjFmMWVkMTQ4ZmM0ZWU1NzNjNGI1ZmJmNTE3YjJiZTgwYTU5MDFjYjczOTdhMmYzOWFhOTJjYTM5ZDM1MmQ3NDk2MmJjMzRjOGNiYmM2NTk2NDQ5YWI5NWNiMDc3NTVlYzQ0MTYwOWU1ZjQ0ODkzZWJjN2Y4OTZkYmFkYTlkMTE0ZGUwY2Q0MjllODdhNTQ0N2I0ZDBhNzgzZGQxNzBmZTQzZDc5NzUzODYyNWQ1MDMxYjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.opBPMx1V_WiMaPgQURjsF4lkDcPv4NKvM4hg0o_PGuHCb7tRUgZDfKuimuAgZ6oITUrA7aZ9yKobt6cQjyzMcg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110351_44_2490_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.183Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InYyM3VQblI2R1FVUVdyanY1YkFGbTBlSFl0ZjV0WkVpa1ZwRDN0UGMwT2FiUFBvbkY4Uk83eExxVTlpcjdYL3VmNHh1dzA2NDllRVpTMk5HQnZ1aWF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMTE0NDFfODJfMjQ4Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmIxZTM4ZjkzYmM0NWEzNGNjMjAyMTNhZjA3MDc2Zjc2OGZhNDIwNDJlZTA3NjcwZjU4N2M5YjllZThkMjc3MGNjZmQyY2QyZTY3OTAxZTM0MGEyYmRiZmUyNjE1NTMyZTFhMzc5ZGIxYzExMDNjZTk1MjMyOTMzM2UxOWMzOGNjY2UxMTUwMDBiZTM4NWUwODA5YTMzY2ZmMTQ3OWQyYzg1OGZiOWUyNjRiZjhhM2U5ODc2MzA4YzVkYjJlNWY4ODE5YTE0ZjI0MWNkYTE0YzllODlhMDZiZWU1MmIzYzZmMzM0ZDgzOTgwZDkzMjM3YmQ1ZmEwNjhjMDlhOTM1OTdmYTIyYjFkY2I5NzE2ZGYwNjM3ODQ4ZGYwYmE2ZDhlYmYyZjAzMDIxM2IwMmRhOTk2ZmM3ZjcwYzk5YTY0MjBhZmU5MDdkYTdmMWQxNTFhNWVlYTA4NWU3NGI4MjljYzkxODQwMTA3ODMzYzJiMGJlMDY1YzRiNmY0ZWQyYmVkMjEzZTk3MWMzYjRjMjRjYTgxOThkZDhjNjFlNDI5ODMzZjBlM2IyZDk5OWYxMjBmNTcxODIzMDM1NDU4NjYwMzg4MDBmYTAxODQ4NDZiMzUwNDA5OTIxYjQ0MjFkNzNmZjI5Y2VlYjljY2NhNDRmNzFlYjQwZjA1MWE0MzhjYzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.410f2t5Oh0gpmgd2-VeKwRBNVEuLTCp6FbityCHIAaRR7mOQSUiuvUPa_dlBS9dJhVglnufRW9dP6MfzH4UKnw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_111441_82_2482_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.186Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9OdG9Za0FqOVJkUU8vS1BHdE5QZEdZaEFZemdLL1BYbW1iSUdTZ3RQRmlpTmh1RzN6WjFWMW1ONUQ0NUFRQ0VzRi92c1JYNXo5MmtrMDJaNzczL3pRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMTE0NDFfODJfMjQ4Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDA3YjU3MjdhYzc5MTYxZGY2MGE0MTNhNjEwM2QyNTQzMWRhNDc4YWM0Njc2OTkyOGI2MmU3MjI4MzNmMDU2OWIyYjM2OWViZGY0MzRmYjk3YjZkNDI4YTAwYTk4MmJmYzdlM2M2MWEyNjk0Y2ZhOTA0MzRkZjk0MDBhMGQ1NjViYmU4MTk1NTA0NGRhMDEwNTlmZDFmOTYzMjM4MTlmNDRmZmQ3ZGRlNTQ5MmIzNzU0MjNlNTAwZDgwOGFiZGIyMmVmODY1MTMwMWQ0ZjE0MmQyMWJjMzNiNDk1OTUwNWY0YzNlZWVlYTg5NWZmMWYxMTIyMTg4M2FkOWI2ODc5MGE1OGJkMTE0MDNhNWEwNWMyYzkyY2YwNDI3NjQ4MTQ2NWUwMGVhNDZmMmVhMzQyMjk1MWI4MjBlZmE3OGE5YTk5YzNiYTMyM2UwZGM2MmQ1NzYwY2MyMGMzZGMyMTY4MDBjOTk2NTYwNDYyODcxMmJlNmU1MTZiMjQ1OTVjZDc4MzYzZDRkNzI1ZjY1YmM3OWIzOWE0Y2U2OWU3ZGJiMmRhOGQzMDY2ZTg3YzhkNmE0NGNhNjVkOGMzYTBjNjdkNDUxNDQxMzMyZmFlZTczNDM1ZWY1YmVkM2NjNjA0ZjE4ODY4Mjc5NjNkNzdiMzdmZWE5ZjcwZGNjYWY1ZjNjZThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3atDUGp0jEWu5sCOiZ6z8Gi8odfP6DREeyousIV9hhEEdDJ0YumcZRE4pzPCualWbrkYKlN7RccLVjkfgikAbg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_111441_82_2482_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.189Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjB4dmFRaCsvK0hCTHRuUEN0OWprZ2lSTGlHTTFtNk1HZXIrSWJvajA3cHBCQnBNZ0Y2RE5QN2JlVFdzSHpxdmE5TjdkMzliSGpidUF5bXRNTDJ0ZTVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMTE0NDFfODJfMjQ4Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzhhMjk1NjhhOTg0MjJkMGFjNmExMDIzZDgzMGM2YTc3ODM2NzNjZjUxOTEwOWQ5ZDhhMzhmYmZlODAxNzU4NmE4ZWIyZWQyOThkYTY5OTg1MTAwOGY0ZGY5MGMwZTVmMmQyYTI1OGNkNjE1NDVkODMzMmVjOTdiM2E5YTRlZTRlNDFlNDFlZDMyYzA2YTgyMzhkMjhlZWE0YWRkNzRmOTk1ZjVhZGViMGIxODYxYmU1NjE5MzFlYjhkNmU0MzY1OWEwZjUzMzQxZGNlNDE4NmQzYjAyMWU4ZThhZjAzMzhmZGU5YjMxN2I4MTRhM2MxMGMzNWRhM2VlMmMxYmMyNzRkZTA2MTM1ODk0ZDY4Zjc4NjRhZGZlZmZmY2QxOTE1OTY0MzVlZWExZDNjOGU4ZjFjMTA0MWU4ZDM0NGMxMWRlNTMzMzc3OWUyYjY5NTU0Njc3MDA5NTUwMTRmZDcwYzVmNTcyOWZkMGNhMmVjZWI3YzUzNjRkNGNhYjhjMDk2MDZiZWY1YzcyNGQ4ZjZlMWMwMTI5Y2UzOWE1NGExYzkwMzI0NjE2MDgyMjk4NzZjYWU2NTZhMmMwMTBjZTk4ZTUwYTM1Mzg5MWI5YmU3NThjYjQzYzkyMmQ0NzMxYzY2YjQ1OTcxNWM5M2YxYTNjODMwODc0OWM5ZmVkMmMxNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.J2n0sXsDJUg0tt7uNLslZPnnZWmKm8I-gM_AsAu820PFXnPKvTfzv2EHyqEqXLWwIvs4uImGOwzXK2ROt6ceMw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_111441_82_2482_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.192Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjVKSWJsNjdWWU40M0ExN1UvZTkxRnlteWZTSGpVZ3dpWVpPbnNFcDlHbFBLMmlkUDhIY2tBRitnV0d1Q0NNVXB5THNXVzF1OVNvUEV1MitCUHoyWllnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMTE0NDFfODJfMjQ4Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmJjYzUyZGM5NWQ3YzIzMWYxMWUyNmE3MWJkZTQ4M2M5Zjc2MjYxMTVjNWM0MzVjMTk4YzliMmQxNTcxNDY3YmY4MjVkMzc0MGY5NTE3MTdmMjYxM2Q1ZjJmNWQ1ZGU5OWJkYmI4N2NiY2JiMjA2MWJhMTU1MDkwY2E5NjcyMjg4Nzg4OTg1ZjFmMTdjM2MyNDQ0NjQ1NzQxM2ZiMGNiZGQ5Y2E0NzkzOWVkZTM4MzQwNzdlMTJhNTRhYTUxNmRhOTMxYWQ1YThhOTFkY2NkNDA1ZTMzMjYxMzBlZTZjMzUzMmUzZDBjMDk1ZmNjZWE2MGI5ZGNjYWYxODQwODMzNGNkOGIwZjc0ZGUzYTYzOTlkM2JhNDk0MDU5NmZkOTRkMGUzNDBkY2EwZDBkZGY5Zjk4MmU4ZDJjZWY0ZDBlZDlkMTdkMmU1NGUwMDAwNjdhODY0MWYwNGUyZGVhODE3MjAzMjk5ZGY5NWNjZjA3ZTE4MmQ2YTRiOWY4NTlmNmI0ZTcxYzI2NGJhNjMzMTg1N2EwOGRmZDY5ZDQ1NTVjMWY0ZTM4MTRhNzMxODBhZDY5Y2NiN2E4NWJhN2YxYzI5ZTgxY2VhMTViMGEwMmZiMDQ2NmYxODE0NzI0NDg0ZDEyMzIwZmE3MTcyNWRjMmViMzY4MjkwYTAzMDUyYmI3NWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rBKXKw5WdkTK7_inOZ0YGZxiTfjaIhtXTC0cAmijaDAzLZXyzhC23WjX1DqhOCw5ZfBex6n7XGX-JGsZfdj8nA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_111441_82_2482_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.195Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtDQ2VpK01lcUNJM2Z2WTdFdmFpOWNaZUQzalQvcXAvdWhtcFd0RFN4WVVHVHdCcjNma2xzQk9TRzhLTzZBdlpaOEdwdlAvY3FWTHJJc0QvcnBxV0ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDRfNDFfMjQyYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWFlNjE5YzlkMDM5MzBmNWZkZTg0MDkwNmIzOWY2NjQ4YWUzODAwNTk3MTgwOTliNWM5ZGViOGQyY2RkYjI5MWU0MDk4MTZlNWJjZDc1NjI4NGJkNjcwZTM0MjdmNTM2NDY1MTI5OWIwY2Y2MmE3NDBmODBhNDc1MTU0MDc2ODVhZTQ0ODgzYmM5MGIyNzlhZjVkNzY3NjJmMjI2N2UwNGVlMDdlMjIyNTM0NzZhYTY3Mjg4ZDUzZTdlNTU5MzkzMmI2M2Y3NTNhNDY3MmM2NjFmNDM4MzA1NGQ0YzczM2NjZGI4ZDI4ZDQwNjdkYmExOTQ4YjI5ZGExODIzN2RmMjc4NmYwMWE2MjQyODExYjNmMTY2MzAyYzMxNGEyM2YxOTk0ZDA1MDE2YzkyMTFjYjM4OGMzYTBhZGFkMzY0MTA2NTA3MzJlMTM0MTRkYmUwM2JiZjQyNzBlMzcwNDU2MTAxNzdlMzAzNjYzZjRmNmQ4NDZiMzQ5MGI4MTk1ZmViY2M1ZjYwNmFmYmJmNDdhZDZlNDU1YzI0M2MwMmIzYzI5N2Y1ZjNkODQ5NDY1NzVlOWNiYjBiMGViOThiOWRiMDhkZjU2YTkwYTkzMjRjOWUyNTk5YWY0MDNkMzVhYWY5YTk0ZmMxOTA4NzEwODZlMGE1ZDE3YjY1YjlkNGM5ZTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9oqbTcbRhJyJ61dP1ruv0f3NMmW30TDY2q5pgfvK-2_gKtRjzQTIf-m_QgipcoimTntzYshb29y-q1PNPRoDFg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104044_41_242b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.198Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkRkYWhKeVdlUlNpTUpjTzRFS1hEcXZDWGczUTJCbzcyMUhBZFk5OE1NR2wrVVlDTDM2RDlkM0xLUEIzNi9hb2tFTFZFd0x0MkFTZEtlc3hsSUx6NmdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDRfNDFfMjQyYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmE3NTdiMWVlODVmYzk1ZTY5NzhiYTVlMTRhZTU3Y2JmOGM0MjM2ZDI1NDIzMWM4NzZkYmZlZWFiNGRhNmMxODZhZTc4MTFjYWZkOWYzNzc1Njk2ZGZmZTQ0ZDNkNDRlOTdiOGZhMDIzNjAyNjIxZjA4ZGJiYjQ4MWU5ZWM1YmM0MDEwNjM0OTU5ZWQ0OTNkOGNiMmYwMmI0ZmIyNjQ3NjU5NzM1NDg3OTVhOGFmZjkzMGQxYTA3MGIyOTNiMzllZjVhODg1MzFmNTEzMDk2Y2UxYWI0NmRhMjQ0MWUxMGZmMjg0YTQ4NWFlOWJlOTc0NzE3NjMyNDRhMjE2ZTJiMDZiM2NkZmUxMGQ0ZTg4ZDBhODM2Y2YxNTM1ZDRiMDRlNjRmZDI3MjZmMDEyZTg3ZDJhOTIzY2NhZmM2MzhmYjdhMTMzMDRkYWRmZjA5ZDZkOTVkN2RmZDg4ZWNmOWI3ODhlYjI1ZTc1NTFhMGY4NTk1OGQwMWE1OWMwYjI5NzFkYzc1NTJiYjc4ZjM4M2YyMmQyNjdlNjg1NDdhMDc3YzQ0ODI4MWNmMDhhNzBhNjIzOTUxZTkzZGE4NmYxZWNkZWI4YzI1Zjc5YWQ4MjkwYmFkYmY1ZWJkMjk1MDdhYTQxN2UyZGRjNjFlZjQxZTAxZDMwMzgwM2U4YjQyYzU5ZmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7RrmwjWvmlCpnsVt7TqmeRyHHxA4tovCZXddaK1E76vl-qHf_Y7xkHsOiJ4ClOCKcFOG_a_AWfQ1A42yQtVNPQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104044_41_242b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.205Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IksyUjM1SUJpSGNUQmpObTl1UWFrRmx4Sm15enBZVCsxMGF5aTNITFJRR1E0QVdIQ2pka2RFL240bVZ2NDRuTzBmcDJodDhXOTEzMEZCaUZaUnhDVnJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDRfNDFfMjQyYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmQyYzBlYzY0ODc3MTYyM2MyZjgwMjg5ZTRhYmJhZDA3MmE1Yzc1MDI3MjkwMTQ1OTc2YmNjN2RjYzNmMmM1NWQ5MGQzNjVkZDY1ODY5ZmIwMjRhNWM4MzVlZGZlODlhM2FiYWIxMDFlMzA5ZjhkZWMxYTI5NjU3NmYxMWNkNmNmMGVkMjQ4NTFmYmU3NzRiMTVmOTcwY2U1YmQ2YTFkZjM2YWNhYjRlNTNmYmY1ZWI2M2I5OTg2ZGE4OTY3OGU0Nzg1NzcxYTQ3MjdmZGVkNDc4ZTQ5MjYxNDgyZTdhMmVmOWFmMTU2OWY0NzY4NjIxYjI1NDRiZGJmYmVkMjFhZjI0NTVmMDg0Yjc0YTMyMWExYWM2YzE0NjAyNmY5MjU5MTA5OTQ1NGU1ZWZlOGY0NjQyNDYzZWJkNDcyZDgzZjAwOGM2NTk1NDViYTkyOTI2NjE1NzJlYjg1YWY0Nzk5ODc0MDBiN2NhOWM3NDc0YzNiYmJkMTMyYTJmNjgzZWZhZGViMzJkMGU4OGVkZTNhMTIzODFlOTZkYmZlM2Y2ZDkzZWQ3ZGI0NzU1NjVhODY4OTk0NzAxMzVjOTIxYzk1OWQzYWFkYmJhZTI2YTY3MGY2YmM1NzRjOTkyZGRiY2QzMzE0OGJjNWU4OTE0Yjg4MGU3MTgxMGUwOGY4NmJiZTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5s9HyKFiIFhRsDHKTyN_WN4gKo9UHTOPCyPjfpWuYN0o34k-0lFTDEi-IcVq8SgjVjxbggne4TyDgiEqgw7mZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104044_41_242b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.208Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJWNkJ1b3JhdFpQbks1QXVFR0xtTjVtOFFENEYrVVpuUkg0Z2w0ZnNqZmxLM1Uvc3lDcUp0ZWhraXlGeWRUNHFMK3NVeG9tVnRQQU8wcEIrSWh2WDB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDRfNDFfMjQyYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2RiZmIyNmNiYjBkYjFiYTkwN2JiZGU0MmIwMWY5NGFhODJkYzFmOTVlYzgxMjljZDgyNzdlNGIzYmRkMGE4NTRlODNiODRkODA2ZmY1NjEzMTFjYTZkY2NkOGVjN2FkMmE4ZDg5YTJlYzE3MmZkMGEyYmFiYWFkNWMyNjRjMWQzNGJhNWZkNWRlZjM5NjZjMmVhYzBjODk5YmQyOGI3YTA0NmEzMWQ5NzY3YTdiMWYwNGQ4YmI5Nzc3NmEwYWRlZjUzMTk1OWMzYWFkZDcyNmJhYzRmY2JkMmMwYjJmNTAyNmJiYTgxNGRlMGM0ZmQ3NGFiNjZkYWQ3OGNjOTFmYzU0YzdhNGZjODk0ODY3MzIwMmVhYjJjYzg1MjVlNjViYWYwMDA0ZTVmN2YyMmYwMmE0Y2I4YmM3MmVkNmJkYWU3ZDE3NTgyZTZlODU3NTFkZjI1YzVlZWY0NTljM2RjMDIwNTk2NDQzODA5ZTdhZTY5MjdiODQ0ZWVmYmE3YjQ5MTQ5MzU3NWI0OTdmZWRmMTY4MGZhODY3NGM2YjBkMjI3MTY4MDBlZDU1YTkzYjQxM2EzZmYxN2M3YzEyMWUyNTZhZThkNTlmOWJjM2IxYWFkZmY0NWE1MDUwODU1OThiZWJlOTZkMTA3Y2ZmYzI0NjA1NjVlODFiZWY2NDdkN2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.b89e2kIqJHUAbVkCYQ11P7O5_zw8LNbYZCehGooDvN-ScJ9cPVSuffsZIvy-Zy0-DXO-U4jUkKxCohnoH0PFAA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104044_41_242b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.211Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVnSnBtMG5LV3BJOUxVS3VpN3BXUUVlS3FpV1hRZVFBYUYwQ0lobVI5UEQyN0lBcmQ1VmNhUVVrYzlKeW1DTzEvZHNIMzNkRGU3YkVLVmZoaW5NWFhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTE1NTlfMjZfMjQxY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDY0YjYyYWExMTBlMjk0MWZkZjU3ZjkzNDY0MDAwYTQwZjU0ZGQ0MWE3NmQ0NTVlZGMxY2FkMWU1NmE0NmFlOWVjNmMzNWYyMWU1OGI0YTFiOWViZmU0OTA3NTY1MTQzZmNkMDhjMzc1MmJiNjhjNWU2NTRkODI3NmU3NWJiMDUxMmUwYmZlMTYyOTVkZWNiNTg0NGIzODYyMTUxNjZkNDYwNDQyZDQ3YjkzOTE5YWE0N2Q2NzU3ZDE0YjIzMThkYWNmNjBiN2RiMDQzM2NmYmY0ZjU0NzY5M2E5MzM1YzlhMjkxYWJmOGIzOWQ5M2U5N2NkMTc3ODY1MjQ4MmUzNTExN2NkYWZiMWVmYmQ1OGQ0MDViY2Y1YTY3NmViMWUwY2M3MGNlNzk1NGE5MThhOTExNmU2ZjAyZWQ2YmZmYmFiMTBjOTZjZGM0NzMxYTI0ZGRiMzU5MzJhNmY0YTM2N2E4ZTRkMGJmYWIxN2IyMmEwOWZmZjc1MjA5MTBlMzAyYWM4MjZiZmE1YzhhMjc2NTVhZTZlMDM2NThhNzQ2ZmQ2NjA3NGU4OTQxYWRjZTE0ZWM4MjEzY2MzOWFmYzM1YzdjZWZkOWE1OGRlZWVkNGIwNmUyODRmYWFjM2MyYWY1NDJiNDBmM2JkMzQ2YjM1NjIxY2MxNzUyODRmMzZmOGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.U6R6eFLJs60j9ItIKmZDmJ-ii7OT9rzcvM1QrfDBGkcetOcztxBSAnKpUAEi_9Sja3oYIJT9CnsqepxUGItv1A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_111559_26_241c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.214Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InU2WDlBcDhIQWM2Sk1UelZIM0trSHNaS3Z0ZXZvOEVLQ2xuZHJFOS94VytwM282SWtOS0hvMUtvRjQrajBRM3VyTkRuT0x3aXpsdGJMR3VqWVlvQ2hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTE1NTlfMjZfMjQxY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTE1NDRjZDAzYjQ5YjAxNTE1N2JkMDU4NDZiYzJiZDZhNTQ0YzhkYjEwMzhkZjk1NzMzMDM2MDI1OTU4ODdjZTQ1OWNlNTM1ZWJiYWFjZjhmZTgyMDBkNDcxODg1NTI2MWU2ODUwNzMyOTM5OWNkMjYyNDZhMDdhMjdiNTA2NDE4OTIzODE0MTlkOTZmZjJmYzZjNmUxNTAxNGIyN2NiMThhNjViOGNmNDRjNTRiYTE2ZTVkNzhiMTk0MmU3MWZlMWZhNmVhZDgwMTY0NzdkNTdlNGVhYTEwYzIxZjJjZjZmMDQzMzhkYjM0MmRmMzZhYmI0Yjc3Yjk2Yzg3Njg0ZGZmZDBmM2MxNzEwNmIwZThiMGVjN2RlZDFjYTZhMTRlYTA3M2Y0YTMzNzgzM2M1ZjY3NTNkZjFiMWExZjQxYjc4ZjA4OTRlM2FkZDBhOGMwZTcxOWJkM2FlZDVjOWMyYWMzNzQxNmIxOWNiMmRmYzYzOGYxYzI5NzdmY2FjM2I4MWIxYWVkZGI1NzlkMWVlNzE0YWUwZWQ2YWQyOTUwZDhjZDMxYTRhYjU0MTRlMjFiYWE5YTFlNzNkNDA4ZDY1MDI0NDFjMjA1Y2M4OTE1ZjRjOTFlODA4ZjI2ZjYzZTRhMzQ0NjJmMGM4OTQ4MzE1MmFiYmUwYjhlYTQyMzI5NDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FrmCiidt_8QsPcQLiyFQjmQHBmttPNVEE455ZNia9cNPxZWV-yteOGTCCo6QTseSwxol5NxqtrYEVCGMKjsVKw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_111559_26_241c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.217Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxTZjdGRmZ3QWh6bG1TdzY5L3M2U3VJZXgzZTBlWU1nVjhVUnFVMEhLYlpQYm1rdE5SaWQ3SVZPOUJ1VmNVbERIMGpvRVBRZzBiNm5WKzAwK2dQTzJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTE1NTlfMjZfMjQxY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWI3ODg5ZmNjMTkyOTFjZTYyOTU4NTQ2MWUxZDNhNjFhYTNhMzkxN2U1Yzg1MGNiNDdhYmRiYzM3MjQxYTM1YzM2MTRiZDRiODc3NDQ0NTljZjRlYThmODAxZDYyZDI5MzM5ODA2NTQ3NmFhZTIzM2Y2NTZhYWEyYWRlNGFjYTYwYTZlNmMyOTk4YzA4ZmUwYTQ5OTgwNTY0MTE2NDMyYjBhNWViMTcxMTk3YmIwZmZhM2Y4N2Y2NWY3MDI3MDRlNjAxYmIzNjRjYmI2NTcxNTcyNWYzOTRmMzA4MjBlMGQxYzcwZDFkYmE0YTVjNDNiODgxMGJlYmNhMTM5OGI3ZWRlMjA5ODhmM2Q2MGVjMTRmY2Q3NDk1YWYxMjhlMzI1MzUzZTkxNTljYWFiN2Y3Y2QxZmI2OGJkZDM2NDQ1MDAzMjdhY2VkNDI1YjdlOTNiMzdlOWFjOWUwYTZkYjJlZDM1MGNjZjY2YTNhYWUxZjNkM2NlOTYwNzRlNmRiOTk3ZGY3ODgzMTUxZTQyNDBlNjE1Mzg1MDc2ZjMwNzYxZGMxZTZmZTJjODllYjNkMjNiNDhhMDQ2OGY2NGFhZjE0MjcwMDlhNmEyMWZiOTc2ZDhiZGZiNzkzMGQ4NzNhMTMzNzI5YzAwMGYzZDQ4MTI0M2IzMjdkYmI3YjY5YTQ1ZWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SHtIWO2uPtXZvaYjdjcXICwOOGl1rwLC-611F7I5o7cSeerplt6QWitZ_RSertc6e5foqIxePEEj1L0wChRDTA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_111559_26_241c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.220Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik52eTJRbFZOQUlwcjVhRStEZ1AvdnF5aEFOV05ESmVUNkZkOUdYOEp1TDhuMzlEKzh2b0xqR3grcitvNU5BblpBWDhYRzBBTHdHNUhrVndDQmI5SU9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTE1NTlfMjZfMjQxY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmZkNDIyZTEwMzQzYzRkMDI2ODQ3ZGY4M2M0YzJmN2I3YmFmYTk5YWEyMzcxZTAzYzQ2YWRjYzkxNzFhNDUxNWQ2OTY1MWRiM2M0OTJiZjRhYTU4NzJjMzg5Y2FkMGU5NmVkZmEzZjc4NGRjOWUzZTFmNThkMzQwYjJjMTE2Y2Q5OTZhYWI0OWYyOGYxZmJmOTQxYmZlN2Q4MDM1NDI1ZjA2ODAwOGQ5MThjYzhmNjc1YWRkNzcyOTNkMDUzOTRkZDA1NDhmODM4ZWVkZDA4MWNlZjE0YTdkMGMzMmUyMDI4MzNkMjk2MDU4ZTgzYjhkYjQ5ZTA0ZmE0OTg4ZTA0MDVjZDBmMTIxYjgwNjVlN2U4ZDRiYjc5ZjJjZDBhY2Y1ZTk4OTg2NWEyOGNjM2VlMmYwZmFkN2M5YmI3OWJmYzQwNmZkN2E1YTYyNDZmNTk5YTUyNmU2YzAzNGNkMDE0MGQ2N2QxMWRlYmMzNmVmYzJjM2YzNGUxYzRiNmRlNGFkMjhkNjZjZGVkOThmYzg2MzRjYjUxMWI3N2Y5MWE5MjhmYjA1MGYyYTI2YTYxNWUzOWJiZjU5NzMwYzZiZTAwM2Q2MDM5MjMzNTY0ODEyOWVhYWQyNzJhNjQ2YWRjMDM4N2RjZTM0NGMzMGQ3YzlkOTVkMmJiZTY5NzVkY2MzM2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RWEzOpHet3XKyKRw6VxfrB7KC37r6bKe7H4d6BNr3nhbxBkZ1MkXVQ7nDo3RYTn1wviv72zrMQ4Xt3n4FKFhjA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_111559_26_241c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.224Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkRwc09NQmZWcldranNmMjVsVzdnOXJiV2hrQk9OS1ViVWpQRjBUa1RGZUc3dENFTkNZRVJYRWYweW9ReWdEeVRHeHh6dmJBenBibC9ybTA2L2ZTZXFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjRfMTVfMjQxYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTdhNzc1MTg3NmUyZTY2YTAxYmI3MDUzZjA3MWM3M2Y2NzY5ZmYzZDYzNjA3MjA0OWQxOWQwMmJkNGUzYTA0NTA4ZTdhMjZiN2FhOTI3YTUxZTA1NGZjNmM5NTEyMzcyNzI4ODk0OTZkOGFhMDZlMTc0MjNjOTNiYmQ5ZDg2ZDgyMTlmMDFhNjE5ZWQxMWI2NjAxYWFjNTI0OGEwMWQzY2U1NzQ1YzAyNzIzZTZkZTM2Y2E3MGRmZWYwZmVkNDc2OWQwNjRmYmYyYWEyODMxZWNiYjEyMzdiYjZhNzBlYTA0MzUxZTBjNTA1OTliMTAyNzdlNThiOWY0YjA2OWNlZWNlM2UyMjc0OTJiMWYzNzkxMTcyZWJiN2M0YmMxMzViZGJhNTNhOTM3NzIyZDgwMDcyMzRiYTMxZjg0NDIzMTg1MGNkODM1YjA2MDUwOTQxZTFhZGE3YmQ3MzMwNDViMDZkNWRkMjNjZDYxZmVkOGI0NDkxYWY5NDY0YzkzNmI5NjAyMGIyOWJiOTBkNTI1Mzk3NTBjNzJmNDEwZWUxMmI2ZTMxZmIyZjdlNmM4ZTBlY2E3MWEwYjkwZTY3ZDk5Zjk4YjNlZDVhY2UzYTRlMjI3YzAyNWI3ZDg1NDYwN2MyZjI1OGEzMDgyNTU1ZGE0ZGVjYWIwODJkY2YyOGJkNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vCnqleaG-upImdwDb3Q_XYHko-aRKMoxa0Y4N73f66R0VCwXWNNC5Sw0ingtF7r6bsRWrOSj2b9o3a2wlQgqcQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103124_15_241b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.227Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkFYZFJNdVhKbmNHQmhYWWtOQ1RSc3FiQ1pWUWlCaXJTOVpMSFJEenJYR0ZGQi9BL2ZUNnhQU21OU2dKeFMzb2s3Y3JvZit0SVRrV1RabFhMaHk3eS9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjRfMTVfMjQxYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2QwN2RhZjQ4ZTEyYWRjZmMxOGU3ZTc5MDI5NGJiM2E4YTI3YTBiMDAwNGNiNGM1MjdjNTk3NDE5ODNmNGY2OTNkZWUwOGEzNjRmMWI4OWVkMDlmNTA5MjBmMWRjODQyNTFmY2ZjZmU4NjY4M2ExMmIxYzcwMDc5ZGJkMTFmZWI4MjA3MjgyM2MwM2JiYTU3MGE3YjY5OGI3MzdjMmRmMjJmMGMwY2EwMTc5NWViYzk2MzY4ZmFmNTViMDMyNmI0NzEyNjRmYTdiODIzYzMxYjZkN2YyMDk5MTg4MmM2MTJhNmRmOTlkZTAzMmNkZmRlZjk3NGFmNzkyNzQ4ZWViMjM2YmIwNDZjZjE2NmU1Yzc3MzY3MTNjMGExNmYxNzcxOGM0YWQ5Yzc1YzhmYWUzOTk4YzAxZjU5YzM5NDgzNDQ0N2FlMmZhNmYyMTBkOWIxZjZhODRiZWQ1M2NkODE2NTMyZTViY2NjYzZiZGU5MTA3MDFlZmI4NTFmZWU5MzVjODhmNGRjYjExOTY4OGU3YzU1Nzg2NGFiNDU3YmE1MGEwNjkxYjFkYTJkYzc0YTQ0NTRiZWIwZTdmZDI2YWQxNWJjMDIyNmU4YWRmNDdhZjE4ZTFkODgwMDY0NmFkNjhlNmQyMzg1N2MwNjRiZTU1Y2MzODlmODY4Zjg2MzcyMDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oTFIEPfG2FXKByuQopm8rfZcETCAHJa4AKlhOcEpDu5pJlPQp1Nb0RjKZflHqFx5e74_JS04pBkITieGhaj_Zg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103124_15_241b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.230Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNWVmVQYjJrMkU2azVMRzJPRUloTFllSUlUUzJkR21TaHFZb0NUUncrcXJEaUkwK21veFpkeEgxNXg1SGxoTmhveW5yRitVY01RWkRIaEV0ZkMzUmh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjRfMTVfMjQxYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQxNjhmZjZkNjdlNDI3NjY4OTYwMGZlYjY5MTA4NWJkNmYzYjg2YzM4MjUwMTNmMzE3YWFlMGJkMmMyNzhjMDk4YjU3MjM2YWI2ZjYwNGNlNTcyYjZmMjRjYWM2ZjhjNTViYjhmOTliNjNhN2EwNDVhN2RmNTQyYzU0MGE5YmM3MmYwOTZhODFjOTllMGFkYjFiNWQ2YjQxMWY0NGU4MmE4N2VhYjhkYmNmZWVjNzhmNWMzZTI1ZTRmZmYwMmIyMmEzYTA3NjliZGJiYmRhZjk3OTNhNjAwODRlNDVjMDU1NzczMjdiMDAzMzRiNjVlMmE1MWRkY2QzMjk3MmZiMWIxMjQyMDJkMTc5OTcxNjMwMjQxYTUxNTBkNGMyYWZmNjA5NzU1YWExNTU5ZDYwZjM0Nzk4NGQ0Y2ZkZjI5NDIyNTg0ZmE4YmQ4MTg3YjIwMGRiODE1ZjY1NDQ1NDZjNjIxYTQ5MjBhMGZiMDNmYTU3ZDdhZTZkMmI5NzI4NDJjZWNkM2I2ZGY3MjM5YmJjNjY0ZjQ2OGM0ZTNiOWE3N2VjMTY4YTViYTFlYjEyOTgyNWZmODVhOTJhZDcyOGFlOTZiOGQxYWQ4YTM1MjM2NzdmY2ViY2UyZmYxOTVkODBlYjcxZmNiNmMwOGFlMDE1ODM2MDZkZjIzMzFjMTYxMDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SqmpvpoJovGgqJGUftAESB5meC-5LAIreK7kErZ1lB_Qg3juQXBZeNjxjXbK8UXG5-C_NmasqzBkzqXhIaWVbA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103124_15_241b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.233Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InRNL2VWUW5tT08rczF6UVBpeHBuMVRpNkZYRnZTVW1sN1dpbXZ3aExXWjh3VmQrWmVZUVRJTm54cFVxNXEwVXJQb2xIV3pqa2R1R1F1Zjl0dm84VnlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjRfMTVfMjQxYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTUzMTVjYzhkODY5MGRmNGM1MjRmMTFkZGRhYTY3MTg0OTU1YjY4N2I1MzIyNWM3NmI2YzQzZmIwZGJhMDdmZTgyMWUzMTM0MWE5NDVlOTBjMTM2YWJhZTYwMTc4NTg3YmQ2MTUwYjc5OWI3NzQ3NWEwMTc3MTVhY2E3MzA5YzlhMjYzODA1YTIxYmUyMGJiZjc4ZDFlZjZmMTljMDQ5ZjFiOWFkMzkzZGIxZTAwOTNiYTkwMjRjYmNmZWI0NjNkNmRmNzM1Y2Y3NjE4ODQ0MzZlYTgyZWYxOTU5Zjk2YjFmZmNiOTljM2MyNmJmMGU3ZThlMzJiMWQ4NTk1NjJmNjA2NjAzNmRiOTFiMDA3MDZhNThjZjJjMDYwOTZlNDNhNzM5YzUwMWQ1ZWYwMWFiNTg3ZGEzMjdkZjlkNGZjMmQ3ZDdkNzAwNzEzODc1Njg4YTAxYjQ5MmE0M2JiYTZmNDA4N2MxY2NkZmYyNzk1ZTUwMThiNmVmZjA5OWJkNmUwOTNhN2EwZWE5YTAwOTBjNjk3MTg4MTQ3OTAxNDhkYTI1MzYzMjE5MjNkYmJkZmI3OWViMDA1MDMzYWY1YmJiZWJkMTE0MGIzYWU3ZTZmYmMwYTJlODQ3ZDk4MTQzNWJjMGNjOTM5ZjU4OThjMTFlZjU0OWU2ODc0Y2RkYTRhYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.r3JMPWyLlZWIbOsUSHtNrzl5OBUbVev-Doanm1dvHec5TO6fasGm60BjUYlZIiRwje0Yy52Rt6TTck1MjSMXCg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103124_15_241b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.236Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBDOWVLMU9xZXJKaTRWUHptK0J5NmY1RDh5d1Rzd1RROGxtbDN0STByV0p1QVllbHN4elMwa3BzZHhkRjJubU1Vd0ZPeGcvZGJNY2NlMEs3bUVTZXN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyN18xMTA1MTVfNzdfMjQ4OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2FkYmU2NzRjNmE3NDkwOWIxNjc4MzRlZTRlNjg3NTUzNzkyZjZlNDFmZDFiODFlNThhMTFmYzUxYjIzOWQ5NTA1MjQxOTI0MzJmODg3NjgzNjZhOGM0N2QxODMzZGM5NzBjNzVlNzg1NjNmYmI4N2RjNmI5ZmU1NjZlYTJiZDk1ZTEwMmExNmM4YTFjYWM5ZGY5ZDkzZjUxMDQ1MGEyNzJlNzI1MWRjYzU3Y2E1MWZmMDk2ZWU1OWYxN2I4ZjZhN2VmNDlkZGFlOTgwMDAwNDBkYWFkMWRmNTU4MWY1ZTQ1ODVkMzY1NWQwODFhY2VlZmViZjZmOTQzODk5YTFmYzFkMmVmMTVhNzlmZGFmZDJkNGIyYTQ4MDlkZGYxNWUwMmY0MzM1ODkwOThlZDg0ZDkzYTU2N2JjMWU1MDM3OTdlNGY2MTZkZDQ5N2FjNWViMjQyODEyYzFiYmJiZjExMzVhYjJkZGUzODNiYjBlY2NkNDY0Mjg1NmZjZTc3MzBhMjQyOGRiZThmZTNhNmNjZGExMTJmZDQwYjFlMzI2MmUyNjNkYWIzY2NlM2VjZGYxOTAyNDhhN2VjN2MzMmQ5NDlmODRlZjY3NWIyODI2NWQzNmRjZmQ0YWQ0NTA0ZGViNDc4YjBhMzAxNWNlN2MyZTEwNDBmNzVmYWUyMzI3MDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Yn1i5_zb7AqOcmT8aI-Uw6XVETnTBhVVwTk9VHsGqnmOOX2DzfczJFwrjfln5cYphz5W6aR-n8W_HNFUdWcklw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230327_110515_77_2489_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.242Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJsbEZ3K3QybWsya1VTVzlyTERvN1NOUGsyeWR3YWd1eE1hN05Ndy9UaUdodHpMUFFFQUxWdzNaaFlvdW9kZ3V3LzhaTHRZeG9WOGM1TVlFVDhtQTN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyN18xMTA1MTVfNzdfMjQ4OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTU5MTg1ZWExM2Q0NGE1NWNhZGI1ZTI5OWQ3ZTE4N2UzNzZkMjM1ZDk2NWY3YzkzZjFjZTczOWQzZmYxNGIyNDc1ZWY5MjFhYTgzNDcxN2Q4MTk0NjE3MDc2MDMxYmRmYjkwZDRhNDdiYjZjZTQ1ZTkzYTE5MWRjZDJkOTI0MDAxZDkyMjA4ZGRmODFiNDVlNzY3MmQxYTY2OWQyZDhiMDg5NDE1MzZkZmY4N2Q2NjVkZmU3M2IwYzQ0YTlhZWY1ZjI2YTQ0NWU4OTFmOTlkODA2YjlhOTc2YTQ0YjhhMWU3OGZhOWM3N2RhNjIzY2Q0NGI4YThjMDJhMDE1MzkzNWQzMmU2NzkzYzA3NzYzZmQwZDdlZDk5YWE2ZTE0MmYyMmRhZjc4MGI1NGM4NWVmODZlNWViOTEzNjAwMmIyYzk2YTIwNzM3OTU0NTkwMzQyNDUyNjExYWFjYjI5YjcwYTY4ODBlNDRlOWI3MDdlMWQ1Mzc1OGZkYzY3ODkyMGJhN2MxOTlhMzAxZDFkODczYzNhYWIxZjllNTU3N2NkNDU4YmUzM2JlOWM1MzAwMjNiNDQwZGI1Y2Q0NGZjZGI3YTY1NzIxYTcxNjlmMGE5Y2FiNDMzYWNjYjVkNTdjMjdkZTk1MGZhNzJiNjYwNDg5ZTE4NjlhMTAyOWNhNzYwMWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3-igwW7vKOLpGznRukStyKrEVXvBaK_Vuxjfp6CiPIfCAlw2e0lX3reQg5Ar0lRGj8GhA1IwJSeqRO0xgI0tGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230327_110515_77_2489_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.246Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNMdFp3L0lTN3JibXQxbXRmbVV3OWRKek5STjI3UkV5UGI3VFdveGpGUEJZcWI5aVo3Q2JtVXd0VU9Rc2dlN0paWDZsNlVKNUV2SEFhY2pYanliTkVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyN18xMTA1MTVfNzdfMjQ4OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmY3OTZjMDA4N2I0NjJiZWVlNWQzYTY3YzBlZDk4MGZkMDZlNGQwNDFhYjRmZTcyYjVmYWI1N2ZlNDZjMGEwOWY4NzkzNGI0YTE2OTcxYzFhZDk4NmU1YWU4YmE2Y2Y4YTc3YmY2YjZmMWU2Y2Q4MWJmODYxZTExNTJmZDBiZmVmMWI5ZDVkOTMxNDNmMzlmMjIzMmI4OTc3YjMwYmQ0YzhhODBmMGNhNDlhMDUwNzc4MWZiZGNiNjZlNTM0NmU2ZThjMWE1OGJkMTU1OGNmYWNhOGFlMDFjNGJmYTc1ZmQ3ZDdlMmRkZjk1ZTQxYjgxNjY1Nzk3OTFjNzQ0OWEzMTc2ZDI3MmI1ZWUzNThmMzBjYjQ1MzdlNTljOTJkYzQ2YmFmYWFlNjRjNTUwYTU0OTg5NmM3N2JhOWQ4OGMzNTgyY2FmZmJlODRlZjQ4NGQ5Yjk5ZTA0YjEwMmNmODdkODYwNWE5ZTc0ODQ2MTQ5MTBjYWU0MzFhMGRmNTRjNzJmMTc3YzNjMWM3ZjBlM2RlMTk4MGJjMTUwNWU3ZTU2MWQ3ZjFiNjQyM2I1YzJiMWYxNGYyYzNlZWQ4ODc4MTdhMzNmZTM2MTE2NTEwNzgwZDBkN2I1MjkwNjQ3ZTczNjM4NTJiNjRiYTQyMDgwNmU0ZTkzNzg2Mjc0MWRkODdmMmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UMtjaY3Ko-4r6xjtEB3Yc-dhWbL_OQWNJFjZp-NLBYsJ-A5Fz2ZkW-xAN8J_1HyeZW_h4-lyqphL5F7HopXMOw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230327_110515_77_2489_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.248Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjFmZHZpaE91YWVSb1AyMUdNMlFnU1lhQVd3SVZsRGdhQU9jcXdha0RPYVc4WkJ3Zlp1UGphSmRxY05iQ1lmejRGNURIODhtYWxKbWVwQTNUZytJdWF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyN18xMTA1MTVfNzdfMjQ4OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODk3YWJmNjZmZTY4YzM4ZTM3NjQxNjAyMDA5ZDA0Mjc1ZTA3OGZiNTAwZTUxNjU5OTE5NDE2Nzg4N2Q5MzVjOTg3NDVjMTZiZTJhNmVmMTBmNjM4MDhhN2FmN2M5NDdiNjhhOGQ4NTdjZDE3MzM2MjJjMTZhOGNjZDYyOGU2ODk0YjdhMzZlNmI5MDY1NGQyMWVkYWRkZGRiMjAwYTE4ZTE2MjExMzBmZjZhNjc1NWY1MDBiZWVlZDRlNmZmMmI1MjdhNzQwMGUxOTg1MGIxN2MxY2Q2ZjczNDE2NTU3YjI4NGVkODFkMWFiYmIxZmY4MTE0N2Q5ZWMzNzJkODgwMjEyODIwYTI0YjI3NDk5M2Q0NjA5ZTQxNWFhYzM5MTA2M2MzZWI4YmZjZTM3ZTVmNTI5ZGMyNjEzZTkwNzgyNzVkYWE5YzYyNmYzYzk2NTRkMmJkMGZiMTEzZWJmYWJjZTY0N2Y1Zjk2MmY4N2IwODk0NGMyNjg2NGVlMjM0ZjkyM2U0YTFhZmM0YzAwZmNlZmNmZDM1MzQ3YzdmNWZiNTJmODZiNTg1YzY3OTJiMjM0OWQwOGRiZjkzZWFlODgxMmEwNWJiMzA2ZTM1NjI3NjkyY2VhNGZkMTVlYWNkMzkwYzc4OTljNzllODNiYWMzYmNmZWM0YmZhZjMwZTk2NGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.l-wxEBDFy4vCVsnbERFHjrNiLU8lw2pIFpAHFAxNYZN68cd5Ta-bfh9Jp71xAuWz4g7KegIYjS8IBWYbh94yuw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230327_110515_77_2489_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.252Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InQwT0lqVG1ZZ0lHSlVydlJtNTRJdjZiMDdrdnlNc0d1NHN3Y05xbWhrWURCYmZwWHM1N3NrSWtSS3lxNUczYWU5OEdGU0dnN2hreVVLNnkvbDRsdnBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxM18xMTEzNDFfOTlfMjI3YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTQxNDVhZWQ5YzVjOWUxZTk4MjA1MjZkY2UzZTNkZjQyOTQ3NmMwYWE3NWQ0ZTdkNjk5NTZlNjllMDQ0NDg0NDM5N2VjMzA4MTdjMDRkOTQ5NzZiYzQxNmE0ODM1NTExNWNkZDIwZmE4NmQxYjUwZGY3M2E3Y2Q3MGYxNzYwZmM1NzljN2M1OWNkZjY3ZjJjYzE3NGQyMGJkYmVkOTEyMDAxMzE2NzZjZmY2ZDYzNzQ0YTcyZjE1Y2I2MjA0YWQwYjZhOGJlYWVkZDM1Yjg4MDc5NWYwZDkxNjk2ZDQwNjJlNzllODJjYzM3MGJjYjNkNjc3ODQ4OGI4Y2JlNzVjYjA0YmY4ZDJlYTZjOTUxYjI0MTlkMmU2NzJkYTQyYmFjODIwMGNjMWU4ZTczYjkyYjZhYzhmMDVhYmEzZTQzZGViZDA4YmNmYzllN2I1Y2ViYmI3NzMxYjQ3M2Q1MWQwZWM0ZTU1M2RiZjg1ODMyMTJjNjhiMmQzYjRjZmM4YWI5OTM3OWY0NmVkM2VlY2RlYmFkOTAzZDFiNTEwMzI4ZmZjYTk1ZmNiYWEyNTdiNmY5YTkzYzQyMjdlZTg3MTEzMDljN2RjODM1YjM5MDNlMDdjNDczODYyYzU5YTk1MmUxNTJiNTdiOGYyNTJjODZmMzA4YTY4MmUwYTVjYzRjZmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OjDQsszDEUohps7YJp_kHAbLuMSevXHUb7TEV4j_U1-BWi9N60-3G_cnp64j_X8mQjqWIxrL9f7wmzFmhV643Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220813_111341_99_227a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.258Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkErMG82bVhRQ1gwcUVZeTFzTzFWMlJVN0EzT2gxSnJpNVN5amlINzZLVXhxV2FRV1FmU1JnNWw5Z1JLWUFVUkJTWlFJZFltajdkVTlzODV2T1hKSFp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxM18xMTEzNDFfOTlfMjI3YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2UyZTVlNTZlZDM1YTU1Yjc5MDFiNTRiNTUzNmQ2NGYzYmM1NmIzZmQyNWZhZGU0YjY5OTM1MzJhN2EyMjJmZjJiMTYyMmVmNDI0YzUwYTllMmU0ZDBlYmQ1MzUxOTkyODBkYmRlMjI0YjE5MjVhMWUwNDc1ODJiZmVkZDQ0YjZmMjYwMGNlZGExMjYxODNjYjliN2UwZjFkNjcyMTJhNzAwYWQxYzAzZDliNjc2MThkMDRhNjczNmU1MTA0MjMwMDFhNjRhYTZjNzNiZGZlZDk0N2RlNTgxYzI4MzIwYTczOTBkNWVkZWE5MjljMTZkZWE4NDBlNTcyN2ZjZTA4Y2YxNzY1YjQxY2MzMzRkZDIxZWY0M2Y2ZTNiYjVjMzg3MjE3Njc2YjM0ZTRiZDc3NmZiZDlmNzQyNDhhZGM0ZTBmOWQ5OGIxMTEzMDFmYzQwYmZkNzBkMWMwZWQ2ODJhNWNlNDFmNjgzYTc0OGNjMjRjMzU0ZWZlZjk0ZDIzMmM4ODdmZWUzZTBjZWEzNTU2ZDlmYjQyYmQ2YjZhMDcyNDNiN2JhM2VkMTQyM2YzYjcwYTc2NjM3N2YyZTg1YmQ5OGU0ZDUyZGVlY2FiZjM0M2NhNjJkMDYxNjBiZGVjNGQ4NWUzYmI0MDEzY2EyNTBmNDY3ZmU1OWQ0ZjI2ZDZjMjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.z9wTh7SIaPEBuZFBEPVU1Lt8ocZvqcs-8qLcLiSqfz_uM6l78NnHS_un1fjBwDMuwgP5cRJgYmyFS8pI2G0OSA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220813_111341_99_227a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.261Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im1qK1Jsd1JtbmwxTTRlYnZTeml6RVRUN0tidVdYT0dJS1JLOVAyc2x0T2tIWUpBRGlGWGwrWmxXcFd2bFp6N2cwQ3hiR0lsZ05rbVVxRkNHbzZ0YlJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxM18xMTEzNDFfOTlfMjI3YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWRjY2JiMzgzOTI1YjFlZGYxMDkwZGY4MTg1MjBiZmE5MGUyMDhkNTgwNDk1MTBhZDU3NTYxOTk0NTZhOGYxNmZjMjFkMDhjNjQxODRlMzJmNDNiM2FlZjNmYzM4YzViNzU4NWU4ZDNlODFkMzM1YjdiYjJkNmFhYmU2ODVhYzcyMWU3ZjZmMzBjNmQ5NDE0YzVmOGQyN2I1ZGJlMWVjMDNkNDNjNzQ3MzlmMDg1N2EwYzAzMjRhNTI0NDFmMTBlY2VhMGFhYzgxYmZlOTE1NWY5NTJkOTgxNzRhOGJhMGE1NDNkYjNlYjJhOTc4MDQ0N2Q1YTU5ZDYxMjhlMjQ0Nzc1ZTk0NzIyZDZhNjNiMDU2MmZiN2NjMGYyZjVlNTZkMjBiMGExNjdkNDczYTQ0MmYyZmQ1MTRlOGFiNGI5YzkzM2RlYmYyNTgzNzcxZDcxMzA2OGFiMGI4MzVjMDM2OTJmNTVmYmE4YmQyOGU0MGI3YTQ4YmE3ZTMyZWFjNjJhNmY5ODczZTMxYmVjMjM4MDZmYzQyZWE3Y2JhOTllNWNmNDFlN2FmNDJhN2VkN2RiZDNmOWE4ZjVjNzM4OWFhYmYwNWFjNzgwNDRiZjFkM2QzYmU4M2VhOWFmYTcxY2YyMGYzMTIxN2M1ZjA2MTIxMTYyNDI4MzJjMDkzYmM2MjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ieGCXnBNAhFDD8qeix5zBJYR5dr2jE1cTAVNgZx7Ly6bci6NMfftzgkUrLHo8GxQe6u7cFtsqifE2fDhR_1V8A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220813_111341_99_227a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.265Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZlcWI0R0tCOFRWbVJsTGhGSEw5azluelhKeXdUZDNZbEI2UkxmbU1yd0R6eE1mNGZUM01iby9RWjd4VjBxMzNJNXdmSktZcGkvNjdvWlROaFlJUjNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxM18xMTEzNDFfOTlfMjI3YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmU4MTdhYTA4YmZjNjVjNTZiZGQwNzBiZGE0ZDQwNTZkZDBiYjU3ZjI3OWEwNzc1NmU4M2Q1NDFkOGRkMDE4N2U5MzM4NzllNTBlMTg4ZTA3OTE0YzVjN2I5OWMwYTQ0M2YxMDhlNGM1MDI4Zjg0ZmNjNGIxN2FhYjg3ZGEzNzI3ZjRlOGZiZTc5NjgyZDUwZDVlMDE1MDk3NGY5Njk1YWViOWVmNmYzMjBlYjQ5MzE1MzIxNWQ1NTY4YTlhNDNkOGFkOGM1ZWE2Mzg0YzY0M2UzYTM1NTMzZGQ0MTdmNDZlMGExMDQ3ZGNhN2NlNDU0MTc2NmE3OTUwZGU1YmMyZDU4NWU0NTkxODU3OTEwN2MyMzdmZTBmYjhjOWI5MjExYWRmZjQ3MzM2MjQ3YzRmZDFjNDc5NTUzOWM0MzlmMDBkNmU4MmEzOTA3YjYzMDM1MDIxNGU4NDFhZWM1ZThiMTMwYzA0NzRkODcxNzg2NDY3YWJmZWUwYjU3YjMyNjZlNDRhMjBjNDJlMGI1ZDM5MjcyYjc0Y2NlMTE0N2E1OTAxYjhlMDQ2MjFkYzZmYjJhOTQ2MjZmNTY0YmVhNzMwNjAyYWE5NDIzZThjZGY2NzdhMTg3NTVjYThmYzA1ZGIwZWY3YzczNWI0MWJlODJhNDQ2ZWE4MTAyNmYwY2ExYzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6dSri3vFrQEV2UHD_8pIUmtub4uSO0UpiMrmq2_WW0hvDSxveQdwWNI74ha6FtN5Z4IGelKbVrzvmyxEGYRS1Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220813_111341_99_227a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.268Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlhsNmVsYzZYODFDekxjYXRvcUhTRGlyeFUyVmRDSWlvK0RNd1cwZ3l2RnVUNndQRlBDOGlzZWF2eGlRU051VWt0YzVYbmhoaHNzNHByQkNqeC9qUUNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDZfNjRfMjQ3Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGM1OTJjNTZmNmI2ODQ3MWU5NTdlYzczYzJlMWRkNGEwZjMzOTk4YmQzYjk4OWI0YWNiYWJlNWY3MWM2YjQyNTEyYjkyZDEyN2I2NGUwMTE4YTZhM2QxNjFkN2U2NTdjMWJjNTJlZTk2NzJhNGQ0MzhjMjMwYWEyNWRkYTE2YjEyY2Y1OTZjNjU2ZDdjNWQ2NzY2ODAzZDVkOWEzZDMyYWUxZjVlMDYxOTg3OWQwYWNhYTk5OTU5M2YzMzkyN2NjMzhhMzI5ZTBiOWMwYmNlZjg1YWFhODU1YzA1ZTk5MmY0NjVlNmY2OTNlOWNhZmI1OGM2NDMzYTkwNTUxZGYwNzQxNTM2N2ZjYTc2OTYxYjM3OWFhZDkxODVlMWY0ZGRhMmUyNjg2MjBiNThmOTNiMWRiMGUwZTg5ODI1NGJkM2Q0ODg2NzAzODY0NGExZTY3ZjM2YzllYjhjNjZiNjAzODRhZGVjMGY2MTcxOTU4YmYxYmEyMzQ2ZDUyN2JkNDkxOTM0NTg4OTRjODFkNDk4Nzc2YWI0Y2U5NjdmMDMzYmRhNjFjOTQ5OTNkYzdhZjE1N2M5NTZmNGRhM2M5ZWM3YzlkZjUzOWUzZDgxZDI3OGYwYTQ4ODFiNjNmNGM4Mzc4YzkxMGRhY2RhZTgxYjBiYzU5ZjE2YjViN2NhZTUwMWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4VvYJm2LsT25GbFOfbmImStTArpIqOls0fm6gZujN6ydWqo_5aNLNEsqXPDZ8XVCTTUFJ12bOgbPzaC80ZrHFw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110206_64_247f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.271Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imw0cXN4WlpjWDJQSmRsZkF1RzJSMTVIemZQM0ZuUDZPWlcvQW9LdDBBTXJRUjUrYTE2NEV2KzN5MlFKQ2c1alRFM2pTWUNpOHNSbVJ3ektqcmtIMGF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDZfNjRfMjQ3Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDM0ZWMxMTM0ZGU2YTQ0MDc3MDM2OWZiNDhlNDFiZGJmY2UwNzhmOGZlOTcxOTQ2YzUzZGMwOWY1YTU2YTU3ZGE3ZTNjNmZhYjcxZGE4MTQ5YmU4Mzg3NmNjNmMwM2U3M2I1MWM2YjRlNjA3ZTg0ZTE4ZDVmNjQ5ZGYxYTA2OGY1ZGZjZTQyNjU1MDA4MDBhZTM3OTA2NWVmNDQ5YzA1NDliMzhhYWQzOWVhYzEzNzRmYjI2MjcyZjgyZGU4MjQxYmM5ZmIyYjYwZWM4MTczNDdiNGViNTM3Y2E5ZWQ3MGMxZDM5Njk0MjRjMjdhYWEyODM1MjI1MDk0N2ZiZmM4Y2FmZGM3NzYyNjcwODcxNmRiNGExMjg2MDJiMjk4YzZkMTNlMWI5Yjg0MDg5ZDQ2NWE2ZGUxYTQ1MzE4YzhiNmQ3OTZjOGViMjM3ZTVmN2ZlODc1MjhjMWI5YmVjYTQ1NjZjMTNlNmNhNWQwYjY3MzY3MjBjOWY4YjMxMjYyYjRlNzQzMzYyZGU1YTI2MzI4NmFiY2RkNjJmMTIxYTM3ZTRhYjU3Y2UxNjlmYTBkMDlmMzlhY2EwMDVlMTU1MjEwM2MwNzRkNzJlZGE5ODY1YWViYmQ2ZWY1MGM0MmU4ZjRmODE4MTI4YmJiY2QyMTYzMjQ2OWUxMDM2NWYyMDgxOWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Bdwaa4_8OS7e1r8kemS4UKpYhxHlKcvlhcVE943Qgq9H18hulrmmLZxEvUMf6flDUfZ3z2mAqRWcNzcgsbjRAw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110206_64_247f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.275Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlVDMWNzRmE4b1E3YVBveUZCeXNtQmRjZ1hlSUVQU0tLdm9saG41b2JhcGpQMExpQkFCVkphTHEyNDBSTVlnRjZqM0YrSjJjcjJWay8xWSt1ci84ZUpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDZfNjRfMjQ3Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWU5ODUyN2EyOGRkNzkzZmU5M2NhZDFmYzdmMWQ2MjQ1ZDk0ZmM5MjMwNjQ4MmU1ODczYTQ5NzQ1YTA4OWNhMmI3Yjg0NGJhMzlhODJhMWIyNzJmNzQ4NzE1NWYzOGQwY2Q0NWZkODgwNmQ0OTM5ZjgzMjNkYjYyOTQyOGRkMTMzYjBhZGM1N2JlODU4ZWVmOGYwMzRhY2M5MzgzY2Q3ZjY4MDgwMWViNDllMjQwZjY3YmI3OGY4YjJhODBjMWRkZjNjYjZiZmMwYmRmMjQ0NDZiYTQ2NWI1YTMzNzQ5MDZhMmM5NTM4NDY2MmE5OTk2NGVhN2M5YzQwMDY5OWRhMTMwOWY4MjQxNzNmOGNlZWQxZGMxMThiMjk2MzA4MGUwZmFhNzlhMmQ0ZmQ0OTU0ZDA0YTc5MzY0NGI2NTY1YzVhMzk1ODViMGI4MGY4NDEyNTcwODE0MmZmOWIwMTgwZjkxODg4NTBiYzQ1Yjk0MGUzYjI2NjkzYzBjMWYyMmIwMWVjYWQxMjA4MDcwMjMzMDc3MDhjZWM0ZjQ5ZmMyOWZmOGNjZTM1Y2EzYmEyM2VmZTA2ZTgxMjE2M2FlYzg2MTY2Yzk2ZWFlOTUwZmIzMDcxMjFkMmU0Mjc1NTU1NDg0ZTM1MDI2ZTYxMjQ1NzEzYWQ4NjIzOTdiYTJlYWU5OTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QryLv7dsS0Y6BHZDAvI1UQlinVnS06uKjmGnAWUL37dKOF6BmQYyrHk5oeADBu4dM6WkWv29kpnzKSIFu7zb-w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110206_64_247f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.278Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlGR3owMk9jSUw2RklrVXJGYldIcnZkSEFxdkM0VnR4SGRleEdQYVJwVUh4dE9McURaaHpvWlE4enhLMWVBdVVaTVVCeXR3UmlyZTJtNTBlZVJGTGZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDZfNjRfMjQ3Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTY0ZmQ4ZDA0NzBkZTkxYmNhM2M3ZmNjNDcyYzIzN2ZjMDJlOTYyZmRmNTQ4MzQ4ZWFmZGE2ZWQyMGRlMjNlNTc0YTRkZWE5Zjc0ZDRjNTQ3Nzg1ZTczM2Y1MzZhMmQwN2MwMDg4OWUyMWNlODE0YTY2NTRlNzE5N2U4Nzk3YWQ3YjJkZGU2NzI0MWYzOTAyMDM5Njc4YzU3NjQyMDczZjNkNDk2NmFiMGMxZTU2YmRmYWM2ZDE5Y2ZiNDhjZmRjZjlmYzZlOTgxOTEyODI2NjQxNDRhMjIwN2MzMmVmZDMxYTE3ZTZhOGYxNGViN2M1OTZmY2ViNWZmMTcyMGQ1ODQ4ZGRiNWU1MjI0NzdjNDNiNTQ3N2U0MDFhMmRkMWY0YjBjYzhiYjE2Yjg4MmE3ZWZiNWRjMzI0ZmIxNTY4YzRmOTk4MzQ3M2YyYmExYjNiM2EwNGU3YjczNjc2YzFiYmZhZGU5OWI0OWFkNWUzMWUxYzA3OWRiOTAxNTM3OTFkNzUxZmQwZTVjMDgwMzM5MzQxNGU5OTYxODExYzI0OTEzN2MwZjBkZDU5MjI3NjQyYzQ0NGUxNjg3ODhlYmYwNTUyNzQwOGM4MzFiNzM2MWI5NjMyYzM5NjczNjVhZmY4YjJkNTgxOTdkZGM2NDg3OTY3YjY3MWMzNmIzM2M4ZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Wx0WLT6_hM6Aj8H3Ziefh2X3mZzSSVQKC4u5tcszYtO1zPdEF5RI1gShyBRXJHLCqCK50Tji-xcuU_h6eS18SA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110206_64_247f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.280Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IktKaFRBbVBicU5FR1g5R01pSDAzcWxnemlMVHNLUk4xWTdnbnNXYWdhQnVHVWtkZUhWOWNRZDNWU0JhVGo2a3Z5b3hXcHpUaDdQSnZteSt4RzUvUmV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyNF8xMTE2NTZfMTZfMjQwN19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGJiMTY1ODY1NTViZDQxM2E1MWI4OGU4MDc1MzFhZDQ4ZjFhM2VkNjUxNDE1OGU5MTRlZjMzMTMyYzQxZDE4M2UwM2VkNmQwN2RmYTJhODdjYWZmNjY4MjY0MGQzZjQxY2YxNWY5MzQyN2Y3NjA0ZTNlOTRmZWU0NDJjOTUwNmE4MWI1NzNmMDliN2E2MDg4ODNmNmY4MjEwOGJlY2M4MjdhYWY0MGE2N2M1NjZiYjFmZDNkMWM2YjA4MjI3MjJlMjE3MzIyNTZjYzk2N2MwZDFhZTE3MTgyYzQ1MTZjODY4NTcwZjE3MTYwNDkzYzk4M2NiNWM2OWNjMzQyNTY2ZDE5NDNjM2FmZDVkNTA5YzE5OTRiMGU5ZTQ4NmQ5YjJmOWViNDU4NzU1ZjBlNmI3NmE2ODhhYTc5YjNiYzM4NzE5YTAzZDIxZjEyMzliMDlmZTBkZmFjNTNlZWViNmYzYzY4YmY2NzBjMjEwYmNhN2JlOGRiZWY4ZWU3Nzg5YzAxOTBlMzI1MjZjYzYxOWYwNWY5ZmRiODdkYTQyOWFlMGE4ODA5ZmNmODIyZDQ3NGExMTczMjFkMDIyZjAwYmIyMTBhYzA4YmYxMTliODUyYTIyNjA3NmJjNTU2NGYxYmMwMWQwZjJkZmM1Y2JhMWZhYzRhYTA1ZDQ1MzJiNzEzMTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZjGXNAmDFmNt1WPcEGsuiiFj8hExU7o_Srtl7XNsjFO1vztZIOMFeRdSL3dGQXHAgc1Y-JX_WDNVStfy6v5b2w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220424_111656_16_2407_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.284Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikxob3lKYlZYS0xpWmFrdit3eG1saGkvRWxpYzQ1RmlYRkE0K1B0OElIbEdGVEJuaWJ4OFNzMWpXVThlSnE0TTZ6WnBxekNOSjIrMDNVNzNmTDZ5TzdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyNF8xMTE2NTZfMTZfMjQwN18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDA0YmY5NGZmNmFjODk2MWJiMzcwMDg3MjE4NmEzODZkMzNhODBmMTY1Y2RiMWRjNmIwNTkzNDI3MDg4ZmExOTk5YjlhMTIzOTViMzNjZmVjYmIwMmM2MGJmZTM2YWY2MjlhOTQzMTdiODdiN2M5ZjAxZmFlOTY5MWIyMjgyOWI0YWNkZjI4ZDQ1N2E4ZjVkZGY5MTdkYzQ1MjI3YjUxYjk1NWEyZTFiNGNlOTQ5ODExZGNjNDczMjQ5NjJjMDlkZjRhOGFmYjgwMjE0MmI4N2I4OTliOGVlZDc3OWE1YjMwNWE5ZmY5NWQ5MzUzODUyYzU4MjA4NTNjMDkzZWI2NzFhYmZjNjU1M2RkMTg0YTdjODc1ZTI2OWRkNWExMjcxNDZkOTg1MGU5OTEyNmZmOWMzNDIzYzhjY2I4ZDRmODViMjdhMDEzMjJmOWU2Mjg2MGFkMjMyMTUxNzFhOTBkYzZhZmQwMDc5MThmY2M4Yzc5MTIwNmZmYWNiNjlkNGE1MGRlNDNiNzQ3NDczYzA2ZjM0ZjkwZjE0ZDk0ZjBmNDIyOWZhMTgwZGNiMjQ2ODI0OGNjNWEyMDM1NWI2MTBlZDJjYjc5MDJkMmU5YTgwNDhmNWQ0MTc5NDdjYzg1NDM5OTk2YTYxNGVmNTg3NDZkNjMyZWQ3OTYwMjE3YjQyY2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6D3XrKf_BJ2mHX-WUXi3n5EfRkVsS5CK304zavrbpsVsc7rC50QqYUju6TwbrtjA77-8eIJtzH-uLFBiaFbTCg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220424_111656_16_2407_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.287Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlRTFAxYnYvQjdTNEpSeUhsS0tGOVBnWlBpZkpoait5dndnMTQvUEF2ZEJjVGMraTZBNkwvT0cvZXdETjFRMng4b2ZKbk9SQTZCSXZKaEdJcnNLYVlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyNF8xMTE2NTZfMTZfMjQwN18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjVkMGUxMWMzNjdjNTMxOTQyODNmMDkyZjUyNTRjNTkyNTkwN2IxZjRiNmE1NWRlOTRlMDdhN2E2ODVhNzlkODZiMDA4NzdiODlkN2JkYTU3MzI5OGMyYmE4ZmQyN2U2Yjk1ODUyNzIzOWRhNzc5OWM3ZWY2ZTVlOTFhMDg5MzFkZThmZDFmZTRjMzkzZTYyZGZhMDk5M2EzZTllNWMxZDUxMjc3MTdhN2U4YjE3NDYxNWJiN2JlYmQ1NmEwZjMzNjY0MDMzMTFhNzg3YzRmZjJlODNjMmM3M2RhY2E4NGVkOWVmYTBjM2Q2NzVmNjU3NGY5YjJjMGM1NzU0YTc0OGNhMzdjZDE0YmJmNGNhNDM2ZjVlZDdlM2U2MzcyOGQ1ODQzYTVhNWM2YmNmZjk1ZWEzOTBhMjY5ZDcwNjczYTBkZDgxMzQ0MTA3ZTJmNTE3MjBjYTM0N2NmMmE2NjZkMDIxNWY2OTBmNzRkOTYyYjY2ZTBkMGQ0MWFlNTg2ZjQ4OTg0ZjFlMTkxOTc1OGFjZjdlMjZiOTA5YjI0MGIxMTMyZmIzNDNlOThiZGMxZjNlMmI4MzZiNGRjYjM0NzI5Yjk1NDY5NjM5ZjY0NmM4NjNmODQ0NTY1YjRjMDg0ZGJmYTNiMTUwYjA0MjRkNDkwMTFjNmM4OWVmODc3MmY5OTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SzTgVsOhpXrTRbxtijqQWWts1HfsnPfw6lWkYai5vd-xLiDl_wtX8ZMdtd8jF2TbZZzCn1HzgUpkTKhOhUYJ4w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220424_111656_16_2407_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.290Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlcrSXBVem5sZlN3Q1I1cEFoOHlIVUVTMXVzQWRPREFEWGpTRmV3UlRrQmxIOHliRGI2cXJLVnVobE5YdWpsZ0tEUUZaNjRBdVVHcXp5TWtxN2RrWFdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyNF8xMTE2NTZfMTZfMjQwN18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTcxN2FlZmU4ZjdiMDlhMWFjZDdkYTUyYTU1ZWVkNzc0ZWUyMDM3NzQyYWUwODA1MzQ4MTNjNzY3Y2I0MjE2MDg4OGFmZmVhZDU5ZTZhZWE2NzZmYWQ2ZWMyZjI3Zjk5NzBkMmZhNTU3NDkzMGRmMDZkNWUyNGEwNDNjNmY0OTA2Mjc0ZGMxYzM0NjdhZDA2ZmJjNWY1ODk4ZDc1YzA4NmQ4NzkwNTEyMTM0NDI4NTViM2NhMDIyZTc2ZmY4MDA4ZDUyMjczYmE2OWE3ZDMwNDFlMTVmZDU5NzJhYmFhY2IwMGNhNWI4YTZhNWYxMGZlZGViYjEwNGEzZWM0ZGE4N2E5ODg3NDY1YmZlNDU5MzJiYTgzMjQwMzVkODJkMTQ0OWNiYzRmN2VjZmY3ZmE5MGFkYTM4YzdmYWJiMDlmMmFjNmMxN2I4OWUyOTQ0ZWJiMmQzZTNjZTAxZWEwOWZhNDAzZDQxYzQ3YTNiNGMzZjJkM2MxMTIzYzFlYjMzMTRjNjU3NzYzNGRhYTIwNWE3MDNlY2JhMmI2ZTg0ZTFmNGExM2E4YjE3MThlYjQ4ZmQ2N2I0ZTExMDg4MmE0ZTExYjA4ZDczMDMwOGIxODA2N2ViOWQ4ODQzMTJjMjVmYWI3MjM3MDRkMWZlZjdhNGJlYmU3YTJhN2YyZTEwZTgwMDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LrNDwkCnKq1_BusRBXmbq8r2eMN3oaTq2YpSPOXslq-RaVZeIcIZr_0NL9ssZU8sXR2eL7VwcMVIJQHsW7wwMg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220424_111656_16_2407_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.293Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlJd2lmYmErc2lrS25HOWV2THZHMmNwNmJZRzV2cnUwTjFVSkJML3pNT3Y1TEg3K3FkUFpTNm5XQjdHRGp5aFRiL1M2TEZRbXR4RW9aSWF2OThjSTZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzdfNDlfMjQ4ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTE4NDdmNmZlMDgxMzQyNmY5NTQyODkxYWE0MjBkNDk4NDE2YjBlYjhiMzk5ZmE2NDdlYjJlODk3MTM1ZDU5NWMxODg2ZTc1MDUwOTZjYTUzNDM3Yjg3YmUyYmM1ZjM4MjdjOGNlNWJmOWQxYTAwOGEyYTdjZGEyZGEzYTE4MDU1MGZmY2M2YjcyZTlmN2RiMDQ0YzM3NjRhNTdhMzBhMTcwMmZlMDg2ZDA3MGExODNhZDlhYWQzNDM3MmMwZWMwYmI5NWRhMzIyZmRjMjhkOTMzZTdlNTdhNzNkM2YyODBiMDQ5N2YxNGI0MGUyMWI5YWVlZTNhMmYzODQ5ZWY1NTM2Y2UyYTRmZGRkYTY2ODUyNDkyNmI5MzA4Yzg5OWU0YjdjYTk1YzMxMDg2YzlmMWE0ODAxYjg4MzM2NDY1ODcxZjhmYjkwYzg3Mzc0OTVhOWNmNDJhMGQyNjc3MDM2YTUyNWFhYzhhOTgxNWFjMmNkODgwOTYyZjFjYTQyNjkyNTE5NDZkNGFhY2E1ODQ3MjQ2OWM5Yzg4MTA0ZWM3ZGM5YzNkZTVjNmVjYzI1ZGRhMGIwMTQwMjVmYWYwZDRmZmY0ZGMzMTExYjE2ZGRiNjAwYzVjNjU1ZTMyN2JmYjE4MWY0ZWFhMGQyNGFiZmRjMzI4NjRjNWY2NDM3YWRhN2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nd12O7YQ69g8Fh0IYsw5Zn1Rz7BX3eVq7pWVvpy2WvCK9kCGt7_oiGJ0-MAHnMyd4fvfvH8NFp2f_C4d7aXcGw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105837_49_248e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.295Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IktPSm9qNmZnbFdpQUhRWWtJUGNESUlMaWVYYzd0NnBnN1N6N1ZiS2xnSUQzalRSMHVzTUY3dE9PcElsZkt5QzRwcDNNSTBIREJjVWt6ZThrVXdBV1hRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzdfNDlfMjQ4ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTgyMWYxODkzNjk1ODdhYmUyMDg3M2IxNmJlZjM3MzkzYzI3YzkxNWJmZjdhZTE5MjMzMTc2ZmVjOGEyOGZlNjNiNGJlYzI4MWY1ODk5NzE2MGMyNjg5NzEwMmQ3OTk3ZjliN2NkNTJmZGNiOGIwMDU2OWY4N2YxOGI3MDI0ODgwMTcxMjE0ZjNiMGRkZDFkM2UwOTUxY2I4MTc4M2I2YTBmMGRlYzVmYjdhMzYyMGZiODBkNWE4YmVmODEyNDFkNmJmZTA2MTdhMWViNGI1ZjUzMmYwODk2NjZlZGU2MmYwYWMwMDk1ZDE5Yjk3MjA1MjMzMzY1NmE1YzM1ZTFmMjlmNDAwZDdiYTIwYTMxODZmM2ExN2MyZTAwMjljZjRjMTMxYmQxNjFmZjFiZWUxOWUwNDY1YTc1MTE2MmE2ZTUyOThmMmI2MmJhNGVmZDdlNWJkOTlhMzFkYWM5MmY2M2M3NDk1MTljMTU5NDBmM2QzZTc5YjIyZThkNjg2NzdmOTMxZjRmMGQ4YzYzYWI2MTJlNmRjNWYwZWVhODYwYWRkZjdhZTZiMmJkNjg3MDMyYzFlZjBlZTRmMjVjMWMzN2Q3M2FhNzdkNTE5M2Y1OWVkMzE3ZGQwMjE1MjNjNzdjY2I0ZGFlY2NkYTVmZWI4YjA2ZmQ1NjgzNzkyMTZiN2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.h-h1hB97UPy0eXjJ_sf5DLd-vrIrs-q_twihbR7y8ihUaxpxvd65zc13CEfw2i-pgU3NS_QXC1KjCCimwBXwkA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105837_49_248e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.298Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikl2SEVtN2tDaE5WMUJDYVlodTJwOXNIUmtITnlDa3N3MkZibzhFWFV3aGJFZVVScHUydmF1dzNpaW1hUXN3Ym5Qa2x4WU1Ub2o2a2NHR2tmTFYzL3hBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzdfNDlfMjQ4ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGY2ZGIzOWJmYTU3MmZlY2IzM2YwYWQyYWRkMDYxOWJhMTA5OWRlMjI5ZDIzMTk5NGEwNDlhOWFiYzlkODM5Yzg4ZmM1NTRmNDNkMTU4NGUyMmEwN2ZmZWVkYjhlZjgwOGUzOWU0ZDgzNDBhMWYzZGIyZjExZjQ1MDNhNDk3MzhlZGUzMjM1YTAyMDY2ZjlkZmEyNTQ3YTgzZDNmMWI3ZDUxMGFiOWE3MmIwNDE1NjkzYTg2YjI3NTEzODY5ODExYWRhNjg4NWE3ZjUxZTI3NGFjMDQ0NzQ3ZTMyYTcxZTZlZWUyZDA5Y2M5ZjIxZGZiMGM5MzIyZTAyNDI5YjA0NmNlMDA3N2VmZmRmMDNlMWQ5NGRmMzdkOWI0YzI3ZjM1ZGM3ODNkMTJkZThiYjJhZGZmM2IzMWM3ZjE1NjdhZGZlNWRmYThhODE4NWE1YTVlNWNmMDI0YjM5MzFhNDQ4NTg5ZDgzNTI0Zjc0NTc5ZDEyMmE0OTZhMzQyZWE5NzMzMTQ3OTJhNzU0OGY3MjJjNjBiZGNlZjcxNjcxN2YyOWU3YjU4NWY1ZDdhNjIxZjVjMmFmYjc0ZGRiMzFjMjZmMDE5MGRkZjY0NzJlZGQ3OTQ1YmQ1YzhiYzhjOTdiMTI3ZjI3MmQ4NjQ0Y2E1Y2ViZDFhMWQyODkxMTI5MTM2NzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rfa8nxPxLlX2YT5ocvi3g_avXA1dr61-eF-ZePxF36RWqskyglpsf6f5G29uA_ek6DTcFAIS9FuNxBFiFgmbkQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105837_49_248e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.301Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVTaTEyMjRia0ZxTzBOeUdZQkFoam5sSVRJVER2Zlg3Q2ZPOUk5SkoyWE1kaW1lUlZXdjRhU2M1T3J6Q0lYVkRvTXZVOGFwZjArSGhhZTVJby83TXlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzdfNDlfMjQ4ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTExNmZiYjdlNzlmM2I5NTY5ZjhkZGE0M2MxZmE3NDM1MzNjYzBmZmExZTUyMTc2NzliOTQ3NTk3YjM2NWQ2ZWRkMGYwNmY1M2RhYzE4ZmEyMWRhNjBhMDM5NTg4MDkzMzY0NTEyMDZjNzAyODFjNWNkYmViM2RhYTZkNWFmMWQyMWM4ZTEwZDcxMDk2NTU4MjZhZTAyZTIxNGQyODJiZTBjMWRjODYwMzdmMjlmN2NmNWUwMDY4Y2IxODk5YjFlMTE0YzI5MGZhM2I5MDJmZWI5YmM2MjlhOWQxM2FiZWRlMjc3MTZiNGQxNjYzZDdhMTk5YWJiNDI1NTJhZDM5MzUxMTczNjNmMTM2ZWZjYmUxYzc5MWY5MmVlYTE1OTllODdiZWJjYWRkMmFjOTA4MzE2MDJlY2JiZDQxODNjNjU5MzlkYTYxZDRhYTcwZTIwYTU2ODM1NmNmYzI5MTg4NjQ5NmJlMWIwNjRhYmI5ZjFiNTk0ZWEzYTc2NWI5YmM5NjI0NDdmNjU2NTI0ODBhNGQ0NGIwNmFmZDIzNDBjMDE1OThkMjIxYjZhZmIyM2M4OWViMmE5OTI4NWQ2ODY3NzlmM2MzNGMzNTI1MzVhYTc1NjUyZmEyNmFiOWI5ZmFmZjJmMDUzNjU5ODA3MDg4ZjhmNWVhODAyZjE5MjVjZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RyL_1sEbzSGni5Qz0Yye8ooqGT9YL8cSlb1cdYmm75d9iuEywnvoLtwsgcaNNAURPKflxxKIJaF30XmphGj2cQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105837_49_248e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.303Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtGVjAzb0M5ejRiK2RJYVpRRXY4V3FhMjUvVUZzaTVUQldpZ3J4ZC9YTzB4cG81MmV5Z3NZV3hGd0hPcXM4UUJraW1mWEF5UFpFSk0vOEdWenN2ekFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxNV8xMDI1NTRfMjNfMjQxYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTg0ZGVlNWE5Y2IyYjk0MTg5Yjc2OTY4YTBjYmNiMGQ3MjExYjM4MWM5OTY4MDEwNGMyNmU2NTQ4MmRmYWQ1NTEwN2YxMTNjODE0MDcyOTAxZjQxYTY2ZGY2MWRiY2E3N2FlYzYxOTkyYzEwNmE0MGQwNGZkNzBlYzQ2NDU3Mzk0N2EwYTE1MzljZmIwNDM4ZmYzMWVmNDM1MjI4ZjY2ZGRhOWJiMDIxNWFlNDI3NTY5MDllNjFmZWExMTNiY2M1MWEwZTM1NDRiOGIwNjZmNWEwZTM2ZTdmNTIwOTFlOTg5YjY1NTFjMzMxMjlhNTAwNWNjZjQ1MDliYmQ2MGI2ODM0YjkwOTVmOTFkNGQyYTI4YmY4Njk4ODZkYjExZDA3M2UzMmRlMjc2ODJiZmVmOGY4OTA3ZWJjZjQ2MzRhMjNlMGU3NzcyZTUwNTQyMmIwMzdmMmNkMDBmOWE0NTk2Mzg4YjFiYTc0MjI5MTZkODJmZTFlNzNhZjM1YTU4YmM0YjNjODA5MjEyM2QzMTM2YWViZjZkY2M1Y2I4Y2Q0YTEzMmE5MThiMDk4Yzg3MGE5YzZhYzEwYTE3MWJhZTkxODcyOGNlNDM5ZDVmMWYyMjQ4MGU1ZGUwMDZhZDE0OWM3YjhhYzFmZmUwOWUwOTI4MjUyYzJhZTc1YTMzMmUzYjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4q28tsvm8M_qZ2A8UY19nmfkBjMN5U8Vjot8ZjwhfmkkRQV2iPFPJR1QftP2u7z8MCxOtNmdhkrdq1OtkRTkag", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220215_102554_23_241b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.308Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFWV21KRWpyOGw5eW9KalRqYzIrbzhNYzRLYUlpTVJLK2dLQkduc2JHc3J5QTIxQmRzQzhBdG9vamg1OU40RzFVUktyR2lzbHFVS1NHdEdlYW1BVm13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxNV8xMDI1NTRfMjNfMjQxYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmVhNDEyZjNmNzc1ODEzYzlkMDBjZjUzMmQzMjFmNmY0YjM1ODYzZDIxZjZjYjRhMGVjMDVkY2QzZjNkZjE3ZmU1YTc5ODBlZmYxZGM1NDE0YTJjODA2YWRiZDA2NjY1Mjc3M2Y3NzhjZmZmMTk2MTQxMmUxZTliM2M1NThkYzBhNzVhNDIxYmMxOTVkMjg4ZTM4NjQyNTU4MmRiM2FlODRhMjk4ZGNlZjYxMmI3OWIyOWI0MWM5YzQzMDQ1YWVmYjllZDAwMWY5ZGE5MWM3NTE2ZTRlYzczYzllYWExZTg3NjMxMzFjNjRjOGFjZGQ2NDA4ODgxZjYzY2NiNjllY2ZiZjlkOTRhOTMwYzQ2OGJlYTczODBkN2I1ZDVjNjJkN2Q4MGQ4MzIzNmM5ZGE0MzJhNTdmNGYxMDAxZTk2MTViZmQzMGJmN2UwZmRiMGRlMTQ3NDdiMGQzODI0YjJlYmZiMjBkZDQ4MjY2MmViNzU1ZTdiYzdjMTM5MzUwOTY3MWU0ZGZjZTdhYzM3NGQ4YTc1NGZiNDIzZjMyODVjNDRmODU1ZGU4ZDA5MDkyMmFiOTU0YTdjNjUxZjY2MjEzYjQxMDk4ZWFhMzNlYzU2Yzg4NzBlOTk3Njg1OGRlMTRmNTk2MTQ3YmU0NGZkZGZmOWE4ODYyZDQ3OTA1M2FlMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XgBO5LE_loFnBIzHYa-f9H6ldB5oQHMquQiJ6lNiW1mROWV-7AtHMKUusOYkZhOAR_lp8feKqcVnalccYXfBtg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220215_102554_23_241b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.311Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IldwRklpa2JDSDZ0bGtiL0trRUxtMk44NmFLckhqL2tocVFTc21uUXZTWmFscS9OWENydVNJOVN6ZlVRV09oU2hlUFYrbnBRZkJBZnZsdXp2V2xFTVdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxNV8xMDI1NTRfMjNfMjQxYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTJjNTkzYmY0NjJkMDAwZTcxZmFlZDBjMTUxZTIwZWQ0MTNmYTNhOWYyMmRhODBiOTc3Y2NjOTRjMTI4OWQ5NzdlY2E2Yzc5MjNlM2I0MjJkZjVjNTU4NDhjZTdjN2NiMWMxMjM1MjEzZWI1ZTc3NzVjZGJmNDc0ODRmZTllMWIxZmNlYzM5ZGFjOWMzYjBhZWE4MjFhZDVlNzQ5NDBhNjg2YmJjOTZkMWNkMTQ4Zjg4MTBhYzY2MDExYmRlZDhmZTc5NzMwMjc5MDA1MjI3NDU2NTEwYTY5MTZlYWM5YjEwMjU0ZTAyMDEyNTU4ZDI4MWE1MGQ2OTkwZmY3YmI3ZWI2MWNhNDQ5ZWI4YTU1ZjY5YTBlZGQ5NjRjYjcyMTkyYzA2NDJhMjU5OGIwNTU5NTUzMDBkMDQxOTQ2NTM1NGU2NDdmM2M3OGJjNDJmMzc3ZjNlMjRiYmRhNDNlMGM0MmM4MjBhMzUxYmZlODExNDk5M2RmMTQ1NDA3OTAxMjBmOGU4ZmE1NjZlYjgwNmJjNGFjY2IyODc1ZDFmYmFjMjc2ODExNGRhMjdlMGU0ZWU3M2E5YTIzZGU5ZjUzZDkwMGU2ZWJkYWU5NWY3NGU1ODBlNzVjZTc1M2IzN2JlMWQyNGJiZjc5NzQ2MTE0YzQwMGRlOTg2MTQ4MGE0ZGQ1OGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.asOgTOfW8HXB5cn8-CBrcAy0nbZ9pFUH0Y5qV8cepv_fJluG1tJe8uNRxatN1NoY4uUrsas1mof4Gh9QVra67A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220215_102554_23_241b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.314Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjdGTVZ5cHZ6bDlzVUtta2luL1dKSUc3Nm1WcEtCQXRablNBQlRzckNESUNxamxBbVdQSWJUclI0Nk1ReTdJb3FMSzFSTllGckZGQzhMTzhybVgwM0lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxNV8xMDI1NTRfMjNfMjQxYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGY1MmRmOTFkM2ZjZTYwNTQ5ZGE5N2YzOGY4MGNlYzg3MWNmYTZiNmZmOGY5NGI3MjFhN2E4NjZmMjAwYmY2ODQ4MTc0Njg1NTI4ZjZmY2E1YzZhY2Q3YzRjY2IzOWY1ZjYyNTg0Y2Q5MjI5ZWZmZWQ3Njg4YjFkMjNkMTljYzk3ODQxOTg5YWZlZmJmZjY0YzYwNDJhMDBmZjIyNWQxMDllYWY4MjQ2M2ViOGRiYzkyZmQzYTEyY2VmMjI1NDU1YTg3ZjM3MzVkZDgzNmU2OTAxODdlMzhiMmJlMjAxOWQ3OTU4ZGRjNWE0YmE4NTEzNmY5ZjMxNmM4NzgwZjI2OTUwZjk1YTNiMWQ3NGRlYmQ0ZmMzMzM4M2U0NDNiNGMxNzQ0OWI5ZDE2MDdjYjgyZWU1Y2QzZjE0ODM4MzU0MGQzY2Y4OTIyNDI0NWM5MjEwMTIyN2EyZDZlMjZiMDQ2NDI4MTIyODUxZjNhMjQwMjVjYTJiNzI0MzgzZTA4MDY2ZTM0YWIwNWY1NmQ3ZjdmYWYyM2ZiYTgwYTZjOWEyMzkwZDU5ODFlNzZmZDJhM2Y1ODQxMDYyZTNjYTA2M2FmZmMzOTQ0MmU0MzQzZTc5YjIwNjYwMjk3ODNlMzEwMDZmZTIyYzgzNjhmYzZkZGM4NTc2ZTc1ZWEzZGUyZTlkYzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NO6diK6H9f_l7a8-mFvTfoC0_71wa1mvu-UN1YFb7ed2Izh6KgeuKNvfP0ank8jujxUe24MCgHoxFqut2-InQQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220215_102554_23_241b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.317Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJMVmFiOGJNZnIzeXJ4clE3Y2MyWTRPRm8yTGdNeDRSWTdYK09TZ202SzZBQ0pQQnBGdGlBWmV3dStGZDMxaVV0VFdzRkRIU3dUYmJCMUR3MmIxajJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMDIzMjhfMzNfMjQ1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGE0YThhNzgyYzdjMzA0NmUyNDAzYTIzYzc1MmFlMzJjZDA5ZjdkN2QwMDU4MWEyZjkwN2M2ODk5MjljN2VmMWIyNDA4ZWIzMzg2M2RkOTU3MDM5YTg0YWNhNTVhMjljNmRmYjc1YmMzOTAwNTljNDE2M2UzNmZhODlhOWY4Y2FkODhiYTI0YTE5NjllYjU5MzA1YThjY2IzY2I3NmQ3ZjZkYmU5OWZhYmFmYzdhNDJhOTQ2MTJhNWJiNzdjNTU3ODhiNDYxN2JiN2I5NWRlYzQ1ODM5MGMzN2I0YjQ0MzE2MzIyZjE2YWQ2MzA4MTBkYTg5NjAzOWQwYmI5YjAyYjA5MTMxMjdiYWM0MjY2ZTBhYWU1NjUzMTUzMGI4MTI1MTg4N2RlZjNkN2Y4NjVhYWE3ZDFiMjg4ZDE2ZTA2ODc4YzdiNDFjZGY4NzJlNTI4OGEwYTE4YjU3NTdkODI0YzZmZTJhZjg1MDEwMGYxNjc3NmQ0ZDUwOWFlN2M4MzczMGY2Y2Y3MTdlNzdmZGVhMGY3YzEzNzhhMjFkYjc0N2UzYTA5YjFkNWQ2MjQ3MjNmNjUzYzZkMzA2NTAyNjE5MGUzMjc2NGMyMGVkNGE1NzAyMjZjNzdjODgyMDMwMTkyMzM4ZjQyMjI2ZDNmYmQyNzliZGE2NDUwZDAyMTFjZmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HhNmngCvWkJ3T8cAZIZUYgG7knpKNJa3SwGlkpr4g8dYjOTJ6EaJvRU4U64ZLIQzcrYRZs5-P4u7ecqjVOlNoA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_102328_33_2451_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.320Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlMwaUlLS3B2WUo4WWNWSkl0QXozT2JlZWQ5Z2hOaDV1ZWNBSEgwM3R4SEZlOUwwNnZRZnpMNHB1L0xXTTYrR05iajFDSGxVdllvMVhmbXI2dXlJRnlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMDIzMjhfMzNfMjQ1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YThhMzZlMGFjNzNiMmJiZTM2NzNjZTY4ZTE2ZjE1YTVmYmRlOTUzZjI3MDhiMWQ0MGE4ODM3Yzg3NjlhNjc5ODllN2VkNWQ3YTAyMjMwOTk2NmJhMGYxZjJhZjgyZWVlZjBhZDUxNmUzODdmNWE4OWM5YjQyYTIzNGY1OGY0NjFiYjRlM2MxNGVkN2IxZjgwYmIzYjNjZjBhYzNlZGI3MGM4YjlmMzgwYzM4NDExZTE0Mjk1NDkxNzMwNjQ0MjIzNjBlYjU3YTFlOGUxNDkzZDQyOGU1NDc5MTEyMmQ5NjcwZmJiMWU5OWFjOGYyY2U5MWJmYmMzOGY1ODhjY2FhNTdmNTY1NjdhYmZlYzAzODIwZTE5ZDIwMzE3Nzc0ZGEyNDRlN2YwOWFmZDgxOTJiNTA5YTk2YmQ4MDc4ZjNmNmNhN2Y0NDFjYmY3ZGE1YjQ0ZTU4NDIxYWY3MTQyY2FkNzIxYjI2MGQzNTE0NjE3ODM5Yzg4ZDY0NGEzYjI5YzdmMWY0YTM1MjRmYThkZWViNmNkZWE4NjgwMWM0OTUxOTgzZTk3YmE5OTQyZjYyNmM3ZjBjY2M1MzI0N2IzOTY3YmZjMjQwZGU0MjU2MTMzM2RjNGNlNzIzOTEwN2RhM2NjOGZlYjJmYmMxMzA0YmEwN2M1YTljMTcxZDMyMjVjOGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2Ph-W4RZvKvwfT1cWSN7JqtTsU7GMvCC7whuuNbRTUWb6gA-k0Ab9zvKX39j_JJeD-v5x1FWAD0UQSxSEHRqGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_102328_33_2451_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.323Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im1BRXQ5NDBIVThock1TRjdGWGVleXc5ZHc3U2RpcmtUSFR6TWIyOEJhZE9iblF3NTVEMzNVeHhYak54NjRxWWFaWTRvUmZCUmJod1duZVRZYTQ3L2xBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMDIzMjhfMzNfMjQ1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzZjZTJmZTY5NTM4MGU5YmY5YzkwMmFhMmM5MjYyNGUwYzY2MDMwZDVhNDU5MzQ4YTVmM2ZiNTcwYzI2MDA1YWI5ODQ0N2JiMWY4M2QzMDAzMmZiMmQ1OWEwMTFhZGJiNTk2MDMzMTgxYjBmY2EzZjg2YmViMTIxOTAzNDJmNzg5ODQ2ZTVkZDllZTc0ZTRlZDNlMmYyYWVlZjkxYWY3YjllMDI3ODM2ODY0YTkwOGUyM2JmMjM2ZTM0MmUyZmIyNDg4ZGU3OGVkYjU2MmRmOGE3OGMwNTc0Zjg5MzJkN2Q4Njg0MGM2ZDkzNGUxODZiZTJmZWQ4MjIyOGYyYjlmZjU0NTViN2RmOWY3NWJiYzRhZmQ0NjZkMTgzNDYwZmRjNDNhYzRiZjA5M2NmMTNlY2Y1YTNlMjY5ZmU1MmZiNjhjNDNkYTllMTVmYmY1MDNlYjNhODk4ZGM3YzJiMzM3NTUyYWFlY2MzZGY0N2YxNWFkYWNmZDMzNjY0YjM2NzU1Mjk0MWI3MDdkZmZlMDRlYmI0MGE5ZWMyMjU2ZDhmZmI0ZmMzNGQwY2Q5YzY5NGM5N2IwMTQwNWU4OWFmZjY0OWMxNWNiMWNlZjc1ZGI3Y2QxMzk0ZjMwMjAwOWY1ZGFlNGYwZWY1NTUxZTUxNGRhMmU1OGU5MGQwN2E5NTE1MzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oO09nivfIKZ9QihuwnsB8HWcoa6q-BI5W7mH-zumVKl4KsKn7bS3gsOTfutzdFDXMUsrBo4G2fH2VAJZIJdLcQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_102328_33_2451_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.326Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNUQzMyeUlmckl1QlZkRk12cTFWczBuNWFSYnpTUnF0TkoreDgzeDJZaWRPbVdUSzliNUszcUhpVVVtU1lUNmVBQUJ1dGFFcDAvZlhkR0tDZmJVbFV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMDIzMjhfMzNfMjQ1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmNmODQzZTUyY2FkNmQyMzAyMjRkOWE2OWVjMmFkM2QyYjdjMGU0Y2IwMzJmNDcwYTRkZjI0NmRiNWM4NGMyYzY2ODI5YTIyYjdjZTE5Njc5NTEwYTAyMzFlOWNjYTY1NTA0MmQ3ZDA1NmUzNTZiYTIyZTExZGJjZjg1ZGVlZDcxOTcxM2JiNjAyMmZiMTRjMzBiOTViOTk2MDFmYjU1MWMwMjkyMjE0ZmIxOWZiN2I4YWJiZDQ1ZWI2ZTJmY2U2MWIxOTM5ZGVkNDc3MGNkN2NlZDExZThkNGE1NDMzODNiYmQ3M2NjNDc2Y2I2MGZmMzQwMmE0MDI1MDAzOGY1YzU3NzY3NGUyOTk4YjA3MjY0MDZlY2E3ZTZkN2UwZTQ1Yzk3MDgxOTgwNTlhOGIwZWE0NDQ2YjAxMWU0MWI0NDcxNDY3NWY1YTE4MjgyMTE2ZTY1NTNjNjkyZDRmMzY1OGM5ZGQ0YTU4MGM0MThiYWRjY2FlZTNjYjQzYTkzZGE2NTg0NmE4ZjA5NmM1MzIwMjVjYThhZGIyODMxYTAxNjVkMjg1YWY2ZTczOWVmOTRkYjI0NzUxMzk1ZDVlMzE4ODI4NmNhYzU3NTNiNmRiOWNhNTdmYjIzNDgwNDA2Y2VkZjA3NWRhNjliNDJkZjZiZDUwMzUyZDc5OGM4YTAyYzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GWXGVt2-LV2Io1zqfCBdVI-J5Uq_9exwPp7du5v-f7ezBDCSePqKlXp_o2oNAYTKeHzi9Lwysaw3ZLlSoCbtsg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_102328_33_2451_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.329Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImVPUTE4ckdsRnFkRFo4dTdDOHdQSkxoV0t1K0FRMHZMVGVJVnBTd1BBQ3I0bmRRNGgrNGIwMXNYbDhNdnZSZm9RaTFvUllKaDlVSjRmUW12djZLUWRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjNfODdfMjQ2NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjkyMDkxODU1NmJhZmZkNDljOGQ5MzJlYjlkNTY1YTIzN2Q4ZWNiN2RkZmNhMDQ1ZWExYmEyZTgzOWE4ZjM5ZmY2ZmY2OTRjZGZmMGE4ODhhZmUyNGRmZDBhMGQyNzZhZDRmNTQ3MjUzODExZDgyZTI3M2M4YjMzYzcyNzhiODY4NTkyNDhjNDM1NDQ4NjE3Yjc1MDI5ZWZjZTlkZGRlMWUyYmU1NzliODRmOGY0ZjQ1MjBlYWJjZWFlNjk5MGJkNTMyNTFiMmVmMjQxYjJkOTk2N2Y1MzM3YmNiMjVjMTVhODZjN2IwNmY5ZDU2MWFmNjRkNDU0ZjVkOGY3N2Y1NzUzMjAwNjA5OGQ4ZWI4ZDJhZmNhMjFkNGU2ZWEyMmZkNTM2YzA1ZGUyNjQ1MjNhNzA5N2YyMmRmMmYyODlkYWI4ZTJmMmNiMzgzNTU1MWNmYWJlZjc2MGM3ZmQ3NTRkOTU5YzA0MTI4ZDE2NzgxNjVkNGI2OWU1YmQ4YzNhMGE3MzgzNzUxNjgyY2MxYzgwNjVjOGU3YmY5NWZiNjQ5NTU3ZThiZTg4MjVjZTIwMmU3YmM0ZWNmNWRkNWRjNmIxOWMwMjMxOGViNzA1MDdhYjk0MmMwMTM1N2RkZjMzZWFiYzRjNzNmOWM1NTdiYTZlOGQ1MWI3MDJkYTYwNzdiMDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.VtGTBz1Zk8Zbx7jwLqvMDGZug4hV-JRykJiFzIhR_w1XuA9b7bZSpKI84DIG87y_f2IW9T6KChs4gSz9ArLb1g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104623_87_2465_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.332Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNNalc5Y2xEdzlNMjJJSzFMWHlMV0VML2hpM2k2cUlXLzVhWHJSeHN1cHpVenVia0tocGdIUGVhZjNKQ3hIUWpoWmV2eHZFTGdLUEIwdm5FM0k2TEFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjNfODdfMjQ2NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGY3YmFiZjhjNTEyZTk0MGU4MTExMTEyZmM4ZmY5MmQ5Y2I4YTJkOTU2OWY5MDU4NDk4YTUzMmFhYzc1Zjk0NDU1YTE4YTEwMGQ4MGExM2IwODU4ZjNlYThjNGNkZTU3YzhlODBlOWE4NGExZWNkNGQzOTNmZmUzY2U2YTE2YzFiNmVlM2IzYWEwMThmZjYyMGNjNTdhOGM1MmU4MDk5YjY1OWQ5ODI0YWEzMmI5YmI0YTAwYjJkMjM2NTY0YTA2MzE5MWFkNzU5ZjUyMTI5MmEzYWU2ZDU3MmEzYjc1NzNkNmE0MWNiMDZlYjkyMDVkNmE4YzhlNDQzZThhOGI1YjFmNzNmODJmMjcyMGYwZGRlMjU1YzcyMjMzNDU3Y2IxMmNjMWQ4NjMxOTE5ZTA2NjMwNTVhOTRhNTE4YjIxYzY1YThlMmQyNDgwYjA5ZTc3ODkwNzdkNzY0YTA5ODVkZDMwMzJkMjlkZjA3MjBlYTYxNWJlNWNjMjQ1MGUyNzljMTc4NWE3NzI1MjkyNGY2M2MxYzYyNDFhNTI1NzUyNGU2Y2FlMjk2MjYwMzAwNTkxMjkyODgxZTY1YjdiZjIwYmJlMzNlOTNhOGI0NTJjZDU0NTJkNDA2NjBkOGVkZTE4NDZkMjEzNDJjZDhlZmZiZWU1ZmNmNTFiNzkyMmU2YzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QSVp72rqjUjWX4aMOaxm7kA6kF1rd5n91dvbZ4HCEXXWqrTMAMQaLEftKVWYNXhBLjT7-6x3JrvtXoFw2DtYkg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104623_87_2465_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.335Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik1tMTlENGxtbldBeXRDcHJKRS9Qc09NS0hxVytlSzZxQWl6bzNjRExEY1p5MS9BejZPSUlKRmxCS2lDNExFeTkzamJ1QVpDQ0p6cTRGV2hiWVVkS2dRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjNfODdfMjQ2NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTBhODBmNGM3YzM0YzFhOTY5MGM1NTE3NTAyYzZkNDIzZjA5MTIyOWFlMzdhODJhMGI1OWI0NGVkMjA4YWE0OTczY2NiZjJlN2U0MjY0ZWFlMTNhMzE3NDNkYTMwMjEzYjY1ZmZjMzQzYzU1NTYwY2ZlNGMxMzcwNWI1MjhhOWIwNDE0NDdhZTMwMDY1MGEzNmJiZjVhYWE5MzBiMmQ3NmQ2Yzg1ZDA3N2Q0ZjMzZTEyMTgyZmNjMmY3ZDEyZGM4NWRhOTUyNzY2NmVkMjM3MDQ2OTE3YWMyMTYzMWRmZjQ4ZTE0Y2FjMTgxN2QwYzQyNDc2ODk0NDg3NDk0ZWZlOWQ4MjRiNTZjYjFkMzBiY2Y2ZDdjMmVkN2ZmNGVkODRiM2U5ZWQwNDAyNjYzODliZDdmNzdlZGJlMTM1MDQzODQyM2RjMGI2OGQwOTIwNmEwNDAzZDU4MzQwNDdmNTEyMmJhNjA1MTAzNWIwOWNiOTZjZTg1MzUxZjVjYzQzYTY3ZTg3MjY4YzY3ZDY1NjVkMTliYzdmZGM4YzVmMTBjNzlmYzViYzQ4NDQ1YjEzZDE5OGEyNzFjOWRmM2Q2NzkzYzZhODVhMGY4ZWYwODExYzVjMzJlNTk2OTE1YzhhYzY3NzU3YmIyNDM0M2Q5Y2I2NjU1MWY0ZWFkZGNmZmFkMWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._yrzt-p53k96zwNappauo8j2rvQ1rEdEHibhJHlfaVnLBPrZ_blW9Y9AHnWvuJ3RupBVqnOxMFzO_GbeFYzuDQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104623_87_2465_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.339Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkdRT1N1V0xpb0VhajBGd0NoS2VwOUI4RHg2SEVhS2xtcG1GWWJBbk1TWk1KeWlZZ0lUNVZSU3oyVjJvL1ZHTjhnd1dBVFo4M29DdFpLZ0pucjZjOGdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjNfODdfMjQ2NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzkyYTk4M2FkOWM5ODE3MGI5YTczOWYxMTM2MjE4YTliODYyOTAxMWZlMDY2MWYyNGRhMTAyOTEyNmVhNGU3OGU5YTIyYTI0MWM3OTIxYmVhZGQ1YWEzMDU2MDE4YmNiMjAwMjE2NTRkOGRhZTdiYTc5MGI0YmJkOTU2OTUyYjAxYTY2ZTIyMGIwMTc0MDQxMjFhMTI2OTM5YTgwMzU3MjZiZjc5NDAyODAzNzVkZWFiMDI4NWJmZmY3M2MzNjE5NjRiNzRmNGI3NWVkMzVjNTY4NzY2ZWRmYTFhNzQ1NDIyMTI4ZjQ3YzBhOGEzZWIxYWUzNTJiZTFiZDYzZmU2YzQ2Njk4YmVkZGRjNTczMzRmYzg5ZjY3YjU0MDQxNmQyZjA4MDkxOTI2OWQ0OTVmMDVjYjVlOTZjMmMzOGI2MGJkMzE4YWE1YTA4YjhlNjc0YTBjNWFmNDlmMWMxOTMxZGQ2NzI1ZDljZTg4ZmI3MzllMTBhNmUxNTFjZWQ5ZGY2ZGQ2NzkzYWFkYjc0OTk0MmQwNDQxZjBkMDViZTNlZmZjYThlMjY2MWEzY2RlYTM2MDhhMjNmM2FkZDBiNDQ0ZWYxYjcxMDhkODQxNzk1MzcxM2Y1N2M1ZGEwMmUwNThmOTU4ZGEzN2JiM2RkMTU3ZDMyZGUxNGI5MWE5OWYwMTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lSionscxGIQLfUHmUkZ2dn_itqAaHaEH_ele2FPOPLltxKQ3h9f_Y3ukVZsFlwITrN-0X_nsp-0GAiZ0hn38OQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104623_87_2465_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.342Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpYS0EyNFJpRDg4Z0NlTTJKblJsRU53VzRHQzJmTDZHTnE4VGdKREg3bDdCRGVSU05YNHBiQm1YbzBXc2pMeXM3MnJjUHZuMkFxdHhldTNKaWpEYXRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMTA0NDdfNDFfMjQ4Nl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjQ2YzU0ZjhhZDY1YTdhNGMxMTg5YTJjMTFhZjlhNmQ3MDcyNGYxNWJmNGFlYTZhY2JlNzg2YzViOTMwYTRkMWU1NzJjNDQ2ZWEwOWI2NGM0MmU4YmNiMmI2Yjg4ZDBkYzk5Yjc1YWYyYTI0ZGQ5ZjBlODM0MmUyMmM3NjIzYjZiMjVmNjViNzI3MTY4ODQ1MDM2M2RlODUyOGM2N2MyZDI5YThhMjVmNjE4YzgyNjk4MzI1MTM4ZTA2MjJhZGU1ZGIzMTBiODllNDA0ZGI2ZWYxYjlmNDIyMThiZDY3ZmM5YTE0ZGRjNzRiZTlkNWI1NDllNDI5OWQ1YzE3Yzk5MjQ4YWNhMDcwNjg4YzVjMWIzMzY2NmVmMTIxMjgzMDNjMjBlODRkYzI1ZDYzZTIyZjdlNTU2ODM0OTVkNjc0YjAzMjQyNzdjYjk2NDEwYjNmMzc3ZTE4YzgzOGI2MjVmZWFiMDQyZTllMzZmMjJlNzQ2YmI0MjkwMDU4MmU3YTA3ZGE2MDFhODIxMjA1NTUyODdiNWZiOTY0YTQ5Y2U2MjY0OGQ4MjhlYWIzYWQzZGEyM2YyOTBjMmNlNDM3YzdkZGIyODNhZTEyZjFmZjUxMWRiYzFkOTNjMmRiNDAwMDUyODExZmNmNmEyMTBkNTYzMjc2YmQ3MWY3YzgyMDFlYjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JhYmFqjQokxLCY0P_AOYR2qVl0Y1veOWA0Wjade2jpj8OTjvvBV9tfo22awo6qhNM2UWDP-7vvUEZ0wqu-jK0Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_110447_41_2486_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.345Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImVkb1dHOXVmS1RlTzlCWHk4bGtEK05LQ3hkQloyTk42SWtST3pFb3lXUzUyeXErNEJ2d3pGaDRINEVrQzVwcFRGYjRMSEl0MG1LTm92TWsxUXcwK3F3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMTA0NDdfNDFfMjQ4Nl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmViZGRkMTllZjA1ODQyODE2ODAxMGM3ZDc1MTdiNjUwMWEyOTdhOGM1NDAwNzIzOThiZTBhOTk5Y2Q0OThmNzBiNzc5NjUwMzI4N2ZjYTVhNGNlYzU2Mjc3MjQzZjdjOTJhZjVmZTVkMTI3YTM3NjkxOWI0YjI0NjNlNWI1OGViODRmZmRmOWVkOWUwOTljMWZiYjAyYTAwMjkyZjhlMDhkN2M3MGRiNmQ2YjdjOWVhYjNjM2I4NjU1MTliNDZiOTczOGY1ZTg4NzZjYmU0M2FhNDY0ZTIxZTBjM2M0YTZlOWUzNzdkZjBhNjZmZDY2YjhlZDA1N2Q0Y2E5YTg4MDU5OWRkMjBiNjM5OGUyN2UyYWVmYWYxZGM5YmM2Mzg5NDE2MjdkOGU1ZGU2MTcwODcyYjRiMzE5NTA5YzcxOTA1ZjJhNzhkMmI4NzgxZGI0NjU1ZWI0YjcyMjIwZjE2YWZhNGU1YWFiMTI2OGVlOGI0ZDNjOGY5MjIzZTcxMjVmMWRlZDRhMWM5YTYxZDBjZmNhM2Y2MDFiMjIyYjZiYmJkZDE2NjRmYjYxMGZmZjhmNWYyZjFhMGZlZWYxMzg2OTg0MjZiYjA2ZmQ0YjZiYWRlYmUwNWNjZGExNDc2YzhiNzJmMTc5ZWY1ODkyMzNiMTliZWRjY2QxMzEyNTM1ZDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1l9lGP-Bglovkxb765DXAQtsi166LWO8B1O469O1ftxl_IpjBRGCjgh8UArhri48vHfL8Ra2q6k9gkXYJHU9Jg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_110447_41_2486_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.351Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InI4MGFycEUwVU9GM081RU8xakQ4Z1pLSG41dVVvak1vMy80MEhCMHRuSnZPem5NZHBVRjVNZi9PeUVBMGNkMWVrQUozdVlEcGhkSmhRd244akkxbWxnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMTA0NDdfNDFfMjQ4Nl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDM5M2QzYmEzMTczZmE1MmE2YTI5NTVkMTg5YWViODQ3MGI4NWEyYjI3OTM0M2IxMjUyYTkwYjUwZmNlNmZmNzZjZjQ1MWI1ZmQ3ZTEyYTcwODQ3YTU3MmUwZGYxYjkyZDNmYWUxY2ExMGUzYTkxOTkyMDQ3ODliMjUyMThmZDg0MDk0MzVhY2FlMzM5NDQ1NzZmMDY0MjhmZDYwN2E4ZTg1YjI4N2JjZTRhYzczYjk1N2VjZjI3Yzk0YWZjMzdkMjNmZjNkOTFhNDA3MmE5YzA2MjBkZmI3ODhjNGY3ZTk0M2IzNWYzYmM5NGMxYzQ3NmJhZWMyNjIyNzZkYmFjM2ViOTI4MzY4NmUyODZjMTYyM2ZhNjM0NzcwZWNkODc0Y2M0MGMyNjVmN2EwNmU0YjgxMGViNjRjNmNjMDU0ZTgzM2RhZDY4Y2FmZjE3YThjMTcwNzcxNGI1YjAwMDU5MTI1ODZjNzc1NjMxYzI3YzViMmJjYjQ5OWYzOTJjNDU4N2YxNjViMmUzMGQwN2QzNDA1OGQ1NmRmNTMyMWY1MmQ3N2IyMTc3MzJiOTFhZTliNzc4NDVlODAzYTFlNThkYTYyZjBkZDA4MDE1MDAwM2EwNzA5NjhlMjE3OWJlZTk2ZjY2ODA4NmZlZDczNDM5NWYyM2IwNjFhN2E2NjNlZTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._iNP_m_Iv0RjQTcrfwA58-QkOUBWd6Lm8_x68hInoh7Wie1ubbCHwkurc-bIovPd2wxEvi5iegc9Tx0sS0dhaA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_110447_41_2486_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.355Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Incxc29xaFd2QUVwdGZ1U081K1N6WHpTcStLeG1YTVVpa0t2d3IvazVGUW1xYXYveWcza3JLeWozWVhJQmdVTkdmVjNOSWd2RzgrMThTaDZTRU4wdUNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMTA0NDdfNDFfMjQ4Nl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWUwMjE2Nzk5YTcyZjlhYTA0ZGIwNDdiYTg3Y2ViZDBiODVhZDEwMGM3ODcyYTY4NzFkZTg0NGUxNDI4NjBiZGJlZWQ3MDI0ODhiYjdkZGM4OWNlMzc1OTBkNmFhZWRhOWIyNWE4YTU5NzA4Nzg4ZTM2YmM5ZTNjNTIwZjVkNGYwZjMzY2E4MjEwNjdhZTUxZjIwMDM4OTIxNThmNzg2NWVmZjMwNThmMzlkN2YyYTcwOTNiNjU1ZjVmZjIxOTFkN2JkMzgzYjNhYmUyZDdlZWE3ZTg3YjZmYTlhM2M1Nzk5ZTI0MjQyYmIzN2Y2MGQ4OTIxOGE2NDQwNmJmYzU1YmQzOTFiNGMyMmY1ZGRiNjJhNzYzNzEyNGI2Y2FkMGZjODBjMWFhYWVjMGQxZWY4YmQxZGYwYTUxNTY3MmM0MDY5MmUzZWNkMTIxZWNkMjc4ODI0YjllZDNiODkxNjk3MTBmODAwZWEyZWNlMmJkM2JkYzAzYjQyZDlhYTBmMjI4N2U3ODdjM2RmMWJhZWFkZTU4YjUyZDcyNTAyZDZjZDM2ZWZhYzJiYWFiMTI4MmM1MDU3ZGJmZDkwZmYxMDBmMjM3YmFiMjNhMzdhOTg2NTZjZThmZWNiN2JkNjI1NDMzZDUxZmFlMjliOGU5OGQxNTliY2Y3MDc0ZjNlYjRhZjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LPQHVoluJrhCrLF8Hmj-UZSIwD2xm5g_OESQmJO36eeFiEQMbmxpA5JdLaDtjBKgK9GAFR8dbHChGvO4mycmrA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_110447_41_2486_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.359Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNHWlhJQWRFd1pKTmk1TW9LOWI2RFNQOWJTVkpJSm9VNEZwditBdHJTZm93QVVoa3dLZ2hPeVJETEVWK1dUVTZGS2loVmdOZ0VnYVh6UzVHNkltV3NnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxOF8xMDI1MTVfMjNfMjQ1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODFiYjhhMGNhOWE1NjUyZmRmNzAyMWU2ZGUxYTZmZjA0NzhiNWUzNTJmZjFiYzlhMWY2NjRmZmRhZTA4YmVkMGQ1NGJjYjgzMjRiZjNjNDMyYWRkZGZjODVlOWRhZDE0OTliYTU2ODBjMGUwMzE1OWY5OWQyZDgxYmNkNGZmZjM0ODczZGU1NDRiNzMxYzY4ZjUyMGE2NGI2MDY4NjllNDMwYjg4NDJmYzczMGRmZTc1ZWRiYzI5ZTAyZDRhN2I4NjJlM2U1OWYyMGU2Y2FjZmRmZDQxNGU5YjRkYmM5MzYwMjI1YjM0M2I3ODE5YzYxMGY3MDM4MTMxMDAwMGIyMjMwMjJhMzU0NGJjYjA2MjlkYjQ4MDIwZWMyMjMwNjRmMDE5MWEwMTE4MGQyMDk2YzIzNGRjNTBhNDVjYWYwNDA3NjBkMzBlYWVhMTk2NmJiMmVhYzZkMTNjODgzZGFiMDVmYTg5YmI4YzMzZGMwOWJkMjMxMjhlNzFmYjEwM2QxY2I2MzEyNGVhMDBiNzUwMTc1NDlmNDRjZWM5ZjA2YmI0OGVkM2UzYTQ2Mzk3MDU2Y2E5MWVkOTA2YTM4NGI0YjMxOWZkYWVjZTJiNGFjZjM1MjhhYjk3NmYzMTA1M2M0ZmE4NDVjNGE4MWYzMWU4ZDE3NWY0Njk1ZDEyNjE1MmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZU-bMygWl5HzFEQnW3eJ4V30fwe31nAbxiDELogpdBZBuDLAIT27VuWWmRqdW5G1PNR4q4mQOETSBcEQd0UyWw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221018_102515_23_2451_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.365Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImdmNTNIbU9BdkxZd2Vad3kxVEQxZFlaUkdjaUtWaVNoRjVGNzZHcmh4ZDY1SC9sc0FXbk5kVlRrMDVkSmFaRmhZSGM2RkNIUXpCRG9MVDhMY28rcWxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxOF8xMDI1MTVfMjNfMjQ1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzlkMzhlMzExMmRkZTgyMmE1NTM3YjYwNmIzZjU2ZWVhYzdhNmNkNWQ0MzVhZTU4OTQxMzllOTE1YzI5Yjk4MzU4MDg4MDk0ZWJmY2RkOTBkZDg4YjI1NjM3NmI2ZDc0MzI4NDU0ZTdiMDY4MGQzMTk5ZGMwYzU5MzcxYjc0OTFhZDJlYzMxMGI1OTJiNGJkOWJkOGQxN2Q4NjQ3YzM2MjA4NWMzNTcwZTllZTVhYjBjYWRhMmI3Y2YxZDlhMWE4MTIxY2NkMTNkNTNkN2ZhYWQwZWFlYWIxYzljMDBmMzBmZTc2MDM2NDBkZGVkMjhjNTNhM2JmOWQ2YjkxOTIzNDJiMGYxZWI4OTk4YjZkZTAyNTVjYjMyYTE1NGVlNTZhOGNkOTljMzg1NDBhZmNkYjA2ZDdkY2MzYTlkODdiMDZlNTVjMzI3OTE2ZTdjYzFiMmM5OTU4MWMxYTE2MzgwMGYxZTA4OGRhMTYwOTAwNmQ3NmIzYmMzODhjZWY1Y2FlNWY4ZmI5MGQ1OWI2ZGIwYjhmZjUxYzU2OGZjNDQ2MGNhODQ5ZmQzNGFkNDBmZGUxZjYyNmRiOGY4OGYxYWY1Y2ViNjkzM2MzOWY1YmI2NTM4NGM0YjY2MjZiODAxMDg5MTczMTQxZGU2M2IxZDYwMzZlNzk2MTE5ODA1OTkxZjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.slrGyd4XKudg587FxIHzXSu7akxBCI14Jlz4NXWndCIgZd1CxeWl0ptib1VStGAZQk0SbfY5reAaq6h--pZjSw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221018_102515_23_2451_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.369Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imsyc1UxL3FNdlR3TjRrT3plczVndzgzR2hodzRwVlNIdlNjaU1MdjBCUW9vM3UrUnNOT3k3ajdmdHJUWDFpYVN5U0NBOExjc3BFQ2wzZHBVT2NoK2h3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxOF8xMDI1MTVfMjNfMjQ1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmZjYTViYjZkZTA2YWIwYWI0ZTgwNTM1ZTkxMWRjMzEyMTk1YWQ0ZGExZDQzNjg5ODUzOTE4ZTJhMjRiMDc1ZmEwMzU3Njk4MzM2ZWY3ZTM3MDA0Y2M1YTRkZjE3MWI5MTM1MDIxN2ExMTQzZjllYWRhNDM1YjA5NjM0YjBkMmIyMGIwNzgxYzJjN2RkMTU4ZTYxOGViZDQ5YWVlY2M5ODk1NTU0N2UwMDU1YTZjYTE3YzljNTU0OGMwZWZkZWRjMDBjYzM0ZmM5ZTM0OWRkZTM4ZWY2NGFiNWZkODJiZDM1ZDU3OTFjMjExZWZjNmVlNmY2ZWRmOTU4YmI2MDI1ZjJjMzYxYmY3NmMxNzI1OTMyMDIyMWQwNzk5M2NhY2RmNGEwZWUxZWI4NWI0N2JiZDA3N2NhMmYxODUxZTEzZWI3OWVhN2Y1NDZmOTMwMzQzMWQyNzcxOTg0YThlYmViM2IxNmJjZjRiMTIzNjBhYTVkMDljZTRiNWJjN2VmMGFiOTgwZDQzODFiZDBiYTY0YWMzMDA3MDc1YTc2OGU3NDJjOGRkNzBjMmE0ODBjMGJhODNiYjRkOTVhMzEyN2ZlYjI5NDc2MzM1NDY5MWI5NDBkNjdiZjc2MWVmMmZkZGRiYzUxYzkwZDBjN2NhZWM2MmMwMmI2ODU4NzQ0MTVlYzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8lwfdkXRX321pocyqSwGX6SpsVUKHg4XK_JD3fx_ra9ffoWRiJfVHygiSz2fqEtgcZdadl6Xdc8tSjcjiB0bGw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221018_102515_23_2451_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.372Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlqcG56bDkxM2wzMHdxOUFiQzBiVzZLVVN6U280cS9WamY2RHRCbnFvb3VQR2F4Z2cvb29TQ1M3OWxJYWZVSTdWZ25sNUZpL1dLUlphdXRtMElLMDBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxOF8xMDI1MTVfMjNfMjQ1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzYxNjU3OTAyODg1MDNmYTI4NDZlYzIxZmIzYTNmNjlhMjIyMGMwMDgzMzkzYTY1YTY0M2I1ZmEzMmJiZjI2NjliYTI1N2NkN2YyZWU0ZmY2ODBhMzEyNDJjMzA2YjY2MGNhYzcxOTMwNDhjZjFjODUzNTczNWRmM2ZjZjZiOGEyMjJjYWRkZDYxNzNmNDNjYzU0NTRkNDExNWZlNjFjMTgxNjEyYzEwYTY1ZDJlNzAxMzBjNWQ4NDliODY4Yzk2MjRjYjQ4YTcxMzVlNWE1Mjk4MmQ4MjMzZGI3ZDVkMDk2NzBlMjA3ODQ0MGNkMDc4YTZiZGVlYjhkODc3MThmOGEyYzA0ODMwOWRhZjBlZWM5NWFhNWZiMzc0OGI3OTEzOTcyN2Y2ZWE3NTA2NzZlMjg1YWIyZjRkYmVlMzgwZjVjZThkOTBjMDFlYmU0OThlNGVmOGNhNmRiYTM5ZDdjZTgzODcyZDRjZWEyMTVhNjA5NTE5N2E5YmUzYTI0MjhmMTk1MDk0ZjQxNWM4OGIzY2FlNTE5NWM5NTFhMWQwYmNiMDNlZjU1OTBhMjY3NWY0NGNhMzVlNDE5ODViM2ZmZGVlZTVkNzUwOTY5MWY3NThhNWZmOGYyOWU3MzBkYjcwMTA5NzgxNTM0MTEyOTU0NGFiY2E5YjFlZjJkZWYxYmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CHfVsuiid2g-tR9GO7thVzM7I2Kkliz5NUwf-p64RTaruzv_yMVYyzwf9w4bO15PDRcGqcBEWp7zd5R8RvZFyA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221018_102515_23_2451_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.378Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJWKzZDNWU1MG4xVk8wYmFjeVRNckhZblI0RzJQbFFDcnMwSThZN09XNC9FWEtkRUQ1dXo3cmVPWU5OWWxpOUliZ2tSTVpkb3ZoWktvQ1lrUE9yQ1FRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIwOF8xMTE4MDBfNDZfMjI1NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWU1NDNlZGE3OGZmNDJhNGE3YjYxMjY3ODExNzI0ZGQ4YjVkZDIxZjg1Y2ViMzAwMmE4MDlmZGIwNjVjY2U5ZDZjNzQ2MjQyNmQyNzg1NjBhYTNjZTMzNTc5ZGY3ZWI3Y2MxYWRhZDAzZmFjMjZkY2NkN2ZhMDhkMjhjNDM2MzhhYmY4NDcwOWVkNmUyOTA5YTczYmY3OThiMzg4NzcxMzkxZTEzMDllZDI5YzgxMjI2ZDY3ZDk3M2Q5Y2M3ZmVhNDdkZDkyMzcxZWE4ZGZmNzFlMTA0ZDRhODYxYmZiMjU1MzJlYjQ4MmRiNGViM2NlMmY2ZWJiNDc4NDRlY2YzNWExN2RjOTEyODZmNmJkZWZiNWE1NWE4ZWQxZDYyYjkzOWQ4NTRkZDIwOTMzNjI3NjhkMzhiNjBlNDcwMmE4ZjgxYTE3MDczY2JlYTJiOWU2ZjVlNmU5NDUyMTQ2YzQwNDJmZWYzNzU5ODgzNzQyZjY4MDI1OTQyNjc4NDhlYWIzOTlhZjA0MDA5YjQzMjg0Yjc3NDNiNzI3Y2FkMzM5NmJhZWMxOTE4YTc5M2MxYjU5OWQ4Mjg0NjIxMjkwODZjNDdkMmVmYzZlMTg1ZWE3OTdkZDdhZTY2OTlkN2JlYWM3ZDY3NmZiZTRlZTAxYzg4YTQzZmZmYzRlZGQwZmE2M2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WWa8Vx-3vY6ebXxOXhlzZX2dEmrYX6CGFqbnQjOljPymRiDxp5vZiH-XDjTwJ7pv8DB5kAi1XPqPXrFGUVgSLA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.382Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhLUi8vZWFoblhwY1luS0hKY0JBdkdRSG5MdFRKekQ1SzBDYnVhS3hURzUzL3dhcFltR3AvazEvTVp2WFFudnowY2dWbjVzd0RsaGtzUnQ2QjZhOG13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIwOF8xMTE4MDBfNDZfMjI1NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2YxMmNjM2I4MTU5NDczNTA5OThmZDM1OGFhZWZhOWNkNGQ5MjUzYTE3OWJkMDVmZWI4NGIzZDhmNWUyOGY0YzE4Zjc5Y2E5NmQ1ODM0ZTIyYjQxNzczZGM0OGMyNzEyM2E4Y2Y0YTM1Y2Q0YTM3YmJjZjFmODkxNzRjODQxZmRlMDlmMTVmZGQ2ZmNlNjUzZjE1YzY0MGQzNzYzOTM1MmExYTRhY2EwYjc5YzAyNzQ4YWIwM2JkNWM3NmUzMjRmNjQ2MzBkN2Y5YWViMWE4MWRiNTc1YTI0YzQwMmQwYThkOGYwZDYxMTgzOTQ4OWMwOTA0ZjM3OTFhYzFlMTczYzk0MmYwM2RmYmFjNTNkNjRmODgyOWIyOGM5ZmExOGFiNmNiYzhjZDM5ZDMzY2UxNzg0ZDgzYTFhMjIyYjVmMGM3ODRhYjljYWZmZTZmNjFiMjc4NzBmOTg2MDk1MjI2ODUyMWVlYTJiM2QwZDE5ZjUxMWU4ZmY4YTYzYmE4NTZjZDJiMjYzZTc3ZDZlMTZkMDBjOWMyMTI3YWM2MWNmMGFkMjg4MGE4MzYxNzUyOWUzNjVjNzkxYWZmMGMyZGNmNmM5ZjdmYmY0NmY5YTUxNTZiZWZlNDg3MjVjODA2NDYzNzU4NjhmOTQyMDVkYmY0ZTg0ZDEwOThkNGFjMDgxNTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mqJerBqMhRSAB3PcL9Wp37Vi2R0mTKemyKLLviiYMqPBKXfialf80GPypXfQgX_4y4PrS6e_0pE3-MIKvFMXQg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.385Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVWajNRT00zbXZQZmVlUTBZNjNicjllNmFwOTkwTGJ1eEhzc05uSkNJRDczSUxMMDhmWFdObkx3eVVGZjZhZUtVU2VyVGF1czNDcDI3RmF6T1lyenFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIwOF8xMTE4MDBfNDZfMjI1NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTNhNDgzMTVlZTc3NWYwZmU1ZWQwYTg1ZDdhOWI1ZGQ3MDBmMDdhODRmMDJhMGRkOWRmY2Y1YWE4MzA5NjdkNjkyZGExOGFiYTg3MmE2YmE5Yjg0MGNjYzFiMDM3MzZiODYxNDFmZWJkZjc2OWEzM2JlYzZhNTg5MGYzNDA4MjlmYjhlZjRlZDNlZDU0ZTBmOTliMjk5NDU5OWRhMmIwNTIwMzA1NDYyNjI5NzYxMDZkNTRiNDRhMWI4NzNmYTEyMWM2NjYwOTNlOTM5ZDhkZGMxYzc5ZjRmYmRkODk0MjQwNDY5MjdmNzUwMDgwZmFlMzhlMGUwNDdlMWY3OGJiYTNiMTJjNDg3NWIxMzgyYjIxZDAzODBiYWI1OGRjNGMyZWU5M2EzOGMyZWUwMzJhMjdkNGVlZTRiY2MxNzFkNDFiYzI0ZDM5OTM0NDE2OTEwNzk3ZTU3YzBiN2E2YjMwNjBkZDk0NzQyN2ZkNGRiOWU0NzFmY2UzZTI2OTY3ZTM3NTU5OWYyZTE1YmQxMmU2ZTMwZTZmZDQ3OTQ5YTVlOTU0ZGY5YTQzZTI1MmRkZjIzYzRiNGI3Zjk5NjhjZmJkMTllMjgyY2IwY2RkYzFhZGE0MGVlMjUxNTlkNzcyYzE0MjUxMDIwMDU3ODk4NDJkYzhmYTliY2MzNTE5Mjk1N2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3heoI51spbkC8uAuiLmwNodZpGpbOL6uVf2pSnwsZzdJ-QReaeW25VBEMqaZ76uZXWgMq2HhVKBcSbuLy8UFUw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.388Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjhNWWRCbTg3MnYrUUhnTlFCaVZ3ZmZjazY5eVU1MW1VMHF0RnljeVU4QWNmTUt3MlZLN0V4MEVZVWh0cHJwUm10akxLSXQ3VDdzWWtBVTR3YWhTcGp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIwOF8xMTE4MDBfNDZfMjI1NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2EwNTY1Njc5OGY4NTE5ZjRlODY3MWEyOWM5NDU4OTUyMDVhM2Y5ZGEzMjI1Yjc0MzQ4MmNiNzQ4YmJkNzNhMjE4ZDMzNzRmZTYwZTQ2NjVjOWMzNDRkMWY4NGI5ZTJhMzJmNjE0ZGEwMmU0MzRhYjI1YWRkNjljN2E1NjI3MDRhNmQ4NzQxZWI2ZDk2NTBiZjQ5MDk3M2ZlZmY5YjU4NzY2YmNkZDU3YTAyMzg5M2QyYzE0MGNlOWU0ZGIyNWI4OTc3NTA2ZTJkYjA5Mjk4YzBjMDBiN2RiNWUwOGVhMzMxMWNlNWM2YzYzOTdhZTM4YzZkZDhmZTI4NDQ0Y2I0ZTBkMWVlNGRmMmZmNjFhZmQ3N2QxNjk3ZmQ1ZjliYzlmMTA2NjIyNGNmYmJmMTFkMWZjZWRiYmJjODhlZjY5Y2RmMWY1MTM4Nzg5NmFmYTQwNjg3ZGFjM2FjOTZmZDdjNTU2MmJjOTVjZmVhNjQ4ZDA3Y2UxMjA1NmUzMGMyYTgzODhiN2JjNWU3ZDM0NzI0OTg1ODA4ZGE5NDUyZGFjNTVjMTcwZTAxOTBlMGEyNTBmYzQ3NWNmZDM4MmI0MGU3NjJjNzdmZjA4MjRkMTYzMjBkNjA3OTI1OWQ2MzQyYzc2ZGI2ZmJmOThmMTQ4NjczNThhYmI1NGE5ZDcyYjU4OTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7ksiCZcacOlFyz3sqGcp5J3QCnTh5u6Q0j60orwYTbmnEkqk7TB2H62tuwaX_gH6WI4zv-SCsfmMUF0m6C4TlA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.391Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik9TbzQ5K0dXenVsdlBvbjl6WS9YcDJNZENlUWFYZ0pUVWF6WForTHo1bm5Ma2pJRFFlbzJhcytyQW9zZ0kycERNWmdtSFhVdkE5TDlpbHRWeVBZYnZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTRfNjdfMjQ0OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODNmM2VhNTBjMzk4NWE2NTViYjczYTg1ZDE4NTQzMGU0OTdmMDk5ZDc0NWZkZjA5ZDNkNTA1OTc4ZTlhMWNkYzEwZjljMTNmOGIzN2FhNDA1NzViY2MzNTMzMDFmMzNhYmFmMzAxYmQ0YjY2ZTgwNmE4ZDJiMTQxNTk1YmJjY2I4ZjFiOTY3ZjM5ZGI4ZTA0NjYzMTEyNWNlMGQ5NzBiYTA1MDQ3ZTBhZTJjMzRlMmEwY2YyMWUxOWUzNjM3ZGY5Mzc0NGY2YzJhNzlkOTFiNjIxNGNkMDBkYTE3NzE4MmFkYmRkNmRjZDNiYTA0NzA4YWU4ZWE3OGNjMzRiYjUxM2QwOWNjMzU5ODk5ZWRlYjI3NmEyNGMxOWNlNzg4NmMzYmJiMjNkN2IzMWEwNzJjMzE3YzIzYjhiMzMyNGIyZjUwYzg5NDc5MTI5ZWI2YWZmZjBiYjBhOTgxZGVkYzMyMDM0ODI4YjI2MDczYjI5YjMwMTcwNzAwYmVjMTNjNjg0MjJlZjY4OGQ2NjdhZmUyOWE0YTExOTZhYzA0ODRlMzk1ZTUzMmQzZTdiMTc1MjcyODUzNDUyOWRmODFmNWFjNGZjYmIzNjgzM2Y3ODFmOWE0MWUzNzk5ZWJkMWYwNDZkOGU5NjE5OTQ0OGRmMjc4MzEyNTQyOTRjYzc4YThlOGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AdTifmOj5762dO2JqVZFO5SmrbMw5XvnytaUr7uDxkI7gryv4Vyj3HrOOW2RG5vPLN8W8MX3xxwoFQZUEhU2nA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.394Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJ2L3hENkNmcTM3WlRjbTBZV2ZSa0hrWTBpYlhBTE5HLzNhVEhZNndvOG9KcDVWRlRhNkEyMzVjZzVNR0lPSzhmc1NLOVc5RVliZzZDbS9qVjZpOUpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTRfNjdfMjQ0OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGNlNTkwMGI2NWY3ZTI5NGJjZjgyNmQ5ZDE1YWQ2ZDNlN2YwNjFiMDYwODI3NDgyZDRiNGUxNDA5YTlmZmM3ZjVhZDI5ZjgwZmI1YmFkODVhNWExMjEyYzk1YTJmYmExMTFkZDQ5MTUwZDlkYmZiZmM4ZmY0ZTJkN2Y4NGYyNzlkNjk3NDExNDcxMDBkNWZjNjEyNzgzYTgxMzNiMjg3MjEwMTU1MTAyNmM0ZjEwZDhmZGQ0MmVjNGMxZTJjZTIzMjhjYTVmZmYyNDU0ZWJhZWYwY2Q1MDU1Nzg0NDZjOTBiNTFiZmNlMjQ0NjU1NWRmYjgyZTM5NzkyYjhmYzk1ODUwZjY2Njc4OWM5ZjRjMWVjZGRmZjg1NzlhMDcxZDQ3NmYzYTNhNWJmZjE3NzI5MzQyZDk1NTFkMDk0ZDhhNDQxNzliYTk2MTg1MTUzNjZiZTAxZDI1ZTM1YjQxZTU5MWExZDU1NjIxODhlNzUwYTRjMWRiMjY5YTZkZTZjNmViZTQ3YzA2NTkxZDQ3MDMzZTQ1NDI3NWU4ZDI5ZTA3N2YxZjY2Mzg0YTUzYTAyMGI5MmRjZmVlMjk5NmQ0YzliYTU3YzJkNDg5MjMzNGQ4ZDc0ZjAyZmIzMDE4NmEzN2YxZDM5M2ExNjVlYWU4Y2RlMjQyMzE1NDdkZWZlZWQ4MjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.k4kugqEcaCf-jrcQGVcp8BCKSWv1JkRZ4mWyZ2bC7HoOROB-geOHsG1gqXKvtsWkwJZJrHR1OewmEpu-c8-pAg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.397Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlhNzh3OXBEWjlSMEZrSUZaaTFlelZOaHRYZEU2ckVPR1RjNVY0ekpSbStBcWU2Nk1RUGszY1MveWVIWDFHRFgreTFGVjJmZnlVbFgyNHd5U1dtbUR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTRfNjdfMjQ0OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzMwOGU1M2ViNjY1YTU4MzUwODM2OGVjMGE4YjQ4YTY1YzVhYTQ3YmE3MTAxNWZkOTJiNjUyYTRlYzgwMGZiNDk0ZGMyYzU2YjI0NjNlYjg3ZDIwNDAxNTM2ZTk0MDQwYmJiYTk3YzU1MmNiNmIwOTk3ZmFiZGUzYjM5OWU3Yzc2MTJmZWM4MjE3YTI4Yzg4OTVmY2UwZDUyYTlhYzQ0YzkyZDA3MjliMWRmYzk5ZjM1ODNmNDkwZWEwY2FkMGZmMWQxNGZmYTc1MTMyN2QxNjMwMjg3OTQ0YmFhNDI0YjI1MTU1MGZjZjY4MWQ5ZDUzMmFlNTYxMTY4NDc5MWI3OTdkOGY3MTZkOTk5YWEwYzUyNGUzY2RjMjRjZWVkYjJiNWZjNzRiYjJmMzZlNDFmMjNhNjRiNDkyOTg4NjI5OGU0OTc4ZmUwZjFlYjZiNzRkYzgzN2M4ZmQ4MDVlYzhhYTk0NzQ3NzJjNDhmYWNiZGE0NDUyYmE5MjA1YWY0NzI2MGUxNGJiZDNjMjlhMTAxOWJlMTg3NGFlNjdhMjllZDM4NmZiYmQ0NDQ2MDNkMzdlNTFlODNlMmU5ZjkxYmJiODhmMTc2YzhiMzZiYmRjODJmZGNkZDQ1NzU2NjRhY2NkODFiNGVjZTYzYjkxMjNlMWI5NDg4ZWFkYmZmZGI1MTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._V6iWfr3JjfsWN04ND4H-FAsJJpbnuQwzFeu5RDCyfwbJc0pChLCm3Lf7CVM7qJhB28MqgFd3z2DH0nLvnyXCg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.400Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9GSk5uZjdKb0k1RDI5OWMwQzY2QXcyY08vYnVjcTI2NVlSN0o5Q2YyVUp6dC91WDhRcThyOTRWY05uNEZoVm9RckdDaEgrT2MyN0VlVEZERkFXaHJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTRfNjdfMjQ0OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2VkMjBhNzdjMTJmZjA2MGFlM2JhMTFhNjdhODAyMzkzYTc4MDdlNzNlYzhiMzE3MDMyYzFiNzBlYmNmZTg1Mzc3OGNkNzRlYTljZWMzZGViNDc3YTJhMjBiODk4NzVhYzQ0MDlkZGExN2Y5ZDgzMjBlYzA1ZmNmNTI2MWZjYzBmZmE1N2E1MzRmNWUyZDAwZGI3MmJlZGY2MTM5ZGNkMTI3ZGY4YzY2M2IyZmQ4ZWU2NDQ0Y2Y5MDliOGQzOWE1YTMwZTRiODJlOTM1MzQ2ZmFjNTc5ZmJhZjdlMTk1ODZiOTg5OWQ3MTlhZTFjYmI1ZDc0NDQzYjI5MGE5YmQ1MWEyNmZlY2FlZTBlYmE5NTQzYjIzMGY0MzUwZWY3YzU5NTlhMThjNThhN2Y1OGI5MTdjOGFlNDFmM2EwNGU5YmVkNmJhMzgxYjllMjAxZjdkYmFmMDVmN2Q5NTRjYzNkMDVhZDVlNWY0YjNhMGJhOGE2MGVlMjdhYTdjZDhiN2E3ZGUwNGQ0MjYzZmQ2YzlmNzE5ZTYyNmNhZDRkM2Q4NmEwYTUyODA2NzI0MGMwYjBjNzQxMzA0ZGUyZjdmYjI1MTNhOTRhMDlhNmNlMWFjZDBkOGVkYTlhZDcwYzNiOTg5YmYwNmU5MTI0ZmZiNTYzMTM3ODE4NGIzMjdhMzQwYjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LvX6M47RKWSYVzVWxIRvEtgG6oRPm-ElNWV_yP-aS6X_8DS9GrtHlhiRJGywNoLZKcFLCXDKdneRnNfDOFD3Hg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.403Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlgyM1A5OXBlQ084Rnljcmp0NlNPQ3B2b1RDcUVRVCtTZkdOY25UaFdJS3Q2NUlIWFBFYnZEWVBsd2RJQTY5SjU3STZ3QjBaWDNQR09GVGk3WndWK2VRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMV8xMDI3NDBfMzFfMjQyYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWY5MmE5ZTk0ODk1MGNhMDdlZjI3OTIwZGUyZjFjNTZlYTY1NTZhNDgwMDBlMjhiYjUyOGE0ZDEyMDEyOWY4YjdjOGExMmEyZWIxMTU2MTJiYzY5ZjZmZTFjNTk1OGM1Njc2MWM4MmY0MTVhMjU4ZmUzNDI4MTc4NTU0ZWM0YjIwZmE3MzllNjA3NDRhZmVmNDVhYWExMWE4Mjc4M2NmOGE4ZThlN2QyMzNiY2QyODg2NTAxZmI4NjNhMGEzMTk3YzNkY2RjYWRlZjcxMjUyNWU1Y2NmZWY3ZTg3NDI4OThlYTcxNjE0ZmFjMTM4ZmM4NjJmMTVlM2JiNzQwOGM5ZDI5M2U1MzI2NGM1MDZiYjYwOGVhYWI1MmE5YTQ0NmYzODhhMjhkM2U2MTZlOWU4ZmI4MjQ2ZmRiNDMzODZjNTVmNDNlMjQ1YWYwMmZjZmFjZTU3ODQ0ZGQ0YjUxMjJjMDUyZDM3ZmZlZTFhOTc1MDc5ODA4OTZhNzZiNTM0ZmE5N2ZmOWUyMzg3NDFiN2JmMWFjNzc4MTU4YzdjY2ZhOTkwMDU4ZjQzNWM4ZTZiZDYwN2M1ZTQ1NDU5ZGRmZWE3ZmI0NjI0NmFhM2E0YzAwNzc0ZTY3ZTRmZmUyNTMxMjliYjU2ZGRhNGRiMzRjMWE0YWRhOWQ2NGZmZjY2ZTRjYTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mAG9wxS4vdqpFgok7eq8sPNP1-AOksSV8-kpSnL6MGwdxtwHsCQ_KiE4Eyd3z4T_ThF62P65Tel8kdOUTHeCPQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220321_102740_31_242b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.406Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImhHS0dsMG9wa1Fvc1I2dktOQVZxdWF3R08xbEI3MHJzVzArRVAxemJmK1Q0c0IyQUthQ3RMRHhEWlhkRnBWN2NVem5PRzVZOVFSeGUxVm9QN0RGaEpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMV8xMDI3NDBfMzFfMjQyYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGE4ZjE4YjU5ZjI3ZDZlODhlYWFmNTFjZDYyMTE3NGYxZWI4OWU5NTU1NmI1NDY2Zjk3MjBiZWM0M2I2ZTQzNzExNzIxNzI1ZTA4Nzk5ZGExODE3MDY3Y2M2MzQ3YTcwMWRiZTRkNjU1YTAyZTc0N2RmZDc2Y2RiNjljMmVlMzJjYjkwZTc2MDQ2MTMwM2RhZjc0MmM3OTM5ZDhjYmRjOTU4MTNlYTkzNWQ1MDQyMTVlMTIxMmIyOTBiODU1ODdlYjNmZmI4ZTMwNzZkNmZjZWUyOGU2ZjdhZGExNTNhNDY0ZWU5OThlMDk0YzJlMjRmYjAyY2VmMTNlNDY5Y2MxYmE5ZTU3YjdlMWI2ZmYwNDM3ZDFlODQxNDQ4YjkxYjRjNzNhZDg1NDEyMmQ5NjQzZmM3ZDU3YjE2OTQ3NTlhMjA1Y2UwODE0OWU4NjJlY2Q1Y2Y3ZTY0ODYxNmY4ZDFjNjg4ZmRjZDdjM2RjNzZlODJlMDFjYjdmNTMzMjhhMWFlYmJiMTBhNGFmNWI3MzA2MTNhMjNkYjFiM2Q2ZGI2OWI0YzQ0ZDdlZDUzZmUyOTE1Mzc0NzkzMjM0YjdiY2YwYjg3YTFiZDllNDk0MGEzODdjMjc4NTM1ZWIxODlhZGRlOTA1MzIyMTkzOTllNGQ2ODRjMjJlOWM5MjA5NWQ2YzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3grereJ5gCnh_pRNhbvt6OueJiw0n6NjqAsVpaGg08f3-yqNkvxjj5OvjzinqHDIixDm9VwJhJgD8wdRR1uPsg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220321_102740_31_242b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.409Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImcrUkpVU3d0QWowVjl2WEZuTkVuQnZzTS9aUk1xdU5pNHNTcDVvY3RvUlkwLzZEdGxHTmE0czMrNTgzc3RsSzUyamxVbnFuRjJGM1VxOHhSVTJWRFVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMV8xMDI3NDBfMzFfMjQyYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmZkYjBiNjRiZDhjZjQ5YjA0Y2VmZmZhNjA2ODY5NjhhNWJhNjljNzkzOGMwY2Q2YzhmY2MwZmFlOGZmNTY0NTgzYWVhNGFiMDVkMTQyMzRiMjIwMTJkOWMzOTgwNmIwZjNkYmUwMzIyMDM4ZDQzOGE3NTYzNDg2MGQxZGFjYmViNzJlNzg4MGQ5MDdmNTliNzM0YjJjOTBkNjYyNDg5NTBhN2U0MDE1MTUyODU3NTlhZGM4Njk5ZWI3YmFhYTc1NDZiMDQzMTBkMjIzNzhkNWY3YzQwMDk2MDc1MGRmZWE0MmEyMmYzMjQ5NzI1ZDkwN2I2YjQ0ODY5MmEyMTMzNTU2OWU4M2Q5MGIxNzNlY2M0MWExYTNmYmFhZmU4YmY3OTEzNTk5NjJhM2NlZWEyYzFjOGU5N2ViOTg3YTIyYWJmMzIwOGI5MDk3M2YzYTM5YjRmMTRjNzdkYjE4NjA1MjUyNDA0Mzc1MzI4NTNlNmE4YjE1NzAwZDViZjJkMjBkZTUwOGZmNGJkZGIwOTljMGRlMDg0ZThlYzQ5YmM5ZjI4Zjk5YWNhMTgwNmY4MTY0ZjAxZGM3MzBkOTM0OWUxYjViZGM0YmNkNzFkZmFjNzM1NzkyOWYyNGFlOTUxODMwOGQ1OThjMTc4MjYyMjU4YzkzNDgyN2RiOWVmODY2Y2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WFQYc3l2Y9cvtz80mpnsPMac_kIiWr3O4lfcychR6YQYpXUk5wbbElmaXfjtW1h5QhKl4wcEVny--5lqyJXUeg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220321_102740_31_242b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.413Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtYcSttZk5KbjZtQ2dkZ3dTbmhGQUpsSVZzb0duU2xWaHVrdVh2Ni9aR0Q0SFhtc24vSEdQQzhTZnY2QzEyRHB6eXU2dGFlWllRc1BhbDZ1OEVaSWV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMV8xMDI3NDBfMzFfMjQyYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzg4NTZjMTZmMDIyMDM4NWQyMTgwNGQ5YjQxN2M1M2VhMTQ5YTliYWU1ZjNlZDRiMjYwMmMwMWI4MjFhYmJkZjQ2ZjQyMmE4NWIxYjY1YmU4MDFkMjI3YTY1NzRiNGVjZTQwNmJkZTFiNWQ2OGVjNWRkMGFiMGU2MDNhZTBkMDJmMjM1ZGNlMWY2ZTJlNzcxYzk2NTMxOWQyNTExMGVjM2MwODFhNDMxNjY5MDZmYjdkNGY0ZTBlMGU1OGFhZWRhOWMzYzAxMDQ1YjUxZGQwOWExNTExNDYxYmRlMTgyMjc2NTNiN2MyNDcyNWM5NTcxNDg5MjU4N2NhZjg1NzdlMTZiOTc5MWRiMjg5ZmU0Mzg3NWNiYzExMzQ1OWQyNThkODExYjAzZTg4YTliYzViMDM1ZGQyNGYxY2Y2YzNjOTkzYzU1NTZhOGM3ODQzMjFlNDQ0ZjMzMjdjYzdhMWM1MTA2NjhjYjA4MTRhNjQ5MjE2YTg4YzZkZjY5YTNlOTRjYThlMjJhYTA0ZGM2MDRmYjg0ZDc2MGI0MzQ5NzNlMjQzMmI0MGRlMzdlYjUwOGFjMTlkMjIxMWE2ZDU1ZmYyMDIzMDZhMjI2NGYzM2M5ZTMxN2EyZTkxODMxNmJjZjBhYjhkNWM0YjVhYjUzNzY3YjQ5MzI5MTI2YzNiYTNlMzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XLNUF0Ag7yNIUd-aJYo0gixWwaBWnNXQS9fho_QhK7Im0deHM3ULmr1mWHRzhDiN9EKjbSrZB5i5y5uMGUhc5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220321_102740_31_242b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.415Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZ0UnliYUFib0JWK3NsOWJvSHNQT0o2MVozU3VMeXFoL0tlWGtQUXVGdG1ZaW9xWE4zSldIN2pCSXFQOURYb2ZyTEtDSnhPQzdjOFZML0p0VFZISThnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjZfMDlfMjQ5ZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODIwMWZhNjFiNGVhY2I4OWU3YWQ5MmYxNmZlNGM3ZmU4MDVhN2Q3OTM2YTMxOWE2ZjIxNDE2YjA5YjY0NjQ4M2Y5OTgxMzExMGM0YjRjN2FkODk1YzNjOGM5MjZjMGQxZmJmNTBjYzY5Y2RjOWI3MDhhM2U5YmVkZTlmZWI4ZTdiMzlkNDlhYmE3YzUzNWZjN2YzOWQ4NWNmY2ZkYzk4NTVhOTQwNWEzYTgxMDE1OTdjYmI3MmM4NTlhYzhmYTQyMjY3NTQ5ZmNiMzU4NWM1M2M4NmYzNWFkMGEwMDUzNjI2MDMwMTdmN2U3MTgyMDQ3YjE2OTIxMDIxODEyZDE1NjlhNDEzYjk0ZDRiMDhjNmJlYmQ0ZDYxZWMxNGM2NDc4YTQ5ZDAzNTEyZDA2MmE1M2Y2NjkyMTM2NWFlZjBjZDYwZDJhMGQ2Mjg0YjRlMWJkYjE1NzcyMzljODM3NTdhNzg3MTVmMTM2OGU3NGI3NzAyNDc5MDZmNjQ5MWNjMzhiOWFiODFmZTE1YjYyOTJjOTg5OGU3ZTFmYjAwYmRkNzIwOTRmMmMyMWMxNDE4MzQ0YjU5YzEyYjllZGViZDAzODkzOWJlNzcwOGMzODY2Mjc0OTk1NmQ5MmY4OTlkZjNhMjA0ZDFjNjY4NmEzMDg4MDhmZDEwMzQ5MDVmZWRkYzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9j5O2vEym_0oAJAXULoMxa_bBOMLQtKvShims-nfz79MzjXBr1q7pf1E-C8lR9MV0GQMhC37Mvht5g_h51mAYA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110126_09_249d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.418Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ijk2M0YveXJhcmJEajlQaWVCUHdsRm5WcnFlQ3B0dFZ0UXlnVkNJMHcrTHJzc1ZBbG1BV1d4OWFFb2dhQ2dsc0lncFJ2TE5TRmpsZ2g0MEpsQ0k3L053PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjZfMDlfMjQ5ZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmUxMTZiYmNmMDgzZDU0MDRmZmY0OGZhNDU4MmVmYThiNTQ4MzEzZTYwY2VkNGVkYTAxNmZjYzYzYTEwNGI2NTdiNmQ5NTFlMTYxOWY2YzA4ZTNjMDQxNjAwYjVlY2EyODRiY2Q5M2RlZjNmN2ZmMTc2Njk4OGFkMjExOTU2YTA4YTMyYmRjYzIwNTc4NGM3OGZjYzcxZTY0NWE1MzAzNDFhZjkzY2Y4NGI2ZmVkY2M3YTNlNzdlYTRjYmZiMjI0YWE3ZDY1MjRjNjM2YzRjYmI0MWZmNjVjMTgwZWRmNjQzMjE4YjE0MWY3YWU3Y2U5OTBmMThmZjBjYjY1ZTdmNDBlNGU3MDc1ZDVlNjg4YjEyZDVjNDhkY2U2NjA2ZmZiMDQyMDU2OThlYmQwMTc0ZmUwZDI4NzBkMDI4OTljNDI1YWEzY2ZmMzM3MDJiMDJkMTNjNWE5MmIzODZjNjQxZTZlMjM5NTQ2ZjllNDNiMTgxN2FlMGE4ODM0YjdmYzZkNDVlNDlkYjE1MDNmZTViNThhZjE5MDdiNDQyN2Q2NWQzMzRmMDEzZTUwZGRhOThhNWE5MzRiZmIwYjE4MDBhZmZhZDk2MDI0YzcxMmYzYjJkMmM3MjkyNWZhMzdjMjE3Y2M4MjFlMjJlY2Q4ZGJiMGIyMTc2ZWRjNTc0MWQ5MjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cLy2TcNk4qQVjFD0u0LRiHsLtydvmMvkBE_tV4LSLeWAJeVgNzn2hr4nswwNgJlkW6vQd6wFTdMZl_W6dKqR1w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110126_09_249d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.420Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNKNTNIaWdDQWVXWWN6MGR4QVV6ZW53V25FNjNVaUtMZTB2bGpuaVhQN1JkMC8wQnpZWW43SFhReVBrdHQxVnJVRTA4enlsdXQyU2h1QWR6M3JrTVVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjZfMDlfMjQ5ZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGNmYWQ1NWI0YzMxMWFjMzNiZDM1NmQxNTE2YWQ1M2M1NzcyOWY2NzdiNDY1MjdlZTgwMDVlYTRiZjAyOTEwOTI3YmJmZDYyMDBiZWRlZGM1NzEzNTdkYjZkMGY3ZDQ1MjEwMjM4ODA3NjUyOWM2MDA0N2VkZGJhZDZkMzYxZGNhYTgxNDgxNGY4NTUzMzI2YTUzZGJhZDc2M2Q3MTY1ZDUzYjUyN2EwZmZiNTkzNTQzZWRjMTkwMzdhOWU3YzgxZmU2OTIxZmMxMmRlNzkyOGFmZTU2YThiZGIzZGRmZGU2M2Q2NDdjNGQzZjFjZjEyN2I5OTk1MGZmYzk1ZjQ5OGNiYTY4YzQ0YzBjMTM2MDA5NWU5ZmE5MGFjOWIwYTdjZTEyODA0ODljNmJlNzRkMWFjNjk2NDc1YTljNjkzMWE1OGU3YTYwZWU0ZDg3NTEyMGQ3YjM2ODJhOTg0MzkxYzk1NjRhMGU5NDZjYjI3ZmZkZjVmMDIxMjdlOGMyNzRiMDM3MWFmNDhjMDJjNzllMDlhY2U5OTExODQ3ODBlZjE5MGY3MWFmNmU5ODdmYTQ0NGQ2MDJjNzdkNjVhZjU1Mjk1NGE5NmVjZmI2NzU0YmRmMzFmYzkyN2NkMmU4M2JhYWQwNTFmYTg4YTljMzMzZjM1MWZjYzE5NzJiMjA3YjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wWWFxOXSMZ7M85WUjlY9prJu9ZVfXk8T3-bO6XkD4fr_9FMmgf0FFtYEbaltoDw_qz0c1hq1UUa7hS-lDdXi0w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110126_09_249d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.423Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InExQnh5NU1hSE5qZ2gwY2FlQ05VTDkyYzIzS0RLdFE3U0FPbThvaEFpb1lac29ZQWg1V3Z6a0prZ3ErOWlKSHRrMW9MSXNoRkZHTGR4a3gxT1dna1ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExOV8xMTAxMjZfMDlfMjQ5ZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWJhZjA3NGRiNTFjY2EzOGFlNWQ3MWM5YjkzNTljM2UxM2VjMTNmZjdiODUyYjkzYjRhNGI1NjhhMWYwMWQwNTRkZGRjN2M1NjUzMzZiZTdjMzNlY2NmMDM4Mjg5N2ZlYTMzMDliNjI0ZjM1ZjAxNWIyOTc2NzBiMDM0NWJmOThlNWY0NzI0MjFkMTIwOWM5OThmMjgxMGEwZWRhMzVlZTVmMzU0MTFiNzlkYzNhNjRmN2ZkNmNmNTMzNDkxMTg1NzJlZGFkMDA5MDA5MmM1ZjJlYmE3ZTM5ZjU5OTI5YjJiZjM4MzVjNzhlNjBlNWQ0MTJkNWI5OTIwY2FhYmY5NzdkNDAwODcxZTAxMjQxYWY4OGRiNDc1MTE5MWEyNDkxNGJlZmNhZWY2MjJhOTZhMWI2Mzg3Zjg4YTY4MjViYjliYTBkOWVmYjNlNmIyOGM5YjhkMTgzZDQ5MGZiMzA5MWMzMGU3ODA5ZTBhNWM0NWRjZTg0YzMzZmJlNTE0ZDI0ODdlN2U5ZDY1NjA4MDA5Zjc0NDRjN2M1YzhhYjY2MjNhZmY0NGJlMTMwMzY1NmY4M2U3NzBmYWNkZjM4YzZhZWE2ZDc1Mzc1YjQ2MDcxZGVkYzM2NDFiYmRlZmY3OTA4ZTg5NjdkNThlYjczMDkzNjJjZDU3ZGM4MzVhN2U0N2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.K3LXs3SDs-D-umZNwktyZnWYIuwyeiaGsV8FZq3gDuQBm62s1DPUYmjZxKhv_MUZlHlbSBR4PYfk67Gad9El9A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221119_110126_09_249d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.426Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IldadnVwK0EvMHhjN2ZacHdoVUVUKys0dTgzcWpOT3FYL0h5UWxtRk9WRFdoZ2lXZi9qOW1mZWs2L1FxcEpNK3BCQ21zNlA2RFFRckpTN003SlBCMDB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExN18xMDU3MDJfNDRfMjQ3NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2VlMTdkNDQ1ZmY5NzgxMzViN2I1N2I5YzI5N2E3OWMxZTMyNjc3ZWU2YzI2ZGY4ZTY0MmY0ZmQ4YzFiNDdlZGJiOTg4YjhlN2E3NWE1ZmI3YzQ2YjUzNzlmYjkzZjJmNjkxZTY5NDdlMTI1NWIxNDc5ZTYwMjRiYTg4NjU1ZmU2YzRkY2Y3Nzc2Y2RkMjNhZjAwZDkzZDcxZWFiNDRjYzVhODQzZWRhMTVhZmRhZGE4MGMxMGYwZTg4ZDZhN2EzZDBjYjhiMWU3YjBiMjcxYzFmZTFlNjBhNzBmMjc3MWQ2MWM1YTQ3MDRiNmIyN2QzNGJjOTBkZDAwMjM0OWM5M2UzMGJlOWNmN2M5OGI1OGFmMzZjMTcyM2M3NmVmZDJlMWJkODRlMGYxZDhjODllNWJkMDhhZjEyYTIyYTc1MjkwYjg0Nzg4NTc5MGJiNGZkMWMwZGRjZGU3NjIxYTc4MzYwNDE2ZTM2YThmNDg3MGUwYzBhZWRhNDI1ZDRiZmRjYTViYmNlMzAxZGU5YzUxMjA1ZjMxOWUzZTRkZTQ3ODdmMWU0MjkwYzJmMDRhMTI4ODVmOWJkNThkODRkZGE2OTM1NDE1N2QzNTNhN2NjNzcwYzFiMDdhOTA4NjE3MDk4NWFlYzMxMjFjZmM5ZDBmNzdmMzJhOTY5MDI1MDg4OWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6YMPJSY7N7AK1ZBnMzOGGPVs8HUeAbEVxZWreyWDUX0aiK--3jW5bloZePZbjR88UK-MpcpdDw1UgSny37xRSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230117_105702_44_2475_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.429Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im84ZDdFUHZtNFRtdGhYKzJRZmJqQkNhRHp2TE9leWRmWHBGVEJIZGFRSWRteXVsZHJKTFNRZzc0cDg2c0RJUnhMRVh2R3pWY1BxSmxFUVNaRzZtcUxRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExN18xMDU3MDJfNDRfMjQ3NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWMzYWYyOWMyODM4ZjE2MmVlZjZkNDVkOWJjYzZkYWJlY2Y4NjIyYWFhZWQ3NTQyZWUzMmQ2YmM1OTc0ZmQzNWE3ZGExNWU3MTY4ZGEzNjM3ZGEyNDQyMDVlZjI1ZDQ5MmFjZGZhNDllOWMxZjBmZTk5ZDllMTc2Mzk3MjM1OGY0OTRlODRkMTM1NzUzY2IxYjQ3ZTYwZjljMDcxNDgzZjllYmJmYTc0Nzc5YWI2ZDhkZDU4MTE4ZWY3YWEzZDNiM2RlMjdmZDE5YWYxMDRkYTY2MzFlNTk3YzE4YzQ4OTY5YmY3MTk2OWY0ZTg2MjFiNDUwZjIyYTFkNTAzMTk3Mzk4Zjg5Nzg5MjE2YmZkNzZmMTYwNGU2Njk0ZDU2OWZlNGNmMWQ5N2RhZGNlYmYzMjdhNzlmOWNlN2JjMzY0YWJmMjZhYmRiNjM5ZWI1NTNhM2EwNmI5YjExNjZhMjIyNmMwMjZiNTJkZDEzYmNiYTUxNTNkYTk0MWZhODliNzczNzJjZjc5YThlOGQ1Yzc0OTEzZWRlOTRjODY4MmU2NTg4M2E4NGM1MDEyMDhlNzg2NmVjZjM5OTQ0MWU0NTkzMzVmMWM0ZjYwN2MwZTI3MzY3ZTc2MDNkZWZhNjQ5NzBmYWM0ODgxMDBmODkzZjE3YTgzYTZjM2E5NzhhMzllZGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nVxK1gEpt_QMa_hv8zuZ2O3PaDnep21umLsSKAxEgq_mJHWLVqRXTaK-Oj_kFMqkfUlEZTyuw5xaceK2ZCP98Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230117_105702_44_2475_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.431Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkZIU0hMbGhmL1BNdjhrcVB5QmlOUGpSRFFuaWtjMHEvdXJRazkvTUl5RDZWaVpFWXlRL3ZMN0laYzRaanFNb1djYWgyOXJ0TE1FdTA4YXQ5WjlxYld3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExN18xMDU3MDJfNDRfMjQ3NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzJmNDlhODE2MTQxYjdhMDViODZmMmE3MjAzM2VmOTQ2NDkwYjRhOTQyMzMwZmE2M2UzMjUwNjc2NTA4YWI5OWI3ZDQ4YWE2MjQ0MDk2NGM5YjgzZTMwNzA1ZDUyMzgyMTFmNWI5MzAzOTRhODc3MzEwNzJhMmFlYmI4MjRkM2NlYjNmNWZjNjFlNWRiZTZhMWFmYWIxYTcwMzMyZGQ2MzA4MDMxZWJjZTcwMWUyOTM5NWIyZDM0ZDlmYjlhOGI3ZTZiZDY4NDEwYzU1Yzc5MTFlZTY5Njk0YjU1ZTljY2VkOWFmNDY4MjQwNWY0NTJlYTM3OTk0MTlmN2U5YzVjYjA1ZTZiMzlkY2U1ZjYzYTRlMjlhZTlmMjQxYTQwZjRhZWYzMTQ1MmI1Y2QzZDNiNzY4ZjE3OWIxYTBiMzkxMWI0OTFhMTlmMWMxOWY0NWMwOWNjMTgyNTIzZWMyNzVhOTQ3MjQxOTMyNGJmZTAwYTQ3ZjI1OTFkMGE2MThiZDE3YzU5ZWJjYzdiNmY4YTI2MzU0OThmMzRkYjRkZTAwZjZlNGI5ODcxNWMzNDM1YWI2MTAyMjIzNjg3YmI1Yjk2MjhmNjI4Njc3OGUzMWNhYmQ5ZjFmNGJhNTcwZmVlMTc3OGVjNWJiNzcyMThjNWI2NzFiNmE0OGIyOGNlYTIxNjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iZjIDohuEk2dsm1Mc601Xsd5ucCPf5p9vKObOwAFwY129uSZN31cCoebs8OuW3jYomN6ybQe9QNUQ-wO-2pgzQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230117_105702_44_2475_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.434Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImdQUkRwendZMklnQkxsdWxaalJqbkhIMXNHbWtsdElKVHovdzVpY3pKeHlMbTFaR2xGWnBiWDcwZ0REcjM4a01zUmh1QjAwZEtFUUZBbWlOQ0hpa2N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExN18xMDU3MDJfNDRfMjQ3NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTYwMTUzOWNiZDZjZjU5YmViOTBkNGM0ZTY5ODRlNTIxOGFjYWU2MTg1YTAwOTM4ZGY1YjUyNGQ3YWVlOTQzMmQzYWE0NmQ3OGNlODA5MDQ1NTA1NjVkMzVlZmJiNjZiNGFlMzgzZDliMjViYWViMzk1MDdiZDA5OWU2MTQ1YjAzMjk4MjYzN2JiMjkxYTM2ODlkYTNiYjk5ZTdkMDU1NGYzZWM3YWQ3ODdiNGFhN2FjNWE1M2Q5NDlhMDFjMjc2ODZjOWM2ODEwOWM2YTc5MDY0YTY5OTNhMDdkNDllZDNhZmQyMzk2ZmUxNTRjZTI1MWZkZTlhYWE5MTI3ZmVjNWNkNWMyYzRlNzY0OWMwZGYxOTVlY2E0MjZlMzhlYjRmYTc0MGUxMjhkYzk3MDBlMmI5OGNkYjNhNTQxNjQ1MGZkMjNjMGQxNjc4MTI2ODE4ODk2MzkyNTVmMzcxNDUzZDNlNGMwYWNhNTNhNmM5MTU2NGU5NzAxNTRlNGQ4OTY2MGY0ZjY3ZTMxOWJjYmEwMzg1M2I3ZmMwZmM1OTQwMGFiYWMyYmU0ZjYzYTFlOWYyMDQ4ZDY1MjQxYjY5MjNmNmFlM2JmOWY3M2FmZDdiNjkxYzFiZDQzZGU0ZGVmMTA1N2U1OWM4NDcxMWZhN2JiZTEzOTg5NzA2YzM4N2E0ZWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LUHLNJPf9KRXlej4XLF8kR0t9tSTm9XuWdH58G_wD1Kce3jGUkK0Uu-5cVLaGXjwP-UiAn0H3388Df-w4xsphg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230117_105702_44_2475_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.437Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkkxUXNvUFpTQ0ZXOWl2Z3E5WjFKek1oTjlPN3liclVLaWsyQzI5VXIvazNFZnl5Q1JBVHFjTllZWDlZZTkzWEhmVEdUK2ZIUVc5dmdtN3hiUmFsajFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzBfNjRfMjRmNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWVjNTc3M2JiOGY0Y2NhOTg1OTc0ZmZlZjFlOGViNGU1YmM4OWI0MjI0NDg2OTEyZWNmZWVmZGRkNmNmODVmMTI0OGU0MTM4YjBjMmYyZWFmMmIxNmE1ZjliN2ExZjFhNjk4ODc0NDg1ZTVjMzFjYjI2N2FiODczYzBiZmRjNDY2MzBiNGRkZTQ5NmU4NzRiY2EyZDI4ZTUyZTE3OGJiY2I3MmZhYzlmMzQzOTc1MzljYTlmYWY2OGVjZjVlMGUxMWIyNTEyZjRmOTU1NjJhODYzYzA3MTM0MjQ3YzhiNzkwNWVmNjg4ZTU1Y2UzNjVkNTEyYzhhZTljZDhlM2I4ZWVlOTdlZWY4ZjM0ZGZjYWI5NGQyZDQ1MjY1OGJhYWMyYTBlMWExZmE2NjEyNjZmZTk2MmNmMjUzNmFkNzhlZTQ4YWZlZjMzNThhMjQ4Yjk0YWU2ZDcxMmQ5Nzc2ZjBkYTFkOTIyNDkzZDNhMjgyOWE0ZjkxZWU1Nzg3YjlkZTBhMGQyMjA3MDY5ZGNmMTkwZTU1YTNiODU2NWRjNWM3YTYxMmI3MjRkZWU2ZWM0YzM5NDA0MjNmYzhmZmRhMDU5ODRiNDViZGM4N2ZkMGI3MGE0ZTU5MTk0ZTUxZWFiMWMwZDI5ZjUyMDE3Mjk1MThjMGI2NzIzMTkyODA2NGM1YTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KjQxV2aew_zMpf4GH_KzHbg8vQ3Nl_RSf--8YtgHOXrBdtjv_p4d1lbpqOa_J4xAGN4W-2yreCW_Ri-AbyA0sA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113530_64_24f5_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.443Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImczQ1Z5TmY3VzdqNDNya0dDcXNXbVE3ek91M0xwbGZrR2o1RWdhSnh1cURoV241RXhKTlROT3pnNGNiUnhiUzE5NTU4UEN1d1BETkVBWG9zYmQxbGRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzBfNjRfMjRmNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzhiNzQxNjYwNGRjY2VjM2E1ZjhkMmI4ZjA1NWZlZjg4MmMyZGFhMTZiOTk2YWU5NzViOTQ4MGY0N2Y1ZDUyMjJiMWE4NjFjMTEwOTY0ZDNiNWM0YzA5ZDQ2ZmQ4MDI1MWI0ZjVlN2JiNDIwZmQzNmRmMzViNzU0MjMxZGVmYjI2NDA2Njg1NzBlYmNiMzg1MjVhOTIwNDI1MTEyZjM3MDQ2MGZiZmE1MWNiMTk3YmNhODRmMDZjNzZmYmMyYmYyNzhhYzc4NTM0ZTRiNzhhZWUyMzI3NzZlMzI3OWY0MDVjMTM1ZDg4ODE2YmQ2ZjI5ZjE5NzEwZWM0N2M4OWE2NTM0NGY5N2UxZjg0NTQxNTE3MGU3OTQ2ZWU0MjJjNGIyMzc5NDFmMmI3YzFkMzQ4ZGIwZjAzN2RiZjNkZjg1MWVlNmY4MTk4YjVmYjdjNTkyMmU3OTc5YmYyYzczYjY0MTI0YTQ4ZGE2MmRlM2NjOTQyN2VjZjU2NWUyMjgwZDdhZmNkYTc1OGU3YWRlMWVkYTBkNGRkMTQzMGY4YjNiMjU2YTUzYzBjZGYyYmI0YzhjYjJlMjQzMDI0NDczMmI4MjhhMDI0OTY4MWY0YTlhN2Q4YzZmZjM0NDdjNDU0NTczYzExZmM3YjBkMmUyNDg0YTA5MDhjOTQ3NWEyZGY5YzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iQTpmbODKJwyZZkO9Qqu64H77DsC6BAJIeP4sP-dUX1P4Rfk3g6zj88UshdB1l6-2inypZpctV3bodopJ8yVZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113530_64_24f5_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.446Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlFkaXdPbEhhelpDOXNSQ0VUZWNCNC9EMXl5Q0dJbnhBeTlYZjZiVjRwWituaklXS3VmV1NpdzhaUHczUkFaL0Q4ZUJrKzFMMXc3TkdrTXBOeUVsOW9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzBfNjRfMjRmNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDRlODZjYTY2ZjE5N2ZjMjg0YTUzYTY0YjgzYjRjMThlOTNlNGEwNjMxODEwNDUxMWJmY2IyYTdhZjlhN2Q4ZDc4ODE2Zjg1MmJjMWM3MTk4MjFlM2UxZmM3ZGY4MmQ4NGI5NGY1NGE3NmU4NjA0MGYzYzJlYjJkNjI2ODFhOWUyZjYyOGY5NDdjM2I4MzZiODM0ZDBmYWIxMTlmZTdlNWE5ZjE0MDg1ZTA4ZTMwZWExYzhjZWExMzM3OWQzZTMwMjk5OGY0OTRjMjA5OWMzMGI3ZDc1ZDJlODlmYTJhMTQ3NTcyYWQwZWM0YWU4MzUyYmE2NjZiYzkyODc1ZWQ2MmE0OTM3OTMxZmVmYmNkODQzYWViOGNiYjVkYjE1MDg1NDI2Nzk0YjQyN2Q0MGU4NjU1M2QxZDI1NjQ2ODVmYmI1N2E5MDhjYzY0NzYxZmZiZDZkZTVmZThkNDgxZjE3MDgzMjA2M2I5OGM3MTU5MGMyZDg2OTVkNGUwODRhODQ1ZDQyNjY0Y2JlZDAyMDNkNTI4ZmVkMzM4MTY2M2FmNTdmMDgwN2ZhODE3NzM1NDgzOThmOWFhNTJlZDE0MGU3Njc5OWZkZTU5MmI0MTBiMWQ0YzBjMGZlYjhjZWM0YzU3MjJkYmUzNGI5YjRiZTVmZTNiNDZlNmI5ZDU1YzEwZDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.L9xp2UguYGb2ebhLK-yYXAcrZ0RXKodVxdm8ZIGGUWIIGAkFpM5NFlFFtGGrnqcqh8IYeaJp2sAfdAiGehZsmg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113530_64_24f5_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.450Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InBoVm16dUY4Tnhhbis2d0tMdnF0VjV0c2ZEZG1uQ3NjZUNDalBIWGJSdUZrb2E3SkpoblVuYm13cUhDU05zSFd3b3JBMDQxeVlMRnpMYWxDVkJxd3BBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEwNl8xMTM1MzBfNjRfMjRmNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTM0NDhhMWNjODE1NTY5NjJkYmI2ZTAxY2RhMGFlMzRjYzVhZjc2MTQ4NDEzMjIzYmVlZDU2NzU2YTgyN2UxMDMzM2Y0ZTE0MTU5MjdjM2I0YmU0NmYzOWJlMzQwNzA1NzhiOGQ0YzkzMjc1MWE4ZjkwMTQwNGMzMDFjZjNmYzU4YWZhMzJjMzE5NTY4NzBmOGJkYjllYTg1MTI3ZGVkMDc4N2JlMDAxMmEwNGU3YmEyN2U5YzBmYmMxZWQ4NWEwZGYwODUyODg5OGFlZTk0ZGE3MTM2NzdlM2ZhOWJlMjc0YjM2N2NjNGNjYTdhMWJmZjkyZDhlMzljMTQwZmZmOTk3N2Q1Yjc1ZjVlNzNlNzE5OWFiMzJhZjdjMjE2OWJjYzVjNGMxMDRjZmE1ZjYxYTlkOWUxMmViNWE5YTM3NmJiN2YxYWIzNDg2MmI4OTUxYWIyZWIyNWQxNzAzNjZiYWI5MzIxZWIyYWMwMmM3YTRjOTRhM2NmOWI2M2I1MDJkNTk2NWNkOGM0YzRmYmM5ZGRkMmM3OTg2ZmRjMDYzYzZmYWNkMjk1NTA2YjdlZGEyMWI4ZjlmZmFmZWRhMDg0MTFkNGZmOWVmOWUyOTU3MTM3MjVjODAzOWQ4Y2MyMjgwMzNiOTRjYzI1YTdjZmI4YzU1YjE2ODZiYjk1NDUzNWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YTfqy8Zmwvzj3XP-ugCz8ZrVdBS5aalBS_FemA_e-bnbRqdN_93Ys3IGU0k7prEDWg1LLU1GMSjDuZLwN-uGiA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113530_64_24f5_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.454Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZNUFF6Q3g5dGZ5bXpzbXkzbVdVVzVlOVlIZFJNWjJocGxQVEdWS1BROXJOK0d1MjNLa2tVd0FGRUg4bGo3aUc4QTN0ZmJ5YVEya1NhZmMvbDh0VG1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjRfNjVfMjQ4OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDk4MzRjM2RiZDE2M2E4Y2U2MWJmN2M3MTRjODlhOWQ0OGExNjUwZjc1NTUxZmU3YzUyZWVhMjBmZWMwNDdkYTQ5MDA1ODIyNWUzNjhhNWU3OGZjYTg0NTQxOGZhZGY5MzY2NWFkNGU4NjMwNmViNDUzYzNhNDBhNGFkODA3YTQzMDM0NTdjMTIxMmQ4ZTIxNDNkMDdjZmVlOWQyZjM2NDBhZjUyNDAxNzA4ZTA5ZmQ4YTg0NmJiZjM4YjAzN2M1MWExNzljYmJhNzMwNzQyMDUyZWQyNjVmOGZlODI0YjUyZGE5NjUyOWI5NzQ4OWE4ZGY5MmEyMmU1ZDE0NWZmYWJhYjQ1ZWQzYmI4N2Q2ODA1N2JkYmU1NWMwODc1MTc2ZWExOWI2ODY5NTk1YjlhM2E2YWFkNDEyNmRlNWExYjQ0M2Q0OWE2ODgzMjQwNTRjMzVkNWI4MTc3Yjk0ZTMwNjE0NTVjZGVjMjU5YWE3OTMxM2Y4ZjQ0YTRlNTQ0N2MyODkzMWQwODc5MDM2ZGZlYzA3Y2I1ZTk1NTFmMzI3ZmNmZTQyNjliYTBjZjJmOWYxNzRjMTA4NTc5NGIxMzNiOWVkOThjNTI0YTA1NmJiMGM3OThhNGRkNGRmNDE5M2JjNzYxYjk5NDY4NGE5MjViOGRhOTZiMzhjZjUzZTU0NjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uDquLhAooBvhs8YbnIYXRi6LlYCXaGnN-pvPxWT6zQrwodWut3Y4LvWVk1kRobcEJqNMzLOzyEdR-IovydLKxA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105824_65_2489_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.457Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im55WDMvTm5IRWFFTlN2SFhZQWZoY1Y1am9taldZaEdDZ1c1QldkWitxSXZ5TnVNRXJ0dkJTZWhlOXd2bkc3ZmU3aUhkeGFvRmljdk9LTW5hUmpxdkh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjRfNjVfMjQ4OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzQzNzk2NGVmZmIwMzljYWQ5NzNhZTYzZTNmZjE4MGY4NTdiYzFjOTI4MGZlNDcxMWRkZjY4MTQzYTgwYjUwY2Q4MDQyYjM5MWIyMmEwZGM5NDBhOWUxNWY0MTZjZjU5NTFmNmZmZjA4NTk0ZTRlZWE0MzA0MDRiNDYwZTJhMWExZTg0MTY5ZWMwYjQ4MDMzMTllOGFkNWQxMjA5MTc5NzY5ZDlkY2I4NjIwZmU2M2E3ZjdkMjIyMDJjYTZkZmIzMzE4NjNlMGUwZDAxOTRhYmI4NTJiNjg4NGQ5MTQxYjJhMWE5ZDVmZDllOTdlZGVmNWEzZDNkYWI0Yjc2Nzk0MzNkOWRlN2Y5YmMxMjU2Y2Q5M2NiZTY1MmEyNTU3NjY2ZTJlNzdiYWFiN2M0NWVhODdjZTVhNTU1MTNjM2JlZmQ1MmQ1MjdiNWZjMGYzNjFlNTU2YzEzZDdlYTExODQxNTUyODk4M2ZlMTYyODNjZWM4YWU4MjI1NTg4NDEyZDMwZTU2ZGMzN2U0ZTA1MjIyOTE5YTk2MDViYTIxMmU4NmRjYzA5ZDBhMTA5Y2VlNTMyYTNlNDZjODU5N2M2ZDUyZjcyYmM3NTAwZTk3MDQ1ZDc0ZGU0OTNjYWFlZWFjMjllNDExOTU0MmJkMWZiM2QxMDI2OTE3MjNhODJmOTNjYjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mcA4BbMY1RaYMia2JFPGdLsdCzVGDdxugYys-vtd1l5HPTx8x5pylbn7Qq6gTK3WoKW_T1FDvFnVyAqTuAThBw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105824_65_2489_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.459Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZWZWgvTFJvZU5lbUt2UUdDdWtPd0JmbUZZaGM1TWhOd3VvTCtLZDYrai9FVVo0d2ttR0tSWVBRUlVoZXBnalJKSHJncDJyeDlUYzVDMk4xcGN6OVB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjRfNjVfMjQ4OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGRjOGNkZWZjMTllYzE5NWVhNjgxMWQzOTJlOWM2NjA0NjM5MGQ5ZTc4MGZkYTUyZGU0NGMxMDFmYmU4ZTA1MDE3NzhmMThkNDk2ZmVjNzI1MTgyOTE5MjBmYTY4ZGVmNTEwN2UyODBhNTIxY2NhY2UwZDU2ZTFiMGViNTc2YTI3M2M5ZDJlZTFlMDhhY2VmMDVlMmE4MmJlN2U5OTJmYjM1MjFiYjAzZmUxMTNhMmFmMGQzOWFlZGQyOGFkNDcwNGVhZGUzNTc4NzM5N2EzZWEzYTFlNDhjOTY3NDQyMWYwNDM0OGFhMDM4ZWRmMjYwZmQ4NzkyYWU1MWI2NjQ1NWVlYWRhOWU3OTBjZDM1NTBkNDBhZWI5ODA0N2E4NWRmNzYwZjdhN2I0NGUwMGZkZWY3ODE0OGM0YWVmODY2YjYxN2I5YjM4MzI0ODYyZTU0NzZiOTliMTk5ODAyNjZiZjQ1NmVjOGQ5NDM5MzNlOWQyY2I4ODFlOTNkYjk1NjViZGY0ZWQ1MzY4YjQzMjNlMjZiZWQxZDYyM2RmMjI1ZjRhYjRhOGFiNjAxYWE5NmMxYTcwNGVlMTkwNzE5NzMyNTUwNjA0N2MyODBjYmE4YTYwNDQ0MmUwZTg3Y2JiOTMwMzFhYmU5MTIyMzhiZGExMmI2NWUxMjc5YTM2YzhkZmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1-1wOvX7rXY_7JTw-dxxfOYpMt0U5ADZkDvxDPA17eVmug73ONdkqGbaVPVOO4fnhlrNtfVEatlwF-F6YYEtIg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105824_65_2489_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.464Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik52cjhEQ01vaDR3anVZQjdpd3Q5M2ZRL0tTU2Mya0dzTDhUWS9ZTTlQR2pVVHdhMGhmQzhsY1UvTklqOE1mYnp2MHVwWks0QXJBcjVESElxQzR5a1Z3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjRfNjVfMjQ4OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGY3MjAwYjA2OWM3ZGZkMzMxZDljZWI3Yjg4NDkxODI3OGYzN2RjNTUxN2U0OTQ3OGZjNTEwODI3NDc5YzI4YmNhYjk1ODAxMzc2ZThjNTQ0Yzk1Zjc2NTY4N2JhMzNmNTExMWFlMzhiMzU1OThkNDc1YmZiZjIwNjE3ZmUyODU2ZWVlOTIzNGJkNGFmODBkNGMwNmJlMjNkNmUxMTY4ZTY2OTAyZjg3YjM2ZWYyZTlmMmVmMzI3NDc1M2I3ODFhODhkYmZjYzg1NzkzYzU0YjNhZDVmZGExM2Q5NTI1YWVmZWY2NDczZjM5NTA1MWE3MTNlMmM3OTVmYTllMGY1Njc0OTZhOWM3ZmZjNzA4ODA2N2I3Nzg1NGU1M2I2ZGQ3ZWJmNjA4YzM4ZTVkYzZjZWM2NjZiZmI0YmYyMjY5NjI0MGQ3ZjY0MWRiNzE4YzBlZTViODVlNzg5ZTgyMWU4ZjYzM2QxODY3NTFkZjg4MWZmMmY5ODI3NGRhMmU4ZmZlMTI0NTM5YjI0ODQ5MDMzNmRiNGM2MTNmZjNmMTQxZmUwMmU4YjczYTlmMTIyNjBmMzUxNGU4NGNlZTg1NGE1NTdiMzAxM2E5MWY5NTUwMWUxMmZmYTY4NThiZGFjNjAzODY5YWI5Njg5NDZhYWY3NWY3YWE4NjI2ZjA1OTZjYzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PR8zgVhgsB_MauzzQQmVwq3BVIKwTdmht5I7oeeAIFKJnUCrK1v24Tyeb_pWyIYrYuKMfqnRk_u7oHVoe14Tpg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105824_65_2489_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.468Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkRqWnp3cXRCUFZCNk1mL1YzYm9GNFR1YnpKamI4bDJmTTdsTlpCSzhDaTVVd3pxOFIxcEVaekNoMFFxaWlaWnUrOXcydFQxTXZ5SjM2a0M0OHNrU1ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxNl8xMTAzMzhfMjNfMjQ2MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTNkNGNjNWIzMjUzMGZjNDVjNjEyM2NmMTkxNjg5N2JhYjI3ZjNhNDZkOGYzODE5Mzc1MDFlZTA5Y2JkOTU2NjU4YjZhMThjZGU0OTNkYjlkYzAzODMzYmU4NzVkMDRiOGQ1NjU4ZTJlNDMxODc4MzkxOTdkNDdjOTQ1OTFmZDNkOWJhZTZkY2Q0MWIwYTBmOTQ5YWQ3ODk1NTRlOTVmYzU2YzNmYmYzMzVkODk4ZGYxMGEzN2NmNjZjNmQ1ZTE1NzM5NTYxY2U2ZmJlMjdmYjBmMDI0ODFkNjhhY2ViZTAzNDQzMTI2MzkzYjY4NTI2N2I4YzcyNjc1ZTMxZDM4YTQ0Yjk0M2QzNjU4OTQ4YjE2YTkyZDVlYjJjMTY1Y2U0NGNlY2M3Y2M2NzAwZmZlM2QyMzI1OWZlYzAxN2E5MDFmYzdhZjc2MTdiODAxNGRlY2FlNTEzMDA4MWMzZDExYTNkNzEzMGNiM2NlMjNhNjAxNjcwY2FiOTRjYTM4MWQ1YTM3YjNjYjg0ZDAzNzBlMmU0Yjc1NjRlMTc4NmMyNzliNDg4NDNhZThiMDUwZGI4ZWUyOWVmOGY4YjFlMGU4MDkzZDBhNDYwYWM0ZjgyY2MwNmMzZTdmZWUyOTEzZWI3ODM1YjczYzljYTgyMTY1ZWVlNmI4NWE3ZGFhOGRjZmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.P4OPw63IKto8ZGix2Ea698vEXLGOCNTFCL63LuGEIvu0AHfIYdlWfGGyEKftQ4CUNMrPNedJLa0b1HT4n3kKag", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221016_110338_23_2461_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.471Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZPMUlGUjBsV0FsNlViK2JLVXNYUGpyU2tvM0dZWDgxdFpUMGtHZEpaRVlsaTZKbkprUDVNR2VHL2h6TmYxQ1FoNTRxSWJTdVpqa0YrMURjcG0zbC93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxNl8xMTAzMzhfMjNfMjQ2MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODExYTczYTI1NzFkYzgxY2I2YTEzMWIzMTRkMGRmZTU1Nzc1MjI3NjljMzBhYTUyNTlmMDU3ZmY4OTMwMjRkZjRhZDdmYzdlNDE0MjQ0NTM0YmFhMzgwMzY2ZjYwYWU2MzU4N2ExZTU0MzBkODMzZmY2ZDY1ZmQ2YmE5MzViYTA2ZjMwN2QwMjkwYTQ0NTM3YmU4N2JmOGMyOGQ4ZjdiNmZmNDMzN2UyYzFlNjgxMGEzYmQ1NWUwM2NmOTc4NTE3ZTAwY2EzMTU1NzQ4ZDA5Nzg3ZDY3NzVkMzM3NmFjNzJkZmM2MmMzYTUwZDBjNmYyM2Y4ZDNhNGJiZjc4MTA0MjIxNjgwNThjMGE3Yzc0MjhiMmUwNDg4OWY1YTkyM2JhYzA2N2Y3ZmM4ODM3YzY3OWM5ODZkZWZjNGQ1N2NkMTM1MDRmMDBhMTUxZWE4MDc5MTQyOTdiZmZmMDlmNTVkYzkzMWE4YjVlOTUzNTJhNDI2ZGEyODM1MWQ4OTk3Yzk4MzAzZTlhYzJkY2U2MzJmYzlmN2M5ODliOTAwMGE3ODNjODc0YzFhYTg0MTBlYTM4NTk2YWFmMDU3Yzc1MWEyM2I5OWIxNmU4OTZiODQyNjcxMzI3MzBmNjRjZTkyNGExY2I5ZTc0YTY4MWE0MWY3M2RlNDE0ZmVlYjdlMWQzNjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3_XTMAEeQDkP6M6ZRrncvvSFyb1WTnkwjAyUnFssYT7qci23zD8zPrJp-W9jda4BJ4fuacWWISC_K1SsxdU3ZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221016_110338_23_2461_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.474Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im1nUGpVVTk3ZytUeFZwVHF0UmVFT05uUzhvTzVvT0Q4WkI3b1pFUGJqdzZ1bjBTN3AwbTducTlSNTNZVVNSQXh3UmZYNXBpaXVSSldhcWVCVHczUVVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxNl8xMTAzMzhfMjNfMjQ2MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVhZjE2M2E0NDc3YzFmMTFjMDIzODViZTY3ODdlNWRjZTRkMTVjZmY3NGQ2ZDlhZjE0YmQ3NzliMGIwMzg5YjhhMzE0MTk4OTE2OTM1MjYzMWEzMTdkN2U1NDA4Yzc4N2Y3MTdjMGY5ZjQyYWY3ZTNlZTY0Njc4ODM0NWRjZGZhOTc1ZTg1YzZhMDRjOGFiYWVlNmY0Njg2YzgxZTA4ZWI1YTRjOTI5MTkzNjZiMjE2ZDZhODY4NTIzNjU4NDkyNDllZGY1NDc2YmYzMGE4ZmNkNmY1YWRjZGJlOTRmMTdkMjgyZTA0MjIxNDc2NzIzM2VjNmQxYWRhODNjY2I2MzMxMmUyZTdjYmVkNjIxNDU1MWJkNDFjMzRjMmEzOTVjOGFjOTc0MWI4Yzc1NDI3NTMzZTI2MjI4NTY2ODcwMTA5ZWFlZTkzYmRkYTA3NGYyOWJiYjg0ZWNmYmQ5NzBhNjM4ZTZlNjkyM2YwNWU4MmY4NWE5MDkyOGJlNTU5Mzk5Zjc0N2YxZjhmNTkzZDEzYmM4NjMyOTU3YzQ2ZTQxYTIwYWFiMmEwYjIzNGVjYTJlY2ZmYTNmZjU5NTM1MGQwZDhlNGI2MjFjOWU1YTY4MWFkYzVhZWNkNWIxZWZkYzgxNDljNzYyMGFlM2U1MjdhMWZkYTliZDY1MzNkZGZjODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8Mvtfa6goyW5eMpX1QgMJeGbGWYqMX_WG-ejDRQS0nsZhZEES-6fjH8vbSNoF6-ZFcrWYbVR77f_eGTvhCDffw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221016_110338_23_2461_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.477Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkRnMzJqN1ZVV09vZ2JpSkZoQTk4Sk5TUnNHazR0dUdvYzlMSDN4WnRjL1ZvSmw4QjdRSXkvUHpicEY5SWwvQ085WjZxYTZsNGR5T0JTbktVdjhGMmNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAxNl8xMTAzMzhfMjNfMjQ2MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDgxYTU1NGJiMTNiZGM0MjkyNjZiNTFkNmMxODI2ZTcyNDI1NDAyZmIyOTU1OTRhOWQxN2IzYzcxY2Y0NzBmNWNhMGNjNDVhYzYxYTU4ZDQzMzYyOWIyYTY0YzVkZDAxMzM5NDIxZmJiYzk4MGI4NWEyMjFmODUwMGY4NzBmODdiZGI3YzhmMzYyNmU3NzI1YTBjYmQ0MzQ5NTZiNDgwNTk0M2U5Nzk2NTExYTBlMmFlYzk5NmM3ZmFiM2E5MjQ3YzIyOTc0NzQ3NGNkOWFmNzI4YzY5MTU3N2U0NDhjMGI5YzQ3YWI5NDhiOWFhYzI4NmYyMTNhM2ViNmIzMDg1NjM2ZGM0NDAzZWZhNjNhYWUzZTVjMGJhZTM0NGNlMmQ5NzYyMWQyZGMyOWI1MzRjY2FhN2NjZDEzMWVkNjI2ZjA5NWQ3OWZjMTg0MTJjMWQxMjk1NzNmZTQ5Y2E0NDY3MGFiYmFmYmE5NDY0ODZiZDcxYTI1OTI5MWEzNmRiMGZhOGU2ZTVhMTBkNTJkZmQyMjYyOWFlMzRlMTI3YWQ1YzAyMzFjOTNjODExYTVhYjgyZWNiZGIzZGI0YzMzMjY0ODcwMzQ4OTZmMjc0ZGU4OWVmOGVlNWRiZWU2YjkwYmM5ZmU4ZTdmMzBhYzQwYjMzMDdiYzBkOGY4NTI2NjRiNzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UeSl9-Oznw4ZZMaXaqUQhBDy3ZVTivz87HCSAb5eymkJi_5vHGwVO9UZha9s_KCpJk_nzHCVuMfnMoHxjpUeQg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221016_110338_23_2461_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.479Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNLRzMvYmZRdlV5UVJJYjdBaDdkMW9MY3NDWWZWU3d4OTEzY3VtRzM4MTRwaXZlK1Q2THdaU0htYWlMOWMzRTRDZ3orcWJwc2x4VXhBajZFaTdxbUtRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMF8xMTExMDZfNTJfMjQyNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzY2ZThhMTE4MjQ3NDAwNTgzODAxN2FjODUxMDA0MDI5MzEyNzg3YzdmODMzOWNiZWE1MWYwMjI0MzgyZWE1YmY0NDhmOWJlNTUwYTQyMDRlYjIzMmMyNmZiYjhmNTIzZDQyZDAyMzAzNjhlZWQ0OTM3MzliM2ZlOWI2ZDZhOTI2NzE3YWYwMGVlZDFjZTQyZGZlZTJjMTkzY2RiMDc3YWM1N2FiNDdkZGJkYjc2NmU0ODM5YTg5ZjJmYmNhYmY0YzUyMTlkMTJlOGQwNzcwMTRhOTE2NWU4NmFkYTNkNWJhYjEyMzY4ZDM3ZWExZjU0ODM4NTdjOGIzZWIxYzM4MmQyNGZiYTljZjdhYmUzYTMyYzQwOTA2YTY0MGNlZjE2N2U0MTlhYThlMzQwNjViNjZlYjg5ZTg4ZmNiOTdmYjhmMDk3MmUyZmM2ZTEyMWMwYTU5NzVhNjdjYjg0NzJjNTRjNzNmNzIzNDEwNzYzNGE2NTUxMjQ3OGVhYzM4YzI2NjAxMDFkOWFkOGUyMzM1ZTZjMDg2ZmJiNzliODk4MTdkZmZkN2NiYjgwYzQxNDY4ZTA0MTkzNzc3ZDBiYzgxMDI0NjhiMGZiOGY2MzUzNDIxODAzYzQ1ODFjMzQ2MmNiNjJiY2JhN2Q1OTY1MDE2NGVkODFmMjY1NmI3YmYwYzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hN2jlafrWN3O_GwRjfZ95DD2jJewiOri8WuzMFm5JDxURMzFsdKeu716bLuhilebXKpeOC3q2TcAwY4CTjZGAA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220320_111106_52_2424_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.482Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IndPU08rVm1xTVlEdGRxV3FMcWt3R3ZpdTFaSEFXc29HZ0lHd2ZQUGZNTDYyN29nZHZRQnZ1ZzQrWENHbDVxLy9RZkZNYXpQZnhMNnZHcmpLdHJpMkl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMF8xMTExMDZfNTJfMjQyNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjViYzllMjg5Nzg1ODUwYTg1MDAwZTY3OTQ3NTE0ZmJmNTZhNzk0OTQ1NmEyODUyMjZiYThlNjdhNGI3NDliOWZiOGIzOWQ1NGRmZDUxNTg0OTMwOTU4ZGJmM2VlZGEyN2E5NDZlZDAzZmU0NWJkMGYyNGQ1MzNlNDc3YWFmNjI2OGFjMTBhNTg5MDUyYzE3ZmY2YTVhNDRmMTZjZGIwZmEwYmMyNmY5MTUwZWFmMTU0MmQxMWNlMzEwYjdjMGU3NTY4NDBhMmYwZmU5ZTBlNGM4MGViZjkxNmE0YzJjMjI2N2I2MTRiZjk2MTUyYTk0YjVjZDhkMTk1NzliOTI3N2NiMzk4MzNiYTQ1ZWIwZGVjMzRhY2Q4MmU5NjA1YThiYzY0MmE3ODZmNGMzMTEwNDg4N2YxY2RjMGVlNWYwOWE1MzU2Nzc1NjQ2Zjg5ZTFiOTQxMjc5MzRkMDE0Y2I2MmUwY2NlOGZjMDM2YThmNWRiYmRkNTU1MjY3YjE3MDVjMzIxYWYxZDY3NjUzNTQ0ODIxYjAzZjE2NWE5MDRjMWFlZWYzZjVlYmRjYmMzZTgwMTA4NTJmZDkzMzk5NTcwZGEyNTYwZjlhYjIyNTY2N2NlYWM3NWU5OGE5MzNiMjFlMGI3MzAyMTJkNjBmNzkzMzc0Y2M4ZWM2MTZkNGM4YmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Sd9GSwyhGaA3ebzsYtWDmyuT1lEfkvuwsx2MimfD1gLWkren13hHhDNPYrBga1cJLl7rpVuUQJ8f6S9Aj3xyOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220320_111106_52_2424_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.485Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImRSdHUvRjdVTEJJWEluM2lseWFEcEpTeERjV09wVW5zYnBNSGJVaGJxNkhUa3NqeXo0Y1plL2FnZzBiZ0p4bEtYZGFFNTZNc1czUm51eUZzaTcvc0J3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMF8xMTExMDZfNTJfMjQyNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDZlMzQ0NDY4ZmIxMTc1MzllYWMzZDRiMDhmZTQ0NTgxNjE5NGMxZGFkODBmODRmOGNiYmRhYzliYzA4Y2ZkYzVmYjczNmJmYzI5YjQ3YzIxNDgwMjM0MmJjMzM5MWNjYWFmZmQ2ODU3ZTc3ODcyODBmNzIxMDhhZTkxMmIyNGM2NjAwYzYzY2RhZmM5YmQwZDk0MWIxZDY4YWRkZmM0NjQzMjAzMDM3N2Y0M2Y5Njc3ZGQ2YzgwZDJkMTZkMjhjZjdkOTc0ZmRlMTg1Nzk5MWVjODFkODM0OTI5MzAzMWJlMWNkYzdmMTFhMGIyNTFkOTNhYTdlNGY2NzBiZWQxMmQyYmM5NGNjMTZhZTgyNmRjYzMyZWQwMDllMTFkYjI4NGU3NGM3MDNlYzE4YjIwY2I1MWY1YTg0MDQ0MmFiODUzMTRjOTJiNmFiNDkzYzViOWJlNmM0OGQ2ZjY0NTU0ZDkwMzkwNDA0NTk3ODFjNjlhZGJhYzkyZmRmYzQ1YTgwNDJmNzAwNTIzZjYxZWY0MWJiYThkMGJmNTkxZjQ0OGJmZThhOGM2NWExOWZjMzY2MDM2N2IzNTNlMTEwYzdkODAzNzkxZThjMjNjYzVjNGQ3ZTkxNmZkMGZjMzFjMGMyMTZhNzdlMTRmYzdiZTBhNWU5NDkwYzE3M2Q1ZWZiMjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8IXFUsHw7CB1WXwnG9s3ZOyxnkYmDU1_rGi4gSCLvXDtV9D7uUhY7zpeJiC7nNTwDokoE3V04o47yWs7621WZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220320_111106_52_2424_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.488Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IktXQXhLN09pM2RpRUFHT3FtSWZGTHJUQ1hHUE1xU29OTzVTM3lSeC9NSWgyTzFxK1VDZzc0aFlqYTJ0OUtGSVFFaUhUUXFZRGx4bFc2WVcrK2QvRWZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMF8xMTExMDZfNTJfMjQyNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmVmNTM4ZmY1NzQ1NmVmZTkwYmNkODNiNzQwYmYyNWNlYWNmMzllOGJhZTUyODU4Yjk0MGNjZDU2OTAyYTUyODY0NjJlYzA3MGZjNTJiNjU4MTBkOGYzYzU4ZTE4MGI0NDdkYjdiMWJlM2Y3OGQyZjNmMjdlNWFjYWRhMTNmYjA2NDc3MGRjODRhNTZkM2MzMDFkYzRjOTZiNTc2MGRiMzVjODljMDc3MzA4NWY3ODE0MTI4NGU0YWVjOWYwNDJmOTdjMmY1ZTBiNDU1NGNlN2RiZjk0NTdkMTRhMmU1ODNmZTA0ZWE4MDAxMTlkNTc3NDU4YmYwYzNjNzYyMmE1YzA4YzhkZDExNTAwOWIxOWEyMDZjNTI3YTdjMGI4MjdlYjBiY2NjNDY1ZDNmZTVmOTY2YzVlYzVkMzdiZWU3Zjk3OGRiNDVjZmZkNmZjMjAzZDQwZjA2NDZmNmRhNWMwYmZjOGRjNjE2OWFjYjZjYTE3NDFjYzU1YmYyZmJiZmNhZjZlZTFmODdkNmRmODdjMjgyZTU2NWUxMjRjZTcwMjliMTBkZGU1NzBmY2M0ZmM4ZTYzNmU2OGNlZTNkOWQwNTFmZjllNWY1MDA5NGEwYmQyZjFiOWYyYTdmOTc3YzVmNjA5NzNiYzdmZGFmODQ1MGUwZmYxZDZlMzNmMmE3YTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.VJGG1omtFFBwYfHJbcLny1nRT5ySVSLjAxx7qEEnxW64ASOE5unpSXWpXluEqhbWrOTa43ah14KO2NDmfYHQLg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220320_111106_52_2424_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.492Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InN3WkFKa1JBSWM4NDc1N2ZGSWtoNWh4aCtsV2UvZGVwSlZGZ0t1OUw5c0NwOE1IYXE1anBPUWJVaE1JSnNxZFpIZ01VL0JnN1RGcWNKckV2NnlGN1B3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMl8xMDU4MzlfODZfMjQ4Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NThjNzZhZDM5YmQwYzcyMjZiYTA1ZGZkZTNkZTBkNmI3YTI1YmY1NDQyNDNiYTY1YmRiYTVlZTFhN2Q1Yzk1MGUyMmVkYjA2OGExZTYwOTM2YjU0NDVhMzFiNjVkYzA4ZWI1NDc4NGViNWE3MTU0ZjQ1MWZlMDk3ZjMwOWYwZDZkNTI2ZjVlMDA0NTQ0NGRkNjg4ZTRhOWU0MDA4MzdlMzEzZDBkMWMyZDVhYTgzMjhiZDNhMTMxYmE0YmRjYzcwOWE0MzRlYTE4MTRhYWNmMGQ1OTU4ZmU3ZTk1ZjdhYzA4ZmE2ODlhZmQ5NDU5ZTNiYzRmYmI5MTdiZjNmMTliZWZlMWQ0NTkzYjZiMDVhODUyZDM2Y2YyZDc3NDc4OTc0NTQ0OGY1ZWE0NjM4OWY5Y2NlNzUwZDQyNzMxODBlMWM1MWJiNTU5ZGE1ZGJjYzUyM2Q1MDRhZjU2OGY3MjU0ZDZiMjdlNjA3OTNiMDM1MDdlZjg0MGE0MmZjMGY3MmIzYzQwNDIxYzJhOWE1YWIwYjIxMTZlOTY4N2FmNDMxOGQ4M2I5MmQzZjBjZTQ5MmZhNDExMDJmMDFlNTg0MGMzNTlkOThmYjRjODk4ZWQxODViMzVmYTcxYjFkNDJiNjE4ZjM2NWNkNjQyZmVmNDdjYmU1YTQxMWM4ZTBjODc0YmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.V2QoArH4i51d314GhjlZyhSBrno96sZXE2fmuvNXgQczQ2JWPlutj3arOXQZpRldPzIH51JlRB-8OjfZusD-jg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221002_105839_86_2482_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.496Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjNVSnQwR2VXN1hFektBeEprNG1ic1c5WHZSNXNFZFdrMG9RZXozdUVqS2RGUlNCdStzK0VqeUh2akphSFBNODZPV1dzb3V3WU84dTdtUkNQSG0wZy9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMl8xMDU4MzlfODZfMjQ4Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWZlOWRiNmQ4MDZmZTg3ZjMwMTI3NzVlMGQ0MjI5YWFiMWE2MGY0OTlkODViNGE2OWUyZGQ4NTU5MGY2NmRjOTRmMDE2NGVkYzkzZmJkMTk0MmVhZGE3N2Q1MzBiNzkzZmNkNmI1YzZlNTVmNWFlYzk2NjAzZmUyYjA4NDBlZWM0YWYwOGEyMGU3MGZjNThiNGU0Yjg4Yzg1MWIzMjMzMDU4MjFiNmQ5YTZjNDJiZWU5NjUwYzAzYzk3ZmY3OGVlYjkwOGI4NWQ4ZjhjOWI4OWQ2NjcyZTQwZmQzZjQzMWNhNzE0MjkyYjI5MTM3ZjRlNjc2OGU5MjNlM2U2Mjk1MDcwYThkOTg5NTViYWQwOTUzNDUzYTIwMTNlYTAzNTM0Y2FlNzFlOTkyOGE5Y2ZkZGNhYWNlMTVjYzczZTdhMTAxYThkNGE5NzMzZmRjZWU0YzhhMzk1ZDQ0ZjFlMGI0NzM5MWUwOGNkNDE3NjkyYmUwN2E1ODkyYTRiZjkyNTEyM2RlMjcxY2MyMTZkMGYxZTNhNTlhYWFmMTQxNDY4M2NiZjA3ZDM1NDAwMmI3ZWU3NjBiOGJhM2ZjMGY3Yzk1YTNjMjA2N2Y0ZDYzMzIwZWY0YjVmZTE5MTIxOTYwZWNmZTgwMGRkZTIzOWQxZGNkOTdiYzAzYzhjZDEzNTBlMjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CfMGIhTY_kfysGE76QacgrgW3PG_b7svWKhuZr8MQegZv-5umz2iu4uRpuR2s-VgxQ1dqhOVEHi4NdOWFdxsog", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221002_105839_86_2482_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.499Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Iks5ZFdyakRlTm14KzA1QWZ4LzVtc013OEdBK0ErWUc0M0dpeGxiajREM1paeDU3N2hhajZlZTllcm5PK2s1YVpqcVRHNVdEZmhlS0pUTm94RStxUnhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMl8xMDU4MzlfODZfMjQ4Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTNjZWI3ODllZDRjZTYwYWFhMDU5NzUxOGY1Yzk5YTMxZDgzOTdlZGIxNjZlOGIzYmZlYzdjOTY3MzFlMmE4MzVmNzQ0ZDU2ZjlmZTRiODYxNzJiMmViNzJjOGZiNjhlYjNmYWIxNDg5YzJjMzEzMjNmYjI2OGM1ZDk1MTNlNzE0ZTdiODE5MTA2NjYyZDM3OWUxMzhlYTI1NGMzMjEzMjMwYjM5NjgwMGI1YzQ5MjUwMDFlNWQxNTAyMDIxNzFlYjQ1YTQyZTRiN2U5ZDc3Y2RmNjk3MWI4MzgxYTRkZDE4MzE5ZTdkOGQ0ZDQyNzU1MjAyNDhlYjkzNDRmY2VlODA2ZjYzZjQyZDMwOTliNjhlNmIxNTJiMThiMzNmOTdkYWNiOGJhMzRhNTIzMTk5MWRiYjIyNzRmOTFjMGFiMzljOGJmY2Y0OWI4NzU0OWM1N2RhYTEzYmMwNThhNjJjZTA1MTZmNThmYTZkNDRhZTU0ZjdjZjMxYzZlODA0MTg2NGI2NjNiYjA0MzI5MjE5NTMyM2MxNDYyYjVkYmUxOThkNWQ1OTFiYmQ4NmZhMWQzZDBkY2JlZDhkYTYxZDcwZWM2YjYzYmFkYmNlODU1MmUzMGFiN2VkMzRjMzhiYTM2NGYyNGE1NDQ2NTNjNDU1YTdmMzliZjI5N2UwZDYyMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.itXNygWS6jUE6Uj137FJ6F2SksqOohoHl0-xBFVErGeiWkg0xQfhz6lRVAsac1PLK4yalvFinXC4RfO8aB-i5g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221002_105839_86_2482_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.501Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjY1WW9XeldMNlIrWVcwVVRhbFpEeHl1bWYzMWUyMkdZeDB6bjRwSWJwQ1QyY2VlY1dmTWpTYVFkQ0R0aWVBcHF5OHJnQUhZeUJMamhNZEdjemw3V1pnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMl8xMDU4MzlfODZfMjQ4Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODA5NDUxNDkyNjhiYTliMjQ3YzVkYWE5MzA0YzZmNzFlZjE4ZWQzZGNlODgxNDQwNGYwYjQyYTE2MjNiZmJjOThiYjk0ZjIwZTBlYmVkNTk4M2NjMTIwNDFlYjZhNTlhNmQ2M2QyM2ZjZmY2ZjVkYWJiZGY4N2VmODllNzRlYWJjNjk3NTA5OTE0ZGMwNzllYjYzMDQ3OTNlMTg4MzQ2NTdlNzU3YjE2ODJjMjlmNThlNzFlMzYxODI0ODQyZTBhNGE4MjJhNjRlODk4YWNkYjJkM2ZiNDU0NjVmYzhiMGRkNWQ2MGExNGM5ZWIwZjM0YzJiZmY3YzAyZjkzMGYzNGY1OGU4NjY2ZDAzMDcyOTJkNzg0ZTcyMzg3MDkwMTZjMWZhNDIwMDVkZDczNDUzOGIzY2Y5N2M2ZDU0ZDgzZDM0YzQxMDcyODIzMzM5MGRjNWJkMTQ5NDdhN2M2M2FmYjI3ZmNmYmUxZDBlMGM1Zjc0OWFhYTc4YThiNTU2YTMyNmZmYzgxOGRjODI2YmRiZTBhZTEzZGFkYjg0ODJlYzU4YzcyNWU4NWQ3ODc2MGU2NmNkMzgyNjk3OTUzOTQ5NTQ2ZDMwNmQ0NWVkMGE0NTRmMzBhNTljMWFhZDk2NTVhYzhlNDI2ODExNmQ0Mjc2MDE4ZjYyM2ZiMTdmMmUzNDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2XIQ43FJ8qgGTkRUDZq7DJghglk3XUpCfNkD-okaFfv6vbuSyy1yWU21T2oBaJXEy8L2U1KszSXBBhg03oEsbg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221002_105839_86_2482_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.506Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkVmMHAxZm1KSWh5azNPRkk4VlEvY0RMVDFienlLbzNpOXcxcmFIUzJDcmw2dkc5RFBFL1RrK0ZTczBKVEtiTERWRnlJVUh1RC9XN3Q1dnMvNWF6QTFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTRfMjlfMjQ2M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjY2NTQ0ZWE2OWQyZmJhNWYwODljN2YyMjMyNDUwMzYxNzdhMjY3NGE3NzU5ZjQ2YjA2NjYxY2NjMDI4OTZkOGE5ZGUzM2NiYjFjZGZjMTVjMjk1NjEzY2FjZTdlMjAwYThjM2Y4MjFjMWRiZmRlMGY1MGJlYWI0N2M1MGNlMWZhMTUxNmM4ODkzMjg4MDIwZWEzYmJkYmQyN2M2OWQ0ODA1ZjE3ZDllMmUxMDA3YTA0ZThhODFiZDIzYjA1ZDc1YmFkOGJlMTEzMmRiZjZkMzU5YzZiY2NiNjllNTMzZjA1MDQxYzlhZmQzY2IwYjQ3NjMwOGNlNTQzYTY2ZjM4YTZiMGRjMmZiYWU2ZjcwM2U0MjUyMWZiNTU2ODM3ZTk1OTNkNmI0NTIxZGI2MzEwY2RhNDJmNjEzZGFlODY0M2M0ODg0NjUzODZmOWEwN2Y4ZjJiMjY2OGM4ODk5MmI0ZmRkMzNkMTQ3NTE3Yjg0NDQzYzkyOGFlNzZjZWFhMGFiZjQxN2YyZGQwYzRlMGJlMmNjMjAzNDhkY2M4NGMyODBlNDBlODgyYjBjZWRkMmZiMWFlNGUwYTNiN2U1MDk0NWJhYmFlY2Y1MjNlMmVmOGMzOWJlMDUzZmJkZmY1YWExZDA1ZDU1NzIwOTBlODFmNDM0ZmYzY2U0MDE1ZjZmYWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.kF67Ozaojk65oSJbqUqYxLFuZbLSvGMDQqAneGbI6YHuVTY9dWneLs3_E63WxuRKdkBbJvd1ey94W8aATVGFCw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103854_29_2463_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.510Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZ2TVpxZTdyOFJZSldjYmtyWERRQkppaFFpWUt5TVpTaTVVaS9zRWdJcUp0VzQzMDFFT3hpbm5vZmlZRExlbVZ6aTNGbERpVWRLR1VqSFdMQUIwS0lnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTRfMjlfMjQ2M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWE3MWI3NGU1YjljN2VhYjk0MDUyMWMzNDI0ZGNhNjBmYzNhZWVmY2Q5OWQ1ZTM1ZDlhYWYwNTBkMTUzNjk4M2M5MjMwYjU2ZGE0MGE3MDdmNWE2NzJmNGRmMDgwNDhmNzNjOGVhOGE4MThmMDBlNmJjMzZhZjM2MzU3ZWY0NDU4MDA5YzZmYmRjODc3MWY4MjlkYWFkZGE5M2JlNzEzMmY1MjllY2VjNDc5MDE1MmMwYTE2NTA2OTlhNDE5YjBhZTU2MDQ2ZDJlOTI3MGU4MDg0OTM1NWNlZmQ5Yzg1NWEyYzE2Nzc1MmZhNmM5MWFiZjAyZWMzY2QyMjgxNDVhYzNjNjZmZDhmMmQyMjY0NWQ4YTAwYjRjODVkOGI2MWFmNGM0MmM2MDAzMjlkOWUwNDYzMzNlYzU2YzQ2MWEzNDEyN2FhZTFmOWU2ZTljZDE5YjcxYzczMWU2MjIwYWQyY2NmZGM5Y2IzNzJhY2Q4M2Q4NTI3ZGNhOGYyZjJiYzllODc2YzFhMzI0ZmJmYjNmNjQ4MzAyZjI5ZTY5YTFmNmM5MGRlNTc5YmQ4NmVjYzJlY2ExZDdmMzI2YjlmM2Q1OGZlMWIwMDUzMmE1ZGZjMjNkODY3MmIzOTMzOTEzMmE4NWJlYjA0MjUxZTcyOGE5NjlkY2FjNDgyOTVmMTQ2NzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LxZNLl7jG-d_eXNkZlxiqqgf8nQSBFnudPurLrvWdtaR7pcdg_eTVn4r9oSvBYRpcybLXDPcJvkdV91b9QyvFQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103854_29_2463_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.514Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9VRmczZjVnY1hOZFJQSjlzTkhGMEE0S29FLzM0Z1dDcmUxSDUzcmNrZzMyMXZzOGsrcURDd01uMjBBdURWejBGcERDSDM0dEtBQTA2Ny92NWJxbDBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTRfMjlfMjQ2M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjU2MjM4YTM0ZjliZjRlZDFiZGZiYjc2OGQyZGJhMzQ0YWVkNDA4NTBkODZjZGFjZGRiMGI0NTJiMzdlN2I3MjZjMjUyYzAyMDdlNTcwNjU2NmY0YTFlMDMyOTYxMDlkOTJjYzc3ZjA0YmUyNzc0NmVmODQxYjQ2M2M1NjM3YWU5NzgzYmUwYTc5OTZlNTZkZWRhZDk0YTdjYjFkYjgwYTVkNWY2YzQwODk5NGU2Y2M3YjhmOTVkZDk1MDE3NjY3ZTczNDc4MDY2MjgwNWViM2ZhYjY3MWUyYTkzZTk0ZTdhOTk5MWVjOGVlMzllYTVlZGI0NTA1NzhmMWI3MDkyNDA2NGVlM2Y3NzZjMTk5NDg3ZmU4NGUwZGU0Y2MxMWU3OGFiN2VhMTA5YzUyYzI3MzUyY2FjNGU2NmQxYmFiNjI4NzcxZjVhMWMyYzNlOTMxNWRkMWM0ZDRiZTIwY2JmMTgxMGZkMWE4ZGRiMTkzZGEzMjI2NzFjZTRhMjE0OGMzZmE1YWM2ZDk0MWRiNTViNGI5NTZlMGY0NzYzZTRjOTA3ZjdiYTQ1ZmIyNzE0MGJlNGMyNGExZDMzMzYzNDg0MjdlODRmMmFkYmYzNmQ0NTc1NDdiNTExMDFlNzQzNjg3NDFlMTExNmQyMzIzZTA2YjhiMTBhY2ViM2UyN2YyNWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7107m6iqEhseFF86IHwn6ajm3FdWpWCfF-Y-j1YXi7IJMjHuCDIZ0v2gMaxFP4CoX-tsUU5HfNY9pVPfvuMDYw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103854_29_2463_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.519Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlU1WVoyODBNdkZXV3gvT1B4UWlCQjZWcnBGaytsYlEzc1AvRWhPeGZPZUVYWG5ZL2l4clRvTXdSTFhhREVtN1dHbnA2STRPWTBhemZicVc5b1Y2NWVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyN18xMDM4NTRfMjlfMjQ2M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWM5NDIyNmVhZWI4OTE0MDBiMWM3OGVkOTZkMTk0ZGQ5YjhlMzBmYTliODQ5NjIzNzk2ZTYyODViOWM1NTFkZGJhYjkxZjIwOTFjMjllM2EzOWUzNjY2MzE2NjY2MDgzMjIzZGYzMTgxYzJlZmE1NmM1NWM1MTg2NDg5NTdkMzU2NzBmMzFiYjNiM2E4YTQ3MDZjMDI4NjM2NjY0NDAwNmFjN2M4OWM4YTE1YzVjMzgxZWRkYmU4NGE3ZjU5NzEwMjhiNzhkZjdiY2NhODJhZTQ2ZDQ5ODk4MWEyYjZjZDRjMjljNDlhNTc3ODRiOGUxODBiZDllZDNmM2I4MjIxNTBhMjJkMTc1YzliZjUxYTRjMzMwOTMzNTY2MTkxMzI4ODJiZjJkM2RhMTNiZWZiN2Y0MTg3NWQ2OTc4ZmFkYzA4M2RlZmU3ZGU3YTE1OGJmOGE4OTJmYzI0ZTIwMDNiZDU4YjA5Y2UyMTM1MTI1OTcwNjMyNjIyOWY3ODk1YjI3N2VjNDY3OWYzNGUzNjYxZDQ2MWFhMjU5NzU0ZTdhYzAwOWFjYzI1ZTc5NjhmZmFlYjY1Y2U5ZmZiYmU0NmVjNGVhNWU2Yzg5NjhkZTIwMjFjYWI0YTBjZmVkYTc2NTIwMmRjOGZkZWJmYjlmNDIzNmJjNWZlNWQ1NmQxYTRlYmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RvbZ_eUx21MVeVabXMqKS390LWmhbnH0wGl-FNK244YyfGE3bnSmW0lZ20dmEDa9S_uDCOkc5-668h6wEpEbZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210327_103854_29_2463_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.525Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlpubUhndUxKKzJWeHdoNzloWXd3TEY3NWoxem9CTk5LYVUySnIzdmkxdTl6VE5jem1ERmFpQnJ3eXoyMUlIV0JKQlIveVczUEZzcTRkY01qM2ZVMVh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMDMyNTZfMjFfMjQzMF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmY1MzU2OWMyOTUyNDc2ODFmY2RhNDYxM2M5YTQ2OTY0YWU1NTk5ZWM0MzE1NjFjMTk0OWYzN2FjMTk1NTdhMDM4MmY4MDc3NzQ0OGEyZmUxNGQ5NGQ1Mzc4NjhkMmEyMTllNTRkNWVkZDc1ODU4OWQzMmE0MmUzMzY2YmY3ZTYyMWU1Yzc2NTQ5YWRmYjg2MWI1OTlkZDgzMzE1NWVmNGYzMWE1YWY5NDEwYTJlYjU1OWY4NDU5MmU3N2MwNTBiZGY5YzMzOTc5MzQyODZhMDI2OGFlMjU2YWQ2N2Y4MGYyYWNjMGRlMmRmMmNlZGJkZGZiNjUyZWJmNDA5Zjc2Y2U5NjdiMjBjODRhMzkxMjlmNjA5NDJlMGVmN2JhMjAyZGY3MmU1YzE0ZGJmOGM5NTZkZjU5MjdmMDI4NmY2MjU4NGRkY2Y3OTVlN2YwYjM0YzJlNmMzYTFkMzk5MmY2MTRjNTk3MmI1OTYxZTBlYWIyMTU2ZDM5MmVhOTBjNWIyNzBiNjI1NGMxYTVhZDY4NGJkOTdiZTFkNDk2NTU5NTBkODUwYTAyMjIyNWU0ZjZjMTY0N2Y1MDlkNTEwNGRkZTNjYWUxZTNjODVjYmExMDhkZDVkYjE2ZmZkY2U1MGE4YWEyNjNkOTZiYmJiMmMzM2U5YTE2ZjBhMTVkNDcxODJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zzO_Zw6nbzi7k-MSVvoyGztoCgs8NfciLMw8JNB-8G33uYi2akYQJe_sTzYhQ5k7v9UUUMYL50JeRAMAdVrnoQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_103256_21_2430_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.527Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imh4VjVSak1CUGtoVWhHWEpZNjM4eUlrU3N2dnNHcUpVTzczRy9IVm9MQlZHa2Y3Mzd1aWMxTVo5a2RMcUxUcFFjc0NNcmQ0Y0hIdThSRkZCUzZwdThnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMDMyNTZfMjFfMjQzMF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODRhNTVjNTc1YmM1MmY1YTNhZDVhNzFiYjMwYmI0YzdiYjc4N2I5YjE0ODgyYmY0NWZhZGY4MGE3NDRlZmZkMzAwZjk5N2RkMDQxZTQ2ZTEyODlkN2FjYWM0MzNlZDY4YzUxZDQwZjY2MmRmOTEzOWYwYzI2YmVhNTg0MmIwYzllZDJiYmU5YTJmNmJiYjE1NTAzMGE5ZTRmNjhiOTQwNGY1YzgzMGM4YjNiZDkzOTFkYzMwNTU4NjEzMjIwN2YxOTE1MzI1NDQxNzMyNDhmMzJjOTdjYjc5ZGI2MDRkNjRjMGZiNzg4ZWFiZjkzZDQxMTAwMTM4M2ZiMjNmY2M4OWE4NDFlZjFlYWVhNzc5YWNlZGRmZWViMWFlYjNlNTE2ODY5N2E3MGRiNDg1YThkMzViODBkODcwNzkyYzYwZTY0MTg3NWY4ODk2Y2JiODI4ZGQ4YzliMTI2MGRhZGVhNzc5ZjYyZTQ4MGJhNGFkYTFiODNjZWYyYzVhMmU1NmQ2ODNiNDkwMzkyNmIzNTQyN2Q4NDNhNWZiZTViNmE0ODk2ZGI5MDg1ZTVkNzhlMzA5ZGU2M2JhMDc0YzMwM2E4NzE4ZGUwMWY4MjZiYjBkYzI2NThkYTU1MTA0MzBlZmFhNzE5MzAzYzhjOWE3ZWRlNTg0ODkzYTMxMmM2NmI1ODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LLUlmP2GSK2fzijqEcP1C1oR00jc9TUCiJl8efm5_Yf9ESAHdIo03-Tv61ciqe95XxlN9S8K59nf_NzX7PJFIQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_103256_21_2430_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.530Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkVRMHhLdmNMT25VeW05LzFBWHhNUDBZcHVSV1h1NjZuQ3VseUlKdDF2L2N1a3dhcmdLdHI3MXR2WHJIWUdNRG5oYkN1K3lxQjQ3SmZNc1k5ckJPQmd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMDMyNTZfMjFfMjQzMF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWQ2NDY0MDZiYzdlNWJhNDcyNmJjYWFhZWVlMzkwOGQ3OGI2MGFmOWEwOGI5ZDY3MGIxZjQzZDY2NjZiNzAyMTNiMGJmMWFiZGQ5YWQzNjAzOTQzMTg3NGY3OTgxMGUwYzBjOGQ5MjFlZTk5MjMxNjM5OTllNjk3YzZkMTM0ZDJiNTZjNmUyMGVkZTkwZTdmMGVhMzFiNjRmMGQ0MDExYzRiZDYzMmU5ZmYxMjk5YWZlMTkwNmY3OWQzZTdjZDU4MmY2NDNjMDc5ZTk2MTAwMGFhNWY2NjFmM2VmNzI5N2Q2NTBmYjJiZTRkNDIyZGFjNGY5OTM2YWEyNmYyNzFhOTcyNTMzOWFmMTk0YWM5MDk3YWQwYjA3OTQ3ZWRlNmE2NTA4YmMxMjk1NWM5YzQ4ZjhkMDM5N2QxZmU0OGI0ZmJlMGY3MDRiN2NlNWNjMmEwNDgzMWYzNTAwZjA5YmNiMTBlNDgwNDJmMTIzODZkOWVjNjZkZmViMzkzMTdiOWE0Yjk3Nzg1YjkwZTY4NDU1MzMxOWNmZTE3YmRlYjNlMmZmNWQzYzA5Yzg4YjA4ZGU4NDYzNjVkYjU5N2ZiZDhjNjMwNjRhNjRiNGU2ZTU5MzEyNDU4YzczMWZlMGY1NTdkMzZlMTlhMWQwOTgxNjkxZTNhNGRkZThiZTNjZjg1ZmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-xerbDXWM5CCNrEbySrePHykGuDe1nBVJrmJIgdSYKoQL95NsZCVi8BYYMg69wVUEM_h4Xr66qXStyo7-u12_w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_103256_21_2430_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.534Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IklyQTk5eWQ0L3lmcER1S3QxYmFvdHZQWUNGbDU0bUNwUVJ0VjF1MUNSeEJxN25QeTI5QjdrSUV1bGcxWHAyYWp4cFM0NzNqNUtMNXErWTVrWmc0TFRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMDMyNTZfMjFfMjQzMF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzVmMGE3YmY5NDU4OWZjZTU5YjE0NjZhOGMzNGI5OWJiNTc3NTFiMjg1OGFkZDAwNzZkMThlZDg0YWRmMGY2NzM5OTc2MDg5MjhiZGFhY2E1YTczMWVjOTE0YWE5MjE3MThiNmNiNmI0ZTFmYmNmZjczNmZmZmI5OTIyYmNhMWY0OWIyMThjMzliYjNlMzMyNzY4ZjYzMjE2ZDYyN2FmNWM4NzA2MDUyNzQ2MDc0NjExOGJjNDRlMDE3ZWRkOTdjZDBiY2QzYjIzN2QwMGUyYjJmYWRjY2MxMTdhNjNmNWVlNWYxMDU1ZDkxMjRhZjJjYWViYzNjOTA1MWY0NTczNzM1NjQ4ZmRhZTk5MGQ1MzRjMmU3YWY0ZGVhNmFiM2YxYzY2NDUwMTcwN2NkMTVmYTM4YmQ1YzFiY2I5NjBmYWNmZGQ4NjNmZTYzNzExNGUyMDg2ZWM0OWE4OTI1MmQ3ODJkYTdhMmE5YjgwMzZkM2EzZTgxNzNlMWM3ZWZhY2YyYjU0MzMwMzRhMTI5MmZhMjUyMzUxYjQ0NGRiODNkYjI1YWI5MjRiN2NjNDY4ZTNlMGMyNWQ1MzkxMTg2NWU3NjIyYTk0NzA2NTYyNmYxYjczYjY1ZTlkOTA1Mzg0YTcxM2Y0M2JjMDQ3YzVhNGYxNzdiYWZjNDM0NDhiZWNmMjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mGDSKqQe41BkoXQqcwfdb8YtCI77c8yG5B0YqZgbfw05_dLKKiiN6qb4A1Rp8TBWrnp6Og3lS0HYsE-wsvkdPQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_103256_21_2430_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.542Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImV6WG5YbHlrVTJXWHRaVE8remRtdEpNdVB6OVVPZXI5TnZnaWkrSHF1TlFJUTd5NVdUUHkxMmd4eHZBaUpBeHlVakxhZzNJRFRPS250eXZCdWVMTjdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjRfMDVfMjQ4NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjViZGY3NjQxNjU4NjE0OTE2NDA4Y2U3ZWRiYzI2OTljYmUzNzIwY2U4YzYzYTU2YmRiZjIwZGMzZmI0ZjNhOTBkNmE2Y2FjM2U0OGFmNzA2YmU1NTBkM2U4N2IyNzkxY2E2OTA2NTRlODc2ZDQ3NDFjMzc4NDdiMTM3ZDc0ZjAwNjVkZTAwZTAyMzRiZWZiODgyZWZhYTE3MmU4YTQ3YThhYzdlOWM5ZTFmYzZkNjhiZTQ0NjVmYjVlNWU1ZmQ5YmYzODllYzRlMjZiNWNkZDUxNzQ4YTcwM2E3N2FmZTY5ZDMyNzMyNGVjNDBlNWNhMTgzMzUwMDM4MGYxZWY0NzhhNjcyNTUzZTZmM2IyNmZkNWEzODM1MWY0ZTgxMjgwODNkMTE3MjgxMDU3NmU0MDZkZmMyZGU5YTAwM2MwYjA2N2Q1YWExYmVlNmNkYzA1ODIyNmVmOWViOTc4NzY1MWE0NzdjYTU1OGVmNDllNjNkOTVmN2U0OWIzOGI4ZGJjNjQ1NzBmMTVjYWVkZThmYTU1NzNkOWFhOTRlYTczZTM3MThhZWM1MDBiNGY5NTQ0YWZhOTM2ZmZkZjJlNGUwZTczOWQ5OGRkMzFkNmM4MWE2YTNhZmJkMjBlNDYxODdlM2IyMzA1M2YxNzQ5MjBiY2RmNTc5MDZlZmYwMTQ2ZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2k-H_1xLnhULx6mcKGG1njmt2Sk4jjjnbi6LzhbQP4hgxbLete5-EWnopTGkrdv7_2ThOrCplJZn_J9abNy6Aw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110224_05_2485_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.546Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im93ZXpRUngxRVlDakwxelRPQnpwL0hCdHk0MjNXbS9TeCtkRncxUjFXSk82UHh4aHVSdW01RHFxbVY0dzJGS3dpRE1qWTJ0NmVEdGdld2ZSMjYzdERRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjRfMDVfMjQ4NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTlhYTA3YzNmM2UyM2E2ZTFmYmM1OGU4ODIwYTM1MDhiYjNiYzE3ODA3OGZlYjM2NDVlMjk3OTliMzFmOTM3MDU1YjQzMmI0NWNiOTJiOGE0MmI0YTM1YzI4YzIyNWIyNThjNjI3YjI2ZGRlM2NhN2UzNzA1MTVlOGE0YTRmYTI4NmZmYjIyYTYwYWJlYjU5ZmE2NzEyOTdkNmIxMmFlYWY4Y2JkZjMwMjU3ZjkwNGUyNGFhODY2MjA0NTMzMjZjY2NlODIxYzBkMjQzZmNiOTM0NWMyZDkyMmVkNmJjMzY5OGM3YWMyOTAzMDY1ZjkxMDgxY2ZkYzI3MzdiMDg5ZWMyMjZmNWI3NmUxNzBjM2YzYTM3YjU3N2ZhNjg2YjVjNDg4MTAzZjBjMDg5Yzg3NTRkYmE4ZWE4Y2Q2MzQ3YmZhNWZkMDgzMzQxYjAxNWQ5YmViY2MzYmZhZThlNTNkYTRiN2NlOTg0NTA2OTNhOTY4NjY1Y2ZhNzNmOThlZGMzODkyYzcwYmE4ZWI4NTk4NTJhOTdjNzFhNjdjMjNjNjMxYzJmMGM3MjE2OWY1YWRhYjVlN2Q4MWMyYWJkYjc1NjU5MGIxZDZjYzJhM2U5ZDY3YTNhN2E3MjAwMGQ3YjUwNGZmYTc5NGIxNjQ1NzFjMmVhMTQ4NGE3MjM5NWRlM2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OCviC0hMrM85kcXuYaVjF9gYrNMHBA5IzkYrft7mD6he__LX4h9284uonf80fLOEtAKM7J7BRJ0jTrbtOwtvPg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110224_05_2485_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.548Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJnRWpJbWJ0NWlmSzZCYTZKK3RpbzZDSWZVU3A4L0I2UlJQN3haMndxZ1cxMVcxbXJBV1YxWUhwYTBVYWg5VFBiOWFTdzlvcmZLSGJjQW5BbVRvcTNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjRfMDVfMjQ4NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGZhMWYyMDc0ZGI2ZmJhZTZlNmQ3NzUwNDhhZTAwYjI1ZjFkZmUzMTE2MWVhMDFhZTcxZGRkNTk4YWJhZWY3NmVkNGZmM2IxMDkzMjk3OGQzZGY0NGJhZDNhNWFkYWYxYjg5MzgxZjgxNjUzZjEwMzM4YzgwM2VlZDU4MTkyOGUwMzJmYmNlMTFjNjIyNDY0OGI0N2Y0ZDlkMzVjZjYyOWNjMDM2YTJkNTBhZGUwODllMzgyMzg5MDE1Y2QyNDE4Y2NmZWZlOTZjNGViN2RiMDY1NjM3ZmZlMTMxYzBjMzFjMjhiYTJlZjEzMDhjNDEwMmM4NjEyNzdlMTJmZWY2ZWQ0NDBjZWYzZjg1MjQzMDc3NTYwZTQ5YWNmNWMyZjM1YTU2MmI0ZWRiOTU5YTRmNmI1OGRkYThiZmRhNjIzODQ0MjY4YjM0ZGI3NjVjZWFhN2JkMGUwODQ5NTQxOTYzN2ZkYzQzNzM0YWE0YzNlMjdjYzc3MmYyZTA3MGEzNjI4YzlkNzdhMzdhMDI0MjgxZjg1ODAyMzYxZjg3NTViMTVkZDdjMmUwZmJmNGIxNWNlNjE4Mjc1ZDM2MzQyNjI0ZTI2ZGJkNTUwZmNiNjI4Y2VmN2U3ZDJjMWZkNTJjMjJiNzhhNDhlZTcxYTM3Mjc3NGY3NThjNjEzYzc0MGVmYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.N5MTF7t-diaMyZWVTxMidK5p8sInoDYVYkSe5xnziHrEFRX0cMgn3teWihIYkifcRBwG6Eerx2X75irer2taYA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110224_05_2485_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.551Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Inp5OHdzMll2NEtjNDVOQUlSMTlJRjlrUk5uUHoxZEFuRDU2djZCaWdRTng4UWk4TkUzeVpEci8wUkdBVTFRZmZObk5uZE5EajJUZDBSTW1CNXRFeXJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjRfMDVfMjQ4NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTExZjAwYjQ1ODA4MjI0M2JlNTk4ZGYxNGJlZjE5YTNiNTQxM2Y5OGVjNDQzNTlhNWE3NTI1MGNkMGRlZjhhMjhhZjg3ZGU4MWZmYzNlMzUyOTcwZGFhNWY2MTEwYjRhMzA3NDVhNGRmZjRlMDI5MjI0YmE2OGU3ODVmZTI3ZDc3NjQwMTRmYTVlOGNjYjVlYTQxNzAxODE3N2RjYWY2NzllNGZhNzE5M2U4OGQyODQ3MmM5YzdiOTFiMzI2ZTQ0NWE2NzVkYjYyODBkYjQyZTNhMmIxMTg5ZWM2NDQ1OTJhMzZjNDgzNWMyYzhhOTUwMWQ1NDRjZDY3YzQwMjExNDcyNjNjMWM0YzZiNjkwMGJiZTUwMTE2YzU4Njc4ZDZlNDIzMDYxZGZhM2M2NDQ2MzRhZWI5ZDQ2N2JjZTkxZTNiZmFmNjM0NTRmNDIwZjQyMTlhNjMzZmRlMzY0MTVkMmI3ZGFjMzQxODgzOWEzNGUyY2I0ZTI5YTU2MTYzOTBiYjg2OGVjODM1NzNmYjQ4YjFiZjFiMTEzZjJjOWY5ZGJkZjUzNzk2Y2E5ZjI2ZDIwN2YzMWU5NjJhN2YwMWZlNDY1MDVmMzU5MTkwYzBhOGVlNTcwNjM4YjdiNTIxODIzMDU5N2M1OWI3NzMyMTUwMGQ3YTQ0N2Q5MDFlM2MwNDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6rqdXTm4YH9i3nOA5ttlPFArcghl8W3GZR7EXrKwJab5X-qlNfd7G1E6YPmZwCGUmHKXxed7TlZJpNi_w7Peqg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110224_05_2485_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.554Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlB6VmxmRE1iWmZPR0UxaDNnRXVjWE0zakgvc3pSd2d3MWZKMTlrcnMvemhHdUEwVFVMaWYvaUZtMHIzbk15dUJOUE41cEtqTVhyRHZPOGNiMkdpdG5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDFfNjZfMjQ2NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjRiNTI5YTIzM2E3YjUzNDllOTczMmU2MjRhYjlkMWE4N2I0MTQ3NjQzYTVkNDdjOGNkNWM0Njg3ZGI5MDBmNTk2OTQzYmZmNmY1NmViYmU4MGUyNDdlODkxYzk3MzMyZDE3NGU1MDJkMmYzMGEwYmMzZTliOTA1MDZlZGRiZTRhNTI1NWMxYmI1NDgyNTg0NjU0OGVkZDE5NTc2ODFhNmZjYzAyM2NjZDI1NmJkNjFhZmExZDcyN2ZjNGNlOTc1NjVhNGFlOGRiNzEzODMwMjhkMmM3NGY5NmQyNjQxODE0Mjc5YzkwZDBiZThjMDkzZTA3YmFjNzBiOTc5ZTY5YzlmMThjY2MyOWU0MTEzN2ExYjliNWYzMjQyNGUxYzU3ZWJiZGFkYTBiZGZmYjZlYzhhOTc3M2I5NjNlMmFkNjgzZTJmNDU1ZmZmMjgyYjZhOWIwNjU2MTAyNGM0NTgwOTdjNjkyZmI1NDU0YzMwNDFmMjA0NjFkN2NjYjgyZTc2NWU0NmYwZWMwYTgxMjA3NjQwMWIwNmU3ODE0Y2Y2OTgzOTQ5YTcyN2ZjZmZlMDQ2MWEzYmUwZmEwMzYxNGU3NTE1YjE1OTdmYzUxMTU0Njc4MTVjN2I3ZDAxYjk1YTBkMTAxMDBiOTQ5ODU4MjMxMzVmNmUyZmQ1MTk0YTMyZmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MtOdzOJuFyT82MsS8qSChHEsN9yZybmP-gnMnGLeBRsuvIFha3_Y24ljtdfPoabDhG0Ggr-Mh27Xm2t0ZtC6lA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102801_66_2464_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.565Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkNqMjkxMU4zOFE2cXo4NHFrMWVmQnc3eHVhaW1iMXdTdEJ6eThyTGpsTGNQcUthaWFYNzBsWDU2eEhYZ1Vsb2RzMlJhWnVVbThYMGVReUJHWllLM053PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDFfNjZfMjQ2NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGRmODcyMzU3ZjA0OGJlNWY5N2QyN2M3YmZlZDFmMzRkNjAwM2I0NDQzNDUzZGVkZDdjODM3MzY0ODAxMWIzODkwMTU1ZmQzOWQ2MmEwZjBlMDUxZjQxYzA1NjMxMGRkODFmYWQyNGNjMzhmYjQ4OThiOTIyNDIwZjkxNDY2ODNkZDdlOGFkMjllNWY5MWMzMDVhODQwNWI4MWVlMzA0MzZiNTdlOWI0ZmE1NGQwNmE1ZGUxYmYxYzJiNzE2OWRlYzkyNGIzMWNjOTM1NWE1MjliNTgzYjg1YjQxZmQyZWZhZGNjMDJiOGE5ODJmYTk2NTgxMzdiZGMzMzU1MTEyYzVjMGRlNDk1YjA1MjMxMzkwN2QxNzBmZmY0YTdiNjYwMzljMDEzYzI5ZTNiYzFiZGIwZWI0ZWQ2MzUwOWFjZTRhNzU1YWJkYjRiOWM2ODIyNDViYmVjOGQyOTI4OWNjMDZmZGQyYWEzY2NhMzFmYmM4NDQ3MDQzNTU1ZjAzMmZhODA1MDY0NTg2N2I0ZjIyYWRlYmZjZDQ0MDkzY2QzOGY3M2M0N2EyNDYxNjgxOGE5MjcwNjZjMGMxNTA2MzcwNGQwYzhiZDk5YWVmZjhhZTgyOWI2ZjM5MGJjYmIzYzY0OGVkMjc4MDcxNDU4YzU3ZDdjOTg1NDE1NDA0MzMwMGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1wmen8p6P40nQPj1_2QgHOukc-uAYBIKne1Y3FLh6OMk8cafzgn19pQqNsqnm_mvLszDgVXsMq0RV0ciyi0pmQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102801_66_2464_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.568Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkZmRjhzaStKVzVoTTBCYjZnTVZtcFVxWDR3eVQ4WGt3OG1qQWlmOVNYSHRuSndNZDArMFlvUmVpUTE5UjY0SkI1cWNzNzNZNlo1bUVXY0k3WEthWDh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDFfNjZfMjQ2NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTk5NDM0OTAyMDI1NmJlZmMxMjZlZGJhM2EzNzQyNzQ2MDJhZGQyNmQ4ZjNiMTI5ZjRlZmFhMTc2Mjk3MTliNDIxNjJiMjY5Yzc1OWUxZTU4ZGU0MmIxOGJkMDFjYzU5Mjc0MWM4NWUwZTgxNzZiN2U3YzY1YjUxMDkyNjY3MGI3OWFiOGZlZWM5YmM3ZDk0NDY1OGI2NTM3NGI1ZTFlZmQ3ZDg5MTZlMGMxYmE0ZWQxZjhmMjAxZDlhMjUyMjVhNGMzY2Y1MTBlOTdjMGMxMTc2OTVjZWI3MWZjODNhM2FmYjdhNzgwMjkyNWQ0MjE3NWZhODgzZGI1NDIzNzE5NWQ0YmZjMjE0NjFhODU2MWUyMzM3YjhhMTI3MWIxNjE5MTU1MmM1MTlhYzk4N2MxZWRkMDM0MGQ0NTc2ODIzNWE3MTE5MzM2NzYzOTJmN2E2MGZjM2M2NTJjN2JkYmZiN2U5M2NlODc4YmIxM2ZmY2U3YzY2OGZhMTVmMjJmOWRlMmYyZjRlMGQwMTg5NWE0YjFkMWJiZjI5MDU4OGM3NTBjYWEwZDJlNjNhYjJmYjljODhkNzJjZTU4N2JmYjJhNDNjZWZiNzUwOWE5MzRiYmQyMTZhMmQ5ZGU2OWZhNGY5N2Y5Y2ZjYmYyYWZlYjBjNGFkODAxMmQzNjZlMDkzODJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.X2uquOWUdwjgk3WWxI9hcjj7iR1P1g5KKDj0QAsNn_CYaiXbfFNEXXx6Xhd51KGDMwPwMX_rD6NBEqusNOGx5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102801_66_2464_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.571Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjEvME5FLzAxWFZDRTc4RzRQMTdOSWoxTWN6Y1paSWZBQTJGb3Q3RW52MzJHVDQyeklXeXRIYnZNMUx2RkFMTHh3eWpBbURQbUVtd1lqUk9sclNCZmxRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDFfNjZfMjQ2NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWMwMGUzMTU1MzhjZjU0Mjk3ZDA1ZTgwZGEyMWEzOTMwNzc4OTQzNWE4NWViNGMwY2RjNGVhYWU2ODkzNDdjOTY2NGI4NmMzOTc1ZGU4YzU3OWU4ZmZmZDFmMGU1YTE5N2Y2ZTRkYzc4ZDZmMmFmZmVmMjJiM2Y3NzVlYjMxNWEwMzE3NmRhYzQ1NmZlOTZhZDZlNjA0NjBmNWYzN2E1Y2NmYTZhNTRlZmViYTdlODUyOGQ3ZjRjNzBiMjJhNWE5NTUwNWIzNmUyMDlkZGExODIwZjhhMjU3ZjZjMjllY2IzYmQ1ODNhNDkyYzM1NzgwNTdmNmQyZDVjYjExZmEwZWVmYTZiNjVjNjlhMzNjOTg2MDNmNTMwYjQyMjNjZWM5NWM1NDIxMjM4NzYxZTJiZWYwNmJlOTQ5ODVkYThkNTkxZWQ4OWIyNWZlYzUzMDJhNGI5OTE2ODlkZTMxNzYzZTFmNjU4OTRiMDlmMmFjMTQ5M2Y0YzBlMDI3MzA5OTRmOTc3NzM3ZGNkZjhmOGViZTk0N2UyYjdlNzRlMGY1ZWQ0MzU0OTc2Mjg1OGMzZjNmOTMzY2M3ZjI5MmI0Y2Y2YjEzZWRhOTRlYWQ0YTk2MjJkYjEwZjcyN2Q1NTdmZjlmMzljYzM0ZmM4ZDhjNTNhOTA1NzVhNDU2NGZjZDY1ZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dDuxSD2tIGOyzSHfaVMrUEPbwhK130H17613dZHjReqOKrHN5kV7uxbKzqhEvMH7sNa5C0qVRwTK_0UqfH5aVw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102801_66_2464_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.576Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjY4RFhCbHlvZmFrNmMvUkg1Wlc4TjdkNll3T2MrV2JtTG9BK2Q0VndVMGRod25mTCtvZGdNOHE3Z1dyODZLTVdLa1VBWVBQWkVzZHB2U3BqZ0NzOVlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjZfNDFfMjQ4NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTVkNmJiNjJhZjM4ZGYxNzUxMDdlYTk3NmQ0ZmYxNjA5YWMyOTVjYWY3M2RjMGIzYzcwZTllMzNkNGNlZTNmYWIwMGNmNTJlOWM0YzZmYmQ2YWIxNTAzYWQ2NTRhOTBhMDVlYWNmZWQ0NzE0NTU5NDVmOTdkOWU5MDU3Y2NlZWVmMDk1ZTI5ZDYyODg3ZjljMjU2YjljNDEyYmY5MTZiNTZlZDdmMTVhYzhmNTNhMTcwMTk1MDNmNmEyZDI0NGE4ZGRhYjRjYzU5N2Y5MDUwYjhhZTVjZjA5N2I0NDg4ZmUzZWJiZGM5ZTYzYjNkODUwODljY2NiZDg2MmMzNDFlNzc4Zjk0N2JkNjc4MmQyYTI3MzBjMDVjOTFmZTM4ZjllZTJhYjY3NDU0MTdkNTc3ZTM4ZGEwN2IzZGQyYzdhMTFmZjdjZWQzNDNjOWRmMWJlNWI3MmExNDc2OWFkZWM4N2QwNGRiMjE1ZTk1Y2U2NGZjNDY2MTZmNTk4YjNkZjZjZDk0YTg2ZGYzYjQwMDMxMWNhYWIyZmM2MWZlNGIzZGFiNWNkNDM0YmZmOTlhZTVhMjc2Y2MwZjNiN2ZkNGQwMWNjMzI5MGJlNzYyZDE4ZTEyZTczN2U2YTRhMjFiYmFmMTM1YmM0NDA4MmE5Y2I4ZjczZTI2N2JiYTBjODc2N2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FlzdQegODM2mkLPHzdNYKQ3pv1CzhO2dGvxUYHNz8fEMtCi4P1enhCjGPqBRbe9gNB-4fP4ANjVgjeb29-CqUQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110226_41_2485_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.585Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im8vWWFWR1lwWlNjdkpzTmxaK1hsOGNyTk5qS3d4Wkx2czlXRlA0TjlmV2YwOVdtNUVVejJlWlRESXlPSDFYaXRkNHp3RThITTZQQXA5Qkk5VHdETmhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjZfNDFfMjQ4NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWJlNGYyZWZkMzVjYzFkNTc0NWZiZTYzMjgxNTUyYjEzMjg3MmI0NzBjNGQyYjQ1MjQ4Yjc1NjRlYzkzN2NmYTg5ODAxOGQ1OWY0NzcxMzBjMjA5NjMwYjgzNWE2YzM0NGJhZTc0NDJiNzAzMzgzYWRjNTU3NTNiM2JkOWRiZWY1Mjg5YzYyMzMzMWRjNzY1N2QxZDY3MzEyOGY5YzIyZTI3YjI2YzNjZDRhYWMyMDU4MzFlMjRmOGEyMjk3NzFlNzc1ODc2NGRhMDIxYjIyNWQ0YzZmMTI2MTZmZGU5YjkwYjZlNmZiZjZmNjhiYzdiY2FhNDhkMGQzMzFlMjFlNTBmMDgzNzA3MWQ5YjZiOTY4ODQ3Nzc3NzUwNTkzZjNmMzZkNTc1ZmJiZTc0MzZlYzVlMzkyMWMwMDEwODNiYWEwNjllMGY0MGMxNDhiY2RmNGJkMTcyMTEzNWJmYjRlZDdmMDk0Y2FjMjgxNmE2ODkxNDQxNWM3ODFmNDNkZjI3NTYzOGFmYzY4NmZkZmQ3NzljZTU0YzFhYzZmNGEyOThmODIzYzg0YWQ2NGUzODQzZTE1MjEwZjY3YWJlNzViYWQ2YWYwNzQzZWJiNWRiYzU0MDUxYzM5Y2NjZjQzMDE2NDRlODFlMjM0OWNmODc5OTQ1ZWU4YTM2NjgwNTk5YzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LGkm-cToaOW-MBxeGJNZmXmaRHYm1oE39AE06fxXlmfcK0IdS2Mmpdkh-aFC7ybVrdoA46SQdz1kah_lPFGk8Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110226_41_2485_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.588Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNES0NVZmJpT1lRWUFqT0ZjSTNsQ1Q3Y0w0S1JwaWxlaDR2NGltUzc3RUYzcDB4SnErR0VCNlhqRHBxY20ycUVaR0pLNVRMSEZ1TU5aMENkc0lxeHh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjZfNDFfMjQ4NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGJlMTM4ODUxMGQ3MDQwNDBiYjJkYjRiN2JkZTljNzg4YmIyNDM0YTBlOWNhNjUxOGNlZjI5Y2I0OGUyNjUyOGZhYzZlNzgyOThkNWE0ZTJhMDljMmYyZDhiZjU2ODUyNGY2OWQwZTBjYjAxY2JmNTEyMTU1ZmUyMWRlNzgxOWIyZTg1OWMyYmUzNzFhZTI5NTJlOGE4NmE4MGQ5OWY5Zjc2ZjA4NjUxZDQyYTk4ODY2NWMxZmNhM2FmOGU5M2U3N2VjNmE5MGY0NTUyNWNhZTNjNDlkZDIwZjAxZWI5NzIyMzI2YjRmODZjNDQ3ZTljZDlhYWI5ZjYyNzI3NGQ3ZDhhZGJmMDMzYjgwMmZkNjFjMjJmMWYxZmMzMzgyNjYwMTVkZTY1MTYxZGU2ZmYzZmNjNzEyZmM4YTQwYTYzYTRkMmI1ZjdiMTJiZWM1Y2VmMjU1ODVlOTY4YmNhY2ZhZDM5YzAwMGQ3YmZjZjc4NTJhNTAyY2Q2ODA1YjhkMzM3YjcwYWJiZTBmZWVhOGI0MGVjNTkwNWNlZmM4OTA4MDQ1ZWI5ODcyZmJjYWM3ZGZiYzRlZDkyYTUzMzhkMGNhZDliMzM3MWJkNDZiYzdkZGM5MjJiMzU3MzI5NjI5NGE3MWYwNWFkZmZlZWRkNWY2YzYzZjFlMWVkMWY3ZjExYWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8z-W9vU8fLPPNKLBGnfna78DPhYSALbvgFTPbbz3JQsPMj_3kW8e5fBPGcxAtL2PRmt9a-Yn1Hj1nBydoFOZIg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110226_41_2485_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.591Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlI3dDJCUEhvY3FaTEhUR2pQNzI5dEwwcDRqL1labGFUNWtya3hOdk5WeWhCVkRVQWxnd3ZtYm0vMTY5TFRjRzQxMVhHWlFaNjVjYkR4M0F3MmFsc1pnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEwNF8xMTAyMjZfNDFfMjQ4NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzAwZWRhOTM5ZjY4ZDZkMjc2NTk5NDAyZDMwYjZmYzAwNWQwYmU2MWY4MGI0MjAxODdhMGMzZDhiM2ViMDk0YmNhZjA3MjczMThlODIzMjUxM2JiMWI5MzIxOTEwMjdhZWY4OGIxYzE0ZTRiNWZhODMyMzc2NzY1NmE1OWI1Mzg4ZmI0M2FhZmE3Y2Q1N2E1M2Q1NDI1NzY3NzNmOTNjOTRiODRhYWJhMWE1YzhkMDRhYWFlMzA5YjIxMzZkMWY4ODljZGM3ZTViMjRlMGEzYzJiYmE0YzZiMGE0MDQ1NGFmZWJjZTQ2MGZmYjQzZmJkNGM2MTcxM2UzMGY1MWExZjI5NTEyMjY0Njk3NDUzZDg5OTI1N2NkNzJjZmFlMjkyMGY0ZGYzN2IyZjFjMTBjZTkxMWNlZWZkNzk4ODIwNTFmMzYwZDc4YmRhZTJlZDE5YjY3OTc2ODA2ZDNmMTZhY2I3ZTM0Mjc3M2QxNGY2MzM1ZjUzYTAwODI1ZTAwMTc4M2NlZWZhYjFiMDkwMGE3MzI2YzZkMGE4OTgyNDJhMDMzZWNmMjcxNmM2MzNkMGMxODFmODg0NDQzZjIwZWNhZDc4N2YxYjQ4M2Q1NzcwZjBmNjg3NDA5ZjNjMGEyYjA2ODIwMjk1MmYxNWJhNDZhZGU4ZGU4MmRiYTQwYTJiMDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PBbIZy_KRin7DSxGKeckCO0HHF4TYGTYFkVvXwIRzr5JnW1IuToa_fJEuiM8r6BeCwmaE06jMDLJLM_jUyK5sg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221104_110226_41_2485_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.594Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImR6eFY5WUdOTmdqdStwV0dqa3dOZ1hCYlc0dHgyaXJzRzB3emt1RzNMWWdBN1pQMmRvVnQyT09BOXVKWnNzTUpQTzNBQlMxTzdPVnl0UVg1WDdhZHZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTIyNThfMTJfMjQxM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGM2ODQxYmU1YzhlYjI2ZTAxNDFmOTJiNDRlMjAzOGJiOTg0YWU1OWQ0YjI3MGNiZjFjMTE1NzBkMmE1ZTI3OWQyMzE1YTA0Mzk3YzQ4ZmM5MDk1YWVhZGY3ZGJiNzMyMTc5OTliMzc1MmYyYTNmZTQxN2MwYjRkYzA5NDQwZDZhZWIyYmYxMGYwZTMxMThhY2MxZTdjZjRmOTdkMWE4YmUzN2U3MDU1ODYwMGMxNTQ2YzUxMzdiN2VlYTNlNzQ2NWIzMjM3NDBmYzI0MjkxMzdkNDdlN2JhZDhmYTFiOTE0OGE4NWU5YzBiNjU4ZmZhNDA4NzJmODdmNDYyM2EzYjE2YWFmMzQzZTkyN2YwNGJmZTcyZjA5ZGE1YzA1NWMzNTdlYTdlYThlMmI0MzRlZmNlMDUwYTMzZTdhMmEwYjZjYWI1NjVmMzhmZDVkMThmMDJjZmFmMmNiOTFjOWFjZjIwMTQ1MjEwMDc3NWNmNDVkZjJjMTdmYjU5Y2RkODRlYWM0ZTVjYTljZWUxZjZhMzM1ZWFjMWI0MTJhNDdkNWFhYjMwMGIzZWMyY2EyYjA3NjU2MWYzM2M2Y2JmOGM1ZWU2MjE4NjRkMWU5YTYxMDk4ZDNjMjQ2OTlhYTE3ZTMwZmY3MGZlNjY4ZTRlZjgwNzkyOTY5NWZmMjVmYWVhZGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tlBRBtHCe84vo5HddTqkWMIO3IxSjePsZ9hyJbFGLi4k1kZdeSsKYEgMl5-JZYYb-_Qxuk-4A75Agn-Rwajo_Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_112258_12_2413_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.596Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik5KYUNqTnlFcWpwaTVEU2I2a2l1NXhyRzd3eXNXbDQ5ZmM0eWFMekVhb3kyUkhyWXpBUlBMbWxmYlRpcWUwWm1EVGdQaWtVN3lRUEFCa2xVNUNCd0h3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTIyNThfMTJfMjQxM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzBiNzQ1ZTMyYjFhNmUxMDVhMGQyZjE5NzMzYzE5Y2Q3YmZmM2E1Yjg1OGYxZTEwMTg5ZWNjNjU5YWI1M2I2NmY3NjdmOGFkN2FmYzA3ODQzMTczNjBkNmRmYTY5OGM0MjhjMjhiYzdhN2RhNWEwOGVlYjM4MGJjNGUxZWY0MDg5MDAzZWEyZTgyMTg2NjExMjk3ZThiZDkzZTI0MzY4NTRkYWJlMjQ0NmFhYTRkNjdlYmI3YWNlY2Y0MjI1NGM2NDQxODFhOWM1M2Q3YTdkMWY5M2QwM2M1OWZiMWMwODYwNDIxNWYzZmJkZjFkMzlhNTQyYjZjNjBkZDEyMDAzMjVjMTViNjgwZDU0MDEwNDEyMjg2ODZjMDlmM2U0MjE1OTNmNmEzMjJkZWE1NDAwODljZWFmNmFmMGIyMTgzOGMxODM4MGUyZWQ5NWVjMzQ1OTQyYTk1ZWFiZWM0NGFlMDIwOWYzYjU1NjY2MDQ4NjQ1Y2MwOTZjZWY2Y2M1MTRjYTVjN2FmNGQwNDUwYTFhZDg3YmU2ZTA3MmNjNGVlY2QwMmEzMWFhOWI0NzU3Y2UwOTg4MmNhMDZiNzI2MTNmYTY0NTQwMzViNzVhODk4NzNiZmU1MzQ3MTVhNTkzOGZkNmQ0ZTgyYTUyYmIzNWUzNjk5OTBmNGRjNGRhNTE5YWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YgoWMvtFv5kHzTbhzkBEZlk8Xzl_FDvc31VezHqxaGpQ9ymAzSZrYOgA_8sc66who5zEl4T-Tm29Ch6xfdVJrQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_112258_12_2413_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.600Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtGQk5vLytDSWFjelpxbHl2dzlBcXB1V2s5Qm9VN2hSQmZFSk5lbmZMaWNqcWtPRUp1OVh0VjBtUjdJN1d5eHFienQ3ZXNCSmVkV2NqTEpCRlFRYTJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTIyNThfMTJfMjQxM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjIxYWJmZmQzM2M1MzA3Y2FhMTQ3OWQxM2JlNmU1YTQ5ODc2YTc3YWE0YzU2MWIwZWE0NDkzZDgwYTlhNjk5YmIwOTBiODA4NTkxYmE5MDhjZWFjYmY5NWM1ZWRhM2UwMTNhOTEzNjQxYjQ5YWZiMTM5YzFhMjZiMmQ4YTA2MDEzZTE1NzZiMGZhODEyNDZhMDQ3YTQ0ZDU0ZDQ5YTQ5YmZhNTZiZmRiZDI1ZTNlOWMyOGU0MDZjZGZlNDNlNzZlMmY1YjY3MjEyMWQ3NDdlNTBmYzE0MmQ2MDJkNWNkNDhkMWQyMzM0NWI1YzExZTVmOWQ0NjI2YzRlYmJkN2VjMWI0MGQzMmE0Y2MxMTViYzBmZTE0NGNiMWVlODZiYzBlZDIwNmQxYWU1NjQyOThjMzJkZDA1ODA1Yjc3OTRjOGRlZTY4MTBkYWQ1OWFiMDIxMjhkZjA1MjQxZjBmOWY1ZmEyNGQ1MTIxNmYyYmE5ZjMxN2U4ODU0YmFmMDA1ZDgyZDI2MGM5NGRmOTQ4NDVlZGI2YTEwMDBlZDM3MWRkM2Q0M2JjZGU1Njc4MDQzZjZkYTM4NjZkMzZjZjFmYmUxY2RjNjY0ZjI5MTExZTAyOTMxMmM1MThiMWIxNzhkNTAxNmY5ZjQyMTY3ZmM4NjUzM2MzMDNmN2RkZmQ3ZDA5YmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jyR1DppvVlW5OWWmqlmfeeDdSCH4T97w0nY4RPEnJkUIsdnXmlIa46BH9_wh6X8WG3ZrUOwm0pefF4Z64OEvLA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_112258_12_2413_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.605Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJEcmsvQVIwVHNGNW9Vb3B6WktXaHpIWUVUYWFhMlZ6NmZTYkJoeHBENy9NS3pLSy9iSGM4cmp4SG1JMzRCVWhNb3M1VmRIbU5mWUZrZDJBQUxiaTRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyMF8xMTIyNThfMTJfMjQxM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzFlMmNlZjVmMTAyZDUwN2NlZWExNDNlOWY1YWYxZTU1NjFhZjdjNmNlZmU2NzZiZWI4YmQ3MTczMmNhZDliNWQ4NzJiZDA4MTFmN2JiMWFmNTBhNjZlNjEyMzYyYjdlZTcyZmU1Y2E0MGEzOTY4ZDU2MGNjMGMwYTgwYmI4MmFhYzQ0NjkxOWEzYTMwMmZkZjQ4ODA0NGM2NTQ4OTMyYzg5MDljMzNiMDVjNzU2YWYxMmE2OTJlNjAxMTg2MjkyYzM4NmIwNWI5ZGU3ZjU4MDgwN2VhYzFkZDNlZjhiZGNjMDcyZmYxN2NjM2JlZWRmNDdlOTI1ZTkyZmU5YTZhNDcyZjFjMzViZGY3NTVhYmUyN2E1M2E5NzQ4NTg3YTY1OGFhZDMxOWJkZTgwNWUzZTc0NGViY2EyMWE3OWYwYjBkY2Y3MDEzMWFhZDc3MzM0YjVkMjBiNmM1NzBlMDRlNWRkMGY2NTE0ZjM3NThhZjY3YmMxOTlkODJjZTdkNzg2MGE3YjZlNzUyN2EyNzgwZDE0Mjk5MTI3ZmNhM2M1YjBhMTE2OGNjNzU0MTJmNTE1NzdlY2NiZWZlOWFjNTBkZTg5Mzg2OTZiY2ZhZjg2ZGFlNTJiOTA1NGRjOGIxNjg2Nzc2ZTUzMzg1ZmQ1YWNiOTQzMGMzMjg2ODI0MTY5NWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.o69jquh0UUXPj_esW3spCe4bp23GbNPeRRBsZuDCn9OGGEi2seVzxwc5AGcg3_LRwQK3v6gu2kCTlySJpTFMPw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220120_112258_12_2413_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.609Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlXVEZTS3p0ZWQ5RlBGdEpIRm4zUS9FVjZ2UzEzSzJJQ1hZS1Fwa1IveUwxK0NzZ0NBZFBmWlowR25qT3hYVU1ZN3Rrc0dOWGlkTkJROXhFVW9zNGZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMTA0MzBfOTJfMjQ0Nl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM2M2M5MmIwODYyYmQ3YjNjYjhlYTQ3MWM4ZjZjZjAxNjUyZmI1MzdlNTI2YjQ1MWU3ZDExMDczNGMyMjM3NGZlY2M3ZGZhNDM4M2QyYzYzOWFhYmU2MTA0Y2JkY2E1OWFiYWQyZTY2Y2JhYjg1YWJiZWFjZDZiNjkzN2VlOTcxZjlmYTMyOTIwMGFkNzFiZDViOTQ2YzRlZWJmMTYyOTk1MGJkNzFkZTljNzkyNDQ4YWM4Mjc1YTc2MTU1YjIyY2E0ZTdkZTU3ZTBmYWI2ZTEyZGNjYmIzZmIzMDBkZGM4NzQ3NjdlNjYwZTk0OWRhZTc4NGFiZmI0OGI0ODIzMjhjNGYxZWQwYmE4ZmE0OGU1NjUyOTQ4NzM4YWE0ODhmMjMwZTljOGFmZDMwMmJiNGQ4NTIzODdjZDJkNjBkODFkYjMyNGJjYjBmNDViNTNhZDM4N2I4OTRkMDI3MGFkMTk0YTY0ZWMxMjk5NDkyNzVmOWIyZjllYjI0N2E4ZjkyZjVhMjlkZjhiNTQyMjU4MDEzNDkwMzRiZmIwOWUyZmZhZThkNTkyOTFmZDYzNTUzMDA1NDI4ZDVjNjkzYzhjYTQ1NjMzMmI1OWJlYWJiM2UzMTgzNWIyOGU0ODQ0ODgzNDE1NDU3NjFhMDlmZjg2ZTU4YWQ4YjRhNTI1OGRiNzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qYKBLkekUPluELNo0l6RlALABl3gNGcswCH51BmXLdsNB8p3EfWo48Kdlb3kMS17Jsxnsg4U4-ucgMx-Zh_51Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_110430_92_2446_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.613Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik1SOGJiVDd4b0tnUkNObGxLbXVxbW9PenhvT25jNHRVaC9sRi9wZHZsQ2R4RWcrKy8ydmwvM3BWS3Y0a3AzQ3ZCcGx5OTVxSnVpanI4Q3pVSHZwUk5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMTA0MzBfOTJfMjQ0Nl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTVjZmMxOTAyYmQ0ODQ2YzU3ZTZkNDIyODEwMWMyMGM5MzNhODdhYWIzYjI3ZDc1ZmUyNjI3ZjFkYzUyZDBjYzIxMTlmZjM2MzczYjcwMDU1MDdkNTE4NTEyZWRiZDc1ZDY2MmJkMWNkZTFlZDFjZDdmNjFmZmZiYWMwYzA4ODNkZjVkN2QwY2UzMGFkNDQyYmYyMDJlZmIwMzdhYzFiMjcyZmI2MDFjOGU4MTIyMDk4Y2VkNjRmYmNjYmQ4ZjlkOTc1M2M2YzdiM2RiNjBmNWE2NDBjNDIxMmRiNGI3ZGIyNzAzNjllYmNmOGFlN2Y0MDBiMzE1NmIxYmMyZjM2YzVkNDVjZjdhMmViNGNiYjcxNThkYTQ1M2FhYjMwZjA3NDE4NDUyZDcwNDcyMGY4YThjNDhkZjljMDU0NDE2YzA2YWY5ZjFjMjM0ZWQ5YTgzOWFkNDI2YjNhMjkyMTU1NzEyZDAxZDc3NmJhMzg1ZWIxNDJjYzFiZmQ5MjY5OWNjNTQ5ZDY2MWY1MGRmMTRhZTAwN2Q3ZWUwYmIyNDFmY2NhNzBlMWE4MmRlMjQ4ZWUzNjI1Y2U4NWUyMmFhN2U0YWU5NTMyYzgzZWY2MGEyZDFmZGE2NjkwMTUwY2EwODBmOTJjNTY3ZjkxYzYyZjc5NWY2ZTEzOGJmMGQwY2RkMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KvXq6LFf_p4f3xt987InHwnTZL4c2DBTsHL_bkA836lroD1eBrQmKYNkRnCRfo3TXZ7LJtUJ6tC_WJ6wNPdoTQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_110430_92_2446_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.616Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InRBL292eCswZThEd2R3MUxlOXhYTG9lYWJ0bVpOY1pLa2EvMlAzSG96KzVWQU5kV0dwOFZSNjQzbnVEYXpOUUt2dzZWeW1RWE1NeUpuMUQ3eFM5NXRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMTA0MzBfOTJfMjQ0Nl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzUwYmI0OWI4YTZmMTFkYzBmOThiZTVkMWZkMWY3NjYwNTAzNzQzM2YyNzFjMTE5MjAzNzEwZGM4YjFhYzAyNGFjMWYzM2NlYmMxOGQwOTE0NWUyODU1MjlmMWVlZjBmNmZiZWY2ZGQ0YTc3OTJkMzVkNmE5ZGIxYjkwYTYxMmNhZTMzZGQ1Mjk1ZDEwNGU1YTEwYzcxYzNkNTU2YmRlZGFjOWQ4MjA2MTMyMWI1ZTM2NWQyMjJjNzc0MzBjYjQ2NGZjMGIyY2VjNGRmZWFjZDNlZTBlY2I3YmM4YjJlZDA2NWEwMWZkODBhMTk3MDI0NDRlMWQyYjAzNzQyNGY4OGIwYTA1NzI3NWRjNjIyYTY0MTgyNjQ3NTFiY2U4ZTViNzA4ZGU2M2Q1ZjFhNjU2N2FlYTIyZjFiNTk2ODBiMWJmNTRlYjI0NjUzMDJhZTUxYTFhMTA3NzVlYmM0YmM0MTEzMTk5NjU5MjhjMTk3YmRiODhhY2NkZThkOGRiMmViN2I0N2RlYmNkOTM0MzRmZWU5ZWMzOTdjMWZlNmJmNjc3ZDgzYmI3Y2VmNWNkODgyZTAzZTg0YzFhY2ZmYTY3MDNiOTNhYjIwZDllMmRjZjI3ZGYzMGE0ODA4NzE2MjQzZTkxNzJiYWE5N2E4OWYwYWMyN2IwNzgwZDRiN2JkMzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sImdTH1qcEeQ5qHDYDTp0PvpcBpehmrH98RKtURuWFXRusws_Cj9A4bpYjH7OxsRtrPHHFPkYSeYVZuZhIkpCw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_110430_92_2446_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.619Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBjMk9XQTFTY3ViQnBSaGhRVHB5TDJ0R2htazkreGFqd09oYm9SWmFhcGt6MzZYN21EVnc2RmFINlZ3RXhRRUg5MHRYSzhJeFVWcTNFK0ZxZjBCc0Z3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMTA0MzBfOTJfMjQ0Nl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmNiNzhhMTYzZjc5N2RkNWJkZmM0NDM2ZWM2NzdjNWM0ZjhlMjI3ZWM5ZDQ4NjgxMjM3M2ZjYTYzMWFlZDc4NGUzZTM0YWMwZmUxZDhmYTliMWNiMmExNDIwMzUyMjdlMDNhZTk4ZjI4NzZhZGI5NWI1NmFlMjg4NGVmN2Y2NzhkZTBmNmZiNzg2MjZlOGE2Y2M3NTgxOGMwM2UxYjQ3ZjU2OTg3MzNkMDQxMjU0MTdlNjM2YzY5MTRiZWVhNzE2ODNmMjA2N2VjNGM2ZjcyMmQ5YjEwYzllZmUzNDZlMjBjNzhlOGNlY2Q2YjRhOTVkYmVlNTc3ZDMwNDgyNGNhNTViNzhmOTg3YWVhMjdjODMyMjBjOTEwNDVhMjkzMTMwZThjMjdkYTAzY2VjMmVjYTFhMzk5OTNiZGJlOTI3MDg4NzI0MDJhYjViM2FhYTQwOGZkYjA2YzQ3OGIwMDExMDQ1ODIwYzM4MzYzM2FiZDdlYzdmYzM0MWE1NTNkNmU2MzQ3NGMzNTQxN2Y0MzYyYWU5YmU4NGE2MTVjZmQ3ZWJiOGVlZDdlZDliZDA2Y2FhNzI1ODUwM2QzYThlZjZkOTk0NTkyYzUyNWQwOWJhYjdiM2E3NjRkYjRhZTE2MmUxYmRiYTRhNTQ0NTkyZGMzYTM2M2Y5ZWUxOWFlYWVmNTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FzauXAw9OrBja8aoZD_hrF6rOfZ-AShd9G2cTCGL4E97sLEps9yI3zY8g7zuk2UwHICqvTbXxwwYc8nY0YB00Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_110430_92_2446_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.623Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkcrajRXQVRVNnNveWlVSVl0TlQvT2ZxVVBtRWpuby80clRUWTBDOVdDOWU4YStXZHZSdkRuemFoSXNqM1ZsQ3luakQ3WFNlODJaZlRsWHgxOTRiWUNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDExMl8xMTI0NDFfNzBfMjQxNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTk5MjdhZjM0ZTA1M2IxNzllN2Q3MmYyYzhiNGRiYWMzYmZlNzljMDhhODY5ZjA1ODhiODViMzkxNjhkZDFjYmZmNDQ4NjdmYTJkZWFhZjEwYTc2ZjEyMDE5NDcxYzczZjcyYzY0ZGVhYjRmYzMzM2U2NjE3Nzg3NGZhMGNlZDg1ZTNlYjJlMGMwYmExZjExZTc3NWE2NzMzOTAxODQzMzA0ZDE5NGRkNmU1ZGIyMDhmYzZjNTMxMGI2OGFiNzhlNDhlNGE3NzI5NDAzNzExZjVmZWRlZDBiZjU3YjE1ODcyNmVlMDc3Njk5NDJmOTgyZTQ1NTYxNDY5ZDYzYjU2NjY2NmNjMTdhNGRjZjhjNGE3YTgzZjE2MTMxMTFjNjY1N2JiMmFiNjEzNmMyNWY0ZmJlMjljZDMzN2Y3OGZjNTZhY2JiMDQ1MGM2YzNkYWE3OTQwNjE5OTUyMzFhZGY4MDA1MTg2MWViZDQxMGM0MzdlYTZlZjJhOWE1N2VhZTE2NWU2MmI3OTU2ZjI0MDE3MzA4OGUwMzhlY2QxYWMxNjU5ZjNiMTQwZGI1YzllNjQ3MjRkYzZmOGQzOTI0ZmIyNjNjZTMxN2UzOTVkMWU4MGZkMzc2ZTQ1MTQzNGViZjcwYTM3NWFiOGNhNjYzZDM3MTI1NjBlOGZkMDYzMDZmZTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cUMItoVzNCmuVLtjrhwFa_LVCRyF9XfktG128bOvstRnG4YECMaQthnR9O7WzlG3ahJ2OQoS2NkIpXoVxS24hg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210112_112441_70_2414_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.626Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik53empwTDdTNFhSQWk1cENhK29rQWlHSjZ5VVoxUmt1WVJBMW00bTd2VWp2eUtkMHdweTFlWkFDbDA1WHRMeVVUcGc4Wm8zYzJ2OCt3WS9oOU5oVzh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDExMl8xMTI0NDFfNzBfMjQxNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWJhMWYyZjcyZDUwNDY2ZGM4M2I3ZDMzZDc2ZjhkMjQ2NThmYjEyY2M0NzVlNTcxYTUwMTBmNjcxYjUyYTIzMGY3YmExYjdiMTVlMWQ3NjQ1YWU2ZGUwNTMzNTNkZjYzNjBhMDY2MmEzMzQ1OWI1YTRiNDE4NGQ3ZTYwYTlhZjc1MzlmMzBkMGQ3MDMxN2NlZDMzMzBjYTUwMzI3ZjYzYzM2NGUyNGViYjhjZjJkMTkwMzc5YWQwZWUzOTA1ZDUwNTg2ZTMyMjY3Y2UzMWI0NDdjMzllODgxODcyOTNkOGM0ODk4OGVmOWU0ZDdlOTk5MDE2MTQwMzEyZTA4YTZhOGI2YTFlZGYxZGZmYmUyNDA5OTE1YjRmNzUyMGRkYTAxNGI1MmY2YmJiYTlhMWJkN2Q3ZmExMTgyMDBlODNjNjljNWU1NTYxNDRiNWE0M2MxMjBkMTk5NTVjNTA0YjY5NDg2MWI5YzJjMGIwZmM0OGI5ZjcyZWRkNDJlN2I3MjE1NzlkN2I5YTY2MGFjYjc3ODhmOWI1NGEwY2VmZTY4ZTQyMTcyMjVjMTU2ZDZjM2NjZmI5OGYxODE1YzdkMDVhZDBhZTE4OTZhYjRlYjY5NDVhYTljZDM2NWQxNzU2N2Q0NzgyN2UzZGUzM2U5YzU5NWY1ODAwMGYyNjZhMTE1ZDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ksYta5n1qZFDAtQuW3MirlwiZ2JfMLqZ0T10ZhKpnDnp6RlvcRmG1KNX9imUFVGFmtl3nGFqLZ2Whp7H8ipItQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210112_112441_70_2414_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.629Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpzSlA0MTA4aEw0dS9zSngwaS9WbnF3eDVxeGNXUnBDajJTTlg3SXk4SnF2NTZqVUpQUFRyanhxQ1ozeStjT3QxajZqNFZPaGVKcGVZMm9XSyt2TE5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDExMl8xMTI0NDFfNzBfMjQxNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njg3OWJkYzdhMjAzNjg4Njc2MjAxMDljZDhiNWQ4ODEwMDEzNzFhMTY0ZjI1MDFmNjYwMDBmMzk3Mjg0ZjUxOTgyZTUxNjA4ZGFhNjNhYmE5ZGNiNDZlZjE4OTJjNmJmMWJkYTY2NjM3YTkzYTg0ZWY3ODI2NzM5YzI4OTVmZjEyZTk1OWExZmU2YTdjNzIxNDI4NjY0ODAzNGZmODY1Y2YxZDU0ZmQ2NGIzNDQ3NDQzZmNiNGI5NzM2YmUzMmVlNzUxYTFmYTNmNGJhZmJhODZmYTg1ZjMxMzQ5ZGFjNTlmZmUxYzQyMjE2ZjQ4ZmZhYjBjMjhhYzA3OTNhNTdiN2E2MDM5Y2I0NTNhZDJkNjA4YTdmOTA1YjE0YmFiMGU3YjhiOTFjMWIwOTE1NDRiZmI5ODlhODY2ODg0NmJhYTMxYzFjYzJkYzVmMjkyY2JhZmM1OTExOTdhZjFkYmI4N2Q4NjVkOTgyOWVlNmE5ZDc3NzA5MzQzOWU3OWE1ODBlMTU4OWMyNGM2MjcyMmM1ZWE4OWIwZjhiOWVjYjg3ZWNmOTAxYTJkMDM4ZjQwYzZlYmYzMTA3NTlhMzM0YWJlN2U4YTAxODJhYWFmYzc2Zjg5YTA0ZWNjYmM3YjllMzkwMGQ0ZjAxM2MzYTVmM2Y3MjExOTI5NTM3MjRkZGQzZDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2Dz_Ha8rPqVVP3oICKHih8yMi2cFhd1VCJbhxSQhVHYYD2zLu-xN3MGfJVuZzGLqSmoCCG4U--km-7lVSALDIA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210112_112441_70_2414_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.631Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlViWmxrV2lkaFpDcU03Z2VzaGdJVGNjTXRDSDAvNmoyTFVaa1ViUlZNVC9UdlF6R1oxd2ZpUDB3KzlFOExhQk1FSFhRNlY4ZDJtQm02SXRHR3dwa1BnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDExMl8xMTI0NDFfNzBfMjQxNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2FhNTU3MjExMDkwMTEwNDA2YmE2OTkxNWVlYjNjM2M3N2FlOTFiMzYyNzE3MmFlM2RlMjA2YzI3MTk0N2ZiMWNjODkyMDc3Yzc4MjQxNDdlNzYwNTJhMzY0ZGFmMWQ4MjljMWZhZDljMjM3YTcwZDNlMzEzMDg2Njc0YWJiOTNmYzMzNjg2MDA5N2ZiNjNjNjJjNmE4MGRjNGYyOGY1ZjIyZWRiZTk0YjJmNjA4OWE3ZTQ3ZGI4N2ZmZWM5ZDk5NmExMmRlMTM5ODEzMmVjZmJhY2MxYWUxOTBhZTFlMmM0YWVjOWE4MTcwMzY4NzZlYTdmZTdkMjA0YmU3MTRkOGNlODNhZTM5ZTc1NDEzYTI1NDhiMDZkZGFmZTI2NWY5OWI1MGE5MjQzZTlmMmJlODA3MTI4NTNlMzc4ZTkxNzUyZWI4YzcwZGE4NGU2NDIyYzAyNGE3NzBmNTMzNTFkZjhiY2U1ZDUzYjFlOGNjYWJjNjgzYTRkY2RkMDdiZmQ4ODkxYTU1MzNiMTIwYTk0OGQ0ZWI2NGIzNzhhMWI1MWU5MTMyNTMyNGI5Zjc3YWNiYjdlZWY0MTkzYzk4OTBhOTZlZTg3MjdkZjQyMTBiMmMzNDZkOGZmZDQ2MTRlNjUyMzEyMjM0NGYzNjFmYzE2ZjVlZWRkZTBmZjkwYzI3MjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IYu8uWy6I8QaCniOX0yAswt3CVCwu5quuqAjHtRKvMveb6OA23ZmtF0wVoboxwz4TU9QDmz3Qv9Ve8xj8wV-gg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210112_112441_70_2414_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.636Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IldQQ0xOSFlEbmVFUDl3REx2TUpkZ1J1eUlLYWlJZVkyd2JlMFh3MFFUQkdIZWFiSWxqSG10b2R1ZjU1c2FXTlp3TlRLMEp4bThIbnVvN2hiMUZlR2hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjlfNjRfMjQ0NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODVlNGM1ZDVjMTZjMDhmOWU2ZTdlZmY4MzQ1OWQzMDQ3M2UzZDY2YTJlODgyZjU2MGI5OTRjZGNmNzgwYjNlYjgyMzFmYWM0OTgxZjhiZDViZmQ0YmZmNzNkOGIxOWI1NWU2YzE0YjIwZDIzMTA0ZWU0NGI1YTFkYjNiYWRlMWU4M2U2NzMyMDBiNmIzMjM2OGQxNWUxMGE2OGI5ODk2N2FlY2FiYjFhODRkMWFjNDhjOTJhMzYzYTc3NmY5MGUyMzIzMzU4NWRmNjM0YTZkZmEyMTAwODRiNWUxMjY1M2Q0NjI1YzQyOWRkZGJhYjhlODAzOTRjNmNhMGQ1MDgxNzY1N2FlNmJlNDA2MmIwYTRjYjIxOTllYWE4OTUxNmUxNmZjNTQ4NDljNmZiNmZkN2M1YzQ0OWZmNjQ3MDY4ZDBlYTMzNzIxYTA3YzE1OTFkZDQzYjI5ZDE2ZGNiYTc0YzIyMjQ2OTdkNjFjMGRmYzljZWNkNDFlYWMzMjBmNmZhY2UyODY5Yjg3Y2YxNGI5NmRjZTEzZjQxODZjYzNiNjAzMjNhMjMzYzdlOTFhYmVmZjE5ZTY2NDJiNGJjOTg2NDY5MWZiNWQwZWRiMTc2ZGIyN2Y0ZGQ0MmMzYzdmZjA0NDQ1YTc5NTM0MGEwODhlNDRiNDk0NTAzNjZjZDBlMjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5KM3NHPPcdrtqoAi0TUtFERLmN3Tb19PvKEPC5BHJsVzhzA2sUHIpFt3yexEvaIrjIz64XlOzt4AUNgQZwtKhg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101429_64_2445_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.643Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBuVG5WSWpaN2lwdy8yTWZRTkZaRS9YWnNrK3FWL2ZLTUVXcGFFQTFCK29wUlAzczduUnE1UU0vQTd5RmVDKzNoUXNMVzk5dzFxbUM0R1RuZ0ljRitnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjlfNjRfMjQ0NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTU5OGU4MzY3MjE3ZGJlZjE5NTcxZmEzMTZlMWRhMTcyZmI2NDc1MTFkNThhYTdjNWI2MzY4ODEwYjQwNDYwMGIwNjYyYmUwZWExOWMxNzNhOTZhZTk4MTI4NDdjYzFlM2QxNGZhNGRiMTVhODFkZDdlNmUzMWM4NGUxOWIyN2I3OWVjNTk4NjAxZDk0NjMxOWY1MDYyZjllZTQyZWFmMThhZDIzYTc2NDAzMDZkMjQ5MDEwNjRkNzg2MzkyNWI0NWEwMWQ4M2NiZDM0Mjg2NGI0ZmVlNmY1YjgwY2Q3YWEwMGVmMWFiN2M5Y2Y5YzY0NGEyZGI5Y2E0YmY2YWE1NGUwZjZiNWY3Nzk5NGM0NmJkYzMwZTFhNThlM2RlYWIxYjM2OTQyZWM3NDQ0NWE4YzQ2NjIwYjE4YjI2Yzk4MzQzNjhmZTZiZWJhNzkyOTA4ZjlhMWMzZjA5OTBlNjg4ZWE2ZDRmODM0YzU1ODNlMzkzMTMxOWIwM2FlY2QxNTExZWQ0NGFhYWYyYWU4ZjMzYzVmZDI1MTk3YzEyZWU4N2Y1NDgwMzc3YmQ5OGU3YjA2ZGZlNmY5NmU2OTc1MWY4YjYzMmMyOWE2MDMxMWU0NjNkNmI0ZWI2NjMxNjI5ZmViMmNhZTE3MTM1OTRjZDRlN2Y2YTk5MzQxODUzZmI0MTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NJCNbB5SrcHsBdhsWPHBJjK6RBP6ZhUYVKS9-mgKayM_Lx-NrQFm02URn2pdkqE6YLnlQ0OWukhe6lPVNV9U0w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101429_64_2445_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.646Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjdCYmwzRGhBZDFxdnBNUFJ5ZXArREU2S2dIeENoNlY5a0JYMUJIOXhJV1hmVmtLOTJTSkpHUXY0RUF3eWhwdG1TZnVjdVc2RkpzN1NWUGlQOTl2c3hRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjlfNjRfMjQ0NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTNhOTAyNDY0ODFjYmUwYjViNDJlNTJlOTkzOTYwZjc0YTExMDY0OGM1ZWYxYjhjYzhkNWViYjIzYjU3YWM0MWQ0Nzc4MzkxYTlhMjJhZjY0OWFhMTEyMWQzYzdhMWE3N2UzMTI1YmRjZWYwNjMwMzM2YzNiODQ3OWFhYjVmZDE4YzM1NTgwMDkxMzM3Njk2NDJhN2JlM2Q5NzQ3ZDFjYmY0ZTMxZTYyNTgxMzVjMTJlNzVmZTQ5MTM2NDE0NjZkODMxZWU5M2I3YjU5OWNlMzlhZDM2MzVjMDYyOGM5Yzk0MzcxMDNmNmVmODRhYWRlZDllYjNiZTk2ZTYwMWQxNzliOWVjMTgxOTZiMzFkMWFlNDQ1Y2U3MWY4MzAxODU0NTQ0ZjZiODY5ZWIwMDQ5Njg3NWFjMzg2ZDBhMjgzYjM5MTRjMTZhZjc3MDk1ZTUyYmFiYjU4NzIxZmE3NzMwZDNjNjU4NTY3NWRmODQ1NjhlMzZlZWZhOWI5NTAyMzc2NjU4NjNiNGM0NDI5ZDAyNDFhMDQ2MjZjZWE5ZWM3NzBiNWRmNTIyNWQxZTNhMjM5N2EzYzBkOWFmMWM3MWQyZjRjMzRlMjJjNDRkMzliMjY5ZTIyOTgwNDIzNDFhM2VjOWJjOWE5Yzg3NTdlMWM2YWNlNWRkNjE2YTY3ODJjY2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6rbcqOXhrqoNGM-Gg0fFiz8Esip7ouqiiStkfchnYfxwk4A77xvequ7Z1PXeLirfXsYtFPd8eTiApFm2lMGvZQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101429_64_2445_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.649Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IitnZExzblZ3b2ZWUlQ1TVVUdWF3RUw4T1lrcXRBczVBY2JVQWdnQVRaMU9OdVVWc1dPYkQ2YVMxZVo2eHB6L3c3UGpYM0pDZjR2aWZ2UUx2VUZOZ3R3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjlfNjRfMjQ0NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWIwODdmNDQyYzkyY2U5MzUwMzBjNWJmY2VlY2E5NTA3M2M3M2I5MzE0OTFlZmJlNTFmYTdiZGU4MDY3ZmJlMmVmYjExYjE3OGRlODY1OTBiYTRhZWUxMTQ5ZjQ5N2FhZTM5NmQ1ZTk4MzI1MWM1NGE2Zjk1OTE2YThiMzFkMzMwNzNhM2YwM2E0NTZlYmI2NDhhMzliM2JkN2Y4MzljNWRiNDE0NjMwYTViODc2N2I5MGNmZTM4MjM3ZDc0MjNhYWMwZWMxNWQ2YmVhOWUwYmI2NzMzOGU1YWYyMjJjYzRhZTExODRkYjM2N2Q2NjY5MTYxNWQxZTFlNjRmYWU1N2E3OTU3OWU4MWE3ZDUwODgyZjk2Yjc1NTU0OGVjMjM5YTM4OTRhMDFkYWE4MDI4OGJiYmE5NTU0MWM4NzU3NzE5ZGZhNjVhZmExYjZmYTU4NmI4MzM1MTRhMGZlZjJlYmUzYTRmZWQ3YmFhZDNhNjE3NjBjY2QyOTBkMTE1N2M0YzZlYWQ0OGU5MTczMjFjMjNlYmNlNjIyNjgwODA5YTBmYWM3ZmFiN2IwYjI2OTRmMWUwMmM5NmUxMjQxOWYyMWRlYjkwZTYxYThmNDZlZGU5NWM5ZGM5ZDBjMDYxMDdiN2FiZTdiZjNkZTZjOWM2M2Y4NmMwNDU4ODJmN2RhMjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6GMRhULtXtEksJRKx7j9bK5dLR74FMjIXcUrIwYmjvchOriNrcdWwAxJs2uY7DZs20WGeFbKwciQxbD7ZTjG7A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101429_64_2445_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.652Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkExQ1JraElqWUJVUXdXbHc2eHcyQ1V6UXZpdm1pYmNxMGcyTjY3V0Ewd1JSZTFwanZlZVNubkVCdzdVT25pT1lpeEtGMkgxUUR2Ulk2cm00cFd3SXhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzVfNTlfMjIyZl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjJkNzNlNGRkYTgzYTA1MmQ1YjJmMjIxOWM2MGJjOWQ3NjcwMzkwNmE1ZTIyYmE5YzQ4MmExNTFmYmM3MGIxZDQ2ZmI1ZDJmOWU4NmMwNzhmZTBjMjdjZWE2ZGY0ZjI1NjY1MGRjZmFhMjM5Y2QzYWQ1ZWE0NWMxNDAyODFmYTMxYjY1ZjRiNWE5M2IzODY3YTFlNGM3MjE1N2Y4MDc0N2MxNWIwZDM2YjI4ZGRhNDU3OTYyZTlhZjg3OGY0NzgwNDY0NzM2YWI3YmIyODMwNWYzMjE2Y2NjN2VjNmRkNjNjZDMzZjFhOGExODRkMzg0NDhhNmI1MWI5MDA5YjcwODQzZWE0ODhhZDg2Y2MwZDg2ZmNhODQ1NmUyN2VkNzY1YTM2ZGViNDM5YThlOWZlYzA0ZGFjMGRjNzdkNjc2NGZkOGYzNTlkY2EzNzFiOGNmMzcyMWY0NWUyNzIzMWYzMGI1MjgyNWI0MmRlZDU0YjllYTQyODkyZjk1N2NjMzY4YTNlZTYzZTk0ZDZlMWZmOWNiYzU5ZmEwMWQ2N2UyZWYzNjdjMWVmYmZjOTM5YjdmZmVkMDZiMjZjYjhlZGI3YWI2Y2U4MmI3NGVjYjA3MTcwYTcxYzY2YmRkZDhhMzRmZDNiZDVhY2NkYjU3MWVjMTQwYjQxODRjNTVhMWQyZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Lm3HtGHBn6K3ejc1k-cYZj_-XGuoAZdETkcQLJQ8OwQcA8qLfHwi7KWXkD7Pfe44W15sFCimRm5LBhd6OC_QrA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104635_59_222f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.654Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik5EMUxGWUZmYUJmR0NnZVdDOTlIRVVZZ0hoa1BVcDQ0SHpTaWdNdDZJTXh6VXpSTWoxamFBcEdiUWdWV0ZvY1pQOXhuUXRUQlR5am4rWncveUdHU1NBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzVfNTlfMjIyZl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzZmMTE1ODEwNzdmMzFjZTNjOThhNjdiZWUzN2UyZTAwNWRmMTY3ODk0MWFhNzc3ZDlmZDIzZjNhZDkxMGQzZDk2YmQ2YTcxYTlmMjU5NTFkOGVkMzgzMDM0NzZkM2M1NWY4NDM4NzJhMTRlYzAxNTMwZTU5MTY2MTk5YWFiM2YxNTJkYmFlYzYwMTViYzhkMDJiZGZiMzUyNGI5NjQ3M2I2MDBiMDc5ZjVkOWY2Y2E2ZWM2OTExM2Q4YmQyYTc2NjZkYjY3ZmUwYjE1Yjg3OTZkMWQzOWRhMzZlOWE5NmUwNGMyNjdkODYyZmMzNTQwM2NiY2UyNzIwYjhjMTU1ODY1ODE5ZGMxMTAwYTM2MTExZGI0NWJjZjY3NDBiNjdjNTY1N2NkYTczYzVlMzc3YTczZGRlZTE4YzQzNzNlMjI5OWRiMmI3M2E2Y2Y0MDliMGQ5NzcyMDIzNTE1NDJhYTU2ZDNmZTlkOWIwNWU4NjA3OTM0YzA3NjJmMjFmNjkyZTIwZjg2MThkYjJiYWQxZjc2YTFlMzBhYTQyMTJmMWY3M2RlMWE1ZDAzMjNiMzU4OTQxYjA2ODQxNGFmOTdjYTNkNGYxZTQxYWIwMTdmNGVkOGZhMWQ2NTljOTkwMDgyMDc2NmUzNmVkZDY4OTkyNjk1YjdhYTQ0MGFkODhjMTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xt9nLOILVmxWmAVw49XNwF7qe97cgyc_24GQO37elRGKvkEtLFBZ8sZr3k2ERnl-ve6VG3Rsu_F9i-VfK-JE6w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104635_59_222f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.660Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9HTy9Fb1p3aGZyNkM3RnZwZWNPdTExYlRZbWNPcGVHU09RVCtkeExCU3o5aVRES3VmbFNKNTNicFZYQ1hrS3N1VmVqRUd1YitOU2JwVjNxRzBsVmNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzVfNTlfMjIyZl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2IwMDJiNDdlNjZhNjAzOTI2NzQwYWRiNWIwMzdkM2FjZmY2NjUzNjI3YTQ1ZjE0Yzc1OTI0MTMzOGViY2ZlOTlkOGRlYjEzM2Y1ZTVlMzI1NzQwNGU5OGI4Mzc3OThlNzg4OGM2ZGMwZGZhZWE1NDJkZDllZmQzMDUyMmM1MGUwN2IyMWI0MWJjMjRkODBhNDFiNWM1YjRkZmRhYWJiNmU4MmEyNWQzODhmOTIzMWMwNGU2NGU5NGEyYzM4NjMwMGFhODc0YTFmOTgwNWI2YzM0Njk5MDUyYjMyMDRkN2IzODJjNjBkNWQ3ZjdmZTlmZDA1ZTAzY2JiMDM2MWM4MTZlMjU2Mjk3MjVhZmVhMzgwZTMwYjgzYTAxZDg1N2M2M2Y3ZTJhNGYwNTdjYjkzODdmZTk0OWQ2ZDAwN2RkYTE1Njk0MDJhY2E1MWQxOTk3MWUzMzdlNTA2YTA0ZGJlMzNhY2I5ZjBmMmQwZmY5YTc0MzEzYWI5NDk0ZTU0MzVlNmIyNDBhZGE4MWUwMmZiYTM4ZmNmMDlkNTdhMDJiZDE2NjE2YjU2NzMwMmQ4MDI5NGFmNDNlZmQwNDBjMzMyNmY1Mzk3NTRiZGVkODU5YmFhZGU5M2IxMWIwYTgzOWY4NzZmNGY1ZWNhNDgxYTEyZDViZmE2MmNlMjU2NWM4ODJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fjJHBhLFFArQONyVqYjfh0pbLB-fBjQGDLA6jpwXndzKFY1Fz0Gmw_RYmnJ9Y5s5eLEz6VQ8kpwf4zDWVFrO-A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104635_59_222f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.664Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlHM3ZHOEdFNGRMSldUUWNRaGFjbjkrNEZvODRMeHJXUUxpMFM2QUVVaVc4VlJDZDIxTDRzWXptWEhvZ2YvTzd2a3UrVk9iNTJSUVRIa2VDOXpZemZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzVfNTlfMjIyZl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWY5OWRjOWNmZjI5ODkwNzg0MDVmMmI1OTQ1ZGQ2NTNmOTFjMjcyYmIyZDgzNmMwYjZlMTIzNTQyN2Q1NmZmZjA4ZTBlZjc2N2I4YTdmNzAzNDQzNGIyODA2NGRmNmEzNmZiN2YwYzRlZWUzZjRhOTQyNDZkMTFmZTIwNzYwMzJkMjRjNDMyYTY4N2ZkNDI2MWVmZGQxYmJjZmM4MjkzMjVlNGFlY2JiZWFkMDlkMzM1MzE2ZTJiYTRkYzFiMTllNjUxYzJlOGU1YzQ0ZGIwNjY3MTE2NGJjY2E5NzdmNjFmMTZkZjdkODczMjY1OGYzNDNjYzY2ZDdiOWJiYzQyMmJjNmNlMjM4OTNmMzU3MzQ4ZDU0ZGE0YTgzYjQzM2Q4NGQ3ZTZjNzQ5ZjUyMjY3NWM4MDkzNjY1NzE3NmFmOWIzMzdkNmZjMTg0NzZkNWM4YmE3ZTZkNDU5YjkxMTRjYzRhNmRjMDdjN2Y5MjQzODdhMjVjZjFiYmJmMTRjNDZkYmE4ZmU2NTg0ZDlkYWJiMThlODM4N2I5NTk2OWZlZDgxYzJmMDY0NGU3ODYyZjA2MTRlODI3MDMwODFhM2M3NDA4NTZjYjllZjI4Mjk5YzU0YTIzNjIxMjY3Yzk4ODRlYTk1YTMzYjcyZDQ4ZTMzYWE2MzViM2UyNzg3MjE2YzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cHojxAnbqyJ-_-xESiC75j6txWd1LtHnFIgf4D-vSozicOCo2h3RQqfltp3_DknZyDwVFPCmJ9pZldHTSwOLGw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104635_59_222f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.667Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjNScDRJWVZqT0llSVNnQjBMNXpKVlRESjRYZ3RzL0pVZWNQZDd1eURJUDFwZkVsblB1K2kzcWZFLzNtVTVEY2FWVCs0YjV4N2ZjcUNvSGVseWNGQVlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTZfOTZfMjQ0OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2M5YTY0YzAxZDY1YjMyYjdjNDhiMjFlZmMzMGQ1NGE5OTI2ZDVhMzliOWQ1NjkwYmYwODI2MzMxNWI4MDNiNGYzNTg1NzFmNWQwMjA2OTE0NzFiNjk0Y2JlZDFiYjlkZDc2YTcyY2ZmZjk3MzlhNDczOWQ1MGEwY2FkZGE0NGUyZjU2ZWZiMGI2ZDNjYjYzMTk4N2I1NzVlNDcxOWU1MDRmYmJlMzZjYWYxMzNmYjE0MzlmZjQxZDAyMWYwY2UzYTUxN2U5ZTlhZmMyNjMzODg1MGZjZmUwZDM4MGY2NDE1MTBmZGQxZmMwOWI2ZTNiMjU2ZjM0MmFiOWE0NzFiOTQ5N2M3ZmEzZWI2N2NjNmRmMDBhMTYwZTMyNzliZjFiZDQ0ZGExZTk1ZDcwYWVmZGRhZTQ1ZDNjN2VjMjAxMzQyN2MzNmNhMmY5NmU5ZmUxYjYxNTMwM2JkMzRhYzg3YWUxNTkyNDg4OWJhYjMwNWE5NjM1MjJkMDU3OTBiZjJjOWQyYTEwYjg5Y2JjNDk5ZjQ3YjM1MDhkODE5ODAzNTZjZGNjZTk5ZGY5NDQ5MTk5MWMxODA0MGI2MTcwNGU1NzNlNjlkOTNhNDUxZGY2NmI0YjM4ZjJhNzM3OGY1Njk2YmQ4ZWE0MmUwYzQ5N2FmNjVjYjY0ZDI2NTc0NWFiNTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yFZysXHdZ-1Zoehu22D_ukekDznu2VwbeMcogauHwR70PNzC3l438DTCi-fZs6dlJ89FJGpjr-2klEdYdjLBJQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102316_96_2448_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.669Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlFobzB3L1VZTGkzSWRQUDNaeFJXTHBXTXlNNDJKQXNlS2o5d1F2WUxoSlJBSXRBbFJUUmVuSm1sekYzeVpqTHR0N0dpNkVRakpGMEs4cTgxTVF4TkV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTZfOTZfMjQ0OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Nzk1Mjg5YmM1YmViM2ViZGRhMTg4OGY0ZjQwOGYxMmU1NTNlOWViZTFlODM2YzVmMzE0MzNjNDc4OWJhMmE2NTQyMzJlNmEwOTQ2YTQ5MjdmMGVmMmM4ZTZlNjU3ZTIzMjBjOGUzMjBjNjg5NmY5YTIwZDZiOTRiMTUxYjM2MWQwYjZkYjllNTVmMWJjMjExYmMzOTBlOGQ0MzBmNzM4ZTFhZjg0MzM3MzUwOWQ4MGViOWY2YTc3MzA0OGQzNWQ2MmYyNTZjZGQ2MGUxNzRmYjM0NDExM2JlZjk3ZDU3N2VkYWVhZThjZWE3MjJmMjg2NmFjM2FlZTk0ZTc4NTNhMzBjNmI3NDYyNzc5ZGM1MTYzMWQxMWI3YjI5NGE5OWUzNDI1ZmY5Y2JhOWI1NTVmYjllODE0YWNhYTAzOGZlMmE2NDc2NDRmYjQ4NTA3ZjkzMGE1OTU5MDZjMmFmNjdmZjMzYmQ0YjA5NTc0ZTAzMTExNmZjZGU5MzY1NzIyNTEyMmY2NTY2NGM0ZDQzODRkNGRhNjEwNTk5ODMxMTc5NTM5YjU3Nzg1MWFlNjRiY2VjZGNiNGEzMTNjOTQ1NTExZTA2ZGQ0Zjc0ZDQ2Zjc0ZWQxYjExZDM5YjA2MjRiNjAzYjhlYjg0ODBlNjM3NGFiZDUzNTEyZTE4ZmIxZDlhZjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ON5zquKiFJs2DYNzPBDP5xnYNAoA3LV_IJl0xl6kNtB2dG3r2j5SOmitStPdfclK5t6YkE84_vxgSkWVbY8l3g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102316_96_2448_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.672Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBOaml3NFR1R1V0NXBWOXd1dXRtVzJieENLWm56TXBuYUpDOThXYkFuS1Q0eEgyR0tJVTlGZktDcWlFb1dFVWpqWWZHaTdhRlpadysvWk1lcGl1b2xBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTZfOTZfMjQ0OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQzYThmNmRhY2YwMmM0MmU0MTJmZGM4NTA1ZDFhNmQ4NzUzZThjZDk0NDI1MmY1MTExN2VkNjI2NTg1NDNkYjdmMzNhZWVlMWI5Y2EzNGUxM2QyMzg0NWQ1NzNiMTNkMTFkYTUxZWI2YzBiZDNjZDlmODY0YWJkNjZiODQ4ZjYyNWYxOGQyNTBiN2M4NGFkN2I0N2MzMTg0YzJlZjFjYWE0NDJiN2RiNGU5ODNmMzM3NjNkZDg5YjczZDA2OWU2ZWM0MmI4MjMxMzU5ZGUyZGY4Y2EyM2NhNzQ5NzhlNzJlNDIwZjJlMzA4ZDAyNzFiYWY0OGIyYWJiMmQxZjRiNzNjMTk5ZjE0OGFlMTc3NWFlMTYxOTAzOTI0Yjc3OTRlYTJhYzJjODE5YjA1Y2I0ZDAyZDkxZDdhMjU0MDkwZWRmNDA5N2NmOWEyZGIwOWRhNGEyNDg0NzEzYzQ0ODAyNGZjMDllOTA4ZjJiNTVjN2I5NjZlZmUzMjlkYWMwMGM4OTUyZGVmYzcyMjc0MjkxMjJmNGJiYTUxNDZmZjFjYmQ0Njk1N2FkYjkxOWMxYWJkYjgxZDc0ZTZkZTQzNmJkYTUxOGJmMzU4NGE5MGIyZDM3YzJjZmMwY2IyNzNiMThjMTU4MjVmYmZmOTRiZmYzMjk5NDU0MTJlZDQ4NjA3MDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gYhcySh5yb2pVuXV7zrhvJM9zDEjmr9Tqe7Vhc6jC_4pREBt5yWz0aCB-BjiBzLChGr0QHfcAvWsjgHnKelswQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102316_96_2448_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.675Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InhUQkZBa01oWjBZWlV1OXNtY1dNbHdKWWtVQkJ6WHlrMU5zUjQ4SllkRkl2TUZZQzRUZFYxQU9WMXZUbWlJa2RkdVhMWEdkUlhkSmVGbEUxTlMzUWFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNF8xMDIzMTZfOTZfMjQ0OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTAyOGQwMGZhZTFkNzk5MTAxY2JlM2UzMzExZjgwYWI0MDUxYmFlYzk5ZDUzMzk3MDhjODQzNjFjNDk5ZjM5MDBlNDRkOTMwMjRlZDhiMWE4NDM4MWUyZjVmNzhkYzU0NzhiNTdlNGEyNmY5OWE5M2YxY2FiMTFjODM2Nzg5MTgwODI5ZGE0ZWZhODg5NDkxNWQzMmY2YTAxMzY4YzJhN2NjMWMxODg5MWJjYWEyMjk5MWFjN2M5YTkyZjM1Y2FkYzUyMDVlMGUwNDk0YjhkMjA5MjI0YmE3ZmYzY2M2ODQyOWM0ZjVkNDJiMTE5YmY3ZTBlZDFkNjE3ZDk3NmQ3ZTAwZDZjNGM2YmNkOWFmYmQzZmJiMWIwNjgyOTE4ODRiODM0Njc4YTgwMWRlYzk0ODM4NzQ2OWI2YWJlNGYzYjcwOWM2OTI5Y2YzYzhmYjdkYzQ1ODUzZGExNGJhYjJjY2IxZjQ5NGQ5ZTJiNzE3MmJkMzk0YTU5MTQ1ZDQ3NGQ5ZTdkODI2ZDAwOTBkN2M2ZjczMjMzZTlhNjI5ZTBiYTNhNzZiZTk0MzNkMmRiNjlhNWFjOWY0M2ZlMDQwODY4MTIyNzc5MzEzZGRmNjk5NGI4YzJjYTY0NDUwOTg1NzNjZmQ2ZGJmY2U4NDkyYWRiNTY0NzFjZmVlZDQzMDRiNGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xp-jRx1Dq0gZ0PRWUNk4YbUw_B3WqKrh3z6S7K1tk5m5tMo_a8MMMIuYRkc47UsbR2TScWL4INLrw5AmMnGCcg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102316_96_2448_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.682Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkZjODVhMFZ2YWVhRFNxR083NnUzaVd3SklqYTNMWkhJdHRxY09JUEwzK1pGN0pPSUJ3ZHR3OStUNkg5MUFBK0Z1QlhuQ3VFRlppWE41Ryt2OWJDQ3dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDRfNTFfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGY0YTM5YTVlYjQxNmZhOWI1ZjU0MDM4NjQ3MWM5MjMxYjA4MmQxMjRiZDE2NjlhZDA2MjhjYjQ5MGMwNWE0ZGVkN2E0NjI5YmVhOWQ1ZDdmNzkyMzkyNDk5YWU1MGUyZTAyNWM2ZThjYjNmZDlmOTQ5Y2E3OTlhOGViZjU5MGM0NTliNDlhMjgxOTdiYmFjMjU2NDA3MGViYjc1MGEyMmQ5MWQ5NzU5MzBmNDg0ZjA3MTAyOTdhODM3YmFlYmMyOGY5ZDQ4ODVmZTgyNTdkY2QxNWM2NWRiMzg4NjdlZjEwMjViYWQ5NjllMWYxYWM4MWM1YzU4YTZiYTQ3ODhlMzkzZGFjNzBjNmYxZmQ4NDE0ZGU3NjZjOTg5MDNjNmYyYWMyZGQ5MzY0ODIzYjNlZDE4OWRkNGM4YzNlZTg3MDI0MGUwZTgyOWM0ODM3NTZjNjhlZWZjMDAzZTNkNDEyM2Y3MTFlZGZiOTIxYzU0ZDI4MjNhMWZjMjFjNmFjMTUzMmU0NWIxMTRlOTRhZTk4NDRiYjNjMjgxYzUwMDdjMWMxMWM4MWNmNDc2MmMyNjFmMjE3ZjNiNzVlOWYzNGU0Y2E4ZjA5MjBmZWY0OWVhN2E1YzBmMjAwOGY3MGNhYzYxZjU3M2ExNzJlN2JkZDFhYmFlNTZiY2M4MzNkNjI2OGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.o_aG7OM8B1UyobP7RutDJs_a3Bij7gU3xiI8RlW0_kEwqlSUtAkARlo3pHpZ563AlbA0ZMZjoDiw96a0hFi4DQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112004_51_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.686Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlhIN2hJTHh4M1B5R28rSTlOSldUMWxPbUFtU1ZtSkJCMnh6TzFBa1Z0eFN4NHI5aUVZUXpKbC90YXIrT3NWM00zWmlIVmMyUU5JcG1QaHIzMThpMEFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDRfNTFfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWMwYjUxOTY3Njk5MTAwYjIxNGU3ZjhhNjZhZTRlNWIyNGVlNDBjYzIwMTY2NjgwOGY4YjA1YWQ3YWE0OTAzODJhM2U5OWQ1NWEwMjM5ZTdjNGY1ZjAzM2I0MmYwNjZmMzA2ZWQwYjdjM2Y2MDE5MmZkYzg3MzcyZWZjYzY4ZjNmNTk5MWU5NTYwN2RkNzBmNGRjYTZlNDdlZmY0ZmQ0YTRiM2ZkM2U0MjQyOWIzZmRkODE3YWNhMDNmODQ0MjIyZjI4M2QzMGJhMzA0MTYwZWE0YzY2YmRmZTY1NTEwMjQ2NDZjYzEyOTQ5MmFmNWEyOWJmMDVmN2Y1NTRhZjY2MTU5ODE0NTYyZGIwZjhmMWNkNjA2ZDMxMWMzNDNhOTBlODYwZWY3ODczZDRkMjdmOTgxMjI5MzQzMTQ5NjFlMzFiNTdlMDk2OGIyNTVkZTY2ZGIyMzQ3YjgzZGQ0Y2QzZjI5OTQ4MjBmZWIxYTBkNzZmMTE2N2EyOGYyMGE1OTM3N2E0ZTQwYTRlMWFmMGUxMmIyMWNkZjEzZTU2YWM1ZjRjNzZkNWM1YmE5OTM0OTQ4ZGIzYzI0Y2QzYjMzOTFlMGU2NmU3MjkxNzRhNzI2MDMzNDMwMjMxYTcyMTE0ZDY1NzA4MDZjODllOTlmMjc5ZWViNWM1M2I5MDg5ZDEyODBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.70znsf7JLmFEA6MlNvWqbKW0_3xLNZsLldpfQj1VTVdQfJMBknkE64AN-uZhJwsL4oyGARFMzuv7UanObvhHuQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112004_51_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.691Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNSUE9jMXFsenZ6TElQMWJhU08yamVrSjA0cmY0NG1YUFRWVk9rSnMxeFZNcEgzS2wyRTFEWjNVcVJRZ3QxOXlvQXBuclIzSDF1VnR6emNiUnAzK2JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDRfNTFfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE5ODZmZTU4NGFlMzhjNGVkNzA2ZDllODYwN2UyYzA0NDZiYzI5NzcxNWIxOTgzZTc4NjZkMzdhNmVmNjA3NGNjMGQzZDk2NDRlZGM4YmM0MDdjNWJhZTRmOTA4ZTVhZTlkZmE5NTYwYzM4YzAwZWRlMzQ2NDZjOTE2N2Q5YTNjNmU3NzhjYmY1MTJhZWZmOTgyY2E4YjNiYTRjMTMyMjQ1ZmUyZGY5ZmZmMjU0ZDAzMjRmZWM4OTM4YjAyZmExNzBmYmZlNzI0MzQxMzAwMzA5MWYwNjY0MTM2MDEzMzg4N2IwOGZiNGZhNzIxMGY2M2E1OGU5NWRjZDM2YmRjNzVhOWRhODEwOWM0OTIwN2QzZmU5YTgzYTk3Y2JkZWYyMGZlMzYwZWQ2NWRhM2Y1MmFkY2FiY2FiYjJiMGI4ZDA4MjMxNTkzZjZmYTVjNGFmY2I5ZmU4MWEzYTBhOTIwMWZjNjQ3MWMzYmQ5OGUzMDk4NWM4YzY3MzU0M2M3Zjg2OWI3MGJkOWY2Y2MwYWY4MTRhNGZjNmJkYjAyNzNiMTY0YWYwNWM0YmI3ZTY2NzQ4OGFiYTVlODI0YTU2ODhhMDYwZjcwZTk2NjcwMzQzMWRjM2UyMGY5MTQxYTFkNWU3MWEzZmM2NWJjNmUyNzVmZDRmN2JkYTViZmI3NGM3MmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.I3pq1Ob8r3ZiWB9PnYjq9bUr65Z80ZF7KIouvp_oTSOP24zqR_SbdAcbFTx5boMbV2cNJycunL7z5DIPaC4l_Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112004_51_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.696Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkNZbng1eXNLd1ZlS1FGLzN4dTBPSmdzQm4rSTJUR0VlUHN1UUhleEtrcEc2UjRyQjNhMnVzdHVPcElhMWM4ejhmZWNjUi9qZ203c2loM1hYSlVLeUVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDRfNTFfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDkyMzRlYTM5NmUyOGI2MGIzZDEyOTRjZGQ0ZTMzYzA2OTkyMTQwYTEzODVlOTU3ZDczMTBlYWJiMjkwODY3NTJiZmEwOGEyODdkOTM2N2RhMDQyZjk1Mjk1ZGUzNzUyODYzMDI3MTZhMWZhZDQzMTdkNGIzNGFkNjk5OTZlNzk4MzUwM2ZkZmM5NTk4YTBiZDc3NzMwN2UxNjYyMzdiMjRiYmQ0ODQ2MWE2NWRiOTAzNDIzMjU5NmNjZWQ2MDUwNmU5NDJmNDE2NWY2MTZhZGU3NjBmY2Q0OTliYmEyOGQ5NzJlMGJlNTM3NzcxMjM4YzhlMmFiNTU5MTQ4MzQxMGZkNGVkYmMyNzliYTJkMTZiNjhjZWNkOWJiZGQ1M2IxMTE0ZjZlMzEyMWY3ZGI0YzhhN2IyNDMwZTdjZTE5MmY5YTliNmM5MjdiZjY3MmY4NDE2OGNkYTE0NDBmMzdkNmY5N2UzNDZhYzMxNmEzNjBjZDhkN2VlMGYxMzEzODdlODE5MGVlMWM5YmU1NGVjMTI0YzQzMjRkZWNhNzJhNGVmNzUzM2U2ZTM3MjEyZThiMjVjNjBhY2NmM2VlOWY0YjVhZjZhMjFjNDMzOWFiNTIyMjY0N2Q0MTg4ZGI5YmMzNzdlOTFhNmM1MjE1YjA4ZmE0NDBkMWNiZmI3OTY4NGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.efszMyGiNI_FiJetDEYH6T1t-6E3oNuCpyZUdUlnGmaw99Iz3joAGEeQ1RG1bk4-24nCRnANWkqnKu_GGq3V4A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112004_51_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.698Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImIyZDBtV0VrdTZrQ3F3Y0YxZFg3WVNyVXlCQlUzazgrU24xQ1RDNWVhRmhCcXZ0eTlod0EvanNialV6Y3hZcVVHQWtkRGZ1VmVqZFU0aXV1K2ErUW1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTJfOTRfMjQ4Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjA2ZjA1ZGNiNTc3NjVhZjhmOWFmYzUwOTZjMjBkNjVkYTkzMmFmZDM4MjVmYmQ3OTQ1NTM2MTI0MzFiNGMyNjYwNGY4OWNlNmIyOTQ0N2EyNTA3ZGZlMTJhYTFiMTJlMTM0MmM2MGFhY2Q5YWY3ZjAxNDdjNDcwNTJmNTk2MmNmZjk4YTE0NWY2ZjUyNTM4ZDkxYjcxMWQ5MDhmZmQ1NTRhNGUwMzIzMzc0YmZhZTM0MTkzNGVlNTJhOTVlYmExNGE1YzkwN2RiZDBmNWU2YzUxMTdlNDgxZGU4YjgyZjIwMzIyM2E3NjRhMjU0YjI0YjIyODQyYWFiYjdmNGJjZGU3MWI1ZjhhNjI2NjQ5ZTFlY2RhMjk5NjU0ZDRkYjVkMzE0NWNlNGU3Mzc3ZGM3NmZhOWZmNWFiMjc0YzZiNmZjYjJmZDg2ZTJkN2FlMjNiOTExOTRkZjAxNWE4M2ZiNTJiYTI5MjUwYWY5YWU3M2JhZDQ1NTIzYmZmOTc5ZDVjOTdlMTFhNzQzYzliNGI0NThlODViNDU0ZWQwNTBjZDM0YmFlMTE5ZWJkY2QwODcyYjQyMjExNGM2ZGYzZjc4M2FhYzg1YmY5NzZjMDA5YjExMmUyZjMyMmNlYjRlYmM2MmI2YmRmNWJmOGM3OGExMTVhMzE4MGMyMWRjOGUyZDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IzZGNE6_MYOgtsZn6zhVIeWpGWk4Vp_1_7jlI-calRZ_nZMzHjLsAgv4o-3YhjpdWjYACtfhBCF1pYrbh2UG8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111012_94_248c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.701Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im8ydWgxdURBUTFGNWFKT2FHYkNqdnhrUG0wQ3hVN242Wjcxdjd6bStZMEVUSVlKaWp2VW9UU3NLZjJPMFNEN2IwYzdBMmVuaE1LV1B4cHhVT0NSeC9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTJfOTRfMjQ4Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjI5YTMxMGMxMjExODc5NjBlMGNjZmM0MDJmNDRmZjhjNGUwMDA5ZWU2ODBiYzc4YWRjYzIyODEwZWU1ZDdiNzBjMTViMjhjZmUzMjZkZTkzNWE1MGUwZTA3Zjc1M2ExOGNhZTY3ODJjZTliZDI5MWEwNDRmY2NjYWUzZGRjNmI5YzgyOGI1NDkzZWI5MmMxMGI2NmVmNzFmZTk1YjBiY2Y1MzdkM2VlY2U0YTMxNGVkYmQwOTI5YTk0ZjQ1NDUxYTJmZjg4MzY3NjA3ZTBjZGMwNDM4NmU2Yzg2NmQyNTUwOGU5OTQxNzQ5Y2YzMThkNjEwNDllNDUwZWIyNjY0OWRmYTlmYmNiNWZlMTc0NjEwYzljMTA3YjNhYzNjMDExNjM2ZDBjNDhlZmJhOTEyNTM5OGMzMjY0MmU1ODY4MmQ4NmE1ZGQ1ZTRjNWI5ZDI0ZDRkODA1N2QyZjUxYmU2YzE1NjkyMDgzZTQyYmZhZDc3NzI0MWNmMjMwY2M1ZWY5ZTNjODVjYzhkZDYzMGUzNjMzNGE5MTFiN2QwODMxY2QyZmJhNWNlMzZkZTlkYWZiZTE4MDI3YWY3YWVjOTljOWU4N2Q2YTYyMjdlYWU2MzczOWY0ZWZkMmE5YWM0Y2NkOGU1NjA2MzhlYjIyOWM3NDliNWM1NTJkZTVmMTA1OTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vfpr_K1NCjH5VDTRgEnLkKEDlgku-RKTesF_XwPtapgH48cX9sJOfJM-VSswUZKskNVCxo1pB3Pp-6UDFk-gtQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111012_94_248c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.703Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjVYSWtwQ0dweVBQaXFGZXU0RHE1eXVRaUhVUzhRNm1FQ284YXA4Z2JYUFY5T3ZXVm5wa1hZL0JBdTlNZGV4TVEwTFNaQ05sck5FdDl5QzRscEp1Y3RBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTJfOTRfMjQ4Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDNlNTJiZWQ0YTZlNGRhMDc5OWYyMDg1MWUyNTY2MDFmZjM5NWYzZThkNDFiNmRhMzUxYzE5NDQzMDA3NTQ5NzgxYzEwNzVjMjI3N2I4NmVmMTA1NDljNTRlMWMyMTM3MjdlNTY0OTM0MDdkMTNhMTIzN2NlYTFhOTE5Yjk0MGFiZjkxNWE5NWYyNzVhOGNiZmI2ZGIzZWFhNTQ0YTczOGIyODkwOGU1NmNkNzk5NzI5YTE4NzY2OGFkN2NjZTJiODJiYWQzMTdjOWNhM2MzMWE5ZTAyMTFjZmQzOTM2ODdjNDUwNmZiZTk4ZDllZGEyMDY2OTY4NWMwYzIwZTQwMDNkZDg2YTljYjYwMmMyZTQ1ODUxZDdmYjgxMTMwZTkwY2NkN2U3ZDUzMTI2ZTc4MWMzMmJhODVhZDMxOWI1ZTg4ODBiY2M0OTA5ZDQ1N2Q1YjFhOGU1ODYyMjA3OGJkOGY4N2ExOGIzZDU2ODdkMjk1NTEyNGQ5YWIxNzlhYWI4ZTZjMzczNTI1Mjc2YjYyYjJhNjlhZWJiMjZiOTc0ZDVkODNjMTFjNDc5YTY4OWVmNWFkODAyNWZjZDM0MjdkZGY3OGVkYzdiOWFlYjJkYThjNTk0NGMwYjY2OTE3YTA0NjQ0NmZiMTFkY2JhMTg4ZjdhNTQ5MDEwZGJjOTIyMmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mREKx8U804CEyvqgzRfBo6MtBSrG06jrz45kvMRS1RTmMImLPIltgRJVEoKizPZiaF996wBPeX31c4EFmEOQcg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111012_94_248c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.707Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhkdlliMi9kNUdzRnk0dFVzT3I2UFBZajRhMW94WWJ1cGN4TmgyUjQ5eFBIejNKWjl2clhZRi9RTDMza3B5MmMzMkhRTWt5SzhyK2EwOWNOeVVReTJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTJfOTRfMjQ4Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI1N2IyODIyNjA3OTY4ZWUxMjc3ZDRjOWI4NzQxNDI3OGU2MDQzOTI2MjE0ZGMyN2Q2MDYyNzNmNTU2NzM2MmQ1MDE1NmQ1MDFjN2MwNmNhNGIxMmU4N2M1ODE2M2YxNzcwNDc3ODFkZWRkNjIwOGQ5NTVhYmY1NTJiZWEzZGMwOTQwMTg2OWEwMDZiMjI0MWE1MDhiOGMyZGQ5MmNmYzk4Nzk2OGQ0NWU3NWM2OGQ4ODZlYTRmMjBlYjFlN2FhZGFiMDQ2NmU4YzVlYWFhNWRjYmYxYjc0NTk3OTE1YWQzMDdmOTBlYTA0ODk1MjExNTZkN2U5ZjdhNjk4NzAzYjhkZjQ0MDc0YWQ0YTZiNzlhNTA0YTBhNWM4NzVkMGEyMDZmMTM2YTQxZjAwNWY5ODgxMDdlMDA4NWM1ZmRiNjQ2NDBjYTNmZTEwMjhiYThiZThiZWFkZTJmZjNiYWY3ODc3OWI4NWMzNTAwMWM2YzQ2YTIxNTk2MjIwM2FhNDNiY2YyOTI1MDM2ODBiZWZhZDE1NTQ5ZDU3Y2ZjM2ZkMDNmMGYzNGY3NzkyY2Q1NTAzOGFiNjIyZmEwNGU0NWFhOWY4NDYyODU0NTNkZmJiNzA1NDA3NGVlNmNjYmNiMmI3NjI0ZmE3YWQ5YWU0MDM0MjA2NGMyMGIzMTA5Mjk1ZDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5esxJlwBj4KnO9d82D1vP2M6yKILz9zazQ2l4ibsgRpweh2ShgVUBu5fHlG-cokDZPkcZ7i9eDCmCtl8PuMb7Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111012_94_248c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.710Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhSNnh2UjFLMjZENHllb1VRZVU4NStWZng2cXBmenpEZENXcDNVTWRhNm9hNmpBS1dsQUVDa3RRbUM1S2U1bEJQSTF5aGVLdDN4K2RFK2xoRWFnUXFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNV8xMTAwNDdfMTZfMjQ4NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTRjOThlY2E3YjgwN2UzZDAzMjg1NTQ5OWMyNWM3NTA1YmMxMmUxNDY2ODEyNDIxOWQxYWNmODMzNDFlNGZkYTY2YTYyNjQ3ODk3MjZlZDViNGNlYTBjNjBiYmEzZTI2NzU2MzRhZWQ5NjdkNzY4NDM3YTAwYzFiMTRiYmZhYTJhMTNlOWYwODYxMzc5ZTEyMzliMjkxZWUxN2ZmNTM2MGU2MjkwNGZhOWUxN2U4Y2VmNTA5ZTM3NDMyM2M2YzgwYzY3YmRjMjc0MDBkNDU0NzNlOTU0MWZiMzZkNWVhOGVhMmE5YTAxMmY0MGI3NDg1YTEwNDQwZjQ1NmU3NjgzZmI4NmM4OWJjNTA4NDcyYTNkM2MxMDQ4ZjIyNTdmMmIyZTRiZGY0N2MyNDY0ODY5OTM5NTRhMWYxMTcyZTU4MTg2MGVkZTNjMGJkYTllYWFkYmU1M2VhZDkxYzQ2OGZhN2FmNDM3NTVlMDMwYTg4ZTM5YjBjNjk1NjhmMTg4YmU3ZjQwNzlkY2QwZTQzYzMzMTE4Y2FjNmE5OGVmZWVjN2MyNTg1MGU4OWEzMzAzZDkwMjI0MmQzMTZjNjVlYmY4NjljOGRjZDM5OTg4YTk4ZTk4MDgyNzhhMjZlNzJiNmY3NWI5MzM3YjBmMDA2NWFiMzVjOWQ0MjRhZTAzYWE1OTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pSsvtDEOVRQ80Cj_1ZKbD0k17WjLDBpXbQEv3Q_6E6l33q52Rfr9mJVnZAHBtJ5aBYAzBf1togEMU7JvaRyvXg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221205_110047_16_2484_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.713Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhCc214WGNjbEdmT1RZTHdxOHFpMy9oWXV6Vm8rTzlXV2psUk81SmpzdENHSElDbWlnV2hsTDZvS2g3VUFCZkhGRXBKb0dqT0ZQTGtUMDBGT241blhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNV8xMTAwNDdfMTZfMjQ4NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWQxZDc1OTNiZWE0MTE4MDg2MDBhZjhlZTc5NzEwZTRkYmUxOTFiOTcxNzE5YTQ0NDIxNjU0YjY2ODkyNDkyZDBkM2QwNjM0NThhNzBkZmE1MzQzODM2ZmNmMjAxMjMwY2E1NmRlNjRkYjFhZjNkMmJmODY2ODBiZDJiYjJhOWMxNzQ2OTJjZGE4MGFkYmVkNTgwM2ZiMzE0NTQ1YjFlNzhmNzY3OGJjYjZlZTcxY2ZhMDFkOGQxNGJhMjMzMmQ5ZjEzOWFjODJhN2NkZWFmYTUwYjY2NGQ1NTBhYTRkZDU0ODQwZGUwMDk5YWY4ZjEwNTIxMzdjYjYyYjY5NjgwOGVjZjM0MGViNDZkNzRhNTEzZTI5ZGM5N2MxMmQ0NDE5ZjBhNGFkNTBjNGU4OTFjMjc5NTJhMDRiY2E2Yjk3MWY1MjE1ZmZjNjA4YzcxZmZhMTgyZDhjNzQ3NGMwYjgwMmUzM2M3MzEyMjNlMDNmMDZmOGE0ZmE2Yjc5MjAzMmEwY2RiNzBhOTI0MTIyNTEyY2VkNTgzMzRjZTNlZmFiNDQ2N2NhZDQxOWIwMGEwMDBjNGE1MDNlZGQ2OGYzYmQ0OTU1NDk4N2MwOGVkMGU0ZjE2Zjg1MDdhYWY1NDY2ZDZlOWQxMGJjMjQ0ZGVhNjVhZTNmMmQxM2RjYmNlYzkzOTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5W3zc5ibv_kcIStfzUGVrNlSfO10VPRgm1kabvG4yciQHm7p6SrHBd-Ei1UMKKPYNdtpsOXhDuXGAIkpX3TDxQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221205_110047_16_2484_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.716Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNkOFp2UnhnamhtaWlwZFJmbWIzckxUdWpjVFI0aUpHMG9aK1pVemlrbVRYK045NDZJODd0SVJQb2YwcEpVWGJiZkowdGUzQjJGQ3lUNk1HTkxlSG9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNV8xMTAwNDdfMTZfMjQ4NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDA5ZjdkNmYxZWQyYWE2ZWE2YTE3ZThiYjg0NzdmNGE0M2QzNjE2YzdhZTU1OWQwMzQ4NzZmMmUwYzNjYzY0OTYwOGNhNDc5M2IyMzk2ODg4YmY3YWU4YzdhYTgwMTU0Yzk0YTBmYWMwMWM0MWFhMGNmZDAyOWY5Zjg1NDk0NjAwOGIyMTU3MGMxODdiZDhiOTYyOWIwNDRiOWVlNjY5MzNjNTkyZGIyOThjOTc1YzA0ZGI0MTVjOGY4ZjZjM2NjMzIzNTcwNjI2NTFjMTQzYmYyYTg2ZThmYWU3NDE0NGY4NWRmOTA5YTM0OTMyNzFkZDZkNzU5MmYwMTUzOTMwMWJkYjE2NzQ4ZTYxMTAyYTlmNDkzN2FkZDg4NGFjZGY5ZDk2M2I2NjlhZDY4OTQzOTYwN2ZhOTk1YjI2OWYyNjNkZTFhZTkwOWNkNDFiOTI5YTNhZjk1OTQzMWRkMjhlYTc3YWUzZGJiZTlkMzdlZWEwMjg0MjA0ZmQwODE5NmU0YWJiMzMxNDNhYjRmMTU2YmZjNjg1YjU5M2E4NzMyZWU5ZjhhM2EwOGIzMmFlNDUxMTEyMjlkOTVjOWY3NWEwZmYxNmQwNTIyZGY4M2JlZmM0NDVjYWY3NjkxYzI0ZTliZjgwOTAxOTRjOWZhODRjNDBmNWE4NGYzZWNlN2RkMDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wFQ9cM8vtGeq2NMBnlpVhoq1QPnu_LvHWhCVvNcbe1AyrKR3FGV1DtDHgVGNnICxyXi4b0mqPy0q5AO-RkhIeA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221205_110047_16_2484_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.719Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlZTVWpjQXd2bWJ5NFgzOXhoVkJhUE02Ny9aNy9zWVk2aFNiWFJraFdqY3duVlFGTzhCOWhHanR2ZVpaS0Vhc1g1QXpseWo3UHl3c2JlaW5SL0xFZVVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNV8xMTAwNDdfMTZfMjQ4NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmE4ZGU2N2YyNTlkZjY3YWUyOTM0OGE1ZWI0NjA3YzBjYTBmMDBiYWIyN2RlYzdiNDViMjMyMDZhM2JjN2ZiNDMzMzJkMzQ2YTNiMGJmODY3OGQ3MzBkZmRkMmMzNTdhMDI1NzllZDA3NDNkOGIwOGY5ODQ4MWY1NzFiYjU2YmE5NzEyOGFlNjUwOGQ4NTI5NWMwNDUzNTllYzk3MGJkMTM2NDRhNmU0MGU3YzU4OGI2ZmFiYWZlODk1MWNhYTA0ZWEwNTFjZDg5MGY0MGE4M2UxMDkzZDUxNzU1MDQ2MDIwNTQzMGQwNTExZGFiYTEzMTM2MGQ4NTA5MzhkMmU3YzVhMzk1NDkzNTI0MTliOTNhNWZjNGVmMjFjOTk3ZWM2NDQxODFiNWFmODM2MjZkNDQ3OWZkNDVhMGQ5MGM3NTVmYjY0N2ZlYWMyMGI4OTY0YTI5YTMzMDU1ODljMzM2ZWNiZjMxN2FkMzAzMGEzYmJmNzUzOTI1Nzg5NDNkODhlYjVkMmZiODY5NDNkOWEyYTNkMzUxZDY2ZTg0NjA0MzQ0N2VlZWU2NjFmM2M0MjJhMmI5YjYwMTgyMDI0MmFkYTRkNmY2MWE2NDEwOGI1NTRjYjZkODZmZTJiNGRhNTE3YjQ1N2U5NDdkMzFiMmYwYTQyMDY1NmRmZTc2YmM5MGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.C4KJ-B7mTkz5Mjq5AV7SkG6UFNLZAYhdCNxOED34WrtpJfCrOTB-pt9xPd_jNhZgu1m6tdP9GzyjRIOu3b5nuA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221205_110047_16_2484_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.722Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVNK21HTHJOeHRMczlpVE84dGdPdXpQaGxCTnJRengxRW9NdjR6dmsrL3QyamJlOE9XOFMxdFZKN1FIRTdqUms4dzg1TVhwUnM3ZDB4RlpkcXk1QUVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDZfNDFfMjQwM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQ5MWE5NWU2NThmOWI2ZDQ1MTliM2M3MGY0NTUzZDczZDQ1NWYxYTcxY2MzNTQyOGM3N2Q2ZDQyMDcwNTg2OTE1ZDk4NjlkNTEzN2JiMDE2YTFkZjkwYzE2ZmM0MDJhY2ZjMDcxNzBhMWMzNGNhYzMwZjMxYzg0NmU0ZmE3ZjA1MjJkOWI1OWFmNWY4ZTgzNmVhYjU3M2M2NTUxODE1MWI4MzdkMGZmMDY5YmFmNmI5ZjY2MGEwODgxYTg4MDczMDkyYmIxOGRhMjJkMjRmODRjM2NiYjdiYjRhMGE1ZjM5NTBmN2RiN2Y5ZjllNTYwZTYwMGQyODBjYjcxNDkzMTgzYmY4YjZiMTZlYmMyMjIwN2VhNzdjY2Y3OGFiMjY3MTUxZGMxNmQzZTI3NzNkOGFmY2ViMDdkM2I2ZWRhOTE3NzRkMTIwMjg1NmI4YTZkOWMwNTg5MjE4MTllZmVlZmI1MDk5ZjJkOTcyNzRlMDBkYjQwMjNmMTg1NWZlMzZhMzYxZmU5NjZlODZjNDczMWFmMGYxODc0NDdlNTExN2QyYjk3MDZiNzU3NTRhNWZkMTU5MWY3YmE4OGZlODQxMDA4NzlkZTljZGMyODAxODAwMGEzNDM3YTU2YTZjMjc5YjZiODNlMjZlOWM5YWFmZDM4ZjAwODYwY2M1NWE0NzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xkScrfZRWMSIx9P90B6LgABC-4FR-_qY1Ome9y3dYXml9IW1pgS4Av7xUqC4EIdP9RxF-Edvn83tlghsJCNfTg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111806_41_2403_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.726Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlhyMGlna1hhbDRFc3I5MXBzbklZWUNOV3FsTkErQnA5Z1NDa1RLRkpiSjYxYXRub1JwNm1WSVRoUlFxSEtlWjdEMHYrbDBZZjBJQ0tuZVgvT3VscGRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDZfNDFfMjQwM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDE2ZjgwN2Q3YTc2MmZmMGUwYzY3ZWVjZGQ2NDYxNTk3ZjcyYWNiMmJmYzU2ZDVmZWRlMWU2ODVlYTU3MGY2ODFhOGRiMGQzOWJlYTc5ZDBlNDdmYzJjN2ZmY2NjYTg4MTMxMGVmZDAzYWU5NDE4OTY1NzU4MzA2Njc5NjlhNjJkOTEyYWM4NjNhOTViMjA0NzE5OTk3ZGNiNDExMmZkMmFhYTY4ZDk2ODJlYWRmMWIzN2NjY2ZjNjkzNjQxYmU2MmMwOTU4ODhkYWYzZTZhZWNmNmU0MWVmNGVjOTBjYmZkZGE1MGJiMTdhYTI4NDY0YWNhNjI0YzU0Mjg5OWI3NWZhNjE2MGIyMzhkMDYzYjYxZDA4N2YyOGU4OTZmMmQ5NGFjODg2Y2NlNmY4Njg2MzZmMjZhYzA0ZDE4YjdhNTAyMTY0ZjJjZGRkODgwOWZkZTdmZTM0NWZkZjM3YjVhMGQwMDJjNTZlNzA3YmJiOWZiZTBkZWQzMWQ4MjQ1N2U1NzNlMTA2MjFjZjg2YzcyYTI0YzQ5OWNjMzIwMDMwZjA3MDc3NDg2OGNhMTFjMTJkZjVlMTAzOTZjMmNiYTcwYzQ3YTJmNThlZWRhYjRhM2RjYzUyYzBlNDY5ZmYxYmU5NjJjNzcxYWYwNWMyMmMwYTAwNDhlOGVkM2E2MWI5MzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6cum9MkK10BxSY6dtqNwpfa1CLjmBQwpqtQPB4kOYNx8b1t5u5TPSNN1F3GQbupIashSiSnUdt_eU48X49_bDw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111806_41_2403_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.729Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZlTWJsdjV0OXNlSVBSN0tjcnUxTGJKK3Q0aGQya0R5VTRSQmJPTG9YaGRFTWF4U3l0S1krSm1BdTk3Y0k3T1FwelZ4WjVBNi9VVU5vM2k0VmxseXN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDZfNDFfMjQwM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjFiMDExMzIxOTdhOTc2ZjFmYTg2MDQ2Nzc1NTExMmI1Yjg3YTdmOGUxZTE1YWUzYmVmYTQ2MDM2ZTBlY2Y5ZGMxYWU0ZGVkODUxNTgwNjk5ZmQwNDViYzlmNzExYWVlMDI1MjJlNmM3YjEyODZmOTc0NzNlNzVjNTdlNDgzMDc5ZGI0NGVkZWI4NDY0ZmE4N2RiNzRkYWI2NmIxZDdkZGUxMWZhMGNjOWIyYWRkYjc2YmVlZDBlMDU3NTA2OTlhMGFjYWVhMDcyNjY0ODk5ODYyNTM3MDA4YWFmY2ZiOTA1ZDgxOWMzNGRhYTI4M2UyNTZhZDNkMjhmZjdlYzU4YWY0OTE4N2JjMzM5NTU3YzUzNWFiZWVmZTAyOTBiZDIyMDdjYjA2M2ZhOTA0ODU2YzY3ZWRlMDk4MmY4ODQ0MmEyZTcwMWQ0MTdhOWMxNTgzOTFlOWFiOGMzMGQ3ZTliYzZlYTJiYjBjYzFkY2M5MzExNmQ1NTA1OThmOTI0YjFjNWI1YmQ5ZDA5MDRjMjdhNzEyM2MzNGZhZWRiODVkZDZhMzU2MWU2ZTZiZGUwY2ExOTk3ZjVkZjljNWI0Y2Y2ZDM1YTg3MzBhN2RiMzQ0OTliYjA3NWZhOGYyYmYxNDIwMDJkYjNjODIxYTQ2ZGVlODk1NjI4ZDg1NTFjM2MwMTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.svfk2rP3GlMVHd9fAcscDg5wA0Rri8gPU0gEEpVotXe63pGX4CUsRaR8RDAwcKRLhvH05G233kLq2g863F579Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111806_41_2403_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.732Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImkrYVo5QnVxaGRnbWcvSk12aXU2cDZBRFNZMEQvOFZVMUR2MHBIbEl1YTNPdGxhU0J2d3NJQnBhcnZnMG5KbFVMamtaZEkyVDkvSm9SNDNzaEY3TGhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDZfNDFfMjQwM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWQwMGQwMDQ4ZjhjZDJiYzQ3Yzc2YzI0MmJlNjE2MDUyMjBhMGYwMGU5OTBmMGE4OGViM2I5MGUzNzQ3MGNmYjI0ZTY3ZGQ4NzVkMTE1M2MwNjQzNGIxZmJhMzliNmE3ZGI3MjczNGRiODc2YTEwNjBkMTZjYTIyZjRkMGMxM2UyNjQ2ODg4ZGJlN2YxNGQxOGM4ZWMzMzU1NzJhMDVjODMyYmEwZjQzZTAxMjk3MDE3YWYwYTQ0NzdiOTNlMWY5NmQwY2ZkYWFiYjZmNzJlMWFkMTdlY2Y3YWZhODM5NmE4MDFmYzQyNjA0Yzg5ZDBmYTg1NzA4M2Y2ZjhmNGQ2OGQzOTJhM2M4MmRjZDAyN2Q1ZGIyNWM0ZWIzYTk3ZDk4ZmE3ZTFkY2RjZjhhNjE4NzFjNjU1MmE3MDgzYjgyODRiNjZiZDk5ZTI2Y2FhZmRlZWQxOWFmZDExOWEzZmQyNjY2YThiYTRiZmM3ZGRlMDk5MzU2OWUyM2MwMTgyOGFiNmZiZWYzNjVlNzM1YjUwNGVkZDM5YjlkZWVkODE0NmZjNmFhZDdhZGFhMzhkZmM1ODlmMzUwY2NhZWY0ZjNhMzgwMzY0ZjUxYTk3NTRlN2Q5NmRiODE2OTA4NzIzNWVlODA2OGE4YzUwZmI0ZDVjYWE5YzZhZjdmMDZkMzVkMjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iRFzL7YMajx7J_FAiB17kuHK1Z2iAnP9yez_T2tT9lNKQkhLXKUxtsL_IoN6I2Pgw7lafksmNoNYRlMIFeZrOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111806_41_2403_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.735Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik92ek9pdTQ5RTJ2MjZVZ1kwbTUreG5aSURzcFZEa2RwY2twanp4KzA4YnR0alQ5KytYU1VPUDBEL3IxeXU1aDlpQ3gvTnhDZjhOVlpBUHZnYldISXJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM2MTNfNzRfMjRiY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2NhZGM0YjFmZDRkNzczMmJkY2NiMGZjMGJlZjMwMzljOTZhMzgxODgzOTBmMmZlOTFmMjIxNjM0MzVhYTRiMDJjMjZhMjMyMGYzNmU2MTlhMDIyMjgyZDJjOTdiYmVlMzMzYWNkODEzOTVmMGMyN2JlMzFiODJiM2JkM2QwNGIyYWFjNjJlOTlhYTYxNGRhNzU3YmI2NzBlOWM4ZDllZjI4M2ExMWM0NTFiYmY1MzJmYTdlNGVhMTg0Zjk1ZDE0OTMzODU4ZjkwMmY3MzA0N2Q1MWVkNmI2YWI2NjFmYWU2OTUyOWExYjRhYjlmMDI4NTg2Y2Y1NzA0MTVlOGM3ODEwMmRlZjdlOGZkN2Y2N2QyZTBkMzcwMGI5ZjQxMmRjMGVhZjJiYzdmOGVjZmI5OThkMWFhNjZmYmE4NmM3MjMxOTNjNmU1MDkyYmE1MTFhMWM3ODEwZTc4MmZkMTc5OGFkMWM1YWIyYmRlZDgwNGRhOTY5NjExMjI4NTAwZTdlODdkMzA4OGE2OWZjZDk0MWRmYWM5MGRhNGM5ODJlODA4MTIyMmY0NWVhMGZjZGQ3Y2M1NWQ0ODQ1OGVjYTA3ZjhjYmVlYjRkMzczNmEwNzM3MmU3NmQ2NzJiMTQ2NWRlYTI4NDMyZjY4NjdhNGM0NmY0OTM3MWZhMDZhZGE5ZjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Be4k6yioedCzJzyeED5tmDN3koQr6jWcNWqRpu35faOiouqbw9XqSJciWB3900W2cvUVdFRjdmLLecGekIm-cQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103613_74_24bc_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.737Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IklkNVpRK05pb2pweDNWcUdpZ09JaGp2a2RGdzkxMDBYdTJSNmNvbnkrdUpPMDd3YmFsVnFtZThDeDRhR3BrZHZqWmovUGREZFZidm11Vit3aEFFenlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM2MTNfNzRfMjRiY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzM2OGMwOTk1MjMxZmIwYjg3YjEwZjZlYjE3OWY5NmI5ZTllNTM5NDRhYWI5MDA1YTE4ZTVkNGE3NDA2YWZmOWZjNmY5ZDlkMjc4MjI2OGNmYTRhZWMxMDA3OGI4MzM0ZTA1MjA2MTVmMzIwMjk0MWJiMDcyMTQxYTMwODE3OWVhYmJhNTQwNGI2NzQ5NzIzMzEwZjNjNjY3NGE0ZGE5MWJjNGNiNzk3MDA1ZjczN2U1MDNkNWRlYTJkYWQ5NzBjYzQ5NDg5ZDFkZjc4ZWI1ODdhYjlkZmI2MzkwY2NlMWQwMmU2M2QyMjYzZDgxZjFkYTI5NDIyZDhhODk4MzdiOGFiMmQ0ODdmZmM4M2U2OThkMDJiOTgzZmFmYjRmNDEwNWI0ODA3MWU0M2ViZjdiOTE3ZTRiZmMxNTA5MjY1OTYzZWY2ZGI2ZDRhZTY3YWU4ZDNlYzQ2MjRlZjc2MGQ4OTQ3NzI0Y2UzZWRkMjg5ZjJhYTU4ZGZhNTc5N2U5NmJlMGU0NjgzNzVlNzIwZDAyOTIxMzA3MDNjNDI4N2JlZDgyZDNiODE0OWU1NDE5ZTQ4NTRlZjE4NDM2OTVjZDc1YjFlMTM1ZDU5ZDE2Y2Y4ZTZiY2QxYzgwYWY3NTJmM2NiZTMxNGM4YmFlMGRjODY5Njg0MjMwNjQ5YzVmNjZlOGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iiU7hF78eAz8kbrhk_vnvQUpeX5XAEbEfhfotlhkkdoGPK48Swm7I2HjwBFMqzWwaXQNLAESwru-nnJ7_sTi2A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103613_74_24bc_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.740Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkdxdHEzSDZZQlJoeEVrVm5wMHFwRHY5eHlydDRZT1YrcVVlMlkwREhCSnlCT3kxRVdwMW1wMDRnKzBSTVYwcjdtT0phVk5VaXprcGI3NHZhNTJ6dVFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM2MTNfNzRfMjRiY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzlmMzhiNjI0NDkxYjlmMjg1ZGM1OGJlNTlhNDI2ODk3ZWRlNmVkMWMyMzAwZTMzNjQyMzg4ZTA3ZjNjNzliNjY2OTY0YjIxZTAwMDM4MzZhODUwMDc1ZDQxOTlhZjJlOGY2YTRjMzE2Y2IwMWEzOTNiOTMwNTE2YjhlNzIwODE2Y2U3ZTQ0NzA5MWJjNmIzYmY0YjIyZmEyNGEyYThlMWU3ZGMzNTRmYWUzNjQzZGFhMjU3YzA3NWNmNmQzNjY3MWM3ZDExM2UwZDEyNmVhMjY5NGZkNDAxMTBjMmIyODU0Y2ViN2U0ZTlkODY3OTMyZTQ0ZDY5NGY5NTY2OTcwNjNhZjRhNmFmZGYwYWE5M2MzZDM2NGY1ZWU0NGFjZDcwZWIyZmVlMDE1MWRlMTExYTBkMTdmNDQ5NzM3Nzk0YWM0N2QwM2RkZjJmMjI5ZGQ3NmY4Njg1NWZiMGE3MWVlNjFiNDA2ZTEzOWMwMWU5M2Q5ZmFlOWZjNWU4ZWNhMTFmNTc5NGI4ZmUzZGY0ZGM0ODczNjllOGVhZjg2ZDU5NzYxMmFmNjY4ODE0NGVjN2QyMTI5MTNhZWNlNGUyODEyYjA1YmNhYjZjNDMwOGY0MDI2Yzk2NDg0MWIxMzQ5Mjk0MTEyMWE0NjFmZjYwNmQ0OGMwZWUwZGY4NmE1NDEwNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.31TsCVdX7KIe8vi2C-qepe-3BNwCoTY0WfvvZEC7hlRHp2vNX8wBpFok0gTVV1xxm5lbzjXceQWbLO-jm2SfmA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103613_74_24bc_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.744Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImdpQW5PcDNhQm1sYytnalFnMHBTS282S0RFYjhHVDNLR2pyaGRQV0hIM3BEdWV5Q0FUR1J5TWpHYlZsaUdxbm10Zmw1cmtEMHp0TUJtL3BpQUUyeHFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM2MTNfNzRfMjRiY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTUzYTJhZDhhNmY2NjRiNmY0ZmMwM2Y0MDdmY2EzNzczZDdjMTY0MzI4YTgyYzVhNDE2OTg5NTRmZDQzYmUxMTdjNTJlOTQwMTJlM2I1ZjBkMTEwMTIyMDhlMDEzNWQwY2QzZjlhMWNlOTMzMWViZDc4NWMwZmYzZjY1ZDM2ZDllOWQ5NzNhZjdlZTMzYTI1ZGUwMDM0NDAzOWUwYzNlZWY4NTMxMWI5MzFkZjVhMTFkYTU2YWRjMGJiMjI3YThmYmY3NWQxZGU2MjEyN2Y0ODRkNGNlNjVmOTljMDZlOWI0YjJjNGY1MTUzMDcwMzU3NDUyYWUzNmE1N2E0ZDdjMjYzNzJhNzI1N2FmMDRiODJkMDMxNTkwZmNmYzZlNDdjOTU3NjJkNGMwZGJhYTAwZTY0MDkyN2E4NGY3NGFjZTJmNjkzYTYyMWNlNTc4MzA5MDY3MzE2YjBmYjA4NGEyZjQxNzhhMWY3MmVmZGY1ZjRlMmM0Y2QxMWZhNWM2OTJhOTg4YjU2N2RhYzQ2Y2FhZGVmODc4OGQ4YzZjOTRmNjlkMDI5YTE5ZGE3MzgwNGVkZjM4ZDc5ZjIzNGZmZDY2MGE4NTdmYTE1ZTY0MDkwNjdkM2FhNTlhOTFkYmQ3MTZjZjM4ZThlODAxMThlMjI0NzJlMTE0ZWNmYTQ2YmZjOTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AoIZbUCvSpSWanDYwJF3Urc0tPn_WwcKk-S3Ui2b4zi_2IuAmvwYsVms9DstFDReY_CiTc-sUx3wEEAq5Pm7zA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103613_74_24bc_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.747Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJEVlN6eXlHMDRFVStJb2JBMExBU3NndEI3MCtLY1lZK3VDbENIR3BENit1cUFFUkpyaGhaUEdEQm5WaEhjUERSYmNzMDV3WUViVTMvUlB5ZFNYWTBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzdfODBfMjIyZl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGMxYzBkNjA0MWJiM2Y1OTJiNTc0YzdiMWU3ODQ1ZTM4OTJjMjQyN2ZmMDE4OTFjMWFkZTgxMjcxOTFlMjgwYmE3MDYyM2Y2YzNhYjdjZjg4N2NlOGUzYjc5NTk5ZmNlMTFkZjBiZGQ0YTJkMjVmY2E0NGM2YmE3ZGJhNmE5Njk1M2E0MmI1YzA3YzEzZDE4ZWRjZWY5YzU2Y2FiYzBhZjY3MmRjMjU1MjEwMThiNzdmY2QxYjc2NWFjNmI0OTVkM2YxYWM0MGY0YjM0YWRkNWRjMzA4YzZjYzQxOThiY2NjNDM5NzhiNDdmYTdkOGIyMDQ1ZjZiMGQ2YWM2ODFhY2VkMDM4MDA4NDkxZDI2OGE0NzM4NTc1NmY4MWIyMDhhY2QxNWQyODUzZmMxMzYwZmFmM2YwM2Y0OGZhZTlhNzhmYzRiNmQwY2U3ZjM5N2U3OGY4NjFmOGExMjJlOGNjZDliYTJkZTEzNjU3NTQ0MDBlOWM0YzFjYWE4MjI3ZWM1MTViM2YwMWI2ZDAwNjk0ZjY1MzhlNTM4ZTcwZWZkMmVkMmIxNmJkOGVlYjYyNDE4MjkyNWNmYzA4ZDE2NGJmYWMyOWM4YjA0ZjE5NjcyMmM1NjU5YjIwYWMwMmI3ZmU5ZThlODE1OGI4MzYzNDAwNzNiNzlhZmQ4Y2MyNjQ2ZGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4tyvv_WZPvylQoNp_dBZ-bIaehvWJMhT3zD2Zv8WOd57youJrZcU6VU3bbfI8RjU5MVzhmlQvJAK0wT9lmlv8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104637_80_222f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.751Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFwSWp0MFVYeWF5Z3ZiYXNPMkkwcnBLV3BUTEZ6ZHNCRlBKNjBOT001enRDNlRyMjNYL0FGbGJ6U003ZllnZkdkQnp4MldmZGhjdVQ4MzVhQ1k4eW9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzdfODBfMjIyZl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTYwZTEwOTE5YWVhNzJiNDA1ODg4MWU5NmJhNDc5ZWZiMWEyYWRmNmU3MDJjZjhhNjEwOTgxYWQxYjY5OGE4ZGQyNmFhYTI3MWZjYzU5MTE1NjJmMjc3ZTI1MjBkODE3MDMxYzhmOTUxNGM1NzI4NjU1ZTNkNDE1ZjlkMWMyMDc4NWFhMmMxNGM1Mzc4ZWZiMmQzOTM2OThhZDdkZDYxZTE1ZjdiOGE4NGYwODFjYzlmYzM4ZTRjZWY5OTIyNWMyYzk4ZjIxOTk4NTM2NjVkYzhlOGRmZDg5NzgyMjg4ODNjZmYxNWEwNWVlYzQwMjE2MTYxNjc4OTljYmFiM2JkNmRlODQ1NDZhNWVkYmQzMjY3YTVlZGFkZDBhZjgyYTgzMzQwNTc3OWIyMmMxNTAzMDY1ZTc4NzdmZDRiOGY5YzAzNmY4N2E4NmUyNGQzZTEyMzIxYmY5OWU4YTI0ZjgxMDEzZDQ3MDU2YTkwYTU4MjE3YzgxYmRjOGEyNGI2MjUxZGQ2MDdmYzIyZWZmY2RkMmI0MmE5MTBlNjk0NWU3OTUyZTRkMmIxN2U4ZjU0YjhiYzBiODZmOTIyYzc1ZDNkYzkyYWNmMzRlNjcyYzliZDlhZDViZWQ0ODA0NWEwZDU3ODhjN2E3MjY4MTNjMjE4NmU5ZjAxZGM4MDgwMDdiMjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.u2BB0SIQu2XEspVsebnUSTgsyzNFZ8-2dLzd9PzXSEtOcbvXjIYN647bYAvmILu-_wANb8jTRQU7-y1Hflwazg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104637_80_222f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.755Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InUyVXdaSU4wQTVlN09XSzh5WjVWVGxXN2UyWkZENjIxMXVycjhKRGpLS3JZUkJhZ00wWlFQd2g5V1ZxOEE2Tnlyb0VHekYxZGhsbUJOVjQzdTBtOTZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzdfODBfMjIyZl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGM2OGI4YzA3ZWIxMjdkZmMyZGRhNDJhYzYxOGI3MTFiMDUzYzM0MGY3ODQzNzZlYTZmYjAzMWY5YzU5NmQ2MTczNTg4NzBmOTFkNmFlZTEzNTE0ZDRkZjQwYmJkNDdkNzUxYmE1ZDcyMzkyYmQ4Y2VlNWJjZWJjYjAyYWVlNTAzMDEwMTYyYTAzNGU1MTk1NWQzMjBjNGU0ZGYwNmMxZDQ1OTNkYWJmNjc2NmExNjdiNjIzZDk3ODgwZWY3YTZjYTg2NmM2MWNjNWU4Y2UxNGViNjRkZDRjMmI3NTY0MTQyNGZiYWM1YzRkMDMzZDhiZjBiZTkyZWIyODRmN2Q2Y2Y2OTAyNTYyNDllZjNjZWI4MzBjMGZlNDRjMjJkNjM1MjU3ZTg0YzRmNDA3ODE1OWZjOGIwNmI5Y2JkZTg5MWZkZWEwYTU1ZTY1YTI2ZGQ2MzU2YjVmZmIwZDk3ZDk4NzA0ZmU2MmM4MTM0YWQ2YTFmZmIyY2MwNzE3NmNjNzIyYWI3YWY5MWZkN2QwMDljMmYyOTA5OTBiMThlYTVkMzBkOTZhYzVjYTE4NmJhMGU5MmIyNjRiZjljMjM0Y2JkNWYzOTEyZDM1ZmIxNGQyNjAxNjQxNzljMWViNjgyYTBkZDI0M2JmN2MyMjhiOGNmMjkwMDFiYzVhMjFkNDc3MGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AF0GFxwC0gVW1YgaAfKNpbJ4DTrElX6rUgyMh86vdrG_PYbeF-8I162Pmd-BN0ceZTP_atjTJdDNu6CQ1c7Pqw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104637_80_222f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.758Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjFZV3pGdEd0bUlSQ1ZEei9JdzlBM2ZkRXhtVE9GZCtUNm1ITEx2MTZMOENEUkJ1S0pLS0FlTllyTml5OXJDbVp4VWdHRFltNDZDN0RqZVdaRno1T1N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDQ2MzdfODBfMjIyZl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTc1MzQxZmEwZTY2MWJlOTQxNGQ2ZjEzZjU1ZDQ5NTk5NGQ3ODVjZjBiODc1NTFlMjBjYjA2MjA2Y2EyZTk3ZDViYTg4YWI4NTZmNWZkNWEyODViYjIwYmIwMmNmY2NiMjY0MWNmZWFkMDMwNjc1ODBkZWFjOTZjY2UyYTA3OTQ3NDJjMzhiNzk4Zjk3ZmI4Y2YzZWQ3MjUzYTQ3NGNjNTYwOGE1Y2VhZmFjOGU4ZTRkODlmY2FmMDkzMjBjNmFkMTMxMmM1YjNlMDY3YzA5MWU2ZWUzMDM3NzhhZDdkMjliYzVjYzgyNGQ4YTQ1ODk3NGJhMGVjOTIwN2JjYTEwZDU1ODBkNzRhN2ZiMWIwYTdiNzdmZTRmNzkwOWFkYzZmNzZlNDFkMDFkYThiYTQxN2Y0NjRkOGFmNDkzNTNkYTc2NzM2ZGQ0ZjViODEwM2VjZTMwODRkN2Y4M2EyM2E2YmYwZWEyNzEzN2RmN2NmYTBlYzI3YzQyMzI3OWU3Nzc5ZTliZWM4NTA4YmE3NDllNDNiYTFmYjk4NjhiZGRlYmFmM2NiMjkxMTFiN2RlNzJlNzQ5ZGJiMTQxZDVkMTM2MThkYTU5MDMwYmQ5ZTU2ZGZkYmUxZjY5Yzk2NTNhZWU4NDdmYTY5OWZhOTBkODkyNTJjMzNlMzE2MTRjNWZlNmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Kv1r1WLIu00jpGaGpRL7ci7iVgltCjUdWL27jqCMyO5B_18doN8Qs-aKisisyd13UhI8FupeL9BFczQK-iOgrg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_104637_80_222f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.761Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkRpcXV6TUh4bVFvOTBSTmlxZDhWVUFMZitNZGZFTUp1c1dUK2l0dHpFa21jZXduVzRwbW5yZXpVK01GemxnQ0lwNjR6NExzeDVTZEk4Q3lIRTlHT0VnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzVfMzlfMjQ3YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWJjZjZhNGE2ZTg5ZjVhODI3MmUyNTQ5ODJlYTA0YmRmZmY5N2NiYzM1OTkzNjY0ZGQ4NWQzM2I3YTZkNjM3MWU2NDE3NjgxNTY3M2JiNGI0ZmZiZmZjMTM4ODJkMTU2YmJhNzZkZWYxYmNjMGNiYTBmMDViZTcyY2Q3ZjhkODIzYWQwMGQ5YTUwNDBhZTQ5N2MwNzdhZDU3MTM5OWJmMjZkYWViYzY4MGFkMTMyODdmYmFkODNjNDc3NjY3ZDdjYWUyMmVhZGFlMGI5ZDczMDU4ZGVmMGRhMGJhYTY3NDdlYjNiMTlhMzg1MjcxOTkxYTg4MTYwNDMzOGI1MmY0NTg1NTBjMjM3N2RjNTFlMjMxMGQyZDNmZGU0ODY1MDYyMWE2MGNlZjYwY2M5YWQ5Yzk5OWZjNDcwNjRjY2UyZmFlYzNhYTkwMzAxN2E1ZmM4NTFmZTI1ZjIyYzE5OGVmNWRlNzlhYzY3OTY5NmRlOTJmNTNmMGJlYjIyYzBmMTcxNzgzMDQ3Mzc2ZWQxNjk0ZmQ4OTM4MDNlYzMwOWU0NTQzN2ZhOTQzNmUyN2YyNDFiOTY1YTEyNzg3YTI5N2I2MmU1ZTAwYjQzZjg4OWYzMDQ1NzhhNTliMGQxZGE5MmY5MzE0YjdlOTJkYTllZjQwNjFlZTMzOTk0MThiZTk1NTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1TuT7fSls7MaeFHBzMMlFPNcVJd2ikpmZ7z20dSm5W7zSmu2w-cHhVVPoEzEMcDvxACLOPQx8ulRWw8IGq_nrw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111235_39_247a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.766Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImprWmRoTmZRT0QvaTlKUkhXWG5hMzZTNmxjQ0htTUFOTEFDOWdSa29YNVlhbVA1elVGR0VxcW1UY0JPdDlXMFo1MGwySUV3VDJjMmJURWpoTVU1ZFdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzVfMzlfMjQ3YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzE2ZjczMzUwMjAyNWY5MzU1ZjBhM2FhNjU1YTIxZDFiOGZiODk5ZGNmOGM5MWVmZjI0ZWVlYzI3MzM5YzBlZDg4ZjM1ZTZiNGI2ODFhODVjZDg3MDRmM2E2ZjhiOTk2NTE4OGQ4NmVhOGMyMWIzYjJjOTY2YTZjOGQwYmMyNzJjYjhhZWRhYWI3YzFhZDYyYjMyYTdjNmNiZDkxMzYwZThmYTg5OTg2NTg2ZjA5ODBlYmI0OGU4ODZmOTZiZjg0M2NhMDI3NGRmNGEwMWU5ODEwZmJhYjM3NjJiY2U1YjI5NDc1NzU1OWE2NDk3MGFmNjYwYTlmMjYzYmIzMjc2ZDYwZjNkMmZjZDEzNDQyODE0Zjg2MzE4YmU2MjNkY2I2NmE4YTNhMDQ3NmYzMzA4ZGE1NDQ5YTg2ODI2NGIxZGNkZWZkYWY3MjkyNzI5MTRhMmQxZjIyY2M1MzhhZDZlYTMzYWQzZGM0ODM3NzRlMGVhMjQ0ZWRlZGZhYWUyNDc0OWRlODFkNzg4Yzg3MzQ0YTZjYmQ1N2Y5YjdmZGU3MjI0M2VkNzZiMWI3Mjk0ODEzMjNjYWFiNGUxNmM2ZTA0YzVlY2Y3OTY2MzAwNmRjN2FiZWQxODBhNDIyODEzMThhMWFhZWZhYWZkNTA0ODdiOWUwNzk4MjhkZGVhZDM3ZmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-6dVz0-upz_DOA9KCKm9-5s51gzsbXsWAbdcfQEZ1QvhWBb7j32yG1nbh6eJ8RfuhnGvqdnqGZ6CwADyFmLuag", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111235_39_247a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.769Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkUwcFd4WkxlcW5wa2p6SXhqMWRjNXl2UlVzNUxDWFdCdjh1cU0ycVAxc0F0VStOaG1kVXRsVjBOaUIybmZGdzNZbW92R1NWemxGRnVzU25EdjBmRFZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzVfMzlfMjQ3YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWM4MzE2ZWRmZWM0ODczZTYxZmM0OWMwMThkMjQ4ZmVlYWM3M2VhMjM5ZTlmY2U0MmZjYzE0MzVjOGZiMmIyZWYwNjAwZjk5ZTA0NTU5MjFkNmM4MDBlOTExMTliMjlhMWYzYjAyZGU1OTkzMmQ0MDRlMmJjNmM2YTA3ZTg1MmU3ZjVkNGI5NWFjZjVkY2ZhNDgxOTMwZGE4MDgwNjk2N2Y2NWY4Y2E4ZmJjZjI4ZjE1OTJhMmY4MTRhNmY5MzBiMzc0YWExZWU2NzA4ZjJmYjAyOGM3NTMxYjQ0ZWNhM2IxMjNiMDFhYzc3ZWVmOWY3YWViMTA0NmRiMjRiZmYwZDZkODYxNjBiOGU2ZGIxOWY1MTljMDg4NTczMjU5ZGQ0OWVjNjY4YTU5Zjc4ODBhZjIzOGZiNzBmMzU0OTI1MDE0MTA4MmJmNGIxN2M4Y2FlNGNjMGYyOTE2N2FmMjc0NDEzNzc4ZmJhMjRlYjU3MjU4MDU1NjUzNjRhYjU2ZDA2NjVlNTNhN2Q4NjQxZGZiYzBkYWIyYzk3NzI3N2ZhYzE3MmRlNDkyMTM4YTcxNWRmYzlkNDhmMTJkZGY5Y2MxZmViZTZiN2ZiNTg0NzY1ZmVhN2FlYTVlMzA2ODc4OWNmYzQyN2EzOTBhODU3MGRlOGJhMWEyNDZhNTVkY2I1NTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SSKySzrL9gc8Nl1yfZT_Z-Mwe-13541OEIXcNl127XwHXWwCQnnfB2q_n6kbihyqamFtS6xQjoxbSMzSGvGymQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111235_39_247a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.773Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkloMk5QZXJwTTY3ajBpcURCVTdiTnB1Mkt5QVBtcUZ5TzdHQ3JTek5naU9ZKzhWUkplOHhDNlJhbEtVc2NXN2RFR3ZhM0lIQU14aXZHekUwREs4Z09BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwN18xMTEyMzVfMzlfMjQ3YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGI4NzZhY2U1OWE1MzQ1OTMxYjFmODhiMzMzMDFmM2IyMWYxZDNhZmVkMmZhZjg1MTQwOWJjNWE4NDI0NGNhMDdjODI2MDEzYjg3MTMxZTMzY2Q4MTYzYzkwMjQzYjA3OTFiYmEzN2UwOTAyOTdmN2I2ODU2MGNiOGQ1ZWI2YzQ5Y2NjZDNmMzY0M2M5ZGFlZDBkNTFkMWI0OGQ5OGI1OTliNzg4NTgyMGVmM2QxNTYwY2Y2MjExOTIwMmQ3OTIyMjBmMjBlZDk2MTNkNjgyY2QyY2EyYzA2MTE2MTIwZDEyN2MwODRjMzEyOWE1NWMxZDc5ZTE3M2YzY2QyOGMxOTQwOTc4OTkyZDY4ZDM2NjIwYmFhNjg4NjdjMjhiOWVmYjkzNjMwZGYxY2QzYmQ0NjIxNWY5MWUyZjlhNTA5MGNiOTY0ZjE4MTQyMGRhNmZmZTA3MDNhYzgxYjU0NmM2MjcyYzlhODdiMmJjNDY1MDg3MzBjZjg1YWMxNWE1ZDJkYjc3OWQwMDU2ZWM4YTMwMjY4MzViYTFmNGYyNGIyZDRmYTRjOWRkNzBhZjIzODgyN2ZkYjU5MmRjMTg2NjljMDU4ZjQxZDhhZDA3ZDEwYTI1NDljMmYyZDlmODQwZmJjZTJkMzU5NGZiY2I1ZjhkYTU3NzVmZTQ0YjY5MDY5ZTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qQm-qH-miGtN2D2NPLqdcQtaeHjCCH5blgyCWkh5LS3SSztplm98Dt3gPlm-WqxI_TGijB0hqfpLX58aMzWOWA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231107_111235_39_247a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.776Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikg5em5naWV3THNUZkM1UitVaEZrV1hFTDhzSzFyTTN5eXRLZWk1YVVVbS8vUHdWUHdGOVI2R092OHFOVFg4MGt2LzVBWVNZTUk2RDRMWEh3U3BIZ3p3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMTA0NTVfMDZfMjQ3Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM0MzAyMjY1MDQ2NDcyNjZmNmVhNDcwZTNkM2NmNGQxNTdlYjczYzViNTA2N2MxYjA5ZjlkNWM2NjJkOGM2NDAxYjQ4MzZmNmExYzhiYmE2MzMyMWJiNmU2OTQ5ZTZkNWVjMWZhZTEyZTVhNjgxODBjYzhjMDdmMjg3NjFiYTdiNDQ0YTRjMjhmY2VhYjY1YmM3YTliZjA0Y2M3ZmEzYjZmNzhjMDI4OTU1MDhhNmM4ZjM1NmNjZGM0MmZhYTE4YWNjYjg5NzVlNjA2NzEzMTA5MTZjNTM4MzRmNWU0NjBhZjAxNTViMWYyOGIzMTNhMDYyNjhkOGRkYWMyMGMxZDkxMTI5Zjg2NzZlNzVkMGM0NzVjNTAzMmVmZDliZjA1N2RmODAzMDFmYzQ0MjkxYTc1ZTM2YWI2YTMxOTNmODFhNDcwMWRiOTc5MTdjZWE5MDM5YmFmMzgyMDZiNmIyY2IzYjNjMzY3ZThjMjQxMGE4ZDEyMDk3OWMxYTZjMGUzOTg3NmIyY2Y0ODllNGFlZjZlMjNhYjdkNjFhODYwOTNjZDQ5MzNiNGViNDEyYTEyNGZkMTAxMGUzMGZkMjUxMjRkMmE4YmYxOGE2ZDIwMDlkYjhmOTc2MjBjYjI1MmMwNTVlNDI5N2RiZjQ0ODIyY2EzZTIxMTQyYTZkMzBhN2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Fz6OokL4w51Z4qDhlObjAbwJr1DumlYZKYKc0te8nJlACYZLgmH8yT7f1APE9rIrkr-vlOIOEsCvgdGX3fLq5g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_110455_06_247b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.780Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imt5ZGhHeTlBaE1DU1VDdWovS29CWlZvRXRVZTkxU2lwWEZxc09oa2tkT3FabldadnZVTmt2MnVDUy94TUYyUEQ4aVhkYU9uc2RkYkg2YTl6ZVhJSzBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMTA0NTVfMDZfMjQ3Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWRiMDZlNmFmYWQwNGRhZmIyMTk2ZjEzNDhkZWQ1NzdjYTgxYzdiNzQzNDU4OTY4MTg4NjYxMTM4M2I1MTE5ZTNjMDRlY2E2NzY3ODk4MDk4NjExNDgyNjA0ODQyNWFmNWZjMjUxODc3MzVjNmFiOWY0N2UxZThmN2E5ZTU2MjZmY2E1MGE3ZjhjM2JkYWU1MWVmYzVhNmQwMjBiMzE3MDkwZDllZmJjOGRkZWFmOWMxMTZlMGZjMDc1NTdkNzY0OTEwMzQ1M2M4NGYzODNkYWJjZGJiNWU5YzYyY2I4ZjM2Yzc4NmJlOWI2YTI5NTdlZDM3Y2UzOGU1ODk2ZjU4YjNkOWQxYTBiMDBlNmEwNmRjNWIzNDE5Y2Y3MDQ3OGZkMGQzNzk1ZTVkZDA4NGZmYzUyOTUwMjVmZDRmYmVjODQ3OWI4MGRlNzI1NjdmN2E0MmRiYmZjNTFiMDA4MjE0MDhkOTRjODQ4YWE4MjJkOTk4Y2M1MThhYjA5YTI5Nzg2Y2QyYzExYTIxMmVmMjMwMzFjYTI1ZGQzZTJmODI5NmNiMWNlY2MyM2YwOGE5NDVlNTE5NzJkNjQzMmJmZjE4MDdkOGM1YWNjMDUzZTBkOGFmOWFmNjA0NDQ5ZjMzMDJkNjhkOTcyZjAyMTlmM2I1NmZlZDAzMDQxOWU2NzBiNWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6GjHQaZdM3wGaQ0PjiHCh1N6xD4X84jdcW6D4DAg2c6M2qAduwXnfYgBvsQ3Tpyfsoa-vpvo6w3m19O8VksQ2g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_110455_06_247b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.785Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjFWOWFXeTlBMnBJVzF5RUZWMml0bnEvck9yTjdSeWVxMFl3UTVxYWRZWnRZSktMeXBKQ05ORy9NanV5aXc2S2VhRy93ZXNWeHlSSmg5Y3k1cE5YbHV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMTA0NTVfMDZfMjQ3Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2U1MDBlNzA0MzRjYWNhZDQ1OWUzNmZkZGJjNDIwNmMxYTkzYmNjY2ZmYTYzMTE1NTNhOTg1MDk4NzA3NWM3Nzg1ZWI2MjFjYmY4Y2Y3ZTE0YmNmY2NhMWNmZGZjYmFhNTZhNmE3M2M5YjUxMTY1ZmUxMzMyNTFhMDYyYzk3MzQ3MmQ2NzcyMjI5ZDJmNTVlYTYwYTlkODE5MGQ0NjY1MDE2OGU4NmEwODQ1OGU4M2EwNWNlMTdlZWQ4NzE3NzQyMDdhZThmYTIwMmYzMGFkZWM0YjBlYTRkYzJiNGYwYTk5MjA3MTk3MjRkM2ZiOTQ4MGEyN2RmMzFjZmVhYjc1ZDI5YWM5NGViMWRmNzA2NjAwMDg4YTVkY2QxNmRiOGNlZTAyNGZkZGYwMDhkMDE0ZDVhMjcyM2EyYmU0NGFkYTQzNjQ4NjcyMDQ3ODhkYjk2MDcwYzA3NTc4ZTEyNTFmZTE5NmQ2ODE2ZjExNWYwNDFkNDI0ZThkYTliMTczOTA3MGFhNDFkMjk0NDk0ZjFlMjQwZjI2MmIzNDU3MWY0YjJhMTNmNWFhMWJlNjIzZWM4ODM3NjQ1MmEzNDg1NGRjODcxZDQyYWVkZWNkMTMyNzcwZmU3MTE2YjEzMDlhNTQ1Zjk1YjUwNGE1NzhkMzFhNDdmNTU5MTc1YTA5NDViMmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Crp-CW7cwPLCoZ69qVdNs3tw93YMwyF93jqJzlmbNpmttADwo3MebZo4brew8i6sQBFGohTPupF5nkQB0szzKQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_110455_06_247b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.789Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Iit5MHJiVnBTWU9PL3BsbkFDdEtkeWlIalJsczRmTWVSM241TU14ZmozanB3dWY1eFpVWjEwSnlqMkVBTDNIVFJQcXVNZGxSNlVNZlg0WkZvR1JwdDRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMTA0NTVfMDZfMjQ3Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTA0MDllM2Y4OGE4MjQwN2QxMTY2MzUzNzE0MmFmOGY3MWNmMzE4YWZiYzg2Y2RlOTAxNTc5MWQ0YjIwNzhmOTljZjBhMTg1OGZkNDAyYjAzYjhjMTVlYjIyODU2NDQzYzRlNGUxMGFmMWNiODk3MjZkZDc5NjA5NTA2NTE4MzI0NjJjMjJjNDhlYWRmYTE2MmQ0ZjYxZGY2NDBkNzc2Zjk2YThlODIwZDJhOGE5NzdjMzQwMDgzMTA1ZTRkMmZkYzU0MDFjYjJmY2ZlYWM4MjVhYjI0NDgzZTk5ODcyZDRhNjdjMDA1NWI0ZDYyNDhiY2NlNWNiMjgyYWIwMjA0YThlNmU1YTM5NzBmOWFiNzAzMmM5MTVhNzk4MWEwOGI0NTUzMGY2ODY5YzMwNDQ0NjIwYzg5ZTcxZmJjYTA3NjBjOTQ1YjY1MTMyMGQ1OTVlZmEyOTAwNzZhYjY2Y2QwMDMyOGM3NzQxMDA5YTQ5ZTg1ZTI5YmUyNWVmYzM0MmVhOTgwMjFmYzZjZmE3ODcyOTJjZDhiYjRkZTRmZmNmYjcyZjg5ZmNiZGQ5M2IwZjliZDlhMTM5YjFmMTMzMDRjOTlkNDk0ZjEzZGNhNmY5YWE5YjkwMWU3NWExYzAwMTNhNTE0NTdhNDc4ZTY3YmQxNDM4YmVjYjg1OGYwZGMwODZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.v0EHRddlKAiw1DjfKLVUtZRYJ8IT2ukZ14iccUdPzgMhRzYii2PZylr1ieUHKazL-eS7lcm9_EsNmC-0NavsVw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_110455_06_247b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.792Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlAzYyt2UEYxWkU4RFArNTNzSm9YcElMazdZV0NuRUVKOThWdXZoRGxXbHJNbytHeEVhc1lKZWw1REtZcG02Mk9GZE10eDlSS2VIOHdvN08raW9qRW9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDlfNjlfMjRjYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2I3MTJmOWM4YjcxNGIyOGUzYzU4MzJkMzg2MDY1ZjQ4MDkzMGI3YjY2OTRlZTJiNjk3YzI4OGEzNGU5NmVhMzRmNGY1ZmQxN2ZmNjcwMjI5NTM2ZjgzMDJmMDhmYmUyZGZiY2E3NzQyZTkxZjRhOTY2YWE0NGVlZmM0YjkyZTA4MWFhOGU3YjgwYjc2MzFhYTNlZjFiZGYxMDVkYTBhZmEzZmRjZjFmNWJlM2Q0ZWZmODFkZmRjZTY1MDNhMTc4MWM4M2ZmOTZhZmE5MTlkNmNhNjFjMTVhNzVmNjhlMWI4MjZmMDAxNmE3ZDgxOTMwYTVmMmViNzcwOWExNGY2ZmZlMmRjY2FjNmYzYmI2NDI0ZDdmMzg5NGUxMTc2NzgzYzllNDgwOGU5NWY5MWEzNGU4ZDQwMWNkN2ViMDRjMGZlMGVmMmJlMzY5Yjg0ZDhjMDdjY2YyYjFjMDk0MTQ5YTA1YWQxOTQzN2IyMTI5YzY2NDY2MWVjMDA4NDUwYWI3MGRlNjc3OTZjMmJlZjEyMTkzZDUyZDNhMjc3YWI0YWJiNGZjNDI0N2IxZmUxNjQ1YjBmNmNkMzJkYmVmZWExY2I2MDFkNmJjZTU0ZDMzNTM2N2Y1MDdhM2E4YWI0NjEwY2RlY2Q3NmI5Y2FiNTgzOGEwNWM3ZjZmNjEzOWRlYzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.48V6HLQ8YrgztuRQFc9qOxrkkJJIJhzQ7MRNDJdNIPhvLOFZMBEm5JdYDnzfSmiskFFJ-0vDhgQmgJNeqnlyRw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103249_69_24ca_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.796Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlpXWTJpcE1neXllMi9MOGpGYUFOMVRBZ2dWMGxCOFR3ZWREY0x6cEE4c2NGVXFvREtBbVRFTHZybHZuVUtEeGhXT2taamQvQkF0c2w5Q1VOZGxqb3hBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDlfNjlfMjRjYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY3OGZhYjRlY2EwODdjNWEyYzU4MTc5ZTI4YzVhYWE5Y2ViZTU1OTg5MmQ0MzZjNmI5ZWY5ZDFjNTlhMTc5ZjhhZTRjNmVlYmYyZjQyYzBlMmMyNWYxZjUxNjExMDkzNTA2OGEwNmNmZmFiN2JjN2IyNjMzOTUwZGFmZTQ0ZjNmMGZkOTJlYmE1ZjdmZTYxY2MyOWVmNTg1MzUwZTgyZjNiMTc2YzE1MDA1MDI4N2M3NmU4NzNlMTNiMWFjNDk1YzRiZGEwZTIwODkxNDg3MTcyY2EwOGJlN2EwYTRjOGY3YjJjMjA0ZTRhNmMxMTQ1OTI1MmM3MDI1NTFhMmU0Y2ZlMTA3YjdmYzExODkxOWFjZjI4YTg3OTg0ZTE1ZGMzZGJhY2JhYzY4OTEyZTVmNTljNmQ1YjYxMTgwNjU5MGFhZWVjNzgxZTYxOTVkNGJhZjMwMDQxNGRlY2Q3MjdhNjk0NGQ4ZWQ5YTI4NDAwZDk0OGQwZTNmNmM4OTE5NDJmMTk0OWFlOGVkYTVmYzQ2YTY4NzgxMWEwMzg4YzVmN2MwODUxM2ViNDk2MGY5MzQ5YTc5OTJlNGQ1Yzg4N2JiZmFlMzU2NGNlZDVmMzE5YzQxYTZhYTEwNjg3OWEzYTMyMzEwNWUyMWUzMmZiNzhkYjk0Y2I1Y2RmOGM2MWI1ODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ChEF_1kP18bXY7Be9kRJUH7p6yk2xudaZJIvgU5dVaVJDvGxEJcsJ1pFys0KqWlAdnVlUQomtfc-pofn24zcsQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103249_69_24ca_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.799Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkdYNnA4VmJhMEpwd1Ixd090ckVSR3VuRlBhMTlweGJmY05JSTYxN3BLQVNJeW5JckZGeVpNbDh3RVlQUmpaSTZCZUVtQnl3cGxPdFhta2VJcE1zYlBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDlfNjlfMjRjYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmNhOWUyM2ZjNTQwMzkwYjNlMmY0NjdiYWI0OTQzMTljM2JkZDM5NGQyMDdjY2ViNTdlNjYwOTU3NWQ2MDZmOTYzYWYwZTY5YWE3NGMxODEyNGI1NmYwZDllNjNjZjY5NmU4YmM3YTY4NGNhMjJjNDU2Y2FlNzk3MDczMGE2OTBmOGJhMTVmOTAzYmZlMTNhNjQ5NTAxYTg3MTVkOWFkODcyMzk5NmU1YTZhZDY2MTFhZGFkZTJlMGQyN2IyZDVmZThlNzNkNWRkYTU3NjQzN2RlYzc3ZGRlMDJiYjI1NTI1ZGE0M2NiYTA5MTcwMzI2ZjhkYWE5NjZmMzI1NThhNzE2Y2Y5MjY0ZTU1MmJkMTcxMjVhNzkwMGM4Mzc5ZjQwMjkzODU3NGRjNTM5YmMzMTRiNzFlYzNhMTkyODY3ZjNkMzg2YWU5ODY3ZjM5N2Y4ZmRiNWU1MWY2NDMxYzI3ZjBlNmI5YWUyMjdjYWIwNzI1NTYxZTJkYWUxMTIwNTAyMjNjYmY3MmQwZjQ0ZjhjYjkzYWFhZjViMWI1YmQ4MzIyNDkzMDFjNDZmOGQyYjcwMWE4YWRjOGY4NDFjOTY0ZjEyZDdmYTBlOGY0ZTEyYTM4ZmQ1YmNjYjA4OTY3YjYxZmEwNzA0Mzk3YTVjODY3YzIzYWMzOTAwNGRkM2M4OGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.08UacUFMifxEAfj-LK0i5_0WB1scd-ztv49AD0vzYLK6lUDf1iqyAP1qLhjCNiFuEdORv7S_dWmKuRYBS90Rbg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103249_69_24ca_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.802Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkM0WHNnSDVCY1hJUGN0WTRSQktHbjhlSGJsVyt0UDI0c2xYQTVZNkxjbnZvemNHMjgxTko1RXVyanR1N2djMGxMTms5RzJ0eEFmT2hKK3BuQWQ5emFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDlfNjlfMjRjYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2U0ZDFkYTU5YTljODBjYzRlOWNhMGUxMjc4NWU1OGNhNWUxMDI0NGEyN2NkMjIxMjA3NDU3MDg4ZjdmYjkyYTM4YmJiODViNTZiMjNiZTIxMWRlYmU4NTVlNDQ5ZjdiYTk0ODk1YTMxYzljMjUwYThkNmE5NDM5OTc2N2Q1NDM1MGNjN2U1ZWVkNGI2YzFmY2ZiN2M5NDFjNzkyNTIzOTAzYTc5ZTg4Y2M4MmJhM2YxZTJkZDU4OWNkMzI5Y2U5ZTk1MTdlMTUzY2Y0NDliMjMwMzU0M2Q4OWVkMTRkNGY4NTE4ZDU3OTM3OGMwYWMxMzA4M2Y5ZGI5ODQ5Yzg5NDYyODRmYjRkMjQ0OGNhZDQwMzJkNjE5MzJmNmQyYjFhOTg5Y2QxZDZmZjZhYzJmMzliMDY3OTdiMWMyNTdkYWNjMjczYWI2Zjk0Y2RjZDRmM2FmNjEzM2Q1MGQ0YTdkOTUwNjg4MzAwYjFhMTY1Yzc2MmMwM2VkY2VlN2YyZjJiZDQyZTc0NjI5MWU2ZTljNWM5MmNmYTY4NjA2M2Y3NGYxMDI0Y2I4ZGE0MjUyOTc3NTk3Y2Q3NWZhOGQ3MDcxMjUzYzFhM2RjMTQzMTMxNmQ1YTk3NWEyOWFjYmIwNDdhYmEzM2E3YzJkNTVmOGE2ZTI0MDU4N2M4YjNmNTJhNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yja_-KKQ_OLS51zPUq61efD1hxKxd23xfXu-wRtvvDUXmtanodGKWl4mkSHkgn_z_vw5TEF6FRJDVCPDfvcnsg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103249_69_24ca_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.805Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikd5TmZMY2o2eFpPNVFLU1h3RnU5NC9qNTVUYkZ4M1FOWnQ1cjh3NXlraXYvRitrdTNmajMzdjNUREZDRWZtZHExU2lrVFlYNzU3TDNLMktRSC9mTll3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyM18xMDM2NTBfODFfMjRhZl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTRkZDg4OTMxNDQ2YTFhMDkyZjRjYjBlNGNhMGRiMTQ2Y2VmZmExZjQxZTNkNGJiYWRmZjcwOTkzYWUzYjVjYWFmYjhiNTI2ZjVhMDk2MTU5NmQzZWNiNTEyYjdhMWIwZmUzZWZjNWU4NDdiNGM2OTdmZDNmNmM4MDQxMzY5ZDgxMmUzMWI3YjgzNDRkZWMwNTQ5ODAxMmQ2ZjM3OGQzNDQ1MWMyNjczZmVmZTE5Y2VjZWFkYzFjYjE1YmE0NmU2MGUwYzA3MTZkNzg1ODIxYzRlNWUxYzg3NjNjYjk2ZTU5NDhhZGIxMGFkN2NhOWQ3NThiNjA0MWU5ZTc2NThlZmNkYjFjMWEyMTA0MTRkY2U0YTNjNGMyNDk0OGU3ZTg5MDFlNTJjMGRhNzM2YmVhNTJmM2ZlY2Y1OTQ0NjE1YTQ1YWFlOWM5ZjhkOWZiZWQ1NWYzMWU4MWM3MWUzNjYwMDg3NDJjODZhNzkzNjhlYzNmNzcyZTIyMzNiOWQ1NGU2MmU3MzBkNjVmYmIyMTAyMGVhYjY2Y2QyYTEzYzA4NTFiMWNhM2YyZWU1NjUyMGMyZjIyMGU1Y2VmMjc2ZWY5YjRlYTFlZjhkNzc2ODIxNWYwNWZiOWM4OTNjYWI3MGEzNjRiZjkzMjMxNWYxZjA1MmY4ZmFkNTRkMGNkNzBiZDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UFBUfWMAhYV2ORhQ9iXFnV3NMlXJB77ppJOgQBkZGIJobtECgCj9eORosKVukJqGGdySiFu3vA_-az-mzg2B7Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231123_103650_81_24af_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.808Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlRja050Q3dWNWhRN0dHWUw3R1JDQ0RlVC9HRnpKYnpPSDBQSEVZZnJiam4vdXNRYlZZSXVBVm1oTVRhODlZWUgvWGVGb3JCWE54UTZ3a1JNVW9JVjRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyM18xMDM2NTBfODFfMjRhZl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDkxZWQ3NTBkYzA0NzU4NmY2NTUzZjRhY2UzZGVhMjczOWFkNjEwYzM4ZmRiZDMwNjQ5N2M0YjJiY2RkNzg5NWI2MjVjMjc4NThmNTk3MzhhNTE0ZTBjMTM2ODQxZmQ2M2NhZDFkOWZkNjg5MzI4OTkwNDVmZTU1MjUyYWI1YTUzM2I5MWVmMDBmNjY2MjRkNWJiYjBlZWI4YWZhZTVmN2ZkMDJlNDNhMWQyODJhODE3ODlhNzEyMDliYjIxYzUxYWIwYjlkNjI3NmE0NGUxOTU0OGEwY2UyMTY5MDkxNDdlMjQyNDM4N2E1ZDJlZDFmODYxNjQwNGEwZmRjZjIzOTBkNjIwOTcwMTdhMGFlYjQwZmJlZjg0MDI3MTBjNWI1Y2ZiM2U1YzJiZmU3YzczZDM4ODQ5OWFkOWJmNGY0M2U5NGYyMDcwMTIxYmYwMmVkYmE1NDlkMWU2NTNlYWM4YjgxYzA5ODE5OTdlZTQ0NGI5OTg0NTU3YjM5Y2ZkOWY1MGRkOTE5NWM5OTUwM2RmZmY5ZTBlN2VhNWVhZDU1YmI0MDM3ZDBlZmU0YzcxZjNlNThlYzJmNjM1YzU1Yzg3OWYzMGZjOGU0NTEyODY4YjcyNGIxOGVjOTMxNmMwY2ZlYWQxMzJiMmFhNDA5ZTA2YzZiNDc2MDA3M2I3OTNiMGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LLoNSoZFqgziQMBNvYbh8nN5NghL8Bs6yk8r6vzFpUAJlASBsOEo-pHlukZmrAgiALwEuXairQet5K57BD-ZAg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231123_103650_81_24af_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.811Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InlPQjZLV1B2QWMrbEQ4QUd2VWc0SmlZNDlZVjJFTEdZaWtKT3E4UnluNWhHMzFoelhHZU9EOXQ1alhwaDN2MlV0NWtqN2pWbEpEZjBTRXJma1lPNVdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyM18xMDM2NTBfODFfMjRhZl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWM0MTE2NjE4N2YxZmE2ZTliMDZmY2UyOWE0ZDVmYmE1M2UwNWMyZjQyYjY2N2U5ZDNlMmIyZjZkODFmYmVkMjg2NDhhYjRjNTk3NGI3MjAxZDM2NzIyNDg5MjMxODc0ZGFiMWMyZWRlZmY5MDNiYzA2OWI5NjkyYjEyNWM3YTk2NTFmNmE4YTFhMmViMWZlMTkyOWFhZWIyNzE0NTFlZGU0OTA5NjMzOTRmM2YyZjU1Y2NhNmQ2N2NjM2I0YmZjZWUyMmJkYTdjMGZkMDdjMTE5NzdiNDYzMzQzY2Y2NGE1OTBjYzFhNWI5YzNjMWExYmYzNjQ0OGQ4MWY2MzQxMDIzZjc5YzcxNzIzYWEwMmRiZjVjMjk1NDJjYzk3YjdhOTM3N2I0ODY3Y2FjNTRmOTMyYjIyMTQwODE3MmNkNTQ4ZGRkMjE0NTYxNmM5ODVjZjUxN2Q1MmMwMTlkNzY4YTk3MTY4MDdjZGZlZDlhMzg4ODM0YjY1NmYyYzMyOGVkZGVlY2Y0NzhlNjg0M2U1YWE2ODQyZjAzZmNiYjE3ODc4MTZiNjYwZDYyNGY2MjliN2Q2MGE3ZjM2MjA1NjMxMzMzNmQ4OGY5ZWQzNjNiOTAyZDQ1MjA0ZjRiNGIwMmRjZGY2YTc5ZDQ2MTk2NzBkMzJlN2FmNWM3YTM3ZWZiZTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Zcy-B8dP3sxPH02Kdq1IQmhS3CQN4iIAJH4Da2jvsmyadZTlbqUzEn54BV511bc8ksTxN8u59dPPUYqlM4_Dkg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231123_103650_81_24af_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.813Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InR5NGRjU01RdXpYcWlHVkFUeC9tK1U4Wk9UOENwVEd1KzlRNUNrc3NyajRZUmtQRlQweTZKY1I5TmlDK1FRdHovQnI3TE9RMU1GTzllT1IwT2haR0RBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyM18xMDM2NTBfODFfMjRhZl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2Y1MzgyOGFmNWM5MjI5YmMwZDQwMDhlYWM4MDRlMzE0ZTNlN2E1MDQ2NTU2ZWY2MWE3MWIxNDQ2ZmFiMzE1NDk3NDY4NTAxNzcxMDIzYjViZDQ0NWFiZDQ3Y2Y3Y2RjNDBkZTA3MWRhMDM0NDc3Yjk1OTEyZjM4ZjE5M2ZlYWMyYmM3YmMxMzVmNDFlNWQ1NTdkMjc1YzE0ODlkYzAxOGVmYzhhYTBmYzQyN2ZmMTMyYzdmOWFiN2YxODRlMWFhYTllMDgyZjUyODhlZmVjMzkwM2MzZTRhYzU3ZjlmZjEzYjgxYWM1YzEwYTk1MzAxNTJhNWViNDBlNWY3ZjE4YWJhNmZhZDgzNDFiMjNjM2EwN2E2MTkwZGMxYTA0MmU0YWVkNTk1ODc3NGE3YTk2MGZhOWMyZDIyYjhjMzIyZjJiNGU3OWVkNmFhMWE5MTliNGExZTIxNDVhMTUxOGYyOWIzZDBlYTRhYjM0Zjc4NjlmMDAyMmQ5YjAxNDBmNGQ5NzNjODEyYTc2ZTVjYzhiM2IwNDVkNDRiZGEyOTRhZDMxOGZhNjU1ZTgwODBjNjY0NmY3ZDFiYmMzYWM3YzlkOTMzOTAwZGRkZjViZjRkNDdiMDdjOGE3NWIyNDNhNDJlOTI4NWRjMjE0Nzg2NTk2YzRiMzRmYTk0ZmI3NGQ1MzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.44hqmDC1b8qLvOnzz19S0D72VM3VlZp_Og4homJ77ofQ4pdOIqF2sAgq-N75n4FrhqkwPaNzZNboDoIEi9VPGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231123_103650_81_24af_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.816Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFxd2Q4eVdqa3AwSVFhanhHSWg5Y0pLZjUrTkNSMm5BNTMxa1FKQm9JRkR0UzR4RjRsQTMxMnRSK2ZDajdnalpKbnVKbEhZZ29aWmZFYWx5Nlg5ay9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjFfNTdfMjQ5Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWFlYzNiNjJkZWZmZWRiNDE1YmFjYTdkYWI5ODA3Mjk1MzNkMTU3MDE4ODI2MDU1YjhmNWMxNGNiMDZmODQ0ZDg1MzE1YjM4ZWI4MTQ4OTVlZjk5MGIxNTkxMjIyM2ZiMDRhMDEwOGYyMDNiZmVjMDJlMTg3MDhhODgwYTllOTg5YmEyZDZhNmU1YWMwOGJhZjE3YmIyMTAwYTIzZTA1MjBkYmQ0ZDUzZTU2NTQ2ZDA0NTdiN2E1ZWZmMWJkMjM2ODE1ZDEwOWRmOGU1MjcwZTEyYzAyNDk0YWRiNjNkNWE0NzBmYWM3MTM1MmQ3NTUyM2YyMzg1NTRkOWQ2MDg2NGVmNTA1MjYyNDkxMjU4YmY1ZTc4MjhhODdjNGRmMDE2YTViM2ZkMmFmMzFjNTY5MDVjZGE0NzE5NTNlOGYwMjA5NjVmMjAxZjIzZmFhMTQ0ZmYzMWEzNmYyNjlhZGZlODFmNzdjMGY1Y2I4YTI1NGVjMDFjN2FkNDgzOWQ1YWNmMGJlMmRiNzdjNzY4MWEyZDRkYmU2MGMyNjQzYzQ0NTNlMjFhNjg4YzU0Y2M4OTE1M2Y4YTk5ZDU1MmQ0NzBlOWYwNTIwNmFlMjg0ZWU2YzBmYTA1MGEyMGY4NDNjZGFlNDJmMzQ1NjkwMWYwYjE2ZGJhZDAwZjNkMjc0NWFhZmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DTdu2itIWLmcKvfIs1qBAEKz0FwxsMtM0zufJ5YgBytBGfZQGDnSNqoNMuQoRKryLabzQI2uDFhQilTTD7uiFQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110421_57_249c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.818Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNubjgvVGZkcVlmTEVEY1ozQnd5WEFDb09seGVyN2gwWFdGZmpIQVh6YmNpcDR2dXBXTVpLY3lsa21JcFpCTE04MEtFL1hXdTUzTWNFSXp4c1hGMVZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjFfNTdfMjQ5Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTAwZDUwMzhjN2Q1MmU3NzdmN2NhZjdmYjRkNjMzOWQwZjdmOWU0NDA0MTI3Zjg5NzMxMDI3YzhhMTA4ZjNkMTdhYmE0OWJjMTgyOTA0Y2RmZDMwY2Y2YzkwMDU3M2JhNDA3OWZiMjVmOTE2YTJhNDBiOGE2MGI0OWRiMWRlMGVlMTNkZmQ0ZTNlMzQ2M2VkMzhjNjQzZjQ1YjYxNmVlOGY1MDQwNjVmYzBhYmVhN2JlNDRlNzE3NzdlZTI0Mjc5ODVlNTZhNmZiYTZkNGI0YWVlNzlkOTI0OGIzNWMwNzliNDg3YzBjOTUxZjkzYzE0NDM2YzZhYjU3Yzc4NTEyNjI3ZWUwODBlMjExN2QwMGJhMzJmZGFkZDQyOWU1MjQ4Yjg1NDI2NDNjNjMzNGI3NmVkNWQwYjZiNGI0NDc3MDE2Nzc0NjdmMmNiZjRlMzAyYThmYzdhZTg3Y2IzMmY5YmJjYjBmN2JkZTk3ODMxNjE5NTVlYjdhN2FhMmZlNDk0NDU2NWUxNmQ1NzFjMTVmYmFjMDhhYjM2MGU5MGYzMmRjYmU1Njk3NTQ5NmViZTRmN2U3ZTUyMzUwMTE1ZjE3MWNkODhlZThhYTgwZWVjZjYyODJjYTg5ZTAyZWY5M2ZhYmY4M2FiNDI4Yjk4MTBhMmI3YzdjZDdmMjkzMjA0NzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Q5gOUsx5LCBO-dZP65EEzmyW-w7VOPZXHdOXWrvOAp2qVJ2Q0eETCEIpMcWOSeVdXo-QAS9zfldPY8u8Ftqs5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110421_57_249c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.821Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJ6ZjIzYXp2SzFabzNmS215VituK2g4K0dNYThEVFpRbTNYYWVOYWlKN3IzUStXNGFPN0Nab1drNVFXUUc1Q3VxRVRoMWVGa3RvMFNzc2VHZEQzVG1RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjFfNTdfMjQ5Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjBjYmJhNTA3OGRjYTMwYjgyY2ViYzE1YWQyNDhiY2RmOTUzNjM0MTg3ZmExZjZiZDE2MjE0Zjk3NTY1MmFhYWJlOWVhMTJlYjlmODllY2QyMGM2ZTM3N2Y3YzBmMmMwODdjYmM3NTA0NDc5NmY4YmEzNmY0MTJkMWIxMDg1NGZiOTBlODBhMTI5NGMzZDA2MWE4ODUwM2I2NWQ3OGJjNWMwNmM0NGYzMmYzOTA4OThlMTY3NjU2MmUzNGU2ODVmYmFmM2IwZjVjNWQxZDliNjg4YjExY2M4NDU2YWYzNzBhYjIwNjg0NjY5NDNmMTEyYjRhMDIxZjA2MmIxZjdhOGE2N2EyODVhZmUwZGQ1NDQ3N2E3Yjk0MGJjZjIyYzQ3MTM0YzExNjAzM2ZjMTc3YWQ0NzAwOWQ2ZmQ3NDlkN2ZhODkxOWFjNzc1MzVkMDYyYTViYTZmM2NlODNjYTU1ZGY3YWE2MGMyMDNmODBmMzEwYTdhZmQ3YWQwYjM2YzBjOWYzZGVjYTZhNzU2ZDRkYTJhYzQyMzZmMzJkZDZjMmJiZDk0OTIwYmVlOTFlMDNiOGEwZjMyZmIyMmIyZmIzMjRkMDAyM2VkOWE5MWUwOGM1MzI3YTM0MDMzOTdmZGJlZTZmZjk3ZWQ2Y2M3ZmZiZmU3NTU2OTA3OWQyNTEyYTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8TXqOV2uV6utQsG1ZS1sfotFYei2Mk6pPbJ1XCi7CbBqNWCRgq6UeOn8ZQD21Sb9aTPa6Iv4AUUoJfqfz0VLrw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110421_57_249c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.823Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjluK2tPV1B2cExkdU1HdEVkWW03dDBKdWpVRXNVMm8zajRsNHV2K1dmbUdqVGRwd2NTaFpIMWMvd3dWZXdxQmhDQ01mNkJENy9ydjJHL1kvTFJBdllnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjFfNTdfMjQ5Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjEzMjYyNWFiNzc4M2M4OWQ0MjllMWI5ZGNkMjBmN2UxYzhiMWVkYzNlODI2OWE2Y2Y2ZTZjNWFjMmI1ODRhYmRkMjUwMzJiYzIxNDQzNTNjZTA5YWE5MmJlNDhkNDVlMjhkNjA2YjMxZTkxMjg4MmNhNTI2MzdlZTE0YjhjZWQ1MDBmNTM1ZjM4Y2EwNWRmMTFhMTljN2EyNWZlZWM4NWZmOGFmMjgyZDZkNjM2YmE1MzM4Y2Q0ZGM0YWJkMjQwNTEyM2UwZGEzZjRlMmQ1MmQ4NDJkYmE4NDk5MjBjZGM3NjkzYTgxNzZhZTY5N2JlNzFiMDNmOWU3Y2JjNzQyMmE2YzUyYzA3YzY0MGU0NjdlYTkyOGE0ZTViOWI4ZDNjNzY3NmM1ODZhOThkNzRhNjAxYTJjZDAxYWY5OTU3MmIxYjc1YTQwZjc2YTRjYTg5OGY1MWNhNDE0ZWRlYWM5OTk0MWRlMTQ2Yzc4NGUyZDliY2ZjYmEzZTcxYjFhM2ZiNmRlMjI3MzM0MmU0M2RkNmNkZjFhMDliZmMzZDM5OWY3ZjgzZTU5MTE4MDc2ZTFlY2UyMTA4ODdlOTQxZDFhYzBiNjFmODJlNTU3Yzg5ZDJmZGJmMjkxNDk1NDRiMWRjMGRlYmYwZDE2MTVjNDI0YjcwOTcxY2UwODJhZDU3MzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1KP_opvOa_1_kb6PTiNW0iZblwru-t2jyHxYBOSZN8vM6N6pxecjznJhEAC1yur3Ckysd4xteo7Fj5VUzNQaMA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110421_57_249c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.826Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZQSXg4VWI3ZjRDVTV5LzRPZHdHRUM4OU9QcEoyZmZLcDZZUHBLZmlwalVZeDVVUnNkWUp4RjZBVU9UamM0UDNDSzJ0ckFodmszU2lQSkRkbERGRExnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MDlfNDZfMjQ1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWIyNmRlNjc2YjFhOTI1MmZiYjZlMWRmYTE0NWM3YWJlYzYxMTZiMTVkOGU0MWU3YTFmMDVkNjE5MTg2MTRkYmMwNDlkNjk1ZGQ3NGVmYzA4MThkYTYxM2EzMTg1ZGUyYzAxNDUzNDEwN2ZhMTY2MzY2MGFkNDMyZDU1ZTU4MWRlYjA0ZjhkN2E0ZTgwMzRmNDAwMTFiZjQ2YWM1NzhiNTExNjQ1NzliYTRmMWZjYjg1ZGE1MWE0ODNiYTcyOTk2YzZjMDQ4NDUyZWQ3OGRlZDExMzdhZmQ1YTY2NDI1MjgxMzYwYmY2MTcwMGQzNWU5ZDNmZGE5MWYwZTA3NDJkM2NkYjMxZTQ2OGMxZDM5MzE5ZDE1YjdlNzk3OTg0ZGNlZmViNWEwNzcyZmM4YThhODhhMjJiOTBjNmMyYWNlZGQxNDZmZjlkNTVjOGY0NzYzZmI2MjM5MDc1OGE0NDMzNzBhZTUwNTQ3NWQ3NDYyMGExODExY2ZjMWM5ZWFhZWY1Y2FmZjIwOTA2YTI2NWVhNDUxM2I3YTMwZjk5YmE5MzFjNWRiYTQ5ZjBkYmFjYTdkMTk5NTdkMWQzZjEyNGQ4OTY5OWY3M2VkYTY5YTIwZDIwZGIzNDEzZjdhZTgzZWJmNmZjNGI1NmNkODUzNTVlNzBiMDUwNzVlYzYzMDg0YzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qUADDdRiNM1CnRN5edMYoNiZiBZcOVmoNh2QP9UpdzLPUyjVkvaUYN5AGf8Ut0NdywUhlzf8EzZaGdhXdEGekw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102509_46_2451_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.829Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlU0MUl1MXF5eEJqenJjaDBndGtFRUlmN25QSEtXTXlOZXZ4QWpVNUpCNWZ0OHc5UllvclVNLy9Pc25rNm5ZVmNiOSs2eDYwbi9XcHJYSXl2cFF0R253PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MDlfNDZfMjQ1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTcyNjRjMjcyODk1NDdmZGNlMzczOTQ3MDU3YjE1ZDYxYTQ5ZDlmZTc5YWQ1NzYyMzQ5NmM2OGEzYmIwMGFmOTQ2YzViYzg0OTAxYTNiNDAwYzFmYjc3YjVhNDE1MmIwMThmZDI3YjYzNGEwOWIxZDZmYmJjY2Q1YTMzYzU5OTJiZjVkYTJjMWYzMjM1OTMwZWFmMjJmNWY3NTIwNjU2ZjY1YjIwMTJhMjc1ZDljNDgzNWRiYzRlNGUzMTczMTI1NGU4ZjA4OGIyNmI4MjQ4NzFlYTE1ZjU2ZDc1ZTI0ODllNWYzYjU3YzY5NzY2ZjI0MGEyMDkzMjgxOTlhYWY0MWM0NTI2MGU0NGFiZjY1ZmIwZjk2NWNiYmEzNWUzMjE3NmRkZDIzMmZlMTY0ZjkyYWE5YjM4N2RiZDlmZTRmMzliMzM1NDgwYjZhNzIyODlhYTM5OTQ1MGM1ZmYxOTcyYzI1MzU3ZWEyZGQ2MTM1MDA0YzAyZWI3ZDliYWVlOTdhMTkxYmIwZGYwOWVhODNlODE5MWM0MGU4MGIxYWJkYjIzY2VhOWIzZTQ4OTAwNmQzOGJiMDY0MGUxYjI5NmQzOTVlNjgxNjA2Zjg0ZDhiNDU1MzMwYzFkNWExM2RmODFlZjM1NjY5OGQzNjEzOWRmNzI0MmUxYjE1NDU0N2M1NjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6jgG4YMuapnq_hYUC6sFKzPn8-ADYgqwbywWY8VZF-MeUFgRezo-DHSVExhUb9XcwBInz8vLlmdntLAascDXJQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102509_46_2451_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.833Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlZEMW5RVndEdEx1S3VmZDQrZkRvRlB1NDNvU2w0TWJYdjVtWXJUeTFMRXlUTlY2QTd2R045ZTkveEMwL3RrMHhsc2NFM2srWWI3SENOempCOFpnTnpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MDlfNDZfMjQ1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODdiNWFjMTMwMjAzYmM0NzVmZTVlMjk3NDNhNTJlYjA0ZTVhMzE5MDM1OWM1ZDJkZTFmMzkwZmZlZWE5Yjg5MmFiNDFmMDU5Zjc0YTIyMDEyZjcwOTliN2RlYTkzZGFlMzNlZTYwNjJkMjZlNTUxZTliMTZlNGQyZDA1MGY2NjQ0M2EzZmY4ZmJjYTI4ZWZmZGQwNGU0M2JjNzIxM2VjZDAwMGVkZjlkN2NjNjE2OTFlNmY4NGQ0OGFiZjhjMjYxMWRiYzMzOTdjNGIwZTc5MGRjM2Q0NjQ0YmEzNDMyZmUyZjdhYTlhYzBmMzI4MzA5ZTZmY2RlMmMxOTI0YWViZDYyZWVmZjBhYjJhYWM2OTE2Y2Q3MmY0MTc1Y2MyMDJhNzM3MzRlYTJjZTAxMjdkZmM0MWFmMjc0ZDk3NWIzODg3MjY2NjJlZGE4Zjg1ZTI0YTAzOTRiODRjYWUyNWRmYWZkNmZjMGRjYjU1MzY1NWJlNmE4M2ZlMmQ2ZmU3MDI3ZDliMGQ4YTkwMjJkY2YxZGU0NDM0ZDFkMjQxOGZhMmFkOWEzY2RlNTdjYzA4MDJkNzkzOTE3NGMxNjJlOTBmM2ZhOGI1YTFmMDRlMWFlNDYwMjljNGEyZWFjM2IwODhlMTY2YzNmY2UwM2ZmY2YyODE5NTZlMDdhNWRhM2UwYTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.e9LaTyxS1G3bpDLIzr_1buEBREwoucjrmmsKLRxfYvgxv59E6C_QAhMXY3Q6I-iLlSb0puEN-CUo26CVrMHxJA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102509_46_2451_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.840Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJValFtcDRPbHA0NUlYYmsrMHJwdEJKMFRNUlFZb2VkaU5JQWxTUjJBY2JRMCsvaGNUVkVGVk1kK1g5Q1pkL3E4L1B2MVhEK0JGdkVVaGlFTTJlZUt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMzMV8xMDI1MDlfNDZfMjQ1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjVhYTMwZTA3ZjM5NjI4YzA0OTVjMDQ2ZmNiMWNiMGM4ZGFiMDAwN2JmMmYwMTVjYWExOWNkZjYyMzQ2ZWM0ZjgxMjRiZTdjZWE4MDFiZjJkODEyMTBhMDhkZGRhZDgzY2EzYjZiMGI4ODI4MTRiOGRjY2U3ZTQzYzRlMGUwODg2NmYwYTk1YjI1ODVmMjNlOTBjYTdjM2VkZmViYzA5ZDdiMTI4YjliODliMDQ4ODY5YjIzNjczNGRmYTg3ZTllZTdmZDMxOGU0MjJkMjU4NjhjYmFhNzBjNjQ3ZjNiMzNjZTE4MTliYTlmYThjNmIyMTAxYWUzNjJlNTBjZmJhMjk1NDQ1ZDA1ZTgxNWE2OGYyMjcyMGJhZTBkMjA4ZDIwNjI1MDk0YTAyN2RkNjc4N2M0MTU0MDg1Y2JkOTlhODE2YTBlZWJhMjIzYzU5NGU1YTVjY2VmZDJjMzE0NjBlZWYwMmJlNDUyYmQxZjcwOWI5NWRiYmE2MjhlYjJjMjBiNzNmMWQ2MmM3YjE1ZjMwYjE1YWJkNzY0NjFiNzMzZjJkNzg0M2MyYmI1MjhiYjcyZWQzNjM4ZDFlMDg4OTI1OTZlOTQyOWJkZmIyMDBlYTM0Yjc2NGQ4NmU0YzkyZWI1NWQzZWEzMGJjZjdjM2E5Nzg2NGIxNWM0OTEzMzJmZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZqTN0ni7gP30BqhdKL508PE_lQajTusM1SGsTSNA5T487DBdiikgPtLfMXwgVJ0bL1XhqaBOOtVDEms34ZfsaQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220331_102509_46_2451_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.842Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Iis1dWVxMDBYNFNrUjdxdTluQWdEV1FFUjVvd3RoeitOQk9HRjE2YnRmVkxkeHZJR2FaY09vUWFmQldQVnVtSXl6RWsvbFRwNmtLSW44SkpHUmYxWmRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NTZfNzRfMjQ0MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjI5NWNjN2RjZDYyZmU2NzZiNjhhNmI0MDlhNjA4NmI1ZTc0ODY3YTgyMGE1ZWUzNDUyYWFkMWVhNWJkMzU4ZWQyYjk2MThhZjg5YjA3Y2FkYTMwMjY3MTgwOTgxY2NiYmE4YjMxNDBjYjQ0Y2Y3N2Y2NWE1Mzk2YWQzODI5MTI2NDM0Yjc5ZmUzYTQyZmVkYjBiYTZlMWJmNGUwMzMyMmU4M2IwYjcyMTBlODUzNTFjYWM1NzZlZDhhOTFmYjY0ZTYxYTYxZjIzYTgwMTQ3MzNmMGY3OGYxOTcwY2JmNDgzNjZmYzdmNGViNDg3MzhkM2QwYWNmZGY3NTQyYWIzYjk3NWE2ZjI1ZWYwYTE5YzE0N2Q2MzIzZDkzMTI0YjI5ZTc0ZTM3ZmFhOGRiZTVlYjA2OThiZTFiMzVjOTRmZWZmNWE4YjIzZTg0YjVmZDczOTM4MTIwYTNmNmM0NDgxZmU0NTllOWI2NWMwM2JmZWY4YjhjNDdkODI2NzkzOWMxOGZhNTAwYTNmNDNlNDlmMjM5ZmM3MDcxZWJlMWViYmRkOThhNmM2MjRiYmIzOGFmZmI3YmEzNWI3NGM4OGJhZGQ1OWI4OTU5MTVlZDQyOTE2ODViMzNjZTYwZGVmOTM4YjBmODIwMmE5YTQ1NzQwYzc2ZTBmY2ZlNjdkY2YwMDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.VlDE0RS0KPlDBgi4L5mPmjteWa5V4E337cKb-lFQzMpl_-vsL6rEIeJHcn9X19r2tAjMPFWKoce1ULATX-NYCg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102656_74_2440_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.845Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IldFdzFhNDA1N3dVZWdMZi9pTnJkMVBCOGVveWNGZitTelRDakJTWVhLT1ZmRFpjZ1MrWkpXdzNsNVdrOThTeW9PMk94TnYyNThWWk1KZ2tJNHlGb2x3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NTZfNzRfMjQ0MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWE4MTIwMzIwZmRlNGZiNWRhMzgwNTAwZjM3MDk4N2I0NTViNTE1NTJlMDJjMWNmYzQwMGRkYTI1ZjlhYzJjMmFiYzlhNWZhOWM3ZDM5ZDA4NWFlMGMyNDQwZTU5MjY5MmIyNTU5MGE3OGI0NTNkOGFlMWRkODE2MjA1NzZhYjk4M2RjNDA4YTkyNzYwZGQ1NjdhZDY1MzcwODVkOTAyMDA5MWZmMWNkOGNhNjg2NmQ4YzMwNDRkY2NjMzcxM2JkNWI1MjUzNDk0NzIwOWQyNmNhYWMzOWFhY2M1NWQwNjNlYzlhMmFkYWVlMDU1MzkwZmE1OWVkNjA1ZWJlMTVmZWRiZGRhZDljNTYyMTMwMThkZTM1MDc2NThhMGI0YzllNTU4ZGZhZDJjMzg5Y2UwMjZiNzZhYWJmZGUwODk0Y2MwMjdjZGFlMWRkNmI3OTZiOTEyMTlmZjQyY2UyYjA4YTFhZDk1ZTk3OTA3YmZlZjYyYjQ1YWY5NzYwOWZjYTdjMjdiMmE1ZjU0MDk5MDQ2ZTRhYjdlYWZjNGI1YmIxOGRmNDM1Y2QwMjI5Y2EwYTdmMGQzNzA0M2U4N2ZhYTA2YjYyMzYzN2QxOTA1ZDM2MzFlNDRmNDUxNjgwYjkxNjY5M2U5ZDQyY2Q0ODg2NzY2Mzg0MmVhNmI3NGQ0MDYxZDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GCDM1iL7sT9WOqnaVa_QtsCkKieSBVa1oHKWL5yQPoVIg7XphSdkURGzTD1OUCiQq7tTC-45T6JbJDYzLSJD1Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102656_74_2440_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.848Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9SVTFBNGJNZjRBOGt2S2doZGtJOUVid0FrWmlmY25TNlFpTFRhN21PdW9qL0VrWHh1aDJJaGh2QmpXU3ZGblNXUVJtSHEzcjBrMUExZG0rOXJaMXJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NTZfNzRfMjQ0MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDRiOTNmZTg5NjRlZTE1ZjlhMTdkODg1Y2M4ZjQ4MzIzNTQ3M2JjYzI1NWJmYzMxZGIwYzFlNWZlMmFhMzNmNTdkMzk4NWIxNTM4MTQxYTA5ZjM3MjEzOWI4NTdhNzE5ZGU4N2I1ZWM1N2QzNmFjZWM2OGRkYWJiMDA3MjdiOGMyYmExY2U5ZGY2OTk5OGViZjdiM2U0OTJmYjZlNzBlMzMxN2U1MGE4N2M3ZDVhMzc1ODVmMjZiYjA3ZmZjNDJiMGMwM2MxZGJlZDBiNjg1Mzg2NjRlZjJmZmRiMTU4ZjcwZTk0ZDMzZjczZGY3NTIyMDA4MGRiNzM5MjlhZTAxOTM5ZmNmMTU0OTU4ZGE4MzEwYWY1NzliMWRjYWIzODgwNmI3ZjQzZWNmZTcwNDkzNGQxOWYxMjhiMzkxZmRiMTk4NjdjNGE3YTZjZGExNWFlM2QyMzYyMzcyNjVmYzI4N2E0MTIzNDcyMWJhZmU1N2NlZjNjYzEzMDA2NWY2ZGFlNDU3OGNmZDNkMTRlZDhjZDUyMDk0MjExYThjOWJkNjY1MTYxNzY1Y2EzM2UzN2Q3MDU3MmIyYzQ2NjgyMTc0ZTkzMDczN2I0NGE5ZWY0ZDkxMzg4NzM4NzNkMjQzNjNhYmQyNWQwZmFkMmI5N2ZiZDQ3ZjVmMTUwYTliMjIzYmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8RwnowboVaW-o3vWLWZjg35rvm1DBQspRYzpKSxArpbGqeGPg1u-0wJUjdNfYR14QhCBM0lg3rlFrqja07EUxA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102656_74_2440_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.850Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjM5TG1HUGd2NnNwOVJqc0VRc09XYVc0bzdIM1ROeFJ5eWdiOHV3UDY1M2hGejk1ZnZ1SWlUWU1XM2M1RGFMMnplWkdUdzVsODdmOG42TzVtSG56UUx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NTZfNzRfMjQ0MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmUzNzEzN2JmMjViNTM0NjE1YWFmMDgzNzRiOGM5Njk2MDVjZTEzMmQ4OThmN2Q3YTllZDM5MTg2ZDlmOGNiMDJlOGU3NzFhZmRlNGE0Zjk1NGU4NDY3NGU1Y2NhMGI2ZjcxMmNlYzA3NmRkYmVkMjI2NTVhYzY5ZGU3ZmVjODk0MDI0MzZjYjRkYzgwMjY3NzQ1ODc4MzBiNGUwOTU4ZTIyNzkzZjlhNWQyN2NkNTJjMWJlZTY3MzZkMWU0Y2NiYWQxZGQyZGVkMTgwMzg0NjQ0NTk4MDk3MDMzYTcwMGVhNjg1OTZjMGIwOTAwYjJlM2MxYmUyNDdlMmUzZDlhYzc4N2U4MjEyMTQ2NjA2YmUwMWI1YmZhNjcyOGQyNTlmMjMzODVhM2E2ZDBiYTg5MTQ2OWMzMTEwMmMzN2NiOTRmZmFiMzNjODFhOWM4YjI0OWU1MjAyMjNkNmE0ZWJjNTRkYWRkZTYzZTA0MzZmZTVhZGMyNjVjMTdkYzZiMmI3MmRiMjMzNTQ2YzA2NjNkZjg0MGM0YTU1MTExMWFkZWMzZTk0YzJlY2IyNWQ5ZGM3NDQxYWMxMTlkNjU1MGQ1NDY1NTE4MTZhNjNmZTAyYTk1NGVlMmVhMzgyZjU0MjNkNTQxYmNlMjNkMDIwMzRkNWI1NzQxMDE0NTVmMjFiMjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eOIUoPtnszcX-GKuhlbL9AbbZTxQckJ9OkV8-M8kbc4MeBX9CCvmhKuLpUehO3OxzQwy2NgR3eTEQDkTJ7jsuA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102656_74_2440_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.853Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImFHdWQvYUtMemo3YStYRXRqVW5sUDMzMzM3ZkdMam5yNlBZdDlLTXp1eGRZSjBHZm1WMENUaDB0eUdETzMwMnRBOWw5SW1pVkVRVHNqSDdJN3NvT1F3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzlfODZfMjQxNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjgyNTAwOGVmYjQwM2EwNGQzMDYxYzU5OTFjNmM1ZTQ4ZjMxMjY0NDdjZGQzZDk2ODFkZWU2NTBhZWFiM2U1ZjdkZTM4MDI3Mjc4NjAzYzk3MjMxODQxOGQzNmRkMjgzNWNhZTU5NWEyNjQ0ODFjOWI0M2Q2MmRjOTg5ZWE4ZmQ4YTNiYTcwM2Y1MjNkYWFkZGJjYzUzMWMyMzJiOWVjZWZjOGQ0ZDBjMjJiOTRhNTAzMWY0ZjQ1M2M5ODc3MzBlMTcwMTM2ODIxMDQwNTcxNGE0Nzg4OTE1MjdmMTc2MTZjOWM3NTAxZWRkNjE3YmU4ZDQxNzgwYTJiMjM4NTE0ZTdiODFlYTM2MmZjNzc2OTAzOGQyOTZkNmE2YTcyZmYyYjNhNzk0MzgyYjgzODVjODg1ZDNjZDQ1NDUxODk0ZGYyMDdkMzMxMWY2MmM0NTlkOTJjNzNlODY2NTJjYTcxYTJhNDBkMDUzMGY4M2JkNGNmODc0NWE2ODFhYjhhOWZkYWQ1YTg2YWJiMDUzODQ0M2Y1ZmU0ZWVhNzExMGE2NjU5NGRiMmEzZjFmZjdkNzdhNGEzODY0ZDQ2N2EzNDQzMTIwZTJjNjE1N2QwOTk0NDEzNzA1NTgwZTRiZDI4OWE5MmIxMzhjMGNjNzc5NjBlZDhhMTA5ZTFlZjcwNDg4YjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cPflhpbqERiAmL6K7nG-RtvfcA-K-G6zrWZVBdpgFfWglgWGmh5KGVGTDVaw0WIG0aBxu7byrrTNNZKRmigfsw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112739_86_2414_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.856Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Inp5ZVh5WUkxc3BTQWYzNzN1QUY3eVBkMzRGSC9TdUlDdG5YWE0wRHdNQWtBOWQ3YkkzZDRnSmtKdzdhSE1pZ2xTSUVmSWhOSjI3d3NaTlh3a1l3aFdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzlfODZfMjQxNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OThjYjk2MTM0MDBjOGZjOThlNDI4NGQ1ODA0NjA0ZDdhNGIxYTJlMjFhMzczMGUzZGUzMzZlNmY3OGRkMmRkMWI3NTA5NGVjMWM2Y2FkMzMyZjFhNTNjMGFkZTM4YWIwNjg1NjQzMDkxNTZhYmIxNjQ0MzQ0MWNhZjUwYTczNmZiM2U3Y2NkZjYxNzA1ZjA4ZTIyNDBmODdlYWM2M2ZlY2QyN2UyYmQ4MGViZmUzNzE2ZTY4OTA4ZjBkM2JhNGYxNWZkNDgxNTZhMTZmYWI3YjBmY2E4ODk4M2JiNjczN2RkYjdkYTEyN2Y4OGI3ZDE0MTY2OWI2NTUxZDA2MWY3ODU3ODY0Y2Q4OWRkNzNkZmZjNjc4M2NmNTdkYWQ5MzYzMzM5MjUzODA1NmIyNTcxM2FmM2IyMWI5N2MyMTc3YjdjYjc0MGIyNWM5ZDUwYTliNzZjOTUzNGIyODRhOThiY2ViOTYxYWUwMDc3OWNiZjc4YzM4NzVlZTk0OTNhMzI0ZWIxMGYwMTllNWM3YWJkMGExMzBhOTUwZDFmNDZjZTc0M2U0Nzc0NTQ3MTFjMGMwODI2NjBhOTQ3NmFmNzk5MDM0ODk4NTM1OGQ2NmM4Zjc4MTkxODI4MDA2MDNmZGNjMTA0NmE0YjJhMWE0MjRhYjg5MmUwMDQ5ZjFiZmU4ZWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.kofF53YJihiAj4O624GM7p_-hU9nrF8a1da8VBUnxt2NhgWpAwHAtcWKNV2EGaHGSCeXF4-3LOYLK2yGzljZtg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112739_86_2414_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.859Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjVFbVNXNUpFR3JpRHZaK2F6ZWdKN3JsRTVhcjA4OGxXZjRFanVZb2x6bGlDU1d3Y2NLVTRtY1dJMzV3dkpHVDRSV0IvMzViYnBHTFNWNzFmNnhvbjJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzlfODZfMjQxNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzJkNzMzZDc5OTM3YzYxZDNlMTFmYzM1N2Y4NzQ0NjU2MDQwMzJkMDM1MjdjNDMzM2E2OTlkNzk1ZTI5MGYxMDRhNDg1OTQ4ZWU5YjU1MTgwMzViYzBkNDY1MWZjMGM5ZDRhMjA4NGQzY2QzZTY2MThkZTljOTkxNTI3ZDk5YzYxZTBiN2Q1MzFjMjJlOWZkODdjMmIxN2Q0NGQwMDlkZmY3NWIxMmZkOTBiNzFhMzhkZDBkYTY2NDQ3MzdlOGYwMDdkMGU1NjEwOTk5MDY2ODgwZTQ1YzBjOWE5ZGJiNzM1YjZjMjMzOTEyY2U2MDY2NDJkMzhmYmRjOWJjNzdlZjEyNzAwN2UzNTc1Mjg1MjIwNTYyM2U1NTc4MmFiYzg2ZDUxM2QxYzY3M2EzYjk0YThmN2QyYjc4YWViMzgyZTM3NmQyZmQwNzM1ZjUyOWVhZjAwNWQ3NzEyY2NiYjQxZmU2NzBiZmZlY2ZhOTQzMWNmNGYzZTdhNjk5NjU1NmNjODFiMjk4MzQwMTY5ZDAxZmY2ODE1ODMwYjUxYzkzNDNhYWJjNmVkMWU0OGI3YTk1Y2MyMDliZDFlOWM5MzZkMTIzNmMxNGI2OTliYTgyN2Q5ODM2MjFhNGIzMDViOGZhZmRiYjdhZWY4YTYwMGNiYTNkNzdlNzM3ZGQ1MjU0NzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hnvLxnvT0wqLYjbpP2uyxfs209LNpqqW1Ztky1Lqz6lOQAblPz3mVCmYR9QmcCOb5TORgvZo8hfPQgwBOYm_NA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112739_86_2414_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.861Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlpvTUFNOVFnMHM1QnB3RTRYV0xTbElOem1uNzdvL25Gbzg4ODliakhxR0VZNEo3REppQWpTV2t3K1MvcURaNXZzS2R2Vy9UcFJjWklwTzYzZEI1Uy93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzlfODZfMjQxNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzEyOTllNTQ1N2UxN2JhODI1MTc4NmY0OWYwYWU5YjRhYmViMGUwMTc5ZDcyN2Y3MmU1YzcwN2I2YzU4ZGZkOGZjZmNlYmViZjAzNGVlYzNiZmRlMmFmZDJjZDlmYWUxYjBjNTQ0MTk1YmZkNDA0ZThhMTc2OGQzZmY0NDg1YTZiMzc2OWIzMGI5ZDcyN2U5MDc0ZWQ0ZmRmZDM2NjdjZDE0NzgxNTBlZTQ2ZWE4ZmI4MmI3ODMzMTIyMDk3NTU1N2EzNDY2OTExNzZiNWNmMTIyMjBiZTVkODVhN2VhOThjOWJlMzY1NzQ4ZjdlYzM0YmRjMjE5YjM0M2VkZDVkYjU4MDFiNGViOWQ2MTJmYWM1ZWQxMWMyMjU3N2E2MGUxNjg5M2VmMDA4YWMzZTQ1NWY3NjM3YWQ0OWIzNTkxNTAxZDg0ZDM4MWI1ZGJmOTUyYmJjNDUzNzgwYWZhZDIwODAzYmE2Yjc5YWJkMzg0M2U2ZDE1MTU2ZjcwOTBiN2VmNjM1ZGZjZDMxN2I4NzIxODY4OWRhYTgwMjJkZjNlNDMxMjEzODFlNTE0NjQ3YzY5NDQ1OWE4NDk2ZmRkNjAwNjEzN2JiMWQyZTQyMWViYzZjMjZhMTJhZGE5OGViMmZiNTg3ZGJmOGNhMmMxZjQ4ZjJmNDU3NjcwZjYyMTcwMzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mg6HEkXASkOdFsJdGTcDGMX07A-c09Mi9tG8IMjIyXIwE6CPrRqYvaub0k7pROlzwj095vUgDXJlHqNWl9pFrQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112739_86_2414_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.864Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImgwcDh2VkdlckhIanpoRlQwMVl2TGZhcDJ0cFQ2R3hRWTFDem9QNFRnNkZzV0lNS1BNVnpMS0RZa1hHa0lzbXpGZzZWanlmNWtzM3ZhL2tmQmFPTFN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTFfODlfMjRjNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTE0YzY5MDFhNzllN2U0ZmU1YmNiODFlZGNjNzgzNDY4Mzc4ZjNkMmUzMzMxOTUyMDEwNTZiYjNjMGIwMTdjNThkNDhmMTc1YWI5YjZjODk0YzVkMmE4MTY1MGUzYzJhODU0OGU3ODFhNDMwZmZjMjYyYWI3ZTQ2MDJjMGY0ODE2NThmNjU5ODYyNDc4OTE4MTVhNTFmZjQ5NzQ5YTgxNTdkNTQ1NmJmYjU1ODU3OWUzM2E3ZDA2ZDhhN2FiZDE5NmZlNjAxYjU3YWM1NmRiNWRkMGNhYjA5ZDVmMDI1NjY3NWNmOGM1ZjBkZTM4YTgxZGEyODEzNmFiNTQ2NzhkMDJjOGQ4MTZkYjFlY2E4YjI2ZWIxMWJhNGE4ODQ4Y2U4NmIyOWMxNGJkMTZlNjQ1MDYzZjExNDE4MzgyMTI4MTk4NTA3YzI3NWE1NjJhZmY2NjEzNDI1YmRiMjhkZDMxZTMwYWFkMTM1ZjQzNDM0OGQwNzYyNmYyZjJhYThjNzEzZDI0ZmEzOGQ1ZGFhOWI1ZTYwMmNmMjc4OTZhMjAyNTg2MGUwOTg4MjE1YjA5NGQ4MWIyYjQ3ZWYwMmFkY2MzZTFhYTY4ZGEyODdjMTliOGYyMGQ4ODk3YjMxZjg0ZjQ1YWUxY2Q4ZDcyMzQ1ZGU0ZDllNjEzNjU0YjgxMzFkNmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ipmdezSm8NqgXydMvk6gpY7lUTGVFmAQ0gzsj9mgaTF_RNsN2v078a9yXg65SZgYyqWYctc31jI_2hJnaTyiaQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103351_89_24c4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.866Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVTc0N0dVBSakZPQ1RvdnFaMVUrQTUvQWo0S2JBdmVXOFlQVWxseDZuRjhQQU0wSENBaGhmcUltVUVKbVBpY0RMQUpYS29JUlpsVVQvaUxzcCtYL2FBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTFfODlfMjRjNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGU2ODI1YTczYWMxYmE1NmMzNzBhOTcyOWQ4ZjgzOGRjNDRkOWFmMDVmOTZhZGFjZjRjOWVkZTI3M2FiY2I3ZGIwMDAxNzVkNDY1OTMxZDZkOGEyMTQ1NGI2NTA4MWFkMDJlMDczNzJhMGUwYzZiMWFlYzZkOWMzOGIyZGQwNDUxMWFiMDdlYmJmOWY2NGRlYzU2NTY2MjVhZWRiNjE3MTliNDY0OTg0ZGI3NTg0YTlhZGRhNTc1NmE2OTg3MjY4NzEyNDY0NTZlODkxN2E1NzFmZGE5MmUyOTk2YTU2MWM5YmI2NWVmYmUzNjQwY2JkOWYwMTRlMjEzZTIyMWNmMWUzNWM4NWYxMTc3MThlMjQ0MGI4NzI0NGQzNGIyMWM0ZmNjNTA0ZWMyNmI3MTNhNjBmYzUxZDE5MzVmOWI3Y2RmMjI1NmUyZDQ4MTAzMTcyOGIwZjJkNzRkYWFlY2UwN2NmZTUxM2MxMzk4MzExNTIwOTU2YjVhNDNmNDYxM2FmODI5OTJhOWZjNjcyNGYxNWJiM2NjYmE4MTcyMzI2N2Y4ZmVjMjhjOGFlYTM3MmMyOWVmMWMyMzAzNWQwMDdmZTZiODI1MzRiYTUwOTcyYTFlMDA0OGU4M2JiMmFjOGU1ZGY2NWEzYzM4ZjgxMDFmNWUzNDQyYzEwYWI0MmM5NzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TxFpW6kRVlChM8f0OSmb9Sga2FYMVNlE_bPlLKAXOUvK_tdBxR_LIej8N9sZrhh0Qh6lqE5476LrRI4sLfFW_g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103351_89_24c4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.869Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkIrVWpKSjBFSnltcFRvcjZzTFdHaEVrU3F3ZzJ4cjZxc0tNOS9zYXpna3Z3TER4VXRsZGZPNVhxR0phbkZmc29CZ3V5d3ZOTTh0TkJGNXowVkJqemtBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTFfODlfMjRjNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzdlOGYwODVhMTQ5ZjMzMDcyYTkyOTEzYzY5YWM3ZTYyOTMzNzc0YjQ5NDRhZWIxODBkNThkNjNkNTIyYTg1NjQ5OGQ2MzJhMGQ0ZGQwNDExYTM0ZjNhZTZkMDA1ZjM3MTYzOTM5N2VkMTQzY2ZmYWMzNjBlYjczOTdmNmVjYzQwNDAzMDE2YmIwY2Y2MDk4NmE4YTgwMzAyMzFkM2U3ODEwODZhZWU1OGRlYjIxNWEyMzFhOTUxYmE1ZGYwODU1MThiOGRmZmIyNWQyNmNjMGI2ZmRkZjk4N2Q4OWZmMjA0ZTljMzQ2MTk5MDE2NjgwMTFkZDJjNzEyMTFiYWViOTdmNDdhYTBlOGMxYzVmZGYwZWQ1YTg2MTlhYWY3MDY5YjFlNWNmMjFmZTc4ZjliZWJjOTAzYThhYTM2ODc4OThiOTFhOTZmYzAyMmIyN2ZkNWZkMmRkNjA5NjE0NmQwZGJkNTUyZGY0M2Q4YWRhZDg4Yjk2NmZhZmQ1OGE4MTY4OTIzYmRiNzg4Yzk5MTM5N2M2Y2M3MzE1MjE2NTZkOTI2ZjNlMzk4ODc3ZWM1YjRkYTIxNzQ5ZWE0MmNmNThkZWEyOGI4NTViZWQxZDg0NmMyZmY4MzYxNzljZjNjYTYwM2I0ZDVhNGEzYTMzNGFiOTlmZDVmMTYzZDIzNDQ2MWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9K2CQwgNbJfMnM5wbdskuV83IQ6kCFZbv66kRIMUiQ6QpbMgTTfyCEv3K7ZDgrtudujL-H1u-4pZaGdflo_mUg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103351_89_24c4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.872Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNoRXhRb2hJcmNRYkswM1hQYUJvRnNPdXFzL2hOY2NBZERrR2VPQ29SSVdNZ0ZzRUpWY3plSFdGRHdZbDh0TklQSmpqYW1UYVBDN3huSnZ1NzdFbGZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTFfODlfMjRjNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjllMjM5NDE1Y2YxYmM0YTY3YjlmM2QwNWExODEyZjgzZTkyYWE2ODdjNjJjY2RkODQwN2JjMTRjMTc5YmFjZmVhZWFiZjMzYzA0ZmU4MTRkMmQ3ZDUwZGRlODJhZmQ0OTQ3OGVlODM2NjFmMDVlZGRjOTRkZTQzODI5ZWMxZThjYWFlNmNhMjExNGY3NDI3NDhiNGVlMmRlNzdkZjNhOGZmYWE5YjEwMjA2OTE2MWMwNmMzYWJjNTU0MjEzNzM1MTk4NjVlZGMwZDA1NzhmMmYzZmQ3MTc2NzkzODVlZTU5MmU4ZGZhOGM0NGMzOTgyM2E3Yzc5MmI4NzQzNDI0YmYyZmFjYmIyYzZmZGQ5ZjllYTZmMDZjYTBjYjIxODVhNTI1MzIyY2U3MDM5OTI1NzM3M2EwYjNlZDg2ZDBhMzMzYTUxMzZhZmU1YmIyMWU0M2NlNzAxNWQ3MWVkMzhkMjI4MjQ2NzcyZjFlNWNjMmMxY2U5MjgyZGQ0ZDU4ZTE3NDRjZTk3NzZmOGI2MWJiNjFmZGJhYjYzOWY3YzVkMjk2YTY4OTkzNjU3ZDAwNmFhOTNkMzJlYjkzMTg2NzA3NjBlNDY5OGIwZTQ4Mjk4MjY3YWRjNDYxNzcyYzg1Njk4NmYzMTc0NjI4ODMxYmQ2MWFlN2JhNzVjMTcxYThjY2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qxcN0fZyFBRIc5DJWvXY-kozAAkr6bRMFC1pNGrSxVBgbbgAvPMBfTuCD-6AOwNqWh9zn1lw4zhxkZfVy5Uf6g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103351_89_24c4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.875Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im1JMkFkYkw2Z0lnWEVsWTdSWkZaY3pneEUyb0N3UWI2Y3U5ajZ1TmJlYkxpTGlBSjdwTjdackVpbXUyVzNUTFYwUmtGZVY5YkVhalJocGJuVzV0VjlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzJfOTZfMjQ0N18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGQxNDg4OWMyODZhZGVkMGI4ZGJmOGI0OWExYjRkMjMwNmJiN2Q0YzNhMDM1NGVjN2ZkNDI1Yjg1NDcwMWI4MTgxYWYzY2RlZTc3MzI5NDE4MjJiNDIzYmFjYzIxOTZjMWFhMmIxY2ViNjkwNWMzNTFjNDg2ZjhmNjk1NDhlMzE5NzBhZGY1NGI2YTY1ZGUwMDE2NzVmMWFiZmRmNjViMmUzYWI4ZmI5YWY4NmFkMTJiZjAwYWJjYjIxNzc4OTZmNjYwYWQ5NzBmM2QzZjA2ZGM5ODBkYTMyYjBlN2JkZGIwOWY2MDViNzU0NjA1NjVhMWE5NTY1ZmE0ZDdkMjM1YmY4YjlkNGMxY2IwODIzMzk5OGUzMzAzZTZhZmNkNmVlNDMzZGQzOWYwOGE1NGM1ODM0NmJmMmQ5OTJlNTY4YmRhZmQzYmVmNGJjNDQyOWVkY2QyYzEwMzQwODUyZGE0M2U3YWU1YTdmZmI2NDUxNWY5YjhmZjE4Y2IwNWQyZjFlYmIzODhlNzIzZjRmOGE4YzllNTRlMTYxODM2OGFlMmJkNzYzZWVjZmM5OTI2ZmM3MzM1NzY2ODc5N2MxZjZkMTVhZDMxMzFjYjA2ODk1OGNlYmY4MzcwNDE3MjUxNzM1NzMwOTYyMDNkOTA0NmY5MGM3MDc2MmEzM2I0ZWFiN2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.r22MoDwyrJsqjUO0hqRMTYOEMJ6NQkH_NhPwQR5AARlFnEnbEzV6kbmYAOH_OSTw_wy8iWEjlhAWtn4eOLl51Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103332_96_2447_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.878Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjhMZjlYZmFLdXhxMWVZa0lzUmNvcUZIQTgvME84NWZPM1JzQTVoNmNPZ1ZRdnhheW8rdTlNbDJLdVlxaGtVRkh4eE9JZ0Z4dW4xUlNIaWVLbVc0S2tnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzJfOTZfMjQ0N19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDRiZGQwNGIwZmVhZGY1YjAxMzUwN2U3ZDc2NzEyODAzMmVkYmM2NThhZTI0Yjc0ZGU0ODBiZTY2OTNlMDIyNDM1NGRhYjUxOGI5ZWJkNjY0NDhiOWE4YTQyYzNkNzRmODIxZjk2YmVhN2I0MzVhYTY3MGY3NDNhZDAyYzIxZGRkODgxN2EyMmMxZGVlM2EzMzA0YWMyZDczZTc0YWNjNWExMDRjODZiNGZlY2JiMDNkZmZlNDRkNDcxYTMyODc2ZTIyYWYxZDE0ODE3ZmY5YTAwZmI4MGJmMjlhNjViYTJhODg5YTM1ODU4Y2YxOTBhMDUwNTJjZDkzMmMzNDI1ZDYzM2EwOTU2ZmFkYzQ1MzA5Y2Q1N2I2M2I4NTE2N2E1MjU4MTIyMDFlMTFlODlhNWE5ZWI0MTQ1MGJlZTc5NzRkOGQ2MzJkZjNiM2M5NDUxNjUyY2JlODk0ZmQ3MjVmNTQ3NjMwMGQ1MTg4YjBhNzExN2NjN2RmMTFhZDhhOWNkNWJlOWExZGIyNmQwMGI3ZTIyZWRjZjhhNTdlM2IxNmMwZWQ4MzRmNWJkMTZlNDZiZmNmYTJmZTM4MjQyMzg0Nzg4ZjhhYmIzNzY1NTZlNjhkNmIwNjBmNzAzN2U0MWFkNzE4N2MzYjNiZjU0MjEzYmE5OTE2NDJhYTE1MjcyMWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3-BaoeglO5rAOLivHCVzT9YIdPnCopTaCESm8GzxDbPTVnoPAU3IBdnupOb5YxZVQaXl73N1QGAOZ_9Mc_FRGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103332_96_2447_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.880Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlWRUdtWkxDbWNKTUNZVkc3NTR3dU94S0EwNXFxZ0RmdE01TklIU0hwdHdHT3RLRXZlaHNPTDZ5QkY3aFZBVjdTZWNqc3NCTE52ZHlXYmhrQm1xc2xRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzJfOTZfMjQ0N18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmRiODA0YzNlNzdkMzJlMWQ4Y2ZhMWExNTlhODAwYzkwNWRlY2Y4NWQ4MzQzMjQ5M2U0MmI4YmFmYmFhMjBmMWUwZjk2ZDZkODFkYWU0YWYzNzA1NDgwMDQ4MTRhY2Q3ODA3NTIzZDE0MTJiNzY5ZTI3YmExMDViOWY4NGMwYjJiNDI0NzYxN2RkODkzYjZhYzZlMjVhYzIyNzYxOWQ3ZDlhYjQ1MTViOTJiNTUzZjY0NzNlNGMxMzI2MzFlYjZmZjVjNGMzNmYzMzVmZGYyYTQwYTYyNjdjNGQzNDViYzFjMDliNGZmMGVhN2RlYjBlNzhlZWY3OWVkMjVlY2NjZjQyYTk0YWI2MjY4YjZjNmY4OTkyMWRmYTk5NmVhZTc3ZGQxMDdiNTBkYzhiYTNiYzFmZjM3NjM0MmMxMmU1OTdjNzI3NzI5OGIyNTJhZTYxY2M5OTc5NGU1YzMzZTJmZDFmYzAxZWM2ZmRjNDg3ZDhmYzQ3ZTIyZDgxYmZjMWI0Zjk3MDU2OTA2MGU2ZTIwODYwZDFjZDA0MWMwYmYwZjk2Mjk1YjgwNGU2OWI1OWFjNmQ1NzdjZWFmMzJkZDZhNzA5ODRiYjFjMGE4MjcyOTJkZTkwMjQ1YzFhMjIzMDllZTE5YzUxZDM2M2YzMjVjMmY1NWZkYTgwZjM0MzhjYWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aZGf2d-Abv1xJF2ltnDBBLJk5B-KmVKjAQP1344JmzVT3l4xb_-eVCcawlqKdRx4wmiuafcTqcYZFVypw9SWbg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103332_96_2447_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.882Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjhGa2liOWsxR3BQcE0vNzhCVEFEbkRhSHMxNXVOR1pEU2lGbzFZalZoK2d3TzdMeWdtbmRPL1hRenFqWjJldkxhcm5LY25DVS8rcmxTR3hHWE1rRWFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEzMV8xMDMzMzJfOTZfMjQ0N18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWI4NjM0NDczYWI0ZTFmODhiZWRjMzliYTI5ZDcyZGRiOWMzNzJhM2NjZTIzMWQ2NGFlYWExMDNjN2RiNWQyYzFlNTczNmNhMmQzZTU1M2Y2NjM1ZjA3ZWMxNzA2ODg1NGFmM2IwMGEwOTc2ZWE2YWYzOTM3ZDEzNmI1YTk4ODQ3M2UzYTczMjVlNzc5ZTQzNjUxNTAxY2Q3MDAzYzJlNGEwNzdjYzc5YmVkMDZiMmE0MWRmMzJkZGY1NjIzNjcwODlmOGMxOTVhZjMzNTM3NmVmNGEyZTBhNzUzN2EyNGU4MDcyOGU0MjAwZWQyNjkyMDJjZTIzODEzZjUyMTQwNzg3MjhhZTQxZTQ4NmJmMDdiZTkzZWQyZGNmOWIyYmE5MDcyNTRmZGFkZDZlNTU1N2YwMTBkZmEwMGI0NjllYzZkODAyY2IwNTBlMTRkOTNhMTMzZjIzNWQ3M2FiZGMzZTI3M2U5YTY0NWI4ZTQ2YjNjN2RlNmVjZDc0YTU3YjEzYTg3ZjYxNWVlYzFmNmU5ZTg3NGE2OWM5MWQ3ZTY2M2U4NDE3NGEzYmRmMGY1YzUxMjVjOWY1MTJkNDg5N2MxYmE0NzhlZDdkNjhmNzM5OTk1MTc3ZmZmM2U0ZmVkMzk3MGEwMDhhODBiZWM1OTBiMWRiYTcyOWUzOGY5M2Y1ZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.L-5uWMptASZIbcTFAhTPEZvQC7ETEeNNIZi_Xt_uFt2gA_NU9dMtZqui_OtaedBe1c7J5up9y9JboT34DoLkxA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220131_103332_96_2447_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.885Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjExTlBRbzNDWStWL2M0TytCREpKbHVWc3pkK3BrNXNVanlHWEN6SHNmb3FHOEpNVkxIbGFPWDhWSlMyYWpiZktMcTZURWhJZmZDUjRreWsvbTYzN2hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMTA4MTZfMjNfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWEzMjJmNzE3MmNjYjJjNjgwNGM4N2E3MWFiNDI4OTRjNDc1Mjk0ZjMxZmJkY2ZkMmVmNTc2ZWFmNDBlNmIwOWM1MGFmZGU5ZDlmNDcyOTZjYjU0NThjYmUxNDczY2VmNTkzNDNhNWNmNDhmNThjMDRkNDEzMGU1MmFmNzgyMTY1NThiMmNmZDA4NmVmNmViOTM0ZDlhMmFmYTk5OTJhYzJlOGIwMDYzZWIwMzk4ODE3NTk1OTJjNmI5YzEyZGY1ZThjNWZiN2U0YmNjZWUyZjVlOTcxODhhOTAwMzEyMWVlMTkxZWM1YTlkNTBkNjRmOWE2NGM3YzQzODA4MjhlOGQxYzBlNzIxNTRmZGJhMmFhMTZiYTIyNjMwNzBmMGJlYjUwYzRjYWVmOTgwNTQ0NDEyZTkxYWY5YWJmZDUwZGI2YzI5YWE4OTYzYzQ5NmY0MTA0NTM4Njk3NmE0MTZmMmVhYjg5MWU5MGEwMDg3MWY3NjZhMzExMTE0NzNiZTMwZGVkNmRmMTA4YjE0MWMwODQ0Y2VhNDQzYzIyYjYwOWZlNDUwOTllMTUzMjQ5MjZkOWMzMjBhYzgxZTk4YmVmZDMyNzRhN2U5OWRlN2RjNzdhNDg3YzE0ZmNlYjRjNDk2OGJiY2ZlMGNiOTE4ZGNkYWY1MTcyYjVlODNhZTllYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.f6Ij8tRA9gVAT0vr92PCsh7_JoQObhUAva6gpPPUo57AGxHB6liLL8yDJvp3xN4kmqUjer8KmpDZ2IYq0ANnQw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_110816_23_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.887Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjR4bWZHZXpVNUhiTVJucGRpTUl1Qm9lR2NVV2ZseEUrWEtZL3BPK2UrTTkrQUdKdXNJS1EwbENQT21tMkhOa3E3Zk93WGxDeE1CdC9LZHBaWU55OHl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMTA4MTZfMjNfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE0ZWQyODY3NDZlZDY1ZTg3Yjk3MmY3YmM5MWMwZjY0MDNkNWIwNmUyMmE5NGQ5NGYxNGIxNDA4OWYwMGEwZmJlNjhmMWVjZjY4N2Q3OTc5MDdlZmExOWE2YzQzMzNkZjY2Mjc5ZDU2ODEzMTQ1OGI3ZDk3NmJmODk1NTczYTRkNmIxYjU2ODYwOGY0NTlkNmNhNmE5NzYzMjUxODk0NDgwNjhiODI0MGY0ZWU5M2FjMTRjMzRkN2MxNjI0YjhiNmYwYjU1NGViN2EwODI4ZjZjMDkxYzhkNmJjOTkyMWU5ZjViNWQ5NTU4OGZlZDQyMzYwNjYyZmUyY2Y5YmNjODNlNGMxZDA2NTVmMTc0YTc2ODUzOTVmODU4ZGYwNDAxN2JjNjQ2YjYxZDliMzBhOTBiYzRkNThhZDI4OTU4MDhkNzFhN2VkYTQ3MjY1YzY3MzhkZmI2YzI2YjBiYTVmYzNkM2FmMDgzNWY3MGNhNTEwZjFmZjRhNmRjODFhYmM4OTY4YzI1MTFlOTNkMjI4MjBiYjY2MmM4YWMzZjk0MDRiNjk1MjE3NGRlNGZkZDczMTJlNTNiNWNkMzQ2MmViNTQyMTJhNjkyZTg5Nzk2ODlmZjQwMTZhYjI3MDg1OGMzMzdhNjAzOGFkYTkzMDBmOTI5ODRmZDQ1YWVlYTNjYmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HNui4OVEEXKfuQ0R9YjvxWhU_LJuy9dX2sOMJJc4GrDM3slO0YNwsg6GpyO-ku-4-Yaa9AqL3nY8XrJWG484Cw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_110816_23_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.890Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkV6ZGR6TzhPYnJFMGJZWmFZU2hnRDNOMUhpRWI2NUxOb0RtcVJRbHdhdXoyTHIzbXlhR1o3T1JQdGRCKzR2YjdocUNTL1NmY3dtWUFFaGlQa3hrVHVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMTA4MTZfMjNfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDg2NDNlNDA2ZTZjNTVmMWQ0ODgzZGZhYjMwNGVjZjc3NWFkNzZhOTUyNDgxODk1NTFiYzZkNmRmNWQzNWUyNWFjMWNmNmQyODUyNjQyOWUyYjkyZmQ3N2FiNzZlNzI0MDE3OGY4MjgzZWQ4NTZiMDI4NjUwMDhiOTgxMjgzMTY5MzdhOTAyNzIxOWYyYThjOTA3ODhlYzNjN2YyMjY5OTBhNjlhMWY1ZGJjZjI4NGUwNjA1M2FlODVjNzIwYTg1M2Y0NGY1ZmMyMWFkYTFjMTgyYWJlN2NiZGQ0MjM5ZGU1NTczNTk1ZTQ1ZThmMGU4ZDM3MjJkMzQzYmU4ZGUwNTc3MGExMTRiNTVlNmExZmNkNjI4ODg1MzljNTlkMmY4YWQ0ZGRhODQ3MmJiYWZjYjA4ZjcwMGZmMzg5MDkyOWNjZWQ4MTA2ZmYyNTBlMzdhZTI2NDc4ODY5MzAxNWVmMTA4NDc1NTA3NGY2NTAyMWEwYzgwMzI0NzMxNDEwYzRmNWUwMzk3NzNiN2RkOGExMWYyZDg3YmFjZTRlZDMwNWYyNzU2NmQwZmI5YzQ3NDgwMmMyMzJjN2JjNmJiYTQ4MzRmZDQ0YzEwZmEyMjQyNDk2MmM5MmQyYWRmMmZlYmFmN2IyZGU1NmUwZDM3YTI1Nzk0YzQ5YTMxNjYyMDQ5N2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lkf4nxAwPaSKEfWir5SCMegxXyucGIxVMJ4_Waql7kWAdDMzhMQU55-rwOOvWGrwhA3jEsLJWIZSQBD25KEquw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_110816_23_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.892Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InAzQStuMFpZd0psQVZvMHhKM3k2aVZiOTlYT01DYkVkT3JWWEFxWGpuMFd3R2x0cWZiVmZxYzU5UGxheTQvTHRxS1NLTkF2K1ZDVklGTlA3VFcydlN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMTA4MTZfMjNfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDFjZjdjYzE2NzkyZTBkNThjOTAyOThjOWNjOWQxNGU1M2Y5Mzg5NWMwNzM4YjM5YTVhMTNmZWMzZWJkOGUxYjNjOTA2YzEwZGM4MmU1NzNmNGE2NmU1YjE3OGM5NWMxNmMxOTQ3NDIzZDljYzM3YTBmMWY3ZGMzMzgzMmJmNWI1YjY4NDNkZTM5NTI2OTc3YzQxNmQxZDFmOTIyNGE0ZjEyZmU1OThlZDQ1OGJkMjQwNjNhOTYxYTBjNTRlZWFmM2I1NDcyYzI3ZTQwMGQ5ZjcxMTk5M2VmMTliODRjNzIzMDI2NTU5NGY1ODMxZjQxYTYxNzYzNTRlZjU0ZjlkMzRmZWNhOGRlZjQ0YTcxZGFkMzZlNGM4MWQyOGU3YTllN2I1MjExMzRlM2I1NTJlMmRlYWQxOTY5Yzg1YTkzMDQ1Y2IzY2YwNGUzZDlkNDE0YTkxMmRiOWY0MWU4ODg1NzA0ZmI0ZDliYTNlMDA1MjAzYWUwMzEyODMwNTg5NDgxOWE4NjI5NGNlOTc3MzU3MzYwMjE3NTEwOGJhNmE0NzJkZjA3ZWJlNDZhZWIxMzcyMmEyOWUzOTRiY2I3M2Q5MDE5M2E4ZDk5ZmE1NDRiY2Q4ODk4NmMzOGU1M2NlZWFhNTU2MTg3ZDczNWVhM2FhNjYwOTRkMjBjZTdjNGZmODBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.onyge6_XeE-hWr03U6V1r4Sq8_d26rS019OzA2nMe6UiQYyc0oLC4ZvW-cOoCmHdra5WzcmqrgtUKNAUi3BMsA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_110816_23_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.895Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IityblRSdXlqaXE3ZGFVUVNIbG5id3NpTy9JQTUrVVBTNkdORGJSQVIzSDBpeFVRcExJZTcxV3c2VlA5ZFcxV0U1d2pEczRhalBlNTV2dlRMMmlIZ2N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTAyNDNfOTFfMjQ5ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzdjZTQxYzYyN2Y3MmVjZjcyOTRhMDAzZjhmN2EwYzk1MjM4MDBlY2RhMmZlYWM4N2VlZDZkZTMyOTcxZmFjMjhjNWM3YWNkZTBlNWE3Yzc1ZWZjNTAyOGI5NWZhODhiYmNmYzM0YTljNjZhMGU2MDIxOTJjZWIwNTNiZGM1ZTQyMzNkNmRmMDJhY2I3ZjQ5ZDQzOTAzYjJhMGNhNWZkNjgxYTI0YmJmYWJiZjIyNzk4OTZjMjJmODlhZmQ3YmViODcxZDNmNDg2NDQ3MTllMTRmZGFlYjA3OTg5MmFiYWRlZmJhZWExMmRlNDVjNjgyY2I2ZDA5ZDAzNzZmMDI4NTc2ODg1YTNjYWFlMjdjYTA0N2ZkZThmMDJkZWQ2Njc0ZTU0ZWIwMDg3YjlhNDBkZGRmYWM2NDQ2N2Y4MmU1YTZmNGRhMTczMjhlZTYxYzE3MDYxNWYyZGViZjQ1ODYyODUyNzBiZjZjMWFmYTY0YTZhYjg3YzM0ZjgzOWU2ZjM3MDc1ZDNmMTMxMGViNGZlZjBiNzQ4MmE1OGM3YWU5Njk0YTg4OWExMWU3YzBhMDk3NDJmZjlmMTFiOTU3NTgwYzNlYTViMjMwYzRkYWVmY2QxZTRjOGY5YWNlNjY5MjVmMTI2ODE4YTIyNDA2ZTZlNzhkNzA3OTJlMjFlOGMwMDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6APZwolgRrR3mce6SQ8yqXJYQ_9lutB0HXVMSYql6f8mL7T1mlOH0rCwP5CinbHJfs-Daxv0DTbpTz4_BVkGKA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110243_91_249e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.898Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ijh6T0F5aHR4Vi8xY3QwNzBCR0x0c3lUdFNmVjVQb0JlVGt1L2lRWEc5eS9SbGdTZkhaa2xlV1VLdkJEcnh1MUo4cldkTEJ0M0g3RjE2TTBIejNrSlNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTAyNDNfOTFfMjQ5ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGM2NTYyNWYxYzYwMTFjZjdhZmQ3NzVjM2VjNmI3MjljY2Y5ODFkMDk4NmE2M2QwNjE5OWNjMDU2ZWNmNzE5OTUxOTNkODFmYjc3MGNjMjExMTFiNjE2YTRkZTdjOTA5NDZmYTY2NmYwMWE4OTk3NjRiZjcxZmFhNjQzN2IwNjNiNDBkNmU4ZWE3NjQwYTIyOTkyNzY5YWQzZjMwMTY5MDA2MjhlNDI0MDBlY2UzODMwNjkwYzYzNjZjMWZjOTI0MjQ4YzM3NWNkNjgwZjJmYmUyMzdkMWU0YTgxYjRiN2JkYzM0MTczYjVkNjQ1ZjE5MTgzN2M5YjIzZjk3NjIzNTY4Y2JmNjVjZWIwNjYzMDUwMWMyYjE2YjQ5OGJkNDhmY2Q1NDk0ZGQxMTVkM2RjM2I0NDQ5ZDBhMWJjZGZhNzJiZDNhMDEzYjEzNTA3ZGI5OTdjZDZlODI0MzY5NDk4YmI2MGY5MjEyM2Y4YTU0NzM1OTI0NTNiMTMyMTdhMGRjODE5NjUxZDRjYmQ2MzBiNGJmZTQ3M2IzYTRmMDIxNGRjZjBjZjAzNjVkZGQ1NzZjN2ZkZjdiNTc2NTkzNmZlZGYwYWRlMzJmOTdiMjUxODVjZGM0NWY1NzkwNzk5MzBjODVjNjY1NzU0MjY1OGM4ZTc5Nzc2YzY2MzQzNjgxYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xLQp5T7Xdlv0o4A0qHMLs7zFSq5oG_IVycehFtKJ29Wz1sGnZhdESr1DS-GqNzh8eo4ADOVliBOqRmQvhkRhxQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110243_91_249e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.900Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9Td2lxN2JDaFk1R1Q2MWd2YkcxUU9vS2MyZDlSMXpNUVIzbkpveHNvNjlPWEJrL0RuUWZycDk2ckhZMEFqUmE4OHorZ293amF0MHhtWDVmNFdBNVhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTAyNDNfOTFfMjQ5ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTYwMzk0MDJlMzFlMDI1NzAzZDk0N2FkYTljNTgyNTU2MDRhMmM3ZTMzMTQ0YjcwMDMwYTgxMGU2NTI5OGY4YzBiNWQ5MmM2Nzc3NjZkMjc0YzZiMWE2MGZmMTljODY2NWE0NzgwZjVmZTk4YjJkNjUzMmIzNjZkOWE5NjRjZTY0MDdjNzA3YjU0YWQyNDE1OTdkOTgxMThmZjg0YjVlYTJlODZkYjYzYmM5OGNjODgzNWUyNTI1YjRlN2E3ODViOGY1OWRkOGQwMTVkYTlhYWQ4MzQxODFiNmU4MmFmODZjZGZmMTczZGU4Y2M1MjhhYWJkOGZmZjc0MGQzYmNhYmJiNjYyZTEyNjc4OGEyZTlhMDZjYTUzMGFmMzA1M2MxYjA3MTYxNjJiN2JkNWRlNDU0OTIwYmI4MGQxODUyN2E4MGFiYWYzOTUxMTdmYjg3YTc5NWJlZGE3NjBjNGJiYWVkMmJjOWI3ZDBjODJmZjE2ZTkzNjhiMzMzNWI4MGU5NjVhMTc1MzU2YzQ4NjUwNjM5NWNlZjE1YzBlZjIxZDdjYmNmMGNiODcwZjcwMWM4MjM3NmZlYTYyZWI3MTZiN2I5OGI3ZWRhMDJkMDQ3YjhhYzZmNzJlODM0NDNmYjM4Y2QxZmJlYmY1ZGJkYThmODQwNTQyZTRiNDU0ZWIwZmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BwqvTktcRp99m-QTeBOlO7EemPJz_onT5kvdUJI6_BE3sfMtdMntjURr3e-0YIOss0OeGVX5VRJcUsIIKFh-_w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110243_91_249e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.903Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhJS2h2ZHJxNmpWZTg2REx4c2ZZRS96aktWdlAzSUE1MkEwVERob05KNFg1Z1Zka2g2eUxqR3FkYnVMM3ZZVDZwTkhjN0tnclRSTlhRMnZZalZySlpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwMV8xMTAyNDNfOTFfMjQ5ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2EzNDZiODlkMTNhMDEyMDdmN2VjNTZjZTlhZDQ1MzNhY2FiNDBiNGJhNDc4MDY4MGRkODkwNWZlNTI3YzRkNGJjZTg2NjliYTQ5MzA5ZDM2ZDc1ZWZkOGZjMzQyOTVlYzlhNDVjZGFiYTYwYmE0ODI2NWYwNmRlMGVkNTExNTcxMzVhZmFhMWJkOWQzZThlYTJlYjQzOTMzMjdhNzg1YmYxYzFjYWIzYmYxNzFjNWQyNDU4MWMzYWRlZWM4ZGQ4NWEzN2Q0ZjA2Mzc3ODVjMmMxMWM5MjZiZjIzNmY5NzMwNzMwYzk4OGEyMzlmZDk2NDE2NWY1YmNjZWU4NzIyODk5Yzc0NWExZjZjMDJmMjZhNjY1OTUwNDY4NGQwNjNlNWU3Yzc4ODU5ZTkxY2IxNjdkODE2MmVmYTdjZTlkYWJlMDg2YmEwZDM0ZWNlNmQ5MzA4NDA5YWNhYmYzZmNjOTdiNTFjYzMyOTA0M2FlYmZkM2EwZjM1NmU2MzY2NmMwNThjMWRlOGY2MzFjMDZjYTdiMzJmMDYyZjlkNGZhNTQ0MjhmYjM2ODcxZTExZWIwZWQyOWU1MzAxNjExMTYwYTcxZWY1NGRjZWY2YzExYzJmNzA4MDcwMzI5NWViODcyNmZjMjFmYmQwMDYyYzdiZDBmNDRhZjUyYmEwM2U5ZGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.k6-O9QVJt4ik4MfzveZt3YJPr3lCWYm2kvAE7OxV2TZlPfZ4Ln_XrzWYFfT168GAVvsF5vKWxVT4PN4iFeNPWw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220301_110243_91_249e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.905Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFJVGl1cnpYYzcwblFTRlNRdHBhK3hIM0dac2Rocjgra1pNYzZjbmhIMjBBYjNQTWhlMWpOSjVkV3NhbzV4aktZSmFHNVRRd0J3OG9Gd3JlTndHY09RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNV8xMTE1NTVfMTRfMjI3NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTZjNzIzZWI4OTE4MTMwNTZjNzI0N2ViYmZjMDQxYTRiMjk2OTI4OTk3ZmRkODNmY2Q4ZWUyN2VjYzc4Y2JjYmJhZTIxZWY4NWUzZjAyYWJjYTcwN2ZkZTNhODJkMDY0MGU4YzllZjY2Yzg4Mzc4MWY2MjQ3ZjcyZTY5OTBmNWFlMGNhNmEwODIyZTNiYmQ0YTY1MDY3NWE0Y2NmNTIyNWY5YTAxODVlNjQ3YTljYTUzZTY3YWQ2NTY4ZWI4ZmM4MmRhOGFiNmZlNDU0MjMxYTkxNmY3YWVlNWZmM2RjMDJjMTZjMjYxMDQyNzIyYjYzOWUxY2UzMGQzMWI2ZmU1MmU0NjcxMDViMzQxZDFlMjVmZGYzZjMxNTRkNGQ4MDE3N2FkZmM0MTU4Mzg5NDcxM2ZkN2FjNjk5ZDVjOTVlMDhiYzFjMThkMjAzYzMwNmUyNTVlYmU3Njk3MmMwYTQyNDY4NTJjNTZkY2NmNjNiZDZlZTIyMjlhMzI3ZDhkNmYyYjE1YzllYWM0NTViY2RmMTIzMzJlNzE1ZDkxM2JiMTQwNTNhYjI4NTM2YTQ2OTI1OTAxOTU0OTQwOTkwZjFlMGY4NzA0MjZlOTJhMTc4YjRiZGMzMTU3ZjBjNzdkNDY5NGE5OWZhZjE1NTI2YmZhNzc4YWM1ZjZkYmY2ZDY4MzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fY86Q6pUP4pgq4sMiQBJF5_aMpBeHs7bopkmodKeCFd0DJoXCURNuQTWmu0EQy1MmuaOwFpj6GCZjAtQ1biNFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220605_111555_14_2274_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.908Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkRhZlhTQi82OXFxTHk4R0dYdWRweXpLUVFESTNONHJ1U2l6ei9ibG5aOG13ZDNycmdKdCs3NnlQNi85UG9IMlBTclE4LzdhMWxLVTZwTTMwRytUYWt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNV8xMTE1NTVfMTRfMjI3NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWE1NmIyMzE2NTY1NGM3NGNkODU5MmUyOTQ3OTA1OWVkYjEyYzU2Y2RjNWJhMGM5MTNmMGFkMzU0Y2QzOGM5MWJiNWM3NDljNmI1OGJhYjFiZDI4OGY0MTAxNGU3N2RmZGM3NzI5MzU5OWI5ZDM4OWRlOTY3MWVlMGExMGEwODFlNGE3YzZjOTQxZjVmMDI5MDFjMjk3ZDNkOWM2ODZhM2NiNjJmZGM1MDI5YThlNjZlODYxNGQzZWVjZjU2NGVhNDU5ZjIyYzY1MDljOTQ5NjBkMjVhNmJmY2Q0NWIyN2ExNTgzMTFmYTAyYTg2NWRkMzJlYmU1ZGEyMzBjODRlODNkYTgzY2QxZmY3NWI2MmEyYjA4Njg5N2Q2NzQ3MWI4ZDFhM2Q1ZjcwNjFlODExYWUyMGQ1ODkyMGI4OWUyNWM4MzY2N2Y1YWYyMDgyYWMxNGNmYjZmOTNhMzhhNTkyN2VjYzhiYjQwZGE4NmUzMjkwNGNiNGMwZjBiZGIxNTE0NWIzZWVhMzcxZDg4MmY5YzUwMDcyMTk5YWE3ZDU5ODk5ZDMyMzcyMDViZmFiNmMxZTdiNzcyNWU5YjI4ZmM0ZGU5ODFmMDkzMDRlNTM5ZmE2ZTI2MTU1NzQ4YzFhYmNkZjVkOTY4MzE0MGNiNzVkNjVmYWEzMWQ0YTg3YjAxZmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uWFoq95hX8KAsrv6c8nXaRsrpS6Ll19w9x9SLZPUTmJ6S1rZF6gqKbU_zsl27WyUMV-NcJNr6x47r5Oo5mBvKQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220605_111555_14_2274_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.911Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJQTHNqeXVJSm4zbGQxdXlwNGFTMGhlbi8rdmpMdVNYZEFlWllSVXpJZG5pMXFBNU1CU1F3SEt3SVAwdTRyWnA0Q0lNWEFabm5sTU45VllSZkxvSG5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNV8xMTE1NTVfMTRfMjI3NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDA1ZWZjNWY5MzJiYTFmZjUzYWE1ODI1YmRiNTcxMzQyYmFlZjlhNTAxZmU4M2JlN2IzZTlmZDYwZWUyOWQ1ZDA2ZTYyMGU4NzZlYjRjNWY3NzQxZWQ2MmEzYTkxYzNlYjM1ZmFhN2QwNzZlMTM1ZTUwOGJiNjkzMzA5MzU0M2NiZjQxZDM5NjI4MjY3YmIxNTg1M2VkYTQ2YmQ0ZTY2OWMzYzdkZTYzN2NjYWMwNjFjNjgwOTk1M2EwNDM5MzZmMzgwOWM4N2M5NDYyMTBmYjE3MTFkNjE1ZTVmYjI4ZmQ1Y2U5MDIwMmYzMGY2NGU1YTg4YTJlMGRhYjc4YTM4NjdkNDUxZmFhY2Q4MjkzMjExNjZiYjlmMDA1ODlmNzNiYzQwZWNkNGNiYTRiYTUwZmU5MzIxZmMxYzIwNTg2Y2ZiNjdmMWNiM2MwZDMxMTU3ZTMyMTYzYzY4ZDIxZDU3MzU1NWY3NWE0OTVjZmE1YjlkYTEwNTM4ODU2MTk0YzUyNmRhZGQ3MDFjZDljNmE0NmViZGJkMTc4MWQ0MDlkM2RjYjdlN2I3ZGYyYWVlZGY4ZTVmN2M3ZjU5ZWFjZGEyMWQxZTRjMjA0MDkzODg2YjIxYTA3Njk3OTQ3NzY0MjcwYjRhMzMyZTBmNWY0YjQxZjI2M2NlZTgwMzg1OWY4OGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1IdKtWhp1VkZ1u1sP7uJzfy7x6jxASbAgrHYd0Fj7VzbS14YVtDZrg43yFhQJxFeBKnnDIFEFQ17NKLXn7xptw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220605_111555_14_2274_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.913Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImI4aGUwaTk2VW4zQTJBc0N5NjNTVkpReXZiazd1RC9ESHZ4RC9TSEUwSk1qa1pQZktuelAvUU9DWmdMOG1ZSUNJT3VjUDBtV0YzZW5Tek9SRnhSSGtBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNV8xMTE1NTVfMTRfMjI3NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTE5MTdhNWIzODJhMTA2NjhhNTdiMDFlNzJjMzgyMzBlYTI4ZTJjYzAyZDllZmYzODRkZWM0YjM4NWQyMmQ4MDQzYjA4Yzc4M2Y4YWE2NWI5NDdhY2VmYWIxMTFjZTZlNDEyNmM3MDI4ZGU5ZGRlZDAyMmE1ZDgxNzgxNTNiMDQ3Mzg3M2ExMjQ5NWIzM2E3MWY2OTM4ZTRlOThkNDMyNmRhZDIwNDlkNzc4M2Q3N2Q1ZDA4NWUwMWM5M2RkMmE0YzE2NjMyYjIwZjc5MWMxYzcyN2FlMDM4OTdkZmE0Zjg4ZWQzZjdiM2E1MDExOThhMGEyYWE2Mjk3MmJjZGYwOWNiZDhjMzdjNzVkYTY0YzkwYjAyNTkxOTAxMTQzYzc0N2Y5MGRjYTczZmIwNDQ5ZjlhODU5YTQ5MDdjNTQwOGUyYmM2YmM4MTYwNGZlZWM5ZTAyNWEyZTMyOWU4M2ExMDA5M2JlODQxYjZjOTQ1ZGMyOGZhZTQ4MjAxMmE4ZGMxZTc3MTNmNmIwM2QzOWMzOGNiNmVlNDBlZmY3NTVhYzdjYzBjYjQwZDNhNDM3YjgwNTNhMWI4NjdiNWY2MWNhMGQzYjRiMGFiOTljNzdhMGM5MzIzODM4NDEzNWFlMTg4MTViYzc3ZTU4ZGRjMjJhNzZlNzg4YzIwNGQ3ODIxOTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4ozxILyxg62-8hilJMGH8KUifTgIsEBdW1bGJkDZ9vQnA0fRcEV6Bd0BmEtd-qknPC3LNaUuVM7OhKHJsQXiZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220605_111555_14_2274_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.916Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikt6TWFZZVVwZUVyc0RITDBQWjNKbkF1bkFqdm8wUHhTSS9QbHRWN1ErZjVTcjU0UG9JSnJtSlBrY2h3SWVBVGc2NmFNRlZWbmtJLzluVnJiRTJ6cVlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTRfNDhfMjQ5Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWJkMDRlZjRmZTExNjM2YTg4ODE0ZjI2ODkyNmVmMmNkNTBlNWNhMjJiMmFhOGY2MTNjZDRkN2U3MjIwZDhjMDc1ODdlYzcwOGIyOWQwMmI5YWJjOGVlOWY0MzIzOGFjYWUyMTRkMjZiZmUwM2U3MzQ2N2Q1YzYyMTg3NjgzOTFhMDVmZjEzYjFhNmEyZDllN2RlM2U1Mzc2OWZhYmRjMjM5OTdkYmFhMWQxZjlkOTU3ZjQ5NmVjMTU3YzRjMGU1ZWEyZTNiMzU2Zjk3YzI4Yjk2OTk4OTRlMTg5MTZiNDMzY2VkOWUwNjY1NzgzOTMxOTQ4N2FlMDkyMjA3MzA5ZDA0YTAyZTUxZTA4OWIwMGU1YTg4NWZlYmVmYjA0ZmI0NTcwZjZjYmE1NWI0YThmMTRhN2QwYzZiNjIwNTQ4NzE4ZGYwODNiMmMwM2UzZmJiZGJkZWRhMmMxNmZjYzNkNDRmODU2OTQzMGZiZWMwMjA3NDcyM2YzMjUwMWE1Njg2YTU4NjY2NThkZjAzNTljODM3MTM2MTllNDk5MWVhMzFmNmYwOWIyZGY1MDY3MjhkZWZiODM3NjkzZDBhNWYxZDExNWU4M2IyN2I5YjFjNjM1ZWRjNDRlN2ZiNzQ4Nzc3NmVmNzUwNGI4NmRiOGU0NWRjYmQyMWM2NmVjYTAzYThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PW2LgVFEaMSTZLNt1QQEQTEpR7Dc8WWQFsEKttx3kOk1bxg_uW_IoKgWcIEtq9fwKC3lGNq1apzXgbHEas4AqA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105914_48_249b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.918Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ijd4RXhYTENiYVBsVkxyK3Y4YWZyNXljcHFPWFRxT1gwMmFVYWVSS1pWUXpvSTR0TTd4R0I1N1BsTmhTbmlYZ0c2K3J1djRVTC9wTVpNZWRBSUNOcitBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTRfNDhfMjQ5Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2VmNjExNzg2YWJmNDk0ODE4NjYxYmMzODZkYmIzNTY4NGM2MDgxNzM3Y2U5MWExOTUxNzFlMjEzNWU1OTlkYWQ0ZGY3OWE1NjZjYmNiNTAzMmQ0ZjA1ZDA3YjRhMmU5YjkxZWZhMmE0ZmE1MWRjYTBhYjU2YWYwNTY4NWZkZDM3YjNhZTdlMTExMTU0OGI4ZWU1MDBmMjMzYTRmNWIzMWY1M2NkYzIzZDhmYWEzZTYxNmRmZDllZjZlMjBkMmNjMDVjZDczMzZiZGVmZjQxZWY4MWNkZGMwNzczMWQ2ODZhMGYxMzkxNGE4MDViYWQ3ODczZmI2MGIzNTMyMzU1NTA2NTZhNWRhYWE1MjZlNjZiYjlkMjIxMmM2ZGQwMTQ0MjVjYzNjNWFlYTg4YmNjZjAzMWExZjkwNDAwOWQxNjYyZDQ5YzY2YjgzNzg0NjEyNzYyM2QzM2VjNWJjYWI3Mjk5YjEyZmVlYjU3YjhhNWFmNjI3NzE3ZTA2M2ViMTNjY2Y3ZjQ4YWFiYTk1NTU0N2FjOWI5NTdkYTkxYTRhN2FhOGYxNGEwN2ZmMzZlNDBmYzVkMDc2ODYyMzE1NDM4ZjQ2NzFjNWJiMjZmNzBkMWRkMDE5Y2ZmMjQ4MjY1M2MyNzA5NjA2MWY5NDE1MTQ5Mzc0MDEwNTEyNWMzMGI4MTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cq718suo1oOw67sfhkfQa2p-Pr6dJJF702OHBWARmIV7KHryQLZyUR4bbTFMSP226vJ3gnXhKZGf_zsQo5LFEg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105914_48_249b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.921Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZPUGJsM1hsVmx1SGF5b2VnRHNCYlpjLzBSc0VpN1RGaXliNjZCSnp1aHcydGFpOCtBWXRxZ3c4TElzd1ErUDBBVTJmOE1JUFdNK0hRQm9ZUm1oOVhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTRfNDhfMjQ5Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGQ1YmM2OThmYjk4ZGE5NGU1ODBmYjcwMGJhYTI2MWEyMzE1YjI3ZjNjODA2Nzc2Y2I0YmRmNjQ2NmYwNThiNzI3MTU3MjM1ZWE1MDNkYWJhMjRmMjUwZDQ2NmYwOTU5ODMxZGU4YjQ3YzUzZWRjZDg5NzczZTM5NWUxNmNkYWQxMTc5ODhmOGIwNjNlZDUyZWMyMjJhYTA0NTg1N2YxZDNkNjdmZTUwMjRhNWM4NjliZDhlOTA2NDExOWM2ZGM1NThiMzU0YmRmZjBhODE1NTFlMGJiNTFiZjdiNzNlNGNlZGY2NThkZDE0ZTkyMGRjMDQxYTZkNDA1NTg3OGNmMmFiZDliZDVkYzZhMTBjNDZhNzQ3NTg1NGVjZDUyYjc4M2IzNzEzMDA3NTRjMmFkODEwMDc1OTIyMTlmYTEyYTg4MTkxNzg0Mjg3NzdlMTE2NjM2MWUzOGE4ZDFmYjkwZjg2ODY0ZjM0MjgxYzFlYzc3ZDY1NDUyMzY3MTcwMDgxMWRkMWU1NTBiYjhkYmFjOTBmMGRjMTcwOTg3ODc1Mzg0ZjU4YWEwNTk3NzEzZTJmMDM3MWY0M2VjYWZmM2Q5YThjNmRhNTUyOGNkMjE0MzdkMjg2YmNhZjNiNGVkNTRmNWFhNzdlZjNlMzVmNzQ1OWEzZGU4ZmZmZDA1MDk0ZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-uEi3mRpFjoaV5y_UfOorFSq6k-ylfPmVzj2mkKqES9QUksJn5Kapg-vBv-SACGoXJZ8uUNipYt8CSYFahaiQA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105914_48_249b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.923Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii8wK1NlVzFPYmtlQ3VIZm9CSWI2NS95SmJwcDBPakFFYk14U282SnpzeUFBSzdUNDZKUXU5bGpzNXZFcEo0WEFGcUNuRjRYTXJpeUxOZDFaajNGNWxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTRfNDhfMjQ5Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzVkODEzMWQzZWE1NDVkNjhiNzcwZWU2MDVhZjI3NDYwNTcwYmMyYWQ5OWE4ZmEwNWNkNDc5YTZmMThmMGU1MzQ5ZTY4MzBiMDJmZWUwZTI4MjNhOWYxMjcxM2Q1Y2EwM2M2M2M3ODIxZjBmOTY5YjUwOGU5MWJlM2E3ODhkOWIwM2NhMzBkNTAxZDg3ZjYwMDY1ZGQ5NTVjMmVjZDliMDExNGEyYTMwZmE4ZjdkMDBiMmUwZmFlZTllZjc2M2Y2ZmZhMzAyMzkxZTNlYzU0MjQyNGM5MWM2ZjRlYWFlMmI1NWM3M2U4ZGI2ZDdjNTFmMzgxODljYzdhMGFkZDZmNTM0NmE3N2M4ZTk3ZWMwOWE4NmY2N2JiNDQ1NDlmMTdjMzM5NWViNWQyOTg4NzM3ODMwOTI1NjA2NGM1MmIwMTNhN2IzNDAwMTMyZmZlZDJjM2EwMDI1MDkxODMwYjcyNDY2MDI2ZGVjY2NjMGZhYWQ2MDBjZDY1NDBmZTUzZGVlNDk5MTQ1YTQzNTc0MDIwMDY1Yzg5YTEzZTgwNjJiNmFmZDFmMzc5YjEwNzU0NjlmZjdhYWNlNWE2ODhjZWQ0ODkyMjcwYTQzMDkwYTJhMjBjMWIzNjVlMzY3MTljZTk5ZjBjMmViZjk3ZjBkNDRiMDhlMGM3N2JiMzI3NDM2OTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2wfACeMlRBcAfYGk4aMQWVcoQJNuTM7DqDuUrX0P92XnvDmbYg0NY2drFz0S6-5RK3SptvVa5UC1d_a8epJO9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105914_48_249b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.926Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkMzaUw0bUlOY1p0a240UlY3d0lDQ2MzUjU3bUF0SGNuQS9GY1d3NTNnOE4ydC81bnhEU0J4Wk92dnU5aVc0ekk1YW1DamNKNkpBMHZzWHpiZ2RuMk13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTFfODFfMjQ4NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTYyZjYxODhjMTI4MGYwNjFmNDk2OWU0MmVkYWQ1MDk5OGJjMmRlYTk4NzM1ZDAzMDkxNmE2NmIxMTliZDZjM2YxNTQwYzZiNzlkYzgyOGM5NTQzYzVkM2M3ZDQ5NjE2MTlhZWRjODJhYWYwYTZjNzE4NGU3ZmEwYzc3NmQyNTFlNTk0YWMyODg0MmE5ZDJiZTFjMmE4MmM1ODI2ZTlmNWI5MWEwY2NjZDJkODM0MTRjMGEyMDBlMmE4ZmY1ODAyMDdlOGE5ZWRjNjAwNTRiYjU3NGQ1OWM1YTdmY2E4NmVhODQ0ZjdlZWM0MTI0MzgzZTIwM2I3MTU3MWNiNGUzODdmYjU1Mjk5Mjg4YjQ4M2E5Njc1NDJiZGI3NjJkMjllZWZlYjYwY2M4MDg3YmM1OWFlZGRiMDhmMGVhNzRhZjZmZjI1Mzk4YjFmZDkzZDBhOTdiYmRkNTI5NjhmMjQ4ZDhlNzAzNjI4NTI2ZjYzMDhjZWVkNTZkYTFmOTQ1M2QwYjM1N2M5YjhmN2IxMTIwNTA5NTNkMWM2MzBmNzEzZjdiM2QxZDQ3Yjk4ZGMxNjVkMjg1YWQ5MWRmZDBmMjEyZTQwYzI1MjZiM2JjMGI3ZjJlMTk2NTU4OGYzMmU1ZGRjOTI5Y2MyMTlkMGY3MDIxYmJjMGRiNzFmZTE3Yzc5Y2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-HjVfyz5eZlN1jZJ575caiPWETcZnQJVUzkTlUN2klAgI2bJNF8Sur5o74f59MGYeCbZqpT0DB3K_2CDnruQog", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110351_81_2485_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.929Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkgzVmUrU0lpdFRZamhKOU1zVlhNVHhSQ1RoWjgveTBXU25sSVNFL1NabXlWZWxCaDg3TUlCUG94dEorYmVRY21mbVQxYUpSZjVhamhkN1p2OC9Penh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTFfODFfMjQ4NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTY2NTIyMzlkNTY4ZGQ2OTgzOTQ1NTQwNmNhNWEwYjk1YTUwMDM3ODU0YmUzMWM0OWVmNGM0YWRmNjJmMjk3NWQ1ODEzYWRjYjI5N2RiOThiM2M4ZTgxNWEyZDZhNjA1ODRjYjVkNDhlZWJlZTUyNTNmNmRlNjdkZjI2MTIwZDQ3ZTgwZTc1ZWFiYTU1NTIyMTQ3OGUwMDg4NjA1YWEzYzNjZWVkM2UyZWU1YTgyYTI5MWY4MmNiYmMzMzI1YzAzNWNmNTU5YzYxOWRmOGEwNmRkODAzOGUwOGFjMzMyMmViNjE2NDY0MzUwMzRiYTY1MjUyYTRkMWFkY2EwYmYxZWVlNmU0MWY0MGU0YjE5MjQwM2MxMDU3ZTU0YTJkN2UwNWEyZDc1N2FjNmE3OGViMWUwYzc0NjMyZjg3MGU5N2EwYjdhY2NkOTE3Y2Y0NDlkMGRhYjcxMzc4M2MxNDJjNDAyYjU1NjU3ZGUwZjhhZTc1NWRmOGViMzA4ODkyNTBjZTQxMjBkN2NlN2I0YjExZTIwYmE3OTI4YmIzYTJlZmMyMzhhOWViM2Q2ZjI1YTNmYjZmNzQxOGRiZmU5MzE0NmEyOWNiMmNlZTYxZWNiZjVjMzgxNDU0YjU5YWVlNDlmMGJmZGQ3NTIwMGNkOTdjNzM0MGU4NTI5MGVlYzI3YzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1vIoDTrtevF8kF6_gQoJJS2gGHfBrkbUH3ROqi2VyM3dsEbLmRpsz62kwrzM3H0_XnAiJUV8D-8_0a0ndd9yWQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110351_81_2485_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.931Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImJzRTNQQWJybU5NcWpHZGNYQXg2L1ZHRjVpcTY2TDgzOUNFb251R1llRElKRTJZOHgxZTEwbCtHMEpQTFl0eUE3RlNvWERaS1hzMFc3YkxKaisvQlp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTFfODFfMjQ4NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTU4NWM5ODExNzU1NDk5YTdhMjQ0ZGVhZTljNDFiNjFiOTlhMmJhZjFhNTk1NmU4N2NjYzNlOGFjMDMyY2I1MTIxMmRmNzFjY2E5OWMyYmVmZmIwZjQyZDQyMGZmNTgyZTdmMTY1YzFlM2EwMmE5N2Q4Yzc4ZWEwZDVhMTQ4ZDMyMjcxNjYzYTI2ZDk2OTM5ODNhMDI1ZWYzMjc0MDU3YjNmYTM0N2E2OWIyMzc1ZmZmNjFkZWYxMmFkZGUzMWE5N2E4ZGEyNGY0NWQ5ZDhjOTA5MDFhYTUzNjUyYjBiZjgwMmM3YjVkNzNkZTcxNDdkNDQwNjJhMDQ2MTU4Y2Q4ZDAyODE5MzcwOWVlOGMxN2MxODJlMDMyZmFiZDE5NGM5MTY3Y2U4MzMzZmU4ZTVmYzBkNDE4YTM3ZTUwMjBhNjUyOThlYmFhN2UyNDk2MTA0OTBlOWQyNGRkNDY4YmMxYmU2ZGM1MzZmZDdlMzU5NmQ4NjAwNGZlOWNjOTdiZTY5ODBhZjI3ZWYwNDFjMTBjYmFlMjY5NDZkNDczN2I1ODA4MjQ0ZDg2NzVjNmM4MzAyZDA1NmYwMjFmNjNjODIwMjgyYzJmYTEyMzg2ZTFlNDQ1MjFjYjk4MmFhZDJmNDg5NjYwZWRmNTQwOWIwZTllZjg5NjJiM2E1NmM1YzQwOWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5ZWoiEySRgzmUgnJW0tilBtm6C81tKbwCSb7K1sbfWgBWpmijSihP1vU93ZXWgUT0ofzjHE2zN-mVdhV8pev0g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110351_81_2485_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.934Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ilp6bDBncHJGTEQzdDZndjJVTkRIVVdIcXRUdHJRMUFMb0NJSHlHb3BFelBDVDZWNTByL2IrZmlCUk15Z2xMRTVqOE5SbWRJOHk3bFpJU1NIWG4wNHJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNl8xMTAzNTFfODFfMjQ4NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGRiNDgxZTRhOWZiYzc4OGVlMGVjYjVjODM5NDhhNzE3OWJhNDA1OWJlNTZmN2FjY2UzZjMxOWJiMzg0YjlhOTAzZDdkNDU0NzFiYjkwZmU1OTcwOTUxNWUwOWZiODNiNzQ0MzhhYTMwOWZhZTI3YmM5ZTY1MjNlZTM3ZjJjODk5YWU1YTdhMzA1Y2EwODA0MzlkODk4MDRhZjk3ZjI1MjdiZDU0MWI4NWIyYWMzOGRhNDBjYWQxMDJiYmM1OTVkZTU1MDRkM2I3NTVjMGFiMjIxODM5NTZkNGFjNjY3MjAwOGI4ZWFmNGFhNTdjYzhhNWM5MTM4ZjVhODUxZWI0YjhjNjA3YjY4MGU1NThiYTgwZmQwNjBiOTgwZWI3ODAyZDllNjdhMmNmMmVmNGU5ODlhNGFlYWU1OTcxZjRiNmI1MjEzMTA5MTg0ZGFmNTY4NGYxZjYxN2EzNjM1Zjk0ZDc2MzBlZTlmNTkwYWQ4ZDhhMzYyNjRiZTkyNWE4YzA4ZGU5NmE0YTJkNjliMDU5YzQ3MTA1MDczNGQ4OGJjZjQ3ZWUwOGIzM2Y2MzYxZjc4MzU1YTUzZGJmM2UzYjE2OTViZTBiY2UyYmZmNmZiZThhMWViOGZmZmZmYWM3OTdiZTRjYjhkOGY1YzU3YjBiY2ViYjViNzhkZDZmYzhhMWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._CMzhXjz1Oh2P2ZExgrvaXO_QnCKudSh0fjTuShH4E1cFaVgDUp2BIVkFr7TQqtkefSrBlQaS1Fo_ISSFThtlA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_110351_81_2485_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.937Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InhoZHJ3ZHp1UnZ2clRmZUlId2llZFNnUkFCWHNhUWpSZ2tzV0NGMVhVNHNzcnloR29ob1dIRzNwL3ExMDVGREVILzBpc3dHQmRNWmJrek12RERjNzdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI5MjRfODdfMjRiOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NThiMWMzOGRlYTM0Zjk2NDZiZWU5OTE1OGMwMjc2NTU1Y2RhYzA3ZWRhNWUzYTIzM2IwNjJhNTBkYTI0ODQxMTRhYWNkMGMzY2FlMzBhNTBjNjI1MTk0ODU0NDJhMWE3ZjQ1ZDA4NjllMWJlYjg1YThhMWU0NDQyNzgyYjBkMTMwYjBlYjViZDI4MTFmNTZjZTY0YmQ1NjQ1MTVkMzJlOGQxMTRkNDFlZmU2ZGMxNWU5YzZmMzUzMDNlMTc2NzY1NDNmMWRhNjg2YzVhNDczODg1MzNkMzlkYTBkNGI4ODNhODYxNzVmYmE2N2IyNzAyYmYzOGFlMzRkMjMxYzFlYWZkODVhODJkOTI1ZjNmNWY5Y2EzYmJlN2E3MjgzMDQ0ZjQzNzg3ZmNlZjI0N2M1NTRlZmQ2YmJiNDllNGNjMDExNTA3ZTljY2Y3N2ZmOTY0NDBlNWYzYjY0YTZjYWZiZWU5MTQ3NGNkN2Y3YWRjYjkyMjM3M2VmOTVkM2MyNjU0ZTIxMjJlOWFjZTliMmQzNjY0MmRkMTE5M2QxMDk1ZjgxMTI2MzU4NGU5OWFmYTliYzljMTgzY2RkNTNiODc0N2Y3ODZjODM0NTU0Nzk3NzM1MmIxMzAzMjcwMWIxZGJhMWQyNWJkNTdiY2QyN2I1NjU5YTg4MTU2YTZkYmY1MDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yimI9E79WVxun0e8L7gQmKKD6uTz3z7M-jEB0xbB0Kd9GwyjVLvNRS_hB3pSjpKSe2HZ4-4BJKHxAr7enCLC4g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102924_87_24b9_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.940Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxkU25xdEtTcVYvVUNiem9pMmJWSFR5V0hpWlI5dnRBaU5DZkIyVHdnMVFpNTlwQXlqSkNFdlVYK0QwOUswblcwOWpYbTEzZFBzbmppR3FNaWlqUmZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI5MjRfODdfMjRiOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmMwYTI0NGQwODY3ZTQxYmNiYjVlZGI1OTA4NTg1MjQ0YjQxOTRjM2E0OTdhYjNiNjQ0MjljYmUxNDQzZWNjMmU4MGFhNTIzM2I3YWZiNzVlNjU3ZmY1ZWE4MjNjZWMxMDQxMjBiMGMyZjFiOTM3NmZjMjg2ZTE1ZjE1MzBmMGU1N2Y3YTZmMTAwOTMzOWExZmRmYjExNjY1NGFjNTQ0Y2EwODQ2MzM2ZWU1YjdiZDczNGJmM2YxNjQ2ZjkxZGMxOGM1M2UwYTNhOGUwNDEyZTAyM2I5OTU5ODkwNWU5MWYwMmY1ZTg3NjQ3ZWMxMzc5YjZkOTExZjQ3ZjAwOTRmYzE3ODE0YmFhZWZkMTg2Y2U4YjE3N2YyNDIzM2QzYjY5ZjU5NjdjOTZjOWI3NDc1ZjliNGNkYWJhNzBkZTNiN2NkM2I3YzI0MDA1NGNhZDdiZTZkNzUxNjZkNWEzZGI4YjUwOThmZDg2YmQyZTgyZDk4OTgwYTE2NjI2NTFhM2RjMjc5YmM5NmMwN2NkZmQwODhiNzk2YTFmNjQyZDAxYzM1NjhiMzBiYjczNjUxMTlmNjBkZjYxYjU2N2NlMDVkMjI1Y2VjZjE1MmZmNGIyMDYyOGNlNmU0OTAzMGJiYzhiNTYzODE0M2VmNjliNTEwZGQ5NmVlZjUyYjE5ZDkyOGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NaGeSajLSAVN0gAFY59RdknsM55BuHCD8zv3xzDchOZHPHWs8RHrjz-ZsxS6Vr7f0eHgk9cNjP6DVQaKHEl53g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102924_87_24b9_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.943Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkxwdkY5T2VQd3hKd2wyQm83a3RVcTd5S2M4U1VENmo5YU5FbGNhVEJtNXNnYmhmMmRseEl3UkJ2TnNUcFBjNENJVXIwUWpiUHlQN3gxN1lPSDFlSWJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI5MjRfODdfMjRiOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGVlOTRhYWRhNmY0ODZmMjI2MTlmYWNlNzY0NTQ1ZjQ0ODhkOTE1MzQ3NzYwMDkxZWFhNmJlNzBmZGQ2OGI4ZmEzMWRlYWEwMzg2ZTk5YTQwZDc5NGE4MzNiOGE0MWE0YjRlZWZmZGQ2MWRjOGJlMzdmYzI1NjU4NTdlNGFhZTIxZDBjMzYxNDQxM2ZlNzFkOThmYjEyOGRjYjk2YTdlNzI4NDZlOWE3ZWY5ZGE4ZTczM2EwODM3NzJjMmEwY2YyOTVmNzQ4YTBlMmUyOTBkZDQyNjlhODVjNDBiYjg4MDljZWYxMjk1M2JhYWJiODMwMDY4YTkyZjU1OTRlNjFjNjA4YjY5ZGE4ZDFiZWE2MzFkNGY5ZmIyNTI3YmM3ZTgyZWVlYzVkNjZlYmNlMTVjYjY1OTEzNzQ1NDliMjRiNzM2NTA3Y2ZjNDIzYzBkZTU2MTc5ZjI3NDQwY2MzZDVkNGVmNDYxZmZiN2RmZjhiOGJkNTQxOThjNmJkNzBjN2ViZDMxNmY5YmNmOGY4YWQzZTkwYjIwNGI2Mzg5MzBiNzYxNzc4YTE0YTIxMzgxY2E5ZDhkNDdhMmUzY2E1M2MzY2JiYTQwYWE5NGJmNGY5MWZjYTNmZjI2YmM0ZTRhNmVkMmQ1ZjZmYTEzNGNkMjI5OGNiMGEzNGFkZGUxYzhiOTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hWpi34YwH6wwX9BH8UfBixiAE0f5InxdOk1VjG_4jgRhrT7ATQAktfUGu0cuPq8oZ0WGOnAWb646Ab8JUKt4bw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102924_87_24b9_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.945Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZISlVXNFFuZG9vZGc3UmhLN2E2R3pwbFNpYW9FYXFNYTczcVFwdDZ1ZU9iUkUrVmpZajRXMDJwd1g0Y3Z3dTByT1NCZ3ZVNTAyT2wwSWIwSFpydkRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI5MjRfODdfMjRiOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWYzMGU5Y2FlNDc5MDhlNGU3MTNkODg4MWM2YmVhMzI3YTJlMTRlOGVmZjgyNjcxMTQzOGQ5OGE2YjZmZTk0ZmRhNjhkYWZkNDdmMzAwMmUzYWViZDkwNzA1Yzc5YjRhM2ViZjczODVjNDhjYThiNTA2NWIzZGNjMjcxNWY4Y2VjZDg5NWU1YTVmMzVkYWY1N2RhNGFhNGFkY2QzMTU4M2NiYzI3MTk1YmU1NTA5ZTIwNzgyMWQ4ZjgyNWY4MzMzNzljMTYxOTE3ODA4MjM2MGQzNDEzYjYxYWZmMTVlMmQwZmNlNmFmNTIyZjg1Mzc4YmQxYmViYWVlNjQxYzg1NzQ0N2QxM2QxYmU0NmZkMTg5YmM1ZjVhMTUyM2ZmODBiOTlmYzc2OWVhMmU3ZmFkNjdhN2FmNjY5ZDM2Njg5OTIzZmY0ZTBhMzZhZGRmOWM0YmQwOGY2NTZjNjQ4MDg1YzlmZjc4YjZmMzk4ZTM2N2JlYmI3MzczYmQwODczMTU1Yjg2NjI4YmI5NTZhYzQzN2I0ZThkMDE0ZTBkNDEzODUwNDlkMDhkZmU0NTVkYjdhY2I0Yzg1NDE4NTJjN2RjMzdkODYxODI4YzdhYTJjOGNjNDQwMjI0ZDgxNmFjYjE4N2FiM2Y5N2Q4Y2Y3YWFhOWY1Yzk1ZjhkZWI0ODA3NDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AfBdrzHy64MbXveppiywqEB4KcbHwICSGz-_83O077Zv2mu-UfbIp27XCU7w1m59E3w3y5bk_TUe3Czg6cP5_Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102924_87_24b9_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.948Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBxSGNvd0FxK3dla0UybFdBR040NVlZVGU3cXpOTnppdXE1Qi9DZGUrL1E3MHhJRnVpZ1JPUkFhK05JT0VYVGI5T2J3YVFRU0gvM2M3RU9aTDR3anNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTM0MzNfNjJfMjRkMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjcwMjdlYjE2Yjg4YTEzZDlkMTc4ZDkwOWM2MzJiODRlYWM0YzAyOWJlN2JmMDUxZjhmNDc0ZGY4NzJmZjk3NzJlMDk1ZGQ0YzAxNDYzMjYzM2Y1NDQxODk5NTY0ZmJiZjA1Zjg4MjJlZDdkYTU1YTI4ZWM2MGNlNDY2MzRlODI2MDI4MTA3YmE2ZDUxNDIwZTJlYWRiNzBiNGY2ZmZkYWMwMWVmZTk2MjBiYzVlZDZjZTNiMTA0Mjc5MDg1MDM4N2E2MzQwZGFmNzdhNjYzMjFjNzEwYjM5Y2NjN2MxMjZjNTQ1ZDQ0ZjQ0NzllNTRlMjBiMWVjZmVjNDMzY2VlOGI4YzVlYTA0ZTExOWY1ZjVlMDhkNWY1Mjc3OWZkZjVkMGYxNzVjZjhjNGJiM2I3NTAxYjkzN2MzNDMwZmJlMWU5MjQwNWY5YjY4YTcxYzJmMmJmMmQzZDY1YmFiNjBkNjAzYzEwOTRhOWMyMGM5YWY3MzEzYzBkYTAwY2E0N2IyMTE0MDhjODliYWQ0ODU1NDMyMDcxMTBmNWI3ODc4MTY1Y2EwOTk5MGFmMjZmM2Y2ODZiMDI0NDFiYmExZWZmNDY5NjY3NWM1MjY2NDZlMzBkYjY5MTY0ZTBlMGQ1MmQ3ZWQ0YjM4NTM2YWMwZWY5ZjA4ZDFkNmNlOTQ5Y2ZiZDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.D6JPBdQDBwRLpqko4Nuc4lscgUUf16sUxGG0uFi4Zqv1kp2_uSKN5dOYH4Aen4U0AWDjEFpemMRZqgAbOhChFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113433_62_24d2_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.951Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkFLQnRWcGpjdWN0RlNmTmlOMUdaWlNOdERrMko5cUhqd0dMYkxTNmErTWFBMzg4YWpsdTRyN1pGbXFoL1V1YW9BVXVLOTN2b1AwUnZvSXIzWmkyNlRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTM0MzNfNjJfMjRkMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDZiMDUxMmNjMjkwNTQ1NjI2MjgxYWExYmQ1ZDY2NGVjMWI4MWY1Nzk0NTU1OWVhMzA1Yjk4N2RlNmNkMzNmMWUzMDE1Y2IyZTVmOTI5MWQ2ODI4MDNlZDZiMTdmZTU3MzExNGM2ZjUxZjM5OGQxNzNjNDc5NjFkZjhlZDUxMDU3NTNhZmMzNDVkNWQ3MjUxOTJhYzc1NWU3M2M0MzkzYTUyMmRiMDVlNWYxMjRjOWI0MmUxODI0YWUxOGMyOGYzMmZkM2Y2MmNjOWMwNmUxOWQ4YzU2YWExMjkzNjRmMmY0ZmNiN2QzNDZjMTZjYmJlYjQyYjc1YzRiNGQ5MDZiODFiYTJkMDY5YTExYzAyZWRkNDhlNjYyN2U2NDZmN2E5OWM5MmE3NDMwMzk5YmQ0NWEzNmFhMzM2MmUyZDY3Y2ZmYWExNGEyYzcyNzI3OGIyOGEyNWQwYWNlODk0NzRlZDYwZDEzYzRhNDUwNTk0YmVmM2U2N2JlYzA3ZDFkMWYwOTRhZTM5YzA5MzNhNjAxMmY4NWYzY2U2ODE0MDdiNzI5NWFkOGU5NTA1NDE3MzFhYjdkMTc1NWRhMzE1NTVkZGFiMGExOWMxOTczYmRhYjA2MWQzZGIwMGQ1NzdhY2RjZWZmNGQ0MjJjZTZhYTE1MGQ3YTE0OWUxOTFjODZhNTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BJt_AZPHseEmic4wGoYP9WTF2m5yCxM0ppLbNN29JUv1OQjPo5OC7scesk-Y_tuUc3T6oit6vTQDsctPpo_zIg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113433_62_24d2_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.954Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFVVUQwbkJzMlE4Y3FCZXNOdHFycXowMGNERjNZeFhldW10cmxlTDhlNGpSN3FxVTVIdldEcHBCL1h1elZtWjJFZk1FYXNZOWhJVWE3TzJTaWpaL0h3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTM0MzNfNjJfMjRkMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM2M2FiYTM0YzJmMzk2OGI0M2NlOWY3M2E4OTIzZDJhZmE4M2YxNTE0Y2Q3ZTY3YzNkMjA3MDA2MDg3MTUzZjY2ZDIzNThjNjQ1MzhlYTA1ODhkMDA2MGIzZDcyNTc3MTE3Y2ZmYTBlN2U2MDA3ZGY2MjE0NmRhNjI3MDJkMGM0MzQ0ZjcyZThmZGVkMzc0M2FhYzFlYjI5OTg4YThlOTQ0OTdiZWRkOGM5ZWQzMzViMzFmOWFhYjRlOTYwOTcwZDkzZmY3OWM4NmNlY2M0MWQ0YmFjYjAxMGM4ZDZjZmY0MGVhNTlmZjg1YWQ4NGJhOGE2YTcyMjZjMTkyMGNjODdkZWUwMzBjNjZiN2FhN2M4MWUyYjIxZGM1ZjIwZjA3NDliNDQ4NmE4OWQ5YzUwMWU5NDM0NDNjNGQ0ZmQwOWQ5NDc3MTdjMzlmYjM3NzVlOTNhYjJhMjI3M2Q5MjBjODcwNmZlZmMzMTEzYWI4NzczN2U3MDkwZTY1MWM0OTE4ZWYzYWIxOTQzYTlmNjQwODk2MDc0M2I2MTNkOWViYTlmZmM0YTZmMjM1ZWZiN2Y0MzQzYThlNTQyZTEzZDZmMjVjMGFmYWFjNTY1ODczYWNkZmU1ZTE3OGJhODYwZmM3NjVjZGM3OGIyY2Y1MWFhZGFmZjZmNGZiMjk5YTNiN2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gIkajuxmwUKAODJ9-D1S-wOgw9C742t30gs_YC5UVgMeUjLlxP_TNx5mZgbJz1982QrpiXH3e8Cm1-XixRt94w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113433_62_24d2_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.957Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjUxTUUrT2JHdmZyakJINE5BODczZVNUaFh0Tk43a1RkN0FQd293N3BKWHJwYW5oa05hK1d3dHdNSXlIWkFIdUkyMkFMNWUzQWo4aGs2YkJ6blRlYVRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTM0MzNfNjJfMjRkMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODgzMjg1NGIwMjZmNDFkODIwZDEyYTQzM2NlMTYxYzkwZTAwNGVhMTNmNmM1ODcwOTI1MGEyZWNhNDQ3MTRmMTVhN2RkNjZmNmJlNjhjZmQ0ZTdiNTVhODdkZTEyNzAzNWFhMTdjYjk4YmViZjdlYTc3MDNmNzdjY2Q0YjY1NDFiMmQ0YmU5ZTU4M2NjMmIwOGU4MDhhNjI5NDMwZjNlMTQ4NTc0YWI4OGU4ZThkNzI1YTE4ZDU1M2QwMzgzMWI5ZTk4MzI5MmJmZDNlMTQyZWU1NDA0NTQxOWExYzY3OWE0OTc0ZTBhOTJlZDIwZTE2ZGQxMDE2ZWJmNDE3NWI2ODRiYTBkMzJiNGMxYmQ1ZDY5YzJhZTk1ZDE4Nzk1ODBhMDVlZjVkYjk4ZGNkZjY0OGI2OTQ2ZTYwNGZhNTAzMGM3NjAzODg4ZDY3NDIyOTA4ZjljZDRmMmQxZDFjMGY0ODNhOTVlZjgxZDE4NzA3OTllMWM3OWZjNzVlMjg1OWQ3MGJiN2FkMTA5ZjRlNGQyNDUyODY2Nzg2OWU3NTExNWNjMjI1OGI2NmRkZGU3M2JhNDUxZGJiZDM3ZTViYjUzZTA1YTBhYzk4MmU4OGY3N2UyN2M1NTllNGNhZGFiYWFkYWNjOThhMWY1ODE4MmE1YzRmN2EzNjAwZWM2NmE0NjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FUSQbrKtftH3kYrAzbjUwo-3xmOXrOfBJdDEO3LK3jG_zMhjtQvkTgipG20S2YqH8JW46X0ga4EVvDp7t7JLFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113433_62_24d2_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.961Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJzZXI1aTU5ZzBoQk5nOGFobGs2V1ZlSUNlWXg4OFpSUVJHT3VJVWYzZlArTjVLcHhZdTVNR1p0U3lpYmhGL2IrYWp2MStBWERiK3lCV2szTE1CTDZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTRfMDVfMjRjNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWViZGFiNzY1MzAxNWY1ZWVlMjliYzU2ZThlNWFjYTgxMTc5ZWE4M2YzZTFlYmM3ZDcwNjYxNDY1NjJiOTlhYWJkMmJhNzJjYzI5YTFiYTQ5YjMyMjRkYmNkYjRiY2NmYWIwYWFkY2JmMTc3M2JmMWVmNTg0YzYyMTQ2YmVjNDUwMjg5MjA4OGE4YmNlOTUzY2YyOGYyYjljMWQ4NzNkYjRlMjY4MTkyODA4YzY0MzQxOWJiOTUxMmJlMjU2NDFlMzdiMDlhM2RiYTk4YjNjMDM4Y2M3ODNhODJmY2Q3MDdhNDE5ZjRiYjJmOWU5OGY5NDMxNzBlMDhlYTZiNGQ1MDkyOTI0MTkzYmJiMGJmNTE3MzY5OGZlOGMwYWZjNzRmOWYyNTdhOTNjYjY4MTFhNzM3YTFmMWNmNTZhOWRjOWFjNTIzODBjMDU2YTljMWVkNmY3NTMxZTgxMTA0YWJiZjVkYWU4MDhiMjI3N2QyN2FmMzEyNTVmZjA2ZGE4NmE4NTI0NzI1Mzc0ZmQyZGI2MmU1MzRiYzdlOTFlNGMyODAyNzVlOTI2OTZmMmM5Y2I3YjM1YWVhMzkwMTc2ZDMwOTM1MTEyMDcxZWU3YmJhNWFmYjRiNzhhZWU1M2I2MWU5Zjk3MGMyMzMzOWVmMjVhOTNlNjQ2MzZjYzliYjI4YWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rJPRfa7_XDz-J3FynuocND3qD2Wfevu_Z843QUq6u2SBY-Joxr-4oPbqzvK2V6ZPaKaZpHYbS6BBUwfjepg1kg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103354_05_24c4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.963Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlZKY1JzdWJyTHVDUzRhamh3SUJLY2NoOWdSM0UvQTdZZG9COEpJVUdjbGV0M3ltKzloS1FmM1dmbVVyTUpKMEJNSVpidytRK2orekQ0VjFwc2kxeWNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTRfMDVfMjRjNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjdiNDY2ZWMxYmRmMzc4MjFhNjdiOGRlNjEyMTM3ZmM0ZGYzZDJhNzhjNGU2OTc5YmJlMjhjNmFiN2E3YjQ1NzEyYThiN2NhMjc3NzYzMDVhOGNlZjJlMDUwNzM5NDVlMzhkM2ViYWQ1MWM0NzM0NjYxODAzZGFlMDE5OTIwYTZiYzNiNDc4MGI0MTI4MTRjNDY2ODhlODQ3NmU4ZmRjOGI5NDA5YTdlZmJlZWE4Yjg4M2RhMjUyOTZiM2IwOWViNjM1MjU3Y2Y3ZDQzZDM3YjBhZWJmYTZlYTlhYmE4Y2I2M2Q4OWVhMDdiYmMyZmExZGUxNzRjYjk0M2RlOGIwMjc0MDI4MTI4MjM1YTE4Y2E0YmNiY2I5YzMyMzkzZmRhNmIyNzMzMDFhMWRlMTg1YWQ5ZWVmODdiZTRmMmQ1ZmEwMzUxMTk3OThkMGViOWMzODBmYjAxZTUwYmVjOWFkZDhjMGZiYjE4OTMwNDc2ODI2ZDljYWVlYmYxYTEyZmIyNjU1MDYzYzIxYmNmOGFjNDU4MDY5MmZhYzNkNjVkYjJlMDgyMjMzODEzZjA0NmM4MWI3YTg5MmU4ODJjNzhiMjdlMjk3ODg4NWI3YmQ3OGU1NjMzMjdmNjNjNTJiMDlhM2MxODIxMWUzMmNiZTNiOWY0YWUyYjNkZWQ2MjY1NGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.op0Q23T227rF6_yLbShQmbndh6Jkxh9TRuTBjMctOSm65mJqwv_u6FuClN_fo0usrSqcaR9BkCGOrsWpjZvPWg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103354_05_24c4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.966Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im05c25sTUZVczM3UForcFdvZXdpMzJxdTR3eS8ydGxsZkw5VXg3elFLa25qM0M2TmxPdi9IN2hHOEs0c2RIVzBzV28vUVhRL0ovQ0ljeGI4Y0hxR2xBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTRfMDVfMjRjNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzAzM2JhYzRlMTAyNjNhZTc0NzQ1YWMwNzA0MTI1OTY0MjMwYmZkZTA3YTgwNjc4MzBhNzgzZjgxNjg4ZDY4NjMyN2UwYzRiOGYzMDQxOWYwODA1ZTMyYTYwYjEzMTZjYmU2MTE4NWY4NDU3NjBiMGM4NjFiZTliODY0NjZmNDUzYjM4N2RjNDc2OTVlOWQyMWM2NzEyYTlmYzZiMWQ2ZGMzNjBhNzA1ZTY3MDRmOTBhMmJiYmY4ZWVlMzA1ZDhmZWY0YjlkODBjM2UzMTY1MTFjNTIwZTViNWJlNTk2ZWJiNjA4NzUxZWEyYzcwNmQ3Zjg2YTEzODBhNjE5YzI1YjdjOTdhOWQwZTk4MTBlMTFiY2RiMDBiMGZiMDQzZTI4ODA4ODg0MzVhZTJjMDRiMWMzZGYzZDcyYzE5YmYzZjFmZDg0OTRiYjgyOTdiYTk1M2Y1ZGUyZGRiODE3YzUzNDljZDkxZTQ4MjRmMDMxYTZmNDg1ODNhOTZlY2I3OGRmZDMzMmIyMTQ1NmZmNGYxYTg3MWYyNmQxYmQ3YTM2MTUxOWY0MmRjYjU5ZGVmZGE0OWE0MzBhNzAzMzU0MDBiYzk0MTQyMmRiYTQxZDQzZmViNzNiOWQyOGRjNWM1ZTViYTQzZDVmMjRhZjE5YWEwNjdjNzMzNmVmYzE1YjU2ODdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zF6BOFStMWzmoESO31aTFbijv15nASCDmo0N7kaS24OwlOMURqebLgSb8z_yFdh_KD9l_2yiyzSjgfxeP1rIwQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103354_05_24c4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.968Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IndZWHg2TS94VW5taUNBUWNSSldHNWMrL0RIQ0FhUDRveDljNjJVNFVkWWFqMWNCTDY2MDlDRlFSRmlBN081V0xmd2IwY0drSUpEWUxYcTNDNVRRbmlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyN18xMDMzNTRfMDVfMjRjNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTJiYzc1ZWI3MDYyNzJjZDM3NzBlMzdhZTNhZjU2YThlMmQzMjY4YTllMjczY2VlMjI4MzgyNmRhOWM1OWUxY2VmOTIxNTViMTZjNWRiMzBhNTFmMzUyNjBlOGRjMjdjYmY3Y2MyNDdkM2Y5NGVmYWE4Y2U5NWRlNzQ1NTk3YTc1OGJmOTFkYjcxMjk0NWVhNzgwZDZmMmUzNmIzMDk1MjYzODM1MzQyNDZkMWJhNDk5MmZhNTg2NDQ1ZDZkMGMxMmIzN2UzMWY5MWJkMDJmMDM4MGFkY2JiMGZkOWYxZDhiOTgxYjAyNGZjMDdlYzE5MmFiZTRhYjhhZGU2ZTRjMGQwOTFhMDdjMTcyODlmMWNmZmEyZGFhYTc4MTFjMWFmZWNhNmUzZjc5YjNhZjM2NmRmZThhZWNkNjNlZjQ1MGFhZDRlNDhmYzFmNjIwZmIwMDUxYmI3NDNiMTczMTNhZDc3MWI1NzZlMTIyZjJkNWEwZDYyNzc0MzVmZTA5MzM3ZGVhZGUyM2QyZWI3M2UwOWFlZjYyYWYxN2UyM2NkODI3NzExMjBkNWY0ODMwMTExMzQyNzFkNmM4YjAyOGZlNmUzNzMwYWVlYWI2NWFjNjc4ZmIzODEwMTlmNzY5MjE0MzQ1ZDIzZjljOGRkMThlZGEwMjhiMTQ1OTdkNDE4MmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OyPx4OtKwsGFWbK1wyNBda-aTtqvInqk8oW_emTu9iUuCiyGzjT9QQDJGOGvYTh7usuEXaBlje-azcL7uQzBOw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231127_103354_05_24c4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.971Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InlxWk10cFBqOUpjbnZUNGRyMkFqTHFvaXBxRzE3d3I4aVZvNzhET2VMU0tMN2xncE5KVDlCVUtudk9TclhPNVFsODZYbWFYYXR4TmV2Yk9aY1lXbVRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxMzhfNjNfMjQxM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGRiOWRlNzVkYzYzOWIzZjIyZDNjMmQwYTQ5MDI3NGJkMmU4ODRiZjhjNmQxMmIyMDY5OTQwZDVhYjhiZmQzN2Q2MGQ4MWYyZDFiMzc3M2E0ZThkZGM4ZGExMWRiNmUxODA5ZmJiZTI3YzI3M2EyNzZkZTIxMWEyYjMxMzY3MDE2ZTI4NGIwYjVlMjRmYzMwNDViZmVjNGZjM2JjZmFjNjM2ZWE2YzZlYjk1ZDNjMjRhMDFkMzhhMTA0OTYxMjc2ZDQxOGJmMTA3NjZiODQ3OWVmMTc2ZjhjYTYxODNlMjkyNmJmYWY2NjA3MjM0OGE3YWRkMWIyMDIwNTc5MmVlMDNiY2ViODZkNjE3YzExMmRlYWZjNThmYmYxZTk5MTdjZWY1ZWM1ZDk2NTkyNmIzNTFjOWRlMTQxYzFhOTVlNTRhM2VmMDI0YWU0M2M3M2ZjYmVhZjZkMGE5MWM5NmU2MTU5ZDZlZjIxOTQzMTZiNGRkZWVjNWE2NmM4ZTk0MGQzYjkyODVkYzc5ZGMwNmRjOWU1M2YwNjE5Njg0ZjAzZDczNmYxZDU2Yzk3YzUyMzc3ZmJlNjcxYmI4ZGViMjYxNzZiYzIyNjAwNWI4MGIxODZkN2I4ZjVjMGU3NGMyOTNjNzU5MzExNjIzZDBjOGNhZWUwZWEwNGIzZTJiNDQzMjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JNKjaSLMJQu_FWAgk3taQIycy1SPczQv8uzfQQXBl2i2uI9yizYFJGjRFEBimH8JJG3ZLWW8CNiYxKyE7aX_8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112138_63_2413_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.974Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJ2b3JCZVFKdU1TYTkzMVQ5b1hnVS95bS9uNzZScG1vUFR5S0RBVGE5MUxkeE1ZbGNneVFCVDUyeG1FWmRZUmZ5OVpWcVZVTFpDeFVNOFc1VjZkZW5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxMzhfNjNfMjQxM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2E1NzU4ZDY0ODhkY2Q3YzhjNGFhOTEzNTY4YjQ4N2RmYzQ2MmFjYTMyZmQ2OWM0ZmE4ZWU1M2ZlYTlmOTUxMWFmODZhMmQwZDAyODgzMTRmNmI1ZjMzZjc3MDUxMTcxNGFhM2NjZmIxMmJkYjFiYzRiOTA1MzYxMjk2ZGE5YTdiOTVjMDJkMDU0YzA1NWZkMGY3MzdiNmU2NmU1MGI2MDEwYjkxZTgzZmE2OTEwN2Q0Zjk0NTEyMGU4ODQzNzk1ODY1Njg5NzM2OTY4N2RlZDA1YjU1MDBmYWFiNDlhOGY2MTE5ODkxNWNhYmEwMjQ2MjE1MWE0MzU2MjdjNDIyMmQ4OGQyMzM2ZDIzMTQwZjU5NzFjZmQ0Zjc3YmVlZWQ0ZjUzZWM5NGVhNTY4NjlmMWUyNzljM2NkYmMxY2JmZGM2NjI4MmZlNzcwMThmYzcwMmI0MjAxMmVmZjA2YjhiMWMyMGM5YjkzZmE2OGVhYjE2MzA0YzRmNTZiMmFjZmFhOGIzYmYzZTI0ZjNjNDI2MTQzNTQ0ZGFhNmFhMmZhZmYzZWIyMDQ4MTNjZmViNzdiZDE0MjMxN2MwODY1NDc2YjE3ZWU2MjExMDg0NTFmOWZiZjMyZDg3Mjk3MTdhZjdmOTc3YWI5MWUzOGU2NDU2ZmFmZmI0OWJhZjQxOGI4YjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QXdD9b43F_vzEqcGTxag9GVfG478UXBoCetPamYL-o_AMJ8ivDJRyCVJdaLAnoMLV2tF8_6o9qhaTCEjfCByaA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112138_63_2413_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.977Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkVCc2ZsOE8yN2lkVU9BU0lybm9aMTdlRzBqcVVlZEdxRFEwMUtSb0UzTEpJeUV4c3lFSkNBQXc1S1ExeXZsUFZOMXpXNHRWRjUzRU9jL3dQRWhDRzd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxMzhfNjNfMjQxM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjg4YmYyNTdkNmZiNDYzYTZlNWQxY2I5ZTVmYTNmMWZlNzljYjBmNGJkMTljMWZlNjUwOWY3ODQ5ODk2ZjRkZDA2ZjE2YzU5Nzk0MDVlNDgxMmExODBlNjQzNzI3ZTE2ZThjOTEzNDBlZDExMmM4ZGQzYTlhNjkwZjRmNjM3YjUwZGQ1YzMyOGQ2YTgyYTljMmQ2YTUzNzZlZjBlYzNlODgzNjNjMzU5YzFlNTI0YTk5ZmM1YjRkYWZmZDkwNDM4NThhMDUyZTdkY2ZhMGVlNzE4MmU0NjYwNmY2YmU2ZWU1ZTg0YzllMWIzNWZhNGFlZmMwMGE2NjFiNDc2YTRiODZjZmJmNWNiNzhjZGM2NWUyYjYzMDg1YWI2ZTgzNzNlYjc1YTM2ODQ3ZWZiOWE5MDQ3YjMwNTdhYTg0MWU4ODU5MTA0YzBmNjBhZTI4ZWI5YzUxNjA0ODIzNDE0ZDg1N2E4ZGU0MzFhYjIwMTM3MTQ5NDY4OGNlOGE3OTdjNDBlZDY0ZjgxNmQzZmIwNjQyMzMwZGVmNTg2ZWIzMWU1MTA4MmQ2ZmE0ODZmYjVjNGExNjBmMjI4MDRiNzlmYzFiMDAwOTE3MzMzOGFkMDFjYjA0MzFjODQyYjI3ZWMwYTRkMjk1N2RjNDI3MTI0NzA5YjY0YTUzZmFhNmE4YzJmNzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FXk3yPLhMBwPoNVRruuU2phMNKGib9vD2LVRmWONro-TAfk2UwrXtiMdNNtpulNCq6GvIToAVuVUJFf1gxQ_UA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112138_63_2413_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.979Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlY3YWgxcGVVNjVEN0gxZnE3MmlRL0FSVFVwK2c5a3RVeFZIZ2d0UnUyNDU0amxMY0JOR0dCaGVPcmk3YlB4Zmo3djZidFkwUVFQenhnQ0RCdnFKMGtRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxMzhfNjNfMjQxM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODMwM2JkYWQ5MGU3NzVkMmU1ZDgyODI4NjVkODAyZjFhZWNkYTEwOTY5ZTlmYzkwYzE5MTZjMzM0ODU2ZGJjNDIzZTFmZDYxZWEzZTZkMzZmNjNmY2NhZjI3YjQ2OWEwNDhjNmI3ZWY5Nzc0NmY1NjIyOWNlNDliY2IxYmIzNGFlYTU1YWRjMTU4NTE3MmQ2MDZiMmFmNjA1YzA3NWY5YWRiNTk5ZjAxYjM3ZDBkOWUyOGU2NTYwZDYyNmJmZDA2OWM0NDY3M2UzZTQ0MDI3YzU2ZDUwNWVlMWYxZmFjOGM0NWY2NDJiYWNhMzA5MTFhOTY3YTE4OThmNmIxOTAyYjQ4ZjFmY2RmZGMzZTU5ZDM3Njg5YTYyMjUyNDU1MmVjODRjODE0ZjhlN2UyNDUzNGFjMWUwNjFiZDllY2RjYjRlN2NiZGEzMjIzMGYyYjI1MDc0OGY5NjgxMzZlYzM1OWYxMDhiMDNmNzY2NDFjZGM5MTEyYjJhOWMwOGYwZGUxZGVkZjAxYmY5ZDM2Y2JhNzAzZjdiMmNiOGI5NmYxNjRkNmE0OWQwNDAyZTBkNmE4Y2NmMGUzYTI0M2JmZTE1YTQxOWY3ZDI3ZTk4Y2YwNDE2NDk1OTI4N2EzY2FkOTgyMDM2YjJiMmQzZjkwODgxOWUxZDlhYWM2MGQxOGZlYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Osi6h9CIFh0worcBznwfcgnJRzZO_iepHMP0z4YwEisMSmqvLTWQTeo9gcAY5QczgCX3R1a6RHHQ-V3EdKrJVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112138_63_2413_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.982Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik5MVnNFYVZ6VGFDZXRJdllUWklPbW1EbnBkNEdrbFRKUTZWR2FmL0FINnFnaFVmUEhTK2FDYjJQWFAzNWpWVytQbHZWM2V2UWdWTTc3Q1FFV053U3FBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNDlfMjJfMjQ5MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDE0ODY1MGRiYmZiNTI2NGY5Y2VhNzUxNWJlZWZkNGEwMWI2NzA3Yjc5MmUwYzZiM2FiMjU4NjRhNDdlZjA5ZjM4MzllMWMxYTZhNWFiM2I5ZmE1NDQ0NTI2MzBkZjNiNDhkMjQ3ZTFhN2NhZGI0MjczNWFiNGZmZTViM2VhY2UwODEwYzY3ZmMwNzkyZDhlMDRmNmY0MDQzMGQ3ZGYwMGViNzU1ODNkMGU2NWNhNTAxMTc5ODg2Njc5YjVhMTM1MmJjNjRmYmZkYzI4NDNlYjJkOTg0ZGUxYmM1NWJhZTRkMzVkZmU4MDY1ZWRiMmQ2NjQzYjY2NjkwOTEyNDcyMTdiZWNlNWU3YjczYTc5NzhkZDdjZWY5YjQ2OWU4ZjFmNDA3YzhlYzViMzFiM2E3NjUxNThmMjcyZjYxYzE4MjU1MjU4YmY0OWNkNDFkM2YwYmIyZWI0YzRjNjgzNTE2NmI4ZWM5OTBmMTJkZjgwMTgyMDZiNzc2ZTFlNTNhYTk0ZTQ1N2QyNjc0ZTIwYzQ2YzYxNDc3OGJkOWIyMTQ3NDE1NjJlOTY5M2E4M2I5ZDM4NmEzNGEwN2FhZGNhMzUxZjZlMDVlODU3NjJiYmZlMTFjMjllNWJlZGQyMWVlZmUzODQ4MTE4ODgzNzhhZjE5NWM0NjE3MDA4ZWNjNjI0NTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.o9N3z6NWxPjHof-VQfF71L-jkgSpCdmx7KLEoZ0v93TCoelRFCEW19kWw74vS3X_VWNGV3EfhPYbzbqAkzE9Tg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110349_22_2490_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.985Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImpBSVVQYXRXeXVzV1I5N0RaS05zMlZKM1VVQVFEeWFaczJSM0R2bDJzMmV2eW5xOEFlaUVuVkV2dm5VaEliV0VnT1V6UFNrekMrRUx2bm9QZDRjR2ZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNDlfMjJfMjQ5MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTdlZTNiMzI0MWM4ZWRmYzAzOWYyNmVmMjQ5YWJkOGYxZjVjZGVjYTI5MzFkMmQ5ZDcxZThjMjVlNGNiYTA4NTk5NzhmNjUzMmMyYmY5YjBkODBkYWE3ZmE4N2Y4YTg5ZmI5OGVmMDI3ODk1YjYwMmQyMDg1YTkwY2M5Y2MzZDQ2NjJiMzliMGNjNGU2ZGYyYjAwMjljY2I4MGQ1ODBlODhlNTMwYjY2ODhkNDI1OTk3ZjYzZDJhNTc5NDJiYzlhMzFmYWZlMjFmNmQxYmJmN2JlNTc2OGUyYmVlY2EwMWU5ZTdmZjI3ZjdkMGYyNjI1OWZiMzhkMzJiNjljODAyZDRjMWVlZWFjYzM5NTY2ZWVhNDcyOGQ2MmU3MGMyNzRkYTUwNjhmMDBlZmJkMjc5NjU3NTg2YjY4NzY4ODkzZWM4ODNjNzFhMjEzNjBmZjRiZWRiNTg4YzA4ZWYzNGJjMTlhMjBkZjVmNzAzYTdlZDJkNGRkODcyNTA1M2E3MWM3NzM2MWEyYzA5MzIwYzlkZmQwNzY4ZmMwYTVhM2M0Y2JlZWM0NDM0MjQ5ZDA1YTg1YWU5OTRiYWU4YWQzOGM4NjZlZTE3NWU1NTcwZTczYTU0YjJkNGEyOTg5ZThkOWFiY2Y2NGRlZGY1NGY5OGY1MDA0NWIxNDA1Zjc0MTU5YzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jYXYe4RVImgVhO6GS0ZMrgeWaRHfPaCwbVMLDWeReopvoR6MMtbYZnfmKPqMwgpf6UcQE3G0IS5_Tl7zRBFmxg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110349_22_2490_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.988Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpNUDg2WUhXUStZajFtdDAyN0hSQWlPcFk1dTA1S2NlVGtCczVaRzJsNzJPV1UvbVI5VFZidE82cE5kUTZ3NHA1U2hkZHYyVDBEU2h1U1pYVC9xMXdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNDlfMjJfMjQ5MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTg3MDBhOTgyYzNjOTg4YjVmMGUzYjg5N2YyZTM0YTQ3MGUyNzgzY2EzNjU2NTA0MWQyOWQ5NjU5MTRkMzg3N2E2ODA1MjNmNjM3MTQ1MDZlOTM3MDAxZDAwMjA3NTJlNWY0YzRlODllYjllYTQ1YzFlODZlOGM2NzA2MTkyNTY2YzU1YjdhZDY1NDQ5ZDMwNDI2MzM3MGYwNmE2YzFlMTgyN2VjYmVjY2YwZGQ1MTU0YTc1ODIwM2ExYjNjMTgwNjEzMjJkM2ViZmQ0N2VhZmRlN2JlYWNiODQ0ZWY4M2M5NThhOGRhY2FjZTRkYmJkZDdlODZhM2FmMTg5YTAwZjhmMjczOTFmZmE0YTI4NWU2ZjYzNWNiZmI1MGY1YjM2ZDc2NTFmMWJlYTUxNGY4MzM3ZjBkMTI5YWIxOGEyNGU2OThlOTBmMzhlOTJlYjQyMzdmNjQ1YjJiNTJiMmQzMzQ0MjJhNGZhYzcyNDUwY2M0ODFiZGMzNGM0OWU4YzBiNjQ4NDE4NzIyMzE1YjQ5YzQ4YmMwZTBjN2Q0NDA4MTNlZmU3ZDFiMGIzNjFhNTk0MTE2YTY0YTg1MjRjM2UwMTVlNjI1ZGRhOGQ1MDIxMDJkNzUyM2NlYjUyODNlODhkZWU3MmVlYjA4ODMwYTgwOTM5YjA2NzUyZDU5ZTE3MDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bzHgxKr_nnTEJddyD3TAsHezpruQFYuLWBNuS2CaI-aGIVWs-fxVlBxCkEzGZcfJFpLg61D7YWLT-kZJFzToBQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110349_22_2490_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.990Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjhiYkdZUFBreURURGsxYll1UHRxWFN1MHFvTWh4dzdsZDlwY1M0UUlETXpjekFUS3NpWGx3Y1JUY0lHaG5WTWlCWmoxK2oxekxPL0Y2d0doSE4zTXJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMV8xMTAzNDlfMjJfMjQ5MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODhhNWM2YTYzNTkzZDJkMGNlMmQwNTQ2YTZkNzhjZWUzNDI1MjcyMmRiZGQ4YzNjODlhMzM0OWJjODFjOTJhMjYyOWQ5OTFlMjliZjc3MWU0NzNhYTdmZTM5N2FhZDJiOTEzZWM0NjFhN2QyNDI0MzViOTZiMzhhNDkyZGY0N2VlMzMwM2I3NzY3YjVmNmM1YzNhMmYzNDI5YTFmNTU3ZWU0NTcwYTRkYmVhMDc3ZjZlODRhYTI0ZTRiZWM4ZWQwMTI3MmVhYTA0NDFjNmRlMGNjYzI4YTUyMGQyODNhNmY4NGViMmU4ODAwNzlmZjZmYWMyZjQ0YTA2MzIxNzQ3ZTcxMDg5ZmFmYjE2NzNmMDUwMmE0YWNmNjUyMmM0ZWRiMzVjMjRkOTAxOTU2ZWNjZTk5MWNiYjI2YTg2MzI4M2M3Mzk3ZjgxMzI3MzcyM2IzN2MzOWRhYzdlNjZiMzhlMWRkNWEzNGM4ZTNjMWU5YzljODU5ZTczZDlhM2I2NzE0NjVkMTM3MDExMzc2OWQ1YWZmNmQzZTc5NzdmY2Y3MGQ5MGJiZjlhNjFlYTQyYjU2NzYzZTkwODM0NGYxMGJlYTIyZGI5NjhhYTc1YmVkYmIyMmE2ZmFiNjM3MzA4ZTVjNWQ2ODAxNjg2YWMyMTBmMDU2OGYyY2Q4MmY0YzQ1YzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uocRrW4bo6Rchwmt_pabKIgTkXuVDyjxOpoBmWMua87-kmhtx1eJKCS8ZF9kFgt0StuIYGNBBgZyLCwpT29Htg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230311_110349_22_2490_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.993Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InRzaEEzcnV1ekxVSjdCR1JkMHUvMmptZVpFQjFTTm5oL1cxYjZWTEpRUlFPbTRuQWcyVnEwdGV5WXQ3Q3lUMENuS0ViQTBnTmtZQWNEU1hNeXFQRU1RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNTZfMDBfMjQ4Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGEyZWMzY2M1MmQxNzkwZmMxN2JlYTUwYjA5NDAwOWRjMzVjNjczODE3OTBmZTIzYTE2NjMxNDAzN2RjZGQwZThmZjg1MDRmOTIyM2ZlNDZjMzZlNTVkNWExYWY0ZDA2YWIwOWUyMzAzOWM3NDMwZGM5ZGNmZWZjNzg2YWZlN2U3MjdmMzkxYmNkNTMyOTIzYmNiMGEwNmU4MGI5YWQ3ZjExZDIxNjRhZTM1NjU1Mzk3NWRkY2QxOGRiN2EzZDcxYzA4N2FhOWEyNGQ5N2U0NzY5NGU5MDE2NTY1NTZmNTM0ZTMxMzE5MGY3NWNmNGZjYjI3ZTQ3MjFiYjE5MjJkNTMxNDhiOTVmZTcwNWVlNDBlYWUxZGE0MzY2YWI2Nzk5NmMwNmVjMTUzMmE0NmFkNmM1Zjg4N2NlMjE1NDRiZTIxNzlkMjJkNGNkYjYyZGVkZThjMWQ2MDA4YTRlMjg0OTZmOWQ2YjJhODk3YTg1OTEyMjA4YzIzOWUwZTdjNDIzNTdiMTQ0ZDcyYTFlMDZmNDM4NWI2YjhlZThmYWIyNDNiMDE3YWY2MGE2ZTM2ZjhhNjVjYmM1MDg1OGM1ZTVmZjc4ODRiOTkwNzE0NjIyNDBkYWQ4NGZmOTI1NmIwODk1ZDczZTQzNDlkZGNmZmU4YWM4YzYxYTIyOTM3YTg3NmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pOaekh1hp-NblFVY1CVZQhALKc9CnMaJYuK-xu2kfKB9TLHmUVYpf6Fai5XnNfElXiltO5QZc6lAuVXPeUdo2w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110356_00_2482_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.995Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik9xcVlPa3JVMS9hZERheVR4eHBFeXlHcjFTV3k3S0Q4eDlwWkNLTVh3ZFdOM3NTS21Xd0kxRUVhYjJHaVhyTkNFM2R1M29SWEErQ0RHdTFod3E3RVhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNTZfMDBfMjQ4Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTZjY2U3NTkzYzE5OTIwMTRhYzAyMWNlZjJmM2ZlYTY2ODA1NzcxYzMyODEzNjc0OGY1N2JmMTRlN2IwMDUwMzIzMGYyMjY2OGM5NTZjM2Y1ZDljMGUyMzdlMTIzZTU1NmQ2NWQ5MWZjMjk4YmQ3NDViMzQ4OTMxMGYxNjlkZjgxNDBkM2QzNzg0ODc4Yjk1NDczYmE5YjNmYTUzNzhhMTk4MTNhNzg0Y2Q3YWZhNjhjYWNmZWE2ZTBiNDZiNDkzZDVkZWUwMGQ3ZWEwOTFmZDU0ZWVjM2YzMTBhNGI4ZmYzOThhNDFjMTY3NzMyZTMxMGM1MGRkOWU3MWQxNmY2ZWNkMDk2NDY0ODg2ODdjNmM1ZGI2MDFmNjc1OGQ0Y2Q3NjMwMGMzZmE5M2IxMWRmZWQzNmYyNTEzMmZhZDA2MWQ4Mjg2Y2Q3ZTAwMGE2YTUyZDVlODMyMTllY2ZkM2ZhNGYyZjk2MzNiMWM4YmM2MzczZmNmYzQyOWE3NjQ2NWQ1ZmQzOGM1YTlhNmUwOGVlYWVjMTcwZjdiYjMxODI1NGZiZTRkNmM2YTMyNWU0ZDQwYjg4ODlmNTdkNDg2OGU1NDZjODNkNmIwZDI2NzdkMmZiNjZhODFmYThiOWE1ZTdiOWY5MTBlZmU5YTA4NmMwNGVmYTg1ZDUxODUyNTRlZGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FnsJZiFhd7Jt41X4vsMvpIWIIpNOiaRcrZJg4n6HvLhFXIvty2xEAIRU9n2rXiPVyiB8XLVoGokuNrjdr6t-vA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110356_00_2482_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.999Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imh1UnJQM0Y3OXl4Y29YeHZIQ3NqaE9vVUdzNmR3Undkd0NRcDdpR0VBYzVHZmZPVWtBcTFHNHgxOXlqdS9JTHFzUWxXY3ljSmJoTmtjUm1kbkRIQWVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNTZfMDBfMjQ4Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDk2NWFkZTVmMWY2NmIwYzI4ODdlNzlmZjg3YTY0NDkyZTAxMmFlOGU4NzI1YmEzOTA0ZTg0N2FmN2ZlYTZjYzQxNzE2NTI3ZGZiYTg1NzM1NjQ5ODhlZjMyMzI1Njg4MGFhNDIyODMxMTBiN2NlOWVkYWZlZWM0NGJlNTFkMzlkZDMyYmRiYzcwMzY2NTBhYjA3ZmM0MDBjNmI1MGEzZTUxNGVmMTFhOTU1NDI0YTg4MDc0M2UzNGVjODcwYmRjNzMzNjE3YWU0MmE1MDYyM2YwMjg3YzgxOGZjYWFhZjU3YzA2NTNlOGFjNjAzNzRlODU1NzMyMzlmNTc2NmI2NjFlNGJhNWM4NDkxMDRlYWRkYmVmYmEwYTYwMzdmZDhmYmU2YjhmZTg5NTg4ZTQ1NjU4Nzc3MzYyMzkxYjg0ZDBjYWU5OWEwMGU5MTBlNWFhYTM1NjNiYTA3OGNjMDE5NGIyMWJlNDIxOTIwZTVmOWJiNjE0ZWIzNzQwMDlhYTU2Zjg3Mjg3MGE3ZWI3Y2FmN2ZhYTdjY2M1ZDMwYTEzMjU5YWM5ZTYwMmZmMmU1NDJiZTFlNDVmNjk4ZmY4NWFjMDQyZGYyYTMwODkxMmYwOGFjNjYxZjhiNGI1Y2M3MzBkNDM0OTFjYzFkY2Y2ZGYyNTUzODI1ZjI5OTI1YjQ3YjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BuXn-BRT7MDxkSi41YIGOcaz81LKUql1ixyqdSU6ufoaz9f8IFUWMwsW2zhk3owAmNNyWsAJZIHG8P_iPs0wDg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110356_00_2482_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.001Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlYvcXpjTkFkS2NQZk5aSkNVOGpJSmxvRjRMY1daeDl6eXg4aE16MzIrbkl6Sjh0aHlFMC9maUhmOVkwRWFTSkJpeEJhWmJIZmgxNUsydzQvcUl1NnlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNTZfMDBfMjQ4Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2JjNmU4NTYwMzk2NmZmYmMzZDE4NGEzMmUzZTM5YmQxMzI3ZmZlMTE4OWNlYWQ1ZTNkY2NmZDQ5NDczNzYxZDNhMTg0MDY3NGJmNjY3YTM4ZmViMTJmNzI4NDQ0MWY4NjE3YmRhYzJjNmNjNWYxYjkxMzU4N2FmMDE4YjVmNmI1YzBlZmNhYjI0ZTNjODkwNDk5NzhkMTJkNDE1NjdmZmE2YmVkMGY5OWJkOGQ2OGNkYjQ3YjRiNjFkZDJiNDQ3MzM0YzQ0OTAxZDk2NmM1MGEwNmI3MjllMWM4N2MxZTFiZDU2YWUxZGIwNzg4OWUxMjQxMDI5MTY4MDcwMjM3ZjE3YWRkNmY2ZWE3ZTFhMDlkNzk2ODczZDdiZmMyNmQzYzQ2YzJhMDNlMmI1NGJmMmM4NzkzOGUxMDAxNjZmMzYyMmJlNjkyMTlkODcyYWQ5ZjVjYzZjYzY2NDJiNjY1ZTk1ZDBmODRkYjUwNTQ2YjIxNzQ2MzgzYjlkMWZhZGIyZDdhNmEyMzIzNDYyZjQyZjBhYTY0NmZlNmMyZTgzMjY3M2E3MTUxZWNlMDk3ZmZkNDhlNmU3NjQwZTQyMTk0ZDViYTczOWE2OWQwZmU2MjQwNTAzOTZlMDQ5NDI2ZDIxNDFlMjRhMjZkZjU3ZjQ1NTdjMzlkZjg5NmFiYTg5ZDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JGxvE5dIyBHX8qLPXOjkNNTGhMqP3x7dGylY65VuXEYXk-5dbUDg-4nEc2BKyX1JbJpQXYAqDxKaANlK363RDA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110356_00_2482_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.004Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikl0dzZQVVYxSWgzV1QzTWg4ejJQOFRqaFdPY2dZY08yU2hnSzRtQUt1NmZ0ZUhwUVRyWERyaVFaSUhpRVJSbnNWZklBYXlIUXc2Ujl0S0VkekgxTjFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwNF8xMDM5MTlfNjlfMjRiNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWZmZDJjNzNhNGFjZWVlZWVmNzdlNmUxNTM1YTA5MjAzNGNkZGYyYzI3YTVkNzM3MWE3M2FlNDEwZWNjMDM4ZmU5YjdmZjBkYjQwZWU4MzA5YzljM2QxZTljN2Q5ODdkNGYxOTQ5OTY3MmFlMWI2YTZlODM2ZjIyMmVlYWIyY2M0OTI4ZDQ1MGE0NWFlMDM4YWYxOTAyMGJiNDgwZWU1ZmNmMTAyMTE1MTU3MTNmNTdjNzQzNjE1OGJkMzI2ZWQyZGEwZTczZmIwMWVjNzRkNjg2ZWE5MDhiOTIzNGFiY2U3NDVhYmRmYTYxMmEwMDRhNzZjNGIzM2JmMDkxMWI4NzA1YTMzMDBjMWJiNzczYjY5ZDlmNTAxZjZjMGNiMGYxYmQ3YTNkZWE4YTBiMWI3MmZlNDBmNmY4MTRhNWNlNGUwYmU4ZTg2YjYzOGY4MjVhN2IzZTZiNDZiNDBmNDE2ODkxYmJlYTBlZmNiOGY4NjAyOWI4YmY2ZWJjZTQxYjE5ZWZjN2JjNTIyMWFhMWMwZTIxYmFjYjA5Y2UzZmI2OWY1YmI0Y2M5ZWU2ODZmOTI2NjYwMjNkZTAxZjIyNmM4MmIyYmU4MmYxMGRjZWVjYzkxMWRkOTJmNzUwMzk1MzlmNTlhNDZlODNiZmZhOGMzMWY5ODAxNjc5NzE3ZWNmNDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KTezQYZsvR8RAmO4fI9g8wjMbE1_rCZIF4V1hm399QtFEjhDsW3DBQe3emB1Kd_DbqM7BLGjyXpB948jfmapkA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240304_103919_69_24b5_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.007Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkdWOVpmeC9ESDJzMVZkUnNFVDdURmtSVGhCZGpucUVUNnAzLzU3cTlkUUovVm9QM2ZvaXRCbUJvdnA4eFJTam9NRytxaSt5dHZzckpPS2RxUWYvOHFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwNF8xMDM5MTlfNjlfMjRiNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWFiYmI2MTAyNzBlZDU1NGMzMDdhYTFhNTY3NTFlYzVjYjNiNjM5OGMxN2U3Y2ZmMTM4YjJlODQwYWM4YTNkZGYzNDUyZDk0NGEzM2RjMzA5NjI5NGI0MTI2MTUzM2FlZjM4NGRmYjAxMjE4ZTY5NzQ5Y2JiY2E5YWJiM2E4Y2E4YzkwMWJkNjRmOTZhOTgyYTEyOGVhOTM1MjI4NDlmOTQ3ZWM0YzUxNjIzMzIzYzRiMjY5NTdjZDZhZmUzZGIyNmM4NzY4ZTU3ZjY2YTU4Y2Q0NWVkMTQ0MzZmZjE4YjEzOGJmNjBhYzEyNDA1ZDk1ZjE4YjZlYWFhNGRmMGRhNzMxMzZmZjM4MDJmYWE5NWQwMGZlNDgyNzcxYjEyOGE1Y2MyODg2YmMxZDlhZWQ2NzRmNDIzOTY5YmY0MDg0NmVjZmQ2ZDQ2NzA1YjY4NWZmNGNjNjkzYjUwNmY4OWJmMjk2YTY5MTJlNjVmMzUwMmE0Yzc1YjNiOWYzNTAwZmNjNGNmOTc0NzFjOGRkY2ExZWM0MzE2NDQ5YTc5NGEzOGUzYzQ5NDY5OWE2ZTdlOWYwYzJkNWVmZGI2YTkzOTE3NTU0ODU4OTA3YWY2NDQzY2Y1YzIyN2RlYTM0YmJmNzU1ODljNWFhMjQ5YmZiYWFjZjc1NzI1NWFjOGRhMzY4NjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.I-1IQesNfvx6Ww_VhS8SRZqbdW9mX-UHbm-XjB-2HWa72iJ9hfJgbYDTtVNS55KNBbBSj2pQUNN0-09nIxQ8pw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240304_103919_69_24b5_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.009Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikw3OGtXUUU5RGRFc01LeDJXVUsxUklUbUM2TUZ4UCsxSWFiQlNjOGtwVFFUMU81MDE5OWtzdCsrNi9qZzlja0szUUE3K2JlejlkWEI5SHlxdG9lcTJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwNF8xMDM5MTlfNjlfMjRiNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTMyZWIyNDgzM2U1MzZhMjNhNWMxYzY0MmE4OTAxOGFhY2U0YTYzN2YwMzJmODI1OGU2YTAzZmNiYmZjOTM3Y2Q1ODc4MjczYjI2Yzg5MTAzNTNiZWU4MzMyMjkzOTQ0NzhmMmNkYWY4MWFlZDE3ZWI1ZjU1NWFkYzRiNDE0ODY3YWFiZjg3YjFmZWI2YzRlZGNlODU2NDVkYmMzNTk5OTc3ZDU2Y2VjZWY2MWYwNDUzZjVmZGI4MTllNGFmYjJlMjMwNTZmNDA0YzQ4MWE0NmI5YTJhMGNmZGQ0MjUyZjhiZDVjNjBhNWFjYTk4OTM3M2UyMWVkZjlmODQyMTMyMzJmNDg1MTkwMjIzYTZhNTIxMTY5ZTQwMmM0YzczMWE0ODQ3NTAwMmNhMjYzNmFiZDZlMDUxOTZmMTMyYzcxZWMxM2VjZDUxNDk4NWU1MDUxZWMyZjJiY2EzZjY2MzY1NGQyYjFkNzVhNzk4NjNmOGVkMzQxODlmODIwMzRjZDAyZTJhOWIyNWEwYTU5NjY5N2VhZTRjNzI0NjRiN2E4ZGEwMDdkNDQ1Y2NlZDg0NDA0MmNkNWI0ZjFkMzZkMmI4YTQ1YWVlNjc1NGIxYTJlMTIyY2EzMWYwMWJiY2E3MzIzYWNhMGU0ZDNjYTY0MGRmNmRiNTk1NWY5ODVjZDA5MzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.F9h4VVKzKIoaNjQnZQ2GIyOB_pO-oNWUz1XEvKzYF-N8NmW_dA2xKK_8kfT9QZAOOUnjwjzSUhB9DWE7W-v59A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240304_103919_69_24b5_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.013Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InIzQTJEQzB0cEt1UUJlKys3aDk5UnV6SG03UVZYZU1jMnN6U3hwL2EyOTcxSGQwanVrTzcrZnBvUkZUZ2w4MzVFTEVSblA2elQ3RFZHbXFPRUJ1K3h3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwNF8xMDM5MTlfNjlfMjRiNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjdjNzZkYzc2NGE4MjRjYTkwM2RlNDU5Yjg4YTMwN2M3NGE1MzI2NDhkODIwMTVjMWM1OGJmNzM3YzJhNWZjMjA5YWIzNDA3YTllM2IyNDY1ODM0NmNjZTg2ZGFmMTMwMjhkOTEyOGRjN2Q3MTM5YTZiNzA2NzYzMTFhZTUxNGVhYzA0MTAwMTAxZjE4ZTQxZmQ1MjBmYjkwOTFjZWE0M2EzZjQ3MTRjYWMxODJjMjk2YmMyYWQ4NTQ2ZTdlZTQ3MTVkY2NlY2VlNGQxNzA5NWNlMjVmY2Q0NzRiZjg4NDIxYTBkNjA3NjY4NDE1OGUzODdhNTlkZjk5OWRlNWRmOGMyZTUwZjVjZjhkYWU2YTBjYWY0MzgzMjI0ZGQxNjViM2VhNTI4ZmM0ODU3NDIzNzMxN2Q2MjlkMzFiMjIzNDUxNWIzM2Q1OTY2MDE1ODk0ZGQyMTE5NTQ3MTRlZGEwYTc4YTVkMGM1M2JmODZhYTg3NzVmZjc5ZDcyZGY4ODJkNTVmZDZkYTUzNGNlODdlMjc1ODM2YmRhNjc0ZDZlMmI0MTgzNjdiNWE2MzI0ODAzMjhkMmU1ZjkzNmViNzk4NjA4YjUzYTZhNjQzZGZlZDA0MjI2ZTMyNzA4YjNkNDkyZDJkMmIwMDAyYzY0Mzc4YTBiZWIzZWMwMDAyODA5NzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AKjTPtQnGoHcIC9OFa5vndC0EApxeyZ0EQP0dcnymRr2KJ4OYTRopw5_bNTJrI98kBka-buSbgc66ELat_9uEw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240304_103919_69_24b5_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.015Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkFtU3NEUHcrc1Byc2l3VTlPd1c1SUROeEV2Zjl4TkVSNkpNOGxORkZhTVhZT0NHUW4rbkI1a0EveFQ2NElWQkVVaHVMUGFJMTNCeCtzdjhIVUFIRzNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMDI4MjJfOTdfMjQzOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjBmNzQzMDk4ZDc2ODc1ZjE0MjEwOTc1YzMzNWI4MDI2MjcxMmU3YzFjNWJiNjQ4NmUxM2ZkN2ZkMDI1OTJmMWY4NzQyNjAzOGU1NzA1Y2Y0NGU2MWQ5ZDZmZjhjMDYxNDljYWRlOTQxYjFhMTdjNGEzN2JkM2UwYWY4NjM1OTIxZjE2ZjBjODUyMGQ1NzgwNDdjOGY1YjQ5ZmJmZTllN2VlNjRmMjI3N2JlMTAzN2U1MDQ4MWNlMWNiM2RmNjM0YmQzMTM3NzM3ODg4MWE3NTNlM2Q1MGJmNGY3YWQ2ZDVmMTQ5MGRlOTk3MDkyYjFlOTQ0ZWUzMjc5NmQyNGM0Y2JmN2RmYTFkODYwYmI4ZDA1OWYyZTBmNTg5MDUwMzliMzRmNDcyNWNjZTQ4ZjI5MDUyZTU4MjM2NGRiODMzOWQxMTU1MTc0YzU2MWVhMGE5MDc0M2IwYzJlZmY1MWI1NTg0ZDY5MjJjMDQzOGM2NzFlMjI1ZWE2YTNjMjZhNTI4MjYwZDBkM2U3ODRiNTNkZTAzZDRkN2M1NTQzMTBhMjc3OTdlMTA4MmNjODM3NjQ2NjE5Yjg4MDZlZGU2NGU3YjdmNTZlNTgxNGY4NzVmMjM0NzQ3NDE3NTE3YzE5MDhjYzJjZmRhZWQxZmZlZDkwYzliZjhlYjQzYzdhZWE5NzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._jN6l5kUKbBSCkUDy-F3Q_p1_EPc7eWFjLQItJUvK4r03ovC0xZTlLFmpTr73Kq1NO_h-n7sfEcZuQuETBgEXA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_102822_97_2439_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.018Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlJsTjl1NlNQWjZjOVpXRm9kYzJXam1DeHBTUHF5V3pCSzVTa1RjUWFqZ2xVVDdBTytLcCtFa2dwNzQvQURvdXNBWk5nM2ZubDVyc3U0anZJRFVWRzB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMDI4MjJfOTdfMjQzOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzFlZmVmZDc1OTVlZTViZTIyNGE4ZjEwZjQxMjE2NDhiODA3NDAzNjM4NjRhYWUyMmEyZTYzOGFjM2JlMmQyNDY4MmE3ZWViMjA0ODBjZGUxMzZkMjljNjI0Nzg3ODg0YmEyOGU3MGUzNTBlZWQzNmJiNDIwNThkMmE2Zjk3YmE1MWRkODUyNDY1MjZkMWY0Mzg0ODI0OTExNzdjMjk3YjdjYmZiYzhiYmU2MTM2YmZlMTQ5M2IwOGIzMzNiMGUwZDM4ZGNhZTY0MjA2MzdlODc0ZjRhZWM1MzE2MDk4NGI2YmU1ZTM0ODNkNDgyMWY3YTg1NGM2ZjFjMmNkNjVhYmFkMjBmODk4MWJmZDNiMjI1ODFlMjRiMTY0M2E5NWM3NjlhOTVlMDE3YWI3YTBiNTNlOGNlNTJmYTNhZDk5OTY5NDdhZmI4MTAzOTAxOGY0OTNjMWRmYzM1OWU4NmE1ODg4NTFiMThmNmU3YTJkZDAxMjQyMGFhZTNlOWI0OTU5ZWY2NmUxNDRiODZiNTg1ZDZjMzQ1MWE3YzAyNzg0YjFjNjhjYzE0OTZlNGNjM2JmNDZjNTVkZjEzODVlZjEzNDVkMTg2M2Q3ZTllMGFkOWNiZDdiMDdhZDYyZmRjOWIxNDdhZTkzM2NlOTFlZmY3YzE3YTA1ZWMyMzQ4ZmI0NjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.42lGibwVlusYHMDM89MDYyXnRKoLVCsq1k2YjNbQFi0AQUfQVloeD1uKAYd48OvVdt6O0ddm5BS1ejzJZsWlhQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_102822_97_2439_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.021Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImZ2T0ZoUEQ3WDIwdFZEMG1zdGVFOFhscU9SWTJsUy9JQXovd1d1SXJWYUdaeGJsaWxEVE92RVBKTDdlaDZOZXpCMEdUQUJvZE5yVHNXOWF3UjRtOFVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMDI4MjJfOTdfMjQzOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTkyOWZmZjA1MGFiODgxNTQ3NGU1Y2U0ODBmMmZhY2UzMTNkN2Q5NWRjMDRkZGRiMzUzYzhhNTJlMzViNDQ2ZTAwZWNiOWNmY2IyMzkyNzVjODk5ZThjZTUyMDVmMWRjMWJjZGNhNTVmNTI0NWYzYWJiM2YwNjA5Y2ZjYjY2M2U1YmZiYWVmYjNkYzU4NTE5MTkyOWU3MjhhYjI5YTE0MzBjYTU4OWRmMWRiZDNlZTY5Y2RhYWZiZDk1OTI5MDEyZGQ1OTMxMzM4MGI1NThiNzNhZDlmOGFjZjQ2ODcyMzI5YzE5NDhhYmI0ZDE1NDY0MGMzMjA0NDFjYjQ3NDQwNWE1NDYzMjYwZTJjZDc5ZDU3ZmUxNGYxYjE0ZDJlY2QwMDE0ODQ4ZmJmYzllNWE0OWY5OTFhNzc3MGQ4YzAwNDdhZjZiNmU2YTQzNDNjYjQxYTMxZTBhZTAxZThhNjMzMGYwMzNjZDQ0OTkxNGRkZTQzZmM1M2U5ZGJiMDNiZjI1YWM1ODZjMjI0NWQ5ZWQ1N2NjYWUyMGViYjM4Yzc0YmM2ZGE0NzJlM2RkYWI3NzZiMjMwNmIzZTMzMzJmZGFiYTcyYWIwYTJjM2M5YjZhNmUxZGU3MWZiYTZkZmQ3MDAyM2E0NTQwMTY5YzgzNzk5ODAyYzBjOTVmY2Q0ZjhiNzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6P47jeFnLUfWxQkUhYQYIyU_12vWRWUxr7O6Seh-maOI9k1_NPdqkQf6Lvj1g5PvPeXSBKULmjzZajgQdi2tgA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_102822_97_2439_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.023Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZZcHFNRG56a2N2MmVJbzJVL1NLbGUySm9lS2Vkc2dPTlBqSzRWUXJ1VmZuMWhuVGgyeGZLdUp2YStsOVlCTFdyZzR1Q1ZvV2tsbk9SRFdVZkc1SWtRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMDI4MjJfOTdfMjQzOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2RjNjY0YWUxOGJiZTBkZTIwYzYwNmQwMTc5YzQ0MmRkODYwNDM3YjIxNzc5ZDMzMDBkZWI5YjhlN2VkMDU1ZDUxYmU3NmU1NGNlNzNjZTg1MzBkMTFhMjM2NjJlMDJiN2RiMmI4ZTUxZTIzM2IzNWQwNDg2M2ExMGQyODY1MDg0Zjk4ZGQ0YTUxYmIwOTdmMzFlNWQ0MTUzNmVlNWZjYzc4OTc1ODczNmMxOGVmY2QzMjNlNjEzNjI5NmMyMjE3OTA4YzY5ZjQ1MzdlZTM5N2I2OGI1ZjY2NzEwMjJiZGU4Yzc2NDFlNWQyZjk1ZDNmZmU3NzMwYjYxNWNmZWIzYzZlZmU1YTQzNDY4YzdhZDNlMTI0OWExMmJhZjg3NjUzNDM5ZDdmYzRiMTVlZjgxMjE5ZmI1NmFkYjZiNmQ1Mjc5MDJiOWUyZGVmNWU5MGU1MzIzNGQ3YzRhMDFhNDg0MDA5YzQwYmIxMzM1OTM5ZWJiNzFkMTNlM2IzYTM5ZWRmYjJiZWY1YjVhYWExNjU2ZDNhNjc3NmM3NzZjYTk1YmMzNGZkNTkwYjVjYTZiZTliYmRjZDBjMTI2ZGIyMTcxMTgzNWRiY2U2MjQxMDQ3MTU0ZGRkMTUxZjE4OWNlNjI2NmIwNzg2M2FmYzk3Njk4NDIyMTQ5NTE1N2Q2ZmI2YjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5qu3tpfhQPvMqbCerNFKnPvStQP19K1vR_15ugyQZdlpTBFS5uyghM62AumNvt7Tnf6Gow6a0kULB81FmW5HUA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_102822_97_2439_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.027Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjJOVTcvQ0RzWTZIYytacDRjSXpyUXEvcWRyL2xLSUh2Q0FzUVQ4UFRMSFVPckJTcUFrNDJsNUs4OGZubmpDeFZlRS9SclFYNXEwNjJVUGwwZGJSTGNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMDMxNDhfNjVfMjRiMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjJhYmM5OWU4ZmFkNjAwOTQxNWQyYzAyY2VhMTE1YTEyNjY5MjU0YmNiMjY1ZTBjZjNmODk4ZDQ0ODFkYTY2MDI0NjEzOGRiNDI0MWVmOThmNWU1OGI0Y2Q5MTM1ZDI0MTNmYTg1ZTE3YjE3ZGYzYTk2YmJiMjY1MzY0NzM0NjMxMjA4NDI1OWQzZDliNDBkY2FmNjBjYmM0OGZkNzJhZDM2YmEwM2RhODlmYmM3ODUyZTVhN2FmZTExMjZiNjdmOTUxODljYjBmNTMwMDgzM2RjODk3ZDMyNTU0ZTExNjYwNGUzNDJmNmVmOTk4MGZiZTRkZTllOGJkN2RmMDExMzgzNTZhMTZkYmY1ZTc1MzAxYTQ0N2VkZDMxZThkNDk3YmMyOGZmMTViYWM1NDQ5ZGI2ZWQ2YjYzN2MwMjMxZTI0ZWI0Yjk0NGMyMmFkYjc2Y2UyY2UxZGI0OTFiODk5ZDg1MTMxMTBhOGRiMTdiYjhlY2ZmNmFkYzg2ZjQ3MTRlNzExMDkzY2ZiYjIwNDg2MDBmODljMWY5NDkwZjZmYzY0NTgwNjU5ODNmYWYyODY0YWExYzAxNTYyNjY2M2E3NjlhNDM5NjNkNmE4M2ZlNDJkYTQ5ZTkzODg0ZDc0ZDk1NzdhMDliMTZiOTFmYmI5YzUzNzg3ZDBkZmNmZTU3ZGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dfL5DCSK0gHHY-RrQWWpa92wWuJFkfCNcionj-KiQTSG2Uz1d98CJ7Khs46BMcE5btceU5lUbHWt1gYWNGS1kQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_103148_65_24b2_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.029Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InprS2U0R0pDNUg4Umk4cGlFTFNZaWNMY2hMdUllTktBMnh5OXVrYTAxYzlIeEdxc29LdUtTaytXYzArVzZ3aVZGNXYzNzNpL2E1cHZPMFhib084VnRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMDMxNDhfNjVfMjRiMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGNkMWRiODA1NDYwYWU5N2ZjZjUzY2I5MWUwMjdmMTc3ZGU3MWFhMzQwN2RmNGUyZDlmMTcwMGIzZGYxM2EwMmQ2NjNmODIwMTYzNGYxNTMzZTRlZGQzMzRhMmVkNmNlODgxMWY1NDQyOTk4NzhmNzNjZDlmMjU5MTE3Y2Q0ODE5MWQ2NWM3MzNlZjIzMzUyMzA1ZGY1MGFkYjY2MTEwNjZlMDc3NTY4ZWUzODNkZWJkYTY0ZTJiN2RhOWJjNjhlYzBlZDU0ZmUwMzc0MjNlZWM0ZDM3MjVjMjhlZjZkNWI4MjhlMGY1MjAzMTk0ODUyZjI1ODQxZmE1MzYyOTcwNTExODU1NDM5NmM2NzlmYTFkN2I4ZjE3NzgxYTgwNTZhNjgzMTc4NjBiOTdiZWE4NTNmMGNjOTgzYjM0YWYwMjI1MDgyN2FjZTgxZjgxNTZiZGIwZWI0MjBiZTIwNzA3ZDZiNzMyNDIyZjE4YjhkMGQ2MjA1MjU0MWYxYzUxYTEwOTIwMzQxYTQ3OWI4ZjZmNDRiMzllZThjNTU3NjU4ZmY5MDljNzkyY2RlNTEyMDBmYzFmOGQwMmEzNmFiMDY5NGY2OWZkMGI4NjlkOGNiZjQyYWRmOTM1YWExY2NlNTFhY2E5YzNmZGVlZTQ4MWIwODJmOTE4Njc5YzBmZGQxNjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6lnbZ2M1kpr_wIWZQIaBFJjenRBXN90M3thHxDZg4-cp4XX3VtVAUsST-arG-lpyAWQGr1j62uTotLjFEpoiKQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_103148_65_24b2_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.033Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InF2aURmanpDYy80YnpLNFowSkhtNUpQdkxJcW5PQ2dOT1l2ZVNTU2s2WGc2RitKRVZvRFAwSlMwV3Z4S2lzZVFpZmxackVVQTBONVVFVVpiZVBrSTBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMDMxNDhfNjVfMjRiMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTQ4OTA2ZmM1YTI2NGVmZmU2MTYyNWJiMTFkNzdhMDQ1NDE5ZGNiN2FkYThiN2M3Yjc0YTZmNzg3NDk2NmQ2MGM4MDIzMTI5NDIzMDMwMDdhM2ZmNjEwNjI2NmIxYmM3OThhZDM3OWQ1MDFhNmYxYTAyY2VmYzQyYjM1OWI3NjVlNjA3ODQxYzJiYTRkZDQxYmU2YmYxZGEwOTM0MDhjOThmZjQ4OTFmOTJhODVlMmZlMGQzNTA3MWQ3NGQxNWIwODM5NTBiYTRjNThhOTgxMmE3OGZhM2VlZDgyM2I4ZWQzOTViNTU1MzUzZDM1YmMyZWM5MjAwNTdkNmE3MGM0ODFmMjBkZmIyOTAwNTZhOGM4MzQxODM0YjcwZmMyODVjZWZiMjQzNjQ1ZmI4NzBkOWNkMjBjNGU4YzJjOWNiNWJhYjI3MWJiMWZlYTZiYzVlYWM4NDY1Zjk1NGU0Y2UwZTdhOTEyZTQ1ZTI1MDc2Nzk4Yzk1OGQzMmMxMzBhODk0ZGQyYjQ3YzhmN2FiYzdjYTc0NmY1NzViOTljN2VhODhjYjFjOGU0ODdiODU0MTlkZmM4NWFlNzgxOWUzODU1MjA4MWUwZGY0ODNmNTUzNzk3NGJlOWI0OWNkYzIxOWMwN2EwMzYyYjQxODk2NGI4MTYyMzBlMWI4ZjMwMTJkNjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vm0HvcfPCSwEgf3jtqXW4oF0qEsW-5VL8Yv9laXLb5f_jj5VofSW7jg3LoMSWcdatXhOIHrHOF-iSLfvUSydIQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_103148_65_24b2_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.036Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxWZUpHemFDdDdVampQWDY2TWVsRmhxTTQ3cG8wS0JObnh2TjVsYStDTUlOaTd5dVp3Ulo5Tk8yK2dxTTBFWG53Zk9aY0laZlVmSGxMaFpDMlhoeXpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMDMxNDhfNjVfMjRiMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTM0ZmYxN2MyNTRjMTRjNGRjNjZkYWFjZDRmYzMwODllZjQzMTEwZTk1NDJiZmU3M2VhOGFhY2NiNGIxOTIzYThmMzFiNzNlYWFmMzEwNzljMjRlZmQ5YTMxYzRhOWUzZjNiZWRjOWZhODhmM2VkMWVjODc3ZmYzOTI5NjcyYjJhOGRlYTVjNjQ0ZTMxNDYxNGU3MmIxOGRlMmU3MzVhNjZlNmU2MGYzNjhlYTdlMzk0ZjFlMzVjYjU1YmY5YTNmMzBiZWJlMzc4NTAzZDk3MGRjYWNkZmRjZWJmM2ZkMjY3ZDhhMGE0MDk4MjUxNmY3NDk3ODA1NTYyNDU3MmI4ZmRmZDI3YjE4NGM5OGJlZjJkZDhhNWUyZDU2OTVmZjkzYTAwNGVlZjY3NmZhOGRhZDVmYjkwYmIwOWY0NjM5ZjIzOTBiMzU1Yzk3NTA5MWFlZGRhNmNlZWI5M2QwNzQyMjJkNzU3NWM5NTIwYWM4OTI3MmVlOWUzMDUzNjllOGEwNjZkYmM5MGU5MDBkY2Q3MjIxODY4ZTE0YjMwNDdhZGVjMTk4ZDg4YjU0OWNmYmU5NDAwMjkwZDRhOTAzNjJiMTg0YmM3ZGUwOGY1Njk0NGRhY2JhOTgzY2I2YTMyNDUyMDFiODQ1ODZlNWNiMWMzNGExMWE1YzBlNmU4NzRhMWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Gg_ZoJOdEI-9LrA1fZcZ7yIXiqijQhoAr3wUoH8Q5r4odhVEOYDAMLgY2eyFbrZC7KiUuq-uiO56pj4pwn4xyw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_103148_65_24b2_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.039Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikk2Um5CVUpOamdrTGhKU0xaajVuaHIwVUdvL29XeldDT01YWkpZM0IxaXd3RWNzaHhRbmJOZVBOR3l0M0d1dUpCQ3p0dEJOczJnVDZsa0M1WEZRYmdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAwMV8xMTEwNTFfMTRfMjQ4M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzI3ODVhOGU5MmQ0NTA2ZmY0YmNiOGM1NDU4MzJhNzM2MDgxMzVkYWZmOTYzZjU1ODkwYzJlN2YxYzc2MzE0Y2JiMTIzNjU4YzBjMzY2ZThlNTIzYjk4MmVmNjg3ZTA3MzdhNjQ1OGU1NjgzYTIwMjcyMTRmNmQzY2VmMmI0YWJmMjEzNzc2MDYxNGM3YjExMWI1MDJmNzQ0OTZiMmM1MWMxZDM2MDAwNThhNDJjMDUyMTNkOTFmZjNjNGExOWQ5MDhiYzU0NDQzOTRmNzZiNTcxNTg5OWJmMTNlMWQ4NDViMTRlMGIwNzE0MWRiZTIwYzQ2NDE3NjYzOTg1YjIxZmExYjhhNjNhZmE0YTAzZTk1NmVkZGY0Y2JkOWVjNmEwYTg0Njk4YWYxNjk5ZGEzMmZjN2E0NTA3ZjQ4OWNkMDJhYzJiNWU2OTY5ZDg1YTk5N2I2MmUyZDlmMjFiNWE4MTQ5YWMwY2YzYmRhMmIwYjUwOTQ1ZmNmZmZlNTZmOTc3ZTlkNmZhNjVkNTdhZWRlMzEzOTA1ZTM0YWI0NGRlNjliM2Q5YTM2MjlmM2IyZDU3ZjM0MmNiZmI3OTViMzY2ZTcyOGUzYmIzNDhjNzMxZTYxMWYyYzQxNDdmNTcyOWYyNjE4ZGEyNWMyNTM0YjcwZDM5N2RmNWNhNWQyMTNlMzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8INlT9EfF21uKlkk-KasiMyFxvgMD_yjC12nYhvyLKqlRxfGIsCKNsaqME9KlYvoFcAYNdVybxH8TPfiXNdC8w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231001_111051_14_2483_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.042Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InVGVzhtM3I3SGFZMkpLYytkU09lbGpsRC82c1BkdXFzVUVpMFJwQi9XYmNmMlN4RlpHRzJNaEFWWVQ2eDBPa1FCYWpsalFjbTEraVd3VGphSXJCckp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAwMV8xMTEwNTFfMTRfMjQ4M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjk3Y2ViNDZjYmY0ZDhlOTg3ZDY1NjcwYmQzOWExODgyNzllYjdmNjY1ZWQ5ZmNiM2NkMTJhZWIzMzc3ZDZlNjAxY2NkMDVjYmZlODIzMzBkOGI2YzViOTUwZjNjNzcwMWZmY2NmZmVjYTVhZGU0NDJkYjNiZGMzYmMzZGI1ZTYyNGI4MGExMzFmM2U2ZjdmOGJhZDJkMGRiNWIwOTU2ODU1ZmEwMDZkOTVlMDdjMzFmZmE2ZDM1NmE2YmYwZDg0N2RmZTYxOTRmNzY4NjE5MzFkMzBkNjA2OGNhODgxZWFhZTIzOTY1NWJhNzA1MTkyZWIyYWI2YmVjOWE0NGVhMWZhMDlmYjhhYjg4OWFkZTgzNTNhN2E4Mjk2N2NiY2IwMWU0MTkzODM5ZGEyNTRmOWI2ZWU2NTFkNGVlMDczOWQ5ODVlMWViODIyNWQwZDQwMDcyNTgxODNkNWQ5OWUzNDA2ZGM1OWQ2NWM0OWIzMmQ0YjE3ZmZkOWFiOWVjZTdiMjhhMWRmMTNhMzFiYzBiYmYwMzY0Zjc4MWRlMzEyNWM3MzY2YzQ1MjRjN2VkMjM1ZDNlN2FiZTJiOThmMzRhYTQ1OWJjMDZmOTNkZmY5YWIwY2U4MzExYzc0MjdiMTQ4YWRiNWI4OGYyOWYyOTk5ZGM2N2RjNzk1Mzc1NTE4NDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ur3F-f6Sc1k5Ek3RO4-bLVBPKFxkgNlR9zyUK7VAt5iVLV9rUpmtpflcJOVesK0voKTVN36cZm1prbqshI06Fw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231001_111051_14_2483_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.045Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IklyWERDcWxvK0JROWd3aDlkVlFNVmNrUndiVGp2QlIzOE40VFliUTg5K01aQ29mSEpwRUxBd2E3clIxVW9FVW5PYTNqMnpIdHFzR3phWGM3dm5zc2FnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAwMV8xMTEwNTFfMTRfMjQ4M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWQxYjRmNWRjYTQyOTBmMDJiNzVjNDFmM2YzZDAyYzYyM2ZjYjQ1YzgxZmQxZDE2YTYxOTVmOTIxNjExZDVjMjE0MjhlOTM0OGMzYzY4M2UxYzVhZmVhNTE1MTI5NzQ5NjA4Y2FiYWJiYTQ0YzI0YmQzMzUyYzY0MjM3MzM3OTE0NDU3M2E0YTQ2ODZhYjc4YTkyNGU4YjYwOTBhZDczN2M1ODIyMzg0YmU5MGRlNWVlOWZkZWI2ZGFjMzg1NWQ2OTE5NzI0NzJlYmZjYzZkZTA0ZTM2ODE3YzJmOTVhNzk1NDlhYzM2MGYyM2VhYTc2ZmVmZmVlOGQxZTQyOGI5NmQ2YzYyYmIzMzNiN2EzYjUxN2NkNjZjZTk1YjIyZWJkYWViNTI3NWU4YmNjYzVhNGZiYmY0NDZkYzY2YTZkYzAwY2I0YTA5MDg2YTQ3NjdmZTQ0ZjRlYTA5NTRhYzE0ZDBjMDQzOTY2ZWJhNTlkYWE0ZjMxY2QwYzVhZjJlYTc3YjY4NTg3ODBkNzEyMWIwYzZlNjIzMjcxOGRhZDgyMmE1ZWE4OWQwM2VjODM5MGJhMzBkMGEyZjJmNDc2NDgyYmM4YmJlMjJiZjE3MjBmZGE4ZmMyOWYyNjMyMDE0ZTdlYjIyYTM0MmMzYTlmYzAwODFjYzMzMGE4YzFiZWJhMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JNNFPiL_5NNwX4Q0rBnrpY91eYdSR-gGYyWmiPOUf3KG0j9D_Mg1sCXLfKo8FgRUA2EtPz1Sd8i-S7CHd6zKsA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231001_111051_14_2483_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.048Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjU5YU5QN2JrY1lnTjRCci9zOHJ4UnEwMFhoMU9ySmZobG5kYjNleEhBdVk5c0RFTTJ1eWJKK1BZOHVzczBvQ2htcS9KT3VERS94c1dBYTFnb2s4d1pRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAwMV8xMTEwNTFfMTRfMjQ4M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDM0YWI2MmUwODk1NDJhNzJhNWMwMDc1MWFkODJhNzc5OWI0NWIwYTFiZmY1ZWJjYTliOWI2MjE2ZTI3MTY0NzhhZTk5NTZiMmMzZmUyMzI5M2M1MGIzMDcxNzcxYzI0MzI3NWQ4NDhmYmE0OWY4ZGEyMGI3MmNlYjk5YTNhMmJmZjdiYzI4M2U5Y2FlNDdmNGE3ZjRhNzk3YjEwZDkyMDUxNDA0ODNiYTEyM2M1NzhhY2M1NWM5MDNhNGMzMmM0MTg4YzcwMDVhMDYwOWIxMGNmOGI4MWZlNzhhMmIxNTA0MjljOTdjMWFlOTM5YmE2OGIxOGNlOTM2NTZlYWIzYTQ2NTNjMzVhMGJiZjViMDA5ZmU0MDJkOGM1ZGE0YzNlNWNjNzhjNzZhMjcyZTZiN2JhZjM1MmM5MmY3OWIwNTE3NmE4Zjk0MTUzNGI4M2Y4YjE3MzFlZjY5YzE2NzcxMzE4YzMxMjIyMmNhOGZkNjliYzc3ZTFmYWFhOGQ3NWUxNDY2MDc1NmJlZmU4ZjUzY2JmNDBiZTc4OTc3OTNhMjVjNTM2OGYzOTE2MTlmZmEwNmQ4MzY4YjA3ZTQ5NTRjM2JjZGU4MzI2OGE0YjBhZmMwMDg0NTFkMDQwNmVmYTJlZWJlNzYzMDg1ZmYyOTUwYTYyODAxYTc1OTdiNDRhYTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uXPTZVRG7YGLfBb_YKACAFlFxnc_CN1LKScdn1SKF5wAiX811Zig-8sj1QyvsZDSG17-4doilyyzcG0yIJkIcw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231001_111051_14_2483_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.051Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IklkSG15R3p5cFV3NkgxMFdQcHQ3ZkFRQTR0TWEySUlNcHhqaDdZWStpQndaUE5GUXlXdjBDUmVjVjk3V2V1RFpWbjVlakp1K1cxOFJuTGtjcnFhRnV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDdfNzJfMjIyM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWMxNGNhMTM5NDc1MTlmZGMxOWQ5YjJhNTBkMGI0ZGEwZjg2ZWM5OWJkMWE1OTJiYWE3MTUyZDdhYmUwOGYwZjg5ZTczYTdmNDRjODUzZjE1NThkNzJmNzk5NTVmZDgzOGI2Nzc4ZDk2OTZmZjAxODI2NWQ1MTFiNjNmNjIwODBlYmQxMzY4ODMxZTQ5MDI0Yzc2ZDg1ZTk3MzhiZWQxZjBmYmVjYjE1NWFlM2MzZGQ3NTgxNjFlN2FkNjU0NTE0ODVkZmE1YTQ0ODU0MzM4N2E1N2M2ZDA3ZDQzYjEyYWU0ZDNkY2MzNTgzYjE3M2ViNzRmN2FkYmJjMDZjOWI5YWNlNGM0N2Q0ODY2MDAxMDY1MTAzMDMzMDBlZGUwNDc1YjY4NzRlMjkyZGYzNzk5NDg0NWJmNmRlMGFhNWM4MTA4YjZmZjdhODZkNzY4M2YyMmRkOGVkMjVjZjA5MTlmYjg1YzhkOTRmZDhiYzIxNzI1NGUwOGU1NDhkMTA0N2VmMTJjNzA1MmFiMTk4MDU3YTVhZTBmZWQ1NTZmNDU2Nzc0NGUwMGZiOGI5OTNlOWVmMGRiMTA2NjhkMzU0ZTZkOTI5ZWE0NWE0ZjhjZmIyN2ZmYmMyYjc5ZjlmZDlhMjYxYTIxMjNmODU4YTM4MjMwOTE1MTgwZWNmOTU5ZTYzNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KsX2mHzRd_TqrTzBoJnAQvknaOsJi3Ht_jKnKjq_Hog1h8hi1jvw3M6XL-LCl9xOuwuuYyqgUu2Hz6sDtkeTOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104047_72_2223_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.054Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Iko4S0JoUlVFdHBVQUdjK0h1UkROTFVvUDUxVFB4M2hNbWhVNlRJR2xGWnhQSWhlRFp4UjVub3Z0cldVUGZVNjl2Um9qRlNBMkdhVHVaeVRNUXoySVBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDdfNzJfMjIyM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjY3MjQ1M2E1ZmY3ZTY5MzBmOWY1ZGEyNWYzZTQwZTExMDNkYzRjNzUyMDU3NTI1ZjNlMmY2YWIwZDY1MDlhZDRjOWZlNjYxYjM3OTVjNmYyMDhkY2NiY2JlZGM2YWNjYWM4NDY0Y2FhOWQ0NzY2ZGYwNDE2ZTZlNTk4NjQwNWU4N2NmNTdhOGI3ZGQ2MzZhZmZiMWFiY2Y2ODEyZDk1ZWFmMTc5YjdjM2I4MzYyNDM3ODdhYzYwODkwNWRlNWI4YTNiMzRkNGEyNzQ2NDhkMDUyODk0OWMyNzk0ZWZhMmQ3YTk0ZDk3MWE3ZWFlOTk4ZDhhMTQ0ZmJjM2U4YzJlOWQzZGY4Mzk0YmVhNDcxZTg0ZGY4ZTAyZDMxNjJkMjRmYjJlMWEyNzU2NTRiYzkxYTUyY2RhYmE5MjY0YjliOGU0YTk5NzE2ZjJjMzUwZTI0ZDg5OTM4N2U1ZjcyYmQzMzNmNWJiYmZkYTAyZWQ4MWM5NjgyMzE0ZmU2NGI4MDk2MDI3OWY4NGY4YmM0MWI1ODFlNWJmODFiNjllZjVkY2MyOGYxODk5OTk4YjVhNzNiODk2Mzg5YmQ5MmZjZDY1N2ExMWI4MjdhMDdiMjM4ZjRmNDc0NGNiOTg2YTE5ZjNkY2I2ODZiZTliYTJlOWYxMmZkMWI5NTEyNjIxMjNkZGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KFxYFhrHpbDulQXmrkVtB0_F9n0hjZzI4cwTJ4zb7zBeeWWO_0bM9r6j6ykA0jyJSOUpiCUX7tQk5SKVutCegg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104047_72_2223_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.056Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlR2M0toeHRlNGhMekRxZm1XUkh3OXJBbytUa1RvNlBLMGp3TWttL3FFSUdjdjhWclV4QlFhdndpenlsT1ZKa2hGRVpROUFDOFdoZlIwQ1NrTlRiUUd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDdfNzJfMjIyM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGU4MzVhZjAzODM2OWRhZjRiMTdhOWVkZGE2MTllOGVmNDY0MmZjZGRhZDgxNzRiZmQ4MzIxMjZkZTI2ZTg0MTMwM2UyY2M2YjRmNTFkMTM2ZDQwODdhMmY0ODJkMTUyMTMwZjllMDYxM2NiZjU4M2RlNDAwMzA5ZTliNDQwZjM1NjI0ZDUzZTAyYWQ5ODMwMzljYTRlMGM4YmE0Njc1MWU1ZGIwMGZkMThhZGUwZWQyMjU0ZmE0ZjRmNjIxYzM4N2ViY2ExZWY5MzUxZWQwOTM0YWVjZTY0Zjc4MGIwMGZmYTU5N2I2Mjg0Y2EwMTc4NWJlMzFhMzJjZWM2OTI1YmM2NmU4OTQzZDQzZTYzMGY2MzliMjM2NjRhOTJhZjBlM2JiNDhlMjQyODM0MTJmZWFkZDZiMDQwOWY2Njg3ODA5ZmU3NWI0OTgxYzEzYzVjNGEwODE1NjlmOTRjODAzMDk1ZjZiZjcwODA5YWRiYzhhMGZlMjdhMjY0ODA1OWZmOTcyYjMzYTQwNmExYTkyNzY1ZWJiMDBjYmUzYjI3MTEwNzU5YjdjNDMzMTAwNDlhYjVjMjQ5ZDZlN2Q5ZDljMWRlYWVjYzAxZjdjNjU1YWM0ZjNjZjczY2QyYzg5ZTM5NTBhZDg5MjljOGMxNmE3YmFkMWE2ODg2ZGRkOWNiZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Os0JDwKm6RcUc5TsrUclfKS5OcXQQa3t12YUXpKsHOfnNqv5kDt_PJuYTlD1WjZvrqeyOS6aCC_1luXqZ5d59w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104047_72_2223_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.059Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InZhNUl6VGk4QVhqUEx3RDZReWlHQ0N4VDhpaUpDa0Z0eitQV1AyUHVvbm1xQWtMb2o3T0dJNTZyN1pBUDVibXhvNkJNTDMya2NySlZmZFJtM3g3d0N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDdfNzJfMjIyM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWIzOTQzNzJlOWU3NjNmNzlhZjJlOGRmOTFiYmRhMTNlZDM5NGMzMDdiYzcwNWU1NjhkNjgwZGJjMWRjYmEwM2E0M2I3MTA2MGQ3YzExODMyZWU3MTk1NjI1MjVmNzYzYjY5ZGJiZGNhMWE5YWNmZTAwYTcwNDZkMDlhYzVmNDI3MTZjZDRiYjZjZTk4ZTA2OTExZTUzYWExOGQ3ZmY0MDJjYzk2NWIzNzk2YWQzNmExNTBiMjFkNTQ4OWY3MDNhYTlmNmI0NGM2MTk0ZDI2MWI1NTMyNmYyMmY4NmM0N2E1NzhhYjQ1OTQ2ZDM4ZDFmZTdiN2JhZmM0MjJlOGFmOTViNTFmODA5ZjhlNWVkNTUzMWMwMDA5NmI0NjAzMmQwOTFhY2FhNDdiY2UwOGUzODk1MzgxMmQ1NjE4MDkyOWZhMjNhNDNkNjY3YTYwOWFjY2RkYTIzY2EyODEzY2I3OWFkNWNkYTI0YTk2ZDQ4Mzg2ZmU0MjVjZDc2NGYyM2JmZmNkMTBmNjNlNjJhZmZkMTM2ZDNmOWI4MTUwODZkNzA0NjllYmRlZjRmYzFlNWQ0NDVlYjc1MTBhMjU4YjRlNGUzZWQ0Zjg5MmE3Yjc3YzMyNmM5NjY1MmE5ZWMzYTIyMTg1YTA1NjFjMTUxMWZiOTQ1MzRlZDRiZGRkNjQ3YWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dnNtaYq3bMHO7L1yG6ZAhGeGPfY6BmalpGeqyg98W7zH87RQOnb27RKPgPVbW17YHpBBJUVawZJ2E7WHMSkrLg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104047_72_2223_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.062Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imw4S0Faek1VUnJ3MFhBM010YW92WkxyYlRaczdGVTF6YktGTDNJTEpCVnA3VDByTDVEdlFqa0djVUtBR0VVMTZTQWJEMXRYNU9sY0hvSTRjdFdjRXRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwOF8xMTAxMTlfOTNfMjRhNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDk0NWI4NjdlZWRlZmM0NzE1MjI1ZWYzYWNhOWMyMTU2OTAyN2Q3NmI3MzMyNDQxN2UzZGRlODFkYmFlODI5OThhMDdjMjgwMWQ0YTk0ODgwYTRjOTAzNGFlNzc2ODY0NWRlNmZmNDk4ZGIyNWQxNDUzYTYyMzIzOTNiOGZkZmE5YjcwNDY0MzEzYWI3OTRlNGUwZGYyMTc3YTlkN2U1MmNmYjg4YjFlMDI1YzU5OTU4Y2U3YWNlZjE2NDU1ZGE1MWEwZTFmOTE4Y2JmZGU5NjNjNDYyODVlMmQ5NjA5MTA5MjA3NjRkZjZiNTFjODhjZDg5MzBhOTQ5ZTM3NjY3MGMwNDBkNWEyNDlmNDkxMTM0ZTdhZTIzODg0MDZjNjYwNDhlNWI5MWY0ZTBiMzNkNjk5NGYxNWNhMmEyNzRhNTRiM2RhNGY2ZGExMDQ1ZmU0Y2E4OTJmOGE0OGRkYzI3ZDdmN2MwYTAyZmIyM2QyN2ZmNjAyYmVkNjc3ZmY0MTBiMGJjMzQ1NzE5MThhNmVkYzc4MTAwMmNkZDMzOWE1NWRhYTY2YmIxYTYxMzRlZTk1YTdiYTBlMmFhNDBiMDM5ZmExY2QwY2M4MzgxNzgwMzQ0MTNjMDA5MjkwY2MxZGM4YzU1MDI1NDdkNTJiNDQ5MGNlNDQ4ZDEwNTkwYzI4NzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XnWVRhovtIiDrtiJU8Bcxbj_5fjHQ4buGGu5DtmXhvKFvYAGVz3f8V1KH5q5Tmi_NKJFo3OcV-te91MHikK_KA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230408_110119_93_24a4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.064Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImJIU2JZRndTV2VHcnFPamZDcDlOZzQ5TW9lU0VYcElaN3gzeHZXcEJ2bXh0ODlvSmxtN011WlJ0MER2OFpEOFJrY25abVprSHN0Nm02MkNMeHNOZ1JBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwOF8xMTAxMTlfOTNfMjRhNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTE2ZGViYzA5NjIwYTc0MWFhNTMwOWY4OWFiY2Q3OTllOTU2Zjk2NjJiYzZkNzBjNGFlYWU1ZTI2YzBkZDc3YzNjYTA5YWZlNzE1ZDU1NTI3YmEyYThhOGJjOTYxMTYxMjM2NmQzY2E1MWFkZjcyYmNkYTk3ZDA1Zjc1NzFiZjgzMWQ3YTBiN2YxMmI3MmZkZGMyYmY0NTAyOTkyM2Y4YjZmMjkzODAyOWJmMWFhZmRmMDdlNjU4MjliMWQ3ZDg4NWI5OWNkOWExOTZiODAxN2I5MmQxYmYxZDdhMDc2NThlNzg0NTViYWEyNjEzY2E4OTY1NDRhZTYxOGM1OTU1NDYwNDg5NmYxZjQwYjc4ZjhmMzg5NGUzNDJlYmM0NTE0NmI5OWIyYWI2ZDQzZmU1ZWE0ZGQyY2E4NzhlOWEyOWMwYTlhM2JhNjBlMGQyMWZjYzllNWJmYTQyMWI1MzFmZTdhMzg1NWM5MWI5ZmM3NDkxZjQ3OWI4YTM0ZTM2NWQ1YjAwNDRkNjA5ZmViOWNkOTA1MDYzYTJhYjNiMmY3OGQwZGJjNTEyZWFjYjEyYWExNTA5NDhhYzdjOGUxMDFhY2IzNGY0YmQ4M2M0NjZkZDExMTM0OTYxYzU5MDQyZDg2NzUzMWNjODBiY2E1MDhmYjg2ZGI2NGVlNGY1NGNmMTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.R9uIb51s5zOlnVRF8OUuIo9uUPffkEOaF5Mcl6TVb6WwamgTJ8G39W612RMkOMOycsESa7bRzsDbbWRl-W7mHg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230408_110119_93_24a4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.068Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImhjMHFJZ0NJSk03N0RPMnEyQUJMaEJQNHNWY25EWDFLaHYxanViYk1vMzFBYW5pTHQwbFZ1ZTZVMmpxUGl5SFdsYWRyM25YbjYrZlJqdzJHVkE4enNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwOF8xMTAxMTlfOTNfMjRhNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWQ3ZjBjMWUxYTMwM2I0MzJkZmE2NjI1ZTZhMjNhNjE2ODA4NTkyYTI1ODU0NjAyYTI0ZTQ1YWQ1YjRhNjViYTQ1N2FlZjg4ZDEwZWJjNTU4OTRlMTRhYTI3OGVlYjRiZjNiYzJhOTFjNjVmOTVmZTg4NzM0MTZjN2JhMzMzNzNiZDIxN2Q1MWI0YjEzYzczMTJjMTAxODUyMzExYjU3ZWJiY2ZkZWRmN2MzZTcwYjMwN2U4ZDVkYmJhOTI4ZjMwOTMwMzhmNTk2ZmQ5ZjZiMjE5Nzk1NDM1YmI5NGJiZmZiYzdhMmQ5Y2RiMzhkNTg5YjMyYTRjNDJhNmM3ZTc5ZDE1NDNjZWVmYmU4ZjY0OWQwZTBiZmNkMWM4MThiZTlkMWE0MmM5ZWZlNTM5OTQ3YzI1ZmQwYzQyODIwNWM1MWI0N2M4ZGJmMGJkYWNmY2NiYzRmMjNkNjkzYjEzNDVhYWI0NGJkZWQxNjZiNGY4OWMyM2U2NTMyYTI3OTlkZGQyM2E5OGEyMDY3Njg1N2YzYzVlNmM5NTJlZTRiMDM2MTRmMTgxMjQxMzY0ZmU4OTZjYTZlMjEzYzY2ODk5YjY1ODQ1NGVjMDExZjY0ZmYyYTRlNDllMmQwM2EyM2Y3ZmZmOTkzYmMwYjg3YzZjMDUzY2RlOTk5MGE1ZmE5YTRmNjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UAZRx1-F-ylnMJV_ZbWN2-sLR9dHELusonvAr_MDBpG1zIXGFS1ZAr2yg7sEot74SzsdsVoQL9gYsy4ec_f35w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230408_110119_93_24a4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.072Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZaRUk3UFJid2RpK1F1Y2g2bW00M05lVUNIa202THl0Q2hqNitDNVRYaVgxQWROaVlpUzdSd0xta3FUZGx1Z0JvUXB4RUFjTWZSRnNRMWkvSExTQkhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwOF8xMTAxMTlfOTNfMjRhNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDIyN2RmMWFmNDhlM2UyOGRkM2E0YTgyYTVmN2I2YTMwN2E2MjM0MDJkMjdlMTdkMjZmNDhhOTExNThhYzQ2NWFjNjU3MTM5Zjk2MjhkMjJlMzk2ZWJjMDY3MTgwM2I1MWQwMjYyYTg4NThkMTk5NWQwNGY5MGExZGRjOGUyNmFmN2QzMTc4OTk3NzY2ZTY5NzExNWY2ZGI1OGQyMzhjNzAwOThiZGJkMGI4ZTZjYTc2N2QxM2I0MzlmM2MzMDhmNzEyYmU4Njk1NWM2MmYzZWI0OGFiZWVmNDY5NDRlOTNlMTZhNzgyYWMyNzRiN2QxNzFiYzkzNzNhODJkODA4NWMwZWE3YzZkYTI5NTYwOGY3Yzg0MDE3MmNhOGVlOTYxZDZiODA4NDNiNDIyOGFjZGE0ZDU3MDAzM2UyYjJhZDViNmRhZmI4ZWNkYjY5YzAxYjBiOWRhYjc2YTQ2MjU0YjlhNWU5NmE2ZGY5YmUwMGUyMGIwNGJkNGY5MWQwNmJkYWQwZWQ5MTE1YzE0N2NkYmQyZWJlZDM5ZjZkMjkyMjFkZGM3ODhhOGEzMTFkYjA5N2EyZDU0YWJjMmIxMTQ0NGZlZjAyMmNkNWI1NjYzYWY3MjQxNDI5NmRmNzNlZjhkOTJhNDhhZTQyOTIwZjc5OWNiZjkzNjAyOGExYTZjOGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gBPZ3aJOt0800qiGyGiDW9Y6q9iJIVx5yevcy4-9iTMzHG92xObpYue1NDTXD5j0KBudAuaiEH07268DUIRwhA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230408_110119_93_24a4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.075Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik5MSWJvQUw5UWpaaUVObGp4N3JOMzFCd1NUY283OUxEcllkYlBLZ2hFRmpWTnNZYVhCTzlPc2cwMHJpUHBmaW9tY0tYL0s5K3JJdVFuT3dKa2Q4Y2dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDE5MjlfMjBfMjQ2MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWVjMmU3YWE3MGZmYmMxNmQzYTJiNmFlODY5Mzc2MzBmZWIzNmI4MTIyMWM1MWI3ZGNlMzZlMzhiYjIzNzVkMWUzYzJkZDM5MDdhMjIwNzNmZjgwNGIyMzQxOWM1MjRiY2ZlN2QxNDdkOTQxZDUzOGVhOWY1ODE2Yzk1NTdlNGU1M2U3NDg1NTUxZTg2Y2RiODY3NzZiNmQ2MmYzMzEzMTc5OGIxYjgxMzJhNDkxNTFjYTcxYzBkYmEzNTc2OWE2OTUyZGU2YzI2Zjc2YWEwNzQ1YWUzZmZkMjIyYzExMjg2ZGIwOGNkNjE2M2U3ZThmZWM0MDczMWU0YjM4NGY4YzIyM2ZkMjgxMmNkYzUzMTUxOTI4Mjg0NmYzN2I1ODQ0NWQzNGEyZjcxMzdjYmNlNGY5OTE4ODg5Y2RkNzZlMWY4NjZhN2U1MDBmM2JjZDlkZDUyYjJhMjJjY2UzZjNlMWZiMzg3NzFlMDk1ZjE5ZWQzZmI2MmIzZjg5YTc5YmNiOThkZjEyZGI5Yzg0NjQwZWQ0M2RjZDc2MDNmOGZlODc4NWI4ODYxYzk1MWM3NGFiZDYwYTlhYzQxYjEyM2ZmMGVmNmRmMDJhOTgyZDY0NGUwNTBlNTM2OGI1OWJmZWQzMzZlYTY1MWQ3YzQ3ZThiNWIwNDc2ZmJlZDFmNjZlYTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jCfIcOIMLkzWevQLxI74BRoaI0F-wFWY1pb6PQIvJmoG_D3UGK8Ig7KmreGaS8i4qRxe82M6EaFNlIABmqKyxg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_101929_20_2460_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.078Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IklCc011RG5nRGZWNnFpc1hDUmJCZDloRlZ0OERzeWZydm9kc0NWcTlERjNZdG8weFhhZXRKeEpkbS9sVG5kWENSM0lCV3NkRUF1S290dVZ2VGtCQ3lRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDE5MjlfMjBfMjQ2MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjczMzBiMjBkM2UzYmJiZDJjZThiNzllMjAyYTRkMGU4MzFlMGM3ZTQ3ODc2MGFmMDYxY2VlNDE5N2NkNzBmYWRiMWY1YWE4MzUzMmNmZjRhNmY4ZmI5NjMxMmVjM2FiZjVlODVhNDg5ODM1MzYyODUxNDMzMmMzZWM0YWQzNDdhNTQ2YTY5NmI1NmMzYTI5NjA2NmI2N2ZhNDQzOGFmNzQ2YmFiODQ4YjBhNDRkMjYxNTEwMTdiMTkyNmFhOTgyOGZlYzVmMjA0ZGViNzJkZGRmYWYyZjhmNWI5YjgzZjk4ZmE2YzY5OGVmYWFjMDNkMmMxNjdiZWJmOWU0YzU5OTY1MjIyMzI1YThmYzdjZjYxZGRmOGI1YmI0MzFkNWFiMWI2NDgyYmMxMDhjN2RhNzhlMmNkYmIxNTY0YmQ2MjgxMDc2N2MzNTk5YzM1MzhjYTdmOWMxYTdjNDJmY2RmYzMxNGM2NzcwMDNhZGIxMDA5OGFkYjg5NGU0ZDQ2ZTljZDIyN2NmY2UzYWFjYTI4ZWU3M2RiM2Y0NjRmYTUxZTYyMTNiYzY0YzI5NDU4OTVmOTM0NDc5ZTU1M2U3MWE4ODI2ODgxY2Y1YTY0YWU1ZjQzOWQ2N2FjOWM0NjVmZTJjY2RmZGJiNzVhY2Y3MjQzMzhjMmU5Yjk2MzJjYmFiZmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QC3TN_ZlaI4JdrR0ozSuKMN6mdN7SBme-MtsE7C6b4nezuPLS6oPXkcfJz9x4MKCL6WIEth3cAq5tu7UXakO9A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_101929_20_2460_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.080Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZWcXBHSUlTNjQ2WHV2ZHkwcGZYb2ZnQzZHS3V2TFpsc3hUOWNMUWV3cVhhQ2hDdkFMOExxUkF5NXRBSTYyVDVXN2FTRHBUekt3VUpOQ1pjVkUveWF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDE5MjlfMjBfMjQ2MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjkyZTJhNzg2MTQ5ODUzYzI2YmEzNDU2ZmM1ODk4NTE4NWY4ODc0YzMwMTJkOWFjMDBmMmI4NmJiNDAyMzczNGJiZjQ2NTUyMDgyNDhiYmM5NmQzZTQ3ZGQyMWRmOGFjY2YzMmYyYjhlZmNlNmFjYjQyZmEwNzI2MDliNWFkZTZlODgxNTYxNjE0OGIxMGJkZjNjZmY1YzU2ZThiODVlNmI1ZTEzYmYzYjExOWExNzllOGIyZjM2YjFiYjgwMWQzNDcyMTllMGUwNjFlMzkyOGU5M2RlYmU0MDMyZGUxZDcxZmIxOWNmNGI2NjVlZjg2MzQzNjcyNmY0ZDUyNDk5NzJlN2IyYjgyZmQ0ZjkzNWVjZDBiYTU3NTU2NThhZTg2ZmJlMTJkNmJkMmVhN2VmYjg1ZjNlZGY3YjM0YmQ0YTIzZmU2YzNiNWY3ODRhMWRmMjMzMTg3NmYwODk0YWZlOTYxYmI4ZTEzYjA3Nzk0Nzg3NTIyZjI4ODVlZTZmMGU3ODgzMzhiYjVmNWNmMGNlMzkyOWQxZDY5NGZlMWUwODg0MjE4MTg5NjUwNjJmYTQ5NmY2MjA3Mzk4ZjcyMGY1MTA0MDI1M2ZiYzEyZDM2ZjdhMWRkY2FiNTUyOTk5YzU0MzYxYTY0MDFjZTk1NzM1MmUwNWIyYjhiZGMxZjJkMTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bF71grFz4fvm5ggtpiDXAqoCgE0gsKLw-pfF0Mv0cBSeN0ymmFFEzOUu86pUEuV2Ze8cizwWeZGPXIGy_m-hug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_101929_20_2460_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.085Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImZVUmptdEFiYk1XSkdGV1o5ME5XOVpBWGJKelpVbWRVTEtsR3d2bHcvNEhkMkVEZWF2MUhNblFyejR5QmRmdDFsSFErTHgzZXpXWEtycU5rQzZyWkV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDE5MjlfMjBfMjQ2MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njk2NDU1NGIyMGZhNmFiNWYzMzU0MjFhMDAwNDUzYjk1YmI3ZjVlMGU2MmJjYjNiMjgwNDVkMjg4ZTU3ZDMxNmQ3ZTY2YTc2Y2M4YTVjYjQ4ZDI5NWUxMTZkNzNkM2Y1ZTlmYjhjODdiMmEyZWJjYjY4MjQ5ZjhkMjM5NjliZmY4NDNmZTJlNzI5MzZiOGQ4MmUwZmYxMmY5MGEyNTEyNmUxYWQ5MzE1YWNiOTIwMGVlZmQzZWEwNjBmN2IzZDBhMDU0YjQxYzE2ZWFiNGQxNmFhZWEwYjI0ODYxOTc5MTFiMDE1NDZkMjFiNjY1MmVlZjA3YzVkZjdhMDgxYzdjZjRiMmEwZmNmODBhNjA0YzQyNzI3M2RmZGM3Yjg0YTVkY2QzMzJjYjk3ZDQ2NTUzYjU2NTVkMDEyZGQzMzEyOTY0NDRhYTBmY2IyZjBiN2EwOTc1NmU0ZTIxZmY0ZTExMzA3OWYwNTQ4OGM0MDc2NTk3YmY1YTA0YWE5MTFkMGM5MDVkZjcxNGRiN2M2Y2E1M2Y0YTM5MGQ5M2ExMjdlOWViN2U0ZWRiZDZiOGUwMzYzMzE3NTc5NGQ4MWViYzhjMGMyYzNlMGI2ZDRhOTJkYTllN2NiN2NkNzUxYmYwYmE1YjkwN2M3YzUwMzIwMTcyYzRiNzc1MGRjM2Q2M2IxMmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AwMiZsPIPRMQFSGkgtb6G8ckSEurk6OcrLjFABUSD8yiQ0CNf9R4-1K9fDFm3E2ANIVlG0PnFvL9VSmb5xsWVQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_101929_20_2460_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.090Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ill1L3lvYW4rdXRIWkZDZUVkbkcxdFdyZHcwcjBCQzFhUWpwVEdLemFNWmt4akx4ZmhMUUMrZVBCcGRlei9DSlBHV2ROd1g1L0FUK3ZKK2hpQmdORHN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjNfOTVfMjQ0OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzdhNTg2YjFjZmRiMzM4ODA5MWQxZjFiMWU2MzE5MmM2MTA1ODMyMWQ1NzFiMjc0NzM0ZGUzOTk4ZmI5Y2E2ZDBjY2E4ZGZkODE0NDFmOTY2Zjg2MWU2OTU3NzdlODAzYjE2MDZmNjkwODQ3YmI2Y2VkMjc2NjA2Y2QyM2MxNThmZDkyZWU3YjYxYjBjZGUyMjI5NGEzYzY1OGVjZTJhMzIwOTRjODFkMjZlM2E3N2YzNjY0OTk4OTY0OTIzZDRhOWMzNWRmYWFiNjRlZWFkY2RlZDRmZDhjNDFhNmI1YWM4ZTJiYjM4YWIwYjhmZmQxNTA3ZmRiYjFmODMyZTUzOGI4MTRmYzE2ODAyOGVjNmVjNjQyMjBlOWI2OTYxMmI1ZTIwNmIxOWMyODRmMDhjM2MwMTc2MWYwNDA2NzMzNDVkMWRiNjAzZGU4ZDY1MjFmMzI5MmExZDM1Mzk2MTQ3NmJjM2E4ZTBkNzVhNTEyMzAxZGI3YjRlMGMzNmJkYzBkNmViODFhZDBmM2U3NTI1ZWU0NzdlNjBlN2M0ZTkyODFjZjI2ZTY4YzJiYjYxNjY1NjcxZjNkODk5M2UzYWUwMWE3YTc2MGU1MWQzYzVhOTdkNzhiYzM0OGE1YWYzMTJiODU5ZThiY2YwNjg1ZWJhNTU3N2FhZTFiZTAxZjQ2OWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EAkrrD9LUrrw154zqyZYsukhLPtZEhgqUq0WjVQgAeJmG640jUmMfrAeKRjgB6jD2CeRZHtFnEKQ1IL3C9n8MQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103423_95_2449_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.095Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRqUGFkL00wTit5V0U2T2FWUmlGYWE4ZjBiblZrYlY3Sm52dXRWMHljVjJVcEJYQTZxak81UThTUVV4YnJWYUsyWkNkWUxVZWdqcE1TUEZIU3NGTVZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjNfOTVfMjQ0OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTBkZTIxNDJiYzJhZDUyNzUzNGQ0OGQ5NTM4OTlmMDI1YTFiM2VlZTFkMDlmOGMyNjEwYjNjNzY2NjQ1ODkxMDY1MzdlMzgyYTdkZDkyMTNmYzI5OWY5MjkzNDMyNmQ2ZmRkNmUxMTEzMzU2YjJiOWZmYTc1ZWQ5MDNlOTBjMmNhYmViOGYzYzY4YzUzNGYwNTQ4N2UxOTAwY2JhN2FiNGJhOGUzMzBhMjJmNzVlNDlhMThmZjNiMmIxZjM5MWMzYjQ1MTRjY2I2OTgxZjQ0ZjkxOTg5MzM2YjJlYzAwOTkyN2Y5NGFhNDhhYWNiZjllM2JmNDJiMzJkNjg2ZmRjMDhlZTcxZTUxN2ZjZWZlNzFkMjBmNWRkMzRiNDA2NjEzNTQ3Y2E5NTdkNjI4MzgwZmU0NjA2YjdiZjJlZmQzN2FjNTNjZGFiNTg4MmI2Njg2ZDYxMDUxNTY2ZWZiYTFlMTc4NGRlYTBmYzM2OTM2ZDFmODE3NjZjZDdiODlhZGQ5ZTJjOWY4MDMyYmY4Y2NmYmQ1NWM2ODMyOTI1OWQ3M2ExNGUxNWU5MDMxODIxYzQyZThhNTAzOTEyNTgxZmE2M2U2ZTQyOGNkZTliODRkOWNkNGY3YmE3Yzg5NmE3ZjI1NDM4OTE3NjMwMDM2YzQyMmY1ZjYxYzMyN2RlNTY2MjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xKNd2rAaEl-wsZTia2uySncxYwWlwjkn4or_slQ0t0OdFVLQ7dgi7XPcgZF1Z9waEYTc8DEMXseQbzdWYdESIA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103423_95_2449_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.097Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxzaE9UbFk2NVRqcVJOMnRqQ2dDUVFNN0xMYXpnRWw2NUhPVjNPMCthM0ZkUzZML1pYMTEvM2VVaDJ1SDdHbEpiOWY0MjBGMTF5ODFmcDlMcmZMd3pnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjNfOTVfMjQ0OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzhjMGM2MDJiNTQyZWFiMWJhNTU2NThiMWVjNTI5NWM2YWEzOGRmYzNiNGMzMGYyZDBiMThmZTI4NDIyZGJhMmM5YjA1MDhlYzc4YWFkOGYwZTFjNTVjNjVkNGE1N2Y0ZmUxNmRjMzA3NjkxOWVjMGMzNGY2ZDZhYTg0MDBhN2M0YWE5NTRhZTllNmM5YmYwNDVlYzAxMDA0YTY0MTk4ZjljOTU2ZGUyYTIzYjkxMWM0NTYyMTQ4M2U5NWZlMTcyNmVjMTc2Mzc4NjMwNTQ3YzdjODYwNmIyYmM3ZDk5OTQ5MGYzMjY5YjI4OTg0NTQxNDVmYzBiMzcxY2M4MjBhMGFlNGE1NDYzYTI4ZDA0ZGRiMmZkNDA5NDM3YjIyZWM3Mjg2MDJlYjc2N2NiNTRhZGYzZmQyMzYwOTYzMzhiZmI3MGVmNmIyNWZkZGIwM2EyMGRiZmJhZTg4Mjc0ODQzY2NhZjNmZjM3ZGEzMjM4MzFiMGM4MDQ0ZTAwNjdiZTlkOWQ3MDc5OTQ2ZWNlZGEwN2I0ZmM0YjVhMjczMjkzZmM2YTQyMjQ3ZTkzZDUwMzQ4NzFhNTkyYjg4NTg4NDk0NTJlMGU2YTRmMDBmM2ZkYzMwMmQ2OTE3MWYyZGU5OWNhNDkwNDlkNTA4YTQ1YWE0YzU5NmE4NmNlNDMxYTA3OTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GNW3ErxoaI8LuS1sPmXfJDaNOtPR5YIRjn4lMTYtKxa9k7lWm-mqtJst8j3LVVHYlCCs0TXc9DuU6TZCdblLyA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103423_95_2449_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.100Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im9CYlV6MjQxdVZNNUlMMWFqRjIxa0ZaWWxrR0EvRENmLzBYNjJTbUhJMEhvQnVTTnpuaHN3MHRlYVJuQk1lUDQwR1B3YkVsdXZ2ekExWEovei93QjFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjNfOTVfMjQ0OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjhhMGM3MzNkYjNhMDQ5NGFlY2U1NWI0Y2UwNmQ2YzI1N2U3YTExNjQyYTVkMGZmMDRiODc1MzY5MWUwMWZkOTRmYzhhMTBlZWQ3ODdkNDI4Nzc2ZjU2OTdiN2I2MTA0ZDM5MzMzMTQ4NmQ2ZGI2MTEwYjRiYmExMTAxMzQ5NzM3NTc5YjZmZTk5ZmMyZTNiYzI2NzcyZWU0MzVhMThhMWRmNjBjNGVhMGNhNmJmZjgyZmI2ZWFjZGEyNjY5NGM1MDhiOTAxZGU2ZDYzM2MxYmE2MjRmZThmMGVkNGZiNGRkNzE1YzQzYTVmNTEwMmQyYTE5MDE2ZDc0ZTUwNDkzNzYyYWNmNDkxZjAzYWEyOTk5MWNhNTEzNTVmNzE4NzgyMjA2YzBhYjZhYmUzYjE4ZTQ4MWMzNDk2Yjg0NTE5M2FkMTYzYzI4OWM4ZDg0YmNjNmM1NmM4NGUwMDE4ZGY2MTU3MjRhMWU1YTc0NDM5Zjc3MjQzZmVlMzYzZTVhNDYzY2Y0MGM1OTBhYjBhM2MxZTE3ODllNjVmNWYxOTM2NzljZjg0MmUwZWNmODM1ZWVhOTA1OTZmM2IxYWE3ZWZlYjlkNGE5M2Q1ZTAxNjYyN2IxZjYzNTlmYWU4OTJiZTBlMjI3ZTg1YmFiMWU4YzdmYzFhMDc1M2ExZDRjNDhhMzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rPzHpaq75nyM3CEDuN4VOU-hRo-dFCG0QBIVHa0vqluViwRYR6XxZURfc_LzYjlCvJL6akdkNQOYg4QJq5eF4g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103423_95_2449_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.103Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Inh0MTFKdS9RQ2YwaWhQMy8vSjF4YVBMQjVIMVpENXAvakVHeDVaOFhNL04xejdPVmkzUFJQZ3QzKzVnSVptMm9KZzhVdFlSM0E1RHNkcFZTS0tvV3ZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDhfNzBfMjQwM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQ3OTU1MjhjYWZkYzc1MWUxNmQwY2ZlYzJiOWE3Y2I3MWEwNGE0MWY4N2Y0NDY3Njk1NDNkZjBjMGZlMGQ0ZDk1NWNkMWRjOTNlYTcwNDMxNGVhMWRjZmViNDdlZTU5YTYyMDc3ODIwOGNkYmNlMTJhY2ZiYzk4NjI5MTk5NTRjMThkZWY5OTA0ZmExZmVhNDdlYjgxNGU1YTU0MmNhODYxZDljYTMzZjEwNjg1NWE3NzgwY2VmZGI5ODIyMWQ5Mzk4MzQ4NzA5NGNiMjVkZWJhZmUxMjhlZTI0NGJjYzIzNzIwMDQwODQyMzg4YzA0MmNmNmUzMzdmYTg1ZTRkMjk2YTA2MjhlMjA2ZGZjM2Q5ZmViNGQ3ZGUwZWM5N2JhYjQxZTBmMGQ1ZGUyZTBmYjQxYjY4MWU3NDM4NzE1Zjg0ZjEzYTFhOTFhYjBjOTE4ZGI2ODVjMjc4ZmExMTYwMDAxOTVjODI5MTY0NDg4MjVmMGE4NDY5OWY2N2YzYmZiZTYwMzAxZWVlN2YwZTliM2I2YTEzMmFjZTY1ZmQ3NDg1NzRhNWYzZDU5YWU2NDJmNDllYWY0YjlmMmFlZDUwMmE4NDA4MjYxYjRkODEzMjcwNWU2MzAwMDRhMTZjNTUzY2RhN2QzMGZjYTU5NzY0ZTY2OTJiOGJmNzM5ZGYzN2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Juatl_5JvQwcwdR25skyIykbpfOl5lR16UyM3l0DORQgwA2N0yhoXZl8KhUofvcCr-_D_NO42iB5wOARcSoy1w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111808_70_2403_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.106Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlhaRUxGa1V3U05vem5rR0JXZDF1WW0weTRnYlBXL3B4OEdIZkFIQWNnVVRVdnhLZ2Y1c2Nqai83UDFhM01xUWxNR0xDQUZNN2lhTm9FSjdUKzJadWhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDhfNzBfMjQwM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzBmZThhNmY4NDcwZWZkMzdjZDU1OWIyMjI5N2E1NDY0YTQ4YjUzZTY4MzFhNmFlY2Q4Yzk4MGQ4N2VhZGY1NDQ3ZjFhMGRjMDdiODZhODc0ZDA5MjNmOTg2NzcxNjIwZjFlYmUwMzAyNDA2MThiYmEwNzYyMTUzMzZmMjg4MGU1OGFhOTVmOTBhNzE1ZTc2N2VhMmZiOWFlYzY0NWFkNzA4NTgxOGNkZTkzM2Q0MjBlYjdmMGZkMzkxMWMyY2E1MzVmZWM5NDY3MzY0ZGNkMjAwOWMwNjE3N2ZlMTRjMTgxMDFkZjI1YWNlZWE3NDgyOGYwOGU1YmYwZDhmYmE0ZTg4ZWQ5MmYyN2E2ZTFhYjMyNTI4NDNmNDhlZTRkYjFjYjc3Y2QzOTQ1ZDQwY2QyNTkyMWVkZmZiNjI3ZGUwYWFjMzNlYWQ2OTc5MmNmOTQ5MGM3MmNmMGIyZTQ1Njg0M2E4MzJkZDYwYjRlNjA0MTRkMDEzM2JmMzY4NzI1ZTk1ZGQwMzc0NWI2ZDQwMGNhNDNkNDJlM2E5NzVhZGI5MzcxNGY4ZGEzNmM0MzY3ZmZiZmNhM2QxOWFmNzM1N2RkZjE0Y2RhMzU4NTg1NTAwZDdiZmFjMzA1YzFlZjUxNTAzNGVjODNiNjUyNzE3Nzk0N2VlZDgzNzI5YjAyMTlmNzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AFEW474vBKC_MSAA8U4GxPzA9SW6vg-1wdY64DJK833I6xbYmEYaQ9TvrBdbxmyemNadRiAyH3XJn36jHvNzBQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111808_70_2403_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.108Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlB3QThETE1zd3dFY1A4bXNjNmVLdGlvSWdjTWpxSjU0SjVhRGFUMVloSHNnclNCeFZGWnljQXU4MU9mNVdFejV6WFcvZHhtWVJXaVhKaXk0ZXhhQjBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDhfNzBfMjQwM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM2NjcxMzVjNTlmMTVlZWRmODVhN2UyM2I5NDQ4NDY4MTM5NTkyYmNiYTZkNWM2NTNkZmVkNDJlNTU4YzdiNmFmYjA1NDFjZGNlYzUyNjg1NGViNDQ5ZGFlNjM2MGY2YjhkOTdiOGYwZjlhYzQzMzQ1MjRjOWUyN2YzZDA1ODdjYWQyYjU0MmEzYmQ1NDgwOGVhMmEwOGNlMzlkNDZhNjljNGVjYjc0ODZmMmZhYTc2NzhlOGU5NTZiNzYxOGRlZDNlNmRlYjk0NzhhMWY2NDAzZTIwY2EzZDIzOTdmZTgxZmYyOWUyZmRhMGE4MGQwYmJkNDI1MTg1ZTE3ZjhhNzg1ZGIxZWVmNjI0OTkwNDM0MWI1ZmNkODI0YmY5Zjg0OGUxZWFhYjdiYzdjMjJmYjgxOWZkY2MzNmViY2Q1YTYzMGFjYjAxNzE2YWMxNTUyMzVlYjg0NDEyNzczYTg4ZGI0ZjNmNDU0YTU3MDU4MGE3MjE0ZTc4NTlmOTViN2Q0ZWE2MzU5ZGQ3OTExMjliNjA1NGM3OWE2YmY3YmQwMmYzNTkyZGQzOTE5MmQwMWZlYTkxMTUwMjg1MzgwOTMxNDQyZjgwZGUyMGE2NTg1M2Q2NDMyNmJkODYyYzM5NDkyMzgxYWY5NjQwY2Y1YzFkZWY2ZmMxZjczMTY0MWY1YTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ONwHSaJBFcxFbt5qSGIoosTCiFIusW1q-XMx0xcOXuw9_WenUEBVyX3_32It69GNyjnFqthSFjJ_nV27aQhKDA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111808_70_2403_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.111Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii9CNnRUb2tTdXZReXo0MTBMVkI4MGd5ZGNmTk85SnNGR3VOZ0ZhbVdVKzlYRDZPY3N2OFdPUTl3MDJGdU5wKzNMVy9IOG5XcXVSa054SndiYW9HY0d3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyNF8xMTE4MDhfNzBfMjQwM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDhiNTkzYjM3NGFmNWExZDM3ZjhhYTkyMjBlNGQyYTU1MDQzODEwYjI5MzhkYzhmNDA4YzkyODY0NDNjN2I2MzgwNjM1NmMzNjYyNzg1MjlkYmFlYzViYjBjZWI5YWQ3NmVkN2VmNzMxNjQ5OTUzNjFlZmIxOTk2MDhmY2JiMTY4MGJlODkxM2IzZGQwY2QwOGE2MGNkZDA0MTVlOWMyMmNlYjExYzMwNzJjY2Q2YWU0ZTBmMzViMDBmODg1MDc0ODJmMGIxYjg1NDE4NjJhNTgxOGVhYWJhZWU5MDg3YWYzOGNhZTNiMTdjZmQwNzlmZDdkYzgwNmM1ZDkzMzlmZDEzM2I0NjQ4Y2E4Njg4MGU0ZTVjNjUxZWIzYzcxNmZjODJkYjFlMTRjZjFjNDZhYjM5YjI2OTJjZDk3Y2QxMDE3Zjg5MzJmOTkyOGUzYWJlMTA0OTU0YTFkYjE3OTI4MzY1ZDg4MzEyN2IwZGFlZWQ1YWI1M2YxOWMxZGJmMDE2Mjk5YzgwODcyNGZmMWVmODJkMTc5M2ZhZmY1YWY4NmFiMTlhMTFiNTAzY2YzMjcxYWJkNjY3NDU1MGUxM2M5MmJkZGQxMmM0ZmIzYzVjMjQxNDU4ZWE4ODg0NmI5MjgzZDVhN2FjYWUyYmFhM2E1MmEyZmZiOWJlNjllMGY5ZDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nTC_4h-lTIo3FLyDjK24Oatnroa5_craSynP0ImdTUUN6fRkZXXcdKOwPKnRTgjRnZEroA511Gum7XeD1KxMwQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220324_111808_70_2403_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.114Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IklNUWpjejg2Tmh6RWFLOVNvMUIvWnA5Z2UySUVPeklPRnNFSFNoMWYwWDRaSWpFckNjK3dXSTFDM1VLL0dWSE9CbUNJM2R5akdHcDBybnhnUnhVWWRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjNfODVfMjQ5Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjQ2ZjYzNWQ4N2RlYTY2ODMzNmViMjc1Y2Y4ZTQyOTY2NmUwY2Y2NDU4NzFhOTA2N2ExYTg2YTAyY2Q0YTEzZjk0MDE3Nzc3YjgwNzQ1Zjk5NTliMWE5YmQ4ODA2ZjVhMGFjNmQ3MjIyYjBjOGViYWQ0NzFhNjZlZTEwZjM5NThmMzRlM2IzZjhkYjZiMjRhNjU3YzM1MzBlODAwMDU3OTlmNzY2NzZhNjE0OTQyNDFkMWE3ZTZlYTdhMGRhZDQ0YmY1OGZhYzVmMjhmMjcxMmE3MjY2MDJjOTcxMWQyNWVhMWU2ZDQ0ZTQwMjRjYjBlM2Q0ZDgwMTY2MWFiNDVjMzc2YTQ0NTg0ZWNlNTVlMzAxOTc3MDMwYjViZTJkNjViNGZiOTUzOTA0OTJjOThhYzc1YTcwYWJhMzYwOGI5ODA5YzAxZDA3OWFiMGQ3ODJhNDZhZjVmMzdhYWYyOTIzNmI2OWNmOGY2OWIxZGI0YzZmOWFlMjgyZmIwZjc5MmExNmEwN2JjNzNhNDIzOTIwNTA2NWI5YmE2Y2M1NWVhNDI3MGRkM2Y4MzQ2OGU0MzZmOTRlZjVlMjc4MzcwZWJiZjI3ZjI3NGRhODAyZjBmYmExZjIwMDkzOTQzY2VkNzg1Y2Q2ZmU5NTkxMjRkNzc2NDdjYjRhMzYzYTE5YmMzYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6zUbf_D5bChLA5AmtSDBjvFOgkSb_5vhX-ivtZXOPWULpJyh31d4x-UXJILwIJH_nxJu3-0xElyMl4tbYVnkyA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110423_85_249c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.118Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjdYRFJISEdSUml0dTZ1blVQem9EMm5jVWp4N1h2K2YrajYraGR5K1ZERjRTL2RIamRlaEVzZTZwNmtkY0hKOHZiOXNYU24xREJQVFBUMmdzUDRvelBRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjNfODVfMjQ5Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjA3ODU3MGM5MTgwNWNiYTRmNjA5ODdiZjdmNmEwZDM4M2ZkOTZjNDUwZjUxMzIzNDUxN2RiNGM3ZjY2OTkzOTRmYjcxZDBhYTZjMGVjNDFkNjlhYTU1NDU2ZmM4OTUzNTZjYTM1ODFjMDgxNjQ0NmUyZTM4Zjc0YzY2YjMwMDc4ZmRmYzNjMDhjZWZhMDIyMTdmNWQwMmNhYThhMTJiNTE2YzMwNWNlYTA0YzYxNGZhODg0N2Q2NGI4NzZhMmU4YjQzOGMxYTRlY2M3YzMxMDJiZGVmNTQ5YTA1NGU4NTkzYzA4Y2E1MzAxMjQ4MjgwODg2MDFiZTUzOTY4ZTYxODNjZWJhYjkxMWFkYjY4ZWVlYTU3MzBmYzE4NWNjZTFhNDZhZWEwZDMxZDdjNGFmNjcyZGMyMzU4MTUwYzNmNjRlYjFhYWY0NTQzMzkzNTkxNjYzMzI4MDY0Mzg5ZjQ2YWE1Y2RjMGYzN2VkOGE3MDU0ZmE3ZWI2YzZkZmIxMzIxODk0MzA2Mzk3YzMwMjdjNDgxM2EyM2VkNWUwOTc5MDczZmRlOGZlYTJkZTlmYTAxZTAzODFhNGNjOGEyY2UyODFlNWFmMWI5ZmJlMTM3NjViMjlkZDY1NGQyM2IyYTYyZTJiYjc4ZDgyODU0MmQ3NzYyNjhhNDM4NjYyY2I5ZGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pcrPY79LpmMw82b7aim6EW3uvrXx9v6twJ5-eJvZAUN5Gk5MQ-dOZ8fjpJnfO1FvyBBcEZ7cujDRru2dL6MkWw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110423_85_249c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.120Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtIcnllM0IyUkxiRU0vMUZnL2hMdEsvUUpaQVRqdUpUYUJrakRrTDBsb1NoRjk0dStML0xoMThvU1JaQUNDMURWdDdWYzJ5NEZTZk52c242NjVzSDVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjNfODVfMjQ5Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODg1NDhmZTY1Zjk1Njk3MzI3MGNkMWI2ZmQxODQ4ZWI0YjEzMjdjMDFlZmRhY2RlMzljZjBjNGNmYTNjMWE0MWI1MmIzMzhmMGJmZDRlOTQxMjY4M2QyMjdjZGJhZTQ0NTE1YWIyYTZhNTRlMTQzN2Y5M2QyZTY4NTUzYzBjODNmNWUxOGQ0OWZiNWVlZmM2NzcxMGY4NTQyZTk2YzBlYzEwNTZjOTc4NWYwNDQzNTkyOTk1YzE0MmZlYWQ5ZGJlZmYyYmExNDM2Mjc1MjU4NDc3YmRlZjVlYjAzY2RmN2UxZTBjYWVkMmE0MzllMDkxYTNiYzA1MzUyZTgxMjk3ZTI2MGQzZDA0YzA0ZTg0Mzk4ZGU3ODgzZjBkYzMyNjY0MjM2ZjY0NThjMzhmOGZmY2MwMzdiM2UzZGQ1N2Q5OWNhZWQ5YWQ1ZmQxZGRiYzAxZmFjZDJmNjNiNjg2MDhkZWY2YzgwMGI3NmY0ZjQwMjcxZWQzYjI3MzVhODI4OTZhY2EyMWI2OTg3ZDFjNTg4OWE3ZGU1Y2MyNWE0NjM3ZDM0NjU3YzdiN2I4NTc5OTM0MGM5OTQ3NjI0ZTM5OTQ5YjhiZGFmNzdiMGUwMjYxOTE4NjZkMzgxZGY3NDlkMDYzOGI1YTBiYTFkY2I4MDJmYTFjYTNlYjcxMzM5NmFkOGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EcgDOF-cx4ibeI2rPlWbpRvnhEnzvew8Kh8z7PRnzhTg5pz_IqM-hYTYwLk6RhJc5k8nO_SCI8OL6DXx7TqpnA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110423_85_249c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.123Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkhOSzluTERIZXFoUmNkbU5SWUhMYzdLRTVmQjVGcGFHT2puRStHNkxQc2daNndORm5HQ2F6cXFPVGRCZWcvdGo4RHN1MU04SVViTVVuWTBoTDVYVjFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMF8xMTA0MjNfODVfMjQ5Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjgwMTg2MzAyMTA4MDIzZmUzZDRkZDU1YTVhZDYyYTQ3ZWQwNDY4YmRmMjkyMDYxYWVlNzA2ZGU1NWQzZmNlY2I1MTNkN2MzZTBjNmUwNzE0NGE3Yjc0NTFkMjM3N2Y2YTQ3NDcwNTk0Y2Q4MzE2OWEwMGQ3Y2M5M2YyNTY0ZDU2NDNhNmIzNTFhM2MyY2QyODc5ZjQ5MzVjYTc3ZWY2OGNkYTJmYTA0NGNjYTE3YzJhMTczN2U1OTgxN2VkNzZkOTBiNTAyYjNmMGU3Y2YyNmEzMjQ5NWJkNTQ1MTg4MTVhYmEwMGE3ZWE4MWE4ODgzNGFiMWY4NDlhYzdiNTE0ZDU2MTRkMjUxOWY1MzBmNjliMDkzZDIxZjA3MTVjNmNjMDI4ODhiNWM3MWI5MTA1Mjc5ZjUwY2UyZmJhMjA0MThjNzM1MDg3OTkzNTJlMzVjMTI4YmExZjRiOThjN2IwMDhkZTg4YzExMjJjNTZiNzRhN2NmMTY4ZGMxY2Q3Nzk5YmRjZWU4YTU3YmI2MzNjNjY2OWEyNjFkNDJmMGNlZmViYTVhNjY5NmM4YjFkY2Y5MjY4NzU5NTcxOWRiNzkwZWI4ZDA5NGNlM2U3NDQxNGM1NjI4ZTY5ZGEwZjQ4YzQwMTgxZTBhODIwOGEzZjg1ZDBkNmUxN2M4ZmFiMmEyMTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SPRBnD6Z0y-f5fR_dbwCqB8Bx5ffDdJkaz4H-_fCzcG4ap9WvlCdNeT45fWKDZeIDz1X6n5zJjHupkTfWYRI4g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220910_110423_85_249c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.126Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjRiaHJ4NGUzUkhOdTJTUlQ4aklyRnlOQkZybVpHV205MFpTVUwxVnVKd2Fhdk4yeXYxRjdTVU9pWWxPVW80cm90T01ieGJtSDlWbGpDb2FLMDQ5anZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzVfODRfMjQ0MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWFkNDM0MDljNGM5NzM4ZjRiNDUwOTQzODMyZDRlYmQ5M2FhNmExMzJmOTc5ZDVjN2RhMDFjMThjMWM1Zjc0OTU0MWY4MWE1Y2Y3MjZjMmQ2NmVjMTNlMjc1OWRjMDQ0NTdhZjdiZmRiZGJkMmUwMTUzNjIyMmM5ODcxZGIwZDJkMGZiZTAxMTgzY2FjMjI4ZTA4Mzg2ZWVlYmE0YTM4YmQwOWI0NzY4NDhlOTU0OTE4NzU5NTJiZjlhNGU1Mjk4YjcyNzhjOTg5YmIyY2Y3MjE5YTQzNjBlMzhjMTQyYTBmZjE2YTViYTRkOTcwOTkyZTJkNGYwNjQ3ODJmYjc4ODM0MGZiYjkzNGViZWNiMTU4NjQzYThmYTA4MjVlYWM3MTZkNGFlZjBiNmFiMDY0NjYwNmFlYzdhNzlmZmZiZDJkNGVlY2Q3ZDg3MjkwMTU0YzcwM2Y4Y2QxZTJmZjcwNDc1NTJlMDI1YmZlNmQ1ZTM0NmZmYmU0NjUwMzQzZWNiYjFhNmEwMjQ5YzIxNDRiNDQ3MTkxNzI0YWNmOGVhMjVjNDllNWRkN2EzYzEwMGMyYzM4MjkwNTBmZjQwYWI3Yjg4ZGUyMWYxNDk0ZWUwOWExOWNlYTU1ZDJkNzlmYjUyOGU3YTE2YjUzNWM0MWI1ZTYyNTUzMTY2OWE1ZTRiNTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.W38wxtEF8gcOGbG8mpIqddX6p3ICYHoUmGK_KjrXsuHHXBDSyV6zh-emStmNmkJwq3YKrP69DknLhkebajIoOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104635_84_2441_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.129Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InRFdVJNYzF2bE52YXMwek9sZWJkSUxmREN3S1pkblBPVTBtN2VXZ2JsSGM3ZytySXEzVUlFRDdrcHFQYm0yVVdGeld0ejFTVHhVWklna25PTVRvcCt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzVfODRfMjQ0MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWUwMjk5M2JmNDg3NjU3NWNmOTY4YTZmMDFhZjdjMmRjMzAzYTU0MTg1ZjA1NzVhNzdhOTdiM2E5YWJhMTBjNjI0NGYzNWVmOThmY2M1NTYwNWE4MmY4OGFiMTllMGI1YjUzZWYwZDk1YmI3NmJlNmE5Nzg0OWNlNjFhMzM3MTM5OGZlOGU3NWU5MzdlMWZlNzJkZmVjMzYzMTZkMTZlY2U3OThkZmNjMGM5ZmMyZmU0M2Y1MjM2NWZhZjBlY2U3NmQxNzAwOTVjM2Q3NjQ3OTY1Y2Q5N2IwZjRhMDRmMmI0OGM0ZTQ3NGQ0ZmYwMjkwZWM5ZDI0YTE0YmNkMTBlYjEzZmIzOTY5Y2FiMzBmNjk0MTVkYTliMTkyZWFjMjVhYWQ5MmRhMGY0YTY5ZmE1ZGQ4YzdhNGJjYzNkZjc4ZTc2ZjAxYmMyOGQ0MGIxOGExOGE3ZWMzZTViOTZmZTZmYTUzOTVmZWNjOWM0MTc3YzA0MDZlZjYyMDExMDliODVhOWM4MThjYzFkNjc2NjI5ZjFiZjY2NmEyNWYwNzM1ZmM3ZjE5YTc5YzA5YjAxYzg1ZjE2MDAxYzQwYmQ5YzhlNzk0YTYyNTg2NTgxNmZkNDQyNThkMDcwZTk0YTYxOWU0ZGY0NDFiZTRjN2U2YTNlNzc1OGZhOTEyMzJiZjZiZWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bTqyiKZPv-0UvYI_sQo-oVVWLaHl50ZB-IHqbdAIypZQG6LgtNwySJu2JdbJDc8roCMebbiNKSSW7VluYoXqLA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104635_84_2441_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.131Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InRnY1d3THJwVGliMHJVYlg1OHBIUUl2R2x2c2dMaXV6eFBCNzNSU3pIMlFkemVPQXVPWVp5VVBuRGIwdEx5U3dmemZGNTNiaWVDNGNQZjgwbnJVZDZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzVfODRfMjQ0MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzFiYzU3YmViZjVkYmY5MGJhMTJkZjlmZjVmNmQ4MjEyMDZlMDlmNWYxYjhmMjc5M2JiYjk5ZWJhNmU3YjY3Y2E3NjVjNTNjYWFhZTJkMGZlZmJkODJjYTAwZWQxOWE2ZjZkYTY3M2I0MzY2M2I3MmE3MGY4OTkzNGE4ZTM4ODliZGUwODM0ZDE5ODk4NzQ3OTkyM2E2Y2M1MjJiOTU3ZDgyOTYyM2Q2NjE2YmI2OWRjOTVhNDJjODEzMDFiYmEwYjMyYWY2ODE5MWExMmQxZGYzODI4N2FiMjExYWY2ZjRkMDNhZDA5NjM5MGViYzZiNzgxMzkzNWU5YzFkZDk4NzYxZTc0MGY1NjdhYzI1MDBiZjMxOTM2MjhjYmJhNDIyYTk2MDg2ZmRiMDg4YzRmNjRiMTcyY2JjZGJjZTZjMWI5ODViMzhlZDMxOGUxZDllNDQ0YTE5NzNhNGU1OGFmMWY0N2IwY2I1MzY5NTlkZDhlN2Q1ZmJiNGRiNzBkYjUzNTk4ZmNhYWZkNjY0MDdhYTJiZmJiMTUxNDEzNzFlYjA3NTUwNWE1MzNiMTUxM2EzMTk5Yjg1ZmI3Nzk4NGI0MWE1NDNiMWIzYTU2ODU2ZDFmMDA1ODUwNjE4YThmYzk1NzNhNjg1NWU5MDQ2OTA2ZDZiOWE5NTYxNGZjMGVjYmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.u6ReryK7IDnmTXrMgMqGgHqoUUKVp95qTdQCikgGvVac2Y4SDP553LVQeRKz4awW35e8hqjyEfU8ZKzqrgLZRg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104635_84_2441_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.134Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InQ5dEpBMWFPQ1ZqTzdvclZzeTBITjVPNVJ1bDVaTU1EdDNmUXBnbzBoZzdoUWVVbDBlcVNtZlp5bjc3aEFVOVIrUjJMdjhtZlZKYzgyU0V0ckM4RUZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzVfODRfMjQ0MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTU5YzY0YTgwZTFmYmRkZmY5YjExYjcxMmFkYmUyOTUyMzQ4Y2E4YTkwZDhjZjUyMDU4MTBhN2M3YmM0MjIzYzQ0YzcyZDE2MzFlMjZhYjYyOGNmYjA4OWVlMDE5OWM2MjEyZTk5NGIxODk0MzAxYWZhZWQ0NzBhNGQ2YmFhYmQ2YTY4ZTM1ZjNjNDc1MGZjNjI5NWFhMTI0NGFiODgyMjFmY2YzMjg0NGQzNjk5OGU5NjVjNWM2YmVjOGZlNDJhNDhiMmQ4ZGRkODQwZGRhNWVlNzc3MDBkOTc3NmExYzdmMWJmOTNkMWM1ZDNkNmIxMmFiN2VmMWM2Yjk3NTUxNDY2MmQ1YmRmZTk1ODdhOWU3MjdmYzkyNDUwMTNjOGZlNzk1YWM4ODY0OTYzN2Y5MzkwZWEzNjIzYWExZWIyNzg3MjNmZDkxNGI1MDY2NzU5MjdmZTFlYjk4N2Y3ZjExN2Q4YzAyOTY2NDA0MmIzNjNkN2RhZDM0ZDUyNzE3MDg4ZGQzNDYxZmU5OTcxMzUxMDNjODY3MDI4ZmUzYzJkNjZkYjE0YmEyMGQ5M2JjZjVmMzY3YWUzNDA3YjljNGVlYTI0NWM5Yjg0OGViNjE2MTc2ZDljMmE3ZGM1MmIyN2JlNjI2OWM2YTY3ODczYTFhMGFjNDkzOWVmY2Y4MzM5ZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KyA0VXrrY5JCmArA1qGAqgCohXXkIH8YAcfsJ5GNUQGv1sOO5gNHA-V8EpL0ecfwfkB8yl1AKjrAttcSp0bIug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104635_84_2441_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.138Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZWdStYT3BRUTRPZS9oYjNSOStDekEzYmtqdVp0ZjYwMXNCaWc2YU9zMDN0MzVVSjUybVYzRjd2REVnRGZpeXh0eUdGdlVTV3hQb0RwLzloZlZXRmpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMDM4MzJfODhfMjQzOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWFhOTZjYTUxZTZhOGVhZDQzNWE0OWM0MGRhYTRmNGNkNDJkN2Y2YzNiYmFkZmVlY2Y5OTIzY2I4MGRhYWUzZjIzMzU5NDI0MmI3MGJmZTA3MzU3MTMyZTNlNjA3YzIzNTQ5MDI3ZDAwZjRjYjM2ZTI1ZGE4NjFjY2Y1YmU0Mzk2ZGZlMTViNzAwODRjMDdhYjgzOTRkM2IzOGNjMjdlNDk2YTNlNDNlMzY2MTE4ZjE5YWNjMmIyNDk2M2ZjZjI1ODJjMDFjNTcwMWNhMThhZDYwMzY0YmU2N2IyM2YxMmIxMmQyOTRmNGNmOGVmMmViYzA0ZjUxZDg1ZWM1NGE4YmIwMGQxMGVlNTgyOGJjNzk3NDRjMzAxNmRjN2E2NzQwZmI2MzVlNDllYjhmZmY0OWViZGFhOTUzMmIxMmM2OTYzN2I5ZTNjNDZiOGE4Mzk2OGY5ZGUzNDIyODAwNDQ0NDMyMWQ4NWMxZDkzMjdhYjk2ZTkwODc1MzFiOWUzNzRhNzBhNzE1Mjc5YmViMzFjYTEwMTI0MjgwZGNiMzZlZDAzYWMzOTIzYjAyM2U4OThmOTkzYmQyM2NhNjRlNmE2ZmI1YmU2Y2Y4NGRlNDA3ZWJlMjNjODhiYzU5ZDhjMmRlNzc1MTNhZmJlMGM0MjcxYmYxMjc3ZjFmMTJkMWFlODBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iQV4_oQGzoTCqqzRGe7SGR-UR7usQ_Us3G1A4j3QCiDMpJJNDIy-EwIlwbwwhEZfahWUo3H9YHJAG8aoeYsIqA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_103832_88_2439_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.141Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjZRNSs4QU9VUmpyWUZYTWV0K3VnR2tROXhzTUNldHd0cUJNQzJESkRGSkc4QUNyZVlVWWVEWi8xanlGZERrSWRZd3habnV1OE4ySjBaNnRxU3FiVmhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMDM4MzJfODhfMjQzOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDFhOTZlMTkzODc1ZWIwMDVlZTIxZjdiZDQwNzI1NGVlNmUzY2M1ZGY1YTE2ZmU5ZjAzMjY3OWUyNjJjMWQ3NWNkMDVkYTg0ZWM4NTg3NWM0Yjk2NDNhZTUyYTUwZGQ2NjVlNWVmYjI0ZmZkMDgyYTQwYTE1MGE0NDE5YTJkYzc4YzMxMGUxZjJkZDRmYWI4ZGEzMTIxZDg3Nzk4OWJkOTdiOTliNmFjNGIwNTJmYTJmNmVhYWI0NDljZGFhMWVmMDcxMzFlNjc4ZDE3ZmFkNTZiN2ZmMWZlNzgxN2E0NTA4ZGQzODVmMTQ1MDA5NTZiNjA2NDQ0YTFkMDE3NWExNTE3OTFhY2UzOWQwMjdlYzhlODNhZDNhMzg5YTE2ZWVjYTBlZGYzZTFjYTE4YTM5MDE2ZDZmNDA2M2JhOTE2NzRlYmZlMjhkNzNmNTMxZjIyZTQxMTVhZDA3ZDc1NzI0NzBkYjZkNDFjMTU0MTA4YjNlYzgyZmYzZTMxNWVhMTI2NDE4YTcyYzhiMGUwMDBiYzFjYTNiNjY0ZGI0OTQ1MDNkYmNiYWI4MWJiNmNmNzNjY2ViNTNmMmYyNGEyMTk4NjA1M2I5MmM2ZmIzNGIyZWI4YTFjOThlNDA2YTFkMzc4OWE5YWEzNDBiMDVlZDQxZjNmNWM1ZWM1ODJkMGRlZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PGnQYJ3oCc_PyfyGzv4YtSrQyDM9531qBo4NJFjz2eVRHiR5e49-W6c_Ky875hHhCmo-sOtcyBuMS1AJBnh46Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_103832_88_2439_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.144Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IndPMjZnNnhLKys1dGdzd0tEdXUwc3lISGUrN3Z3Z0RoaWZTZHdKSzJTTkRMSmJyeWkrSThRcGYxaCthZUxzTDg5Z3h6bWFOL0MzMjJSNlhJUFo5Rld3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMDM4MzJfODhfMjQzOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTAyYmE2OGY3YzUyMjFlNjc1NzA2NTNkYjVlOTUxOTlhMzViZGY4OWU1ZjQ5YTAwZWZlNTZhODdkZjAxZWU1OWQzMTMwZGYwZDZjODE1NzBmZTE3ZDc3MzI0ZTJmOWI3ZGY5ZjVhNTkzYTI5YTE5ZmM2Nzk2MzljNmRjMmNiYTVkODM3NjQ2NjZkNDZlNzE4MGYzM2E4NGZkZmYxMWNjZjQxNDM4Mzg4NTBmODgyNjhkMmQ1ZDUyNzA3NTI4YWM5YzJhOWIzNWVhNzMyZjQ3ODI4MTQ3NjYyMTE1YjE1YzZiNTNhNGIzMTNhOWM1ZGIxZjY4YTIxNTRiYWU1NDI5NzI0YTE2ZjFiY2FhNGIxZTYzODhiM2M5NGVhYTViOWUxOTNiMTNmMGI0MzNjYTA0MGIyNmU0MWM4ZjZmOTU0NjNjMDI4ZDA2OGI4ZDkyZTQ3MDc1NmI3YThjOGIzMjY0YjBiYmZkMzllMGVjMGI5ZTQzNGNlMjk1NjZiZDZkNjIzMGM3N2EyMDY5NzUzODJjNjYxYjE3ODUwYjc1OTI3NDIyOTc0MTllMDM1ZjRhZDE3YmFhNTY2ZmY1NmMwNjU3ZWYxMDU3ZTQzNTM5ZTU3YmM4MGVkYzJlYzIyZjg4ZGMzZTI2OTRlMzM1YTFmMDBiNzQ5NmIzMWU2OGVjNWM3YTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GkuJsiiT2UIWagOLfMT-pg1_1IhJtkluFBhTppsau39KEibnUUtpm6_RZO4KLGz5pkLZ5g5JFSri0i9N3gAiOw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_103832_88_2439_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.148Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkdyK3JsejMwZnVyU25Dd09uaHBtdHNROTJPNThsUE1BR3MyWENCU0FiMTRYRE1xKy8rSkdVT1FyR1pUTzNobFJiMzlwSi9hQUJNd3k5UklIaGM4WXBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyOF8xMDM4MzJfODhfMjQzOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTVmZWM4YjA2ZmQxNWZiNDlmN2VmOWYzNTI0NmIzMmQ5YTBkODdkZTE0ZjQ2ZjNhYTAwZjc0M2QxNDM3YjYyN2M3MDRmMjI1ZmZiNGIxNDYyMzE5NzE2ZmY3NTc4Yjc2N2FiNjVjNjBiZTZkZWY4YmNlYjEyYjhmYzA4MjliZDA1YTE3YmYzYjRjY2U4MWRmM2VmOTAwZGNiNzIxZTAzN2VjMTFhOGZhZmExNDIzOGJjYjgxMWQ5MDIwMTM1ZmZlYTdhYTc0MjE2ODhmMmEwNjA3NzUyMTBhYzM1YTVkMjFjYmJhZGJkZmE3ZGJhZWRkM2FhMTdlNGNlZDk3NDNkYjNiOWRkNWIwMmUxMzVlOTFkNWI3MzUwYjgwYzQ2NWQ3NDk5MTlhNDk5NzhlZWEwNjcwZTU1MGFhMjBkYzgzYjIxYjk5MjAyY2NlOWM3MWFhMzZmMTRmNTY2YWI5NjUzMjNjOGZlYjVlNTg1MzdjMmM4NDZjN2NjOTc0ZTI2NWJkODBiYWQzODg4ZWE0ZmY5MmQ3ZThiM2NkODI0YjM4ZjI3YmMwYzhmMTYzOTk2YzZjY2QyNTY0NzA3ODY0YzlmMmQyNWM4MThlMjc5NzZhZDg5ODY2MTY3NmVhZTU1ZDdmODgxNjUxNDdkMjQyYjVmYzIxYWE4MzdjMTAzYzUyNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Z8vNSgEVePiFe2Ba2l_5gqFF0NKLc4B8fdF0wlNhwjXCw7y8eBWEZsVPK6Os40UMIwZbx0ACtQVF9KoqKg2WlQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231128_103832_88_2439_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.151Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InMvMldsTHBNemdIU1FlclpLMTBBK2RkbFBNQmk5V2dlMU1qSjBaNE02MlFtSTRHekhHaVY2QWVaaEk1RTZBMFVaUlIyanFRUEp6S3VBL09CVGlWb1FnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTMyMTJfNDlfMjRmMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2U1OTAzMTc2OGFjYmQ5ZTdmM2I5YzYzMDY2ODU1OTIzNWUxZmZiYjZjMzk1YzY1YzIwMTk5YjRkODgwZDIwMWM4NWMwY2MxNzZhOTBiMTljNzhmZmIyMjMxOTg5NGYzZjcwMTU1YTJjYzEyYTcwMjc0N2E4YmI4OGE4ZjljNGIyMmJjOGRmMzIwYThhODU2OTEyNTgyYWFhNjI0MzcwOTA2NWU5MDgwNzY1YjM4MDYyODEzYzVhNDhiYjZkZjkyY2EyMjA0NzEzNGVmNWY2ZWYwNzg4YzMzOWFhMDYzODgxOTc0OTY4MjUyMzFjZTZlZjc2NzgxMTE2M2I0MWExNzk3YjJiN2VjMzc3NzRlMTU5NzAxN2I1NTE3MDRjNmE3MzA2ZjNlNmFmNGQ1OTc3MmU3NDRkNjM4MWYxNTNkNGYxZDc4NjFiMTkxYjFhN2ExZjQwZjQ0MDg1YjYwODA4NTg5ZDI2Njk1M2MyZDA4YzA1NGU2Mjk0MDNmOWI4ZGFhNzBhODg5OGQxNzE4OGE4MTBhZWYzOTU1ZDA4NjUxYjAwNzliYmNkMDQwYjJkYTRmMjk1ZTYzZDNkYjc2NWUzZjMwNWYxYWFkMTRkZDQ2ODgyOWRhOGE3YjUyMDBjMzVkZGFmYjg5MTc4OTM1YmY5YjI5ZDZmM2IxYTZmOGJiNTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.apw--ozKcIA0NGggNryjm1JiDtKxWM4ZF4INpMekjKEKD3skV4hXfGQ3uoeq6xw0Ko2th-8ZfuDYCxEc4VVYLw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113212_49_24f1_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.155Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjQwNEp4RnBDclFyc2dTUDhya1JUSzIydHMrRE9Kd2trZG1LS0hLMlJpU3liY296MC9iUmxwNFJKbTRiOGxING11YTZJZTJpZDkyYTBwZExhV2k3cVVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTMyMTJfNDlfMjRmMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTc3MWJhZGRlYzdjMThiNzI5ZjM4ZWY2NmNkZTk5OTE1ZjE4NmQ0MDg2N2E1ZTU3YzU4OTRjNmZhMzI0ZDQwNjE5MGYzYzI5NmFlZDZiMTY1YzZiNjA3OThlZTZiZTliMDRmMTkzNWE0ODNkZWMxYzRlNjNlOWYxYzYzZTZhMzJhMTkyNzcwZjFjZDc0NDNlYjk2YzY4ZGZiOTdhMDgzMjY0Y2NhYjJjZWYxMWU2NmQzNTAxODkxZTIxY2U5YjdiYTY4OWZhZjI0MDk3YWQxM2I5MTg4ZTQ1M2I4MTcxY2M0ZTc2NDlhOWM1NTkyMzgwNGIzMGQ3N2Q0MzNiMmNmODhlYWJlNTQzNDAxNTc3ZWQzZTUxODE3MTllNWEzNGQ0Mjc0OWUzOGQwMjEzM2ZlNzMxYTg5ZWJhYWQ3MzM3MTBlMGJmMTRiZmFjMThiYTMxZGQ2OGM2ODUyZTUzMDdiOTZlNzE0N2FmMWExYzE3MzQzMDA4M2ZiNjFjZjYxMTM4NThlNDI1ZjA1ZDY4NTM2YjA2YzI5NjcxMGE1NDMzMzc3MzJkMzUzNTBiY2RlYTk5NWI1M2ExMTNlYzU3MmMwZjQ4YzU0ZDgwNWIwMTk1MzE2YjU5NTVjNjEzN2I0MjA4Njg1MDliMWIwYzU0NDllNDZlNWExMjA5MzMwZGM0MzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uMrJ362AsH6s6cuKnXzwxHgWQezvIp9m1Vq3Y_ACEDpjz6_kFeIobsYj4i8aSLiTUGMXX7xlUtG1aUc1sAlNSQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113212_49_24f1_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.159Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRXcXg2cHYvSjRpWjN4dFNWaVZXY2I3aXBVdnVKOVA3S0pxa1FRb3RuNmcva1pZU3l1R2I0eVJWMzhLVVpxR2tmOWNaQXF4a0IzcjJZbTNFS2JOS2tRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTMyMTJfNDlfMjRmMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTc1MzZhMmUxMGYyYmNkM2M2ZmZiZDJhN2JiNDQ4NGViZWE1YjJhNjBhMTk3Yzk0OTM2NGU3ODEyZTk0YmNlNmU3ZmQ0ZDZlNGU5OGRjZGU5YjA5ZGI4NzhiNzUwYzU3YTA4NGU0ODliOTg4ZmFkN2EyNTg4Y2M1N2I4Y2M1YjhmYTQ4ZWUyNDIxMjE4YzExODU5M2E2MGNmN2VkYjEyZWEwZTI2Njk3OTMyODMyNTAzMmU5MzRlMDFhMzE4OThiNWExNDRmN2EyZTJiNTMzMWI0NTFmM2RiYTUyY2ZiNWEzOTM0NTZjMjBjM2NlZjY2NDA0MWM2YzA0NDk2NjRmMjVlODc4MDYyNWRkZjc4Yzg1YjhmMjU3ZjcxZGQ5YmI2NTMxZmMyYzc2NWQ4MTYxOTg0ZjUxOWZmZTY3NGE5NTQ4N2ZjMjk4MDJlOTk1YTBmMTdhZjc0MTgxZGYwNWMzNDViMjM1YWUyMDI3N2RjNDA3YjI2MGY0YWRkNWM0NzdmYjE5YjY3MDAyOGE1OTBkMWVkODY0NmNlZDk2Mjc4YWRiNDBkN2U1YTg3YTZjMmRmNjMxZTVlYjI2NTYwNDg5MzJjMjBiMzJhOGI5MmQzNzQ1Y2I4NjJjNDJiZmFlNjA4NjlmNGU3NzM5ODY3ODYwM2ZlZjUzYWExZmFjMzMyNGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iOIi7WePruxskIY8fdzw9p9H2vLCO4KD8OXN1k8pjeThTuRus_zAQ8_3EitkhG-ywWsUp2iQa_vsqCC91pN9iQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113212_49_24f1_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.162Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imgwa205alR3ZFdob2o4dk9vZUVoTjVIb0xMSVZqYXNvMFYzWVU5ME81MjAxdWhDc0xFNHNPUmxjYUJ3T3hDT05DM0NVa1ovQmsyT0hQTXlXTzd2d1lRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTMyMTJfNDlfMjRmMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTk1ZGYwYzRkYzVjNjYxZDY1Mjg2YjQ3MDJlNWQ1ZjFiNDY0MDNkNjI1ODRlZDA0NTU3ZGNjZmY0MTgxZGFlMThhYTNhMTZiMDhiYzM1MzBmMjM1NmI1NzBlOWIxZGEyMWIxMTk0YzhmM2QyOTBiOWY0NmZhNjdhY2M1MjQxMzJkMGZkNDc0ZjJlN2YxMzNlOGY2NmY0YjM2MTc3NWVlYmY0ODFmNzBiMjFkMjVhMWI2ZWUxNDYwZTQxYjQ5NDcwYTI1NWEzOTk3ZWZmOWVhYTUzYzZhY2JkMWU4ZGZmNThmYTExM2VkNjVlZTUyNmFjZThmOWRjOTJmZTgzYzBhZGNlNzM1ZmU0ODI3NTY0YjAwN2NhZjU5NTNjMzg2MGY4Nzk3NGFlZTE3YWYwMDE4YzFlOTUxZGQwYWRhNmI0Yzk0ODhlZTQ1MDgwZDU3YzY5ZDRjNjUxN2MyOWQzYjNmNjRjNWMxOTNkMDhkZmVlZWU0ZjViMDZhZDZjYzRkM2VjOGE5YTA4ODI1NzFiOTdlMWM2YTFiY2Y3YzVlNjg2MzE1ZTIwNTBjZmZhZTYyNTBlMzhiOWY5ZjM2MTg2NzYwNTcxMDFjMzAwOTA2ZjMxZWQ3NWI5OTc5ZWJlNDg3ZjJjZjRmNTk4ZDQzNjdhMWUwN2M0YTJjODNlNWMzYTUwNmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.g9m0aOQgs2aDqLkMc-udgYdNHJ5K3_bU6-PcM04nPMemgt9vnDC_HCEdM1WDvJV_vtatAE3K5TiSLhPsEpQ64A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113212_49_24f1_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.165Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZFaVJSWWpTaVFEWlc2UzdqK3JWUHNCS1BMSEVzQWhyZ09iUC9yWkRUNjV5dUZKd05XNC94a2paZklOa0p3Z0lIOUVpWFhjQVNDOFVHdVhOMXIrdkpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwOF8xMDMzMjRfNTNfMjQxZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjJkNmI3MGJhZGJiNTcxMzg2NDQyYmEyNTdhMWNlODNmYzIyNWViNGIxMmZlMzc0MzEwNzUxYzZiN2YxMTQ1NTNlNDBiNDkxMjU2MjI0ODM4OTQ1MjlkNjg4MWU0YjhiNWY3N2YwNWUyMmNmNTMxM2NiYTBjNTMwYWI4MmQyMDA0ODE4ZDQ2MGMyZWU5NDU0ZDNiM2YzN2I4Y2NhMDUxZjIyNWU4MTViMzBmYWQzOTdjNWYzMjk3ZDBiN2Y4N2Q3YmRlZTFhMzEwYTRiNTA4NTc0MmVhM2Q0OTFkNTEyMmQ0N2JlMmQ3NGJlZDRlMzYxMjQwYjUzNmYyNjZlNmU2Mzk0MTA3ZDZiZTA0ZmFkZDg3ZWEwZjZmMzE1ZTMzZDMyYTA0MWRmYTM5N2UxZTdhMmYwOTNkZjUzOTBiZmZlMzQ4Y2E5YWQ5MGEzNjVhNGM5YTJhMzhiY2IxNDYyMGNkOWE2YjFiNTkzNDc4YmNlMjUxY2RlNmI1NzBlZDRiOTAzOTFlYzNlN2Q1ZjQ0ZDM0YTc0MTVkNGVmZGVkMGEzYzNlNzI5MWIwNTNiMGE4MzhmMDQyMjIwNzNhN2RkMmM2ZTEwNjNjMjY5MDM1NzBmNWViZGVlYjdkMmRkYWJhODExYTJmZTY5N2YyZmRlMDcwMjgwOGIyOTQxZmFlMDcwYjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.viyAo2KB1YpDQRxWrsAtMzujVBR4-g1PeQkz8EaeiTo-RPPehva5uuXP8QdO6BiTwTV-wQHPUIAQP_4O-7ah5g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220308_103324_53_241e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.167Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFTSmlPdUpjU3pCV2JIOTI5NWUrY3BPTEUvSUtiSkw1UWVwR1ZDVHJnbmh6OFNtQ1NteDhDQnVaN05sSE5zZlhhbGxPZWVMQ3JGMktYUlJRd0V2bTBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwOF8xMDMzMjRfNTNfMjQxZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmNlOTBkODMwNGU2NzUwZjExYWY5MGI5YTllZjg5MDMxOTBlMjlhZWExODcwNzk3OTg5ZGI5MDE1ZmNiMzY1Y2NjYTE1YmMyMzgwZGYxMjE0MTc2NzQwNjhlYmEzNjEwMzZkZWQxOTQxZTcxZGQ4NjBiYjYyYWM3MzJkMDYxZGUwNTA2MTEwMzI5Nzk0NThlYmYyZjMzMDdjMWViY2JjYzU2M2MwNGQ2ZjBiYjY2ZWNhNTM4ODJhZjBmNjU1Yjc1NDY2NDJhMWM4OTk3ZjMzOWY3N2YwNmEwMTdhNzE5Njg5NTk1ZWNjNzcxYjJiYzdjMzM1ZDlkNWMwMTE5OTk2MWRlNmNmZWFjOTU4YjJjZDY5MWZkNmIyZWNjYzJjYmY5NTI5ZmE3OGRlOTdmMzQ2YzJlMjRlNzQwYzU5MTEzNjhiMWQwOWZmYzQ3ZTdmZDE2Yjk1MDcxNmY2YjY1YmVmMDQ5OTgxYzZkMjhmMzhlOTQyMThlMTA2Yjc0ZDgyYWViNmFiNmFjNTJkNzE0ZWU1Yzg0ODJmODI3Y2JhMTI0NjRiNWNjODc0NzgzYmYwM2IyMzRiN2U2YWQ4MWM1YTEwNjBkN2QxYzA0ZTA3NjNmMzM2ZmIwZjE2NDFlZjVkMDRjZDQzYjc1ZDRmNDg3NDIzYTU4YzhkZDc0Y2MwMjYzOTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CCHzKKbLNAvBTIylxR2e00MWMhxeRsP0TPKf6sp5OkqvOrGe5pY417vs4cfG4COXU2AojWfsTDp1dCCR-r1JZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220308_103324_53_241e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.170Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imo2cUFkUm1hc0d3Sk9PLzR4WVBqZFJsanlqd2JaaWowc3g5OE0xVHpuTHhjRGlpQ2pmbTBSa1crZStsQmdpd3RlZ2ZVT3V3bE9SR0E2NU5ZS0VqeFVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwOF8xMDMzMjRfNTNfMjQxZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2JmM2NmNjYxYzQ4OWFiOWE2MTUzNWY0NDdiNTcxMWQ2MTlkMzFmM2QzYjZiYmI5ZjJkNjVmOGJkY2MyZjNmYjU4MzY0Y2QyMDQwZDExMTkwODVhOGJmMTg1ZmZlNjFjN2Q4ZWVkY2JmYzUwZGM3ODkxZDMyZGFlYTY0MjM4ODQ4OTQ5OTNiZjM1YTNlZTBjNmJkZDkwODNkYWVlMjRmMGM0ZjA3NmE5ZDNhNzkzODE0NjRkYWIyM2EwYjU2YTAyNTUxMThkNDhkZGUxNDk5ZjllMDNhMTZiOGFiNGJiYWY4ZWVlNWY1ODIyZTNiMjZlMzY4ZGVlNGNhYjRlMDAxMjc0MTkxYzJhMTdmMGNjY2Y4OWEzOGE0ZThjZTYxOTJkNDczYzU1Mjc2MDRhMDQ5MWJkN2IyYjhhM2FjMzY3YmFkMDZhMDFjYjYzZDA0ZjVmM2I2MjMzNTAxODMzYmEzMWRjZTI2MWJjODc2ODViNWM4MWNiMDMxYzI1YjdmMWZmYTYzZGUzZGUzZTgxNGM3YmRiMTk0MjU2NGM0YzFjZWI0YjdlNmVkOTRjOWZjNDc2MTRjYTE0ZWVmMzVmMmRjYWQyZjk4YzgwODZjYzk0NzBjMTE3NjczZGI2YzQyNzk1YzEzODM3NDM2MDVmZjgwMzJiYTllMTRhYTFmMDUxZTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3ccOJMOnxLK5PqnqKlwwSqRFFlsip4smwX7CrDUVabCclwE0K4mAiFBsS7eSk5a5F7GgVuYI5Fu9xJy81mMh9g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220308_103324_53_241e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.173Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZ4YlV1N2RuclhCTWVkSFBnTTRlTjFXMW5FNS9pMFpKRmVZUVM1eThLRitxSVNOdUhXbC9JZzUwakVwaEhhMUFHQTNHTk5pUWt4YTNZZXVWR3JiQ0x3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwOF8xMDMzMjRfNTNfMjQxZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Nzg3YWZiMjljMmNiOGUxZDJiZTQzYjdjMDNmNWZmOTJjNDE2NmZlMmMwNzg3MTdhMzgzNzU2NTUxZjU1NGRmZmE3NTljYjRjNWYyZjJkYTJhN2U2ODJiMmFmN2VjMWExMWVhZWMyZWQ3YzQwZjNhYmYwNzYyNTRhMGZmNjdhNjk5ODIxZTYyNGMxNmJiZjUwZjZiMDQ0OGY5MzY4ZTVlNjYxNTFmOGQzNDk2Nzk3MjRkYjg5MTE3YTQwMTQ1YTg5MWIwZWQ0NThmNDlkNzM0YTE1ZTZmN2RiZjRlMmEzMDU1ZGY4OTgyOWY4ODE1OTVmYjYxZjY5ZTNmM2JlMWE2NGExMWZlZWJjYzdkODM3ZmE5MmZjOTJiOGRiYzNlOTEyNjZkMjY3YmFkZjQ1YTM0OWUxMTNjYjdmYTUwZDhjMjNlZjJlODUxMmRhNDczNzRiODMyNGNmNzNkNzg0MGQ4ZjJlOWQxNWZhNjRiZjBjODg0NTg2OGVkOTE2MTFmNDMzNGE2YmMzOGU2ZTQxODNlMWFkOTA0YjY4MmRiNjRiMjhmODg1Zjc2MjE0ZmE2MGYwMWE1ZWJhMjY5MzBlMTI5ZmFkZWNlNjU2MTJiMDJlNjNlM2QwYTI0YWJlZTE4NjMzN2Q4YzJhOTNmOTIzNGI1MjZjYmNiMjU3NTQ3ZWU2MDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._J8v_49zUXxoDYewQgTZkzqZl0-sNbCqQx3nhfL0lVypXatLMCwRJRmdERLLkl0himRF0CJMZ8H-Z4wpmX2RNw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220308_103324_53_241e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.176Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InhUdUZ2ZGgvSHBIMGpCN1l2YXRITTN1T2ZJRXhXMFZlVjVUUHk1ZGN4MlNWT1JRZjRJUGJodkpYa3NEakR1cHhpdjJPK0dhV2RlY1ZPZU96MXV5clNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTNfNzZfMjI3Nl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjUzMDgwOGE1YjBhMGUyMDM2NDRjNTgxNDlkNDUzY2Q3ODZmMWYxNzQ3Nzg0MTY5OGRmN2M5NjU5NTgwNDA1OGZjOTQyNGYwOTJiMjdiNmVkMjdhZTExYjkzYWVlZDY3MmU2YWU2YzI3ZTlkNWYxMWM3YmUxMzk3YTYwNjJhNGUwNjc0ZjZkNWI3NzViNzk4MmQ1YWQyNGM3YTk1NTFiM2IyMjZmNWE4MmEzZDg1MjgxNjU5MmQ1YTgwY2Q1NWY5MjQxNTY0MjNlMTlhN2U4MzM3OGQ0YTc5YmE4ZDc0NTE4NjYyYzkzMjdiZmU0NDk3MDNiY2JhMmU0ZGM3NTg2MDUyMmYyMGJjZTc4OTRiYmI0MDdiOWRlN2ZjNzdkZDAxZmJiY2U2MjU0YWNmN2UxOGFhM2NjNTYxMThkNzFiNjJjM2VkMjJhMjRlZjE0OTMxOGM4NDQ5NzQ5OTNiNzRiZTM2MDJhODJmMjgyZTc5NmY4ZGEyNGVkMjMxYmVkMDJkOTFiZjRhMjk3ZTVmYTU4YWQ1OWQ3NTQ4NTBjM2IxOWM5MjQzYjI5YjIxMTgwZDg3YmMwYTA0NjFjNjE0OGY0YmI4M2JkMDQyYWIzOGVlYTUxZWJhY2U2OGJkYjc1OThmNWIyMTdhZTAzOGJhNDczYWM3Y2E2YzRhMGFkY2NkMjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.o8KUi9kLt9klHLljCRkidaoVDrIoGU3ol616V1796l2ph62ivpxSP22sUP62d9jGqaQvhJAgXPchurEGGuGsgg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104713_76_2276_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.178Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImhCbnFzT3VkNzBJS29TQkROTUtkbTBvN1JFenEraC9zU0JpQmsraWVPVFpvc3R3VXp1eGVmN2U5ek00a2lKb01qMHRpUEo4TkQyaWN2NUZRajM2SFRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTNfNzZfMjI3Nl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjJjYWMyNTE1MjdmZGQxNzlmZjM2YzFkMWUwZjgxZWM5ZjFlNDkyODQxZjMyNDIyYjMxMzgzMzFjOTYxYjk3NzMxNzE1NDAyMjA5MzM2YWI4MzRjYTZmN2NmMTk4Y2ZkZTg0OWIwNGY4ZDEyNzZhMjg0ZmIzMTdmNmIwMGE3OGFkZTFjYmFjMzA1ZWE1YWI5ODQyYWIxZjY0ODU4ZjlkZTkwYTQ4ODY4ZmQ1MTVmODVkN2UzYjIwYTQzMTIzOGRlYTQyODYyZmU0YzNlODAzMWJhNWU3ODAwZDFmOGIzNDc2ZTE1NTMyOTJhZTA2M2RhY2I5Zjk1ODM4YjdmYWFhNjBiMGVkZjg2NTFlYzc5NTY3MTA3NTVmM2U1OWQ5Yzc4N2Y1MGFiNzU2YzFhNTMzMzU0ZWRmZWY0ZDQwOGI1YjEzZmExMjM5MDZhNWM3ZTEwN2Y5ZWUyNzQ3MDNkOWRmMTkyMzYzNTRmNzQ5Njc5YTA5YzQ1YWNlNGI1MzcyMTAyNWNkNDlhM2JjM2RhNWJiMTZjZDMzMGYyMmUxYzA4MTEzNjMyMmEzOTZjMDQ1ZGIwMjgyMzZkNzRhMzdkZjMxNDQ4NTY4ZDExYzBiZmRhZGRjOWRlMWRhZTRhMmUzNTQyM2E2MGY2ZDY3OGFlZWVjODU2ZmVmOTI1ODVhNzU4OThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Dd5OfqoPm7vBLgHoy3MZzVOMIDMxcd6SrVD5K-HC7Ep1qJLQYW8a5sNObZBz0bTINlfUULFCDG0DnkA7lYgqnQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104713_76_2276_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.181Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InA0RWg2S1VabGNpSEtuaE1uTXduaFhHUStlL3R4ZG8zRGxzTUU2MHBBTW1EZDUxVWxyZERNMFZoNk0xVXhnYzk4Kzh0Z1g0OXoxVXYwQ0FDTTg3eGtnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTNfNzZfMjI3Nl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmVkMjU4YTFmZTVmOTc3ZTFmY2JiN2JkOTQ5ZTFiZDAyNzk0YjE2ZWFmN2QzMTNjNGI2NmRjYzEyNWJmOTlkMjRjZDAxOGU0YmIxYjFjNTBiNDllNGQ4MmU3YTZkMjhmZjIzNTEzNjU2NDc2Y2RkZWVhZjg1MzVjZGIzYTM0N2E0OWJlNWY1Y2I4YzVjYTA0ZjI4MjY2OGNkM2Q0YTY4MWFmZDBmYmJkYzZiZmU0NjhlN2Q5NzU4NDIwNGNhMTRjYWM5YjgwM2U2NjllYWVjZmFlMGNmMWQ3NTViNTJjYWU3MzM3N2IzNjUwZTY2ZDczYWNjZTU2MWY1ZWVjOGMwOGRjMzhmMmI4MjA0NzY0YTk2NTU1MzUxMjMxMDZmZDZhNzVmNWFmOTk5MWM2OTBhZDIwMDliODU5YmNhYjdmMjEwYjA3ZGUxMWFlOWRiNTUwZGQyNGQ2ZjBkZjAyOWJiNTFjN2U3Nzg0OTBmY2Q2ZDY0MDBjZjBjODJmMzc2OTBhNzMwZmZkYWEyNWRiNTE3MzEwYzIyYWVhNDc1YjY3Nzg3ZmI1ZWJlM2QyNzhlM2RlYjU1YjBhY2JiOGU0ZjM5ZDZlNmNjYzYzMjc5MDM5YjAyYzFiZTRhOGY4YWQ0OTljYjAwOGNhMzIzNzBjOWIxNmQ4MDgyYmQxODFiZTFlMzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zIU3K-Fmfr7TH-elzihJgbPka3fd5eQyrSkwvA9azv8_C2GUoJvFMRV8O1CY_Phuf80XgppDHEOugmb-wIKTBw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104713_76_2276_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.184Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtvTzh2N1RaZ0pRZy9xRTZNUkVzZ0J4eHlPQnk2cmRMbDdhaGcrUUs2MFg4QndnbEhTSStSUWhDMkZoVVExQ3UwZ0xHd2ZBSDVjZGRwOUlWWFhld3lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTNfNzZfMjI3Nl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzNmYTEyMmE5M2VjNDQwYWZmOTdmODZhZWI0MmVmMDRmZGYwZGRjNTE4MWIzNjZhNjQwMzVhNjk4ZWE4ZWMwM2MzNDdlYjFlN2YzZjBlM2MzMWRlMTY0MzczNTRiN2FiZTkwMTkxZmI0ZDYxOTc4MmJlNjRjNzU2YWM2YjFmYWNmN2VjM2RkZmU1ODZmYmQ5MjY1NTRjNGQxYTI1NzI1NGU3NWZhNzhiZjZiNTY1YjViYmExZTNmOTBlNDAyNzkyY2YyYzc1ZGVmZTk5ZjY3ODRkZThmN2IwNjQ2MmViMWU3ODBhMDQzZDUxYmU4ODFiYmZiOWFlNDhiZjBjODAxYTlkNTRmNjQ5MmExNDQwNTNhMjViYzhiYmI0OTZhYzUyNmJiODdmY2YzMDY4ZTkyNTMwNGY3YzE0MGZjYTExMjg4OWI3ZWY3NTMxMmEwOTM3NWZiMjE1OTI2ZGQzYTBmNTA0M2M5NDQwMGUxZTJmYTAzYjYxN2JhZjI5ODJjMGMwYTczYWQyM2M3NWVjMGQzNzdjN2UyZWQ5NzAyN2JkOTcwZmQ4ZWYyYzg1YmFlODllNmQ4OWRlMDkyMGRmZDFlNjlmNGZkNWU3NzQ0NjNlY2NjOWRmYWI1ZDlkZDBhNzI2OGYzZTBlYjE0ODMwNzE3MGI0OTQzZWNhYjc2NjI4M2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.r86MEzP_g3PhWFroXfxcpb6HqGAiDiNSHOd-dr3KQGgMvTZIFp5BP9wsqXmE8KQPN7OItxvCuNyd8GbSFShOOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104713_76_2276_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.187Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjFiNzdyeTZORjNTQVNaNFkyTFFrZ2w4UUNXVzB2VkZWK0JCd0U3d3F2L08rTUJKU21Mdis4bitUOUgya2ZLZFJSL3EyemRManZGUVFkY1NtQWYvRVhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MjhfNjRfMjQwY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTlhOGVjM2EyZDE0MTBjZTI1ODY1M2Q5NDI5ODliYWRiYTk3NTc0OGIxY2Y3MTkyNzUyNWFkZmE3Y2QxODEwNjU2NTY5NzM4MmZlZmUxZWY0NzZhNzc1ZDk1OTRmOGJmZjhhOGFmMTcxMTNmYzRjOTRhN2MwMmQxYzY2MzRiMGY1OWE4MTA1NDVmOGM4NGY1OTI0OWIxOGZmZDY2YTAzZTI1MjNiMTJkYTk1YWI4ZDkzOGY4ODVkNDQ4ZmQyYzI1NzlmYTI3NzA4Y2UzYmNhNWJmZjQ5MDlhMzVlMGVlMzNkN2I3MzgwMDcwZGQ5NzBjYjBiMWJiNTNjYzEzYzk3YzBiYWEyMjk2MTVlNWZmODZlOWVhZTBiNWI2ZDA2ZWJhZWY0MjgzMTE1Mjk2OGE0MDk4MWY3M2I5ZTVjYzg2NTMxNWViMzdkM2Q0MmEwZGVjZjNhMzk3YTc4MDExNWVlMmE4ZWFkZWNiOTA4YzIwNjhlYjdiZjY1YjE5NTQ2MTI0YzkyM2MzN2MyZTMwNjQyMzk1NzA0NWMxZWRhYTE1OTA5OTUwZjUxZWE2MDk0Zjg2ZjAwNmIxNjc5M2E4M2QxODRjMWFjMjExZGVhZTgyMzI5OWM2MWRlOWU1NzMxOTdhNTVjNzQzOTExMjI5MTI1NGZlNDI1OGZkNTg1NjUwNmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XPmiXzSosuyV7OOcxqhi_cAkAevRUhz3CyWjP2JfcRGggEEG2Jhl10xKOajK5c5kSCPqOiRPbVAeHNXZL6RBgQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111628_64_240c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.190Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlBvQjdUd3QyNEtXb1p4YjJERnR6TWhhNURuaEk0eWw0b29kYzFFdHVZSEowa3BSNlhHNXFaSGNZVkhZUi9zd3J3NnNwR0ZvYlR5UFNGTmh6djhTTVJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MjhfNjRfMjQwY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTY2Y2M5MzEwMzA0ZTNmN2JmZDg5ZDc2MmMwNDg2ODIxNTg0YzE4MmE1NTU3Y2YyMDMyYWYwNmIxMjRkNTg0YzU0YmFiOWUzOTBlYmJkZmEwZGY3MmQ1ZGZiMGIxODY4NTA3YTE0OTM2YjNlZTIxZDQ0YjJiNWJhMDRkYWY3OTdlMGY2OTQ5Njk4ZDY4NGNmODM1MzYxMmRjNTQ3YzFlNmE1MTVlY2NjYTJkOGE3NjZjYjQzOWVhZmUyY2VlNmM5MjM0N2U0ZWQ1NDY4ZmJlY2U4YjE0NTM4MThlOWVhZGM3NDZlODUzMGY2MzA3N2JkNTc1NDliOGIyYjlhMGNiZTQ4MWEzNGViYWU3NzZkNGNhN2ExYTQ4YjkxMGVjZDYzNzE1NjE0NTlmNzZjZjQ5M2IxNDNhMjllMzI4MDE3YzgwZDUwZTNiNDkyZmI1YzM2M2QwMWYxZjllMTRlMzI2OWQ3NjQ0NGQwZWU0MjYxM2Y5ZjM1NTUxYjU0NmZlNzRmMTE4YzRjZjU0ZWE1NTE5YjBiMjM0ZjA3MWU3ZTA3M2QxNzIxYzc1YjA3MDUwMWViZGVhMzg3NGU0NGVjZWIyYTMyYzYxMDBlNjg5M2Q3MmJlZjcyYjQ2OGEwOTdhZTQxMDcwMzM1NmNjYmZjYjAyMmEwNzg0ZmFlNGIxNTYzZTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._24nQIYgTRdFmQPYS4P_RPat8Ynrn_w5313yvoovS6gV-ubhHbnCyiAKTLWQlapwXqXI-2-dG2z4HN1MYmAGOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111628_64_240c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.193Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkJUNEdWNWgxcm12U0Z4eE9STElZN3JYRUNVY3ZXcytpSVRQWEt5R2hIbFNTRGQzWW9Bbnp2LzV4d1Bud1QwUkhBSVBtUEh3b3dLcG5uRGxlQVhqbmhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MjhfNjRfMjQwY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjA4M2E2ODczZjU2Y2VjYTFmYzk1MjI5MDUxZWIyODg2MzQ3ZDk4NzczZDVjOGU3MDc4MDU0ZjYwYjUzN2FlY2YzZDVhYjhiMjgyNjUxMTA2OTFmNzRjMjI2ZDE5YzEyZmY4ZWU0OGIwODMyMDcyZGIyMGZmZTg1NDFlYWUxYzVhN2FiZDEwZTdjOTk0Zjk2OWI2NTc2MzIxMTRiY2JiN2EyM2E3NDZiYTMyYTc0ZTFkODcyYTRmZTBhMjdmYWZiMGQ4ZWUxNWQ1NjQ0MGEzMDE2ZmRlNGI4ZDEwNzI3NmM3MDViZjExZWYwNDIxOWM4OGM1YjFjYjQ4YTM0MTdhYzU2N2QwNDNlMjk1OTg0Y2ZmZmZkZjhkOGY1ZjM1ZmNmMGY0ZDEzZDQ5ZGQ3ZDMxNWQ4NzEwYmQ4NmE4ZWQ2OWY3YjZjMTU2ZDhjNzFkODJjNWNiNTI3ZDU5YjY4ZWE3YzBlYzRlNDgzMzE1ZWNjMGMwZTQ1MzdhODlkN2YyNmM3ZjUwMzI0NDExMjY0N2EwOWE1ZGQxZmRjYzM0NDNhZjg0MzkxOGZiZjMyYjJkNmIzNjQ5Mjc4OTZmZDZlYmZjZmRjMzM1NDYzODlkMGY4ZjM0M2ExOWU2NzYxZWFlOGIzODhlNzA5Y2JjMjUyYTZhMmU1Y2M1MTI1ZmQwODIwY2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Smd_R3ZgMHvUdutOoSPvRPWXpPWuwFMfKRsT07T_XtGbwVXXSv0UhIHTmxKyl2fJFtzaWA2ieME1v8V6WSWeZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111628_64_240c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.196Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imh5dm00WjFXdTQvL0gwOW9uSGVCL0F1cS9wNllvSTZ4blNmSnFMMDYyOFFabFBnc052ekk1RzZtNldUUXB0cWtRbHN6cTNxL3RQbnNJMjZUUXNWcTN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MjhfNjRfMjQwY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDIwNDg5NDE2NzlmODBmMTQyMzhjZjgyYTJjNTUzOTRjYmNmZTdmYzMxZDgxYmRmMjMwODA0ZjE5Mjg5YzE0NjVhMTUxMjRkYjMyOTFjMzYxYzUxMTQ1MmM0N2I2YmM4MmIzYjM0OGEwZWMxNzI5NDFjYTVkNWUwZjRmYTIwODc2NTczZTk3MjM2NWRmMmYzODkzZWU3MzYwNWE1MDFiYzI4MzRlNzIyYzFlM2MyMmJlYjNkMGQyM2FkOTUxNmIyYWRjM2U5YjY5NzQyYmFjZTY3NTlhZThhZDUxZDhjY2U0YzlkMWViZWQ5MjgxNzMxMmVmOWMxZWY0ODRkOTI5YjVjNWJiZTM4OTZkMGNhMDljMGFhNjI4ZWY5YmFjYWZmZjZiYTEwM2VmMjhkYmVlOWVmODM0ODljZWNjN2EwNTY5OGJmOGRiNWZmMWZmMWU5OWI3MWU3NTA2OGRkNmIzNWVlNzY1NzQ5NTE2ZWNjN2Q1Y2I4YzI4MzFlZDc3MWJiN2FjN2E4YjAxNjAxMDg2MmRmNTBjZWVmYWRlNjQ4OWU4ZjBjNzNiNjY5YTliM2IyNTgxMWYxMzE3Mzc3MzQ5ZDE1NDliNGUyMWU1YjQ3MzhkMjIxNmJhMWNlNmEzNzk5MGI5OWU1MTE3ZWZhZTUzZjQ4OTM0NzRjMDRjMmZlNDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.v08PDd3crPYhYz-_qPx9qwMw8Hd_qjYt_2RWk2JQy3pBtV04K277VR4XHNbKZ7bTAbmHmHbX6Z-53Pp1ffxiFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111628_64_240c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.199Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtLS3UvbER5dWxaSnhWbmxkekkwMUY4bTErcW1jMHBZV1NueWNCVnU2OXpsRHVreGwzQUdHVWlpZHYrNDFWZkVOUElHODIzNk9wbW1jWkhaMFJLVWFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMzNfOTJfMjQwM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTY1ZDk5NmZiYzBkNTUyOTliOThiMGRkZTRhYzFmODQ5NTlmODk0MTVmYjA2N2NmYjFmOGIxNzc0MzZjOGI0NGE0YzgzNzA3NTY4NTI2ODQwMjE5ZjdhZjgyMzVmOWVmNTI1NjlmZjMyNWM0MzU4NDYyMDJmODA3ZWVlZDM1YTE2OTMyMDE0YzM3ZDkwMzM0YzY2Yjc5YmRmNTBiOTA3MTZhYzM2YTM3MzVlY2ViMWVhMzI4MWZmNzFhZGFiZGQ0NzNiMGI5YTM5M2IxNWJjODg4NmFhYmJlOTkwZDdhZTJlZjdiZTQyMDk4NmUwN2IwODUwOGE1MGU1NmQ5ZDM1MTYxMzVkMDhiZTNiOWQ1YjRkNGM1ZjUzNmE5YWMxMDgwZjNjYjQwZmI2NjkyMDUzM2ExNGJkY2JlNmY5NDU3MDE1MTk2ODQzNWI3YmJiMDc4ZDY2NDJhYjU2Njk1MjM1OTZjMmQxNmRkNGJmYWRjNDBhNGQzYjFlNzU5Y2Y1YjljYmY3Y2Q2YWNmOTA5ODExZTlkNWZiNzY3MzViODdiYWU0YmUzODE5MGMxYjM4NjRjODk0MWIxMDhkNzhiMmJhYzQ3NDRmOTgyNDI5NDVhZDU0N2Q3NDRlMTNkOWJiODE0ODQ4ZWI5YmEyMjJkNWJhZjUzMjIxYTQ0MzY2YTA4YzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0DrVNoh2oGzwL5MVg2fKvjosya9ZpNvUT88_ugwZAxMEqK2Wp8ExYLMVSPnyC63kgd1O-VHpINJcrSD69hpiNA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112033_92_2403_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.201Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im9Pb2ZBY0RZT0Y5Z1Vyb1ZUNVpoWHFIWFJtV0RydmFnU1lpSWdGK3h0ZC93VzFHMC83TVU0Z3h4ZDhqMGlyNVRBTCtwVXhYbTZZY1lVa3BlcHlqbkJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMzNfOTJfMjQwM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDhmZjEzYmFkMGRmNzFmZDhkMjA0MmNhYmRjY2JkZjc3NDNmNTUxMGRmMDk1Mjg3NGU5NzI5Yjk2NGM2MDM1OWE5M2E4ZTg2ZjRjMTU5YjY1OTQyYjYzMDNjYWUzYTQ1YzcwZDdjZjlhZTBkZDc1YjI5NDJhMmMwYTE3ZWJjMDcxNjg5NWYyMmJlZjJiOTRjODlmN2ZkYmY5NzNiZTFmYzM5NDMyYTZkNjZkN2I3ZTlhMTkzOTA1M2YwMDQxODY0NzQ0NjRlNTk5ZWI4MjgyZmMyZjQzYTJiZDcyODMyMmVmNmE3MWU4NTIyNDExZmZkMTUwMDgyOTk2YzAxN2RkNGFkNTMxNGVjNzc5M2E3M2Y3ZTZiYTkxZDFiYWQzMWQ2ZjZiMTJiNTQ2OTJhMDhhZWRlYjM5OTQyNDgyODlmZGNhMDllN2Q1OTU2YTdjY2ZkYTVhZmEyNWIxMGFjN2U4YmEwNmUzODUzYjJmZDdlYzE4YjEzOTIwM2JiZGNjODczNmQ0NjVlZjkyZTA4OThjZTQ0NmY2YTQ2NTY3NDdkNzVmMDZhZTVhNmEyMWQ4YjFhNzU5NDZkYzAwMDFlNjIyMTQ1ZTk1ZWE3NjlkNWJlMzkwM2Q4MzE5MTdlYTgwZWUyZDA4YmVhYWE3OWQ2YWI0MjNmNzU2YjBlYzgwYTE5ZDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AEVDLu4pAa3gLt8tDv8aouN_DThC61QIl7GWYe7LsuwTfbIHEQnX5sIQLGNOKRgS-hdqbTohmPnRYUYJwf8J_A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112033_92_2403_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.204Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikkya0pRQVlRUFdHY0pZVTRqQ3V5OU1WRm83MEdyT2FqVmI2bkI1aklsWi95UnFUaXRYSjNnUWx0ZnJGckdPeUJ6N1orVFFZVTBxUjhPZXNHbkJiQXNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMzNfOTJfMjQwM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2Q2OTRlMjA0ZDNmYzM5ZDBlODc4MjkyMzZmNTJjOTJjMDllYjFjYzlkNjFjY2QxNWMxYWVjMDgzZDNhYTI5MjM1YWM3NWJmNDI0YzY3Y2RhMmQyOWFmM2IwOGMwYmQwZDRjNGJkODY1ZDM3ODcyZTgzNWM2NGFiOGI2YjBkOWViNjcyYmYyNTFhNmM1MzQ3MjhlNDE3ZTE3ZWU1ODQzYWQzODRhYWIzNTgwOWZkYjRjZmI3YmUyMzZjMGRkY2NkODJhZWI5YTY0YjRiODc2YTAxMGJmOTQ3ZmNlMmY0NTEzMDcyZGRhZTlmMWQyNDJjYzRiM2I5NjdmMWMyYjgzYzM2ZjQ1MTI3MzRkNmVjNTk2ODU5ODc0MjY2NTQ2NTdjM2IwMWIwNzVhNTRlNWIwNjM2ZDY5MmMzZjJlN2Y5ODAwZGJiYTMyZGNhYTJmMjhjYzJjMzRhNDljZTQzOWEwMjQ1ZGEyOWU5Mzk0ODY4YTBjYWU2M2I4MTM1N2M2NmZhZjY5YTY5OTIwMTllZTU4OWQzZDBhNmRlNmQ1ZjUwYjJjN2NhYTY2ZmQxMzUxNWE3ODlkNjY3ZGQ3ODc2YjZlMzMxZmEwODZjNzZmODEzZTE2OWVkMDVjNzk2MTU1MjE0MjBlOWE2ZDI4ZjdiMDc0Njc0ZDJkOTkzMTRmMmY2ODFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3-f6j35e5KC0cwxsm3iRfhBj9LI0ViKPydDy1YQPUNYsCgyWxabj_xb8j4k4HM4RY-PebtCpOwZ1VgnkobBkdQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112033_92_2403_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.207Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkxvRVhtOFI2emQ2dVVqYTJQcDJhMDZ1NjNDOEFuRjhWNzYzNWsyVUFWSzVJVVZ1UXdtaHJHUXBWMWZHaE5Dd1cwQkdtUUVrVXE3aHl4TFZFYXZvd0lRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMzNfOTJfMjQwM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjdlZDZhMmZlM2VmNmQ3M2RjMzJhZjU1NzNjNjFiODg2NWYwMzI3ZDAyODUzNTc1MTI2NmI1YjE0NGNiOGZjZWY3ZWEyNzQ1Y2NiZGE2ZjcxOGQwNjhiYmY2YzkyMDYzZmQ3NjY3OTJkN2ZmOGQ1YWQ5YmIyYjMyNmMxMWJjMzE5OTk5NDk0MTBmMTAyYWQ3MjZjNTk3MzU5ODljYzJmYzYxYTE4ZDA3YzAxNDE3NzljYTIyNDJmMDhjMzkyNjYwYzIwMmVhNGExZDIwN2I1YTBmYTJhZTM0NGNjNGVhNjYwM2EzMGQwZmY2MTBkNGM1ZjNjNjAyNWQ3MjY4YTliZTE5MzNiNTk2ZWYzYjdmODlhMTI2NTM5MTI1MDhlZTZmZDViMDBjMDI5NGQwN2NhNTBjMjZlZGVkNDRhMGE5NWRiN2I5YjQwODQ3N2UzOWY1NTAyNzk0NTYyOWU0MzIxNGM1ODExYzFmYmZhZDcxY2MxZjVlNzhjZWM1OGFhNzcyNzhhMjkwYzA0MzQzYzUwM2VjYWY1ODI4Mzk3ZWUzOTdiOGI4MTE4MzJhMjJlMjdiMzkyMjEyZTQzNWY0MTc1ZWY1YTIzNjQ4NzU0ZDE1NDBhOTIxYTQ4NjZmZDE1MzMwNmZlYmU5NWZhZTUxNDQyNmM0ODFlOWMxOTY5ODg4MjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wCGzPeUsffq19v2Dg2loa5Sfv5uVpIw0wqPAnM5yWTni2iS6shDH_6J4BEIBoIyuTqBhGgAZg3dk4D_0C0QzGw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112033_92_2403_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.210Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlordUpCNElFWFB2SWhnR084VHBveXlxR1ZEeVU4SDdSUkxWc2R3a0Vic1BmN1NoQnl5L05uL3NxQnlzNVM5RDhGTUxUdGhiaGl4VnRkcWQzbEJIYUZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxOV8xMDI2MTFfNzlfMjQyYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTMwZDBlYTZiOGRhNGNkN2FjNmE5NjY3ZDgzN2VmMDg1OGFhNGQyMjZjOWY2OTc1NjJjYmNmMmIzYjY3ZmVkOTIyNjRmYmI0MjhlZmEyYTQ4Nzg2YzAwNzEzNWI0ZGExNjgwZDlkN2M3Mjk4YmRiMTc0NmU2Y2E2ZTc5NmQwYWIxZGE5ZjA3MjY3NmI4Y2ZkMGQzMzc2N2UxOThiY2Q3YzAxOTg2ZWY1MzE1NGUzOTU2OTdiNjVkYzc1OTg3MjlhYzliNjc5ZTRkZTUzMjBhOGExZmRmZTZmZGMzZDQxYmUxMTI3NGNkMzgyYmY1NWFjNDhkYjQ0YTMxNzI4YTViM2ZjYWM3ZjZiM2U5NzJkMTcwNDBkNDkzNjEyYzU1ZGVlMmY4ZWMzOTc1OTc0NDY3MTNhMzJkOGJmNGZhYjk5ZDJmMDBkNjBjMTAzYzI2NDEwZjE5MzEwNTUwYTY1NGQwNDg4Yjk1YjdlN2FmZjdkMGFlMWVjYmVmYmM5M2MzOGFjNDRhZGI5ZmVjYjA5MWE3MmY3MzQ0MmQyMDllYzhhYjE5MDQxZjhmMzUzMWQ0N2ExNjIwN2Q2OWZhOGIxMTg2Y2Y2OTA1MWU3MzI1ZmE3YzViZTljZjZhYjUwYmU3OGJlM2RjNWJmY2FkNjMyOWZkYmU0YTg0NzJkNDQ4YTIzYjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7K6t3K-RN95Y31YLAVzuZvuMqO0nhVzYCkmdQv6dDBR349FGoDanDCn5-9Nwlh-grlNw3icdZzKOSt3JaCXGAQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220219_102611_79_242b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.214Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpUbFZrWHIySnhBVnJIc1BhMGgrbWEweTB5b1JKY2trQmNYNkFMdGt3OFFuSXJxMVNIeXk5SEFYaVEzQzl2ZEtwaE03d2lDMFBubXNlN2xON3hCdzJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxOV8xMDI2MTFfNzlfMjQyYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWZkNDJlYzg2YjJiYzllNWYzNTdjNGU4NjYxZGQ0OGZlZDY0NjIzOTkyYWY2NWZmNTU1ODFmM2ZlZjExODY5MjUyNjU5MDhiYTQ5Y2Q4ZTM0N2JmN2UyMmY0NTFmZGJhMzc3ZGZmMDkyOGZjYjM2Mzg0NzRkMmM3YTVmODc1ZTRhMzI4MTAyMTBkNDBkMGYyM2M1NTFiMzUwNGI4ZmMwMWZiOGE4MjlmOTFjYTYwNWQxYjY0MmVkYWQ5ZGM3ODBlNWQ4YzVmYWQ3YTc5Y2IzOTA3N2ExOTg3MjExYjRmMjJhYTVlM2I5YmE3NzNjM2UxNThlOGE2NTE0MTkxMmFkMzQ4MzBmYmMwNGJlODUxOTUwZWQ0OTJlYTY3YjU5NmM3NTk1NzgwODEzMzQxZmRkOTY0NWY0MjA1ZjQ1N2Q0ZWYzNzM0ZmM4ZTA1YWViN2Y1OTUwYjExYTdiNDczNzcxZTYwYmFjMzMxM2RlMjliMDEyNjk1ZmE1ZTYwZWYwOTNiYjlhMTk5YjczNTdlMjFmNzY4MTU3NjFlMDdjOTU5ODdiZDg3NDJjY2VkZDU0ODdlYzk4Y2Y2NmFiYTlmYjM4MzgxNjRjNGQ4YjRmN2FiZjU3NjE4M2MzNTBkYmM0YWM3YjAzNjg2YjU1YjRhZDZiMThkZDg1NDE2M2I5N2ExNmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qSTY0YOdQwZ-M2p7MuwlRyVihSx_lWLSUM2INLA8UjNKrQguV9VH2UYGX_HR0ERaDQwxj3n8fewaubMT_23Ewg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220219_102611_79_242b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.217Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjdQZ0JISVUvV3RqMHIwWkc4enFtR1o1c2Y5VkJpSitQcU1rb00zQS9tRW0vYzJKMHVUWmVUN0tDT1NQT0hxM01nUHA3ZTVuT2ZwMmpVN0toeEJQOFZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxOV8xMDI2MTFfNzlfMjQyYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODg1ZjVlYzg1NWIwZDJjMjMwN2NlNDA1ZWU4ZWIwYzAzYmQ5MTE5Y2RhZjhjOTFkOGRiNzdiZDBjOTM2ZDIzYThjYWZlOGNmZTljMTMyZDkwYTg4YjMwYmVhN2I1NmEzYWRhNzRmZGYzMDRiZTQzNjg1YjZhMjdhMGZlOTkxMGVmNTgyMmFhNjBiOGIyYWIxZmE3OWRhYjg4NDljYWU0MTVhMzg4ZGQzNjhlMGQ0NmMxNTQ5MTM4MWIwZWU3YmI0ZDcwZjQ5NzQxMjA4ZmFhZWRiMmZhZTYwZWI1MDdhOWRkZDUyYjUxNzlkMmQwMGVhZDExMTc0ZjJjYTMyZjgwMGJjOGQ2ZTQyOTYzZTBkZjcyZjE2OTBjZWI4Mjg4YjAyYzE5Njk5YmU2M2Q1NzVjYjFjZTczOGI2N2I2OGUzMTQ1MDUwMDkwNGRlMjFkYWU0MWQ0NTkxNzZjNzcxYjFmZjNiMTYzMzc3YjA0YjgwMWYzMDMwNmFkODQxNjIzNjQxZjczOTBmMDMwNzkzNDcxZWJhMGQ2YTIyY2JhYjY1MmVmNGQ5OWZiNDVmYjY0NTA0NTExYjhjZDFjM2QzYTJkZjQ1OWVlMjU5MDZkZWZiMzk0MTAwNjE3Nzc5ODExM2M2YzAxN2VlZTNlZjJmMmI3YTdiOWE4NDdjZDE0MDVjOTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tU63BbuQ4Rzp93Ypzz2zNy3DcDrEg7ed1QDmAKriyBw4a8kj0k2l-IkMIvZxmnniwHJFU0lZqHLJTvHuZjCu7w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220219_102611_79_242b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.219Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im5wNVk3b1U5SDhhYmd1NXBLL3Y0WkRTdXRvNHF6YXFqdm1RV0JDZHZYQWs5MldGd0pHODk1eGV6UGVnek44ZGFNVGlJQ0duVFJFbGJPZXVSU3c2MDdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxOV8xMDI2MTFfNzlfMjQyYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzg3OWVhNWM3NzU4ZWJkZjEyMjBjNzdiMGVkMGM3MDkwZGU1OWU4MmYwNDU3MjFhMTZmYzg2ZTIyOTg2ZmNhOTMxMmNkNTQ3MGQyM2U4MDdhZDJhZmRjNGI1OThhNTgxYjMyM2VlMjA2NGRhOWM4MGEwZTBhNDFmMTNmNDA2ODcwMmU0MzdkMDUwYTkwNzE4NzE4OTA2MGJjNzY4NzRlMDU4YjIzNTk4OTM5ZTFiYmNhYjcyMTZmNTRlZDg4MzljZGQ5OWY1NGZmZTc3MjM1YzRiNmM1MTRlYTliNWNiZjRkYmJhNDhkYzBlMjIwMjVlOGY3YTg0NjRmY2RlMmQ1MzE0NDBiNDIxMWZkOTU1YzBjNGZjYTBlZTdkMThiNWVhMThkOGQyZjE2Yzk3OGNlMmU3YWM5ZDM4YzYxZGQ3NThmNWJlMmFhOWFhYzk2NmExMzVhZjlkMTViYTAyZWI5MThhYjhhMmRhNDc5MmQ2NmQ0NWIwMTFjZmU1Yjk5NjQyMTUyYzhlMmEzODk5M2JjYWY2MTI3OGU4ZDA3ZWIyNWQ2NDU2NWJlNDBhMDQzNTdkODI1ZDIzNWM5MzZkMzI1N2M1MDY5MGUzNGE0NTliY2RiOGEyMjY5YTcyZWI3YTE5ODgzYTI1NmI0MTE1MmRjY2RlYTEwMDIxNDJkMDc5MjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-XUwZERuaRs6k-AZaTJvDZToLRUwbGlsVOdF9-qoMT7MSssFTk0HufGUkxRSJDguGPGB8NsZzCFlPPVqu0cylw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220219_102611_79_242b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.222Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imx4WWdwbEhUc0N6V0V0T3I1R3BWVm1LOHlYM3ZGalhIb2xxRnJ3S3AvNHN6Rit3UU96enJLQ1NkaENsZ0hnUXMvRzhDbm1OT3B1ekV2Z0FMRS9kRHJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjlfNjNfMjRjNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGE5NjhmZGUwM2M4Y2UyZmYwZDA4MTYyM2ZmMDI0ZDc2NGE4MzNmMmQ5NGQ0MTg3NzQzMTg4MDdjYWEzYTM2YzZiNDNjN2MxYWY2OTRlZmYyNDU1OWRiNDFkMDQxYTQ2YTc4ODUxZDE1YTA5YzEyNzAzMDg2ODRlNGJmZTViZDgwZTRkOGQ4MWE4ZTI4NWM2ZTg3MzZjNTc0YTg3MDZmNjE2OGU3MWE0YzZkZDA0MzdjYTNjODBiNTM0ZmY5Y2Q2NGQ4YjhhNGYwMTRiMjU4YjdmYWYwZjU5MDVhMzNlYTljYTNkYjFmZDkwOWIyOGI5YzMyZWIwNzQ2ZDE3NzMwYmFjOTkxYzhiZmQxMWRhM2JiNDM2OTQwMmFmYTg2YTI1ZTViYzVhMGY1MDgwYzQyYTRkZmQ2Y2Y4OWFkYWE2NzY0MTRjOTlmYmEyYmYxZTE4ODAxNTg5OGQ4ZmU4ZDMwZTBkM2JlMjBhZjgxOTRjZjVhYjAxNGQ3ZWFkNDcyZTExYmNhNzgzNDE3YmQzZWZlYzFhNWEzOWE2NDI1ZjAxODUxOWJjNmNlMzY4NzZhM2FhMWY0ZjZhMzQ2Mzg0MTZiMjQyMGM2MDQyMmU0NDg2NDczMTJhMmMxYWM3NDk2YzE4M2Y2ZmE2ZTc0YWY0NzgwZmE1MjI4NGFiOGI2NWZjMjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._2QVokL5_OhfcC73ZNsEr2-5UwjT83FQ1po4U_93mg2irgpi5WfiWy7yzy6uUwsRi7nEOPiYljgvUh_bBubFag", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102929_63_24c4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.226Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkUrd25KZG81UmZiSk4ydVJTN21LY09nZXdyMnZxSm5VejlQeUozRUdNb3FOU1Y2ZWdLeHBjMGlDQUJnNEhtNXVSYWEvSzJlSW1WanVUVnhXWW9mRHVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjlfNjNfMjRjNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWIxYmI4NjFjZGIyNzQ5YmY0NWU3ZmEzZmIzMjQxYTIwMGFiODYyMjI4ZTA2MTg2ZjhkODQ4NmZlNDQwMDg5YTFkY2ZmNjI2MTJhMGUyZjRmMDliNTQ1ZmY3ZTNjMGZiMWUzYzIxNzNiNGIwN2I0OTk5OGQxZTMyMzA2MWY5NmE0YTY0MDA5Y2RhYTJmN2EyZTI3NWVmNzJhMzkwYmU2M2UxNmE4NzRlNjdkOTMwMTM4YjY3YzIzNWRjMzRmZjkxMGQyNDc4M2FjNmY0MjNlNGIyODZmNmQ3YTg2NDgzODM0MDc5NTBmY2NiN2IxOTc4NDEyM2Q0ZWI3ZTc0OTgxN2E2NDE3Yjc5YmI1MWQxMWUyMzMzZjM0ZDZmMzIxYjEzYjViNzM3ZTk5MzMyNmQ1MWY2OWI1M2E2MDZiYWM5MTIzMTllZjEyM2MzNmRiZGJlODIwYzU0Y2M0ZTU0N2VlZTg0MGQ3YzMzMDlkYzEwNjBjNWJmYTIwMTRmM2I4MzJlMmNmOGIwZDYzMTU5NzE3YWUwMWMxM2ZjMTk0NTdhNmRiMDQ4MTJkODIxMTdkYTZmNTMyYmI5MDJlZGQxZTcwNGIxYTlhOTE0MmNmODhiN2Q0ZTMyODI5Zjg5ZWQ1YWM3ZjBiNWU3MjdiM2ZhYjRlMDJlM2I2NzJhMTc1NTM5ZGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Oec87bPtbr-H_dexHCS6_V47nuM7RIoNLwMk0200GoMznP-4RTaHkzAMGP4SXD8T4bkBjtq5NEwOpYYp6rHcTQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102929_63_24c4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.228Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IklxL3AzYzN4U1FWbG9sTFE4VmQzODhpbnBFSGxsdUtlaElDR2ZBNmxoWE4xM3JhUmhMampIRUsrNDM5d3h3NmJTbnNZMDhjci9FZU5lZGpPc2tKbWJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjlfNjNfMjRjNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjYwZDNjNzJjOTZjYjYyY2RlNTlmMDYwMTM5ZTBkODA5NTA2MmNjZjRmMTk1YzU3OWZlZGU5YWQ5ZmZkMjIwY2U2MmU0MTFjYmZlYmUwZjVmNTU0MTFkNjM2OGFlN2UwZDFjOTQzYjVkODllM2Y1NDE1NTNlNmViMjI4MTk5OGQ0ZGY5NDdmMjE1MzIwZGEyZjk5YjRjNjNlOTdhNzY4ZDZlMzk2ZDM0NzE5NDYxOTU5MjY0YjYwNDI0M2I3NmM4YzJiNzcxZDdkYTJjZTMxOTIzZWEzZDUwMmE1NGIxZmY3ZjRjOWM0ZWVlYjI4OThiY2RhZjg5ZWI0N2FhYjAxYmMwZTI4MmM0ZGE5YWUzNjBlZTg3ZTdhMWZjMjFjNTljZWE1YWYwMmVhYzUyOGEzODkyZTY5M2RkYmE5Yzc1ODI4YjAzYTdkZmY3ODUyOTA0Nzg5M2NmOTFjNmRlMTAwMmNhNjZmYTNmYzhjODY4YzMzMmI2NzI2N2I5OWQ1N2Y1YmNmZWNiODYxZTQxNmNiZmI0Y2FkZTU1NWZlZmUxYzc3NTZhNTIwNzJiMjFkZjEwODIzODJlMDY1MGQ0NTkwZTVjMTJlZTBiMzRiNjExOWJkMzEwNjczYTU0MzRkNDQ3YjZkYzdkODhhOWFhZDcyYmNiYmQyNzY4NTk3NjlkODNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.glfj1weW09dJF6JWeTDmXLt8xrGqMDkDh9W1T3ZqhW8aIHoXOQ8ez4ogWoQKnvRNJ3dMq7F-GN4EYqEshnurBg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102929_63_24c4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.231Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkxmcGcyRVZsU2puZGVscDNLTk5LeXJubUJQYVBXTkRIYVFzWlFqUUl3ZHZBTXNFeUhQcWo3bllJOWpFQWlRT045L09va2tyeVBUUXZ2bVlYWWRpdEpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjlfNjNfMjRjNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTRhMmI4ZjUwZDczZTc0NThlYWYzYTExZWE0ZGVmZTk3ZDQwODA2MGRlNmI3OTJlM2YxYmI2NzBjYzAwMmYwMTU1MDUxODQxZTFjM2NjMTY5N2RjNTczYjA0NGUzNzQ3MzNjYTMzNTQ0YTYwZGI2NGU2OWE4NmQ2ZDY3M2FlMjJlYTFhN2RmZGM5NzBjYTJjZTE4MDg2YjU1OTFmMTI5OGY2OTY2OTAzNWViMjcwZDdjODc0ODE4ZThmMmUzMGMyMjcyNjNhM2NmNTkzMDJiMjdmOGE1MDI3NGI5ZTllYzZkMGM2OWE0NGMwYjVmMWI3Y2EwYjQyMTI0ZmE4YmYwNGRlZTY0ZjA0YTFkYzY4YTVjNTY2N2JlYzgzY2U3MjhhNGI1ZWI0MDc0NWYxYmQ4Mzk5M2VjYjc1OWUyOTIyZTViMzJjY2RiMWM2OTZiYTZlZGMzYmVjZmQ5MTU3MTc5ZGM4OTY0YmM0MTc2NzAyOTQ1MjgwM2Y3NDZhMjU1NzkwYTM3MDNkNmE3NmNhM2NkODMxNDFhMTIzNWMwOWRhNjBjMDdkODFkYzRlY2Q4MTBiZDliNDFhOWFkZTJlM2JmYmUxOTRmNzM1NDU3ZGIyMTA3NzNiNjdhYTNiYWUxYzBkMjE4NTQzMTQzNDMyYzc3OTVlNjg2ODZiNGQzNWIxZjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YAdiUBLAZyhgsTc66hM-eSdRxt10d2BntynY0IufUqep7oeKOCCzYELhVo_FVsGoOjnJfiBhTPAYswhAwQF2Ag", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102929_63_24c4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.234Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjVLYzZETTJ5elV0bGZUbmFFTTZuSktVSGd5NDZVbURjNzV4Ly9KcnhUSFB3K2VCZEE5YmpTSW5EOFdzTTVXYkpGUFNHVFZGYWJ2dnBkMkVUcHJZTnNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjFfMjdfMjQyN19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjlmYzBkYjA4NWJlNjJlMDczMzI2ZTI4ZWJkYTM4Mjg0NTE0ZGJkMDhiMTNkNmI1ZTQzMTczODc5ZThhYWRiYmI4MWVjOWM0OGFmNGNmMzRhOTQxNjc3ODAxZGI4M2FjNTYxMjNkYmI0NzVjMDc0ZDk0MmM0YTI2ZmFhNGU5NTRiMGRlYTM0MjcxZjFiY2RkNzQwY2I5MTUyN2MwMGM5NWIyOGJmNzM1OTMzNmM4NTZiZDYzN2MyYmY3ZTA0NmViNmUxOWZiYjZmYThkYWQ5ZWRjZTU1YWZkMzk5ZDFiMzQ3MTljMTI2MGE4MjhlNjE4MTI3YjE0YWRjZDczZmU3NzY3OTIxMTM4ZTdjZmEzNDM2ZTM1YzA1NTk5MDM2M2MwMTMyMTk5ZTM1NjgxZmQ4OGVjMTA3MThlM2U5MDJkMzhmMjc1NTRkOTJiZDJlYTg4NjBhMGMwMDBlODJkYzAzMTUxNzBmZmU0MGRhZTYxNzg0YmJhMDM4MDliYzA0NzhhOTkxYWY3M2JhZWMwMzRmNTg3YzE2YTRjNzlhMjAxM2E0ZDZmZmJhNzE0YzY2N2U5MzQ3Y2QxOWQxOWI1YzhiNDY4ZGMwYzI2ODc0MmI3OTJmOGUwN2U0NTdkZjUzZWYxYzNlNWQ5OGQ5M2VlOGI5NmJjNTgwYjg1NDhiZjcyNDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zvpm0QIgl-wKozttzRSj7Ue0ygMsGOcw-_hPWBBUHXelZRb1wf_dIlXwkQMcYPMX9BBQn_C3cFQI-_U9BzQ3zA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102821_27_2427_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.238Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlMzMUZWQ05nRTBodXcxUlJnZkY4R1l4Y3JhZWNQaURrYWtHY3U4c2xmaGtRVnNnRG45TXVsUGY5bTYycFZHT1RVRmY1eElxKzZVWDVON0ZvUGZmRDFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjFfMjdfMjQyN18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDlhZTllOTk5MGE3MjI5ODZkODZlZjFiNTczZTgxOTEyNTI4NWFjM2JlMjcyZDRhYzJjZDdhNDU5YThjZDQzNzM5ZjdhYTk3ZjRmMGRkZTllNWMyMzNkNzBmNjZkZTQwMDk2M2MzNzI2NTY5NWJlODA5MWVlN2EyZjI0Zjk4ZWU5YTg0MmExMjMxNzdlNTI3YWI0MzA5OTJlNTFjZGI5YTQwOThmNWU3MGE1YmM2NjRjNDhkMTI4YmJlNTdiZjA5YjBlOGVhM2MxNmU4ZmQ3MDU0YmM5NTAxM2RlYmQyZjFhMDI1NjI4YTIxNTU2MjM4YmFmN2RkN2Y5YmVkNGNkNjVmOWUzMzE5NGM2Nzk5ZWI5MjYxZWRhM2FiYTBkNGM2NGYxZThhMGU0YjA5ZTNmMmZjZGFiYTc0ZDc4MzMzZGZhN2FjOTNkZjUzM2E0NGRkN2RkMGNiMzM0N2M1NGEwNWZmMmJkYzM2ZmM0MjljMTdmMGRkOTJkNTVjMjRhZTNmM2Y2NmYxY2I2MWI5NmY4ZGEyZjkyZjgzZmRmZjFkMGE2ZjIzYWNlNmI3MjE4YWNjZDQ5ODJmNDNiMjdlMGU2OTk0OGY5N2NjODVjNzQ4MTE3MTRlOWJkMzZiNWRiMWI5ZDkwNGU0MWE2Njc3MzE5ZDQ1YzliMDZlNjhlNjc4MWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IsxwBeGdQ83mAdDuPnp-UNhOqvaOhooSCeRqMjNKA1m7qqYjbBUXThLI9_ym_6dP1xKRAlfmAQ2mNsadrtt-uw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102821_27_2427_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.241Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjNUTXNPdk9YVDdHdWNWaHM5QjJBK1gwdTlGSStpMWdhbm10bS9UZEhXTDVIcHZiSHFuMVdWbzQzU3RsdXZSTTdFQW9Wb0VEMlRWNmFmbGVLNFpnMkVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjFfMjdfMjQyN18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODAyMjgyNjViNTQyYzI3ODRhZGQxY2MyN2E0NGVmZDIzOGVlM2ZkYTBjODZkZGZjZjlmMGMxNzRiMjZjNmIzMzhhYzY4M2JkOGFiYjljZWM1OGQ0ZTM2ZmZhN2Y2NTJhMDhlMWZmMWM1MmFhN2JiZTEyZWEzMDc4OWE0ZGExODIxNGZjNjJiNmRmYzY3OWYzZmIwZWJjMGRhYjZhYzI1MGRhMDljNmFkYTU2MWZmOWU5ZTBkYjg2N2FkYTVjZmJmMmNlOWU2OGFiZGE4MGFkY2NhMmJhY2U3NjAxMWEyNDg1MDE3ZWMzYmFiNzA4ZjU4YTEwZTg4N2Y2Njk0MDM0ZGQ5MzVhMjkzOWNiZWY1NjJmMTkxNDBkNGMwMzg3ZWQzNzRjMDc2MGRlMTkwMmVlMmI1MGY4Yzk2NGEwYTVkZTcxNmY0MjdhODQ2ZGRiZmM5MmFlYzkwYzBkZjhlNTZiMzUzYzg5MzdiN2JkMDM1YjA0YjM2YTcxY2ZiYTg3Yjg2YjllNGVkNjk5Y2JiMjg3YmFhYTEzYzkyNDdmZjU3MzY1OGVhNTRmZmVlN2RhOTdkYTZlZDZiZTQwMzhkMzZmNjk0NWE2Mzc5NjE4ODM1MmZkOWZlNWI5ZmVmNjE1NTE5OGU5MWQ3ZWFkMjU2NzRlODZjNThkYjU3YzMxZjZlY2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.krVWjA4qJ5LHdKkCYAzl3TrVN1HWvU2J1NtNtGmYvsefHCnepxOycgyn_df8NBFGQf7lY7IV5RFtTncoabqVcA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102821_27_2427_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.244Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InhMWnVwb3JFc0x2TW40NDd4NXlpMUkvUy9tRXZ4cFRvMmY2V3N1ZVhQcDNoK0tGbFNMeWxLUkdnaE95Y2xLTTJCY1NDa2VyZGdzTnhFYmhUQ1lvYVRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjFfMjdfMjQyN18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWQ1NGE5YzljYzUxYzAxMTA5ZDQ2Zjk4MmI0NzFlMjhkZjlkNTYwMjYxMTU1NWE3NTRiN2I5ZjkyNjE4ZTRlOTk0M2FlZmRmMzE1MTIwMTU3OGU3NzY3N2E2MDQxNGE5YzcwMDJhNzgyNjJmNWJlMDdjM2EyZWVmOTYyZjIzZWQzMGIxNTA1OTY2ZDljYjYwZTI5ZDk5MTc0MzVmYzcxYTA0NmRjMDAzZmM0ZmFiZmUwY2UyYjhjMGQ3ZWNiMWViN2U3YTNiYWIzOWMwZjA5OTBiYmQxZjQ4MDIzMTUwYWU1ZTI0YTBjYmU2NzVmZjEyYWMzMWM2MWJjMjY4NGI4ZWRlN2U1ZmM5MTJhYTZlYjI4N2M5YWE5MDBiYThiMjI5OTA1ODJjOWIzNTE4NWI2ZmQyZmFkZTMzYTk2ZjlkZjkwYWQ3MDEzMWM2NTFjNDRkOTYwYmNlNDE3MDRjNjIzZWFkOTU2M2RlNTUzYjFlYjcyYWU1YTQwNGM3ZjM1ZjJiYzA2ZTAyOGIwMjFlZjkzMzlkMGQ1MGZhZTBmZjlmZTk3MWFlZDYyMWRlNTNhZWU4YjFhZTU3MTEyYzNiNzNjOTFmYzAyMTVmMzMwYzM2ZmQ0YjJiNjAxOWM0MjU5MTJjNzRlNDBmNTk2NTA4N2Q1MzlhNTQyZmIxN2VlOWY4ZjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bpzBxoncEw8ygIPPY_w1mTDTPDXWmZJSpn7YmrANqKxu8oPhzFmFULzTTya27CF9rWoTGM5ELJIlVmbd4xLiMQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102821_27_2427_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.247Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik9OTmhXemlwTHpjNUtmL0xLbDRMemp5YmNZL1EydjRFNU4rNGo5cjZoeDRaanlqT2ZncDB0NFVOSUgzTW9qaVdSZTNaN3RBVG5ibWxpYTg0bkM5bjh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzlfODVfMjQ4ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTU0MTMyYmEwMmNkNDUzNmQ5ZTEwYTQxNDBiYzdlZGFiMDE0NWRiYWE5MjQyYjRjNDJkNDViYWI3MTk1OWZlMmYzYmI1MDcxNDdkMDU0MDAzNjViNzE0NDQ3MTUwYmNmNjcyY2YwYzM2YjhlNDNkNzdmYjljYTA5ZTU0YzkxMDQ2M2ExYmY0MjFlYTM0M2JjOWQ5NWEzMzA1MzljMTJkZTZiYTA3ZDYwNjFjODFmMTJkZTc2ZDYyNjMwOTA2NDQ1ZDIzNzczNTAzMDA1MTRjYTdhOWFhMjhjOTEwMWQ1ZWI4YzY3MDQxM2UyOGM4MmEzNzdkNmM5ZjdhMjg4NGE0MGRmZGExOTkxY2NhZjI2MTZmMjA3MjJiOWE2NjRlOGIzMjMwNGE1NDY1YjU4ZTZlOTc2MDU3NGNiNmFlYjc1Mzk2MDg5YjZiYzA5ZGMxZmQ0MzY3NDE3ODdiNmQzN2I3ZWY4ZDczOGZmM2I2MDQ0NTgzZjBlNmVmM2NmZDU0NjA0MzZkMmExMWVhZDI4Y2E3MWU5Mjc0ZGUyNmM1YzAwY2VjYTQ5YTBjN2Q5OWU4ZTA0YmQxNTM4M2M2ZDQ5ZmVhYzNhMTY1YzkxYWNlMThmZGYxMjdkYjViMDQyMzU5ZGMwOGY4NzFlMjgyM2FjMDk0ZThiNGJiOWQ0MTRiZDVhZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.R6pUbB509xGFDtKoCVDbi2CbhmwN6RYwCz8fvveGCddFjC_SQBZqYr-H8wiSEzy-VOBCI7td_5BxHgc8VckGhQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105839_85_248e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.250Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjgwNHM5aXlEK0czL0tqVEMvZlRXcEkvZ2YrRU5aM3oxMUFCSi9WdkptaTF5VmNXTmZzSXVrbjU4YVN3OVdTSW54RDlhOWgwQzIvREYvTTNDVEkwaWp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzlfODVfMjQ4ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjljNzEwZTcyYWEyNTk3ZTFjZjcxZGJlODJlMjY1MzI2YTg5Zjg1Y2Y1Mjk2ZTYwZTYxMjE4ZWJlZTA2YWU1Y2FjMDQ3MGUyNjgwMTkxYTUwY2M3NzZmM2NmYzRmYTdmZWMwZDZhMmY1ZDZhYTFjNjI0YmJjODJhNTM0YzVhNWRiZTY1NDEyMGQ0ZWM4MzcyNzVkNjE5M2UyMThkYmJjNTRhYTRmN2UzOGMwODgzOTA1ZTNjN2YyZWMxZTc4MzcwMzI1MGU2NmU2YjBlMDkyNWZjNDYxNWQ1OGNjYTRiYjg5Y2QzYmViMTU5ZWM1NzEyYzgwOGUzZjM5NGY0NGFjODc2NGNmOTI2MDQ2ZmQzYWRmYTYyYjk5MGFmMTY1MWU4MmIwNWE2MTc1NTMzOWE2MDBiNDA1MzY2ZDRhZjVkZmYxNGU1MzExZTM1ZjU4NzFjZTQ3ZWI5ZmM4OTYxZDdjZmY4MDc2MDhlNjMzOTQ2OWY0MzA2MGZmYzllZjg1M2E1YjgxZjUzNjUzMzBkY2VhNDAyYTkyYTE1MzE0NWQwNjQyZDRiNzZkODljMjc2NzVlZTMxYTI3Y2YwYmQ2NTQwZjk5ZjgzNzVjZTMxMGQ3ZWJkYTE3MzZkMDI4NWYzMGI4ODIzYTQ5MjQzMmZlNTYwNzdhOTNlMzUzMzkyM2IwYzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qeeagk38BZrRh9Nkw-XZDVSl5oEldVN2v7LA9zfBSfNfsEoNAooki5V59t1p9tUoVrsl52MrMn2MWWs1ewUJ8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105839_85_248e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.254Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpFMDhHM1VURlNPUTlYcDdwZm5VTDZMbnNRaHBNb3NiODFrcllUcnFkakF2WjFxN21nN0V4WUZQOEpieWROSEJha1lYaHB2Z1A1WlNsN0pCZzNiY013PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzlfODVfMjQ4ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGI3YmRiOTYwNDIwMGRhMGE2YTUzZjA3NDlhN2YwNDBiOGI0Y2EzNTllMTA3NDc3M2UwNWY2ZjZjZWI4MTEwMzQwNDlkZDNjYTJmOWI1YTEzNGFkMTM4ZWYzMWI5Yzg0MTIzY2ExYzM1Y2VjMTZlYzRjYWJlMDM3OGIxMGQ2ODhiODY5ZDI3ZjE5OGJhNmE2NTljMGRkZDVlNzliYWZkNzI0OWNhOWZmYjk4OWYwOTEyNzk2NmUyODdmZGU3Zjc5MWFhNmI2YjlmZWVkZWI0Y2Y0NTdjM2YxMjkxYjFkNjNmNmI4MjY0YzIxMWJiMmE4MGQ2M2Y4OTI1ZjFmMjM2MDhhNDVkOWViYmZjNTQ0YzhiZWE3MDUyYTQ3YzMyODkzNDJkMWE2ODdmMTBiZGY2OGU4YWMwMzIxZTljNjQ3MmMyMTNiYTQxYzQ2ODFkMGRlZTMwYTE2MjkwYjAzOTA3NTk3ZDVhMGE3OTU2N2RlZDA5ZGI3MjlmNDkxMDE1ODRmOGE1NTE1YjhmN2U3ODk1NDgyNzNmYWY0ZTA2OTY2NjVlYmQzYTY3OWUwNTA5YmZkZGVhOGY2Zjk0N2NjODJjZWZjNzY0ZmEwMGNiYjg3Yjk4YTI1ZTdjOTg5ZjUxNzhhMGU1ODAxMDYxNTA1MmMwYTdkZDdmMWFhN2VmMjJmYjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qi9Zy_IfBDG4olSIXe6lPOXWcL15nDrdBXV99VMiQ7Fpqa4CTEI8Gr4HXKx0p5OgCFPvT31vn0G3He0A3gfwkg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105839_85_248e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.257Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik1lN1pxRUFPYlR2RkNHQmFFeHhQKzZ0ODNzaHo2eFJON0N4d3VVNlFUVVY5V2F3MTB6L3RZYkNhL2FhYnpmZmdPbUpBVU10VEVtUzhxY0h2TVNYUDJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMwNV8xMDU4MzlfODVfMjQ4ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzBhNTllNGU0MmM5YmIyMzM0MTNjMDE4NjgyNTU2ZWY0MmYwZGEyNjVjMjgzMDIxNzA1MDE3N2MwMDJkMTIzZGFhMTg5MTU4NTNkODNiYTFlY2YxMzFiODRhODcxMWVhNWVmZGMxM2JmYTM4YTlhYzliMGYyMGJiYjRmOTU3NGZmZWIyZjg0YmY4ZTFkM2I0ZDUzNWU1N2IzM2I4MGRkYWM0NzYxODc2ZDVlOTlmNDdmYjkzN2NjMTczYzkwYjMzMWJmZDEzNjU5ZDAzZTZkZWRiMDk1ZGU4MGZmNDRhMzViNjBhZjE5YzU5MDllYzVkYThiZjU3NzE0NzQ1ZDM3ODdhMWY1YTYxODdjNDk4NGJkNDYzM2M4NGIyNmY2MzI3MTYxOWE0NmJkNjg4OTIzMmNmODI2OWJmOWUxM2U0ZTJiN2ZiNGE2Yjg3ZjMyMzJiZTg2NzdhYWQxN2MyNGUwNzc2Y2ZkZGU0ZGY5ZmM5OGQ0NmEwZTJhYjgwNzk5NTQxMjEyYjI3MzA5ZWFjZDY4MmRjMmQ2OTUwMzkwNjM5ZjYzODU2YTJjMjdhZjU5YmY2ZDhkMWRmM2RmMDI4NzZmMmUyOTljZDQzNmFmOTVmOTk5ZmQyNzYwNWE2NzM1N2ZmY2U2OGVmMmRlMzMzZjg0NDUyOTkyNWMxNDk0MzZjZTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PZgS2aj1qs36vbT1VoO1HVN2KIlNvHyFkA7yoq3kii2jShNLvGyQjQwJ0DpL3SVXTFSjCNnNt5O-RAPt_ySVeg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220305_105839_85_248e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.260Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImdPYjhWck1DT1RSUklsb3dJMG11N1hFNEFMOStTL3ZFUmFmYXJpaXhaRkFwM3ozZXhlb2hBc2FlOUNwVjJtZjZuNXkrZFBzcEJUcjQzK3FsbjVjcmxnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTdfMDFfMjI3OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQ5NWZhZTViMDc1N2QzMDM3YTFhYzVhN2NmYzlkOGE0OWFiMTg1OWJkODAwMGI3ZWQwN2UxMzRjOGM4MzNkNGZkNzkyMDg4ZDJmMDk1ZmU1OWRjOGM4ZmJjZDkxMDQxYzMxMjhkZDMwMmY2ZTdhZTEwZDYwMTg3YjE2ZmY0ODhjM2MwODc0MmNiMzJlMDI0N2NlZjljYWJmMzc0MDY3ODc3N2VkN2FmMmE1MGMyZTcxNDVhZmI0N2RlYjI2YzJlMDA5NTc4YmJhM2NhMDhkYzVmZTM0ZjFlOTdkZjRlNzkzMGMzNWFiYmMwMWZlYzUwZjE4ODJiOGU3MDY4YmVkYmVhNWVkYmMwYjQ2YTYwNzY4NGRiNDI5OTVlMDBhNDNkM2ZlOTMyODgwMGQyNDE3OTgyY2JkYjRjYWU4ZjE2MGExNjI2NjA1Mjg2YjVhOThjZTczYmY3MTk0ZTBlMjhhNWNhYTU5MGMwN2U0YWY0ZDljYzZhZmUwMjcwZDZjM2NmNjdjYThhY2QyMWE5NDNjNTE2ZTM0YzMxZDM2YWVjOWJhOWQ2Mjc3ZDJlZjUxMThiMGQ0YWQ1ODIzYmM3ZTllNzFkYjI2ODkwZWQ0MGY5NDYyMjQyODY3N2I3MDNjN2M0OTdmNzcxYjM3NTUwOWEwZDFhYjE1Y2Q3NWQ5ODZlZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ohQ6TZp9lYXrMAXFT60zvg-4N_2uk2UWjxW2O9PLZfsMTCOyFvOSOmvVfaDN3uSAgb6sD_o85p_XlIhcQcK2Pw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104257_01_2278_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.263Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im9Tb2pwNDVMbHRnRTVFNlJKVTdVME9SV0NpZWM0RE4rN3EwNWdUZ2JvN3BjZWF1NnhKOXkrNGRzVjY0VUVpYTh1RzhzelowbktCQU5hNG5wbjZUUXFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTdfMDFfMjI3OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmE5YjE1ODgwZTYyMTZmYWYxZTFiNjI3MjgyZGUzMWQ4Njc0MTMwYjQ2ZGZkZDdmMjZiYWI2ODY2YjBkMThmOGU5ODFmMWE4OTFiOWYwODdmZmQwOWRkMzFlY2ViMGU4YmQ3MGU4MGM3ZGJlOTY4NjU4MWVjY2RhYjQ1Y2IwYzhjNzM1YzRmZDhkYmEwMjVjNzVmZTY3MDA1YjdiOTc1ZDMwMDdkYzljYjkzZWVlYjU2MjFmOGQ2NTg4N2QwZmQyZTc2NTIyZGU3MmUxOGY3Y2FhZTdmZmYyZDk4ZDNlYzI2MDEwOWUxOWM3OTA4ZjU3Y2FlOTg3NzU5NzE1MTI2YTA2YjA2MzkxYmRjNTFjNTcxYTE5Y2Q0Mzk5YjE2NGQzNWM4NjI2OGJmNGU0NDNiYjU2YjhiNmFiMmQwMjVlOTRkMzYwYzhhM2YzMzVjZjBhOTIwMGJjMTI0ODhlZGY0Mjk2MzI3OTliZDU5NjA1MWE5YTBjZWM1N2UyN2FiMGE1ODc5ZTAxNWI5NWNkYTllNDQwMzFhODRhOGE1YTE3MDkwNmQ3OGJjYzI3Nzk0ZWFiNDkxMTEzMDc0MzQ0NjQ0NmQ1NTk1MDY1MDkwNzU5ZjE4ZjZlMzlmYmMzZGI4MWVjMjY1ZGI0OWU3YjYxZTllODExMTlhYmI4YmQzMTg1MTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MOlcLCMiQK1hPcIB1vv6ot6Dk35gkrpk0bvLo0NkIs98Cw3nq0-XES0BFSLz-iwxkHWhNDr4pfBbaUN7giT_KA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104257_01_2278_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.266Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ino0STJaeEo2NXI0UjVXMWk3QXYvb0szdFZKWVRBZk5QNEMxVldoREI3MUMzTmFTdWpCKzEvbWd4Z0JGM3VvMVJLM2NUOW5ueEZtbk5pT01McGUzSGhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTdfMDFfMjI3OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTZjNDI4ZWZkMGE5MjA2ZWQ0MjcwNjNjOWY3ZjAzNWQxMGJiYzA2M2E5MmI2MmNlYWVmNDY1MDIzODQ5NTQyMDgzMWE4NTQ0YzEwNzBjZGVlYzI5NmEyOTBiMTM3NzkyZmFhMzNlNDg2NmI1Yzc3YTc2ZjUwYzU3MTgyODIwZThmMWRkNDQ2OTUxNDhkODA3NmZkZmEwZGRmYzY4YTkyNTQ0NThkZTgxMWM3YzYwYTc0YzRkZDk0ZjM1ZTEzMWMxNjQ5ZWM5Y2Q0NjQxYzljMjM4Nzc2YzFhYjVjZGY0ZjQ3NmQ3MThjMGVkMGM3NTYwYjNkN2YwMGE3YTE1NDk3OGUyZDkxNzQwOWMzNzAyOGUxNDA5ZmMyNmZiOTMzMDJmOWMxZWE1NGIwNWIzMzJhZmMxYmI2NTljOGYwYzE3MTlmOGViMzliOTcyM2Q3MmY3ZWM1NmEyZDczMDM1MDk0MDk3NGNjZGZhOWQzZGM0MWNmOTAxZjE1ZTBkNDM0NGY3NTI2MjJiMDc5MjY5MjcwNzkwN2I3YTg5MzU2NzVjOTQxM2I1NzhhMjE3NWNhMzM2M2YzOGRkY2MyYzg1ZTQzMzIzODU5NTM5ZTkwYjljNDlhNjM2MWVmZGVkNzExMjE4MDRhODQyNTFiNTUwZTI0MzI4YTg0OGI4OGIwZGQ5MjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xm2BX7gpA1mnax0QYtQE_NjA8srzRInuTXSOSSE_0Xatt5csSGP9bCx8jvSJp-3yb6w9eRdFE1BqdqfSX2fVxQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104257_01_2278_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.268Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlZ4MGxiTDB5NWdrZmt2L2d6enpCdVpFclAwSW9mR3hFYXFqU1JwamhLTElCZmJ3NUd3UjF4OXYxMGN3K096Nmx0cm9Wa25UZTN6bFVrYnk5R294eERnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTEyNV8xMDQyNTdfMDFfMjI3OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2E3MTNkNDYyYzg4NzUxYWQwNTZlZTA2OTc1Yjg1OTMwNzZiOTk4YjU4N2IxZjlhYjg5OTZmZjU1YTJjOGRhZjhiMTIzMzIyNWFiZjdlM2NkN2NiNTk1MDliZjA5ZmMyYTYyMDdiMDg0MjBiMzYxYzJhZTVmZjU4YTZmNzVhZWU5YWEzNGM2NTljYmU2ZmZlYmI5YTRjMTIyNmJlY2NiMDY1NjE0NzA0MTg2NTE2NmI1N2E1ZWVmYTVjNDA1N2RjOWMzY2RhNzNjZTJmNmNlMTM0Y2M1NTJkM2Q3NGIwN2QwZGQ3OGQ2Zjk2NmI0OGUzM2E4ZDI4NmEyYWU5YmMyOThlOGQyN2YxYzk0OTk4ZjAyZDFkMTA5ZWUyMzE4ZjYxMWYwOTMxODZmZTRhZTQ0OTdkMDc5YzU3MWU5ZGVkOWJkNGZjYmIzNDA3ZGU4NWMxNmNlNWM4N2E3ZTE3YmIxZmQzODFlNjdlYTZhZTRmMzMyNzYyYjFkZTlmOWYzMDA1MDYyMmRkZGFmZGM0YTNmZTAwZWQzMzFlNjkzMmVmMDY5ZDYyNzc2NTI3NjkxZDViZmNlY2FkYjdlMTdhZWI3MzIxOWM4YjVkNmY1MDY3NjY2Zjc0OTJiZmJiOWIyZGM1MjJlZmM1NTBlMjIwYTVhZTY2YjRjYjYwNmZkZjg3MThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.X-thLUwqy9PrKWMVM8lpP-nGySXrnnj518_A6-RSz5vAWP9SqqSg5pKxE339iFOp8ifY9uDLJcMuY8x9KYdYfw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201125_104257_01_2278_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.271Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFjdi8renNReXU3WlpaM1dlWWtJdWlvWTZWY1Z1V1JQd05KaGdpeDFzNzBseXExWnRyaDQvNE96VjJnZ3U0ZHF4RnJmR01PdjNJZlM0cEtyOWR5UGV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNF8xMDI4NDNfNDlfMjQ1NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTNkZjQzOTI4MTkwMTQyOTEyOWU2MTFhYzliMTY2YmM2ZGE1OTM5Mzg0OGIwZjY0ZTM3OWEwMmI4NTI3MjY0ZDkwMjNiMmQxNzk1MzMxNWE0YWY2YWU2NGQ3Yzg3ZDljODk5NjlmNWRhNjJhMGFlMmEzYmEwYjExZDFlYTU5MDM1ZTMwOTRjNDIyZjM0OTcxY2FhMDUxNjAzOWJkMjhhMzI3OWVlMzYxNzFkYTE4NjNhY2U1NWI3OTA3ZDk1NWNkMGRhZTRmOTQ3ZDdlN2VhNDc0ZTJiY2E0ZDY5OWYzOThlMmI3YzU0ZmViZGVjNTFlNDJmNjgwZDJmNDBhOWM2YWI1ZmE1ZjNjNTg1NDE5N2ViY2I0NDk4NDRhZWY4MDM4YzEzMDlhOTlkMTZjYzE3YWZiNjcyNmFhMTVlZmU1MTFlZGZkODQ5NDkyNDUwZTczYmRhZWMwNmFhM2VkMzU3MTlhNDkyMjhkMmE0NTdkYWNhNWI5ZmUxOTEwMzdlNDNkN2ZjYjI2MmYzNjM0NDg4NzllOTRmMzBhM2I4YmZkMDE5ZDQ0MmE5NWU0N2Q1YzZiNzAxNjk5NmMwNWI5MGUwZTY4MGU1OGY0ZGVmZmY5MzVkZjY1NzYwNjYzYTE0ZWQ0N2Q5ZGQ1MzRkZTBlZGQ5OGY5ZTU0ZTA2MDMxNTYxMzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.kUTZb9XxabGZbijM78QKikRrW9Goit51xReim6YAp-mgc5mjfdiVJv5Au04iO7aw8_BZcIatiA1FxBRcXjWUxw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230824_102843_49_2455_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.274Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlZHVUpkbHpRVEt6RzMraHJZVSszeGZHblFUbi9HdHI2NUdxalltSlE1RDNjei9UdjVvMkpsMEpsL1V3S0FDejNFSjhGR0wwTmI2QWd4cmh1SHZzdTRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNF8xMDI4NDNfNDlfMjQ1NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTdlMzkxNjVmNGNiY2M2NWY1M2FhYzRkYjg2N2JlZTNkNDAxM2NjM2EyMjEyMzI3NGE4YmJlMmE5ZDFlODQ3NjUzODdiMGVkYTcxNTNjOGUzZmQ4OGM5YzYzMzUyNDE0ZDIyNDdkZWMxN2M4MmQ4MjRiMTBlODkwZGI3NDA3ZmE2MmRmMjRiZWM0NTE5OTcxYTlmZWVhMGVmMDIwYjRjYzk0ZTFiYzU0NTg5ZmZkNDE5YWNlODk3YzQ4M2I4NmY0MDcxYjg2OTIyMzAyZDg0MTYwN2E3NzM4MzYwMjdkM2YxOGFjNDc1MTcxYTI3NmIyMzYzZjY3ZGU4NzQ4Y2FlOTcyN2IzNmNlNmE3YmRlZDA1OGI4ZGY4ZWQ3YWMwODQwZjI2N2ViZDdiNzE5OTRiNjcwZjZhMThlMDZjYWMwNDA1NmJkOWNlMGJiNzk2Y2Q2ODc2MmM0MzdjZGFiYzVhYTMwZWYyNTE3ZDgxZDM3NDVmMTc2YzhkNjI3YzEyNmVmMDk0ZmJjMTU2NmQ1Y2JlNDUyODIwZTFkMzcxZWI3MjQ5OGZjYzkwYTdhNmE5NjMwZDNlMTAyZWM3OTUyM2RjZjMzOTY4ZDQyZTc3NzQxNmZjNjE2YTYyNjVlZGJjZWJlMTljYzI5ODliNzE1NGVhOWY2NjM5MTkxMWYwODlkNGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.af_6SxhPPuDT19d79OBjA62xr99Tw763OFmwGnvcwNPmxgyTm5bNurqyNM3_Fr1jJvYxhTm25RlcKu46WUhrNg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230824_102843_49_2455_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.276Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik8zLzVtcHVsRGI2cnFoaUl5enRMVXBnVlNSNmtJSUZxcUdZZVppRnhvdktJVGJDUXdGZ0x3dHpmQkJ3c01OZjZucUNBSDkxVU9JZm9wbHdKSXErZkV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNF8xMDI4NDNfNDlfMjQ1NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTg3YzFkYjI2Y2UxZTAyZDQxMTE1ZjgzZTIyYjYxZTEwMGUxYTYzMzIwODFhZDIwYjQyYTg5ZDk3NjViOGI0NjYzNDU2ZjdiMmI5NmE4NjMzYmRmMGZjMjdhZjI5ZjI0MTkzMTJkNzI1YzVmNjRjZDM3MGVmZDRmZTNmYWE1MzEyMGVhNTk1ZjFlMjkyYmZkODM5OTliMDM5YzE2MjA5MDc0MDM3MzJhNzU1MGM2MGY1ZGU3MjQwNmJjZWNlNWJlYjhhYTYzZDFkODMzYjJiNGIyODBjOTQzNWFjNjA1ZDVkMDViYzhkODQzYWNiNGMwMWZjNmIxZTY0ZDg3MTVlYjAyZmU2NzExYjgyYTdmNjE2YThkMDYyZGI0MzNmYTQ0YWU4MDFhYzQwMDllYzVlNzZkMDUxMjU0ZWMxYzQ1OGM3ZmFmMDk0YjkxYWY0ZGY2ZmM3NTFlMmMzYjFlNGMzODJhMjBmNTY4YjMyODFmMThkMTg4OTFhOWFmZTA0NDVlNDU0NWZiMjA3OGZhMjJkZjVmMTVjNDI0NjFmMDVkYmNhYjA0OTM4Y2UwY2ZiMjMxMjczYjhjYjk3ODBiMDAxOTJmMTk4ZmVlNDU3ODBjNTAyYzI5ZTMxZWQ5YzdmMDBmODcyNjNiNzNiODVlY2NmZTM2ZDg4YzE3MTdkYTQ1OGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ykydDB9BwrcT2fwMobOYqKyEmCQrt8ZU77k_lP-Amp60jNSPmR3gPoBLTCoLgNqdUio_GFamUvKP3a_Yl-aW6A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230824_102843_49_2455_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.279Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkxMdEYxWjJwMjdEYm5qVHd1SmxvMyszZXltNHhpQ2hCTm44ei9BblY5YVVUTmNydDcybzRZQlVaM2JESlI1QnFpZEtFWWVzclVoQ3V1K01KeXRpR0xRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNF8xMDI4NDNfNDlfMjQ1NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjU2M2Y0MjVmMjc2OTc2M2ZmYzg2Yzg1ZDRjOGZlOTVjMzk1ZDgzMTMxNmQ5YzljMjVlY2UzNDMyYjZlMDM0MGI1ZmIxOTE2NzMzNDE2MmM3YTgzMmE5YzI1MmZiNWJkNWFmYTM5MGExMmVjYTNjYzEwODZhYWMzYmIyNDZlZmY2YmNkYjEzYjI4MWEyZTg2YTBlYTM4ZjIyZGYwNGM0NjBlMzM3ZTMyNjQzMmVkOWMxNjJkODAxN2U4ODc5MzNiMTg5MGMyNzQxNjRiYzI4NjFiNjRlYzYwODAxMzBjYTRjZTZjYWFhNjdhOWFjYzE1NDFkNGJmMGYyNzM0MmI2NDBiYTI2NzkxMzM5ZmRjOWRkZjBmZmY1ZmViOWU5ZDQzMmViMDFmZTNmZGE2ZDQzZTBhMmNhNWJkNTIwMjM1MDRlOTNiZTQxNzEyZDNhMDExNGRjOTZkMDcwNzAxNGVmYjllYzA2ZjU3ZjRhMmY0OWNmN2E2YjYxMTM2YzMwNmM4NmI3Nzg4OWQ5MmEyMWYxNTQ1MGMzMDgxY2NmZjFkYjEzY2NlNmExMjExMjNlMzZiNDNjNjE4MTZkYjY0ZGFkYjAyNTNkMmQ3NjRjODhhOWE2YTlhMzYzYzdjNjQxZTViMTZhOTJjOTRmOTQ0OGQ5YTJkZTc4ZWI2OGQ2ZjkxMGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zrpkCve3qNy-iAX_-wksP9wWyKBfJtG6UETHBLfA1N6KFLb6LaKcJzTFg4bblT6QrTxhsiq3gtdaS92w752-vA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230824_102843_49_2455_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.281Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjhGZGJKS3lmenNIcVVRNkl2aDVlRUI0U0xDWmg0ZHR2Yjk1cUMvNWsyOEsvcVgyMjhUbDhOZ3J4S1Zpck9ncjdNb3RKRWQybmpyM1QycUNtczJGUnZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA0NTlfNTBfMjQ3NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODFlNDZjMWZlZTAyZDU2NzRkNjczNzM3YjZkN2QyMzQ0YzUxZDQyNzBhNTU3MDA1NTY3NzQxOWE2NjZhYzJhOTEzZjk3ZTM3Yjc2MDdiMDk5NDZmOGYyZjJmMzJlMTg4MzNiMTZlZDlmOGNiZDdjNjA0YjA4NDEyNDdjNDQ1MWQ5MmNhODMzYThjY2JjMDQxNjVhYmU2YmNiODc3MzE0MDI3NzlkM2I1NTUyYzk5YmE2NGZlNjFiMjJkZTA0YzZiZTA2ODIwMzZjYjQwN2EyMzhhNzI1NmZkNzQ4N2VlYzBkMDBkODgyNTA5ZDUyMjNhZmRmMTc5NWRjOWU2YmM5YzMwYWE1NWQ4MWJjMWQ2MjM5MjUzNThkNjFkMmZjYmYyMzZjNzc3Mzg2ZmMzNmNjZWUyOWM5ZWEzMGQ1NWFhYzA3NjM2N2YzYzdjY2QyYWU5MDY2YmI3OGExNTcyMzExNDQ1ZmNlN2FkNTFjODA2NjYzNjEyZmVkNWU3ZDRkYmM4NDNmNDVhZGQwNzk5NGIzZTNhZWJhZTYxNmRiZGZkYWQxYmQ5ZjJjYjc3ODQzZTFjZjJhNTFhZjQ4YjFjOTQxYjM2YzYxMDRiMWM2NDVjMWU4ODM1MjBkNjFjYTRjYTlkZTg0MmViNjExNTI2MzVkYTcyYzFjN2RmODZhYTNjN2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SHt5tUAzc9Yi77JFDx7mCYwLvndWU2Gm0A07o4ELdRGPCihnwWHWZT-MGwFwAVVJ6_ytbYNg8Unjot-8l31vjA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110459_50_2475_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.284Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjN3TUh3Umc1aDVKaEVtSDNkUTlyM3libzJwYVJDYUltUTF5ZmdNRC9ZQjRmSllRZGhuSkJmSHFPVW82cS9DZHlkUHJEeXl5M3N5Zi9OK0k5YXhsVEhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA0NTlfNTBfMjQ3NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2IwZWU5MjMxYTU0Y2VmNDBkNDE1ZmI5MjE5ZTc3M2NmMmZhMmJjNWQ4YjcyZTFlMjJkMDYxN2JiYzU0YjZjMTBmNzhkNzBhYWEyMzNkZWIzOWU2NDcyZWI2OGFiMzU3Mjg3ZDMzMmRhYjRiMzdmMmNiNTg1YTY1NGIwMjY2NTIxNzhlYTBmM2Q5MDRlOTE3OTI3NmQ0MWNiYTc4NWU1NzZmNDM0M2FjNWU0MTM5YjEyZmQ5MGM2YjM3YTUwNjkyNDVmYjVlNTg1OWEwMzNhY2IyNjA0NmY5MTA4MTU3ZTU2N2MxNjNjZTA1ZmM0Y2NlOGQ4NGQ3MmQ2MzBiOTNmNzFmN2YxN2I4MTM0MzY2NDBmNTgwZTg4NzdkZDJlNjdhZDBjNGQ5ODQ2YmM1NWQwNTJmYWQyMWNiZDIxN2FmMmExYzI5YmI3ZWRhMThjOTM4ODM0NjJiNWI2OWQ1Y2I1ZTVlYjc4MzJlOTgzODI3N2U1NzQ5ZDJmYjg3MGU2Yzc0N2NmZTIzOTI5NGRmNzI1ZWY3YjJkMzVlNTQ3MDFiM2I1OWRiMjRmODFjNjk2Yjc4MmYxMzA1MzgzMDA3NWJiZWIyZmNlODExZjIyZGIyODQ5Yzk1OGI2N2U2MmViNzY1ZjRkNjY3MGMxZThmMGJiNWJmZjg1OTc4NDY3MDVhZmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.z6NphMJ86H_d4720AAQ0mnD46Qv37Jm-M701zuX_7kI3zhk6BsWKIC1QSOFQPiDoYoorOQ_POjIkKAc2LVjIKg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110459_50_2475_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.287Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlNFUHYwT29SZEtRTXZFNVlJY1h4NjhIQjlMZ1pqZWs2V25PL3V2Y1M4dGNMdUFSaWRLSzNuUEQwL3hBdDlPSFl2Tm5UbkxrZGsvT05QVXlmWEpFcFJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA0NTlfNTBfMjQ3NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDM3NTRkNThiYjFlNmQyNmVkZDc5NDhjM2RhYjA0NWU5NWNjODZhODYwYzJkYzMzYjI0NTY5OTYzNGE3NjQ2ZjQ3MTYxM2MxODQyN2JlYjAxMTQ0ZDY2ZjFmZGIyYjEwNjhhNmZmMzllYmQ0MDkyMGQ1MjdiOWIwNTg4MGRkZTBjOGM3ZDU0NDc1OTNjZWEyZGI5NTM0MzU0ZDNiMjFkOGY0N2I5Y2UwMTg0OWJhNDAwOGE2ZjkxOWJhMjdhNDVjYmIyOGQ1YmMwZGE2MzVjZmRmYWJlMTgwODU0ZDFmNTE5MzQyYjIxMWFlYTM1MTI0ZDRkOGM5YTMzMjA4MjgzMGZkODI5YjU5M2FjNjBlYTMyOTg5ZGY1YmZkZDk5NDk0Yjg4YzEzMTVhM2ZjMzdmYzg4NTg4ZDA0ZThmMjFhYTdhOTM2OWNmYTA2ZWIzZjZlOThkMDg2MTQxZTE4ZGE0MWM5OGZiNTllODBkYjk5YzVjZWQ5MjJkZDEzOGY5NjYzOGQ4ODE2YjhhZjQ4OGNjY2IxYWY4MjY4NzRhNjM3ZmM5NjA2NzE2NTQyNzhiNjk2OTc1ZGU2MmI3NTU0YWM4ZjgyNDk2YWQ4NzVjYTU3M2MyMmMxMjAwY2U0Yzk0YWU3MTJlZGI0N2UyNWMyYjI0ZTJmMDkyMmQ3Njk5NmFmYzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LeibTSLpfzD8b0gCc935NSM587VZV9ynUZrD-TewoeJZ92faJLYsCiLS6YzblnoSJp6ZELfZIr-67wS7tZtiRA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110459_50_2475_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.290Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkNGQmVmdytpMmIxdW8yanVDdGVMSmdkeUNhRU5jWFBkT3ZGSFBNU2UwOHYweFpDcVNEMFlGTnFkVlVGdndLT0pFZXFpZS9RMTRTZWgyK3dobU5KYUl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTA0NTlfNTBfMjQ3NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGYyM2Y5NzYyMzllODAxNzE2NDMzYWI2MGU0MGQ1OTI2OWE2MDhjNzIxOGFmNzA5OTA4NjI1OTk1MmMxODcyNDdlODk5NzUwYzNiYTUyY2U4ODk5NDYwOGE4MjY2ZDcxMWJlODRlYmIzOWQxZDcxOTFlOTBmYmRlYzM5YmQ0YmRmNTNlNTQyODBhODE3MDViMzI0OGEyNGVjYmU4MjIxNmRiZGI3MTQ0ZmJkNTAwZjcwNjJmNTdlN2YyN2I3YWJmOTAzN2UzMjg0OWUzZjdkYWQ3NTBiZjBhNjRiZWIwNzcxN2U5Y2E1NGZiN2ZiMzdkZjlhZGVhZDg1N2E2YTViNWE2ODY0YWM4Yjc2YTE4Y2Q0MDAyNTkyMTY5ZTQ1MWFlMWVmZGM1YjEwNjA1YmMyNTBmNTU4ZTdmN2U2Mjk1ZDMxZTg3YzlhNmQxNDhmYmVmZjFmNTI0MDE1ZmE0MDViMWMyZjczYjRjMGNiMzAyODIzYTk0MDVlMWI5NDIyZjM2ZjMzNzdiOWVhMTM5ZjA2ODlkNTFlNTE3YTA5ZjMwM2JkMTllYTZiNjM2ODVjOWIwZGU5NWQ5NzkwYTg5NTk3MzdmMjYzNTMyZjFiYmJjMzM4ZjQyNmZhMTU5YmQ2YmM5NWVjMDI1MmE1ZWNjYjE1ODI3N2Y5MWFmMDliZjNmNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.U_o8PPcqw16-tx8LRR4JW1FiXhlFWs46x3GsnLkV9yjknleIIIhUX7rGo6MoBML5hEguV6QZHINfku8Lq9OJ-g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_110459_50_2475_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.293Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImdybGFvUzhrZUthVGtIcGsrdmFXS1FSVE8rRjdxemFMNkMzenR2V05seElsbG1CdzhOSTBwcEMwMG9CcUhFWXJWNDZHTEJyYlBCeFNrbFBiajMySEx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxNDBfOTNfMjQxM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTI5MTM4MTY4OTZlZjEwNTc3YzNjNmMwN2M4YmUwNGM0M2RjMDM1NDZiYjI4OTA4NTY5MmYxZWJiY2YyZTg0ZDM4NjRjZDk3NjJmZjRlMWM2MDc5YTQ0MWRhMGVlNDJmYzZlZWIzZTcwZmEyY2M2Y2QwZDJjMzM3NDY2MGI3ZTFiYzM4ZjZmODg4NDA5ZjFjMTAwOTUyOTIyOTM5ODg3YWU3NTNjYzgyNDlhMDRlNmE2Mzg4MzMyYzE3ODc2M2JhMWZlMjgwODAxMjYzNjg0ODhjNGY2OGUwODZjY2FjMWNjZmIzNDEzMTViNDVlNTU1MDAwNjlkMzc2OGIyMGExNWVjMDdkMjViMDMyNTliZTFhNTAyYjg2NWEwOTg2Y2Q5ZDgxNzM1MzY1ZDE3Y2EzNWQ0MWU0MDc4ZGQwYjAwMjc0MzYyZTFhNDUwMTczZGI0N2Q4Yjg0Njg0ZDljYzE1MTM2ZjgzYTI1YjgzYzE4ZmZjZDY0Y2ZjY2IyNDNjNWJkYzRiNmM4YWYwYTJjOGNhYjU2NGM2OWY3MmE1NzNkZGZmNTkxYTA1NWNkNDcyOWM0NzQ5ZmZjNzc2NzcwZGViZTE1NGJiYjUwMmM2NDExODk1NjM2NWU2MDBhMzY0NzI4NTY0ZDc5ZmE1Y2FlOWE4N2E3ZjVjN2UzNDUwZjhhMjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zrVR2SdYWTpa5KbKHkOHarY-Lz3632UhOvHEPQ6jslgPpyFnZNF-I_1W7iqtFnn7gF4tN1oGPefiU46f0uybtg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112140_93_2413_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.295Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ims3UXU2NG50NkNUb0QvaTZYU3FmZFUyRGNLOVJWMWpPdnZYWDViM1lOd2FuNENTRVd6eGZMbkpGeG5GdVhERWVGV0dKZlJlbUlZY2F4MmlPK2RvS2VBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxNDBfOTNfMjQxM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzFmOTIwYjIwMTg2YjEzM2QyODRkZmRmNjA5NmJlY2RlOGMwY2Y2Yjk0Y2E5MTA5Y2I2ZDBlOWM5N2Y3N2JkOWQxYmQ2ODJjYmZkMTMyZDUwYmIwMjNlNjQ3NDdkYTNjNDFiY2VjYWY2ZjVkODQ2MmI5ZTNmZWVkNGIxYjAxM2VkMTFkMDIyOWQ1N2IwYTRmYWE3YmZiYjgwMDljYzBhMTNiNzI3YWY4YmMwYmFkYTc4Y2Q5MDg0ZWY0MmEwODliN2E5NDcyODVlM2VkZDJlYTg4YmNiNGUwMWIzMGJkZWViYzFiMmIxZmUwMzY2NjQ1OTJkMjVmNDYzZjA1YWIzZDhiZDRjYmJlYjhjNTIyOTNlMmYyMTEzODEzODQ2ODY2YmE5NTNhMzI1YzhkYzgyMWUzNGJhNmE0MjQ2ODQ1ZTFhNWU0NjZmOGJiNTUzZTA0MDI5NWE2ZWE2YjJkMTFkMDkxZDIxMDlhZjE1OTc5MjYyMDgxMDFhOThmMjcxODU5YTZiM2U1ZWRkZTE4YTE5NTQxNmU2ZWQ1YjJkMTRjNzJjY2U4ZTIwYmUyNmQxYWQ1ZGU0NzVlY2MzZjk0MjAwNTg3YjA3YjZhYWFlNTBjMWUyMzc4MDU0Y2E4Mjk5N2FjMGRhODUxZGJiNzAzYmRmZTMwNTQzOTk2ZjE1OThjZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yBkQzPqlweqLyGhBVfEwP6ZgkIed91MKjcmtZPQPneZAu7me4gz1H4VfQIC6TODyNRiweiTA-bTo9rTYC7-OsA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112140_93_2413_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.298Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkdMNzdoTTNnSlJPdGF1SWVHOTNNVVVHUVg2eEszajZkVTFwSjlUVnBEQ0RDRTBTNUdpem5nMnJ6cTBCUU00WjhGOGZvUTNKZUkvRkV2dGdUa1JpWDh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxNDBfOTNfMjQxM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2FjMjhmNTlmOTdmZDEzOGQxNWNhNTAxM2NiM2JmMjc3N2UxNjllODhiYjUxMDZhMjc5NTFkNTlmY2E2YWY4NjI0NWRlYjU4MmI4MTcyNjA2MTNkYmM4NWM2Yzc4MDg0YmU0OTVjOWI5ZmY3NzI2NjNmOTAxMmNjZTQ1MTQ1Y2JlMDJhZjUzNzY3MDFhZTdlZGUwZjQ3NTU1ZDJmYjU5NzE5Y2I2YzNhMDEyZjljZWNhZjQ1YWY1MjJhODE2YTU3MDdkZWM3OWQ5NmZlZTNhYzRhZWUyMjhhZDQ1NTFhOTg1YWFjNmQ4Y2Q4YzdiMDZiYzQzYTg4ZjFiMjYwMWZkYjE2ODQyYTAyMWQwMzljNjQxNjZhM2RkNTVkMWFlYjZiYmFlNzRiYjZjOTc2ZGRkNGIyMDQwN2NlNDIyZTY5NzE3YWY3MzJiOTIwNDFkYjBhN2NkM2ZhOTgxMzBhNzM2ZTEyOTMxMTg1MGY2MmFlY2RiNzU0OTFhNzI4NDFlNjIwNmMwZmI5MDk1M2ExYjg5N2Y2ZGU1ODEwZmQ2NGUyZGMwZDY5NjA5ZTIzZTI5NWFkZTg0ZTA1OTM2MTZlZGJjYzU5NGI3ZjdlNWExMzVlNjA1ZDgxYzdhNmU5ZmUxMGI5ZjllMzhiMjJiZDRmZDE4ZTg5ZmZhOTJiNGY0YTM1M2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9h3vEyRlc6oJhFGIB3L4Hlc_JEy-R0kFKQKlBkr6nOo_bCFz30G4kWOgc51xpMOKP5HF2hmhGa1NZqbKpezKrQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112140_93_2413_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.300Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlE4cER4T1FjaThBSTlzS0o5UmQxYXluSGpTY1F3R21XSllPd0Rsd05WTTlFU214L1doOUJRTnQ2U1BUaFIyaUVIZzdPajFKajNhZmdGeENDblc3clpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyNl8xMTIxNDBfOTNfMjQxM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODgyZGM1ZTgyNDhhYzJjZTNiMzkzYzEzODE3ZTE5NTc1MDJkMDk4NTZiM2NlNjRhMWM3ZTgxNDJmOWM0MDc0MmYzNDZjYTU2NGExMzQzNjczZmViMzAyYTQ4NDkxNWJlOWE2YjIyZTdiNTI4MTQ2YTQ0NTJhNTA3YjFkZDYwY2UzMzgzZTMwNzdhZGY3NzBlNDRlNDExNjNkMTEyOGJhMDgwNTU5YmJjNDc1M2Q3ZjYwMGFjNmQ5ZDVjNTJjYzU3ZGY1MzM0ODNhZGIxMzY2NTk0OTgzYjY2MDgxNDE5ODU3NzYzODA0MjljYzUwYTgzOGViNmM1MmRiMTlmMDY1ZWExZjM5ZjE5ZmMzZmIzY2VhMzQzZTM5YjkxMTNjNTFjNTdlM2YwNzAxMDJhNGFiZDUwM2Q1NmQ1NGZjMTc5Y2U3NzEwNDdmMTc0ZWJiZWI3NWJjYTFlMTVhMDJiM2M1ZWFiMDQyMjQ5MjU1MjI5MThhMzk3YzA2NDdjY2Q1ZWYwZjVkYWJjOTYwNmMwZTFiMzY0MDViNWFjNmY0ZTY2MjY5NzUzZjcxNzg0MzI5OGQxNzViZWNjNWY0YzU4MTA5ZGZmYzU1MTY0MGNmMWU4YWU1ODliNTE0N2JkNDI2NTk5ZDIxMzc3YTVkOWM4NTQyZDI5ZDg0NmI2ZWFhMDVlODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IVZX4ZeoMr0HpEWA8HUVsafo3zxPogZhdvOf2tZFhZMNw5sqwjCEo1Q-eKdQD8BkhhSP5Kt_bjnSTJKeoCHyRg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211126_112140_93_2413_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.303Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjZLTkJXRXJXUHN1NDBzbkRoZElFUHNyN2JyVEVVdS9xcEVPMWNxS2wzMXUrcUlRU3lTR1c0OTB3S2lzRllBc3Y3Yi9oSEF1WVNKdkFJRWl3SGJZelhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOV8xMTA2MzFfMjhfMjQ3Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTg4NzY0OGU2ZjI4MDBkMmM5NWQ1YWVkM2Q5NGE2ODZmZWFkN2U4NzBiMDdkZjZhY2M5OWQ5YzUxZWVkNDA2ZTNlNDhlNmY2NGMyNDRlZmI4OTMwYzI2Y2I5MWRjMzgxZjk4MTc5NDNjNWU1YzA0OTgzNTZkMjgzYTJkYjI4NGNjNDQ1OWI3MzFiMTQ4ODk4YzI2YTQ3Njk2OWMyMTQ0NjliMGRiMzBiN2JiNWU2ZGVlY2JlYzczNjEzZjkxNDllN2M4OWMwZjFiN2UyNmY1YzE0MzNmNDUyMzM0ODU0NTBhMjFlYzhjMzljNGNkNzgyN2VkYzY5NWFhMzQwYThlNmYyMWI4MjA5Y2YzOTRkNTBlZjVmMmIzNGMxYmJlYjkyMzlmNTkzODU3MWI2MjlmNTkxNGI5MzhiMmJhMjczMDM4NDQxNjEwMWZlODlmZjgwNjA0MGQ4Y2UwZWM5YjRkNzUwZDVhNWQ4ZDIzZjkyZGFlNThhYmNmZTQ0NTdhZmQzMDhjZjQyNWVhZjBhZmJhZGZlZTA5Y2MyNjlkM2VjNTFlZWI0NDU4YzdmNTU1OGUzYjI3NjM1MWM0ZmViZWVkNjU5ZjhhMTQ3YjQ4ODMzMWEwZTlkZDJlODgwZTlkZDcyMjAyMWJmOWQzZmFkZjJhNzc3MmVlNmExMGU2MzY1NDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Pxzf7_u7Ig5wFva90vYTIUWQK5kYUkmzPrgTDA06h_PMm0jQktK1rpoF0L9JImuTiw99TNHh0KSFjoQ-vGcHRA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230419_110631_28_247f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.306Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ijg1TnNaS1dYa3lnSnlTc1RkVmhjb08wR3FSN3lGYmoxUWxjZEM1Ry9SMWxxNFl5THVQK1E3MEliUEwzN1BYMmJFRUZQSVhPeDl0VkhQd1dzdU9HZEdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOV8xMTA2MzFfMjhfMjQ3Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDk2MGUxYjEwM2RlYWUxYWFlNjUxMzJjYWM4ZDg5NDUyZmUzYTViOWIwMjUzMThhNTU3MThjOWVmMjg4YWEwYjhkMzMwYmU3ZmQwNjcxYTdkYmE3MTE0OWQ4NmFiNGY4YmE3YjBlYTIzNTRmMjI4ZDY2NGJkOTg3NTZlMmY2YmViODRiM2E2MDgyMGU2NTU1ZDBjYjMzYzE2ZDY5N2E4NmZjYTE4M2EzZWZiZDM4YmU5N2Y3OTEyZDY0ZWVjMzI5NWEyNGQ0MmYwZWI0NGU5ZThhY2E1ZDg0NGY0MTgyZTNhMTExYmI5ZWQ1NzJjMTBmYWI1ODEwNTMwOTQ4NzU3YzZmNDRlN2IzOTU5NGY1YTUzNThlYjIxYWY5NmQwYjRlYTBmYzVjOWYzNDMyYWY1MWE5ZWYyYTgxNzhiY2JhNDcyMzUyZmI5NDNmYjJlY2NkNzAxNjFhYWYwMTJlNGRjMTdkNzJjM2IyNWRlMjI3ODY5MGZkZTI5ZGQ0M2ZjYmI1ZWNmMDQ2ZmU3ODEwNDAwZjMwZjlhMWJkNWRmNTIyNTViMzhiYzM4MWY5Yzc3Y2FkMWQ3MjNiYTU1NTY1MDA3MzYxYTRhYzI2MTIwNTQ4NWNhMzc5MTVhZTJhN2UwMTdlN2M3MjI3YzIyYTc0OTUzODhkMDYwZjQ5MGRhOWM1MWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.57vEB9Do8IWd1vSGtuZPrAjrVsXjhE8bpgaJ-o89OJGdZvKA1W-zNDIHxeX2BrNk9bOVTRUjw_xVbeSODOIhGA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230419_110631_28_247f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.309Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IktBTTdnZm55WGs0N05kRjJsdU9NR3VpeE42RmVaclRGTVdkWlV1cDlIRVoraWxyWGZ2Si9DNFEyM08vdEZmcHloMUVocWZTbDh0WURJUHNHajRGamR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOV8xMTA2MzFfMjhfMjQ3Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDYyY2I3NjlkODllMTY2NjY2NWIxYWM3OWQ0YTkyOTU2ODViM2FjZTZlMDJmOWNjNGY0YjM0YzRmNDRhODA0MjMxZDZiMTYwYTY0ZDAyZWM1MjlmOTNiZTI3ZmQyZTg3ZDMyZGRjMjU2NGQ3ZDRhZDMzZTg4NGI2ZWQ3YTM5MTNjNDYyN2I5OWY3ZTNmODNlMTFiZjk2OTY0YzVmY2JiZjlhYTg3YzUzMjU1MTZlOTczNDgzODEwY2FjZTc0OGUyMDhiMjJkY2ZlMTM1NWY4ODQxNjMyZDhiOGVmYjIzYTc5ZDA5ODVmNTQwNWZlODY0YTdhZTRkNGMxZmIzNTE4MGJjZDA1ZDdmZDBlY2ZkMzE5MzNjYTg5NDJhOTkyMWU4ZjE0NTI2NzYxODU1YmVlNmNjMDc3Y2ZkN2NlOWU2N2Y5ZGM1NGY5ZjYyNzM4YzFjMmNlZTI4OTk1MDRhMzcwMTg3NDc1YjhjYjI0OWNhYmFjNTg3Nzg3YWUzNjUwOWVjNWU5OTA5YmJiMzI3MzgxOTk2OThjMWExZDU0NzQxY2ZhZjg4ZTc4OTk5YWY4YWVlOWYxMzU4N2RmZDg4YjQ3ZDNkZGM4ODM0ODMyZGE3MmI5OGJmMTQyNTk4MTgzOTNlNjM2Y2FhYjQyOTEyY2NkOTZjMzQ4ZWMzM2M4ODMxNjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Uw9ir6DYJuQmaMBvxleaV-Dfvu_V1_u7c0UzmczWbh913r4pftu0iS1vShWrZo2VbvNdPz-PMvnGr46256_83A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230419_110631_28_247f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.311Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjR1VkpkWFgrdzVDTDRMR0J1K2ZiMzlCZkZHU3I1ZW5Hdkt5bnAvY28yRGdNdDJidmsyUXlFNnVWRC9sZlF1a1h4OHpoSU9oVkJhYkN2ckxPVXYrQXpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOV8xMTA2MzFfMjhfMjQ3Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDEzZmQyYWY4NzE0ZGY4NWZhNDM0YmQ0NDI2N2UzZmY0M2RiOGZlOWM4OThmOTg4M2ZkMjUxY2NkZTA2OWIwZDg1MzVmZjM4MjhkYjA0NzUxM2NkYTdmM2Q3ZGJiOWRiOGMzNjk1OWRkNTE2MjM0ZWYyZWFmZDgzMmY0MDBjN2E2MzkyN2E0YmQ0N2RmZTRmZjc3MzQ3MTk1N2VkYWQ4ZmJkZmQ0NGM3NzgyNjIyZGVmNmNhMzBiNDc3YjQxZmI1ZjEyMzJlMWZmMWY5ZjY2Mjg4MTYwZWQ2NTliN2FlNWY3YWY5ZDE5OTllNmM1MmFkODQ3N2Y3ZjU0MDVhOTc5ZTVjODIxNDMyM2I4NjUxODkxY2QwOGNjMzkyMGMxZjBlNzM2OTg2YmU0YzQxY2JlY2IyNDM1ZDAyOWQ5ZGFhMmIyNzRiYjdmMWViMDM3NmNmMmYzOGFmZTQ3NjkwNjJkNmI5YTgzNDAzNDY0YzMyYjE4Zjk1NTM0YjJmZGRhODg3M2Q2MjExN2RiYWY3MWU4MjNlMmJlNzRjYmExZGQ0M2RhZDIxNWM5MjQxNGM5ZDI0ZjExZGQ2YTRhM2FmNDI2OTI2MDA5NTgxZWM1YmM3YzBhYTc4Yjk5YzUwMWFhM2ZiZjY5NDZlNDZmY2NmODBjYWNiMGExNTdiNGE0ODg4MjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8xMpBEA_iw2TiVOQIFMofjZG8q0jOtdq26xUBWi7m2nDzDyD0NUPRWZfcgqATGK87bzxjF2CxD8aLcq7qXEdSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230419_110631_28_247f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.314Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InJrMThlWEZZUFJWL0NZRGZGQVFXT2twb3RLdmVvMTdRN2ZDY0tzYjAwZmQ2K1lCUW1BY1Nlc25ESlB3d1ZkYXVKRTN2azkvSUJRTjBNS2NqNklRNU5nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMDQ4NDlfMjVfMjI1OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTgxZDMyYTdhMjgzMWJiYmMzMjVmOTczZDk4ZWFlNzE5NGUxMTkxYjY0ZDM3NWIzZDVmMjM2OGU5NTlmOWU1Y2UzMjA2Y2YxZjcyNWQwNDA5MmJjOGZjYzM4NzAxNWZhMTA1MmUzZDViZGJmMmZkY2RiMDJkNzI5MjZjOTY3ODEyYjhmNDdhYTA1YmJiNjRjOGI4YjBiZDYyZWVmZmMzN2E2ZjYwMWUyYmE0OWFlOGJjMjY2ZDhjZGY5NGUzYTVlZTllZjc1NmE5ZTI2MWNjNmUwOTY5YzViMjA4NzY4OTdiMzc0OTlkZWIzZjcyMzU5NDFiMzcyYjk3NTI1ZGYxZTg3ZGFhYjg4M2EwYzYzZGZjNDllMjQ5NDBmMTUyYjJlZjZjNTkwMjhlMjdmMWJlZGIxNTI5N2YxZjViY2FlNDNlNzlkNDc5YTc3ZWNhNDhhYWQ4YzBjN2E1ZDNlZmNkZjgzZGIyNjcxZDcwMjE1MzNkNDc1NmM3MmYzYmNjMDAyYmI5ZWFjOGU1NzFkM2Y0ODNkNDRlNTBhNDU3ZjZlMTUwYzU0OTljMzBjNWZmOGMwZTNjYjZhNjRlMTMwZjM5MzNiMjMxNGEwYzEzMzkwZjYwZGZhZDkxMTBjMGFiYmEyOWY3MWJmYzVlNDllNDY3YzgyMzRlM2UyYjA3MGUyMmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MeL_tbqh2exgDu3Hd13JstFA1QfTnFK5_OOUiXeUawXAYadovqFgT7WnjMp1378XAIg7yXg1cwfXR681mKZFzg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_104849_25_2259_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.317Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlVOME1LMzFUQkx2YVFhVkJCVERINjdudU5lMXZmVWw0dTlWYzBaK2dIUTlubEdZaGYwSlFVZHhHYUg5VXdrOGlXZFFSQmJOWTdmYXFOQnZRTUhWSmd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMDQ4NDlfMjVfMjI1OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDIyYzg3NGQ3NWYwYWJlYjg1M2E5MTcyOTVjMTgwZGFkZjM2OWNmNTU3ZGFlMWUwN2VkZTllMTMxNTVjOWZmNjI2Yjg5NDY0MWIyZWMwOThiYWM4ZGI3NzZkNjdjN2VmN2VmYzc5MWVjMjBlMzIwYTIyMjRjNzg5OTFlMGQ3ZmFjNjk4ZjVmN2MxNzU1MDY0MTAwYmY5MTk4ZThlODRiYjI2MjY2ZmY1NWMyOWMzZjg2ZWQ4OTE1MjNjNjI2ZmJhZmNlZThjNGI4MjkyZmE2YmUzZTU3MWI4MzIyMmYxNzVhMTdjMTM3ZmExYmVhZTViNTljYTc2MWU2NmRmY2M3NzEyMjkzZjVmMTcxMWM0ZDliMzc3MjQwZTk1NGY2NjhlNGJmNDgzMDQzMWI5ZGFiN2Y2NzExZGE1NGUyZTk4MTUyMDdlZTlhMDMxN2I1NGNlMmYxM2JmY2VjNThlMTNhOTliNWI0ODFmMDc5MDJkOWI1ZmRjMDc4ZDAwMzUxZmNjNDNiMDQwMmIyNmRkYjA5MWM3OTEwM2JiYzQ0NzZhN2Q5Yjc0YTdkYjk4MTdkNzU2MjA5ZjE5YTY0Zjk5ZDhmOThmNzY4N2I4MGQyNzA5NGI4NGYzMGM1ZTQ3M2RkMzBiNmE0OTdjOGY3YzNlYWEzODYwMGQ4NjJlMDUzOWY2MzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lclB_P63v6vGDjjSjr_F5NeISB8vjvMHy3cSs2RoySqKZQC2icMlDD3R10qYyTkaEB--AHAz1XRELjOmVbPskg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_104849_25_2259_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.320Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imh3RDhMdmppekt2c1VNZngyRHVCeHNlL2JsR211QVEyZ3pSanUxN3VKNTR3MEtxSUdqVXVUODNBdUdsNjJObFNxQUp4UEhZd2YvRk9EOEt4MVJPVjhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMDQ4NDlfMjVfMjI1OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODRlZTI0MTExMzc0MTEyYmYyMDNlY2IwYzhiYzA0OGJlZTIxNDYwOGJmMjRiMzJkZjE2ZTNmMTYyMmQzYzZmZjZjODc5OWEyZDc3YTRhOGRkMTRlNGIxOWVmZGZjOTc1YTFlY2RmZWFiYjU0N2M0ZThiY2ExNGUyNDEyN2E5ZDBjOGEzOWMxMjMxMjFhNjFkOGFiY2NlZWY0YmViZWQ5N2ExNjVlNTAyYzZhNmMwOTQ0YmY2Nzg0OGU2NWM3OWEzMTI5YmFmOTRiMDAzZDJiMjAyYzNhNTNiNjRhZmVhZmIwNWExYTA0YmYzZjAyODQ0MTgxN2Y0NTBlNTM1NDg0Y2E1MDc1NzI0ZmYwMjJlMmVhMmRmNDVmYzNlYTViYzJlNGQyNWVmNDQyZTk3ZWU4Y2E3ODY5YTk4MDJlMDhmNDEzNmEyNmFkZWQ1ZjMxMDNhZWFiYjM2NWVjOWMzNWU0YTMyODE4NzgyZjYzMTI2NzkwNTAxODI5ZDBhZWE5MGQ4MDcyM2Q1MDM0NTI4NzNhM2Q5YTg5YmIwZDVhYjVlZWM0MDlmODk5MzkwZTNjZjA4MGYwNjgyNjQyNGY1ODgxZGI3Y2I1YTNhZGUzYWVhZmJlZjhhMmUyNmFjZDdkOTQ1MjkyZmIzNzI0ZjU3OTY5OWVlMjY4ZDZhNjU0MTE0NDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sWVT35ZmKmfKV2BFMmEIgL0ZLQAJVr7Pgjpi1c97EJOYZRCtIa3ghStrrGdiE6-wGZ-C14xJUpK66cMcLkpG4g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_104849_25_2259_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.323Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImZORGwzd0c3alprb3d4VUZCN00wUVhPcnViR1hlbVM0ZUZBbDJYYy81dE5MMHpZQ0xQK1NDMFpDbHBlejRjK3kzSDFRaERiQ2xiWGhESzNUcFdKalpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMDQ4NDlfMjVfMjI1OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGZiNTQzZjE2ZmEzZDc3NTFlNWEyZDZhMjRiNTNkYzg1NDExZDVmMzNlYzMxOWViMGEwZjA4NTVlODkzZTUyNDQ4NTdhZWY0MDYxNTA4ZDJmOGIxMTM1ODk4OWZiMDNiN2FkZTQzYmNkZjI1MGQ2M2Q2ZDBhMGE1OGQ4NDYxMDEwMDJkNDRlZjVkODU4NDk5MTNkNmFhZjU3OGU1ZTJlNDE0M2ZhYTQ5MGQ3YmQ3NzdjYTcyYTc2ZjM5NjA5YWUzNjkzNzY4MjVmZWUzMWViNmEwN2E5NzM2MDJiMmJiZDhhYTQ4NjEyMjYwMTU3NmQ1NjFmZTY2MDQyMDFmYTU1MWM5OTllZjJiODQwN2M1ZjM3ODg1MmI2MDJiMjUzN2RiN2ExNDNlZDZkMjdhYzJhZDNlYTBhOGYyOWJlMGEzYWEwMzdlZjUyOGJkYjIwYjE3NWYxOWFjNTQ0ZGI3NWVjOTZiYzk3ZmVjZTY4YTBjZTZiYmZkNzEzM2E2ZTdiMzJjZWY3ZmU1NDE4YWZlZGZmMzA4NjI4NWJkMDlmNjBhYmQwOTEyNTVlOTdhZDA3OTUxZWY5YWMwYTMxMzNkMzQwNjc4NTBiNGY1YTJlNzExMzUzYzhkNzIxMTE0NmM3YzcyZTRmNjliOWQ0OWZiZjAyYjY0ZTkxNTBmOTIzYTM4NGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2_ai8459v9wMy9Y6NtGlWLwjIvDeQQB8-LYDRiNksvltx9dItW2FbO0VNbfTP8btBdjr3d_hbKLxP6CfvC-DaQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_104849_25_2259_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.325Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InExcjVybnJIUUVMMTF4TmFYVVd5bExzejU0QjE4NzI0N0t1TTliTnlVWWd0Si9iWE0wOXJKS2xoWGpXbXI4Z3M5ZnhxUVZ0amtEc2czeGxPR21YUHZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwMV8xMTAzNTRfNTRfMjQ3NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTA3Zjg2MDYyNTM0ZWQzODJkZDZkMTcyMTQ2ZWEzYjkyNmUwN2FlMWRkMzhlZmVlNWZlZTgwZjMyNTE0ZTAyMjc5ZGIwZDdjMWU1NDFjZjI0Njg4NzI2MzJmMDAyYTllYjdkNDk3MTQ4NTAzMjNhMjNlYmM1MzFkMWI3ZDhjZWUwMDA2NDcyNDU5MTM2MGI0NzZlODg1NjlmOTE4MDcwZmMyMmE0OWQ0MTExYjYyYWFlNmMyNjlmOTMwZDA0ZWFiMDg3OWVkOWMzMTdlYzQ1ZDNhN2I3MDExODQzMDRhYWQ1YWU2YjU2NDcyMDBkYTFiNDE1MGI1YmZkZTFlMzI5OTQzMjc2ZTA2Mjg2MGQ1N2FkOWYwNDc5MGVjY2YzZDFkMDRjOGJlOWYxYTZhMTBhZjNkMjdhNjdhMzFmMDAyZmI1YjFhOTRmOTQ5ZGU1OTlhZjI2NWU1ZGRmYjE0NDI5ZWRiMDE0MTliZDU3NzM2NWQxNzQzYTg5OWVjMjIyMzEwMzBhYThkN2Y3NWVkYTE3OWE5OGIyZGI4ZTE4NTY0OTUxY2ZlMjc2Zjc3ZWVmNGJkYjBiODUyMGY1NzBiZDNjNDgzMjRkYWZhMGQ0MGIxNjhlYTY3NWMxZmVjYzhlNmE5ZjBhYzYyMDU2Mjk4NGE3MWEwNWViNTlkNjk3ZjM1NWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SMYEwVNPHxF8rITe1XwUzBiBl3jOz8VJrp1Xyl5dNTd9FJOQFMQ1qscXFxq_clZC5TqMTT87dI4aL5oWQYEdTQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230201_110354_54_2475_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.328Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjFncGR6eDJsQzZlL08yaUlaRTA3alJ0M1QvRXNwT1dvMlVub1U3bGRzRllmU09sMW52b21DSXVrUmtHOGNRQUtaZis1K3VSUkIzNWtVQ1lVbDdPTVl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwMV8xMTAzNTRfNTRfMjQ3NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjMzNjA1YWNmY2MzNDg3ZmViYjA2M2VlNTM3ODAwNjYxN2IxOWZjY2VjY2ExMjliNzEzOTBjN2NmYzNmODBjNDI5MDA3ZmQ2ZDE4MGJjYjliZmNiNjE3Yjk5MmExZDJlMDg4M2Q2YWI2M2MwZjI0NTQ2MTJlYTg0N2RjZmY0NjNjMWY3ZjkyZDJhZjk3YTQ4OTIxZjc5YmNjNTM2YzljNGNiNGZlYjBjNDdkN2ExMDExZWVkYThmYzIwNGI0NmMxMTFkYjJlZjJjY2M4YWI2MGY1MGVmMWYwMjdiNWMxYmY1Nzg5YjcwMWE2NTM3YWJiMWM0NWQwNTI4MjM3YzcyMDhmY2IyNzg1NjYyZWVlYTg1ZmIwOGEwMjBmYWQ1MTgwYjgyODg1NDAwZTg2MGFiMDk5NmI4MzQ1MjQwZTg5YzQ1YjVkZjhjZDkyYWY0NDM2Nzk0NmE0MDg1M2IyZDhjZjlkMTliNTljNmQzYWM0YjJhNWIyMmYyNzZjMTNmNjc1ZDM0ZmE4MDdlYmY4ZDBlODdhNTQyMWMxZTNmMWVhOTE2NjkwNTc4MmQ2ODQ0YTIxMzk1YjhiOGZhYWQ1MmQ1NjZhNDNmYzM3YjVjMzVlZGQ0NDlmMTkyN2QyZDQ2ZmE2NmI1MjE4ZDViZTQ3ZWRiYzZmNzY0ZDIyYzFkZGMwYjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.94dd3_N3cwQKD4IWXoqA_NMiR--JUhEe5kdzr-c63hHIexTsHsMQt7JKrVuwGlnBbWBCPSZ2MuDt6_9lXcmsxA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230201_110354_54_2475_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.332Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im5jeDRnMzlmMlhBWU1rcFlmMVpqdmNUcjVjRFcxT0UyL1o5NDUwdzk1N09oUERiY3cvRVRGUzU5V29uVmFWYmVnaGU5enhZZW9Kb1pUMFp2ZFM5U05BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwMV8xMTAzNTRfNTRfMjQ3NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjJjNTJhNGVkOGFiYWFmNmQ4ZjE5NzE3MmU3OWYwZGFlMTFiMGU2M2Q4MWE4ZTk4NmU3ZGUxYTQ1YTI1ZmQ2MTliOTE5ZjI4MzcxMzc5MTM3NmJjM2U3ZWUzZDQ3NGQ3MGYzODJhNzA1ODBjNGZkNGViMzg3Yzk2OTAzOWM5MjI5Mjg1NmYzYjcwMGRlZmNhNTVjMTAyMDdkNTkwMjQzZGFkZTE2NjI2NjYyYTAyM2EyOTBjN2I0M2ZjYjkzMTdhZmIzMWE5MWMzNjdiMzBmM2JhNjNiZmYwMDdkZTgzZjQxYTg0MjVhODRhYzBlY2U0N2ZhODJmNzE0ODk4NTgyZDUwYWVjMGQ5OGEwNGNiYTdiNmJkM2RmYmI2MmQ2NmIwMGFlOWE2MTc0NmFhMjVlMzE2ODc0ZWY1ZjkwZjFmMTI4YTIwZTNmMGJhYTNmYjg3NzkwZjk5YmE1MjU2MTQ5MjE2MDZlZjM0ZTRhZGI0MDE1MjljN2JjYWQwZjdmZTM3NTc3OWVmZjBiYTlhNzNmYjQzOTZkYzQyMzE2OWYxMzFiOTFmY2JiZjgxYmVhMjFhYzk2MWFhMWEzMzZmMzY3YTY2MDRiYjJkNmIwMTRjYzhlZjcwZWJiY2Y2NzNjYTc0NTUzNDY3YWY1ZTc1YmViYWQ4MDg0MjhhY2RkMjExZWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xQVJOWuyfGowfow0wCTeRNVSVkjtcgSLnDlXOXV0AMFkXcaYAyqXTXdkym5LSLhqcJM7SUnmkHjKtt7JmjxXeg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230201_110354_54_2475_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.335Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlJHRlRzQS9WQWsvTWVaVjVGSzVtallybUU0MlB4dHdOQnR1aUdEOTZxRk9kNW9rTGNYRWxoWmdCOTdIYVVNaEVKeG9ldFlJMzZ4dkQwcE5KbXdnV1JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwMV8xMTAzNTRfNTRfMjQ3NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDRkNTE2ZjQ3NTU2MTk4MmNmYzFhMThhZmIyZDBmMzNkNDg3ZGNhZTdkYTJiZjg5ZTAxOTkxYTM2MDY4MzBiNWJhMjFjZDJiMzIzNjBiMWEyZDI1ODc3MWJmZWM3YmVmZGQwOWY4YmJkYjdmNjcxY2IxNzM5ZTUzOTgzMDc2NzQyOGZlZGMzMmQ0M2VkOTViMjA0MGQ4MDVhNjhiNTZjODUzYjYwMDk0N2VhYjA3NGQwY2YwNzUxYTQ2OTQxZjljMDYyM2E4YmNlZTgzNTFiMmY3ZTk4ZTY5Nzk3ODdmNzYwNDRlMTQyYjM4YTNlZTM2ZmI3ZWMwNjAwZmFkNjAxMGFjMGQzODgyNTA3ODU0YTFkMjkzNzg5MmNhYTZhYjgzMDhlN2ZlYTczYjU3N2NiZjhiMGI1NDkzNjRjM2VjOGJmNjRkM2FjYTAyNTgzNzY4ZjhkYjY3MjUzZTZlYjBlM2I4N2U5YjU4MDgzYWZhM2JkMjhlN2JmMDliM2QwMDRiNmMzOTE4MWZhYWE3YTQ4ZTAzNGIyNWVlOWZkMjhiNWU5NzQ1ZTg0N2Q2MGQ1Y2QwYTI2ZDg5Mzk2OGFlZWUwNzQ3MGY3OGJlMDc0MTAzMjczNTliMzQ1ZTc1ZTE3ZTBmYTRkNjU4ZWJlMTU4YWY3ZTZkN2IyZjIwNWVmYWNiN2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wzMf4j_6rMKfeLjGBzI6ZftaFT4TemPpbHJMg20UNvzT6lPnYkNLJHYubPFd892I05DSXmWG-7C9FvjgqBC2tA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230201_110354_54_2475_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.338Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkkwZXQ2MGZma0lTeXBvc05LTG1CWHVYKzl3T09sQmsvV3R4MnVSNHFIR0xPbElDUHJjbGxPUHhjcEpEd09QUXBsdm9mSnJUOVpRWktIMjBiZWJ5aXRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjFfNjdfMjQxYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2E4YzFiNjg3NmY1NmZhZWY2YjBhZDkyZTMwN2M5M2M5Yzg0ZDVhN2VmYzJiZDRhNzMxZTBiNGRiZDIwNzY5NzQ3OTg1ZTk4MzI1ZTYyOTcxMDljYzIzZmY3OWFjYmFkM2MxMjA3ZWY0ODFhOGIzNzVlZGY0MTNlYmY4Yzg0MzBmZjg5NDdiZTE1NGRiMzkzYTQ2ZTBlZDY5NjQxYzVkOTI4N2RkOGJkYmM4ZjdmNzRhNTRmYmMwOGIwZjEwNmFlOTczYzdjNjZiNGM1Yjk3MzRhNTYzNzlmZTAyMTQ4MGYzYWI3YjBlODMzMGU2ZWU2YjA0MjA1OTNkZGFlM2Q5ODBjZDk3MDY0NGY0MGM1NGYwYmU2NzEwMzE0ZjMzMTU1ZWY0NzZjMjhkOGI3YTE0NmEzNWU2ZDZlY2IwNzk1NzJmZWYwMTg4YjhlZTVmNDJkZjFhMjRmZDAzZGYwY2VkMGIwYjNkMTZiYmIyMDU1MmNkNTA0MWIzZjk5NThkMGE4M2E0OWEwMjZlOWRkNDgxMWUwZTk0MjVlZjk5ZDVlZWM1MWZjNjdhYjY2MDI2YzA0Mjc4YmJkZjU4NmE3MTA0M2IxNTZlNTNmMzY1ZmMzOWNmYmI2OWU2Njk5MDI3MjY0YTg1MTRiMjRiZTAwZDI3OWQ0Mjc0OGIxYTRhMzIwYWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.an6kygOyElzO_I1_M_roCLV6VcgsPRqlNZOBp7oIg4Zjgsc0Gc0Xc66-XUgnUFCr-oI-Wy2duyjayWpBf68yWg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103121_67_241b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.341Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlpWeUI2QVBPUHEweFFoT1NaMDI0T3dRQ1ZabTF0SFUrZU1rZE9oUjkreGlCQUY4azJOWVB1SEVVckl2MWtjTW0zYmVvb3kxMXBETVRsMmR2aXowWTBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjFfNjdfMjQxYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDc5OWEyNTM2ODU0ZmRlMjc3MDkxMjQ4Nzg4OTI1MGVmZjRlMWFlZmQyYjU3NDVlM2Y1YjkxMWMzNGViZmI5ZGMyYzVkMGQyOWQwYmE4NjhjNTA1Njc5MTM2OTgxMzBiM2JiMzU4YWY3NWJkZjA2NGJhYzQyZTMzNGRmNjRhNjJjNDcxZjIyYjJhMTBlMmQ3NTc1MDg2NmMwMTNhNDUwYTcxMmE2NWMyODhkNDIwYmRhZGVlODE1NzQxZDIyNWFhNzY3YmRjOGU1MmEyYTBiY2UwY2MyYWVjMTM1ZWE4YmUwMWFhYzBkNzQ5YmRkZGYzNjdlMzNlYzFjNzE3YTNkYTI3NTZkMGNlN2MxZDIxNGIzYzYzZDQzMzUxNGM5YTUxOTFjYzU3MTJmMzk4MmY2OGQ2YzcxYzRmYjhhOTUzNzhmZWVmYmM0MmY0ZTgyMWM0NGQ2M2RmNjA3ZWUxMjdmNzk4NjE1MDAzMTBhNzJjZmM5YWUwZWM1NmJlMDI1OTI2NTg3NjU3Y2RmM2EwYTk5MGJlY2NlNTliMTRjYzdjMWY0NjdmMDNkYzA2YTI4YjY2NzA1YzYzZWRlYjVhMDkyODFlM2ZmMmJhYzgzZWJiNjBjZGUyNjhjZDQzMjAwM2ZmOTFiZGQ3MDQ4MDNjNWZlNzcwZGQxMWNkY2RiYjZhOTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.W04Q0et1ozzdMFpb35hO2MDI3Zu3PUeHP17Rgd_l9RXIcNJaKQ5Cz8xifBl15HpmEvni3ElmIs9RpakwF3w12Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103121_67_241b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.344Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InUwM0w0MlZpWFFuRkdEYmRmbHI5NGxaUTBkRWsvODgvcjkxeVU2NW5kN1FMMmYvaE9sWEFRU1NET2dMTkdLVWJ3UWVHcEpoZHdVSTJ1aWs4YVVLeVZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjFfNjdfMjQxYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjE5NjA4NGZmYjk5Mjk3MjBhOGNhOTdiOTg4NDIyNWM5NjkyMDMyN2RlOTIwM2FiN2RlZWQwZWI3NTNhMzgzMzdlODY3ODIyMjVjODRjOTQzMWRkOTU1MmNjNDY2YjgyMjM2ZWI2N2EwZjFkZGMyZGIzYzIyZDI2MzI1YzM1ZmY1ZWQwNjMyYTAxM2M5MGE4YTk3MGU1ZjE2Nzk3YjVlOTVkMTlmOTI1NDcwODAzMDMzZDFhODM5MTcyNDY3NmZjZjVmMDExMzgzM2I5MDk4NDRhMjcyY2Q4NDE4ZmFkNzQ3MTExOGQxOTg2YjU0ZmNkYjk4ZDJlN2Q4OTRmN2QyNmNiYzgyZTc2MDBkYjIxZmY5ZjcyYzg1ZDRiYzA0ZDI0ODgyMWRhYjIyYjRlNThmYTVlMWUxMTAwM2VkYzk3NjlhMjJhYmI0N2IxMDVjMGFmZTIwMTNjYTk3Mjk5OTdmYjY1ZjVmNzMxOWE1NGIwYjg4ZjMzYTdhMmIyNGYwMWVjYjY1Y2Y4ZjE1Y2YyOTYxM2ZmMWYwNjQ4MDQ3NWQ2ZWUxZTYyZDVmNzdkYmJmZmQ4OWJjZDBkODk2Mjk0NzhiNzRmMzIwYzBkNDM1YzNmOGIxYjI5MmEyMDlkN2IwNzJlNWM3Mjk5NWI0MjBjYTk0MTEyZjBjOTRjYTcwMWQ0MGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HQQsD7-CzUmd7AQDMEZjrF6ZWVO5uUhynCFl8bwIKRwi8jt2h7Iiq5B1uHt7enlIrAx6TtJgB0fDR8xCClFhOg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103121_67_241b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.347Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFJaG5YUm54WlVoVnJBNWQ4ekJRQmFKOHhZWmk5TEtFa2c5VW42Wk92Q3lwcUgrNlZsM3BQV2h1MnViWTRObVdXNlZoVXo0Z0FlUityck5rdnJhOVBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxOF8xMDMxMjFfNjdfMjQxYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjQ2YmE0ZDJlMjEyMDFlZWE0Njc2YzI4ZjFiNjg3YjY3ZjY2MzBiMDNjMDBiNzVkYmFlOWQ3N2U1YTM5Zjk3NzE4ZDY3MDA2Nzc5NDU4MWE0Nzg2YjUwZWMwNDM5MTVhZGZmOTZjNTdiN2RjNzFkZjI2MzU5MGM2MjE2NDA4NjkzMThmZjZjMWM0MzkyZmQzMjM1ZjE3NWY3MWQ1ZGI4MDliMjYzNTllZjcyMjRmZWZiYzY3MjUzM2NhMWJhZmRiMzdlNzgzZDU2MGY3NDIzYjQzOTZhNGFiZTZhNWYxODA4M2E0YjQxMjc5OWRmZGEzMDFmOTQ5OGI4MjI4Y2YwMTRmZWVhZjBmZTAxYjQwODFkNTYyNmZmMDczYmExNTJmNDAwNGRmYTA5MjM0ZmY4YmEwMzMwMzIwYzYyYWZlMzdlZWFjNzcyYzA1MTFiMWJhMDA1NGM5ODdjMWVhMjkzMTkxNjA2ZmIyNGI0MmYxZWEyNDM3NTY4OWU2Mjc5MjVmMGY3NzU5N2E2NzlhYzc4NTIxMmQxODhmYWJjOTllOTRhN2U3OTgzNTEwZWE4MmU1MmNhNGJmYzg5ZjE1NDExNDU0NzQ3ZThiMjQ1NjAwNzA0NGViMzQzOGFhMWNmYjA4YjI0NTM2MzkyZmJmZDljN2EyMTQ3MDk3MWI2OWJlMjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UJM2M5D2bmH_ks-f7BF5f06YQ-g72JGSalFc1Zm7pQn3BwwqYEOdVd8G0dkJ-c-DG9lDit_DVA1ZnWPs568aOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_103121_67_241b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.350Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkNQZ2VzdnZITGhyNGNHM2NZQnZvR091SGdJZ0wwSEdsUUV6cVB6V0xlNWZYc2UyNlFvU2NwNCtIRVExS3poeVdmVkNKOWJ5MDJzZjN1SmdjQjlRdGRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNl8xMTAyNThfNzBfMjQ5OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDA3YmM4OWUzMGU2Y2VlMjQ3ODNjNTZlN2Y0MmVlNTFmM2EwMTFmYWUwMmI3NGIxODAwODA0ZTE2YWZhMTRkZThiYjY1OWUzNDNiNjllZTg3MThhNTI4ZmE1MzA3MjhiOTUxMDVjYWFkMThkNzEyNDEwNDFmYTVkZjNjMGIwZWJmNjA3NjUzMjczYjNiYTY0NTUwNWQyMjBiNDQ1NWU4ZWE1YzBjOWI0ZjU4MWY3ODdjNTRlMDEwY2EzYWY1ZWVlZGZhMmUzMWY5MjIyNWU3ZDVhNjkyMjlhMzRlZjY4YWUwZjA2NWRhYjg5NmM5NDViZTJhNzA2OWE4NGFiNWU0NWY0ZWZkYzI1MDJjNmIyMmUzYjU0ZDE3OGM5ZDcyMjlhYTEyMjFmOTExMGI0ZGNiMGM1NTI4NzlmZGNlZjM0NzYzMzIzYTQwYjJiYjk2YmE4ZmZmZGE2YThhMjAyNDAyNjcxNDExZTA1MTg1ZTJkZjkzZDA5NzFjMTUwYmZiNzY3MWU3NmM2Mzg4OTE0MzdjOWQwMDlhZjE4MWMwMzAxZjQ4ZmRiMjNhYTA4MzYwZjc3OTgyYzk0MTYwMjM3MTU1MDU5MDAyZmQxOWE5ZDY2ZDQ4NWU5NDlhMjYyZmJlMDdkODA1MzI0MGIzMTIyZWI1MjE3Mjg0MGZmZGQ2ZTAyMDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._MeRM-C9NnqfG0CydjuwHZIE7qQ3p1_cKQWSzTz3A_M0J4b4-M9ryyTSeLwwajxHwRCbJxRXVmf1M9BR8YPRfw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230116_110258_70_2499_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.353Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjZydnNrdmlmVzExOFhkZ0txTXNzWlEyZ3lnMC9JMFMvOWJxeTFHdHRRVmY3RXF4Ymg1cGZHQjBmZ3BsdG1XdTQwanFWMGR1elB5c0FnQU5WQW5wUnNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNl8xMTAyNThfNzBfMjQ5OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzgxMGU2NmI5MGJjMTgwMjM4MWQxMzA3NjkyOTgwMGYwYzUwM2JhMGY1NjMzNzcxYjJjMGIxNTE3NTgxNDBjMzhmZjgwNGY4ZjY5NjU1YzBhZDgzYzMzN2U5NjRjNTRmYWY1ODkzYTQ0ZWE3ZWRjOTdjYjNlMjE4NTZjNjM3YmE1YjRlYTBlZDBiMjM5NzcxMjNkYWI1NTk0Nzc4Y2M2NGM5MzIxZWMwYzE4ZGZjYTI4YmEzNGE0NzdhYjQzYWEwY2NkMmU3YWJjMDdjYjZkOWUxOWNmNDYxM2Q3Y2Y5Y2JiNzllNGNmYmM0OTVhN2ZmZTZhMzQ0MGQ1NTNjZDY0OWI2MDJhNDA2NDQxODVhYzM4N2Q1MWFiYzgwYzUwM2IwNzhkYTM4YWM3YzI2YjJjODI2YWExYTMwZmUyYTcwMDM3ZGQxYzNmOTZhM2IzYjc4OGY3MjRlMzQ5MmZjZTUzMWQ2NDEzNjU0ZTQ2MjEwZTVmYjUyY2U3Zjk0MTJkYjk3YTgzNjE0MTRkN2VlY2E3MzM1M2NlMjNkY2Q1ZWJhNjg3MWZkNjM1NTI4MzNhYWY1N2QzMDNhMzQ2MzdmYTIwMjdjOGJhNTkzY2VlZDU0ZWZlMGY1MjYwYWNjNDZhN2Y0NGY4NGE5MjNkYTJhZjQ5MzE0OGFiNDI1YzA1ZmMxMzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FL82OUeWBLdFiF0KSXGEOfvmbJTbU8l9xeZx0cwq_F6ULeJw9L8g1cxkT12Zlvlql397GmA_CCPKx4i8rUH2lw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230116_110258_70_2499_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.356Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkNWRGRlQ0tKREVZMysySWF6WnN0OTdaSTdSaXJUZzUwcTdPUHdXVkJCSGgxMWVmWnpkOFJKQ0tpQzU2UmQvV2hiZU9OeURQUWVLcDZtK1diaXJKK09RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNl8xMTAyNThfNzBfMjQ5OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OThjMDczNzZmNTA5ZDk5MTIyY2M3OGJmMWYwMTY0MDQwYTBjOWFlNmM4YjY1OGM4YjZmNjUwMjM2MWZmYzY5OWUxOWFkMzVlN2MwMWQ2YzI5N2MyYTI1Yjg4OGM3M2U2ZWVjMDI5MTVmNTEyNDJlZmUwNDEzNzFlNzZlMmExZmZmMDk3M2Y4ZjJiOTUyMzQzMGFhYTY2NmJjNTg4NmNkZmNjYzhmMjY0NmRlMmEwNWVhMTUzNTEyZGZlYmM4ODNmNGM0NDA2ZmEwMzMyMmEwZTdiMDYzNDQ4MjBhODhmNjlhNGFkMzNjZDU4MDZkNmQ1OTRhMWYyZjg5YzljNDc4OTRkNDUzZjRjYzA3MGQ2MzVjMjllM2U3MmY5NjMyM2Y1YTFmZTJiOWNlMWYyNzc2ZDNhZjNkZmEzNjcwOTEzMTZmNDY4NDE5NzYzZDU1Y2Y4NzcwZGRlMWRiYzgyZjU2YmFhYmQ0NjllOGY2NTIyOTEzY2IyZjI5NzVhOTAxM2UwZGZmNWZiZGI2MTUwNjcwNjQ4NmY3NzBkYjViZjY1YjM0YTFmYzE5NzhjZTMyMzBhY2ZjNmM5YjMyNTgxMDVhODYxOWMxZTczNmMwMDgzYjVlMmRmN2Y5NWIxMmExZGRmMzNhNzMwOTA5MmRhNTMyNTdhY2I0N2RiNmY5Njc5Y2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OeDBfbY7cpp9ADdNmjYgarDWyo5UqfCBRei7uWqK5Rj1H72Q2R3oCywPFJkddo2J8dBNpbMSIFek313YNGbHRw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230116_110258_70_2499_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.359Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Inh0bTZPSTJUT0dpZUkrbFFYeHdqWmhoVlFucGorUGRrNVhaQ2R2eWM1aXFWVGg4NEhadExXVDVpNHBQUm8wSE9VU1d1TkxmQ0Rtb0kxZTh0eWtOc2JBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNl8xMTAyNThfNzBfMjQ5OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTNmZGM4ZWZmM2FmZTY2MGVmYjFkYzkzYjQyZTg0ODFmNDNlZTRhZDY5MzQyMWEwNGI5NzhlNWQyYzEyNTNjMTYxZjljMjQwYjEzZjRhMGI0YjM4Y2Q1ODUxNjQ1YWQwMDc4OGZjMDRjMzdhMzVhMzU4NjhiOGU1OWQxYjM0ZmJhNjlkZTViZmU2Zjg4NGI0NmFjZWQ5NTBmMDllMGNjNDkzZmYyZjFiNzhlZjA1ZjUwMmVjZDE0YzZhNzFjMDllMWY0NjM4NDhkM2I1NTljNTk0MTVlOWIxZDQ0ODZhNTJmOGMwM2I1MWYxNjZiNTEyNWE1Y2VjZjU0MDdkYzRiOGFlNjQ5MGI3NGM5MzZhODZlMDM1ODY3MGVmYWE1NGQzYzk0MjNiYzBhYzMwYjM5MTNmZDcxNzM1YjMxZDRlOGE4Nzc5ZWY0OTk0MTAwN2VmOWM2OGM0Mjk3MGQzNWY2NmQxNTZmOWQyOTQ3NTZlZTg4ZDgyYmVjZTY4OGMzZmMzNWZhYjI3NmE4NmJhMmE0MzQ3NmJmMjI1NzFkZjJjNjQ5MDgxOWFjMmFhZmY0NzJiOGMxODI4ZDI2NjM2ZTI1NmEwMjBkZTRlN2JkYmUxYWM3ZWU0ZmI3OTRmNzAwYTBiYzRhNzI2ZDM5NDllYzk4YmZkYTQ5ZTNjZGRkYmNlNDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5UZBezRuBgJ0_CA0gVBIlaK_kON3K1eCT66kBB5UMJws8kEhdiGoBiULB4AmDu-NcUZhwC6hxs1Tr-nXeG6w2g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230116_110258_70_2499_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.362Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImhrTkJuZENDUnM3NytFWG1DQTlCeDdoSmovMlVHSE94eDZpbENGd1FhVGJsT0VIQVpPZUFuV0hMSUhZOXVkZWhlckd0MkRKWkVNNER1YnV4Wkh2Y1ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDMzMjFfMTJfMjQ1Nl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmY1NzVjMWNmMWNhZjg4NTk0NGUzM2RiOTEyN2ZlMmRkYzY0ZWI2YjQ0NzdhMDUwNGM5OWQ3MDAzOWE5ZGNiNGE0NjJkZmZhYmVjZGY2NjRjMGQ5NjM3MTE0YjFmYTJlZTFiYTZhNWU4M2RjYjE0NmYwY2I5MjJkZDU1OTZkZTlhZmUxMjkxNGVhODJiNzVhZjIwYWMyZDkxOTA4NmJjOGZlZjY5NDExMDQ5MDRiMzEzZDg5MjZmZWJiMTI0ZDY4YTExOGEyMjVhMjY1MjRlMmYwNDRmOTdhODViYWFjM2JiY2UwMTg5N2JmZDk0YjEzOTcyODAzYmQ0OGVkOGRkNzI3MzdkZjYwYmZlZjQ2NTVmNzIzZWNlMzY5MjMwNTUwMDkzZGFhMTk2MTdiOWMxMTE4YTgwMzhhMzlhYTZjOWVlNTNmMzNjYzk2YmFmZTdhMjFlNjg4MjcwNDQzYzlhYjYzNDA5M2I5N2VhMjkwNGVjZTQ3YjIwMGY2ODU4ZjAwYmJjNmQ5ZjNhMTUxYzA5YWQ2NzQ2OGUxYTAzNmYwMzY5MDZjMzM0MmE2OTQwYWI0NDQ0ZjY0N2FjZTY4NGJlZjBkMDQzMzc2ZTYwZDY5MjM1M2JiYzk2N2Y1Yjk2N2MxYzBmYjg5ZjFiMDAyMTEyYjY2YTBmM2VlNTExODdlODNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.i24k3P9R2IZVMmDYoYRTljpRxhB7Wih5xXe0VKXnMse9jvKdKKAoGgRwqw-IOmnDiU3P7D94KlfywE-1MP4LXw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_103321_12_2456_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.366Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imw5MHBZVExpYnZEdVprZFZDS2tSbW4waGlBU05LRE1vWjVCSEwvM2JJQlZ2MHJvZ2xSV1hUenEwNWRZSU1jMzJGS0tvMitibmxiVG55Z0tlbTVZUElnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDMzMjFfMTJfMjQ1Nl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTIzZTQyNGI5MjBlMTNlZjBlNWY2ODExMTRjYzk3MGYxZjlmY2Q2OGU5MWM2YjhhMTIwYmU3OTdiNThlN2Y1YjlmM2MxNjJmNWFhYWQxY2ZmNzlkYzNlOGJlMWRkZDkxODQ3YTA5MWJmMmQyMjY3N2Y4ZjJmYmU1ODE4MzY2MDMxY2UyMDJhM2IzNjI4NjI0MjEyNDczYTg2ZDA0OWI0NDZiNjQ5OWE5YmI1OGZiODI0YWQxNGQ3YjQ4YTRkZGI5NjMzYzdkYzIwODNkM2MwZGI2ZTg0MTUzNmQwMjU4NzJhNGMyZmM1ZTQ4ZWI5YWI4OTYxNmIzMTA1OTkzNzhiZThiZDQwYTNlMDJhZWZlOWRkZTU2MjBiNjJhNTcxZmYxZWZmY2Q0ZjljMjE2NTEzMWM2ZTI2NTJmYTJmZTcyNzljNGEyMzQ3ODdhMDA4YjE3NDg3MTIzZGQwMTRiNzkwMjliOThmYzFiYzc0MWNiY2FkZjI4NjNmYTZlNjEwZWZjZTkwYjg1NjM0OTBkNTQ1NGJhNzBlNmQxNjZiZWQ1Y2QwNzM3OTc5NmE5NThiOGIxYzEyYmQxMjNhOWZlYTA0OWM1ZTVhNGI0OWY4NzdhY2FkMmRhNTJhMmMzNmVjOGEwNTQ3MGFkZWMwYjhmMmI1ZDU1NmI3OTM1OWFhNjBkZWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eFi0XI7KVhH49WJHIuYQlZd3AxRYCU6pFhNiHSALKhc8iHEvgep0IBEKEPsj3nRnACiSBGvAT7Mal-MSQriNFw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_103321_12_2456_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.368Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRFMW8zUnBOR1hvbyt1OXZ1UGVQc3Q3UTkrQ25lZXBHQWF1V05IMWRiaU5IYWNJSmhxZ3ZHNmJBMnhrejZkOUltWVJDZlNnbHJQdGpzaWVjRTNLU2JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDMzMjFfMTJfMjQ1Nl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTE2OTAzNTVmYzI3Y2ViODVlNjcxYjVhMWE2ZGQ5YzUzZWFjNjhlNjBiN2FlZWIyODQyZGI5ZmQ4ZGU3OTU4NGMyNDk5N2ViNWJkMzgxYTVhYmUyYjQzMTJmYTIwOGY3OTRmNjgxYWFlYWNjM2UyN2M0Yjc1ZWRlMGJiODg1NmFjYzY3YjgwNjUzZDY2M2M4ZjM2OGMwNGRiODYzMzNjOGM3NDhkOWE0ZmZhMmNiZjE3YmM1MWM4NTM1MjhhMWI3MTgzNTg1N2FjZGY5MDgxYWRhZThlZWI0MDQwZTA1NTVhMmUwYzFkOTExYTY2OGM4ZGI5OTQ4ZDA2MGY3ZjU2ZGUzYjhhNzNlMjcxNWY5YjQxYTZlYzVkZDU0OWExNGJjZGMwZjBlZjY1OTUyYzc3ZWJjZjMwN2NkZmJlNzJmYTIwOGVhNzIzYjJmNTI2NGU2MTcyMmNhODYzYmIwOWY0MjBlZDg2YmM0NGQ5NGRhMzRhMjQ1MjkzNWZhZTFhNWMxMDU2NDM3OTY4NzA3MTQxOWM1ZjhiOWI0MWY0ODRlM2Q1M2Q3NzJkNWQ2MWZiY2U1ZWM3Y2JmZDg4NjJlNjExYTQxZWRlOGJkMjk1MjQ3Yjc0Mzc4YWQ4OTc2ZjY2ZGQ0Yzg4YWY0YTkxOTM2NzZjMWU5NTdhYTQyYTVmYzlkZjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ki7U5CCsIYfaxYYRYypoN5pYnLQJKIG6tc8unOUiiAprQXLv-8TF0U0Me7ojZKJYKeGeSOyy2kB8drOgTNoMdw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_103321_12_2456_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.372Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IndRVjdJZlhuYjlUK1pVOHplcU1xV0x3WllpYXNyQVZ1d2xuSDVvcU1Gcm5xQTNNcTd5Z0VxK2VjYWFFS3BPdXlCbm90QzlESWE0RTNUYnFEZzhkN2xnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDMzMjFfMTJfMjQ1Nl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzQ1M2ZhN2RhN2U5YzUzYTBlN2U0MjE0ZGNkNjNkMDc5OTNlODY4MjA1MDQwNGM4ZTZlY2FkOTZhMDc4Y2M2YjhiN2Q4NTViMjM4M2FhYzU0NTA1NDVhMWQ4YzhlYWMyYTNiYzdjMjYxOTk5MDBjNGM1YmY5ZjY0ZWQ4NGIyMTAzZTkyMjQ0Y2I5YTUzNThkODY0Nzc4ZGYzNGVjMDA1ZmZlNzZlZDAzZWEyNTkxNjJhNGQ2NDE0ZDA1ZDAxYWMwMWEzMmU1MjkwMmIxNDJiNmE1ZTBiYTk1N2E3M2NmOWMyNWY4MmM4YmZiYzI5ZGU2OGVmZTI5MjRiNjg0ZGYxZWRkODA4OTRkOGZhNWQ4ZDNmOWE4ZjFlNjI0NzMyOTZhMDc4Mjg2YmE0ZWRjYmM0OWZhZmFhNmEzMjJmNzkwYmFlM2M5NTM0MmQyYmYyMDdiOWRiNDgxMjRlZmJlODNhZTg1M2JjZGNmYTc1YThhNmI1ZWQzMDgzZjBkNTk3YTIyYTkwNmQ2ZGI3OTI3YzI2ZDc2NTEzMmJkMGE5YjY1M2E2NjBkMDA4NGYxOWFlZGNkMDQ5MGQ1N2E3ZDFlY2QzMzUzMmYxMDc4M2E0NDVmZDhhZDQ1ZmE4OWNjZDJkMGQ5YzUwMzEyZDhjNzAyN2RkZGM2MzI2NmQ3NmMyNWFmZDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tBGf1jjJo1T6lKppn-CUCZkRvUy5Xvnx-lVi8uM8v4cBPCOuxKAynATve4AoQJjXCRTKqR4MJtnA6kkQxUMugQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_103321_12_2456_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.375Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlQvYVgyNlZsam0wQ2ROcW02a0pIajE3Ykc2U0IvcG1leDEySzF1bHB0YmxnM1dCY2h5SExSdFJaRGdERXlXZldEQ3haQjI3ai9keC9ZMndlU2tPRFhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDNfMDZfMjQzOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2ZlODkxNDNmZGY4MGZmYWU2YWQyZTAxZGY5ZDZiMGRjYzQ4ZTUzNTkwMzFkMDllYTIyZGZlOTYzYWE4NjM5NzhiZjEyNDhlZjNkZWVmMDFkY2QxODFlMmYzZmY1ZDMxMGY1ZmQyZTU2ZTIwMDc5ZjJkMTZkMTE0NDlkYWQ1NGRhMjE0YzAxZDg4NTQwZDI3NDY3NzI0NzdmY2Y1MDQ5ZTVmYzJjYWI0N2ExMDhkMzExZGM3NjFmYzViZjdhMmRiNTU2MGYwOTZlNjIwNWI5ZGE0ZTU5Y2JmNWM5MWQxNjNlNjRiNWI0ZTg1YWQ0MDhmNGQyYzk5Mzc1NWM0ZjE3NzA4MDM2ODQ5ZWFlYzZmYmM3ZWVhYWE1NTNhMzZjMWUxZjcxYjZlYjE5NTZiZjFiMDM1OWQzYzA3NTdjNWI1MTc1NmE1ODllMDM4NWM2NjU1YzkxYTMwNzNlYTIxYmVmMDgzMGJlZjcyZTU1YzEyNGVjYWY0Y2ViZjIxMjNhMWFiMzBjMWM5ZGFjZDcyYWUzMjJiM2VlZWI5ZjI2MTk1OTMzMTIwZWRmZmY0NjExNzhmNzE3Y2ZiMjAyMGE4ZmNiYzU0Njk5M2E4ODllNjQxYTNhNTFhOTc5MTBkNTM4YjhmYjFkMTJmNjJmYmZjMmM5NmQzNGRhN2FjODFkZTUyNzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.j6zfBoVgdW7og1T1xvKDCIZohEQT_LZHPOj2VytamV2Sr6Om-ab1yj6o6jk8bO1nuH5w2EYGrT9wNoq7mc9Teg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103343_06_2439_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.378Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IldBRTdtNmIvR283WEYrSWlXYVkwTnBxNjh5U2VYZGlQOHNTbmZ0QWxUeXMvMEFOY2EvaFdFVzhYMld3WkR5b215Ym51dG5ncEY5VlNMaU5TQ2thNm9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDNfMDZfMjQzOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmNhOWQwMjI5ZjI1NzA4NGJmMDNlNzJkZmE4ODhjODExOWNhMTk1ZWQ2ZmNiNmMzZWViYmFhNTAzYTdmYTZhNTMxOWFlOWJmZTRkYjgxMDFlZTMxY2QxMmQ5MDIxZGUyMzY5MzkzYWZlMDkzMjE5MTdjYzczYzQwYmFjMDM2ZGRjYmU1OTM3MDgzNGYwYWY0ZTg5NGFjZGE3NTg1MDVhMmJlN2Q3OWU2MGFjOTNjODdiMDU0OTEyM2FjM2E3Njk1OGRhNzI0NWIwYWQ3M2I1ZjIwNWFjZjA4YTM3YmQ4OWE3MGY4ODYzMWM3YWUzMDZmOTFmMWI4MGNjZWMxNDdhMzUzYjU1ZDczOGJjMzI3NDdiNDA4YmFmMTA0ZGU5NWNiMDYwMGM4ZTYyOTM2YzA1ZjA2MTA2ZDFhOGRmM2RkNjc3YmE5YTJlZDcwNTk0ZWE4NmUwMjhmNWQ0ZWU2MjllYmVjZWU0YzQwOWIyZjc5MjdmYjhkOTI5ZDI4NjQyY2FiZTRkMzhhNjczYzJhNGVmNGZiYWYxYjQwMDJkZmQzMDhkNjIxM2RmNDAxNTYzNTg2Y2JhYTUyNGE5MGJiMjJlMTQ2Y2RmOGM0ZmQyMzk1ZDg0MTQ5ZWE3NjFjMGMxNGQ3MTRjZDUzYzM1OTRmMTM2MDdmYWIzM2ZhMWMzYmVkZWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.akZnEMYpKxxjGfLd8XAYzxFVvycRmQyAoUEgU8myZ5Ua4rMKc28HruHOUVghhr0CWmSQbBxbbiR1RQDgm-KVhQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103343_06_2439_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.380Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjRSbVZmU29jZ2pEMzVoTis1K0xZWXZPak5hNTU1enFTRGxTc0RyTkwrbFpQNXdjbjZlOWtwRWxOc1BFYjQ2Ynp0WHFwU3pueEJZTFphRHJXSjhIZWZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDNfMDZfMjQzOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTQzZTRkY2ExMDVjOTI4ZmIxMzA1ZDQxYTM5NTliNmNmOWJkYmYwNThiZjE3NGQ2YWFhNzk5NTYxZGIzZjk5MTVhMzdjMGUzMTc4Njc5NTgwMGFjOTgwNjc3NDYyZGIxYmM5MjVhMTFhYmM3ODFkNDYwZTEwZmZlODk5MjRmMzEzZGFmMDQ5NjRmOWM5YTMxZTZiYWJkYTM0NTI0YmQ1ODEzZTJhZTliNTIwNzQyM2Q5YjNhMDhmYzFlYjNjNjZjYTViMDU2MTBjYzMwYzI5ZGZmZjk1ZTAxZDY0ZTM2ODI0ZjczMjBhMTIyOWRhODQyMWNlYjE2M2RiOTFlMTJlOWFiZDVjM2ZlYTZlMzA0MjRmZWNhYWY2ZjMyZjY5MDM5ZDc1YTU1MDgzZmE4NTc5ZjY5MWVjYzA0Nzc5YTgyYjdlMmZkMWUyY2E3ZTdhYzkzMDgzZmM1MjJkYjVkYTI2MWFhYjJhMzNhM2ZjZmFlYzVhNzhkMmNjMmIzZTAzZjhmYmZmNzM5ZGE2YjRkNzU4ZjMwODNkZTE4NTcxOTIwYTVjZDUyOTE2ZDA0YzI3MjM1N2ViMzc0ZjUxYjg5YzMzNWI4N2UyOWMzZjJkYzUwYzMzYWNkMWM1NGIxY2M1NjBiOTY0ZTZjYmIwMzEwMjBiYjRjODdhNjAzMjljNGNiYmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6Ao4D0_i5Ex7hoYsub7kb3ENx_rmsAJGbtRpGHdovpIUMfi96R2EDWfFNbYylXLZwh6gigxCIHIwZXcL8H-1Dg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103343_06_2439_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.383Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imk1Rm5BYkJFT0MvOWtvSEsrUzhmOGwzbjk0VFdNOFRqVU9iWjArVkZONjJ5RHIzNWRMRUY1R2M0UzZZL2lIUGpyK1RvNjJCQ1Q4eHFQTjcwVWYzcFB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDNfMDZfMjQzOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDA1ZDIzYjAyZTFhMTVmMWFhYzRmOGZlNTgxZTBiN2YyMTQ1ZWNlOTAzNGQyMDFmNjZjN2U3MGMyY2EyMmU0YTA2MzI0OGM5NDU1N2MxZjBjZTU1ODNlN2I5NDg2OWQ1ZDZlMDQ1Yzk4MWQ4ZmFlZjkxNDI4M2Y5YzQxNWQzNzc5NzBhMzc0MzI1MjcxM2JmYjgwN2RjNjc4YTQyN2MwYWUwOTQ5OGRkMmQxOWRjMmRhZWQ3YjZjMjkyYmU4NDRlMzJiMzg1MDcxZGFjMmFjYWIxODc2NDg2NmRiNzA0ZGYwMDUxYzc4NDY1NDJlMmNlZmU5NTM2ZGYyZDcwZjc3OTk4MDljYThlMGQxYjQ3MjdmMDM3NzFiOTc3YWNmM2Y4NjFhZDhkNWQ3OGM1MDJkMGRlZWRiMWY2YTMyODRkOGUzZGYzYzMwNmEyYTVhZDk0YTYyZDc0MTdiZTE1MTQwMzNhZjBkZjc0Zjc1ZjJjODgyNTg1M2I4ZmMyNjdkMzFmYWI1OTI1Mzk4YTMwYWQ0MGMzZTUwMDJiZjI2M2U3MjVmMDE4Yzk2YjM0OTMwMTE1NjFhM2Q1ZDViY2EyYzQ4MWE5OTdmYjNiNzAwYWYxNjJhODFhMGQ3YzYxZTI4NWM1MWEzMjdmOGQ2MzQ5MzhjZDc5MzkyZWJjNzcyYzU5YWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MAjFO4uvSAO_6inP70Nkr9ki19fB-dwyy713gLHKFvsWNXXzcDgK2TEFw7KfRdHKa251H_e7Co77VhlH-6-xDQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103343_06_2439_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.386Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImkvZjFoSFhlU0NzdXVZMnZ2VTk1akpOSDU1cDI1TUs1U00vWVJKVTBpMExBQVM1RllwSmtLU0xwMWcrSmhxZ0liMG43cEx0OEpVVVlMQ2YwOVk2cm5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTZfOThfMjQ1M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmQxNzRhZmI2Njg4ZDBhNTUzZDJjMjcwNDUwN2NkMGQ3MDQwYTI5ZDgyN2FkZTkwZDRmNTNlYWEyZjM1YjVhY2JlZmFmOTU2N2ViYmM3ZmU2Y2MwYjgwM2EyMWMyOTJmZTEyOTRjMzI0ZDAzYTY0NWU4MmExMDE0MTdkODczNDcwMGUwODRiNGVkNmUxMWExMTk1YzI2NTY1MjBiYzlhN2E3OTk4OWM0YzY4OTY5YWFkZTBiZWM1NTcwMDQ5ZDhhY2UyNTMwZWJjZTNjMTIxZTQzNmY3ZjgyZGZmMjllZGRiZTQ1ZDg4NzJmOTM0ZDI5YzQ2YmQ0MzFhMjI1MTU5ZDJkNTE1OTYwMDE4Yjk1MTc4Mzk2Mzk3ZTg2MGU3NzBmNGI0ZmZhZjYzMzY1Zjk4NzcwYjg3ZjA4MzBhNDg2YTNjNGQ1MTAwNDdmOTg2MWNlMGM3NzJhNDBlZDY2OGQzZWFkNGZlMTdlMjAwY2YzYzZhZmU3ZjZkYjVhNTZjODQ4YmQyYjE5YTZlNGRjMmUyMDlmOWRhMDBkMDJkNzcxYWRjM2M2NTdmMTk3YmRiMDA5NGVlZmY3OWM5NDljMjhiNTMxMjAxNTZkYjE3ZjQ5YjI3NmMxYmI3NWY3NjA3YTg1MjE2ZWY1NzlkNWY0ZWM4MmEwOGVlZjUyYTczMGMxNWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hn59IMV0xSpdJcn7NIShDsK5gKL3z5UBcTTgN1So4twL0MxfX8zRBYYvbY7mFCBGfaIFZO0Jtl-uRvdX1eO33A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103656_98_2453_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.389Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ijg2MHJlUDQ3dzNtSmZxQUcvbEQxWHBvNi96RnRzK0FhZHQ4OXFpb1B2TmQwTjZ1T1p4UFBwSTlyS2h2eTRobm16NjE4VG5JR0ViQ2RPWFVMNGcwRnlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTZfOThfMjQ1M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQwODBjMTEyZTQ3NTFmMzE4MWI3ZTNkOGEwYzNkOTBhYzMyMzBiZmE3NmU1MDA4NWJlNjU3ODU3M2MyNmI0NmU4ZWMxZWYyMzY3OTYzNTc0NTdjZGMwNjI1ODMwOWNkODY1ZjM1NGMwNGI3ZGUxMGRlNTNhZDIzOTA5YWI3MmE2ZDFmNjIwYjdlYTAzOWMwODhmOTNkZTAzODUwMzk5Y2VhY2RhOTMzZmI4MTQ1ZDNiMzE3MmUxMmJiNzY5ZmI2ZDdkYWUxY2U0OGNjZWUzYTQwZTIxMDY0YTc3OTA5ZjI3OGVkNTVkNWJlM2QyZWQ4NDQ3ZDNiNGZiYzkyNWIwM2JjYzFmYzQ2NTMxOWIyN2Y5NjQ3MDZiNjg5ODQ3ODM3ZTU2Yjg2YzljYzU4NjRkNWJlOGQzNDI2YzMxZjI4YmM2ZjNkNDg5M2IyZGM0ZDc0YmE3MDUwZjhiOTg4NjYzZTU1MjM1MmNjZDI4MGRlYjNkMmM5MTBiYmQ4NWYyMWUwMjA5OGNkYmJhMjFkOTU0OWMyYTQ2MTgwYzQ5M2UwYWFiNWZmYTJmNDcyMzcyNmUwYzExZDI4NjA1N2MyYjEzYjY0YWU3MjU2NjY2MjZmYzdlZjk3MTJlNmMwZjYwZDc5MjBiYmI1N2Q1MzViYzZlOTFkYzAyYjk0ODUwNzY4MzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ynnyx8rAK227GTkMm-fZz-q29sj6qo_iomq86A16-q8eZm6znOpwqh4ZaFl2jHhqPnU6-NxsXSaPCuFBwU1ABw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103656_98_2453_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.392Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImRrWjhTQTFzbGNTd0plM3ZneTZZNWJwVk9qSWhydG51dXFZOTAxVndkTTI4RDNtOTFIUHM0K24vSlNweGhuUmIyTHR5b3JaUlNMaVRwSkxrSTh6d3JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTZfOThfMjQ1M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzg2YmMxNGQ2OWM5MjMzYTFkYzM0NzE3YzJlN2I2NGJhMDE2ZDFjNWE5YzQxYmM4YjRkNzU0YmIyM2RiOTJjN2I1YTBhOThmMTk2NGE3NzdkOTJkY2JhM2I1MzlmMTVlM2RjMWI1MDI5OGZhNDU4NmEzOWM0MzMwZWZjZGMxZjhhMDNkNDFkMGJjOGI4NzQ0NzdiMGNhYWY0OGFkODMyMTZhYWVjMjE2ODhkNTVlMDhhZDdlNDY3MjQ0MDhkMzI1ODUxMDUzY2M5ODExZjlkNThjOWJmYjI4MzQ2OTRhNzg5NWY5YTZhNGExMzMyMGE3ZDkyMDEwODlhYjM4NjAzZTM0Njc0YTk5ZjkwZTMwMjFjYzFiZDIyNzA3ZjcxN2NlN2VmODk4ZDUwNjJlYzliNmNiZjU2MjFlNTYwNjJlNTExNjMxZjBmYzg1OTg0ODU3ZTMzMTFhODQzNDU0MzZjYzVhYTJjZTgyMTlkMGEzZTA0YTVjNjZlMzJlY2ZkMDc3YTViMDE2ZDU3OGEyMWE1NjRmMTY3YmMwY2RjNjJhMjVhM2JkMzkyNWE5Y2Q0OWM5ZWIzMzI1ZGE4MDliZDQ4NGQ1NzMxNGVhODUzOTdiMDkwYWY3ZjhmYjE2ZmIzMTgyZTYyMjdmZjc3NmQzZDdlMDMyNTE2MjFiZGFiYTFjZTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HeMGlYizlbsftZxuozm0aiMIEghonWndSjkZcrZwCVaFMhfXuku-YoYAUmdLl3WPkPvhbl7XBX-LwEyrmYrpVg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103656_98_2453_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.394Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imp2UGdxTWUvdG0wMG5ubzFiVjlmYml6SmpWVCs4VTU4RVpQSXpnQk9PQUdRcC92WmxZelJwVDV4R2lETi8rWE5qOUFUUkIrQUZsMmYyTzR4bFJDcGlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTZfOThfMjQ1M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODNlMzJmZTlmZTc4MjNlZmVkNDk0YTJhZmE5NTA5NDNlNWM4OTAzODRkMDE5MWQxODhlYWIzNWE3YWYxMjNjYjg0OGIxYjRlMTliZTc2M2VmMDZkNzJiMzU1M2RkNjY5NjQ1MjIwYzQwN2QwM2YwY2M5ZGM0NjMyYTAwYTQ2MzY4ZDllYmI0NWZmMGYwODM4ZmY0YWM3MzA4ZjMxYjIzMzJkMWU5ODZkY2IxYzMyMzZkNzgzN2IwNjhkNGMwYjZhMmE5ZGU5NzM0ZDk1NGMyOWE4ZDllNjM2YWJlZThhMDY1YWU3NTZkNWMyMWQ5MjE2MGI4MzEyZWQ4YzQ5NTU4MDIxZGVjN2NkYTYzNzhiZTA3NGNjMDhlZjRiZDBlYWE3NTBlNDA5NjM3ZGM3MmViOGU2ODVlZmEyMzdmY2RlZjkzM2ZlM2FiNmJlNWQ4YThhOWRjODc2NWRjYjJjNWNhZmZmOTFlMjk2YTA1ZGZkOGZkYTM5NWVkZDY4YTM1ZjNmOGEzMDJmODJiNzFmZTczMzI0MTIxOTEzZDNmMGYxYzYzYzAxNTRkOGZiNmI5NGQ1YmJlM2M2YmI5MzE1YTdiMjM0MWRlNWIyMmRiOTc3YzRmOGZlOWU3NWNlYWZjNDEwYzNhODdlMGJkZTFkMjk0ZjUyMjkzZjcyNzllOTE4NDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RjTxylHrRyE3Mt8bhHFBcWj8JNXLb4QmB9ECX6_-hdLhKZBIw_PEeY7k08o3RuMvrCIc-j3zuoJH0HPiq2Y1iw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103656_98_2453_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.397Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImR4YkVqNXpodkNtTmhzNG1XM0cwcXRZcTBYR2QyL1pPY3JzUWszL0tsaStwQloxVTlUK1Rnb0FpRXA1UDJ1VWd4dDc0dFlkQ3ZQMzc2RHpiWWhvdExnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTBfNjVfMjRjM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzlkYjgyMjE1YjhjOWJiMzVlMWQ0Njg0MTIxZTEwOTYzZGQwNjgxYWRmZGM2ODQzNDRkN2QxYzAyMjQ5YjcyMDJmMzJmNGQ3YjE2YmQwNThhYzlmZjdkMDNmNDhlMmJhYjQ1Y2Y2YWRiYTM3NzY1NzU5N2EzYjE5ODJhMTJlMDBjZjYwM2Q5ZjhlYmNlOWEzNTU5ZTdiOTFjYmJiOTIzYmJkZmFhMmZhMThlNGM1OTRiMzhlOTc4NDc2NjM1NjAzODc2YThlZjQ5OWJjMTAwYTY4MGRiYmJmMGQ4ZGQxMGE0ZWFjZDVjMjU2ZDkyYmQ4YzFjZDY0ZTljMDc0NDJlYzk2NDBkOTk2MGY5ODA2YWI5YzdhYjc5YTcxZjc5NjNhZWIwMTZhZmJmMThjNWM5MThlMDI3NzYzYTgxNDAyYmYwYmZhMzViYTQ2ZTRjZDliYjNmMmVlNzJmODA3NzNlNGY3NGE5ZTgxODAxYWJkMjk2ZjQ5ODI0YzRhZTkyNmNhOThlZGRkN2NiYmZkMDUxZGEyYjkyYzg0NjM1YmI3NzAxMGE0MDQzMDIyY2UxODJmYjA5MjA0ZTllNzY2YWUyOGEzMjUzNGVhYjJhZTFlMmY4MDZiYWQyZTU2NDZhZGRmOTM5YTExN2M2NjI4NzMyMmFmNDkxYzhhYTE4ZmMwMTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UMH16hKFmfEocoTAfN0IeE_oNpYipwZSA8oXtHCFY2bnVasSrRrUvEWWQwO3f7XEsZnCWGFk1l7I3aIshGblQg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102950_65_24c3_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.399Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii90U1JtNW5NWXcrb3B2Tm1taytBUXdKY280UmU3MnB0VlJHclFoZHlkWGNSTUFHa1NrZ283MmhOZDY3Q2pVdHY4RnA3UzJTWnJDRjhhanFLSFAybnVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTBfNjVfMjRjM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2EwNzEzOWU1ODhhNDMwYmFiMzQzNDY1NGFiYTQyODY2YzI0NTA1MjQ0MmZlYzM4OTlkMDQwMzM2YzUzNmEyOTc0NTAwMGQ1ZGUyZDdiNGY5ZGNlMWZjYjEzMmU3MTRmZDI3NzZmNmI3OGFhNGYwZjJmMDBlNjQ5ZjI2YjVhMmYwYTQ4YmNmNzNjN2RjNzU0NWYwZmNlNDdlN2MyMDc2MzY0NDUxY2QwYzk0MTBmMmM3MTIzZTEwZjgyMmQ0YmFiNTZkMmY5ZGY2ZjBiNjBjY2I3YjlhZWU2Y2U3YTBmMTE2MWQ3YTRmOTliYmIzNTI1MmRiYjRkNzMxMmY4MzMyNTlhNjM1Nzg3NWE4ZWQ1NzliNDEyYTZmOTBmNGM3M2MzZjY1NGY2YWJiZGFkMjk3ZmRmNGVhMjkzNDI1NmE1MDAxM2NkZTBhNzFkY2RiNGQyZDEyZDFiNjY0Y2I2ZjRkOTgyZTBhZGIxM2ZmMDM1Yjc4Yjc2YzJhZjYwZTlhNTAyZDZmODE5YjA3YzAxNDZlM2RkZTRmOTExNWFkNTdmNzE0MGU5ZTZlYmM4NjdlMmJlYzMyM2M3NGQ1YjllODc1NjRkMjZiMTE5MTk3NjI5Yjg5MGVhNmQzYmYwYzlhMDJmN2Y5M2VjYWIyOThkYTZiYWRmMTBlZTM4ZDBhMmIyNWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5H6WGebXA5f1p9ijk1MX4LZaIs-cA6OTI1c_h23XUWX4YygnYTU215QlCZIzkDTZxnUYZHSQ1U116-GM7HI30Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102950_65_24c3_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.402Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlBVUlNubVlrUTJqT3FjN2lMRkdhaW50S2xTYkptQk9WdU9LQ2ZsNUJ2WFFlbndrZFlTWnh2RlIzbllvK09OOEN3RHdmbis3S09ONlJLbndWaC9OUGt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTBfNjVfMjRjM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDdjN2Y1NDBiYzQ3ZGVkZWYyNTVjZmI4ZGNlZGE5MTNlNjk3MTEwODczYmUzNzdlZmEzN2I5OWFmMzM0NjljODk3NTZjNjJlMmU4NzcwNTVmZGM5MjQ2YjdiYzZjNGY3YzMxNDc0ZWMwNGFkY2NhYWM3ZDVjOGQ5Mzk4ZWZmOWRmZjA4ZDY1NDhkN2RjNWNmOWM0NWQyZTU2YjI5NzBkNWIxNzg3ZDU1ZTk0MzUyM2Q4ZmZjZjEwZWM4NGFhMzgyMDJhNzJhNzc2YjZmY2VjOGRjMTA3ZWYwZTk2MDQwOTIyZTA0MmI3NDM3NTVmOTY4ZWU4MGY5OGRkMmQxNDdiMWEwN2ZhNGM5ZjhlNjE5ZGI2YjU1NjNhMmViZDBlOTU0ZTdkZTA1NGNjZTNjNmNiNGMzNmUwMTRmMDM2ZDQwZmU2NzU5Y2Q1MTQ2YWFmMTE2NWE4MjA3NTNmMmIyMTVhMDQ4ZTM0OTkxMmVmMmRiNDczM2M2M2U5MWI0MTQ5YzU2ZjFiODVkZWZkYzQ3MGZhOTAwYWRmZmQxNjZjNWM1YWJjMGYzNmM4MzE1MGI5ZWRjZDM4YmQzNjI2Zjg0NzRiMTc1ZWFjMzA4NTM3ZDEzYzAzYzE1Zjk5YzY1YTRlMjc5ZWViNDQzYmNmNzBlYzdjOGJlYjM1MDg3ZWZjMDEyOWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4S3Np_O6itQTsT98LvkkiKcg3xHERU6SUgIUxL-cJ5qA5hKWK5BJBswslguNjlOgtCrVXFNuNGgpa204foW35A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102950_65_24c3_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.405Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkJFM2s1SEZrclpST09nb1c2WTRlTnJ2QXBtTytsN0dSNncvNExFem1WODZ3NXgvOGhacUx5Zmh0Um9Zbnk4TlY1UUNiSXp0UGRlRGxrQnprNjZ4ZEVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwM18xMDI5NTBfNjVfMjRjM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzE4NmZkMGU0M2E2MDZlODAyNTBhNmNiYzI1ODk4YzIxMjViMjQ5MTdiZTk2ZDU2MTA2OTJlZDI1OGZmM2I5YWIyNTY3OGE4ZmFhYTIxYzA3MGFlNzU0NjY1Y2ZlY2FhY2M2MzU2MWRjZTY1MjFlOTVmYTUwNmNhY2ZkMGQ1MzFkZjM3ZDVjNzczNGZlYzllNjg4NzA5NzJkMzkwNmYxOGYxZjc4ZGFhYmQwMDRmZDIyZmZhOTA4MDlhZDdhMzMzMmQ2MDk3ODk4OGE3NmY3YTc0YTA5YzE3YTNjZjg3YTAxMDFmNzA0ZDA0NzZjMTQ1YjkwNDdjMzdiYzg2YWQ0MTA0Mjg5NDUzZWRkOTNkOTkwYzdjOTQ1NzM4NDQyNGM2YjlkZmQ4MGVmNTcwYjdjYjcxZTY1MjI3OWIxYzY1NGI5MzhlMDc5MzA1YTJlM2JhNDU0ODNmODhlNTkwN2Y4YTJhNDg2N2VmYTg1YzlkNjM4NmE0MmE3ZWEwODdiNjE0ZjlmYTY0ZWFlYjEwZmU0NDg4YzcyNjg4Y2IxMzU0MTVmNTdhODgxMjkzNjkwMDIzYzgwMWJmM2E5ZjlmMDc0M2FmNTRhYTcxYWJiZGUwODdmM2YwMmM0M2EwNWFjMTYzNDVlZmQxNjdmZjk0ZDE3MGQ3NzVlM2IzMDEzOWRiMWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cvGt6KYJJ9ckOV-qO1f0Lk8AspygWBzmJsx6PZvayfvAhfC7PC7hj2cOyy8Q6avS-HxDjtNecBjuxHUlfTBbeA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230603_102950_65_24c3_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.408Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlYwVW1idjRRdC9nQ2c1K21sTlR1N3poa2tjVDhkcmtDUkp2Sy9FS2NaSkF1b3FNemZjYnlWTVkxa1VzS01PRm1SOStFeFdra25PeitMOUVoR1pOTS9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMTEyMTJfOTdfMjQ3M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTQ4MWY0ODU2YTI0YzMzYTQzOTJlMDhiZWY3Njc4YWVkMzA0Y2QwNzg4NDY0YzM2ZGEyZmY2MzExYTQ5MDZiOGJjNGFjYWRjMWJhNTc3MWM1MmQzMWI5M2M3ZmRiMDExOWI5MGFmMWZkNmEwN2IzYzk4MGMwMTIwN2RlNTJmZGJiMjVmYjBjOWY5NTQxOGQ4MmZmYjAxMzZiZmQzNmQ3YTJiZjM4NDEyYTRjYTEzZWE3YThkZDUzZTk5YmU3YThlODA2ODZjYmIzM2YzMTlmNmFkYTg3NzY4ZTdkOTcxMjVhMjJmZmUwNWQwZDI0N2FjM2Y4NzE3NTAxNGQ3MTRjNzVmZTdmYTBmMDk0MDQ5ZWE2MzA0NjRkOGFlNjJiMTJlYmU0ZTQ0MjlhMzY4ODMwZDM3ZGRiMWJmODhmZTk3NGU4MGNlNTRiOTc5M2QyMmRjZWMwOGQ5MjU5NzNhZDdjZWY5ZTA0Y2FkZTc0N2EyNGExOGFhMjIxOTFiMTQ0YWY0N2Y4NmQxN2MxNTA4ZTk1NGY1YzA5MWQwNmE2MmZjMTIzNmEwMGIyOTBiYmViYmE2MDQ3MWM3OWEzYTZiODNlMWJlM2VhOTIwZmM1MjQyYzJhYTczMjNjMTcxY2RkODQ0YTE3ZDQ1NGExNDliNjY0ZWU0OGM4MjVlMzYxM2FmZjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1ENCmFbRuKESURzc5EG4kg9RmnSBz7hxb8qnm-N3PwHmR99ch9s8vPvcoFVKx4hxSSOoV5UMddI-Bv0hbUZzrA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_111212_97_2473_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.411Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBmWHdYNmkrajA4YlBlbkpLQTZuMi9OaU5uNkVtMXg0VnZheHpYUHhQYlphSTRjTmZpMVJMeitLRDFPZEZWOVcwS1M1VDdwc1h0c2IxdUg4T3VpUElRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMTEyMTJfOTdfMjQ3M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjgwZjY3NWZjMzZkY2Y3NzRkZDUyMzAzYmY2ZDk2Mjg3MmUzMjE1ZWMxZWVlODFiMTFhZmNlYjE3NGFjOGVmNmY1M2VjMzhkZTY2NmExYzZjZDg4OWEzZWE5Mjc0OWYzMTU2ZDg0NzFlY2MxZmQ5MDUxOWI1ODA5MzhkZGMyZTg2ZTZjNjllZDc1ZTBlMGU2Y2U2YmI2Y2YxODU3MjM3NzRmYTc1MGU2ZDI0MTkwODVmNGI5N2FkYjAyOWE4NzRkYmMyMWM1MDMzOGM0ZDlhMGY4ZjgxMGQ2NDk2NmQ2M2NhNDM0MmY0YTgyOTMwNWZiZDg2MDBmY2YwNjU2Mjk5NTk0ZDNmZjM5ZjBjNjMyM2E2YmRkNDA4N2Y2MWU2NTFiM2U2MWQxN2NhYjIwMTEzYzZlYzFiNDc2ZWY0NWMwNDY1YjhjZmZjZDY2NDk5MzMyODBiMjA2ZGZlZmM2NjE4NjdkYTJmMTcwOTA1ZDA0NGMwN2JlZDk5MzQ0ZWM0ZWMyZTEwNzI2NjdjNjlhOTBhNjc0NDJjMjNhYmU0ZDA4MDIzOGFmMDc0ODJlNjFjY2Y3NzEzYWY0Y2Q3ZTdjYmIxYWQ3NTFhNTk3YjAyOWU0Nzg5YTI0OGQ3ZjllNzMyMmMwMDgzNjE5YTc5ODczMDZlZmExNDJjZDJmZjM4Yzc2YjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qgFjhF-JSuj27n6KFIqzHDCNF4cVyqoLDvn7Wbf4ZhCXlQbywO8Y4Tt8bwCgDBXeH5utz_97eyRjflbSpqk6yw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_111212_97_2473_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.414Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlNzaWswTk14WXBObHBXQ250RHM2OXNSV3hjZGxCZ1IveHBrYmljWE5odXU2b2htMUFYT1BDNmdEcmFzbzNaaVdQM2JnMld2cGVMbzRQWVBvL3MyR1pnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMTEyMTJfOTdfMjQ3M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzQyY2QzMGUyY2FjMDM5MWU1M2U1ZGZiMGIyYzc1MWY4MGRjMzI3N2M2ZjkwMDRiMTBkYjU4MzdhMmY1NjVlZWM0ZjE3MmJiMWIwYmY4ZmY1YjhkMjczYTgxY2U3NDQ1ZmI1MGIwM2UwMTJiNGI0ZDA5YmE4MDhiNTJiNDFjNGIwZTE0NzAyY2M2Y2RlYjVjOWRmMjM3NjUzMmVjMDlkYTA5NzRiZmFjMmVlZGZlZWRhMjk4ZmM1NWU3ZjVhMjQyMTc0MDQ4YjYzMGQ2YjdmNmU1MzA0N2VjYWFlODkzODY2YTExZjFmZmQ0Zjc5NTI5ZDVhNWZiNDRlNWI4NmYzMjE1ZmRiOWNmNjQ0OTNiMGNiODhiYTA3Yjg2NTMwYTM2NjU0ZGVlNmQ2YzYyZjZlZWY3ZWFjZTk0MDY5MTUxMjFjZDkxNDlhZjZhZTkyMGY1NjNjOTg2OGY0MGFlZDFkNmNjM2I1MjA5NTQwY2IwMDQyOGUyNWVkYjFhZTdlY2ZlNTJkNTIyYTQ3ZGRhNTdiMTdkMzBhN2RlMWJkNzQ2YzM4OTkyZjVjYzk2YjEzNTlhYTlkODQ4MDRlZjE3NWFjMTk2ZjRjYzlhNjRhZGNjZTU5ZGE0YWFiYTAxMjg5Y2JhYTU0YWZiZWZlYTVmMDFiNzI1MDlmMzY0NjE4MDA3MWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cxF9JPSOE1rL6ugV9NqI7FxMe532rri2Zd2VLmVb_huP63Ud5L8mgYahR4jUcMBeN_EYaMMONBll9MkEF8baMA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_111212_97_2473_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.416Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxSQ3dEcmdjWUxvM1h4elJyMzE2T0NHalN6OElwRllNWnNoY3psTktQQVN4aU90UGNQUlBaRkxvMHdUK0w2ckt2TUxjejUzdXoxVVYrMWtCM0UwTEl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTExMV8xMTEyMTJfOTdfMjQ3M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWQzODhlYTIwOWEwZjZiODk3NWUwNzgwYzdiMDEzMzE0M2MwZjUxN2I2MTI3MjYyMWM0NGYzOTZiNDkxOGY5ZjkyZTA3ZWMwYTQxMzhjMGUzZDgwZTI4MGVmOWQ5NzdkOGFmMjc4YzQwMjNlNWRjM2Y0NGQzYzc3ZDMxZmE3MjFjY2I1OGJlMmUxNmJhMjFmNjcxYTgwM2YzNTM5NWM2OTQ3ODkxMTUzZjdlMmFhODM2NzVlNjhhOWYwYmQxYjU0ZDQ4NjVlYTE0NTM5ODVmZjExYWExOTI1ZjIyMTgzNmJiZTQ5MDdlMTdjMjIzNzY1NmEyZTZjOTc2YjQwZWZjMjliMWNiMmVkZjdkZjA2NjhkMjAyYjkxN2U3ZGQ2OTdmNzNiMzYwNjU4YTE5ZWM2OTQ3ODM4ZGI5NGZkNDlhZDBmM2VjMzExYmRkYWRiYzIxZTE5M2FlZWZmNjhiNjhiNTc3MGZjM2QzZDdkNTBiOTkxMzlmMDM5ZmYxNGFkNjQyMjJjMDNlNDIyYjJmNzY3Y2Y4NjkxMmZjNmM1OTZlYTIxZTU2Y2EwYzI4MzhhZWIzZjQwNjJlODRmZTU1OTI5MGJjNDJmOTBmYmZhM2E5M2NkMTBmNDdjZjNkNzJlYjU1MTkzOWQyN2E1YjlmYTAzYjI2NzM4MjdjZTlmN2Q4Y2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4dS-6zJ_gQwOWHTiOo7E6bQN8pW7cNYI-z9hUfF-aoV8vASz82wVeYCnozfxUl9ovoct9yzG7GHcOjxqNuu8Gg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231111_111212_97_2473_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.419Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImEzUzlwL3JkcXBQOExtMFRta0M5bTRVdUovNlNuTVhtb3Uwdm1qUnc5VzJyVWh6alhVMXllRDJydm5WTlk0QmRSNGtzRnJ0M0lEaDBhQnVqdzROQU9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjdfNjVfMjQyZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTk2NmMzNjg1ZDI4YmExMmEyNGY1YTE5ZjQ3ZGEzMjFlNDlmOGFiYmZlZTA3NDZlZTExMDc5ZWZiNmIwOTA0MDQyYjZhOWMwZWRkMmIwYWI1YWQyMTZhNTY0NTY2ZDRhZTI2OTU0YWE2NGQ5OGU1ZGNjYmJmOTQyOTFlNjM0MTAzMGJjNDZkOGMyZjc0NWU3NDlkODUyMTFjZTA2YmMwODk4NDkzMTU4ZDRmZTYzZDQ2YWUxZDAwNWQzZGE1YTU3NzBkOGQzMmVjYTEyZTExY2ViODI4ZTAyM2M2NGFiNWFlZDU0OGJjOTAwNmVkZmM2Mjk0ODA0MDI2OGNlMWM1ZmI0YTRjOWU5ZDk1YzRiMWQ2MDBkMTNiNzhjZWQwMTg1ZDNhNTQ0NWFlYmRjNmE3NTJkM2JlYzA0ODY0NmUyYTA3OTA0YmQxZDc5MGMwYTRhNDNkNTNmNThhODQ2ZDFhNTUzYTJlZTk5Mjk0OTBhYTM1M2M5NTU0N2MwM2JiNWVmZjVhZWE4MGExZjZkMTE0MDc4MzM0MGNjMzRjNzgwYjM1ZDFmZTVjZjRhOTVlYjNkMTBhNmVmODE1NTZmMmJlZjg0YTE4Y2M4ODAyOWQ1ODRiMWZhNTU1NjA3NjQ4MDgxNTc0M2UyMjI5NDY1YTViZDVjMmE0YjIxMjE3Nzc3NDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XQ9ehcP_xrohSpRArJS8W9qR-2dS5K_1LIIhKCvEJuhZdzm5Z8YCNIye38JqCeaerwRfwO7xc9YX5BN_rIc3UQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102727_65_242d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.421Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikg5VHZpaXo2WUgrWnp2MVpwbTZ1SkNWUHh0S08zZUFxTlpiczY2eUUwb2s4RldZZkNYK3NrUjc1QzhuNFdjVWcxVyt2QXd0RUJRa1JObnFpRDJGZktnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjdfNjVfMjQyZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjk3NTQ2NzkxNWJhOTUzMTQ3ZWIyM2Q1OTdlM2U3M2I4OTFjOTBkMzI5MTNiNTYxZWY3YWNlOTQ4MDJiMmU1OTIzNDRkYTlmNDdlYmVkMjQ1MWQwYTY5MDhmZDEwNjExM2UwNmJmN2NlNTAxZjc4ZjNkMDdkYTY5YTJjMTU3NjhlYTNkZmM2ODU5ZjI1MDdlYzVkNjg5OGExODlhMzI4NjE2ZjIxZmY5NDdhNmQxNmJiMzM0Y2VjODAyNTYyMDJmMTVjM2JhNWJiMjA5ZTEyOTc0YTg2ZTU5ODBlNzRlNGM0ZmE4NWViNGIwMDEwOGVhNGZlNWU4ZTEwNDE0ZDFiNjRhYWFlNWMwOWE1MjBmMmM0MDg4ZjE2MTYyOGIyNjg1OWQyYWRhN2NiNDIyZTEyOTIwMGMyMGM4YTVmYzJiMWE3MDllOTY4MjkxMWU0MjEyNDhjMjFlOTVmMDkyNmM3YjY5ZmQ4ODA5YzhmNTNjOThmNjZkZWRhMjRkMTBiMzk2ZmI4ZmNjZDRiYzE5ZGZkNzE0Njc0OTExY2Y3NGQ0Mzg2Y2I5ZjBmZWUwN2Q3ZTRhODkzMzI0ODgzZjJhOTViZjk3YTYzODNmZmE3MTg4ZDcwNmYxN2IyNDNiZGNiZTY1MmQ0Zjk2NmY5OTY3M2U0ODY3MjEyOWFmNWMxYjc5MTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bvRJoPTWgVKroTcaVQjrNoaA8m_Xoq2JHdusXB_eKMS-vo0cYHoVfh4f7h9jSWZzgeQCaOkL2xwL3DOyItYMrQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102727_65_242d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.424Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjE2WHIwVTBkb0pZTFN5WWRLRVVNRzNwWjF6VmR4aVpvTzlTU2s5M2dXdHBrTmhvN0h3MC9DMzh6K2tpQmZZUzVVcWZ2cTFQUzgwUzEyT0xVc21sZ3ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjdfNjVfMjQyZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjM5YjA2ZTA1ZTVmNmMyMzU4NDIxYzg0OGI0OWVmMmFiOWNhN2FmNWYxZTVjYWRjZDYzZTU4Y2IwODllYzc3OWRkYThkOTNlYjQzNzgxMTA0ZTgzNmY4ZjliZjdiMGUzZjViYTBlMjU5NGFlNzQ5ZTEwOTY4ZDJmYWEzNDI0ZDVmNDA1MGI0ODE1YTUyY2I2MDMyZGVmMzJkNzViYzNhMTlkMGJhMzkyYzliYjYzMmY5Yjk3NWNkMmFjYTE5NTkyN2MwN2I2Yjc5ZmUwNTM1OWUyOTU0NDMyN2FmM2QwNDE2MGUxYjM0MjZlMjEwYzFjMGE5N2E3MzVkN2E2MzMzYjBmOWM5Mjk2NmM1NGM1MWI3MTg4OTQ4OTJiYmE1Y2RmMWM2ODQyZmRjOTI3MmY5MjFhNGZiYjFmOTUwOWExNzRjMWExNjY5YWY1YmI2ZTViZjcxYmJlZWU0Yzk4Y2U4MDAwYmI0Zjc0M2JmNzEyZDY4Y2U1ZjAyNjcyYWY1MzkxOWRjNDZhNzBiNmQ5MWM4Y2JhODBlM2M2NDJjNTNlYzg5OWVlZWZjZDZlNWIxNTBhMGFlMTU0M2E0YzljMjIxMDBjMWJhNDdlN2ZiMjk3YjMzMTg4ODAwNDZkY2I0OWZkYmVhZDE2ODEwZDM4YWQyMmExMjY0NjE3OTY1ZjVkM2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sCgl4ga7hNfUI4fR9LNdWICDrOGZ_ImEJS7Mlt0EGOlSSqxID-I7lcZ5UIK657EInEVpUP4gRwslR2OCuKIFaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102727_65_242d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.427Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkFtTDl5Ylcyc2FhTVpGd01LcEU0QmJDbS9UVHdUK0JrU014R0d3Ni9OQjBhb29GUXNHZVVIaGtUdlM2ek9wZmE5Z01CRmpMME0zdE5wa1pUOTRRczJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjdfNjVfMjQyZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzA4MzBlNTBjNWEzZmQ1MzU4ZGUyM2RmZGYzMDViZmRkZGQyYjA4ODBiMGJhYjIwNTY1ZTY5YTNlNjQyYzhmMmEwYjdiMzdmZDQxYzEzZGIzMjUxZmU4ODk5NmVjMDI5NDYxMDk3ZjI3YmVkYzhmM2E2MjYzN2Y2ODA0NjQxMzdiOTNlYzhkNWFiMmMxNTBjNmVhMmZiMjE4ZDQ3MjhkMDc3NzM4ZDkyNWRhZTYxY2EwMTBjZDJjMGJjMjYxNDFkZTEzZTBmNDk5NGRjYTA5NTFjZGI5YzA4MWI0ZGQ4OWJkNDhmMDQxZjZiN2MzYTViNzJlZjU2NmRhYmI2MjQwOTAwZGM1OGZjZjU1NjNkODM2ZmI3YjJmOGJhYzQ3MDFlZjQ3NzBiMTg4YTY2ZGYzZTQxZGU5YjkxNGYxOTQyYTA1OTQ5YWVmNjQ0NzE1ZDU3MDdhMWUzZmI1Njg5YmNkNDViMzExOGRlYTI0ZWI3ZmNjNzhkZjk1ZmZmMTBkMjc1MDY3YTQ1MmU5MTcyMDI4MzllMmUyODQzYTQ2ZmExODQ5MDFlODdmZGI1MTRjMTdhMjc4ZDNmYmZhNzAyNzFmYTAwODQwOTQ2MzNmZDRkZGYxMDk1MzFiY2Y4MzZkM2RjYTI2MzI0YmVkOWM0OWZjMDI2M2QzMzdlMmZlYTRjMWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FIU-s9BdcaQmZshlCPy7UlH1kL4quDGZV9-im91QmvnZH2UnRt_GV307E9MaUckBltO7UpgJ9aCx6Fjduim1xg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102727_65_242d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.430Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImhBdjluUVcvN1FNYllqQXdrbW41eUoyVHZWSHk2KzdBMjlkTG0yRUZXMTdGcEtIcWtvdVdOUE1neEkwS1NpMHVSSzd6UU5BaTYyR0dCN0daNVFkNTdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzBfNzhfMjI2Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGJhYTVjNjM2NmU3MjMzZjk1MDI5YjNmYmFlY2M1NWUwNmM0MWFhY2Q3NzMyNTJiYmRkZDE0MzZmZjM2ZTgwNGMwMTQwMzA0YjE1N2YyMDVlYWNiMTU5ZWE5ZGRkOTZjNTAyMGZiZjY5ODk5N2M1YzM2NjY5OTNhOWQ1M2NjYTk1ZTJiNmNkZjcxZWUyZWM1Yjk1MzMxMjU2NzQ4MDJmOTYzMDk3MzY5NDY5YTdmZDJmNGNkZDFlZjM0MTBkZDg5MTk5NTQxNDE1NmRlMDY5NTFhODk5ZGVkZjQyYzJmNDc0ZGE2MzExNDUwYWRkZWMzOTkzZGMwODdhNTYzZWY1ZWJiM2RkZTE0NTkxMjFjODMxZTRiZGE3MWQwNzI2NTc3ODNlZWNhNzA1ODFiZDgwZjY0NjlkYjNmMDZkYzIxOTg1NjFmNWM5MDNjMjA1NGQwYWM2YTVkMjM0ZGJkNDRjYmE3MjUxMDZjMzUyMjUyNWI1NWY3OTlhMDU3ZjUwOTI1YWMxYWM3NTgwMWMxN2E4MDVjNzg4YmY4YTY3NjE4NWE5OGZmYWIyZDIwZDJlYjM1MjdjMDhmM2Y2ODdhOGIwZDY2MGMyOTQ2OGY0M2UyNDcwZmMxNzZiNjI1NGIwZjU3MDlhYWQ0Mjg0MmExMmZmZGQ0YWM3YzgxZmI0ZWU4YjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UwTF2TNte09UOXwwivDAR5WbBxK-3B1SfKj3c9o3i4BtItCpvsQMK0qSIwvcBZTPFMq0LBe_S6e6uBMVLwWGOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105030_78_2262_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.433Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ijk1TEtkOGtLMDYxb3Q3QmVaNzVFYnRCcU9DZ2w3dDZ4NzQzZWFSQmtLaHcwVHpRVzBBMXB6T2NDWFNsNXdpbmVqc0dUWHRodEJFUDY2RG0zcVR1N3ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzBfNzhfMjI2Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM2ZDNiNDVmZDc1OWM2OGRjYmJmODFmZDYzYTVlNzQxZDY0MzE0NmZkMGE1NDQxMjdmOGVhNGMzMWRlMDc3NTI0OGEwMjg2MGE1ZTBiNDA1OTNmYzUxMjZmMDUzMjEwMTU1YjFkMmM2YzYzYTQwYzlhYjU1NWVjZTVmZWRkYTg0NjU4YjA5ODA5ZWM4ZWUzMDM3MDk1ZjIzNGRkODliNmU3ZWUyOGYyN2Q1ZDUwNDZkYWYwNzI1NzA2NzY4OTU5Mzc1NjI2YjM1MmZhY2U1ZDVjMzQwNjhiNjIwY2NhN2I2ZjBmMDMyZTA0Nzc3OTIwYWIyNzljNDZmM2QyMDhmOWE1NGY1ZGI2YzNmNGQ2ZDJkOTJjNTY0YjczYjAxZjMzODUxMWIyMmNiYTJhODIxODM3MDc0ZmI0NGEyMmUwZmZiNDk0NjAxZDM0MjA4ODY5ZTI0YjgxNzJjODc2MTg4ZGY4YTk1MDAwZThmZmI5ZTllMTI1MDBkMzFhYWM1YWJkZjBjM2E5MTg0NTUwZGViOGEzNDY4OWFkMWYzYzI2OThhNzVmOGYwODJiNDEyMTg3NTFmOTc3ZWQ4MTM2OTZiOTM1NWQwMmQ4NDRiYzk3MGUwNDhkMzk0MzA1MzMxOGQ3OWExODUzYzE0MDcxODkzNTU5ZjlhYzY3NzFlMDc0MzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GWAU_e6jqKBTlyR0tTfnkHMGrPkGPqNzDfDfn_mI76h4Ah6fJN1Rhs17_sJwnpUSiduKYkmoe4aurcOSrHoytw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105030_78_2262_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.435Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikg2VkJ3WGJXRVVVaXVaWkhRVDRQRUZURm1kdkwxRUdzeUFLaVlMRFdTOWVXUURoQU92M0VnbWpRMnAyQk1KblVsNFhwSGJsM3V1K3lWY0xXVUxsM2dRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzBfNzhfMjI2Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTU5M2VkODNjZjU3YWQxMjUxYjdjNWY4ZmY3MjMzMGU5YTkzOWQ5YjVlZTkyMTBlMzE3MTE4ODBhZjg1ODQyMGI4ZGI0MzI4ZjEyZDE0MmE0OTA2ZGY1MGFlNGMyNGUwYTRkNGZhZTI3MDUyNDNhYWExMTBlNTMyMWM4YjM5ZWQzNTA3ZDZmOTAxYzMyNWUwNzE3MmViMjRlNzMzZWIxNjVkMWMzZDRiYzQzY2NlZmQyYzEyN2EwNzE5NTY3NDI5MTkxY2FmNjc4MDYwZDg1MjE3NjFjZTMzNDg1NzFkMTE0OTFhNjk4ZDlmN2JiMTM3MWMyMWY0MWJiZDE4Y2Q5ZDYxM2VkNGRhNTM0OTI1Y2JmNmFhZmU2YTA1Y2M5MGY0Nzk3MDY4NGQ0NzJlN2U2ZGY1MTMwNzUxOGI5OGM5N2I0MzVmMWYyMTc3MTA5NmEzOTlkY2JmODU1ZWRiYTZlY2EzMjA0OGU5MzU2NjY3OTY4Nzg4ZTU2ODllNGI3YTgxYjNhMzM0NGM1ZTYxMTM5NzFlNzIzM2FkYWVjYmVjOWUzYzEyZWM3M2M2ODU1YjdlYzg2OWE1ODVlNDc1NGY0MDYwYjUyN2JjZGFlYzg4MGViZWFiY2JmMWIwYWVmYmVlMTk3N2UxMzk4ZWJhNjAwY2UxOTI3ZjEyNTMyMjhmZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aUko6KYlCiqXjciGNfQsttLXGftzc-fP9SMShZANtHFXHREDwiwmdFKG4YGEWRYzzHBA5fSyudz1lwpyUj2rdg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105030_78_2262_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.438Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkJnaHNBM0VObTNwL3hYMjBOWGlKYkJocnlLSEE5aE5YSnBZMVBYaEpra0E2VHUyM0R2U2trYkp2akRxRndrOStXU3RrRUFaTmpIS1JrTFBJQmxKMGV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzBfNzhfMjI2Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODZmNmM0MjdmMTRhZDk1YTg4NzQxZWRlNDM3MzYzNzZkZGY5ODRkNjhhNmVmNTA5MjI2YzQyYTlhZDIwMDI3OTgwYmMyMjM5MzRhZjhjYmEwYjQ4ZGM3YjJmOTJiMmMzMGRmYjdjZTBkN2FlY2JhMGM4MmU5NjhmYzU2YzMzNzNmYzdkMmU0ZjA1ODhkOGIwNTM2YWE2OGVhNzQ1OGYxNTA3NzIxOWU0MTVhMTM0YjAxM2FiZjRlODc2YmNhODZmOTExYzk4ZTk5MjEzMmVlNzM0ZDM5ZWYwODIwYTcyMzYxMWU3MDlkYTAzOWY3MGM5MjFmZmNmNjc5N2U4ZDM0NzdlMjgxZGJhMGVjOTUxZTZkYjY3N2FkNmFjOTZjNTQ4NjY4NWY1MDgzZDQ3ZjFjMTliYTBhYjZlODdiYjQyZmZkYzVkY2VlY2MzZWEzOWY0ZjAwOTc1MzU2MDk5MzEyNzFkN2Y1YzEwNjU2Yjg3YzkxNWIxZjVhYTU1OTI5MTVjYjg1OGRhYWRmNTk3OGZlODAzYzAzYjg1ZjdiZmQ3ZTllOWU2NjkxNGUxYWZiY2I0NDQzNTc0ZTdlMDA3ZTllMzUxMDNjYzBhNzYyZDA3YTQ3YjM1NTU3OGFmMjExZjgxMWJhZDBkY2QzNWM2NTZmYzI2NzFjYzFlN2VlZjAxN2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.t7K-DicQ_iXG2ycO9Cc1rKSQXIGKLG4sGvkKYb4uqDgYQQ-eGCKsyFOTwnx1H1N0zCPqRbAhraG2eV1FWgS-DA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105030_78_2262_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.440Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlJtMTFsLzdoNGdTWVB0dnpyUDhISjA2UVZXa3hZN0pjZWFzYjdPejM3SGFleGVuSjZTVVppWEwwZ29VekdvR2RGNmhWOWR2a3B6WU1mYnF2R3k2bXBRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjJfMDJfMjQyM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjFiMTg3YWIyZDIwZjQwOGI3M2QxMWM1MTliNjIzZmU1NmZlMWNmYWE3ZjkyYzY1NjRkYzA4NDdmZGFhYTljMTY2NzEzYjUyMmRhMDE5ZmNiNjk4NWNiMDg4NWIwZWRmZmUyYWE0OWE4Mjg2ZjA3ZTJhODU1ZjBmMGMzY2QxZGM3ODVjNDViMzMwYzdkZjUyMDdmYjFhOGU0ZjU0MGEwZmQxNmQ3NTMxNTYzNWUzYWQ2YjUwMzA1ZWIxNmMyODkxMTg2MTQ1OGE4NzI4NDJlOWI4Njg4ZjQ3OWQ2OTQyMjY0NWE0OTkwMzAzMWEwYzcxZWMyMzRiOTU5M2QwMGExMWQwYTIzMDc5MzdhMzVjZWEzZjU3ZDRlMTI1NWVmMGE0ZjliMTk0NDM5MWEzYTYxN2M1ZWI3Njc5ODZkN2Y4NGU0ZWQ0NTMwNzA4YTE4NTQ1MjE5OTcyNmY0OGI0MTMwNDBmN2UxMTNhODI1OTE0YTI3OTdhMWEwYWZkOTAwZDM2OTE5MWYyMDAwMjgwODRkNzYwODk3MTQ4NDBkM2JiNmUxYjBhNjgxOGVmOGYyNjhkZjM1ZDJmMDgzMjgwNDRjNmI5ZTEyYWVjNWEzNGUyNzA4ZTU1NjVkOTczY2Q1YjI5N2UwZjc2ZWNhMmZkNDFhYTBkMmI0NjY2OGI4NmNmMGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ECOXtCnH_qOXzRdw5AgXliAME1N1DiujvWgwBsFGKjnkf987U0aIWCGkOUThF4piiKeNjHTZEcol9s2Yrau1HA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102922_02_2423_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.443Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im1wTXN0Z05OeWJzVWlUYXNLNUFRaVdzYk9rWnQ2NTdkcG1IMWRna0k2QkM3YzJPYmhjZC92aE0ybnVuMnBIeXVYbjhXVTVOdHdwc0FPcmdNY25VOGtnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjJfMDJfMjQyM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDU4YjIwZjZlNDRiZjA1NTZhNGYwOThmM2I5MDg2NjA4NzBhYThkYzIzMjI0MGQ2OTkzNzc1MTNmMTQwMzUzYzMwMzdlMDI1ZDJjNTY0NGIxMTQzZjkzNDVjYjU4YjBkMDFkYjliOGUwMTQzNTdiNjQ1OTIyNTBmMWIzNGMxZTE1OTAxYTRhOTIxOTBkMjY1M2U1NDQ5NTVmNmEzMDM4NjhmYjE4YWY2MDQ3ZDI5OTc0ZGM3NTBjOWYyMTA3MTRmM2Y3MTNiMmVhYzFlYTRkMWU4Y2FmNDU4N2Y1NTk3Yzk4NDk2YmYzOTlmMTIwYjAwNzM0MWRkOGIxMTBiNjdkMjMwYWFhNjU0ZjBkNmY0MDMwZmE3ZDNjM2VmNzQ3OGUyNGIyNzE5NmZkNTNkOTRkNjJkMTYwODdhNDI2MmI4Y2ZjM2IxZGFjNjliMzFkZDliYjFjOGFjZjg3NzBjMjgxNmU1ZGI2OTAzMTMwYjY2YjQyMGYzZThmNGI1ZmVjMTA0MTc5NDAxMGNjNGEzNWM3ODU0ZjZiYWJmYzQ1MjUxODc3YTRmMDU0OTJlOTFjMGFmN2Q0YmFiYTllOTliMzkyOWFlMzgyOTUzNTg3ZmNlNmJjZDdhMjQ4ODk1ZDAwNTYyOWZiZjM5ZDQyNWU4ODgyYjI3Y2M4ZmUxNDU3YTk5YmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AIPbUPzaN8rEfHpXvUIjvFNe5R13duHF0xrfh9ynbcvbvqjHBwRQXfOKAjNtTSy1JVsYZ3UCgNiKIPuajTgmgg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102922_02_2423_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.445Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjhNalhwcEVmS1p5VWdtZHVBUjUvb2F2djdhT0Y3ZDBBL1NXNW1NcVFZWmF2c2pGWm9BcUtHNEhkak5XdVJnYkVjOVpkMS9wcXBlV3lXTU11YzBKeXZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjJfMDJfMjQyM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjk0MzliNGM4ZTVmNzU5YmJiZDgzN2U1MDhhNThhNGEyMDQzOGJmOGM4MjViNDZjZWFiYzVlZTcyZGYwN2EzNDc4ODYxMjI3NDdmNmEwMDZhODJjNDdlYTNhZTU4MjE0NjlkNDE5Y2JlY2Y1MGUxM2JkMzBhOWFkMzBiMTM1MTRhZTI3N2U0ZDQ1ZDIzNzlmZGQwZGM0OWJlNWMxNDQxOGUyMmY1MDU4MmUwNTE5MGM4MzA5OWY1ZjdlODE4MTE0Mjk5ZDg3OWI4MjVmM2NmMDZiY2FmOTA5MDg4MzcyNmE1ODU1MTdhYTIyMDRjZTVmNDJkNTE3ZGM1NjRlYmU3YWIzZDAwODdlOWI2Yjk0ZTY2NDE0NDVlNjYwNDRiY2Y4YmY0NTFhOTVjOGQwNWQ1OThmZDJjYTA5OGRjNDQ2N2JiYTY5YTYzYWZhODczMTM1YzQwNzU3MzE5MTA3ODZjOTUyNWUyOTI4ZDg4OTBhMWI4ODMzYjA0MjNmNzU2MzYzNGMzOGQzOGMxYjliNzYxN2I5MGY4NmIzZjNhMDFjYjBmNGQyMjAyYjRlMjhhNGNhM2FkMTEzYTIwMzNhNDQ0MmZlNTBiODYyOTIwNTAwNTU3ZWY4ZDJlZmQ3YjAxYzE1N2VjYjBhNGE0M2IzM2RlZTk4Y2YwNWE0ODBhNjg4MmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ReLjzjKycyzcyh535803svjvUmXrXHz7oSNOVfpks18JwV3KHLk0N4qpBgDd4D1E6XF-U21GOQ2XrEtx7Kwyqg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102922_02_2423_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.448Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikh1OGdFNCsxeWVSRXhXSWt1VVhadWtydFV3bUxOK0sxakxLaHpwVVVjL3lsMXl1blBtNzRsSDRseWJqRC9sdHNubSs5SDc2QkdHa0w3bGVXdWRpS2JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjJfMDJfMjQyM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTUxNWUyNGViMWFiNmYxNWY5ZTRlNzM1Y2M3OGM0ODYxYjVjODZlMjI3YmY4Yzk5MjRiNzA5NmMxYzllZGY1ODM0OWJjMmM2MDMwMzMwNzU2ZmE5ODg1MDRkNzU4OTM4ZDgzNzkzMWM2YjBhNDhiMGRlMTE0ZTUwMmQxNzlhMjNmZGQ5YWQ1NWRlZmExMDhiNjM3YWVjOWEyYzNlMzQ1NjNhY2Q4YTcwZDg3OTM2YzU4YmM2YzMzYTM3MmQ0NGZkMWQzYTYxNzRkYWFhYWNhNjEzODE5OGZlNjZhZGU2MWMwNmViZmJkNzA2M2NhMjc0YmNmNmVjYzgxODcxODg3YTczMWFmM2QwNzMzZWNmYmJjMjM3MTlkOTgxYjcyODdlYTQ2MmNjOGM3YjMzZjgxNWM2ZTRiZGQ2YjczNDY0OWY4NTE3NTkyMGRjZjZjNTlhZTgyNGFiZWFiNzZhOWJiMWM4Mjc2YzhmMmYzNGQwODdmNmU2MDdmMTZjMGVjMWFiMzEwNTJkMDI0MzE0MDQ0ZTFjNDM3NjUwMTZkMjQ1MTU2OWZmYjNkYmViZjAwNTAyOTFjYmZkZDZjYTAxZGVmYWNmMTI1Y2NiNTUxYzI2ZTA3MGJjMzdmYzc1YjUyMjBhZjY3YjhiZTg0YzMyMjc5YjFmNWE3YjcwNDJjNTExM2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4LlGHGkIFU7ABAMgvAj6Cnj3Mt9nN3n7SsaqIwHBrJp9a9dWsVJ0FlQQFIK0gs12_rtl47ZPLB0nE0VzQimGjw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102922_02_2423_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.451Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IldzMzJHdGlDVDdySjVSc2ExTFF0a0JDQS9zT3hQSmV0WkhlNHh0T0JxL1pqR0ttbDA4TTlaSG9RTGhTcTc1S0xsaXFzdWRlQzNDczA1bHlpMVA5bzNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMDMwMzlfNDBfMjQ0OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTJhYWMzN2U2ZTQ0MDIzMDQ1YWI5MzExZThkNWUwM2I5YzA1YmNmNTY4YzE4NzFmMjM5OTdmMzBjOWJhNWUxZmE1NTYyNWEwYjNlOGVlZjNmOTZiZjdkNWY4Nzk1ZmU4OThkNzJmYmY5M2YyMzg4MTQ4ZjgyYWYyNmYxMmM4YWQwZWNmNWM0MWJjNjcwNGJkZjBlZDQxYzU2ZGY0NzdkZjAyOGU2YzRlN2RkZDA5N2U2MTMyYzM5NWRkNDlmMzU2ZDMyYmJlNWM3MTcyMWRjYWRmOTAzNjgzNTI1MWI2NDEwMDA4MTE2NmNhMTA5YTRjOWVmNDllMmQ5YjhkOGQ0OTRhYjNiNzlmZGQ3MTVjMGYwZTVlNTc1YTE3NTdlMmE0OTYzMWMzNTBjMGRkMGZjZjk4ZTVjZjYwYzA1Y2RiMzNjYzk4Y2Q3YWUxNjIxYmVlY2NhZWIwMTkyNjkwZmY2YzZhMzdmNDBlOTVjYmM0YWQ5NWU1ZDQyZmIxNjcwOGY2NWY2MDY1NTg4MmU5N2Y2MGMzOTgzOTRkMDdiNjAxMWUzOTU4NTFlOTc5YjgxM2E3M2ZmZWYwMDkxOGQwMTI2YmZlMDVhYjc5ODA3Y2Q2ZmQzMTczMzZlODcxYmIxODhjNmNhYjU2N2ZkYjYxMTVjNDVlZTE2M2QzNjJjMTM4MmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1qwwaWUYL2Rn6BMJ4w_2MvCcp5HSji9Bzh2U__ogKEcWfTbkVdY1HHMdYn_B4qxbZ0j_OteKuef7eFLkimwc7w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_103039_40_2448_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.453Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtwZUoyT3pXUjNScy9aTnhWdG4yMHdnRWZRVFRScUtkeUdNdlJDNy9JODhBV1QrMXlZR3FpU0xLTXZwTi8rSzhCYzlYWDlEUHl1L0R6OGFZbElLeHF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMDMwMzlfNDBfMjQ0OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE4ODBhMGI5MzFiNWJiODcyZTczNmRlNjJhYzkwZTJlMjQ1OGZlZjAxOTljZDMxYjFlYTY2YTQ2OTYzNTM2N2Y1ZjM4M2RiMDZjZTYzMWU3MjE1MjlmMjBjZGQ1Mjc1MzUyMmVhMDdmYTE1MDJjN2ZiNjYwOTg2MGNmZWU3NGNmZWRjNWYwMWYyZDk4YjRmZjhhZWZjM2Q4MDllZmRhZmZmODBkOGIxNDhkYTBkMjNjNzkzZmQ0NDA5MWViYTlhYzdiYmQ3MzdiZGFjYmUyMzI2ZTgyOWUzZGU4MjZlOGRlMTE4YTgyOTNmNjBmM2UxZTZlZTRjMDkzNjJjOGU0YWE2NjhmOTUzMzg5NmVhZDJhM2EyZjI1YWU3ZGM5YWU0MmY4OWRiZGVmNzdlMmI0ODMyODg4MWZkYzIzNTYyYWZlNzNkNGFlMjk1NWI1ZDAxMjY5NTJiY2RmN2U3NzFiZGZmNWQ4NjRjZWFmYmUwNDc2NzQ0Zjg1Yjc1MmQxMDg2MjhhZTZjMThlM2YwNjE4MDNlMTM4MmY0ODViZGYwMTQwZWFmN2NjMzk2OTg5OGRjMzMyZTk5NDdmOGE4N2U5YjAyMTA4NmExYzAzNmNlYjIxZmQ4MGQyYzUyMzk4ZDM2ZDQ2OTc4YTIxZjc2NzYxMDhhNjhjMmJlZmZlZjQzN2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Dwa4AWDWGdItKq6pLQG1th43dMX9A0xBUDB-6PgCxJhlE2mluMMio4OqCI5zX1uQMM4zxh2cTOgjDrliccF16g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_103039_40_2448_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.456Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImRXWXNwYU1Xdmg2c0xvQmlNREpoMEtIQXpCclBZTHduSW50OTRMMnN6WklGZlU0eDZqVFd3QlJieFVZODJMcWNIcWR1OUpiYk5ZTDQ1bHVzRzU1Q0VBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMDMwMzlfNDBfMjQ0OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzExMGI5NmIxZGU4ZGI5MjkzY2Q4MjQ0NTlmZGUyMzFmODZjMTZhZGNmZjJlYzAwNmY5ZDljMjFiM2E1ZTg2ZDllODczODM0NjljNjAxZTJjYmExMTRkYjViYTU1OTc3NzIzN2M3NWYwM2M2YWE4OTZiNzQxYzJkZThlMTMwMTdkMWQ2MjVkNzZlNTFhZWE0OTc2MGZmMjFmYTg1MDAyMzRiNDczNGZiYTBhN2RjYmJmYmJjOWY3NzIwY2VhMWM1MWY1NGMyZDU1YTdjOWE3ZjllMjlhZWNmMzViYmI5ZjIzMTlmOWQ1YzVjNjEzMTRjYTdjMzc2MTIwZjdjM2JmOWQ0NTBlZjE4Y2ZiNTcyZGU3ZmJiMWI1NDhiN2Q4M2NiZmVjNjE5ZDY5MTA5ZjA0YTM5MGFjOGIwZjZkZDYzMDExNDZhODY2MmU4M2ExYWFkZTVmOWYwMjBiYWMxYTMwMTNmODk2ZmNiY2Q5NDZhZmYxN2NjMzFjMjFjZWMxZGNiNmIyNjgyZDU1N2JmYzQ5NzcwNzRkZmUwNGY2MGE3OTBjYjQ2ODU5YmMzYjZkZGY3MmY2Y2FhMmVhYWFjMjBmZDZmZDIxNTBmZmNlNDk1NTQ1NjZmMGFhYzQwZDlkNWM1ODc1ODdlMmY2MDY3MzcwMDBlNzE5ZGRhNDYzNzMyMDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CbQsZ7OKyD8fqj3pZkVBDCSXT50vD0BGsqoe8wkAZFXhQxRHupXXdeJ7IoMzPH0r140r0LcuXgii49sFGceIKw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_103039_40_2448_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.459Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjB1dVVnZ0R4QmdSQloxdjlmNnJJQU1TS2hJYkpmL2xlMnFjcGgwQUM4N2VRYUpQRHNwVzlWVUxOdFlaTnZzaEJlZ2ZUNmRVYjU2M2JuLy9TU0FJRjZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMDMwMzlfNDBfMjQ0OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmMxMmIwMjA4YTdkZmRmZmJmZGQ2YTY4NDM0YjdkMTY3N2RhZWY4Y2Y0YTAxYzU4OWJjYjhkNDU0ZjE1MjYxYjE2NjJhMWI1MDYwOTkwZTA4MzlmY2VhYmQ1MzRlMGI1MzI0Mjg0YWQxNjE5N2UxMDM3YTY1OTRkZjlmYjhhZDM4NzdhYTRhMDU3ZTYzMTc2NTM0NzY5NDkwMjk2ODUzNzFmODczZjEyNDRmNDI0OTVhMWM1MzJiYTdlMGYxNDM0MzM1NjY5MjM0ZDg1NWFlNjcwNGE4NDAyMzA0Y2UxYWZhOGI4ZjNlM2VmYmI2NGRmMThkODBlYWI2Y2NlZGFkZGM0Yjc5M2E3NWI1ZGNhOWMxMDM0ZGU3ZjFhMjExNWFkMGIyYjMxYzEzMjIwNjZiN2I0Y2M4ZTU5NjAyZmQwMmM0YmE4ODI0M2Y2M2YzMTFlYmFiZjkzYTdhZGVkYTAzMjY3MzE5YzkyOTliN2JhZTUyMTRlZWNiNDlmMjE1YmU0NGYzZGY3OGQ0NTg5NTlmZWEyODQ0YTViNWQ5ZjE1NjBlMTY2NGY5N2YwMDI4YzVmYzNlMWEwNThlNzJmZGI1YmY1OGNmOTc1ODM5OGEzNTIyYmRmODVjZDVmZTVmOTdkYzE2NTI2MzQzOWIxNjVkODcyM2U4NGIwMmQ4NzBjMjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BA8JMcLkcCp59HH0G7wVVJX9a8QuzxASkyOCG_kPlbw3ndCfUfAr0zzJOwFyrJfw0V6wvyd5nVmTq-LotnJnuQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_103039_40_2448_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.461Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBrYzcrL3BoS09XOW5TK1FZZTJkeU5CdEhTQkxGQW9uRmtJc3RBTmRTakxaUjkxYmJoVjY0WmRkNWh5cTlLT2ZyMEdsZ3NHTVRDWWJFWVdaY1c4YWRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMDMyMDFfODhfMjQxNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTJhMjhkZTc2MzhiOGM0OWMxNzBlNGY2MWM3M2IxYjVkMDA2ODg0MDhmZDUwM2JhM2M2MzNjZDM5NzFlY2VkMDMwNzBjOTI5YjIwNWYwOTY2ZjA4NzllNzRhMjk3YzM1MGNlOTMyYTZiMDhiOGU4NjVkZWZjM2EwYTEzZGRmOTI4MzU5NmNlNzQyODFmZjYwZTZmOTE1YTNiMTNjZmJlZGYyNTNhNGViNjMwZDU3YzRjNzRkZDI0MDYwYzEzNDJkZjYxZjcyZWMxZjQ2ZThlZmU0ZThlYzcyNWMwZjY4ZWRjNDFlNTU0MWQxNjFjZTg4YzU4MjFmNGQ0YmRlMGE3ODAxNWY4OWM5N2YyYmM4OTcwMzBhZjkyOGNiMGIyZGI3MGY1YjBkY2U3NjAxMzZkMGRjZmYxYjE2MzhlOWE2N2QzZGU3Mjg2YzhiOTA5M2YyYTQ3OTA4ZGIyMmY2MTk0NDIxYTZmZTY0YmM5YzczNTE5NzJkZTg4ZmU5NzA1ZWRmMzljMDkyYWQyMmU0MzJjNzhjNTM1ZWVjNzczZmIzNjk2OWRlODgwMjk3ZDgwNzY0OGI4ODMxYjhlODM4ZTQyYTNkYjIzZDg0N2M2Y2NlZmFhNmUwZTUwNDc3ZTE1Yjc3ZWU4MWFhOGIwZGVlMzA4OGM2YzBmYjhmZWE0OWY1MTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OI3fzM7uW-d6qbW06qh6uVKgSDeFDpXxXdc6QFt-6_eqKHOCtImwj64D31vhq_djjmanzu5peT2kjWz2KgnH9g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_103201_88_2415_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.465Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjBWcW8yTTF5OVpxZTNxZDMxc2RuV2ttYXMzSDcxeURmQnJxRlhnT3RhVzUzME9EbVN5cXlWMVV3Umxob0tvRDdkS0ZVSXBCaHJtTXowZHRJVVNKMFdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMDMyMDFfODhfMjQxNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWRjNTk1NTc0NmExMDE4NzE3MDI1NTdmZjhkYzRhNDMxMWMxNTE3YTczYWNlZTI0NTc5ZTY5ZDY5NDNkMGY4ZTAyODk5YTg5NjlmYzE4Y2UyMDU3MjEwNDk1YmRhNDJhM2NjMzZkNTZmNzAwMTgxZDAzMTA2ZWUzNWVmYjg1OGRhODYzMTc3NmY0Y2MxYTY4YWU2MjM4YzhkZjViMjg5Nzg1ZjdhZGUzMjQ2ZDI3NzE1ZDVhZjdiZjI0YjViNzMyYzU1YjgwM2QwMjY2ZDc1YjNjY2MyOWU5MjljMThiOWI3ZDg1MGEyNWNkNzBmZTUzODc2NWY3ZWVhMmUwMzJjNjcwNDc3ZTQxM2Q5MTE2NmQ5NzdkZWQ1NTZlYjRjNDc0MzM1OTE5YTQ1NWRlOGYwMjY3NmUzZTZhZGE4MTUwZGI4NzdhYzkyYjVkNDhjZjcxNjJkOGZiODk2NWUyZDcwODdjMDg0ZDI1ZTNiOGJjN2E4YmMxMDhjNzUxMDY4M2I1ZWE4ZjdlNTJlZjQ0YjI2MTUwY2ExNzdmNjhhMzZiYTZiNWE3YjFhMWE2MGM4MzkyMmY4MzI1NzlmNGVkYjI1YzU2MDlhNjE1MzcxN2YyZjgxZWEyOTljNTQ5OWFlZjVmNGQzOGM4Y2VlNWUzMjYzODAzNzg3NWY2YzA5NGE4Y2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3Pw2XYM58DnAbsZWJ69DJyVfsCtFRXypxN8cVo2KHp8z6gEATfIz8Z0P3JPCG4gHHbyPBjOo7devCYaSBX42Nw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_103201_88_2415_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.468Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlBNVllBV3V3UkZXSGhpMnpxK042L2wwNThrOE9lQUFXMVlrT1ZUK1lPOXI5aW4xQTIrZW1IdWNTRk5VNHdwVHMvdTBVRHZqUkhDY0RpZXNsNEpGZTR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMDMyMDFfODhfMjQxNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njg4NWJiMjFlYjg4OGY5NDlkNDNkMDNkN2EyZjZmMDU0ZTVjNTYxYmIyOWFlNDM1ZGMzODg4ZTM0MmMyOTc3Yjc3OGVhZDk0ZGQ5OWVjYmVhN2I4MWNkNzI5N2I1NDcwN2YyYzM5MjFlMjg2YWE5ZTAxMmEyNzBmNjIyYWEzNGI0MjBmYzc5ZTUyZmY2NzYyYTNlYzY1Yjk2OWI3MzU4Y2IxNzI4YTRhNzJkM2VmM2Y5N2EzNzQzNTM1MjdjMTMwYTA5YmNkOTYyM2U1NzhmYWMyYTQ5NzBiMDYwYzEwNzUyMzU1NzAzMjdlNTcyZTNjZTRmODE1MDcxNjcyMTQwMjQxMTdlNWVmNDBlYjQzMjM3YzNkMDlkN2M0ZDQwYTQ5YzJlNTFlYTA5YTFlZGM0M2Y3NzRlZDg5YzlmNDUxNTZmOWJlYmUwZmQ5NWUwNGFmN2Q1MTM3NGI0NmJkZDI5MmQ2ZWZjMTE1NDIyOGVlYjdhMzk1NjhhZTZmNjkwNWEyYjU2YTNiYTc4YTEyOGZkOTU2ZDYxMGFmZjQxNWM2ZTVhNWNkMmVhNDY1YzBjM2UyMTNmYjM3MmQ0MWRkMjY5Y2M4ZWJmZTgwMDdjZjg4N2FmZmJlNDY1OTQ3MGY1NjIxMWM1ZjIwMTRlY2Q4MDZlNDZkNjQ4Y2M1ZWI1OGExZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.n7FN7M3SKREZZYH8lkVqtYNXhwMBHo-Vy0VA39zzUTaTNlvsbECllb1KW8T-KGwJWpYtn8Kq6ArIPjJvkAlYcQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_103201_88_2415_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.472Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik9YbVJtdWpWSnhFWG1HYU8xVDlrRklNcFoweGdYRXZkUmpUamFzdnNMYlNCVG4wbnZtZERuMk5aUWJVbWd2UlZqWncrUzlnU0FJdFRaTFVQeWI5WkF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMDMyMDFfODhfMjQxNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmNkYzNlZGNiZmY2NTEzMGYyYzAxMjcyNDIxZTg4ZjZlZGRhZDNhZDRlZTA4YjdmOWJiMzRlZjY1MTY5MTBhOWU5ZDg0YTg5NGJhODEyNzEzM2NmOTIxMmQ5OGJmNjY0NjcxOTU1ZmZhZDAwNzgyZjEwOGJkODEyNjE2ZmNhN2Q5NmExYWI3NzdiMzcyY2E5YmQ0YmM4NzZlMjg3NWIwMDBmZGMzMTQ1OWQyOTY0MDc3YmFjOTZjMmVjZTk1ZjAzZWQ1NGNmYTVhNzFiMzJmYTk5ZWQwOGJkYjdjMzFlNTczNjdhNjAyZjNiY2FjMWI0ZGJhNDRlOTg5MWYxMDNhZjkyMTdkYTM1ZWY5ZmFjNzU3ODYyNmQ5MGRlNjM3MGZlODk1NmE4NzI2MGRjMjYyZjllNzMwMDRiNDZmNTBkNzQ5ZWVlZjYxNDIxNjU4N2VlNzdmOWE3OWQxYTc4NjljZTEyY2M0MmY3YTAxZDQ3NGQwYTcyMGM0MmE0ZDQyOGI1ZGEzYTRkZTdjMzE4OTI5NWQ2YzY3YTkyMWIxZWExZWUwODAwZGMwZGFlNzA5MWRmNjkxNTZkZjhmODliNmI4ZDQ4ZmI2OTVjNTM5YTI5NDAwMWMyMzg5MmIzNWQ3ODJjNjljMWExOGFhMTJkZDEyZDcxYjg2MjBmMzI5MDM3N2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2oWQQ0gnsqlYlLhP5ltISQdzoAomlD4hZ3r_EEn0PZUlmSkTJDlFMLgaHF_0hpsw3X0P_lCI40WoKi-rYJoWXg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_103201_88_2415_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.475Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlRENFhmc2hnd2FsTWpJaktsTnoxbDRTSmtBU0Vxb3JqTkhPQXdyei8reFFTWUpiN0xIZVFNRGhmV3hmeTBvaHhVVnovVTJIWWZ3ZzE5bkRURjFkT0xBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDlfODZfMjRhYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjE5OTM1ZDZhODIzZmJmMjFkODFlYzdhMjFmMThlY2U5OGYyNDUyOTZlMjYwM2Q1OTU0NTJlMDVjNWNkNGNiZWIxYmMzMmNhNDY1ZTFmZGEyMGJjMDI3ODkxNDU1NTA5MmU0NzhmYjAyYWJkNjBiMGY2OTE1MDhiMTYxNmE1NjQzZWVmNWI5OTdiNTEwM2Y0MzViYzUyOTM2Y2E1ZTAwMzk5YjhiNGYyNDJlOGVkMTA0MjkxOTYyZjQ3MDRkZDVhMjk3NmEyNmYzMTM2N2Q2ZGYyMWRlOWVkMGMwYTVlMTk4ZGVkOGUyNTk4MzExZDViYWRkYjg5MWMzNjE1M2UyODAwNzUyZDhiMjk3MTAzNzVjYzk2MTYxMTMwZDViZjBhY2VkYTUzZmU4Mzg3OTc1NzZmODEwYjJlYjNkOGFlMjA0OTFhMTk1MjhjYTQ1YWFlODc1MGQzZmE1YWMwOTdjNGFiODlmYTU2MmE4NjlkMjBjMDkxYzhhZWY3M2YwNjk5NjZlMWVjOTMxMmE4YWI1YjFmY2UwMmYzZWFiNjcyMDU1YTJmZWM2MjI1ZTU0ZTBhN2VlM2M0MDBhZmRjOTNmNjc0ODdkZDcxN2JiZmQxNjU4ZTFkZWFkNDMxZDQ5YWRlYzVhNTViZDEyNmE4MGVlZTlmNTE3N2M4ZjE0OTUyMjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Lzhzmnem2nAKzKksMgXxmgVXsd-iXHSsSGGcuiKfS5WMqKqzND839ly67nI6f9hmiLDtnrPTMXCBYRVRH4qYbQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113149_86_24aa_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.478Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IktPemRpSXRzSTFrTmNvUUlvZFlyWDJhSVZRUVBBa0x0dkxDYmN2RHpYK2RjTlgyVTBBYVIxemI4VEY5Q3RmN2Y1dE9LVzh4SFhhTXJyakFtamZUbEhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDlfODZfMjRhYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQwYzNmYjk5ODk3MzFmZWU2ZjZhYmQzYTcwYWExOTg2NTJmZTQ2MDdiMzk2M2M2ZDlkZTcxOTFkN2U3MjA5NjhjZTM1MDU5ZDQ0YWRhODNlZmFmNjM5NTg5ODI4N2QyYTk3M2I4MzhmMzA5NzhhYzkzOTQ1MmFjN2RiMWQ1OWQ4YTdiZTNiM2U0NGI2YTA2ZWFkNjUzMWI5Mjc0YmU4OTc2ZTY4M2YyYTJmOWJiMGU0YTNhOTQ4MzU4YWViNWUzZGY1ODQ3YWE2ZmY3ZTljNzFjNzdiNTNhNzRkNDIwMzBlZTgwNGY2YmFhYTI4ODU4MmQxZWE0ZGVkNjc1Zjg2MmE2Mjc2YjkxMWIxMDU2OTY4YjE4ZjQ1ZjJkYjFhOTA0NzhmNmNkNTE0ZWEyZWY1NTYzMWU1YTUxMjBjMjQzOThlNmNkNzJiOTBjNzNjNWJhZDViNzA1ZjBhMTY0ZGY5MTA4ZTA0ZmIyZjM0MjhmZjIzODAwZDY1OGVlYzc2Nzk2OWVlYTQ2ZTdkNWU0NzFhODVkMzA4MGEwOGNhYzgzYTE0NjI3Y2U2YTY4NDYxMDVjNDE4NjA1NTczYjBlMWY2M2RlYmFlMmNkZDM4YTZmYjE2YjkzYmJlYjM1NWEyMzYzNzdmNGNiNTAzNGEwYjQzNTk4NmYyMWMzY2Y0YmE2MDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Y3uybPXz_VdBZhegc1PDc37PLjQ_xcc4OwTQmcUPExc6r0TA02yRlksy94aBV2eV_uboZEETZJ8yAmwwUNgt7g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113149_86_24aa_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.480Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjNOLzdML3E3V2FMcnc5NDRPUzA0UDFScThJN0Y2NEY1Wko2KzAvMEtUZzF5Q1pxakRQcnI2Nkxrc1pta0hRSHhveXdEb3BadnJUZFIxbnlLWExxdTJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDlfODZfMjRhYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWQ2MTlhM2I1OTBlODgwZTE3MTk1NWUyZDgwOGQ0NTIyYTNjOWZiZWQyYTk5NDc5NmFjZTUyMTQ1ZDBlY2E2MWJiNGE2YTk3NDU2MjA0MjhjYzhjODRhNTI2ZWQ4ZWYwYjAyYmQ5ODIzZGI1YWUxNDg3N2NhMmRjN2NkODhmMjUyOGU5MGRhY2RjMjUxY2Q3MmM3YTQwOTQwODBhMWZmZTkwMmE0ZGY2NjNmMGRlNzkzOTc1OTRjZDkwMWYxOGZkYzMxNTY2YWI0NGM4MWQxM2Y3NDU1NWU3NmU4ZjMzMTU4NTMzNTdmM2NmMDVmYmY2ZmQxN2I4Mjc2NjJjZWEwYTc0NGRiYWJlOTBhOWFlMWYwM2ZkMjEwYTMzNWNhYTFhMzM2OGU5NGRmZGMyZjk0ZTc1ZDY2MTBkMTI2Yjc2M2U2Mzg1NDI0NWU3Nzc5ZWFiZWRlZGVjYWZmYmI4MjQzYmZkNmRjNmVmNmM4MjYzMzU5ZTRmNWQwNzQ3YmFiYTA2NTJmMTA2NDk5ZDk0ZjYyM2MwYzExZTEwNTEzNDUyOWVhMjdhNTk0YWIxN2IzMTVjNWI2Mjc0M2I3MzcyNjEwMTMxMDNlYzBhZTA0NWZmZWY2MDlhYzAxZWY0NWZlZWQwNmRlOGI2MDk4MTVhNzFjODM0MDk5YWNlZjJkYWU2ODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OE0GratUjz5p3Mz6l9jAoaKt2v8ETpKMip2kPKYDG1udG1doSYHAXBZayhJEfoWxDZqt_6ScgC_6upKOQzq8kw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113149_86_24aa_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.484Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxwYWFLaWtPVFQxRmllK0xoS0gvRmhUL1VQRHo0UG5MdzRFY01lanFrdjdzRlUxcHBxZkFYZUx2THc0Z2thekxxbnJsSTUvOUoxSEJ5SVE4M3MvajRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDlfODZfMjRhYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2RmYTk5MWI1YWEwMzlkNTk5NTFhZmFmZDFkMjQ4N2YxN2I0NmJhMzY3N2Q5ZmMzYzU0Njk2OGI3NTkxNTdmY2U3YmIyY2E0ZmFjMWRlMmI0Y2NiZDNjMjRjOGMzZDMzYjA2YmJhMGIyNzQzMWIyZjVlYTkwMGMxYjdiOTk5MmYyNTJmYmE2OThhYTJkODVjNjIwODAzNDAxMTdmOWE3ZGY4MzgzMWI1NjY0NWMwNWJkODQ3M2Q3MWVlNTQyNDVjNmNjMzVkNGJmMWE4MTY2NmNmYzEwNWZhM2M2Y2MwYzI4OThjNDY1NTcxZDk0YWQ2ZmUzY2NkMjYwNGM5YmVjMzVmYWY1ZGMzNDUzNzcxNjAzNjJlYzc5NjJmOTk1MDBjYjc4MTMwN2UyMTE3ZDhmMGQ2OTlhYThiYThmYjcyNzBmNjg5YjdlMmIwYTQxMGJkMjEzODc2MDM5ZjZmNjdhOWM2ZjA2NjM2Y2JlYmJlOGEwMDJhZWE0ODYzYzc2OWEzNzJkYTdlNDg3MjQxN2YwOWQ3NmY1MzBjMGFiNzUyMjAyYWQ0OTUzMmRkYjFiMGJlYjJjM2NmZTc5MWYxMjQ0YmU1MjIyMTdjYTFlZmIwNmE5YzU2YTFhZjMxN2RkNGE0MDA2NGU3ODllNDkwZWU5YjEwNDZkYmU2N2Y0OTBmYzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KkfijeDYmI6sTr4hmVNmAunQ0wMs1i2Q6E96COsP0iGHIf6aVA6tK4ySvV40YR3EFxZTpArlF55mV8opxo2SRw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113149_86_24aa_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.488Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkFwZjRTSTU3dUFYRFV5WExDSFJSVnZLV2FIUWVhbVNFZCtDUTNjVHY2M3NHbmxqS3hCblJpaEs4T29Bd3ZLTFhXdDlCaE41TW5rQ0xTaGxzczBXd0VRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzVfNzhfMjQ4MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjNmMTI1MTQyNDY5Mzk4YTg1MmVkNmVlN2MxYTZiNTlkYzhmNTFkODA2NWViZDQ4OWNjZTRiM2M2ZGNkOGMwMjI0NmY4YmQ2MWI2ZjFmNGNjMTcyNGU0MDkzMjRmY2VkNjIxZTA2NzViMWY4YjIxNDMwNjViMGMzODJjMGI2ZGM0YWI3YTM1NzIyNzJkY2Y1ZjBlOWMzMThlMjkzMGYwMGZjYjMyYjVjZWY5MzcyM2FjOGFmODNjNDdlYmQ3NWM4ZTEwMmMxYjFjNTIyNmU0NmZhZjVmZTc0MzA3YzRjYWYyYjYwNmI0NWU4ODMzNWJhMzhkMzU4NTM5YWI3MGVkMjFjN2EwMDBhZDJmNmQzNTQzYTY5ZmM3OGNkMzFmYTllNzM2ZjQ2OTYyNTNmNmUyMmM0ZGQ2M2E4Nzk2MzZhYzc0OTkzODQ5YmE4ZjQ4NTE3NDYyMDVlMDNmZTc2MDU3NzMzZDA2NGFlODg3ZDc2ZDdmMTEyZTA4NGVjYjcwNWNmYTExNmQ1ZjFmZGI3MzFhNTg5MTkwNzY2NjFiMzJmNGE2MTk2YzFhYTZjYTNhNzkxMTkxYzkwOGUyNTVjMjM2NTU1NWJjMDBlYWY5ZTJjYzMzODc5Mzk1NGQ2N2NhNzBjMmIyY2RjZDk1M2RhOGFiMGU4OTJhMmE0YTZhN2YwNWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DsJi-Vo-mVIfGsYPKOtvQm90LJtvFEu99ZHxFqt5nyz4RKyq3bQqqpLtBy4r6dMwY2pkBysGIeMRhnxOJ-b0FA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110435_78_2481_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.490Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjdIaXZodXlnYVloS09FR2FRc1FxQlFWNHM3SzQ4cUhBU0Zoektab3RseWNXam90c0psRkZUR2dkMXpqUVV0d3dCSTJXejBFaXIxZ1IxWlNMakNYUjZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzVfNzhfMjQ4MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGRmN2E4NjQ1Mjk1N2RhZDFlZjU0ZmM0ZmNhNWFmMDVhZjUzNzM0MmIxM2I1YmUwOGRiNjBkZjRhN2E0NTNlYTY1NTY0M2IxODQ2MGY3ZGYxYjZlZTFkN2M0Y2E5MzI5NjM1NWE3MmE5NzMyMjk3YmE3YTc3YmFlYTIyZWVjNTVjMTg1NDY2Mjc2MGVlYjIyNDA2NzljNDE5NGRhMzg5YzI1NGIxNWZiZjg3MTA1YmI3ZjVhZmJhY2Y1OThmN2FhM2NkNDU4N2RlZDYyYTM1YWFkNWM0MTc5MWUwNDE0YmUyMTAxOWYwZTRjYjk4ZjY3MWQ4NDhlZDM0YjQyN2ZmYWU5YWM5ZWI1YWNmNTgzODNmZTQyYTI4MDY0ZTMzY2I5MDYzY2I3OTZkZjBlYWZkZWQzMDUxNmY5ZTUwNTM3MjE3YjhlY2JlMTZkNzRhMDMyMDU4OTM5MThlY2QxNWE5NzhlMmExOGFlYjZjNmE5OWUwNWU4YjdmYzA0NTBiN2FmMTc5YzAwNDVkNmI2ZGVhYjA2ODJhMDA3YzkxZWRiYzIzZTI0Njg1NDM2NzU5MTJmMDg0M2U4MzBhZWZmYzQ1YTY3MDI5ODg0ODM5NDUzYjI5ZWYyZjk1NjJjYWQ0N2U1YzhmYTBlNTdjYjNmZDY2ZmUxMTllNzRmZGVhMTFlNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Tz3GKeAQ6BY8eaH-5_PRsmnisHInfjUJnHGJ8D6emM6K3F-mFlpRfyj2YFyc2ykOE7FA5cc375hNg4cxDytnbg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110435_78_2481_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.494Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ilc5Z0NnWlRja2tVbzhDQWFmOGJXbVAycWxIR0xyR0tpTkMrcC9OaERDcXRNQU9sWFRzdTZ6MEMrTyt0UnpiYlNGaVI5WWVGbjVLbnpLYUtmcTBYdCt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzVfNzhfMjQ4MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTkzNmI3ODAzOTZkYzYzMDY3NWM5MjU4ZmFiZTc0OTc4MTUzOGE1NDRmODA5OGQ4YjJjYWQ2NGYzZjk3MzU5ZDA3MmIzOTc3OWQ2ZTVlYjA3ZGIxZWQ4Yjk3YjY1MmNhYWIxNTYzNTg2YjY4YTU2NDZiNGFiNzA3NmQ0YjY4OTIyZGQxNjdmMDQ5ZWZhNTY3N2ZkOTYwODhhYWJlZjEyYTUxMDUxNDAwZDUwZGFiOGUzZDFmZTBmZmFiYTQ5NmY1MWVhNDhhMjA5OWVlNzBiZmYyMGQzMTYwZTNiZTA2ZTRmZTRkMzBiZmEwZGZkMjkyZDU5NjM5YTcxMDBjOWIyMDIzMmQ4NDk4YmVkZmM5MjU5MDA2OGNjNThlODMyYWI5NGRmNDgyMmIzNDJlZmI5ZjUwYzg2MTliZWUxNTQ0NGM0MDBkMDU5MjBjYzgzNDk1YTQ5MDczZWE0Mzk1MmFlODRlMTY3ZGQwNDVmOGIyMjViN2ZlZWIyMGM4ZmNkY2Y5N2Y2MTUwZjM0YTExMTZlM2ZhODFlOWJjZjIyZGUzNzIyMWQ3MWNlODA1MDg1NDEzNTM3ODc3MGZlMDUxNGQ4MThkODQ5YzkyYTMwYTQ3YzQwNTMzZTg5YTY0MTFkYTMwNDE4MDQ4MGFhYmZhYjE5YWZkMzc5OGFhODgzMzBmYTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oZMK0JBnCKPza6eXeF4LOYeP7oqpCMI2aEnRPvGg1tDewOZNY70AQ9zBhztpkHZDQ3pd-INkJT2KTJCPwbMapA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110435_78_2481_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.497Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFXdmY5VDVlRXBIZFgxWjEwMjkxaFZTRTczL2RyZUFJK3dZM0M5bGlrb042ZU5FZ0VCaDEzTUVzSTNPVjdLZUpvTjgrblBBd09PSFRFNGR5YXlHanZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzVfNzhfMjQ4MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY3NDljNGQ1OGEzNGIxNmY5MDgyNDIwNDhjMjdkMGNlNGQ5OTE5NjNiN2ViMGJlZTA1NDM2Y2ZmNzQ0Zjc2YjJlZjRlYTNkYWU1MGZiNTFiODk2YTljYzFkYjllOWUzMjUyYjk0NjgwZDEwMWJjNTY3ZTJjNGEyMGZjMzVlYzQ5YTJmYjM3MTExZmUyZDkzNjcxZWY3NDg3ZjNhYTJhZWYzODYzMGZiOWUxNzMzNmNhZGVkNmE4NTE2MDdiZjg4MmRmNmVlNzMzNjc0OWExYzlkZGNiNjA4NDQzMzJhNDg2OTg3ZTVjMzU0YjIzMGViM2U2MzI4NDlmZmU1YWJlYWJhMjkwOWNhZGQ5YTY2MWE2M2ZmMDIxM2U0MjVhMzUxNjU5Y2EyNTc4MmU0NWNlOWQxODM0NmE1ZWZmYWFmMWI1OTBiZDg1MzA4NDI1YTY3NDA5MTI5OTc4NDZhODAzMjk2OTY5ZTFlNTJiOTc1NTIyMjEzOWYwYmI4YWEyZTc4OWQ0NTQwMjFiMjNlNjhkNjJkZGFlYzJmZTY5ZGRhYzIyYWEwNzUyYzQ1NWYzMzZiYTk5Y2I3ZjQwZmVhYjA2MTkzYWE3YWUyZmIxYjBmNjhkNDk0NjY3YzEzYjgwNDFkOWE1MjZhYWNiZWNhNjViZTczMzRhMDQ1MDBmNWMzOWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OcXHdjNNIRGGH9utmuN0rqNNYvtF1tuVFt-q9nrizqgb9YYV9MbvIXS5Ioh1jYSuacn2DlHOl5vIMsi1K1dQ5g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110435_78_2481_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.500Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlcyZUx5YW0wTnBQdlJqc3hUMmZnMzJCSVdVWFZZVkVydmZGVVh1VHFvbms5cmpJUDluLy9KYkZiWnl1cE9pWGpHY205SmpRc3pEVzJiUCtZZGpGa3Z3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyOV8xMTEzMDRfNThfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTJmZTA0Mzc5YTY2MjkxYjY2MDA1Yjc2YTA0MGNkZjlmYTAzZTA3Y2Y5ZTc5YTRiYzJkNjQ3NGMzMTA5NmIyOGJkZWE4MDliODZmOGMzNmZmYmUzYzBmMTY1YzkzMjhmNTczOWUyZWE1MmZhNmE2NmU3MGJiMjFlNmM1NWQyMDdlYzQ3YjJmNTQ2YjM0NTU4YzdlYmJmNmQ4ZjUzZjI1NGQ1Zjg5Yjc1OWE5YTFlYjg5ZjI3ZTkwNGFmZTFkMWE0MjllMWVjZTM2ZGU1OWM1MTM3M2IzMmU0YzcwMTZkYWVlNjVkY2RjMGRkMjBlNjZkODI0ODM3MWFiYzBiODhmZDk5ZWY0NTQ0MGMwNjU2YWYyNzRmZGY0ODE5NjhjNmExYTY3NjM2YjczMzM3ZmNiZjUwMzA3ZTEyNTAxNDNjOTY5Zjg1ZjU2MmI3MmRiNzc2YzZjZDI4YWQzMGJmM2M0MzMzNmZlZTgwNjAwNDlkYmQ5ZDViYTYyMTQ1OTMyYTQ2YWI5ZjJiMWMzM2ZmZTJjNjcwYmQxZTkyZGExZGVlN2M1MzU3MzRjMDViNjAyOTBjMjQwZTc2YjA0MDQwNDlmM2I1ZWVhNjgwY2E0ODA3ZTg5YzYxZGRkMjgxYzRiZDc1YzAyZjRkYTFiMmI2NWUzMzE5YjExODRmZjVkZDI2YWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4u0FSZnIwH4Igh41UVPsvtqh203kPMcRLm0DGK77EbVtx--t3yMNRbtiNdkhpNa3rXd_F9NPP41mZsV5rS6b8A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220429_111304_58_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.503Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtEQ1pETi9MZkJnRVdJeE1VRlhXcVNuRkhzMWdqT1dtL3BFQ29ISUp6U1A2NmlLSDFaNStCb0dHZlFmT1ozdkh0S3dEUStCRWdEV3ppbUJVY3ptKzVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyOV8xMTEzMDRfNThfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGUzNDUzNWMyYmJmMGJlYTY2ZTVmZjkxNDQxNjBkYWU4ZWFiYTViOGRiNGNiMmFhYzZmZjBjOGEyMjRjOTkzYjFlYTMzY2JiYjMyNGM3NjNlNTI3ZDU2ZjRkNDgxNDQ0ZmVlYzhiZjg5NmJjYmFhZmYyNzcxMDU4MGE2MWFhMmI4MTJlMGRmODBkNzgzYmJlY2ZlYmQ4NTFiMWRlZWRjMjhiZTNiMGM3NTA1ZmI0MmNmNjE1OTBlNzgwNzdiMjQ1MDg5NWM4NTQzZmI5NTMxY2E5MDc3NjE5OGY0MmZlMjk4ZmMzYjczZjNkOGFjZGZmMTUwOWY2MDY1ZDAxOTRmYmNmNDU1NWEzMzQwNWY5NzRmNGNiNmYxM2Y5ZGEyZDlkNTViMTgwM2IyY2IyMmE1ZjQyOGYwMDVjZGQzZjZkZjYzMzZkNzEwMTEyM2YzOGFlMmI0NTc1MTlmYmRlOTlhOTlmY2I4MTk2MDRlNWZmOGMyNTUzMTUzMGM0MWI1MjhiNDZmNWRiMmJmNzZlOTk4MzE2ODg4OGQ4YWJkYzM1YTU3M2JiNzE1ZWJlMDVjN2VkNTk0YTZiMTMwMzQxMzZhZjYxZmIwNjgyMWFkMDU0ZGYyMjVhMzAxYWMxMjI4Y2NlZjZiNmU5OGY5MmI3ZTQ2MGM3ZTk5MjViYjI2NzI5NzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-Au0iMekigYXmGu24VFzw4FnS_05p-I6lSf7TFINGR0cQRvI9FuIJJKs67VKX_eOGM3rMF5c07iwGnBK-vSk3Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220429_111304_58_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.506Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlZlNFZqRVVmb3lYREVzd0dBemRudG5tU1g4SSsvZWxZZWxtMWhnSU9ZV2RTaEZuMXJOKzBuK1d4TGRFeXJYWGZ6RnJ3VlZvRkVWV0NXL0YxeVVOQWdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyOV8xMTEzMDRfNThfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDg0NDE4ODg1YjE5ZjI1ZjRlM2YxNTM4OWY4NmY5NjY3ZTBiMGFiNWJmYjIzZTkyNTJhMzg2YzRkYzM3OTc4YjA3MDY3ZmYyNDUzMzE0ZDY3OWZlZTM5NTk4MGY1NTViYjliZmI2NmQ1Mzk3OTZjZGY1ODFjMDAyMzVkMjU2MzMyMDliMWQ1NTcyYzc2OTBjNDE2ZGQ1MTdkNmM3NzIwMzBkYzJkMDg5MmJiOWJjMTVhZmI2YWNlMTVmYjBlOTgyMmZhNGViZmI1NDMwYzBjMzFhMGIwMTdhNTg3MDgzNDgwMDgwMWM2OGNiNmE5ZDg4ZWUwYmEwYzE4ZjA0MTBjYjBiNWZmOTZlYTk1YTg2YjkyZTE3YzI0NjNjNmU3ZjZkM2Y0MTAxZThiYjY2MGJlYzM5MzRiMDI4NzZjMmZjZDA0OWVlMzRjZGZhMGFlNThiOWNhMTMyNjBlODkyZTRjNjhiM2RhZmFlNjkyYTk3ZjUzYmVhNWU5Y2VjNWYwNzVkNGE0NDFmYjhiZTFjMDg5ZDFlMzgzN2FkY2QyZWQyZmU3ODUxYzU0ZWVlYzY5NzZhZjc3ZTFkYTdhY2ZjMmNlOWJiNTk5N2RlY2YxMDE3NTMxMDM5NDMyM2Y1NjUyZGY3Yzc4ZjQ4YjBiYjA2NjlmNTFhYmEyOTFlMzRmYTdjNzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LJ5Z05V-hK40C_2euvYgSp27F1AOzEl_yj7PPvFMvScCR1fqfXCo5VY9PDTKcL6DlYyXXQJtEA4uncmpBHh4uA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220429_111304_58_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.509Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjJFOXRnNWo3L1h4NFhMVndrRE9iQXV6cmprOHpsaUpsYzFKb2RNWWNRNzRoRVY1MWlpQ2lRbzdxd1lRWG5VUFBmWlNCL05GcGliM0RDS1ZFaFJQT1FRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyOV8xMTEzMDRfNThfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjhiODgyNTA3NWU5ODBlZWNmZTIwYWVkZTQ2Mzc5Y2RhYTZlODgxMjBiZGM1OTNkOTViNDkzYWM0MjhjMzQyNGY4ZTEzYjQ4OTIyYmVhOTA1MWUzYmU0ZmMzNzU4ZGQxNjg0YmM1MTY5ZjRkMzEwNzAzYTdiNTlhYjc3YjM1YzYxYWU5YWYwYTNjM2RmNmMxOWQ3YTNmYzRlOTllZjI2MjRkZjc2OWU1YTBhYTVhYTk5NDU3MjExYTJhNjFlMzU1MzA0M2VhYTdjYjBhNjMwMTdhZmI1ZWM1MmFhMDA0ZWMzNDhmMDM2YjhmYWUwZWNiMWY3MTEzZjJhMDNjYjQ3MzkxZDNlY2E2NTBkZDM1NjI4ZmYzM2U2NTNlYWNkZWMzZDU4OGY3N2QzNmQ5MjU4MDRhYWIwOTVhOTE3OTI4OWY0YWFlMzZiNWJkY2NkM2JhN2Q5NTg4OTFiZmQ2MmM5Zjg5MDdiZGI4NDk4ZDhjODdlNmU2M2I5YTJmYzU4YTM0NjNjZWUyMjgyMGY3NDcwNTE1NmRkYjIyYzllZmI4MTMxZWViZjg1MTE2ZjgwZGI5ZmE3YjgzNjgxOWI4NjEwZjZiMjcxYjFkMTI5NDVlNWYzMmIzNzMyZmE1MTE5YjAzMmUwN2U2MDlhODk1NmY4Y2JlYzE5NDgzNTI5ZmVjMWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yOyVy0W5SO0Z2AJky9gQnvwzgCgeGG-0Hlo_3pJ7ouAqA27t_Q0bV2tjvtBZTrrXKKAMafB7eXO8EJMLfszaGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220429_111304_58_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.512Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImZGKzVJWE9nWXlXZ0Q0ME8wKzdzU2dEaThmUTcrVndOZTZnaEtISkRTRVlJOU1uMGhQcFlmVlJEcURLYTlibHc1R3RMeTJkM2ZHTlVvRFR6eG9QQURRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzBfOTRfMjQ2NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWQ5ZjNhODMxMGM3YjdlZjQwNzhiYmEzZTJhOTQ4MDk1Mzg2YTdiMTAzMzYyMWIzZGJlMDAxZDFjN2EwNGEwYzc1Yzc3NmZjOGViNzhlNjc2ODBlMTMwMDY0MDcxOWY2YjRhNjY5MTQzZjVhY2E4OWMxNTljZTA5YmRkYzY4ZDQyMjkyNTU2Yjc0YzM5MjQ0N2YyNzcyNDJiMDBiM2NkOGQzMGJhZjdiNmViZWVmYmE3ZGZjZGRlYmUyNmVhMjAwZDFhZjE3YWViYmRlMGVjMGMyODNlODljM2Q4NjhjOGE4ZjQyNjIzNzY2ZTVlNmJhZWFmMjc3Njg0NjVhNjNlN2I4OWNhN2Q1ZGIzOTcwZTUyOWVkMTVmMDJlMmY3MTFhMjM1M2EwNmQ5YWFiZGYyMjRjM2VlOTk3ZWIxNGIyNDY3ZTViMWZhMzZlYzRhNTllY2EwNjZmOWQxN2ZlNjM0MTg1OTI0Yjc1MDAzNzlmMTJmOWMwYWM2YjRmYmE0MmRlYTA0OWI1MGVhNTU0NzcxNmIzMDFjMGU1ZmQ3YjFjMWYzMjAzYjcwYjlhZWQ0N2VhYWIxOTA5ZWIzOGFhYTA3OTM4NTZhZTc2MjQ1ZTkwYjA2YjNlMDM1MWFmNWY3MjI2ZjZmZGFmOTViMGQ1NDEwYmViMjhhYTI0MzQ1OTRmNDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EKMGhAatEQ35OwIgs-t7ArtI7wo15kYPEV-Gwm8LdcCVWRFyml7FKkxWzhJNKywX5Au81FSkdoCiYi1VZt7HxA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102430_94_2465_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.514Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii9neU9FMkVaWWMyY0NFQVBZRUVnRkFWNWpBai9QZjBQc0J3a1VRMHdFSmhvZzB5eG5xeEcyTXI1RVZYZlZwekFyTS9OZURrTEhuT0x6czd1Y01FTjBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzBfOTRfMjQ2NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTE2ZjY3OGIzZjYwYjY1MWVjZWM4MmM3OTM2ZGM1YjYyMjNlMTkyNmE2MmNmNDdlY2YyMWMwMTMwZWIxYjg1Mjc2OThkYjU2ZGNkMDEyNTAzNDQ4NDExYzRkZjE1Y2UyYjU2YzM4MDliZmFlYjVkM2YzYmNmNmJmNWU3ODQ5YjVlM2ZjMTQ3YTNiNGZhYTVhOGFlNWU5MDBiMWNjZWRkYmNiY2YxOTY2ZjY0NmY3OGEwM2RjMmJjZDE1Y2VhNzk1MmNlNmU1Y2I1MDdiMzJhMjRkMTIxYmQ1Mjk4ZDQwYWI5MzQ2YzRhZTVkOWIzZDgzZWUxZmEwNjE2Yzc5MmVlNTAzMWU4NTAwMDk5MmMzNTg4NTljZmIxMzIyZjA4NzBmMmY3NGM1NTAyY2RkNjlhYjFmNjI5MWFiNGRhNWZjYWRjMjRhODRjODMyNjkxNzliYjNkOGIwYjZhZDdjMzY4MTBkYjFhZDg5OWQ0OTdlZmZlMmRjYmM2YzI5ZDY5MjEyMjY0OThjMjNjMzZiNjZjZTQzMjYzM2Y0MDNjZDc0ODljMDEyMjljMDEzOWJiMDNkYWQzNGUwNzQ3MTIyZTc0MTNmMTIzODQ2ZGQzNjhiMDdiNzgxY2E0ODY5NjA3MjY4ZWMyMjZhZWUxZWM4ZTJlNjM1NTQzMzY1MjAwOTI0ZmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GxNtvi7pwQOMAaVafk6seXpJicB9gbbZ0bAPVjNt3x7pPokDSRUM6ipNsvn-dUBglbBTr9zHne7nzlMJbuJN0Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102430_94_2465_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.517Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlVaTlhiYlhMU3F2YUdaeGgwVkowMjRwdllaSE0za0dvQ28zd01ZNWJjYUlQTWNOVFh3QlFvMDd6OXIvc0xhK0FmaExEaFlhNHJQcWVMOFZTWld0elB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzBfOTRfMjQ2NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWFhMDAyY2Y2MDMzMGVhMjliMTYzMTM0YThiM2FlZDQ4MjVlZTRjMTk4MWUwZDA1ZDhkNjliYzc0NDVkZGE2OWQ5Nzk4YjNmZDkyYTg0NGJmOTc0MWVkZDZlYjY4NGMwZmE2MmVmYjUzNzc1N2JjNjdjNWM5ZmZiNzc2OTM3M2I0YTYyZWExNDJiM2FmOWU4NjAyZjlhOTQ2OGFhZWQ4MDkyZDM0M2M1OTMxMThlMDBkNmE0N2I1ZTUxMTE1M2VlNzM4NDI0MjZiMGFiNGFmZWRkMjAwNjAzZTgxMTczY2FkMDM5MWZjMjM5ZWJkNjQ2M2E4ZTFlYzQ4ZjE3ZjA5ZDcwMTQ0Y2I5Y2Q2NDBmMzUwY2U4ODEwNTk4YjBjZDgxZDY5YmVjOWE5MWRmN2RiNmJmY2YyZTdjNjk3YmUwYWNiNDRjNTBlYmViMzY3MzJkNDYwYmQ1NGVmMjM0NWE5MTFjNWVkZDZlNGNiMTYyZTRlNmJhM2MwODJjN2E0NWZkZmZhZmQ2NmI2ODBjYzZkYWY4OWYxYTQ3MmYzZjU4ZGU1NDViODcyMjYyZTM5MjNlNGVjYzAxYzFiNGExNjA0MTFkZTUxNTAxNmRmYzE2MzVlM2YzODFhNzAzYjBkMTEzNjgxYWUzMjdmM2QyM2JlNmRiN2U0YzFhY2EzZTAyNjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rjyDh0RcYAHo9_D_NQhnWBElpGR3mCmsBi9NOitMXRwu64PNmC2bmVnat-hlhrmT_f_SVeTF2SL4UXWbweh_Wg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102430_94_2465_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.520Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InYwMW9XT2NadXlLYTA4RnV6bFRXN2tZVTBqMEU1K0VmU25PbnEyTUt2K3RaUDdadnBvMVNPZzJxeVhZRkJVUUZjM29yMTFvaWVJRWZXMExiTWdIeUlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzBfOTRfMjQ2NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzU1ZTg5NWQ4OTJkYWY4MTM1Y2Y3ZDY1MDhhMzI0YTRiZTg4MzY4ZGQwYmJhYmVmMDk3MmI0NjIzNGUxNGE0OWQyOTJhNTEwMjI4YTNjZWI3NDE4MDI5YjA2NGJiNmE4MjFkYzg3ODE4NmM5NGQ2ZDhmNDZhYzQ0Y2E2ZTFhMDM5ZmJiNmNjNWMxNmM0NGFhNDgwZDA4MDhlMmEzMmI5MzU2NGEwMWRkZjQxYWFjZTU5NmNlM2Y5YjJiOTc1MDU2Y2JiMDM2OTE5ZTg1N2FjZDA5MmMzOWY5ZWNiYzg0ZWQ5ZTBmNWJjNzE1ZDM3NWEwNjIwY2YyYTdkNTAxNGY0NmVlNjRkY2ZmMzZjZjU2M2I1MzQzMTk3YjM1YzQ1MWUzNGJiNzExNDc4MTdjZDE0MGFmYjEwN2Y0ZGU5Nzk2NDExZmYxOWE3NjYzY2U5ZTNhNWNiOWJkYjhkODU1ZTNlMGY3NzUwMDFlODRmOGY0YmQ3OTNkMmFlY2JiNzdjYzIzOGFkNzg0ZjVlODE1NzE5NGJjMmQxNjQ0ZjU3YTQ1YmM3MDZlNjYzNzUwMDc4NTM0Mzg1MzA1ZjkxZWY3ODMyYTcwYWI1ODYxMGFjNzIzMWQ1ZDM0M2EyODYwNGJkOTQzMjQyMzRiMzI5YTNlNDUwZTFiZjAzZjUxMGY2Y2I0MGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EIL_CXSahM_e-b_hSbd8_hVkFHOk5-P6A48ANsFVeMKknDoFVy0dQy3ohLFwxypDkhDlxDTwC3wnkLnKSrAodQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102430_94_2465_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.522Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlRDVWcxVVRhUlhNNGdxc1NZMGZKRjAzTE5Fa3lkRlo3Z2NjOXFCZjNNaXB5TGVDcTFJT2lnWXNSMVhOMzNjRjh4Q092ZTRiUU5TWjNOT1BoSVFINDNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDNfNTVfMjQ1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDUzM2E2MGM1ZGY4MTFiMTVmZDAyMGJmNGIzZmUyNzllYzYyMGM5MjNiYWFjZjBmMzRmZDhjMTY1MDkzOTliMDY1MTVhNGQxZjgwY2Q2NzA0Yzg1Y2I0MTgzMmRlNDVmNzUwNmJhNGVjNjYxZWM5MmFkY2YyZTdhMzdmM2Q1ODQwNGY4YzM3ZWUwOTVhYzkyYjBkNDE5ODZkNTVjOTQyODg4NmZlYzg3YzQyNjU5MThlY2JhNmVkNTY2OWNhYjU0ODY3NjJkN2UyNmRjNWJkOTY3NTcwNTUwNDEwYmI2MWU0Zjc4ODc0MjA4ZGVjMWQzNTFlYTZjNDlhMWJlMTk1NjdhOWFmMWQ2MWJiYTE1ZTFkODg0OGY3MDAyZWNkOTQ5ZjczNzhlNmQ5NTA1ZDk4MzU1NDlkOTEzMjRiZDc5Yjc4MjcyZWI2MTRjN2VhMjE1OWRlOGZjMjA0MzBjMzg3YmU4NjRkMDM3OWE2YTdiMzZkMDIxNzljOGU2OGYzNzgxNWNiMGMwN2Y1OTQxYTY0ZTIwYzI5OWY0MGYzOTY4MTNkZTY5OGJhMGZlNDYwYjUwMWIzZGNiN2M3YjM3ODExMjk4NzgwNjdiMjQ3NTE2ODQ4NWM4YzI5MDRkY2UwMmJjMmQyNWJiM2RlOWIwZjNiNDVlZWExYzJiM2IwYTFmNzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-Q26bDE9LRUS65RZKQKhdDmGYKy3wRcxuBs5j7fnQbFPv7MJnPKExg5SHG7pcX1NSKOThJ5w1wPFpfRa0Q7LHA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102503_55_2451_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.526Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ijl3TUpGS2VLcGcyN21iVytUOHVHTjZOZzVydWpld3pMQjZibFRFOVQ4YWlQK1VZL2RjeU5aUWs3R21ZcXNLOGQ5SGxvVllVa1NMeGNLYUw3eWRaa0RnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDNfNTVfMjQ1MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGFhN2ZhOTEwYTgxM2E1NDkyMmY4OGIzZTRmNTU1ZWIwNGVjMmY5ODBjNTliNDUzMGQzNWY3NDZjMzkyMzU4YjM1NzVhZmY3NzdhZGZmNjljZjY3M2UxZjY0NTdlM2EzMWVmYjA1YjZiYjgwZTgyMWJiZDM5ZTZkNjg2NzQ0MTkzM2U0OTk4YTU5NDJhOWJkODhiMTdiOGY2ZjhhOWUwZjQ5YzUyYThkMTNkNjNjZmM0ZTlkODE5OGE3NmM3N2ZkYjExZjI3NDQzYzhhODI1N2JmNjI3YmYxODAxNzk5NWQyNmIyOTM4OTlhMTY5YmY5YTdiNzQ3NmUyOTNjOWE4NjljMTRkY2E1NTNmMjE4Mjc4NWNiNjJjN2UwN2QyZjZkMDU1YmMyZDViMDVmZTk2M2VhNGFlYTIxYmRlYTQ0NGRkMmZlN2Q2ODZiOGEzYjRjYjJlNTYyYmNkYjhkNGNlZmUxMzJhZmI4NzYxYTNhNGFiY2I5MjA2NDFjNjUyYjM1N2IzMzRkNDRiYTNmMjEwNGViZTZjZWNjNjFmNWYxY2YyNGY0Y2RiYzNjMzljZjY1ZmViYWQxYmFmN2E1YmE2ZWQyYzYyODk4Yzc1OTlkMmMzMmI0NjNjMDlhYmQyMTU2YjA4N2RkMGRiMmFjNTJkMDk5ZTRjZGRjNTQ3ODhlZjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.l09XqDyjGU6zn1xwfl43-XJuXczP7SiI9PHX3dddCpefG2nHxb7dsBfab-ttjOuY6GRny8kho4xDKHPnDcoCEw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102503_55_2451_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.529Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IklaNkJhTHBnTHVLNk1VanlCVmduTUczQ2o4TU9JaXNpS3dWWThKNDB2VTJqelZ6VFRZT0Flbjg0bVZZM2g0YlpoNUhBbkl2bTlOMnFucm9FZ3ZlM3hBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDNfNTVfMjQ1MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQ3MmM2MGU2ODkyNjExMjE2NDkwOGUxMmEyNmUxMzZhOTI3ZjRmY2I5Yzk1MTg5OGY5M2M4NmM5NDZlZmE3N2Y5MjQzZTg5MTk0YmJlMDdjYjgyNjJmMTJmNjM3NjNjZTZkNGMyMmQ2OGM5M2Y0OTc4NGU3NTJiZDE4NzI0N2EyYzk2NDRiYjdkYjk1ZjUxODQ2YjE4Y2EwNjkwNjVmNzUxMjViOWMyYjUyNDQ4Yjc2ODM1NjRlY2I4MzhkYzZmMDY1NzU5NzgxNGQ1MjQ5OGM1MmMzZmUwZmU4ZWUwZjA1YTc2MDEyNjExNmMzYzgxNGM1MmExNmJlZDk5YjdjOWZlZjZkMzljNzJmNDg3Njk2ZDExOWM3MjgyMjk2MTc5MWIxYWIzMjE4OThiMjQ4NmY2MjAwZWM5MDBlZGVkOGNmMDFlYmVlYmQ2NjdkZWRmNDhhYjE0MGU4MzRhNzY0YWIzNjc1NmY1NGNkMzdiMDI0Y2ExNjM5MGFkNmMyNDEzNGFiNDkxNTFhZGI3OTU1NjFmMjM4YjFlNTI0MzYzMmZjMmJlYmUxNTEwODNiODViZjEzNDY1NDM4NTJmNzQyM2M4YzQ5N2MyMjFjMjU1NzAwYzkwZGE3NmJlMTRhMThkNDA2ZGUzOTdhNzg0YjJjOTJlOTYyZTAwYjM1MTk0ZjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fBqgDNRGPr1R2lEyxUaUoJlWieTXFewiEmzWKOU6Diz0m42B-g1OMALZWUQi3xt3dKkKJQpPeg5BQqFsHfC3eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102503_55_2451_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.532Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikpuemxmb25udlBzb3RYMEx4RG5za2F6YTVPNElScHNBSi9PZ3VjajhPMEhTRkJSb1Jud2U3VmZnRzF5MlovRTlBMUgxMjMyenhYRHRSZDRqNEdZWEpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwMV8xMDI1MDNfNTVfMjQ1MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTEwZDY3MDdmYjRjZjhlYTUyYmEyZjY1ZDM2M2QzNDM1YWFhNmJlNjIwMmQ5Y2M3NTQyYjUzNDI4YzlkNDAxMjNiNGQwY2FjYTc1MGU3OTVlZGQ1MWZmN2ExZDgyZDlmODI0Y2U4ZjIwYTE5NzI1NjU4MTFlYWMwNjA0MTVkMDk0OWJlY2MxNTZjOTUxZGI0ZGIxOWExOWNkOWVhODNiMTgwOTI2NjZhZjhjNmEzYjFmM2MxYTI3YjdhNDk1MjUxMzEwNDZlNmQ0ZDY0NDBkYzJmM2Q4ZjM0ZDA0OGUxZGEyZjZhMzg0NTc1ZTIxN2JiMjgxZmFjOTcwYjNiN2M2ZTEwMjc2NDhkZDNlM2YxZGJlZWQ3OWNiOTVmODY2ZWU1MTNkZWQzYzI3ZWI0ZDIzOTcyNzE4ZDZiMDYzMDI2NWY0OTk2N2Q0MTFmODMyODJiMzkwNzVkYTFiY2IxZjE1NWZiOTc5MTNjMDE3MjU0NjNlOTQwMmU1NjQwODVjMjg5MWE3YjZhYWQyOThlYmM5NmRjODkzZmFhN2NjYjYyMDBkMWE3MjYyYWQzMzZjNGI2ZjU3MjRjZDkwNTY2YzJjOGZhYjk4NzgwMTcwMmVjNTA5MzVlOWFkODYzY2UwNjEzZmRmZTE0ZjBlYzAyZGQzZjI4Nzc4MTdiOTE5NjVjYTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UDJcu0WFLcgWTHP_fEFxqgqZZ95d4D8d4l2dCi1BDhVMUfeONc_k7Pi5GGNToYCYIT5pOcTDoL1cLSymXQj7Bg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102503_55_2451_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.534Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InRqQldIVTNxYkFzQTFrbXBCT1FhZmNWM1pEdTB6WlNBWGdxREVBUXpjQ0JtOGlReGsyc2lzVTNqL0piQTBpeUl4dDFYdkVyK0lHUnFDTU1BbXdOYmpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDJfNTNfMjQ1OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTVhOGRlYjQ4YjRmMTllNDZkZDkzNDA3NDNjMzdiMTVhZDVjNjcwM2VjZGQ2ZmI0Mjc0NjE0ZGJkMjAxOGUwNWI0YjMzYWI1YTUzNTM0ZWFiMDU2ZmM4NzBkNjE2MzA2YjNlMzVjMWRkYTRiNDhiNTcyMGI2Y2JlZTZhOWIzZjEwMGNhMzgyZjYwZjk3ZDlkNDcwYzVmNjk3NTBhYWRiYjA0NGNlN2RhNjc5Y2ZmMWNlMmJmYzU2NjE5YjNiNDY2YjY3ZGEyNGJkMWQyNjJlYzJiNjViY2I0NzZjYTA3YTg5MGQzNWE2N2UwMGJhNzc3MDcwNjc1NzI2ZTE5YTcwNzUwMjE1N2JkMmE3NjRiYzk4NTNhNWFmYjk0M2MxMjc0Y2EzMWQ5OWJjN2RhMzI3NjY3YTFkOTJiZTY0ZTJjMWQ5ODgzMGU0OGQ1ZjgzMGE1OGM2OTYwZWZkMTA1NzMxOTA0ODQ1OWRmMDE1NzM5YTdkZTdlYjg5NjgyNjU0MmM2OGMxNWIyM2EwMTRhOTg1OTUzOGU5NDMzNTUwOWZkN2UxNjA2YzllNWYwOTM1ZWFhNmM5NWY0MzU4OGJhNzU0YjU0YmE1ODlhMDU0MTQwM2VkMzgzNTAyNjI4NTRhYzBiYTVhZTUyYzU1ODFiYTJiNTViNThjMmI5Njg4NTg1Y2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ptbX43MWgxiQ2_QsOfP3FSBbGNwhk9bmALTt6mpv8tBsjrP-QF6yYlFneswtilEucNBlSaMBSAwLCoH31VCO3Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103642_53_2458_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.537Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRUS0FRZGd4L3E5V1pLaFhxcHZFcE0zNkdiblQrSWlSM3BlNDBldXJZamg0VHBmTjlBa1ZzVUpFb2wxVEJjQU9OdDArS2lxeExYdlllQ3NmZFIyaUJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDJfNTNfMjQ1OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWFkN2QyOGYyZjMwMTcxYmM4ZGFmNTA3ODE0YTEzMTM5MTY0YWIxZTQwZjQxMzUxNGEwYjAyZGMxZTlhYWViNWYzYjlmZmU0MjFlNDM1MzE1ZTE2YTZmMDRmMmUwZDBjOTRjZjY3MGUwZjFlNzdkODQ1MTIyNTkxNDZiN2UwYmY4MjliNzQ2NTI1ZTM2ZTY1YWE4N2M5OTlhZjQ4MjllNzk1Y2Q3M2QwNTE5MTQ0MjQyNWZkNDgwMWY5M2U1N2Q1NTJiMTE3NGU3YTUxNGU2NjVhYmZiMDY4ZWQyMGU1OGJmODIxZWQ5NDRhODkwYTY4NzMwMjhlNDJiMjQwYTk1ZGJhNzdiMGEwMDM2NzUyZjRlOWU5OGNkOWM2MDE2OTFiNDRhZWI3NjIwMTYwNGNkNzRhNGExOTE4ZjlkNDhhMWJlMmVjN2I1MjZhN2RhOTYwMTViMWEwYTgyYTYxM2IxNDA0OTQwNDI5MDJhOTZlNzE0Yjg5NTdhM2JmMWZiOTY1OWU1NmQyZDgyZGViNjRlYjk1M2I4NmJlNzUyNmQ2YjQ1NWQ3ZDAzOTBmNjQ3NzhlMDM4MjEyNjVlODMwY2YxZDMzYTNiNmEwNDAzNmNkNDBlZTU4ZWIxZGZmYWE2OTA4YjAwYzg1YTkxYzVmYWY3YzI4YTgwNWIzMGIwYzBkMzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9d1y01Sevl-c8_utnQTn9BhLJDdezuI6XpLiZ6x86Dizc3FxShTY-Id2ZJkAPEXPlOWyUm5j9IgnHZKdJgdgUA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103642_53_2458_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.540Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpZbWVrTDViYUdLbjBBMmFJeHQ2N3R4R2RCU25KNU9ZYjlBTnlydFdaRU1YcExJeVhFYW1XN0NMZTkzSG9kRGI4OCtObWNkd0IxSlRYN2d1NW9NNjFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDJfNTNfMjQ1OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWU3NTgwYTRlZTMzZjMxMTM1MjQ5NTdjMGVmOThkMDU5ZmM4MTliNzkzZjk2ZjQ0OTY1YTUyOTE2MGI5ZGNjNDgyZmNmZTcwZTM4MDRhMDg0M2FkNTk5OWVlMWU2NzkwZDA5ODQwNWQwMDk5NDc1YmMwYmM0MmFlNTdhYmM5YjJkM2M0NTViNWE1ZDdiNGEwMDczZjhkMmFmYzVkNzAyYjE4Mjg0YTcxYTM5OTg4ZTk3ZTg5OTExMDI2ODAwOWJiYjU1YTU0YTdiODI1NWI3ZjhkYTJhNjE0NTQyZTMxZjhiMTJiMDg0ODdjZGFjMTYxZjg5MTRlNTVhM2EwNDU5ODcyN2FiOGE2ZWIwYjc0YzU2MGEwYWY5NjNjMmZhMjcxOWNiNzQ1ODAyYThhYmRkYWY0MDZlYTBiZWFkYmZiZTVmNjdmYWZmZTRiN2Q2NjViMDU5MzU3MmE2YzQ0M2JkZDFmMDdiNWU5NzZiMTViZmQwMmU5ODU2OTFmZjRkNzFkNjBiZGViYjAzNjUyMGUxZGZkYmRiNzU5NjAwNzZjMjJkMGJlNmNkN2IwMTNkN2Y2YmEzYTUzNzE3ZGUwZGY2YWJhZWY3YmRhNWUwM2RjZmViOTk5NjFhMjAyZTExNTM3ZDQ5NjY0N2JkMmZlNDE1ZDg2NTk3YmViNjUxNGVjMmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.b50H2dfs0FiJrgaGB52fXcOVb-Oz2zMzShhhzXMfeHSjtymlDt8dqzx9UpN2_rIqaCiK1kcqbmO4HxSAfS_epg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103642_53_2458_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.543Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImlNU1R6M2R0YkNlbTdTbDRFQW9hd2phekhnWEtCVE94Y1dFZHFNa1BwQnZ1NWRVc1dRRGtzZWdHdWZxcWRaTVFmLzhQKzY0dmZUQk1YYW4xVUR2dzlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDJfNTNfMjQ1OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGIwZmQ2ZjJiMzI5ZmVmMzdjMzlkZTZlNTYxZGZhZGNlMWQxMDE4OGIwOTM1ZTRjNDc3YmNiODdmZmYzNDkxMGNhMmQxOTk0NWYxMzFiNTE1YTZiMjg3NDNlNGViYTZkNjY2YzE3ZGViNTI2Y2ZlYWFhY2MwOTIyMzlmYzBiYzlmN2YyNGVmYTJkMzIyY2Q0YWFjMWNmOGE3OWI5MmZjMjhjOTg5OTFlMTZkMDIzZjU3NDA0NjZkNmY1NjdkZjdmMTQ4ZmE0NzNhYTcwOWJhNjVjNzA5NGQxYjc2OWI2NzhiYWQ1ZDkzODc4NTJlMThmNTg0ZTVkMDA2MjBlZjU3MzgwYjZjZDlkNTI2NzdlNjg3MDJlMmU5ZTNkODFhYTBmNjcwZGEyOTY0NGY3NjBlYmUxN2Q4M2I2NjhiOWZjOTc1MDJkMzcyODYyZWYyOTZmYThmMTQ0Y2Y5YjI4NzkyNGQ2YzZkMGE2MjhlMTRmZjE5YWUwZDNiOTQyOWZmNWNlOTQzMzVjMjg4Y2NhYWRmYmE3YTk3MGJmNjFkZmM2NTU2YTcwMjU5N2Q5YmM5MDNmZDYyNTI1NjQ1ZTVjM2YzZjFlNmFjMDBiNDMwYTJhOWJjMTkzMzQxMGM4OGYzODk4NjUxMTc2NjRmZjMxZmI0OGQ5OTQyZjFkMmUwY2VjZDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NjBn0xYqHSq3tQKASaArgWVxSdzGnSe5M4S6Kap1K1ZXGHvv4ciRMcxkBqLqSQKCahLoEFwyfBZAOXGoKpa6tA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103642_53_2458_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.545Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InEvdGR2UVBqd3Y2QmJFeFdvSjFxV2NNYTB5QVMwbGNidWxqNlVuMCs1cFQwTlFkdXJpQXhXa0prSXhxMmFheldKRWwrRDg4Y2pIM1NmTC9JN282bndnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEzMF8xMTI5NDRfMjVfMjQxY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmU2ZDYwMDI0ZDQzODBiMjBlNTM4ODJkYTllNjUxZjYxODg1YjkxMGYyNWVlODZmODI2YmJiYzI5Yjg0OTkwYjllNWYxM2Y2ZDcyOWQ0Y2U1ZGJjOThkYzhiNThhMDU2ODgxZjQyMzQyMDhjMjgwZGFiZjAxZjY0YzY0MTNlZDBjNWFhZTdiNWUyZDc4YTNlZmY0MzBjYTVjYjA5MWU2MGFiYWExOWY1ZWE4ZTA1M2Y1NWM2Y2Q3YTkxYWIxODI1YmY5NDhhNzBjZjg2MzNmZDY0ZGE3YzQ5OTY2NDQxYTFiZmMzYmEyNzRiM2IwMzlmMjZmNWU4YmE0NDU1MTkwNTc5MTUxZGY1ZmI5MjkzZGNiYmZlZmM5ZWU3MmJhMDVhYjA1ODRlNWFkY2JiMTg5YjQ0YzNhYWM5NTU0ZTk2ZTI4OWJmYTgxNzI1MTgyMzQ4YmI0NWIyYjQxZGRkYzBhMzY1MzQxNTJmOWQ4NDZiYmU5MWM2NzMyMjExZjBhOTY0MDVmZjFmM2JjYzQyNjI1MmJiNWQxYThkNjRhZGE1ZWYxYTE1MzE3MTU5MzIwMmU5YmNiYzYxYzdjNjI2NGE0ZjJlMzE1NWU5NDZiODUwNWFlOTJjZDI0MDA3M2Q3YWZmMmQ4YTVkZGNlYzdiZTkwMjZiMDRkYTdlNzA1NTMwMjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ahQse0zmcvKVsI3AgAw-ciLep3G769NayO1sXpXH4_lR-HFLK08tisS8SMnTvnCYIjFoEpwweG-rSNLMlB38NA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210130_112944_25_241c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.548Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjdIbkVMdzdYbWxYbCs4MUU1aitLbWVWekNtT0VyZ3lObWQ2QmtpQyt2TzVtLzZ0S2xmUmltUUJJdFVPUStFMWFWWFBhaW4xL1RvZzM2SVpsWitCRUZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEzMF8xMTI5NDRfMjVfMjQxY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzhjOTRjODQ2NjRjMjU0YzYzYzQzNTM1NGMyNGM4YjFhYTJmYjQ0ZGNmMGMxYzI4OTNjYjQzMDM2NjJhMzFlMTczMzBlYmI1NGM3YTBmOTFhYTAwZjc2ZmNkZDM2OWRiNjI3ZGY4MjM0ZmU0Mzc4NjFhYjUyMjNmMjk5ZjQxMzU0MDdkZTA4OWVhNmE0ODdkYzNjMGVhNDc4NTIzZGQ3NWI2YzEwYmY4NWU0OGU0MDUyZTk1MmQ5YTI3NzBkZjg5M2ExNjM0OGQxMTVkZTZhYjdiOWYxMWJlOWFhMjI4Y2E0MmQ5ODU1Mzg0MThjNWM2YjAwMTNlNjQwNjUyYWVhYjhkN2JkYTc1NWU3YWM0OTM5NDc2YzZiNmUwNTM0YzdiMmRiNGZiZDcxOGFmODQ5OWVlMTY5NTE1N2ZkNzk4NGM2ZjNhZmVkODVmNGE5MTQxMjZkNTY1Mjg5MWM3ZjMxNWZmOTk2Y2I3MzhhODNlYTE3MWM3NWQ3MmNjYjhiOTQxNWJiY2Q5ODNhOGQ3MzJkNGE4ZDA2ZTJlNWMwZmRmYmI5YTliMjYzODU1MDg5NGVhNTg3NGU1MDU5MzQ0YzRlZWI1MWU2YzU5NTBjZDJiM2Y3OTUyZWZmZWVlMDljNTM1YTEyNWVjNjZkMDlmZGQ4YmFiYTdmODM1ZjM5ODYyZDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KnnkrnEQZUeJpUWhps--LhapYkvGfFEemMUHrb0u0K17Dn2FEYQIK8isJvieesU12Y3FjlCaIYVYwNXtZkaOXA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210130_112944_25_241c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.551Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFYTHY1dVZXRjNtNndvc0RCTGJtWG5lZzFOUnVCMFFQdjl6bGZTTnltMXl3cGc1eFpWM2lZNTRHbU00TEkvNFNNdnRWOStMOFhvZXdGZlVSTzRMbnFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEzMF8xMTI5NDRfMjVfMjQxY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGFjODVlY2YwMTA2YTZlZGZlOWNmYzg4MDA0NTJjZGI5NmJkZjRkMWQ2ZWQ2MGUyNjM2OGY4NGNkYjY3ZTA3MDIxMDY3ZjcxZTFkYmM4YWYxOTQyMzUxNjNiZDY2MjI1NmE4ZmRiOGEwOGU3ZmUwNmRlNzNiZWQ2ODZlYjA4MmYzZTU4ZTM1NzVkMzMxZGJmMTcxZWM0ZmExYWZkZDA1NDA2N2MxYmQwMTc3Y2JhNDI1MTA0MTdjZWY5MWE3ZDM2ZDI2ZDkxMjdjMmY5YzJiMDA2ZTI2OGEzNDYwZjIxNzhlY2E4YjcxNzQ5ZWZjYjcxMWRmNmNhNjk1NDNhOGZkZWE2ZmYxN2JmY2QwZThiZjdhNTM2MjM3MmJmOThjN2U3ZWE2NmYzMzQ4MTIzN2QzZDcyYjIyNjZhNDNmZGEyMmRjZDlhZDQ2YjM3NmQzNmQzMWIyZGIxNjdkOGUzZGQ2NmE1OWZiZjVkNGZiYWQ1ZGM0Nzk2ZDFlYmY1ZmU4ODMyZDRkNjMzMzU1YTI5MmY2ZTYzZDVkNjMzY2Y2OTdhZTgyYTg2MTZiZmYyN2NkMDY4ZmJiOWIzMDkxZTNhMThiMjMyYzFiNmUwMmQ1YzRiMWFmZWU3M2U5MjI4ZTQ5ODAyYWZiYzgzN2NhMjNiYmRjOWEwM2I5MTU0ZjlkMzFkZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gjrUZFaN5IGf3GYfygMUZ_tLvB_h0ehUAOqCu9fNEPQhHP1twHKL3Y8VgDrlezx7ql_ntTBjFsPkIEb7T0zfrA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210130_112944_25_241c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.554Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Indudklrb1hUQnFWMXU1aVE1cVpyZEpqWGtTdmFPQTdMNFpod3JrbVJVVFRudklCajRITjF2M1dhZFF3cE9ySEhOa2ZkdUpkMFoxRm5URHFoUVVLamd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEzMF8xMTI5NDRfMjVfMjQxY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWFhOWMyMjY3MzljNzRiZmE4MzMxY2NjNzEzYWM4ZjcyNzViYWRlZjliNmIzODVjNjVhMmIyYjdmZWQyM2IwNmY3N2M4NTk5N2FkMTkwZGQyYzIwNjcyZWI3MzNkMTM2MGQ5MTA5YTc3ODJiMjFlYmViZDllZDRjZjc1YjdmOGQ4ZWM3Njc2Y2E4ZWNhNjhlYzU1NzZjZDlkMGM0NzE4Y2MxNzEzOTlhMmRiYThhNWFlOTE5M2JkOWQzZmM4NzgxMGE4YjhkMmI2YTZiNWQ2MmUxNTRmYTBlMzEyOTQxNDdlMThiM2Y1MDI1NzNhZmU3NTAyNGMxNjBhNjZjYmVlZmViMjY3M2RkOWVkYmY4YWM4MDEyMmM4Y2FjNGZiNmU0N2M5NTQxYWNiZWQxNTZhNzA0ZDJhZDIzYjI4MWZiN2Y2NTMyYWEzMGY0ODdmZmJjZmRjYTFmYTRkZGUxMDZkYmIyZWM0NjhlMTA1ZmRmY2Q0MDgwMDA0MDcyYWNhZGJmODY1NjAzMjU1NWUyZDYzOTU3MTFiOGFlYWUyYmYwYTlmNjA3MmU1YzE5MzkwZjNhNzI3ZjNkNmQ4ZGY1MjNiNGExYjA0YTczZTIxYjIzZWNjNjk1YmIzZTRlOWRlMDkwMjE5YzA3ZmRkMmUzOTJlMzk4N2YyMTQ4MWYyNTZmZmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MUuiLm1cbqDaC46TAHDDqsly8vOR_dYg73_8FXZEdNRg0jTqFISIBjo7LR4o4-6EIgnjLmIfFf8NAh_qBrUrwg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210130_112944_25_241c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.557Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxsekphUGNTQ1Y0Z0wvVzk0UDRmK0RvTDdVa2VBblNmT1NPU3RlRGp4VEJTd2lPMjczbGx3MVBXQW0xYXFUSTZlWVN3Vzg4MU9lT3JSZ2VWbTg1dTJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MTFfODFfMjQ5NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2NlMGI4NTAwMDRhOTkxZGM3ODU0NDdmYzMzMWQxOGU4YTZjMTM1MWIxYWY4MWM4NDMxOWFjNjY2YWRjNzliODIyMjdiYTIyYzhjYmMyMGE2MjQwMDkzZTBjNzQ1NDE5NzU1ZTMxZDdkYWMzOGM5YzUwNjU3YjdjOTA0YzJlZWZlYTljMTRlYjhlNzVjODIxZWRkMTBiYWE4N2ZkZTk4NGJmNWQyOTRmNzUyZGVlNDQzNjFiYWQzNTMzZWRiZWZiZGI3YTZiYTkxOThhYWMxYzc2ZDFjNWM1ZDQ5NmYzM2UwZWVjZDE4NTkxYzI2N2MyMzlhOTAyMjY1NTEyMjZhZTVjOWQ0YzM1OWNmMjIxZmE2MjBiOTBlZDkzNzg2NDA1MmQ4OWQ5ZDQ5NjgwMDNhMDlmNzNjY2FjMjIwMGQ1MWQ0MDg5OGJlMDg2NjRiNzE5NTQxODhkYmNhN2QzYzE4NjVlMTkwMDI0Zjg5ZWQ5NDA0MDgwZGQzYTMwOGFjNGRiZGMyMTk4YThhMGY1ZGZjOGY5NTk0ODg2YzI4M2FmMDAzNjBlZWJjZjg2YmRlYzc1YjI3NTJjYzJhY2Q0NzY2ZjY4ZjllNGZmZWJhZDVhNGFmOTVhMWNkMGVjMWQ5NGY1OTg5ZGJhOTYyYTE2MTg4NzNlNmNhNGRjNThjZGUyODFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vQ6ncJkGlsH42wNeNvDPdVlXGe1x5fakKRujdhZO3s6spQbupNAqgbRq1ISyencOpefk_Y0O4xpMWLJi1KIu4Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105811_81_2495_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.560Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImJFeVFzVnQ3eEljMHFrdkJwUG5Zb2RLVHFjSDNsbm5GWVF1ZjJZYXpDeTdpV2dwaDVGclNMZ1laWVo3SmdPSXhRcWVGVDZzbGVHMi9IM2RWTE10R0p3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MTFfODFfMjQ5NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDM1ZjM3Nzk0MTI4ZDMzODc1ZGI1MGQwZDliYjgzNzVkOWMwMjJkYjkxNTJiMzlkMjc0MmM2NzYxOGQ2NzNiNDg3ZWM5OGU4MzFjYWJiNGFhNDEzNDJhMzE0MTBlNDAyMGY1MzdlYjRiMmVlNjFjYzI1ZGE1ZDJiOGFjNzMwOGIxNTNmODkzMjk2YWYxNWUwYjFhMTNlMjU1YzI0MzA5ODY3NmExMzQ4ZThlMDg5Yjc0Y2MzNDA4MjRiNTQ3ZGIyMGI5NjdlNDAwOTNkZDIzNGIxY2NlYzJiM2Y2YWEwMWUwMmQ3ODBhODFiMGU0OTUxNzBiNGZlMzVmOGQ2NDUwNTllZDE4ZWZiMDNlYmQ0OTlhOGFlMDY1ZTA5NTJmOGUyZGNiMzc4ZmZlMWVlZTJhZWMwNWM1ZGNhNGJkZTI1ZDU3ZmI0MWEyZjYyMDI0MGFhOGNkMTI2ZjI0MGQ5YjBkZDgyZTAwNzU5NzE5ZThjNTcxNmRjODE0ZWU4OGNmZTI3YmExMzhkNjAzMjQ0YTg2OTRkMjYzYjM0NjBhNzZhNWI3NTQzY2RlNmM3ZWE5M2ZiODY5YWQwMjMxOGY1NmE3YmFkMGJiOTM1ZDc5NmNjNGM3ZjgyZTQzZGMzZmUzZTExMmQ1ZTVjZTdlOGYyMGZmNDY2ZmQyNDA5MjVjMzIzODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nbtKtSBi79BqU1sg9IhHF-jhMzwlWFjg0HDLqBQNm7R5cCfcs0NAbMKpk55dJIRXSMmnay6Knx_dPD74h8ecLg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105811_81_2495_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.562Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InZhVVpoVUh5ZW4rUkR3RmoyOXE0OFprQlovMW4xRU43UzlDUmJiWU85N2ZybDl2SVlQZTdRV201QUZJOG9EaHJCNGZrN3pkdEFUOEFUVzVtQWRjNVlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MTFfODFfMjQ5NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjg2MTY3MmVjZWQ3YmU3YWYzMWIyYWZiZTY3NzM0YTI2NzYyZmViNjQyY2NhYWFkMDFiYzIyM2Q1YjVjZDIzNjM3MGMyZWZmZmI0MmQ3ZTZiOWY3Y2U5ZDFhYWYzNzEyYTFjNDA3M2RjOTMzYTc0NGM3NjIxZWJkYjFlODczNWZjODBlMWFmMWE3ZWI5MjI5YjU3MTMwMTM5OTExNjcwYjllZDY5ZmI5OTE1ODdjZWY2MjVhNjUyYjY3M2Q5ZDIwNzVlMTYwYjVkMWRhOTVlZmY2MTgwYTAyMjRhOTMzNTNhNDY5N2Q3NTc4NDk5NWE0NWEwZWVmYzZlMzQ5MzQ3NWRkOTg3MTdmMjMzNzI0NTNjZGY3ZjU5MzUwOTJlNWQ5YzQyYTM0OWM2ODE0OGJkNWY2OTYzMTQyZDdlMjg0NGFkNzY5MjAwMTRmMzc2MTdmMGUwYjM4NzU5MjRhNzgxYWM4MGJiOTBjZDNiZDM4MjlkNzc1YjNjNTllODM2OGFhN2Q5MDY3MzhmODg2ZDJkMjNlZTVhNjQ1ODE4ODk1NjdiOWM3MzVlMGZkYTJkN2U5YTEzNjBiNDBjNGQ5NDFiZmRhZTliZjQ3Y2EzZTcxZjczNGM5ODY1MTM3MzYwZGU5YmM3OWI1MjM1YmI3ZjZjMDRkYmYwOGQ0NjUxMmM1MWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NfFlx0qXtkENiPA8e6yEiC0r6Shgz8aWLGllXGW8zPLoOq6mzdV9vzgqefrX8_pkyh6fMIWkG9-a9zdQiebbtQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105811_81_2495_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.565Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImJJMXk3aWhBZVUwNVpOUFBveUtnak9OMVJSOXNjNEF5MnlaWG1sdklFc2ZGMFNuUWh5WUMrUUhvN1NaSzVYWXJtcmI5RllPYkdCenprOWRFd0JDaTVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyMF8xMDU4MTFfODFfMjQ5NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjQzMzM0MGY2N2IwZTM5ZDIwOWE4ZDkyMzM0MmQ5OGRjYTA5NDg1NzgyNTkxNWE0MDQ5OTNjZjJlNTRkZDQ3ZGY4OTYxMTQxYWVjODU4NGZmNTJkNWZlYTQzY2EyZjA0MTZhY2I0YTM5M2Q1YTYwMmZmNWZlYWZjOTAyMzY4NTQzZjU5ZjVlNzEwMjFlYzA0ZDliNWQ5OTAyOWFlZjRjNjBjODgwOWNmODk4NmFmZDgyOTcxNDc0NjQxZmNiNGJkYmNhNTY5OTk1ZjNhNDU1NWZhOTIyMjgzOTI2ZTlmZDM0NzEwZmQ2MTQwYTZkYTI1MTFjZGY0NjM4MzA3ZjIxN2QxMmZlNmM1MWNkOWU1MTljMmRlYWM3OGJjYjVkMTlkMWI2ZjFjYzU5NzJjYzNhYWYzYzM0YjU4ZGE4ZDEzZTVmMGM2NzI2YTdmYzU3NTA1MDI1ZmRkYjc4ZGIwNmQ1NDBmMDhjZGQ1MmIzMTY4NThiNTllYzM5ZDRjNjBjYTgwODQzMmE5ZjZlNWQ4YzM0MzQ2NzZkODczMjhmMWQ3Y2I1YjE2YTQ1NDE2ZTJiMDJjZWI4MTRiNzJhNzNkYTVhMTUyNDQzOTgxYzlmYjY0MmI3NjcwYTQ0OTMyMjk4YzNkYTI0NzIwZGFlZmNjYTkyMjJmODMzZWMyZTQwMjZiMzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WOjn8T9RU4nH6Va1513ldaPsinsi2gqmGgT9Wvrg4oyCA3O-EPgYtlJCrCtTwJMKmqtVG6yupFhMBljVxe6BTw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220920_105811_81_2495_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.568Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtTQjRwV001N1lyMGxCb2VDeGtxZjB6SEFHTTJ5UGdSWW90L3RXdXp2VzRmTjMvajJyaFNld1k0WkdUL0M5dUt4TnlERWhxR3FmNWdXUy9hSGQzbGJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjBfNTFfMjQwMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDQ1ZjY5ZmFmOTczNDk4OTQ1NTE4ODMyZTVjMGQ4MmRhYWFjODM5Y2E0NTBjOWIxNDEyN2Q0MTI3OTMwZWRhZjIzYTAxOGQ2NTUxOGIyY2FjZjdjZDc4NDk0OThkODE2YTAwYzIxNjdiYmU3YTE5NTQ4YmQ2OWMwZTU3NGNiYzU1YTcwMzc4MTgyYTk3MGM5OTUzNjAxNmQ0NDliNGMwNTE1MGNmY2E5MWI1ZGI5OWUwY2M0NmM2MGU4MjkwYzNlYzE5ZWQ4OTM4OTc1Nzc0ZjQxOTgwYTZhY2I0ZTI1YjNhZTBiY2NmYjk2YmE2YzJlYjYxZmU3NGYwZTZiOWZkODAxY2VjZDI5ZjYwMTI5ZmFjMjMwZWM1ZmY0MzM5ZDRiMDZmNDNjOWUzMWUyMDAxMzI5ZTZmM2JkMTEzZmQzYjQ3MjlmOTEyZjY4Yjg2ODQwNmVkNTZkZWNkZDcyMWRiOTFjYTM2NGY0ZjM1YTIyM2UyNmEyMzFjYjc1NDNiY2YzNjI0MzQwOGUyZmY4NDdlZGFjMGRkODZhMDEwY2IyYWU0OWM1M2NmYjk2M2RkYWU0OTM2YzlhZjViNTI1ZGIwNzVlZDgyNTY1NWE5NGFlNDJlOTE1NDZlNWZhY2MzZjIwOGRiNjRkNzkxNGZmZDhlMWQ2ZjJiNDRkMTg2MDMzODlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wKRdUQv7wsN3C49etYvTniqr5OT3c1YWO3KvW9Y4KAQCJj5w6ul5TqtsaRsov4P-YHkoDX5dE-oY_OntMn3b5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112620_51_2401_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.570Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkY5YzU2RUsvTi9icGRJazlGLzRWNUpCRndkbUpmVEtYMHROUG83RWo2czZzMGFydHNQWUNOTFFwUU5tempVWHhXOUZzZ2RxQ0xReDcxTkw0NDZiU3VBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjBfNTFfMjQwMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQ5OTc4NWFlOWQxNDAzY2QyMmE0MWQyODk0YTkxNGQ5ZWZmODY0ODQwNGI2NzI3NGFlMDlhNDI0MTZiM2VlODBiZmU1MTViNmQzNTg2OWUyOGE3YjJkMmUwNWI4MzNmOGVkMGM0YmUzOTRjMDBhMTExYTkzNzRmOTgxZmJhYmFmYmJmMGJjZWI0MDE4NDFmZmJkZTEyZTFlNDAzMWQyNDk0NDdmYWM3ZGE1ZDk5MTkyOWE0ZjFiNTFmNjg4MWYyN2U5NjRkNzMwZjY4NGU2MWY4Y2Y5NWUyNjAzYWQxMmNmNDg1YWFlODIzMWY2NTQzMTRkODc2OGI1NzUyNzZmYjUwNDkxNjQwMGExYzY2N2FiMTJlMjc4YTljOTI5NmMxNGUzMTZhZjFlZTJmMjJkMWIzMzc1MzllMTE4OGZhZjM5MjllMDcwZDhiZDUyNzIzM2U3ODdkNDJhM2I2ZmM4ODU4YTUxODVlMzQyMDAyNDYwMmJlYTFmOTQ2MGU1MzU5NjM3OTU5MjNkNDQ0ZTRmNDRkZTA2NGZkZWJkMmE4OTFjNmRlOWZlYzE1ZGE2OTJjYmFlNGQxODEwZmIxOWViMWQ2ZWI0MTBiZjMwMTg1NGUyMDViZjdiNGEyMTg5YmU1MTRhNzY4NGNiOGZhYzAzZmUzMmI1ZWE4ZjQ0ZGMyNTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Y80E8jvg_pv9fosnbwU-QgFJXNGMH_rkH37vbsInsR45tWg0kBUTddO9Ck10SGu83MRd8SyxvyXFlr0HlOmpJQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112620_51_2401_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.574Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpLWUl3anBYZ25nSUJPRWovcmxSaGV0T0ZCUGZGbUhUZUJwRlZNTng2Q2owQWsvOG9yU3FyQnNKUEI5cC9qQTNVT21yL1pPTVBhWXdZalZDeFZzMEtRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjBfNTFfMjQwMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDIyYzkxMmNhMTQ0MjcxOTU0MzBiNGM1YjBkMmM4MzVkZjhmNTIzN2YyMjQxMmU5MjhiNjI2YjQwMzU1MjkwMDk1YWQ0NDViOGYwOGMxZTU5MzQyOWUwMDViYjJkNTZkNzZiYTc4MTI1ZDA4MDExMTE4MDM0MzFjYTY4MTliMGFjOTgyNjhlZTc0ZjdjMDRiOWNhNTllMTFiNjkzZDFjYWJiNDI4MWZjY2MyNGNlYTdjYjczMjdlYmRlMWFjZTY3NjU2YTI0NDBmNjIzNTJmOTYxMWM3YjFkZGY2MjRkMTY0ZjU4MzQ1MTI2YjY1NzYwMGI1OWEwZWY4ZDMyMzNkODkzZjY5MzY5MmFmNTYwMzg1YmZhZTFiZjE5ZGY5YTJiZjdjYjljNGY0MGFiMDMyNDM3MTAxZjgxOThjZmY4ZDRkZjI5OWNjOWViNjE5NzAzOTQyNDFjNzFlNWYxZGEwMTUzODJiNDRhMjJjYjVjMGVjYzVkZDFiNzExMjE0OWQ5MDM3NjU0OWUyYjE0YzEyZmU2NWU2OTk5Zjk3YjZhOTk3MzYxNWZiZGRlNzJhMTMyODg4MDQzMzE0ZTQ5MTBjM2FmYjQ0NzBjOWEyZDg2ZDNlY2Y4YmUzNTU1OTI2NDVmOWE5ZDM3M2FiMWFhNzJjNTIxNWI1ZmUzZTk5MWZmMDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fPVz3lqAPwuJ8lcmQIrR-zGv9tp0fvloAaMiNQWGNC2pOsX7d85sjByIKVbrCnarJ1dnYxtu7uJC4LMobHmf4A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112620_51_2401_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.577Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImNYZkhpRUlpc1FwbExxY1ovOVB4aHlCTXhnV3cyS3JGTWtGVy9ZZXdEa3pGQVplUUphZVpuQlRGMGMvdzlCSjk5UTRXUHp0V3cyWXZISThTVVVBYVJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjBfNTFfMjQwMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTA5N2Y2ZDhjMGY3MjBiNzRhNWFjNjBmNTY1NWI4YzBlOTQ3OGUwMWVlMzgyYTA1MDhlN2QyYjNhY2ZmZGY4YzIzMTM3ZGNmM2VkZGU4MTkwYzc5NzM2ZjFiMTY0OWEyYzZhM2I4ZDY3ZGU0OGE0NjA1Y2NhZmQ2NTFlMTAwZjk5NDgyN2FiY2YzYWU3MzYyYmM0NGUxOWQzZGY4MzUyNGZjMWZmOTBkODc5ZTFjMzhlZjQ5YjkwMzdlNWU5MjQ2MjMyZmEyNGQ1Mzc3ZDAzNjg3MjEyODYwNzc5OTMzYzQ3ZDc4MjI5Y2I0MTVlNjY1M2M1ZDQ5N2QwNzllZTM1MjNiODdhN2VlZWI3ODA4ODkzMDE3NTRiNjQ1M2ZiZDY0M2ZiZTE1ZGY4MGRiM2MxYzI5NTAzZmY0NGUzOWI4NWE5NDhjNzQzNDc2NjRjMjRjNjdmODc0NjcxY2YyN2NlMGYyY2FiYWZlYWJkYzkzZTcxMzVmYmE0MjczYzdlMGRhMWIzOGRlYTQyZTdhYWI4ZWU3NmZlYmVjNDRjM2I1YTcxZWVlNTQyMzFmYzFhNWNjM2I2N2FlMjQyNWY3OGIxNWViNDI1NTMwYWY5YTFjODQ4ODdiMmZiOTJhYzMyMWVlYjAxYzg1OGRmZGIxNzA4YmY0NDcyMDdmODNmZTIyMWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eSXLAGyqI712emdCojNmmMfJbir82RMvu45JyP4GovizI-HICYkBZXeg19xUJYDbuZira9aBeH-CgB2PXCXR1w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112620_51_2401_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.581Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InpUdjFiejhCeEFQMjAwRkIycmNEOERyNWRQVnUzOEdCNEdMZ0FwcmZqOXlvU1NaT1M3dFpQeitPNWVtQlBTZ0JtbTBlVkl5Z0NXMXVSYzdyZ2lYYnNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjZfMjVfMjQ0OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjMxZGU1OGQzODJlOWZlMjFkZTNkOTNkNDlkNzM0NzJlYWUwNjczYTcwMjdkYzA0Yzk5M2JkMGQ1YjUxMmZhNmRhOTI0ZGE1NjM2MGJmMmExOWQ5ZjE4ZTI4OWMzZDYwN2YzNGMwMDhkZGRhYWMwZjNjMGUzMDliYzU3ZDJlZWQwNDU4OWQxMmUwNmUxYmQyMTJkYzBiYzBhYWEyMGE2ODA5YzZjMzJlZWIyYzMzMzg1NzEyOWRmMmVlNjE5NTg4ZmI5ZTJiNTNhZjhmMWZiNzc4MTBhNzVhMTEwYmQ3ZWQ0NDVhOTE3NjY5ZGE1NjAyMzNiMDA4NmE5Mjc0YzA1NDJlY2JmY2E0NDQ5YWYyNDc0ZDdhYjEwNGI2Nzg0ZDNkN2NlMTEwZDc5NWM0NDQzZGY1ZGY1ZDNlNzI4NDU5ZDc0ZjA0ZDQxMjhhOWM4ZGZjMzk5NzkyMzhlOGU2OWE1N2I1Y2RmNDZmNGFlMjI3OGRmZTdmNzZlNjFlNDA0ZmQyOGFmMDdjMjk0OTVmY2Y4NjE4OGFjYTcxMzdmNjc4MjAwMmIwZTJhYTQ1MjhlMjVlZDE4NTdjMDQwMTBlNWMzNGI0YjhmMGNmMmRlMDVlOTZlMjQyZDg2YmNhNTJiY2IwZGE1MDA1MGUzMGIwMGU2MTU4NGNjNGY4MDNmZjNhNzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FihskaybHjDofoOPn_WWKvIMaurr0WEp96TDXjWwUDrBV2OQFQr0baIWg_Tn0b62hrKwQOhPxQDHvqAGTf6J2Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103426_25_2449_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.584Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlZ0N0Q2Q2U4ZlNuZGdaRGNzV1lPVXlTbHh2b1pyd1hpQkhJc01LYWJ1d2xEOCsyWC92S01uNGx4SW9DWFk4eWJPTk9TOWhlSWZ1YnF4TXFNZVVac0NBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjZfMjVfMjQ0OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGJmNTE4NjEyZmYyMTZkNmVkMTkzODE5ZmYzMGFkNTk4ZTJlMGJjMWQ3NzFjMThlYzU2M2FjYjJhOGYzODdjNWEyODhkYzkyNTI3YWE1NzMzYzQwODhhYTM4MWE3MTY1NDc2MTFiZmUyYTdjZjEzYWU1NTlmYzZmNjBhZTVmZmI0MWUyMmYzYjZjNWM3MDIzODM0OGJhYmE1NzBhYzM1YmEwYjhjMzJhOWMxMWUwMDg5Yzg5YTkyYmIyNDBiNzY2M2ZjNjVkZTczMmU0OWUxMjI1ZmVmZjBjZTY4MDc2YjQ5YTYyNzBhYTkxZDhlODVkZWFiMDM4ZDVhMzA4NzlmNjhiZDk0ODMyNGE0ZTY2M2ZkZmQyNWNiYTc0MDJjOGE0ZjExZjAwYmViNGNiZTYzZGJjNWU4NWNjYjI1NDAzMzgyYzU1YmQwNTQ1ODAxOWQ1MTFkNDJiYTY1ZjllYjUxNTkxZjdhMWNhYjQxYWY3OWNjNWEzMTY3ZGFkYTZlNjQyM2FlM2I0ZTdjNmRhOTY3M2ZjMTAxZGJhN2Y0NjQwMTVhMmUzZGI2ODczMjNkZDRjMGNjMWQ2Mjc3ZDYyOTliY2VmNGU2MmVmNDE5MjliMTlhYzBkODllZWJmNjg4ZDUzNWI0MDkzZmZmZDYwYWM1YmU5Mjk3OTQ2MDY1OGVkNGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.A1sLFIUgcdPxYQHR1zkRkB2iMrl13svLAC7uxsYa4Njga2gDVfuElM776tsK1r5w_nuJ1tcQZVzrrjtR5frAcg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103426_25_2449_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.587Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii8xZDJucU5lUG9jWDVhTkR2b1ZHTlNCdEp3ckswUnBtWVVqZndxWndmODNESFRiOSsrdTRjelZUSjhwWlBoOUduOVpYU1hocnloeElFS1ZGcktHcTJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjZfMjVfMjQ0OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzBiZDE5NzAxYzMzNzRhNjZlYTU4ODExNzFhZThiODVmODdkMzRiZDRmMDVmZmY1YjMxOTZkMWU0OTQ4MmZlYjQxZjVlYjY2OTZiZDhjMWQzNGZjYWUyYjdhNmJiODQ1MjY3YzAyMGQ1MThmYTI5NGQ5NDMwYTBiNGE2NjNmNDNkMDc5OGU5OTI3MmJlYzMzZjBkMzdiNTEwMDJjZGJhOGNmNDVmZTRlMjBhNmU0ZjlhMjc5NTgzMzlhNzY2MGU0MGVlNmZiZTc3YWRiNTQwMmFmYjRhNGNmZDJkODkwN2Y5NzdiODc3YTc1MTRkODE2NjI4NTBhMzc4ODk4ZTMxODYxMjNjOWZmYzU0NDMwYTE3ZmE0ZWMwNjVhZTM4YWM0N2ExMzhhM2Q4YzcwZDEyNmY4NjUwNGVhMDg0ZDgzNWU0MDM1YzEyZWJkZjJkNzdhODA3Mjk1ZWZhZTk5MzAyYWM0ODEzM2E4NjU0MTg2MzQzN2JlYjM0MTFmNmMxNjkyZjNmYTAwNmIzZjlmYzZlMzc4MTMxMmI2ZTI0NDYwNTYzZjUxOGQ5YmU3NzRkYjRmZTMyMjlkZmY1YmRjYzZiMTdjMDZhMmY3ZjBlZjBlYmRkN2JlMjFlYzA2OWFlMjY5OGEzOTg2ZjZmNmFmYTRlNzI1OWVlN2YxNzU3ZDEzMjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TDpP6Yz05kY3wQiE2EazwqiUryfQ4dcapXoE15bCKhCrnA9QTLG97S8rYXy5XT-Uav6hYvEIP9fN_pX2Koqqng", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103426_25_2449_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.589Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkVQV056TS9RRG1laU9lWS8yaUExeElXalphUTVNTE83M2RWa0dZSGRTdUs1QmdYUVRYMy9JL3FVcU9GODZ5N1hVSUhwbENNRnpOZHdod21qc0RxL29nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyMV8xMDM0MjZfMjVfMjQ0OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWFjMjMxNGZmMDBhYzBkNzBjZTBlYzQxMGZhZTc3Zjk1ZDJjMGY1YjRjYTk2Y2NkN2Q0MGEzMTZkNGVmZmRjN2Q2MTM1ZjdlMTgwZjc1ZjYwOWMzN2Q0MDk3OWEyODI0MzhiZGYzODI4OGY0NzJkYWJjNTI3MWU5Nzg2ZTc2ZmEzMDE5MDAyYzcwMmNhZWM1ZmUyNGM2ZjVhMDdiZDg0YTFiYTg1ZGM2MWE0MjdiNTZhMzVkNDVhZGQxMjVlOGZlZmQ3OWJkNTAwMTQ1MWQ0ZTIwY2Q2Y2MyZmY0ZTBhOWQyZjIxZWIwMjY4MmZjNmNjNTRlYmMyM2FlZGE3ZjdkNjkyMDA5MzY5MDA5ZTg1YjE4NTc0MzMyN2ZlYmU3M2U0OTgzOTBiZTkyM2FhMzM5ZDdmNWMyYTI0Y2VmN2I5ZDkwNGIyYWI0Y2IzNTI0MWJmOGI0MzgyNDM5YzA0NDJjNDUxMDhkNWNmYmVkMTY3MDc1Mzg5OWUyZTJhNWM5MmY3M2M2YjlmMGE4ZmQ4MmM3ZTkyYjA0YTVlMDUxZGUyYzQ4MzViMjJhNzUyZGEyMGE5Mjg3MzNiN2VhMjNhOTI4MTY4YzJmMGY2OTlhYTJlMTIwMzA2YmVlZDEzNzk0MWJkN2Y1YzNhMWNlZTdiMDExM2JmMmQxMTUxMWMyNGQ5ZGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._ZC6Et2RUSQwfsle4lP_Yh9yX3kblNyvd__oyaiMZZ3xayKzhQngJjw2StUI_sKapK-gZO82ahparS6Z9f47lA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211121_103426_25_2449_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.592Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InUxL3JZaG81R0NMblUyTWVOWFNYRG5Lbm9Gd1pJNGtjRlZTSk1CdlJNaVJKZjhyK3AzN0R4VGpxeHJzdHFzOVdPdUZPb1VPSHZHV3pqOGJzMVM4aWlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyOF8xMTE4MzJfNThfMjQyNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjgwYmQyNWQ1MDU5ZTcwOGFmZjdjNTU2NGMzYWFlYmMzMTI3ODk4ZGNlMWQ2NGYyNjIyNTAyMjQ2MGIyN2RmN2E4YTZiMWQxNTc2NDc0ZDNjMjZlNzM3NzNiZTk3MzJlMGNmMzk3MDMxYWQwNmQ0M2U0YmVlOTNkNzkxYjVkMzk3MTAyNmU1YzYxMTA4N2NkMTYwMGNkOTAyMTM2MjY5ZGY5ZDEyYmY3YjI4YzM3NWQwM2M4ZjY5YmVmYTAxNjFhOWUzZDQ4MzgzYWI3MzYyY2RmYzg3YWZjMDQ0NzZkNTBmMWIxN2JhODUyOTYwNzdkYjcwZWZhYjI4MzA3NjcxYWZjOTBlMjJiOWQ4NGZhNTU4OTQwMzc3MTAzNTEzYTFhYjgyNjBhNDQ2YThiZjYwZjAyMzJhZDIxOTgxNzA5MGYyNjMyZWZmZjdhMDQyMTJiYTE2YWNkZjVhMTkyNWI2YmE4OWFmMDBkYTZkNzcwNmM1MjJmM2MwY2I1MDlkY2JkYjYwYTVmY2VlYzhiYzM4ODYwYWNlYzU0NzlmMzBkNGY2ZmIyYzUyNTZmOGJiMTE0MTY4MzEzOWU3ZDcwNDY0NGM3NTVmYzljMGJhYTVkNmI4N2EwZDZlMjc5YzIyZTA4NGNlZjk1YjdlMTk0MzA4ZDlmNmM2YzYzYWE1NTQ4NzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Upnh_1s7S50-u2phoESzlTJjB9CwbpC60BdN8_z7PjGZZGcaM7LYzXBObpN_jOjJ_z1WrkxM-MIuxeVXon-jRA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220328_111832_58_2424_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.595Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InV4bW43MTdmTjJyRWE4QmxpVWVZS2dZZEdFU2lPQUtqQ01Ubm9ReTRCZ1BFVUVTQU1ubmJrZk85QUdONWtHek5kakJiWFlTVUtUTkV6N0ZnVDJEMzdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyOF8xMTE4MzJfNThfMjQyNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTQ3M2JhMzUxM2NiOTUzZjUyNTZhODNjZDVhN2FiZDk4ODdhYjE4MjVkYzdmOGE2Yzk4ZmNjMWQ1YTM0ZGJhZWZhYjFlMGFiM2ZmOWFmODhkYmY1N2RjMGU0Y2E5OWZlODdiNzBlZWEzYzAyMDE0NmRkMGM0ZTIyY2Q5MTBhMjQ3NmY5NTExNzI5OWM2YTQ2ZjE4MjhiODc3ZWVlMzBmMjcwMjIzZDVhZjBlMjFhMDQxODdkYzU4NDZmZDkwYmFjN2U5ODIyYTNiMmVjNmQ2ZGU1ZDZiNWI2OWY1MWI4MDkxZTk4ODJhOWIzZGQ1YWY4ZmY0ZmQ4MGU3Y2QyOTBhYWIwZGIyZGJhM2FhMjZkNDQ5ZDY3MzRiZjFkMGQ3NjkxNWFmNjMzMjhkMjhhMGE2NjFjM2ZiYTllN2VjMDhmNGYyMWVhMWJmNjMwNDEyMmNlMGU5NzY0OWZkMGExMmM2MjUzNGZiYmJlYmY2N2NhZmE2YjEzNzg4M2I4MmFiNjZlNTk3Y2YxNWIxY2RlZTYxYjZlNWM4NWRjZDY4ZDQzNjVjYzM2ZmMzMzk1ZTA2ZTQ0ODE0YmRkZjYxMDgxNGVhNGM2ZTA0YTUzYjIzZTY5NTAxZjJiZGVhYThmZmZmMmE5MjgyMjNlZDIzOTgwNzQzY2Y1NjEyMDM4ODlmNGNjYTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.f0MnGcUywQDQQ7vmTuB58aSZR9fctiQ6KeXQiT2RpBgO2GqSZCqc_ydRLriY72Zrt_Z0dseCJ4Lg0UBQKVmn_Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220328_111832_58_2424_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.598Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjJVVmQvVmZWY2JFRkRucnVoQWdmTHY3cnZXZ3IvL3h5ekpYalNJYkJGRXNJWmlmNVVMMU1JVjdMV1BmZ2Fmenl6ZUtOUk9XK0w0U2o1ODQ3bjYvRWlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyOF8xMTE4MzJfNThfMjQyNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjk0NjRiOThlZTczZGI2MTM3NjhjMmQyZjBkMjdkNmM1YmI2MjJhODA5NGZlMjIzMTE2NTgxOWQ4MTZjMzk3YTVkMWUxOWM2OTMyMTIxN2RkOWNjMGFhODk2MGMyYTM3OTAzYjBkNzQxYTMxNWU4M2IyMDQyMzI2NThjNDRjNmRhNjY3YTcxMmU5Njc3MDY0NTUyNmJkYjRjNzEzYWZmNmVkZmFhNWQwOWFlMTUzZTM2ZTlmZmVhZGU2ZGYzMDk5MzU4NTAxNmY0YjI1NmEzMjAwMWE4NjQ0NjI3M2ZkMmE1NGNlMDVmYTdjMGZlMGE4NTgwYmYwYzc3MTFkYTQzMjMxODE1YjE2ODBmY2JiYWQxZWQxYTM2N2Y3NjA3NTA4M2U2ZmQyYzAyY2U3MDBkNjk2YjdhOWIwZTA0NGJhZDQwNjVkMWVlNzE5NTJlNWY2MWJiNmY3OGExOTlhMTAyZWUyMGI5YzBkOGExYjBjM2ZkZWU3ZGE0MGI4ZjhjNThlZmVjOWUxNjkzZGIwZDgyYWRjNTA1MjQyMmI4ZDQwNDdhMzQyMjJiMWJmZjcxMTYzYjI2MTI4NmRkNmJiZjk5OTNiMTQ3YzY1MmY2YjNkYWQxMGJiMGUwNWQ5NWI0MmFhZmQ0MDUzYjNjM2IzNjcyOTFjNGM3OGIwYWExMDUyMmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.U02jE2Y2u6fyMsZz29udsnRG-qTFG1b_BTsIdI5j15tsLyoE3cCem4DK9jUpsdidulLJC8PR2H-6Aq73HNKOVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220328_111832_58_2424_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.367Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjMyU2JzVVd1UWpTY0plZVZCMkx4SWZLWk5zMTJDNVNOUU5lcFNkVnpFQVl6N2RIbUF6NUtXZnZJNzNQZm5uK0N1L1dvdVVTdGk2Ly9LMGI5Q0dBc3NnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyOF8xMTE4MzJfNThfMjQyNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDQ5MjQ1OWQ0MGMyZjlhYzUxNTczNWNhMjkzZjc3MTI0OGY4MjVmOGVhNzZkMjYzNTM5NTg0MTgzZDdkMDgyMGUyMjUyZjA0ZmY2YjI2MWY0YjYzYzI0NWE2YWYwMzYyOTE0ZGVhZTAyN2RjZDJiZThjYjc3NDAwZTExNmI4ZmI1NjAyM2M0MjZhNDk5OTZhNGY0MmQzNzEyMzE5ODQ0NzQzZGMzMTVjZDRjY2U2YmQwODZiNGFjZTBhMzRhMjFhMDk2NjgxYWExNTk2Y2FjYWRmZjIyZTE0YjdlMGEzNWFmN2U0NWMxMDM0MGI4NzJiN2JmNmM1YTA2MTNlMWU4NWEzM2E2N2RmZGM3YjYxOWRkNGUxOWZmMTUzOTAzMTM1Yjg1ZmUyN2JjNzA4NTk1NmY3OTFmMjk4MGI0YTMzNTRhZTYwZTZmMTlkMGNiNzkyZGUzZmVmZjUxZTc3YWFhMmI1MTI3YjMwMjQ3N2ViYWI5MTJlOTRmYmY5MjAzZmRkMTNjOGFkYzE5ZmU5YjVmYmU5NTdhODE3ZWNmNGUyNWE3ZDlkZWQ2YjRhYzVkZjdmYWU1Y2M0YTdlMWQzNmE3MDBhNDhiODc5YzEzMmE5NzExZGJmNTZhMjE0ZTc3M2ZjNzJhODRjYTU5YTg5ZGQzNWFiMTI0MzEyMjllNWQ4ZjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gvoJIXKgfZsmSx411dVDwhDJcbr0XCskFT1vEB7BBI4ZDQ-MzkOCpSkX4HB5QCK7B9owWTVuuXiHukNnXkvGAg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220328_111832_58_2424_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.370Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImpPcXgyWnZqMGtnOENHNDNEZnYwM3hDaHB4bXFzYzY0SjNkOTlBVjNEWXRnWEh5T1BVYTV3OU5tam1jekpDQ2xsa04yWmRFMFFOMjdMQ1VvMWlGVVl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI2MzVfNzdfMjQ1OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQ2Zjc2MWYyM2ZhOGY5ZDc0N2MxNzBlZjUyOThkNjk1OGE2NjI3MGZiMmEwZGMwMmZjNzNhNjU2OTlkOGExMzNjNWUzOWU2ZjUxYWMyNWU0ZWZlZjdhMDRmNDg1N2FjMDYxMGZhYzJmOTU1MDM4ODg1NTU5MWMxNWE1ZDg5YmVkODIyZjQ5NWM1YWVjMjQxOGE0NGViMjlhOGFhODM3ZTRhOTg3OTMxZTllOTc2YjlmNzgwNDNmNzg0MjE2N2RhNzM2NmQ3ZjNhOGM2MGMzN2VjMmU2M2QyNzk5ZWE4OGVhZjcwYTI0OTFjMzQ2ZDAzZWFhYzA4YWMwYzhhYmJhMmVmMWI2ZWQyZjQ4YzBhYzY5MDlkMzA0ZDY4MDcxMTdhZmUwOGEzNGExNTgwYTliN2M5NmJiNzE4MWE4MmQ3NmM2YTNmZWE0OWRlNjhhZDEwYzYwYWRjNDExYmRkZGNhM2IwODAzNTVhODEwYTkwYWY5Mzc0ZDUwYTVmZjFlMjBhN2IwZDc4ZmNhNzVmMjg1YTgyODEzOTg5MGYwYzgyNzYxNGI0MTkwZTY4ZGNjN2IzMmQzNWQyN2Y2NTY5OWU1MTIyMTBjMDUxNjgwN2Q0NDVkYWNiM2E4ODY3OTA4NjIwNTNhMGQxYTI4ZmQzNGIwYjAzOTIzYjVjZGQ0MTdmOTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8Ph8I3VPxJzc9qXihn9gWakJ1VOqv7tM71zWpRO0o_idVDUKWP373sj7CBiQenUoKowryXMDdzleEXlcfL-EBQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102635_77_2459_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.373Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkI4aGt4K0Z6d3BacS8vWGp1d2RZV0dSV1llRGRjZEZhdFRoZkdhWmZIWTRqOXJ4ZWdLb203TDdUSHlBakt0Z2grVE5kVzVxMjYrdDhtU29EbThBb1V3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI2MzVfNzdfMjQ1OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTA1Y2ZmMWU2NjU5ZGYxNTM5NTg2MjEzNWFlMDhlMWRjZDNiYWE0N2UyOWI2NDE5ZWI1YjkwMzYxNGNlNDhlMjJjOGFhYjQ3MmZmMDY1MDg4MDhhZGMxOTEyYWUyMDlhZWUxNTIzNjk0YzcyMDhjNTg4MjlkZTc5OTY1ODQ5MWZmOGQwMjAwYzQ1NzgxOTUxYmM4ODIzOGE1MjNlNDg1MzM4YTNmZGZkZWY5MDdlOGY0NDlmYzAyZmEzYmVkM2I4YTczYzg2ZjJkMGIzNTU2YWJhMzVhYjY2MDY2YzUxZjJjYTUyYjdjMTc1NGU4ZTMxODRjZmU0NWVmY2RmMDE4ZDlmOTlhMjMzYTM5YTMzMmE2YTA1M2I0MDgyNGQ2MDA5ZTU4YzlkMmMyYjJhOGM1ZDU1Yjc2MmI3YTQ1MzRlNzZiNWI5ZDVhMWJkODIwZTZjZmRmZWRhNzZlMGQyYWJkNzU0OTgzNzEzN2UxMTlhNzg1N2JhOGU5MmIxZDMxNWNlMDkyYTc5MzdiMDhmZWVhYzYzOTY0OGE3OGI1MzEyNThjNWZkMmRjYTlkZTc4ZTk3ODE0MTMwNjc3ZTdjZmFjMTBkNDZkYzcwNThlNThjYWQzNzgzOTJjMDIxYjY3MWE0ZDMwNDAyYTRhNzI3OGY2Mjc2MTZiYzFmMWExMDllNjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LXoYgoouf5PagdsWnY3UYWZLQrKruQ8tMtZegS9AnJVA7tWRY3U2REUNZ5aKlt7U-reuW1Y7ka7QHe-EsVWIxw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102635_77_2459_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.379Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImxVV3lMTGFSY29mL2RCaURjRkRjdnhsSnp1TzByUTE1K0FubzZEUjJTbERRV0tSUTBRZDRUWkVKWG9lTm5ldjQvdUJYM0NjSkUwQytKRHIzVFd2REt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI2MzVfNzdfMjQ1OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTllNWNlYTk0OGJmMmM4YTVkYzMyZTg2OGVhNWQ0NTllZmMzMTNiM2FhZTY1MGQ5YjRmMmQwMGRkOTJiZGQ5MWVhOWZlYWRhNWMwZGU1ZjE0N2JjNjdmMTc2MjdjYjRhYjFhYjEyM2IwYTkxN2I3MzNlNTIyZjFlYzlkMGY0NGE3OTkyZjAxNDNkZDJlOGM1MTI4OWM0MjRhZGM5ZTZmOTViMmJhZTcxZDRiZDUzYTk4NmUxNDBhYjQ5N2RiYzJmY2EzYzg0YTc1NDBhM2E1YWM4ODEzZWM3NGQyNTY4Njk5N2JmZjY3YTU4N2Q2MzFkM2Q5YTk4NDZiZWI3MzU1MWY3ZjYyMGYyMzZlYjJhNWRhMzU5YWUyMmZhZWYyYzQ1NjU2OGNkNmM1MDNmN2Y4OGM2MGRjYTIwOGQ4ZTI4YjZmMzgzYzgxODI2YzczNGZiYmFkZTVhNGEwMzE1MjhjYWE0MzQxOTU3YTc1ZjJhYmM3ZWY3YWVlZmRkOGZlY2RlYjYwNmE0OTk1ZjZlMTJiYmIzZjdmYWJiYmZmYTRkNjE2YTNiYTVmNjZlYzZhZTQyNzY4ZDM4NjMxYjVkNzc1NzI2MjllODQ1NDkzZjBmNGU2ZmM5OTY3MTU5ODYxMzhkZTBmOGU0Zjk2Y2UxODA3MmNjNWZjNTc0NWViYmQ4Y2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8lzeDLBlB91_tB_ICn2_cze8tHVEN_JDqDvaB3jpCS92xw6_YLFJ9C9cwS1VIDt3YgSU8DYAHDFi6MoGEqkwTQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102635_77_2459_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.384Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImhwSHgrODk0cElyeW4zcnAxTDRPVy9PNTl2elVrTm03dXRQczVVYlozL0J3aCtqMlBBTUJJS2xkVWs3S01nUTZGekg5MDBwL1lUNTBvdS9VM09ZWTNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxNl8xMDI2MzVfNzdfMjQ1OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDJiNTljYWQzNWYwOWFlN2VmZmU3ZTZjMTU3OTNmNmFkNGE4NDUyM2Q3ZGU0YzQ2Njg4M2QxOWVkYTZhMDM2YzE0NTMyY2NhMmU0MmJlMGYwYjRiNTBjMmE3NWNmNjI5YTQ2YWQ3ODk1YWY1NDBlYzg3ZDMwZmU1ZDdkY2JkZTY2MDVhOGFhMGIwZWNkNTI5ZDk5MzE1NmI4MWM1ZjA5ZTVhM2FjYjBlNjdhODg4OWQ1ODgwNjE3YmM2MzM2OGVjMzMyZjZmMDRhMWVjOTk4OGUxM2Q4NjI4M2ZiZmM2ZjNkZjNiNmI0Nzg5M2I2NGVhNzVmYjdmODcwODk1NjkwYjY1MmM0Yjk5YTRlNjNmODhiZGE3NWIyODUzYjkwN2E0NjBjNGUzM2MzY2I2ZjIxNzdhN2Q4Yjg4M2U0MTczMTY1MzczMDIzYmQ4MGRkMDdjNzRiNzVlODIyODYzZjUwNWJlOTA1NDM3MjdhYWFkMzc3YjNjODZhZjc0NDc5MjA2NzdkYzNhZDViY2U0YzU4MzdiMzdjZjcyYjMwOGU0OWFiOGY2ZWE2OTk4Yjg4NGMwOGIzODE5YjNmZTgzMjM5ZWNiMmQyZDRjNzM0OTczOTdiYTU0NGNmMzhlY2IxZTMyMzQzM2FlYWQyMTM1YTM3NGFmZGU3Y2U3YTFkNTljYjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JzgtMvo1Dj6gKx29LxsmsT2HdeBQO7yvBlSPMOiatKuCyAucD5pmlLD12iD8KFGRDwvhVCcVEkXXCcgjxGYb4Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230616_102635_77_2459_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.387Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InI1RHFPaGJtT2pzbWRia3lMdi9aNlhaVDVVb2FQZG5tUk53ZW5HeGxBM05OeGVpS0pFNkRGeWR4WXhUSXh5dkdUMGw1RTE2QW9PaHBDaU9JRU1Jb2lRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNF8xMTA3NTFfMjNfMjI3YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWZkZTkwMGJmMjE5ZDRjZWY1N2UwNGMzMDk5ZDM3ZTNhMGZmYTA3NDBiNTQxNGVmYzE3OGI4MWFkZDViNmNmZDg2YzBmYjI1YzIyM2ExZjhkZTA2ZWFiODIxMTczOTI3MGMyNmM2NDE5NzczMDgyODRhYTRjYmIwMTdkNGE3ZTMzNTAwYmRlOWZhOWE4OWQ5N2I0N2UwN2IzOTUzNDY5MTlkYTExNGM3MzFhZjhiNjI2Y2Q4MzA0YmEzNDFmZTRlMTMyNTgxMTk4NTdkMGUxODQzMDNmYTJkMmQ1ZWM2M2UyYTUwMzE3NGE5ZTIzNmI3OTA4NGU3ODYzZjY3NTY5Y2E3Y2RjYTgwNjY2YWQ1MWZkZTc4MTYwZDBkYzMyZTQxNmE2N2U2ODkxOWQwZjM2OTZmZTMwMzE1NzA1Yzk2ODAxZTM1MjdkZmUxMTc3MDVkNDQ5OWIxNDEyNzcxNjBmYjgzYWJmOGNiYzUyNTM4ZTZhNzA3ZDBhNzMyYzQzMjgzMmMyNmFmZjY5OWNlZTlkNmZhZTFmMjQ1NGZmNjc5Mzk1MzcxM2UxZTZlNDVkNGE4M2FlMDkwMzQwODcxYTQxOGU1N2EwMTFkZmZhY2NlODE2ZjljZGFkYjBkODk4YzhjMGZmOTc4NGUzNjgyMzIzOTc1MjBhMTc5MTAzZGI2M2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DTM4cyxpsU8rz1o6YZ2g1fo5gdOYH0UYxhvBUbqVrsQoiEhzgJMBZ5cL6eOVQ84N-G-a8zchht5QvlVEwc1lYA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230114_110751_23_227a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.390Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InpHdW0wRVBnalVEamtaNTRTNmJIKzJHbHFveFZiQ0c1bi9mMTIzMzZ1NllkaGs5cUdnMUIrZzNtQ3dDRFE3U3BmRVNMelNWQkNXYTZxRlM5OFhRU2dRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNF8xMTA3NTFfMjNfMjI3YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmQ3NmNiZmY1N2I1MzNhMjVlYjc3ZDFjOGE5OWQyY2U4OGIyYzI0ZWRkMWExNzY3NjkxNjNjZTc3YzU4YTZjNTQxOGRiNjk5NDY5NTVhMTM3MTRkZTM5ZDZiYmRkZWY1MTdlNDM1NWZhMGIzZjM1NWQ2OWVkY2U1M2VlZjBiZWZiNTM1ZGVlOTA3Yzc2MWM5NTU4MmRlMzUyMzkwOTMyYTBkOGUwNWNlMTc2MjE1YzM5OTRiNGVjYWViYTE5N2I2M2E2Zjg5MjRiYTZlZmQxZWQxMzhhYzc1ZWU1ZDJlMjg2ZTM3ZDBkYTM5NmVjN2UyYmVmMzg4MWE3N2MwZjQwNjViZWE4NzU1Y2VmMTIzNzU0Mzg1ODYxYzhlZmUzZTcxOGI4OWEyMzUxMDEwZjYxMTIzZTJkOWE2YjI4MmVhYmMzODAxNzdhNjg3YTg2NWVjMTJkZjExNGE3Y2M2OTYwOTljYzdiZjMyNWQ1OGU2ODZmYjBiZDZjY2JlZTMwYjNmZWM0MDMyZWNhNzM2NGFmMmIxNTljYzFmMDJhY2M1MTQzN2M5MGJiNGU2YjQxYWJkYTI0OTU2YmYyMDVhYzUzODhjZDYyMTIyNWY4NTIwNzFiNGE2ZTE3MTNhYmMxMjA3YWQ3NDhkNDYxMzVjZGI5NTU4NjY0NGYwZWI4NmRkZWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nzg_TuWYgFGghZP1Ndqv7nZjtko2S82lOG1XDYhP4ckVU253iIKoSmzSJkCZNO_QVsOfL8hC9hZeEyOB0oE8-A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230114_110751_23_227a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.393Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjhjVzNpVldteVlPb2ZVeURFTCtHSWVoc1ROdmxZZkUxK3FYTVVVenIzU3lwcEE0OUU0U0RWSEVJaGNPTjVRZ0NCczdVQ3g0TkNkR2xXcWVDRXNXYWdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNF8xMTA3NTFfMjNfMjI3YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODIzYjFiNzkxYWU2NjAyYjNjMDhhM2VmOWI3ZTY5YmE4ZTc2MTIyZGU5NmUwNmRjY2RiOGNjZjQxNmIwZDFiYjM5MDVjN2I5MzdiNmYzMGQ1YTFkODM2NjVjNjM4Yjg5ODJjZThkZDdkNTZlMjNjOGE4MDcwOWQ3MzVlMTMxNzM3Mzc3MTljZWM5NDNlNTJhZWMwNTU1ZmU5MDAyMzY0YjBkMjNlODQwOTgyNTk4M2FhZTc2YTliMDk0OTJlNGY3NGQ0OWEwNDBjNzg1YjQzNmFmNzdlNjdmNGFlNzliYWZiYTI5ZTQ4Y2Y2ZDE5ODZlMWUxM2IzMjU4NmUzYWNlYzhhMTkxN2FmYThlM2I4NWVjNWI0OTM2YjRiNGFhZmNiNTY0OGNjMGMwOTgwNGUxZWM5NTI4Nzc4MTFmYmYwMTU4Nzc0ZTI3ODAzZTEzYzcwYzJiMzRiN2JhZmJmODQ4OGMyZTQwZGE5ZjNjM2RmN2QxYjZjMDM0MzhiMzBlZWZlNmQwZmM3YzM3NTU1NTQ0NWViZWFlY2UxNWQwOWZiYWNjMWQxNjk4YjcyMTRkZTFlZTBkYTEzMzMxODlhYTJmNjVkMjljNDZhOTgzOTNkNTU0ZGZhOTJmY2JhZWJhYmI4YmMyYjQ3NDlkNmNjYjU4NGNiMjg2YWQzMjYzZWYxNTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JekdstfC5hzWb0wkQTTP-ZY_0sAca0rgN5tan6KSxKbMeRkvWeg4-yHtBvAvKcPpuMz4_kJuL90TDQy1VdpinA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230114_110751_23_227a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.397Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlArS08wMnRZSlc2czV0UzhjVjJDall3VzhOajh2MnpuaUlqSVZQaHhUYTFnVnAxWHZRZkV0M2F4MTk1aVh4b3dkaXhkQ1N2QlRHVkR2SG1MdzVQY0x3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDExNF8xMTA3NTFfMjNfMjI3YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjUwZGU4MzEwYmMwMzczODkyMmVmNDUwZjdiY2E0ZmEzMmUzNjdkN2E1YmMzNzE1MDBkMDE3M2M3YzJiZmQ3YjdhMmViZmM5ZGQ4OWY4MWZlNTE3MzY3ZDYxN2Y2ZjdmNjJmN2JmY2E1MGUzNjM0NjkzODU5NTNhMjUwODQyYWYxYzg2ZjU5MDc5MTY0ZDEyN2FiY2U1MThhZTRhYjM0M2ZlYTgzYjdhNTlhZjFlMTc5OGFhNTk0YWM5ZDYxZDRlZjAxNzA4ZTI0ZjU1MWM0Nzk2MDFhM2RhZTI2ZTE5MzgxYzQ5ZTY3ZjQ5M2Y1ZmNiZDI2YWI0NjBiOTc4ZDkzNTBiMzllNTc1MjQwODUyMDFhZTVmZWE0MGJkNmNkZjBlYWY5MTZlN2YxOWE5ZDJlZjhjMzQ0YTM3YTc0OTQxMjU1MjY3NmM0M2UxN2RhNjQ5ZjNkNzgzNTNlMThiYjMyZjY5YTQ2M2FjZGY4ZWQ4Mjk4MTc0NjU2YTY3YzJlYjBmNzFmZDViOTlmYjk1NWQ4NzMwNjY0N2Y3ZTMzZTc2NTRlODY5MmI5OTM0OTdkNmFmZmRhNWY0OWZiMzYyNjkwN2E1Y2MyMTUyNjI0ODJhNGM1YjVlZTM3YjAwMWQ2Mzc4Y2Y1OTI1NWMzNWU3ZGQzMjRjZWM4MjlhYTgyMWNkOTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3on-IfdbN0jrrRMpMlPQdWn4x_TTy9L1fvM4ZOu0VnfwkN7kw0pH4f3w31ulc-Ce7Rs8n2hqeeVKfj6EzipYbw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230114_110751_23_227a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.404Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlUySjFVUWJ6bk1Ob0NucVJMbGhuZ3Q1RlFSZUVQVVdueGJyL0lPei9jNzluSkIrSm1WVlBWMFRUU2o3ZkJNOUxlRDJkRldsa3kvLzhLdG0xclluSlFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyOV8xMDIxNDdfNjNfMjQyYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWZhMzdjMjdhYTgwZTkxZmNkYzY1MWE5M2YzM2M1OGVjOWJjMmFkZTZmNzllNjRiMDZiOGVlYThhMGMyNmRiOGRjNWU5NDQ1M2E5ZTc1YzA4MTU5MjgwZDIyZmM5NzE3YmY2MDMzMGZjZWM1MDdhYjhjMjg5MTFhMmZjNWFjNjRjN2E1OGQwMzY2NjQzN2ZmMjI3NmNiNGFhMDUzMTNiN2YzZWE3NDY5ZmE0YWJhZDM2OGUyMDQwYzdhY2MyZjk3MTU0ZjA1YjU3NzE5N2FlNWMxNTVhOWZhODQxMTliNmQ0ZTIxZDE5YWFkNjkzNDA4ZDI3MjE0ZGQ3OGY3MTM1MTc1YmJiNTE0MGY0YTI5OTlmM2FhNTJiMTdmYzIxNjczYTA3NDBiMTdhYmZjYzgwYWVkMTI1OTdjNDdhYjc1Mzc0NjM4NWFhMWY1ZmNiYzQwOTMzZThlZDcxZjM5MDIzZmU1MjdiNjc2MzhkZTg0NTE1Y2E0ZjJmMjE5ZDhjNDhiNzVlOGMyMWJmYzJlZDcwOTgyNWE3YzBmODM0ZGY0M2JkZTFjYWQ5ZjU1YWM1YWM1NTI5NmJlNGIyM2ZhNmU4MmUxYmIxYmJkODhiMzUxODc2YzAwN2Q1MzU5M2MzNTJkZGE4M2QxNjRlOWQ3MzEzMDcyMmJkZGEwMjdjNDM0NTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EqMHPZ-Qbmx3gk4gaz6LPxvwu8Q8gM4QEyCoCAZpDz_HtF8mKuRJqHaclEJ7XI635Fnww9Ei55pxj3T4GJjjSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230529_102147_63_242b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.407Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImJDWGJIQWN6V2VVN1U5SWhxSzlWS255WUZnanBlaXNKUlNPUzgvM2NCUmpMVkdOSWczWTJ0UHBpRXlVUzk3VjYyMEwrSGQzMmZBc01VZXNyOS9TakZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyOV8xMDIxNDdfNjNfMjQyYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI5MTgwZjNmYzAzNGVjODNlOWM4YWJiMGRkMTJmMDU1MmQwZDQ0NjA5ZjI4ZmViYWVkODdlYWM0NmNkZDM5YjMwODI5NjM1NmZmZGUyY2JlOTE0OWUxNzQyMGFhNTkxZDRlZmU2NTY5MGM3MTA4ZTliNmRkNzIyOGU0YTZmODQzZTM2YTMwNWM2ZTY3MWM2ZjI1M2I1ZWU0ZGNjY2JhNWNiN2I3Yzk5NTZkN2FmYjc1NThlNzhhMDhiYzNlYWJiOTI5YTM2MzRjM2E5ZmM1YzczMDQzZWU3NjQwODZlZDVkODA2MGViNzlmYmM0ZGM4OWE3NDljOTFiM2FmY2MzMTcxNWE3NDU3MWQzODdhNGQyMWFjYjdmMzZlMmViZjU0YjhhZGQ2MjdmOWIzNDM2NmQ3ZDMwNjNmNDQyMjVjNWNlYjNlOGExMTdjMDVmOGVhNzkxYTExZjdiNzIzZDE5YTZhYzQyMjE2ZGRkNmU0NDc0MzBlNTBjYjU0NTgzZjg4ZWU2MjM2ZjM2ZmYxOTY2Y2UzOWYxMjZiMjQ3NjkyNWNmNzM3NjI4NTY1OWI3YjBjYWNhMjhlMDZlZTViZWU1YWFmNWY5NDIxYTkwYTgxZTY5MmYyMTZjODZiOGZjODY0NjBlNGZjY2RlNjM0ODJjYzNjZjVhNmJiNjYyMzczMjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HugWNcPPSKaxoJqQ2KlGYSQ0Q4xVrl-BpZ2p0txrjWoRT9xxJJ_ZlYC18dCviwITH2smenLEJw5or4Z056exPg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230529_102147_63_242b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.409Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlZjaXN2b2dydEdCM3diY3c2KzVyWGZ1em9ESVZVSktQdC9HTUs0WEJzS29DZHZ3ekdkbi90L1pjbnJ2WGdmYmYxazFycTVmazN2RlgrZUJZYjNuMG1nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyOV8xMDIxNDdfNjNfMjQyYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTIxYjgzNjFhZjhiNTI5Y2ZmZWQ3MGFlZGQ4OGYzNDY4MDg3NTU2Mjc2ZjAyMjY0OTYxNWE0ODJjMDI4N2JhOThhOTFhNDAyODE5NWQxZTYwMGRmMTMwYmY1YjJmNzkwNjA5OGJlYWE2ZWRjMjA5YjcwMzk0ZDI3ODg5YmRmNzk0NWNiMzYzZjI1MTVmOGQ1ZTI4ODk1ODc0NjkzNzI0ODQxNDU3ZTVmNjU5YjQ3OWViOGNkNDUwMTM3ZmIzNzBkNDM5Y2E5MThiZjM5ZjY4YTM1YzI1YTcyZDY5OTFmNmU4MmI4NDk0MmEyMGNjZWI0ZjQwNTQxOWNlMjAzMTM4ZGMzYzVmOTM1ZTYxOGQwYWQ3NjhkNjE2MjU0YWFjMzZjNDkxOGM2MTVhODg4NDdkYTE2OTE2ZGE3ODE4MzRkNjEyOWUyZjIyZTcyNTg0NzZkMDZjZTAyOGQyZGMzZmVkNTQyZTM3Y2FlNDI4NzgxNGNlMzg1ODcwMzhlN2E1ODNkNGM5YmQzODBlZTJjNGNhZmI0MzU3ZjI3ZDdiYjk4ZWNiMzFiYTg2ZmI3YzNmYzQ0YzNjMTA5OWJmYTJjOWI0NDE2NzY2N2Y1YzBlM2QxMDgzYTk3MTg3ZTcyNDNiNzk5Yzc0MGY1MzRhNjI3NzExNjhjYzIwZDU5MjQ5OTk0YWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KNnY5h6itim3083nCan1GZlwR2HrwdLfnyxUqYgHeBRlcmw-g4hKkCFOSsH1hfjX4M1buueqOTqTH2KO0czy7Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230529_102147_63_242b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.412Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImlHTDZXemY5NzM3d0t3NUwyQ0hBR1U2cTFQNnNZQ21YT0l0OUltRHZsVitSdTgrdkFPd29hSjU2dklPK0hYN2dvOTFRNWFCNTJFK2JUamYraVA0QnFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyOV8xMDIxNDdfNjNfMjQyYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTNiNDdlNWFhNTExOTI4YzNjNWY1ODg4ZGMzY2RlOGI4OWJmMmZiYTJhNTZjNzA5N2MzNmY2NTRmN2Y4NzUxNDBjNDc2N2M5ZDRiYjA4MjA0N2Q1YTBjMjgzMDllNGQwMzhkMTg5MTgxOTJmZmIyNjViOTdjYmNkOGFkNzA5YTQ4YTA1MTAyOTEyYzhkZDViZTNkMTI4Yjg1MzU4NDE2OWM4ZWU4ZGVkZDFkZGUzMTc4Y2FlNjVlZjVjMDcwOTY2ZDU5NjQwMTVmOWNlMGU1MzYwYzNkYjA1M2U4ZGIzOWEwZjhmOWI2MWFiNjA2Y2MzNTI5NmQ5NjU0MTIyZjJlYzc1NDkwMmJhOWEwNmU3MTJmODllMGQ2ZmJiMmYzOWI3YWM0YmExNmE0ZGI1MDQ2ODI3ZmJiNGU1MjU0NGE3NzAwY2I4MDNiMjBjYzNmNTQ0NTA5NjEwZjg5ODk2NzczODFlM2IxNDkyOWFmMDI1ZDdhNDFlMjZiMDdiOWI1NDliZGY2NTIyOGViZWY1MDNmN2ZjMGQzNWVkNmQ0OWIzOTJjZWMxY2VlNmE0OWE0ZmE0N2Q4YzQ3YTM1ZWYzMmNhMTEyZTM5M2Q4MDI4MjU1ODM3NDNhYzc3M2MxNzExZjc4NDEwNTJhODRhOGVlNzk1OTcwZjEzMzhmM2MyNmMzNTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-qjemvpU6owaid1G_B7WUEEonCQ3m5gz6KCgaHHeogzuq3AreoBxezlt2i_POn_5ts2URLHlpk9pmnPZuOrjoA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230529_102147_63_242b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.415Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImdzMVFOUk9FTTdZby83R25vS1h6SWQyU3pwNVhnbTlFYUYzQUlJdTZQZFlhaWlvejFrUG5FUnI4YW9EV1ZybEdycHEzMzVCUEZDdE5JRXpMV29acWFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMl8xMDI4MTlfNTdfMjQyN18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTM4OTk1ZGIwOWYyMWY0MDg4NmY4ZDg0NWFmNjVmYTY2MjkyNWI2NmE2M2NhZTAwNWE1MzE1MWFkNmE0ODQ4YjQ2ZTI2NTM3YTgzYTNhNDEwYTlmMDJmZThmN2VmNzk4MTZmMDJkNDM1MDg5MWZhZTQxOTg0MGU2OTkxOTZmZGUxODExYTA2OTAwNjA5NjNlNThiMmE1MzU1NWRlMDgyNjEyNWMwZTdkOWM4ZTIxM2Q5OGMxMzE0NTQ0MzAzMmQ1MzU3ZWVjYjI1YmViNGEyMzU0YzZjY2FlMWQwMzk5YTRlNzJjODNjMzYyZTVjMGE0YWE0NTMyNDBkOTYzYjQ1MmQxOGMwMzI3ODU2ZDljZDJjNjExNmM2ZDQyM2U5YzMxY2UzZTY5ZjE4N2U1YmEyYWNhY2ExZjY0OTVlZTliODU4N2Y4NjY4MjNkMGFmNTczMjNjYjE3ZDRhNDkyODdiNGNlYmRlNGFmMzIwNjUzNzhhYjJmYjBiNTZhODRkZDNkM2RmOTVlMWY5N2MwOTdjNDUxZWQ5NmQzMmIyOTIxNmRjMTM2N2RjOWFiMDlkMDA3ZWE4ZDdhMDljMjFlMTBlODAzNTMxMWM0MzEwMDllNDBhMzkyMDk2M2U3Njc3MGMyYjBmM2Q5OGUxZjNhODI1OTZhODM0N2Q5NjQyYjEwNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Pej5hOx_OvNrf3F2gVYzi27LjB8lnEk8LdSTFmp6vFtomP0bSKuFbiJabHZEynD4jGQ_oL_FTh_ZhN032tKB6g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220322_102819_57_2427_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.420Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InBRZERZSUtrcEM5NHVncnJsQVhreUdRWWduSkUrNlZGdmhzeWt6L2pFS1FLV2pld0xpL041eGxuVWtGSXpzV2pZVGJzTG53ZkJxK1JqKzVoN3dOY3NnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMl8xMDI4MTlfNTdfMjQyN19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTE3ODAyMWJmNGNkZTE0ZDZkNDg2YmZjNTkyNTk5ZWRlMjNkNWYwNjJmODRlMGNiYTI2NWE2NTYxMTAzZWRhNTEwZmYwZWVkODliZTUyZDA4OGNiMzI1ZTMzNDVjYTA1ZTliNWJjMDhkODcyNmJjZDYxMDVkODg4MmZhNjExZGRmYzYwOGZlMjlkYmVmY2MxNmU1ZTRmOWY4NjJhODQzNTBlMGFiZjc5N2ZkNTRhODlhNjBhNDZiZmYwM2Q5OTc3N2JmMTQyOWQ2ZGM5YWE3MjRlYmE4ZjYxNmVmZWM1OWQzYjI5ZWI2NTFkOGVjODdiODRkMTU3ZjBmNGQyYmMzNGZmZjMzZWRhYzNkYzVjZjU2NjE5MGRhOTRlNjFhMGMxMmIzNTI1YTcyZDJiYTJkNGVmMWM4MGQzY2E3OTczZWRjZjUwNjdkYTRmMzY0YzY3OThjNDM0MGI3ZmE3NzE0MGIwZWQyZGRlMmYyNjk2NjcwMzMyMjllNTI1NzJiNTA3YTVhZmRkNWMyN2IwZDU3N2ZlZTEzNmJkNTRlNDc0ODc1NzQ5NmQ1YTczZjEwOTI4M2E4NmFmZGZlYjI0NTFiYTE4NDg2N2E2MTEwMDRhOWMxOGRiMjhhMjJiY2FmM2QyMTdhYTVlOWZjYWI3ZDhhMzVmY2EwNzJmZjU1NzI4NzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QH3bboK6Hzf9rReiNqcfG_rAGbhq-VVle9FvolfB6ioW-U5NQtYcwjjxtE2lIXgiJtBFx5ZFnusAGrBtWrzsmQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220322_102819_57_2427_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.423Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IldOMi9xUHRkc0JlQjRsTXRKMm9MSWFxTFBXbE9GTnk1R2RORVdiTDdXNFEyOC9TY2NYQW9kbGYyVGZxbHJzLzc3bUhlSURnc2xyOFBCYmE5VVJSNGJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMl8xMDI4MTlfNTdfMjQyN18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGZlODU1Mjg1MzU4MjQ2NTg4OGU3ODBlM2IyMzc4YWZjODFhY2ZkMGE1MmZjNjRjMjRlYTVmYzRlMjI2YzQ3ZmE2ZDY4YWZmMmZhNTcxMDcyYmI4Y2ViZDIyZGZkMjEzYjYwYzQ2ZGU0OWRhMjZkYjZkOTFlMmZiYTRlZTI2MDZjNTEzYTgzNjI5MDVjNWY3NmJmMmVkNDQwZmQ4ZDM2MTg3MTRiNjc4MzI4YjU2Nzg0YjhkYWRmYzE5OGI5MTIzOGEwNGYyYTI2ZmY3YjgzN2NhY2Q4MDYyYjM2MGZmYTk5YmQyNDVmNjAzMDZlZDhhMGJlOTkxMjhjM2QxMTUwNDNlMzAyYWUzYzY0YmFjYjg1YzcyMzg4ZDUzNDliNGRmYTZiZmM4YjYwYzE3YWVhYWE4ZjUyNjNiMTEyNTQwNDQ2ZmNjNjY4NDU2MzJlNTNmYmU5NmE0ZTM3MGQyZWVhNjZjYWIzYjc2YTM5Njk0YzA4NTZiMDYxMTJkZDc3N2EyZjY5ZGYwMWExZDM0MWYzYWE5MDNjOTk1YTJjMWM1OGVjNmZkY2ZmNzJjZjNiYjIwZjU1NGZhYmU4YmIzNDExMmZhODQxNzU0MTRjZWJmNzYzMDMwNmY3YmQyNTQ2YzM1ZjJkY2E1NDZmODlhN2Q1NmMwOWYwMGNhOWQ1MGNhYWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oTkyUkRguu7qcrY1Aop8AJMlJAtp3mpPHNRYrl21BqwAuFNA5c6MfzAIB1r9UQoItabH3AJanfrXTOIlVUvLVQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220322_102819_57_2427_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.426Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjBkQSsxd2o1WCswR2VVd2V5ZFQyTmNCUnVWZ1BhU2pldy9ia1VtWnNUOVRmTGFPemg0M1RwcHRQYnlqd3dkdlRZcFlEQm1neFgxeU5SRjRUbXJYRkVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyMl8xMDI4MTlfNTdfMjQyN18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmZjNzQ4ZTJhMGI0MTg5ZTg3Yzc5YmMzZmQ0MDEzYmM4ZjllNTM5MWFkOTMxODhjNzkzYWY0NjY4ZWY5MzRjMzhmYjJjZTZhYThiNWVhNWMyYjMyNzdhZWY3ZTU4YTk5OTRlY2Q5NDQyYTU0NWNlYmU5MDA1MmFlMTViZGJkZDA3YTcxMDk1ZGQ5MGNhNTcwZGRjNjRiODIyYzVjYTA1ZjkyMzNmYTUyYWZiYzFkZTRjZDc4NzhlNGU5OWQxYWZkMWU0ODA3Njc4OWU5OWQyYjViZDg3YmI0MDE3NWIwOWFiZTJiMjhjMzQxZDBiY2NjN2NmOGU5MTAwODQzNmFmYWVmNmIxNDBiZmI1ZDFlNzAzZjk5NDRkYjM2NTZiM2Q2MmQ1NTdjNDFlYWYzMWI5ZGM5MmZiOGNiZTQ3OWFhOWRlMjA3NzRiN2VlOWYyMTI3NmUxYzQ5NWU0NTk0MDczNDBjNDhlNTIxYjljOTc2YjVkMzBlMWY2MjA3MjJiYWZkYTc2OGI0YWQ1ZjFmZTc5YTZhMDAwOGM2ZTZkN2UwYzgzMGMyZGRkYzBmZWIxZmYwNjZjMThlYjAwNzg5YzMxMzFiMzJjMzliN2I1OTY3NDQxYTg2YzkwNzBjZDIxMGJlMTYzMWYyOGUxOTA0YzliYzI0Yzg0ZmNjNDJmYzZlMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zHiCSrQfWmqHI5Fak75VlYEKP7xMC52CJ_9Ylhjq5WLf4tYSTSSXWOUpTlUu2O8ki8ceR63C1O3sXCpQ0nYOLQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220322_102819_57_2427_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.428Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjdJZ0FLZFlkdWtXL0Y4ekl2ODl1cjEyQ3dnK3ZZL0wwYy91MCtSeGlhVjNzTjZINVZ5amxNSmdha0tJZmlwTldwVjh0c0VXR0ZMMmxGTE5KQ0MvM1dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI0NDZfMjJfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGE5MGMxMGE1YTM4ODc0Mjc3OTcyN2I1YzZmMTA2OTU1ODI0Y2Q4MGQ3NmU2NmQ0NDI1MTE4MDM5ZGIzYTE3NTMzZmQzMjIzNjM1MDhhNTk3ODRiYmM3NmU0Mzg4ZDY1NzIxNDFmZTQzY2I2MDE0ODYzYzFhNTE4NGE5ZGI4MjQxNjJlZjlhZjk3YTM5MGU4ODBjODY2YTM3MzkyNTJhMmI3ZjhkODMyOWMwNzc2YzBkZTA3MDUzZDM3YmFlYmM5NGRmZjhmMmZmYWZhZmFmZWZiNWFkYWM5NGM4NTYyNmZmMzAwMjJlZTNmYmFjMzAzYWFlZTU5MjEwM2NhOWE2ZWE1M2UxMDczZGYyYjZhYjE4MzZjYTg0OTNhMTQ5MGMyYzQ0ZWNhZmRiYjE0YTIwMDY2OTMyMzU5NGUwNTFmNWIyYzVjNjU0ZThjM2Y5YTE2MjBhMDljZThhMmY0NWM4YTdhMzg0OWQwYjUxY2ViMTQ1NzhiN2ZjNTQyNzhlNTVlZTBjNDA2YzZiOTMyOGMxODIwNjI0MWQwMzQzNDAwODY0NTllNWU1Njk2NjY4ZjRmM2Q3YTkxMTUxNTljYzNiNTBhYWVhOWQxNGEwYjBhYTMyZTUzMzk3ODMzMWEwN2E5MDY3YmEyNjFjMjdmMzZiZjc4NjRkOWY4NGI5ZjJjNzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xrVwWmE0AsxfAX2FkhxemtWzcPbW_bwZ8RGFmUkcWWvq6T595Sw5MxntcruDsDDQHEnaGXiRDn1iPJIDAz_uxw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112446_22_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.432Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImJvWlFVczdjVTN2bXFmVlVnVHQ0K3dBN0M2UWNRWWVoVVZPNnZaeHVma1dOZ1BQbTBVV0drcnRTZ29HQUtHOFBWdG5GNEZNNFZmWVpxVFdQdm5ibUlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI0NDZfMjJfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzZhMDg3OWRlMmFkNDVkYmNhZTdhMmM3YzVhZmM0YjY4MWQ5MjkzMGJhMjQ1ZTI3MmE1ZDdjNTc0N2E3MGE2NWNjZmI4NjJmN2ZkYTU2MmMxZWMzMGVjOWE0ODI5NDg1Yjg1OTU1M2U3NmEzZTU3YjZlZmQwYWU5ZDJjZGY1OTgyZTY0Y2I4MmY0NzBhY2QyNWY3Mzk2NWVjNGQ1ZTc1YTQwMTE3MzM0NmE4YTU0YzFhYzIyMmVkZGMzYTU5NGY0Yjc4YzIyZmIwYWJiMzk3MTI0YjU1ODJmNmM1NDBjNzA2NmFlMzk3YjgwZWMzYzQ5MWEyNjczMDFjOGEzOGI3N2E2M2RiZjRhOTZlMDMzNTY2NDM4NTlmZTMyYjJlY2I4MjhmZjE0ZjEzOTBlOTdmNTRlMmU2ZmNiYzYzNjM4ZjE0NzkxMDAyZWVmNGUzY2M5MmY1YzUxZWJiMDdmYmFjZTA0YTVlOTEzNjI1ZGNmZTc4MDgyMDczMzljOTA4ZmY4M2U3M2RmZTQ0NTI5ODcxOTY1ZDY1Yzg5NjY2NjRiNmFmZmY3YzU4ZjM5MzE2YzE0M2JhZTQ5ZTQ5NTNjNWIxYTIwMDYzNjI4ZmJlNGY3OTJmMTM0ZjhlNWQyZjc2NmIyYTE0ZTJjMjIzZDk4NmU4YTA0MjU4YTE0NTcyODgwNjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jVHTmz-1MUV21NZLLooC2mYUFljjdILTlZ3juKtnKD0EOJz33WIHY87IjmQUUsvnu8hdm8PDpIBjcXsvDvHi4Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112446_22_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.436Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlBDY3ZUOUxtQnlSQkNlc1JqZVBsMVJLd0hIaWp4RUdOeXFPTzFYb0RReWg4ZGh6R1VXdmczMndpSmxmUFNmOFora2N2b1l4YzNJSDQ2eHN2dlluaGVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI0NDZfMjJfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzVjYjUxNjZkNjk4MmYyYzY4ZDM5YjljMjZmNjRmODk1MWI1NWNmYzIzNjA2YWQ0NDM2M2YxZjFhZjI1YTRhZTIyOWJiYjc5Y2IxMzBkZGJiOTY0NzI3NWVhZGRlYmM2MDdkOTY1MmQyMTFmZGRhNGM0NzQyMDAxOGY0ZmRmMWQ2ZTcwOTEyNjllNTMzNzFkN2RjYTc4NjA5MzJjZDQ2MGY1OTBjZDAzMWVmODEzZDIyMTU4OWY2OTM1NmQyZmE0OGFlZTk2Nzc3ODBkZjMxMGEzNTg3M2Q1NjY4NzIzYjVhODk1ZGEyNGU5YTRkOGMyMWIxMWRjZGRiZGY5ZDFiMmM1OGI5NGM4ZWY1ZjY4NjA0YTQzMzE4ZGUzOThlZmRkMjUyZTAxOTA4OTBmYjczNzQ4ODI5MmZmYzJlNTM2ZjBhNDBjMzJmOGMyN2M5MjMxOWVkM2M1MTMyNTBkMzY4NTc4NWM1NjU0OThmMzliYjg5ZjM0ZDgxYzg0MWI0ODJiZGVkODc3YmEwYWE1YjRkYTM0OWQ4YjZmMWQ4MzYzN2UxMDU3ZTcyMDRmYTFlNTkxNjRmYzE5ZjUyNTIzZDhlNGI3MDBiZDIzZjZmZWJmNThkOWZlZTE3NjJlNDAyOWNiZjlmOTE5OWIwNzg2ZGM2YWQ0ZDUxMzgxYzEyNTlkOTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CjexfFxORdEVFT1jLTt8VowjHDeO0_hr7EQZFMwJ7gkKcUun7dqwmuxUDYSdlSjl1GM2IniyiXpat5S3SIrKdQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112446_22_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.441Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlB4ZTQ2UkIrK3B4dDBFVDMvRFBaV3YrSHNFa1hmVzY1cW13bmtRWnI3WThidFNtek9FbEU3OHQ5a3Z1VWdkOTNOTFNsbUZKTnZ4eGc1WXFmVTFzallBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI0NDZfMjJfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWUyNTBhMWJjOTczMTVhNDkzNjU2NTk1YWY5Y2Q0MDNjYzEyZjVmNjcxYTgyZDFkOTBmZWNlZTM2MGFkOGFkM2JjNGUxNzg0ZDAyYWRjMmY4MDk1ZmU4MjUzMDU1Y2RmNDgzMzVmNWJjYmZhZDg4YTNhMDllYThjMWU2YjJiYzllYWRmZTQ4ZWY2MmIxNGQ5ODllODNjZDExNTBjMDgxNjRhN2JlNWM3MTFiODA2ZWJiNTczNjhmNTJjNWNkNjJiYTc4MDFiNGFmODQ5NGVjYTcxZGZmNWYwMjMzN2IxNWEyMTI0ZWRkNGM4ZTVlYTQyOTQ2MjQwZDE3M2I3ZGQ3YTJjZTg1OGY1NzA5ZTMxNmJlNmYzNzMzMjkyNjMwMWU1ZDIwYTg1MjFjODI2ZDM5MWI2YTk3YTc1YzBiZGY3ZThiZTA3YzkyZjgxMGRlZDRhYTcxOGRkNjU0MzY2Njk4YTAyMmI3YmFiY2Q1OGM4ZjQ3NTFhNGMwMmQ5MTE1ODQwYTM0ZmZlZjhkOTgwM2QzZWQyMjY3MTQ4NDAxMWUwYjEwOGFlNDgwMThhZWJiMzM1YTQ4MWQ2NWQ4ZTkyOWIzNzIyZDRlNzBhOTlmNWNiOGRhZTIxNTBiZWE1NjY2MmExZTFiMThhNmQyOTVkMmJkMmU5ZGZjNDgzOTk2MDdjZDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4C7DF0HTGEATzjo6Trg74BlP6nuaX_7GI1y2Dy_zj5Qq7mac0wz-E2xCa3hqOd_SfJqZPqEHNhxhHf6GvT1Hkw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112446_22_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.444Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlNKcC9GamhEcm9Ld25EM3N3VEpjY1N0Mm1UNDJKOFlCMlF6c0s2amM3ZWhLRktJM0lhcEJIYUt6Q1RkTDM5V09tUVYyb09pOUhNYm0yUk9ZRDhiZ2JBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDQ1MTJfNzBfMjIzM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjEyYWRhMjk3NGUwYjg2ODQxZTAyZTM2ZWQ1YzZkYzU1YmM0Mzg1Yzg2ZDJjODk1NjdjNDMxMDVjNjI4YjhmZDdlNDA1Zjg2OTVjN2Y2NGRhYTY2NWU3NzEzYzRkMzY0M2UwM2IxZGYxZTQyYmRmNGRmNTY4N2MzNTcxZjIyMDBlMzlhOGRkN2UzZDg3MzU2M2VmZjRmODg2MDdlYzJiOTBmOTkwYjI0MTZjZTA2Y2FjYjhhZjA3ZWMwOGY2ZWMyZmM0NjBiZTc1ZDljZGY3MGIwZGI2NGE5OTIxNzNjNDNhMjk4YjcwM2UwZmQyZDJlNGNlZTQ5ZTMyZDQ2NTcxZTUzZWY3OWRjYzIyMmE3OGExMDMyZWEyYzcxODY0MGMzYWE0Zjk2ZGNkNTFhYzFjYzRhYjM4Y2MxMjEzODNkNDA3ODI5ZGRhZmVlMTY4ZmQzNThkMDFiZGU0ZTUzMTdjOWNhNjVjNjNmYjMyZjRiOTczNThlNmM2YTFkMzlhNTUxYWQ1ZmM5MmM0MDQwOTNhODIxODc2MDdmNzNlZGUwMTMzYmM3ZmEwODY0NTVhZGFiNGE3NGFjYjVmZGViOWE4Y2U3ZTU0NGFhYWUyZWRjYmRlMzAzMWM4NzIwNjRkMjAwNTZhZmNhNjQyZDM0YWU0MDg1YjcyZTRmOWFlMTFkMjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jsOSZPJnDmFdAOd32heJ5tejTMAk7uIancLSX7ztdchnFiYAbFQljfGXmy50jrVqCaC9mO_CUpBpqp8OmhqoWA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_104512_70_2233_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.447Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImpqeUNMOEJBSnJoNDVLOVo2WGtTa3h5aklRT0lwL1Z0VFF3TWc2NFhtVFBWbWwzQUwxaDRTNUhRTnV2bG0rRmYxRGliYTh0VVROZlZlOEo3Z3dWb2lnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDQ1MTJfNzBfMjIzM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDk0M2YxMWQ3NDNiOTJmZjhiMzcwNzIzZmZmZDk2NjM0MGJjOTk5MWRjOWE0ZDkwMDM0MmVmM2QwZGY2NDU1YjNkYzhmMjI0MjcwZDA5YzQyMTY4YzczNDFjNmMyYjM2YTMzYmViN2IxOGZiM2Q5NzA3MDI3MWM4YjFjYWE3NjQ1MmNhYzA2MzEwZjY0NGRjMjNkNGUyMDI1ZTM4NDZkMzBlMjU3Y2MzMzZmNGYwYWEyYWM4MDM5YTNiZDZkNGViODNhM2M5ODVjNjVmODAwNTNjNTg2Mjc1NmJlOWFhNzk4NDE5MjUyZWNiYmU5NGVkZjYyNzg0ZjRlOWY0MWE2NGE2NDEzNGIwOTg1MTFhZTc5MzkwZGZhZDViOTk0MzYwZTdkY2NmNDg3NjhjMmU2OTdjNGZhNWNkYzFlYzYzYjFkNjIwMDZiMzBhNmM1OTU1YmZlMDcyYjAwMDIyYjlmOTM0YzRjZTVmODRiMWNiYWMxNDY3MmI1MmNjNDExMjlmZThiY2YxM2ZmYzgxMzJkZjdhNGQ5MmUzN2E5Y2I2YzFmNGI0YjRiN2RiNDk4MjJiNWY0MTc3YmYxYzEwYTFhZGYwNzFkNGIzMWMzMDM4MDY4ODZmNDk4ODYxNTk3MzJiMWExODM3M2QxYmZhNzMzNTgwM2RjM2Q0ZWM4YmZmYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iChRD0Hr6Snai29bT6WApm9RZ-OTRcvD1wpUbLvIkYZ4eBVPvuuZbciuKD6O-ILRy1TF4ptDMA3oqp1Tq1bDXw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_104512_70_2233_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.450Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlhEUDFBcU5RZkZLZ3FjckpOcnIrWnNVQXNxVHhoRnI0R3Z3S2R4YmV2VVFTR2ZNQU5TZktjN2tLcjRhV1NHb2lnU1NGbkxwR3VxOE9pY3Fsa3NlRFhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDQ1MTJfNzBfMjIzM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjM5Mjc3M2FjZDcwMTM3OTQ0ZDYwYWIxYzY4ZTU2ZGVjZTI0NDNiNWUwMWE3ZDQyNzI1ZjMzYjVjN2ZiZWU5MmY2YzBjOWM5OGNmOWJiNTVjYmQzMTUxOTFhNWUxNWQwYzg3Nzg4NWE5MzUwNTFlMjRjNzQ4MDdkMjRjZTA0MTVhNDY5MmYyY2JiZjY2OThiMjhhYmJiNTBmNjNjNzFhMTY2NGUxOWU4NzFhMWNkM2NhOTU1MGZhYTc1MjFkMjM5NGRhYjI4NzVhMzI3NWM3Y2I3OWJmMTEyY2I1MzdlOWZkMjdlYjk1NDQ1YzBlNjI1ODY1NWMwZjI4YzM3NjY4OTI2YzYwNjliOGVhYTY1NTZiMWY3YmMyMzczZDY0MjcyZWI4NDgxMmJiMjcxZGZkYjNiM2MxMzIzODFjNTg1MzI3NDE3NTViODNhMzdhMDIzNjczOTI1YjQ3OTFlZjZhNjAzNDgyYTI1MjMxNmUzMGQ0ZTQ1MGI5NjFiMGZiYzI3OTY5YWZiZTkzZGNjMWI2M2E1NmI2NzljZWNmZGVjYzY0NTI1MWMxYmYyNjIyMjUxY2Y5NGY1MjA4ZDRkYWIwYTk2YmVkYTk3NTMxMDIwOTdhNjc4YjUxYWE0MzFlNzhlYTRmYTQ1ZmNlYmY1NzJmZTRjMWU2ZTdmZTVmOGYyZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1MyMZ10Vbu9mEmEyabf54bam9kY-gvkNgFDudnnoj3qWRwEAaPPlRwT3X1MJjPgGfH8tRu5CZXG7crR8aDxTAA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_104512_70_2233_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.453Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlEwbHpJbFBQeTVFNlNzNDN2TUo5U3VobFdSS000MVYwdGdya2ZNbmMyV1M4VmloRjN3UzU4REJLdHhqSlJZaFN2VjNESDlnZDVsMzNTcE0vMjdNai93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDQ1MTJfNzBfMjIzM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjFmMmE3YTMyYjEwMWU0MGY5MTBkNTdhMGQ5NWI1MGI3YWY4NThkNDFmMGFhMGU1NzVlYzhjYjA5NDc3NzUxY2I3ZmM5NzU2MjY4OTZhNGEwNjZmYzVjMmJkYjk5MzE1NmUwMDY0YjhlOTNkMGI5NDhkMGNmMmNhOTUzOWFhNTlkZTI2ODFjMzlmN2FkNDA1NWU1Y2VkZWIxNTk4ZmNiNjRmN2ZkOTQ5N2VkYjI0YmVhYmE4MTk2YWIyNGJmYjMyNzkwNWU4ODQ3ODg5MjEzMWM5OThjNmNlZmY4YjlkMTc3YWM3N2I4NTdmNzNiYjQwNzM3NzM5YTgyYzllYWYxMjVjNTU3MTY5NTUxYzMyN2VhOGUwYTdmNDM1YTVhYjk4ZDVhOGMwYWViZjZmNjNlOGVlYTc5OThkNDEwNjllYTFhODI2Y2IyMzZkMjA1YzVhNGJkYWQxZDg4ZjkyNWY5ZDllMmFlOTc4Njg4ZGM3YmE4ODkzNzMyMzVmMDVlNDI2MjQ0YjIwZDQxOTU5N2ZjZTUwNjk3ZmY5ZDQ1NzdkNDMzZDQ3Yzk1YWY0N2FjY2EzOWQ2NDdmODY3M2M4NThjNzAzY2IwZDI3NDhkMGIwZWMzZTYzMjc3Njc2OGM0NmJjMjU1MTYxMDA2MDE0Yjc5OTA1Nzk1MjgwOWI2MDE5MDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HCUrrWhLfXAhR00-FetLwVArO2z-WJaCn1a6QkrKbU8LJm2zarOUiLoItBfXBbdKQyr2wF2XNhWAfeq_A7PVNA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_104512_70_2233_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.456Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im1vd3pmQk5FNDJWaHRRRENSVXVYOXlnTS9CdGkyb2krRmhKUGJqNFZ2bGVGZUhuUzZyOEZZU0tmMUdHeVh2U0xMWGMzUFVRWVNXcW54VE5jZlN1WlhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTVfNjdfMjQwYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzhiY2Q1MGIyZWYwMjc1OTgzODExMWNjZTJkM2VkMDFhNGQyNWU5N2Q2ODlkYTc3NDFkMWU3ZDFjNjI4ZGYwZWQyMjgwNDNhNjlhOWI1ZmIzOGE0NzQzZGQxZDdlY2ZjZmJkMjBkMTU3ZDgzZWViOTI4NTM3Mjg2NjNjNzc0MzVhMjQ4YThkNWYxNjViODk2ZjY0ZjZjMDQ3ZGRlZTk1ZDA2ZDk5MmNhMDI1OGM4NjEzMGVmZTYyZGNmNGQ4MGVkZDZiNWQ0ZDYzY2U1YjhjMmJkMDZmNDAwOTRlNWM4YjJiOWVmZjdkYmZhNjc2MTJjNWQzYjg4NmNlYTM0Yzc4OGJjZWU0NjRkMmJlYWY0YWY2ZjY4OWIwOTY3OWI3YTRmYjFjYmJiYTdmNWM2YmI0YzNlNTAxN2IwYjgxMWNkZDVlZTY2ZGQyNDgzOTE2ODY5YTk0NGIwY2FhNjQwZjYxZWZlMGQxZDE4MjhmMTQ3YjRlNWNmYjViY2MwMTBjNzM1NWFmMzM0ZTg0MDMxOGUyZmI5YzljN2YxNTdmOTBkYzFmZmY4YzMzYTA5ZGE2ZTE3ZmQ2M2Y5YWFiYWM1YmY5NjYzYjgxMTRhMmU0NGI0NDJlMWQ0NzM2OTEzOWU5M2RhNWUyZTlkOTljNTcxOTdkMmFlZmQ4NTVmZWVjNTVjMmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mq4i6So-_YFlSG22iE4qJO5ZuTdKXa-2M4WgIpLliBtPKEPzReYkpJIDNVAOocll5DtvOVbCDJq6lGw75Q5RZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113115_67_240a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.458Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjFGZ2xTRFY4TEdSM1hGbXVNcVdiQzQrem9XcG90bTM5TFN4Q0VqYzA5c3g0WXRFNktZK1RkUWlVSHR5UVNybDBjWHlMZ1dLbWVCVWtQcjRqRmxCZUFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTVfNjdfMjQwYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzM5ZDliODMzZjMwOThkYTQ3N2UwMmYzOWQzM2VkODMyZWM3ZWNhMDU2ZDg1ZmExMmY2MWE0MDQxNzM2Yjc0NDBiMDE1YzAwYzgwNDVjOGEzNjRlNjllMjljYmMyMTZmMjc0YjYyMDNmZjc0MWJhMmE2ZTY4ZWM2NTkxZTRkZTRhNjM5MzIyN2U4Zjk4MzhkMGZlODI3ZDE1YzYzMDM1Y2UzNjU2Y2QyN2I5NjRjOGY1NjFmYzk1ZDA5NzM2YTE4NjRiMTczNDUzZWEyNTJhNzc2NDExN2YxMWNjZTYzOTUxZmM0YjQzY2FkZTY2YjNlOGZkNzNmZmFlNmMyMGZjOTBjMjZjOTJiNjE0ZDg0MmM2MzVhOTAwMTk4YzdmNDY2YzI3MjgzZmMxODhhY2U2MjZlOGRmMTc0ZjFhN2JlM2Q1MTkyYzllZTYyYzdlZGNlZWZmMGUyOTZiOWUxNTFkMDQ3ZGE4NDc1NjVkMGZmZmY4ZDIxZjg3MTllYjFiYjYwMGEwOTk5ZjMyM2QyNGE5YzQ1NDQyYmE1NWFhMmRhYzYyNTljY2JkMjVlYTM5YjdlNmNhYjUyMjYxZWQ1ZWFmMDEzYjE3YzZmZmIwZjc5YTllZmUzMmNiYTFkYzYwYzU3YzlmNjU3YzYwYTlhMzViZDQyM2Q4ODE4NzRiZjljNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hzkF8E-oOi56Z6oW7PSG0ys9VGfsTOqBkxhhn4RF4owElOlC4rdtEr-LIyp4GD0G7eqc0DZk6yucUHhsOZlHpA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113115_67_240a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.461Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlowZm1QTXh1U3lHWG8rSm5tckR6U3FKa3MwNUl0UEVQYXAwbFA4NXIxdXFhWVlLMEVDYVF0dk11L2lyVXVHMUZFTTkwOEtrcDMrVysvN24rekorQXdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTVfNjdfMjQwYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWI0MTkwZGJjMTgzNDIyN2FkZTFlMjI4ZmRlNjBjODIzMTIxYmFiYzJiY2M4NjI5Mzg1NmZjMzQ4MGZkZTYxZTE1NWMwMmFmNzFiNWEyNmE4OTY1NWJlMDU1NDE4MWJiZTQxMTdlMmNjY2RlMzYxYTE4MjdkMDNlN2FmNWZlNmY4OTkzMWJjMmE2NThhNjFlNzMwYmVjYTIwMzBlMTYxZDc3YTQyYjNiMDU4NTY2NTNjZWUxYTIwYTQ3ZGFiODRkM2ExOWYzOWI1YjQ3Y2IyMzMwM2JiYzdjMmRiZmY1ZmE4MzlmYWZkNTQ1ZmNjMWE5MzNmNDE0ZTc5ZjYyNjY2NWY4OTY2MGRlOTc2ZTFlNjlkZTQ2MjFkOTlmZGYzYWIwYjE0NjliM2QxMTkxNTRiZDVkNGQ2ZGU3ZmQxYTMyY2RjYTc5ZGRkYTU2N2Q2ZDljYjdlMDAyMTE1OTNjMWNkYTcyMDI2YzhkYzgwZjMyNzhiMmQ3MWRjYTk0NzVhNzA3ZjdiZTg2YTRhMDgwYzdmNzQ5ZGMxMWMyZDJkNTBkZjQ4MzMyODZkM2Y1YzBhYzZkN2Q3MTBmNjkwNTc2OWU2OGI0ZTJmYzI2MzIyYjVlYjg5MGQ4ZWFmZWJjMTgwYjk0OTljZTFiOWE0MGYzMmRhNzQxMDZiNmUyNjYzNWFjNWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0K0IBztyBFvrYmT7RfEhGwKLkjYF0RDZWQ2ntHFt2fYjG_jXiURHnb7H-iSddpzjqmXXWs2irJGDF911giKPdQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113115_67_240a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.464Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InVBcFFtN1pnNFYrNDdzbmFVQ1c0bXk2T0pWYVZHL3F4UHJOQVk4RWtLTEtmbndoWndkbUZNNm05SVprbzRFRUV6a2hhRThvV09vaEFSQ2dqeDJGM1pnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTVfNjdfMjQwYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2JkZjZjZTliN2Y3Y2FmMDgzMWNmZTRjMDM0NDFhOTg2NGQzNDdlMTllN2VjMDhjYWIzYTRjZDY5NGQxODljNDllOTE0NDQ4MjczMDc1Y2ZlYWY5ZjVlMDgyMDljZDhlMzFlYjI0ZTM2M2Q1N2M3ODBmMWJiNjFjZjg5MmU1N2NlZjkzYzJhMDNiNjQxYjkxNjk0YzhlMzc2ZmJlODE3MTQxODc0Y2JlYjI1MjYyMGUxYWIxNDY1ZjFlYTk5NTczNWUxMWJiNWFiZmIyNWYzMWE4MmQ3MTMxOGIzNDVhMTRhOGUzNmUyNzUyZDdjYzAwOWFlNDcyMTVlN2YxZmYzNTU5NTQ1Y2M1OTkxYWJiZjJmNzRhZjc0OTVjZGExYjQ2NzM0MWNiNWYwMmJlMWUyOTY3MzkxYzA2ZGNiZGRiYmFjOGFhNmY3YjM1NmQyMWVlOWI0MGY3NDdmMmIyN2FkYThlYTI5ZGI5YTJhY2NmNDFhNTViOWZlMDE1NmZiOWM4Y2U1YTJmMjhiY2FhYzA4MjMxMGZlY2YzZjYyMmJiNDkzZDM0N2VlNWU4OTQ2ZjhhYTk4YjgwYTBjNWE3NjUzNGY2NWRhY2JlYjgyZGVjN2U3NGQ4M2VlZDYyYzJkZDhmYWQ4NjlhNDQ5N2Q5MTM1MTk3NTllZmI1ZjQxY2NkMDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RFYIyPy3B-NYVN3_3BqgSM_4xxGPp2IFQ4WwBsPwKfDH50i0vLhzYSTKTd255DFRm9Dc0nZggtaMf4vY3OkCaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113115_67_240a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.469Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IllHRmlONXBselBVbU9JWjJCbkVIdWZ2TEd5S1QxSHJmMm1ybUJBRGVRMUZUMld5bWc4ZWJmQ0JmQmswNmRVWFF1VTZpN0FwZlFHdEhFOTBuQWhaWU9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjlfNjdfMjQxMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTNkNDExMjhjYThiMTdiODU4YTNkOTYxNjYwYTZiZjFhMzVmNDk0NGMzOTA3M2E1ZTJkZmVhOTFjMTA1ZTAyYzk1ZTc0YThhNzUzNzM4NTI4ZWM1Mzg1ZWE5OGY1YTc3YmVmNzg4ZmExNzgyNWQ3OGU5ODMzNzliYmJlZjA2NGUxODZmNGRhMTg3NmMwNTY3OTBjY2RiNTAyYjE3YWIxM2QyYWY0MTIzY2FlZGJhZDU2MTZmYTYyNjBkNjVmOGFmYTQwZTFhMjA0ZDY0YzM2NjJjOTEyMDEwNTYyZmUzZTIxZDA2YWY2MzE2NjE5YzViMmMwZGRhOGFmNjRiNzlhNGJkMzVkOTI2YzBlZmVmMDQ0NTEyZjRiNDYzYjFiMGQ5MDc3M2FlNWViMTE2Y2NmN2Y0YzU1NDIzZGY3YzZmMWNjMWZiY2YwZjk0YzI4ZGRjYmQzNzU0NjRkOGQ0OTMxMjM4NjBjNzk4ZDQ4NDRhNTQxZjEyNTlmOWRiZTMyNDcyYzA4MmMxNDEwMGJhOTgwZThhMDQ2Y2YyZDVjYmYzNGFlMjcyN2VhZmJlODVhOTEzOTc3NzgyNTkzM2ZhNTUxYTA1ODczZTlkYjNlZWRlMDViZDY5OTc1ZDFmODkzMDg2ZDkyZDFjN2I4MzliM2YzMGMwZTZjZGMyMGNiMzE5ODNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.B0meydx8ab4KbRTeJTPXOYCGJ06jyRF8lhqtHl80cCJrRIZoWr_nGQbMJ4VjcBH7ysdUNGaCiAn4aJNZwTi6dQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113129_67_2412_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.472Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjBEZmltZWF2VFhBRnBPbCtVU25kMFVkZS80bjBqRjNxekZmbDg4OEpvOHh3ei9wak90Y0tlL2RmQVNDWllDRmR6ME9va05QVTRNM2tOWHFGcVBLai9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjlfNjdfMjQxMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzU2NzQyY2JiZWMyZGE2YzY0Nzc1Nzk4NWY4MjkwYmQwNjdiYzFhZmY5MmIzZGI5OWY1ZWY3NGExNGZjZjBkZmJjMWU2NmFmNmI5YmI0YzdkNTU1MzNjZDM1MDNmOGVmZmZjYzExYzA2OTgxN2M0Y2ZiZDJkZDcxZGI0Y2Y4N2Q0MzVlZDQ4NzRlZTFhNTRjOTViNzkyZDRhOGEyMzM2NzAyZDI4NDdhZTY0ZTM1Mjc2MWJkMjhhMTI1ODBkOWZmYjQ0OGU3MmI2YjBjMmQyMjA4MDZiYjQ4MmMxM2RmNDZhYjgzYjZkY2M0MzczYTI3MmJjMjc0OGM3NTkzMjA4YzEzYTY1MDJmYzJjNGI5NjI2ZDlhYjc5YjVkYjlhZTFhZGM3OGQ1ZDMxYmQ4NjI4ZGJlZjEzYWRiNThiOTY1NzdhNzRkZTFiZjZhYWFlMGVmOTRkOWQyOThhNTk1MTQzZTIwOTg4NzI5ZDY2ZjUzOWQ4M2MwNTIzYTBhYWVlZjY0NWI0ZjZiNjZkYjE2MDY3ZmFhMWY3MTcxODFlODM0YzY4ZDQwZjAyNzhkOGQzMGJjZThiZTQ4YzhlZDIyNTkzMWY1YzIwZmEwMDZjOTZmNjkwYzk2ZTk4OGNiOThhMzI0ZmQ0N2IzMzg4MWZmNWM1NjRmNjBkMmRmZGJkYWRkM2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KdN2gJKskmi4xDUVzLFAqdvWTWwd3jsVUgCQTjmXA6w422CeFmZikSkhSy5toQfvLWpmkbp5ghYflL8FdKViuQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113129_67_2412_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.475Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InMyRkZuQmV5cGVpUFJ0eWNmYWZ1ZGRmK0VJSUV1OGR2TW9vNTFWOWZadlA1RjBiLzVZOG5xRWRkNW9kZWl0ZzM4bjNiUlg5MG8yODA4SGdmZm1ONGFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjlfNjdfMjQxMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTU2YzVmMTI2MGE4NDJkNjk4YWFjMzVjMWM1ZTc0YzAzZmEzODUzMmE3NDE5ZDIxMWI5ODE5ZDNmYjI2YzhhODZhYWUyOGI2YzM1ZWVhNGMwMmMyODY1ZWNiYWMyNzM1NjU1YmJlYTkxNWNiNTc5MjZmOTU1ZDg0YWIzNWMwNzM5NzUwYjJiNjQwMDk2MTlmYjI1ZjY2NmM1Mjg0YWFiNmRmNmVmZWQzODA3ZTkwNGJiNTA0ZmY5NmIzNjI0ZDE2YmYxMTM0MjllZTYwMWMyMjk2NWY3MmMzNzZmOTc3MmY2ZjFlNDM4MTIzNTE0YzFmZWVkZTU1MjJjYWI2NmIyY2E1OTFiYmU3ZGZiOWMwM2FhOWM2MGM0OGY0YTkzNTYxMmI2OGY5OTgzNWQyZTFhMGFjODJjMDg0NTViYzU3MTc5NTU5MzVkN2NiNmEyN2VjYTk0ZjgyYWRjMzExMWQ0YjVlNWY4ZTY4NzkyNmVhMzc4YmFmYzM2YzBkYjQ2YWVjMGE0NzFkZWM2ZTdhMTZjYmU3MTU4NTM0YmQ2NDQ5ZWFhNDI2MzA1NTEyYjlmNGM0NWY2OTc0NzkyZGM5NTM1NTQ4MGNjOGZhZmEzNTI4NTNiZTZjYjg1MGJhYjE5NDFmYjdiYmU4ZTczZjhlZmRmZTZlZjc4OTFkMmU0ZDgyYThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aaVUpKQQEa7Q-8-aAPCVBuceZ4xl4WnKf6o_Qb-ICkhaIglpTuCswyjSGkz5RxWhkFDe1UmGXSJ8m8f51-ZPFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113129_67_2412_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.480Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlRQNHhFQUZPWmVLa2htUE5pb2VneDlWNHZWT1QxaDBObXRsZEt0bWhJQ1BoOWxjT05nVzlZVUFBZXZtemZNYSs1YkpsRkJPcnVkK0tEY3Fnb3JyK3ZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjlfNjdfMjQxMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTQwYWRkMWI1ODM2NzA1MDkzMDhjMDliMmI4ZGJlNDZkYjQ5YjFmYTg0OTQyM2E1ODE5MzI5MzVkYTFhMmUxNGJjZDIxYTFlNzM0YTQzZDI4YmRmMGMwOTJjMmUyYzkzZjYxZTBiN2EwMmEwNzI3OWQ1MjVjMzZmMDQ2ZWU1NWI3NDdiYTM3M2FkNzhhMjRjZTdiZjE3MjVmODgxMzlkMWI3MTBjYzdjNGIxNTNjNDExNmFiYmM1NmQ3MTkxZjFlN2FkNmM2MTJiNTBhZTdkOWViZWVhZThhYzE3ZDY3ZDYwMzMzMzExOGVlYmQzODNlYTQ4ZjdhZWY5N2U1ZGRmN2ExZTQwODhkNWVkNjg5NjkxNjgzMTNkNjQ4Y2QxY2Q4YTQxZDNmM2I0ODEwNjdmM2U4Yzc0ZTE5NDExMWIyYWMyNGRmNzg1OWYwYjM3MzlhNzIxMjk3ZWI0ZTMwMDljYzY2YTJkNTE1MzNjZGE3ZWUxN2U3OTAyMTMyNTI0NWIyN2NmNTIwOTNkZDE2NjdiYTNmNmU2MmQyMTZmOTAxNGZlYTM3YWIwNmViYTRlNDljOTExZDU0ZmQ0MmM3M2ZlMzY0OTkzZmE4NmNkNDFmM2NlYmU1NzdhNWE5ZjhmYzllNzI3MGU0MzYwYTI2MTU1OGY3ZDJkM2Y3NGVjZDlmYTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YW-ck0PZ_ROaN1yQAYNZedJr-9ffn8barRU62_6dSUfLTSzBLkFGHMKyrBwzh7dOWHj05UFQPMC89-Ys0gKYKg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113129_67_2412_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.484Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Imk4cllSYk96a3Zna3krNmNBZm9JQkJQN1JnZkxhc2lqK3NzR1pwMnl5WUJaV2VrY0ZxcjFDZlBhdElURDNhdFNDM2VBeWY2ZWpxSHNDNjBXelQ1YVNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMThfOTVfMjQxM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzA2YmRjNzY2YTExOTIxMTk5ODMzZTE3MTY1NDQwNjgzNTQ1YmY1OTcwZTA0YzAxNzA2MmNiNzNkZDI4NjhiZmZkZjgyZDBjMDk4MWY1OTMyNDZhYTcyYmMxZWM2YmMxMmZlN2Q5ZDAyYTA5YzNkNzk1NzE3MDYyZGM3YTAwOGQxYmI3Y2MyNTA1Zjg4ZjA1NGMwNjM1ZmIyZDlmN2U5NTU1YWUyMDFiYTMyYjVlMDM3MzQ3YzA4ZTJjNzBhYWI1MTc0MGVkYzgwYjNjMGYxZTgzZjkzNDg5Zjg3ZWYwZWMyZmZkYzc1MWNhMmRkODg3YTAyNGRlOTAzMzQ4ZTA5N2Y0ZjQ4NDdkNjljYzFiMzM5NjQ4Mjk4NDVkNTE4OWY1ZWNmMTQ1NmFhOTY2MzlhMDRlYWQ0ZDIyNWU2OGU2NGU4NGFhYTgwZDM0YzU0MDZlNjNjNDE1NzVlY2I3ZjlmOTdjODI5NmIzMzhlMTI2ZDc2ZWU2ODkzODdiNzY2NzFlZmI0N2Q3MmM4YzJkMzhlYTQ0NGVkNmZhZGM0YTM5MjcxNGI5Mjc2MGZmOTg4YTlhNWM5ODU3M2Q2Mjg0NzQ2YzhlMzEwOWNjN2JmYjEyNGEwODZiMGM2YzA2NjJhMjcxOWFjNWRhYjIwYjA3ZDFjYTJmNWJhY2Y0ZmJkZGNiZDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BHy0XM1ly6Vj7iAki1egh8QI6BRs0v9xmQMMaV6hkGrmWHlKvPM78m86nPspKbViVMuKd8XhxwnVyxl_CFE8eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111318_95_2413_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.487Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJhekhHZVdIdlhRNWkyTStFYXFnK0hJRkhaY2ZuSnVGd1hWNTZaK253NEpYS1gzRW1hdmppMVZBdWt3L3BIU1l2bnFwcXlQSlFvbUFnWjZTLzV0MCt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMThfOTVfMjQxM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjZjY2EyYTFhNjMzNTY0ZGZlMjUyOWExODA2M2ZjNjRmOGE0NWM5OWNkNWYyY2VlODg3ZWY2NThhYjIxZWM0MGUxNDM0ODZmOTc1Y2JmN2FmMWQzNzg4MDAwODE1NDdkYWYzNmI1NmVlMDkwMzgwNmZiMjVmNTMzNTQ3ZGM5OGJiNDE3Y2E4NDljZjkxN2IyY2E0YjU0ZDhlNTU2YzcyMTUxY2UyOGY0ZWEwOGM1OGZjYzRkNDViMmNiMDkxYmYwNTlhMDM3MDMyMzE0OGE3ZmQxODlhOGQ1ZTM1YWY1OGQ1MzEwMTAxOWJjMjQ1OTBkODc5NDgzY2VmYzhhNGFjY2NiZWQwMzBhMzg0ZWExMGE5Yjc0MzZkN2FkZDc4YjEwMWM0YTE1ODM5YzAzYzI4ZmJhOWYyMzQwZTliOGExYzU0NmQ3MTdmODY1M2YzOWI1NGZiZGM5ZmIwMDhlNjAxMmRmM2Y3ZTM0ZGY4NWViOGE0NzQ3OGQ4MDExMzQ3MjE3NzczZWZkNjU0MDBkODkxNmJiMTczN2YzNzJkZTlkZDNkYmM0OTAyZDY1NzRlNzQzOWQ3ZDgwZmJjM2NhMjYxZTBlYzE3OWEyZGJkYmZiODlmYWQ4Y2JlYjBmODVhN2RjNzU0MmNkZGMzODgxN2FiMmU2YmFiN2MyMGNhNmU2MWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sSpB00OmqgQbpRPStFl9PbqtElOIIVJos0OAEtFwZ7HpY-5q1ncl_NHOHIKfu-qCbkhpb9MF0IGInUiSMWdiyQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111318_95_2413_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.490Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlRQN0orRjU3YmY1QTFVSTNUSS9wM1ovRlZpemlFVXdDZTVoaEVYWm8xTGkrZmFhRjRua3dUR1hYakUxV1Nad3EyTldSVTFJUG5GQ2NsOHVFcmhPM0JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMThfOTVfMjQxM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI2YWQxNGNmZWQ4NzQ5Njc0M2Y5ZmM0YjIxNmQ5ODkzOWE4YjJmYWQ2MDE0ODE5Yjk2YmRiMGE3OGRjNDJjNjFlNTU1ZDhmYTM1ODk0MmNmY2MyYzFmNDVmOWM0YzZkNTA2ODRlZjBhMDRjMzM4NTI1YmI4NjVmZGNkMzA1MjA1NTI2NzJlYWEzODAyN2JkZjJjZTE2MWQ4YmYwYWRmMTIyZTZjNjYzZjkxNjc0MWYwYjU2YTUwMGEwNzgwMTc0Y2ZmMjQ1NzBhZWE1YzAwZTkwNDY4YjEzNmEyMjFjMTZjMTA4ZmFjNmI5MWQ0ZDM4ZWZkY2E5NjkwNmU4YWYzNWZhYWExNTQ4ZTFkMzMyYzUyOWRmNjg5Y2M4OGE5MGI4NDYzNThmODZiYTM5MDUxNTRkNmY4MzdhOGUwMGM3OWVkZmZlNjEyYzQ1ZmZiMjZmZDUwMzFkNTAyOTVjZjVjMzRlNzA5YzFjN2MxZTVhZGY5ZjMwMDI3MmFiZTVkZjI5MTM1OGRkNWI4ZDkxYTEyMTMwNzIzNjdlZWVkNjkwYWU4MmE1OWE3OWViYzJlNTAwYWFmNzEyM2NhNTkzZjU1YzM0N2RiMDYzMGI2OTdhNzUyMjg5ZmE3YTMzMjRiMmI0YjUxNWFkM2U1MzEzM2FlZDUyNzA2M2VkZjM1ZjM3YzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AYfI2iHQQr9vXYa8zKkX27FwVV1-fFQzkWvGFb5ujHQD7mPozSUf_-9ejuQKauv49WXGcvuJ9h6jmdOjJ5ZNmA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111318_95_2413_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.495Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkpVdXdBRDB6YXJpLzVtQzdVY3pOVkhtYTZYelBKWGF2dmI3eHViWDMzN1Q0Z1k4WXU1NFFocm15VjZ3KzJRS2RERzl5L0xZbFRoV21XdHFxaE1reDN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMThfOTVfMjQxM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjc2YTZhN2M0NTM4MTY1MGMwYWRkNGQ0ZjQxNGYxMDkxODgwYjA5NzBlZWJmMjljNDcwNmIzYjc2MzA0MTc0ZTIyZmQ4ODg5NjMxMmM0N2QzNmIxMDgyYWNkOGZhMmM5Y2RkZmJkYWQzYmNkMDJhOWI0ZjkwZWE5OWQ2M2FhNTM3NmRjZjI0ZDc4NjBjYTUzNWIxMjBiMDI3NDE1MDE2NzA0NjI5MTNkNmMwOWQ5YWJlNmNmNDAyN2YzMDNjZWEzNGM5MDFmMzEzYjYwYmU5ODZiNjRiNjY1ZGM5YzRjOWMyMmE2ZTllM2ZkZGI4NjEzMTZlZjA4NThmNGEwMmVlNTIzNWMyYzBiZDhjNWNhZWZkM2NhOTNmZDY3NzY0ZmRmMzA1N2JlMWRiNjRiMzE0MTk3YmJmMWExYTVhZDk0ZWQwNzZjNTc5ODIyN2Q3ZTU0NTQwMWZiNjY2ZWE3MDY4YmY1ZWFkY2NiMjE0YmZiYzA4MjhkMmE2Y2NkOTIwZjMyZDI3NDNlOTNmMGUzZTI0YzVhMDY1NDUzOTkwOWZhMjY1ZTgyNjhlMGQ0ZDFjZTcwMThkZmU0MjVlODc5MDlmZDVmMGI1ZjFmNzZjYjM4NzM5NTVkMmNhNGNiNTM2MGEzNThkMWM1ZGZlMGVlMmQzMzAyNjFmYmUyYTk2NTA0NWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.86ylQVAru3bPgpFauTrz4bpaUApESAPIbEJ9w14IMu9Xxj1dRLFHsFfWr6Q7j8wqilyLp26VU4itJkjcTd0ddw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111318_95_2413_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.505Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InVCR1pXN1VIZ0tyTFBvMTJjZEVYZHZ2dnRDTmpIamFDRjIxdlVneWdycVZCcTE0ZVBCdVVPWSt3YUJ0aUhLNytoTXFTRHRlaktQQ1FZay9jaTFsYXdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMTEzMDlfODZfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWM3NDc2N2U1MTk3MWEwZjM4MjJmNTU2Zjg1ZWFlMDNhZGE3ZWQ0Y2M3OWYzNTE1ZGZmNjhhYjIyODlkY2Y5MDdjY2JmY2UyYmNhNjZhYjA5NzUyMzdmNTkwNjgzMjJiM2M5NWYxZTE3NDlhYWU3NmE3MDBmZDQ2Mzg5NGQyMTJiZGI1MWM2MzE2ZjIzMDRmNjEyNjMyODhlZGViMjhkNDcwMGU1NDg4OWIzMGM2NDY0MDMwMzdkY2M5NDBkNTZlMTczYzBkODIxY2IzNzUxMjEyNzUyOGY2MTU2MjQzNWNkZjgxNjFmMWI1OTRkZmZkYzIwMTdiNjA1ZmUxYWVmNzhiNmYwNGU2NDNmOTI1ODdiZmExNGJjNGI0NTliOTZjMzZjMzNkMTg2ZDkwZjQyMWUzOGQ1ZjBiOTUyODU0YTA5YzAxYzU2YzZmNWM0YTQyMTI3YzIwNTRmYmU2YTBkZDkxY2U3OWU5MGVlN2MwNzViMTgxNWI4ZmRlYzc3NzQ3ZGMzZGZkYzBmMTU2NjllNTBhMDkwNmQxZTdlNzczMWE4NmJlMzc2MmQ1ZjcwNjQ3ZWY0ODU2MGI0ZGUyMjkwYmZiOGQxYzIxMjE4ZTU0OWI5N2FiY2Y1MGEwYjU0N2ZmMGE0ZDU1N2FiZTdlZjRhMTViZmI5MDBmYjI0YWZkYjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.T3Lemthic8DV4hvds7cduu5SFK2cx17tC19HoTqW42uNSZYnnUHzyXH0YU5SFLcu2wvNUCJWHSkKctghcQ8u8w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_111309_86_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.508Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ii9Tbm9xRDZTSHlkNXNxTEhkUm8zTXNzUUVQTU54d0VPTDgxcjgwNE9NbVpnNGFqWURFVVdtODFrSy9GMC8yNFhJQldlZ2RvQVl0SlZHamlLWENzM0JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMTEzMDlfODZfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQ4M2E3ZjhmMGNmZTIwOTM0ZTU5Y2JmMzdiMzU3ZTljODViMTY0Y2U4MzBkMDUwZTUyN2I5YzAzYmNjODM1MWI2YjdhNmYzYWE3YmNmZmJlZGMzNTdmMjBhM2Q1MDNhYmIzOTExYjI3Y2ZkNjRlMGEwOTM0NzI1OWY1ZTBjMzVmNjJhYmJlNDFkZTZkYzZhYjFlY2Y2MDhkMmJjZTc4OWM1OGRmNjliNTI1YjEzYjRkMDc4ZGNmYWNjY2FmOGRhNDMyZmYzOTA4NzBlOGYxZWMzNDMwMTQ4ODY5NmNhYjI1YWVlZGVmYTBhOTFlYzFmYTJkZjkxZGJhM2QxOGI1OGNkYmY5NWU5Y2UwNWNmOWYzZDM5M2MzYjNiMzQwOGVkMTc1NWNhOGVjMDc3NDU4ZThkYWUyN2MyYWYyMmY0YzExYTcyY2NjZDVhNDk0YWVjODBkZTRlZDk0ZmMyYTE5OTRjMjQ2NGI5YmFhMWU1OThmOTRhZTg2NWIyOTY1YWJhZDRmOWQ0NmZlYThhNTVmMGUwY2QxYzhlYmU5ZDA1ODY3NmM1YWRkYWViZDkyM2M1OWY4NjNmMWIxOTY0ZDJhMzAwZTQ2ZDBiYTcyOTViMTNmYzdlNzhlZTVkYTdjZDU4ZWUyY2Y3MmNkZTE2Y2Q4Y2VjMDMyYThjZTc4MDc1ZWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8UCiHPtvbXtlDv-Gb_H3YD7XWKTtCIjnAq7W49LPIWP4oE42y2jV-TMoDBROvkQ5OVEac2ZDa68rOznioXDmvQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_111309_86_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.510Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik51eS9DSlM4ZENlZ0RYU2ZuUm43TjUweUlraHBGblN3TUxza3lCYmt0ejlkOWx4bVA1bWw3NmdBRGZBc2dIL295WnNsNld0WktHS05DRm5XOEFtZm1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMTEzMDlfODZfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGQzNTcyZWIwY2M0YWRkY2MzOGYwMWNiZGYyMzczN2Y5MmE5YjlhODJkY2M4OTMyZDljNGU0OTkzZWIwNDhhMTQ1OGYwNmJjZGVlYmY5NDA4YmIzMTNkZTgyNjliNmQ4NjY0YmQzYzM3M2Q4NzFhYTEzZDcwNDQ0ZjMyNDkwMjM3NDRiN2E2MGI2MTdhNWE0NzgzOTgyM2E0NDkwNzNlYjczY2Q5MDMyZTYzZTEzNDVhNjNjZmE5M2FiM2UyMzQ3MWM0ZDdlMjVmZWZmYzI5NzRmMGE4YzkzNmE2NjcyOWNhZmViYTJiYzhmOTgyYmZhZWNhNGZiMDk2ZDhiODUyYTEwNzlkNzk1ZDY1MTVhMjY4YTNiZDhmMjMyNjVhMTJkZDUzNTYyNjc0NzBiMWE0YzdlZDRlZjllMWNjZTE3MDFkYTA2ZjA4MDYxNDYwZWY5ZTUzYmIwY2ZmNjJmNjY2YzY1NmUxNTc1MDJhMGVmYjZjODgwMzAwZDJlYzc0Y2E1MGE2ZWM1NDI4NWRiZjVmZTdiNmYzOWU5ZDY2ZGVjYjZkYjdjMjY1NWFkNjNlZThkMjQ2ZTRlMzhmM2EzNzBjOWE4NDZmMjNhZGUxMWUwYjE1YTQ5ZTU1Y2U5Y2QwZTIwMTQ1ZDAzZGNhODAzNjlkNjhiMDQyY2QzNjJhNWE3YzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6Yfroq4KlT1H0o8UWi6c1ccAN3e2NYcwsgQbgqfnfxpwmd0DYZ7rbDl8_LtADypVQHfXeWRpVfubTzfzKoO-RQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_111309_86_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.514Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjdNaXREQ0NvOG81YjNSWWhiZkVSL1JlS2NLVVFjeGFtb1FtNkk2bGkvY1drK2NLMmZNenUvQk9uRUNZQlREUzJGbExOaVVlallwekVLbytmTzN1UmJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMTEzMDlfODZfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzNiNzhiMWRmNTkzNzc0ZDQ2MjZiODA4NjIwYWI2NWVkZjBmMjVkNjc5YTFmYzYxZThlMDBmNTM4OWYwMTU1ZDJhZjA1NWQ4YmFmNTQyYmVjYmY4YjE2ZmVkZjk3MWRkYjgzNTZjNzc5NmI2YmI3ZDQxNTdjODM3NTJlM2VmYTE1OWNmNDM4NTk4NGJjYzM1ZTdlN2EwN2JlMmM1OTFhZThiODYyOTk0ZGNlN2NkN2UzOGIwZmNiNDc3ZjNhZGRkNDM0ZTkzNDE2MWYxNGViYWNhM2MwMTg0YjlkYThjZThkZmZhNjg1NjEzYmMwZGVlNTNiZGQ1OWE0ZWNjNzI0MmI5M2I2MzRiOTgzN2VjYzUyMjY0NzhjZTg3MmRkZmNiYTBlZTIwMDljMDY2NDg3M2Q2OWIwZjkwMjA4Y2RhNDI4MDhlODQxNDFiYThhMTk3MjYyNWNiOWIxODUxZWQxMWY0Yjg1ZmU2NzU4MTgwZTVkNjEzNjcxNTQ1MDdlZWYwNjU2MWRjOTliZDIyZDE0OGUxMGJiZWNmYWVmODQ3NWUwYTQ5Zjk5M2UyNGU2ZjJhZWQ5OTU5YmZkYWJlMzJhNjQ2ODE0MzIyNmE1MjM0OTQzNjcyOWM0ZjVkYjA4ZTNmN2E5ZGFkMjVmZDY5YmE0NGRlMjMxNTBmZGU4ZDk0MmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mL9iKsNSXkz3reKKLvB5ikd31h912sigQBJnzVAQRWYGdnrjG3FmSQdCXz53JaeJMJHW8HHhy79nLIexwbtDpQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_111309_86_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.524Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJvQ2toNk9FdEw3ck9VZ1N3RGZtR3BkVkk4dzBYNkZtaXBNS3BDYWc4TXpVMlB5dlI0N0k3YWtXdDNzeUxVanVrRWhoVWNidzllUEt0MEUvZkJDRHVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjVfMDBfMjQ0OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGFhYjM2MWY0MzAxNGQwYWQ1ZmZlNTJjNTYxNDAwZjdlNWY4MzRlM2U5NGFmODlhNTA2ZTk3ZmE4NmJkZTEzZGViZjRhYWZmNTA1MThmYTc1MzVkNGE1M2NiZjJjOTgxMThmY2ZkNjI1MmJkZGFkY2FjZmNiNWU2MTNkZTFlNzUwZGYyYTk3NzU4ZjA5YTUwNDM3MjYwZDNlZTViNTI2YWFlNmE0NGFmMTdmNDNiNzc3ZjBmMDMwMTA2NGZlZTU4MzNmNTZjNGIxZGRjMzg0MDEyMDJiYzBiOWQyMjcwYmE1YzA3NDIwZWQ1NzEyODA4OWYxZjEwYjIzYWNiZjM1ZTZjNjBlMzNjOGU0YjliYWI3NjgwZmI4ZTVlYjAwM2I0NDYyYTMwOTMwNWY1YmE4MTRiYWQ2ODUyMDJiZjMzYWE1NWJhMmJmZTAwNGVmZGZhODQwNjViMDRiY2Y3YTcxZDUwM2Q5ZDk3ZmY2NGYwOGM4ZWMwNzViZGY1MzgyYTg2Y2NmZGViOWMwNTIzM2U3ODI5YzdiOTQ5NzMwZTliYzI2YzEwOTRlNWZhYmZiYjg0MGYzZDAxZmFkYTM0M2U3NzQwOWQ5YmYxY2YyMTFjYjE2M2U0ZDU4MDQwNGU3N2RkYzUzYTg1YmFkNzdmMDM5MWY4ZDlmYzdiYTMyNzdhYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.VfROQ-x0HnqRHAYqJAoWr8DS-HgFIfOgjiS3xyT90bqlkUEmEMwFECuT4NtarpiEiXTghc_4BwVpe3KWTT6cOw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103425_00_2449_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.530Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im9DRFZRWUg0R1VFZHY3ekxkanRkMWFaTzBpVlE1cjJ2TzdPNjJFWlN2OFk2UmVjVXJlOC9MbXFVYmNjM3NpcitGR3c2bnNndldTdm5qdE9uczhhTW1RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjVfMDBfMjQ0OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTNhZjVkOGQ5MjQxNzBkOTA1MDZlOGUzYmM4YjY2NWI4OWY4ZGJiMTNhN2YzYzYyYmNhZmZlMzE0Zjk5MGY1OTRjYTgyYzA4YjEzYmEzYzEyNTlmMjcwYjA0ODI4MTYxMDBhMDUxZWU2YTIxM2YzNTU4Zjg3NjMyNmNhYzE5MTIwZjA2YTMwM2ZhYzBjYTQ1MDNiNWM1NGExZThkZGI1ZGMyNGQ0YmNlYjk5ZDg1N2RiODBkY2FmNGJhMTYzNDRlNzU3Y2I1MzIzNDQ3NzVmOGM3YjE2ZmI0MTJjMWQ0ZTBiNWZjYTUxMzZjNTk5YmQyNTlkMWMxMTE2YWU2ZmMzY2U3NDY2YTk1ZDJjZDIwMzAzNjkwZjgxNjQ1MTU4OGRhYTU4ZmJmMzUzZDc1MzM1YTQ5NTdiNTcyZjY5MGFmNThkNTQxNGI0NmY0NjFmMjkyYjBkMzY0NzA0ZjAzZjBlMDY4YTdjNjNhOWU5ZDg1ZGI2Y2E3N2EwNDA1ODk0Zjk0NjQ0NTMyN2IzYTg1YzY0MTI2ODg2MGRlNTdiMzVlY2JiZGNjOTE5ZDVkOWEwYmNhOTI4ZmI1NzFkYzI2MDJjY2M0YjY5ZDBlZDIwN2FhNDQ5NmEwMGM2ZmQzNGVkMjBmZDcyN2IyOGUzYTAzOTZlNTBhMWMwMWUxYTE2ZTE5ZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.soPsQUkIh09nMyRlfYWTpOYjQwvD6weBKXEWNFbpwDIUxL25VqgPW72Ri6nSpexxDf3EUAY68oSqSiCCaEj4og", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103425_00_2449_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.538Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImVxVi9FVlVNSTVvQy83MVdiemxtUHl5SXdKSlVkbTdOM09Kby9FZitQTHJvWmFDWmJ2U1gxWEQrQWZYMnRMaEZVdzNrSEhjY2tESWFFZXR5SitWLzZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjVfMDBfMjQ0OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDQ0MTUxZThhMzIxMjBiZTJjZDc3NDFhYmQzMmQ5OTRmNzg3ZjE1ZTQ4MTBhZGNiMzIzYjdkNDE4OGNjYjI1Y2RiMTJiMDYyNDMxMzlkMTNjNDMxYjkxZTdiODgzMDVhZTM4OGJjOWIyZjM1MTQwMzdjNGYwMjE1YjY2MjY4MDgwMGQzYWE3NjA1MDA4MTA3ZjkyMzhlMGYxNWJkYzAyOGJlYzg4NzNjZWExZjg3YzVlMWI5OTlkNDY0YzU5ZjM1NGY1ZWZhMzA2ODkxMTE3NWQ1OTI2OWIyNjViN2Q0OGNiNGY4YzBlODg0ZmY1YThlOGEyYTQ2YjAxNDBmZDY2MWI2YWJjOTE5YWQzMzBjMzAxNWU1YmU0NzQ4ZDczMGE4OTUyYTAzMWI1YWRjZGY4YTM2NzNjOGY3ZmY3ODg1MzNlN2Q4YmQyNzRjMDRiNTM0MjIyYTA4YTZjNGVhNjZiNDdmMWI0NTFlNzFhZTE4Zjc0NzFiOWMwMzBhZTZjNTJhOTRjNWYyZjZjMTY0ZmZkYmU2NWM3YTc3NjBjY2Q5ZWU3OTE5MzA5YTM1NWQ0ZmFlOGNjYjM3MDk4NWJlN2NjNWE0MWZlNTFmOTUxNWM2MjM5M2U5ZWFmN2I2NTFhYjNhODhhNmUyN2I3N2UwNTg1NGZkYmVjMzY4MTYwYTQ5OGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Q7Pgq6SiddzaO-heIQDUON4yyHwCpVpw7UyVCfOOL7cAQdFJg3WjaLXSEUnvnVRlp0RTxEiPtC-GALGOwcV2kQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103425_00_2449_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.547Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkN5MnNkcnRCVE42RWpTZUZJcE01bWNCdWtndUIvUlZYaXNyYnlzTUlYa3c2d29CVTR5aHYwLzhzTHdaOU95dWg2Rm9mMCtnV3NYRGlKOXdJYzUyZlFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjVfMDBfMjQ0OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2NhMDFiMWZmNTBmZTUzNmUyYmEwMzUyMDFkNzZkYjhkZTY3MDBmNmRiMjdkMjdiOGNiYTlmZTIzYzNiYTVlMTc2MzY3YzM5MWU5MTFmNGY3ZmI4MjE4MWIyYTkyN2RmMjA2NzNhM2ZjN2QwNDRiM2RhMGY2OTlmMjlkMzg1YTk5NmEzYjk4NjBjNGQ2NmU3ZTI4YzZmNWMxMTg5YmE2MWQ0OGY3M2RlM2IzNWJjNDViNTg3ZjViOGJiOTJhYzU5NzM5YjVkNzliMzgxN2NmYjk0NzQ5Y2VhNWFhNmZmMDFmNzY4YmEzOTZlM2E0MjM4OGM3ZWFkMmQxZDIwMTJkNzhiMzgyODE0ZDM1MzdjNzViMWQyOGM2ZmNiY2VlYzg2NDBkNzkyZTM0NzE1MjRjOTMwMWVhZTc3MmI1ZTM0MzAyMzMyZmQxYWExYjYxNTUyZTgwNGYyMGRjZjRhYzY1MGU5ZTAwNGE3N2VmNDE3ZDg2MWQxNThiNDk0NzJkZmIyY2Y2YzE0Y2Q2YTViZTg0OWZjM2FiNGEwM2E2NjE1ZDJhZGMxMWM1N2I5MzU0OGUzYmViZGExYzZmNmJkYzcxZmQ2NTI0NWJmMzJjZmQ3MTg2ZWE3OGY3MWJhOGJmNWI0YzYyMmI4YzNhNjE1YzdmM2Q2ZDIzNWZmNzdkOWQ3NmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.A3tU2cmWQidUAMHR0TGTcE1xob43xDi5A8Q_KtkoymmXByIRlXSK28OPPDi6yTxbZLvbHhkdtm3vyLZy3Q2x-g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103425_00_2449_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.551Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkVxT1dsOGp4bFlhZTVWcU1zazNEL3h5alhoWjc3ajZ4TjZTd3ZqSjVvYWhBSDQ5YUZvc1g3QmFVdGZDN2IvQXZnYjU3OTErNmcza2ZPdFliM0d6dXdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTFfMzZfMjQyNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDQzZGY1OTM1NzQ2ZTViNTkzNDQ2YjdmYmRlYjA2OTJmM2QwMmUzYjQyMTMyZWY4NDczMzI3YjQ4Yjk2YzM0NzM4YWI2NjljYWI3YjcwMzQxODRmNjAyZTBhZGM4NmYwY2JmNWMyZWI4NzY2NWVjYWQzN2QxYzM4ZDZlZTUwYTI1ZjA4MDczODExZTAyYTdhZDRiZDY5OGVmMGMyYjZhMzI3ZTc3OGIxYzEzNTM2ZjgwMzQwNTQ0OGJhYTI4YTVjMzJiNmRlYTRiOWY5N2VmZTZiZDJmNmEwZmM0OTcxZmNhZjMxYTc0NDNjYmE4YzJlNmY1ZjhiZDI5MjAwYzVlZThmZjM2NjdiMjA2Y2M4MjBlN2FlMWM2M2IyZmJhNGU4NWIzMzVjNTc3YmM1NWVjNzU4NWM0YjhlZGM2MmJjNjU5NDVmZTI4NTJhZWJmMGZlZmFjYTZkOTcwYjA3ZjYyODRiZTBlMzk5ZmZiN2VkYWIyNmJiZTNlZjNiNWI5NGM5NzZhNDhlYzhhYTliNWZjYmYzZGVmNTY2ZmQ1ZDc2MDg2M2FhODU3ZDQ3NzI5NDE1NDgxYzZlNGM0MGNlMzNjODE3ODEyMDBjNTJkMWM3MWM3MjYyYzc4YjVhMjI1NjAzNTVhNmFhMjVhMDIwMzUxMDdmM2JkZmQ2MTc5Yzc4MmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rlmh0m-bHegtuB1OyoeKg-YPxeIZkuibvo2I8c2EUANnl6aLowxpOjE_ALfbSmQMaXcjrV99Lnl8LjBH6BWfaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112811_36_2424_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.566Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkdKOVpkTTRmdEJLaXgrMktqMlhPaDE5OW9vNXhxeEJZZk92TEZuNnZOY0dhRDB3MXlySWlnUlhSMFdlTHBJZ2JXY0gxc1lFZHF5OTRxSFRVZ3YxeHV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTFfMzZfMjQyNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njg3Mzk2YzM0MTliZjQwYWU5MDFmOTJjYzg1YWIyMWM0ZjZiNTQ1NmQzNDQ3NTg1MWJmMjMxMWRmMDAxMjUxNjFmZjkxYWE0N2FiYTFmNGFhOGEyNTg1YjQxZmRkMzk2MDEyOTlkY2IxZTNhNzI5Mjc5ZGZiOTUyZTllOGZhZWNjZDc1NmM0NTYwZDRiYjA1YjFlNmQyM2NiMTM0MzFhMTc0MGM4ZWEyNzVmMTI0ZmU1Y2UxNDZhNTJlY2E3OTYzZTQ3ODkwN2YyY2ZlYThkZWQxNzczMTk5NDViNGIxZTI5OGZmZjY0NzZjNjk5MWQ0NTVhNWM4MzNiOWQ5ODVjMjE2OGRhYjU1ZTI2OGM2NGY5NDc5NWZkZjdjZGVlZTk3Yzg2MzNlODllNWQzOTNhMmVjNzYzZjY5ZTg0Y2I3ZWU4ZjNhZDUxMDEzMDg5Nzg0Njk4ZGI5OGI5ZWFkZjMyNmFjMTRhNDljYTY3NmNjMjUzMzc2NjE4MDI2YzEyYjQyOWVjODU5MjY4YzFiOTRkZGVlNzdjZGNmMWFjY2U1OTI0MjQwZmYyMmExYzlmY2M0Njg5NGMxMmE3ZGNjNTNmMjM5NjJjMjEyZjEzODlhZWM3MjljZTJiNzQ5MzhlZTRlN2Y0NjhmNDBiYmQ0OWZjMzkxZjY0OWMzNzQ3MDc5YjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.esaSNbhJodoHHYuZuGcffbS9-jZW9s_0WqCIFDsABmV5nDYO0spXDarh8GGiySEA6RjVzmR7_Krd5kFJsPnVhQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112811_36_2424_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.570Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Inh1TEVTRTJBbzVSSEROc1lJQ2E1QlR5ZlplSXhEcnV2RHVjbEVwRUM2ZkJWSVExZlRRbVB1QlFXWDN5VlcxL0tRQmFnaW9DcURFY1Ztdm9BS1kvVC93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTFfMzZfMjQyNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzE4Mjg5YmEyMWJhZTczM2U5NmYxZDdhNTNhNjAyMGMxOTU5NTM3NTM2ODNjMzlkMWRhY2EwOGJhOTMxY2MwZDZiZDJmMDUxY2U1YTdlMzBiNTA3YTZhNGI3ZWFhYTE1MjBiZGI0MDI2Njk1NDc4NGZhMzgxMDEzNDIyNWZlOTgxNjM1ZTcwZmNmZTRkZWQ1ZWNmMmE0MjM3NmUzNzBmNjNiMmNmNTgyNTIwMGNjNDIwNTE0ZDQxZDQ0NjEwNTk3NTc1NTc4ZjljNDA2ZTdkNmYyMWMwZTk5YzIxNGE4ZGEyNzdlMWFhOTZkZjlkMzQzNTYzNTk4MmIzZjVlNjAzM2VlMGVlZDIxNjQyZDgxMjY1ZWZkYjAyNjRjYWU4OWE4ODk3ZDJjNjZiYWE1OTM0Y2U5OWNlZmRjMmZkZTFlN2MwODNjYWJmZjc5NjBiMjBlNTlhNzJhMjUxNWEyMDNlZjc1N2RlNjAxMzFjMjkwODgxMjhkMmVlMjhlZTBlZGQ3Y2JmMjZhZjUwZjFiYWEyNzk1NzE1YWRkM2YwZjBlZTE1YjI3MjkyZDU1M2NkMDY5MDlkNDkxODI4ZTBjN2E2MDM5MmNkOWFhMTg3ZDZiYzcwMWU1YmVkODUxZjI5ZjhiZDMyMjk1NDlhZWJkMTYwMTFlOWMwZTc1NzY4NDlkNTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.a4-rvqDMiBFb9ysDWN-vktmRtsK5BODqeDRbpjw-2o72xfFtvKKE7F3mgdUaBQhkwFn0g_0MT8GJ5oZlRXNT6w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112811_36_2424_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.573Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InF4V2VKZWZ4SEtPQUhqNHA0ZUV2d0JWTjlBdDVpRldzdGVIck82NytrSWUzTXdQY0kwdk9kWW5OUk5JY05DMlFSNDhyOHpILzgyNU92SmhFWmtUdEt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTFfMzZfMjQyNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDBiOWEyY2EyZjM1NWNkYWM5ODYyZTE2NTVkNmMwYTg2YThhZjdiZDcyOTBmMGQzMzQ2MjRiNDFkNDM5YzBhNTNkYzk5ODMzYTk2ZGQwZjdjMzJmYmUwZDlhNTBhY2VmNzYwNDIwODgwYmY5MDFlZWYyYjNhNGRlOTE0YjhmNGRkN2ZiMDIyMjZlNjJkOTczY2ZmZGVhNTJmZWZlZTU2Y2M0MTkzYjYxMDFmMjViMTA4ZmNjMjRmYTc4MTU2ZTA1ZGUwMjI0ODM5YjkwMTk1ZDA0NTg5ZDM5NDRjOGQzMmVhMzliMDVkYTIyZTRlMDAyNGU1NWQxZWViNTJmNDczNGRiZmVjMmExMmVjMWVjNmZlMWY1NGMwMDQ4Y2VhOThjZjg5NDQxNmU2YjhmNGE0ZWNlMGZjMmUzMGM2MDg0MTVmMzlhOThjNzJjZGYyMDU2NWJiNTcwZmM3MDlhYjA1NGU0NTBlZjIxZjhmNWZmNjVhMjNlZDIxNDZkYTUwNmRmYWIwZjYzZTAwOTFlMTYxMTNkYjg4NWZmYTg1ZTI0ZDZmNTY4NzI2ODA4NGM3Y2MwYjlhMWRjZDIxMzQ2YTJhYzc4Y2ZmNzA3MWU5MTRmMzA4MTdlZGMzOGIzYTRkZDBhYTQyMDUxMWI0MDZmNTc4OTY4NzA4Y2NlZjAyZDU4ODdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.w_4zmBnekWacd0nbd-WaYUfvGmqreNA21uqdaPlpAx3MPh5ezjbFVbMdZhhmU3kTjua9JIKIKe614ACZry0sew", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112811_36_2424_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.586Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImNBTjE3NHZub0xEdlQ0cjFoUG8xbWhnSkd2M0xEb2FqZ2doY3p5b1QweWhkMEtia282TjEzcGZUZTliOCt1WkNqNWJuNzFtUVBvRXpoNE9Ec0VGd1BnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzNfMTVfMjQ4Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmM1ZjEzOTU2OTE5MTIxMGM5MjE4NzA3MjllNzdlYzc4NGI4N2Y2MzcwNzc2NWU5MDc3NmRkZjdkMjBkOTM1NDMxZDNkNTgyZjA4ODM0NDI5MzUzYTYxNzI0OGNmODI1NGFkY2YwOTQzMjI1NmJhYWYzOTk5YmI5OGEzMzZlZjMyYTU3NDkwNjdhNjA0OWNmYWRjY2FhZjdjYThiZTM3ZDhjZGMzNmNhMWVkNTRiNmNhMDliNjFlNDBjNWJhNmFjMDY1YzFlZmY3MTE0MGZhOGNjZmE0NTFhYjAxYzFmMDk4Njc4MjEzYmQ3ZGQ3Mjk3OWIzZjNiNTQwODMxZmNlM2EyYzgzMjZhZTRkM2ExOTg5MGIxMDllYmZmYWUxMTQ1NjI1MzMzOWNlNjMzZTNlZjUyNTA2NWFmYmVhMGM2ZWQ1ZTgzNGYzOGJkMzY4NmYxZDcyZWY3MDdmMTUwMDA2OWJiOWI3OTNkODRlZDMzNDlmNzY4MGExOGFiZWRlMGQ5ZjMwMWNmNGY5OWQ0ZmVhMzM5ODk4M2M5ZjRkNjI4NWU3OWViNzhkNzYzNmM5MWY1NDFmNTA4NDk0YWJkOWYxODMxNGM4ZTlmY2RkMzY4ZGQzMDA3ZWYwMDFlYTk2ODY0MDNhYTZiM2YxNTIzYTBiNWU2Y2Y4YmVjYTRkOWQ2MDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9xwwiCZ7XDvwYWIEJSSpTW5Ifg55gKMiyl55-X5wzKbiQS8Jl5uceWTzcXVPeA1lNxJQvl3UyGXuaayMBqub6A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110133_15_248c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.589Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImJPUkdOWko5TUVJQ2xUcURYQ1M1Rk5IcTU1ay9TamUrUk9GdWtqaUl1cHNFVzNnczNyZkZyWmhtblY4eVlDazBVai9VaEJrVlJkK2FoZE5yZjRUbHVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzNfMTVfMjQ4Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGExYzAzYzg1MGQ1ZjBiMDM2YmU3ODZhNDU3ZWY5ZTQ5MGUxYWE1NDYxMWVlZjQwNGI5YWEzNWIzOGIwNjFjNzAzNTA0NmI3ODMxNmIzOTE2NTMzZmFmMGY5NzA1NTlhNTM4ZjcwYzQ5ZmVjNzNkNGI3NzEwZWZhNmFiOTQ2MmEyYzU1ZjMyMDIwNjFiMWQ4ZTY1ZjE5NjI1MDdmY2M3YzBhMWYwZWQyZWUyOTI1ZTEwYmFjMGM5MjllMTg2ZGNlZWRhMjEzMzk4MTQyNmRlNjdjMjdmYmI0NzkzNWE4NWFlZmIyZDkxNmM3ZDVjMDljOTJmMGJhMWI2ZTVmZDAxYzg5MWVkMTgyYjNjZjU0ZDViNTBmN2M3YTkyMWRhNTI1ODcyZDc2ZjAzMzgxZWI1YzYyOGU1Y2E5N2Q3ZTZhZjViMmFjNmE2ZWViYTRjZGMyYTdmNmE5ZThlMGQzOWI4MWY3YTM5YzkwOWJjYWNhNTg4NTA1ZWZlOTBhMmM4MGFkNmM5ZDliZWVjMjJjYTJiYzViMTJmYjM0OTk0YTUwZGVkMzllZDEwNzg2NjEzNTY5YzkxNGVmNzU5ZjY0MjU3ZDQ3YmMxZTAwMTZjMGI3MTY5YjVmODg1ZmRkMGIwZTI3MTI2NWY3OTk2YjhlYjVmMjE4ZDQ2ZGNlNDVlYjFkYzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nw_G_Zu7VeCHpbCIyPhGPlw4_mCQ1tztonK0RCaLgd_BYn7QCJgYsNACUf1h1mT0d56y-Yjp-G-u9H4weU9DuA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110133_15_248c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.592Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImM5OE16a2hKeEppL1Y4dTk4V0dBYWZLQjJobG1mY3hBMUJaYTE0cWZVVWZRWURxTzdYRFpBRWo2dUNCZDhHZ3d5WWpEV1ZyM2g4bmo4OGgwUlU1K2R3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzNfMTVfMjQ4Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGI4MWIwYjJhMjRhMzYzYWJmMDU2ZjQ4ZmZjZTAyYWQ2NTk5ZWI0YzlkOTk4OTM3YjY4ZTJlMGJlYTBiM2NjM2FlMzQxNDE5NTVhNWU1MzQxMjY3ZGQ1YWVmOGIxOWNkMzZjNDhhZDU0M2Y5MDY4MWJkMTlmYWFmYTMyOWRhNjU1ZjZhOTU4NmU3YWJmMjdmNTViMTliMzQ4NWQ0NjRmMzdmOGYyNmE2NDQ1Y2NhNjdhODAzNjU1Yjg1ZmJmNmY2MGJiN2VhZWM4N2ZiYmRjYTFhZGYwNGIwZjU4YWQ3N2M1ZDcwNmRmNGYxYjU0YTBmNDEwODU5Y2ZiMDNiMTI2ZmU1NTc0MDllMmM1NzMzZGEwNjNkODZkNTBkZDM5NTY5YTI0ZDhhZmNmMmZkYjdkMjg1YjNjYWJmMzlhZjg1ODY1NDE1YTRlMzc2NjA3OGIzMjFmN2EwZmQzYjVlMTA5YWE2NzZjMGExYmJmNzcwYWUwNjNkODQ2ZjE1MDg0MzIxNjAwYmMxYWY3MjExNjIwNDEzM2ViYmM4ODNlZmY0NjM4MWIwY2RhZWIzMjFkYzMxZmZkNDFlZDI1NWQ5OWFiYzVkNzE2MmU0ZGY1ZTc1Zjc5ZDNiMDAwNWE3NTQyZGQ0OTEwYzRmOGIxY2YwNGUxZDI1M2Y4MDkzNWQ4ZDBlZjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-IU_hROwMvYwlsSUmNvOfTLpC6jbIIMjayg1u-lFd3EqnKrDD1LEsesBwGg3fFrxCWCDSnC71eUSCzY8_N19qQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110133_15_248c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.609Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkpVUk5nT1ZXVHg5N3FJOUx4eDFuQXRMclRoakEzN0VvZGZYb29OYy9mL3d6eWN5SVNpbVNvTmJFbGovWVVIdTA4S2ozTFFzZWNRejBDdjdjcERFMmN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzNfMTVfMjQ4Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY1MmQ4N2I2ZDYzYmM5YmRiNzZlMTFjYjAyOTViYTk0ZDFkY2UwNmUxYWYzMjY0MzExMzI1ODkzNDAzZmU1ZGUxZDU3MWZkZDdhM2MzOGExOGY1ZjBlMjIxMDJkZjdkZTRjMjc3OWZhZmE0YzYxOWI1MmI2MDkzYThjNjAzMGFiMmZmNDY2NWE4NTM2YWM4MjE3NTM1ODlmZjMzZmM0OWMyMmRjN2FiMDhjNGIzYWU3NmI4ODA4ZjI5ZGRhNDA0MzBmOTIyYjNhYTBlY2U2ODg4NjFkZjhhZmM0YTYyNzc0NDhlNGE3N2MxYzM3ZmQ0N2Y2Njk2ZWM3YzhjODNlMzQ3MWQxY2FlY2I1MmM0OWQyZWM5NDE3ZmVkZjkzYTg1YzMyYzRmOTQxMWExMmNlMWQzNjAwODYyZjZhZWE1YWZkYmRiYTI0YWI1N2I4YjQyZDZmZDg2MTI2NTBhYTY4NjIxYTVmY2ZiMTc2ZDQ4NzkwNjM0MWIwYTgyZjZjNWZjYTg4M2Y2OWNjZTA4MGFhODYzODdhZWY0ZDEwYjYzMmMzN2U4OTI1YWQ3MzIzZGQ2YTBlMDJmZWFiYzdlODY3M2YwMWM4NDc0NDVlNzA0ZWQ0OWVhZjU5ODA1YjMzYjNiYTQ2N2U1ODNjYWUwMDlhMjAzZGQ5ZjYwNTk3ZGE4ZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.g8KaCcg0CMm0qc-t0wInio7r4JK_0skOqFlzHMFkC4KG9jICO_QT0el7J6c9u-_cp7S3aaDVwvXIeJUp6TwMNQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110133_15_248c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.614Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkhsVUNwVUp6TlNqeFdaZzMybEhBVWRhRUt6QzJWak0rd3dWZ3BINS9XWUcvN2oybWtUREQ4dDErTU55b1pCZmw3T01HS01CK1UvQS91ZXcvSm5xU1FnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMV8xMDMwNDZfNDRfMjRiMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTRlNWVlZTA4YTI5ZDQ3NjljZjAyOTVkZTYyZmEwZmUyNzc3NzFlMmE0OTQ5OGYwODUyYzdlMjVlZDgxYzJlYzI0Nzg5ZGRiZjhjZDhmODdiY2NhODU3YzVmMGEwNTkxN2U0MGI1YmI4ZmI4NGE0ZmM3ZTIzNGNiMzg0MGVlN2I5Mjg2ZDdjOTE2NzczOGQzZDIyNGMyM2QxZWEyOGU4ZDQ0ODEwN2NjOWZmNWYxOWUyOTU5YjQ1NTA5OTk2Nzg5MjE2NTVjYmRjNDVkOTIzN2E0ZTQ1MzgwMzUxZDBmYzc2MjI0MjMwZDViZTc2OTgxNTFkM2NkNTU0OGI3NjZhODUyNzk3OWE1NWFkM2IwMTU3MTQzOWYzNDM2NWFmOWJmM2EwYjFjMGMyYTIwMmRlYmYyYzZmMWQ3YTY4MGYyNTI1NjY4ZjIwMWEwMTgwOTM2YzI3ZGVjN2M0MTIzMTcyYzRiYmIxNzNiMTgxYTJlZjg0ZGRiOWMzNWI1ZDZhNWFjNWQ3MDEwNDg1Yjk4MjM3ZjA4YmI2YjE3NDUyNzJhNmJmZjQzYTM1ZGQ5MTIwNjgyNjNhNjk0MmQwN2Q1ZjdlODc0MjhlZTgzZGNlMTJlM2Q2ZDhiNGIyMGY0ODk3NWM2NmQzNWQzODYzNGMxMjdlOGQ3Zjc2YWE5NTgxMTRlZjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.o_cai1z4gpYadUFNtjhpHzz3jZu7o-uUY-6Ste2P2PESfLAWx9-kX29PGlPM6nZJLfF69dc3TMidXJrZTdKfrw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230911_103046_44_24b2_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.617Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkVUQzFXdGU5b3hWSmM3WjBWbDFFdTMzMWlyVVZpQ3FGdEt4ZzFRMGpuODBtR0NLdTY4eUovRm5WTmZtSWs1dkFIeDFxSGxzZ0JrdzMrNTFCeVJ3TURnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMV8xMDMwNDZfNDRfMjRiMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWIwMzBlOWZiMDA1MTI0YjgwMTlmZTEzMTg5YjVjYjc1NjExZjQ0OTRlYzdmZWZjYWJiMTE5NjE5NDcwNzI1ODhlYTBkNjZlM2E2MGNmMjZjZmRhZGYyYmEyMzZjNWNjOWFkMTMxNjYzN2RmZTAyMThlNjA5ZDM4YTViZDQxZGQ0ZTY1Y2JkYjk1YzE5MmUwZDYwZTliNjM5NWJhODVmMzkwYTc2OTA3MjViMTk2ZjZlZDRlZTdhNjliMGI2OTE0M2VhNmYzMTc3MGEzY2JhNzdhOTM3ZmI5MTlhNjMyMTczMzM1NGIzMzNhODI4MWYxOWMxNjRlN2M5ODI1NTI1YzBlODgyYTBmMjM4ZTgwNGZiYTQ1MTYwYWE2ZTIwNjNhMWQ4OGUwYzhlY2E0NDQ3ZjE3MmM0MTQ4MzNjOGRkNjljMzIxMWQzMWVlYjU3Y2FiMjRiOTBlOTJmNDZiMGY5ZDJiN2M3Mjc5ZGRiNWU1MzIyNWVmOGU0MTJmYWRiNThjNWVmYTY5NzIwYTkyZTdhMjA2YjA1OTZjYzZjZjRmNjNmZTIzNDY2ZGRlZmEzOGY0ZTM2NDZkOTlkMjBkMjliMmU4NzEyNTBlOTg2NWMxNGQzNTJlY2M3N2E5NTU1Mzg1NDFkNmJiZDJjMGM3MmMwZGNlYzkwZDE0YmQ5MDFmZTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QQaD5X7GZF2kFT4CYigTTqMjJ50kjGaU_6pPuFHwoCP_woKWce_iqJtMztOWd6e_VVgSpRh6GEpBajD76Mhehg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230911_103046_44_24b2_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.620Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im1lZ1BiSFN3ZEYyS3NJZHp0ZXl4eFM5cVlJekUyOWpCaG1pQmtvY1N1bXZJYVFUbThXVHU1Uisxd00wVG9qUDJtdHZhT2dsYS9iMUhUQ21jSWRMdjFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMV8xMDMwNDZfNDRfMjRiMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjM4OTg3ZDI3OTYxMmU3YTJlY2UzN2U0NmQ5NzdlNmNjMTU1MDVlYzJmYzc4ZmQ3N2VkMzZiNmQ4ZWNlZGYwNTM5ZmUzN2UzYWQwNjQ2YjkyN2E1NWRmZGJkMWFmM2FmOWNmYTFlN2E1NDlhNTQ1ODhiZDFhNjNkYjk3MTBiMGNhOTgyYWQ0MWMwYTA4ODVkYTBlM2E0YTE4ZTJjZjBjODUxNDg4MjA2NjBkYjE5ZjdlNDU2Yjk2MTllMTcxMmQ2YjJmMjM5ZjBkYmJjY2U0YzkxZDA0ZjExYzA3YjlkOTUwMjgwNTZjMTU5N2FhOGRjM2ZjZjI0NjBkNmY5YzIyZjJhYzI4OTJhMmU0YjM1MDhhY2I4ODZiN2JjZmRhMzQ5NjFmZDE5YmQzZGUwYWM2NTk0ODY2OTc3MTRiMjAyOTI5YWJkNmMzZDE2ZDUxMWE2OTlmYjBiZjRhODZkNGRkYjk4OGVlZGY1NjkyNjU5MDBiNmUxZjQ0OWY3ZDhjNzQzNjA3OTE2MmYyZjI3ZDU3MWM5NGM5MTE3MDMzZDFjYzJhNGIxNmQ1YjllYmRmNjBhZjIwZGRiYzAzMTE5OWY2NTYzNWFlZDQwNjY0Yjc4OWQ1MTYwMjBmOWUxOWZkM2YzYjExMDg0ODI5ZGYwYjVhMmM3ODY1Nzk0YTViMDA0ZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-uw_IcSTdJ1Z2i5ongBTJ0EdShrbNOcCTR7fLAuT1efqIslLJmNs3qSuvz3sv9eyR7SgKfO22MfZlNYa2ZQWYw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230911_103046_44_24b2_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.627Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImtiTVQzb0xkT2Y5dTZmbWpOZ1V6SitGNXR4ZDcxVUh5Njd0S21MSHJqM3hMcGxWQkZBTFM2MTIvRTdNdUt6T2xWNGlKcnFUdmQ0NU9TMEpYV1VJcWxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMV8xMDMwNDZfNDRfMjRiMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGJkYmVkY2Y0NGNmMjhmMzhkMzI0MmE4MTIxODg3MDY0OTg2ZTY4ZGYzMzc3NDE0MjY2ZjZkZmY2NTY0MTIyMWM3MWM1NjQ0OThjOTc5ZDkwZDYzY2RmOTEwYzIwZThmODZjMzc0ZmYwYWRhYTBhNTgxZWEzYjQxZmM0YjBmMmU3MTUwZGExMzVkNjVhZGY3NWE4NzIwN2VjZjJlYjUzMGY1NWVhN2U4YjBkYzU1NGU1YTIxZTUzYzU3MzBiZDI0NDNjMTIxMDExZjQ3N2Q0M2FhYzc5NjdjZWYxNDQ2MTEyZDY0M2JhOGIwNTFiYjgyMGI3MTcyOTE0NDMzNWI5NWM0NDA1NTBhM2Y1YTNkNmZiOTA4Njg0YzEzYTg2OTUxZjk1YzgxNjhiMzIzZmRkNjVlYjRmMWQ2MmJhMWZlZGU4OWM4NTEyY2U2MzI3MDkzYTFlODFiZWMxMDNhOGYxZDg2YTAwNTU2MzVjNTc0ZWMyYjRjZTYzNmVkZTkxMjUyYzRlODI4MDVhMmFkZWEyMTM2NjQ4ODAxNTY3YWMwYTA0MzVjNDVhNzg1MTZjNDZiODEzOGViMzg0YjY2MzkyNzE5NmMwYjdkYWQyMzRkNjI0MTZhOTgwMTg3YWFiM2NhYjgzNzQ2N2M5OTk4NDAzOWQxZDM5NmRiMDEyMmI2MzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tnC9775qpr7i-CfhflSlPl0-RV9uQmABxTiapQXLkWJqfI8iGdTK8FEdI2qlGXahHyzqcIOF5K-1Mo3Rb9U7-g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230911_103046_44_24b2_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.630Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InVrOW5PZCt1eW16ZUpiWks5eGJpZmYwbjlOd2FTOWt3Q0VoTWg4WjdJZkQyTHlvbHRPejVmSEhYcEhwaUdDanR4MmVLdHdrVFI2MHZtdGM1YndQYnpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwNF8xMTE2NTdfMjJfMjQwMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzRkZjI4ZmMyNjYyMmJkZDE2ZTk3MDUxZWJiM2I4M2VjZTAwNGJhOGMyNjVlZWVkM2YwNzY2OTVjNDBkNmUxNWIxMjUwNjYzMjgwM2MyYmY2YzRhZjQ1NjA1ODI1YTZhYjdhMmMzYjc2NmFkZTI1NWY1YTM1NzA3N2FmYzM4MzZjYzUwZmNhY2M0NTFjMDVkNjk4ZGMzODk0OTViNWZmYWQ4YTA5YWY5ZDI1YmU1NWY3OWY0M2ZiMDIwNjY3N2ZjNjU3ZWMxNTU4ZDZiY2IwYzJiNmMzNjIwMTIxYTgzM2JjNGZjZWJiZjY3ZTkwYWQ5ZjdhNWI0ZjI2MDY3ZDU1MGY1Njk0NjJiM2U0YjRiMGM5NmFlMDJjZjdhOTkwODBhNWJmNWFmNWYzM2NjN2U4YWRiMmI1YTNlYWM4ZTk4YmI1MTAyYTM3ZWU4YmE1Y2VjNGJhOTk2MDQ0NTFjY2QxYzgzZjg3YTZkOTYwOWE2YzU4ZTZhNmU5ZDQ1YzNlZTJiMzE0N2NiMTQ2OTA2YzE1ZWNjMTI5NDc5M2ExNTc0YzQ0YzcwNDhlMmQ4ZjUxZjAzMTg2MWI4YmNmY2I2NzRhNjI4YzZkNDhiYzJiYTgzNTQxNGY3MDJjNTM0NWZmNWZjYzFkMGI3OTAyNGMwNDU0M2Q2ZGFjMzczZDE0NWUzZTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vYCGB2M9pRTgEf-mRDwVs3jlYHe-F73n-YknOjpSDSZUFXTpjEsuvWJ-QnM9ltXeOFk0K7lR3HeS55Y5tkkWEA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220704_111657_22_2402_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.634Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlVDVEVRWjBBczRlcjgxU1l5cXlIZXNzWCtMOEx2Z20vVFRrck91Sm9FSE1yMFR4RTQwNFFDQzh5MFpEK1dlQTh0T3BVbVgwczJxZGlFUEtBVnBOU25RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwNF8xMTE2NTdfMjJfMjQwMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzY5ZmVjMWEwNDY2MTY3YzczZGJmNGE2MTAzYTQwOTNiN2E5MDI3OTQxYjgyYzExNDAzNWU3MTlhZDVmYjYxNDFiNDk0ZjMyMGY5ODE4YmM2YTI2YTkyNjgxNTQ4YmZmOTNiNzgyOTU1NWMwNmE1YzFlNDc5OTI1YzU4YTI1MGVlY2RlYjEzZDI0NzA4YzQ2NDkzYWY1N2FlMTg1MWRlNjRhNzE0NDk2N2E0MTIyYTE1OGUwZDBlOGU5MGVhNzA4ZGU0MTNiZGEwOTNmMDQwNmEzMTA1YTgxNzcyMDQzZTY1ODJkNzBiOTA1NDgzNDMwZDNkNTliN2MxODhmZWNkNzk3YmFlNWM1ZTJmOTI5ZTA0OGVmODAwYWMwNTBkMDYwMmJkYTc2Zjg4ZTUwMmFkZGNkMzU1MTgzMDVhMzgwNzQ5OWNkZDFlOTM4MjgxYzg2NGIwNjgzYzhhZTdmNDVlZGM3ODkyMDllM2Q1NzQ0N2EzNjgyZGEwMjI3ZGJmMWQwZTMxMzIzYTA0YTE4M2I0YTdhYjM5MjFhODU5NWRkOGI1YWJjNzlkZjkyOTUwMTQ5YTZiNDdhNmE4YTY0NjdmMWQ3NDY4MTZkMjMyYzYyMjUzMjAxNTJhYWRlYWQxMGU2Y2Q4ZDU5N2RhOTcwZDVjZTk4NDJlMTEzYzQzYjQwMDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ySv-6-axYWd-y92KUrstAuoEDGvt1EUbJaAu_kwSQrrWyzvz8d3fujmzfDw5DwwuC70Yd_iTWEI6ZAuRJ-1dTw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220704_111657_22_2402_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.639Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjlNbG9kZE1XMEhrUXlGQjBpczRJM3U5Y0FodFZJdG5aR3NVck9hTGtvc0ZJWFhvN2M4Q2xZUkI0QkZGejltZFNIdTUyc2ptK0VxQnRVUTFOSEtBYWtBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwNF8xMTE2NTdfMjJfMjQwMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmU2M2E4OTI3ZWJhZjNjYmYxMmEyNDQ4ZmYyYjk5MWJiYzFlM2JkMTUxZGI5OGVjOTY5MzYzMGI2MDM5ZmZiNTk5YmEwYmM2NzliMjcyNDRkOTgzOGE3NjBjOGY5Nzk2NTQwNWI5NDdjNmM0ZTA0MWEwZGE0NzRjZWY0OWM1MTAyYTU0ODY1MDU3YWNiYzdjNGY4MGVmNjk1Nzc3NTgzMzgxYzZiYzZiYzYxNTFlYmE0MWUxNjIyYTNkYjYzZjkzM2FkYWZjY2EzODMyNWIzNDhiOWViZjRmZDg3MmFmZTA3MjBiZGI5NTdjMTFmM2UwZGMwMjg2MzhmY2E0NGMwNTIxNDYwNmEzMTliM2E2MDNlMWUxOTcyNTM1MGExY2YzODBkODQwYjZiZTAwYTA1NWQ3Y2M0YjQ1ODRhODJjNTZlY2VlOTA1YzAzNGFlZjJlZGRiZTg3YTExYWYxZWY4NmNlODljYmVmYTE3NGNkODEzNjE3NzA1ODU1YzhkYzY3ZjZjNmMxNTE1ZTgxZTEzMDg1OTczODAyYTViMmJlZjZiODE5M2I4YWE1MGZlMWJkYmEwZjA0NDE5NTg5YmNkNDQ3ZmU0ZWRmOWYyODYxNzg3ZmQzODU1ZmEyMjZhZDM5ZGEzMjkyMDM2ZmJmYWY4MzUyNzFiZTM3MGRlMjMwOWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wApXtUwHJ_SLRU76B_YpbMejhmKMhvZHzKpeb0hL9arW-O5jf_t0Kh8b_seHpu1fbcBRfAcfNXru_MWp-1k0ew", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220704_111657_22_2402_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.644Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImZWMUtzT3pPeitqQkN6V1h1M1B6YXlFRWd3TFI2b2E3elZXTExvUk5zd3lQd1cyNVRRUDJVa1RXanlWVmlSMVJjSFB0VmdIbUMzdWNTSko4ZkJKTll3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwNF8xMTE2NTdfMjJfMjQwMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWYyMmM2ODc3ZWQ3OWNkZTMwZGNkZmFlNTNlMjlhMDljMDk1NGRkMmFlOTI5MjMwZjllZWMwODg4ZjE1ZGEzNmRmMzE4YjIwMzBlZTEzYjZhYmQ0ZGZmNmY2ZTk0YjBjZGIwZmNjMTBjMDhmZDM2YjIyNzUzN2NhNzk3M2RhZTdhODhlMWJjMzUyY2ZkYzQyMTNlN2FlMjQ4MWVkNGZhYTc2ODNmY2IyNDA1ZjE2YjE1MWEyOWVhMGY5NTY3YTc2NTc0ODdmNzFmY2JlY2U3MDE1MDc1YmYxYjYwYzgwNjBjNzdjNDYzMzM0ZWYzNWUzZDc0MmY3YWEwMzFlOTUxZGIyZDI2ZjllN2I5N2MyMjU5YWYzM2Y1ODY0Mzk5NjRiNzE4MzE5ODY3ZTVmZDMxMDQ2NGY3ZGMyODY1OGI2YzU0YTE0NzIyMjMwMDkyY2EwNDNjNjk4OWU4NjFjYjVlZWUzMzAzZTgyMGZlZGRjOTk2ZDNkMTdlODU3M2FkYjliODQyMTc0YjczMWJjYzY0Y2ZlZTNhODg5YTk2MDM3ZDk3ZGM3MzE3ZjQwNTZhZmJhNDYwOTY1YzZmZDkzZjY5NGJkYTc1ZDgwOGU5ZmVjYjFmYzVkYTkyZDA0OTQxMDljNjBiMjAzYzI2MzgxZjhmYmViMjUwMzdiZWY2YmIxODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-3JTIVikWuQ2grRgfUPOg7tvqgTmIGqYBM60s0S1otPSGCij-jv3T2qjVpM80-t58uY_OooNf_hFrIOeVBVwMw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220704_111657_22_2402_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.647Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImtuaVFXR3g3VUlTYmNyV1lYQkQ1NEVMejJmU1lrVzRCSWlHSFZlbkU2ek95NU9UN0l5aGNwcFlxSUd4N01USStYZHRIVWxrRU5HQ25lRlBDSUhxNVNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDVfNTFfMjIyM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTcxZmU2MzdhYzc1NTY3NDVjMGM5OTE2YjJmYTA3ZDgwNGJmOGUzMmFlODlhOWExZjEzNzkyZmVhYTdiY2MwNjNmOTIwMDMwODcyY2JkMTEzNzdiMzY4N2Q4ZDVmZjM5MzVmY2FmOWU2NDA1MGEzMzdiOGRlYjk1NWY3ODc1ODBlNThlYmFmZmIzYzBhMzkyMzdhOGM1NDU5M2MzM2Q1NWE3NzI4YTQ1MjMwYzQ2MzZiYjQ4YTQ3MzZlOTRlYjBhODU3YjhiNDRiMzY1NWZlYzE4ZGJlMzNlNDhiYWJjY2E2MjllMmI4ZGY4ZjNlMjk2YzZhMGU5NDk2YWQ0ODMyMTdjNjg0YWI2Y2ZlM2NlNjUzZDFhNmQ3ZjJhODJiYTE2NjkyZDllN2U4NDY4YTY2N2VjNDQ0MDNlZGQ4NGFjMjI0ZDQzNDQ4NDg0ZGZmZWU0NGIwY2MzZWIwMTQwNzg1MTg5YmRmNDkxYTI4YzEzNzkwYjM2MDg1YzI2ZTk5ZjcxOWZmZDNmN2M5NDUyZTRmYzM3YzczMzI1MzIwZTc5NDA0NTY1MzExYjNiYzA1MzE3NjQ5YjY3NmFmZjc0MDQ5ZGYxYWM5NmMzMTkwOGZiNGNmMWJiODQ1OGEyNGNlZjU3YzU4YTQ1ZWUxMDVmMWM2YWY4MWNjZWY4YjExODIwMzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.H0TbOpCgo_bqq4huxX-vIwLWtKMnhTAcCNRJY17TXJs7K15SdKJkLaldIEWa9zpm5vBcAwtfhv5sS8vixDKbKQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104045_51_2223_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.651Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InlVS0dCVUYzUVdMZmwydDlYZ0twbWRUUmpEVFhZbnRCODlXZ2l6SVRldi9hV2ZibWdrbEVpbnNDVEtPaUFSd2N2YzhXaUpmbitmVFhIdTBjcWNTQ2VRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDVfNTFfMjIyM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmNhYmE3OTZiNGJiNmM4NDFiZTg4Nzc2N2U0NDNkMzJlYWU0OGE0MzhmMzQ4MzI0NjJjYTZkMTVkY2E1YWY5NTIyMWE3OGUyMGY3ZDlmY2Q4MzIyOTkxNmM2ZDJhMGMwOWY5OTMxYWM4MzQ1ZDA5ODVhOGFlYmRiMDJhMjdhY2RjMjU4ZTg2NDkxODcxNWRhMzk1ZmMwMzQzNzg4ZmI4NjUwYTg5MTRmYmVlMGFhM2VlNDA4ZDA2MDI3NjYxNDc3NzJmOGE1MDNhY2EyMTZiMGYwMjdkMzBiMTQzMDhhOThlZmFkN2VhYWFhODcxMzhkNDkxY2QwZTYwODNlZmQ2ZmRjMTcyY2MxMTUxOTJmMTk1ZGY3ZDZmN2Q0ODVkMjE1ZDEwNzU3OTkwYTIzNzNiZmI3YTY5MDVhYmY5NGQ2M2Q3ZjhiOWViODRmODg2NzZmN2NmZmEwYmVjYjliYjc1NDc5ZWQ5ZWEzOTVmOTUyZDRhYzg5NzRkZTJkODcxNmI4OGVmOWNkMGNkMjc5NjRjYThjOTAwMTA1YjE2YWJlNjE4MDQ0N2UyODNmMDlmZDFmNDMxODk5Zjk3ZjU0Y2ExZWU4ZDNlZWNhMTE2MzNjOWQzOTk5ZTFiMmY5NGE0M2FiYWVhMGMzNDQ5MGVmNmZlODlmZjhkMjNlNTU2MzE3MzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cX7RqzojCR-bNkTkbL4LlktYevXy_ON-w2sRAHFWihwftqORVAlxf4GKoK84v5x5Aec3yj4k9kJMLXmeadrjHw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104045_51_2223_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.654Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlFCVHdIZmV4RzhpSEdOdHdXTVFZc0JoTmZkNVNYN1UzMmZiMEFFcW9HenFsQkYveWJXS2JVQlFFamRsZ2pmQk1tcStpZCtTWGtQaGt1UFdMRFU3YStRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDVfNTFfMjIyM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjIwM2IzMDQyYTRmOWYxYjYzZWYwNzI3OTJmZTViZGMzNzNhOTIxZmUyNGJkOGI5OGVjODA2YWM1MjdmN2ZkM2M1YjZkMDk2YTAxYjM3ZDk0OGEyMTA1YzU2ZTA4MDgwZWE2YzRjODFmZDZiMGMxY2QzYjY2ZWJkMzI2YTQxNDFiNzRiZTU1OWU0M2NkY2I3OGE4NjMzNzU2MDVmYmY5MWJkYWRhNDIyNDlhYjMxYmM0NmQxOTIwZTRjYzQxM2FjM2RmYzY2OTA1ZjBlOGZhNzM3MjYyYzBkYzJlMjRlZWJhMWVkYWNiMDJiZTdiNTU5N2E1NjFjZWY2YmE4OTU3ODY3ZWYzNTE4OThhODhkYTJhNGQ3Mzk2ZjQxNGNjOGZhZWJhN2M5MmE3MjFlODUzYTEyZjBlN2UwMDZiYjcxYjZkOGUyOGEzNDdkYzRkODQ0OThlNjkxOGU4MDIzMGEwMDQzMTMxZGFlYmUxMjgyODA3M2NlMTk4M2ExYjk0MmE3NWY5MTI0N2YwMWEzZTBkN2IxODgyNjIwMmNlY2FjNTU3NTJhYzM5MTJjMThmMWEzNWViYjU3ZDNmM2ZjYmUzMmM2MTk0NGQ1NWVhMWI1YjU2ZDBmOWI5YzQxN2VkMGI0OTBlMmQ3MGRiOTcwZjM2ZGJkZDg1MWZmZTkwMDg4NzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fBvrwxn0DilcCUuAHzlcmeQjsG70VhptFx6Kh3n7Y9B4oO8e4Cb07FlxaUUpGz5azCf2qsG3dx4a09lLXwi6qg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104045_51_2223_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.662Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjN4WDhVSTdmdE81OTBVZ2NaTWFoNXJEVE1MVHd6M0N5QUJlMFAwV1F6OG55aTYvTWhMNlB6cW1IdzZ1NmtSWTRCVXlOTlNtZEFRdDRNdDlwUDJBc0RnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkxOF8xMDQwNDVfNTFfMjIyM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmIwZWJjNTFkNjRiNGFlZTNhYjZhYTJjNjAxZTNjYjEyNDJjN2IyY2RlZGMyNjlmZWZlNGI4MTgyYTc1YTA0MzI0YjQ0YWY1ZTAxYmFkZTZkNjc2MTY2NGI3MzQwNzVjZGUzZjJhZWExMGQxYjUzODY2Yzk2NmEyYzA3YmZiZjkzZTg1NWZmNjFjMmU0YzgyNmUxZmI3MTc2MmQxMDE1MTg2ZDc5ZmNjNDg3NzQyYTE1YjIzNzRkZjQ5OThkNzNkNjAxNjg1MWM0Y2Y4Y2NlZWY4M2UyNzUxZWVlNGE1ODM3NDI3MzRhMmRhNmIwMmIxYzY1MDVmN2YyMWNjNmY5NmI0ODk3N2I1M2ZkOWI1YTI0MTYyZDczMjdiZWM3ZTkzMWM0ODIwYzk0MWM1NTc1MjA4NzI4Yjk0M2JkNmNlYTAxMzIwOGE1NjQ0ZmZhOTg3OTc3YzA3YWFmMmIyMDUyNmViODgyYTlhMmU0ZTZkZmNmN2EyZjcxMzJjYzJmNmI0OGJhOThmZDc2ODU4NjVjNTBhNzVhMTZkYTRiOWYyYjZkY2RhZmU5MjU4MzkwMTdkMGJhN2I2N2FlYmEwY2ViZDU1MGE2MTFjZmE2ZmFkNjAwMTlmNzEzZjNmY2FmYWJiNzQ1YjUwZjQ2ZjM2YmE3YWJjZmU1MDZiODgzNjM5NmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.K9R7_np1UokLq7elO4icxMy7YQq-tSL1Cg99fHI5tDajrzcUmiLh5SYllbxvYk_XCtideqfv-cDKPqpYdBB9fA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200918_104045_51_2223_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.667Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Iml2cVhET0krQnJESjkyTzVMVE1wcmVWbndNQ1NncEVoY3R5VGs5MFl0M2ZPQ01FcVFmNzFoQW1valBQNHNHaVg5NkFxSU1pcjFRVk1HVHpuZkdRM2N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzNfNTVfMjQxM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTA2ZTdmZGM5Yjk2N2JmYjdjZGVlZDljZDA5M2ZmMTlkMDFjY2NkZTc3OTU4MjVkMjA5OGMyNzhiOTQyM2QxMmQyMTU3MTU1MzA3Y2U5NmUyOTZjNGRkM2FiMDk3MDVkYzRkZTlkODA2YjhjODJkMjg2Yzg0N2I5ZTYxZGI2ZjAyZTQ5YmZiYTZjNDZmOWUyNDVmMDc5YTQ4NDM5MjIyOGNkMTM5ODI4NzYwZmM1NWU4MGMxZjVmZGZhNGE4MDNkZDM1MTI4YmY4OTEwZGY4YWVhMzQ5ZWM3YTVkMjQzZGY3NzkzYTVhYWJmYTE2ZjI1YTg0ZWY0YTUzNDliZmVlMjRkMThiMmQ2YmFkMTc5NzJhZjgyOTk4YjQ3MmY1ODRmYTY5ZmUyZWZlM2MxYWQ0MmQxMjBhMTE1YmQ4ODI3YTY0N2IzMzdjY2RkOGNjMjQxN2M5OTdmYTYxMzNjMWUwNWU2ZmNkYzliNzAwZjNkYWFjYWUwMWQxYjIzMDFkMWI1ZmFlYzU2YTI5N2RiNjA0NTcxZTFmYTI2YjlhZmQ2ZDg0NjgxNzQ3ZGQzZTRkNGUwMGU2MzMxOTM1NDUzMGI5MjExNTk3ZWMzYWY0OGQ3YmRjMzQ1NTcxYzhiODZhMThhNTllN2YzZDBjNzNmNzRiNzYwMjY0OWMyZWQ1Mjc2MDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qXdi_tcrDUq9pgLslg2STiE74IWN44mc7j_Dt-5IzL0gtUiCNiirnJbEHd8PyUXfpW8grIMdHVaYmBlkDaFZ-Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112633_55_2413_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.671Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Imp1YWhnMmlrWmZ0OE1ibG9NOUZyWjRwN25HWEpScWZ2MDVFTEIrSSszMFI5ajQ4bTFNWlp5dVZVK0hpb0RJZnY5U05WSlZwK3ZIWU5TSDVMUDgwdC93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzNfNTVfMjQxM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGMwZWYyNTdiOWY5N2Y0OTY3NjA2YWVjNTc0MmE5ZDQyNWI5ODViMTJmNTU2YjQ1MTk5OWM3ODljYjJlYTFiNWMwMTgwNzdlNTAxNTM2MjE2ZWI4ZmViMzFlODVjYmUxNWY5NWIwMzZhNGNjZGY2MzVjOGU4NTU2MTBhNmY5MzUzMTllYTE5ZTkyMjgzZGJlODE1ZWFmYzcyZDNkMzg4NzMxMDU4ODgzOGQ2NTI2OTg2NjI3NjgzODNmNDIyYTI0NTJkNmJlMjYwN2Q1NDIzMWJmNTMzYjUyYTg5Zjg1YzJhYTg1ODk1Y2Q3ZDE2NjUyMWNjMmRmZGRiYzk1Zjc0YWE0YjIyY2Y4NWY5NDZlOTg5ZTI0YWNkMTU4ZDliMmMwNWQyNWQ4NjQyNDkyNzQyMjJhZGZhNGE3ZTQ0YjAzOGFiYThhN2QzODMxOGUyYjFkMzBmYWM3NzU5ZGRmOGJkZDZkZmVhNzU4ODU1YjEzZWFkZWNhZWNkYzAyNmE5MDU4YjIzYWVmYWM2ZTkxMWFkZTUyMTQ4ZmQxNzZkZDYxNjVlYjIxNDc2YWQ4OTU1ODcwOTZmOTY2Y2JkMDNlODkwNGQ5YmFlNjM3NTU2NWYyOWExM2Q3OTBlNmM3ZDEwMzA1MWI2ZjcwODNkZGJlZDk2NDQ5NWU4M2EwMjVlNmUzOTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NIW0R2xf2AKFiORP_5FI_HJVNwAbXx7KEahVzW3VFWVErgTXhHXFwr8TQUmrbPtmpw3n3fGB-npRI7XZgooYGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112633_55_2413_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.681Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImxKVGE1L1UrS1d0TUhMMENKem04eEdraXdkZWFMUmcyQWU0YVlHVExYS2pzcDE3bko1Zkpteml5dEJhWnB5YklQTkh6TFFBL2lWQ3V3YnRXMzQ2WkZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzNfNTVfMjQxM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmJlZmE3ZTkzNDE0MTU0ZDYwMjNmNGMwMDFlOTNhM2JmZDYxYzhiZWYzNzc1NTVkZmRkNDZhNDBlNmI4Y2M4Mjk5NjRjOTNmZDc5MTNiOTdlMmRiOGZkYzU3NWY2OTU5NmI5MjMyNzFiOGE1YTBkNDRjYjQ3NzExNWRiMThkM2M5MjE2MWY2NzBmMDhhZjk4ZmVkN2FmYmRmMDQ5ZmI4NzNmMzNkMmE5YWI0N2U3NzhkMmMxYzYzZTViMTZjZTk5YjhmYzI3MTYxYTZkZjg2YWRkYjcwYzEwMjczYTNlNDc0OWNkZmEyYjI3MWRjNzA5ODQ0ZjJmOGJmNzMyZTNmYzViYTZjZWRlN2MzYzZiOTAxYmQ3YTJjODQ3OGQyMGIyNzdiZDY1ODMxNGE4MWVjYWMwMzM1NTUwODAzNGQ5ZWIwZmM2NTc4NDk4NTYwZjYyOGNjODcwMzZmYzc0YjM2ZDQyOWI1OWJjZjc0YmFkYzQwNDhmN2E1YTFjNDgwOWU2ODNmMTk3NGMwYjBhODc3MDU2YTc0YjM0MjFlNWFjZDY2NjU3ZjUwNzdkMjY5MzgwMGUzZWYzY2MxZTVhMzU5NGNhYmI0NzFlMDY5Njk2NGQxZDdkMjFlMjIwZDViMjlhNjEzZDYxNWIxZTcyNTNmZDcxY2Y0OGEwMjg1YWI4YTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.20qR5RG8HSdKwyfz53SiAcTnVfUK_z2LgfvO8NaW4ksJL4ZFuJfZN1dNTJKp3a1JzLyw7UvJjT3BJF4r005PtA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112633_55_2413_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.685Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikh4RUdScmZPc29BUGJhZUo2N1FQQkpWYVZWbjRVRUtCdll4RXlxTkJ6TGhDcmZBQWRpMTZnc0pOcEJhTmQrdzV5Y2pSZTNHVDY5Y1IvMjF5OFpra2d3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzNfNTVfMjQxM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OThiNDgzYzY2NGZiYmMzNzZjOWRiODFiZmE1YmE1YWIzNzhiMTllMzdhOGRhYmJmYWQxZmI0NDgyMGU4OTJiNDhhODUxNGM0N2RiZTMyMWM4MmM3NTRiY2JhZTljZDZkYzNjNzVjOTdlZDYzNWJjMzQwNzc2YzY5YTJjZWI1NTUxODg4ZjZmZjIxOTQ2MTAxOWVkYTllNmQyNzdjZGIwNDQzZTIwMmZhY2EzZWUyNTM4YTgzNjUzMGIyYjg2OWY3YWQ5ZTBiMGVjYmUyZmY2MGE1ZDNmOGRjNWU2M2E5YTljMzAyOGU5OTY0NWI1MmIwMjkzMzMxN2Y3NWFiMWI3ODAwODdjOTJlNDhhYzE5ZDQwOWNkNzQ0YWYxYzg1NjcyYjFkMDgxZmEzNmMxNDdiYzMzYjIxZmIxMWQzM2YzMmJmYzljYjMwYzJkZGZlODJmZjc4ZmQ2NTc0NDBkZmNhMDUyYWMzM2FjMTNjMTgyZmNkMmIyNjFjZDM2MzVmNjIxMjczMjAzNjc2OGI2ZTQxZTA4M2Q4YTg1NDZhZWMxNTYwMmJlMmI4ZmFjYWYzN2RiYjNkOGNiYTA3ODMxYjBhNDdhNDE5ZDRjNWIwMzA3M2M4MGEwOTQ0YThiMzExNjVmZmIxNTE1NDE1MjBiODdkYTI3Mzc2Zjc2OTVlMzgxYzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ncO_w2Zxh6ynCQpbqvEfFQAoO9peQhgzcORCpO7XB8MFwrtISD1TCEzTZN_W0nnEyS82AMOFRJTLF8aVJYeq_g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112633_55_2413_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.687Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im1KUWJUR09FVFZVaXd6dmg3UzUyL21pSW9IUzgzRkF3Qkt6SlBFTDlyQUhpOWlUUkthS3NVS1lzd2d4TUJ6RXltOGswSHBub1Rtd1dFR2QvQlltWURnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzhfODhfMjQyYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjZkOTE0ZGMxOGI3Nzk0Y2M0NTZhNzcxMWRiMDVlNTQ2YjdjMmRjMWMzZjI1N2I0ZjgxMmU0ZWJkNGMyNzc1N2U5MjEyMDllZTlhMzdkMjQ2NjFhYjQ3YTA4NzhiZjM3ZWViYTUyOGQ2MWJkN2FiYWY4ZjAxMWQwM2QzMjc5ZGJkNzYxMzg5MTdhNzZmNTM1ZTczYzQxNzA0NjFlMmNiMTdhOTU1ZjUyNGYzMjg0ZWY3MDkzYTE4MzdlZjJiODI4MzY0MzQzOWExZjMwNmIxODM3OGE5Y2ZiMmRhNmNiNTAyYWYxMWY5MjY3OTFkNWU0NjEwMTZkOTk2NGI1OWVmNmZkYjJkZGNhY2Q4NmNmZTVkZmNjZjkzYTkyMDNjYTE5ZTAyZDIzNmViYTlhNWI4ZGQ0MzM0YzY4ODQxM2EwNDg5YmZlOWQ2YzcyMzRjM2U3ZTdlNjJjM2ZiODBmMTVhYjVmNzgzZGM3N2NlNjBiNGM1NTczMzRkMjIzZTJhMzg1YzE2YjhjNDUyMTIzMmU5NDVmYzU3ZTRmOTdmZjVlNzM0Y2UzNGFhOTMwNTU5MWMzOGExYjQxYjI3MWY5MDIzODc3YTVlMmZmMmFkNGMxZTE2OTg1ZDFkZTdhNGQ0ZTcxNGE1Njc2NjMwZjM3NzJlOGYwOGRlMTM4ZTRmNzg3ZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UhTHuDSTpu4RpUGOiW94rtS9E1o_HhHrkhxy9rJoyanM8zHMJazXFct-8nahd5d8k3w6zchjAUmBceIDaubMNA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103838_88_242a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.691Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InlOaVJjZ0R1K0pzTUdUMldjeUpHWUpNZUl6bGVSSzNDVHplcEJpSXl2RVlsbUladE1MODVjejdPcE4wTXdnczgrUno0KzhPODhER3dHV3JXZ3lvck9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzhfODhfMjQyYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTJkZWM1NGM5NDFhZTlkNzI2MGY2ODcwMGFjZDJkNzU0Mzg0MmZhNjE2NzI4ZmFhNGYxNWRmNGY3MzkxN2FkOGFhZTBkMTVmZDUwYjNmZjdhZjM2ZWE1NTFiNDk0OGI3NzYxNmRkMDVjNDgyN2ZmMDdmMzJmZjVlYjFkNGE1YmIwOGY4OTI1NTlmZjUzMzk2MWQ4ODg0NDg5ODQxNmRlZTVlMGY1Y2IxM2Q0ZWFlNTAxMDhhMzdhMzA1YWFlNWNlZGMwMjIzZTk2YTAwODEzMWY2YmFlOTdiNWZiZjYzMjE3YWEwYTg2MWNlOTNmY2ViYmQyMTllZjQwM2ViMjk2ZTM4OTEzZjFhODRhYzVlNDM1NGQ1YjBlMmEyNjdmOTIyZmMzMjhhYzgzMDBhMmU2NzM4NmZjNTg1ZDRmMDJlODM5YjdhNGFiODg3ODVmYmI1NzFlYzEyMDEwMzM0ZWUyNWFmZDY4NzhiZTljMzYyMTczMTQyY2ZiNWUxYWZiYTNjOWJjYzA2ZjEwZWNmNWI3N2EzNjBmNDc0ZmJjMGUyNmE3ZTZjMDI2YmJjZmY4ZTMyZDdlZmQzOTA1OTQxMjEyNDEwMGZhZjg4ZTFmMzk4MmY3NTQ0ZDgwYzgyMmFmYjhmODc2ZWM0OTczNmM5ZjlhODU4ZjY3MDdmZDE5ZDliMzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.w-mrgEd8yK7ofU94Cg5YUxaBhcZFUTke8dHEOV3-gpvXapZFd394fkDpLHBgrLO28dO3cGn1r4aUYfKiyil4gg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103838_88_242a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.693Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ijk1eTdTaTcrQnR4REkybTNHZmdVT1A4ZmU0Vmh2NEY4NlUyRG5TU2lva0FrTTV3M0FmWkRvSHJ4cGNFdVRXc2d0a2pOK1IzUnBwcFMvVWJ6Z1N3T2tRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzhfODhfMjQyYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjAyYzk4YTNkMTA2YWNmYmQ4NjI1OTYzZWIxZGQzZGRmNjEyM2NhOTAwOGYwMzVjMmU4ZmExMGMyZDZhNmU4NjA5N2UzNjFmNDlmOTY4MTFiN2NhMzBjMzg5ZjVkZjRlMGZmMjYyZTkwNjc3MWFhMTgyMDVjN2E0ZDcxZDZiOTJlNDNiNmQ3NjZkNWMyZjJlYmRmMTI1YzI5ZGU5ZWFhNmQ4NzliYmVlOWU2ZmExYTA2YWVlNmEyMDU4NjViOTAwNzg4MDI3ODliNDQ4NjcxMDM2NDk4ZWZhNjU5MzRjNDcxMTU4ZDE4NzczMTYxMjY4NmFhMTlkMzBmZDZjMTY0YTM2NDFmZWViNmJkNzcyOWZlZjcxZTNjNzU3N2FkNTY4ZWQ5YmU4MDdiODhjOTdjMzQ2OGJjM2E0ZmM5OWJkMDU5NThmODMxMTZjYjAxZmMzYTE2MzQ2MzJhMGJlZjFkZjAwYzE4ZWQyZmNlODE5YWU1ZjFlMzdjYmYwMjVmODNkNjU5ZGViNjNlODA3NWVkODMyMWY5MTQxMTk1MTBlODE4NTM0OTQ3NTQzYWIxYmM5NTE1OTI2ZDAzYWYxOGEyNGQ1YTJhMjc5ODY2NjYxZGNmYzNlNmFiZDRhM2Y0OTdkYWZiNDA4ZmRjOTg2MTIxNDc3NzBhNGY4MDg0MDYzODFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zv89_jjQcxOIlrIAgvGDXc1vgvJ0oUSrZdVivi658-lFFbdLKx35icLG5GAhO1w0RdBKjTS9CKRG_30GsXOA2g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103838_88_242a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.705Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkRXUHZVWWNPY2RqZlkrdG1KRFZCTk5LYkl1N3ZhMTBTMkxiL2pTOVhmSjJEQzJ3SlF1aDgrZWRzTU5TWGdUWkhDeEp0UG1Ra3ZnOWVoS1NmZGhOamJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzhfODhfMjQyYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2YwOWIyNWIxNmZjMDgwZDA2NDUzMTdhZDBmMDIzMTgwMGIyNTVhY2M4YTI3N2MzYTcxNTI5ODY3OTAxZDkxZmVmZTcwMjY5Y2YwNmRiMmEyOTlkYmRiMGZhNDMwOTY0ZGFmZDZmN2U3MGNkOGExY2RjNTU2NzI0MjdhYjY0YWZjMGM5MGM1YTliMzJhYzEyMmRmZDQ5OTJiMTU2ZGY4MWU3MWM5YTU4NjJkZjQ3OTFmNTZjMzA1YmYzZDNlNDE0YTAxYjg4ZjUyY2JiYzFiMGE2MGFiMjExYWU3NGZkNTI5ZDlhNWE1ZjAyMmI2ZmZjY2U4NDBjYmYwN2JkNGI4M2I1YmEyOTk0MTRhOTQ0YzJkM2EzOTk3MjVjZmUyMWI4ZmQ4ZTI1NGJjNGY3YTY0MzYyYjBiYThlMDNiYWVjZjczZDNkODVhNWQ2MjI5ZTYwN2FmOGNjMmFmNTcxZGQxZWFkZDczMzUxZjU4Y2RiZGEzYTgyNjZhMDcwMmQ0ZWQwZjBlMTk4Y2U0ODc5ZjlhODYyYzY4ZjJkZTZmMGY2YmY3MThkMWEyZGM3MWI4NDEyMTdiM2I1YjhjYmRlMGEzOGM0N2I1NjRkNjhiY2RhYjdkYTQ1NzcyNGNkZjgxMDVhMjY5Zjc4ZmI5Zjg5MTdhYmM3OGRhMDYzOGViZjJkN2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2XPXPywgm2bVl18TO9aFviycFxyI0Z_HHHDvOUBJrp4REP56z9WUcCr2yxYBNzGwSQEWeoYjrC6F45UF0-bETA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103838_88_242a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.708Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjlsT21hbkJ6U1hja0RrbUw0RmdNQ2xZLzJoWXlsaFZZQWZ6aVJHRGZqdVRaT1BUS2NsRlpGZU95ZFRZWU15NFFtY212SHJIeWFFUHNmeHdub0NENUlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwM18xMDM0MjZfMjlfMjRjNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVkMjA1MDZlMTM5YTg2YmQ0MzA0MTg4MDk0Mzg5NmMwYjU1ZDMyZjgyZTZhMjA2ZDQ1MWYxNmI5ZDA5YTc4YThmYWM2NTgwYmFmZTBjMDRkMmFlNjI4NWE0YWIwYjI0NjIyZTM3ZmY2NDZiNjAyYTM1MTRlZDA0OWVkODE4OTNkZmFmYzY0MTM2NzExYTE4YmQ0OGFjN2Q5NDhiMWM0MmE2Yjg1ZmViZGE0NTc2OTViYTg5MzVjNTEzODNkODRkM2ZmNGUyOTNhOWM0YWViZTQ0ZDFlMDQwOTc3NWI1NGQ3MmNiMWU4Mjc5OGVlMzAyN2NiYjIwZGUyMGQyZmI4ZjJhMmM0ZmU2MzEzZjg0YmEwMzQzMDgxMWI1NmE4YWQ3YTliOTEyNTA0ZTE5MjZkNjhlNjU5MzNmYTBkM2E1ODZkZGQ3OTFhMzFhZDEzN2YzMWQ0NzVjN2M2ZmY5MGYzMmRlN2Q5NTE0ZDJlYWJjODJjMjI5OWU0N2Y1YmNkMTQ5ZTIyYTgzZjY5MjhiNjU0MjIxNjkyMDJmZTM1NGRlZDRjMzUzZDI5Y2I0ZWRkMWY4NzZhNDk0MTcwZThjZmVhZTA3MzQ5YTY3N2JiNmE2OGVkMTExNzMzZWI4NzM1MTdjNDIyM2E1ZDU5NWU5N2ExM2JjM2M4MmEzOTZkMWUzZjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8Jr1KYGxC7FgRGyEyTgtypRiDTS-qZeqdCc3ojikT49tO3JR4NGC_bAYyQcSiTBDF1rLANV9JuGZPua3qQRsbg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240203_103426_29_24c4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.711Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InIzQ2FPZ21xTlFlZTNTaDFJamkvTkcrVzdjMWtvSjE5OEVNbHFuOUd0azZoaWhUSmpJMldVc0dqNzJUQkd3WUc3NitCOUZFL09iSSs3aFB6Uy9KeFhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwM18xMDM0MjZfMjlfMjRjNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzFhYjMxMTU0ZDBiYzMxODZjNGI1ZGExZmQwZTQ1NWRlOTI3N2U2MGJhOTQ2ZWJmMTY2ZDhjNGNjYTRlNWM3YjMzMjJjYTUzY2MxZjMwMjYyNTczNWQ0OGU0MzI1MDNiMzM0NzNmMzE0NGYzNjU2OWFiNWMyNGM1YTFkYmQ5MjhiNDE1YmIzZmJmYTI5NWJiZDhhMGQ4N2E5NGJmYWYzYTI0OGUyOTQ5ZGIxMTZlYWU0MzM2ZmNlMmIwYTkwNTQyYmY4OTY4ZDFlYjg4YjgzZTFlYTNiMGU0MmMxZDg4YjU2MzhjMzQzYzQ1ZDg4YTVlNjA4YjBkZGI2Y2ZjMTExYmRlZGU4YmI0YmU1Y2E2ZjFkYzBkZWM1NTc1NzJkZWYyNjE2YmE0MTUwMGJjYzc0MDcxODkzZDkwZWY1OWU1NGI5NmUwYjczYTM4Y2E4MzJmZWFkNzJjMzdhYzViMTM1YjZjNDM3MTgwZGZhYTkxN2E0Y2U1MTRhNDcxYTBjYTVlNGU3YWQyNDBkZmU4ZTk0OWIxODMxNzRkZGQ2YTgxMzBhMDdiYjc1ZWRjMDBiNDYxM2YzMWNhMzg5YjM3OTk1YTVkM2NkMmYyYTA4ZjVjYzg5ZmMzNzAyMzQ5ZmJhMjViM2ZkNmIyMzg5M2RmZDQzMDQxOTMzOTA4ODkyMjJiYjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7afaXRBYZpBTcd1Ds3d05SdtRGGFl-ZDyVCEPMwICj656ctDw4RoMNdQakmrfog7CdgV-d4o3Xry4kfqTbIXzg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240203_103426_29_24c4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.714Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlM2N1NBSkNTM3BVdUZJTzFOWUFRZDRaVUhUVTFwZVo0YWliWlFWYUNpVGhIdGtxVDR0RjZYSUYrVUFlK25aejRnTjJNZkFqVXlLWndmcGUydzNmbWRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwM18xMDM0MjZfMjlfMjRjNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGU0NDZlZjY0MmFiODMyZmU2NGE0ZTliM2JlNGEwYjYxOTczYjJmYTEyZDgzNTg0MzRmYWNkNjJlODhiZTcxYzJlZWFmNzJiMTA0YTY4NDkwZjUwOTM1MWEzMDQ1ODcwNjBhOTExMWZmZWQ4MGJkMTgzY2NjODM5ZTg3MWFjNWY5MjcyZjVlOTFmMTEzMDI3ZDExYTFlZmRlZDNjMmEyN2RiYzBkZjJhNDIxNTRlM2ZlOTE0OWFjNzA0MWU3OTZlYzdhZjYzMGU5Nzc5MDdkODIwMzg1MTVmNzI1MGUyZDE3ZjU2N2ExNDk0MjJiYTVmZTZlMDNhZjhmODE3ZDUyODEwZTIyN2Q4YjY5MmQ5YTEyZmJiNDcwZDFhMjYxYWVjNTU1NWMxZTE4YjY5ODEyYzFlODYzZGFjOTA1M2JlMGNkNzBhMTkwMWFhZjc5YmIwMDRlYWIwNmFmYzc5NmQ2ZGE1YjYxOTdhOTdhOGUyYWJhZjU3ZDMwN2QxMzIxMmRiMTY2ZTE4M2NmMTIxNDYzMDQwZWQ2NzZhZDY2YzM5MmVkMThiMWU2ZDExMDA0NWUyZDFlOTQ0ZmUzMTg4NzM2ZGIxNjI0MzkyNTg3NzM5MWZiZjUxZjM1MTBlZDY5OTFjMDJjZjRhNzcwNWNjN2Q0OGNlNDI3MTQ3NjA4ZTQ2MjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1RWLDZKllqYGVPZCvIEn9_ywKhRgMGXzfxCuPT4701SVTEXwHT9TYZM0XCmtpcEazNBE-7OlTWAWClJuuTe4Sg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240203_103426_29_24c4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.720Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjRwQzVkenNCRERpVVJnR1Q5UDZ0YkVOZUc1d25GMlNJczNEbkxPWEl5WWlMY1lCSW81U1AwRmFmUDNIQjhzWFBlSTJ4QTU2VldlcmE3Y3M4TlJmMmJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwM18xMDM0MjZfMjlfMjRjNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTQ5ZTdhNjJmOWM1NzU1MTUzODBlMjE3YmFkOGYzYTA5NzVmZWRhNGIzZGYzOTBiOTA4ZmQ2NzAzNjE2NDk3ZjU5NzFjMmYyNmQzYzMwYjI5MzU1ZWYwNmRiYjU4ZDgzZWNhMDAxYmQyZTUyOTQ2ZDIxMjE1MWQwMWZjYTdmNTA0YjU0ZGMzNzkyNTc5NGI3YWZlYjFlNmM3ZmYyNTg2NWNiYTY2ZDA5OTUzNzQwYjYyYTJkNWQ3MTg4MmJiODZlZmVkNzIxNGZmMmYyMTc3ODJhODkwNWNiYzlhYzk4NjgwZTU4NjEzZTYyMWJiNWY5ZTYxMmYyYTM2YjZiYTJlZjUyYzU0ZmRjZjhkZjJmNWE0YmRlMzk0MGI4ZGI1N2MwZDkxOTJmZTAyYjA1ODMzNTEwMjc0MzgyZmZmYWVjNmFhYzgwN2MzNDVmM2E2M2I2YTIzNjA3OGJlMDk0YTBhZGYwMTg0ZGE3NjE5YTk2ZjRjNjljMzhkMGUyY2E0NjQ5M2Y0Nzc0ODc5YTc2MjA4Zjg3MzY2MmViMmM4NzQ4MTIyYWZiNGE5NTg4ZjE3ZGZjZmJiOWE2YTdiNDdjOGE4NDY2NjcxMTVlNDE4MjZlNzI1ODI5MzI5ZWFkYmFhZDRhNWE2NjM5ODI0M2FhMDYwNTMzODFhZjk2MDkyZjlkYjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ClVw-svLSNDb6GmImwNymHVSeW_Xb2qnoNiYDudna7fULXaA3xzTwkNfAa3V8SuiIAJMs9BL3equ6qBjbf-q6w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240203_103426_29_24c4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.722Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJYMkRreFJhWFJxR1VmZUgzWnFqNFEyWE1zbzU5OWsxWlErWHRQckFRZG9seVVGdS9KdFNuVjI3bmpWekwzODNKd0oxZW9TSTJxcUtBalpoRmI0cHNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjRfMzFfMjQyM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGIxNDcwOTBmNmI1ZTkzMTA3NzNiMmFlZTk5M2IxZmVmZDI4YWJjMDM2YWMyZGIxYjljYThjNjgyZjMwNjYyZTY2YjIwNGE1MmVhNWVlNGRhYjNiMjQ3OThhYTNiZDhhZGIyNjlhNjNkZmRlYzQ4NDJhNzIyZjVmNzM0YWMwZTU2MDNiNWQzZDgwZjYxODAwYTlhODIwN2ZkYTc2OGE5ZjY2ZGU4NTcwMzU4NmE2ZjBjYmQ3Mjg4Mzc5ZDQ2NDY3YWI1MDAxNzJjMTU4ZjM4YjAzOTkyOGJiNDc3NmI3NmIxZTEzODliMWNjZTFhNmUwYjcyN2QzMzIxODQ3MDc0M2IxMjgzYTNkMDgzMTM1MDgyZjhjYmQ2NDY4NmZiZWRkZjNkYzY1OGQzZWFhYWQwN2IwMjdkMGRmY2JhMTljYmJhMDNiZGExMTIxOGM0NGExZmM1ZmUzOWNhYjAzMDA3OTEzOTVhZGExY2RkNDlhYzY1MGZlNmU0YzA1NzNiMzI3ZDg5MGJhZGUyMTljYzBiMGJhNTI2YzUyN2Y1MzAzYTUwYTRlZGQ0OTc0NjQ0MDU2NjE5MTU1M2ExYzBlNzE5YTFiNTVhODhkYjg2MTY1MGQ1MjQ5MzMzOTJlNDVkYjFjZTcwNTM3NmNmZTkyZTE3NDE2YmE4ZGUzN2I0YjFkMDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YZkf1LnYgODk3Bi2BQByAKODdUfJxKUhiYVZJjgcK6W6UlB6pAgAkeRt1nqsuaCq9s4m4kfeMIN2gQYeuo_xjA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102924_31_2423_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.726Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImFVSk5tNTZFODNoK0NWTmhSSUdCak0zRWdlZHBiS0VVbUFvckwwdG5QUUNCSGkydkFhbHA1WHYvcm9TN0V1RVRJTWMzWHBvc3lwZCs5M3lZdHlVdGlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjRfMzFfMjQyM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDM5NjFlZjU5OTczNDE5N2EyYjY4Y2MxMzU5YWJjYzgzNWVlYmQ1OTI2OGQ2ZDZmOWZhMWViMjMyMDY1ZTRlN2Y1YWI5YTZmNWViNjdhN2U4ZjhmNWQzMjc1N2QxNjFjN2RmOWY1ZDljN2U5ZWJjNjE5MmZiMmRmNGM4ODlmMmQxZTBkNGYxMTYzNTJlNGNlYzkwMDdhODI3OWY0MDYxOTc2YjE1MjgyM2FjN2I4NzYyMWY3ZDBhZmQ3YzViNWI2YjAwZDI2MGFhMjVlZWUwYjU1OGRjZGMzOTE3MTBlZmQzOTI0MzUzZGY0MzE0NDU3ZWQ3M2NjOWFlOTM0ZTczM2FlNTdiMGJmYTQ1ODIyOGM2MmM0ZGNiMDYwZDc5ZjMwNGNiMWZlYmQyZDVkZmQ4MzVjNjQ2MWU1MzA3ZDJhZWNjNTM4Yjg3Y2Y3YmZjMGMyN2RmNjA3N2ZjZmU5OTdhMDg1ODRmNDRlYjk3MTczOTQxMDZhYjEzYzc1ZmFkMGNmOWRhNTMwZGMyODkyMTdmN2VkM2VhOWVjNmY3N2VjMjNmNmE2YTgwZjQ0ODU2ZGNkYmE5NGVlODI5YmY5M2EyOWU1ODRiMjc4NTZkM2QwNmI1YTUxNTFjN2I4NzNhMzk2MGRjODJkMWI5YTU0OWY3YmMwZTdjZGM2YzUyZGM4OGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ePhzkxNNK-1jy5Aocr5-AHmhJhU4WvEwDej8HW4INpnRK5zT6jDZSQBA1yDkO1RHmYG68R4s9UFSQ6JJC1-Hlg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102924_31_2423_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.729Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjlQcmd6blhXUnY3ejJIWVNrSVhsU21VZkJPN2hqdXU3RDBpL1o3OEc2Qjd4TTFxS1N0N3VvVlJyWGhxRE1ON2J5eDRRRjg3UWhHcDlvVE50eXV2QVJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjRfMzFfMjQyM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWUyODAzNmE0YjU4Yzg4MDJhMTNhZDU2MGRjNDBhYzdjNGIzYzliZmE0NDQxZjM5OWY4ODFkYjMxNjY3N2Y3YTJkMWU1OThkOWM2Y2I2NWRhNzVmOGRhNzNjM2FkYWY3YzRiNmU4YTE5Yzc3ZGY5MmJmMTc0NzQ2ZDg3NjYzMjc5ZWYzODkzZjUwZTU4ZGFlMjExMTU3MGJhZTIxN2MyNTkyZDM1YmY5MzZjMDliMzQwNDMyMGFmZjc1NjMxOTM1YjdjZjMwNmY2MzM5NGZkN2I4Nzg2NTNmNGVmZTE4MTQxNWFkNGJmODlmYjkxZDU0Njk1MTFjMzlmMmI5OTdjMTUzMTI1Y2FhZDFkZDViMGU1NjI5ZThmNTkwYzJmZDJlNDhlYzcxNmQwZjQwNDRkNTY1MDY0MTk2MDhiOTRkMjZiMGNlNDdhZGM1OGY2YjEyY2Q3ZDFlNDk5ZmQzZTg3NDNlODNkNWExODJiOWE5NzA5NWZjNzczNjdkZTJmOTQxMjYwMTljMDg5NzQ5ZGM1NTIwN2I5MjgxZjQ3MTg0YWEyYmNhZDliZDhlN2E0YjhkMWM5YWE4NWFmMmQzYWMwYTE3ZTEyNzUxODQ2YTFlMzY4YzNlZjUzMGU2MDgzMDdhZGEzYTgzYTc5NzhlMGQxMjI1OGJhODVlNTk1ZjczNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XTxMObd55NYYSbli7oewBcQEPzIH3lkz3OQ0uFb30GpsdEtQ6cgP-lcpx8hns37nYbAT3L0xriaOh74tsDk5oA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102924_31_2423_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.733Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InhsNlhoc1B4UkZxU2M0M2tvaWVKUVRnUjRqK2ZoanVZMWdmY1ZFdGxLNFFNTmhiNWFEMnY1Nlp3djBwR2ptbktEaDdacWZHcnhkTWpUNlNRMHl0ajFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYyMF8xMDI5MjRfMzFfMjQyM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmZmZWUzYzJhOWIxMGNkYjg5MTc1ODYwZDdhNmUxNzVkNWEyOGQwNzQ5ZWNlN2QzMDVkZDI3MjgzZjQ2NzMyZjUyMzZlM2Q1OWM5OTNjNDE5NmIyNjQ5M2E3ODg0NDI2NDg0ZWFmNzYwODZlYzYxOWJmNzU0YjllNDg0MjIyMzA3YjNmN2Q5ZDU3MzY0MGRiMzg3YjdmM2RlNDNiMWQ4NTMyOTAwZDZkZjRjYTFmMDhmZjIwNWRjYzhlZjI0YWE1ZjQ1MjAzN2MyZGJiYWUxMGVmNDJlZWE1ODM3Zjc5MTk5ODFlZjIyNmRiZDZiMDE0OGMxNjM2M2Q2MzA0MTg2MzAzMDkwOGE1ZGUwY2QxZTViOGI5MDI3MjcwNTI3ZmJhZjI4MGI1ZjMxODUxYjI4N2U4MzNmNDVhMjU0MjM1NzE2ZjhjY2VmYWQ2Mjg3ZGU1NTU4OTYxYmE5NjFkZmYwNjVlOGY5YmY2MGViZDYzNTU0OTRmZWMwM2FjYmM2NzE1YmFmNzM3Yjk3OWU3MzhlM2UyYzQwNmU5MzdiY2I4MzA1NGViZWUzMDAyMWIzYzM3ZWQ4OTVhMGM4OWIyMzI4ZjY3NjhmMWQ1YmVlNmEyMDlhMDEzOTU2ZTdjYTFkZDdjNzMyMmVlMWM2Y2ZiNjU0MzIzZWRhZTRmOGZhODZjODRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.X1jEHyxkAPRmXJuwDZX-lV5vuPHRd9BigpNrkAxMoBb76H5ZQq5J3thH7YYtkl2GF4fqo4ZsZ9PRgBM_4MHTEA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220620_102924_31_2423_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.737Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjdxY2t5WHYxUVFENlhuTktTNVpZbGlCTjRaQTZQSTIrUmczd2wrS0tiMk9mNFc0ME9XVkdMcDdEMlZJVm5ZM08rcjZucklnTS80ZmU2cFN1WUsrY2h3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDVfMTNfMjQ4Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWFmNjYxNzY5NTdlNDRhNDMzYzMwMDFmMzA2YTY3NmE1NGNjYjhjYmE5MmYxNjdhNTM5YzM2YTg0YTUyMGE0NzgwZmU4MmY5YTY0NDczY2UxYWZjOGFjOTg1NWY4NDQ5YTdhNWM3ODJlZDRiOWY1MzFiNDc4NTkyY2JjNDIzZGFhOWZkM2RlYjE3ZTg1MTUwMzUwMzIzYjNkMDI5OTJiODA0YTQ2ODlmOWU0Zjk2OTUzNjQ1OTQzNWUyY2YyZTU4MDBhMDRhM2E3NzIyZTA1YmQ2MTcwODcxYWUyNWE4ZGZiZTllYmQ0NDNjZDVhNDY3NjU0Y2VmYTI3NTUzYzQzNzc1MzEwNjliYzI1NWIwMjA0MjRiZDE3MzA0NmNlMDY5MzBiMGUyZGUyZWYyMThhZGM1MDkwMmI3N2M1NTNlMzZlM2Y3NjFlMTFjNzU5YzdlNWJmNDcxNWQ3NGI4OTcyMWQ1NGVhMWRhMjU3NjU4MmU2M2U4NGZiNTA0NGI0M2QzYzgxOGZlYWEzOTM5ZmVhZjNhMWMzODJiZmQ1ZjE4ZDRiZmQzZjI2ZTQ5OGRkNDI1ZmVkYjk2YzRiOGI0NTQ1ZGRkMDk5ZDhlMmJlMWM0NTFjM2FhNDU3YzZkOTE5MTdiZjJkMDIwYTczM2YyMTgxN2I0YjRmYmU5NTg5MGQ1NmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5dpoTefCa4snTBfxi1Uq9qiokMStXlqFNE_WvIwMOmTyTxO7wwp78ww8b5rH_XKaZ_xP2FJDJgTOvTlhi0bloA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110505_13_248f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.740Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImtSZnExeVl3cmRCNTlxV21pMGZDRzNmamM1d2pqL0VaTXk2Q1M1SmEzSzhwYnd4L0daU2lFQU1lM010MWd4QmlEb2NCb1RWUitVZUtJNUJqM3B4aDZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDVfMTNfMjQ4Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjE3OGM4MTBjY2U1NzdjYWQ1NWI5ZWI0ZWZhOTlhNzYxMmY1NTE3NTNjNzI2MjdjMmQxN2E0NDkxNGU2ZWEyYTc1OWY0OWY3YTcxYzU4YjRmNDg4ZjlhNGFhNTU1ZGY0M2NhMTkxMDQ3M2IwYjk5YjJmZTljYmQ5NzhiOTdlNTAxZDQwMTAyMDIzYjU2OTUwNDY3OGIxMGMyMGZmZWJlYWYyMjI3YTBjYzJlNGMxN2QwYzg2MDFhMjg1MTBjMzNlZTQ3OGE1Y2I1OTk3ZmQ2MzgzYmZiM2RkNGMyYzBjNmFjNGE3YTVkYTgwZTk2ZDY4MGM2MGEyMmFiNDRjMDdiYjg5YWI5NGRhOTVmMmNhNjJlYTk4YzY2YjVkYjM2Yjc2MDQwZjRlZGIzZThmNDc2NmViM2RkNzZhYjMyODBhNzE4MjgyZTE2M2NjMTgyNDBiMmIyMjM4MjY3YmE2YzNiMGEwOWEyMDNiYmIxNzJkYjEyM2JhZDUzOTNjNjlkMzQ2ODRhODRiMzBiZDNkMDgxM2NjYjllZDVhYWU5YzY2NzU3Mjc1YjM1MzI2MzA3NWVjN2Y0NGVkYzUyYmZlMGQ2ZWRiOGFhZDBmNzQwYWI3NGJiODAzMjUyNjg5NTI4ZTIyZDNjNDBmOTRiY2JjZDFiZDY5M2NiM2YzYjFmZDQ0NjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jOYYr2QXVt7kOgzni51XWxPjqvQfXxrsUmpfTx9E_M1QXZdP1W5PUlHIxaDnZCmvbID40U9_5hCxz8on3lgG5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110505_13_248f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.744Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IktKL0NGUVZpQzNMcnFEZTBkMEtLc1lpTU9VdTBvSnErNjNrWkVYL3hvclJneHNsSEdyMnE4aHZyVFYwSE94L1JiLzBaMUMzYnB1MkJkWVJsR0FJTTRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDVfMTNfMjQ4Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjA5Y2NhYTg4Y2ZhZjUzMWZhOGRiNGRjNmQ2YzlkMjE3MmNjOGJmM2UxM2JjY2ZjYTZlZjk3ZmIzYWIzNjI5M2MwZTQ3YTJlZjBjNTZkNWJmN2M2MDMzNDU1YTcwOGUyMGY3YTQ2MjBlYTgwODU1Zjc3ZTI0ODY5YzNlMjU2NTA3YTM1M2I4ZjI0OTFkOWVjMDg2NTM5ZmExNWJhNzEzZjEyYmQ3ZDI4Mjc1ZWI1MjVlMjliNWUxMGZkOTExMDljY2ExMjAzZTg0YmRhYmE3MzJjMzFjNzUzMzM0ZmIyZTk5ZTJkMjY2M2EyODcxZTJiNjFmNDAxNTJkY2FlYzAwNGY1YjdkMWJhNTEwODBiOWZkMGFlYTM3YzE1NGE1NDI0NTJlYjRiNmE0NjBlNTU0MzY3NGQ2MjY4MDhhMWUyZmY5MDMzYTE0Njg4MmUwZmM1YTYzZjk3NzJjY2E4Y2I1MTgxMTA5NTQzNGI4NGQ4ZjU3NmM0NGE2MWRlY2NkOWU5MjA3MDk2ZDMyY2E4ZjFmOTMzYzU0MTIwOTBkMzMwMmJkM2Y0ZGQ2N2ExMTY4MWQyYzlhYzZmZjU0NmY2OTllNGI5ZjM2YTI5ODFmMTYzMGIwYjg3YTM4YTk5YjU1ZmJkYmZiOWZlYTBkYTY5MTU3ZTgwZWFlOWI4NWRlMjllMDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Qi_R2aw8PwVAWgWAilK39I6uzvIeR6JvyqDgg9G3-TIjSRPAKwHszRQppr7K7wSF9gFbDTgb3Gso2Vnh-FWHcA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110505_13_248f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.747Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Imd5QTlIU1FkWFdpK2pmNlVWMFV2NDdOa3hTZmJkeXA1aGRxRkJWaUFyalhRVkRqNVdhd3c5aHAxT3RwWDl0RVlDNGhuRHRYaVIvU2QycWpGWjB2QzdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDVfMTNfMjQ4Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDQxY2M4NzIzMTlkYzJlZTIzODBkOTgwZDI0NzZhZTdjYzBmOTJkNjM3MDlhZTMyMDQ0ODUwNmM2ZDRiOTllYTJiMDI0YzRhZjAzZTM0NTdiOTMyZWEwNWMyMjg3NTRmYjVhNGM1N2Y0OTUwZTY2OTNjZmJmNjJmNWY3ZGVkOGIxZGI0ZDczYjY0ODgzYmViZDQyMzM5OTRhODMyMThmN2VhZDNiYTQ4ZWM0ZjdkMzBlOGVmMjc1MTk2NWE5MzhkMWJiMzY0Yzg5NTAxMTdkY2Y4OGRiY2U3ZWZkMjE4NThjMGU2OWEyYzhiYzYwOWIzMjlmNGIxYzU3MDA3YThhYWJhMTdhYzVmOTMyMmY1NzJlZWFkNWYzMjYzYjNkZGY0OTQyNmIyYmZmODcyMDA5ODI4Y2QzODg5NzAzM2U1NTY4OTJjOWJhN2QzMTIzMzNjODJlYzA4NGRhNzcwYjdlMzgyMmY4ZGFjZGFjODdmN2U2ZGM4N2VhMzRjNWZiMmUzYTA5ZGI3ZDEyNGZkMDBiZTcxMmVjMzI3YzAzMDk2NjI1NzBiZTI5NjJhNmNlMjc5ZGQ2M2ViZjg0NGFmY2I3ZWYwMGYxMmQ2M2NjMTA2ZjIzNGEzOTE5YjljNzdjMWNkYWNiYjM5ODc2MjkxOWFhYzA2ZTBkOWI0YTdjNWE2ZDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._x-cANIiWOt68TeutJwc6fuXaeCy7daR4mfbmTWuO4v3a0aCuZm8sB8Kpuoj7YZfzSfn0_zZLnzkyCApzAqDrA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110505_13_248f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.750Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlRkNTNJeWlsZ3p0RFVZcVRMa2tFdExsTStLWTQ0MzlsVFZnUFEwdFRadXBKdEZpMW5nb1NGMU1oVi9ZQWIxVHJQWUVlbTN0b3JjWG5rSXAwRFRCdFhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxMF8xMDM2MzBfNzNfMjIwYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTlmY2E3NDYwYzhiNzFmMTIyNTExNzI5MDdjZWMxODc5NDM2OTA0Zjk0YmVjNGY5NGRhM2M1YzQ4NDIyNTllZTUzMzg4MDI1NDFlMGRiN2JhZDg1NjllNWYwZTEwZDQ3NTU0MjMyZWIyNmJlYmQyNjJiZjE1ODRmNzc4ZWQyOTAzMDlmYTYxZTAyNTY4MGIzZGNkMTU4MjhiNGU3ZWVhZWI2ODYwYjc0OGZkODNlZmVkMzY0MGM3YjZlNmMwOWEwNzQyNjU3NTk4NzViY2NjN2UzYmExZmNmNGNlOGRlNDlhNjRmOWQ3MDU1NGE0MjBiNGMwYjU4ZDA3Zjg4NDcwYmE0NzdhNzRkMGU4MWFlMGJmMDg5NDEzM2I1NDJjZTQwZjczNDk5OGEzZTJkNjk4MjcyNGYxMzM1Mjk1M2MwY2QwNmNiMmY4OWE5ZWUwYjJmOTgwNGY1YWQ5NTBlNjFjZTQ0MjMzYTkyOWUzNWZkNzcyYjk1ZmFhZGVlODNjZmVjMmU0NWJhYTk3MmE1MDAwMWFmYzVmMjJlY2YzYTdmMDgxMDIxNDkwNTIxNjE3NWM4MmI2ZDdjNzJmOWNkODQ0Njg4MjQ4ZGU5NDkwZGI0ZWI1YzBkNGFiMWQ5YTlmYjVhNWFjYmI0YmM2ZWQ5YzM3OGMxNzZmMjhmYWJiNjBhZDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.E5VEiCT2DXOcNUZm9ZO4D4n6YoaNi8hfpSCvZi852FyJUR4rOX0jbV0MKfvdwrWNF0E-EM4lDI8-7hz7k0SnoA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210810_103630_73_220b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.753Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkprWi80cWVFVXA5S3ViSXh3QkRXRmhBZWZESUp1Ymp1Z1N6RkJxaTRDcDhnWEUxeDQ4VkxOOTdBaHpvSFR3RmwrdnlVaG9SRzB2ei9QMjl0N1dGZVFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxMF8xMDM2MzBfNzNfMjIwYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTdjMDZkNzk0MWVjYWE1NGUwZjIyMGFiZGU0N2FmMDNkZDlmNzcwYjkxZDljMjNhNDg0ZjNiYTU0ZTk4NTdiMWUxM2MwYzIzYWE3YTRmMWQ0ZDRlOTg3YjQ4YzlkZTY5YTZlMzZlZTU3MDE4M2Y0ZGY3NDRkNGRjMWRlMTMzNTMxYTM3NDFjYjQ4MTFkM2YxY2Y2ZWRmMDcyODYwYzdhZTExMDU4ZGIxZTAyMTM2YWQ0ZTEzMjRkOTNmY2ZiMzQ0YWZiNjA5OThkN2ZjNDViNmU1NGY0YjkxNTU4NTY1YjJmMTQ2ZDA3M2RiYzdjMmI2NjE1NTk3MmUyN2NlZjllNDM2MjYwOTFkMzY3ZGJkMWUyNzVhYmRlYzA3N2YyZWM3YThmZGRhM2FkZjJkZDJmYTFlNjI5MThlMWZiZTI4YmQ1NzE4NDMzOTMwYjVlMDA3MTg2MTVkMWNkZDA1NDg0YzZiZjIxMDBiNzE2YjNiN2ZiYTczZjEyMTAzMzU0ZDM3MTUxY2QwMTdlOThlMGU3ZTE3MjFmYTBiNTVhZmVmMmJiNzMzYWUyMjBjMzc1ZTFmOWM0MDZkMDAxMGUyMThjZmQ4NTZkZDE2NzM5ZDBmZjc5ZTg2NGVkNjc4OGUwNTZlNGNlNjE1YTMyNjkwNjY3ODdhY2RhNmVhYTdjYjNlNWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sIWTtZ0RUTTyg0p9rLoJdy-QmKO5nWwDAozyjFAbkWLBUmgXXvoG9lVzdSwNjQH0PxqslmAJFWeA8iPy3Gg8PA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210810_103630_73_220b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.756Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkRmd2VIVTVEL3E2Nzh3WlJVQnBBY3h4REwzdWlJU09seW5PWFJ6K2RUZEs1Ums0b0o0Ukh6Q2ZjZE1SaEs3YmduQXQxNVlPRVhWd01rS1VHTks0L2lnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxMF8xMDM2MzBfNzNfMjIwYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDJhNGFjM2Q1Njg4YTkzZjg4MjE1OWMwYzE5M2E2Y2Q1ODRiYWU0MDYzZTZhYWUzNjdmOGVjYTM0YTBlYTM5Mjc1OTQxNDNjMjU5YjdlMjFhMjhiZWY4OTBlOWRkM2ExMjI5NDliNjliMDQxY2M1YzgxZmM3NTY1N2ZhZTVhMDA5NWYzMDA5ZjIwNjFkZGU5OTZjYjQ2ZGRmMDVkMjVhYjY5ODkxYThjYzM4Y2Y0MWQ1YzVhODg2ZTI3MjA5YjBjOTJiYjc3NGU1MmY1MGQxZmQxYzE1ZWMyMzM3ZTI3NTBlYTE2YjAxZTc3MzA2YWFlMTJlN2NmMDUyM2FlOTFhNzExNTI1OTI2M2E3ZWM3ZTBkYzgzYWI3YTI4MDIyMTZlNzUwMTYwOTIwMTY2NWMwYWFkOTBmNTgxOWM2MjRjNTE2MmIxMTVlN2M2MWRmMmVjNWY0YzQxZjU0OGY1MzNjNDkwMGIyYWUzMGVlNTc3YjQ3N2UyZTUxZDRjM2FiNDhjYTc4OTQxZGRlZjYxYTgxOWUzNTVkODdhMmFlNTE1OWQ1ZTQ3MTg4Mjg4ZTRjZmVkOGQwMWM5MjUxYzkxZTMwNTcwYzY0N2RjNDdiNmRhOTBmMzEwOTdmZWNhMDM5M2NmZmQ3MDA1MjMwNzg5ZmE4OGUxMDVkN2NjODU5YWZmNjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.h99t7hAg-sRXg2YLSsD2dlH3C13SUe3-K5f0D5X4d-5nEN4gTmoxLAdevTq75vYBt_42j45sUwwAwAZmEYn4XQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210810_103630_73_220b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.759Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImhOMnpzSk8xOC9WN0hZSk0vZkJDUjkyL3dEK3ZwSDRFWUZZc01pWmxwajFPUHFXMGZXZ1lZK2hIYWN0Q21RQUlFWGpNaFAvRDc3VUdKaFZhcHZ0U0ZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDgxMF8xMDM2MzBfNzNfMjIwYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzVhZGM2MTdlMDFlMTI5ZjU0MWNlY2YxY2YwMDVkODFjODkwYzU3ZTBlNDhmMGI4NmJkYzc4MjQ0Y2QxMWJhYTgwNDhkMzBlM2RlNGVmOWMzYzZkNWI0NzBlOTUxZWVkNTFmMmNjMTkzOTU1MTc5YzE3NzliOTZmNTE5NWE3MWNiYzA2NjE1MzE0Y2I4ZTI5NTI4Y2EyZjA5OThiZDA1ZmQ1NTk1MzM4OWQwZTE5MmViZjYxZWMyNTQxZDVhMjliMjllMTNkODU5ODI5YjEzZmMyNWIxNjkwODZjMzUxOTcwNDQ0MmY1NmE4OThlMjBjMTc1OWVmYzhmMTJhYjM4YjhjMzRjMDVjYmFhMDI0OWJkMTRiYTI1MjcxMjU5Njg4MmZjOGE3ZWM1ZWI3ODFlN2VlODQ4MGY4OGY4ZDQwZjUwM2Y5Y2MzZDMxZmE5ZGQyMTIxN2ZkMDlkMjk4NjM1Yzk0YmNmNTU0OWZmZmRiZTIyNTAzYzZjYzhhNDIzN2JjNjhjZmJiYmUxNDk1NTEzNzg3NDU4MDE3MjQ5YzY0MDUyMzI3NTc0NGZkYjg5NzM5YmI5MTBiODYwYjlmZGMyMTY1YTEyNjAwZjE5OTgyM2NhZTcxZWQxYmEwMmRmMThjNmI4ZTVhNWEwNGI5NjBlNjNiNmE3YWZjNTVhZGIxMDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YBREJ2triDIB7VYIu5K3NncFVe3nsdFTZteJaKxBDrq24Lag4qkixoOKjCUQ2dZiFi0_Md3qWRhsYeu08VK_Mg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210810_103630_73_220b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.762Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im1XY3ZwOFNSakxVVEQyUmkwQzZBVERDU1dYUS9ScSs4WkNQK0RUdkhyKzBweGNuS1lLWWJMQTFnd0NnQXFXTzMrb2RLUnRoNkpJcWQxd3BtSDZjdzdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcwMV8xMTI4NTZfMjZfMjQwNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODYzOWNjMjQyNDVjOWI2ZGE4MDlkYjVhMTE2MmM4ZDg2MDJkNDJiYzY3MWYzMTMwODMxNjhiYzhmZDY5YTMyNjA5MDc5YzA0NzkwNGI5YTdjMzIwZTYwOTA4NGMxNmE1OGJlODBkOWJkODM3OGY4Njc3MGQyZGQyZDAxMWVkMzZkYTMyYjliMGNhZGJkZmYxZDMxMmU1Y2E5OTA3ODJlODQ1OTI3MzA0ZGI3ODVlOWYxZmE3NDM1ODA1N2Y2ZTQ0NThkN2NkZTg0OGU5ZjYyOTllN2NhY2FmMWFmYjJkOTI4NzYzODg3Yjc0ZGZmMDgzODIwYTM3M2E5MjBlMjI1YTRhOTY5NDhkMDIwYmFhNzE2NGQzM2ZiYzkzNmU5ZjkzYWU1OTA3ZmVmOWRjZGM5OTRiNzgzNmU0YzZlOGUyNTY0YTkwY2EzOTEzOWE5ODYyODRiNjU1YTgzYjdiZWQyYTgwOWNjYjY5MmU2NTIzODRmYmU3NDQ1MjE2YWRkYjQ5ZWQ0ODA4MDIxZGNkYzU0ODMyYTEzNzljZmFiNjc0ZDQzYjZiZTVlZDY0Mzg5Y2E4OWE4M2EyNjEzYjg0ODQ0MTQxYWFlZmQ3MzVlOWFiZmVlNmVhY2FhNzk3N2QwZTQ0NWYwNzQxOTZkMjE4MmUyM2E0Yzg2YzhiZTU0ODQwYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jX7xppkrYDtvMlFn7dl3KPpRJizHXFUX5aLc6QnNQ4XI0bfH8PTQcLkFuHBHv0iHvopK6BDAsqlqJooEwB_Nig", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210701_112856_26_2405_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.765Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkZFUEhzNG93dTVhaUhMT1dmS0lnWXF0c2NmRG1Wd0tpUXVvM3VJSkRwcjZsNGtNejlQOHBpUTFyM3hUMFA2Mk9QMTM5Q0YxamJZNXpkc3RyRExvWUpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcwMV8xMTI4NTZfMjZfMjQwNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2VhMjAzYjI5Y2YzOGE4NzY0ODM0ZDc0NzkzNTc5MzM4Mzc1NjM4YTgyZjc3YjMyNTE5NjQxNzg0ZmUxNTUxOWMyODQzMmFlYTViMDU0MjBjN2UyYzUxZTViNTY4OTEyMGM4NjNmNDVjYmJiYTA5ZTkyNmQxMWVkZTUxYjhkMDY3NmM4MjFiMzkwYjI1ZTE0OTYxYzkyOTlhNWVkZDMzYWQ1NDc0NGJhNDY2Y2Y1MmVmNTNjYjNkYjllYjZkNWFlMDUwOWEyNmI1ZTgxMjI3NjVjZjUxYmQwZTg3MjhhM2VkMDQ2YWVlYzU3YWM5NjJhNTYzNDc4NmMwNTVlMDQ5NmQ4ZmFjNTI4OTI3NjA3M2U4M2Q3ZjAwMmU1ODNlODc1OWRlN2E3ZDQ0Y2JmM2E5OWNkNDhlNWNlNTQ0NDViODM3NzY4MmFmODA4NDdmODZkNDg4ZjM3YWY4MGQ3MWEzZDVjNTBlMzBmZjVlNmNhNWQyNmUwZWQzZjcyMDg4Y2E4MTYyYzM2NGI2YTFhNTNhZTc5YmMyZDJiMmJhMjE3YmNlNzMyZDJlYjBhYjM2MGEzNjFiZmJlZmQxMDZlMDkwMDlhNzE5MmE4YWFjMTU3Njc3OWQxNDM0YjI0MDc3ZGRlODUwODE1N2FlMTFkMDVkMmJlMWU0NDExZmRjMzE5YzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zjC4FBmPTCw_Q9qyk-U8tvVjDwwOR3XIQyNJ5zmsc1RA9W2_Xmn4u083USzDI7A5rR_SnuiSsMk3ES20JtTd1Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210701_112856_26_2405_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.768Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjB0UTVNQmlYdG5wcHo5cGlUb3ZOYlhuekFBUjBWc2c1WWpicXBGaHc1SXk3N2FmZVFpY1J6dEpzOTFBSGVNSVJIQnZ2blJqTmZCV0VQUFVYdGJkWENBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcwMV8xMTI4NTZfMjZfMjQwNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWYxYjU0YzhjYzcxNDc3ZjAzMGE2ZWEzYmM3MmY2NzAyNzNiMDExNzkxMzVlZGFkYzc3N2M2YWNiNjlmMDA4OGRkZmRhNTcyMTBmMmI3NWM5MDI1Nzg2NmViYWJkMzMxNjEwMTM5MjNiYTBmYmRiYTBhMjA1MTNlNzQ5ZDNlOTM2ZjU1NzA2Yjk1NWRkY2ZkYmYxMjgxYTNmOGQyM2I0NmE1YTEyMjViNWRiNGM1ZDUxMTljYTQzYjMyMzhkN2ZhMmE0ZWYyNzU0YTAxOTNiNDg1YzVlYjVlM2M2MDI3NDYzZTUwMjYxMWY3ZTZhMDQ0NThlN2VjZTg0Yjg1ZjZhYmEzMzMxZDg4YzI0MzRlOGMwZDU2MTE0NjgzZWEyNDRmMjgxM2M3MTBkYmUxMDdhOTM2YTZjOGQ1ZmIyZDJhYWU1YzY0Mzc2Mjg4YjBmNmEwMDI4ZGExYjg4NzFjZjE4MGViNTVjNjY4ZTY2OGVmN2VjYjRmZDFhZWI4ZWNiMDhmMzFhZjYxMDAyOWIzZTBhZTY4YzZjNTY3YmNiZWU4ZGQ4MjA5MGZhYTI5N2Y4MmIyMTJlNGZiZWM2MDRiYTAzZWZlNzAzYWEwMjdmNmQyYmVlZWFmZmVkNzVhNmFkZDA0NzZiZWE0YmY0NmFmMzQ4NDU5YTU5NjAzYjdjYTFhMjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WzJP_fv7BUGnEov6BKr2-GFr_KmZkFM7zsckEgMaQaASW1pCQblnx19WNcXeUfu4BovcoBb6kSx_2fzV-B6pcw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210701_112856_26_2405_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.772Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkswSUVnZ3l4UTRxRUhZM2ExaVQ2V0pEbklQdzk5Z3Z2Y1AyRnZHaSt3TDFZNVZiR0JDd21lTHN4aVdRaGN3Q0FzU0hhQVBQOHBKMFJzY2ZOWmZzekhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcwMV8xMTI4NTZfMjZfMjQwNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzlkOWE5YTdlZTU2YzdkMWI3MWY5OTNkNWViMGZmNjA5MzlkYTAxYjc5YWMzMmViYzkxYjU5ZTEwMTQzYzQ4YTQ3NDk4NDgyZWRkMTM5MjBlY2U0YWExOGVmMGI0ZGJmOTkyN2M3NDgwMmU2MGNjYWZmODg4MzU2MDFiOGU2ODBhNTUyYWM2M2UzZGJjNmU1ZTM0NmU0ODFlMDRhM2EyYzYxNTNhYzZkYjQxYzFmNTNjMjg3YzIyZDEwMzFiZWM1Y2U3YWU3MTUzYTM2NzUxZmEyZDJkNjI3OTI1YTFhMTFiYjE0NDgwMWE0NjRhNjMzNjYwYzhiNzg4YTc1MTQ0NzRmZmI2MzNjM2RjNzJmZDAyYzgxNTcwMzAwYzRjYTM2MmJiMmIzYjMyMTZkZTU2NmNkZjJkYWEzOGU1YjMyNGNkMDk2MWMwNzhhNWVlNmQzZTkyMzc3OTczNTk1ZTAzMDZjMDk4MmY5ZGJmODNhMGE4YTYxYzUzNDRjOWY1ZDQ4MmMzOTYyYWQ3NDNhZTFlYjg5NjU1N2U0ZGQ4YjkxMTg2M2UwYzQxNTE5MGY3YjQ4YzcwZDQ2NGQ0MzAzOWRiZWFlNjVlNGFlYmE0ZjNjNGQ2NGQzNTI1OWI0N2IwYjdmNTEwNWEzZDIwYWE0YzBjNWMwMWY4YjdmOTJmYTRjNjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pCCHy4fG7gVs50SkLqPF1jVLcx_AKK34PRE-iUQYtSjCOJ6y1PVFDTxC21b7MVaMtRMOggt9fFWi_Z06KuvagQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210701_112856_26_2405_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.775Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IldSOE43bDVPM0F5S2g2NlBKcW1Jd0NxZThlVzNzb0hGSjIxZThTU3JobTZrRlpqakkzc2NZK3plN2ZLZEJwckNramJXSUdHMGRNM25iRWRHR3p2SWNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNTBfNDJfMjI3NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODM0ZTFjNTI3YjcxMTBlOGZjNzYzMjY5MWFjMWM0MzRlNzg5ZDc1ODY5ZGI3N2RiOWViOWU1OTA1ZWI0YWI2MzRhODU4NGJkNWY2Y2E3MjgzZTdiZDg3YjIzMjNlOTFiOTU3NWQ3ZTExMzgxZGRiNzlkZTgzMTBjZjBiNGIwYjRhMGJjYjM0NmFhNzc3OWUxNWI3Y2QwNGZkMjY0ZjZiMTE5OWVhOGJlODU0YjUzYzhmNjRkYmZhNTVjYjVhNjMzMWY4MDI3MThlYjUwOTcwM2ZmZjJlZDliZmNkMGI0YWJkYjNiYjMzYjk0OTc2ZmMwYTIwZjQyYmMxOTRjNWNlNjI5Y2Y5Y2NjYTBlYWEyZjI5MjUzZWU0NGQwYjM2OWI5YTkwZTk0ZjAzZTkyZGE4ZTk0MGQ1YWRkNzliMDFjYWRlYjYxMmRlZWQ1MTlhZDI3MmYwNjJmMmI1MTExYjI5YTQzMGViODRmMWU1NWQ1MjVjZTFiMDVlNTY0M2UwYjU2OWU0M2VkMGFkM2FhMTI1YjVjN2NlMGIxOTNmYWM4ZWZjMTQyZGZlNmMyMDJiODUzNzY5YjVmYWRjMjU1OTc4NTIzMTZlNzk1Njg5OTRjOGViNDk1ZjIwZjYxODU4N2I0YzMyZTBiMTZmYjQ5NzUzODBlN2NhNDJjZjZmNmQ1MzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CRD_LVLC2SlvsiLKRq047e7CQxOuroQCjFYYIfOuWhYntgGEyPkgP80fggNiIxq1_r2doNgRc3U-Dhi0XO5POA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111350_42_2274_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.781Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImNTNk1IYVB6THFQSXJKMUpHZ2pJeTdzRklZUnZMYUxURVhlVXdnZW13eEpwdm9ScTFuNEVtSE9LNnZ1bzZqWnlzdU9odldJSmhwUFMvSTNUMUw2VmxRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNTBfNDJfMjI3NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmZjNDE4NzJlNzI0YjA1NmFjZGQwMGEyNmI4MDc1ODUwYmE1NTI0NzI2MjRmZDMxNTQ1M2YwYWQ0YmU1ZTc0OTE5OTlhNTEwZDQ3YzZlMGJmM2M0NGExMDBkYTM2M2IzZGYwNjViZWFmMTcxYjMzOWYxNmQ2YmUwYjFlZTUwNmI1YTBlYTE5ZmI0OWE3M2Y3MzVmZmE4OGQ0YjkzNDI4ZGVkMTY0NjM2ZGI1NWY2YzdlMjU4NGRjZWUwOGQ0NjZhNjBmYzJhMGZhYjY0MjlkY2U0NmEyYjVmYjllNWNkNzBhMjE5MTU3OTk3NGVkNDFkYmU0Zjk5Y2VjMmIzN2M1MDYxNjE4NWFmNmE2YzJiMDI0YWQ3NTY3ZjU5NDZiOGVmOWVkOGVhZjA3NjdhOGM5ZGYyN2NiMDhkMzhlNTIzMGUwNjBlYzhiMmJiYjg5Mzk5YzI5YmZjOTBlZDVkMGM4ZGQ2ZTNhZDg1Zjg2YWVjMGY5N2QzODQ2OGUzMWE0YmFkZjRiMDI5NmVkOWQzNzNiZjgyZjdkMjM4NTJjNmFhZGQ4NGZmYTUxMTkzNDM3ZTg2NmRlOTUyY2I1NGMyY2JkOWVkNWJiNDE4YzYyOTgxNjBmZjRkZDc5NWVhYWNjMWM5MDRlZjY0OTA1OTUzMWMwZGNlYjZkZjdmYmNjZGU3ZjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EQXU4ucpIQQ9bOIgYnxza7WRbZpqN0KVchNQTofS0TtZK5ACRhSDX3jPi7fojqUshvLNoD3LajwLeSL2tEBaWg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111350_42_2274_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.784Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImNzeFNUZ2pPcmF3YUx3UzNlcCsvNWRsaFFaTnJRSTRnZ0FmczhsS2NSeXNaelN0NHZaTWV0eDJtTmVCU0dHaWkzWXBRUlpWUUtJSlQySUFKTVcxRER3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNTBfNDJfMjI3NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODE4NzYxYjFmMTdjNzBiOGMxODE4NjM4NTA3NDY3Nzg0ZGRmMzI3NWQ2ZDNjMzUwZGFhNjBmZjNmZDQ1YWI3N2FjMGI1YTIyMmFiZTQ5ZTBhMjU4NTA0OGE5ZmIyODhiMmMyMTgyYTNhZjJiODljMDIxMDY5NGU2NmZlOTE2M2ZhMmZjYmM1OWVlZGYxYjQ3MGE3NzJjYmFhZGQxYzU4YWI2ZTRjMDYyZWY0M2RhNzBhOWM3ZjQ1ZDVjYWY4OGQ0NWRmZjZhNTljYzI2YWRiMjBkZWFjOTdiYjBmMzcwM2E2MDY4MGY0ODQ0YTljMDE2NmE5OGI3MzcwMDhjYjlkNmU1ZjY2MTVkMzQ5MjMzNmMwNWE0ZTVkYWU1Yjg1NzU3NDk0ZTU0M2U2ZGFiZjVlMTNhMmQwYWU1MTdjMDU5NTcwNDdkNGRkZTg1MGNlNjM2M2Q0MTlhMDk2ZTIzNDhkMmQ0ZTVmODk0MjE0YjM4NTBmMTgwYWMzZmI0NDZiMDE2ZTZkNGJkNzEzZDIxOWRiZWM4MDMzZTE0ZTFiZWIwMTQzYzJiMDkzNDg4ZTUwYjU1OTA4ZmNiNDUwMjdjNjdhY2ZiNzEyOGI4Y2YwY2RkZmE5YmQzMzAyMzE3YTEwZjFmNDA0ODllMzExZTA1YmU2NWUzODllZTFlNWQyMWRiZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AYGfkWpDPof0OUZ_5UI-E_AEAQCBZ3ASGeLfhLxbwwQRevzZ_dnB05R4JQfJoRSG4TboanYgA37JfxRP9NN7Qg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111350_42_2274_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.789Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjFaUmlnWUoxbjQwSkYrcHNxQmhkWXpQb25MV1g0ck5tRU1TMUR1Y1lJZEhCNlJZQ09qejljeXRwaFNOeWduTnVWWHNDRGRTV3JGZnlEbmoyUGZVT1hBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIyN18xMTEzNTBfNDJfMjI3NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODYzYmM0YjVkZGU1YWZjNDRlNGRmOTA2NmRkZGYzMjgzNTY3YjNmY2E5ZGUzMjNjN2NjYmRlZDFhZjk4ODA0ZDMzNWQyZmI2MWQxYTk2OTIzYjBmYjNlOWFlODY1MGMwZTBhYTJjMDU0YWNmMDUzOGY1NWNkNzZkNmJmODBhNjM2MDZkMzQyODIyMDMyYjRmZTU0YzI0ZDcyZjIxNzE4OGE0NmJmMmI4MmNlZTQzZDcxMTJmYjQwZGM3NzA1N2Q0MmU0OGVhZDM2NGViYjNkZmQ4YWU1NDNmY2YzOTYwYjZiODhlNWNlNjY0Y2U4ODVjNDM5MjcxMWYyNjRhZGNjYTAzZDk1MGU1ZTA3ZDg5ZjU5OWE2MWFhYmRlNjNkY2MxODU1ZThkOTg1ZDIzMjEwODY4ZTUzMzAyZmIwYTlkY2U0M2ZmN2FlMzhiYjZiODc0NTc5MjliODdjZmU4NmY2ZTc0NDIzMWE5OTI3ODQ5YmI5ZjE4YzFlYjBhZDI2ZGE0M2NiZmQwNmQ1ZGIzZDMzODA5ZGRiYWY1MGI0MmZkZjQ4NzBhOWUxNWFhMGI2YzM1MzNlMDZhNTM4Yzc5NWU0MDA0MDM5MDI3Yjg4NjI3YmI2M2EzMzdkYzgwNDI0YTA1YTBjYWE3ZDE2ZTQxMzBlYjJjOTc2NGFkYmMxNzk5YWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hionp6zvI3HS0d9gzVyjWjxdcIqdS2ZuU9rlTwc6xn394gDoJvlTeHF2Nn7OIdAqnoQ9V0o8VTNfALGaqDWCeQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220227_111350_42_2274_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.792Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlI5Qm91WW1lbkl2UEJBZVNBdTRDVUJKNkhQZ1Q3cTFEZDlZNTNwa1d3MVo0V2tsT2NRTEd3S2VNNDkwbmNEL3BNOW4ycG9tSno0aStJZlBNWE4zd2JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0MzdfOTlfMjRiM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGU4ZDFlNjg3NzI3MjZmZjY4NjdkOWE1NmVjMDY4YzIyMjhiNjJkMGEwYTEwMzg3MTMyMzI0YjY0MzU1N2JhMTczYzcxNmUwYTMxNTA2Mzg1MjJlNzRkMjJhMjQ4OGNiNTQzZDJjMzY5ZjI1NTFhMTRjOTU1YTIxN2ViYTJjOWI4YTViZjU3ZGI0ZWE3OWE3YjE0ZjYyNGMxN2ViMzY1ZWZkZGYyMjlhMjU3ZTI2Yjg2MDVmNTY1ZjYxMTAwYTAxMTlhMDFiMzczNTE0MjdkOGRmYmU0YmI0MDJmNWEyMTBkY2VjOTEyNGQ4ZGIwMjFmYjI0Mjg0NGRhNGJhNTNiMjlhYTcyYTVhYzRkZDFiNjZkNzMyZGU2ZmU0OWNjOTI1ZTQwNmZjZWFlZDAyMGM4MTk1NGMzMjEzMzdkY2RiOWQxYWQwNzY3MTcxZmVjN2MyOTVjY2FhNDlkMTg2NWU1ZjJjYjdmOTNkNTM1YjEwZTExNWIyNzY5NjM3MGJlNjM2MjZmMTUyNTc3MDBiYTMyZWM5ZWI1YjAwNWNkNGRkYmU4MWJjN2Y0YjNlYWM4MzdjYjE0NDg1OGFhYWJmMGQzZTVkOWJkNDUxYzY1NTQzNmRmMzdiYzFmNmUyMzY5OTk4N2QzMTRjZDlmZjBiMmY3NGZkZjA1MDg1YWZmYzIwOTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6X5IPwCQ_0gEiisj6Nuh1uH1Z7gy4sfMd78ZBHAgAG3Vu2T5If5TdIymr-M4S62VwKyM8nqOssASAqXi-CRi2A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103437_99_24b3_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.795Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkxpSjNyVmQxOEdTOVFNNmhBMFBFSmk2ZmNLZW90U0VUT3ltSHVnUlB4dXZJTktmWk1oT1ZKeUJmczdiSzRJZUR5NFFwNUdsTi95WkM1RUpuUVJzNkJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0MzdfOTlfMjRiM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODRkYmFmNzg4NGZkNDcwYTBhYWMyYzU2MjU4NzU3MTJhZjNhMmIyYzQ0YmZjMTljYzEwMWU3ZmVlMThkYzQ3Zjg5ZDdiYzNjZjg0YTk2NzI5MzVlZDYwOTcwZTA0NWMwMjI4ZWZhNWI0NjQzMzg0ODIyMjU4YjljNGVjYzMwZWMzMDJmM2I0MGNkYzcyODJiNmQ4ZTFhNzkwOTdjOTRkNDViYjI3ZTFhNDA0ZjBhYzlmMGExZWFjNzZkNDBmNWQ0OTIxYTUyOTFiYmQ4NGY0Y2E2ZjAyNWFmZTg4ZDdlODE0ODkwZGNiMzRkNDQwM2U3ODEzYzhmMDU2YjQ1MjhlZDYwZjYwMDg1NmMxYjY3ZGY5N2YwYWNmMGUxZGY2NzczOTU3M2Q0MjkyZDUyMmU0OTY1YTM2YmQ0MzRjMjNiZTcyZjk1NzY2ZTUwZDYxOTY2YzM2OTFhNTYxM2UzZmNjMWU5ZGJiYmI2OTMwYzAyYjE0NGI1YWUyZTY1NzkyOWE3MzAzNGIyOWQzOTgwYjhkZmZmNjk0ODlhNTAxOGZmNDJhM2MwNjM0YWMyMWQ2NDMzNjViMzQ2YmRmODZmMTQwZmU3MTMzYTc1MDYwMzQwMjU2YmY1MDg2OWM0MjI1ZmViZDFjMWViZjAxNTkxM2RkNzUzMGJiYTY0ZDkzNGQ5NGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pCRUFWL-PCZmAqBKqS8F6gZz8qokmoNt9TF9PEFdcekPCLKQAj0LdJLprsVrVKchEKrG9vY1Hj7pdmHFEKdnxw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103437_99_24b3_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.798Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IktoWkgzUy9kMlBNeG16NmtzTm5kak5FWFNZbnlBK0lsODdiSElOcFRUS1RIVXozNU81SDM5eGZaZmpVN0RaT1NFRC8rRHFVTzkvUE5EWWJuMjBmN01RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0MzdfOTlfMjRiM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjA4MjMzNjQxNzk0ZTUxY2Y2MzBlMjM4NGNhNTNiYzVkMTJjN2UyNmM2MTU2MDQ3NGI2ZjkzNzMzMDAxNTdmNmEwMDhjMGNkOTVjNGU0MWExYzdmMDI5NDI0ODJmMmQ5ZmRmNTBlZjU3YTM4NTIzMzg1MTllMDBmNWY4MzIzMGMxOTdlMzBlOTBlZjllOGM3MTM4ZWNlYWZlYTlmNjdhNmFhMjMwMDk4YjI2NmRjZWZhOWZmZjhiMTUwMjI3MTg3MTliMDA2MDkyY2ZhZGE5Njg4YjIxYTc1OGRmZDY0OTUyY2UzMzEyMTA0MjUxZDMyNDc2ZTkyZDM3ZWFlNTExYWUwODY1NTBiZTIwZTFkMjkzZjI3NGYxMTM0MjdmODg2YzYzMmZjNjdkOTAzYmYwNGIxNThjNDJlODY1NDQyNTUwMGUwMjNhNTE1Yzg1YWJiZDJiMmQwNjg3ZjI5MGNmY2NhZjY4ZjAzYjViYjYzYTE2OGQ4NTAxNTE4ODVlMWRlZTdiNzg2ZGM0OTA3YWFkZDM5NjJjMmEyNDk0NTZkYjNjY2FhODc5YWFmZjJhMTJlZmQwOTNhYTVkY2NhNGNhMmU2MTIyMjQ1Y2RmMGZiZTA1M2UzZmUzNDRmYTBiODAxOWJjYTU0ZTRhZDViOWY0NTU0Y2Y2YThmOTYwY2MzMzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MpWQeKf_bXtz4LQ9XcFkNWBbn04EQlHi6jpXIS5p0yvOGkdmpSz_8hpFwQqyQ_0GvkXYkiUd23c2inOfTgphOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103437_99_24b3_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.800Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InBsRXJhM05rZEpPOVhCODlvME5mWFhBR0lyNS8xcUJCM0VFMjVwekd5NC9tUFU1a0VuUWhDcDR6cGsxc1FzbC9DTjlzVHUyZGZ5OWV5cXFJL3hYcUxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0MzdfOTlfMjRiM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDA1ZmIzYWZiZDlhOTA5YjMzZjIyYTk2YTYzZGI3Y2IzMTkxYTdjNTc5MTAxNjM5ODlkYzk4NWNlYmU1YjAxMTVmYTdhMzFmZmFhMWY0YmVmOGJmZWRiYzI1ODdlOTUzYzA3YWMxMWY1ODE0NGE1NGQ0YjA1NzFiZWNjMmIyZGViMGRlNWQxODNiNjdhMzM1ODQ5Y2ZkY2EzYmYyMTFkY2U2Nzk4YTc3MjQ2MGZmMjU0ZGM4M2Q3YmM5ZDJjZjA4MmU4MzM4YmExNzYxOTQwYzFjZjYyYzdjYjdmOTJlODlmNTFhMWZlZjdmOTE4OTUxZTk1NzljYzYyNjM3NjFiZTQ3NWNjNTUyZGE2NGFmYzUyYTBiZmZkZWFlNzVlOTMxNjFmMzczNDA4MDczMzNjYmUyYWRhY2YwMGQyZjFiNjQyNzRjZTU1MDBiZjc2Mzg4ZGNiNjZlZDY4NjQ2YTJmZGE5YzllZjliZTgyYzY4YjgwZmE5YWE2NzNjYTM3NjQ3Y2YxMzEzMDU4NGM2YjI5ZGFiODdlZTY1YzJmNTRhOTY2YWFlMDdkMmUxYmVkNDc5NThiMzFjYzUyMGJlYTA1ODgxZTgwYzI0NjkxYmFhYTEwYzQ5Nzc5MWY4OTg4OGM1NDg5YjFlMWJhMTAwYWE4MDkxODY2ODAyMGFkZjFlNDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3tSsy76onIx90oPMH_r1qekUn5_2v04n8fkmB6cjGkGla-W-QJNPwNWA9MKYJXfesJTPEc5fwnB5_JTJrUBrYQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103437_99_24b3_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.803Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjdBTnpZK2t4R3hhcTE3UjltNmF1SVhvN1NSejUyRGVmaGk4R0NkdUVEaldDTENDWUhVclp4YWFkM3ovRUpNZDdRemdTcnhEY3JPVVE3TTVzVUMrcXdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyOF8xMTIyMDFfNThfMjI3NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDQ3MDI4ODk3NWNiYjgyMTU3ZWI4ZGE4Njk1ZmFiNzlhNGI4Y2IzZmM3NjFlZDRlZjc3NmRmZTQ2MjY4MDk2NzRjNjIzZTI1OTdjYzJiNmVjYjMzMTkxODU3ZGZkYzUxNGUxZjExNmZhZDU0NzMxOGUwNGQ4ZDNlZTg5MTk3NzRmNjVkNGE4NTI1NmEzZDhlMjk5ZjBiOWVjNmIwODU0ZTAzNzUwNGY2YzMyNjI0NDlmNTdiMjZkZGVkZDczMmQyN2FiMzdlZDA3NGMxM2EzZTIyZDhiYWFiYjU0OTA3ZjdlNDVkNzRkMWU1ZjZkY2YyYWI1NTdmYTllNGIzODEwMWY5OWU2NzQwMGVmZDlhYWI0MzkzNDhiZTgzNmEyMGQxYjMyNWUzYzVkYmMxZjA1Yzg1ZTVhN2U1OWM0YmIyNzJiNzA4NjRkZDNiMDJhNzMyYTY5OWY4N2Y3YWNiMDkwZDYwMmI3NDkzMGI2NTNjMmFlMDNhMTEwOGRiNDUzNjE5ZjY3ZmNmMzI2MjU2YzRjNTA3MmZlMzBmODQ2M2Q2ZmE4ZGIzNWE4YmQ3Y2ZlZTk2YmIzMjU0MjljMzkxZDhmZTJmMjZkZDQxNDE4OWI5MzUxZjQ3MWNlZDEzMGYwM2EyNDFmNmJkZTJhYmRmZjFhMDI0Mjg3OWFhMjA3OWZiNDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Lj-WBeRqXQFPGnA6Mmfk-fPb6mLmcOWFz9mvMF3a_RM9K1qGYZVqBcF11A5jhi1ncob6mDa3l9FWnejQjJrriQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211128_112201_58_2274_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.806Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikl2Ylp4YmdtNDcwMFY5T3NjaVpzZkZUVmpHZnhZZkNFT292eVY2VVBTaWpPV2RyMU9qM2VsMzJ1VlF6UjFrY2VVSVdKYlVERnZ0MDdpOTdyVTJJYlBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyOF8xMTIyMDFfNThfMjI3NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjZhMmM4NTMxMDM1ZDk4ZmJiYzczODVkNDllYzRkNmYwNDU1MGRiZDljMzRmYWVmNzMzYzc4ZjQwZWY4OTIyMzAxODdlM2UwMDIyZGMxMDUwMDVmNGZkNWU3ODIyNzFiZjdjM2FiMmNhMjFjNjBkNGI0YTVmMDI0MDgxODYxNWU2ZGM3YzEyOWFiNDNjMjY3MTI4ODI1ZjAzNTIzMTkzOTQ3Y2E5OTAyOGI3NTJmYTJlNDQwNGI5Njc4NzU1OTNhNzYwNmFkYzQzZDVmODNlNmI2MDE4Mjk5N2VkNjA1NDQ1ZDkxZTcwYzZhZDQwNDY5ZWU0ODgzODJmZDZhMmExM2ZmNGRmNTZmMmQwODQwZTYyNmIyYjdjNGQyZTkxODJmODI4MTEzNWUyNWY0NzQyNTdlMjg2YjYyZWI3NWY1MjBiMTY0ZDczYzgwNWIyMjQ0ZTQ2YThmZTQ0ZGFiM2Y2ZDZiN2UwMjYwZDM1ODcxOGU2YjVhMDNjYzRlZTM2Y2JkYTI4NmQxOWZhMjIxNTI1ODI0NGIzNmQ2MzViMTZlZjgxNjFkNjBlZTU1ZDJhZTVmZmVlNzQ4OWU1NjdmNjY3M2Q1YjBmMjYxMzdkMDYzM2EyOTExYzcyNWQwZTUzOGQxMTdlYzJkYTk4MTYwZjVhNWFhMmQ4NzhlMTFlNGIwNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wo0wRRlTCW-xU17eNRHhssFbly7bDtS7iCx6gL3haRca4jTYUsNFUrxAXiWdQAJy2p12EDIwLUkGshB_-HvqPw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211128_112201_58_2274_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.809Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImZsbzBqZUJyN2JzWFdrUVRJemZGM2E3SXNWNzFnU1BwcW02aFZMNU1QaEFCaFRDWWNaRmlHWE5uNk9UV3dNL0wvbmw4dDlZa21wQXMzQlcwU3EzakNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyOF8xMTIyMDFfNThfMjI3NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjEwOTFkODRiM2UwZTAxMmVhN2QwZDFjNjVmOGQyMzg5NTlhZDI3ZDU1NzFiZDI5MGY2ZmIwOTU0NjZkODlkM2ZkODhlZDhlYWIyYTRmYmE1ZDljMGQ3OTgzNmU0MTBlMjNhMDhiMjFlMDAyMzExOTc0MTYzOWY0NGE5MmMxOTU5OGJhOThiZmI3Y2U4NDhlYWI0ODEyODM3MjZiYmM3ZDQwZDYzN2QyOTVkZTc4MTQ4NDJmMjRjMGQzZTYyYjU5NTVjZDJlZDM0ZThmNjA0OWJiNzFkZmQ4YzJhOGVkMDdlNjJlMzNlNzJjMGQ2YzM5OWJkOTU1M2I5MGVhMjU3MTQxNTE0M2MyZjQzZTYyMDY0ZTg0N2JjYzkwMmYwM2RhYWIwMWY2NWM0N2Y0YWUxMDBiNGZkMjdkMThlNzE1NzNmM2M1MTA3MzZkZWM2YjYxZGE5ZWM4Y2ZkMDhjODczMjk4YzVjMjBlODVkMmE3YTE4MzRjZjg3NmM4ZDFlOGU2MzM5NTFlZDBlNDU5OTFlNjQ0YjdiNThjOTM4MTUzNzFiYjlkYTEwODA4MmExZGRmNDlkY2ZlNmVhMWM4ZTg4YTIxOTFiODZkNWYyM2I3YTA3YzBhYjRjZDY0YzNiZjA2MWM3NDkyNTYyNmNkZDcxNTIzNDYzMGVmMzA2Y2Y0ZGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jIPAEe_ZGfann04xwNDwQIAopFdgnG3MrsL4p0JQe-jwxYBCzQ-_1IcJpyM7XKbCTY8mBwjFDjYDiPbe-pMVqQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211128_112201_58_2274_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.812Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkkwM3VPNWRWcTltazMvMERDdWhDUXFWN3dQengrSjF1aDNLU3hnNVRUWWZtdkoyZFMwZ05rZ0xrU0lPLzJBMUh5K2xzUkUxM2pENW9LVU9EeUo1eHN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTEyOF8xMTIyMDFfNThfMjI3NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTIxMzhiZjY2ZTllOGNkNWM3ZWY5NjNmYTY1ZTc1OWJmMTExYmRlMDg0MTY2OGY1ODFkMGQzNDZjZDY5ZTU5ZWRhY2QzZDhjN2MwZTZhZmIwMWFmMTFjMDBkNzc1ZTAzMzM5NTI3MWIwZjNhM2NkNmYzYmUzZGM1MzE4NmVlYWJjZjQ0Y2RjOGJmMmM3MzMxYWZmMWNkYzlhZWQxNjRiN2ViMjE0Y2FmYWRjNTg0ZTdlZjYxNDYzMjJkY2Q4MzBlZDMyMThlYWVmMjBmZjk5ZDAxMTk5MDBhODUyYTVlODA5Mzk0MTMzNDg4MDUyMThjOGYxMzZmNjllNzhmYmI2NjYyMGRhNjA4Zjc4MTg0MmQ3ZTk1Njg0MWYxZGM0ZGZlYzQzNzZhYTVlZGY2M2U2MTBhYzc2N2Y4MDNiYWY5YTlkN2QxNjI1NDcxNjNjOTYyMDg1YWViZGFmMjI0MmMxY2E3Yzc4ODhlN2FiZTYyNTYxODdlNzEyYjRmMzRhOTFlNzEyZDQyYjE3OGMwYTQzMzc3YjgwMmI2NGZkNDg5ZjZkZjNjZjI1ZjZhYzc2NjQ3NGVlYTBiNmI2Mjk0ODJkNGNjMDg2YjVlYjdhYTk2NTNmMWZiODhmNzI4MjYyNjNiNDdiNTlhYjYxNDA1ZjQ1ZjM0NDVhM2Q2YTM1YmI2ODZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3GhTMALPXsfytz_SPnweh4dIWK7CUaGhLNMVsdCNKI9PTtfAVINE7MOEZHjfz4_iofHE1UC8tTJRXX9TC74vMg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211128_112201_58_2274_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.815Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlRtVkh5QTdvNGhaekw1UnA1Z0JuekQ4WW53Zzl3dWppcEJxT3l0cGQ4Tm5LSkIvM0MrTFlQeEM2eXg0S3ZQOExNWnZaVGRreHpqUWdkNEhiSUxVZjRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTMwNTRfMDZfMjQwN19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODE2NzQ0MjNkODkzMjQ1Njg0ZjJhMWI4NjMyYjIxMGIwM2FlZTk5NmZjOGI2NzIxNTMyZTc3MGQ3YWIzMWFlNWU0MWRkNWNmNzU2NzQyNThhZGI3ZWI0ZTljNzEyYTRkYzMzM2JiODU2MDc1OGYxYzZmMzAwNzI3OGM1OTgzNWQzY2Q4ZDVmN2Q4YWEwMmMxMTI1NTY3MGRmNDFlZTI3ZDE1ZThhZWVhYzc3OWE3OWI0YjU2NmJmY2Q4ZTk2MGM3OTVjNTYyNmQ1NTNjYThmYzBlMWZmZTZmYTc3ZjlkZjlmZTNiZWM0NWM2NTk0NTk2MzlhODc2ZWRlYzc1MzAzMWE0NWM2ODIzMTU4ZjYzYTE2YWUyMGQwMmE1YzhhNjcwMTRmYjgzOWQyN2JmNGZhYmUwNGU2MTk0YTkyMmZiODhlZTJkMzhkMmI3ZTc2MWU2NzcwMGZhY2U2NWNkZjE1MjE3YWMyOTViNjhkMjhjZTdjNTVjNWQxOGU2N2M1YmYzNGMzNDBhYjE5NWE4Y2U1NGM3MTc2ZTU2NDFmY2U5ZTdiNzU1NWQwOTkwYmQ4ZmFhN2ZkZTFjZjljNjc2YzlkNjMyNDNlMmFmNmU3MzI1MDM0NDBkYzBmMzkxMmVmMDU4ZGQyMjcwNjBiNTdjNzJjNmE1ZDJlMTI4MTk5NDc0MjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dh0lbgu5apNAym6ePQh2yTWtuJG7f7B9wV3FoJqVJ_BD911weZVj8GHxVqmtHC5ceaIkO0YJgeW3xvdbklPwlQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_113054_06_2407_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.818Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkROQkV2T2ZYSDV4b3NuTlBabGdRVVUwb1Q3L09peEV0V3hXSmdjMk9pK3BFNlhkWkpURlR4NW00UzcvTU42RHRhblYwUC9MdmxXdVU1V2NJUGZXSVdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTMwNTRfMDZfMjQwN18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTdkMzUwMzBjYzQ3YzU3NzkyNTJmNDAwMGMxMzNkNTc0NmQ3NjNkOTQ0NjYyZmFjMTg0ZTRiY2ZjYjZhZTM5YjU0ZTgwZWJhMTE3MjJlODkwNzdiY2ZiNzgwMjNlNDcxOWZhYWFhNDllYTk5NWFhN2EyYzBiMzA2MzI0M2M3NGYwYTRjMTBjNDAwMjFlNmFmOTQ3NzNlMGJjYzcxZjBkODAwMjE0NmU0ZDRmNDljNzIzMGJjYTAwZjNhNGFjZWYwMmU4ZWU0MTQ3YmVkYWY4MzdhNmQ4M2RhM2FjODJmYmVmOTA3ODgxYjRiMjU4MjA3YmUzMmZkYWI5Y2NkOTM0NTA4Mjg0MDVlYjgyZjJiMTljZWY3OGEwMzI4ZDAxNmI2NDZmYzZhN2YwYWM5MTE2YjFlZmMzMjk0NDdlMThiMWQ2NzhiMTE4ZGRiM2U1MDBhNTAwZTI5YTI4ZjVjZWM1NGMyNTBjNzRmOTMzMDEwYTMwOTBiZmY1MTYzZTM3NGU4YWFjNDZjYjY5M2ZhZDk0Y2E2YTMxMDgzNjA3ZjU0YTQ3NTY3YzY3Nzc2MTlkYWVmZGYwNzcwYzFmYWEwYWRkYTcwZjEyNzVlMDBkNjY5NjQzM2UxYTg3NjQ4NTQ5ODNlODBlZjFmMjNmOTM4NzI4OGMzM2ZjYmVjZDJkZDMxZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.V3H7yyQZblE6JtlXJDway8K2SMabePd67EuR-e26Fhgj0X-wqNwm1WNeb0vH9kaSB3VD9eBjKcKu9w9EBykU8Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_113054_06_2407_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.820Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImY5SVY5ZmdHMkxRbTNLUXpYYzFjVnl6OUNwbEdFOGh4bnIrMnBPMHUwdVlpcldkbGVsWDNFeTJaQ2FxS3FuYkczR2wxKzBNVTRibU9QSzRQU2RMSmdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTMwNTRfMDZfMjQwN18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGJmYTFhNWZiOTU5MWUyOTZiODE0MGFlMjcxMGEyMGI5MzhmNjY3NjAyODQxMThjZTc1MTZhYWYxMzg1ZmVmNzc5ZWQxYzIwZTRmYWMzNzE2MGM3ZDFiNzBkNTM2ZTZiNmExNzhkYjgyZWNiZmMzYmRkYjA2M2JiZjkwYjkxMTg2OTY1NzZjN2E2MTAyMmYxZjU0YzAyMzg2NGNlN2UwODE0ZjBlZTAxYzgzMWNmNzU2ZjczN2U0NDJjOGYzMzVjN2YwOTgzMzE0NTBiYjg5ODJkMDY1MzJmMDNlZGM4NmI5MmIwM2JmYTM5NTVkZWU2MzBmNmRlZmExNzlmNWJlNzdmZTM3YjMzY2MyMzI4OGZiMWViYjMzYTQ0MzI0M2NkMzNhYTJjMTFkNGM5NDE2MzE5YjBiN2Q1YTg0Y2UwZWYzZjY5OTg3OGI1MWU0NTY1MzI2ODI3MmM3MzExMTI1NGRjOTEyYzY1ZTliMjhmNTBjMzQ2ZjE5ZGYyZGViNjQ1YThlZDI0MjBjYTE0MjI2NGU3ZWU3ODlhZGU0M2RiOGJhMTgwNGJhODM4ZjAwYmUyMDQwYjE0YmE2NTgwYmMzOTM1MjZmYjFjMTJhMTgzOTM0YTMzODZjOWJmMTRmOWNlMWEyMjAwMjMxYmE1ZTU0OTMxMjZkNTE2NTc1OWY3ZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XxpDQRszmamk9BX49tV2EfNWhZfQClBGGrUKDH07v2eznosQvnI4l2q0dG_jpq-wRKlQa65Bml8UjfrNcOs6Eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_113054_06_2407_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.823Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ilh4cHU2ZW9vbTBYN2N0MUM0RWFWUUFPQllvUGprSnNvUzBIUFRtZTYxbk52RTlUS2ZhMVdtYTZIQ3VBWkxoZEQ3eHh0enpxaGw5dFZWNzNOKzhPdjhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTMwNTRfMDZfMjQwN18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWNkZmZlMWZmOTMxZjZlYjZjYzMzMTk3YjNlMDQ5YjRiZmVjMmJkY2IzZjZkODg1ODA4OTc5MTRhNzk1ZWUxMWJkNWMwNWFmMGYwZDFkNTNmNDk0Yjk0YzEyYTI1YzQwYzA1ODA5N2ZmNTk4M2FlZjkwYWJjZGRmZDlkOWMzMzNlOTAzMzU0NzUwMWMzYzM3OGRiYWZhMjk1OTE1MDdiZGFmOGQyZjAzYzNiMTlkY2RhZTliOGE3MGVlOTA1MWY4NzNlYjRkZWUwZDllOTRkOTlkOGQ3MDcyOTkyMzI0YmM3MWFiNmJmYzNjYWRkNTdlZTEwZjQwMTA1ZGUwYTEyMDZkYTE0ZmUzMzc4N2JjMmZkMmY2ZjUxNDdiZmYyNzhmMTkxMDgxNTM5ZDAxMTU5ZTc3MmQ1ZDkzNWYzYWE0MTU3ZjNlMjIzNDFjOGYxMjA3MmI5MjUxNTY5MWEyZTA2ODVkYzI4Y2E1YTQxOGNmYTMwMmQ1Y2UzMWQ0Mjc1YjYxYjA3ZTE4MWUzNDM3ODg0OTk2MWRmOGE0MjQ3NzI4M2EyNjc3ODkwNGIyNDI5ZWJiMzFjOTQ2Mjk0OGJjOTQ2ZWY5ZjFmYjRhMjhlMzQ5YWQ5YTNhOTk0NjQ2OGQzZmE1N2Y4NDQ2ZThkMjcwMjdjNGE2MjM2NmUxMDExYmU5ODRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vg5YYaT4F7piqBWUHhmLaCG2gQs9vIXeSpBvPx-8Md0_DHOy4EmBb8POdoA3_wfy8RX2yK7l7ttaceYwT4CSzw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_113054_06_2407_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.825Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InIySTUyWEQ1RDl5cjZHcVpnRkEzU1lNSm9QYklobG8vQ1BzZkUwYnN4WnRCaTVCU0RTa0EwZzIyOUtBczdUVTAxM2JtSlBsWFhleDdCeGxuSGR2OVNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwOF8xMDI5NTFfMjlfMjQyZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDZiNzBhMzU2NWJjMWI5OTI3MjNjY2ZjZGFjMDkyNjE5ZWY3N2ZkNDE0YmI2OWQzNzkwZmVhZmM3ZWFlOTNlOWM5MWRjZGNmY2Y3YWY2ZGYwNTBhNjM5MmVjMmM4MDZmYjc2NWY2ZDllZGU4Nzc1ZTMzNTljYzVmN2M3M2RmMTY5MDE2MTBlNjBhMDE3MjU1OTM2MGIzMzQ2MGFjMDE4ZTQ5MDZhM2E2ZjI3ZWFkOGVmNzYzZTIyOTk4ZmU2Mjc3Zjc5YzFkYWFmNDY2NWMyYTNjZTgzY2E4OGI5MTJkOTMyYzhjZTk4ODFjZWNjNDlmNDBkNjFkNGI4ZTY2NzQ5ZDkzNDBjZmFmODdmNDhiZTYwZjZlMTEzNjFhNmY0ZDkyYjJlOWZlMmMxNWJkMDk1NWZiODVjN2M5MmZiOTkxYWIzMDhiOTYzOThkZDI3NTMzMTAzZTliNzY2YWMxN2MxMTQxYzRjMmE1OTc3ZDE2NGQwM2UyZmQyZGVkYWI2MTUwNTkxNjE4NGQyYWRhYTE2ODQwNTdiMDU1ZGI4YWZiNzMzMjFjMGFlNWI3NjNkMjA2MTBjMzkyZTJjMDdhNDBiNjI2MTA0Y2M0NDM0MmY2ZjhjZTRjMjQ0MGViMzVlM2UxZDNjNzliMTQwYjczZWFlOThjNzgzNDg3Y2ZjODQxN2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.G61wKEHrxFQghRpuDkYMScAvnGQ_YTu3_3Ev5qPOz9f1Gu_fUtA6e8sp1qWDNE26TUv2U4wNhytM_wQvPcG5Xw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230708_102951_29_242d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.828Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlBUNENhOGo1Y3ZQekdRNXlIQ1BLSEFxL3Zha0IwUVIvSWF0SXRIa29EQlhDaldicmlyNEUyZFFPakxzQ1hvcWRVRHpGR2oyN2ZjVXY1ZjhIV0NZbURnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwOF8xMDI5NTFfMjlfMjQyZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzdjMDU2MmRmMTUwYmI2NWU0NmVjNjY5ZDU5OWZkNGFmYTgwZmU2Nzg1MjczMGE2NWU3NjA2NzUwZjljZTYwZmY2MTExY2NkZDFlNjU2NDBiYTY3NWJmMDU5YWMzMjZlNjI5Yzk2ZjQwYWI1OGIzNmU4NzAwYWE3MGY5MTAwNjI1NmEwMDM0NTMzYzBlMzJlMWQ1NWQzN2JlODQ4YjFlN2FlNjY5ZWYzNDJlOTg5MTM4MmI2NDk3MGM5NGI2YjAwNDYzOTAwNzk2YWY5ZTIxOTI3YTE0NjcyOTk4MDFjZTAxYmIyZmY1MDBlMjE5MjdiNjVmNjg3NDY3MzZkMGJiMjUyMmYzNDcyN2IzYmNhN2U4NDIxMmM0NTU1N2Y1NGFkMDliOGJlM2Q3NTA0ZWQyYzBjMmI3OWEyODAzNmMzNWMyZTBiZTc2YmExNmEzMDQ0NmNkYTc3Y2I2ZjE1ZjE1NmJkMjBlZmRkZWQ1OWVjNDE1MGI1NGYxNDgwZDM4Y2Y4MWVkMzU5OWNiMmZlNTY0MGM2NmE4Mzg3N2RjNzNkOTIwNWQ5NDUyZWZkNGQxNTQwZjFkMjY1ZGQyZjA1NDYwODgxZTg3ODEwMmM4Y2UwNjVhOTdjODhmY2FjZDM1MjY2NDMyYmYwZTAwNTk4ZDExYmVhZmMyMDZlZTZhMmRmZTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Tleh3wgp_dyiDK5FFzD540L29w_1MYz7p4bdzelsA0D1f3kOd757K2RaOMqVfdAPUqhIszqmJMiqtT0tm0_9Eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230708_102951_29_242d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.833Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Iml1blh1bHpLNmQ3d1NJbWdvNXRZQ0NObk9ROUFNSmgvRHFiOGY5NlozenZJZ3dDQmt4T2pUNFNwRWxLSGlsT3V5OXNRN3dHSkVnZ3h2V0UySk5BeWd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwOF8xMDI5NTFfMjlfMjQyZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWZmNmNiMjYzNTMxY2YzZDljNmQ2NzgwZDRjYmExNWFlYWYxNzBiZDI5MGRiN2I3ZjkxZjVkMGIyYTczMTZjNTQzYTg2NWZiYmM4NTA4YzcyNjAwNjdjMjM5ODQ1NDhhYzY5OWZkMWJmN2JjZDFlZDNhZGI4NzBiYWI4MmM4NjVhZThhYTM4M2RlZWIxODA0ZWQzYzJmNTExYWUyZWViZjYzOGJmZGViYmQwYjlhOWRiNDEwMDY4NGMxMWFlYjZiOTIyZDBiMmU0Mjg3NTQ3YmQzMjFlOTA2YzY3NzQzZjZjM2JjNjlkYzM5YjUwZjdjODFjMDk2ZDYyMjI2NzVhMjg5NTkyZTU3ZWM2YjA4YmZiYTVlN2U4ZjRmZDE5NWVlZGI2MGE2Mjk2Y2M4N2YxYWZhY2RkMDdlZGU2NTNjMjk0Y2UxZDcxMGMwMzZlMWQwODc4MjNlYmFmMzY0OWUzNDdiMzQ0NTZmNTRjNzcwODEwYTgyMGViNTgyYTcyZjRiNzljNmRmMmNkODhiNjRlZTUwZWVhYTNkOWE3MjYxN2Q0NWNhMDM1MTExN2Y5ZGU2NjM1M2JiOGY4YmJhMjE2YTFiNzAyYWQxOTAyMTA4ODZmNGI1ZmY1NDFmMGQ5MDk4NGQ2NjA2MGY4OGU5YmZkOTQ2YzkyMDQ1ZDMzMjQwZTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OqZziDYcZI_Wq_a1wF875IG5uQjR9viTTd4hPBTD9OzHKsZYZBTEJY6L3E8VNsiiRoadLZBcoib94pJTl1FY0w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230708_102951_29_242d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.836Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkVPQkVZd2hjd2IzR3ZQd21jVHNJSGUyUy9laEFFWVJRME5NdTdKSmNQd2FhbjZmTnhPaGw4VXM5RnBnaTAyb1BpSlNsc0ttejdPdzZLU0pJQjZHZi9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwOF8xMDI5NTFfMjlfMjQyZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmZiOGNiYjM0YmJiZjkxMmExMGZjZTg4MzUwODgxYWVjZmQ5ODEwY2MyMGUxZTIyZWExZDE1MmM1MTFlZDgyZDFmYTljNmZiZGFhOWE2MDE1YzM3MWZhYjJlZGZhYTJiZGQxYjZkNmViOWY5YWEzYzg0MTc2NDAzMGQ5NGU4ZjNkMDVkM2FiZmVkMTY2ZmY3ODEwM2RiMjdmZmE4ZWE5YmNlYmJlODA5NGNiMTU5MWU1ZDc5NTYzOWEwZjUxYWM4YjQ5YzcwYmU3MWVjMjQxNGJiMDg0Yzc3YmZlOTdiNTE5N2YwNjk1MzUwMTZlY2EzZWVmODQwYTIwMjNlZjQxODU5NDhhOGY5MmJkY2YzNDMwZGJmNjNlYjcwYmM5MGE5MzU2M2Q5ZTA1YzRhZmM5NWEwZGM3NTc3ZjNhNGNmZTI0MzhiOTJkMDA5NDE0ODViNWVmM2JmYmU1NGE0YzIyNmVkMjk3ZGM2MjNlNTFiYzM0NzUwMTBiMzYyM2MzOWYyNGRkMWZhOGU5MzY5NTA2NDQ0OTliYjg4YTdmOTBkZDNiNmJmNzUxNTNkMzdiMTZkZmFhNjNmYjkzYWQ0NzdmY2Q4YjQwNDNkYjE5NTg5ZmNkNmZiNjVmNGE4OWY5MjllOGQ0NTk3MTMwZjQ1OTUxOGNhMjhjNzAxZjRmZDQ1NmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JXzEtqo7prPOk1WdaaMbFR2xsgeMcEjrOA63DHaLhuWkGUlmCykv0mw9WN1rSudZlwSysrH7l1P0cLINkeXxjg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230708_102951_29_242d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.840Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ii9DbEoycVpxN1FYOGpaSm1IZFg4L1FvYWRRZlVqSjd4UDNSc05mRzE4NEl0bEt2bXdpV3l0M3JpUW5xZFRWb3JjYXZWVkFQcjgzT05IVDhVOU5GOXRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwNV8xMDM0MzNfNTBfMjQ1ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTY0YTdiYzFhMjVkYTgyYWQxYjIwNDEzZDY0YzMyNDhhOGEwYTgwNDMzMjYyNWE1ODk2MThhOTQwNDY1OTFkNmYyYTU2ZjZlMGM0ZTk3NjNlMTM2ZDQxZjRhNzY3NzhlYzUzYmMxOWFmZDAzM2U1MWM4YTQ5MWNjZmUyYTk3NTQ4YWQ0MWQzOGU5MmVhN2Q4ZTQ4MTFjNDY4YzQxYzYwYjVhNDM5YzVmOTA5ODJkNDE1OWQyOThhMDcwYjYzMWNhYzBkM2MyMTVlNzQzNDJhZjJiMWQyZWY1NGJkN2M2ZmY4ZWRlOWU4NzMwYTIyZjgzNjhkMWQ0OThmNWNlYTliZGRiODNhNjVhZTViYjBjNzA3NDJjYjE5ZmNiOTYyMTgxZTgwYTIzMGE1Y2I0NjZhMjRjY2RlMzhjMjFkYzlkNmZjYzI4Y2U3ZDc1NzVjNjE3MTllODdmOGQyMTUyMDA0MGU4MGUzMTgwNTgzMDM1ZjEzYTY5OGRjMjZiODkyNmNiNWRlNGI4MWE2MWM4NmFlY2IzMGU1ZjM1N2QyMDI1MmVlM2Q4MzQ4MGRiOGU2NDExOWUzNmMyNzBjYzNiYWRjYmY3NDU0YTczNzI5ZThkZDExMjgxOGRmMTM1YTM2ZjVjMDNkOGUwYmFmNzBhYWM2YjIzZjA3YjFmMWJlMjM4OTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uMlfDDL6ANQvVAhZX6MK0PJ_GOjGjWirF5fPgEFvuG2dktFDh05YSG2aRlKB_ZQgOK0oXGO4PxaOq3JYM9AB8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210405_103433_50_245e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.842Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkhCYnV5MDRNM3pzOFgvL2FBcXcxS0QwdnV1UDlXM1hhaHdJUXEyTE1TT3VYOG5jL1ZaMk5wL0l3aDU3WWh1S2lvTEhwekN2NVlHNXVoYXNLd2tCMmV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwNV8xMDM0MzNfNTBfMjQ1ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjE1NjUyNjg5NzRmZjFkNjE2ZDQxY2Q1OGQ4OTg0MmI3Njg1ODQ1NmZhZjdmN2YzNzQ2NjkzMWMwNmY4NjVkZTgyNGQxODc0ZGVjN2EzOTRjMjYwNTcxOTY3YzQ5OTA0NzAwMWQwNWEyYjBhNDhmYWFjOWZiOTY5YmFiMjhlOWNlM2ZkNDkzMmRkZDhhODM1NWZkNmRhZmRkYzQyNTYyNGM4ZTA2NzBkMGM2Mjc0NGMyOTg5Y2NmNmY0MmJlMzNhMjhhZDI0YjlkNWFiYzRmNDJmZWEyMWVkYmIzNjBhYmQ3YmM5YjM0ZGZiMjNhZmE0ZmJiZGFmY2I2Nzc2OTAxNTMzNWM1MmNkYzc4MjM0N2ZjOWNkZDdkYTIxNTNjMDFkNzkwODI3ZWU0YTZjNzUyNDMyZDU1YTgzZmE4ZTc4NjI2NzQ0NDFhODBmYTMzNDU4ZjFiNGUwODQwYzhjMmIzY2I4OTJhMjViMTMxZjJmNWYyZTliYTAyYzIyNTUxMjdlYTAyZGRiMjExNzdkY2UxNmQyMzdhZGIwNjcyYzkwNjE4ODlmZjRmMzY3MjI4NzIwZDdlNTYwMzJmZTQxYzI5MDk4MWFhNDEyYmIwMDc2ZjVmYzk3ZTA2ODg3ZmVhMjhlNjg5NmUyZWJhOGRjNGM1NTQwM2MzMTMzMjM0ZmZlYzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-e2fvwMuH1Q9qTPrXmC7dIbSq68XcbM-JHISBSDrioqMv3-HUL6S5rm-sXZSfglUlNkhRZJ6JMxkuz5OVMbG7g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210405_103433_50_245e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.845Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImMweTNHZ1JuQTF5YkZERE5ORDNXNDZ6eUNjN1VSeWpBbGtjUVRqUEpiN1ZxYTd5eUZpcEdGNUFMWmZMbHRhN2J5dVRMRHM2ZHE5Z0hiYmlpc3RoTTVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwNV8xMDM0MzNfNTBfMjQ1ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2I5YTE4ZmYxNWYxZjYyYmY3NTY5OTU5MDdhYWYxMTJlNzI3YzA2NDgzNmU1YmQ5Zjk4NDhhYzdlOGY1MTFiY2UzMmRiN2E4OWEzY2YwMTllNGE1NGM0Yjg5Mjg4MDEwNDM5MjE4MzU2OTc3NmQxN2FhMWY3ODZkNDBmZjc3MDkwYWE5MzZhMDIzMGRkNTFiNjQyOGJmNjEyN2Y1YTBmYTg4NjY0NWM2N2FlYWMyZmQ2YTJhYzE4OTIzNmQxMTRlMDFjNDVkOWYxZmQ0Yjk5NzM5ODk1YjRjZTllZGU3ZWNjYTUxYzZmZjc4OWJhZjJmNDJiZjQ5NzM3YmE4MGI4ZWQ2N2NmNzIyNWZiNjZkOWNjYTNjZWVjODIxNDkwOTlmYjI5MTI0YTE0ZmM1OThmZDE2YmU0OTQ0ZWRmOTc1MjRmNjUyYjNlYjg2YmRkNzQ0MTBiMmUzOTcyZTVlODlhYzAwZmNhZTI5NjM0ZTdkYjEwOGQyMGE0YjRjNDZlMzkyYzZiOTk2OTdmMWVlMWQyYmU3OWM1NzY4MjZiNWE3ZWE5YTZhYjFiZGJkZjIyMTUyNzhmODg2MmRmNTY0MzEwMmU3YzFiYTQ0ODU1ZTYwNjE3ZGFhOWIwMTY3NDM3MDhkZjdjZGU2N2U3ZjZmMTEyZjMwY2NiM2NjZjdlMTM4Y2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tYKZNhzq8QUP6VJ33LX8q6d1XCehI-oY87Xhen7CxoQtvDldYiA0lNfgmPsYuC0qxd_fILKQ2zXvXbB_Ftj2DQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210405_103433_50_245e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.847Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImZUY0ZXMzhVTmhUcWNYUGNBM3lOMGp1SGVKS3JKRnFBM1VLQ1IzbTc4dVhaL1MvQkpxQ3hXcDNVVUVWRTBXK1EvT3NoRnk1amtBYVBHUVg0TmdHQUdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQwNV8xMDM0MzNfNTBfMjQ1ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2VjN2UxYmJiYmRiODQ2ZGY0OGRmZWYyZmRmMDk1NThmNDM4NWYwMDRhOWE0MDM1ZDVmYzljOGQyZjVjMTA0NDdjYzQ4OTk1ZWY1MzdjYjJmMWZjOTBhZTMyYzVhOTM1Zjk3MDFlNzgzMDczOWEzOTdjMWJkYmY3NjMxZGJiODMxZGU0ZTkyNWJlZGMzZjRiOWUxYzZjODNhMzczYzI0YzFkYWZjOWFjNzgzMTk2OTJiMDJkMDVkMzExM2FiNmEyNmU5ODY3NzRhYzBiYjA5YjNhYWJmOTdjYTE0MTA5Y2VjZGIzMTgzYmFjYjRmYThjOTA5ZGRkZjY4NDA3ZjllNWZjZjYzOTRjMGVjZWFjYTcyYmVmYzZjNmIwMjMzN2I1NWFmMTlhZjFjOTZlZjExNzFkN2YxNDU4Nzg3ZTNmMDFhMzA2NTYxYzkyMGE1NTliMTc5NzdkMTU3ZTE0ZjgxYzFlMDUxODc0MDJkZDQ0ZTdjYzI4N2Q4YjE4ZjRjNGUwYTgxMzQzMTg5NmZjOTQ5MDZhMDU3NThmNmJmYmI3OTE3ODQwZDUwODQ1ZWI0NzIyMWE3N2ZlYmFmNWRhOGVjNjhjNjAwYTg3NWExOWJkNjQwOGE2OTdkY2FiZmVhMTkyNzQ1YWFiZTM4MDEzNzNhNzBlMDI2MjJkZjBhYTQyY2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.53w_2lsibw_Zje0VA_NA3hHOfAyFvaBqHxiBufOTVjVMY5E95-pD9sP_fQnifILnJWBCunSlvX4Ehg6PBi4RRQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210405_103433_50_245e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.850Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjErbkExcFpTZXBGQ0p0aHNXMDFucStaNmpOMFdJZzhoQk9tSXpjYkdsVVVadFk3YmorUDE0dW5DTzUzSFdZVFFEWUtrbUo3YWw5RExJc0J5c2pKZXJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNF8xMDIzMDhfOTdfMjQxZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTc3YjQ4YjEwYzA5OWNkZjIyNWNjNjJmNDllNzUzNGQwZDY3MjNkNGI3NGVhNmY1NDdlMjBmMjM2ZGJiMDIzYTg4NzAzY2ZjZTk1NjQ5OGYxYjkzMzFlZDFmMTVmZWRmOWI0N2VhNzAwMjY4OGIzYWJjNDU5Mzg1Mjg3MGI2YzFlMzJjYjk1YjdhNjkwMmIwY2Q0ODBiN2FjYzFmY2FiNThhOThhNjkxYThiMTFiZDIwNzUwNDFjODc2MDE1ZTdjZmNlZjQ5NjEyZTJhOWRiZGVhMGY1NmYzMTI3YWEyNDFmOThiOTZkZDNmZjYwNmYyMjdmODQ4NzA4MGZiZjA0OTFhNjU2Zjc2ZDA0YWU1ODFkYjMxNzYxMmZlOTE1Y2JjM2UzMmUzZmRhMmM1MzllZDNiODc0MTdlY2JjMWFkMzlhZjU0YzkyMTkyNjJmN2FjZDhkYmQ0MWFhYjkzM2I5OTczOWZhYzg2Y2FkZTllYzVlNGU2ZWFjNTM3MWE1OTQxNTQxOTcxOGNlNWE3NmRiOTVjYmI2ZTdkNTRmYTk0ZDdjOTIwYTJkZTgxZDRiOGVhZmZiMDhjZWYxZTAxODg0YTU3ZDRkMjg2NjNiNTI0OTU2NjA1YzFhMDdiZTFmZWYyZWQ2MGEwZjE1ZDU5ODIxNDhhMjI5NzNkZGY4YTFmYWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7UW3NumIxqRJF5QfmXmMbSefteOmCqDU5On8cgWWrqT2wArKYsmSDTnKdNcH2q8hRLM0bykctP3WenXV80dUwg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230224_102308_97_241d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.853Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik1uOWJ1bHA5dHlnZ1h2RkJkNk92c1I0T3RKS2NnL0FCOHA0WElPY21NTmFIZUR5Vkk5enZ3Ny9SWmw0OWhNRkowSU9nY09teVdhMlRIR0RyM2pGSVNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNF8xMDIzMDhfOTdfMjQxZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTgxMTI0ODM1YTNhYjkyMzE1YTg4MWU3OTE2OGExMmQ2N2UzNjQwNjdkZDcwYmQzOGFjNDVjYTA4OTNkOTE1YmNjNjRmYWZjNDEzMThiYjIwZDRmMGE2ZTNhNmQwNzZkYTlmZjc2MGJkNmU2ZGQ1YmVhMWY4YTEwMDVlNzQyYTlmM2ExYWU2YTk2ODAxNjI3MWVjMGIxYTRmNjljOTMwOGY5ZjkxODM3ZWZlMzJlYzIzZWUwNWJjZGYyNDY5NDVmNjdlYzU0OWIwZTlkZDQ1Mjc4ODM4MjAxYTU3OTY3YjAwNDU0YWExNmVhNGRmZWQxZTBkNzA5ZmRhZDFhOGQ0NjExNmRmNGRiNWYyNzNjNmJkOTc0Mzc2N2NjMGEzYzNiYTlhMWVlYTY5ZWQ5ODZiNjRiNDA4MmM2MjMxMDBhZmI4NTlkNzExNGU1MTBkZDhmN2I2NjYwOTA1NjU2NmMwM2Q4NmJjMjQ5ZTIzOWZmY2ZiMTVkNTBmNjlhMmE3NTE5ODYwMjk0Y2EwYWQwNGVhNzJhZDhlOTQxN2U5MWViNDllMWYyZjEwNzMxMzBhMDVjZTU0NTE3NWQyNWJjYWMwNTgyMWU3YTIwZWRlMDEwMzQ1Zjc0M2I4OWYyY2I4ZjA5YjI3MTI1NDJlMjJhYWY2ZmExMTk3NTc5MmNkYTFjYjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wnTwyGI130I0x7OSoc2LQHYKEWOFVjNNpOkFDlJMMnK3mzmH1JFpOkNGTMfHW5OvALzUpBlOr3AZS2dr-XPL8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230224_102308_97_241d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.856Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik5WNERDU3FSd3hXbk9TamY2QVd0YTVDVStCaWJzTGY3eTRVREdLYjN4eHQ4R3VjU2F3QkY4UHpXcndlVTAvNmhKaXdhTEErOEV4MEJqK2w5bnlMZmFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNF8xMDIzMDhfOTdfMjQxZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzIwNmU0YjFhNWY4ZWQwMTA1NGI1ZTJlNDViMTQ3M2RmOWFmNThhNWRiYmFhMWJlYmMwMmQwMWY3MzNlYzY4YmUzNDdkODVlNTViMzY2MWY4ZmVjMTc4ZGM2ZjM3ZmRlZTA5YTM3NjgyMDQyNWQyYmU3OThiOTBmNGM4MmYwZDMzNGUyNTZjZTM1YmJmOWZmOTkxNzhmNDk5M2MwZjA5MzAwMjM2YjM5NGNlYmY5MWYyZGZmZjkwMDUxY2M4MmYwMjE2OTMwZTkxNzk0NmY0Y2U4NGVkZDE5M2JhNDhlOWYwYTM1MjM4ZDM0YmJmNmI2NTI0YzJhNTRmNGNiZmI1YTQ5N2MxNWM0MTRiMThiNTA3NmFmYTE5YjZlNzdjOWNhM2Q1N2E0YmE4NDY0N2QzM2QxMTA4ODhkZjViMmJhYzQ1MWRkNGI1NTljZmRkYjA0YTA2YmRkOWI2YWNjMjk4OTNkOGYzMzA4YTZhMGRjN2JlYzQyMmVhNzFlNTUxNGY2ZmVkZmRlNWYyZDkzZjM2YTQ2M2E3N2E4YmFkMGRhOGFmZDgxNmQ3YmQ5MTZhNmFiYmQ3NmIyN2QwNTBjMjBmNjczODU5ZmQyYmIwY2VhZmI3NGEwODYyNWE1NmJjZGY5YmIyMWRmMmE0Y2IxZjlhMDQzMzQwMDkyYjFhMDdjODhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7nX-1QfGt1zPF11SRfIV-5C3zO8_AfntnXI9YfCZ437VktJMrBFErbJjyjz7sR2lDI58calwasvxo2s5UxYnfA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230224_102308_97_241d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.861Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjE5Y200OGtmUUszYm16UHZ5VFVSc2JXTGpPbWthMGtrclgyc3Z5VHdDc00rT0l1TGp3SG91emRtYjI5aXo0a25aMnhSMFZlZ0JRWk5uRTZoSUtSS3dnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIyNF8xMDIzMDhfOTdfMjQxZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTEzNDIyZTAyNmMyN2FlMWZhMDhiYmRkZGUxMzUwZWY4OWVmNWViNDNjZDgyNTU3MDJkMjIxMjRiMjg1Y2FmZmY1NTFkOWQ0YTE0NTRkODY1YmRmMTY4YTJmODJkNTdkYWM1ZWZmNWVhZjJjODRkYzhlZTQyMjE5Y2ZlZDExN2Y1NGRkMjIwOGMzYTBkODU1ZTJiZmNmMjkwMjIwYmFiZDc5Mjc5MTJlYWFhOGNmOWQ4Y2IxYWJiZTJmYWMyNmE4ZDYwODBiOWZhMzMyMzlmOTUyZWI3MjE3NTljZGUzYzVkODA4MGY2ODkwNGY4MDE0YjI3MWEwYzgzOWRlYWMyNGM3OTExZmU1OTEzMDQ3OTNlNjA5OTJmZjRiYWM1M2JjYTcxM2M2ZWViMDc2NDM5ZTc3NjM2ODRhOWQ5ODZkOTllNjZiMTM5ZmNhYmVjZjc0Nzc1MjgyNDc5ZTZmMjkyYjQwZTgyYTIzYzUzZjI1OGYyZjE2NWM1YTg4NmU5YTdkZDJlMjU2MWQ3ODA1ZjhlNTRlMWYyOTU5NGRjYzk2MTUxMmEwZjZiNTc3ZDIwYWMwZDEwODdmOGVjMGMxNTRhODdkMDUzNmY3NjJkZTU3ZGYzNGZjZmJmMjQ3YTRlNmJkNzJhODI5YmE4MTMwNGM5MzNmM2QxZGFmNWMxYzJkNWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PXRPZSqt8VU_kozF-59R-9DqTezNBMy8myAQ8EZZSY9BR3VbEw1JD1x2ZyiGkprnsx-2dPq4NiVcQX1KOZUpTQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230224_102308_97_241d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.864Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjBTUFE4eXIzaitQT0FiNEE2T3JpK1cwWGtMbllvYmVVQ0hwUDZFRDlOZ0JrYXNZeHNtZ1ZFUjlLR3Fub2dpbTRUQTVESVlzWEJ6VjBHQldoMktrM3JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzNfMTFfMjQ2NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVmN2VkN2ZkMmJjMjU3MzEwNjZhN2YwZDM2ZWViZTdkMTQ3MzcwY2FjYjI2YzRlNTcyZTdlZjA5YjA2OTk1ZTVkYjFlNmExMTFiNDM0NjA1MDkzNGM2MzczMjkwMTJkOTI3Nzg4Zjk1OGZiMzU4ZmRjMTlmNzlkNjNlOTRlZTE0ZjE3MTE5MzNmOWVmNGJhYzI5OTNkODk2YmQ3NzNkZTAyM2UwYmVkZmI5YTgyZDFmMDVlZWNlYTdjNGQ5MzMwMmQ4NTM1M2Q4ZDAzN2M5YzhlNWRjOTU2ODdlNzgyMThmZGZmYmRhYTNhMmNjN2JkN2E1ZmFmZTdlMTg4MjVjZmE2NTFhZTUzOGQ1MjIyMjRmOTk3Y2I5ZDMxMjdmMjY3NzFiNWM4ZTRiYWJmODM5YThhMDMzZWVmZmQ4NmRhYWViYjYxMGM3NDQ5MDc5MGIzZjU1N2ZlOTk4ZWNjZmZiYjQ3MmIxMjNlNmRjMTNmMzFiYzQ5ZDU5NDllMjRhNDI4Y2NjNzc1NzRiOWY5MjY1MWE5NjRmOWQxMzIyNTY4NTM5MjUyYjkzZGMzZGQ5NzIzYTc5ZjU1MmYwY2JiYjliMDg4MTM2MjBkYTUzODhiMWY2ODdhYWZkZDliNmViZmQ2NTc1YzdiM2U0YWQxYjI0OGQxZDY1ZTY3YTViNzdiZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Gi2zMaSsJKRcSqrPsyZlcAuqH622MJD8idzn_4A7xUCAtaTiDwUMBjiciqfucvG_90du4UYSd4ZjMewg4ERz9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102433_11_2465_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.867Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InVUUWo0b3VDaFltTU5wTWNSTm1Fa0xSTHdobkYrVmJvaTBReXh1dTQxSlEyb0RyVFhmQlUyU2ZQMmxhblBJUFBQcDhqT3FXaXl3TngvWlJ5YWJiN0NRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzNfMTFfMjQ2NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzk3ZjQ3MDljYzhmMDFkY2MxODE4ZDRlNjEzYTM2MGMwMzdhYTM0ZmJmYWRiNzNiMmQwYTE4MmNhYzI0MWE0OTg4NTZjY2ZjNzY3ODZmMzJiYTBkNzVjYzk1YjU0ODdmNWJmNDYxOWZkOThkOWNjOWJiMTlhYWNmYTk2NzVhYjE4Yzc0MTZlOTQ1ZGVjY2M4ZjcyOGRjYWQ0NzNlZjQzYTU2NmIyMDdkNDI2N2QzOWNmNDI4MWY5Y2IyOWFkNGMwMWM5M2Q4MjY4MzM2ZTc3ZGNhOTVkMjU0MWJhYWMxNWY4NTg0NjIwMGRlNWY1ZDEzN2M0OTZkZjZkYWNjMGFiZmY1ZTNmMWI1OTNjMzJiMzg1NDVjNDdjNDU4MzgxOGY0YjdiYjliYjY4ZDI3Nzk1NjhmMDdiMzg4MTNiMmU4OTAxZDRjODJlZTAzZTJkNzNkOGZiYjA3ZjIxNTU3OGQ3YzAzNDVlYTQ2MWQzZjM2MzM5YWQwMzg4MzVkYmQ0ODg0Zjc1ZGRmYjY5Y2UxMmIwZjJlNGIwNDlhMDEwYzMyZjY5NzBkZTc0ZDYwNzBlOTk2M2UxNjQxODY1MTk0ODQwODNjMmEzODgzOGQ0N2Q2ZGUyMzAxMjQ4Y2Q0OWE0ZDQ1NTRhOTEyN2I1NzA3YjRhZDU4M2ZiMGE2N2E0ZDcwZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aDN31wa14qRHh5lUDqt8IMWEmevV3mVUxvmF_4jeXAv3elRdI2Yb6BZu1EF5a6IPGC6KgvVNXredWwKS9vWtdw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102433_11_2465_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.870Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikd4ZHZOY0pJRnlaWUdnZkw3SHNXSHMzMTVmOW02RmJNY3d2eGtiZU5Db1doU3JKMmJ2OGtCSDRyUmFvZFF2aGFOaE9WaWFFd09GRGhJTGt6dFdyMlB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzNfMTFfMjQ2NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTlmNzVjZDQyNDlmYjYzMjE0YjJjN2Y3YzFiMmVlNjBjOTY1MDJiNzYzMDhhNDQ5N2JjODljNjM5NWJmYjZlMDc5Y2RmM2U1YWEwNGQ4ODc1YTY4YWFiYjIyNDRkNWI5OTU4ZmYwYjBlMDYxMjgwMjA2ZDRhOWQ3NTRhZDU4YzExMjA1M2Y0OTllN2UyNzMxMDM4Y2U5ZWU1OTQ3OTdhMTkyNmZhOTgzOWQ0NzI5MmM0NTM2ZDliY2EzOTdmOWY1MDEwYjRjMzIzYjg4YWM0MzBhMmYwZjBiOTY1NjQyYmM5NmJjNmMwYzc0NGZkNzRlYTMxOTI1ZDdkOGUxYzU5OTAzYzIyNmE0NmY1ZGJkNGRhNTc1ZmVlM2VhMTE5YmMyZDZlZmQ5ODA1Y2MwMmY2OGMyZWM2YzdmOGE1MTliZDg2OGUxZjY5MWIyYmEwOTk3N2UwZWZjMjY4YzgyMWM5YmU4M2Y4MDkwOWZmYmE2MTQ3NjkyNGIyOTFjNTY4YmM0ZjM3YWNjZTY5YWU1YTc1ZGMxOTc4YzdiYzcxZTY2MGQyMTJjZTM3YjRmYWFmMzE0OThiYjM0NTc4OGJhZDNkZjU5OThjYTZmODBkMjM5NDY4OGVjNTQ5NWQ3ZTUxMTdkMTZiYjFmZDZhYzE3YTcwYzY0YzM1YzY4NzM0ZjQyN2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LlpAUU-rKlwrCnGuVM7_pwUsK6AfR3nJ8mkX0FuNsBcQqMbnm2qU93MJd3K4ZWNsQYvDfFHad885zYwxWHEhgg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102433_11_2465_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.873Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InhCbFp4cFV4S3k4UlMrQnFUWWRrS2YrMTRXaTBhUnNqVGdBNHprVUl0T1l4QkpyNlhEZmpzMTZPUG1seTRnamR0T2MyTkw2QmsyZEtPYVhSc3MzTWl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI0MzNfMTFfMjQ2NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzQzZDQzNjc2ZTc4MTFkYjE3OWRlZjFiMmU1OTAwMzljMjg1YzZhMDk3M2FjNzRjMWVlNzc4NWQ1YzM0Mzg3NDhhYjA4NWQ3MTNiZmUwMzhhOWI1ODViOWY0OTVlM2UxMGJiMmNlOGRhNWFhYTdiN2IyMDcyZDI1OGUzOGQzZGY3ZDdmMTdhOGRmZWUzMTViYzIyYmE1YjEzYjA5NmI1ZTMxNzYwNzdiOTI2MjgxYWY4MGI1ODUwNWQyNzdiOTIxZDkzOTg4NWYwMDVhMjIxZTFhMjhiY2QzZTI3ODMxZTgxNWJmOWY2ZWNiYzNkZTBjMDFkNzZjYTdjMTBhNzEzNTZkMzQxMjgyMzgzMjlmZjY5ZmJhMWQwZjMzYTJmZWMwNmM4YzM3ZWU5ZWI3MDI1Y2Y5ZjRhNjM2OTRmMjg5ODYyMmMyMjQ4OTFlMTI0N2NlOWVhNGQ2N2E5YjBmNDhjNWI1ZDdhMjU3ZDczZTc4NTM1OTEwM2ViOThmOTZmYmRkMTIwODIyNzNkZThhMTkxZWYwZTU3NmY4YjgxYTA3Y2MyMDM5ZjRjNWYzNGRjNGY2ZmEzMmUzNzJiYWMxODg5YWIwNjlhNDIxMThhY2FmYzRkZDgxYWFiZTgwN2ZhNDU1NzJhZTdjZjMyZTc0OGNkYTMyYzhhOThhMDI3ZWQ1OTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.car9TuBRUBjfk5Y3LLOyhnvFvZ6R-2CvTGscfLdDn7QaK00_eI1ibc_gImeoyORLaGjVIO3gVm98_9dN1GSjHQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102433_11_2465_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.877Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlkrSjRmaVY3Q1M0WTMzRU9sWkd3Z3RMdEpYWHBneW9pam5qbjRYY2NUS1lRRmI5Mi8vVnJXZDlIOTFHQmtZRFRSdnh5WWN3RXo2SWc5dXBEcXBFb213PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDZfOTVfMjRlMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDYzMTJkOWM5N2E5OTcxZjI3YjUyODg0MTc4NzNhNzkzYWE2YmQ2YTQ3MWFjOTM3OWI5ZWI1ZDU2MTQ0NTY4ZmY5MzQ1OTcwMzRiNTNkMzI4NWUwMTg3MjY3YWVlMTc5YmM5NDQ4ZWFjNDBhZGE4MDZmZTIxMDVmMWUzYzI2MDY3MWFjN2FkYmYxNmE4OGE3NGIzM2U1MzExYjBhM2JiMWU4NjlmNzUxYzNkYjU5Yzk4NmY3NGE2ZDdjYmIzMWI0ODhhY2NmMThhN2EyNWE2MmZiMjAzM2FlMjg5MGUzMzA1MmYwOTcyOWE5NzA2MGVlNDIxNzExYzZhODk1ZjQ3Yjk3NDUyZmM2ZjcyYmVlNDk4ODYwOTExOGJjYmM0ZmY5MmFjZDM0MmZiMDhlNzU5ZjIxZTg3OWVjMmQ0MTRiNTVhNGU3ZjZiNWFmNzE0YWM3OTJkODIwNTcwNzllZTFiZGVlMTU5NGI5OWFjN2VmOTM3MzE0ZGVhYmRlOTlkMzIyNTM3ZDAzOTJlNzEwYjg5YWYyZGVhZTJmZGRkMmRmY2U0NGRkNTkzZGU5NDA4NGY0NjI1NzgzZmEzODIxZTQzNzI3MWMzNjY5MmMxMTIzZjFhY2VmNTY0NzE0MTM2N2Y1ODA3NTE3YzQ1NDMxN2VmNzNjMmFjZWU4MTEyZWE5NjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xqZHFF2vYX6-btHD6LapiO3e0vTzPBCH-vWIg66PPhAEs6qmES5AGGP5wdpjGLaUj90LuGzJAm_RlUhjC3OxjA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113446_95_24e1_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.880Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImtSdnJNNzVkeUQ1bEdVaGNGc3Y1SzdGNHFFQVlkb2tENUwyblpwRzBuTjJvU1ZJUUVqMXNKcVhLSThzNnZqVHdlK0I3K2RxN3JLa0MrcFdUeVd5alhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDZfOTVfMjRlMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTczMmZjMDlkYWEyNDRjYjc4OTBlYTcwODdiMzQxZDYzN2JjOWZjZGE5ZGI0MWNmNDBhZGRkYmQ1YzkwOGY0N2MwMzdmYjYyMDUzODJmMDA1YTdlZjQyZWUxODBmYTU3MDBmMDMxNjY0MzZkOTc4MjcwMzk1ZTZjYjdiYzYxZDBiZmQzMGIyYWVkZGQzNzAyODgyMDY3Yjk1NDllNmJiZjdjOTFiMTQ1OWMyODcwMWUwMDQyYjI4ODA4M2NkMzQ1YTBlZGVhYjY1ZjljOTM3ZDUwMTUzMTQ0NzcyNzU4MjJkMTBkMzkzODJjMzBlNGM2NzE1NTk4NDExMDI4ODRlYjFjNjY2YTgxMjk2YWE3NzMxYTIzNjkwMGE3NzNhMjEyZjUyODVkMWVhYThkM2I2ODRlOTE2ZjdjZjA0MWZmNzdkZWNkNGE0NmQwMjBlMzdhNTU3ZmI0NjllY2QwY2JkNGI5NTYyNDIyMjYzODE5MTJlYjc5MGM1MzdhNzU0NzczMDQ0NjhhMGJjZjIxNWNmYTQxZGYxNjBkNGIxNjcwZWY0MDRjODhlMzBiN2E3NDZmZDA0MzkzMDI4OGQ0YmVmM2MwYmZjMjAwN2M4MGY1MTJiZjRlM2I0YzFiMzZiYzFmZjI4OGUzOGNhOTllZDJjZjFmYTFmNGJiMzhhYmFjNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CzMvc_t0OmBWR6NuvEZv9s6UuQ3WCMxjc5_2Z-7j4eVoHtT1xhI4mmMmiPN_iRBFLmMJs2K6P9hgIUIjSMLsGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113446_95_24e1_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.883Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlBiN3c0TGxQMW5DT1A5K2NPNVE3Z05JNWc1cmx1Y0tTbW55QkcxdUlDUFdXWTY2TVZBR014SWJYOHRwb21SZUxkeTN6bnJBbTFvZmZla1Q0Q0JYbFZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDZfOTVfMjRlMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDA3Mzg5N2M4NTc1ZTJlMTI3OGM0N2E5YjcxM2YzNTgwZGU4ODgwNTQ0ZGZiY2UzYTBiNjEzZDZiYTAzYTE3NTJiM2UyZWZmMjgwYmRmZTU5NjdmYjc3NDcwZDBkMTY5YjdkNzJmNDE0NGVkN2E0MGRjNDkxMjc1YTcwOGMxZDFiNzM4OWM0NmRjOWUzOTFhZmIzNzkwZmNhMjM1MGQ0MWJkNDYwOGI1OWJlZGRlYmI3ZTA5YjIzNWY5MTYxOGZhMDg2YjEwMzc2OGUzYzhmMGVmZDc4MzFlNDZjNmU1OWY2Njg2MzRjM2U5NDMxNjkzMzQzMDUwNWY2Mzg2OTEwODE1NmIzODNkMzFiZjcxMzNlMzFkM2IzZTNhOWExMmE0ZWFmOGY0YTBjMWFlNGQzNjY0Y2VmOGI0YWQzYTc2YThhMjQ5MDZmOTZmZWIyY2M3ODFlNDJlYjJhYWVmMjYyYzY1YTQ3ZWVjMDEwMjI0ZDhjYjFjZDI2YmUyZDNmZGE5ZjA4Y2IyMjZiYTE1YWUxYTI0YWUzNzQxNWE2ZTZlYzBlMTRiMzRhYmVjZGI1Mzg1YWYxMWQ0YmYyZmU3MWVhYmQxZjBmMTllOTQ3MjZjNDQyNjcxMDUxNjFiODM2ODhkOGI3NWJmMjc0ZTczZDllYjIzN2Q4NTE2YTIwNTUxOWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xKycBAIB9FY29W-E0uTqd8TKYeWKsMonWOXrBsDAzdH-crq944oVPI_Y6BScL5I-te7Nq0DiFwMXbfhPT7SoAA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113446_95_24e1_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.888Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IllTS2ZvVzMyVVU1NkVQcHMzbW1ieHpqOFU5SEtycnJvVEtwVXF6MHBLdkUwdnRIdzdUcTJadzdNaG9FZHh1WS9qZU9sTFd2LzJmRjQ2eUdHMFkzdzl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDZfOTVfMjRlMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDc3MjNmNWI2YTM2MzM0NTZmOWUwOTE2ZTExMzI4Yjk5YjBhOWViNDIyM2E5ZWViZTk3MThiYTE0OGIwYzIyOGIwZTM1MzQ3NDdkY2FiYzBkMTdkZTk4OTc5MTY5ODE5YjRmMzhkNmNlYTJhMDJlNGUyMjc1YjcxYTliMmVjYTRhZTUyMzc3Y2JjY2M5MWUwMDdhMTc4MmQ4MjBhZWE0Mjc5ZGMzZDM3OTgxMjQzZmZmZGFmNDVlYzdiZjgzYjU2ODNjY2YwNzE5OWI1YTJmZWU1MWJmNTZlMjgwM2ZiNDc0ODMyNDZiYWUxY2YzODQzYjFiOGQzOTE0ZTE1OGIwMjIzOGM3ZDIyMDQwMDZmMjNhMmY4M2ViNTI3MzQ5MTQxZWQ5MjQ1NjdlYjI3OWQ0YjJiYWVlNGFhMTE5OGJhYzM0MzZhZjBkMmE4ZDkzZDhhNzNjZjhlMjk2ZDJkZmY0NDZkMzE2Nzc2N2Y4ZTNlOWE0Yjk1YWZlNmU2YTFiNjQxZTQ5M2FhNzJmNjM0NGViMDE1NTU3YmVmYWNiZDRjOGNmYTVhZDUwODc5MDFkMzhiMjE5Y2RlNzMxMWEyZTZjMTlmZjJjNGJjNWZlOGQxNjRkOTlhNTA2MDBiM2I4MjRhNmFmZTZmNjRkODY2MWQ4Yjk0YTFlMGVlYTQ0NWUzNGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ku4Oq6567lom5lVA-vxY2ocVBk8v4mBFEWEbF57Mzrlt8OnQTcNEPcHKTksu48OEriGnZHUivvq9ANUWSrEfwA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113446_95_24e1_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.891Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InVCQ0dVVUwzQjFML3RTZUl3U0o0TU5LUk5SckdaYzRCYThZTjkxSTRORUsyL0I4SFRtWFZQM1JiT2tyQmpub2VScGlnTUs3RTEyNWNFRmVZTVhJWHdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDlfNTdfMjQ3N19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWQ5MzE4NjRiOWU3NDBiZDhjY2RjNTFlMDNmNjJhMjZjOWY4NDgwNWViMmQ1ZTZmNTYyMmE4YjM1ZmJiZTY0NGI5MjNkY2VkMDdlNDU0ZDE4ZTEyMGUyMmM1NjI0YWQ5NGQ4MjZlZDhmZWIzZjk2ZjA5ZjhiZWYzY2I0Y2JkMTg2ZThlN2Y3YjkwYTUyY2IxYzJjYjZiZjMyY2JhYzc2YjU3N2FiMjUzZTViZDNiY2VjNWY1NzU2Yzc4ZjlkMGY1MDNkYWMxMWVmYjk5M2NkMGViNjA0YTgwZWJjN2U0MGE5Y2EzZjFlMDA5ZTAwOTI3YWIwMWQwZjM5NTQ3MTk2NWRmY2I3MDcwNzAxZWQ0MGZlMzg2M2Y3ODMxNGZlMWZiZDY1NmZmMzczMzA1ZWUwNmFmZTlkNmRkYTE4MTNmZDc1YjM1NzVlNjQ5YTljNWI4OWNjNzU5MTkwMzZmNTNhNTgxOTk5YmZlMDE1MDhiNTNhYTc1ZmJhOGMyNWE5MGZmNGU3MDE5NTVkNTNiNmY5M2RlNmQ5M2U5MjVjMzg5OWIxNTFmZjI1ZGI3NTQ3YTk4MjI3NWYwNjlkZDY2ZWU1Y2UzODlhM2YxY2RiZWEzNzJlMDBhNjJmY2JkODFlOTEyMWExYmFjYTRkNTVhMTUwODM2ZDM2Y2NkYWNjYzM3NGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.43zIzuEvee_mO3YKKt9nj2tfpP6ufvjEYmesqGUl0Kj5PmHIHYZuM4p53YXQEKtjA0DTuo0L8wBQv4sYa8EigA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110649_57_2477_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.894Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJ1N3ZLSDlHcmVqcDFkeThYbUphSXRKUlQxNXNGSlRDbzA0Z2VPNW4yTkdNVnozRjNyNUR1VFhJSU5nMWsyNmxQTEZFbmMza1VXV01NRGY4VkxpN0pBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDlfNTdfMjQ3N18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGYxMTI5NjdkNTdjMDdjZjg3YmZjNDE3MGJlMWI2YzVmMmYwYjZhYTY0ZDdhYmVlOWVkZTliYTcwOGI5YzY2YTM2ZTc0Y2FhZDZjMDY1NDQ4MWIzM2ZlMGI2MzZkYmZjN2U2ZDE5OTM0ZTQ4YzIyNmJlOTA5Nzg2N2VmMWI3MzA5Y2RlZDE1ZmRkYjcyNmM1YTNjMWUzZjAyZmJmYjQyNTk3OTE5MGMwMmU0NzVmYmNlN2E2NzE1YzIyZWJmOWQ4NzVkYzljMzU2Yjc2MDAwZDAxYzVlOGIyZTZmZjdhYWM4YzVmZjM0MTljODFmYjNiZWM0MTQ0OGRhNjEzYTM2NWM4ZTA5OThlMjY1Yjg2OTA4NTlkOTU4YWI0MzFiZjllM2JhYzU0ODc3OGNiNDc3MzczZGNhZjY5NzYzOTA1Njg4ZDIwMmY2N2Y4MjFkNGMxMTY0NWRhMjFmOWY0NzEyZTJjNGMyMTVmZjM1Y2ZjNTE2ZmNkNmUyNWIwMmM0NTI0M2Y4ODQ2ODQ0MTgxMjQwNzk2OTk4NmRlYjkyNDczNTAzZmMwYjU5OWViNjc0OWRkMzI0NDEzMDQ0Y2FjNmE5YmFlMDZjODYzMWZmNjE0ZDRiNWIyZDFiNTQ3ZDAzZTg3MTQzNWVmMDUzY2E1NDdmNDE2YTQ1ZGNhZDJlZTg3MmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.g9mNvdqxa4wm8JQ32g7FWS61dYzAmQpyabMD2Y0nCdz0y3vNiBGv3LdyyOu-QdvwaRM6Al7hui0mw84YYTt0iA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110649_57_2477_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.900Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikkyd0pIcVpNdk9OaW9hV2N6dGNabzRUMXM0NFArZTA5ajZmK0gwOWUxZkZPazZJWVRQQ1VIMHpxTVJuQ1l6dTB1UUFqTTl3anhvTzg2cmEzYi9zRXdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDlfNTdfMjQ3N18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTE1NGVmYzczOGY1NDRhZmMxY2YwODc2ZjEwMGZjY2FhY2QzYzczYWVmN2M2ZTE1ZmIzNDI0ZGVmNzgxOThiODFjMTk4ODA3ODZmNjQ1MDA1ZjY1YTM2ZDIwMDg3NGQ4ZTdiMTBlMzk0MDU0ZTEyOGY0MjJkY2I1NTNlM2JiMDQyYmY2MGNmOGNjZjg2ZGEwZTg2NDAyNjM5YWViMGY1MDEzMDM4MDE5M2M2OWU1M2IzMmE4NTgzMjhlYjQxYmMyY2Y2YmEwYTI0YTM0OTc4MzdhYmEyOTBmNDU1M2NhNzFjMDQ0MDBlZDJiODE5ZGYxZjVlNDAzNzEwNWRhMmQ3ZjlkMGFiYTJhY2NlMzVmY2UyNzFiMjczNTQ1OWFkOGFiNGFjNGE5ODg1ZjAwYjRmNGYwMjk0NmM5ZDk4MmZjZjFhZTM3NmE2NDQ5M2IwOTJjMzAwNTdlNDczMzg3NTY1MGY2NDMxYmNkY2I5ZGVmYjQzZjI1OWMzYjRjOTc2OTlkYjA2NjczZDNhZmY1ZDYzMWUyZjc2NWZiYWVkZTg5NmMzMjk0YmYwYWM1MWQzMWI0NWEwYmI1ODVhMmMwYjc5Y2Y2ZjQzMmY4OTYyMDM2NmJjNjA2YWFkYWE1NjA1OWI3ZWFmNjAwOWVkMzUzNTkwOThkNGNiOTM2NTAxZDdlMWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.d8EpcbifVMMYSMZv5-jUft-MgwtBf3tVuU4DZsoxTgPKbwQpdrrw_vKG-s5MxSRqMpA3VJew0SSYr4g2Nr8DrQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110649_57_2477_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.905Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IitMNFArbVpzZXFTa3NaUXlPdnk5UklBdDRTZnpXWXVza0RFRjR0R1pnaUFBUVRmbjcxZEYrdHlaeURNUmptVVZmcnF1MGJxN05ESk4xTGNDcWFsemVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDlfNTdfMjQ3N18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2Q0YWVjNTQzMjJjYTFiMGJmYjJjODEwZWQwNGM2MzEyMWVlZWNmOTU3YzBkNmQxNjIxZTZmNDM5Y2I3ZDQyY2ZhNzg3ODA3ODA1ZDAyYTc4NGU4N2I5NTI1NTQwYWYwNmEwMzcwNWZlYmQ2Mzc0NjAyNjQ2NGM5Y2M2MzA0ODQ5NWIwZmRkMzIxN2I0MTM1ZjkxMWMzZmY2NDM0YjIxMGJkNjkyYWJkZTI3OTIxNGRmZjY2NGYxZGM0Y2RlZmM1ZTM1ZmNjMWVhNjlhN2I3MGNiYzcwYmNiZTk3ZWI2YTIwMDJiZWFmNDUxZjA2NDA1MmNmZGI5YzdlMzE4OWMxYjJiNWUwMDY2YzBmMWNhNDQ1YjY2MjgwZTY5Yjg1YjcwMjkyNjM4NTZiYTRhNTFlNTdiOTgwMWU3OTMyM2QyNjQ4NWZmMTJjOTc2YThkZmUzZDIyN2E4M2FhMWIwOTI3OWY4YTczMjIzNDQwNDYwMjY3NjI4NDdhMmFkZTQ5NGM0ZWE4M2Y5MzJlZjQ1YzkzNDRiYTZlZTUyMTBhNDQ5ZGE1ODBlZjFmZTU1Y2YzMTA2ZDFkNGExODhlNGMyZjQzMzkxZTMxOTcyMjVhY2MxYTBmZWJkMzBkYjhjM2NkZDgyYjI0NjU0MzJhYTE0OGVmNzBmOTE2Mzk1MmFkZTM4ZTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0TnL7aGui5qZqoQvnLYZxjjohKSnIP-TycG5yr081gNGaIf4L6XB4iZt5HLkqSDPI48lVsy2UASkrBpujHK4wA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110649_57_2477_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.908Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ill1ank5alg5NVRtMTF6L0d6U3FwaW93YmdKZXMxalVtQXhLNGgwTlB1MStvZEcwS2VTUGdMcEFtOXVXb3pyNnRZa3lnVCtJejFqU2tEeHIrR3JEd2dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjVfNzdfMjQ2NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTJhOTU3MWJlNzc4OTI3ZjhjMjQ4NjhlY2FjMmIzN2QwNzdlNTQyMDBlOGQ2MjA0MzhlNWUxYmRlZDcwNzhlNjEyZTVlYzUzNWQyMWNhYzMwMTM3ZWIxOGUyMzVlZDdkNTllNGE3MjU5NmYwNDk3OWFlMzI2NDEyYjgxYTIwMzQ0NGFlYzgxYzk2ZDI0ZTI3NzJkMzE0NTljMTMyYTc3Mjc5MTI0ZmIyY2ZhMzA1ZGQ0Nzg4ZTVmNjc3YjJlYzM2ZTEwYThlYmRhMDQzNzEzOWUxMTNjMjM3ZmE3ODM2YmRlYmM4YmRiYjU4ZDljMGRmMzYxMmYzOTBkZmQ3NzA5MzU4NGQwODc5NzlmZWVjZGI2ZmIwNTg0NWJmYjEwNTY2ZjRkMDkzM2ZhZTExNjNhMTQ3MjlkODQwMzRmNjhlZGZiNzc5ZTRjNDU1ZWNmYjEwZTY2NTc5NTY3YTI4NWZlN2FmN2E5YzI0ZGRlODYyN2VlMjMzODhkZmUyNzZkNjdjNWU0MzBmY2I5NzA2YmY2YjNmMzhmNTAwMGYwMTdkYmZjMmM4NzkyOThhZWIxMzliZTEzMmY0NzliZDAyNmZiNzNhMjFkOTE4NGQ1MjAxOGQ5MmZjZGM1Y2ZhZDJhYjJhMjQwZTU4MTQzMWZlYjExYjFlZWI5YzY4MTdkZWVjODdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xCnTJjo-gygMNXKMFksjqfBq6oR0R1oUPePPhxDGrhZbwDLrfixfOakfJjsaD6b84YFxSVFuPfm_hzuZff67Fg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104625_77_2465_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.911Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ikx2bFpkcEZSNTdkVnNVanU1QTJYYjhmdTc0LzlGY2h0am9zMmliZVVqNlNhRDgzODlyTFF3OXRuaFJFdjJ6NGZXazV4WFU2S0F3dDB4MmljUVNsMzN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjVfNzdfMjQ2NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGNjYTg5OTFjNjQwZWJhYzZjNmFjOTRiMjhjZTIwY2EyOTgwMDE4NDdiZWEzMDZiMjc5MjNlOTNlZTQwZWMwZWYxZWEyNzFlYmI5MmNiYmUyYzZlN2ZjOTlmMDM1NjIwZDkzZDgxNmIwZTkyYzg4YmM4MmRjYjA1NDg1ZWE1OThmY2VlMTUyYjdhNTYxMDgyMmI4NmI0YTliNTUyNDllMzliZjhmNDAwMTgxM2Q4NTg5ZWRiMmYxYTAxOWM3ZmViMTU3YTM2YWMzYmVlYTJhMjNiNTQ5YzFlNzIwMTc5NGExZmEzOTk3YzhjNjE4NWRiMjYyN2RiMWQ2ZThlMTFhNTk5Mjg2MjEyYmRlM2ZjM2Q4ZWIwMGJmNDA4MzA4YmE4N2MzM2EzZTQyMjY3M2YyOGIzYTMwYjZkN2EyOGFiYjU2OTg2YjNiOTE4YzUwYmJmZWE4NmQ3NzA3MGYwMzA0ZDYyZWQxMjJmYzA5ZDQ0ZTRhYjQ1ZmNjYmM4ZmNlNmEwOGU0MTQwYzQ0NWNlOTc2YjNjNWE0MzYwZDRiNjQ3NDhjMDdmNzZkMWE5NDkwYjdlNjU0ODIwNjA0YjQzMWYxMmI0NzBjYjkyNjNhMjg3ZWM1MDc1NzVlNjBiNWZjYjkxODY2YTM0OTJmMGNiYTcxNmQ3Zjk5ZGQ4Njk3ZTIxNDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WVXoo817D9Yn768KDoZl6B5DBopgQnh9pAA45-W2Slwfc5UBeHKmt3zcoSQ07bFFauG8XdiUUkW2MqcIEN71bw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104625_77_2465_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.916Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IlhvL1hCbVVDS01VUnJjd3U0Qk5wemZaOWJqWXFtamFXVlhDYnVDMTBqWHdMMTk5a2Y2dUhFREZqS3d0ZTYrdmJYbVplR1NnVEJhSU9MU1h4QkZnVmV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjVfNzdfMjQ2NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWIxNmFmZDBlNjk3Mzk2ODgyZjgyZjJjYzFjMzJhZjUyZmU5ZDc5MjI1OTRhNTA4NjEzMThiMzY0NTYxNGI0MDA1MmIzYWRjODgyNjhiNGYzZWQ3Yzg0NWVjOTcwZTAxNjI1Y2FiZmU3N2MyODYyNGYzM2JkY2RhNzA0ZmQwNjFlNzRlNDUxYjQ2YjViZjE1M2U0NmVlNjc4YzA1NGVlMmRiMzA1ZTdkZTA4N2Y1ZGNiYmNhMGI3MGExMTNjMDRjYWVlZDhkZWI0MTAxYmQyMDc0YjM1NTJiOTdlZTg1MDc1ODI2NzY1ODc2NzJiZDAwNGUwMDk1NmM5NWE3ZDc3ZTlkMTlhYTU0MmMzMDE0ODYxMjc0OTM4N2Q1YTZkZGZkZWFkOGNiMTg5ODdiYzBhYzRiNGVhOGRjZjI4ODFhMjA5MGM1ZmY0YWEyN2NkZWRlNzU4OTM5MWRlZWIxZDc3NGViYzRlMTU0OTc2NDBlODU4ZDAxN2M5MzUzZTZhMGM1MjQyZDEwMjc0M2RiYjc2MmM0MGI0OGFhNTU3Mzk1Mzc4ZTkyYzc0OGQ2ZDYzYTgxY2I5ZDIzM2Y1ODg4MzVmZGQyZjdjYzA4ODNhNDIxZDJlMWYzOTU2NGIyNzEyMjFhYjQyMDBhNDQwZGUyMmY0NmEzNjc2MDIyY2UyNWNhNTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LyDu21IEt-7FWX7Ztha0B5Pi_LJLTg-Fx3CQojVWbZgeVGGNVS2lLFd496SCDx4B9GvxTVMxTekqvAip5BUKMg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104625_77_2465_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.922Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik5QMEs2d2VRTk9rWXhWNysyZ2xXbXRPOHY4dG5Xbk1pdW4wL0ZiRGk5ZHNvTzZDa29TdHJHSWFBTDJRcmV4VDRGd2o4eHpQYUQ1VjNrUUlKeForK1pBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMyMl8xMDQ2MjVfNzdfMjQ2NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzFiMmMwMWYyZTdiNjY1YTdiYWI1MjQ2ODYyMDZjNzliYzZjMWZjYTlkOGJkN2I2MDNhOTA3ODhjY2EzNWJhMGZjNmFkN2QxNTZjOTkwNjQwOThmOWE0YmYwMWIwMzcxMmI1MjMxYjJhNmVhODFmOTFlNWZhNTYwZjk5MDY2ZWRhOWIyZmQ3ZjlhY2RkYzVhNjY1ODAyZmFkY2RkZjI0ZDcwMTkxOGE5NzU0OTg0MmJjNGJkMmEyNTc2ZDBhYzc1ODgwZjg5ZDNjMzNiYTE4OTdlYjhmMjdjMmE4MzQ5MzIxNWFiZGRjZjA3NTcxNDhjZTA5MDU1YjY4M2JmN2MzOGM5YzllOWZlY2U3M2ZjM2U1NGQ0NWYxZDE2MTU5OTc0ZmNhMmZhY2I5ODc2MmI1NmMwOWIzN2I3MmRkNmZjODY3MTI1NTZlZmFkMmFiMjM3MDU0NzVjMGUyYWUyMDRkZWU4NjFhMDc4NzU0ZWU0MjBhNmEzN2IwNGJhYmIzZDdiMzIzMDA1MzcxMmE3MThhOTcwYTEwMzU3MzBhNTM3YmNjYTlhZDQwNGU3ZmQ1MzBiNzAyZmZjMDhjZjViZGIzNjVlN2NiN2Q4NDA5YzE0YjAzZmZmNGQzMTg4NjQyOTA2YzQyZWU2NDM0NGExZTAxMTg0MWE2NmY2NGY0NzMwZDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.73e5IWAf0UZoMag7JCiuoGA_ZMKqe1J3KOZGcjAuIwHX4dDTyeNxRSq4sSCtoqdN2nSyAqZl2IRyFb6NW9etSQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240322_104625_77_2465_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.925Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im95N3ZtMmdtOE5JYU1tWmFOQ2JZdFZ5d09UMUtjUHptb0RkT0NrbHJleHJlYk1PSXZCVFBvcm5FcEEwb2FrNkw0bjJ2MXVhRWk4eXc3TTFmdDFzOEJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDRfMzVfMjQ3Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmI0NTIzNGY2NjM4MWNkZDgyN2YzZWU3N2E5N2IxZDc4MTU3MDUyNmNhMWQwOTgxZjMwOTA2MmEwYTI5ODA0ZmVlZDc4ZTFmMTE5NjNiYjEwNTI3ZmRkYzMwMmMxM2JhOTJiNjgxOTQzNDY5ZjYxZjlkY2EyNDVkMjA5MGYwMzM0MTExODI5YmIzMzMwNWZmNWRmMTU3ZDJhM2NjZDQ4MTUwMjcwZTMzNTNlNzNlMzVkZjk0MDY1YzY0YjQ3ZTg3ZjU5ZDA4MWIyOGYxMThmY2I0ZjgyMmMxNjJmMmZjMjFjNmQyYTY2YWJmNmY5OWY4MWQ5NDVhYmJiMzY5ZmY0NzQ3OTlkMmE3Y2RjZGQxNTA1MWQ0MTcyNWI4OWViN2JlMjVhZTA1Y2Q0ZmUyOTc2ZmYxNjc4NTgxNzE2Y2MwMTBiYTc1MGZhNDg2MmNjODcyODZmODFmY2M3NDk3NjY2OGUyYjAzNDdjMDFmZGVlZTg3NjExYmZiMWFkOWJkZWFiZTQ0YjNiYjI3YTA4YzRlNWZjMmM0YzEzODM5OWI4Njk1ODBiNmFiMmMxNmFkMjEzMDU4YTUyMjZhNDdlNGM0N2NhYTUwNGM0NWUzZjJmYjcwM2E4NDU4MDY5NmRlYmY2ZWRkNGFhNmY5Nzc1OTlhMTE2ZWVlY2MxMGVlMjgzM2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Edc-M5NbG46YKFSiudMBxhbu44zjNci8iLa0xfJmjgRGQgmPUMI9Tyrtt7dodNgDNT5LyP4pxgPAGd4DVyM4Rw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110204_35_247f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.927Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Inp6YXZIWXo5Vk1CbkpyODBtY1RGTzZwOFpvbGVFYW1UdldFYlFleERJSEZUSUFXZlRDb2dzeithanRnTVZJYnNwNnVBNXlNQXgxUnhYaWJOTWVsdkpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDRfMzVfMjQ3Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTY1ZmUyODZiYTBhNmRlNjE1NDI1MTBmMGQzNGY2MmUwYmE4OTk3OWQ1MTMzZGU2NWEzNGZiMDhlYmI3ZmQ4NmQ2NmU5NGEzMGRkNmYzMjJkOWExMzI4NjRkZWQ4NGJkZDI4Yjc4MGI2MmZkNGM5NGQ3ZTA5MDIyODNlODFkZGEyZmYwMGRiOTJjNzlmZDk0ODMxYzc5N2U4ZWU4Y2UyZjVmM2I0MWJmZGNiMjRlYWFkZWNiOWYyZTdmZjNiNjRmOTI0M2FkNTRjZDdiY2UzYTFmM2M2NTk1MGVkY2FiMmE5YzI4ODdiNmRmZTM3MTJmYThhM2Q3ODljMzNmMmZiZTNjMGU0MzZiOGE4YTY1Njg4NmY2ZjIyZTg0NmNmMDhmZTc2NDc4NzBlNTg3Y2I2NTQ2ZjViN2QzNWVhZTA5NGIxZjBlNTA2ODI5NWFiZGExNTE4OWZlYzIxZDY0NzVlZWJhYjFiN2YyZjdlMTI1MDNmOWVhNmJjMjlhYmFiYWE4NGE2ZDk1YWQ2OGI1OTdmNDUxYTM4ZmJmOTg4MjI3OTE0MDNlYzkxMTEyOTk5YjJlZDI2Njg5NjlhMThhMDVmMjA1YWM3ZmQwNjNkNDhlMDViZmQ2MTNiMDg3MjZhNTBiYTMzMDlmMmE1ZTA4Y2I3MGJiYTdhNjUyNzRhODBlZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZhhiSsUK8EAaNioQq_qJobTiy5VeyQuQGstkcQb5oEMmKGrJAoG7apKUkwb0U8x4u5lpnaLZnaCt4MV9UppOSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110204_35_247f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.930Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJwUG9VOHN4My9BTG94a0s0cHhvL0JGOGw1U2VTNGdzU0xpZ3lsU1BNazNEOHV0Z2Y0SGZ6bytMSG9MYVdJYlczak8vK3JQODFERXVPbDJ3SldGcVpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDRfMzVfMjQ3Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDZkNmQ0OTE5YTc1YWQxOGU4YWUxNDhhM2Q0OGZiNzFhZGJmZDVlNTJkNTQ0YjMxZTY0NzY3MmE5NjRhM2IxMmMwYWE4Y2VjMWNjYmU2Yzc3ODllNDdiYzYyMDUxNzZmY2UzNWQ1ZDdjZTdlNjI5NjAwNzBkZmIwOGM5YzI4NjQ5YWZlZTMzZjJhNjYxOGZmYjEyMDRiY2UzYmEyNjZiZjIxNzVmODhhMjBjNTM3NGNmYWRjYmU1NTVkODY5YTMwNGU5MjgxZDA3NjMzNjcwZmFjYzFkMTY5Zjc2MGEzMjFjOWQ2NDI3MDZlYmIwYzY3OTY5ZWNlOTA4ZGQ5ZDExOGE0ODk5ZDFmZWUxY2Y0MGRjMTk5MjQzZGI5YTljM2YwNTJiZWM5MjQ5MTFjNzYxMTljNzI2ZmUwNjQyYzNkMDQ5YjFjYzhhY2M3OGJkNzdlYTVjOTZjMmRiYTM5NjU0NGVkMWMyMWYwOThjOWRiOTNmOWViYmMyNjI2MWEyOGY5OWRlMTdlYTAwMjE5YjdlNTU0YmUyZDBkNDk4MjA2ZmJlYWNkN2FjMmJhNDBhZWFhYjgwMDkyMTE0NTUwMzlhYWVjZWRlNzVjZWU2MDYxMzJkNzM4M2UyYjI2YTBhMDlhYmY0NzdlZDMwMDllOTU0NDIxMGIwODE0YmMyODYxYjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lwyhyXUYM2qppxgnS7HhPSo58sijDwHTpv7T7fJxvluSv5NNm-Mvatt0M4PkVJVBRITNHjQMT6I24US9S0frjQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110204_35_247f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.933Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkRQaCtpL1M3R0tXY1JuYWNaZzZ1N1RFNi9JVVNTMkc1OFJPUDNjUm8xaTJ3OE1wZGlRNjRITThqWEJVNlliVGVSOVI5bTRheTk0aFhZdjBvTmY3cFpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxMl8xMTAyMDRfMzVfMjQ3Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTcxMWRhZjBlMDM5OWFiYTUxMDk3ODgyMzljMTQ3YTgxYjAxNmRmNmNiMmY5NWUwMTA5ZTRiMDE1ZTc0NDYyNWRkOTM0MjZiYTBiMjg2NDA2Y2IyNzJlYTg2MTBiZmNiNGM3OGUzMzk1OGYwYWJiM2ExOGE2MjYyZWRkMTY2NTRjM2Y1MTUwMDNlZWY3ZmRhMWRhOGMwYmU4NjM4ZDZmYWZiMTY0NzM3NTg0NDVjZDE1MDcxNDRmMDM4NGYxODNjNGNhNTAzZDA3Y2FlNmY5N2NkM2NkZDNkNmE3ZTk2MDUzNTEwMjhhOTQ4YTJiZDBmNTU1MmRlZTg3MDFkZDE5ODk5YWYxMjUwYzgzMGUwNmI4NDFmMmE4OGQ0MjdjNWUxNzQyZGMzYjYyYzRkZGNhZjg2ZjNhYzdhZTVjMTA5NDQxZjc4NjhhOTdjZjRiZGNkODRlMzc1MzU2MWZhOTM4ZTc2OGJhMjI3NTAxNjI0ODM0OTAxYzViMmQ2NzU2MDg4NGZhZThmYjY4MWU2ZDQ5MDc3YTEwZTY0MzE2ZWRmOWZjMGY4MGU2YTQ5Nzk3M2Q0MjliNzNjZjZjM2Y5OTFjZjBjZGRiODhkMmExZTQ3OGEyNmNmYjJhZmFjZGRiMjFlYzhiYzkxMGY5MjY3NDUyMWM0ZjJhYmVjYmViYzJmOTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hwXE4rR2rMDtRzDf6T4XVvtFToONVDJinVNhGfRXCFQoqVOBt7JGFCCGgGeLGCX2kFXWlmlBZo2qKDOsIpXjlg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220912_110204_35_247f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.937Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkJTNE12MWRJaERCVFljK3hBRFhIbHBxMHlKbkFOdG1pNHdtR2ZxTHpKbGxRSGxKeVRjc1k5VmNhS0Z6cVcvWEw3QnFJVCswQUgreFBUOUx0Vnl6VDFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzZfNTdfMjQyYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDMyZWY1ODU5ZjE2NzVkYWVhOTVmODZlZDM2MzQ5OWY0ZDIwM2E1YWU2Y2NlYjZjMzBjMWRmNjg5MGZhZTIyNGE0MjU4ZGU2ZjQzNjI1NDllNzI5NzIxOWUwMmE4ZTJjMTZhY2I5YjAzNWQ5N2E3MTcyMDIyYjAwOTg0Njg5ZjE4YzcyYTkyODc1N2E4ZmQ1OTU3MjRhOWIxM2IzMTQ0MTM4YTQwYmNhNzBhNGU5ZTgzZjcxODViNzc4OWNmN2IwNjlkYzc1ZjlmNjI3MTI3ZmIwNmIxMjQwNjhhYzYzNGMzOWNiMGE0OWY4ZGFlOTVhY2JhOGYwYzhmN2NjMmVmMDk5MDM1ODgxM2YzOTk0MDlhMjQ0ODVjNDA2MTJkODg4NzBjMzkwNjIzMGNjN2Y0YWY2ZTUxN2E2NDY2N2FiYjVkMjljYzZlNDJmYTQxY2RlMGI0ZGQ1NGNhMzE3MTAwNmY4NmZlNGFhZDE2YmMzZWMxNDA1Y2M3MGE2NzkxYzVlM2Q4ZThkNzM4ZjBjMmFmMWMzODU1MWNjMzU2Y2M4ZDU5YjU4Y2U1OGRlZTNiZGFmZGViNGE3YWMzODM1MzY2YWIyNTAzZGM5MDQxY2UwYmE2MjAzNTZmNDBkNzlhZjM1OWY2YmMwZDc4MmIzZmY4YmJjNjZiY2E3OTVmZTFkZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Md9olTVDUSsrLDKg4gSqCpnjiuWw9SvHUE3S9P9hkJtu1l-hMBkWTUHkJQTAsL2QnPMY-zfq3inv-B3I30d6zg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103836_57_242a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.941Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InorMU5GMnZpTUJ2RGFoemxaWTAvOStkYnU5UkdvWWVrUGthanB6RG10STNIejhzaWNnTHY3aHdLdk9POE1aL2dYbDZOWkZweGlydVZmQ2ZXWVpwTnNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzZfNTdfMjQyYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmMwMDc0NzU4NGUwOGVhNDEzMWExYWNmOTkxNDFlZTk4NjEzOGExMzVhZmMyMzNmOGRjYjQyODJkZmVmNWZjNTU4YTk5MzhjYWYzZjE5ZDRjNjBkNjYwNDJlZTYyOWU3ZTJmMmU2YzJkNWZlODdmOWRjYTMwMDkyN2YxNDY4MDBiOGY4NTUxYjVmZDBlZTA2YThkOGNjODI4Y2YxYjZmNzI2ZDg3YTM5ZWMwZTdiNDVmODY1MWRjMzEzNTkyYzEyMGYxN2FjZDhmYWUyNTIwZmY5MzdjMWM1NGQ4MmIxNGQzY2VlNDZkNTk5NzUzYzNiZjlmNzkxNGM5MDkyZTA1YTZlNWY5MDkwOTA4NDVkNzAxZjM3YTBmMzc5YjEwZjM1MTA3MGRjODdlYjM5MjM3NDJhY2Q1N2JjNzEzYTA2NzBmMWQ0N2NmNGJhZWRhN2M4YWVlMDU4ZmJhOThiYjQ5ZTU2N2U3YWZhZDU3NTZkNjlkMTNkOTE0YzNlMWIxNDIzYzhjMmZjMmY2MThjMTQ2OTcwYjY3YmI1MzRjOGIxNDRiODRhZGU4NTUyZDc1NTc5Mzc1Y2I0OTU0YzJhOGY2NDU2MzIxNjBlY2FiNDdhOGUzODY4OWFhYTZhNWE0MTRiMmFjZTU5NTIyZTY4NzA3NTI3NTU3YWRkMDI2MWNiNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hyfb-Gpf8S_KmkEAd0bNknrWd-MKraCc1ZZ79Xq4d_iyRJ5bgzMN6wZLCH3ksybEVp622qp84TZ-G4guYe5F4Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103836_57_242a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.945Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InZLK3Z5ZU8vaGJ5VTdkSytncnFMbGhiOGhhWEUyVjhUU0tOajMzQkk1UGk1Ri9ZMlhJU1ZsOGplZHpFY0QvT0xmbS9mb1h6U2QzMDlVRnJxZUxUVndBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzZfNTdfMjQyYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY1YTBmYTE4ODYzZWIwZTRjNzViZTk4NjVjMTdiYzdhMTc4ODdlZmYyNmQyMTM1ZTIyZmU5MmM5NjZjZTRkMzk4MTAzYjIxM2QyMTU3Yjk2NGZiZTExYjEyYzEzNWIxMTM0ZDM0ZWY1ZjJiNWNjNDE3ZWViNjQ2NTA4NDU5Yjk2NDk5YzU4ZDZjMTFjOGQyY2EwODZhZmNlYzI1NTJlMzcyZTk3MWU2MjIyYTlmNzIyMmMwNjY4MjllZDUxNWYyMGFkMGQyMmUwYWVjMzIxMTAzNWUxMTQ5YzJiMTMwZjY1YmI2ZjQ3NGRhNGEzNGE0YTc3MWM0ZjMxMzMwMmQ2NTVkYjA4N2E2Y2I5NDY4OTkyMjhjNjBmZDc5Yjc3MWRjYjNiMGZhYjU1MDBhNDlhM2JlNzdjZDI3OGZkMTFlOGE1NTE2NjY4NDAwYWZlYmQyZGZkODVmMjFlMzc5NmQxOTQ5ZGVlY2MyNmE5ZDllODIxMTQ4YWM3NDQ3YTFmZjMwODNiZDE0MDAzMzYzNGVkYTk3YjI3NmUxNmNlNTViOWI2N2Y5OGFiNTRiYmJjZThlZDY4NTE5YjhkYTY2YThmZGFjYTdjZjdiNzBjMjdmMmM0ZjJhODlkMjY3Mjg0ZDhmYmQ5M2VjYzRmMzk3ZTA0YjVjZGQwMWUxNDg0OTU4NGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.r4NkN9fhS_36x_-Z8sakMcJ1Lp4aKGEvCNl6Oeao5RxNpLYWy61KAQhJVIKgEU2U-w-3SnbmaJjjtK2LRfj9Dw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103836_57_242a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.948Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImpkUUxiNHAwNjBCazUzS00wNTRwM0tGK0hXUEFKcklIL0VsakEvbGd0WHNoVzFKMFIyWWNobW1KbXdlSGF6QjVmYm9WSkFHc2FLSk9ZWUlLeVQ3dG9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM4MzZfNTdfMjQyYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGE0NWU1OTBlYTU0OTI3YzFhNWNlNmUwYTFiMmE1YmEyMDU4NjgzOThmZjdkMWJkNjczODNlODI1NWU5MDkzNDA5Y2QxMTk4MjU0ODY2YTU1MTI3MDI4YWZkOGUxMzA1MTdlZTU1ODkyM2IxMDI4ZWI5NjZjZjQ3MTU4OGY1ZTk1NTg3ZTIzNGI1NWI3ZTNkNzg0NzM3M2I3YTc1YjYzNzdiODE5ODQ3NWNjMmUwYzJmMGQ2NWM5MTA5NzcwYzYxZDk1MTZkZGZiMWU3YmQxMjI3YWI0OGMzMDAzNWE4YjhhYjM1ZWFhZjAyNDJkNjIxYWQzNDFiN2NhOWQ1ZTU4NTlmNWViM2UyNDA5MTdmOTcyNTQzNDYyOWJlNjk5Njg1ZDUxNzFlNzk1YzAzZmRkZTViZGUzMDI3ZGU0OTlhMzRhOTgzN2VjMjcxZmU1YmZlOTg4M2FjZTRlYTBiMjUzYzdkOGVlNjE1OWU3N2I1ZDE1OWM4OTlhMzc2MTgxOWUyYTcwZGNmMTMzNjQ4ZDQ5NjI5Mzg2Mjg2MDBiYWQ0OGUwNmRiYmRhM2MxMzliMjMzMjQ1YWRhNzIzYjEwZDhkMjA2ZTc0YjE3MWVjNzg4Y2EyMTk4MzlkMTBkZjFhNTU0ZDkxZGMwMzVlNzc0MWFiOWRlZTcxNzRkYTBhOGIyMjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qbPTRlDFrd5pJ_tS0Cr6sKXE2mBj8Klk9uW-KTyDa-OYdPBTsBzD5CPjUPgjIs0dXjVILvXPoeRGWub96y8tHA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103836_57_242a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.951Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkFtMjFkdVhBUzFjbEtPaVZidklNUG1VbmdubXQ1VytDdkU2ZThNMndySnR5dmhKbXQyM1VSRlFPOVB1VXU3QjVhd0YwQWo0L0NLZGEwYitiQVdLUlJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTNfNzBfMjQyNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjI1YTk2MGY5NGFlMWJiNmRiYWIzYzJmZGEyNzYxNmM2ZjUwNGU4ODgwNjViZGU0NmQ2ZDY1ZGM1NWFkNDY0MjlkYjMxM2M1NjlkZjU5MGNjZTk2ZjhmMmJhZDlmYjE0NjVmYTZlM2Y4YjZlYzYwZmMyNzU3YjcxMjRmNGZiZjczZTJkYzcxYjQ5OTNmNjcxODZkZjVmMTJiM2M3NDczZGJlYzI0YjU0MTc5ZmRlYzI5MDJlNWNjMzhiODhhZjY2NmRlYjc5MWRkYjA3NjgwZDQ0YmVkODZkZGJlNTZkOTgxZThiNjcxMzU1NDI2NjQ4MGU4MWRhMWNiMmMwYjQwMmIzZjY4MTE2N2RmNTE5ZjE5MWUwMjYyOWI5ZmE2NjM5YzJlZGY4YjFiZWUxNDY2MDU1YTdkNWQ4ZTgxYzVmZWEzMThmYmNlMjE4NTc3YTc3ZWU2ZTJiNTE3NzcxYTQzZTRkZGZiNDQyMWFlZTFjNjZiYmQ4NWEwNjhhYjE0MTAwYzg4MGM0NDNmZTc1ODYzYmYzZjcwNTk0YzRjOWY4YTRiYTkxNmFhNmI4ZmZjNWI3MzIwM2M2YjFiMjU2MGQyNDZjMDQ2YTI3OTUxMDUzZDE0YTY4MjlhODY3YTE0ZTI5ZmRkYTJmMzBkMTc4YjIyMzNmMzkxOTlmMjAwNzAyNGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.56LKo3J_nBNJioY4FFg4Sk3li_oMwpymcoeB9CV61-MEmpCkcX-N2fKTLgPlZIQAODMS_s8HM54AbeSOU3MjyA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112813_70_2424_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.954Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImxWeWdDVUZoYVEwVTRSWVZHdHZoWURvQmdzSnJPaDhMdFh6d09sRkhlelRweHNxL250L2JEUG1VSlppemM0bHVBemFnS1IxMHd6MjF5VnhBKzUvNHVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTNfNzBfMjQyNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDVhNTEwNGNlMzU1YzJkNjRjZWI3NTQ3ZDZlMTdlODdhNWJiMDIyYjVkODMwNTVhMjRlNmUzYjhmNTZjZTM5Nzk2ZDdkNzdhM2ZiOWYwYjQ3MjAxY2YyYjM4MGU4ZjM4YWJlN2JjZjYwNTFjNDk4ZjlmNGMwM2FmZGEyNDE1OWE4MDU4MWU0MWIzNWYzNGIzMmY1ODI3NTI2NWI4YWQyMDA0ZmIzYmRhM2QxOThiMmIzNzAyNDNmYjIxMTFmZWRiYWE4YzE0NTk2MGE2MGI3YTBiMWRjN2MyYzlmZDRjNTM3NGU0Njk5MzI4MzQ5MWQyZTY3OWY2ZDRiYWVhOWZkMjFiYTgwMTBmMzY1NGVlNDY1ODcwY2EyMzVkMDM2OGQ2OGM5MzNiMGQ5OWI3MGNjYjkxMTEyZjUzMjZjMWM0MzVmNTBmYjA0MDZkN2JjNGY0ZDlmZTY1YTkwN2Y1ZjllNTMzOWY0OGU4MTE0ZjJhOWRkYWM4NDc3OWIxMjBlY2VjOWQ0NTlmNGU2ZDlhNjUzMDc0MTdmYTlmZjU3ZGZmN2E0ZTA5NDBjMjljZTI5OWU3MGE4ZWU3ZWU1NDY4NDlkOWFhNjFhZTkxZmFkNThkOWU2YjIzZmNiOWU2ZDU3MzRmMzlkZGQyMjQ4MTE0Y2RhMzAyODhmMjU0ZmE4MWVjYzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rvcGINNYFK6S7SdqKWJVIM6dThiB429z89GEwAmQrA5-XwuwDsg7lKxHjCdBkd0aUoRubDtR9jC9rSvv3afwsg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112813_70_2424_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.961Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkI2bmJIZE0wZWNsZnlyZDJJRDkxMXVieUN1aUVDbUpabzdrK3NXYXJibGVCOVRQdXRCcVVUb05OcGpVc2lta2NBOEVUR1M2QVFBOW9pc0xORk1qM3pRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTNfNzBfMjQyNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDkxZjBiNjc1YTQzMDA1MWM5Y2Y3NTVmY2EwOGVmZDVlYTUwYTJkN2NlOTg4ZjkzNDBkZGIwODhjYTIyY2M1Mzc3YjIwOTY2NDg3YmY1NDM4ZWRmMTA5OWE3MWEyNDc0M2VlNmU4MWFiNGIyMTcyN2YwNWVlZWYzNDFlMTdiNmM4MWMxOTYzZTk2ZjgyZGU1NjA2NTQ0NDVmMTIyNzc3YThhMDRkOGJiN2U3OTM1ZjQ1ZDk0YjE2NmM0MGJjMjEzMTUyMjNhYzIzNWFlMzljMzE5Njg2NmU4OTUyNmI1NjMyMDM4OGU0ZDU5OGZhN2Q4Y2Q1YmQ0MDM3ODdiMmVmZjdmYTk4MzJlMzY3YTMwMDU2MmYwMDZiMTA5Mzk5ZmQzMmI3MmY5NTYxNTUyMGUxMzFjMDRkMTQ0NzQ3NmI3NGE1ODZlODA4NWQ5MjZhYmY5YzIxMjRjY2EyNmViN2FmYjAxYzM5YTNlNzQ2OWZmMzc0YmE2ZTNjNzA0YzEyODdhYTExMzRjMmM4NGVhNTYwNmQwYjBjN2VkYTJjOTlkMTE4YTFkYjNjNGZhMTg0NTI0OTI0NzFhYWNjZGUwMjRjYWM5YzA4MWFkZGU1NjZkNmMzMWZkNzgyMDdkMTI0NzgwZWIyZjY1Y2MwMmY3MTA0YWNjNDg3OGZjZjY0NmNlYmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jyVjCdOES0aFU2AAu6A1GhkyY4fkSalc7kIIgYB7jCGncbv8CQP-LMxBS3MfE_EmaPv6uL3S-BIq5qGSc0WVRw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112813_70_2424_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.965Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IklId0VqRkI1dXVCZlA3ZXpHeWs1emtIckE0NDhOd0VFZW9taDNVa1pLb0gzWmdSaEJGYTMrbmQxbW5sNTBBcDE5VnFoVTM0S3NwOHpvYjFGMjBuMWFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIwOF8xMTI4MTNfNzBfMjQyNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDdlMDY2NmRmNmM1ZjI4MjE2ZjRmZWFhNzliOWY0YWM2MzhmYjUzN2Q2MzRhOWY1MjU4M2JkMTI5ZDA4YWU3Y2FjY2UxMDQ4ZDY1OTQxOTQyNjExYjJiZWU2M2JkMTZlZGI2ZDRhZGVkNjc0Njc1MTAzYmVlMzA0Yzc4MGZjNTQ4NGNhYWU2ZGY1M2ZkZjBhM2Q0ZTEyOTRjOTZiYzE4N2E1ZTE1YmMxZmM0ZGE4NTE5MmQwMzMwYzI3NTMxZjRhN2UwZTM3YzcwMmYxYTAyOTA2ZmFkM2JiMjUxZmMzYmQwMzAwMjkxZDcyNmQzY2MzNDg0ODQ5NGY0NDc3YTNkOTE0NDgwMDUzZWYzOWY2ODNjZTlkYjJlNWQ5ZWVhNjk4YjNlZWUzNjgxNzhmYjJjOWIyY2IyYTFjY2I0ZjIxM2JiMGNjNmJmM2NlZGE5YmY5Y2E5MDJhMTFjYjM1MmQ1ZDdiNDI2ZjQwZTBkOTEyYWUyZTRmN2JkMmYxNDA1ZmY3NTAwMzUzMTYyYzUxYmI1YThjYmVmMzk2ZjA3ZGQ1ZGEwYWY3ZDhiYmRjMTgzNzlmY2NhNGNhMzY1NzFkMzYyNGQ4MWQwOTI4M2Q0MDQ2M2U1M2YwOTgxNGQwODFhMDFkYmUwZGNlYjJkNjEzNjBhODc1NTk3ZjkwOTgyMTYzYjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.86LOW9JkH6p1R_EON4vCqnz3wjTZPYwFoxLi0-jLqX9gXSVSWImyrU7ArxbJ8JaZeN6xp2Iu2DJ1ke0LzF-Raw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210208_112813_70_2424_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.968Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjNBVDJ2K0QwSk9DS1IrYzNrMjJKZ2VMdHU2Sk5QSmUxd2gzWlFHV0JBK0MwN3Z6cStCaG03Y3ZXVnNGR1lFdkZISnFJTDAzMjNKZkRPSUVwalJnTkhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI1NTVfMjdfMjQyMF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODRjMjFlNzdlZDEyYTQ4ZjFjOGVhNDIxMzgyZGVkZTVlMWU4ODRmZmRmNmMzNjkxY2U0MzA1NjQ5NjNmODQ4MTdhNWU2ZTc3ZmZiNDE3OGNiOTE4ZDE1YjdjODVmYTQ5MGY3OGFhMjhlM2E5Nzg5YWU3Y2JkMmNhMGI4NTg2Yjg0MGI2Yzc3NzhjODNkNTFjMTE3NzUyNjY1MjAwN2M5MjljYTM2MGRlMjBiNmVlODJjMGU5ZDU3NmFjNDc0YTcxMjJjZTkwNzMzOGIxZGQzMjdlNjU5NjBhYWE1ZTY3OTJhZjdjMGZjZGE2MTZlODc0NTA1MTg3M2M5MTA4MTBmNTk2YmE3MjI0MDhiOWYyODUyMGE5NDUzOTFkZWI5ZTc2ZWUzODc0MzBiMWUxMWNmNmRkMGIxNWQyNTdlYmYyYjQ2ZTRkNmFjZWVjMDZjN2E0ZGVjMWNiYjNmMDEwY2YzNTIyZGY4OTc5N2YyYmFkZGU5NzNjODU3Y2UxNDQzMjE4ZTRjZDIwNjc5ZjAyOTU3MTIyZjlkZDY0NzNmMzQ5OWQ4ZjNhMTAxYmNlMmEzZmUxMGI1ZjE2M2ViMzQ1ZWU2NTdmY2JhOTJlYjE4YmRlZmNmMjIzMGMzOWM4MDgzYjU1ZjM2YTUwOGQ3NmExZTM2YmIxODA3NGE3NTczNjRiYThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8xK4su7zOUKsQfpCOfToorI6jYo374wOfJ5SqIE3Xci2OFC4498RkOhpGRko0zMR2f0R7dwPkfu5_w9IZY2LMg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102555_27_2420_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.970Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkF1ZERZMG9RZXdIK1BNOXpOUVM2dDkzdlhpZFhkNTdDT2xmTVhKV3craUIydXdnOHFhczV6c080VWxFbm92RHgzRllEYVAvRTN0LzlTTVBGZHJEcWNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI1NTVfMjdfMjQyMF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjQ0YTg2MWMwZjViNGI4MzA4MmU5N2JjMzZhNDUwM2JlYjI2YjJlYWU3MWE0MzhiN2NlNjExYzU1ZjUzMTJjMzhmMTJmNTdhYmIyMjZkODJjNTQ3MmQ0YzExZDI1M2U2ZTk5MGI3MzQ2NWRkNDIyYWYwOGU0NWZjZDFhYWRiOGFiNzQ1Nzk3NDYwNjhjNTE5MGU3YmZkZjY2MWVlZmFhNmVjNGZjNmEwZTgzNDcyYTRkNmJlYTA1ZjBlZjc0NWM2ODRjNDU2NWMwZmEyM2UxODA1ZjRkNTE3ZTU2ODEwYjY2MzhkMTdhYTFlYjlhM2QwNmMzZDY5NWVlOTM3NDdiNGNhODA2YTRlNGY0MDZhYjhhMTZjNWU2NWIyNWEzZGRkMjc2MmJjMmU1MjI1YmQ3YzgzMjU1ZmRlMTcxZDU4NTIyY2VkY2EzOTdjODY4MzA1ODRlMmMyNDI3ZjgzOWI5YTc0YjcyNzc5NjY0MjdhNjRiOTk4ZmZhNjdmZTExOWYyNDVjODhlZGMzYzc5YjNiNjAzMjk2NWE5MWFmNTI0YTUzZjkxOTRiMDQ2NmFjMzVlODZlNjk5OGIyOTYyY2MzMDNkMmU0MmM3MTZjNzEwNTcwMTE1NzI2MTZiMzI1MWVjNWIxZDJlNzdiZTA5NzI4YzM2YTc4MTUxNWU5NzUzNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.40xCwpna-ydBAtomQJug8-tBh_8TV1_BEBAz-RBhxv8x075AkvU4hHehrsOxotOduOXYXb7taoqLNl33HIK_Ug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102555_27_2420_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.975Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InhySzQ1U0lkcXUyM3l6VmFEdEJJRnJLS1ZDWUN4TURFZlYyTUpLYld6cXVuZW91aWN1cCt4dXNxYWVZeGlDVHE1djhiMWx4Y2lITWlicExQajhoblBRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI1NTVfMjdfMjQyMF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTViZmIyYzExMGNiYzRiNTRjOTRiN2ZjOGNkMzUzODg2NDc0YTllOGY3YzU3YTdlZjJiZDZkZTcwNzQxNmFmNzliMWFhZDNiN2ZkNjYwYTczZGU5MGNmZjYwODlmMWY0YTc2OGE4ZmIyNDc1NWJiY2E3MWU2MmQ4NzNkMjg2NTMwNzg2M2U2YWQyN2I4MWY1MmNmZTRmNDY1ZDM5ZDg2ZTU5NTA5NGI3NGM3NDIzNDFiZWVmZjZhZGE3OTRlYmU0ZThkZjJhYzdhNDRjMjUwNTMyYjA0YTg3ZGRhNTMxYjVkNWY4Y2MwNGU1YWY1NGVlOTYzMWViYmViNTU0ODFhYTNmMzMzMzEzZGIxMmFkY2EyZmZhNTg0MDIyYTI2NTYwYzYyNmExNmU1OWNiY2M3ZTdjNWU5OWMxODk5OTIzMzdmN2Q2M2UzOWI1MjA0Mzk5ZWRjMjg0MWU2NDhmMDIwM2I4OWM5MjkxMjg3NDg0YzVjMTRiMmQ4MmNjNjIxZDRhMTY3ZjAwMThjMWMzMjg2ZWJmM2I3ODdlMGUzMTg1ZmQ3MmUwYzcyYTg3NGE5M2U0OTY0NTRmMTQ5Mjk4ODJlNGUxYjJhMWY5OGI4ZmM1NTQyODEzNzFkYThlYzNiYzNiNmE3YTkzZTQxMzM1ZmEwNWYxYWNkZWJlYjY4OWQ0YjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.88g2gwfWUxt8SpQw7SDxb1KWIzJXuLrHQaKqX4j4EM-uEcOL0LwSb-c0R0vvQiK6dq8zzz-vCvulpILLA7pYDw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102555_27_2420_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.980Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6InpwaVRYWERyaEhvU3g1YlBJVXVRUWFpeUJ5WVh0eXQ0SVF0TTZ5RS9EZ1ZuT3UyK1VLaFNEYzg3aEJuL2x2SEV6eXM2eXprVGR2RFlFWStEZFRzdW1nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI1NTVfMjdfMjQyMF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTg0OThlYjljMjc3MGQyMDE3YzdhODI4NjNlMWZhMjEyNTM3ZWRjNGYxOTJmYzgyOGUyODJmYWE3MWZhNmE4N2JiNjMyYjIzMzI4YWZlM2M3ZjRlOWI2YWI4YzA0MGM5YWVhNjU4MWY0MjQ3OTU3MjcyY2ZmMWM0OWRkNmQ2MDFiMzY1ZDQyN2M3MWVjMDAyZmJjNzE0ZDNjNzA1YWQyYTAxYTJmNGZlOTQzYmNjOWQ3OGVkN2YzYTFmNjVlN2Y0Mzk4YzdjZThkOTZjNTFmNDFmNmYxMTVkNGUzYWIzOThjNmM5ZGRlMzVhOWI3ZGVmYWQ5YjEzZTgwNTk4MjdmYTAyYTRjMjc4NWY3ZmE0NDM1MDIwODUxOTM2YzFlYzJmNTM5NmJhNjFmNTk4MmJhZGFlMTdlYjJhNTk4MDY2NmIyNDg2OGY0YmZmZTJlODE2NmEyNDQ4MDM2ZjhkOTMzZjc1NWY4NTkxMDQ5NTMzN2RjM2Q0NDY5MzgyZjUxYzM2MDI5ZjFlYmVhZTMxNmE0ODk1N2Y4YWYxMjFmZTUwNjcxZTE1MTliZWJlNzY2MDAzZGE3ZTRiZWQyZjEyN2ZhMjk4NTA1ZWQ5M2E5OTM2YTI0YTA0NjIwOTNhYThkMzEyOTdhZWUwNjdlMDIxNTkwMmUyNWRiODAzNjU2ZjRkNDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6uFAXAMMhsnVBBRDH-CDLehIMAHwdD718GP1UUhDtE2jsbkfLR1YMAccRGmiHsBP_oJQhov5kF2vnGjyDYTIyQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102555_27_2420_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.983Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Ik1naW9Lanl2bWdQRjZRUTkvT09uYXhDa01uaFh5dHd2U21zcFN1cDI2bnd3STN0aGFZVy8rcTZRc3R4RGJmVnhNU1drR0ZuZDVBb3BLOHRObTA4VENBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjdfMzJfMjQxMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzM1YWJiODE2NTM0ZTkxNzE4N2Y0MjgwZWM5MzFkMzA5ZjY2ODY2ZjU3ZDI1NzkxZjFmMWNmZWVmNDE4MjFhNjBiMzA4MTY3YTBkMjZiZjQ3NGMxOTU2YzQ5NzNjMThiNTVkMGQ1Yzg3YTdlMjE1YzY0NzRjMmE4NTRhZmYxYzYwN2NiODgwZDUxMGI5OGM2NTAyMWYxNTU2ODFmOTc5MThmNGE3YzE5YTE0YjY5NmU5NDMxNWM4Mzc2ZjkzMzgzMGE2YTVhYTUxNmZiNGUwMmY4YTRmNTkxMDlmNDZkZmUyOTI0MjI0YWZjNTNkYjQxMmE0MTgxNjAwNGY2N2YwODgwZjQxZTM3Zjg2NjM5N2RkOWEyYTI2MjY0NjcyZjBiYWRhNDY0YmRmZWQyMTkwZTAzODkyMmE3MThmYmZmZDUwNTk4NTZjYWVhZjBmMjc5NzE5MTk0MzZmOTZlNDJjNmM5ZjNiMmQxZTk0YmMwMzFiZjk2MzgyYjQ4NjAzN2Q0NWJiYTE5MmM2MzhkOWU4NmJlNTk2ZjYzMTMyMGM2ODRmYTI1NjQzZDA1MDQ5ZTEzYTAxMWI5M2Y3OGRmNjQ0Yzg1YWZjZjc4ODE4YTVkYTZkZTQzYjYzZGEzMzE1MTA4MmM3MDVlZTk2MTRhMDYxZGZiZTcyMmFlMzg4ODM2YjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gQsUeARVSqcNrXxqL4IUmeL9w9S5rKfYsVklYP9jxbrpYd2DtklDXe8PNFyHmOpidPgCRgmBFxBL6Xu-F3GU0Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113127_32_2412_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.986Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImtLOHNxQUNnVFB4ZWdZdTVXNFBJenRNWlJqd3FjN1NNbTVJQ0JUWGl0UUhOdGRSNG5vTUdMMDVMRDhGOTh0QnFyeXB6UGJlZlJrMVdhWDFSR1FlTVhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjdfMzJfMjQxMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTI2NTc5OGE5ZmU4YWU1ODk2YTViZWY0MDYyNmMyMjY0M2JiYzQ0NWJhNTJiNTk4ZjVlYTg5YTY5YTk2ZDRmYmRhNjUyNGEyN2UwZmQwZDY0NTdhNjFkMWNkZDNmOWVlNDU1OTQyNzYxYzJiZmY4OTYyZWEzYmMxMTJjNDQyNzRlOWQ3YzljZTIxNTc4MmQ3YWMxMmRjMmY2YjI4ZmNkYmYzNDU2ODljNTA0NjEzYzNkMjIwN2MxMDViYTBjNzY5ZmNlZTcwNDNkMDlmOTY5MjBmZjJlMzdjNTllNDk1OTU2MDVhZjg5M2I3NDhmYzU4OTA0OTlmMzYxOGQ3NTc5MTNiOTJhMzI0YTlhZTA0MTRjODEzYWYzMjQ5ZGMyNTAyNTQ0MzZhNzMwZGQ5ODJkNWE5YmI2ZjQ0YThhM2ZmODU1YjAwNDUyZThjMDM1NTdlZjc5ZDc5MGE3ODA1OGM3YTc1NTE0OTBjMTE1YzEzZDBiMTJmNzEzN2UzYjUxYzc3NTkxOTQ1ZjNhNmMzZTIyOTc2YmUyZDQ5YTFjZDI0NjFmY2ZkYzQzM2RiMWY4YzFlZjc5OWFjNDhiZGQ5MDkyN2Q5YTM1N2VlZjVmZTJiN2Q0MDRiMThlYjgxZmZiZWE5MGQ4NWRjMjY3NjA3MTQ1NTU5NTRmYzFkZmJkYWQzY2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HkPJtvp8fCqOYKT6C7R-66QiHHGqQoY8KFeNZLcJ5RihZhX06XHb5qJjWuIqMdw7bKIIVY2-MV-tEVw8SK1Yjw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113127_32_2412_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.988Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6Im54ajJIaXBqVjBONktKU1lOLzdScTRHcytienlVQzVWcnZ2eWpvTFkwT0Jadkltay9XYS9HNmRiRm5VeHdBeWE1bExiUHJPeSsxckxzMDI4NVpxZnRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjdfMzJfMjQxMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmNlZDI0NWJiMzVmMmM3OGQxZmU1MzY2MGY5OTcwY2Y3YmJiYWQxZDdiZWEwODIwYmU4MTQ4YjIzZjc3YzNkMDg0MDQ5Mzg0ODhlODgxNjEyMmRkZjhkMGU3ODdhMzc0YzFlYmM1YTk5M2FlMmYyZDIzYWI5ZjA3NWExMDA1NDNlYjllMmNmYjc3ZmYxZmJiYmE4NDg3ZDQ5NTBmZjE5ODNmYjU0MjA1ZmM0NDliNmE2NWM2NTdhM2JhN2RjMjE0NjU1ZDM2ZjMyOGQxNWIyNjliNTY1ZDI4YWY3YzFiM2E4OGQyZTIwZmI0NGI2MmI2YTk3ZDcwNjgzZThjNzk5N2I3NzAyMWY0N2M0NzdiNmM5YmE2ZjAzNWViOTMyYzc1MWUzMDA1MjU5OTYyODBiN2VmMzVlNmZjMmQwZGI5ZjYwMmE4NmQ3MTI4YjkzNjkxZDJkNGQxYWU0MmJkMmQ2YjljMWQ1NWNhYzlmYTQ5ZWEzZDRkNjUwZjQyZmVmYWIxNTRiMmRlMTFlYTc4MTUxMTNmY2IzMmU0YzkxODkxNTgzN2FlMDZlMTIwNDA5YjJiZTUwZTNlMWZkMTRkYjk0NTA0ZTM5NzM2YjhmYWNjODg4NTYzYjA1YTdhNWExZTI2NjM0OTg3Y2M5Y2E2MDRhMGQxNjUyOTBjNGVjMWVlMjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Sn1HvJlZjks3Xv1tqieWuPRqwtPiADP_lrfATmHSGVUosnwEacvhfr3XpxVQkEVwzF-UBOZB5t7xZDmQhB_leA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113127_32_2412_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.991Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImRIQUxIcFIxNWhXL1pPamZpdytyZEpMTVRUKzN5YnhCNXVkUGxzNkI0RVRVQjZMcno0Sys0VWtVaGpDdDV2aXltZkgwdnZTTWtLR2RwaFZsUXhWd0dRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMDkyNl8xMTMxMjdfMzJfMjQxMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTIzYjdkOWVmYjZiMGY5OGMzZDIwYTU4NjYxZjM1Y2YwNjkwNmQzMGU1YWU1Nzg0ZDgwYmExY2Q4MTUwNDY3ZDg0MWFkNjcwOGUxZmFlNDU4NTZlY2I1MGJlZmM4MDEyMWE2ODJiMjg3OTUwYzkyOGIxMzVjYjUzNGYwYmU4OTE1ZTE4ZDcwMzcwMGM4OTdlN2YwYjRiN2YwNmViNDBiM2M2ZGU1MjhhZjJmODFhMWM2N2Q4OWMzYWE0YTMyMTA0YWVhOWJjZjNmZWRlNzE0MWFhNDYzMTgyODBjZWYwYjQ5ZTc3ZGEwOWViNTQ3Y2VhZGE4ODZhYjQ5OGU4NDJiYjRlM2ExYjllZjNmY2I3OGUwZDQ5MzM0Yzc0OTk5NTYzMzE1NDhjNmM2ZjY4ZmRiZTQ5ODZjM2YwMGE4MTAzODFlYmE1MzYzZGRlODQxMGNiN2M0MGJhZWM2NjUwMjNjMTBmMTIxMmVmYmQwOGEzMzg0OTFlNjQ5ZjljZmJkYjljZGNkZDZkNjk3MWVlMjdlZTVjOTFhZDJjN2U0NzNjNWVkNWFlYWMwMzk3MjlmYzBjYTY2ZTA4MjMxMWQxZTdhY2Y3NmI4MTFhOWE4NTIyZGJlYWFkNmM2MzE2NjBhNzBhMjhkYmEyNjk5NTNjYTBhMDMwNzNkODE2ZTM3MjMzYjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IylmPZMMcvGVuT8dtZEj-memQlw_T-RXgFygaHs0nl-iYZkzbb-_n461dmAX3cbmNJuu1MA0OWG5WsXNry26pw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20200926_113127_32_2412_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.994Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IkgyK2pLdDUvT1lyVjRvTFowYndyU2RMTnNVNm5wMVpneEE3OVN3eEM0MUtPVzhwdWRDbUlXcEVDUldZRTVTSUY5dVErWThDTm9ncGwrY1ExQnJhRVZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMTA3NDNfMDlfMjQ4OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2M3NjVjOThiZWZkMjQ1ZDU4MjM5YWRiMmYzYTI0NGYwMGJkNmNjZmJlZGU1Yzk1ODBhNzA0MDk0ODM0MDIzMmZjMzQ2YTQyMmUwZjNmNjI1NzcxOGRmM2EwNTk2MWNmNzEyZTNmMDk1M2ZhNTNmMjgxNjc5NGY2YjVjYWEyM2FmYTc2ODNhMDExZmE3ZWE1ZjVjOWNiYjBhYTk4YzU3OGY5MDU2MWZhMzUyMjVkNzRmNzExZjVjZjVmOTU0NjY2ZDhjNjBkY2JkZTQ4MmNmOTc4ZWIwN2Y2MTY3NWQ0YWQ3Y2I5YzU2ODIzNjYxYjBhMzBhNDUyMWE1ZTA5ZGJkYzFmYjRmZmVmMTJmMWE5NDA5ZjJiZmY0Y2YyYjkyZDY2YjM4ZThhYTMxYWYyZTkzYWFiOTU2NjE4NWMxNWZjNDEzMTUwY2I4YTUyMWY3MTViMzAwYTQ4OWRhNjA2ZTZlZGUwYmQ2OTA2NTc0NDI2NmY4MGYyZTMwZWU3MjhjODI1ODA5ZWZkM2MyYmE4MWI5N2MzNTM5NzIzYWNlMzA3YTAyMWI1OGMwYjE1MGJlMDhlOTIzZmM3OWUwYTIyYWY5NjQzNzYyNTRkNjFlMzVjM2JlMjgyMDQxNDA3NjA4ZTI0YTIwZTA5OGNlY2Y0Nzg1MzkwZWQ3MzI3Y2JlZWQ2NzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xjqU3m6NkGtiLCbe4HM7n2TFwdt_BhhZZ-Zn89DZAi54dXUzz7YxP6HpuzHBYEoxikifWru_pYJMzCz6155IOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_110743_09_2488_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:32.998Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6IjBZakYraldYL3FOQ1JOMVhrMHVLNWFMSk8zcXU2YTBDaEgzZHBERGJtVTNMMVVwaGdIK1QwS1RlU01IUituUlI0YTNYTmRDS2RhVG9rcEgyTEs2c0dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMTA3NDNfMDlfMjQ4OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDc0NDExODY0NjBkZjM3NmMzYzM3M2EwNjRhMGIzM2IwYzJjMGU0Y2NmYmZlZmFhYjlmODUyNzY1MThhNTE1MmVlMzVkOWRkMzJjYzY0MDA2NGMxMTY5YjExZmViNDlkNTk3MjI5YmI0NmM1NWE0ZTQ1MTM5NzMxODgyYTlkODQ0ODg2MTgxZjZiMWJmN2MxZmRjYTcyMmQ2NGM0OTdkMDM2YjU2YTA0NTI2YjlmZWZiOTUyZmExYzA3YWIwYmM4ZDQyNWE3MDA5NjQ4YjgxNWQwZmVjY2QzN2MyNzlhMTA2NTg5NjA1YzM0YzRhZmE4M2IyMjMyOGQxZGQyZDAyNjgwZmQ1ZjZmYmZiMTMxYzE3OGIwZTQxMTU0ZDI1YzhlYmM2M2MzZWUzNzdlNGQzNDZhODAzMTRiNWJmNmExMWUwNWQ4M2IzOTg5NzFkZjJmN2M0OTEyYzI1MTRhMzRhZjFhMjFiZDQzNzA4YzU4ZGZiZGU5NThjYTEyNjU0OWE0ZGU3YjM4NjYwY2MzODU2ZjA2YjEwNWNlY2NkOGFlYTQwNzRjZjQ1YjAxZTZlNmQ2ZDVlMDMyNjMzNGY4Njk2M2EzZWQ2NDkwYTViZjk1ZWNkYTkyNjliYTk5MTE0Nzc2ZjQ1YzcyNmQ5ODQxMzZmMTM0NzgwMDk5ZWIwN2RlNzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FX5-qHnGMUVVbZDp9vj7aJgWgQ8F4Ans0ERGM4SWr0J_OIph7vpneqZ0Wkz-p6yuHikbmPpOSfNa-mU9vCN_JQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_110743_09_2488_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.003Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IndjSjVsakhQdHpaYVp3M3BvbVJWT1ZOT2ExWXJSc3U3WTM0cUh5Qlk4eHVlU3Q3TXJGVTd3Q0FQeHErV3RKMnJkdnZXWFZXVng4T1NpZFd6c09vcUR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMTA3NDNfMDlfMjQ4OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmZlZWY0N2ZhNzYwZTdiYzM4YmRmYjgwNTRlMjIzYzY3NzRlMzkxMjQ3N2NiYmFhNTE1YTNhODk3MGI2NDdmYWUwYzVhM2ZiODc0OTY5ODZmZjMwMDgzM2FhOGQ4MGRlMDc0MDE4N2FhNzY1YTU3OGM1MjliNmI4ZmQ0OTg1ZmJlNzBjZTUzYjVjZTM0ZDA1NTJhMWY4NmRjM2VjYzU3ZDU2ZDJiYmU0YTg5MWQ1ZThhOTQ3MjczYWFjNzM0YTA3MDc0ZjI1OWE0NGQ5ODVlY2M4MWY3ODQ0MmQzYWVmZWU2MjM2ZDMzMTFmZDNkNzY5MjA1MmMyYjIwNTgzNzU1ZTFiYWRhYTBhNjk2Y2U0YzJiNmM3OWIxMjY5YmMxNmJiNDZmZjYwNGJiY2NiZTRlMDlkYTE5OGEwMzdjYjUzNzNjMjIxMjE3YjdiNjg0YzRjNGRhMTY0ZjUyYWJiZDJlYWI4ZDhiZTAxMDFlNzU1YjliYzBiZjBlNWNmMjEwMjY2ZDZiNjZhMTA1MDhiMzMxNzM5ZGEyYjQ4N2E1YjdiNmRkMTNkZjA1NjQ2ODExNTJlYzYzYzc3MDVlYTIyM2NjNTUyM2M4N2ZkM2ZlYTYwMTAzY2U3MDc0ZGY3ODZmYTJiNzgzMGQ2MTM5Y2IwOTA1NWM2ZTg5ZDI0NGY4YjMxZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ft8n-cLgll9X-aIDXwLkPRyMy7IaEKe5diup2e2-lvhPMXufiJUnLkcDCc0OPyQK06lQcps3MsLlhenqV2j79g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_110743_09_2488_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.006Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImVTUE1lcVE3YXJUSEF3YmJUZ1ZhT3RUWmJDQVljSitxbUZndzdYL212MzhHSGtybzdwbXZkdzhDMkhIRk9FT3lCb25vNldJQm1UdE5CZnBZT3VsVkRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMTA3NDNfMDlfMjQ4OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTgxYjhlZTUxYjZhZGRkYTZjMjI3MzU3ODhmMTI3NWMwMzRkNDZkODY0M2NiNTkwZGQxMjJmNmFmNWYzMjkxYjc2MmQ5N2QzZWUxNjE5ZDcyODYwMTkxY2QxOTI3MWNhZjk5MWYyNWEzYmMwZWI1MmMwOTdlMDViNDhkZDUxYTMzNGFhZjY5MGI4ZjMwZGRhMWY2Zjk4MWNmYjAwYzNmNDJiNmIyNGY5ODU3Njc3MGUzYTI4OWRlOTJmNGViNGU3MTM5MGJkOTllOTdmZjE1ZmIzODY1NWZiMDRlMTQzZmNkZmIwYzc4NTA3ZDU1M2NiNTYxZWFlMWQxMzQ0MGRiYjgwZmFmYmEwZDcyZTY5M2MwOWM2NDM4ZDA2ZDY1NjIwNmE0MjIwZGM3ZjJjMWRjYmUwNzdmYjY5MmI1NTg3MThhYTVjZTRjYWY3MGViYTcxMTM1MDRmYThkN2M0MTRiMTg2OTkzYTgwZGMyNWZjNGUyNmMxODg3OGZmYmM2MGUwYzQzNGExNmZhMzcyNjQ2M2U5MTc5YTg2Zjg4MmE2ZmY3NWJmOGEyNTBhMDA0OTYzOGE5NGFkODhkZjg2YWU3ZDhjYmE0Y2RmZTc1OTE4MWVhZWU5YzhlZmZjZTlmYWFkNzRkYTJmYTlmMWY1NjQyOTM0Y2I3Y2IyNTdkZDkxZWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PG72dZ_8yTpFC8Uz8n_kPbGBxrbqclDOV3sSGnAZULT0g5RGdyQf5VJJfKjQE1h3cvo9nKyRmQe2OQmPXMPeNQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_110743_09_2488_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.008Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9MZUNFNUFSZTVkZmNXMlYxcm55T25ndnV0UXlGY3J1SzBJQnJtOVVrY2drR2IyamR1Ym5FNGthb1FOdzZxUmlrTVFYR1VKaFNGc0l0THlYNjlmSDRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDMwMDdfNjRfMjRjYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTUzY2I1N2Q1MDg2MDg0ZTA4YjMxNDM0YmMyNjNlMzQwNTliNzI1NWQ4NjZlMmMyZjUxZWMzMTczOTc4ODEyNDY0OWIzZjRlZWM4MmZhMDcyNWMyYmIxOGZiNGRlZDhjNDM4NDQ5MWM2OTk1NGNhMTk2ZGQwZTJiOTYxYThkZWJkNTM5OTE5NjAxNDI0OGZhNDYwMGJmMGZlN2U3ODYxNmE1MTlmZjRhODQ0MmIyMmJmOTYzMTU0MTBiNzM5ZTgwY2NhNGVlNGQxZjAxNWYzZWVkYjkxNWUzZmNmMWFkOTdiYTVjNTRlZmJhZmU5NzdiMjIxNGUwOGFmYjgzZjg3ZTY0NDZjYTFhYmM3YzJkYjM1MWZlYzhjMTM5MTIyOGM5NmFmMWI4ODljODAzZjQzZjg0MjVhYTkwMWMwNzY5NmIwYTc1Y2U0ZWFhMWY3YzM0ODljZjliNDU1MzRmMjMzZTQwY2M1MzlhZGUxMzc1ODI4NmZjNmU5MjA4OTliMTk4NzQ5M2RmMTA3NDAwMzcwNzRhYWM2NjIxNzg3ZWFmMWRlNWUwMmE0ZWY2YzU3YzdkZTg5MzNhNGZkN2Y4ZjE3ZjMwNDE5NjQ5NzM0M2QyYzUxYTVlYjc2MWQ5MzQyYjZmZWYzZWRjN2I5YWU5ZjE1OWVjMjNhOTVkMWRhYmZmNGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0UETKftgZ1YSKQ1lgLLeGBb3dHp_jMyW7orb3n5JAJc0MqLc6FT1CBcF0ZLj2ODC2zB9aXU4ttVdMmKe2hcN6Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_103007_64_24ca_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.011Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlFRSHRDditOdHhhMkRvNlg4aXFXUkthTXdLdnVISVJkSHdVRmIxdnlrV1NvYWRMcHVkaHliY2Z2eFYwQnMzc0lBODNaUDZwTXBnN05zY2FoeHpDM0p3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDMwMDdfNjRfMjRjYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWUyOGJkNzc3NmMxMzUzMmQ5ZDUxMTMzYWZiMDhkZTQ0YzExMWQyN2RmOGM2MmJhYWVhYjU0OTM4ODRjNmQ0YmQyNzczZDEyOGJjMjA5YjBlZTNiMzE3N2M1MzY2NTI4YWQ0MmQ3MjY3OTdmMmQ3ZTRhNzgxZmFkNGQ1YjM0ODcwMTMyMWZkZTYzMDFmZWNhMmI5NTYwMmY3NzVmNTRmZDk4MGU1ZGQ3NjVkYTJiODEyODJkMDZmMmRjMTE0MTRiYmRlZmI0ZTRkNDcwOTdjZmQ5MWE3OTBmMDRlYmM3OTliZTU0MDE5MzkwM2M1NGEyNjE5YjNiNDhkYjBjYmRiMGY0ZjdiZjk4MDY4ZTI5NjRiZTAyM2JjNDQ5Mzk4ODkzNTUyOTQzMTczMWE3ODliNzRmZDQ5Yzk2YWEyZGVhYjU0NDVjY2FmNDYwYTdhMzdmNmRjOGQ3NzQ1MmJiZjQ3MjU1ODZmOWFmNTdmNTEwOTNhNjE4NjU4ZGFhMzhhMDQ4OWYzYmNkNjJkYzE4N2RlYWM1MTMxNWI2YjU4ZmFmYzQ0ODQ5MjJiMTFhNzZlYmQ2MTk2MDMwMjU5ODM0OTdlMDBiNDgwMTI1ZjUwNDQwMmU3YjVhMzFkN2EwYTA3MzZlNWRmY2RjYmRjN2JlOTc4MzMyZmRhOGE0NDMwYTlkMjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rMxfGaMQJgtxC9hbDMigLSBBHb8AwWH0c9mORsiarIG_T7eDiH8p4ZcQ9LbxqfDX-EqDk8WG2AmZdmHsC3ZPZg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_103007_64_24ca_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.014Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkZpanFUMTVwV3RtRkhhVTgwaXlpb2tBZytuL3BFcUxiZFIzQUs4Z0Fxd0Q2YlZPZm0xOHhqSkdYS0xHRFhVUlczaWR5MnVPVks4blFVbUsvb3lQVDFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDMwMDdfNjRfMjRjYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjc3YmQyMDgxZjViYzgzMjQxOGJjZTU4NWJjN2NkZDJhNWU5NWQ0ZmE5OTE2Y2IyNDMwNzI0M2E1MTI1Yzg0NzRhNTQzZGMzYmI5ZGRjZmI2YjY3NzZkNzJmYjA5MjVlYjZjMjk2OWQ5ZDZlZDVlZGRjMmFmYWQwZWE5YzQwYTFiMTIxMTQwNDQwMWZmYjI2MzU4YTM1NTg2YzQ2M2FjNDZiZDRjODk5ZDg4ZDRhODg5MWZhMDc5YTc1MDA2ZjAzMzVmNmJlYzYwNmI4NjNlZWNhZjk3N2MxNDU3ZjFhNzVjNzViMmI1NDM2ZjhlZjNkODE2YTdhM2E5ODc1ZDkzNDE5ODUwNjVhZDBiOTcxNGY4ZmQzYmQ1M2Y5YWFhZDc0MjA0MjBkMzVhZDlkNjI2MWQxODkzYTA3MDhmN2I4ZGQxMWZjNjUxOGJmYjJmZjQ1NmZkMzIzOGVlOWRhNmMwN2FlMWZhMTFmYzgyMGZhNTBkM2UxZjk5ZGNhMmUwZTFhYmQ4Mjk1NzI5MjIzOGQ2MmUwMGNiMzdiOTEzNDlhNTNlNzg0YjU5ZjI4YmNmNWI1MjQ1OTAzNDRhOWQ5NGRmNGM4NTRhMThjYTU5MjVlMTZlZmI2YTE3ZWU2OTE4NWE0NzJiYjQ1MTkxMTk3OTM0MWM4M2ZiODcwNThiZjE2MWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.N-nVUvJXfa5HNC3j1LzIl_GOvpy9nsmUk-FPM8KnSPbdEwTjriTPsNjlkOjufS7JwEXvuOeMWKY9KiiHT0uUdA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_103007_64_24ca_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.017Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxjY3doRkcvWHJKYkdlSmdrUmdxcFNjTkY1L25tYi9nK1Z1Z2hMSjBxN1RwN3hlOW1LaitCY3pQV1lJU2YxMUdKeUZ4dXVZdm4rdzd0ZU9jUGdndFFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNl8xMDMwMDdfNjRfMjRjYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjIxYTJlYzg0ZWUwMzdkM2UzMWU3ZDI1NjhhNjNiNmMzODdmZDU5N2Q3MDI0YTE3MzE2Y2VhZTA1OTg2ZGY5OWJkZmUyNTUwYmYxZmU0MzIzNWY3OGU4OGMzMThjYzBmNzIzZDAxMjE2ODc2Y2M5MDk3NmIxNDM1YjRlMTI4N2ZjODc0ZTIwZWZlMTc3MGE1MGM0OWI0OTVlYWNiZmFkYTI3NjliZmM2YjIyMjUzYTg3YWQ2ODRiNTlhMDEzYzE5YjA4YjQ3OTM4MjE3MjliY2VmMzE2MDM5ZmE0YzkzZmVkZDcwOWEzNDNlNWU5NjUwMGIwZDNhNGMyMDA1Mjk5MzEzNTU3NmFkYjljMGQ2OGRlMjliMTZlYTlmOTE3YzNkZWUzOGZmMmQ4OGE5MWE0MzY1ZDE0OWMyZGJkNjQ1MjA4ZmI3YmU0NjNmMDk2MDgwZDAzNmQ0MjU1MGMwNWVkYTczMTYxM2VkZGMxNTI4Y2E2MTJiNmI3MGNiOTExZmNkYjliYzQyOTY4NmI0YWI4ZTQ5ZTk1ZWZmZDE1ZTI2NDZkOTZkMWZkM2JkOTQ1ZTQzNmUxNmRhMDY0M2Y5ZTY2MDU1MzA0YmRmOWI2MDlkMGY4ZDc1YjY5OTY4NWMyYmQwN2MyZTlmYTkxZThkOGYxZTI1NjdmZjIxMDY3OGY3NzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.q2aRH90hZshhtzYniWSOj5rgCMl3qw1GU1BWk6zCb1Wf67Kpw6NrPtp1qT4ezORS-mh-pMmedi1NLFPyt4dABw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230526_103007_64_24ca_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.019Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJWTE4vbjRLQmEvcDhwSEhkcmpyWVZWTXdJTHNyWmpDK2Z4eGJpQ1U2dmo5OU5pa3FhaVVUZkI3VGRQUkJXWUl0b0pPbDQvaXk5QmNiazRlUXJZaHpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMjBfNjdfMjQ1OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGUyZmI0YjNiN2JjYWUwMGIzM2E0MTE2ZTAxNzA3OTQ1NGQxNmRhODBkZmI1N2NmODUyYWQ2NTM5NGFjZGIzYTFlN2E2MDAzOTQ3YjFiYjc4ODFmZTU4MGYzYTI2MTNkMGU1MTI5YTlkODM4OGM1ZDQ1MGNhMjIwMDFjNmU0MTM1OWI3ZWMyNWQ5NmExNzc2MWNjNzQ2NmI4YTE4ZWU4NjM5OGEyNDJhM2MzOWJkZmM4MWY0NWJlODNjNTQ4YWNjYWRhZTA0MjQxNGUyOWFkODNkOTFlYTZlNzMzZmU3NzA2OTZjYmVhYmQ4Y2VjMTcwNzY5YjY4MmM0MmQ4NmY4ODNkMTU1OWY2YjhkM2U1ODMzYjlkNmVmYTJkZTdjY2RiNDU1MWY1ZmMwMzFmNDlhMmVjMTU4MTc4ZDJkMWY2NmI2MDNmYzVhMzllOGFlNjE2ODU4YjdhNzllMDVhZGZkN2FlYThmODIyZjc2MzUyNjk5Y2NmNGQxOTUyZGI5ZjQ5YzY3ZmVmZDIwY2VmZjI3YzcyM2Q5MzUwZDU1MGJjN2U0MjdiMjNmYmFjZjMxOTZmOTVhNzc0ZTg0NjljOWM5ZTRiYTMyOWY2Njc4Y2FjY2QwYjUwMjE5MWIyZjg1MGE5ZDJlY2JmYTg4MzYwNGFkYTVlZGZhMWRmNjQyMjNlNzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZAXseHJcpdZJLAJSMtHPbM6wyPRg_YwNk-FMU-zTG29udx4VxB0NusES8tWH9HKpMK7AbAMqPidrhpup79XQFw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102020_67_2459_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.022Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ilh0M0NEQzBtWWpOZkpjbTl1TkVJTnA5cTlnUEFGNTlhUDkrVWVqVzRNc0thVWUraDBhc1hpanhreDZGbi9FZlhCNFJ5SkxjVDltS3FSTWJLajVNQ1lRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMjBfNjdfMjQ1OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODM1ZGUwNWZmMzg1OGNlNWQ0MzQzMjRhOWUyY2I1NDdmNmExYzkwM2I0OTU1Njk3Zjg3ZjQyNGY0NDA4M2FmMWMzMjM1YTA3YmJhOThmMzkzZTRkZDcyZjFhZTExOWMyNzg5NmY1ZGNjNWUwODYxZGFjMDY5YjQyY2E3MTgxNTAxZDg2NWIxZTY3MGJlYjc4YTIwZWEzOWZjMmYyZmVlMjg2NzBkMTAwYTVlMzNhM2UwZDI1MDBhZTEwM2M1MzZiY2I3M2U0MTFiYmYyNTY0NmE1NDNmYjdmOTY2MzE0YTBkNzIzOTIzNDMyNTE0NjBiMzJiYzM0MzY4YjEyZTY4MzY1OTBhODgwNDdlNTlmYjMyNmQ5NGQ1YThlODY2ODk4MGVhYjE0ZDE1NmE0MjY1MzA2MmQ5OTRiNDllODNmYTkzZDQ5MjZkOWZiMTZkMjI1YzE4MTczODI5YTUxNGEzOGVjZmE5YWQyYTdjYjVhODY0Y2Q0NDk3MGMxNWVjNDIxMDI5NDQwZWEwNzgzOTdjNDVjYzg4Yzg4NjFhNTNkMmNmNGNjMTMwNjcyODlhMDYzZGQzYWVhMmIwMzQ2NDJiNGIzY2MzYzU3ZTA0OTU5YTJlYWIxZjM1MWRiMjYxMGQ3ZGViNzIyZWNlODIwMWQzNjRhODM3YWQ4N2RiYTUwMDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AR9qyDh3umNj13pMP_IG3XoM5kNP8f2lIYALtcyuJsHERqWbQz5kwGwB_MdZPjC-iR5Y-nI6eXCjzGo3iP7dZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102020_67_2459_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.026Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVPeHFJUy8rUzNFN28wcFR1MnZCVHhob3drOEdveGt4aE51OFBKTHlkeU1SWDlNakZUdVdSNDVSc0ZsWDI3Uk5RcUgxNXNEeTZycU5yUFFJQVRXS2R3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMjBfNjdfMjQ1OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzY5ZDFlYTViYTNkNDY5MTQ5OTg1ZDIxZDVjZWVjZTAzMDFkODgwZDllNmJjYjIwNWZmY2MxYjQyNTE3OTgwNGYwYTk1ZDQ5ZDkyNmJiYzI5OTI2MWE4MmE2OTk2YmI4MWU0ZjUyYmRmNGFjNGMyZTJlYTQ2N2QwMzc2ZDIwMDc0NDI2ZjAxZDA5MTBiZDgwMTk2NGJmNmNlODcyNThiYTkyY2MyMjYwMmYwY2U4MmYxZDIwY2Q2MTMyZjgxMGI0NTQ4NzA1NjI4ZDU2YzExOTYxY2U5MGQyNGZlNDgxZWQwNDhiZjhkMjQ3M2MwYzMwYzhjODExOTNlZjNhZDkxOTZhZDkxOGQ5MzAxYTVmZDY4Mzk1ZTdkMDk0ZDBkMzAzNWZhNTJiNzViN2FjMGEzYzQ0OTY3MTQ4YjQ1NDcyYjVkYTllY2RhODRiZTlmYjc5NjE2ZGE4YjU1YjcyOGRkOTRiYmFiZjA0ZTlhYWM0NjM0NjU5NjZkOGFhN2Y0ZTI0NjU2N2ZjYWY4YWQyNGExNGUzY2NjMTc3NGNhMzEwMzU1ZDg3NGI3ZmNmYTA3ZmYxZTE0MTIxYWZlZDc4NDBlOTg5ZmY5ODkzMTJjNDczZjRmMjZjYzRmNDdjM2RhZDI2ODFhOTkzMDUxMWIwNDZkNDhkNTU4NDU5ZDAxZDU4ZTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BC2_iqGWe4_bxu1WbyL9mdWpzkCmwkmfFCFmBqLZuMB7zyXVtTiq02cdul0aePyO2FfGmDnuG7g6KK1NLkI4Dw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102020_67_2459_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.028Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjVHWFRxSzhnTmhEb1IzcmppS0ROR09UT1d0aCtaS2l3Y0JEdlpqRGhWYlM0RVZsNHNNeXBPK1E2eFlMeTBvMi9idG5xcE5wYTVRVmpyemszL2RweWJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMjBfNjdfMjQ1OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjE0YjUyMzU2MTVmMzk4ZTNiMWNmNDhjOTkzYzllZjRkMDlmNzYyZTE3NTIzZWFhM2MzNTA3N2QzODQzZDI4ZDE5NDQzOTgxMWE3NzcwZmNlMWJkYzBkMmNlNDVjN2EwNmY5YmI1M2FiNWE4YTE4NTczMzVhY2M2OWJiNTBhZjk3M2QwNTcwOTI1MjBkNmI5MDVlYWJmMTJjMWExYWI2YTQ3OWU2Y2IyODg5YjgyNzU1MGY4ZmM4OWQzOTUyNGViYWQxYmZkODMxY2NlOTQ1OTI2ODcyYjU5ZWQ2MDFjMzU3ZDYyOWEzMTgyNjZmMmMzNGU3NjlmOTgzM2FmYWVmODA1OGE0NDc3NWI3MWFhMGIxZjE0YWM0YTAwNjBlMzZjMDY2MTA3NDY3NjYwMTk5MGQzMWU3ZTYxZTJmNDQ1MjRlM2Y4YmM5ZGFlMDk1ZjRhMmMxMTg0MGE2NWVlYWYxNTAyZTcyZWFhZGY1ZmM5NTc2ZjJlNWU3Mzc4NzRmNTlmN2VlZTlmYTU4MzA5M2I5ODA0MzI2YzMzM2NlOWJiMTEzODlkNWFhZWE0YmY2MDVlNmM4NjdhNTVmOTdhODkxOTFmM2VmZWU4NWQ4ZWEwM2ZhYjI3ZTIyNGQ4MWJjMDFmZDVlMTJlY2ExZDFjYWQ4Mzg4NmI4MTVhYjdmOTA4Y2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bLRA7BTNICvR_RPytMD-Nx0P9RUnBh1oQ0Tv6I2mq7_6yWgRGEPCHSka2dLJl6RveoJvgX-QkvNtit-VSDvqIQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102020_67_2459_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.031Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImRuamdjQXdIUHEyTCtsYk15ZVQrYytsVElrNXl4RjJYK1Nkcmh4RFpIWnFQL0RLd3NpaTZaWml4bWtvWHMvcjFUV0tTYWplSENaWEVGM1YxOFM3NGJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MDlfNTBfMjRiY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjExNTk4MjcxOGM1MzYzZTAxZDNhOWIxOTk5YmYyZWE0ZmY4YjkzNDU2YzM3NjJhNGY1OGY3NjE1NDc0OGQ3MGIzYmZiYjRhYzI4YTFkZDI0YzJiZTE1MGIzNjhiYjg0MWRmZWE2NGNmMzRhZTJmOGVjMjRhMjQ0YzcwZGQwYzkzOTFkZjJmMTUzMDVlOWI3ZmFkZTdhZTIzYjBlMTE0NmFlYmZiODRjYTVmMGJiODllMzhhMGRiMGQ1YTc5MWY1M2VmNDk4MWI4Y2IwMWYwZTc3MmY1M2VkNDg3YjJlNjNjM2U4YmE1OWZjYTU3ODIwYmE5MTFhYzc4NzZjMjQ2ODk1N2M3NDVhZDg0MjE5YTVlM2E5MjYzZmY1ZmZmZjBlYjJlOGNlZTNjYjZjY2VkODY5MDU1OTIxMTVmYjIyODRhYWEwYjM4NzUxMWY4MTBkNGMwZDFjODA1ZTA1M2ZjNzlmMDc1N2Q3MGEwN2M3ZmZjOGVhZWNiYWE1ZWEzNzg1M2Q3ZGE1NjhlNTQzY2E4YjI2YTQzYjg5NzM3Yzk4ZWE3ZmM2MWIxYzU1YmFlNTRjNGY1NWRjOGZmMTY3NmNkNTY4Zjg3NTY0OWRiYzk4MjQzZDE3MWRmNGM2ZWUxMjk5ZTcwYmJjYTE2ZmQ0ZmVhNGRmNDNiMDdhMzIyNWExOTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0c65NzJUzK_ilEqSJad1pJWs0UCkYt80_fBJ7midF4Pe0amYuU2I-Aa5DPOqDrvIilPw8RPhjMWpCGpvKrR8-g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103909_50_24bc_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.034Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImM0UFhPRW8raHQ3dVkzcEJkeEZWZTlHUUZiZUNnT25xMlhqRUl0UXl0blNVL1NBU3d3V2NzRm85cm5tZG4yMmZmU2hqMTVpM0VnRGh4cmJ6UERyTzlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MDlfNTBfMjRiY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmE1ZGVlNGU4MWQ0OWM5MGE3NzRkMTgzZDM0NmZiZjA1MjMyMWI1NzRiYzQ0NTc2NWUwYzAwYzBjMTliOGNiMTliZWE1NzU4N2Q0NWNiZDU0ZTEyY2MyNGEwY2YxZmE5MTQ5N2YxNTVhNTQxNzU1ZTAzNjc2NTYxMDhhZTEzNzM2OWE5YThjMDE0MTU2ODE5NGE0ZmM4NDY1MjU0MWU1ODUyZmU0NjIwZDBhMjFhNWM4MGFlYjA2OWVmNjg3MmE3MTUzMjA3NWIzMjJiZTViNGZlNWIyNGMyNTI3MzFlNjEzOGFlODA2M2MzZmNkNjUxNTA2MTc2NGFhYjBhNDA4OTVhMTM1MzFjMWRjYTBhNWYxZjhjOTQ2Mjk5OTU3Njk1YzVjYTliODY0YjA5MjMxZWFkN2FkNjk1YjNjZDI5MzEwMDYyYmZmOTU3ZGUzOWNiMzM2ZTVlNWVjMDBiZDA3MjdjYmUxMzM3YzAwNzgxYjA0ODUyNjgxZGMwZDUwOWY4MmYxZDllMWMxOGFiZDAyMTAzYmQwZGEyOWQ0N2ExYzVkMGE2MzY3MDA1MmJjOTdmYWM5OWI0YjJlOGQ3Nzc4ODNkYzUzODZkZmI2ZDJmNDUxNTZkOTA1YWU2ZDZmZmE0MDcxZWE5ZWNiNmJmZWVmZmM4ODQxZThiOTFhZmM5MThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EwhFZf4jJqp-fs0T4LbTNtdeTtZiDvgricd2RW2G55Bb2Q33zf_yP4RRJSmxlUZN6KeSroRYgX-FR_VeSVJPMQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103909_50_24bc_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.037Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlpKYU5CQ1JOdVJzQ25XK1NySEQvVzlnTzFCUExMK3c0RE8wTVkxcXJLRkVBNWErcVphRU9NeGNTTG1iWUJUdXVzWnFoQi9JU2p0bzB3NGxZS3ovZ0RRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MDlfNTBfMjRiY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjA2YzM0MTUyNWYwZjFmNGFjMzE4MjFiY2ZmNmEwYTQ1MTZlNWFkYTM0ZWI2OTg3ZDZiOTliODYwMGI5MDY3NzRiN2Y4NDdkMDk1YzNiODg4MTE0MzRiZDQ5MjBlN2M1YzI2NzFkNTc4ZTU0MWE4ODljMzU5MDUzNmNmNTIxN2Q3NGMxMjc0OTBmYzdkNWVkODJiNTc3ODRiMTg0ZTU2ZjA0NzBmYmQ0ZjA3YzUzMTdjMTQyZmJlNGMwNGMxMGQ2YTc5NTdkMTAwNWMwM2RhMzg0NWM0OTY0YWM2MjE5NTZmNjgyODNlMjM2Nzk1MzQ3ZGMyOGMwMDg1ODgwZTA2NDIzMzM1ZDBhMWIyM2MzNzM2ODgxMzEzZWRiNjMzOWYzZTc0NTc5MWRjZWFhNGFhYTYwMGI3YTBhZTI2ODZiNTAxOWRjNzc0OTlkMTVjMjgxNzllNjI5YTg5YmYxNzRiNGQ1ZDBjMWNlYzVlNDExY2JiMmFhYTEzNDY2NDEzZDNmNDExZmQ5MDk4MzM2ZmFlMjE2MzYzOWIzYWEwYzI1NzAzNGEyMmZhYjU0NmRkNDUxZDgyZGZhZTBiMjdiYjJhZjZkZGY4MjI2ZjlkY2Q2ZjcwYjkyZmIwNTNjMTM4ZTIyNGI0ZmY0Y2RkMjQyZGYzYjE5NDIxZTg1NWU0NGMyMTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cPjs0rI-wejOyGDSZymoTiBTdcCWXucTtne_HWc9rSA373Bn5YWXzgoAyGwP4fl8zRqa-GVlHa5r8wb3Wrzxaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103909_50_24bc_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.040Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJOV3Q4MkN6bldjTTdKV08wRFN3T3JrQXFZOWQvbWtnUXJqWHZKQjNCdDNxUzc2TnQ4S2F2NE5rS0VROStrTXhaSnhST1FWMit2bEY2d1JTbUFnZjlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MDlfNTBfMjRiY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzExZDc3MzExYmM3ZTk5M2FjZTVmNThiNTRlNThlZmJmMGFiMjVkM2I5YWFkYzI4OGYzMjVjZDZiZjM2ZjRlYjI1NTQ4YjEzZTRiMjJiZmIxZTA5NDllZjdhZWY0NTI2ZWE4OGEwYTEzMjNkOGM5MTg5NjNjODU1ZWM5M2YxNmE5YjhjNmQyYWVhZmU5ZTc2ZmExMDQwNzE2ODUyZmQ2ZjJhNWJlNGU4ZWU1OTM4Zjk0YzZkYTVhYTNhM2VkOTE2MmM4NWY3ZjJlZDkzNjdiNTU1ZTllZTAxZmVjMzJhYjhjNjliYzAzZWYzZDZhZWFiMjI3NDNiMzc5M2EzODY4YzI1MjQ0N2ZiZDM4M2VlMjJlNGI0NjZhMmUwZjBiYmI2ODExM2VhMDEyNDg2OTUzZjYzNjY2OTI4M2ZhZWNhY2E5N2U3MGJjOGYwMzYxMGUxOTdhMTdjYzgzMjEzNDhjMTk3ZGU4MGIzZDU0YWE1NzFhNjkzZTRiYTJhZDJlYTY3MWEwOGRjN2MwOTEyMzgxOGNjZjQwY2Q1MWQxNDVmY2U3MDVjNmZjNzhlYmY0ODk5NTMyYmQyNjA2YzQyZTc5ZjI0MjI2NDNhNTRhZTBkYjA3Y2JjMGM5OTYxZDNlNjZmMWQwM2M4OGJkMWFlMzA2MzczYjM5MTg5Zjk5ZTZmZTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Vj6-9gXoPBt-lldJILxoieaLzgXqwVrqKAUGizM96tgJOuUGwU0U0WLS8oyTf1rOru2Nh62cCPUSorXPSjyjHg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103909_50_24bc_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.042Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlPUnRaclBQY1YwNk41WGx3bDYycDFzdnFrd1YzY2l1OFFSWEViL2o2MkVEdmN0amMzQzJqcnV6eWtSK1V3N25yeGxzc3l4bzl0R09xMjV1dkJRc21BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDRfNDNfMjQ3Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjJkZDdjNzBjMTNmN2I0NmIyMDNlMTExYTI3OTFhMzRiZGJiM2MxODMyYWUwYWU5ZTcwYTBkMTM3MzM4NDdhNzdhODNkMDYzMWE5ZmE0ZWViNDVjOTBhZTc2MWY0M2QzNTEyNTM3Yzk3ZmQzZjJmZTIwMDJkNmZiMmMyNzU4OTQyMGRkMzNjZGZmMjk4ODNiZWVlYTI0MDNhNGJlZjdhMmUzM2ZiNGNmODYzYjAzYjE4MGE0MWYzNGM4NWU1YTNiYjFmNWQ1MzVhZjMwYzdiZWUxZWJiYjM0YTZjYWNlZGY4Mjk0NDZiNjgxZjQ2MDRmYjczMDllZWMwMjgyMzcxNjQyNjkwM2E5N2IyMTA0NjU3ZDE3ZmU1ZmUxY2IxZTBmNjQwNDlkNjM0ZTFkMTRhZjUzZGRjZTk4ZDBhNTNiODMwZjQ2ODc5ZjU3N2UxZDhiYzU5ZDIxZjRkMDBmYjBiZGRjNWE2Mjc5OWQ0OTViOWRjOWFiMDQ2ZjNkODJiYTE1M2RiZmUyY2VlYzY2M2E5YWUyMmZlNWI2MjczMmFiNWU4MzE5MmUyMGUzOGUwNzEyZmY2NWJkODFjYmUxMzQwZThiNzQ2NjgxZWRmM2M3YTZiY2IzNjllOTA5MDU1YTNiZWYwOGI4M2U4NTBjYWUzMjA4ODZmZWM3NmZhNmMzYTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UmJFZSMx6nPvWsMfVRtCcVXK4o_5EzNdmAAmrMc0LJ6PASfqqLnp7pr4tSRf67M0W940CiQBCM04_ribR3B3dA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111144_43_247f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.047Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikk4c0RtRy80bDQzbU9BbzQxMVlDc2tCbERtb1ZzOXozMWUrVU5EM0JkLzFNcEJkY2F2VS9SL0V5clFlWjhiS2pJdk1wclc2MmVWcHJGNGR1UWFEd2FnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDRfNDNfMjQ3Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjRkNjU5YzNlY2YzZTU5OTU2NDllOWUxZjcxZjhhOGU3NTVhZGJlZDcyMzRhNWExMmE4YjFlZmY0YjVjMTkzZGNkZGNlMzU0NDliNWY1MDM1MjEyNjQzM2JhYjhkM2FmMDRiOWU1MWZlM2IyNDExNjU3MmYxMzQwYWQ5MTE0OWIwZTFiYTc2MTk0YjgzMmJhMmNmNDczYzU0YzE5NmZkYWVhOTllOTk5Zjk4YTNmMzU0ZjJhMGViNjI2Y2MzY2IyMjk3MmZlZDQ1ODk5ODlkOWFjZDNhOTcxY2Y4ODc4NTYxNmZiYzhjZDM2NThlODYyZjQ0MjIyOGY4NWJkMjkxYTUzNDcwNjc0MmZjMmUwZmFmMzAwZTFjMWVhZTlhNDNjOWE5MzU1MTY3ZTRmYjVjNzI4OTc0ZGE5MDkxOTY1MTgzNTdmYzY2OGQ0NzU1MzU0N2YxYzI1MGFmYmI1ODIyZTEwNDBkNjE0MjI5MjA0MjAzNzkxMTNiOTI2MDk5ZTM3MDc3NjI1ZjU0YTI4MzU3ZTUyNmVjYjc2NTA0ZjUwMDdiMzk5OGU2NWU1MGM2YzM2MWVmZDk0MDIwYmM4Y2VjNzNiMWZhODkwYzE2NjA0Y2E5YjYxNGExYjBmZDhiOTE3N2Y0MzFlNDJjNzZlZWFmYmI1ZThhNTZhOGRmOTBhNzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8ZYKKKMAygEDh0zBAbGzPmgAOl2A-6Y3uxMvuW9bW-JMO4zETENyTYl2ZxVa4SG4iKe09BI4g5FBBculL0tEcA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111144_43_247f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.049Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9RZDBiYjZmMUdUWUF1YUVRZ3ZLWDRLdnhHSCtKNzJjVFlNK3AwMU5ZOU1sVTBLUnkyVEZpVFdoZ3Fld0N6VGpLbWticzJrZTNHRWtUZFlKeVhxMVlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDRfNDNfMjQ3Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzdmMGI2Njc3NzQyMTc2NzQzZjAyMGMzMTU4NDYyNDAyNWM1NDViYzIzMTM1ZDgzM2NlNDNmMWU4MDNhZDUzMzY1NjFhMjhjMGI5NTdkOGFkOTZlZWRjNDFkM2EzZTY5YzRmNDM1YWQ4M2NjNjlkMDI1ZTZkODFhYzdkNWQwZTIwYTA3OTk2OWZiMWMxMTM1MWU2NDE5YjZhNWZkMDUxM2RjZTY5MmI0YjNmOTFmNWM2YmMzMjZjOWIwODFlNDBhOTUyZGE0ZTJhNThiMThkZDRhNGE2ZTI2MDc2ZmIxMWIyNmNlMzZhNDk1NGFhODFhN2U0NmU0YzcyODI0OTMwZDNjNDJjNTJhNGRhZTBkZDQ5Yjc5NzgxYTMwZTQxMjllZjBiYjAyMWZiNzQ2NWRiNzA5NTI5ZDczMmU4YTQwYjJhNjdkNzVlZGMxYzhiMmMyZjgwYmJmZTkxOWY1YjBkZWNjOWIxZDE5NjdlNWM3MDdlMWQ1NDg5N2E5ZDZjNTZjZjE0MWU3NWY5OWRhNjhlNWFiMTgxMTlhYmM0YjI1NTg4MTUyYzkyNzY3NTk4YTAwOTU4NzY3ODg2NDMzMjVmYTc0ZWEzYTlhZDQ2NTkxYWMwNDczNGY5ZWNjNmRkZmFjNGIzMTBlODA5MzVhODNlMmY4NDA2Yjc1Y2IyMzViNWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aajjlUnH7LIdddT1w2vsuZGmL2towxUz4j64joXyHfaB5bmns9S27nlf0afzNfxcMGCpS_CKiq9_41VXawzSxg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111144_43_247f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.052Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik5oMFlvQUZrUlRwR1V3ZlJXM25JMHZzcFFYS0RsbXRjK0tkV1RwWWpxbEpsNUlLTWFjRzUwbXNJRDVmUGUxTVB2VGV0VzMyWjUzTTdDK2xkNy9jWERRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDRfNDNfMjQ3Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzIwZmRkMTM5MDMxMGJlYmJmNzBmMTk0ZjAzM2Y1MjNjZmYyMTY1ODQ1MDUzNWE1ZTFiMDkyYWIyZWFiN2EwNjM2ZmM5MDZmZjJmNzk2MWRkYjQyNjEzNDk2ZWI2Y2E2M2ZjZjhjZDY3OGJlYTNjODRmNmRjNjQ4YTU1MmU5ODAyODU3MTBiOGMyNWM5YWFkYTM3MzczMWVmOWFhMjI0NGE0MWQ0Y2VkNWQ2OTAxYmEzYjk3ZmJkZTNlYTlhYWYzNzZmNzVkOTFjZjVmMjljYmQ1MWIzMDlmYTQzZjcwZmUwYjZiMzk2NDQ0MTA4MDhhZTI2NWY0NjNiODA1ZmU5NWVmMDBmZWY1OTUwZWNkNTcxZWQ0NWIyZDcyYzA3NjQ2Y2YwNjNlMWRkN2UzNDc4Yzg1Mjc5ZGEwM2Q4MGJhMTg2MzI2MjU2OWVmMGEyNjg0NjQ3YWY4Y2M3NmU1NDgwOGQ3YzY1MzZiODVmNDdmMTQ5MDgyMDhjNDQ1M2U4ZjI5OTlmYzhhMGIwZDY3NTg4ZjNlMzJlYTIzODYzYzliZGU4YWM1ZWE2ODQ3N2YyOGRkY2ZmM2M1Mzg3MzRiNGJkODE3YjQ0ZjUyMTk2ZDJmMDdmY2M0ZDg2NmQ3ZjJkN2FiYjY3OWZlY2NjMTc1MTVmYTEzZjhlMWYxYzlhOGY5OTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5B5qzuaxJb7dFxB02wKKMhDb1m3HOaawQfX2f_utUyP2vW1JS9Xu1p4I08CtQcOaa_ktiUDglq2tG0iNLbGd2A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111144_43_247f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.055Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtYRDFNVW41YXJYTjQwWHR0eXFvSm96M3lHaVVWNzZDblVHVytyTXJsa3d4Z0lSZERhOGNxVWNueFpIVFlpd1VjQWtZK1p0L2VGdkM5cUpDV2pWdk1nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTVfMTZfMjQyMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTJhZjZlMWZhZmQ2YzMwMDBlYWIwMzkyYjkxOWZhZmM4YjU0NGFjZjE3MWEwZGJmODZhZGE4ZTQwMjI1MGI4MzdlNmQzODQ3YmQwZGFhMzk2MzBlNTZmYTUzMmY3ZGUxZmRjYjM0OTI4YmU0OTExZmZhY2ZkZTZkYjIwMTk5ZjI2OGEwYjRhMzQwMjIyNmVkMDUyZTNiZmE5ZTVkMDQ2NmFlZmQzZTZhZGMwMDc0ZGZkYTAzZjNkYjliODM1Mzk0YjEzNzRiOTVmNTBmYTdkZmU1YmMzMDNjOGQ3ODA2NDk5ODdlOTg3Zjc1YWY3YTEyYjdhMGIwZDcwZWNkYzQzZThlMzgzOTk4Yjk5MjMxNWUxNDEyZjdlZWI1NDFiNzgyZWVmZTNkNWM1NWViZTUyODhiYjVhZGUzZTFiMjAwMTUxN2YxZjA4NWI3MTVjMzgyN2JmOWUzZjlmYTdhNTMzMDliMjQ4ZDA4MTY2YWJkNTZmZjUxYzI3Njg0NjVkM2IxOGQyNWVjN2NhOTE4YzEyNGZhOWZmMmNjMGU5MDA2ODhkZmNkOGZlOGZiODY4NjA2NDEzZTM4ZTc2NWNjMGE1NTMyNDcyNjBhMDk4NDQ4ZTUzZDdjNmVjYzkwYWY0MDNiZGMyMTQ5NmNiNTRmYzhhODk1NGVkOGU5NzIzYjAxN2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DUKXFAmmWad7DC1skV16U4feMvnTVYHS2OiP5sGRa_aEez39sQwJNw3dTyOn1opTCpKYSTldDyKDoLz2x7pHuA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103455_16_2421_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.059Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik02bm5JYzJPWGJDVyswLzNXakx1LzNBWXE0bUJlY1FnTTYxKzE0OVE3QS9UbUk1V3dWY1RvUGJMQ01waTlQQm9FT1JKdnU2NWVycm5MYkZJc3F1cnNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTVfMTZfMjQyMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTI1ZTIyNTZhYjQ2ODdiNWVhMGZjZTA2YTk5Mjg5YmUxMzRjY2Q3NGRiOTFlZWVlOTk3MDZiZjBkZjNhZjg3MTRjZTU1OGZjMDI4YmYxMzAxMWQzYjBkNjFiMzc2OGExYTcwMDQ3NDNmY2U1ODZhYmI3ZDg0NjJiYTAxYTZhMTEzMjE0MWU4MmFhZjdiMGQzZDAzMTdmOTM1MWE4YWU0OWRkNjNmMzc1ZjMxNWZjNWJkZmRlZWRhZDQxZjViOTdlMWVhMzU4YjU0M2EwNjUzOTE2YTM2NzZiMTkzMTA0MzBiNTc0OGI5YzI4NDUyOGFkOWM3M2NkMDY1YWJjNDkwODVmYjU4ZDZkNzNlMjBlZjJjY2EwOTAxM2ViYzUwZjVhZWNkNjVlNjc4NmExNTA0YTQ3YmE1MjRiNGQ0OGM1MjE4NWIzZDE4OWRmYzQwMGNmNzM3YzQxMzIwM2NlZGE2OTU2NGY3MzEwYTQ4YWY2ZDE4ZTY1MTgxNDRkMDc3YjA5ZDY4YzRiODdlZGQwNjllNWJmMGE0MTU5NDMxMmExYWJlYmZmMGZmM2M3MjM4MTE4NmQwM2VjYTBlZmIyMzI0ZDUxZmNlODMwY2E1MDU2ZTE4ZGFmZWYwYWQ1ODlmODBmNDYzZGY2YzY1N2Y0NmU5ZWIxOTI1ZDNiOTIxMTZiNDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qir5RcRTMpaNjpdLetbnmQ6NmZccAUUS866324pjaOmlsFUKRat8xBctQ9CKBHaWMqZneFV2l4ryzfp4Wanqbw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103455_16_2421_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.061Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhoVG81dldKQUhpTWl0bUVrOW1kNUZkNW5Ub1ZKZVZxQWVqNWZVNEgzUGlHUG10VEV1Y0hwajdVSHBOenFqTFlQU294RkhCRmZYSzg5cnoxdW4xUWlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTVfMTZfMjQyMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzNjMzk1MTBmOTE0NmMxYjI3MDFjNjEwYWU4OTkxNDliODdlOTdjODcwZWE2OWEyZDUyZTc3ZDAwMDNlZmFmZGJlNjE4ZWRhN2RmMzM1MGVmNTlkNmVmZTg0MzdlYzk1MDg2ODhmNWVlOTZlMTg4MTQyNjU1NDcxNmIyNzJmZjJiNjY4NTVlZTFlMzc1ZjBhMTg1MDMwM2ZjMjA2Mzc1NWNkNjMxNWM1OWFkODU2YzU4M2U5N2FiMTIyYzAyYWI1ZTZmNGY0YmI1ZGNjYjBkZWE5MDRmNTRjNzk4ODJhOTAwY2M5NGYxMTM1MjQ0N2Y5YTgxNjUxZTk5MTUwYWJiNDI0MTY0ZTI5OTU3MzYzZTU5NTY2MDA0NzYyNTZhZjUyMjkwOWVmNzcwMGQzNjQ1YzA4NWU2MjBhZjliYTc1NDBhZDE2MDUzMzQzOWI2OGFiNTJhN2NiZjBhYjcyM2EwMDE2NGQwZjY4NjhhZmYxNTYyMTgzYzQ1ZTE3NDk1ZjMzOTY3MWUwZDM4MWI0ZGFkNWE2NWI5NmQxMjBjMTIxNzAwOTY5OTQ4MzI1MjJlYmE0YTg0ZDk5MDU2MDAzYjhmMzlkMDQ3MDQ2N2QxNDRhZWZjNGVlMzUwMDU4ODI4MzRhZDMxNDFlNzdlNTU2YTVkOGQwNzllNzc3MzhhOTZjY2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iomdHbSv1SjeuB43x8Ra01FPBZbSQw_W-62hrguilUpraXdVxoccOJV6csYEdG-6i9Ubfa0qrYLFk_r4tmlmZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103455_16_2421_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.064Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im5tNGRvR0lqRy83WjZnWTNSZ1lIWnFiaE4reXg1d3FJLzlhcjhjMU9vMWM5ZDJzU0xVTlVjREIrS1plNVNsWnIzbzA0TnJBVEhMbkRGdWhXZ2Y0cjRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTVfMTZfMjQyMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmYyYTUzZmQ3ZjNiODVjOGMxMzVkMGFlOTEwYjhkYWQ1M2NhMDBiZjZjYWE5MTg3ZjIxZDJjYTRmOGRkNjg1ZWI0YTQ0N2ZmODJjMGY2ZWUyNzZjNzdhNjI4ODFmZTVlZDkyNWQ3NTQzODk3NmU3MjczMmZkYTY1NTA2NWY0MDc5M2VkNmMyMGFkNzliNjUzYzk1M2IxMWJiMDk2YTFjOTc4NjUzOTNkYzZlNDU5ODM0OTUzN2E4NzgwM2EwMGRhZTQyMmI3NjdkMzQ0OGE0ZjQ5NWVlZjJkZWMzN2ZmYTI1YzUxODcwODgxNDljYzBjNTViOGFjYTRiMTIyMDdjNWRiOWE5MWJiYjQ1NDQ4OTIyODE1NTFkMDgwNTgzNGY5NTUxMzQyNjAxYjFjNmQ4ODE3YzkyM2Q3N2M5NzQ3MDg3ZDEyZjllYjcyNzljMTMxYzdiODNjZjczYWY2OTFjZmNiZGI1NjBlYWRlYTJiMmRmYTE2YWEzZjg2MTZlYzY4Yzc5Nzc5NzVmNzI5OTg3ZDM5ZjdjN2Y2ZjExZjljYzZhMWExZWY2ODMyMGIwNDI0MGI2NTA0ZmUyZTNiZTg0M2Q1M2NiM2MyYjJmNTJkNjBmYjNmZWI5MTNhYjdkMzQ2NmQ2ZDUyYzI3ZjViM2YxMjQxMDc4YTI2MGQ4NzQ0ZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sF5McQ3cn5oUZ_2IiMeFDLZquVxGKWuudKDlJYD-GiLD68D0MtBYt6XxmDDnkgmFVwomxVU9sSeEAFi2brySow", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103455_16_2421_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.068Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZ4S0JST0JDeGw4M050NHV4N3B3YzRzZTYydVhsYWgwZEtjWkFRZTBUc1p3VFYyS1hvdkxJM0NsY21tSjFxL3ZVSXFtb0hrbWQ3WXBzWWYvcWlTdGJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0NDBfMjhfMjRiM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODI3ZmQxOThhZTk2YjBkZDNjZTZlYjlmMzg4ZDRjNmZhNGI3MWEzYTE0M2UzMDRkMGIyYmJhYWMxMjc4MjVjMDYwZjkwYmNhODZjNTM5Mzk3ZWZjMWViOTM2MDgyYzU5MmRmOWJiMzVkYzRmZTdlNTRlZGU4MDdjOGIxZGYzMzg0MThiOWU4NDdlNWY5MjM4MzMzMzU4NzFjNDc5ZGNmM2Y0MjdiNGYwNGVkYmFkOGI1Mzc0MDE3YzJkYTVjMThlNzI0MGM0NjkzMjA2NTk4N2ZhNzA3ZWZhZTFjYjZiOGQ1NjJiMWFkNzRiNmE5NzUyZjYyNWZiMzkzZTg4NTA1YzFiYjFiNGVlMDU5MWU2NzViMGM1ZmU2OGQ3ZmVmN2I0YzExZDJiYmZiMjczMjhiOTlmMTQ2YzQxYjBhNGE0YWMyN2FkNzZhY2YyMDk2ZjE5ZTk4Yjc1NjZkMmY3MmViNWRhN2FhZGZjMzcwMDRmMTFmZDZmZGQ5NTdhMWUzOWNlNDQwNDRlMjAxN2I0OGVkZTRlNDkzNjg0MzYwNDA3MTJhYjE4ZTAyYjBlNjI0ZjJlYTc4ZThmMmFkMWE4YWI4MGY3NDI4NGE2ODg2Y2FhOGQ1MGNmZmFkMzIwOTg3MTRkZGE4MTlmOTI1YTA0NDRjOTUxZDRhYTc5YzNlM2I0NTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ueJqP-9Ip5Y3JLrBbhXxmzE_tKmhQGPCzvIt3lNgG_uQxKoB_V4V5zlAC2-1ZGnFJxf22qgDRTzro9kyTZibRg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103440_28_24b3_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.071Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZRQVRQZmFnbXQ2YkFsdUFXMW41eHdqVlVNN3VpRTdtdHF2d1FadjgwYWs2TzQ2a2IzMkZ5dlZIQU9mU1U4K212aWprZWdBM1EvOVJsZCs0SmtEeFlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0NDBfMjhfMjRiM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGJiYTIyNzVlOGM5NTUwY2Y4MDk5NWU4YTRmMDYxYjIwMDQwYjQwM2FmODczN2ExMmJkOWNkMTBlZjgxMzg2MDU4YmNkOTI4ZTllN2FjNzczNGYyOTA2ZGM2OTY2NWMyZjA2OTQ3ZDViYmQzN2U5ZDU1MjU3YzEzYWU0YzdjNmEyNjA2NDhhZGQ4N2Q3NTBlYTZiY2E1MmEyOTgwOTU5MjI4YmRmNWNlZWY1NWZjYmNkMDY1MmU5ZGVhNzhhNDhiNTAzYjY1MmYwYWM0ZTM3ZDdmMDY0ODgwZDhiYTA2MjFmYmZiY2VjZDRlNGRmYjBkZDMxYzNkNTE1NmZmOTE2MmI4MTRmZTUyZDI2YWU2ZDRkNDg5NjYyNzIzODQxZDVkZTIxMjMxMTlkMDFmZDNkNmZhN2RiYWFjNWM3NzVlM2UwZjQzN2E0YWJkNWUxZmExMzE0MGU5NTFhNjAxMWNlOTQzZTgwODdkMWU4ZDExMjMzN2I4NmNlMTkwNGY3YjMxMzYxNWJhNTI5ZWEzNDVhNDQxMWVlMGEyOGNmZDVhY2M0YmY0MzMzY2ZlMDU5NzNhOTYyYjIxMDIzZGE3MGIzNGFkNzEwZTI2NDZlNDZlYzAzMGI3ZDhlMDM2MWQwMDNmOWM0MjM3ODI1ZmUwOTM4NTMxN2E5YmM2Yzg2OGViMTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zcgzZrCmaEHSw9xurFN8RG6Y2jB0V-mCspmxns_hjZUm3yzgWX2-CFMG_M45LqbKIMHllEOTXMxvIbXoiDX8JQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103440_28_24b3_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.078Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpKbVpPOWpwdmFua050NkhyNHE1UkJXSm5wS0tNUDhVMkdXVXd5VUxqdms5RU9oeEVCbVlTeVVwVzNVSlpxMG9FSjlhMDdmcFZ0OXFxUDVjcDh6RFFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0NDBfMjhfMjRiM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQwMTgyNWE1NzNkMzk5N2I0NjJiN2M4YzIwZTdmNTI1ZjhmYTg3YmQ5YzY5MmVjYjdmN2MyYjQ2OWM5ODU1NzNjNjE4NzY0YzY0NGZjYzg4MmU3NDM2MDI4NzI0ZjMyZTNjOTY5YWNmYmJjNmI1MTYxN2UwZjM2ZWQ0Yzk3NTg0YWVlMDYzNmU0Y2U1ZTY0OTMxZWFjNWY3YTM0MDg1NzAwN2U0MzlkYWYyMjU3ZDY1MGE3MzBjMmNjYTIxMDQ4OTU0ZGYwMzBkMGIxZmJlMjFmMmRkZGE1MWFlMTBmNzNlODg2NDFiZTJiYWJmMTk4MGI0ZDVjNjRkMWMxMGVlNmRkMTYxYThiZDgzOTY1OTVhMmQ1MDlhNGEzNzIwYjFmZGI3MTJhM2YzZThiNWI1NTg0MGU1ZTZiNjQzNzAxZTg3YzNhMDZlMDY0ZGM2OGMzNGVlOTI1ODc1MDFiZDVhNWQzZDQ2NzMwMTRiODhjZWVlNDY5YzQ0OGQwMTkyMzAxNjQxODZjMjIxMzczZmE3YTVjOTFmYWMxNDczNTM2ZDA1ZDY0OTgyYzQwYjkyOGQ0NmJjNjQxOTQ5OTA1Y2YwZWEzNmUzYjMzMjNkNTk3OGMzZjQ5YWU4NWZjMzhkZjI4NWQ4Y2Y4ZGVlMTM4NTkzNjRkMWNlOWExNzBhMGE3NGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.72URjSSuTeWeUy7UTokrNQV_1V-G3ZPmoOHM7alA8BDnLdLKYvDDmBEoeJ6GylFHJ9yI9FVSTyu132LLsQu_FQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103440_28_24b3_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.082Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjQrM2RleWwxUGxnRUU0S3ExbW11TU9mL2VrMkZpM0FqZ1Q5ZjF2RkhadEhORGMrL0tLWjJHRWErbko3eEpKSUVNeWdENnk1aG1vWkhxak5CaThwNitRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM0NDBfMjhfMjRiM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjEzYzM4YzViMDk3NmU0MjMzYWY3ZmQ4MWNjZmZmNjMyMDFiNjE3MGM5YzViYjEwY2FhNWM5ODdhYzgwMjk5MDE1N2RiNmQxMTJjN2M1YjJiY2MyOTc0Yjk3NWRiMTg5YjBhMGJiY2ZkZjg1OTMyN2JlNmY1MDFkNDYzYjNlYzgzZTlhMzY0NDA0N2NmMDU5NWM5NWNiNjFkOTRiNzE1ZjgyMDhhYzA2MjNlOTIxNTRjZmJiYjQ0MWM2MTgyODFmMDJlNTAwM2U4OWYzYWNhMGEzZTA0M2U2NWRjMDAyNzJjMGYyNTU1ZmM4NWQzMWYxYTEyOTgzYzI4YjBiOWE0ODlmNDBhZWI2NWJhYmNmNzUzYTVjMWRhODBkZjMwMDcxN2Q2ZTM0M2FiZWNiNjM0YjlhOWMwN2QxMjI0ZDVlMWUxMDc3NzQ0NmEyZjk5NjAwMGJkY2NjYTlhZGFiN2MxNTU4ZTI3ZDY5Mzk5NmI4MTQ0M2MxYzY3M2Y5ZTVmOTljMzk2YWVkNmEwY2FhNzk5MTFkNjI4MTIzZWZkMDllMmNjMjc3YmUwNjZjNjcwYzMwYmJmZTBiNTg4NzQ2YmRmZGU4ZjMzZDQ3OWJlNTg0ZGJiM2ZhNWNiYzA2YTc4ZDQ3MmM4ZjRkM2MzZGYwNTNmOGRkMzNmY2JhZGU1ODNlZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MQ74Y9WhTb2R2K5ut4u2skopsIHuOidP62NaY_49LyTJfHoTGM4myh6mLUJXf62tRSnYHqbfRHyheChQIYjIOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103440_28_24b3_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.085Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjdiT2ZTZUlQVmpEWiszZjhJemRGV1pvM3JhcytzdUEwTkdVdDV4VnU0NHR4MDVrSWhBcmFlV3ZzVXVXaC9HT3VsZnVEVjhMRTdFM0c3YlVSUUpBVjdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTFfODJfMjRmOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODM5YmUwMjU4MGI4ZDIwM2NkNjI1MTJiODY0ZDdmMTUyZTU2ZTQ0YmY2OGE3ZmFhMTdlODE2NTRkYjgwYWJiYmM3YTUzMDM4YzFhNTY2MzliZDY2YzQyNTE3NGFlZmI5NDc1YTg1ZDVkYzEzYzlkY2JlMGUyYjMzNjJiYTI0ZjdlMGE1MzJhODU5YzRiZTQ0NWQ2MWZkOTk0ZTA2ZDY4ZGZjZDgwY2I2ZjQ0Mzc2ZTIxZTAzMDQwNGFkZmI0NzdjYjQ2Njk4YzJkZjA4NmM1MDE1NGExN2YyZTNmMTIwNjkzYTI0ZGJlODUyMzI2MDQ2NWM0YjZmYTRmZTBlMjE2NWRlYWNkMTZjNDMyZTJjZmM4OTY3YmYxZDg1YWRkODM4MDRmNTYzMGUzNmZmZmU1OTY2OTQ1NjE3OWQ0ZTYzYWM0MDU2ZjRkYWM0MWM5ZWVmNzVjYzlhYmJiZDIzMTQzOTlkZjQ1MWNkMWU1YjhkMGM1ZDI3MWEzMmZiYzNjZjI4MjBmZjdmNWVjYTM4YWFkMzFiYzFjYjc1ODE2MTFhZWUyOThmNjQ5YmM0NWZlYjI1Yjk1MDk2NjNiN2M4OWQ0ZGFiZDQyN2I4NDcwZTdhMDU5Mzc4NDg0N2UxZjM3OGNhNzExYTRmNThlMDA4ZWEyODA5YmQzOTEyZWY2YzBhY2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.aw0ghhT15VBsGD5zYXKoQGZ5wcVUt2wfkMBgtwE-zu9ocnOdjCLBdgbhqOpXuhD9wWjXPUhM7CtuOFsbeUfeGA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113311_82_24f9_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.090Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJVODQvVXRMK2xBcGlnd3hOa25oRm1EK0V4V2pGSlVIMFo4L0xKcjRDRWtDWklHNE04cHJUcUVFVk8vbzNTK3VMc2hvWjFoMkJpdW9OZFdmTXZHV1dnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTFfODJfMjRmOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjMyZWQwZWU2OTdiYzcwYmYwMGI4YzcwNDIxZmVlZjU2YzkyOTQ1NWFmZDhlNzk2NzFhNWJhZTU5MmY5ZWU4MmVkZjhiYzdjYWFlYzFmMjkwNDk2MGJiM2I0ZTIyOGJjZTk4NjRhYjk2ODA2Yzc1ZjY1MjA3MjAyZDVkMmUzNWNkZWE5N2FlNjBlZDM5OTY1MWNhMTZhYjU5NmUyYjRjNDQxOTdhNGRmZTNlOGZjY2FhNzU0NGZhNTQ0NTU4OGI4OTBjOWNhZjU1Y2Q0NjJkM2RkNTNlNGQ3MmEzNzc1NmU0OTQyZjVmZjdiMzc0ZTgxMjg4YzU1OGI0MTMwMDA1MTNkNTA5NjdkMWIxZjg4OTExODkwMzgwZmFjNTdkNmU5MzkzMGI3MjQ4OGFkMWRkYTUwNjhmMGYwZTYwMTVkN2IwZjA1MzA1NzY4MWY5YTJjMDdiZTlkMThhYWE3MjBkMWZmYWJjNDZmZGI4YjNiMGJjMTVhN2RhMWYzNTMzYzA0NzUyYmQ2MzYyYTM2NmMxZjE0ZDEwMTkyZjNlZjcxZWQ0N2VjNzlmYzdjZTk1MDQ4NjkwYTA0YzI4YmE2ZGQwYmJkZWYzOTA0OWVhMGY2NWMyMTI2ODg0YzkwODkzODA2ZjBlNDlmMjQ2MTc2OTczM2U0YWZhZTA3YmRmMGIzY2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.kefKjfGs_CGhI3kAosSlb1-pJX9bWp4wGdvsrojAvodtmERWRIbS3vB4gEixsLkGbu2KLa7FKAtRm54mejcGRA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113311_82_24f9_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.094Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImQ0ZU1FakdLcmxpc1EzaHhHU3d4Z2ROTEkxK05SZHhZRHZhaFNwNzByaUlrMGJaOTduY1VlVUdwc290TDRJTHhodkpGdkJ3Zy9iWmFiVkhzOHZoN3RRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTFfODJfMjRmOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGViODg3NDY5MGJjMWJmNWI4ZTQ0ZmQ4NjQ5ZmMzZWI1YjdjMWJmMTI4Yjc5YjQ2MDg1N2ZiMTQzZWY1NjliOGY4YzkyZmI3ZThkMDJmYmE0MTVkMDIzZTYwNDM1MTlmNGJmMjg3NWU4NjMzZjczNDkwNjEzZjA5MWQzZWY1OGE3ZWJiYzQ2YjBkOWUyNGJkMWE3ZjE0MTg0MjA2YzE4NzU4NTZiZTI0YThkY2I3ZGM2MDBhMjAzMDI1ZTZmMzliZDFlZjZiOWFkMDk4MjVjZGEwNWZlZGU1N2M3ZDY3MTZlOGM2NzY2NDczYzRiMWU2MTRlYzg3NjA1YzJkMGYxYzQxNDU4ZWE2NmY4OTU1YTA3ODUyOWRlYzZiM2JjYmEyNTI0N2YxN2Y4YzdiMTA3YjA2NTNhODU2NTdmYzQxMzk2MGMyZjBjNDI4YjQ0OTcyNWQ3OTEzOWZmNjdkODk2OTcyMWQwNDEyYTVmZjgyMGI5NDBmNzIwODhjZTFiOWNkMzE0Nzk5NjgyN2EyNGQyODQzNTFlMDlmYTA0MTZlZTMwMzkwNDUwNWNjMDg2YzVlOWNkNDRlYWNjNzlhYTI3NDU1OGQxZTY5ZDQyMTc5YmQzZjkxZGUwYWQ0MjVjNTY3MzE5MWY3ZDkzZGRjZWI0ZWIzMWZlNjlkYjRjYWJmOGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Hd91Qn7CSgbj7tXkBThJEMjq7SjH83yFaI1iIhAuhV-1qWCjxP4MNz-pEvRnHakA4WF-k0oKhoDEm9APCjGJzg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113311_82_24f9_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.096Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImJCZmFZTmZ0VTAzK3FBd0RiYVp0MndLTzhKQzFrS1FmMDFFNFU2MkQzdlNjZWcwdkY5VGQxK3Rkd1ZXdlo3Q1dXcFlZelpsN3U4eDVkc1d6UTNScE13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTFfODJfMjRmOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzdhMmUyNzY1NTcwMWMwZWU0YzdjMGYyYzYxNmJhMWNmYTFkNjg1YTExMjZlYzJhMDE4MzM5N2IyODdjNDAzYmVjN2JhMTcxZTQ1YmYzZGNkZThhMmI5ZjA2MzgzYTkyYzcxODJhMTY3Y2QwNDA3NWQwZWNlMTE3YjJiZDMwMDRmZDEzMGMxYTE0ZTkyNmU4NWViNTkyYWU0N2JlNmI4YjI5YWVjMGJmYjY5ODYyNzgwODQ0OGQ5ODkwOGE1N2I0NzVlZWYyYzY0OGQ2YjI5MmE0NWI1M2I2Zjc5ODExOWJhNjBkYzkyNzBhZThkMWI2ODYxMDUxZjZjNGU5ZTY1MzAzMTdiYmVlMTE3MTYzZmE2YmJkMzFmMjQxOWE2OTg2YmUwMGNlY2Y0MWZkMzRmMDQ2M2JkMzgxN2ViNTFhMzYwNDk5YTgxMTMwNmZlZWM5OTQ2MzdjYjZiMDJmNDY4NjBlNjI1NjFlOGVhNzNkYjJlNDdlYWVlNDU5MGM0MmMxNjQ4OGI5NmNhNWYwNTU4ZjZiZmExMjQyNDNmMjllMTNlMWU5MDk0MDNjNDE5MmQ1NWU0ZjUyNjI2ODgwOWQ0MzNmMmI4OTA3ODUzMDk1Yzk0ODMxZDFhMWFhNTMwNjFiNGYzNjhhNWI2MWRmZjBjMmYxYWJiZmU1MTc4NGQ3NGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ygmGTKKg6y4n04JAxZtOlQ_bt6iLBmdYFKB4hkSMGq2S4hfOhBsLHUNRF-daLdcZSy8SAYdWAjP3nr1oOsaj8Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113311_82_24f9_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.099Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ing5ZWVKZzNlaDlqU0Q0ZVpxUk9UT3RXTzU0aHJzdTcrLzFkTkVhdXJiUG00Z0IzZTlyWFRPTzlRN2JjOWNDdWNYVThBSkEzWmJyeG9KRUgyVDczdDlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxNl8xMDM5NDVfODJfMjQ2NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTdmNTY5NTE5OGY1MzdlYTc5NzhjYzliMzFlY2VlZmZlNzQxODMwYzE0NWNhNjEzZDNjZmZlYmY0NTVlMmIwOTYyZjAyNGUxMThmMjRjYzlhYzg0NTg1NGY1NDA4MWNlY2NiMjJkZDAyYzIzNjU5MWRiY2NjMjcxYmUzY2IxMzA2NWZkYzAwNWQzZTNlYWFkMzc1NmUyYjc0NjEwYjRjYjE5YmI4ZjVkMzhjYTA5Y2JlYjBjM2Q5ZDRlYTg0NWRlYWRhYjUxNDhjOGI3YmY3MjkwMGYxNmQ0MTA1MGFkMjMzNDRiNmU2ODUzZjU3ZDQzOTM4NDczYzQ4NGMyMTUwMzA3NDExM2E5ZDFjMzJmMDBjZWUwM2NiOTgzNjg0ZjM5YTkwNThlZTExZjE3Y2VjY2ZkNDZjNzBmOWI4MWQ1MjViNmRkYTYyMzdiMjQ0NGVhMjU0Yzc3ZTcwNWFmYzMyMDMzZjRkNGYxODU1MDAyZDJjODYyMjRkMDIxMzQ5YTlhMDIzZDk1MDY1ZTI3MWE0ZGUzODg3MThlMGViNTk2MGU5ZTA2Yzk1ZWNmZGE4ZTRmOTc2ZjFmMzFkMzlkMDc0NGQwZDc5Y2RkZWJmZjY1MjY4NGY5N2ZkNDA5MGQxZTYyMjg4ZTU0N2ZlN2YzYTNiNTk3MDA4MGI2MjJhMjRmNjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nQMwGE8PxOvBbkIuAfkSlbW6YEdH5INijP7TznTYymOpZk_AhZOhkyozvrkK0Kc-GjnoE40LSbq68DpWszMzxg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210416_103945_82_2465_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.102Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkgrYmdQVzAvSVpzK3ROb2J1SDQrd0ZnSnk1K2lIcnlqQm5CR0ZTZzU3NGY0dkY2WkNGTnpQOFZmK0Z1aVR1WGxRalpydndLNUtkcWo2eURzOWJ1Ym93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxNl8xMDM5NDVfODJfMjQ2NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTNiYjJmOTAxODY0NGQ5Y2EwMGIwYjJmOGYyZWZmOWZiYzNhOWRkNWRlNWRhZTU2NzFjNTBhZTM1ODFmOWU4NWM5MDA2MjRkNjI1NTZhYzNhODFjNWRiMGQ3YzFhNTBlOGI2NDRiNjQ5ZTczMDVmNjZmN2ViNTE2YjU2ODJiZjVlM2FkODczNDRiNzI5ZjVjMGEwMzI4ODljMGQ3YjdlMTUwODFhOGVkY2M0MjY5MzU1YTZlNDU5MTQ3ODZjZGQ2MDA3NmZhYWRlNTZhNTA3NTQ1OTZkMmFkZDY4NzViMGJlN2YxMWE1MzViMjA3MjcyMzFiNGYxNDVkNzRiYWYxNDA2MTE2ODZiM2YxMjA2ZGVlNTM3M2ZkYjk3OTc0ZGNkOTI2MTIxNzE1YTkzNjkxOTg4NTYxMmIyZjY5NjUyYzRjYTI2ZDM2MzY4NTExNjczZTk0MTI0MjVjZGZkNzdmMDFiY2RlMWYxYzE3NGRiN2M5YmJhMmY5MThjNjU3NmYyYTljOTI3Y2RhZTNiZDI3YzFjMDQwMjAyNzcyZGNhZmFiZTQ4MDBhOWU2ZGM5ODJlZTA4MzI2NTMyZTgyYmMzM2E1OTFkNmJiNGQ2MTRkNTNhZDZmY2NhNDBhYzgyMzQwMjJlZjA3YTc5NGRlYjU3NDUyN2M2M2Y4NTBmOThhMDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2XQDFzMwxzaF3MJXr-0XRsYLsGM-hSQX-2vCsO0dBWjUKbZo6H51PBzVtWlWO3td29zNkwnKvPfBhbTHK4ioUA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210416_103945_82_2465_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.104Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Inl5c0VRZlVjNW5HdDFXSjBNTUZFN3JlTFlLcENhTDRvMzF1YVFQMVI2YmVJVDNQS1pPUWRGRVZyZVdMMTA0cDVNQUZ4eFN2VEh0WVZMWjBOdHU4Z1VBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxNl8xMDM5NDVfODJfMjQ2NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjliZTFmNTkyMTMyYzVmNzViZTY3OTE2NTQyOWUxYzY2NDU5YWI4MTU5Y2VjM2Y1MzQwMWFjYzk4Y2FkZmRiOTllNDM4OWFkN2U4NDYyZTVkZjZjNWNhYjlmM2JiNmNlNmE5MTRkYWNjY2IwOTAzNjUzNzNlODZkYWMwNjc0MzUwZDA5ZTI4NmJjZjE2YjQ1NzBjNjMwMjYwNTVhZmEyYTUzMmE5YjgwODNiOGMxZGQ5ZTAyZDM0NWFkNDQ0OGU0N2E0MmM5OTQzZGQ3MmFkMGFhMWRlOTNiYTg4OTY1N2FjZGRmYWFiODkyNWQzMTk1ZGM5MTU4MzUzZDc2MGJmOTI2MDRiZjRiNWY1ZGVlYzY1MjI5YmM0OGJhZjJhMzhiZTdhYmEyMjk2ZjdkMDA0MDA4ZDAyZjY5ZGYzOGZhNTljYzk5YWU2MTg1ODU1OTE5Yzg1NzBlOTMxOTI3MjQ0ZjU4Yjg0MWYwYTQ0YTg1OWZhY2E2MjhjOGQwOTgyM2IyNTdhZmE3Y2NiZGIxYWE1YTE1YTkxZWJhMzM0NjBiNjNhMWQxMTMzZjg5NWEyODNiYTJiOGUwZmRkOTIwZTgzOGRkYTk3ZDRiMDNmM2YxMDUxODQxZDQwNDU4ZWNiYjg2NjdlMWY3MDJjMjMwMWMyMjg5YmU2ODQ0M2U1NzQ0NTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.V21Fc4gMVz0Ri6erceNqiWs6sBGJTTHF0xj4kxP3Wdas3wsXcpD1fQwkrK5IT5BTqidrYQbzJM-RTqfuJqYU-g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210416_103945_82_2465_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.107Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFlejY0NzFvWnJXUGU5QmJIekdJMW15SENkV2hFYkROdm5WWDFPUks4Y3JhZHBSWUlvVE85bE1TM0NQWVowR0pTUytrbjlvTHJGYjJpOHFyNnpaakZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxNl8xMDM5NDVfODJfMjQ2NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDZjNTBlMzIyZjRiMzU4NWFlZTRjNDMwNWFmYjc5YTc1YTkyYzk5NzE3ZDg0MWM3ZDk3YzExODNlYjFkYWMyODk0OWFhNjU4NGI4MDYxNDc4MTBlMWRiZDQzOTVhMzQxNmFmODE1YTFlYzAzMzNmYzY5MWFlZGMxMWU0MmMwODQ0OWMwYWE3ZTNkMDc2YzZmZGE2ZGExMzE0NjE0NTVlNjEwZDYyMDBjYTVmOWM0Y2VmNGEzMmZkNzIzMjM1OGI5ZWU5ZTM3YjgxYjdjNGVkMTVhODA3ZjQ5MjkzODNhMjAzZTg0MDhlYTNkODdmNzNiMTdhMGYzNjlkNDM1MWE2ZDVhNDAwOGY5NjhhNWE1YjdkNzg2NTk2YjcyYTAwMzJhMDIxOWI1OWZiNmZhOWJhOTdiODBjZjc3ODJmMDUyYjRkZWU1OTgwMWEyNTVmYjVmOTA4MGVhNGViOTgxNjAyZTI0M2I4MDM1ZGQ0ZTQ4ZDhhMTk2NzlmZmY1MGQ3NmRhODViNzA5M2JhNTA2OWY4NTAwODhkMzRlNzE3MDgwZGQ5ZDZlOTI2OTlhMWY1ODcxMzMxZjNlZDFmMDI0NTE2MzNhZGUzYzM3OTk2OGUzZmVlNDIwODFmMjc3MjFkZTUwYTc3MmNlYTA3MTQ1ZDQ1ZGUwMmJhNzY3MDliMTIzODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WHYNjrlp4svwixymRjN1F9Ke-nbfoVJb9E9YfRCLrSg0nDMrjDdE87DiI5I_mHXD4mTukPbYkcToHa81bjM9kw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210416_103945_82_2465_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.110Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9ZR1RtTDVqS0JoanVpVHNiaUczeFNnbEViQWFickhRdGd2YWJzWVAyVlp1MmR0cHNSM3cvZ3l0aExlcEY3N1Jma1NVN2RjcFh3V3R5WWV2NjJRc2ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMl8xMTI2MzFfOTNfMjQwNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDdiYmM2ZGU4NzM2MmYxNzg5NDUwMDE4NzkwNWY4NWJlMDQ2Nzc5NmIxMTU4M2IzMGNiODM2MjA1NTIwNmRlMDA2NTliOWNiY2YwZWUwNTdjZWE1NmMzNzNhMmNiMThmODQxNjdhZjZmOGE3YmE4YWYxMTVkYTZlMTdjZDNkNThlYmUyYzE1ZWQxZjlmZTg0NmRiYmQ2OWQwNzFhYTUzZTY2ZmQ0MmFlMGE3MDIzZTk1MmI5ZTc3YzY0MTkxZGE2NTZmY2YxN2Q1MDM3M2E2OGI5NTJkNWIxZDhjZjg3ZGI1OWYyOTZmNmE3ZTRiMjA0YTU5MDRjOTc2YTM0ODUxZGQ3ZThjYjRlY2NmZTQ4MzY0OGQzNGEzOTZkMmEwZTA5OThiMzI0N2ZiNDE1Y2RlZWY0ZWRlYWI1NTMyYmI1MDM4MWZhNTljZGNmMjg1MDczZDQyMTE1NzY4NzQ5OTYzMTk2MDdmNGJlYTdiM2YwMDEwOTFkNTM0ZjMwZjg3N2FmMWNhYjM2MWNmNTI3NzExODU4NGJjYjJjNzliOWEyMmJlZjZhY2ZmNzU4MThmYTYyNGIwZDQxMmFkZmZmMWY1OWYxNDM2NjEzMDczODAxNTE5NWJmNDQ0OGFmMDYyYmY0MjQ5NGRkZGFjY2VhY2JjNmUxZTk0NjI1NDM0NjhjYjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oCu2LTBgesurd3_uZdBYy7OwvGOPgW7CCeCfB2Mt2CBlbNBeoD6eEqTGw8gMvUhRAHBFykJTj1OYWIqX00tjAQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211022_112631_93_2406_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.113Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImdGSGtsR1RIT21BQThWeGFuZExzWUVrNEhscWdTeEJocnUrblFXSmE5VXBJb0FjeHkrRWtiYWMydVJPRlVtNWVWSTNQbEtrNjlNRm1KKzA1NDBkUFZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMl8xMTI2MzFfOTNfMjQwNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjM4ODliZjQ4MDk5ZTQ0ZTAwZTU5MDZhYjgwMmFkMDY0MmQyMWEyZmY1MTBhZWFiMjgxZGZhNTE4YzVhNzBkNWU1ZDYxY2E1N2IyMjMyNWE4ZDY3MzMzNzdjYTY0ZWI1NTg5NzljZTQ3ZDA0YzAyY2I0NzJjNmZmZDA1NzIzNjhjMGVhY2U4NmZlNmI4ZDBmODVhZWQyOTdjNGZmZGU0ZDk0YWExMDUyZjYxNDEyNzI5ZWY1MDE0MTk1OTZmZjVkZGQ5ODU3ZDMwYzEyMmJjZGU2YTkyZDUyYmIxNjJkOGExZGJmNGQ2ZTFhZDQxNGJjNmMxYTE0MDFlMGNhOWUxNmU1ZTEwYjEzMTI1ZjEzYmE2MDFmZDE1NGI1MTNjNDBmZjZkYjcxNTE2NmY1MjY0NDQwOWU2NWZjMzdiYzczNTMzZTAxNDc2YWI4ZWQ5ZDAyZGE1ODFjY2JkMGNiNzcxYjdmNDI5Yzk0NzAzMjQ5MDZkZTc5MTljNDcyNGFmMmY1ZWVhMTM1M2IyZmZiMjRiZDdhOTEzMDdmZjk2ZmU0YWJjMzVhNmFlNTE2ZmU1N2NjNWQ1MzRkOTJhNDBhY2I2NWRmZjhhNjI1ZjQxMjg3YWVlYTk0ZmUzOWNlZjYxNzdiODVkNmQ4Y2ZmYzUyZGZlYTAzZWZhYjM1YWFkYWYxM2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eMY1VOAwb63krclzq_s4H3zBHHzZE2JdOPAlP3Yh5s8HgKSMyjD7OEklfE6A4pne569SYxoNXGlVp8tjXcjfJg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211022_112631_93_2406_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.116Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjdtVk5MVXFQSjZIVHZkRW5QaTBvMXF6WTFEK1JHSGh1RUQ3N3c5T3Y0VEdob0owYytMZ2tPc09jSlNndXI1MGJjVFZ4bDFMK3dGZVlqcHZMRE9VclF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMl8xMTI2MzFfOTNfMjQwNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjRjZTYwOTI5OTg1NDhhMDM5NDhjYmYwNWU1ZDkyYmQ4OWRjYWJjNjhkZDBiNDBiNTQyZWI1MjIwODdjYTBjYzAwZDU0ZmUzYTAyMGQ0ODM5NjBmNTZjMGRiMjJjNjEyOTdlY2Y3NWRjNzI4ZWFjMmJiOTM3OTc2ZDFiZmMyY2MyZmMyMmM5ZDYyODMyY2YyYzM2NjhmNTg0ZGZjNGM4OTg1NjJhOWFmZDkzOWFiYjcyZDc4OWZlMDcxOGE3NTcwZDI2YzBlYjQ1ZTcwOTI5ZDlkZGFhMmE0ODdmN2I5ZjJkNGE5NjI2MGRiNmI5M2ZkZjRkNGNkN2QxMmY3MmYyMGM4ZmZiODQ1MWJmMjQxZDk3NTYwNjExOGY3MzQ5MzAzNzZjMDI3MmRhYTNhYTZiZDI3YThkNmU1NDBjODU0MmYwZGQ2ZmIyZmMxNTIyYzEwZWM3ZDhlZDNhM2FkODkzNzA0NmNiZTg0N2Q1ZGQ5ZDE2YjZjNDY0MGZhMzI1M2ViOGU1OWE2ZGM0MmRlM2I2MGIxOGI3NjkyM2RhNGNmYjA2NWEyZmQzNzI1M2IwNTliY2ZjNTk5YmI0NTdkMTQ4YTM4NjFkNmJjMWY5NDNlMWMzZTc0ODkzMDU4ZTBiZWIwYTMyZTIyNmFiOWE3YjhjMjFjZTc1NzU1ODMxZTIxODZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DMbNBQlPfO8yjRUrUAMxDEPimeth_pzxqcn2heWp90ZPd1wpyag2fITycVH5JgS3o7ypswPOXB72_fNj5y7P2A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211022_112631_93_2406_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.120Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNOZFlKY2EydzhNSlBaOFFyQi9USktYS1YvOUwvMmpBam1kcVBRaXpYeHgyTmVNeWZBMXExODZkM1ozUmtZdzMwYXl3ZjZmVlg2M1J5dngzZFVScFlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMl8xMTI2MzFfOTNfMjQwNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjZiZTU2YmY1ZjhiOGYyOTFjM2Q3ODVkY2UyNmU3MzVmZGEwZDFhODNmNzlhMWU4NTNlYzE4OTgxOTk1ZGViYTk0OTAwZGZiY2M4ODg4MTEwOTk3NWZjNTYyMTJmZmZmMTdhZWQ1NjY4ZTA4NTA1YTAxNjc3ZGIyMzgxMWZiYzk1MWIxNjNkNWI2NWYyN2JlYjEyNjEzNmEwNWFkNWVkM2M3MzI4OGI3ZjMzYzVmYzBmMmJmOGU1M2VlMzc1MzU3M2QyYjQ4N2UyODRlNTY4YjI5ODQ5MjhkYTBiZDI5NDUxODBkNTA1MDAyODJkMTU1ZDUzNTg3MWZjMDQ3MDA4YzI5NzhmNGJhMTZhYzBmODg0YWQzNWE4MDE1YzM5MzQ4ZDcwOGI1MTgzY2IyZDMyYTQ4MWMyNTI1MWVjYTJlYTg5NTBmM2MwN2NlZjFkMWNiMDhkNzg2YmNiODdmMDJiNjIyNjU4MTdjNDAyNWI2OGE5YzBlM2UyNjM4ZTU3OTc0MDg2ZGYwZjdjNDE4ZWE0MWJjNjg3MWM0NWUwNmMwYjhhOTFmODU1NmI4YjgzYjU0YmExNDBhMzc3N2I4ZjkwYzQ0YzYxMzI3OWEyOGM0OGI4Y2I0YTY4ZmU0NjlhNmNjMjIzZDMxMTAzN2IwY2ViYjIwNTFhZjlhZmU3MzE0N2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NTp_nlOnhROJz5rHftFIa01HhD4MrgbNB5J175q3E3uZc_mq-aVUUgexP97UfCut1X4nBFctBWEpMJmjnx5gPg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211022_112631_93_2406_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.124Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBrdE9rcU1ZdnpEajlLRXRLZ1pSUE1HL2s1d2dUQVhoZEcxUDZFWDJZdkxvZzB5YmdBSjlkM2pvV09XN2RrbFBuUmYzSU9COFk0OU85ZFYyNVlGbVdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTIzMDhfODZfMjI3Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWYwZWI2Yjc5MjFhNDc4MTY4Zjk3ZmYzODA4NThlNWRjMjFiOWVlNDkxOTQyMDExMjUwYjVjZWZiZjI1YWFlZTExYmQ0MzE4N2MxMThiZmJhZDg5OWY1MjQ0YzJiNTYwZmY0ZDVmOWFlYjQ4ODMwNTM0NzlmZjdlNDI5OTM0MjZkODlhNDdjODJlZTg3OGNiZDA3YjExNmQxNGY1ZTU4NDEwYzg3ZTBhZGVhNTgxZjU2N2VjY2Y1OGQ2NjNiNTM4NGIzOGY3YmFkY2RmYTFkMTJlZmJiM2ZiZjNmMDBlMGVhMTRkNjExMzAyMTgyZWM4NzE5MTc2M2U5YzBiZmI3ZDMyMTU4YzE4Y2IyOGMzMDljZWFlZjcwOGE3MTBjNDQ5NzUxNTZkN2E2YWU4NGU1N2NmNDM0NzA4OGM2N2RiOGM3N2MyZWI0YjAyNzc1NWUwYzVjNDJjM2Y1NzM4Nzc3MzAzMzVhNDdhYTg5MDdhNDYwMGMyZDllYzAyZjI5MjI3YTM2ZmRiOGI4ZTY2YmNmZGM4YjQ5Y2YxZTkyMjUwNjNjNjM2MjcyMTdmN2MxNzQ4MDU0NDVlN2VmZDM0YTgwMTA4YWY1YjJlNDNmZWI4MjdkYjZlZmEzZmY4MDQxYzAwMmE3YTg1NmY2YmU3ZjQ0Y2U5ZmYyN2I3ZjJlMDg4ZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zr_U0_EXR1-6EV6AD9dSMfTVv_gqcSIUikqwbrkywEt07MSncpuCLX9XyUbhElQwVPOp3gaDQoFVYFaN_9TFvA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_112308_86_227c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.127Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZpdVlJN25qZDZYZkVCVit4dTQ1TUNBY1czdjNYKy9PcXF5SnVyZGdjU1QrenQvWWpueTdqRUV4TlR5Ykd4WmVOcklLZW5NbkFSa3oxN3hibVpWblFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTIzMDhfODZfMjI3Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGU3ZGZiYWFjNGVmNGMxM2I3M2Q5Y2IwOGNkNGVmMTY0ZTFkMzVmZTlmMzI5MjcwMTZlNTQ5NGRiZDMwMjNkN2E4ZDg4MDdjYThlNDAyMDBmMTY5NzZhNjJhZTE0NmNmNWJlYjkyZDE3ZWY1MGVkMjRiN2VjMWNmYzFhZDEzMDQxZmFkZWU2ZDc0NzI1YTYyODAwMDZiMDdmOWE1YzcxMzk1ODQyNGYwYjEzYjZjZjc4ZjgxMWU2Y2IxMWUxYzFlZDI2NTIzMDFjMTMzMWEzODgyNGZmZWI2NjdkMmJlN2IxM2ExNmMxMzFhNmQyMmY4ODZjYzQxMmM0NWE1ZGNlOWZjMGQxMGFhZjU1N2UzNjRiZjE5MmI0NTVjNjhhZDMwYTM5YmI4NjljMzVlMzk5NmI3ODEzMDI4YjA5ODdmMTliYmU0NjI4YzMzZDgxZjljMTI5NDc2M2JiNWIyZDA0M2MzNzE2YjEzOTY2ODc4MjhlZTNjZmZkZWU0Nzc1MmE4YTc2MTkwNWRhYjk4MjA5Y2YyNWUzYjUyN2FkOTNlOWIyN2EwMjkwMjBjMWEwOGQ3ODA5NGE4ZWY5ZTFmZjhmNDM1MTIyMDdiYzA4Y2VlNjQ1ZjczMTlmZDU1YzdiMTI3MzMwMTM0MTVkMjM3ZWU5M2RiMDg5YTYzZDY2NzRkYmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dAKaUf003BW59RZM13MSDtDQPKbwEPh73i6Xne_-DT022DBKVCx7_nKQ_3pvpg2WyGOsDhSdPezbFlsKEmiMSw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_112308_86_227c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.133Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjNPTm03aTR3eEFJVGRyS3FqaHFYcmRKTnorOE1SdkpKUzRpdUVJWWVWK1hnTFBEM3FtamI2MjIyV28remhWRW5zTEU3MEtXSFo2K1c4emdya1FkM2dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTIzMDhfODZfMjI3Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjAwODRhZjMxNmFjNzM2NzkxNDhiMTRiMDI3MGIyMjRlMDU0MjQyYzc0NzVlZDZkMTA3N2U0MWNjZTk1ODFjOTVjZmEyNDRkMmU5MTAxYmZkYTA3ZGMzMGQ5ZTE0NjdjM2Q0MjhiNmEwNjNhZDQwNWFjYmVlZmU1N2JkODgwZDg1M2FlNzAzOTY5ODIwM2I1MmQ3NmIwNzI0OWIzNjQ3NjJjNWU4NjMxYjVmYWEyYzI2MTgzNDhmNmVmMGMxNzI5OTY2MjQwMTgwN2Y0YWFiNzEzNjQwZWRhZTNjMDBkM2E1OGI2ZTYxYzdmZDgxMzUzN2QzYzAyOTRkYzQ4M2RhNTQ3ZWU1ZjdiOWU5MTI5YjkzMWE1MjY1NjcyNDIzZjBjMzUxYTQ1YmIyODY1YTcxNzBlNjE0YWJhZWQ4M2Y5NGI1ZjJkNGM5NTE5ZTcwYjk4MTQ2NDE5ODkzYjhiNTBiYmFjNjg3ZmFkYTA1NDAzNmIyZDhmZjk2YzgwN2MzODdmN2FjNTMxNmVmZWNhODU4MDVkZDNjZmQ1NjJmZWMwNzQxMDE1Y2Y0M2M3MmY1OWM5YzA2MGEwMTM4OTFmNDA0YWFlNDNhMmZmYjRhYmUwOTNjZDY2MWM4NWYzYThiOTJiM2M5NTI3NzNkYjU2OTJjMWRlYzVkZjIxNmRiNDZjOTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.N0SbGvVNoC_PSXXzGoqjA1V4N7S6dHYdPIt29qyLvyn9WXWKDdsGTjQJbuqh1aKUzaVdSrD6uXeBYqjuZ751zw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_112308_86_227c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.136Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlRLZDB4TERZODZDMVBJcXB2VjVLRmpYNnhLa1YrQW9xTHdGam9YNXNqdVJtclljUmlkbnZQSWU4SHNOT3pWVXdLYlZGR3FrYmd2a09wSkF4U1huakFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMTIzMDhfODZfMjI3Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzg4ZGMxMGExNjZhODk3NDVjODYzODk5ZDU4MTEwNDkxZDEzYzQxYWVhMGY4NDU5MzNmZDY5ODYwZTY1ZmI2Y2RmZDY4ZTQ0NWJhOGNmOGUzZjAxZjQ3NjFjZTRhOWIyODU3NTQzMGI0ZGNlMTU1YjE5Yzk2NmRkZGM1NDhhZTViYWYxYjY3MWU4MmJjNzk4ZGE5ODBhMmM0NzgyZGMwM2M2NDc2MmE5MTk1NzcxNjYyZmQwMzRiMjZhMDg1YjU1ZjhhZTIyZmY1NjZmOTU2M2U0N2YyMzRmOTc0MjcxMjVmMDA0NWQ4NWQ1MWI1YzdkM2FiYTExNTZmY2FiNzhjZWMzN2NhN2VhNmJiMzRiOTdiZTRiMzEwMTYxZWJhMzhmYTM2NTIxZjc1ZGM0MDk3M2VlOWYzYWUzYjk3MGI3ODFlZGJhYzc5ODBiZGY1N2E0Y2VhMzIxY2ZiNWRlYWFlM2MyNGE4NzdlN2Q1NzRhMjJiOGY1ZTUwOTU2NWEzMGQxNDFkZTFkMGMxN2QxOGQ5Y2VjZTAyYTg4M2YzNTY0ZGFhNGZjNTBkZGM4NzU0NTQzM2MzZTEyMmY0NDdjN2VhYmE3OTk0ODQ1ZTU0NDIwYTBkMWQ2OGNkMGUxYmYyYzNkZjRjMjQwNGNkYTFiYzQxYzUyZjhlYzZmNDE4MDliN2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7vJgwAtzTMf-a2QYLyA8pLOz0SCjxwQbNHX0vs-ScMFYQ8QXxJGTyXLlohgUPjmDfvEpoCOTnvaBNx4IHEXGsw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_112308_86_227c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.138Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBiQ0ZJVTJtUDIwWitKZC9wUU0xMXlOSXRzOHFLYkoxemVNS1NwbjNEb1UyUTlGejdrcGpuTVBzUVhaSVNRdmNRRnNlWm02cklIRzlJajJLdUQyWWl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyNV8xMDQ2NTVfMzRfMjI2NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzIwOTg4Yjk0M2MyZDdjNTYwNzUyYjI0YjlmYmRhMzA1MzZjMzNlMmEwMzU2NTk5Mjc5NGRhNGZjZmZmMGQ2M2U1ZWY5MjRmYzljMjRiNzQ0N2U3NWUwYzMxYzE4ODg1YmNkMWI1YzE0MjU3NzU2MDMwNTYxYTE1MzhlMmVhZDFkOTVlMDVlZjEwNWY4MTllZjYxMmViZDQ3NjNjMDBjMGM2ZTI1M2QzYjQ5YWYyOWJmZjllOTM2ZjZhY2E1NTMwMmNkMWZlMTA0Njg3MzQ0YmI5ZWMwMDQzNDM5MzczODkyYWZjMTNmYzRiMmIxZjI4MmVlYjVkMmJlN2Y4YzdkNmI4MDJiNDY0YTQxMGExMGQ2YjUxOWE0OWFkNDcwMjUwOTc2ZDc3ZmIzZTZiMDhmYmRhNTlmMzI3OTQ0NTUyYzlmOTljZDA1YzYyOGZjYjZmZDQ4ZjhhNDIxZGMyZGI1OTI1ZDJmY2I3MDM0ZGUyNDhkZjUxZDU3MGE5Nzk4ZjA2NmMyZTE1ZWM0NTU1M2UzMGU5MzNmMzljZDkwYjkyZGVhNjM2ZWM3M2EzOTk5ZGQzYzhhZmU4OGQ3ZGJjMmFiMmE1YTQ0NDkxOTkzY2EwNjUxZTRkNmU2MTRlODY5NWJiMjg4ZWMyMmQ0YTdjYzYwZDZkNGEzNWFhMDIxMzIyNTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gMpDorBJTdYQjcY8rVE3XDr-6sDgcAvLDNspJSS2jebN5aVEQng2Jrb3OLoXPELqso_bpWSiNlobOiSW5q6MDg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210225_104655_34_2264_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.141Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlNa29lQ21ESmczMU53M2dsV3JLdWE4c05xVnAxS294dFpmZXpuZ2VFenpSRDlSbmlVRVFoRjZyYTVGREhQbXJ3L0RIcFl0MCtrVGVVQk91WmNLcmxRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyNV8xMDQ2NTVfMzRfMjI2NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTM1MmIzNjVmODMwZjYwNmUxOWExMGNiOTQwNWZhMDJjYzg2OWU5MDJkYTcxYmFmYjMyNTE1NzI2NzU0ZWU3MjJiZWVlOTgxODZlZGU1MjNkODVkZjJmOTY2MzNiODQzMmU5Yzg4YTU3NTkxY2FiMzAzNzExMjljYThiMDZmZjU1MGU2MWYxZjc0YmE0OTBiZmU1NzgxZWE3NTdlMDgzYjZlOTEyZjI3NWM3OWQ2NjQ4NjY5YTQwZTQxMTU3MWEyZmFhMWUxOTZjMmU1NTUwOGEzN2M3NjRlZjEwN2I0MmJmNWMwMTdkY2ViYmU1NGFjMjYxNGRhMGRjNzMxYjFhNjY4OGZhMjdlODcyNzk1MzE4YzQzOTFhMjhiNmVjNWM4NTJjMWM4ZGZmMTlhYzYwYTQ4MTQyNmEzODI1OWU3ZThiYjQzMGU1N2RlMjAxMzZmNWFmZjVkMzBiOTg1YzhiZGEyOWZiNjM1MWIyY2ZiYjI5YjkwN2FkZmRlYjRiNjE4ZmYxYWI1NzFhMGJmZDkxY2U4NmM0NGEyYTdjNGI0NTY3MDEwZmFiZTgwMWJlMTcyNDZiYzA3MzU3MmM5NWQ4MzFkZjdlZGU5ZjhjYWFiMjAzZWFkY2I3YjljY2NiMmE2MmRmZjJkYjc1OWJhY2UzMjg1YTdiNDRmOWQ3M2IwZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ErrzSbcZfxZ4ChYi0X6QOtXiMN2OCuQddRSXUQLEdApk_BlUSRIxl5bX0W1-OXEUY3KfvcGxJzRUMJLz3gCtHQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210225_104655_34_2264_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.144Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InRJbE42amp0U3NVU1BsR3FPY1lweGh1eWRiL1U0Z0tlZDJwR0xxQ1dyaCtGTldEMk93VGtNN3NFT2ovcTlTbSsvMncydERHbGRoclRqZy91bHp0WDVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyNV8xMDQ2NTVfMzRfMjI2NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGRiNDhmOGZlMjAzODNkZTA2YjNhNmZiOTkyZmUxMjUxODUxODE1NGM4OTRhMzM1MWFkNDExMjUzYWZjNjkyZjczZTRmZjM5YjA5NmIyOWY1NmI0YWZmYzJmNTgwNzM5MWJkNGVkNmFhYmMyMzMzMDUyYWUwMTM1MjI1MGM4NDMxMDEzY2E5Y2MxNDAzZTY4YWI3ZmEwZWJiOWM1MzMwN2ZkOGQ1YjgyMzJmODA1OWRhYzcyMDlkZTgwZDJhODVkYjUyMzE4MTNkNGE5NGQxM2I0ODRlNWNjMGRlNTU4OTM5ZGY5ZjdmY2UxMzBjYmU5MWZkOTdmNzQyMTE4YzZjNGIwZTEzNDNhNDJiZjU1ODBlZThmNGE3ZmQ5OTdlYmVkZjYxNjc2NWM3YTc4NDUxYmQ1YTY4OTI0NWY2NDhmN2IyOTJhMDNjNzQ5MmMwOWFmMzg1ZDNhMDU5NTNlYTM4NGM1ODhlMjAyYTBmNGIwYzljY2MwNTdjMDhhOTQ5MGY0ZDdlMTllNTVkZmJkYzk5YTA5N2Q4ZmMzZGY4MzlmODRkZDA4ZGUwMzAyODhmNzE2N2Y3ZjYzNjVhMDNhNmFmOWRhYjFhMjFjZGNjMmY5NzMxYjNjOTUzNGMwZmUwYTRlNTE0ZTY5MWNlYWZjOGM1Njk3OWE0NWEzNGE1MTIzYjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vnmC-CfntB2M1CRci7PUMstToYhD8ADTc8NtAWYync3Bav0_UUC4-kjU8QwfDUgvLcH7W8LNDBcpAtb3_udhOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210225_104655_34_2264_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.147Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxCeHo1RzlHU3g2Y3BqYUVhY2xLQTM1M0o2TnlCalQ1ZHZ3cHdidFlKZjQxTUV0OW9nS0dvYTQ0Vkw4My9ub1FaenY2QWtYSGJMUzA5T05UZCtYRWx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDIyNV8xMDQ2NTVfMzRfMjI2NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjk5ZjEwNWY3YjM3ZGE0Yzk4OWE3NWIyOGJjYmUxNTY1OWE4ZGRkYTJlNDVjOTYzYjVhOTdhZmI1ZjdiNTMyNmVhOTQxYWI1Mzk5MTkxODZjZWQ3YjE5ZGE4YWZkOTA4YjM5YzM3YjU0OTVmMGE5YzRiN2U1ZDBjOWEyOTEyYmI3NTM3YjNlYTFiYTliYTg4MjNkOTkwYThjYTZmNDEyMmM2N2VmYjZiNjI5ZTBjNzg2NGUyMGI5ZThlMjBlYTdmNmQ1YTdkM2E1MzE0OTE0NThjNGMzYzEyN2E1OGU5MzU3MTdkYWMyZjMyOTBmMzk2OTQ4NTNmNjhlM2E2MWQxNjRhNTUxMTMyOTM2ZGZjYzEwNmY3ZTZjZWM4YjQwODIwYTg1ZWVkZDQ0ZDE0ODQxMWUzM2U4NjMwZjlmODcxNjdkNzVlMzYzMmQyMTA1NDhhZDJmNTNjNTZiZGM4YTY3NGM2MDlkMzZkMDY3YTU4YmJmM2JlYzVhMDBiNWVmZjZiNmI1NDkzZThhOTVjMDU4ZDMxMDZmY2Y2YWQxMGFkODM1MzY1ODNlMGZhZTM5ODU2ZjA1M2MxNDA1OTQ3MWY4ZjA0N2M0MjUxMDAwYjE4NGQ0NjczMzVhYjg1M2M0Mjc4MzNkMGI3YTg4ODdhNjE1NDI4NjlmNzY1Y2QyNzhhMzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5dkE319sqMmYgmRQOsfOF6oNnZPMds56hpxLOhPphl77aZgux9jOVF4i6HXDgAShSDE_-Sv_Z4Rb8L0i9qtp8Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210225_104655_34_2264_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.150Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IllHeFFYODRHVmxhc2xuNHN6K05BVlZLMFoyR3o5QW9EencxcUVKNXF6UWdHdVlrTHgrVnd1WGhCVlJHK0N2RlBQY1NXMU82VzljTWpGaFcxZ3N6bGVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMTExMTZfODdfMjQ3NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODYyYjM2ODdmMjQxM2I0YmZiOTBlMzllOTcwMDUwNWI4NWExZmIzMzA4ZDQ3NWI3ZDVlYWIzZWNkYzUyODk1NDAxZGY0OGE1YzU3ZjIzYmU2YTc3OTE4NjI5ZmJhMGRiOWVmMGU4NGYyYWUzNjE4NmZkOWQ1OTIwODlkNTg3Njk4ZGZhMjYxZTliZTEzY2UzM2NlYzJjYjNiYmMwMzEwY2I3ZGQyNmJlYWI4N2EyZGQ0MjNlMzllY2NjYTdhMzVjZDY3NzY3MjNjNGJiZDMwNDhiZGVjOGYzYzIzY2YxZTRjMjkwZTY3YjdkNDljMGE0NjRhMTViMGZjNDdmYmE1ZTJmODg3NThhN2JkNWI1ZmRmYjZkNWRhMjQzOGY0MDk4YTFiYWJlZWY1MTM2ZjUxZGM5YzY4ZWY3MDNmOGFiYjA4ZTIyMWJkZmJmZGQ0NjFjYjNkNDVlNzAzMzQ1NDRlNzcwMmI1NmExOTczMDc3ZGZiZWM4MGUxNzM2ZTNkYzAzM2ExYmMwNmYwNjMxNDk5ZGYyYWZhYzU5Y2VhNDNlYTBkNmI4ZmJkM2MyMWE2MjNjNTI0OWU3OTQ3NWFjMWExNjI3ZGRkODVjM2U2ZjM1NzdkYTUwODkzNGYxZDZiNDM1NWJiNWI1MzRhZjg5YzMxYzU0ZmZmY2JiMDU1YjVlNTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.fiAOwubJ6FooVPNppvE_mOTR8zc9pJOcd6yxUJOHmdcb_tJwJ-pFLIIrwOn9N57P3zHuUFIQxiXQ1ttrP2Xj6w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_111116_87_2475_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.152Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImtkdnBKdWU5UCtxRTg0RnFrVlRqWGhJaVdGOUp3d09sa2o4NmlCZmt3ZTlIdUNKY2xHd3d6VHZKSkpzNXdHeUVPbVdRNk1wL2kvTGRCTmpHdnE5UTN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMTExMTZfODdfMjQ3NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTVkZDdjNDVjZmUwZjgzODBlMTBjYTYxZWM5MzU0MjIzMmIxZDg4OTNhMjAyYjcyYzllODM1YzJiM2U1OTFjYTM3YzJmZWU1YmI0Y2NhZWVmNmVhYWM1MmY1YWM2NzU5MDliMDg0MTliMmNiMmE0OTg1MjgwYmIxOTczMmZjNjA2OGQ1MTFhZDA5ODA2Y2M4YzJlYjk3NjI5ZmVlNTM5YzY0ZTc5NjEyMGE0MWVkNWJmOGUzYWM1OWJjZjFjOGZlMTE5YzYyNDFkYzMxNDY3OGNlM2UzMDdlOGY5NTQwNDAzNDQ1NTE3MzVjNTc3Y2U2NWVkZTU2NjExYzcwMThiOTJlZGEzNmUxYmJhZmMwM2QxMTBmY2I4NWIxYmQ1OTg0Y2U3ZGY3YTM0ZjQ1N2I1Y2ZhNzAwMWNiYmMxNzc1MTZlNTA3YTMzYzFhNWY4NGY4YjIyOGI3OWEzNjA5NmJkZmI0ZjE1ZTcwZTVmNWZhMDRlNWYwYWUwOGFjYzlmY2Q3ZmRjNDFhMTZhYzFkOTk0YTkxYTllYzQ3OWNlNGQ3Y2VlYTBhYWNkNDFlMWRlYTcwMTUxZDliOGZhM2ZkMTczMDlmYmY1Yjg2ZWIwMzdlNjk3OGQyOTYyMWUyMTRjMWM2MmMxMTI0M2EyNjk5OTQzMWRmMzZjOTk2ZjRmMTAxYmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.g2mXQiYR9-1UInwv6YBV8GW1FWQXhoXvWAusD8ff5eou55_lR6tGlxkFYd_FIQyEGOhAST8y5pzgVR34QY6msA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_111116_87_2475_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.155Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNJR1hkV1JsN2trMmlTN0krUlRtVjJTYThFaTRtZWdwaVNnZnFPc0hlbkRNenoyVytvWUE1QlpGd3QwemVVY2ZTNTRUVFJUd1BWL2ZvUjF6aThXUmpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMTExMTZfODdfMjQ3NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzhmMjZmYzEzOTJhYzAxMTc0Y2NmOTY0Yzk4YjgxYTEzMDZiZGVhN2ZmMjQwYmViNTUwOWUwZjU1OWJhMzgyNGMxNDZmMmNjNmFjNTVjMzZlOGFkYThhYWYwYjcwYWZkMWZhOTI1NmYxMTdmOGFiM2Y0ZTVlODdlODczMGFlMmI0MGFiMDRjMTRhZWJkMDI0ZDcyZmFkOGE5NGRjNDQ2Njg3MzFkOTIyMzQ1ZmZlYjFiMzQyYzI5ZDQ1M2Y4M2NmN2QyYWQ0YmM5NTI1ZWFmNTI4ZTBkN2Y1YjE1NTQzY2UxMDRiYmY1NjJlZWJmZDc3NTk5MWI1OWQzZWQzNGI1ZmM1Y2U0NjdmYjkxYmJmNDQxNTI3NGVjYjNjNWQyNjZlMGQ1OTU4YzljYjU1OTQxMDEyYjQ5NTMzMjQ2NTQwMTAyYWYwNjA1ODBlOTBkNjRjZGI1M2I2YjhhZTliNzRlYTc0ZWE4M2Q4OWU3NTk1YjZlYTY1ZDRiNjU1NTk5NzFmY2Q0MDVlMTY1OGIwYjI5ZTkxYTQ2M2FkZDZlMzJmZWJjMTEzMTBjNjcyODRkNzI5N2FkNzgwZjMzNTcyMDM3NzY2Y2QxYTdmZjkyMjM5OGJjZWUzMzViYjA4MjEzYjI5YmViZTRkMTY2ZDhlNWVmZDJjOTVhZDUyMmUzNjVmY2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.o6g20MxjC5TuhQaF1KMxeS8fPk4lNIBxgkV6keyVRlEerQuPq07vwE3hGgfxWVyoqn29VWTXfSW4TCj3jluO4Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_111116_87_2475_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.158Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlpJN3B2STcvOFpzajVxOUw4WXFqaGtjRmxDOGZTRGxmdk9nTkRqd2dpNitPdytxRWtBTUlUTnlXWnlWenJleGs0QlFYSi9XRU9hckhqcWk3UXVQZEhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkxMl8xMTExMTZfODdfMjQ3NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzUzMjY0ZTkzMTM4MzM0ZWIyZGE4ZDdmNGQxMjBiYjMxMGRlNDE3OTIxN2MyMDA3ZjUzNWViNTUwNjU2NzVlZDBlZjJkMDUzZGQ0NDFlNjJmY2NmZjc3NjFjODVmOTFhNmM1ZGU5NWI1NWNmMzlmMTcyYjc0MzRlOGIxNTQzMGVmYzMyYTFjZDMwNjcwNDBiNjhmNDk1Yjg3ZjNmMmM1ZWU2MGU4Mjk3NjdhZDZlODBiZTg3NTVlMTcxYWVjNjViNTdhZDRmN2IwODJkOTU4NWQ4YmIyZWE1MWZmNTcxZTUwNTg2NGZhNGUzOWM4OTIyNTNlZDA2OTUyYTk5YjMxMDUzZGY4OWJiMDM1NzQ5ZTFmYWQyN2NmMTZlZGNlOTYyMzUwNGI3OTEzNWE4MzM2YzYwNWI5YTYxMzVjNWUzNTc5MmFlNmYzY2UwNzhhYzVlZDczNTM1M2QyMTVkNGY3MTA3Y2E1MjJhYjY4NGViNzQyYzY3ZDYyYTFiMmE2YWIxY2NlY2FkYTFhMTg1MTU4OGM0NzYzYjZlYzc4NTZlZGRkZGNkODdkNzg2YmJkN2M3YzgwYzIwODRlMmRlYzljMTNlMmUxZWQzMGYxYTY0NGJmMTlkMzcyZWNjZWM2NzRiMzU3MDk0M2I3YmYxZWU5MThiYjVjMjhhYWRjZDVkZTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tek_38TChvnC4xJUolgqTpsrBumZKu1-DMO_7rg6eahj1V2RlMo2a_sZdmnJbOwbIZ61obRA1dobakr7POk9KQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230912_111116_87_2475_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.160Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImUvYU9IMkpWcWtlOHY1YmJ5UFE3K0tTZHV2c0NhWjBpWDA5a0xJbnorc1c4aW80ZmdKS3lhQzNybEdiTjYrV21ta0dZYWFTait0emdBUGxkSGFyZUZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDJfMjFfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGI3YTBmZTk3ZmIxOWI5MTdkMzVjY2Q5YmM0YTc1ZWJjYTBjZGE3OWU1MjBlMDI3OTMxMTYwZTNkNGQ4MmRlNDc2OTAzMDU3Yjc1ZDI2ZjFmY2E2NDg0ZDU1NjNkN2IwZDMxYTgyMzViZTI5Y2YxMTMxOTBkNmEzZDViNzUxMTcwN2MxY2RmZjc3Nzg1MTkxNzhlYmU1YzdhMzM5NmM2ZTFiZDE1OTg1ZTk2ZjExNTk1MWU4NGEzZmJmMTkwYjczNTA0NzQzMDdmNTZlYTFkZjY1ZTA5MGVjMmYxOTZlN2ZlNDgwNjQzMjMwZTgyOWQyMGQwNmI1ZGZhZGJhNDFiYThjODIxYmU5Yzk3Y2Q0NzAwZjg0ZTJmOWY4YTJhNmU4NTlkODgyNTkwMzhjMTRmNzc1NTA1OTBiZjRkMTQ3MmY5YmU0OTExZWRlYTgyYmEzN2M4ZTUxYzU0NTUyNThkZWEyM2NkNzZmZmI2YjEyNmQyNTMyYjc0OWQxOTUyODgwNzE4ZGZlZGRhODZiMGY0NjUzZjg4NWFkMzA2N2UyNDU4M2JkODJiOGM4MTlkYTE3MDM4YTJkY2FkMjM3NzhiM2IyYTAzYWUwNjJjMWY4Yzg0ODBjN2IzMjA4NTk5ODYwMDYzMzYzY2Q1MmMzYTgxMTc3ZThlZjM0ZTQwMzA0YmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4dE8sCKjlsO9PWNl-0tzN_G-fM7oqVLyDXuAFRwyZNC4iaNXq3p-v3CPVAUyOUuhsXHSzi-bDiWokHE51beDGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112002_21_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.163Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkxTcksxWEYwMU9JZFYzSUU0aC9nNmw2SGZmUjNtMHNxb2o0TC9nSkU1MTlCZnQzN283WVdtbjVsZlFGRGRaSWVOaDJJUDZ6eDNOR2RWWTBQTEp0V1hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDJfMjFfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWZhZDdhODZiY2NhY2Y1Y2QxMmJjMDM5ZWVhMDY5YjY3ZWE3YTNhZDMwZDI5NmQ3MGFlNTRlNGFiMWY5NTc1NjdjMmUxNmNmNWMwZjJmYTBmMmEwY2VhYjhmMjNiYWM0NmM5NzJmOTI4OTAzZDg0YTBmYmUzMDIxZTFiMjE2ZDZjZGNmMTliN2EzMmZlODhhYjhhN2U4N2QwNDhjNTY0Zjc5ZDQ3NmRhN2M2ZjA5MjY3N2RjOWFjODBlMTU0ODg3OTdlNjlhYWE4MTRiMDYzZDllZjBkY2E0MjlkYTA3NTI5YWVlOTJjZDg2YTk1ZTY4ZWI3YWVhMDk5N2M0MDY5NjlmMGNiYmFlZDE1MWM0YTAxZWFlODYyNDAxYzhjYzk4NDE3YzBhMjFjYjYyZWQ5ZTAzNmNiZTVkZWMyZTkxNDQ2MTFkNDg2OTZhOGYzYzI2YTYzZjVhYWE4MWMwODg2MzRlMjJkMzZhNDIxZjY0MGE1Zjg5ZWUzYTkwNTFjZjQxZmRhNjgyYTY0NGQ3NjU5ZDAyZWRmOGI0OWNhYWZkMWNjOGRkOTIxNWZkNDI5YjA1YzZiNzNlM2ZmMTRmNGQ2ZjZiOWEzYmI2ODc3NGE2NmJlZDJlYzIzYjljNjZhNjlhNjEwMTY5MjZmMzI5ZDRkZGI0MjcwOThkZWNkNGM4NTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HjIyQodf_0Fp6TUr9DgIHXcK5z_Zq9DO2nJ8xXo9rbpn5APf6E4SREL6axzMkeXitbPmiq1cIpBgD8Oi3E1xSQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112002_21_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.165Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlVaVnNnY0t2QWh4aHBQZ045Y0kwNkg1NzhZSmNlaGVTL0htMDJPWGg0Y3U2cE5uT0F5clVnT25MaFFzb05KbEl4Um51b0pmRkNIVmNGSU1ZMXdTSW1RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDJfMjFfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWI5ZGMzNWM1YjcwNTdjNGM2Mzk0ZmEwZGZjZmMxMzQxYmYzOWM3NDY4MDllMDNmNzY5ZmQwMjBiZDM1ZDIyMjkzYTc5MWIxYjc4Njk2ZmE5NDEzNjM5ZjZiNzEyZTJiYTI2NjBmODhmMWJiYTY2NzNlMzg0ZmJlZmE2ODA3YmE0YzFiY2JmMjlkYjliY2Y2Y2U2ZjFkNmZjYTc2ODBmN2Y5MGRiMTMyZjBjNjg5MTYxYzYxZjlhZWI4YWRjN2Q1YzZhODJkY2YyMzk2OGFjMzFiNTNkNTczNWQ2NDNhYWQyYzJiMmI0YjA4MWJhZjQ4YjM4NGMzODZkZTBkN2Y4Y2RkMDUzMTBiNDI5YzdkYzNjNTg3YWM2MWVmZWNhNTgxNjkzZGI5NWIzODM4OTE1MjBlMDRlOGIxZTZiNjU2NzU5YjU5NTQwYTEwNGM3M2MxMTRmMjk5MjUyZmY0OWIxYmFlNGYyNzA4MjgzZDA2NjRkODcxMmU4MmM1N2ViZDZhNGJjNTk4NzNmZjk4ZTkyMWFiYTE1OTZmZDgwYTJkMzkyYTEwN2YyNWQ2ZTgxNWUwNjg2ODI4MGUzMDg0MmRiNjhkNzFjYzg1NGVhZTY5OTlhNTM4YjkyZjVjYjVlMzY3ZjY0ODM1NTg1ZWE5MTliYmRjZjMyN2E3MTUwODIyZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.461c4GY21ckwhr2kpzrW4dWMB1BHP3AcT0Z70rm6jDvzqPBvAU6TkNw_1wdZbQzBH-lobZ_q7GzctvMitG8yGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112002_21_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.169Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IklMMVhiYitQL3FrMzVSRC9HcSs1M1FxZ2VwZXdQWVliWERsSEhBeklIa2ZnQjIrSkN2b241OWtUYXlNMFNqZjRzd2JHTm42SWU3MzRUdDJOWDYyeG1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDIxMV8xMTIwMDJfMjFfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGIzMWE2Y2JmMjcxNjliYjViYTNjZDg4NWJkOGE3NDI1MWM2NWQ2MDAyY2YzZTliNzBlYWZlZTMxZWVlYTU3Zjk5YzNlNDRlOWU0ZjM0ZTQxNmQ4NDY4ODVhY2Q3ZTQ1MDVhMjhkMThiNjMzZGMxYzUwNzBmODBkOTU4ZjgzOGVhZWQ2OTZmNTYyM2M4NmQ3NWI4MzE5ODI5MGJiNGE0YjE5MWRjMWQ5MGI0NjIyODg1ODlkMjcwZDZhMGFhYmFmMTdkY2Y2OTVlZWMzNjQ0MTAzNjg4Zjg3N2FiMDE2NDg2MjA2OTZkZWFmNzIxY2IyYTVmZjg5ZGY4Mjg4NjBhNzdjZTViNDY0MDg2ZTAwYzI4NmMwYTE0OWE4N2ZmYWM4ZTc3N2Y0ZDRhYjU0OWZjYmNiMDkwZDY1N2M2NTQwNjY1ZjgxMDNiMDZiNGExNGFlNzJiMjVhOGI3NjYzY2Y2NjgzOGJkMjg0NzBiMjRmMGEwNmU1YmQwMTZiN2RhOTQxNzdiNzc3NjU3MjFiZDY1NWFmODg3NzUxODY4ODgxMGZhYmM5ZjQ3ZWJkMWYxYjM1MDk1NTQzN2YxZTVhOTc4MDdjNzUwYzUxNmY0MjVmNzY1Njk2NTVhMDEwYTJmOWNhNjA1MDcxMGY1NjcyYWJhMGQzZmFjNDMxZjAyYzYwY2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.K6irXYydzDb1HXs0s4GKq8s6usCaovGKIb31Y_Crb9isIP_HupTaRGAJuofYPguk1f7lWiOsw208LCcBYy7TlA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220211_112002_21_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.172Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJyd080ZFkrQjdYM3h3Qjg0aFE5TXM1VXlGYVRUT0g2b2VnRk1OZmdaNkFyTXhYM3NGU29kU09GR25WZ2dBVi9PZ1BqS3k3c0ROZjd6U2hINTNkKytRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTAyMzFfODlfMjQ5ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTM5OWMzYjQxM2Q0ZjRlYzMyNzczNzlmMGM5NWE2OWQ1ZTI2YjVlYzc2NDE2YWQ4MDVjYjMzMDAzMmQ3MGJkM2NmNmRlMTJkMTgxN2E4MDQ3MWZjZGJmYWU5ZWFmYTc4ZWU4MjRjYmZmNzk0OGE2NjEwZWU0ZDM3M2MwZWJmYTYwMzY2Y2Q2ZGQyMmJkZjEwZjE3ZjdlMDllYzE4NDFkYWI0NTYzNThkNTljMTdlMDg3YzkyODNmMjg3ODJiMTdmMThiODFjODg5NDRhNGRkZTkzMDlmMzI1MmZkN2IxOWI0OWYyNjBiMDQ1ODMwNjFlMGI4NGQ5YzY2MDNiMWEwNjg1ZWVkMTYxYTE4ZjBhNDdhOTdlNzI3YTVhNTI1NzMwMTA0YjRmZGVhZjM1OTk1OTE0MWVkMDViMzE5OTEwZjkyYzFhMTY3MDFhNjNlNTA1YzM4NjAwOTlmYzE2ZDM0MzY2YjNiNzczNWMwZDA1MjlhMjU5N2M1Y2IzYzE5MDBmM2MxNTU5ZDBkNjBhOWY0ODExZTI3Nzk1YjI5YmYwZmIxMjkxOTY4NWQ2M2ZkM2RmMmNjNDA1M2IxY2UzNDg3MTVhNjBkMDgwMWVmMjhmZjg4ZDU3YjRkMDI1ZjEyZmU0ZmMzYmMzOTMyNDJiOGIzZmY2MWU4ODJiNGVjMWU0MjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3TgnQSRRg9N5KyLFNTCJU_OSHuEmyZ---Aiq8Kc1jptFGMWEq15Qsyf5DPJE7VPlWb53Ix2r2W1XS01Nc4agQA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110231_89_249e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.175Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImVzcDFWYzYvQ1lnR3VlZ1dhU0ZTdVdnZ2hZNGVqemZnbHRVVTkrMmxRVnYvSXV4MWhWN2VUZHdWQ3IyY2IyRWYwRFc4MXk1YVpOTjhOVFBpWXUrNDN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTAyMzFfODlfMjQ5ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTEzNzFmNWEzNzM3OGY1ZGUxMDA1NzQyNTgyMDM1OWQ0NmZiYmNlYzk2OGM4MDI0ZWJmNWJmNjlkNjkyZGQwY2NmMzUyODMwZDI4ZGI5ODk1YTNiYmRjY2FjZGRhNzZiOGFiOGQxM2MyOTZjMTYxMDVjMTNiYzY0NDM2OWI1MGNjNmFiZDgwYzMxYWY4NjRkOWU3OTU5ZjhjODJkOWY3NTkwNDk4YWZlMTNkNmMyOWFkZDdiNTE2MWJhNjI0YzdkNDhhZTIyM2FkNjMyYmIwMjk4M2JlNTVmMmEyYzM4NmMwNWI3MTI0OGY1NzYwMzE4YjU0M2RmMjE4NWQ3NjFmNWIwYzAwMDY1Yjg4NjE2NjFjMzdjZDc5MTM4NjU3ZTAzZDc0ZDYwNWU2YzRkN2ZhMWJiZTA3NGNmNDRkZDYxZDNjMzVkNGU2OWFjMTE4NDU2NmY4ZWFkOWY0Njk2NGMxZmY1MDVkYWVkY2M3YWVjYzY2OTg0NDczYjNkZDM5N2JiZmE3MWQ3YzVjN2ZlNmY5ZDNhODM1NWUwNmNjNzU2NTE4MDY4NWIzYmFiMjk1YmIyMmZjYWI4OWQwMDQ4N2VhZGRhOGFkZTMxNzM2MmM4NTI0ZGQ4NDk4YWFmN2ZiZjg2Y2RlODM5ODk0M2UwOWI5M2RkNjA0ZGEyZTZlOWYzZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mDpWGw_aqZSHvzBwHBXmsAWljPeJPEALlMMgfSnQfA4jLUcqr0jHGjNaY1m8auhQWFNHT8Kp-vTpp3wY4W-6sA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110231_89_249e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.177Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlVVbVlibmg2ZlBOYnlrRlYvZURHNUR1ZlN2UEVPWkVFT0JXb0tjcmgrT1dPQkJTc0huaGVId0lqY3dGSGVZZEw2UDFVMUhrLzBmcFdDdWFKRGhjLzlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTAyMzFfODlfMjQ5ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDBhNzZkNmZkYWUwNWFiYzA0NzQzYjQzMjUzMmI0N2VhYzJkZGVmMDZmNGNkZGY5ZTA3OTEzYzZjMjJhMWRkMjNlMzk2YWNmZTY5YTQzNTc3ZmMwNzFlMzNiYTJkYWFmNTBkNzg1NThiMTQ4NGI4NWU4YTk2Mjc0OTcwYWE2ZTEyNzE1NTE1MmViYjBhMWNjZmViMWVhMzhjNmYwYTAwM2NhM2ViNzQzNzUwNjE5N2UyZjY1MDgzODBmOGU0YzMyZWMyMmY4NGJlMjRlZjE1MTZiOTc5NTlkODBiZDAzZDMyZTY4NzExOTgxMmI5MzI5YWY2YjIxYTc0YTViNjI0NDE4NDNhZjBhMjRhYzJlN2I2OTBlNGJmY2NjNTU4M2EyMmZmN2MxZDE0NzA1YzU4OGNiYTE4ZGJkM2NhYzQ0NjJhM2YzNGFmN2ExZTRiZTU3OTBlODk0MTRjYTJlMzcxZTkxNWM0ZmUwMmE4OWQ4Njk2Njg1ODEwYzZmODU2ZjI4Y2QzMzcwYzg5ODM0YzUxYTY2NGE3YTJlMjI2OWIwYzhjOGU0MTI5NTA4OWFhYjcwMjU1NTkyYWY0ZWZhYTUyMWZlZTRhMjVkMmQ2MzQyZTk1NDgxZDc3YjkwYjEyZDM0NjA4MmUxZTk0NjE1MzJmYzdlNTYzY2I5NjY4OWFjMzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sPDBx7FDBlD6HiZFhqDCLOcq2q10DLZXZVslEgUOzbA5UX7HWBsxuDf6k3OaMWgA_lsXttGo7UNwzwtlMsZ0aQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110231_89_249e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.180Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkpxM2w5U3plVzZYM1ZxSTV5ZVcrZzYrMDNyK2RoVlJmWHEvUEc5d3o0cm5hNjBucU1DWkJMZm44dkRsM0xRNHpSWmF5eWZhTTF1TTZCTldRTXQveHVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTAyMzFfODlfMjQ5ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTdkMzcxYzk5MGUxN2MyZWFmMjQyYzY2ZjM3YTEzYzgwNTI5ZDYyMDBkN2I0NWFlNDJiM2FhYWZiZjkzMjNjOWUwMGRhYmVhOWY1NWJhODc2NDBlNzUxMzNmNTQwMmIyMWQyNDY2MzkwOGFlN2VlZGViOGIxNTA0NTUyZTliMDAwNjg3NThlMzVkYTViNzU3NDUzYWE1MGYxNDI3MmMwMmJkN2MzOWRlZDMzYzhjMTNlMDlhMjY3Y2Y4ZWE5OTE1YzE1NmU2NWI3OTMwMGM4ZTc2NWZjMjA3ZjYzNTk1YzU5Y2E5YzIxZDU3MTg4ZTJkMGQ4MmU1YTdmMDk5NDNmNGJhNmUwM2RiODNiMTczMWU1ZDkxMDY4ODM4ODQ5ZGJkZDdkOTk3YzYzNzFjNGM1Njk2ZTA5NDM0NDZlOTAxNThkMGFkYWQ4YWJiMzI4MjI2ZGE1MmRkMTFhOGE2ZmVmOTJlODYxZTI1YWM5Yzg1ZGIyNzRlZTg0NDdmZTE3Y2MwODcxNmNlNzM3YTNmMGU3MjM1MzdkZDhjMzU1OWI3NmE2MDliMWNhNWRkNTI4NWU5YWU2NmFlN2Y0MGEzZDlkN2VlMjg4YmFiMjc3MjU3MTIzMzVhZjkyM2ExOTYxYTNjNTdjN2Y2MWUzMjQ3MTFmN2FkNWZiZDg0YTU4MjA4MThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3t9D97cENaciYmVaID7z7awa6tYuvB1hCvGt0Safi_XT-gZgVQkjOX6ylhLuGz3PtUlzw5xvL9ieZP1XScxYcw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110231_89_249e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.182Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNEV2hiR3VBSEY3aGREMnJmYzREUExaYndUTkQydGg0L0NvWTNoOWVoT2hOSkRic3pRV0tZU29xNjlwV0JqTVZWV0UwaVR3d1U0R0w1U0pmWmJITEV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI3NTlfNzBfMjI3Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI2NzA5ZjNmZTYwOWMwMDI4MjU3NTExMWM1M2FiOTZmM2M1MjRkZTQxM2E2NDM1OWFlMDYxZDhlOTZjYmY5OTdmYjJlYWE5OGY5NzVjZjgyNTlkMzg1ZGFhZjMxMjA0YmNjMzAyN2ZmZWQ0NzQxYTg5NTdlYjkxOGFmNmY5MDY1ZmRhMmVmMDgyODRmYmFkYThiMzkzZjYyMTg3YjExY2YxZTliMzVhYmY0NDRjMjM0NDFiOWU3NzdhYmQ2OTNjOTIwNjgwNTMyYzY0ODhhOTM4YzY2YmZiNmRkYTk5ZGQ2ZTViODdiZTM1ZWY5MzBhNzI5NmI3N2VmMWU3YWRmMzcxNTQzNmUxNDdlODYxYWVlYWJiZjdiNzc2ZTkzZmRkNjcwNzkyMTliZDBkMjA0MjAzNDIxNDlmZjA3OTZkNDU4NTE5NDc2ZTk1MDZjMzBlNmQ0YWViNzdjMTE1MmUzNDM5YTQ4YmIxNjExNmQzZDNjZTkzZWVmODhlNDVkOGE2OGI4N2E4MmZmOGNiYTMxMjZkOTIwY2Q5NjU2YTY2N2UyMjZiYTBiOGI2OTg0NjE5NzkxYTlmNTIzYThiZmVjODgwMDg2NGQwZjlhNGQyMjFiOGY4ZTZkMDA2ZWJmNjU4YThjZTRlYTgxZTZiYmNlOWNjOWNiOWE3YWJkYTg0YThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7hw1wZkLTX9jyX-ELs_lYU5w_qu2M2DaRbPdn9ty9nRM2HrPjNAnxgMakIkjy34RXK5DN9Odi4eciS-ETgZu8A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112759_70_227c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.185Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im5pRkZSOE42MUVrWVRYSGdhS1pONU90UnBDVkdNQ29vZUdRRnk1V2RuRkt3K1lhNUxMNW1FNjZISmhSUWkwRExNMTBsZG1ibTExMzRVR2FwSm94K09nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI3NTlfNzBfMjI3Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTliY2ZjOTlhMzU1NmYxNjllOTBhYWE0MzcxOGVjYjFjZTE2MGFjMjBiZGFlYWE2ZmNkYzkzZTc0MzdlN2VmNmQ4NGJkMTJhOWI5N2IyNzg1YTcwNmQ3OGM0OWU1MDhmYjdjZWM1ZDc2MDFmZmIzOGQ2NTgzNDBlNTZhYzRjNGRiZWY3YjM0YTE4YjVmYTM2YjZmMzFhMmMwOTgyOTNjYWJjZTU0ZjQwNTU4MGQ0ODliYTI1NTk2ZGIxZjVjMDY5ZjQyMTY2MGY3YWY3MTk0Y2FiYWFmZjExYzczZjk2NzhmMDY1OGJlMTJjZGYwMTU5MGZjNDk0OWMzM2JiOTg5M2UwOGJmNGQ4NmMzZTdiMmM2ZGQyYmE5OTNlYzMxODM5YTc5Y2RjNGYxNDE4MjBhMTU3MTNjOWY5OTlkNWFiZmQyNzQ4ODIxZDM0NmY0NTkyNDMxNDgxYmRkZTQyNmI0NTNmNzI1YWI3MjQ3NDI4YjE5Mjk4Y2I2NmU2YTlhZTZkYmRjZjViNTE0NWMzN2I5ZTExYjUyODdmMzkwZmM1ZjUzNmUwYjdiYzEzYzUzYzRiMjA3N2QxODhjZDBiNmU3MTViOGEyZjNjYzYyMzEzZDU4ZGIwOTVmZWVjZTlhNDIzODFjNjI1ZGI0NmExMTAwNDVkNGViZjM5OGNiYWFkMzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4X-12qKUt3UOREZSJT7E53veBZfjFBFJ6wQDk5ieFv3mnYCKrlGW5X8HtKJmAcdGsM7hHUKJ2j_HQWH_igmKow", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112759_70_227c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.188Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImgxYzYvOVJPL1JiTkRzQjQyTUlKbjM2cG94Myt4TzZqQ2tUL1drbDRoa0JxeDdJOW5ZUVQ5TUlWaEVKVk9hNDhYMTJOQVp1YUJNdkdDUU9sQStmampnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI3NTlfNzBfMjI3Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDdjODQ1ZTA0MGY1MDIxYTc3MjRhNTA2OWIwMDZmNDA3MTNlZTk2ZWFlYTMyNWFhZDlkNWM4YzFkYjcxMzNiOTU5NjkxOTQ2NzYyZDBiOGEyYTQ0OTc4NTc0YzVlNmY0ZGQ4NmMxOGZmYTkwOTlmMGY5NmM1YzUyMmU3MmJjOWU1OTQ1MDQ4MjMzMTdjYzE5ZTUyODA2NWFiODFhNGQxMzAzZGU5NzQwZGI3ZmVhZGFlZDNhNjExMDQyNmU4YmFlOGU2ZTQyZTlhYTg2MWMwN2RmZmQ3Y2I5MWEwMTE1YWI2MDBhMzQwMDdlZjU2MDE1OWIxYjIxYzhmYzNmMjZmYWFlZTcyMGJlMDZmYzYzZDQ5YmNiYWJmM2U1YWQ4ZGE2YmU3ZjhhMDE2MzQwYjRkMDE3YTBmN2VjY2ExOWFiMzQ4Y2M0ZjQwZmFiMWEyNjgyYWJiYjFlOTg3MzdkM2MwMmMwNDIwMjM0MmZmMzk2YTkxM2FiYmJhMTVhYWY3OWQ1YjNiZTIwMzExN2M3MTVkZDc1MzYwMTExMGY1ZTZkZjAwMGRmZWI5ZWIzZTU5MmNlNmNhZTNiOWJjNDFjMTIxMzU2YmJkZjUwYjQ3NWFiODEyYzYzYzcxYjZhMDI0ODkwMDcyMzJhZDc3MWFmOWNhM2RhZTJmNjIwZjI1YWY4YjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PZrL0kHXv6vcjOAykA2ax0YyCu-B1OtnxgIAhhDH6zfTsLG9n0KOY5pXSg7DGVtftleDZ-NRDScdi4n_xD9nHQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112759_70_227c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.191Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZqLzI4TWtwL0lORDkrTmdscTRlWVdiNCtHRU54RC9XNVV3UVRXalBqMStqdk83YWp5TnJ5Vy9DMGk1T2NncEMrbld5RGJreDI1N0F6a0VtK2FYck1nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI3NTlfNzBfMjI3Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGY2Y2Q1ODVmZTljNjFmMWY3YzhjYzE5YzY1OWQzMDMyYzFiNzY2ZDM5YWM2MDg3NGMzMTkzZjVmOGI1ZmRiZTgwNDY1YzYwMjRkZDYyYTI3YzVlYmViM2FlZjcwODBhMzZkOGQ2M2QwOGNmZjhjYmUwZjE5MTlkZTQzYTlmZjkyMzllZWU1NGU5NGE4MGY1YTc5OWIwNGVkYmI0NDA2NDlmYjEzNWRjNThhOGRiY2E4ZTBiYTkwYjExN2FiNWM5MmE1ODY5MDVkMWI0NjliOTZjYjc1YTNiNzA1ODE3MzMyODg2NTdkMjI2ODI5ODMxNjAyNjJlMmRkNDk0ODM3MTc4ZGIzYTkxODYwNjhkYmI4ZDBjMTVhMWZjZWRhODA3NmFhNzRlNjQ2OTk1ODdiNWJjZmUzODUwZDNhODk5MDIzNmUxMDcxOTc2ZmI0OWFhMDJkMmVjNWQ0MmJmOWQzNWVjYmZjZGZhMWM2YzE2ZDg2MGZlNTFjOWRiM2YxZDUwMGI3MTM1ZDJjNTlhYWJiZTIzMDY5Y2I0MTY4MzcxNDc2NTc4ODk1OTQ3NWEwYjFhYzBkOTliYmMyNzRiOTUzNGQwYTZmOGM0Y2MyZGM0Y2IwMzNmODMzNmVmZjAwNjkzNzRmOTA0ZjM5NjVhMzMyZDAxMTE2MjNmNDA0NTYwOWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wMioGvV0QPoOxHTc_HfiJiOneS7Jg5TVdAFE29otFd4cSCkonnBxM6LVvGNe-0xRtHhOS9TRyOjvqi8_mIcG_A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112759_70_227c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.195Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRxdzdWeS9IckQ3anlBY2RwN2lxMTk2MHVjMDRLbi9Mbml1N0ZXekE0b0YwMFR0aWxDQ3daSXpmVHJUZm01S3htNXdSaklPTkpmMjd6cjgxQnc0QThnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTJfMzNfMjQ5Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQ5OGRiODQ3MDE1MWY2ZmMxZWIyNzIzNjNiMTZiNmFmMjhjYTAxMDJjNDQyNjI5ZDUzNjgwNzExZTBiMWExZmZkODdkNGE5ZTllMWUwOWNkYmViYzM5OTk1NmEyYjk2Mzg4OGZkMmMxZWI4NWI0MjUxYTcxN2E4MzkyMDBmYjEzYmUxMDlmOTI3NjQyZjg1MWI0NTA5OWUyYTMyNGQzOWU1ZDE3NzM5NzIyMDU2ZjIyNjk4ZDg2NTRjZTM4ZGY5MGZlZmZmMGRiNzQ5MDM2MjU1MjgxMzVhODYyOTUyODljNjBkM2YzODZmNWMzZjRlYWU2YTVmY2IzZDk0M2EwYjViMWZiNjZhOWZkNzdmOTMxY2E0NWRjN2Q3ZTUwNGY1ZDU3NDkwNWM2N2NmNjNmYmMxZGYwMTNhMmIyMWY3Y2RkNWM5N2E3MDhmY2NmOGRjNGRhYTdkZTdhOWQ2MmFmNWNmYTUzNzcwNjc4Y2E5M2Q4MDJmNzM0NzcyOTgzNjliM2IwMDVhOTY0ZjI2N2E0ZTI0MDQyNWMxNjA0ODE3MDRmY2E4ZDZiMjlhMmY1NDc1NjA0ZmZiMWFmNjhjMGRmNjQ2NDI5MzFkZGU4YTU5ZjYzZjgwZTI1NDY4MjcwZDg1ZTEyZTM4Y2RlMjMzNDM3ODMwMmU2M2QyYzA1ZjE0MDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jExUSY7kCdHr8f_ZIA9cKTRTVCF6pqIum8SaMQy5SRjrUF8zCB0t216LWJVfmwWAVPHUTH09VODjiN-TcoY-1w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105912_33_249b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.198Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im96bWtRZktTam5CU0FNWFByUXppUWt1STRsYUJSSDR1NnBPWFlqWG9vbVFibHlqVGMxNmxVZUpDdHZldU5nZW55VHRBbzZ5THlnb0c1T1p4N1RUejBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTJfMzNfMjQ5Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGY4MGZkZTc1YTVlMWU5NDljOGU2ODkwMWQ2NWQ3NGI0OWJmM2U4OTg4NWQzMDE4MGI3NGU3Y2Y5NzIxMjlmMWNjNjcyODY2YzVhYTA4MWU4ZWUzMjFlMTAzNGQ4MzhkN2I3ZjI2MzJjMjNjYTIyNGI3N2RhMjU2NjMzZjM2NmRlYjczNTNiOGNiOTA2YWQwZDJmYmM5MTYwY2JiNmE1Njc0MDRlYjNiZGFmMWQwMWY5NDgwNTNjMTg4YjYxOWE1OWEzZDk2OGJmY2FkYWI2N2Q2M2IyMmJlMGVkNTI2ZDQ0M2ViZGE5MDA4ZmE1OWIyMjU4Yjk5MmFlN2NhMDEzMWM0NjM2MGUwNWU5ZTE1NGZlNmFlOTQ2ZmZkYmQ3NDgxOTRlZmYxOTYwMDQ4ZDAwMDUyNTIwYjQwY2IyYjQxMDFhYzE4MTEzM2NkMzIwMzU2NzE2Y2JiMDhkZmQ1ZmVlNzM4MTMyM2M5MmQxZTM4N2UwZGFlZGVkNWFkOWZjZGM0N2JmMjA1ZTdjMmFlMTkwNzM1YzljNmJkZTk3MGFlZTQ3ODk0NGY0OWU0Mjc2NDlmMTkxYWI5ZTQwZTMzNTg2N2YzMDdiYWRmYjZjMjQ3YmU3MWE0NDA3MTM0MzNkOTM2ZGFkN2EzMjEzZGUyNGJlNjE2MTJjZDI0OWZlYTViMjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FPGy4gCDw5pbrjGEKugSPs80Ezwp27OSVDOkoHosDrfczlduxILsVOL80giHTpMpsscdaNvvTwz1uBdceiUbSw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105912_33_249b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.201Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImhPMWFsMmhWdGdlMnBXS0hTSHdPOVdDS3dtdVN6N3FJZStKR3hIYm5SUG84aE5obTUxZHpZelJHZkhDcktzbmxpcGdYUEJZQTNPelFtbVNtUmk2RnJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTJfMzNfMjQ5Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTkzODc0MzgyNWIzMGI1OWFkNDZiZTNhZmE4ZjYyYTI1MWY3ZjIxYzg4ZGEwZTQxMzYzMzU2NTVmN2U3NTUxNDliNjg0OWJlZmExYzUyMGM3OWJiNDIxZGQzOTUwY2U5ZWUyNDcxNGIzYTY3NDk0NTkzNWJkNWUwYjgyMGYwOWZhZDA4MTMxZTY2OGRjMzAxOWJlMTA1YmRhMDllNDQ2ZDFjMGZiMWVkMWNkN2E2M2JhOWE4NjcwMDdmZTQ3M2RmNzJjNDdkNWNhNGYzZDA5ODUzZTk1YjYwYmFkNGU5MGYzYThlMTRiMzBiMjMzMmE5MGE3NmI2ZWE2OWQyZGVjZTkxYWIzNTg0MDdkNDViMDc3ODUyMThjZjMwMmE4OGU5N2JiMzJiYTkxY2ZjMmU2ZDI2MTg4NzE0ZGNkYWYyN2Y3MzM5NDJkNGUwZmY2ZTA1MDQ4YjMxZmVkYjViYWFjNzYzNzgwY2JlNWRiMDU0ZmY5MjFiMzI2ZDdmODA4N2ViODZhM2M4ODkwOGZjZDViNmQ3NDNlMTg2N2IzMzI1YjkwNDlkMjU5YWZmNTFmNTIxYTY3ZWY2YmI2N2M3ZWU1MWZkYWQzNDNmNTYxYTZiY2E1OGQ3MTRmZjQ5NzdjNDA3MWIwOTYyNTRhNzg1NTIwNWNmMDc1Njk4MjNlNjI4MzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OPy_8y-IDVcvg0yLHx1KEFDrBzoU_GuS0jv3Thxg6LjPtsqZe8w_QQEaIZckoN-a4iruj0optQhVX_L9MoxQEw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105912_33_249b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.204Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjIwK2puY29EcXhLSDJnYU5OQkJoRjFpTWdRdXJrVzV5NDAvMUw4eUdHdnQrdEJKckROTXdleUJvL3FnODVBYm54bEdFd1JQNExoenZkdldxRVdvMndBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMDU5MTJfMzNfMjQ5Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzMxMDM0OTJhZWYwZGZhOGVlNTI4OGQyY2M1NzYwM2E0MDQ2Y2NkMTU1YzY4OTg4OTUxZWZmZTA5NWM4ZDg1ZjkwMDVhZjg5M2Q0NzBjMWY0ODExMjQwZjliYzE2NmU1YTkxMTRhOTMzZTA5NzlmYjk0NDNjMWI1ZDQyY2I0NTJlMTk4YzEyY2U1MjRhYWNlODMzNjdlYmQyMDEwM2RjMDYxMTQ2NDkxNDllZmYyNmIyZDM2Yzg5MjQzMmIwOTNjZTE5ZjE5MjcyMWRkMGQ0YzNlMDg3Mzg1MGRkZWM0ODJmNzUwMmEzNjNhN2FjY2E5YTMwY2ZmYjIyMDgxODhiNTliZGExMDk2OGVjMjgwMDkxZjdjYmU0MzQ1NDE4OWUyYjI4Yjc1ZmZlODhlNmIwZGRlNDI0YmMwYjFkNmQ0OWY1NTIyM2UwM2E2ZjhkYWEwYjUwNGQ3YjllODQ0ZmQzYWY1MDY0MDc0ZDJiODVmMWZiYWVhYjI4MzdjYzI1MzQ1MjJkMzI3ZGI5ODc2MmY1NmQ4MDljMjE2NTFmODg5ODQzZGRhNTgxMGUwZmJiMzJjZjExYzMwNWQ5YmVkNTUwOTZmNmMzOWU3ODhkMjBlZGJkMDlkMDE0ZjljYjYyZjI3MDgyZWU2YTY5MTVlYzNhZjk2N2ZiMjcxNTY5NmE0ZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.65XsWuipb-yHRaFAc4kqwhmWTK3DZ2xRDPCCgSrLVD895dxEbJNRgLRtrka5oJNfiss1ulo_-_PhZ-N3HfJjtg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_105912_33_249b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.207Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNDY3Fnd0Q5LzA5WlIzVWlWMGZzdDh0aURJSXg1Q3EyZEY2R2tYWExMbmhURXIxaWVRbVRLanhDeG8ySGdtS3gyOGZpbUFOUzdpNzN2dWVLQkVFWVVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDNfMDZfMjI3Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmFmMmJjMTgxZTJhZjdlOGEyZGI5NWZiNTMxNjk3ODkzZTUxNjRmNzcxMDY2ZWQ4OWI3Yjk4MzA1ZDcxYTFkYWU2MTdlNzFiNjk2MzJkNTM0MmE1MjMxNjgxZDY5MDhlY2I0MDYwZTBmZmVjOTkwMjc3OTEzNGIxODVkYzE3NzI3ODQ1YzNkNDk3OTBiYzY0Y2E0ZjBkM2IwMjg2OTYxMTc0YzNmZTYwMDNiMjM2Y2MwOTQ0MTA2MDMxMzg5YWVkNmVlNTMxYWEwZTJiMjAyMDRlNjk0YmY5MjkxY2EzNzMwNjUxZTM0Y2IwZDVkYTkzZjAzMzliZTA4NDI1N2QyYTIxNjIxYWY3NjFkZWFlM2JkNTBlODZjNjhjYWM0ZmU3Zjk1ODM4YzI5ZjMxZjY2OWQ0MTMzZTNlYjY0OTYxMDkyODc5OWQyNmY1ZWViZjMwZWRiZjA5NTMwOGQzMjQxMjc2NjMyNjAxZDY0OGExNDZkNzVhODMzNTZmZmFkZTQzODFiYThkNjM0Y2Y4NmFhMTE1OGJiMzU3M2E3OTliOWE4ZTVhNTBmNDMwOTU3YmViNmU2ZTFiNmE1OTU5NjZjY2IzZGU3ZjM5MDViZWI4NjcwNTM0MTJmM2JjMmMwODVjYmU3YzhhMmRlYWUwMzU3ODRkNWIzMTFhN2I4MGNjY2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZfErZAAIq0aFxSqTPRd0D2kF8G90C7S6xbTB8IgchKjrdIY5HT_im_HFJFQmCUqmn9PmFNdkKAef-_D6NXOk2w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113143_06_227b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.212Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjNmUWwrVXBhdE1aTmE1MGhWajNCUFQ0RllSRFprSGxWT1NvZHhvTEdEVk9BWkNhVDAwdnhTZmhWUnVPUmY5VHh3ODZYb3ZjQzdCT1RYM25KRU5IK0lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDNfMDZfMjI3Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGFjYTk0NjA1ZDUwZGIyMDYxMTAwZmY4YmRkNjcyMzAyMmY2YTUzZTE3NzNjODBhYzdlZjI3MTRkYmI1NTUyZTE3MDE0MjNhMWY2MmQ5OGYzNDk1Y2E5MDY1MGI0NGYzMGZiMDFhZmJmYjRlM2Q0MDI5OGUyNzQyOTZmM2ZmMGY3MDRmNTlkMGM4ZGIyOGEzZjQ1NzQxNzFlNWY4N2M2ZDVhY2U1YjZkYzk0YTc4Y2RhMDdlYzQyOWI4NDU2MjRlY2NkMGIxZGUwMjI4NzA3MmMyMDUzZjgyMGZhMGVjMDg0ZWVkNWI3MWZjNGM0ODc2ZWE3MTliOTEyODBlYWVjNzNlMGViMDUyNDE3ZWQxN2M2YjlmYzNmZWNkNDA4Y2YzOTdmNjQzZDlkZmZiYmU3YzE0N2MwNzZlMjBhODE0ZDdiNDQwYzhkMzc1MTljZTQ4OWQ0YjE0MzIzYzY0Y2Y1OWNlODRlNTE2MjRlZTZlYTRiYWZmMzIwY2IxYmIzMzdjZDVmZTY2NTg2MTgzOWYzNTE4N2M1MjM5YWQ0YTA0OWU0M2VmODAzMDkzZGE4NDJhNmUyOTUyOTIzZGIwMDZlN2RhMzNjNWI1ZTVhMjQ4ZWFkNTM2NzNkNmU5Y2EwYmI1MDBmMDFiMTVkYWE4MjM1MTBhODc2OWIxN2E0ZDM0NDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TWYUPdlObX3XTImZuJlC5Gq4nU8-n9QJHUfF4tdYoClFqyS6wHpZkduzbIqqYqpu1is5Xr9ekTSma854AEgHEA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113143_06_227b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.215Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjFiMGlCNWFjck5rQ1djVjIrczIxYnoyYVNGbzdQUEQyUkI4SHRyUU96RkdZNjFNZGVFSjVVOUtRanNTczdaa0tvWTVPdDdKSmFLV1ExTTd5OE9ZOXlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDNfMDZfMjI3Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDAzNGJlN2FjNzE3NmU2MWUzOTA5ZWQ3NjY0Y2U0ODNkYWY3MjM5M2RiMGI2MTE0NmIzNzdkZDUzMGM3M2YzODBiMDg3MmE5OWU1N2EzNGY4NTFhYjA0YWI1N2VjYTY2MTUwMDBkMDk4MDc4ZmY0ODgwOTY3ZDkxNWIzNGRhMTZhYTIzOWRlZTA5MmQ3N2MxNzIyNjUxYTBhYWExN2QyYWExZDZiOTZkMmYxNzRhZjBkN2MyMTEzZGRlODc0NTRiOTJlMTc4OWRiNjllYWU0ZmJmMTQ5NGEyMzhiODMzMzcwODRiODEzOWY0YThmZGM2ZmQzODhmYzA2NWQ5OGYzM2M5Y2FlNTM3YjI2ODFkNDBlMzFkODlkN2UwYTBmOWE3Y2M5MjE4ZTM0MjRiYzkwOGMzNGZlNGRjY2Y2ZThjNmFjYWIyNDgzZjFlMzU2OGY2OGZjODA2MDUwMGI2NDYzZGFhMzE0ZjM2M2Y3Y2RmOTYzM2MxZjZiMDYyODIyNTUyNmQ2ODZkYzk2Mjc0YTgzNjJkZTYwN2NkYTk4ZmE2MDZmOGEwNWFhODZiYTUzOTQwYzAwYTYzOTNmMzBmMDA0N2Q0NmNkMTdiYTEzMzViZGVjNTNmMjk3NTYwNTAxYWE3NDMwMDExNGQwMWM0YjgzMDM0ZTRhYWE4NjgzNWQ0YjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.scKqjVzV5Wlj7tyFDK4YEbmuNq45BTZtpKiL-0U4qBwemHkuNqKI3rfd8jnpo76O_3BKEPYs7Qrjd4mc348prQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113143_06_227b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.218Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNweVNuQWp5a2VhZWhxaVhPL1NNcGJGbWorbkQyUnVlbVhTSEFoR0o3eHBIRWlJRlhyYWZrSG1ycEdUUGVQVEVIbmxOZ2FaaWNwSmNNQUZkZjJMVWxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDNfMDZfMjI3Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGJkNjVjNGY2N2VlZjA5Mzk0ZmFlMzAzMTkwYmViYWIxOTQyOWUxNWMxZGE0ZGZjOTRlZDQ4OWE5YWU0MmI5MWZiM2IzNGM2Y2Y5NDYwMzE4MjI3MTAxZmY0YTRjNDBmMTQ5MDEyNzgyMDU1ODlmNGUxMWFlMTI1MzJhZGQ0YjRlOTMzODRjMmZlMDYwZGUwYTJlMjdiNWJjYjU4NWIxMTQ5M2I1OTE0OTY2YThhYjAyN2M3MWZmYWMwOWJkOTEwOTY4MjA2OTkwMGZiNmVmMWQ3ZDVjNzE0YjlhMWVlM2IzODQ0ZjIzN2E4MmFjN2ZlZjA0NjJjZTk2OGZjZTc4N2UwYTgzZGQyMGEzMDQ2OGRiODBjMzM2ZWQxYzkzZTg2NmNhZGZiYjYwZmMzZGVhMWQ3MTAwZTRmMmY3ZGE1NGNlY2M1Y2MxZGU0NDAyMGMxZDViMjEzYzZlZmMxOTk0OGY5MmE1OGRiNDdjNTQwNWJkZGM0YWRmYjFkNThjZjQwOWZhZDFmY2NmYWYwYTEzYWY5MjQwZThiYTRiMTg5M2YxY2RjN2E4NDRiMTljNzMwOTUzODUyZmUxMTQ3ODMyZmNhNDY2MGJlNzc0MmZmY2U5MTIwZjdjMDM4YzcxN2NiMzc3OTdlNjVkMjBkZjM3Njc3MTM4Njc2ZTAxMDdiMTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EZFwRldwoUkII-JIFu9ie8mF9t_7Y7_CJB4wRGom8UGo3dN0GvKK7z_AoOlj-0axyEkcofTnctq5KANB7BSQGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113143_06_227b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.221Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFwdzJOS3F1aG5mdEt0WEU4REpQMi9wclhqeGIyYWZwckdDRStHTG1IazdiSW1oTHNxNEQ2QktWMWRLaHU4MUtaTkNUTHo4R3d6NU9wQWpWdjVUZ1FRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDRfNjJfMjRlMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjZhZGY5ZmFhM2MwNTJlYzY3NmFjYmZkMDYxMGY1MDNhM2I3YzU2OTc2NWYxMjAxYmMwN2QyZDg2ZGVmMGYxNTVmNjJlOGViYjAxZDU5MWI4NWIyMGJiZTJlMDQxMWM5ZTFiNmFkZmNhNWRmZDU3NmFhNGM3Y2QzOTIyNDJhN2EyYzcwNjcwZWE0ZDAyZDg2MTQ0ZGQ2YzQxYzBhMzYxYmNjYTEzZDhiMGUyMTEyNzgxYTQxZjBmODc5YmYzNmRiYWM2MWE5NDFiNGYwNjJkZDYzZDUyM2ZiYTg4ODgwMTEyZjZkM2Q3ODMyZTk4NWQxNGVhOGIyODgyNzk3ZDQwODg4MjJhNzE1ZWExODViMmJlOTk1NjAxNzQyNThkMTcwYmViZDE1NjE4ZDdjNDcxMjI2YjNkNGFkOTA2MGY5YWUzOWUzNjMxMmZlOWQxNDgyYzAyYTBkMTNjMTFmZTVmNGMzM2ZlODM1ZDI0N2FlYzdlNmZkMmYzYzJlYTliYTQ1NWRmMjg0NmY4ZGFjOGM3YTU5MDdhNzdlOTVkMWRmZmIzNzkzZTVhYjM4Yzc0NWUzODY1MjRjZTMwZjM1NmRlYjU0MzRmZjU1MDkxOGEzNjJmYTE4N2JmODU5ZjBlN2U3NjRlMzJhNWRjZGRmYWEyMzJiNGI3ODRhMWEzOGFhZTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UpF9cjvmvneTDWIHl8rBrvfFBe8Vk48wDowJRhoFOqmlUcJuDPQtqsV4RgsRjS_Wj7h-WxSJKxX5EXjOAeVokg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113444_62_24e1_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.224Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJhY0dFYTJRQy80czRJNklUbkVCZ0RwVGRjSTRzTldMZW9CeVhGL1JzdzhjM2tWb1YwSTEzMDAwTWJkSUdpTVVDUkdNNTVEeFRWQlo4cWVNRGxGNmVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDRfNjJfMjRlMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODhhNDMyYzBkNjFmZDUzNWNkNDJiNTMzZmI3ODQzYTQyZDRmZjdkNWI2OGU4Yjk4Zjk0NDNmODdlODY2MjY1NDRlZmEwYjk3OGFiYWExZTk2ZGMyYzUwNWIxMjJhYmNlNDUyYTE1NGI1Y2NmNzk0NTc1NjgzYmMzZDM4NWE2YWIwZGMwOGNjMzY1OGU2ZDJlMTYzOGFiNzRhZDU0YWJiZmI0NWJkNDU3Yzg4Yzc2YzUyNGQ4ZDg5OTA4OGY2NmM3MTFmZTY5YzllMzU4MjQ5ZGNlMWI3NGU4YzE0MmMxMjRiOTM1NzRjYzE2Y2NkNTRiOWI2MGU4ZTdlNjkzMTFmNDE0MTQ0MjNmMWIyMWRlMTcwZGQ3YzlhZDVmY2I3ODQ3NGMwMDVmMzQ5N2M2YjA1YTBkNzk2NGMzYzBmZDBjZTg3MTczZTcwNTQyM2M0ODdkZjIwOTMzZGMxMDIxN2Q5MWM4ZDdhZGNjY2VlMDUxYWRkMDIxYTU5NDU0YWMwY2Q0NzUxNzAzZDk5NzhjZTcwM2NiOTMwMTMwNmI5ZWM2ZWU3ZDVkNDZmMTcyMjkwNjU1YjMyYjY5ZTViZDY4ZTkwNDZmMTE3ZmVjOWQ2YzMwYTZkYThiYjc1YjEwODQwYzdiZmZkNjdhNWE4OGM0OTg1ZjgyYThmYTEzZjUwYWMwNjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rrBwzygjEKgbTEbQkSDOwD3fI18d2opt-5go7a_5y_jh0xFe2KX-eJ6ppTlhpVRusUFmGF1AD9Qi4A08GfUxCw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113444_62_24e1_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.227Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImFQcTgxcWx5cVlpT2pkS0tiZm1uNWFGekx0dzBLTDB3V3MwOUorSlhXWjRFZHBBV3lsVUJqUW9sVVdVZWJjOHh0a0lsOExiL2kzZ3R2R1dPOWRjRDRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDRfNjJfMjRlMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWVjOWUyMmVhOGRiNmExMjlkOTY1ZWY4Mjg5M2QxZDIzNGMwYTRmMzhjNjFhYTdmZmU0MGMwNDVjZTRkNjgwMDhjOGExMDVlNDFmN2JmODY0M2RhNmY2MWNmNTM3NDFkM2VmZTIxM2RiMTVjMGQ4NTlkZmUwZDQ1YzQyYjJiZWIxYzU0ZjA1YjZlZWYwMzUwNTg1NjgwZjExODMzYzM2MjQxNzhlOTY2YWQzMTZiMDVkOTllZmEwM2Y3ZTcwYzY4N2NiNTdkMDFmMDhiOGQ5ZDE5OWVhOWUwMzgzZTJiYTkzZjM1NTFhODgyNWI3MDk1MjY5ODEyMDdkYzBkMjIzZjZhYjdhNmU4ZjVlNTI4NmZkODY2ODcwNGZlOGYzZWYxOGJhOGYzY2EyZjI4YzNiNDc0NmM4ZTBkYWYyMjdjZjAwZmVkZGY4MDdhNmFlNGNlOTFlOTQ0NzA0OTNlOGI3YTU1MTEzY2FhMGZjMmI0ZGEzZjg5ZWVmNWFjZmM4MWU4YjgzMmE2Y2VjYjUwYzZhNjM5YjEzODljZWJiYjJlZWE3NjAyM2ZkZGQyMTM4MGFkOTcwYWRlNTAyYTNjNzk4MWVkN2Y1MTAyZmJlYWZhYWJhYjVlZjcxNjk2YTM1MGIyNDU1ODRkOTc1MWNkYTBmMDgzZWQyZWFmN2Y3NDRjZDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Za0L1AVx-0Ail4IFtTL-emclg4Zb7luw-c3IXcX8X7W8eVQR53OQyBC8mHECzuUUGRWhTmAqqTqPwTeT4Iu4SA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113444_62_24e1_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.230Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IklrTDlpMzdBRm92dkI1MWRFOCtEU2g1MzdXK0dQdElyT09FUHU5VWdZNktsblpXSVZSYllRQWxpaFlCVkp3cVlvNCtHaXZXTW5KMGdXVCtncGs0QWJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExMV8xMTM0NDRfNjJfMjRlMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWQwYWJkNGRlMDIwMGRkYzJhNTg1Y2ZmNThjODhmNjRjZGU3ZDQ2NzE5NmRhYmUyNTJmYzIyOWE1YjY0MzcwNWUwZWFkYzhjYmM1Y2RkZmNiMGQxNjkwNTJkYjg0Y2ZhOWQ0ZjIwYmJhNGRlZjY1MDgyM2JkODY5NWEwYjg1Y2NjMGIxMzJmNGQwNTEwYzBlMjJiZWJmNjgxY2FkYzJiYjQyMjUwYWE1ODZiN2IwZjVlYjMyNGNlNGE0NTJlNTQzYzZlMWE3YzdiOWY3MzM3OWRkMDRhNzZhYzA1MTAyZjczOTg4ZGFkNGMwZWQ4YjQ0MjQ3ZDNlMTY4YTdkMGYxNDQ1OGE4YTExNzA2OGI5NmU0YTMxYzRkMWM0NjMyNzg1MWNjYzllNjUwNWExNWY0NDU4MjBlYzg0YjliMTZmZjQxNzEzMjBlYmNhNTMzY2FkZjEwMTI4ZTU1ZjA1YjQ3M2NjOThlYTVjNDkwNDdjMmVlZjhmMzU1MmUwMjk5NzIzNzAyNjVjNmM0NGEwMGY4NjhhN2EwOWNiNWMyNzhiYjgwOTNmZTk2NzU5YzcwNWZhZTA4MGJkNTg4MjM0NWU1NGM4MWE2MDcyMjVkYzQ3NjYyMDRkOTE5M2UwNTJlOWRlYjVkOGZkY2QyYzMyZDc3ZWRlNmNjYTlmZTAxMjc0NWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lNCsUh_4B06pNNN-GdN3dSnbfuZdl9skKXRS2mHZh6t6Rdc7ZFn_RHVT6HxT-zeIs5DlLj4uFr98j7Iy0QoPDQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240111_113444_62_24e1_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.234Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im83SGZpK3dKNlVZdnZWdkpQTXVTOW1mZTVvMGloRXZSTVIxUWRsYzE4a1VxQmpuRGNDR1pIZDBVVHd6U1hmay9WUXVHTTJYeXJIVWE4UEovQTlJK1F3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjJfNzdfMjQwMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzFmM2I0YjVhOTE5NjA1MmExN2MxYTczNzY4MDRiOWZjOGJlZDIxN2NhYzYzN2M3OTIwZTZhOThhY2JiZTFmNGQ2ODM2YjNlZDhmYTgyOGE4MDk3ODViZWMzNzNjMjRmMDgzZDJhYzdlN2FmZmJjNzhjYzAxMmQ1YzczMTY5ODE5Mzg0ZmQxNTg3NGY0MmU5ZDM4ZWVkNTUxYTRmZGVhNjFiYzg5ODkyNjM4ZGRjZDhiMjY3MTUwY2FjYTAwYjAyN2MzYWZlYzdmZGQwZGY2Njc1YTA5ZjRhNmQ3ZDMyMGFjMTVkYjI2OTc3ZDc5OWY1MjlhZjZiYWU2YjUzN2FiZTE2ZjcxODc2NDUyNGNiZjFhMDAyMmE5ODhkYzQ4NTkyODIxNTU3NGQzNmNjZGRkOGVlZTg2MmJhZjdjOTcxZWU0ZGJjNTg2YjJlOTA4MmJjOTMyMzBiNDQ1N2FjZWJkYzJjYzg0ZDU0OWMyNDg2OGNiYzkwOTllMjU1NTVhNzExZjYwOWVjNjFkZjAxYTFlMzRjNTUwOGEzNzA3YjdlMDY2NmFjNWMxOTljZmUyZTM1Yzk2YzI3MzE2OGNhYWJjMGM5ZTY0ODU2ODcyYjM2YmZmZGJjM2ZiYzJiZjI1MzVjNDYyOTY5ZjM3OGFkYWI4NDY5NmFjMDUwOTM0NjE0ZTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.R9FF8KwPCMclvIce-jCYx9anv_nbP0TpQdcVQZSUllsKoe20Ip9aS-Oh1011wSadbpt1woItsIxKEl5cKT0E6A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112622_77_2401_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.237Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRSM0dlK1VHLy9HVkxObDlkajVzaVZkSG5aZHpRVXV5NzJ1a0JTV2l0OUhvRVlZcUtCYmp0bmZIQ2FUMHJqaEpsMlEvYnpzVkxiWkpJd1RqM215RkpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjJfNzdfMjQwMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDZlYTI2NTU0N2NiNjIxMDU1MThhMGVmZWJmMzczM2RmZWQ0NjdmYjZiNmY1ZGMwYWNmZWFjYzcyYmU3ZTAyNDIxYmM0ZGU1Zjg3MjNlNTNhMjhhMWZmZTk4NTQzNDFhZGMzYmY0YzkxNTYwOGI5ZmI2MmUwZDQxZmFiMmVkZGMwNTk5NzY2MzNmODkzNzY0MzcyNGVlYjY2MjIwMWFmYTljZTExNTk3MjY0NzNhODE4NzEzMGRmZTM1MTc3OGQxNDNhMzFhODVjZDFlYWU4MmM4Yjg4MGE1NGViZjE5YTllZTJiNzQzYzUwMjQwYjExMWFjN2RhOTgwYTExZDA3ODZiNjU3OTc1OTQyOWUxNjY2ZTY4MjAyMTFjZDlmNzQ2YTZjYjZhMzAwMzJlODM4MTFlZWRjN2I0ZDczYjExNzQwYzMzOTU0MTdmOTFmNzRkYTZiMjk5MTRhYThlZjU4N2E0YTBhZTU1ZGUwNTM5MTkxZGMyNGNhZDNlMWVhYmI2ODNiN2VhOTRiOTFlMzczNWI2ZGYxZjRmNmJkZTQyMjgzYzFmNGE1ZjhmMWU0ZmI1MTAwODRiODZiYjFlYjAxZmY4NjQ0YzAyNTFmZGQ0MTRiZjc5YWY3ZTcxMDcxYjUzMTk2YTlmYWQyZjNhYmNmOTA1NjY4MjliYmFiMjUyOTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RgYfHFMqwqZyhWAt-TKi0ILzz4shqJ6eEEohXz9O0bHwDDyZFhOmcf0iOSH4v00LE5Gm0XPNT_I2QLm6ebOSQQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112622_77_2401_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.240Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InltRkNuMmdKS3BSMFluVEd3cEVwckgwT3Bnc0RVYWl4RUxYSWZjN0NhREtPaTd2bjE5MmJ4L0xONWVaeWF6TGdlSE5aMkFDcGR3dVk2RnowSitsOUl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjJfNzdfMjQwMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2VjMzdiMThmYzdhNzAyZjE0Y2FlNjQzNjE2YjA0NWEzOWQyNGRjMGJhY2ExNGRjODQ2YTE1NjM0MDE4MGM0ZmMxNDA0YThkN2MzYTg0NTBkYTVmNDRkZGYxMWYwZGU5NmM4YTAxYzkxNjhjODBjMmM1MTMzNWVkNGM5MDA2NDk0MWU1Y2Q3YTE4ZGNlYjczZDM1MjFiYzcwY2M3OGI3YzI2OTViMDNmMWMzODc1NjAxM2Y2ZDY2YjE0NmYwMTQ4YzFiZWY0ODFkMjU3MzZkZGU1NDAzYmM2MWZkNzk0MGQ0ZWE2Njc1MzE1MjIwMGRhN2UzNzk3ODljYmVhZDAxMTkwYTE4MDE1YjQzZjkyZDEyOTNmMWFjZTUzOTZkZmRjNWUwMTRhZDgxMDJlNjFlMWE4NzBiZGMzNTk1NTRlN2QyNzcyMTA0NDVjNTExMzk1ZGQ2YTg4NjAzNTdmMDczYjc1NjliODJiNmNhYjUwZjM2N2YwNWZjNDRlYjRkNDZjOWViZTZiNDcwNWE0MGU5NDcwMWQ0YmFmYjg0NTgyOGRiYWNiNzIwNzljMjRlZjI3YmJhMWE1NjFhN2QxYzIzMmY0YzI0OTk2Y2ExODY2YTM4ZDc1OGIxZmYzODM2Zjk2Nzk4ZTI1ZjgxNmEyMjg2ZjZiZTEwY2Y2NzZkYTQ2ZDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.non9UDPWgvTVqfyIUZb04GUkPseYjvKJNe_OscY0EQi-9J_8-V1egi-Idp4zFKWtLQQqrcgw2qJifGmFsa4L8w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112622_77_2401_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.243Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlZqY2pFbUF1OEpLSG9Ra3JOTVcxYWdPVmhKWVlaeWtXZFJNbnFjcnBSM29kNVV3SHF5TEVrTzR4QjBTK3p0OThFMlk5SU1BYXRhYTBvRUlXUmRsaldnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMyMV8xMTI2MjJfNzdfMjQwMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVhODNkNWVjNjQ0MjQ2ZjM1N2NiNDdhYzY2NWJhNjYyZmUyYWM4NjUwMWY3NzFhNTdkNGY0MmUzZDY4YTVmNGEzNWExMzMwNDFmMTQ1OTJhZjkyYWQyYWFhMDc5NmY1MzQxNGIyMzUwM2UwN2M2NTdmMzFmOTkzM2NhNjFiYzM3MWZjOTFiOThjZTUyMTg3OGMzMzc4YWJhM2M2ZDYzMzdkZjAwOWFmNDRlNzg0Yjc4YjBkYjY4MmU3ZTA0YTJhNDIwNjc1ZTZkMzgxNmQ4OTljMWFiN2I1M2ZiMDcwMzkxMjk1NTUyNjUwZjdjNGJmMzNlMWU5ODc4YjRhMmVlYWNhMThkYWUxOTA0MWZjMzhlYjVmZjU1MjhkMmFmMTcxNzdkNTZhNTM0NDNjY2I2ZjMwZmY3ZjdiZTM4MDM1Y2NkOWY1MGUwNWU5ZDNjNDk4ZmFmOWViZWMxODQ3YWIyM2ZhMDA1YWFjY2I0MGIzZjRjYjI2ODdhODgwZTk2ODRiMjkyNzg3M2M2ZjcyNDc4Y2YyMzkxM2YwMjE2OTRkNzYwMGUzZjJjYzZjZTcwODIxZGQwMjcwN2MwMzhjNTE1NzYyZmUzZGFhZTg5NDA1OWQwNmJlOTM4NDFiZjI2NGJlNTVlZTk1ZTU5NjYyOTkzYzZiZjc3NzcxMzg5NjYwMDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ocWkH21ulgpnKEiDQUE6ddfxRvBybBXC9XlaPptNublSgX3WH7kMDkG1YLgTCyxyeqZh7YXDSAqJZY2pRTIoOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210321_112622_77_2401_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.246Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IklBZldiWDAySlp5YjFIeHNGS3NkelAyR3B5Qno3Ni9Dbzg3Q0NsZlRYTkJOOGZ5RzB3TkNjZFdzd1VKanF1NXRLMGhPNUtkbTUvcjFZVFBxZk1rMTVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjJfMzFfMjQ4OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTdhZTFlOTA1MTEyNDY3ZDZlMDVhNmRiNDExOGFhZmZlZDVlNjMzYzNiODYxZGUyODU0NmM4YWUxM2U5Y2U1N2E3NWE3ZTY2YzFmYmM2MGNhNjE4YzQyY2NhYjZiMDJmZjdiOTdmZjU3NmI4MzlhMjYwZmNhNTZjZTA1OWY4MTBlOTU5ZjU4OGNkMDU5Yzk3OTQyN2NiNDUwNjkzYWUyNTc5NGIwN2U3ZWRlMmJmNTYyNzMxNWMzYTI2ZmZhYjEzOTA0NmVhNDc1MWNmMzI0NzU2MDk1YjEzYTQ2MDU3ZWRmYTA4NWNjZTMxNGEyZDA5Yzg2OGRlYzRiN2QyNGQzNjkyNTVkOGEwMGYwZTlkZmQ4NmJjOTRiNTcyYzNlNzNhZjRkNGJjNTA4OTYzNjNlZmY0NjEwNjhiYTg2YTg4M2EwNmU4M2VkZWY1ZGZjZWU0OTBhZTc3MDkyMzc2ZmMxOGYyODc0Y2U4NzJiNDU0ZTVkY2IxZTBhZjYwMTg0OWJlMTdlOGIxYTlmNzdhMGU5MGZlZTFkOTZmM2FlZDk2Mzc1MmNmZWJmMzM4MWQ0YjNiZmMwZWQzYzRjMGQ2M2M1NmI2NTUwYjg4NmMxNDZlNzcxNTAyZWRhZDVmYTY0MDkxOTc5Y2QxMWI0ZWRjY2VlOTZlMmRiNWVkYTE2YzI4YzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.l5rK7eoY05vQLC_8K4SuBUn60rvU5B4VJyNpZ2iae7atDgfh3BqqCSWsQxqTbaHPpgdsquR3_0rOmLVWBXprRA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105822_31_2489_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.249Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImIxSUpWTDZGaXZEUFdlYU9jdEFwcEFIYXJCcCt3QTUwb0FpU1lXcmgyU2FYZVpaYzBkOGRFUGNyNm9ZMUEvWmlNdUh2dTdkQyt4Ymd4VEJmeGxrbVFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjJfMzFfMjQ4OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWJiZDhjYzY1MmFmYzg1NGM4NmYyZWMwN2ViZmYyY2JhYzY3ODljMjAxY2M1N2I4YTUyNGIyOGM3ODFmZWM2ZGNlOGI4YjZmNzg4NmJiNzczODEzNTkyZDdhYTVmNTliZDE3YTM2Zjk0NjU1OWUwMTUzZWY1MzM4OWU3ZmEzMDA3MGFmMWMxYTU5OWI4ZDliNGIzYzQzZGIwNGRiNjBhN2JhMTQ5ODQ1Njk2NmQwZmU0NTM0NzlkODRkOTY1OGZkYTFiZjRjYTFjNzQxZjNkNDY0NDA1NjMyMWY4MTE0Y2EyODBkYTBlY2NkNzFkZDc1NTdlMTdjMWMzMDVmMzAyMGMwNmVmNWE2MTZmYjBlN2EwNWYyNGJkMDlmNDExZmM2ZTkyYjRkZWY0ZDY0Njg3NTgyNDA4NDZlNjdmM2RmNTAzN2RmMWExOTE0Mjk3OTA2YTYwYTQ0MTIwY2EzMGMwOWZkZmE1Njg2NjNmZjhmYTNmY2E2YTRlYzllOTgzNzJhY2IzZTc3NmViODhiYzdhNGJjMjU4NTUyMTAzZWQ2ODJhODYwODkzYzVmYWNkYzhlMzFjYTBmODcxODJhY2Y5M2E0MDI5NDIzMjM3ZTZjNjU1YjJmOTNmNzk0MGRkMGE5ODFkMmUxMDIxNmE1Y2VlNmNlYzRiODAwYTk3NTNkYjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NKV_eQjQZGGD-1HbKNUKRZ_NLIKdJNxjjkQmZyeh6-nKXPjMaEL-p1MRYdCoZ9VxWQzxBUPX__YYm-FtT05ZQw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105822_31_2489_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.251Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkI1WWJnR0RmOXp4VHc1Z3BWSHZWR1JTdWxXL0dJUEpNUXVtNGlKSmlNbEt4TzJTbFMrS2FITDF1cDBxUzlLbUFBNFEzSmYxT1Z4RUdXWVFJcEFRUkNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjJfMzFfMjQ4OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGYzMjZlY2FlZGI5NWMxMDZiYmQ2ZjE2OTIwNTU5YmMwYzc0MmRmMTI4ZGFjMTI1MTBmYmJjZjYyOTRiMTRlODlhZTVjOTQ1ZDM0ZmY0OGQyNjU0ZGZiNmQ0MTBlYzJhZjIzYzllYzg5N2Q3YzZhMTE5OGVhZjFlYTc2YWRkZWExYTRlMjg5MTEwYTg1ZDJkMGZjYTYzZjY3ODhiYWEyZTY5YzQ0MDk2ZDJjNmMzZDRmZTc4MGMxYTk4YWFlZDQ4NGEyYzYzMjZmNzhjODZlNGRjZDU0YjZiZDJmODk0ZTY0ZGUzNmI5NWE4MjFkMzYzMDRhMTIwMjU4OGU2ZmU0NTEwYTk4YWQ3NTQ1ODJmNjJjODkyMjE5ZDNhNGI4NDMyZDFkODI5NGU0OWJmNzVhNDYwNjMwNTg5NWNiZTAzNDNhNzRiYmEzOWQwNjM0OTA2NGQ5NzQxMDQ2OWFiOGE2YTYyNjE2ZWZjZmE4MTlhNzJlYWMxOGE5MDI1NWNkOTY0NmZjY2UyMTc5YzFkYTBmZDNmOWYzMjIzNDYxMWFjZjQ2ZDI5MDVkNDRlMjVkNjM1OGQ4OGVkNDIyYjAyODkyMTI4Y2NlMjBkZTI1ZGMzMjI2MjNkOGVmMDZjMjJiODc1NTk1YTEzMThkNmRkNzc0YjMyNmVjMzNlMWU1YzJlYzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JQ9zH2fwWfcPA0_z-GpKCjFpv45xHDcIG34sITdqJ6siPzpq3i6EnDwNIXBeIJ8I8aFfK5XbsUZ_0WM2MPV0dA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105822_31_2489_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.254Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImhkZmNLejhzTWY0ejh1WVg3WjFrclVWZnFGS21jaExpdG1jYkhnYnM2YU5HWXFZeWExMXY2Z3NvVEMxcER1UGxSK1RuNjVGTHdvUTFxRDBGdXN3MVZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyMV8xMDU4MjJfMzFfMjQ4OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWYzZmZlOGIzZWNhNzRlZTE2MGY0ZDA5Mzc0YTU5MDM4ODY4ZGVhNjgwN2FjNTRjNTQ3YWFlZDgzOWY2YjBkNDg5N2ZhZDQ5NWY0ZTU2NzhiZGJiMzE3NWVjYzgyM2MzNDc0NzdiNDU1MzE1ZTAyYjdmM2VlNjAyYmQyMjFmNzg3NTg3MzViOGI3NjQzYjFkNzI1OWE2NGVlOTFmNjFlYzcxMTk5MzhiNGMyNzBjNGE2ODhhNDNkYjA1MmU3ODVjZDY0ZTg5MTU5NTIzNTdjYzA2NWU5MmI2NzU1YmYwMDYyYzFhMDZjYjM4N2IwM2VhODZkMzJlMjc5NDYwMjhjNWVlYmM3ZGQxZWIyNTc0ZWEyNTRhZjg1YTI0N2E5ODA3ZThjMGM2NGNmM2FiMGJjNjBhOGIzOTAzY2QyOWU2NjRlZTEwMjM0M2ZmN2I2ZmQxODA5ZDA4M2M5MDYxNWQxNTgyYTRhOTQ0MTcxMjQ0MzJhNTZkMGIzNTFjY2NiY2Y1YThjZmRmODI1MjBlZWM4MDczM2RkZDZiMWFlZTU5YTJlNzg1ZThlMDQwYjNiYThlOTM0NWQ0NmQ0NjYxNjg4YjBhOWQ4NWQyZWZmYjUxZDg3NjQ0NDNlNjNhYmY0M2JlZjJhNWVlZTRhZDE3MzJjMWY2ZDYzNDJhN2JkZTM1NmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.akDR3Qu9Nj_C9sPdtVcxIkFNNQKqptjHe1iEVcMGmYRIBqQSC8C7RzfWyXpnkPLv2lPcwuflU3pFwI007q_3yQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221121_105822_31_2489_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.257Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBKNHA0YUVCaVd2UVArSnBnQlcwR1N0SmR0WDlueHoyODFlekVvMDFreEFielNxd1FJbmhYcCtCZ0dNTzBFeFR5T3grcllpNTFJWnFKSW0xYmlGQzBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExMzdfODdfMjQ4Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTQ5YjgxNmFjZjE2ODk0MGZhOTExOTU4MWI1N2Q3Mzg1NjlhMGE0NDBjZTQ4NDg0Yjc5YzlhOTAwZTM2MGI0ZDZjNjg0ZTA2YjdiYmVhODIyOGNlZGYyNWI5YjU0NTU0ODk0NTQxODJmZjgwODlkOTAyYTBjMWY2ZjA2MDY4MzA4MzFhOGVlNWVlOWI1MTdhZDFlZWUzM2QwYzc3OGM5MDNlMmQ2ZDAzYzcyNGI0MDkzNTdmMGZlN2IxNzYwNDc2YTNjZGRlODVhZjU5ZjJmNWQxNzBhNDUxNjhmODNkZjc1YTFiNjM1MTQyYzEwODcyYzYzOGZiNGYwM2Y4NjE3MWJjOTM5ZThlYWMxMzFmMDYxMTExOTlmOWUwNDE5NzYwY2NkMzVlZDBiYjk2M2M0NWNlYzI2N2IxNTUxNDU5ZDMzOTU0MjAxM2IwZmU5YWJhMmQ2ZmI3MmY2MGRlNzUzY2ZmOTM5YjQwYTNlYjk5NjdlZjQ4YjBjMjFlMGJhODAxZDkzZmU1YmE5OTk3YjVjNWIyN2QyMjRhZTQ1MDNjMDcxNDRjZDQxY2I0YTI1OTlhNWJlOWE4ZmY5MjA2Nzc0Njc1NTkxZDJkMTBiN2Q5NDdjNTRmZWM2NWFmOWIzYWNhODg3NzkzOWZmZWY2Y2U2YzY2MjU2MDJhZDlhNDIxNzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qVqgBmXpTrObvYS71U-KdZgdRr6vjc7CWVVVdCLQblDYWarIRQw43QAO2fGaYXETndEPMHA8Ldl2gjNXlRP-UA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111137_87_248f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.260Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxpcWV3dy81Z2t0Z0x5cGRuZDMzWW83NlJUQ1FobEswMk9EUjdyaTVxQUxITUFpVi9nWHE1MkwvazBiQzF0NW5uZWtWVUNBOXkxODFMWUJJVytRaFZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExMzdfODdfMjQ4Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTNhYTY1OTY5YzNjNzEyMDk2ZGUxM2E5ZmY5MjRmZmYzYjY0YjY3YmMyZDVmNjFhYjVmMjc4NWNlZmU5MTc0MGIwNjRlZjY3NWY0ZGJlNmJhZDlmNWIzZWE3ZmY1MmViMmE0NTlmMTA1NDk5OWNjYTljMmIyNGJkMjUyMjQ2MDNlYzIwNmM1YTE4YWY5ZGIzOWQ2NWRhYzFhNGVhOTIyNDBkZTQxYjI3NzE0OGU1NzAxYzhlNThhYmIxMWQ1ZDkxNWEyMmFlODc4ZDZjYjhkZmU3NTdjYTRiYWI4ZjJlMGJlYjQxYmM2MzVhMWU1NGZlOWJlNThkMjMwM2VmMGRjYjQ3NzJiMjQ4MTJmZTYzZjExZmExY2NmMDA0ZWZkMWM2NThjOGQ1NzZiMmM3YjUzY2YxZTY3YTMxOWNhZGEzZDBlN2U0OTZmZTY5ZDQxOGM4MWYzMjg5Mzk4Y2I0ZjYxN2JjZWM5MzhhYjZjNTdjOGY2NzdhODJiNTY1Y2MxOTM2YzhjMmE1MTM5MzQ5YjRhODYxN2Y4N2JjNzk2MDc0NzM2NzViNmE3MzNiNzc2ZGE1MWI2NDEzMDg4MDAwYzk0OGEzOTViY2UzNjIwMGVjYjgzYzAxZDc4ZDlkMjlkN2JjODUyZDhjYzRjZWY0OGM4OWQ5ZTNkY2FhODRhMWVjOWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zmoyBZVX0ILgNNDahtgFcWFDqYBrLMXcoxUw-k8BGaxstJ6LUHt3iYOTF8U_zl0VVsP9Oh47Rcd0VjzP1U3uXw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111137_87_248f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.263Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJaMFVlVDZhUDdVMG1uNUhDM2RMV2FxZ1c1bUhqbEoyS3FZRk1iUGhzZE1QWDNXdGY4M3BEaVJWckJDN1doeVY1dmM5a0hsVHNUMEpnQzNHT2NHRStRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExMzdfODdfMjQ4Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTI1MGRjNDQ2ZTI0NjBhNzI2NzI4NGY4NmM5MmQxNWNjZjVjOTQzZjZiNzM2OGI3ZGJlMWY4NTFmYzkzZjJhYzExMzY3MTcwNmNiNGFhZGRjYWJlMDFiY2M5MTMyNDg4ZTZhMDg3NmYxZmI3ZWYwZmQ4ZWY4MmI0NjNlNmM5ODc1ZTAzNzZhZDkwYjhkNDQ2MDMxMjA1MmQ4NTkxOWNlMDhjMzkxZTU5NWJmYTBlMGYwZjU2ZWIzM2RjODA3ZGU1MjA0YTc1OTM1MDdkMDI2MmM3MzFjMzhkOTMzNjJmMmZkN2RjN2VmY2U2MTNiZjU1YmJiNWIzYmJlZjkwNWUyMmQ4NzMwNWY4NmRlN2JkNzdkNWRhNWUzNjI3YTZhZTllODQ3YTE3OTI4M2RmM2M1N2M3ZTg5ZDViZWUwMWZmYjNkOGMzZDJhMTM1MzAyNTk5M2EwZmVmOThjZTljZTFkNzY5ODJkZTY1ODY3NTZmNDQxNGY5MDA0YWVlNmVkNjk0NjBkOTQ5ZTk0NWQwYWRhZTYwMjM0NzNlY2EyNWNlMDQwY2I3M2NhYzFiNDcwNmNiN2U2MWNhOTQ4ZTQyNmYwNDViNzI3OTdmY2Q2YTM5MTU1ODkzNTY5ZmEzYzI1MTdjYjlhYzRmOTIzYzdlZDVkMTNjOWM2ZmFjZGQwN2I0MGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NNPuQOE4p3nrJiemvsLpm6tvShYIDDFjhJYvuyVxAniaJgIAYSkTNx0A-g4y3LdjA8OtnRtabmuyeRqH6DqgDg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111137_87_248f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.265Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImJCLzFHaTQ3anFZTzZtT2tzaVluWU9ibDVBNlUwMndwR1BQT3JtOFI3ZVcyYURQT0RjWlZxdGJYU0tLTGxiUDlaVis1R0NXbVlzYzhGWG1EcnpIcXdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExMzdfODdfMjQ4Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTdhYjFjZjJkYzdjNTZhNzk4YWMxZjVmZTY4MWNmM2JiNTUyNDg3ZDFjMjhiY2UzNzZhOGJkM2I0NWJmZjFiM2NmMmI4M2Y5M2EzMjMzZmU4N2ViYzU0NzAwOWZhNjRmNDc2YmU4Y2I2YjczMzBjMGY1NDZlYTg3MWNlZjFlMWI4NzRmNDQ4OGYxY2UxODdjYzU3NjM3YzhlOGM0NmVmMjg5MTlmMWE5MGUxMDc2MTNkMzk5MDg2NjEyZmRhZTY1N2M4NGEwNjE0ZDQwZTIwMTE3ZjBiNDQ4MTIyYjk5OTgxYjc4Y2MxMjViZWNiY2JkYzFhOTI1MDgwNzAwYWNmMjZjMzUyM2NjODQ2NWU1ODA4YTkxZDdkZjNiMjUzNmEzYTBjNGNkZWE0YWE2N2MwMmIyYTEzNjkzYTUzOGY2YWRlMGQ4MDY0YTAzZDgwMDNhODA5ZTc5MTRlZWEyOGRjN2M5OGRkYTQ5ZWQ2NDRkNWQ0YTNhNDU4YzY3ZjU0NDI5YTBlNjVkZjk4OWVlNDFjMDg3MjVjZThkZGJmM2YzM2ZkNWM1NjhkNTRiZGRjM2RjODJmNzAzZDFiMWEyMzhmMTI1Y2U5MjAwYzQ0NTQ0ZTVhYjZhYWNjNzQ3OGI4MzRjNzIxMTBkMDE2YjBjOGY4MzM1NmMyNTEwNjdhZmI1YzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.E3kwoXAoO61KlNAJMHXIATx5NlaDFbFVlhUnEzadBVFEvPb2MtwN4_JQapwgt-cmqhOH6O1tk2NYwkefleK65A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111137_87_248f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.268Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9vRUtYcDhBSTMwQkxUVlNvUzZIclQyZjdvcEJQOXBKL3Q0ZkRkdzJid0lMN1JLeUpaSk44VmU0RHRianMwRDNuTi9aT0FuaDQ0aHVRVzB4ODk4WmVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcyNl8xMDM2NDNfNThfMjQ1ZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTZhM2JmYTZiNjYyYzdhNjdjMjgzY2NjZWQ5Yjg2NDYxYTE4NTk1YmNjMmNjM2FlNGYzZjJjZmRlYTk1YTkzYWVjMjkwMTYwMjY0MjYzZDRhMzdkMzNiYmMyZGNhYTNmM2ZiYmI2ZDBjZjgxMWFiZjQ5ZmMxMjdkZGMwMzkwMTkwZmY2OTdhYmVjYjRkZTcyMzU1OTkyMDJmYTI0YmE1MTc3MGMyMjE3OTQyNzE3ODNmZjEyOTFjMzcxOGJiNjY5NWQyZWIzOWJmOGFkOGVjNTgxZWY4NDFjZjgxOGJjMjUyZWY1MDQ2ZjMxMjVkNGI4NjY1MTNmNzc3ZDZlODQzYzE0MDY0NmI2YTBmMzQ4NDY2MDkyMTlmZmFlZmM1Y2YwZWZhODFmZDQyYTUyZmY1ZmYxMTZlY2EzN2RiNzRlZjVjMDE5NmIzYjUwOTQxNTI0MTU3YjAxY2MxOGRjMGQ5N2JlMWE5ZjkzM2IyMDRiOWE5ODhhMDc2MTQ1NzZmZDRiZmZmZjJlYmQwNzA5Y2ZlZDA2NDg2NTMwYjk1NTYwZWUxNDc2NzlkOTVhNWFhOTk3OTEwOGM1MzA5ZjY1ZWIyZGNhNzUzYTM2NThiODQwOWY3ZjcxZjQ4OGE5Y2Q5ZTNjNmYxODc2MGQwZWFjYmJlMzM3NTdlMTA5MzZlMmI3M2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FROvZluW1BDsjd7e3k21qThgUkQo-tewG60X3kXyD4S5aW8PXXH-ngXAc8m6FWe3tzT_RNv-tHTWFfdkeU9lzg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210726_103643_58_245d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.271Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im1CUU5xS2hoWDJtNTJEbmpXaFNEQU91dUx0QXQ4NVQ4MDMydmJLWnR3dU5LZkFZWWRYSmN4T3FXQVVmbU5OM0VPY1o1U2ZMS1pMby9DTUdLWFFwZjhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcyNl8xMDM2NDNfNThfMjQ1ZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDk1OTkxMTAxYTg4OTM3MDcwZDVlZTE5N2MxYzUxZjdhMzE0OTBhMjlmYTE4NmI4NWE2NDZmNmI5MmIzYzJlZjUyOGE3YjIzMmRkYzcxZTc5M2ZjNWYzNGMxNmExZjg0MzZiYzExMTNlODM4MWZkNWVkM2ZlYjM3ZjE1NmFmMGEyYzVmMjc4MWRlMGFjZjMxMTllNzgzODRkOTgxN2YyYzBiNzljM2IyOTBlN2QzNmQwMzRjNTdkNTYyOWFiZGU1OWI0YmFhODFjYmEzM2M2ZjEyYzgwM2MxMjU2M2RiY2E1MzA1MGU1ZDRjN2ZjMjI4ODY3ZDU1NmUzODY0MGU0NjQzNzA3MjQ2YTEzYzIxNjc0ZDE1NWYwODdjMTFlZDcwNTFmNzJmMGVlYmYxNzRiNjg3ZjcxYTg0MzM5NjMxODJiNTUzMTEzZTQ3NWEyYzhmY2VkZTdlZmYyMjZjMmM1MDA0YTUwZWQyNDViMGFjNGNkNzRlMTU3MjRhZDE4ZGQ0MTgzMjliZmNiMDc2MTgxMGYyNGJhMTRiY2NkODhjZDU0Mzc0NWY1NDMwMTQwZGQ4MThjYzA2YWUyNWY2NzU1NDRmM2UyMThjYjYzYmVjOTIxMTY4Nzg4MTJlZmY1NWMxOTA1ZGViMTk4MjVkNjExZjEwMDY0MTA3MWMxMmMyZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6OodDmxW_Y1-703myzGV9YbWGRYKzti7yORd4S4yX-RnoAznMufMAEzFaVOqvC87JQbzSvU4CFusWh_G0_Dr1w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210726_103643_58_245d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.274Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJDYkVRTVJJVFpOYjkxK2J6SDZQaitYaUFaZ1JkMnNlbjc2aDdUME9SUitxSnlQYlJ5dXRBSllibXNwNTdVZmsvcEFZSm1IdE1FejRpRHJsM3RXWTVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcyNl8xMDM2NDNfNThfMjQ1ZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODlmYTVhYTE4OTNkMTE3MjI1ZjdlYWFkZGY0MzkwM2RkNTgxYTIxNmY4MDc1ZGE1YWNiYjA5YzhmNGJhY2JiY2ZmZmYyZWI4MGQxYzNlZTNjODk4N2YwN2JlNThiMDIzODg0MGM5NGU2ZDQ1MTdlNGI0MzRjZjQ5MjJjNmY4OGE0ZWNjYTM2MWY5ZTQ5OTY2Y2IyZjVlNDVhN2Q1ZGU2NjEyNTI5OWIzOGY3OTE1NDU5Mzk4NDUzZjk4ZjQxZmYyMzE0NWRiNDE2YjQyNWRhZGJmOTNmNWMyNDM0Y2YzMTVlMjM3MTZjYmY1MWE2NTk1ZDlmYzUyODgwYTNlNzRiZTY1MWJmNDQ4ZmZhOTMxNTVlYzRjNDBlNzQyMWFjNTlmMzM1NjJiMjg3NjdhZmUzOGM4ZmNiZjczYzc3NWExZDA1YmY1YWI5YTUxZGY2N2Q5YTg5NTk5NjMxYjM2NzNkZTA0OTViYjFjODRiZGMxOTc5MWQyYjI2Mzc1Mzk5NzcyNTNmYTQ3OWMyOWRmZWNmODhhNDY1YjZjYWZhNDE3OGY1MTY2MDA0NTI1NzY0NDczMTBmZGMwZjIwNmRkNzFiMGMxZDk1OGZjYThiYzU1ZDg0NmIyNmYzMWY5MGU0ZTNjMTY4M2UwNDAzMThkN2YwMmVjOWI3YTNjMWM4MGUyMzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AU0cN68PHOJVhUGc1Qwit_CmQmFgUgvLmJZFULlr8eBvOT47Ww19AgDQr7s2RWJxe-uzzmkIdvDg_W3pAaa_QA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210726_103643_58_245d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.276Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxxNjYvaENGUjhTcEwvT1paRmp4aTNGaS91QytvYUhBaEpCbUluTDl3SW1oYjVPMXhIRysvN3lHMnlIVDZpQ05YdG9nM2t6Y09obGdGOEtPZ0Q5UldRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcyNl8xMDM2NDNfNThfMjQ1ZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmQxM2RkNTI3NzFiNTE5NzY2NWY1YjAzNDkyYjgxZTQ2ZjdjZjk0ZGY1NzdiZTEzMTJlMDZhNzBmNzUzMjJkMTg4NWFmOGQyMTNjMGU0Y2QxYzg0Zjg2OWU2MDBmZjRjMWFhNzZjNGE5Y2U0ZmNkNmQwNWY4MWViYTUyOGU5N2I4NWFlNjY0MzhjYmY2YTY5YTIxYTZiYmFlZDFmMTM2MjQwYTg3OWYyOTQ2N2Y0ODRlZGFiNjQwYzNjNWQ2N2VhOTRiNmU3NzFmM2JhNWRkMWMzYTY1YzllMGI4Mzg3OTQ3MWJkNzZlNDE5MTY1NzgyNjZlM2JhNGE5MmJiMmY5M2VkNWFiOTRmYTdlOTMyNjVmNTg0NzBkZTAzZGI3MTAwYWE0NGVjZWYzNDU3NWNjNzcwNWZkOGE5MzYzYmI0MTU5MTc5NGZjZDhmMjFmOTU2ZWFiNmI0MTc0MWQ4Zjc0NjEwYzUwYTQzY2E4MWI0MDYwNzZhYzM3YjJlMGM1ZjdjZjJmNmI2ZDFiMzA2ZGE0YWJiMDlmZjg2NzY0MWEyZTM1ZGQzNjA0Y2U2NmQ4Y2VjN2UwNzMyYWZkMjhlODE0MDI0NmIyMGIyYmUzOWE5NDNlNzgzMWUyZGI2NWExMWQ1NWZmZjU2OGEzYTJkNGI3ODg0Y2Q4NmU2MTg1M2UzZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.h5efTFDqSwz8ltxwP0HS36obYY2uP5vwaoQ8m0F8Y3YxCE88YTVrkEMwu4VzUjgpTbHcM1jUOdcmdNEYHU2f5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210726_103643_58_245d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.279Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxqS2dJVTVEVHEyQ20zLzkvOW96UEY4SW02VXdvZWZkTGFGdkU1dFZPMytsVUNQN1liTGNiSGNLbVJBdUU5ays3MFh2MzgzVnBGQmZEcjA0Uml4MFZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDZfNTFfMjQzOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzAwMDgzNWYzNjgyODQ5YzIwOTM0MTMyMDc2ZGU5Nzg0MzkxMTY5ZjMzZmI3MTYxMzNjY2FjYWQzY2ZkMWI4ZTJlZDM2ZjVhNTQ4ZDVlYzVmODVkMDU5NzgyMTJlM2NjYWI5YjE5ZjE0MmFmM2M2OGEwNDIxMzI4Mjc1MTU0NWRkNjViMTU0NmVlODgxZGRmMmUzOGFiMmZhMjAxN2FlZmI5OTc0YjM4ZjRmOGU3MjFhZDk4Mzc5ZTdkOGIxMzMzMGE0OWY5ZTZiZGQ0NWY0MTJhMGQ5NmE2M2ZiZWU1ZjA2YThkZWI2ZTZlMTUyNTM1ZjIwZTI3YWY2MjVhZmQyZjA5ZTVlMDAyZGY3MGZlOGU2OWY5NjkwOTY0MWU1YTRmNzcwY2FlY2VmNjAxMTdlNzdkZmRkNTFlOTk0MjU3OTQ3YmU2OTQyMDdmNjJjYTIxMWE3MTEwMTkyNjk1NDA0MjhjNzdlYzA4MGE1NjkzMjBlMDg3ZDc5M2ExZjVmNmQzY2Q3NTA2NjFlMTg5ZjkxM2VhOWU2OGFmZGFjYjQwZDhjZDdiYmNkOTg0OTRlZWI3NmVhYTVhOGUzN2RkNzU5NDExNmNlZDk0ZDRmYjU4NWJiNzBiN2Y5NTUzOGNhMGQzYTFlZTNkZGE3MTUwZWNhMjI5MWYzNzA0OWNmNTE0ZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2-vcDVLz9Uj2_qZbGrNLBga9Pdd1Zthuwc5ix2WP8QCom3zZGr7wdf1I_aH9NrLrgUMcj7WLq_f1XgKOTN8s6Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111006_51_2438_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.281Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImJSa2plVGRQUXlyd3g5b1VsclQ0aUg3U2VueGVWVWl6aURJOEkxV212VWp2aExYOFZrVmh4YlZTb3lxdU0vTUdjRWV3RmkyTXJjdFM1Qks1VEZBMktnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDZfNTFfMjQzOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDc1ZWJhMGZkNjkxMjRiYWI2MzQ1YTZjMWFjOGJhZjdhYzAyMjdlZmEwYjE5ZDRjMTM5MzdjMDc1NmU5ZGI5NDliNGU2YTIxMjY0MGMxMmVlZDg4YWRmMWZmZjE0MmVjMjI0MzFiNzQxNWIxYTVlMTk4YjliYzM3MjQxNjlkYzlmYjJlODIxMzY1YWE0Y2FmNmJjZjZkYzkyZmVjMzdlZTQ4NTE4ZTNiNGU2NTAzZTc4ZGJmM2FmMTM1ZGFjNjRiYmQxYzRjZGRmOWI4YmY1ODVjM2JjMGIyYTdkYzkyMDJkZjRkMDA3NjgxOWNiNzYyZDU2MjExYjQxZjJiMGM4YTMzMjIxODQzNjkxOTRjNjc4YmZjMjdhNjUxMWI2NmU1YzA4NGYxMDc4YjgwN2I1MjVhNjc4ZjQ2Mzk4MmRmZGJjNGQ5NGJiMjZhOGIwNzc5ODA4NzI3ZDM1YjRkZWQ3Y2NkMGI0Y2E0NDM4NWZlZGVjYzFlYThkMDczODgxOWI5ZjVlODI0MWIzNmU4NWQwNzM4ZTA5YzZmYjllYzM1NTI1YzY4M2Y5NjM2OTJjMDFmYTYzNmUxYmE4OTAxODM2ZDFhNjMxMDlhNjFjYTY1MDBhNDMzOWFlZjAwOTY1ODgzMmVjODlmNzBlOWRhMmQ4NWMyN2Q4ZGI0N2QwNTcxYzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AnYGthG1O3jswb335OPHrb9n1DySz6x3m5ibl_oG6tXi__WpoeukTJa9jEaSnrxhfZpxzncKiLb4Irm0ZZBbJw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111006_51_2438_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.284Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imx3UjRzTzdnaTVvZUJ6UTVkL2pPTkdwOEdsRUZKRjEvdW00bG5vRDNhWU42L2Mvd2pwL0YrcDB3OElpR3hNZnB6MFBtTFd2cmhnZDNHMXlsU1F3aW1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDZfNTFfMjQzOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGNiNzk5YWNlZDBkZDViNmRjMmQ1MWEyY2E1NmJiMzgyN2Y3MzhiNjRiZjBhOGNmZTY4NmUxZTUwZGNmNGIyNjI3ZWFkMTRjMTFjNmNiM2M3OGEzZjA2ODQzYTY1MmE0YzgwOTkwZDEzNmE2ZGI5ZmRkNDA4MzFkZjM2ZWI2NDYzOWU5OGE4NTZmMDIyZDQxNzM4MDE1ZDc2MzQzNWU4ZTA0YjRkOTk3ZTg5MzQ5OTA0MTkyYzA1NDEwYWY0NTg5ZmY0MjY2OWY1ZDU2OTBmMzFmZjk0ODYwNTNlMTJiNmExYzBiMmM0NzlhMmRmYmNmNGExMTBkN2ExODQ0YmE5MzgxZDc4ZjE0ZTlkY2NlYTk4Y2U1Njk3NGJhZThiZDAyZTQ3YmI0ZGRmNzliMzM2YjI3MTcyYWMwNGUzYzY5ZWVkODk1ZjFhOGVlNThiZGJjNDJmMTM2YThiYWM1OTRjYmY5MGQwYTJhZGU1ZDhjOWQ3MjdkNzM1MTJiNjgxMmYxOWMzNWRmMmRjYmEyYWEzNjg0YWE3YjU3ZTRlMTdlNGQ2NTBhMGY0YWI0MzllN2E2NzkxMGRkMDZiOWNlNDQzNDQ1MWM4YmM5MjAxNGFjYjRkOWI1YjVkM2ZhMGJhZGE2ZGIwZTA5ZGUyZWNjNTRjNmQxMDhhYmM0N2UyZDJmNzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0UFSl2yCp8ZMzvqjTKPHO3IZGQufr5Lq_zV-9VtIUmNZ52jll4cgdO2mIUPlUOMzlD6IyEXGMcZYU_8Gulgl9g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111006_51_2438_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.287Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZYS2RqWFNjcys4N2tOOVFvVVdya0ladzkzT1ZDdVJJbDMzMjNzdXRDZHhoemZUQnJzRWxLcDQ4SENWV2hsY01QRGdiN0V4aUpZN0I0cDJmOTZpTlFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDZfNTFfMjQzOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDgzZWUxNDUzMzc1NDVlN2M2MTM4N2YzNDRlYjU0OWRkMWVjYzcyMGVjNWEzY2E3YWQ5ZDMxODgxMDhiNWY5NjFkNDcyZGU3NWJiMDU4NTQyZDgxYjM5MGY4NTk5NTFmNDNiMmQ4OWQzOWZmZWM1YWU2NTJhNWZhYzY5OWM3OTE0Mjg5MjE5NWEwMzNhODc3NzVhNDBlZDRmNzM3ODI1NWJkNzc2NWQ5YTgzNDBjOWMzYjFmNThmZmZlOGY3MjYxNTU5NTAwYTZkNWQzMjI5NGZkNzhjYmNkYTEyYTQ3MjFmMzM1YjQ3NzUwMjAwNTAzZGEwMzM2ZGExZWZlMjIwNjc4ZDQxZjc0MTlmYjIzMTU4ODVkYTI4YTNhM2E5NzAxOTQxZTU0MjVmOTQzMTI1Y2Q0ZDVlMWZkZDdlOTQxYmMwZmEzNjBkZWM1OWZkNTdjYzg1ODAzNDFlOTM3MjE3NjgxZDllMDBjYjJkZjgyMzM2MDJlYWQzNDEwYjhjOGY2OGI0OTE1YTM5MWQ5YTUzZWI1YzE3YjZlZDZmNjBlMzExY2IyZWJmNWIxZThjYmFmNWQ0OTMwMzJiZGJiZWI3ODE0NzQ5ODVkZjgzYzRkYzZlNGI3MmI4ZmRjZTkyMmY3YzQxMWFhYzk4MDRkZjdjZjJmY2I3MTAxZDRlYmRjMDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TcM2XaPEI5D3JzR_TkNMPQIPDVY0XmeuqKQqhp34Ib0lTZHTgtQjjYBpCW7IV5hmNaXB0q6GpYTh09j0AKoJqQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111006_51_2438_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.290Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikg4R1g3QXR0SDBLbmdYVXUzVlNnNUJUNkRYdW53N1cvY2hoM3pEUkJZcjhNdmVueHdrd2htQUVmdnluQkhSSHFDSWVFR3BtWFJDZHJpRitQdFBZL2tRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTlfMzJfMjQ1M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzdmMDMwY2QyZjAzNGQwNmI5OGJmYjlkMDMyMDZlYjIyNTk0MTI2ODQwYjcwZWNmY2MyZmU1NmE4NzMyZjcyMTc0NjYzMmEyNTcwNGIxOTlmYjUyYmQwZTdkZTNjY2RlNmVhZjg1ZWQ1NzY4ODQ2NzcwOGIxYWU0MDE4NWFmMjA3MmQ4MjBmYjgwNTJjZDA5YmVmNDM0YmQwYzQ3NTNjMDBlMzQ4MTdhOTk1N2Y5YTkxNzlhZmJhNTZjZGNmMzE3MzM3NzQyMjNkMGY4MTYwNmFiZDRlNjRmYmFiMGYwZmM5MWI5ODUxYzM3OWJjN2E3NmY0YjM0ZmM2N2E0MDYyYjkwOTlmZTVkOWJmMGZlYzBmOGFiMTkyOGQ3YjU2NzAyMDRkM2U4OTkyMGFjNTYxZTBlMjllMWFjMTZjODNiYWRlYWUzOGM5MzljNmMzNjg3ZmE5Y2U3ZWJmNzEyNmUzNGI5MjE2YmU4MjdkZTQzMmFhNjQ2NWUzNzQwOTBjNDk0ZDFmMDY4ZTNkZTQwMWM3OTc5ZTc5ODc5ZmQ0NzViMTVmNjc3ZDhjNThlNTE4NmM0MzBjNTA4YjI0NGU4NDhhZDJiMGI2MmJiNmZlMDFjMTQzNmFjMjAwMzYyMjU4OWM0ZjNlNmU0YmRkYTYyZTZmN2ZmY2JiMzZmMTY5YThiNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lbZH6x43aII5BfST9OLgNLhpJUD3v1Cihpe4HP00WAlmuEZDDX5bjCKvNxPkx8hDK3Bus-nv3dqEeg5kvPCgUg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102659_32_2453_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.293Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZ1MUtRMVhhMkY3WlpUcXJtTEFZV01RWjZzSmh3QS8vajlkU2FhYzFjTkdQWk1TUkkrZ1Y3ZzdlUEpRaWlnVDhnOU16QUh3LzArazRoSmgvTHIxR01BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTlfMzJfMjQ1M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTEyNDU1NmI0YTdkN2EwOTMyMzJhODA3YzYzYWE5MzBiMWM3MDE1MDYzMDI0ZGJlZTQ4MGYzOTZhNGI1ZGE0NjY2YTY5OGRhODk2Mjk5M2Y1ZjhmZGFmNjM5MDYyNTJlNzg0ZmQ4NDVhNjU1ZWZlZDlkYzY1MmVkNjgwZGE1NzUzMDg5MDBmZjkzMzVjZmRlODY2YTQ4YzY0ODQ5ZTg2MjM5ZWViMWYyYzZhNmUwMTdkNWMzNTUzNjIyOTY2MTVjMmRlNjA5NzZkZjE1NGIwNjRmNzM1N2UxMTQzZjE0N2Y0MjVkOThmOGM3YjNlNTk0M2FjZjljMzIzZmJkYzhhMWFlOTNmZTcwZTBjNjhhMDFlM2I0Njk5YTJmYWUxNmRmMTJmMjY4MjEzOWU3OTJmNWJkNTU2NGIzMTJhMDc2OGE4ZmRlZWVkN2ZhNTQ5YjRjYzgyZWM3ZjE5NjhmYjhmM2RmOTE2MDUxMzA5MmRiZWUxOGI5MzUzZTViYWZlYWRiOWFmMmZiMjdlNTQzMmU3MGI1Y2U1NTFlODg2YjYxMDljMTZlN2NiNGU2N2RmMjYyODE5MzUyZjY0YjdhMzA5NDhhMDBkYjY2MGRjOTdkZGNmMTkzNTNlMTBhN2FlMzdhODdjMDE0OTJjOTkzMWJiN2Q0MDUxY2I1NDBiMGYzOWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lUoqWMGhfupwp9E-59d2JWY6DIkD-huWV4ofuggBANcP2hQMzY8KLD0k3MbQArtojYmQJWdVfdnDvqwSpT15DQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102659_32_2453_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.297Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im5Xa1VkckhYd3VXWUF3Ymh2OFJzT0FLWlVaY01jeXdwOFpwdm1HdWs2OWNzNlZiTVpYZXdrbERVeGZkWVJLWEFPZmh1bTRla3ZBU1VDOE05N3FpVDJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTlfMzJfMjQ1M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2NlZDI5MjU2YmNlZWEzZDI0NzE1NWY2OWQ3N2Q2ZGRkYzY3YmI3ZTQwZTQxMDNjMjc5YmM3YTFlOTkwMmU5NDkwYzhjZGVmZDg3Y2U5Y2Y2MDg0ZWQzNWU5M2RlMWRhNzRlYjBiMzM5NmMzZGU3OGY0ZTAzZTNkNDA1YjM2YjQ0MjQ2N2RlODFiZjE1NTY3ZGEwODBkYzkyNGY1NzUwNTM5MDljZjU4ODRmNmYyNDEzOTQzZTNmMjU0MGIyMThiMmExNDgwOTJhYWRlOTQyNzUyMzVmZWJlYTkwYzhiNGYxZDM0MGU4MDhlODQ0YTdiN2NjMTliMmU4NGFiOGFlZTkxNTI1ZjQzMjE1Y2QzMmJkZTIxMWFiOGMzNWM2NDU4MzZiZDE0MWEyZmQ1MDJlYWJiZmVmOTA5NzA4ZjVhYzg2OWI0MTlmYzczOTYwNzYyMDdlNjBiM2JmOWMxZDEwYTI1NWRhNWUxMTkwNTNmNTc4YTJiODFmMzk5YjdjNzMxODJmMWU5ZWRhNTVlMGI1ZDA5OTg1MmI1N2MyZTQ0OWIyZmFiZWI4YmZhMzU5NjYwYjM2ODdkMzI1MzEwN2RjYWVlY2JiZGUzYjg2ZTI0NTYxNDBlYzFlNjMxYjYyNjNhZDkyN2M3YzA0YmM0MWNkMzg4MjU2YTEzNWEwMGM1NGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-T_aY5HYgafOxWpQeGIcaqIh9spvPd6Qn6UDXQv10ix4aJr3eX30ERCXGLXEnKTvSEh6pwMPd8E_RC0I7EwZvw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102659_32_2453_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.303Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im92b0FHQ3ZFb1FicmYyejJyRFMrTHkzZkRNbHNDdisxZGpCZE1zUEZQZkNQdTRDS3kyQXpydVVPZFVNNm9rN3RFcnBWaXM3OStzK0lQdklGMGxQbEl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTlfMzJfMjQ1M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjFkMzY3ZTQxZDZmNDJiZjczY2EyMjFmNzhlNjhhNTBiMGU4ZDcwNTI5ZWIzMTUxYjIzODFiMjUyOTRjMTg3ZGM0NzY4N2VjYjEwZjVlZTc2NGVhNzc1ODg5MjFlMjBmNWRmYjQ5NGQwZjcyOWFkMWNlNTc1Y2FkNzQzMjhmNmI4NGM1MjQ3ZjdkOGY2OGQ0ODRiMDdkNTU0ODFlZGIxMjI2Y2Y5OTAyMTRkZWUyNTYyMGFkMjU4YTE5NmQyNTgwYTM4MGYxYzA2OWExNWIxMTNlOWVjNjUyZmY5ZTc1ZDdkMjFiNjQxM2NiNjhlNWYxZGNhZTM2MmVlMjNhZDI5NTJhOTUxMjdjYjI2MzRiY2ExZDk3NGUwMTZiNWE1YTVkODRjNmZhZDlhNjI0ZjFiODRkNmIyNjM1Mzg3MTk0MzdkNzU0M2RjYzliMjNhZTVhOTkzZjIzNmI5M2M2OWQyMjY5MDRhODRhMjYxOTA1ZjUxZmFkODAyZjcyOGZlZjNlNGViNGFhZjVlYTJjZDc5YTBhZTNkYWUxNmZiM2ViMWExODU0OGYxNmRmMTlmNzQ3YmVjYzJkYjFmNTFhOWE5ZDQyZmIyYzdhZDNmYjVjMzRiNmUyYTY5MzY0ZTVmMzQxY2ZmYmEzMTFjNThiMDE3ZjA2NDBjMmZiMTliNjhkNWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rDUJXTzJWdYahG3qCNZv5e1Gy0OSs4GA1oLrt-wu76TMjQiECuiJVORIGp7V3O7QQJrZODxvE8sUGnXj7MipcQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102659_32_2453_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.306Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9rRkQ5OTB3b2J6alFCNy91aUhtN0ZKODBBdGNiUTJjVFlZQW5vZHB3RHkyTGdLalZab3FLUFM0TkMrU3ArOVdDTjd2MlJObmNEVmV2S0duYmVQVGxnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUxNl8xMDMwMzNfOTZfMjRiNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmFjMDI1OWJiZjU1YWZlZmMyNzVlZDQyMTNmODNkOGQzNDBlYTY4ZDEzMGRkYTZiZTg5YThkNTVlNjM0ZTM2MDliNTQ1M2RhYmYwZmRmN2U5MTZkZGQxMGU5OWM5OTY3ZWE4NDlhMDRjYmFlNzZiYzU1MTU4OTI1NDc5NmM3MjQxYzU3NjY5YTQxNTUzNzc1YjcxZjNkOTA4MmRiYmYwNGI0Y2I3NmFkMDZlNGMyMTkxNTc2ZjQ1NjE3Nzc3ZDU1NTc4NmQ1MTkwYTM2NjY5ZTFjNTUwNzYyNTcyMWM0NGFkMzRjOGE4OTZmMTE1ZjJlMjIzNDYwYmE4MTQ1M2FkOTRlOTE2Zjc0Njg5NWNiYWQwNDA0MzU3MGRhNjY0OTliMzlhMzA2NGQyZGIyODZjMDk1ZTM3NWI1MjhlYzAxYmViM2RhY2E2NDExYWMzMjJjMzVlZDQ0N2YxMWY4Y2YwZGM3MmQzZTUyNzI0ZTIxNWQ3OGI3OTlkZDIxNTQxZTBlZDhlZGM1ZWQyYWZiYjc3NTYxZWZjODJjMmVhYzc2Yzc0N2IwOTRhNjg2Y2I4MGFmYTczYzgwOGE5NGU3NjEyY2U2Y2ZhNDE3NDIyNTA2MTE4MzcwNDQyMTNiZTc4MGZjY2ExNGZiNjAyOTA1NWU2OTdiNDM2ODU0YTY2ZTYwMjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2fs462aTc54tQpHA_OwdQG8BWnlhbgM31pR0GoBNMXxCWhcieZAkNDY-anR05SLil7G0M9A1Su3TUQcjnua08g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230516_103033_96_24b4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.308Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJ1SGNFSTh6NWdFN1dCWVk3VUZjZGxwN21JKzlVOWxLQ0cvUVJFSjJNSUNSR1NsZ2dVcEsyZndXb3NpejRxdStSQUkzdFBDYVpqRDM1Y2ZsL2J4aWZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUxNl8xMDMwMzNfOTZfMjRiNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjI1YTA3ZDljZGU4YzcwMTdjYTgxOTNiZTQ3MDg3MjU2OGYzZTI1YWRhNjk2ZDgwMDdhMWM5NDEyY2NjZmNmMjgyM2IyNjkwODg0NzVkZmNhOTM3YjQ0NThhOTIzNDVjNmMxY2JjMGI0YTg3MTlmMGQxMGE0NDM2MjZiODg0YzNjZTQxMmYwZTc2NjM1NDY5NDRhMDU4YTQ5NWNkOTIyZDM2MDY1NDg2M2ZjZTM3OTFiMzc3YzhiZWViMmExNjhmNWNjYjIzOTVmOTM2NGIwMDhmODRmNWMwZGIwODI5ZGUzMzJiNzJhMDFlOThiNzIyYTllMWM0NTNiOTJjN2U1YTFkYzFhN2I3OWNhNzBhY2NmNjdhN2QxODhmOTRkZjI1ZDI2Njc2N2FmOTBmOGZlMWE0ODMxNzQ0NDIzMmE2NGQzN2EyYmJkNmFhYjJiNTgzYWU4OTFkYzA1NDdjNzEzZDY1NmE4MmZlYzY4NDMxZWEzZWZmMTA4OGY3OTA5ZTRmNmQ3YWMwZjQ0MTM2Mjk2YWI2M2ZjYWFkNDM5NDg1MTMwYWRjYTdkNTMwZGMxYmQxZWUzZmExZjRkZjkwZDNhY2Y0N2FhOWQ3ZmVhNTg2MDI1ODgzNWVlMmRmMzZkOWRiYWE4YjE1ZjllYzBiNTJhMTNmYjZkNWIzNWI3MDQxOWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LLw4TkQn8ZJLDypQVh6Rfek6zzVpp-BWsFhTONHvkYfIWumGjlVnTSm6ZUwOXwvNqP5nRhQNvNaYIheaClyX8g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230516_103033_96_24b4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.311Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IldIVEh0QXU4dGUwTUNPcExrU3MxM2o1Qlhhc1NGL0dBK2xXUk4zakVVWFB6aXN1Y2tYUjJ5VFZrVGlaY1R0QjhvNE1KUXE0VHNZSlI2TVVtcm5YZ2R3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUxNl8xMDMwMzNfOTZfMjRiNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mzg2MWNkNTdiMDEzMDdlODI4MDNjMjQxZTEwYTcyYTAzYTZiYzhlYzA4ZTIwZDg4NjY0MjFiNTJiYTgxMzRlNDdjNzE1YzZhMzlmMTU4YzNhZWM4ZDY5MmE4YjdmZDFiNDk2ZTUyMGE3Y2QwMjcwYjQzZmE0ZjY3MDRmY2QxNmQ2YzNhODg2NTEyZGI2M2I1NDRlY2VkZjI1YzZhNjZiNjJmODU1YmRkMTAwNWM5NDAwOTQwZmRiZTQ4Y2YyY2EzMGM4NTQ0YTVkZGZkMDVjNzI4MTRhOGU1NTZmNjI2ZDg4MTRhYWQxZDEyNjI0Y2IwYTE0YzM2NmUxYzgwNzgyZjQyNDdkZTE5YmU0Zjk2NWExZTgyZmQ2ZTg1MDJkY2ZmMWNkZGRhMGJiNGJlMjAxMGU1YjgwYWI4NDhiZmNlM2UxMTc2ZmE4NDBkNmM0MTY5ZTFmNmEyYjc3MTQ1YmI1NjUyM2U2YWZiZmUyN2ZlYmQ0YTc5OTQ5MjIwMjA5MmIzYmRlNGJmZDc1M2NhY2ZhNGU1OGJmODI1YjQwZjU4MzFlNDRlY2FkNzU2ZGEwMjYzYzU4OGEzNWY2NDg5NzZhYTIyNTM3MWQwZTkwYzE1NjRlMjQ0NWE0MWZmMDk0NzJiMzUzZTZkMWM2ZmNhYzVmOGIzMjhlMDFhYWUwZmJjMjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Sc1EytPVcL229M4g6aEScyd07szgtsAEEpN1BJiWl_R_2PiDZ1ycsjpMYea4XCsUi-Lr50fJzZf8bNqL_oWYJw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230516_103033_96_24b4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.314Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFsa2s3aTZKeHFVdWZoNzUzeVh3WjBDNHU4Z3ByamEzL0VHSlNzSXFDVzZLdEFaVzlLa1FtVGtZRlpzaGdnRXlUNWMrdlowV2k5SDR1N1VrOWdsRkxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUxNl8xMDMwMzNfOTZfMjRiNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDJjNjMzZjU5MTQ3YThlMzIzODMzNGVlMGIwODE3ZjZmOWJjY2MzZWYxMTRjZDA4ZjkxZmNiMTlmNGUxMDUwNjEwZGQ4MjQ3ODA5MmE4YzRhNGQ2MWI5MDM0ZDBkN2QxNGZjYmEwNDNhMWQ5ZmIzNjE0OTgyZTYxOWNmN2U1NTY5ZTc2NTBkMDQ5MDIwZjkyZjQ2N2ViNDM3MDQxZjg2Zjg0MDk0ZDg3NmZjZjEzYzdkYjc0Nzg4M2ZmNDA5MWU4YzdmNTcxOGQ5Y2IyNTI1M2ZmZTI4YzM1Mzg5MDljZDMyMjY5NTNkNjU4ZTgxZjczMTRjN2VhNDJmMDI4ZTE4OTAxN2Q3NmZhNWZmNjk0Mzc5ZGZkZDA5NzczNWU3YzFjYzRhNWZhMDJhZDhhYWE5NTMzMDgzNWM1ZDZjODUyZmYyZGM2ODZkOWIxN2U5MGY0NzI0MzljMWI2NGUyOGI4ZmQ0ZGJhOTMzN2RmN2YyMTA3OWVhZWM1NGE4NjljNzkyOTBlMmZmYmRkMDdkZDcxNWYyNWU1ODdmZjAzNmJkOTBkZDRmZGJlZTY4ZDU3ZDM5ZDY4MDViOTZmODYxMDczMmQ2MTA3Zjg5ZWIyOGQzMjVkMGY1MTdmMGNiNGIzZWMwZTc1NmUyNTlmOWZhZGE3MDE4MTE5MmJhMDZhYmU5NzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EZhgCj9ftayrHTmR6resDxf24EEJYobMqi3LK4SuGFuo_1Kv2FkBBxdy646lAfIRmipLWQ_QuNmgL2OzEaN2Pw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230516_103033_96_24b4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.322Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IisxRkRwQkR6WnRRWWU1RC96aVhnekxkVTVjVHNwUkwvcDJkdGlVdm5zcVNDcXVlOWw3cWEweG9zYWhkRjNqRkJvMUxZOUt0R3cwM2lVM0tXVVM1ejVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDBfMDZfMjQ1OF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjY1ZTQ3ZWRiNjdjNjcxNzQxNzVlZjIwZjQyZjcwOTRmN2ZjOTZkNWM1N2E0MTI2MDk1MjhjMjM2NjBjNTVhNDBkNDI0YWZjNzg0Y2U4MmI0NTE0YmFjZTQxYzU1NTU5ZGY4ZjMzNDQyY2Q1ZGI4YThjMDhkNDhiMTQ1MDM3MzhlZGZjY2ZiYTVjYjZlMWFjNjEwZDFkNGI5NDBhNGIzZDYzNjgyM2JkZjYxODBlYWY3ZDViMjQ5YWE0ZjY4YTkzNDdhNTVkMGJjYWE1NTU0YjM0MzUxNzY5YjM1ZjA5YTRjNTc0NTRiOTg5ZTZlOWQ4MTFmZWMxZmQ2OGFmNzZmZmYzMDgwMTU3YThkMjYzODY2NmVmZDkxYmRlZmI5ZjZlNGUxYjVkMzMzNDNiMjdkM2RiZDQ0NjBjZWE3MjIxN2VmMGFhMjk1ZGZlZDY5Mzk0NzZhODUxMmIwMTU5M2ZiY2E5YjEwY2Q2ZTA3NTQ5ODhlMjJhMDhhNDU4NmQ1OTQ4NzdmZjgwMjJlYTc1NDUzNDhmMTQxZjNiNjA3MjE0ZDIyZmE3NzgwNDZkZGYzOWVmOTk2YmM1MzY4OTkyMWRmYjg1MWUwZDE1MDJiN2RhYmZhZDkwNDQ4MGRkNDE3NTdiMjEyOTRhMjljYjJlMzczNDFjNzZmMzJmZGYzODM3MjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8SEGXhDqR4Me0kwXkHajPgqWMbQsf-YoyE23DcqqsOqfS31bi2AU4veswJIVICMRdy-12VpIaIqWdVpOjJocKg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103640_06_2458_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.326Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InhaVlJteldIcy9NK2NhcWlIT3Exek5mWFNYS2JsMDZVMkhENHdHdjFqSFJsNHNTckdYMlNDa3JTWEt5dWVUbmlPb1NtRExKbHRtWWpUUUs4VGdNK2JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDBfMDZfMjQ1OF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWZlY2JjODU5NzdjZTMxZTIwYWU4NmZiZmI3OWY4Njc1YTlmMDA3ODQxMTU1ZmUyNmU0ZTdlNjNkMmYxMzcyYmZkYjI0Y2FmNmY1ODU5NTc0YTYyMDQ0OTg5MGVhODI0ZTU3NmQzNTQzYmRmMmNiMzJhZjc2NzRiZTFmYWViODVkOGVlOTJjZjJhNzg1MDhjYTBmMGZhZjRlODNkZjBjYzRlMDQ5Nzc0MDI1YmJkZWZlZjQyNTFkOWUzOWEyNzAwZjFlZjFjNDliNjM3ZDdiOTZiYjk0Y2FlMzE2MjU3NmIwNzA0NmJkODRhMjNjMGNjYzZhZTUyZjFiZDU1Mzg3MTJlMzZhZDQ2N2MxZTY4MmNiNzVjYzIxNzQ1NTA1ODU4MzhiNDJhNmUyOTRhZjcxMDdiYzM5MTUyZDVhMDhmNWNkMjY1NmI4ZTU5ODY0NDA0ZTAwYjkzZTlhNmQ2MTk1MTRmZTEzNDgxNTNlZGU5MDAzZGJlZDA5Yzk2YzI4Y2RjZDlhNmY1MWQ4NzJiNjY0NmE2ZTNjN2I4Y2FjMThhMWVkMWQ5MDQ4OWNjZmMwYjdmMjY4MDQ2MjlkZWUyMDA1Y2MwMzNiZGJlYzgxZDcxNTdjZDExYzhmOGU0NDFjODA4NDdiNDcyYTE2ZjJlZWY3YzFmNzMwZjhlZDgzMTU2ZWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.78xukEuz3ODWKJq9o81hyqJkDTlwczruQtesWoI2_4jLPzNjgfTOYzZnOIpSc2cvebjyGgtXSfy5J3UqJLq5Rg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103640_06_2458_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.329Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im04U1ltZXZjQ1VwcDJkZWZ2NEdwREhFVTFXUTdxTDRINWd4cUU4WWVzN0lLUW1PNE1tUkllbC9DU2U1eSs2c2FKUkRBdkMwVmh6OGhCUUpZd3NQdmx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDBfMDZfMjQ1OF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDk5OGZhZmUyNTI5YjE0ZDc3OWY3NGQ0NGM2ODU5M2I4MmRlMTQ5Y2QzZWIwOWVjMGZlOTg0ZmM0NDE1ZGU5ZmEzNGY2NDJlZmIzZTBjNDlmYjc4NTZkNDE1MzcxOTU4YzgyNzRlYmYyMWRlYjg0MTBiMGE2MjBiMWRlYTVmMTBkMzlmYWQ0MWMxYzY0N2VmZGJkYTIyZjA5NzJjNDg2NzVkYTQ4OTg0NTU4M2FkOTFmMmU5ZTFmNjViYTAzMGM2ZWZmMWYxODVmYzIzYWRjYzA5MDg1MjQ4ZjQyNDYxMTI0M2E5Njg2YTAzZDQxYjI2MDQ0NThlZWZhOTJkOTc2YjJmNGU3MjU5OWJmODcwYjY3ZGQ1NjI2ZDMzZTA2NmI4ZDA2ZGQxMDRiZTk4MjU2MDhjMWU5MjBiZDc3NGFmZmRmY2FjZTBhZmU0ZjIzMWUzOGZlZTViY2YwMTIzMTE0MGEwZmQ2OWY1ODQzYmY1NjYwOGIyZWFmY2U2Yzg5NWUyNjllNDkzMzA0MjM5NTY0ZmFjM2ViYmY4MTQ1YWIxMjY1ZDEwZTQ0ZDU2NjE5ZjE4MTkyNTdkMDQwODgwYTg2NWJiZjYzZDBlNTgzNWY0ZTJhY2MwOGVhNGEzMTc0ZmY2MzkyMTYwNDE5NzllMjNjNmEzOWNhMjMyYjAzNjVlOTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6Xz7NFqnwW7ou34hkQGNi4udJDvr56D0NxIeMYLo19rDI34p3Q7kWrJ2R9q7Dvs7CXXR1pSnaeig7nk0nhSGBQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103640_06_2458_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.332Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjhDYmpRbHVRa1g0MVM4WjdJVHg3YXFnc0ZZR0E0S0ZrV2NqMVJvZUdsR01Qc3k1ZjRrc0tsa1ZCYjRJM21tSVJTWkNqb3VaUGd2UUVjSGhBQlk0SzJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTExNV8xMDM2NDBfMDZfMjQ1OF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njk4N2MyOGI2ODQ1MjY1NDE1MzViMDZjNDRmMjdjN2Q0OTMzZTNmMDNiZjY1YTk3ZGVkZTIwZDc3NGI0Y2QzOTdjODE4YmJkZWZjZmU0ZDY0YTU2ZWUwMjcxNjc5NDRmZDZiZmEwMWU4NTc4YzQ2N2JjMGEzZDY1YjFiNGRjMDZkMGEyZTEzMzM1MDU4NGM1NmQyZTRhZjdlYTE0ZGYxZDAyNjAwNTA5ZWE2YWZkYjJiNzgzOWVhMTBkNzQ1ZTJlNzQ5ZjA2NWFkNjMwZjVjZGE1YjA3ZTA4YzNmOWQ1MTgxNjg4NjQyMDI3OWQ4MWI1NzhkMjYyYzZlMWNhOTdjNWVlMzhjY2FkNDE5ODFmZGJmNjhlOTUwZGUyODI0Nzk2MDhlMWJkNmU4MGFkMjE2ZmVmZjU3OWQ0ZjhhMGUwNjc0ZWRmNDRiYTU1YjRlZWU5NTdmMzFiMTdkODZjODkxNjE1NzE2ZGMxNGY1Yjk2ZjdmNjhiOGY1YTgyZTk4NDQyOGMwMTgxM2VhMzQ1OGU4MTVmMjNjOTJhMzJlNTIzNDFiYzYzMDlhZjlkYWJiMTM0NTQ3OGQ2OWNmMjcxOGMwOWFhYjg5YjY0YTVlY2U3NDIyMzcwYTc3MTBhNTAwZjc0YTgwYTM4ZjQxYmZkODRkYTAwY2Q1N2ZkMDhiZTc3YmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9GxyYK6JJpj-9lSv76mX1H-RfOSbb8EpuH_JfMFUd6jj8vgcE0Ja7qBIDZNE0gPmFNBX5cG3GPLXDzupMNCY6w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211115_103640_06_2458_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.335Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImV0dDVXMy8zV2drVm1QWGp4U2p1WmZRV0VhRmtCMTVnUm5BenpTQUlVZitQb1locmZ4U0E2SmhxaGZPT2tRTXlmblRQNVlDd0lRdVYxVzVneHdpd0FBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTNfMzJfMjQwYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDJhZDk3NTYxOWY2YjczZTg2N2MwYWVlYzg5Y2E0NDdiMTJhZDI5YzRkMGI3ZGQ1ZWRkMDU4OWQ2ODllZGUzZjQxN2EwZTM4YjRhMjk5YjQ0NDJmYzAyYzJlZWY3MThhZWY1ODU4NzQxYWVmYzRjNzkzZTUzMjYyYWEwNDIyZTVhNjQxZTEzNjM2MmRkOWE1MGNkYTYzNTE1NTJlZDkzNTRiN2EwNTU1ZTUwZWQ1NTQ0ODBkMmIxYjg4NTFiOGZjNjg1MTNkYmIyNTE1OGY5NTI0ZjljNWFjNzEzMmJkNWNhMTk5MDliMTkyZmI1YzJiNzIxMTE2MGFhMWQ5NDZhZDdlYjNmNjczZTZhYWIxZDEwNzdiNjY5MTU3YTg4NWVkN2U2MzA3NTMzYThkNzYwYjNiZWZiYzQ1MjhjZTFkMmY0ZTNjYzlhYzI3ZDI3ZjY4ZGYzZGI5ZTUwODZiYTg1YWM2ODdkODFjN2IwOTE2NGNhYzVjMWVhNDg1NmY3NzAzYjhlMGY0MDY5MjE1NjBmYTliMmQ3ZTg5NWZlNGNmNzVmM2JhZTc3ZDcwN2UzZGRiZjUxYjczNDQ0YmE5MDVkYmJlYjZiZTRmNzI1ZjdhYTY5OWFlOTkxZTg0OGQwMTUyODU1MjQwODRhOTRlZWU5NWU0MmQwYWMxN2VkNzI0YWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SxOqDxZTwK3zn-hn-IpMA5O2BmQdonQy1D42r4Rdlbj5X27RSpEZeLcsQRT_iJBie450ArbaEbtYGvEW2p2d7w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113113_32_240a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.340Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNFYkpVM0xJMW51QkRUYnVVMU9qckN1ZFJKejJDUTlvSS8xcmk5OTcyQm5UZ0JKT0VMaThTMUNnMjlIMVFmRnY0UFp3c0F1ZGh2bG5Wby9vZGF2MDB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTNfMzJfMjQwYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjNhNDY2ZTM3ZmNmZjFhYTQ4MDY4ZDRmMGViNTVkYmQ0MmM4OWFiMThiM2UwOTg5ZGRjZDE2NTRiMjg0ZDg0ZWJmMTQwN2EzNDE1OTJkOTZkYjEwMjBlZWU2MjFlNDY1ODkxNjFhMjc0ZjMxODAzMjcyYmI3NGYzMTk5OTJmYzAwZTJmOTBiYzg5N2M2MTIzNzU1ZGU1N2U4YjZiN2U4MjEyNmE2MThhNjQzNTEwOTVmYzdjMDhjNDY3NTE5MGQ2NDc1MjQ2ZmE5YmNlMDU1MzQzY2RhYzBhMzQ1YzIzNTBlM2E0MWU3OGZiNzE3YjY1NWVmODQyOGNiOWI2Mzc3NmExZWQzNzQ5MDZlNDZmYWU3NGEyMjQwM2U2MWU0OWZiZjM0YjgxNDJkNzQ5NjQ1MGZkNDFiYTg5OTFhZTMyYTM1ZjFjN2YzOGJiZDUwZTNmYzk5MjMwNWM5NDI2NmU0YjIxOGQ3YjI2ODYwZDZmNjQyYzM4NjE0YjRlZWExYTZkMDg1MWUyYjMzYThlMzE2YmFjNjQ5MDJiMDUzYWRkODIzMzE3N2RkYmUxYzIwNDA0YzMzNjA3YzMzMGNiYTgyZmUzMGY3NzIyMTJhZjViYmMyZTNhMmJmMDE1MTczMDcxZTE0Nzg1MzU0N2ZlOTMzNTcyOGZiODVlZTNjMDdmOGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.j397AR9iM-K-lZ-Xa-uexsPWl3XRyvds2MiYhgF_06Q-lP69M6jIZJar2EmjPJ0JwPjYdS1xTgiM6sHaz_U0nA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113113_32_240a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.343Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpDcEtZZXpQSVZhYWNOVDZsanYrU3RrRlVITE1acEg1aE9rQWZuZG1ueTJ3QXdaejFQUS8xZTdQM1dKTC9rNk5UeVpFYW10MHh1KzJhcWhDWkF3ZUZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTNfMzJfMjQwYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmYyZjBmYmM1ZDY1MmE3Zjc5ZjliNjhmODE2YzhiMmNmMzNkNjc4YjI3MzMwMjJlOWM1MDQ0NjgwYTVkMzg1MzgwMTk3NGIxMzM5ZDU4MjgzZTk4N2UzMWRlMjMxMWY4MjU3NDhkZGU1NWQ5ZmNjMGI2MzlkZTgyYTUzNmE0OTdlNjc1MGVmYmY0ODBkMTg0NGI4MjBkZmFmZmZkYzU1ZjliOGJlNDQ0MjU0MTYwMGYxNjBmOTQyMGMwNTIyZWVjZDAzZTE3MWRiNDRiOGIwYWFmNTJkNmUyNGZlZWMyYzZiYWMzMTQ3ZDNjZTNlOWMxZDQ3YmZkNjg3MTZhOWI4YzIyNDZkNjhiMjM1MDE4ZTQwYWEyNTgyNTE2MmNmM2M1MzA0MzI4YmM2MWQ5MzBlNDAyOGFmOWNhOGQ5YzcyNDk0MTQ5MThkYmU5OTRhODIyMzlmZDFkNjgzMjdkZmFmOGQyNTU1NTM4MmMzOGZkZTM3ZjRhNWJiMWE1MDc2YmIwYjVmMDM4OThjMDIxODkzMjcwNGQzNGExNTRkYWY3YzlkNzk4MTIzMmIwMmEwMzZhMGZlMzk1MzE0OGI3MWM0Yzk2Y2M0NDJlNDc3MGI4Yzk5NjNkNjhmZjcwNWE2YTExNjA2ZjFkM2M4YWU2YzRhMTA4ZTAzMDdiNWEwM2UxOThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pzn0sd5wvrmRFyxyaG2cwU6cGZgHmdqPe3lYk9tpExBBQW5jfI_KOk7nMJaUYCnmLCA2QexY98u_q5_uAR1mTQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113113_32_240a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.346Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImFQN2NUR0ZXczlVUEwyWW1vVzQrZ0RqV1FzT2JRYVRSeUhxczVheGJnTWNVQlE3R0ZJRi9hVU01ZmhZN0Z1ZlZ6OGIzUGZXOGxyZDB1ZUE3MDcwRGt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyMl8xMTMxMTNfMzJfMjQwYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTVmNGExNWMxZmY2NDRjOTRmMmFkZDYyNjA3Njk4YzA5ODI5MjhiNjEzMjc2Y2E0ZDRlZWMzMzI5MjJmZjViYzBkMmE2NjRlZmI1MmFjNmQ2MDVhYjJmZDE0OGRlNjYwNTBlOGFkZDJjMGI2ZTUzMzZkOWY1Y2RkNzNlYzY0ZTc4ZmZlMDA3ODI1NzM4NjVhYWU5YTFiMWMyYjY5NWFlNjA5NjM5NGI0MmI1ZmNkZWIyY2MzMjA2N2MyZjBhYmRmNjM0YzM4NGRlMmE1NDA3NDhkMGNmNzk4NWFlZDVhODM0YjU0OWE0M2JlZjdhMzdmYzAyN2UyOTBhYjIzMGUwZGJjMWJhYzVlNWM2YzE0Y2IwMjQzNjljODQ4M2VkMDljMjQxNWZjNDQ1N2IxNjY1NTgyNGVmZWEyNTFlNzFmZTEyYzcyMmMzY2I0M2Q3YmVjYWU3N2I5ZGJkMzZlZTk0OTZiZDkwNTYxODM5Y2VlNzk0MmNkODJkNTI5YTU1NGQ0MzljYmNkNDAyZWIxODAzODc4MjViY2M3YjFkMjg5ZGZmY2E3Mjg0MTkxZTA5MDY0MTk5MzE3Zjg0ZDY2YzEwMDI2NjUzMjg0MjllMTVhZTZlNGZiNzM5NjlkMDc5YTFlNGE1MDM3MGVhMTBjNTVmMDgyNjIxOGI1ZDIxOWY1MzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ke0jSWtJNFooz1NGC4JJ387F0OD-bz4Rt91hMs8VQ9XwBm4M183ScaYSJkTUyNWPaD0gpKHc1ixnDaknxrxTYQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210122_113113_32_240a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.349Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkNIRmo3L1Q4S2c0N056K3ZHWC9uZnJGbnJjUUtlaXNPSWFyTU1CMVhBM1VqQ1hSd2t5dDBxaFRmd0ZtaHF2UmI0anFWUkJnQ1RxRjNQZm56R3lPZWh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMV8xMDMwMDZfODdfMjQzOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTIxODkyZGUzOTJhZGM1ZmEyNzFjNWJiYzljNWJkMzhiZDBkYWVhNTZjNGU0MDY4MGVmZjkxMzUwNDg4ODEyNzVkOWM2M2I0ZDhlYzQ5Y2VjN2Y0MGZiM2ZhMjA2OTJhM2FhZGNkYTE5Mjk5ZmIwY2JlZGJiYTY3ZWFlMzkxYjQzNjZlZjQ5OTkxYzE0M2E5NGQ3NmZhOWNiZmRmZmJlMjAxMGJkNjM3MjQ2ZTFkMDFhNGE4NWIyZGE5MDY5ZDlmOWYzOGYxODg2OGViYjVhMGJlMTZjNWM2ZmRlYTEwMGVhMjkwOTYzNzhhZjBjMjg4YzdmZjgzZjMzYjc0MmM1NzQ1M2UyNmQxMTM1MzFmMjBkNDE1YTE1NzgwNzVkMzY0MGJjNWYxNjhjZTJhZTk4MGYyN2Y5ZmQzYzE3Y2RkYzYxYzhkODcxMDc3OTA5ZDM5Y2UzZTM4OTU1NzNjNWU1ZGFhZDJhMjcwYTc3MTc3OGQ4MTY5ZDAzMTM0OTE3ZjAzMGVhMmE0OTAwZGMyNTc0YTJhZjdhY2JiMzdkNjFiOTdlYzgxMjlkNjk1ZTNmM2NlMGVlOGYxMWM3MzZmNmU5ZTRhOTBmMDJlNjg2ZGU1ZDdjMjE4ZTNiNTkwNzE2YWY3MDRlYjk5YWZkNTE0ZTc2MTQ2MTg3Njk0YTk2MzMwZTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.s7LyAvbz0-EO4u96EBoosWdI3SbL7Klze_50-bOEsMRZrz9j9hy6PJpN1ifBKGZHjl4YYn-RJyGISnPDROQIGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221001_103006_87_2439_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.354Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik5nbEw5dGlaZnVlbjY5N1Q1MmRXZDdwV0ZKQVR0VGpMbUtQUVNEbTRBY05QeDlXYjAxdit4ekROM0NUQ2VXVTVaM0xwU0dWTGM2MlJUbnVRM0Y3dnpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMV8xMDMwMDZfODdfMjQzOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2RjNDQ1NTZmMjI3ZGY1YzVlMzQxNTkwMDViMTYzNTRlY2Y2NTYwZjlkOWE3M2YwZmNhYzc4YTE4ZjQxOWIxNTU2MjE2OGMwYWM0ZjJkYjNmMmYxZjgxNjM4NmE1N2RmNDNkMmMxMDU1NmQzMDk0MjI3OGVjMTYyODI0ODdmMDMyZmM3NTNiNWY0ZTFlYjE5N2VkN2M0ZDViZTdmZDkxYzcyZjFhZjVmZTE4NDJiZjRhMDQ0YmY4YWVmNWQ3M2Q0YzYwZjhiMTU2YjUxYmQ2MWIxZWE3MjVlNjZkNWJjMTI0YzU2M2Q1OGIwMmUwZGViMmQwMWRkMWQyZDA0NGYyNTRjODUzOTg2MjYwMzdkZTg2YzQxMGRiODYxZDY2MmMzODk2NDk4Zjg5MTlhMmRkNDZmMTVlZjhlNzk0Nzc3OWYzZDVlZmE1OGNmZGE0MjRjNDBlNDUwNzk5ODU1YmE3ODU0YTMxMzFmOTFmMDliY2E5OGQyMjUxNDJlYjQzNzYwNTg3YmQwMzllZDMxMzMzYmJhODE0ZDZlNDMxZGFmNjI1MjQ4NDYyMTEzM2YzZTlmZTU5YzVkYWQ5NmViZTYwYzUwMWZiYWEyNTk1NmMzNWEwYzA3NzFmZWZmNjNmMTMwMzlmMjFjZGRiNTE2YjI5M2QxOThiYmYwNTI5NWZkYWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UdiDsX3k2XbfcQW9bwG-hwL0KvwyvzmTE5W9PCljsAs4oL1O3bJyR0tRDF4gzezIsMewY9KS3pX6AriSfBXk4g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221001_103006_87_2439_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.357Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik9LNi9LaXE3TmxvT1pOZm1uWGUyblhkS3ZnQ0FIaTkwb2oxL24rbDFZM1FoZGVNSW50VnR1N3hZWWhCdzdlQjkzL2RVWnJDYkk4QnBVZVhQQ1dxQXB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMV8xMDMwMDZfODdfMjQzOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDMwZjZhYjUzZDU2NmJmYTNmNTkwNmRhYmVlZjVmNjNiNGJlMDA4OTkzNjI2ZjVkYjliZDk2OTkxNGNiZjcwMDZjMzU3MzA0NTI5ZmVhMDViODc5NmQwZGY5ZWI2NDU5ZTZjZTMyMWQwODdlNThmZGVjNWI1MGVhMGVkMThlMTQ4MjlmOGIxYzhiNGQ1NTkyNjFlNGY4NTQwM2E0OTIxZTYwN2ZhOTYxNGE3ZjhjNzdjMzI4NmRlMmE2MmE1YmMyYWMwZjZkNjc1MWUzYjRhNmIzMzc0ZGRkNWU0YmMzMWYyMTE3YjA4MjEzMmJkMjA2YTI0NTFhMmQzODQyNzc5ZjQxODA0ZTJjMzkyZWNiOTQ0YWNmZDk0ZTIwMjZlZjA5Yjg2ZTczZWU0NDk5NWMzYjliNjhiMTE3OTEwMjY4NGM4Yjc1NDA4NTVmMTczMjI0NjBmZmUyNmExNWE4MmE4MmI4N2E5NWU0NmI3ZDYyMzBlODFiMDVlZjMzMGUyODE5ZDMzMmMwOTNkYTVlYmRiMzk1ZmE3YzU3OTMxYzA2MWVhYWFjMWFiNjlkNzQzMWQxMmNlYTUxYWZmMzI3MDkwZTYyYjA1NTI5OGEyNzdiZDg4Yjc2MWY0NTIyMzE5ODc4NGExMDQzZjJkMmIyOGVjNWUzYzUwZTVlMzUyMmQ3ZTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jaUvhUXSUMvYZgG6Ns7fCsVFQ7vDstfdSkUaxK7LJbdkzrN-eZ6uNFYhKG8faf5K94hVRT_rXDyvV1RKNhdvYw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221001_103006_87_2439_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.360Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjcrbjZDU1JURW96SVo3Z2FsdWNvWjhJczNFdVVQbVppVmxJR3hRR3dHNVJ1cUpoM0ZUV3M3K3k3MFp2bFUweFRya3IxaGIvL01zZzZuaUd4ck4yMWpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTAwMV8xMDMwMDZfODdfMjQzOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODFkMmI4NTQ4NGE1ZWE0NjhjMjcwZDM2YTdmOGE4ZDBiMDdjNDE1YWM3MjlmNWY2YzNhYjZlZDU4NDAzOWE1NjMxMTQ5ZWRjYWI0OWU0M2IwMmFiZjE2MTQ2ZDE1Yjg0NGNjMTUxYTdhZGEwMGZhZjgxZmY3MGEwYzQ0OTVmNjBmYThkNmY5NjEwMDBjNDBkZmM5MmVhOGI5M2E3ZTAzYzgxNTY5MWZkNWFiNWZmZWE0NmJhN2NlMjAzZmEzMDlhNWY1YzZiYTJhYjNiNmI0MDdiYWQxYjIyNDEyNGMyMjA5MjI5MGRkMTBhM2ExNmNlOTQ0OTE1ZDg4MTU3MzVlMGM3NTI2MmQxM2QzZTlhYjkwOWI3OTdlNWM0YWQyNzVmOTc3NDU3NmJmNzU1OGVjNjI5YzcyNzUxNjRiMDViYjQ3Nzk2Yjk5NjU1ZmFkNWZhNDQ4MzcxM2RkNWI5MzE0YTViZDhjZGRiNjgzZTk1MmQ1MmM5ZDNiYmY3YTI1YThjYzJkY2ZlMjAwOGQ2ZDdkZWM2YzRlOTEyOWJhODU5MzY3NmJhNzE5MzZhMzhjMTUxYTlkNDk3ZTA0OGNmZTcwZGI2ZjlmZTQ2M2NiOTA1MWRlMzc1YjM2OGQ4NDdlNGZkNGNmMDA4NmQzMzI2OGYxNWM0YmIzMzBkMmU3ZTVmMDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mikZKLVtuMQhWTM1xHGtNWdN5iA8qkAjSw0YIkNxGR1rIDOjZKRg0ZA9gy7gj8nz-FP7ZnOZI20yj0omz_3FYg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221001_103006_87_2439_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.364Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRXMTQ0L29ySzM4bG5sTWhaSitqTzF6M3FHei9BT2tWdzRoa0huZEl6aUZ4aVROdldDQ3IwSEJmYi91b1RVMUU1SXhMMkpDUE1JbEg2eGs4OFBUY05BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTAyMTJfMDNfMjQ3NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2NjOTU0NzI0MDdmYjI3YTI3M2YwOTUyMjUyMzU2NjRjNmQ5YmYzMTMwMDdlNmYzNjAzN2U1ZTdjOWZhNmMzY2ZlMTNjMDQ3YjRiMDY4MDY0YTU2MWUxNDkyN2VkNGY0MTAzMTIwNzNmY2NkNzhkNGNiMDk1ZjIyMDNlZmFiNTgwN2ZjZWI4MTI3Yzg0NDEzODVhNDcxMjIyNWZkMTEzMTk5MDMxMTQ1MDgzMGJmOTM3ZDFjZjQyMTAzZjg5NzQ4OWY4OGI0Y2Q1ODE4ODc5MDA0NTY0NmMwYjc3MTVhMjRkYTJjNmJkZWE4NGUyY2E5MWNhZmUyMzcxYjZhOTkzZTk0Y2E4Nzg5OTkwNDhkMDMyNTkwMTUwMjFjMmU1ZTRjNzRlZjQ1ZWViYTAxZWMzNTExNjAxZDk4MjBiM2Y3YzJmMWQ3ZDNhY2IwMzM0NjU3MDZhOWJkOTE5ZTg5YjYyMmZlOTc5ODBhZTgzMTViZWZkMjlkZmI3MWNhOGNhZjNiY2Q3ZjYwYjEwYjFkN2MzOGIzZGFhYzE3OTMyZGFmOGZiOWJkNGZlN2QxOGU1OTUwMjkyMjYxNTcxZTZiZTI1MWUzYTIzNWQzNzM3NTk0ODgwYzNjMzg3MWVjNTdkMzIzZTY5MTkzNWQ1NTAxMjgzNWNmMDI0NTczM2ViMDNlMDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qVmzmUb6lp-jsxI1I3XymPMgB641SMcWKg1oncogVuXpbgnNXKjnRvQmP18oIx2SLox7bPfudILM9K64Sxptbw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110212_03_2475_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.367Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVOM1k4Vzc0RGtjWnZUNjc3UGI4Z0ljelZBcm5HRFpMSFJ4dlRQS3ByekNZZ29RbjFJSHpPNmpRUGVPQU9MbzhjWUJiM3E4RWtkWnIyUG1NSXFwVTNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTAyMTJfMDNfMjQ3NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDllNzQ2OTk3NTVjNWI1MTRhYmMyNTlmMDZmODkwMjRmNjlmN2MyMGE2ZmY2YmU1ZTk0ZmQwNGE4ZDc5NTE1YzQ4OTMyNGE1ZDA3NmJkNWVmMDNmYzBhZWM4NWZmODYxNzIzMjYwMGQ3NjFmZmNiODI1YTRmN2U1NzMxNDIzYmQwNmY2OTVkMTE4MzI1Y2JmOGEwMjNlZDQ2NjM0YjRjYWU1NmI2NTMwOTc2OTM5MTg4MGFmZGIzMDg3NTBhMmM1MzhkNTdlOWE1OTg4MWE4Y2ZjMTI5MTY2Y2E0OWUzMTdhMDEzMmNkNWI0ODE3MjZmOWQ2YmYwNGU0Yzk4NGU4MmM2NDZlNThkNWU3MjFjMDRmODJlZjdhNjAzNjlmYmMwOTQ3MmI3OTZkMTExZmVjODg2NjBhMmY4MzU2Y2RhMTY3NDJiMjNmNTFhMWQxODExYzVjOTlmMjMwYWZiM2RjYTczYjg3NjcxODY1M2YwMmUyNTVhYmE0MmQ5Mzg5NmRhNjQwM2VmNzU4OGI1YjViODA2Zjc2ZGIzYjM2NGI5ZjRmNmEwN2VkMDM5NzllYjg3MmZhMDY0MmQzOWZkMDBhZjVhY2Y4ZDBlZDQ0MDhjMmRlMzJjMTQ1MzM3ZjhmY2IwMDE3YmU0MDUwYzQxNDhlYmFlYWY5YTE0ZWEwMTNhY2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.A2hHS8e3mPpPsRGUtYygfBn5_Yc0RC6kELozWYNep8RoUIacOLkhsGD6X2f7hc93Qr1CXw6-kwIzPH_rMQ6ZEg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110212_03_2475_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.370Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlpMblRiakRqeklXMEtsOXRiRjdWMGlqdFBLc2k1ZWlZdDN6NnhZdDloNVdObjAvVlpKSEtwZzU3T3NNVWxJZG9iV21hSmFTTTVsVEppczZHaXBkY1NnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTAyMTJfMDNfMjQ3NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWM5ZjQ5N2E4M2M4MTI1Y2RlNzM0MDk3ZjBiOWYyMzQ1ZGNhNDU1ZjgzNTQ2ZjNhOGRiMDJjZmNhNWFhZDU2NDU4YzJkNTU2NzIyNWNlN2VmMTJhZGEwMmI5OWJmODcyYWQwMDVjYzg0NDRhNzg3YjllNDZhZGQ5YjE2ZDVkZDhjMzk4MjM4YTc0ZTk0ZmIwN2JiOTE0ZTE0ZjgxNzRjZGFhZTE2MDQzYmM4OTMzYWM1NTYwNzJjYmFjYzU1YzE4ZTQ0ZDhlZTM1N2ZlMTc1NGJiNDFlYjI1Y2QzMTI5NWM0YzRjMTJhOGNmNzFhMGQxMzQ0YjA2ZDUwYjBjNzM2YzYzODQyMWQ5MDk0ZTdlMGU0NDhmNjA4OWI4NGYyMjg2NTM5NTZhOTk4MTI5ZWRiZTBkNjdjYTc5MTJkNmE1MjBiYjJiMmEzM2M0MDNhZDllNjcxMTk5OGIwOTY0YmY0YTk5MWQzZWVmMDhjNTEwYTY4NzBlMDQzZTc1MDQzOTc2MWQ4MWNjOWFhZWVjYWYzMzViYmMyNTlkODY0Yzk0MTg3NWExY2I3NmFkNTc1Mjk0ODE4YTUwNjA1YzBlNDU3NDUyYWM3MDk2NzVhMGVkZTViOWJlMjJkYTQxMjRmOWRkNjIyYzgzNTJhNmQ5NWNmMTA3MTE5YThhYzQwYTg2MDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.09zV78cm4zDPkizzYwUSEi4xhcPcZtMG-1OmQs4qaJXBNgVGNz9oP4w3QRgTwU0CLyFUPaaEzL9oaLQaXl8Kvw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110212_03_2475_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.373Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlRhV0cyTzZlaXhhWitzSzFmRkIwTitaQkM3Y2hiREdBcEdybXJkWHZmeUdSb0NBVWNxQ3UveU1TTTJJSTI3VkcraDAxTzZXdWpyYnFhMWtoOGN6bFF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyOF8xMTAyMTJfMDNfMjQ3NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2UwY2M4Yzc1Mjg5NzQ4OGFmYzRiN2RiNGEzMmY2MTc1Y2Q1YjM0ZDY1NTcxMzMzODFlODdmOWQwZmRkNTJmM2VhYmJhNDVjYjUwMDNiNTgwZjFiNTgyNjQwZTA5MzZlZjJkMWY3NDE2MDMxYTA4NGJiMDliOWIyOTY3MmE0MDdiMzgwOGE0NjNlOTQzYThkZjAxZDVmMGU0NzYzNGNlNzk3MzYzOWFkZWZmMDFjODgyYmUzODNlYzdjYjU1MmY4NjgwNjU4ZTUzMWIzNDA1MWYyY2JjMWNmMzk0NGVlZDlmZDljMTVlZTk0NWI3NWRmYTE5ZjAxMmYzZjVlMzViYzk5ZGU2NGUzMDExZDBjMTE3N2MxYzljMzliMjM4Mzc4NjAwMDAzMzQxNDEwNzljZTVmNTc2YTE2ZmM4N2EwYmVmNGE0ZWM2MGQ3Y2M1OGM0NzBmNWU2Y2MyNTI3ZjU3OGI1MDgzN2ZiY2UwMzAxNjA4YWUwZTMzNThmOThiMjE5OTJlZjliMDJhMjQyN2EzMzg5NWQzMjViNGFmN2RiNjkyYjk2OWE2NmYzMDRmYjc1MjI3MmJmMmExZjhkYTQ5YWYzYzEyN2JlNTRlOTQyNjIzODcxMzc1YzYyZDU5ZDE0ZjA2NjFiMDYxODdkYTZlNjllNzkwZTE3ZDU0OTcwNDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AN6hhKDQZ7kS357ds9ekQ4WI9zWALvWwQJTQhY114fqP2_nk6Xyyl27RWMn-qZ1Fobbp0TnHXn9CSw_Py-qBAg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221128_110212_03_2475_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.375Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imo0dTUwN1l1UHNrbE8xWHR6TFk5S2duMlRNa24yUjFueDJqditTRzhCQ3pnUjF3V2RaQzc2czFaekU2cXJEdEMvbWQwLy9VblFuVDVCMURUNi9mSElBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExNl8xMTAxMjNfMTlfMjQ3ZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWNhZDRmMDBkODU3NjI5Yzc4ZmI4ZjYzYmFhOWE5NTBmNGUwMjE1YzU2MmY5YmNjOGMzMzgzNjI2NmY4MTI0ZGZkZDdlY2ZlOWMzOGQyMjdmOGIyYjQzOGJhMWQyM2VhZTA5NTBhMWM4ZTU0M2I5N2RkNjFlZmZhYjQ3MTA5NDMwMTUyMGQ3NDY1ZmYzZTczMzhmNTg2ODNkMjljZmFiNjJjY2E0ZTgyNTdjMDQzZWU1MDJjZjBiOWNiZTBmMTUxNjgxMGMzZTE1Yzg3Njg2N2ZmMGQ2N2U0MzA5MzFmYWU0NDM4YjQ4NWJkNGJkOWVkNmEwMGZkY2Q2ZDMzNzM1ZDNiZGZkOTMxNDYwMmIwZWY0MDA5MGYyNzBhZDExODc3ODMzZmFjNTgyZjA5YWJiMmJkMDY1MmU1ZTgzYTU2YTlhMWEzMjgyY2IxY2NlNzYyOTc5YTVkNDliZWYyZDVjYTNhYzY4MTY1Y2IxZDc4ZjAxYTNiMDA1YTk4ZGQ2N2JjYzdhMGY2OTNhZGM0Zjc3Yjk1NDRhZWM1Y2RlNTFhYmRlM2ExZGQyYjlmOGNhNzJiMzAxMjg0MmE5ZWY0NThkYjg3NDdkMGE1NThjMWRhYWYyNzJjZDg0N2NhZjI2OGY1MGE1MGQwMTEzYThmYThmMDAzNWFlZWQxMTZjM2NlOThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.R4zXLSdGjWPR7tK7Wb-lfm00QZFt8e7wO3Lxevoxa6pdXhOf1Fez_xqOrEwmO74xqlH3NCajUsNeN0sWKBheyQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221116_110123_19_247d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.378Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxXRU5IVFBraHRZUGszR1BTL0QvWnJidjdMenpJVVJ5MGRkU1UzQ1U0MWlwYzNxWk03eHNiVWxyM3hRSUdiRG84Q3dNZHA5aVVaR3NEaFpxcVUxN1RnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExNl8xMTAxMjNfMTlfMjQ3ZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTdhOTU2YTFlZjIwNmUxNmZlYzMwZThlNTk0MTU4NDRhZTczNDlhYTk5MjI5MTNjZDQ4OTU3MWRkZGFhNmJlMjk0ZTA2YmYzOWQ5MmM4N2ViZTFlNzg3ODZkODMxYWVlNjQzNzk3MGEyMjAzZTJjMzlmMjA3YmQzNjQ5NjVlYzJiMmE5MDA3YTEyZDc3Yzc5MDY3Y2FkYmVjMjMxODU4OTk0MGU0MjlhNDY3NDQ5Mzk2NDk0N2YyYTQ4Y2IzNjFjMTE2MzYyN2RlZTcwNTE2ODQ5NzkzZGYyNjk3MjlmNDcwMDUyOWVkMTJiNjMyZjczYjVmOTJjN2U3YjQxNTJhMWQ2YmJlMGM0MGE5Y2UzYmZmYTk0MGY3NGE1OTQwNGVjNDI5YTA4NDZlYTRmNjNkNjc3MDY5NDkyYTQ4NDU5ZjRiMWY4ZjAxMTMzZTU3Y2ZmN2UyYjVlN2VhMDViNThmODRlMjJjNTQ1MjkxZTM1MjhjMGM2Y2ZmN2M4Yzc5ODU5N2E0MmE0MTZlODllYjdhYzMxOTVlYmRkNzY1OGQ5YTNjZDk3OGE3ZWU3M2MzOTFmOTRiNjdiNDg4MGY3MTM0MzE1MjFjMGMwMDY2ZTJlY2M5NmRhYTQzOGE3OTFlYzMxNjViODBmOGE4M2NhYmQxNzFlNTlhMDlmYTFjMDNhMWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GHHX9JMDaNU5lrl7Mf3BuHykk7cwLkVzX6FFLg8qlVDopJOi-OlNGCGQlS40jjuzU5rWfbHu5jreI2cy78g1AA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221116_110123_19_247d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.381Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Iks2bVkyYkhMeFBsdlByVXAxSm5wQTRWbWtSQUhUTjJwMGRoTkdXSWg5RDViNEcwS1l2b2Y3am5jcHR4dmYwOFI0bVR5Z2E4UUliYjNTUlVudHBEMDd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExNl8xMTAxMjNfMTlfMjQ3ZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2I5ZjBmMWRkOTlmNDM2MzE4YWZjYWEzYjRhOTZiOGQ0ZDkyMGVkZmNlZTY4ZTg5ZmZiZDNiZmRmNThkMjYxZmJjYzNiMDRkNTRhODBjMjZjOWI0MzNhOTNhZDgzZTJiZjViMzU4M2Y0NmYwYTk0ZjNiMTBjYzA3ODNiMzEzYjcwYTM2M2RiNmVhNmU5NDM4MDJiYmQ5Zjc3YTIyOWUyMzBjNmU4ZWJhMWQ1NjgyZDRlN2U3YzJhMTc1NzlhYWRmOTA0ZDcxZmNmNDViMjZkNTM2OTM3OGEzZWMwMzI2ZmU5NDUyNjJmMjA0MzBhMDFhMmNmOGQ1ODZmYzUyODg2YTdiOWQ5ODQ2MDIxOWI2MTc3YzA4OGE1MWM5Y2E4NWY1MWNmNmJmMjgwMzcyYzA0MmZhMjA2NWVlZjc1M2UyNzdiM2QxMjE4OGM3MDc0NWI3NTU1MWM1NzZkYWM1Y2RkMWJmM2FhZDJkMTgyMTdhMGM1MzhjOTNiZmE4ZGQ5NzljODEyZDllMzU3ZDQxMWU3MjAwN2JkYWIwNGUyZDFjNGIwMzYxNjcxYzI5MWRjODAwZTY0NzA4ZjI2NzViODljYzAxZWRlM2U2YTFmMzRiOTc0NjNjMDZjMGM5YjM3NmU5ZDU0ODZmZGQ2M2Y3MGNlNmU3ZmQ4MTM2ZmE1Yzk3MDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.VVItLiGIwI5y6-EJCTbBOettsqvQOMd4GsNm3SC9xJZyUKu95GvjTPIgwGtb2pXpsfmImmyC3j6iPo3wvn_afA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221116_110123_19_247d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.383Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlhhS0FhYkVoeXVkRXU2UFFiSUJ0OVlLdjZRTFVuY21ja0hWOCtDQTNodm5HRDdaVHZ3TytTWWJISEc5SFJyWUNEeVU5Vlhya0xXVmgyKy9kMU9qK3ZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTExNl8xMTAxMjNfMTlfMjQ3ZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjhiZGU1Mzg4OWQzZGI1MGQ5ODlmMGZmNTg3N2I3ODQ1MWU5ODMxYWMxODU2MmZjMjIyYzA1YzQ2OTJjNDQ2NDYxZThkNDhkMjNmZTYzZWUwMmNkNDgzMTViZDQ5NmM3ZTY3ZWMxZDU5Yjc2N2FiMjc5MWZmMGNmNWVjYWExOWZmMTlkZmY4ZjE0MmE4MzcxNzVkNWQwY2YwN2YyYWQ1ODE3ZDFlMGJiMDMwYjdjMGJlZWQ4ZTBlNDNhOGU4OTdkNzczMWUyYjJiMjEyNjE0NTc2NzEyM2VhZTE1ZDIzOTEwZjdlNjRhODZmZWM3NDA0YTE4M2YzYjUyNGRiODQ4ZmRjNGY5OGUwMDIxYjkzZGIxN2NlNThjOThiOTVjZWYwY2NhYTRkZjBlYjUxM2UwYmY3MDFlNmViY2Q1NTZkMGVkYzNmOGExMzUxZGZmMGUyMjQ5NWFiZmQzMWQ4YmFmMDQwZjQ0YWMwODc4MjM0NThmM2Q0N2MxN2VkYzRhZmVkMDgxNGEwZTNlM2UyNGQ1OTNiMTdkNmM1NDUxYzUxYzZjMTFhMWI5NjZhNzVjNGNkZGZhMjdkYTE4YWM2OGJiMTA5MzJhNjMwMzFlNTE1ZmQ2ZWE0MjNiYzBmNWRkNGYwMGU1ODkwMmZmMWVmZTUwM2UzZGY3ZDczOTBkMDE4ZWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.C6x16QOIMqX2iM_lrAJrkU6JOsBq-taiX5M4Yru2aSwyi4wjrmsVUSNQXdTpDRcn3IjjUJgrNChDBs85mZp0Jw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221116_110123_19_247d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.387Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ilk0Mis2blRCZHNaS2c4NHNtd0FSc0w3WVJMUWc3anVLZ0g3U1FPSzhLNFpuajE0bkVGWkJ2VEF4dGdoU0pTanJ6S2FNM3J4NlkvRE9wSm1vQlFYUVFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMV8xMTIzMTFfODhfMjI3YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzQ3OTIwYTg2ODA0ODg0OTBhNDlmZTYyNWJiMzk3NGQ1ZmE4ZTVhNDY4ZGE5ZmIyOTU5MDRmZDE5YjZmOTNjZWUzZDJmZDBlZDNiNzQ0OWEwMTM2ZTI2NWY1ZTRjNzE5NzIzMjcxYTlhODEzZTlkYzlmMDcyNDNiNzI3ZWJjMGQ2Y2JhNWM5ZmE4NDJhNzhhMWIyM2VjZjFjNWMzMTllMTM5YjJjZWY0M2EzYzUyZTcxYmU2MzI4Nzg0ZGNiMjg3YzFmZWY0ZjBlNjc4ZDBhODAwNWViYzFlODZkMDI4NjcxODQ2MTBhNDEwMTk3ZTUwZTc2MzJiMTRlNmZjMWZjZjMyZjVmZTYxNzQ5NzY0YTQ3ZDdmMjA4MzgwOTAwNGYyYzZkNjNmYTUwZTZhZTE4MjA5NDNkNTBjMjY2MjkwNWQxNWE5NGFjMjM1NTk2YWZhMjE5NmU2N2UwNGYzOTE3YzZkOGY0ZTZhZGVmYWRmY2VhOTVjOTAxZDhiM2JhMjIxZjExODZhMjFjZmJjMDcwMjJiMjIwMTdjOTZlY2Y2MmE2ODc4YmViMDRmNGRhZTViZGNmYzU3MzU5YzJjMzQ5MGUyMjhkN2ZkNmRkMmQ2MmVkYjI5YmJhNmNiNDY5Nzg0ZTViNDRlZTc0NjNmMzg5YmUzZDAwMmI4NDhjZTgyOGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.29T4z0HmLO0TYYGOoITjt_yMOxmK53chMYS8-hlZ5txcyYNMWmiXQy0c7fcshgHE6WRBaG6b6iLNopyaOcG_tw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211021_112311_88_227a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.390Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImJXY3U2ZFlJekFxa1FBS29KdzlzUXJCTzlWdkxINlpIVVhRYTFSb0FVTmNLa2dMb3Y2S2dYd3ZuUUNKTStsKy9pS21EcU96WTg3dmMrbWpFdGlMVEFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMV8xMTIzMTFfODhfMjI3YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQ1MjY5Mzg5MzljYmVjMWJmMGM0OGNiYTU1MDdhMGE5YjRjY2MzMWMwOWRmNDFhMjczN2QwOGVmNjBjMDFjYjdmNjM3MzIyMDAyMGI1NTFkMTczYzE5ZDMwMzNkOGFlYTlkODYwMzQxNmY2ZDlmMzg5M2VlMzFmNjcyZDFmNzgzYzY5MTc3YjNlZGRlYWE5MjViMmYxYTg3YTk5YzUzMjM0N2JhZTE2ZGI2ZjgzN2NhZjcwYjFhNDcxN2Q5MmUwMDk3NzgyNjVjYWY3YzIyYTVhZDYyZDExYTNiYWRkNDJlMTJkOTUxMWRlYTUzNWY2ZmZjNDgyMjc3MzQ5NzY2OGE3NTYzMTRjNjE4ZmI0OWVlYTIyODM4NGU2NTAxMThlYTIzN2MzNDhkNWQyMGRmODBhNmJkYzdlNjE1MWMzNjU2NzRiYmYyMWM3ZmZjNDc2ZjA3ZTRmNGQ0NWRhOGUzMGNhNDU2ZWUwNmMzNGVjZDY4MDIyYzhlODNkN2YwOTAyNGRjNmFhNjI3NTkyOWRiNzQzNTFiZWU5NmJmODA5Zjk3Y2M0NWI1NDc5YmU5NzdkODVmZGYwZDUyZWM5ODA3OTFjZjdhZmZmOTJkODIwMTYwMGQwMGUyYjIyNGNhMDY3YmQ1ZGRkY2I2YjU1N2ZhMjVhZDg5YWJlMDNjZWY2MDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1EvegWc99mW7nExqROcvVz_IrYcghjpsj8_zL6dh3ujG4_6LBSpeiPhrArDswRX7Zjwh65_RYRWy75YvcdoaXA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211021_112311_88_227a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.394Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9CNk5zNHE0U1JWa3djQ1hHZEFBeFg2R3VjT1NERWxWZll3VUdIUjduT2NkZS9kZE04QXQ5ckd1Rnl2LzZvcHFqSngwWlhtY1RJd21IYy9rUFA1cktnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMV8xMTIzMTFfODhfMjI3YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzUwYjEwMTRkZTY5M2NhYTYyZjkzM2IwNGIxOWRiMjhlZjEyNTUwMmIxNDY1NTc5Yzc4MThmYmM4ODUyYjdlMzkzMTRhMjNlNTQ1MTBkYmUwMDFmMWQxMjVjMDFiZTA5Nzc3MTFhZTY5ZmJkOTJmMGY0MDg4NDZiYWNhMjViZjk4MGFjOWM0YjkzMjZmZjdjNDA4MTkzMWQ2OGRjZTc1MDAxYmIyZmIyYTE4NWY1NTJiYWRkODM1NWE3ODkyYTAxODg2ZDk0ZjM4ZjE5MzEzMGYwNjdjYmQzMTAyYzU1MTdlZjdhOTM2YjU1Nzc5ZjFhN2EzMDJjYWZhNjU0ODY3NDMzNWVlODMyNzJkZDVhMmYwNTlmYTAxZTU5MTI4OGNlZGQ1ZTk3YzQ1MjA2N2Q3NzJmYzhhYTkwZjUzODM0MjJhYjZiNjA3OTIxMTc1MGJjMjQ0NjJiMjVlOGNkMjEzY2ZlMGY0ZWVjNmQ2YzE4OGYyMzNjMTBhYjIzYjNmNjkzZmI4ZmUwZDk2MmQzZjEzZDk0NGYyZDNhY2I0NzM5Njg0Nzk4MmZkYjRhYzhiOTE5ODVmNWZhZTMwOGQ0ODFkZmJmNDRkMTc3Yzk5NDU0MTY4MzA0MWU1MDIyNGM0ZDBmYzdjNzQ1YmM1YWQ0OWIzY2NlOWFmMjY5NWQ2NGI1YTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eKB6rSodR-bJubWxKL7Ha_Jg0exyKJ4yyOsgt70hTTfvxVFJg7humOyZYCziVZvwhNu0dceCicMieMEoWd3lXg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211021_112311_88_227a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.402Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhBRGZKMFl5djVQSzNMZ1p5cDFVNGRqOEZZdHhJRlIxcFZnNUVMMDRCYXZSNmpVMEdtVE9jUXVSTFRxMFVvejBYODY1aVZvVlJaelAwaHRnN2twM3FRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMTAyMV8xMTIzMTFfODhfMjI3YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTZiYWIzNWU2ZWMzZmZiNDIzOGY0ZWNhY2ZhODJmNTM0MTI1MzBmMDFhNzUyYzU5ZmM4MzUzOGQxNTZiMzNiNjUyZmYyZjRjMzFhYzQ3MWRjODNkZjhlNTdjNmYyYjNhYTA3NDRkOWJkMzM5ODJmODA4M2FiMGRiN2UzNzgzNWMwYzk2ZDBjYWFhYzNkNTk3MjliNWUwNzVmZDUzZGU4ZmExY2FhNWVlOWNlNGQyMWY2N2EwNzY2ZWQ4MjVjNDA4OTc1Y2YwYWU1YjkzZTkyOGM4MzMyMDgyM2MwOTI3ZWFjZjQwOTBhYTM3YmMxODI3YTkzNDYyNGMxY2M2ZDE3Yjk1NWUyMDM0ZTBkNDg3M2RkNTIwZWIxMGVmOWUyZmRlODE3NGY3M2JiZTZlMzQyNDQ3ZjE0MTNmZDcxYjJkODIzOWY1ZDliMWU4OTQ4MmRjY2Y3ZGVmOWNhOThkYTA5Mjc0OGRjYjI1YTk1N2Y0MjMyNGY1NjU1NzJhMmJlZDY3OGE5NzYzODc2Y2FlMWU0MjMxNDUwYzkyYmY1NmY1MmUzNDZkMjM0NTgxOTg1MjAzNjI4MmJlYjAxNDk1Nzk5ZmJjNzUwMjdhNjg5NzFjMTQ2YjU5YTE4NWNjOWJmNjA1YjQyOTk1YWY0MDlkN2NiMzRhN2M3YWI2MDMwM2FlMjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.D8sPN7dNK63s_9HXka8k1bQ6-b7CEEwsObom2Igq5vnCXyjs8vE4RpRIS9ybdVDiCgU4g8IO71xaxOAhdwJSOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20211021_112311_88_227a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.406Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IktIU2NwVUN5UFZZZHdidzhDbWFVVzBmVVVuNm9wcDdjWHBXcFIxdzd0b1pJSXFQU0dKdjdXVEtHb096VkI3anZTWEp4dEdnbVhrOEd1UkRkWUMyM2VRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDJfMThfMjQ5NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGU5NmQ1ZmJhZDZjYTU1NzY5NGFiZmY4NDE1MjA3Y2U4YTJkYWNiNzE0ZjVmY2UwZDc5MDkxNzQ5OTQ3ZGEwMDA5NDI0NzkxZWI1YmJhMDBhMTM0NGY3NTM5MDc2NGM5OGM3YWY4YjFmNDA2MjUxZGY5MWZlMjVmZWE2MzE1MDNhZGE5OGJiYzI1MmE3ZDNmZDIwOTA5ZWI1ZGY3ZGI1MzA3YmI0Nzg3NDRhNTkxYmY1ZGFhNWFiMzJkMzU0NTAyMmFlMmYxZDEwYjRhODlhMTNjOThkN2JiNDU0MDgxZDFiZmZmODNmMjZiZjQ1OWZiY2ViNmY1ZDhiZjczZDE3MWMzYWZlMmFmNzVlNThiNTc5Zjk5NWU5ODRmYWU5MjA2MjEyNmY4MzNlZjZlYjgxOTZlMWFmMTdiYmVhNDI3ZjI5YjI5MWNkMDY4MTlmZTdlYTJhODczN2Q3ZjBiNTVkMDE2YzYzNjI0MDFiOWQ1YzZjYTJmZTZiZjcyYmQ0ZWM3MTA2YzdkNmQ5NzkzM2FlMmUwYzBjNTFmMmQyMjdiN2VmODYxYTBiODU4MmZkYzgxNjQ3ODg3MzhmNWIwYjBkODI5NjI2MmMxMWI1YzQ3ZTAyYjM3MzZlOWEwNjA0OGFlMjUzMGFjODA2NzU0NDNmZGM5ODQxMzQ0YWRjZjdjOTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vLqm6ZNO7Kt-NGhqJ9VlEZ9kMQ7qHGfpJdxdf54TowqCFW9oZPvayplsEyx9F0jQebx_NbFOjxzU89aHezx5MA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105942_18_2495_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.409Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImJha1RqR2VTaE80Tk5peXpVbisvWjgxWUZIcGw3K2NaNkNqbmpqR2szN2MrVmkyTC9XSHoxQ09uL2xQZldFRTZObGt4aTAwSWo0UlljNnFoK1NFUGhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDJfMThfMjQ5NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTI5NWU2MTk2NzU3MDgyODVkZDE5MTc2NDk2ZDRjNzIxZDhmZDc1ODAxMjExMWQ1MDU3ZDg5NWNhMjdmMTE0MmQ5NWUwNTg0YmY5Y2Q1N2E1ZmJlYzA0YzNiNjJkYjFjZTg0OGNlMmY2MTBmYTRhZTJiMjEyOGM2YWQzNjg4MmI4OGIyOWYzYjBjMzFkZWFmZTgyY2FkNmQyYTAwMTJhYWUzMTZjMjA3MzA1MzQ1ODliZWMwYjVjZmNmNGYzMzg3NDIwYmZhZGNlZDZmZGU1ZDk3ZmJmODExNTA4NjRjODUyNTVkOTgyYjIyMjI3M2E3MGY4MmU4YWJiN2E2ZjhhMTAzMWJjZGZlNzQzNzAxNDcwMzk5MjA0NmYxYjMxYmU1MmMxMTMxMzFhOTcwM2JmOTViOTE3ZTNmZmNiYWNkMTA0MzM2NWU0YzIzMDI4YWQ2ZGFhNmY2YzBmNThmOGIwMWUyZGMzMjc1NmEwNWZkZWVmY2ViMzJmNWFkYjcyOTE3MzJlOTg1OWY0YjFlZWUzOTgyYWZkZWRiOWEwMGFiODVhMTQ3N2JiY2NkYmZlMzc3ZDJiYzIyY2VkMzZlMzY2Y2IyZTc4MTYxY2NlNDA1MzMxNzkzZTMzMTI0ZjY1NTBiOGFhZjZiYWI0YzQ1M2Q0ODg2YzU4MmM0ZTQzNmUxMWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZLvZiMwuCnxUfGfTjDtsVI_mRo99e7rIMGTrSnGr9TeTx0NPVkIjHagaFavPKFo8OfvSnxoHsgxjachYnw5HKQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105942_18_2495_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.412Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJ3b3U0NE1WazZEa241OWZXcWU3bENoMmQ0dHJhT1AzbC8xRlVGVlVpNy9Eckk4WUhERDB5WEFDTFNXZ2VSRFlNY1d2RzlkaGc4b2NNcTVFWkQ5bDdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDJfMThfMjQ5NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjQ1YWI4YzAyMGE0NWExZjY0MDRmYzIwYzkzNjMwM2ZjOTZkMWRhMWU4MjFjZWVkNDIwN2Q5M2ViNzQ0NTU0MzQxOThkYWRhYjBmODYyNDU0NWE0ODU3NWEyMTExYWQyNTAwYWRkMjRkNTU4NTFjYzE3NTUxODEwOTZhOTE4ZjE5ZWI3NDRmMjY4YmFmNjYyMmI4OTY5OGExYzhhZDliZjY2YmFkYTY4ZGVlNjM1M2IxYTRjYzhlNzI5NDRjMDM2NWMwM2JjMDc1ZWIxN2QyYTIxYTk0MTMxZjM3OGZmMGEwNGZmYWM4NTM0MzEyNDZkYWY0YjgyZjc5YjBkNWFmNDU5NzhiN2RmYTEzN2M5ZGU1MTFmNDY2MTk5ZjlkYWNlOWE2MjdjNjIzM2JkN2YxMWVmMDdjYTMyNjBhOTVhN2M4ZDljZTk2ZDk2MDE5MmY4NWIwNTFjMTY0MzcxZjA3Njk2MDI0ZTM2MzBiZmU2NzZiZDg4OTQwNDFlNWI4MzkzNjMxMzdhYTg0MjNiZTQ0MWZjZTE2NzAzNjYxMjU5ZjdjZmY3ODZhYWMzODQyYWMyZGI3YzZiNWFkNDdjOTM0OTAwZDk5NDM2NDE5MzYwNzE0OTA0YTdlOTQwMjY0MzFjNmIwMGJjNDkwOGY4M2MwZTRlMmZlOGFkNGQwODcxYTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nGtrRECdRKNeM1oUuDkmkKcVr1KLfkwYCpre0UpW8ioCsiZmck7Izq6OgWCO9gEM170SPcgGgwyzg5NZlekNCA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105942_18_2495_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.416Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImpBWnlrMGZCNE5tRjk1V01CZWpDZEx4dExCSlF4MW9lYVB3Uml2WmV1a3FyV2ZIajZjUlZVRWxidVY5aHB4eFJsUVZBc3ZKNy9mOGJOK0pTMWtSbHpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDU5NDJfMThfMjQ5NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTQ2NDUyMDVmYWYxZGE4ZTM3MjhmYzhjM2ZjZmQ5MzVhYWQxZGQ2ZWEwNDFkOGJlOGQyOTRmNjhkYjU4OGU2ODhhZWZiNjM0MDI1Yjc1MWRkY2IxYTg1NmQ3Yjk0ZWE2ODlhNWJkMTY1ZTM2M2I4OTZlMzA5ODI4MTg4Njg4MWIwNWY3MjFiZjUwNDk4NDkzYzNlZmRlNDlhNjdhY2Y4MTM0MzU4NTFhMGRiODg0NmY0MGI3ZDVlZWY5YTYxYzdlMDM1MmU4NTM1NDM3ZDdiMDQ1YTRlMDEwMDVmNmY0M2FjMWExNDBkZTEzOWFhMTI5OGEwNjEyZTVmN2YzZGU1ZGNmMzA5N2NkNDYxMDQ1MmJjNDU0M2FmYjZkMjQwY2ZmOTIwNTc0ZDcyZjRhZDhhNjU0MGMyN2U2MmY3OTU4YjFlM2IxMWMyNWIzMWMzYjMzMzZmODUyMDQ1OGQ2MmI5NjM2YmI4ZDk1ODRlN2VjNGIyYjM5ODgzN2QyMTI2YmU0MGQ5NDlmNGE4M2Q4ZmE2MTE3NjQwZjgzOGVkYWM4MTdlNGNhZWU1NmNkNzcwM2JjOWRmNGYyZTAxNzk4MWU2M2Y5ZjE5ODU5N2VmOTMzYzgyNjMwZjVjYTJiZTg4Y2YzZjhiODI1YjI1MjUzZWI5ZWJkMTQ2Mjk5MGU5YjY5ZWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WFluFZ1vA0uz58GJVAl0s8eESZ-FVpAxLNx_H_t2IOf-27fyDlDnYC0Op5Hb5DVxjAeQ35Tiky3Y4BkAMngbDQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_105942_18_2495_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.420Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InBialN2aHZqOEhVaVgvSVN3VjhWN0xEREVNbWZKK1BxOWhWTGUyMjZIZVlISVREbkFkelVnRG05VWNodk5YelJyWDRUTUsxem1hR3NucXVadmVWZHdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDMyNTJfNTJfMjQ1YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjU2YjA1YWZiMmE5MTA0MGVkNGY1ZDNlOTRlZmIxNDU3MGFmNDRlM2Q5ZDIxYjEyYmVkMmRlMmU3N2M2NDVjMTcxYzUwNzU2ZDM4MmQ0YmE2YmIwM2I3MjM5ZGM4YTc1ODUyZmExOWQ1ZWNmZWJhYmNiNGUyZDY4OTViNGU4ODUxMDA2MjAzZDY5MGVjMzM4OGQzZDk3NGEyYTk2ZDdiNWM4MWMwYjhkZGQ4OGRiMTJjYzk5NGU0Njc1M2Q0NzU2NjBiMWNmOTQzMjFhMWYxMTljOTRlYmZhMThjMDAyMjc2ZjQ2OTgzMzBlZTE2ODYyODg5MjdiNzJmNWEyOWZhMjQ3OTY2ZThmMzNkYWUxN2RhNWRhMTM0NTAwYzgxOTMyMDZlNjZmNjA3MTZhNzA2NTE1MjVkNzhiMWE3ZmRkNGIyNjBmZDliMGVkOWExMjFiM2JjOGNmNDQ2NTQ3Y2JmY2M0ODA5YTAxODFlZThkMTZiODViYzljMGEzMjgwYjg5YWJkMGVhYjgzMmFkOGUzNjk5NWQyZTUwZmM4NWZiMzk1MzVhOTJhZTMzYjNhODhiOTdmZmZiMGY0Y2MxY2Y1ZDY0ZmFiMGY0MTQ3NTU2ZmMzNGM3MWU3Y2FlYmRiNDk4ZjBhODJkMTM1MjkwMTVlYTU0MGE0MGM3OWFjMTNhMTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Vf32kMJUO8YanFn6_FseSO_9gWijE65cuGycQPwnAdoOKq5PPU6KLgb3gu8J9MRWnLMr3rbcUU3HifFgbBgyvw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103252_52_245a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.423Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlJpUU1nK0lEOHFhUmRRM1F5Q0JtTW1oNVh1Rkl2Zmg3M1ZDZ1BhQ3MzOS93QnFRNUNMNWx1SlZiRUNsRnV2NkxHSUpBbzBFaStobFV5bWMzU08yc0xRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDMyNTJfNTJfMjQ1YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWM0MWIyNDVkZDllZTE1MmQyOGVmMGM4YzM5NTE0ZjM0OGRkNjQwY2MwYTI5OTgyY2I5M2M5MDExMTY5NzNmZjU5N2QyN2I4YjYxZjA1YTk1MWI2NWUzYmZkNzQ2NjVmZTU5OGVjZmFmZWU3NjlmNzk3MTY1OThjYWEyNzg0ZjcyNjY3NmI1ZjMwNzNmZmU0ZWRmZjc3ZDgwNDI4YmE5ZjNhYTJkZGU4NjgwYjIwYTMxZTQyOWI5MDE4YjAyYTU4MTI3ZWY4ZGM1MDg1YWQxNjdiOTU5MzE0ZWI2ZjU4ZTE4Zjk4NTM3YzZmZTAxMTgzYmRjYTcwZTc5MTg2YzhkMmQ3MzZjNGRjYzRkYzhjOGY5ZTY5NmNkYjU5Mjk3ZjRmNGU4YzBjZjdiMGVjMGNmOWQ5ODFiYWJkZmRhYWZjYmQwNDhkZTJjNjJiMGI1NDQ5NWE3MzVmNzIzNTRmMmY5YmM4NTczMDdhYzVlNzk4NWYwNGFhZGI5ZDI0MTBiZTg2ZDViYmY4MDM1MDA5ZTdiZmEyZGZkZTJhODE4NjZiMzVlZDVmNTViN2Y5NzgyNTY4MmZhNTRlZjYwMTc2NTdmNGI5YTQxNzViMmRmYTFjY2Y5OGMxMThmMWNkZjE3NjYxNzMxNDVmMmUyZDZiN2YwYTA3Njk2MDQ2NzE5ODgzYzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HFHO1VuQL0vU6uNecOCEQ_tavIuxLz-aj9tC6iXpzmjT3dCsnw3ZXWRbY2MbQZ06kxXI1b9ErcYCT8ZehkiprA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103252_52_245a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.426Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IllqbWhaUFVzYVd1UWlTZmhzOXVKV21qdlhzMW5JOTkrdWRva3pjbmV6Sk5hbkpVNTVTblR4bjRxR2V5VmY3dkNTMjdCaVhBbG1hVEV2N2U2NFpyd1B3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDMyNTJfNTJfMjQ1YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTM5MjE3Njg2ZTNmN2ZmYWYxOGIwOTM0NjVhODM2YzRmOGEyYThhODYzYzg3MmFkMmQ5OTZmMzg0MDI4NTM2ZmRkMGVkNjljNWRkOTFhOWJiYjVjNThiOWQ4MGViNjk1MmZiZDEzNzJkMjY5MjRiNjRhZjYzYzEyMzZjNzgzM2QzOWE1MzY4ODg3NzY2NjgwOWMxNTYxN2UzNzBjOTM1MjAwM2I4YjEyZjU1MDJlYWEzNDdmMjBlMDhhMWNkMDQzZDJmZjU5Y2Q1MGE3ZDA2MzI5MjRlMmM0NDlmYWVlYzM1MmY4MTZjMThkNTI1ZTU5ZDE0MDFjOTYwMmE2ODhlZTAwNmI2YWU3ODMyYTNjMzc0ZjBmZDY5M2JiODMyNjU1Njc4NTc4ODYxNjRlN2UwYjc2ZTE1ZjRkNDJhMjNlNGNlMmI0MjQ3ZTZhMmM1MWYyYjc2YjRkZjQ5ZTAzNTkwNTNiOTY5NGY3MWZjY2E0ZTMyNDY4YjYwYWNhOTU2MGZhYzVjNDgwMTI3NjM1N2QzMzFjZjU5M2ZlY2E4ZmU1OTEyMGUzYzk5ZjY1NDMzMzY4NGVlNTE2NWQxY2U5ZGU2MWE5MWEzOWNhNmUwOGFhMjJhZjU0OTIyMjM3MDFmNDU3ZDlhNDRlMDU3MTJiZTE4YWIyNWRhOGIxM2M3OTJmZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NCxYl3cVzPIUN7-3QhTwrdQZhpjPMfwvxw1ABFYuDpf4TWCEJkzB7Dd9l-MWxqtYE8mp1Nh8GaE2RFenwBD3dw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103252_52_245a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.428Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpTTVBhV1NQd0Z6V0Fvc0hFQlArdzVyb05iaXFyeSs2d3dwaXRMVjVlclAyRU9JYmRnWjZISnlVYTR6OXA4YUVsM2R2MktreG55Q2ZHVUxnZWViQll3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDYyN18xMDMyNTJfNTJfMjQ1YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTJiM2ExOWM4OGNlZDU4YmRkYzk1MTllOGQyODA5MTBkZWRmNDNkOTEzNmNlODMwMTJmNzZjNzVmOTRlNjc2YmFkMTMxMGY4NjIwNjAxNDIwZjU0YTg5MzNhNmY5MDM5OWE4MjVmZDc0MzUwZTA3ZTNiNDQ4ZGI1MWUxYzk5MjFjNGE1YWI4M2I2N2IxNDViYjRiNmY0OWY2MWFhMDhlNjE5MjBmNzk4OWU3Mzg1YWYyNTU2YjAyYzk5NGY1MTBjN2Y3NjIyNTY3MTkwNzJhYjQzYmI3NmVlMzdmOGU1MmI3YTI2YTViOGU0NmMwOWI4ZjlmNzUxOTA2MjcyM2E3ZTMyOGE4ODk2OWQ4ZDNiNTYxMDc5NDQxMzJlYzg2N2MxNWUyYmRlNjgwODgzMDA2Y2EyZDQ2NzZmZjdhMmU2NzdiMTMyMjY4ZDg1NDQ3MGRmY2E1ZDhkMGEwNTUyY2EwZWViZjM5MzNiMjY1YzhiZDhiZjFiYmEwMzdlNGFiYzlkMzJhMWU4OGE4ODhlYTJiMWIyZTc1YTA5NTIxM2EzNTg3MWIxMDExZGQwYTAzOThiYTc1NDRkNDRlNTlhYTI4MDYzYzk5ODQxY2JkNDZjYzAyYzUwNzBmYjkyODQ2MTJmNmZiYWUwMmQyNGI3NDc4YWM3MWEwMDJjOGJlNDUxMTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0B4ZH-ncpQv7GRlJwFncWlR8XPOG79SeipBOt47Qlg3YxWW5CbQHZzztFb3h5RfivgxBYiqkJEmVh6mrJv5lOA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103252_52_245a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.431Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImsyUlM5bXFBL1NsZXF2Lzg0cUxGTGZVZm9ETnhjUXlSRk5qOElWcUNnbEV5U1BnWWpMUmlDWFRvNndUZzEwc09GZ0dCSUtLSjdSdHFzR0hDcmFzcnZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTExOV8xMDMzNTJfMjFfMjIzNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjI2MmY5YTMxNDVmOTJhZDkxOTc2MWFhMDU1YTU1ZmZjZjEwOTMzODIzNmYxZTYxMDI3YWRmZGRjODVhZTQxZDYxNTg2YTRmYWU1N2I5NTM4NTUzYTZkMDFlNTljYzU0YzEwZTA5YWY4MTA0NWE4MjlkNzE4MjU5YThhYzk1YjQ2MWY0ZjA3ZTY1YjBlNzMwMzZiMGVmZmU1NGUxN2MzZGVjNDQ1OWU0OWUzYmY1NDI3ODFiYTFkMjVmNmU4MDViNDUwZTM5NjVkN2FmZDVlMjgzZTFiNDIzOGI2NmNkM2VmMTE5Y2IyN2UzNDc1YjMyY2E3NzQwYjY2NDFlZWZjNzZjNDc4MTFjZWVkZjgzY2Y4YjZkNTUxNjAyOTZlMDMzMDVkMzQ4NTI4YmRkNmUyZDNlMTc1YWI1MTIxNjUwZTFhN2RhNTFkODM3MDFlYjUyNzM2YTIwZWM5NTJlYWM0NWRjZmQ1MzA4Mzg2MDAzMzIwMTE3NWMyNGE1OWRkZGZjNWUwMzczMzgzOTZmMTFjYTI5YzhiYzdhZjFiZDU5ZjlhYzE0MjIxZTEyZThhYjdmODRiNjFmMDVkNTIwM2YwMjc2Y2VlZDg1ZmY1ZjM4MDVhYmU1YmIzNDc5MmMwNTAwMmU0NGFhODIwZjk5MDE5YTA4OTYwZDFkNTlkOWZhNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1BZT-3SeGWXSt8Yrs79-sA_Itmhn2RN1AhY9tVYciN13rJo0Ik14ZWxBjuDGf1XJibx79_tyAWizgPasu2BDZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201119_103352_21_2235_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.434Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFDNXdzbVBidGtsa1FnNTNBVXNLVXhUOVdYOEtiZDZjZHVVN2xXMW80bHdvQzdzaDFzdGRIaVZwU2lad0ZkZDBsQm5XTVM3ZDE3RkJOT3hFRjBZNXFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTExOV8xMDMzNTJfMjFfMjIzNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDFiNjRkMTM4N2E2ODU3NjhlOWIzNTBmNTg1MTExZTM1ZjczZTIzNTFmMTJiYWNlNDFlMTJkZDY2ZjE5ZjJjYmE5ZjdkM2U1MjJhODVlMmEzYjM0MzQ1NzI1ZDk5MWJjY2FjZGM3NjNmNmYyYWUwY2NhNTM4YWVlMDgzZmRjYzRkZmE3N2ZjZmMwODM1OTYxYjE0YzA2ZWQxMmZjOGZkNDQ4ZDZmOTU1ZDFjM2FiODJmMDJjNDg3Y2RlNjA1ZDg4YjIyODU4ZDk4MmU4OGZmMGVmMmRhMTdhNGRlMjBhNjI1NGQzZDMxYTYzMzdlZGE0MjhiM2Q1MTY1MDQwNjY2YzgzZjhhOTJiZjE2NDQ5NDE5MjdlNzMxZTQxMTQ3YjdlN2VjYmJlODNhYjgwMjBlMTc2NGQxNDk2MTQzZjdlN2Y0Yjg2OTI5OTdjMWU3YjdmZWJkYzY3YTQ5NDQ5YWI0MjE0YmJkZGFiNTg1MDg2NjY0YThmOTEyMWNmMDIyY2RmMTM4MWY2OGI1ZTNjNTYxOGYxOTZkZmJlNTc4ZTUyYTAwMDM4OTllNjNmNzg5YjY5ZGM0OGI1YWIwNmVmOTdiZmEyZjQ4YjZhNmI5NTZhZWE0OTY1Yjk0ZDZiMWEyZDgwZmZjMmUwMWI4NDkzNzY1YjI0N2M5YTUxNjZkMTQ2ZmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._jY7UWBTneMYYXq7Bjv32-L-XGCtjFk7qAWRk-cpdskWKVBTJdXjUN5nus4gzbLuQGMpL-kRofJQsytaCgGKlw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201119_103352_21_2235_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.442Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJyaDBqdEN5c2dqMm0zbitabXI0MlFKZm1RWHlNTHVoZnZyanNoUC9VbFo3cXp1T2RqZ0FwOSs4MUMyZ2UwRlk3NUN6ckN5VGhOQ1ZQUjRkZHdIM2RBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTExOV8xMDMzNTJfMjFfMjIzNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWM2MDRiNDNjNzY2OGI1YzA3Mjk5NzM0NGRmMWU2YmQ5ZTliNmRjZmNhM2EzMjgzYzU1MmJkZWY2MDFkM2QxYjI5M2M4YzA1Y2VhZWQzNDViYjdhNWRmNmIwNDhlNmEzY2NjMzc2MDdhMjQ4ZmFkN2VkOTA4OGRmYzhlZmI2NTRhYTI0ZTViODQwY2MwNjIwZGRjM2U4OTc4NDBhYzlmMDMxYjBlMWNkZWExNTBmYzE4Y2ExNWY3YmNmY2IwNzlkNDRmYjBiYTlmYjgyZmJjMjZhOTk2MjhiYTY0NzAxNzA4ZWJjNzNhOGJmNTkwOWUwOTY4ZDVmNGUzOTc1NDQzMGFiYjg1YWVmNTIxNDZlMWY3NTM4ZGViOTM0MmZiN2JiMzBlZDcyNTY3MjNlNzhhODU4M2VhNzViZWYwYTc1ZDc3NjQzNDMwMDZkYThhMTczM2EzYjlmNzIwZjk5Y2Q1MGI2ZTIyNTFhNDI3ODUzZjEyMjU2Mjc2NDkzYzdmOWUxODMyM2M3YzA3ZWFmMWYyYzBjOGYyMzk2NzE3NDY4MDRiMmRhYjJmNGFiYjYyNDlhNmRhMjM4YWFlYmJmZDNlMjhhOWNiYmZiMjY3MDQwMWMxNzBlNjY1ODE2YTM4NzI5ZGVkNWJiYjA4NWNhMThkNzk1MmRhN2JkNTNjNjVkMGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.p2_KFL-xLIZ0XvjrtjKz_hgxbubsYzl6sdGa1kF6GMgGbSR5xYpBR9SyZvrOQvboGPKK5Oix7DjcyWRxFdRE3g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201119_103352_21_2235_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.445Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkMxQ3htMFRiVzhmaEx0bitaWHZLYWM3bEhGakF0cjZ2a0NtUURTdXFFODVRbHB5R1AwUFZqajJUcnhQcC9wWHhUZ0FHQU1QM2NXdlVNSFZuUGVOZVN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTExOV8xMDMzNTJfMjFfMjIzNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2VlMjIwMmEzMGM0ZTRmOGEzYjEwMGFiNDk4MGNlMjBlYTUzMDVhZWNiNzk5NjlmMWNiMmQ0MTE3YzA5ZjAzNDJmNmNmZTRkZTU3OTUzNjRmNzZhODY2MTgxNjAxMzZjY2JkOTc3ODM1ZGFkZTM5MDBjMThjYTkwOGY0NWYwMDNkMTNkYzBjZDUzNGU0ZWFhZjU5YWQ1MTE4MTc5YTk1YzRkYTdmODk1MDMzOTQyNmZjOTQ3MmJkMzRlYzU1NmI1MTJjNTMyMjgwMWE3YzY2MzQxMTI4ODg0ZjAyMDFiYTViMWJlYmI2M2U0NDI5NTc2ZDY4YTljYzdjZDlhZjY2ZmQwMDNkMjBmNTllZGJhN2NjODAzZGU2ODM3MGFiMDI4YWE1N2FmOTQwNmNkZjFkYWM2MjFlODA3NzEzYzMyY2U1YzNkY2FiZDM4ZGYxNDdmMTQxNjJlYzRjZDQwNmU4ODA5YmJlOGUyMDkxZGE1MDA5OGRhNGVhYjE1N2IxZjBkZTBjMzgyYWQyNjUzODY0MzBmZjk2NjJlNmZhOGNjZGY1ZGRkYjcwMDliYmE1NzhjNGIyMWQyNzkzNDg5M2JlMGViMjQ1NGRjN2JhNzQyNTY3NjIwOTUwNTI4NGZmMWE2NTUwNGZlY2Y3MGRkMzIwMDA4ZmQzN2ZhYzM5NDA4OWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4lX6k94k1685eV9Jqs0xKfBCB9mc9n4Ckpjn__In-H2VRQ8VMVwMvIZ-TKvx4jXGFp2XsHjCRP6LTzxUFMuXwA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201119_103352_21_2235_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.448Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBLeHI5ZnRSQnoxU2REcHdwQm5WTTdkL3JXZTVVZXRzZlp6RGIzVkRqMm1PL1VXZ0xkSWU3T2xnZUZlZEtkUU50NHNnVzlYRitNYVROWE5HYVZtajB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5NDhfMTlfMjQ3N19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTZhZWJmZTFkNmQyNTUzZjVjOTdhMjA5NmQyYTk3OWM5MTU4NjJkMGQ5NzFmZDBjZDNjYmQyNDM0Mjg1NTVhZWRjMDI3Y2E2MDZhZDE3NmJjNTNjYWYxOGM5MDk2MjYzNTcxMjQzYTU5ODljZGI5N2IwNTUyMDY1MDlmNzVjZGJiOWVmYWFiZDgwNGUxYjI2MjllYjg5ZGI3ZmEyNDFlOWJkMDI5OWI4ZjBhZjE5NzNhNzJhZjEyMTNhZjlmYWUyYzI2Y2IyYjIzNGU0YzdjZGE4YTNjYTU3ZThjY2MwMzFjYTM0ZDhjNTczMjM3OWUxNWU3MTg4MzcxMTQxZjUzNmE4ODNhYTk4YmQ2YWI1MjhjMDBiYjQzMzc5MDZjNWU3MTVmNTRhMmViOGQ3ZTNmNDMxMTU0MGQzOGNlNjZiNDdhZWNlZTUwYWI4OTAyZjZiNGM5NGE5M2YwOWYxNmYzZGNiYjg1YTQ2Mzg5Y2YwZGEwZDQwNzk1M2RjZTEyZWIwNjRhODIxMjcxNTFjYjY0MzNhZTUwN2ZiNjU1YmQ1MzU5OWQzMWM1MWNiZTdlM2FjZDg2ZGRmNDg2ZTczMjVmMjVlMmJjZTRhNDhkODA4MjUzZjk1NDNlMmI1YTdkYTk2OWZjZWMzMGE3OTJlYzU5ZThkZDZiZDJjMjNmOWQyZTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sxeFytxXE54mW0tiFNx67s6QtUvrT4mh1GOnmYILZ3c7eRw1lCqq1eo59kWZ1IEl16W0q10VKejSyilWzgTDeg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105948_19_2477_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.451Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkNPVWJScmZZZEVlY2d2L25jcUowR2RiU1JxWFR0c1gxMGpXalcyajNaV2FSMXJ2Rm9zU0YzL1FTaGhmU2Fxb3R1MEhDS2x4aGRXd2NHcmtFM3lHTE13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5NDhfMTlfMjQ3N18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjNjNTUwM2Y5OWRmOTMzN2ZlNDA5NzYzZmEwZWFiOWM0ZTRiOWNiOGVkMmVmNjY3MTk5N2UxMTYyNDJhZTc4NjNkYjQ3ODNjZjdiMjRhNzE1NWUxOGRmOTJiNjg2Mzk0ZWNiNDg4ZDVlYTIwZjE0Yzk1YmIzOGMxNTEzNGQxNjNmMWQ5OTg3ODBhOWIzOTFiOTAyMzQ2YzQwZTI2MTBkOGRhOGNjYTg4YzUwZGI3MDkwNjFmZmQ3NWIxZWRiZWM4YjkxNTVmNzg5NGI1YjAwMWEzNjk1YzgyY2MxOTRkNDQ3NTlmN2YzZTQ3ZmM4OTUwY2M0MTNmNGRhNDAwNjRhMTAzN2UwNTQ4YmE0NzNkNWU4ZTliYmYzZmQ5NWVmYTA2OWJkYjU3NzU3MDdiYjc0ZDRlNzQ0NjdiNmJjYTM0YmQzMDc5YTZmMTMxY2YzZGVjMjk1YjMzZjAyZGM4NmIyMDZhMDkzYTgxMTAzNTJlYmExMmYzNjI1OWI1YTAxNzlmZWZjNDk4YWMzYjJmZTE0OGUxNmQ0YWZjZDA5MjZjNWE0NDA5NDQwN2Q0YzVjMzA0MDVlN2IyMmZhNDFjZWYxODAyZWJhNjczOGY3YTVlOTE5ZmNmYWRlNjVlYWQ0MWQxMDI2M2RiNGMwZWYwMTliZmVjNzYzNjMxMmJjNjgzMWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1P2q9XQWsCHWrHGyKoQcNdCHESjpi5F01tgd_xklUc8oN_dV6y-DMbgU8EXhhOMoMrKtESyMZ_5qLWhTH-SzXw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105948_19_2477_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.457Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRBeXNsR2ZZb0hEZVVFTk5xdFFZM0FBVUZmNVd3REd0VUthcUJJc3k5a0FHek1EZXVybW5NS2RSQXM1anpvdVZUbXU0Q1B0eHFtejFMSjZYOWdXOHdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5NDhfMTlfMjQ3N18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY5ZmYyZjJiMmUyYzNmYzkxZDUzMWRhZDg2MmU0ZTAyMWVjMjg2NDZlZjg5YTFjMTQ0ZDcxZjM4NTY5ZDhkMDUyMTJlNTAyMjZhNjVhYjlmOGI2MTcyMzU3ZmE0MWYxMzgzZDMxNTJmY2JjMzhjZTdiOGMyMWFlZjI0MjAwMGY1YzBlMzk1NzkyNzU0MTEzOTU4YmVhM2YzZTU5OTVjZThmODBkNTRkNjE2OGY0ZGUwMWE5YmI2OTc5YjBhOGFlNzViZDM2YTM2OWQwODdkYmY5NjkwYTRjNTcwMzhjNzQ0OTExYTY5Zjk4YTkxZmE1NzQwNmZiZDljNWU0M2JhZTUzYjk2YmVjOTc0NmRiOTY4YzEzYmYwMzQ2NjkxNDZjZTc4ODE0ZDJjZDRkOTlmYzBkZDQ4MGY4NTc1Y2JmMjVlNGNjZjZiNjg2MjdmODY4ODdlZDMyMTVlMTQ0OWZiOTg4YjQ1MGM4ZTAyYTcwOGIxMWY5ZmVmODJiNGY3MzY1ZmQzMmE4YWFlY2JhNjkxYmNiM2QxMzU4NGE1MTc2MmQ3MGM0NzJlYWQ5NTMxNGYxNjI2MmRiOTVkODgxYTY0MmMxOGI4ODkzMjkxMGY3MWMwMzJjMTkwOGY0MDVlNmVhMmExMDUyYTQ0NjA4ZTFhNjM2NDY4Y2QwYTNmN2NhYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7FKIsFV462gXsmBJBQNdGRyEPZKf1WUl3-dbkOGrnWxZAf34rS5Qiv_dA4hjSSQ5oAOoPsB3RbAoJXN9R3I7-A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105948_19_2477_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.463Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlSTlJoNlVOeldXa3JIMG1PTXhYQTUyWGVOYjhFdm5qNWJOL1VFai9KcmhVMG9XdWpFb0hVcFlCMkpPdm5iRTd0SzNOV0ZPNWh3N1RoK01nTXNiVktnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDMyN18xMDU5NDhfMTlfMjQ3N18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmY3NzM1MDk3ODkyNzA0Zjk2YjUxNDg3Y2ZjMzI4MGJlY2EzNWExMWJkODJlZTc2ZmMxMzk3N2FjZTVhYTc5NGY4MzE4ZDRhZDNlOGQxYmJmMjFlODAyODZmZjc2M2FlNTZkMzk3MGYwOTc0Y2Y3YjRmOGQ1ZGU2Y2JmNmZjMGM2OTFmOGUwODNiMTQwMmY3NDI0YjVjNmMzOGMzYTExMjNmNzYwODE2ZmJmNGZkZjEzYTMxMjJiMDAyZGJjYWEyMTkxY2JiZWM5OTZhOGYyM2Y0NDNmNmYwMzU1ZTkwMTUwNTUyZGRmZWIzNjVhYTM0YzE5MzVhYjk0NzhkNjU0NWE0OGY0MzA1OTQxOGZlMDY2MWM1MDA0ZDZiZTIwNWQxY2Q4NDllNWJiZjM0MzExNjA2OWFlMzBhMmIxNjIwYzIwMDk4MzM5MTI4MDU2OGFlYzc2M2M2YTY5MzcxZjFlMWU0M2YwODhlNDNiZTdkMGMyZDU2ZTM4N2E4M2FhYmRhMDIwNmIyZWU0N2VlMTZkN2I1ZjZkYmJjOGQ0MGFmODJjYzNmMmMwNTViYTUwMGY0OWQ1NzE5MzU4ZDQ2NTFlOGUyZGU1N2U4M2IyODVmZTA2Nzk0ODkwZTUxMDQ2MWUyNmY2MTZmNjRiN2VkN2IxYTAxZjQ1YTM5YzJkYmRlMGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7SWUx7djW8lZSb94ZKgADp-ydg4Sl6Xr_euDvOaEHL9mMITfbqN2sUbBmEujICMUK4RGWZt6yqa7_hPHEpA9uw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220327_105948_19_2477_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.466Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IndBUHpNeTdNSG5pc0dzRThLbTdVOGRyRGJGY3VsTGpFQlJTV0pxWmZqN3h5VU1UMmlzWVBLUHdxR25PRVdJR3RTZ1QzTlBVeTFBZitjUWZkaEIzelB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDMwMDVfMTBfMjQzOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDUwYzBhOWI0YzA2OTU2ZTQ0NTRlZTNhNjhlOTQ1ODRkOGU4YTJkM2U2ODJjZTJlNTgxZDM3ODhkODFiNmNjZGZiY2QyNjU3YTk1YTQyM2ZhZmVmNmMyOWM5ZmYxZmNkYmZlNjUwNjIyZGRkZDRmNzJiYjE3YzVhOGMxMjIwNGQ1ZjE3MzhmZjc0MjhmMDk5N2NkNTNlMTM2OGZhNjgxZmUyNTU3YWE4MWRhNmJjYzhmYWQ2OWIxOWY0YzgxODkzMTBkMzBiNDRlMzRjOTE0OTIyYjM5ODI5ODk3OGZjOGFlMDc2MzVlZmYxNTQ5MzVlYjc4YmMzYzAzZjg0NTY2NDNlZGEzZDg1OTJhOWIzNDhiNDUyNTg4OWU2YmM4ZjBhNzYwOWQyZDYzZTRiNzhmNGFjMGY1MjcwODc3NzRkY2UwOGMyODI3YTFjYTllZTNlNDM5MjdmMWU5Y2ZlNGRhMzBmNjM1ZGJiN2JkNzA0MGE0MDU0NzcwNGE4YmQ5NGI1YjAwMTQ3MzQ3MDk1YjU4Y2UzMjI2NTdlMzBlNmJhNDNhMTBlNDA3MzNmYWQzYTA1OTczMmU5NDQ1M2RhMjI1MzFjMzM1NzAzNGI0Y2NkMDllODlkMTM3OWFlYmEwOWNkM2I0YzE3MjY4ZGMzZmIzOTIzY2M2ZDUzZmRlNDU3ZjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LN1qZFEhZNOUSaIoY1MXueFmS5rOi-tCpqAdkpPU5PMSX_qGcWxScr9DWABghYPeqFc1rWMl9W76PWfaLhq_eQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_103005_10_2439_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.469Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InAyaUp1eDRXTHZJMi9uanBSYnBQUGpPN0d0QytFVWNwTGwyV0IzWXNTOHNWeVVVR1ZJWnZab0xBOFVHbjArSm1GbENIWEhuWUZ4U2o0N1U3ZTNzQi93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDMwMDVfMTBfMjQzOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTEwMWY0MjNmMjRiNmZmZmNjNzhhNWNkNWI4ZGE3ODg2YzlkMzhmZDhhMDY4ZDVlODlhMjViMTc1NjRjOWMxNDg1NDdkMGU2MGY5OWRjOTA5MjVhZjY0Mjc4YmQwNzRkZjk0ZTM0YjU0MDZjYzlmMmJjM2IxOThjNDg4OWY4ZDcxNTFkNjY1NWM2YWRmNTFmYWE2MzdiN2RhN2ZhODk4NzYxYWIzYmUxNWFiNjU3YWIwNmExZjQ4ZWIxODcxOTEwNmU5NGM2YjMyYWMwMDUxNjAwZWE2Yzc2OWRiNTk2ZWYxNGM5YTExYjc1MTJjYzEzMTY4MGRkZDU2OGI4MzNjYzYzNzMxYzA0ODFkNGI0NzI1NWFhMWEzODJlY2E2NTJhZTJiZTUyNTU2OGMwMmJjN2M0MGEwZWE1MGVlMzY4NzA0NmU3OTU1ODgxYzkzNTBjZWZkZDJlZWY2NDQ0NWNiODk4OWY1YmI5NTgwNTgwYmMzNjRkNDdmMmJhOTExN2RkM2IwYzI3YTUyODJmOGYwN2JhY2Q0MDQ4YTAzOGEyY2IyNjM3ODcyYWI2NGMxMTBmMjAwZDczYzU1Nzc1OGQwZjhjZjlkNTRiMTdiZmY3MjY5MWFkNzZkOTkxZDg0OTk5ZDc5NDgzYjE3Nzk0N2UzMDJlNjIzNzllNGQzMzI1MWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yjjDIJ-aPeU6NqStVqje_bNS44cxA-3BnzpM5U1FoEHBUhjyOQaJ1IRcp7vGloZ9Z0JbeBUDVNkPEB1E-aWG3Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_103005_10_2439_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.472Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJCbm1RcldPQlFwaWJiYTlESS8rNXA3S0loWFcvOU9pMU5xM0hMNk1jV2tzakVRcVhEZ28vYk1YT05tMW9sbEFhcUlpUGNlbTRkc1VBeURuYUFNL2x3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDMwMDVfMTBfMjQzOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWJiODE0OTZiNmVkZDZiOTkxZWE2Njg1ZjIxMTBjZDI4MzE0MGZlOTQyNjkzNWY3MWJlOGRlYjhmNjBmMjIxODdlYzBmMzJkZTlkMzY0ZGZmNmJiMzBlNWFhOWU4Zjk2ZjgxMDU5MDYwODYxOGM1ZjAwNWRlNjQwOWQ3YjUyOWVjNWI4MTcwNTBjYzAwOTcyMzMxNjg0MDQ1MTRhM2FkMzVlYjE5Y2M5MzViNzMzYWRkMzhkYTE4MjU4ZmYyMmM4MWY2OGI3ZGRjNjRhYmFhZmU3YTIwZjllMWQwNTZjZGQxZGM2MDMwZTAwZWRhODQzYTNkNGIyZThiNDhlYzU3ZGJmZmYwZDJjY2JkNTRjNjk0ZDNiYjhiNmU0OWIwMTJmNDlhNmQ5MTIxNDhjN2MyMGExNzdiOTlmNGVlMDMzMzNjYTJiMGY4Y2RhOGE1OThjNGU2NTQ4MzBkMDg4NTZhMzcyYTk3NWQ1YzYyZWUzYzcyOTk5M2JjZTYxNmNhYzVjZjI4YzA5Zjk4MTg5ZTc0MzA2ZjA5NjE4Nzk0ZDExZGEwMDg5NDE3OWVhMTQ3NDFmZDNhYjU1YzU2MDRlNmYzYTY4NDU2YzVlZmVjZTRmYTkyZDcwYTU2MzNiMWRkNTk3Mzg3YzE5ZGYwOTJjMDlhZjU3OTg0MGM3YjgzMmJhMWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xYg6iXpn8yXgAYJLVizAKSmmF67ryMa0tOgiYukb_rN8uWWKyG8gtX4pfP1sGIM4QdwkBVffIY3e1lY3XhsWWA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_103005_10_2439_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.476Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZ0MVoxKzFidVhRV0hwajdxQnhiS3FVY3R3ME1BMXFqNXFlekRqc2tXNExYRU9sMmNFWXM3bHVKeWtIYms3ZUdUSEdpUlpUL1dOUE94bUM5TWRRVDZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOV8xMDMwMDVfMTBfMjQzOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVlM2I1NTcxNmM2YTZiNmI1M2QxNTk4NDFkMTFlMjg4M2Q0N2E5NWM0OTA4NzQwOTkxNWI0YmNkN2MzMzRlZTNjMjM0ZWMzZjI5NTNlMjc1OTIzNzA4MWVkZTMyMTAwNzdmZGJlYzgyMDJhYWI0ZjRhMDI1MzI2NGZkOGU0NTI5ZDJkZWU2ZjE3ZjgxNmFmNzU2MWI5MTZiZTk5ZDAxMjM4ZjYyZTdmMjRkYWRkZjEwYzhmYjg2ODgwOWIxMzZkNzIxYjhmNmExZDg0MjM0N2RjNWNiZmMwMTM5ZWIyMDBlZDA3NGQxNTg4YzE2NWFhY2EyNmE2YTUzNmIwZTFjYzAwNDA0NWVlOTJlZjZhYWY0MTdiYWYwZDUxNmNiNTM2ZDAyMTU5N2I4MmI3M2YzOGQ1OWQzYjFlMmEwMGE1OTUwZGM0MzM3ZGU5YWEwYTQ3MGY1ZGNjNzczZjlhN2U0NWE5YjY2ZTcwNTI3YWU5ODA4NjZiOTIwZWEwZDYwMzdhNTE1YTE3ZmI3NzFjMjY4NWFhMzRmYTliZjk2NzMzMDQwMGZlMmMwNDI5OGViMDZmMjZmYTMwNmI0NjhkYTI3ODM3YmVhYzViMzdmNGUzYWYyYjczMjVmZWEzZjgwMThjYWYzOWQ3N2YzNTA1YTcwYzYzMWRjNjVkZThiMzk0ZGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sG-uLWpiR_qctQLVgOiEkRMNRplqapgSwIVr8PdnLlezbrSesxxBUmzZc9e9tdGrUxEvni2gePrPum8rlLI_aQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230309_103005_10_2439_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.479Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRzZkZZdVBzaVNUODJIZDFYN1pIYzJlY1pjUEYxOFVMYXNIbitabGlJKzdFK0xTdG91T1MwUWNjc3RUWTROTVlrYkxsamo2VjhHZHdoTCticFlnaUV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDZfNDlfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTQyM2Y0MzEyNjg1MDZlNjViNDM3MmEyMjVjODM3ODQ2MDBkNGRiYmFmNWJkNGQwZTM0ZmE2NTBmZWY0YTI4MDFhYmQ1OWU4OTc1OGI5MWJmMTg2M2M5NmQ2OGJhNTBjNWIxZTkwZjFmMDhkYzBlZDk0ZDRjZDdjNDY0MjhjZmZmNTE3M2NiMjA2YjNlMTk0MDRjODlkY2FjYTc3M2YxMzRlMWNiNWFiYjllMzQ5NTlhOWY4Yjc5YTEzMWIwYWU3MTZhZDcxMmY2MzliZTcyZWEyMDc1NWRkMGIzMmRhNDBjOTViN2NkNWEyYmM3NTgxODhiOTliZGZiM2U4YzUzMzQyMjdkYzQ3Y2QxZjk0NmIzOTAxMWY5Y2U5ZmQyNDI0NDg4MzZmNzZhODFiMzM5NzY5ZWZkYmE5MjViYWU3MTg5ZTRkMmE4MmFlMzNmOGZmODg3NTY3MzA0ZmRkMDYxNGM5NWNhNjgzNmYzYTgzNDc2ZDVlMWVlZDgyMWNjMjI5NTYyMGY2YjJhNDdiZDc0ZTliZTliYjczN2FlZDdjNzEyZjM0NmZlODFhZDMxN2E3YWFjMzI3ZWE3N2E1NGNmYjA3YzU0Zjc2YTJjNzU3NmU2M2Y2OGMxZGUxMzMyYTY0ZTY3MmQwMjBkYjVmM2Q3NzcxNjUxMmEwY2EzNWU1MjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uYfptsEd6ZTkIvr8zul8NLtbUXktZ8CPIXm7P037uZRGD1LbtxRU_O4EIVZIzptwMoMYCFCivr8OnN45XjJihA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112646_49_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.483Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IllVNXBqUVJ5SWJJRi9jYkhjdWRxbmQyc1BsWW4yUzZFZlBKMUJ6eVVOR2EwQnVseEVMdnFEQWdLemFEL0dMdG1ldWQ4N1I4T0UxcHhpVmo1SkZyYW53PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDZfNDlfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWE3MWE5NDZhOWJhNzhjYzhiZDUzZjYxN2U2NDAwOTU4MWVhNDE4Mzc4YWQyMjIzZDc5YWQzNDA4OGIzNTA2MGYzZjRhYjBkNmZjNGQ4Y2UyMzdkYTMwYjE0YTNkN2JiMjg4ZTYyYTk3ZDNjNWM3MDE5MWQ2NWY2MjliNGE2Zjg1MWEwNzEwOTFkYmMzY2M4NTgxYjYyZjU2YzQzOGYzZWE3ZWZkYjk5NzNhOWUwZTVkZGI5N2RiNDRlNjMwYmVmMjVjNzQ4YWU4ZjcxYTNiNmFkZTI0NTNjZTcyOTVjNDAyMDcxOWJkZWVlNDdiZjU2MGQyN2Q4NWU5YzA0MWI0NWRiYTUxMGJkMzkyYTY5MzQzNDI3ZDdlOTA0MDg1YjcyZTg3YmYzMTM2NjY0ZDI1NmY3YTA5YWU5NWFmOGMwYzQ5MzIwOWQ5YzJlOTgwNTI5MTcwNWEwYmFlYTQ1Yjg5NjFlNmVlZTU0ZmE5YTBhZDI4YWUyNTFiNzlmNjJjODVlOGQ0ODFjYzY3MmRjOTJiNTQyMjM1YWNhOTY0NTI0NWRjNjQ1MWQ5NjgzNGU3YjdlNDg2NzY2ZDgzNTdjYmRhN2YwYTUwZmNlYzUwN2JkYjk0ODRiMGEwZDI1YjkyZTUyN2U1Nzc3MWI4YWE1ZWFhOGVjYjZhMmQwOWE5NzdjNGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CxSYMkXVtL7A07yqHSUYATT3Tv565KqHXRVH2iYo_Sq7v-SNidOk8emeGXr2ipyZzTL25lDKQz-kdbQp-CpRnQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112646_49_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.486Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZFSTRTVzlDNFpzYUh2dWRNRkc0TEJNZWNQUDhJcUUyQ3UvRDJZc2F1WFpESTBHc3Y3OEQ4Ui9YMjFrTW9uL29aZlhmM3g0dGpob080dDhtWWFqbnVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDZfNDlfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzBkYjJmY2I1YjVlZmIwMGZmNGFiMDU1ZDUyY2JkOTM5OTZlZjI2YTQ1OTk3YzZkNmNmMDAzOGQ1NGQwMTU0YjEzYjgzNmIyNWY0ZDBjYTRkNmNmMjcyOTliMTFmYTAzN2ExMmY5MzZjOTMzYjQyYWQ0OGE1MDFiMzc5YzllZjI4NTMxNDlhYWJjYzVjMTNiZjlkMDA3YmU3ZjM1ZTJhY2QzYWMwYjM5MWIxOTk1YTc3MDc2ZWFjZTFkYThkZTJlMzMwZmYwY2M2MTk1MDM3OWY5ZTVhYjE0Yjg5ZDM4NWVlZjBiNzdlZWVmMTMyY2JmNGZhN2I5YmMzNjFkNDNlYTI0YTNiOTdiNjg2OWM2YTBhNTMxNzYzMjgxNDUyZWIwYTE3MjA5NWUwYzZlYThlNTBkNjkzNGU1YmIyMzZhYjhlNWRmODcxNGQ1ZDFkNzZjNWQxNDU5MzhiNTI1ZTA1ZTNjZDM5NzgyYTA0NWY4NDc3ZWE4NTA5M2RiZWVkZTc3MGU4ZGY3MzMxMDYzMGMxZDVlNWU2ZjY0YjVhNmNhNTFjZDY5ZDQzNTQxMGJiYTk0YTZiNDEwMGMzNDY0YTA4ZTIxZGIxNTkwNzFmMjIxYWU1OGVjMDJmOTJmYjZlMDE1ODM3NzhlMmVjN2VjYjYzYzgyZDczNjRjMjI2MTJhNjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Om0z6yEmtdofvkyywlsAPkMPEbvmseQhqV06UMQQP5cnxNq7EPuTWOznQoAaU8fryOvBZIC0uKW47PDFiNfFOw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112646_49_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.489Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InczUGJhbjRFdDdjYzc4ckpvajRYL1h2QmZLNDM5djNTYzZTSkk3bG1mUW1OZ3dpZXNncXFwM3dBTzFlUGxWWnRLQlAyRTZhcDZiYUwwZ1R0T1pMeUVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDZfNDlfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGM0ZTBlNDU5MDc5ZTI2OTE0YjM5YTU5MzBlNzU5OGRlYmYyOWUxMjk4Mjk1ZjYzMjUzYWYwNDUyNzc0NDM0YmE2MzY2NmVlMWUxZjBiYWUzOGU3YzQyMDFlMzc3YzFkZDI3Y2IwZDZiZGI3MjJhY2I5MWM2MmNjM2IwZGViYTU0NmJlNDM4N2FiMjZkODliMDY1YmQ1YzMzN2Y3OGVjZDQxYTAyZTIzMWVmOWRlOTdiODRhMDc5YmFjOThhOGViMjM3N2VmZTBkYWQ1ZWVmMDg5NmM0MDFjNDU1OGI2ZWViYjdiNzNiODgxOWUzMWEwNWQxZmYwOTBhZTMxMjdmNWVmZWM1YWQ2MzEyYmM2NDdhMDUxZTNlNzA2ZTI4ZmVkYTc2NWU5ZmM5Njk0YzQ3ZGI0ODJmMDU2ZDg2OGNiZmY3Y2E0YmMxNDljYTAzODg4Y2QwMjRjYTA0ZTllZTJjZjBhNmYwMDY3MTQ0ZDdjNmM1ZDAwMTI0ODA5MjgxZGU0NjBhMzA0NmJkYWNhZjQ3YWFkNTQwMDE1ZGI3ZjUxMDQ4MzBlNTIyZjA5ZjMxYmYxMTQ2ZGQxZmUwODNlNGY4ODI1MTlhOWQ5MWYzZDE3MmVhMTZkYTJiMzUxYmVjOGE0MmQ5NGNkN2ZkNzdiMGZmOWZhYzMxOTc5NWQ4NzEzNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.37gG0LoHl4X-5jks1xCPTRW8feSHuvj8Ue8AxHf7wTT6YaPtQ7P6407rH1TX6EBrjiY7fHkWCIx9dhCjBktNNw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112646_49_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.492Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InF3TFFpYU1oYVJTdDNmREU0ZE03dnRHZU12VDdScmZwMUpKSW83NlUwcm9oRW5ielFjMHBWaTVsbXBicUdMRUJSZXZoZWF5UnlqRGhKREtIN0ZqYzl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTJfODFfMjQyMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWFkZDNhN2ExYjhiNTMwNjZiZjhlNjRhNzdhNTJlZWZkY2NkZjgwMjhlOGZjY2I1MmRiOTNkZTc4ODE4YjFmNTkxNjBkODZiZGRiYzkwZjQyYTYyNzhkMGNhNGE3YTJhODliODJmZDRhMDdkNzRmY2FjODRmOGI0MDYzZjM1YmQ4NmI5MjI1YWQ1MzExYjI3MzhmODk5ODZlMWIwZGMwODVmMjJjMzEwZWYxZjZkMTFjOGMzODNhNGFjZDlmZGMzYWQyOWI5ZWVjYTIzOTQ1NGJlYjgxYmJiYzdhMDI4ZjZlZjI2OGU5N2IzZjgyMGVhOTc5MGViNWMwYjdiODZkYTgwOTU1ZWMyOWQ5MzcwMmE4YWU4NWE1NTdmMDlmZWUwY2QxNDAyMDY0ODM2YTI0MjBlZGQ1NTdhYTMwYjRhOWU0OTMyMDhjMGY5MDhjY2NiZDA1NzEwOGEyZjgyMDI4NWM3YzAwNjg0MjUwNzc5M2I5ZTRkMjAxZTBhNTY4OWQyNTNhMGJmMDhhYTI1MGMxNzg0NzYzYWY1ZDc3OGVkYzkxNTM4Y2MxNjVjOTVhNmNhMWU0YzgyMjZkZjY4ZDJjZmI2NTY2NmQ2YzEzOTZiMzQxMDU0YTlmOGExYjExN2YyOWNiZDhkZDI1NzlmNDBmNjEyNjY4YWIxZWMzY2E1ODFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ekwVIZuHF3ooHRhoyrfybxfcuriLoz74jSpj14cCOEtqEpai_PQcFOuj1w6aNORF73xweN-9iW3q6RXa4STTxw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103452_81_2421_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.499Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlplZDlCMnkvcnRMMTlFK1lSb2NGYlEyTlhlMlJlVlJvbWRiOXlYR2p5YlR1YXdNSmx1KzliODVCcFBCQ2VHVXVXUmVIZWlwQURpQktTWUlnVklmbUlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTJfODFfMjQyMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmFhNTZhMWFjNGE2OTAxMzZiZWMwNDM1NGVhOGU3YzE5ZGQxNmExZDZkZTBiN2RkMGY3NWYyMzIxNDBiMWRmODUwNjk3MjBjOWI0ODIxZjdiYWMyMDkwMTdjYTNmYWM4ZTM4ZTQxNjIzZDU5ODk5NjMxYTllNjkxMjUxMTQxM2I2MDQ5MTcyMjhlNmRmMzEyYzNhMDViMDg0MWUyNTQyY2U1MjNkMDc3OTU4NWQ1NGMwMTk2OThkNTdhMWJlNjViY2UxMWJhZThmOTZjMDY2ZTA2N2NjMzBkMWVmZWU5ZTljOGRmZmIwYjg0MTA5Mjk0NzE0YTQ4YmEzZWM1MTRhYWE4MzNjZjYzMTBkNzI0YWEwZTg0NTJmZDY2YTY1ZmY2M2M3ZThhZWEwOTU1ZmNjOGQyZjdiMjc4ZGMyNDI5YzUyOWRhNTI0MmY0MjU0MTk4YzVlMWY4ZTM2NjRhODY1MTlhMTc5ODE1NDlhYjg5NTgwM2M5NTQxNTBlNmVkNzQyNGVkMjc0YjUwZjg3OTBiNmUzYmMxZmY5MTJkNWZhMDA2ZjhmOTEzNWQ1ODQ1MDhjNjAyZmEyY2JjODlmNTJkMDI5ZTQ3M2NjZjlmOTNkNWMwYjU5MjdlODM1YjA4YmNkZGI0YTI3YjhjMjczZTcwNmE2M2MxYjE4MTE4NDBhYTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.P6DI336cHz9k8k2upCa14zsE0xH-tErmHgNZdn6lK6UvR67u1qqQFcywkdxD3FRDLuRjuCKC-6kyVA3KBgEcnQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103452_81_2421_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.506Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjdzOXp4N0xqV1FpTmhxaGRPSExER2RNTUIyaE92cDg1c3IrQnNPWHBlNU5UOEZwSUNMR1ozTWJNZWo1eTBjODJ0VkVCOHBEY3ROcGFJdXlFcVZRbWRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTJfODFfMjQyMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2Q4YmM1YThmMGYyNmI0NmIyZmVmZjQ4NWY2MGY2YzQ2ZDAxN2NhNTc5NjhmM2Q3M2MzNzE5NmYwMDVlNjFiYmI1NTBmMmRkZTk5Y2Q1NzhmYTRjNDc2NzlkZGU5MTgwMjFkZjI3NjFiNTZmYjg1Yzc4MjU4MjZiODVmMGNkZDJlNGYxNzE5MWQ4NWJhMmJiYjE0NjEyMDY5NDFhNzdjNWM2OTc0ZWI4Zjk5MzI0NjkwMDllMTNhY2Q2OTgxMmQ3Nzc5MzZiODRjMWMxMDIzMGJmM2ZkNDBmMzdkOTVlNzY1YTU5Zjk4NWI2MTI1ODcxOTE3MzExNGI4ZWQ5ZDA0ZmM4ZGRmNDIyNGU0ZmNiNTZhNjkyMjM0YzIzYzk2OWVlMGYzZTA2YTZiMmI2Nzc2N2YyMmY1ZTU5ZWU5YmZkMDk1YzJjMjc2YmJiNjliZGNlZWM0MTA1NjU1NGNlNTJkZmVmZTcwNDRhYmY0NDJmNTY4MDExYTNlMmE4NzkxMzc3ZmQwMDE1YTIwNjY0MmQ1NjA1NzcyZTRjZGYyMTBkYzAxYzAwODUzNjA5ZjA0OTE5OWNiM2UzYjhiOGYyMmNkMGUwY2RkM2FhMWRlMTQzM2QwN2RhMGFkMzY1ZDI0NDJjNzg3NGU3NmU1ODA5YWNiMGEyYjg2NzQ5ZDc1OTg5ZDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5QphV7KPqHf9FbgWePAA0yg0x6KoOG-P8YfxvzZjWrpGmZqlvHs-fnk09TECgF4jz2fLPYXI8rEio89R2orq1A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103452_81_2421_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.510Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkQ4UVd6N2JhTFV3d1RHWmtTU01wdEx6M0U1Umd4eitQdFhFRGNhMVBiaWdpdHdTVVpIV3YySkZZcGp3Vlg2Y29FRXpUUlR5Y3pNbkpMeUdSY0thdW93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxNV8xMDM0NTJfODFfMjQyMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzQ1MWFjYWE1NTVkYmNiNDlhNTM5MDIxMDI5YzFiZWFjY2RlZjFhYTU4ZmU4ODU3OWI4YmFlMjM2OTA5YjE5ZTViY2YxODM3ZTYzYjA1NGRjZjg3MzI1NjkxZjA3ZGI5NzBiZGM1OWMwYzc4ODFmNTM3ZTJmZDQyOTM5Nzk5NTgzOGZlZmIzMjY5MGNiNTY0MjAxMDMwODJkM2YyMzc3YWVjMDhhNzMzNzY0MDc0YWE0YTFjNGIwNWQ2NmEyMjI3MzE2ZGIwZjc5OTZkZDY0NWQ1ZThmZmU3NDY0MDE2ZDkwZTdlOGFlNWQ1ZGViNWU5NzVlY2RkOTQ4NzYyNmNkZWMzOTMzMzU4NDY4YWIyNmMyOTBkNTVlOTYzMTdkZmU0MjAzZDNjNmYxNThiOTMyMTJlYTAwMzU5NmZjNTQyODdmMDdmMjRiYzUwZmE4OWRhZTI5NTk4YTJiYWQwMmEwMjY5ZDE2Yjc4OWRiMWI1NjNjYTI1NzU5NGIzYmMwMmE5Yzk1N2UwOTRhN2YyYWJhYTg3YmZmZDYyOTA0OTk3YzkyM2NhZDVlODc2NzkxMzRlMTkwYzg4MGI5ZGRlODU4ZGJlNDExZDExNWYzZDVmZjc0NjVmY2RjYjFjY2Y3NGM1MjcyOGE1NGFlODQ1MzgxMjY0Zjc3MDhlYTRjMDNkZmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rS4yTJ5sKltppNlqVv7gHNizqG5EIkoXoqlI5-91bk-YUlxBIbBcMt07Qocd7REPSAVhzfDuBHhRP1YqE6K4-A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210315_103452_81_2421_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.513Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik5ybHpEN0dXeWEzd3pMWGJnMGFPOEFJQmdsbEVYWkswSXQ4YjV2ZEc5aFhrRi9UODJjTCtsMlpDMURJVVZYbFNRUEMrWG9UYlMzVGU1eVlySU5hNmF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDhfODRfMjQxNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWJkNjk4ZjVhOWM1ZTc1YjQyY2Y0YTM5N2UwYjc0ZTU4NWRlYTk1NTBiYjllMmNiYmI2YzNiYjE3MmVlZDgwNGIxYTlhYWIxYWVmZGFiMWE5YTIzZTEyNDQ4Y2VjZDNmZjE5ODhjODhjN2UxYWJmZGEzM2YxM2ZlYzE0ZGJjNzIzMTZhYTVlNGRjODE5MjQ2MzAxY2YxYjliNjZiZjg3MTQxYTlmMDI0ODJiZjlhOTBkYzY0YzE5N2VmNTAwMjI0ZjJmZjJjYzUzNTRkM2Q5OWUwYmM5NzVkZWU3MjQxMzQyMDYzZjI0YjFmN2U4Mzg2NGJmYWMxYTIzNjUyY2YwNjFhMDMyNDJjZjhiZmY5OGIwNDVkOGIzOTk3OWIwODUyNGE3NzJhOGM3Mzc3M2UzOTM2NGFhMzc2ZWIxYmVlM2ExNjJhOWI5N2YwYjI3ZGNhZTRkZGQ0NzQ5Mzg0ZGY2NWQzMWMxYmU3OWQ0NjU3ZTZjZGM3YThjNmM0NzU3OTc1NTIzMWI0OTM3MGNlMGNlZGFkMmEzYTI2YTQzNGVmNTgxNjFmMzk1YWIwNWI0NmQ0ODYxYjU1OGZlMGE4Y2M4NmMwZDZhMzNhNDYxY2JhNTRiMmFmZTU5OGNiMDBkZDRkODg2OWI1ZDUxZTZlZGMzMmI0NTRlMzU4ZGRjOTQ3ZDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1mNV_n5JkpDm5m5aVYU70M3idteQ_5h05G7-auZDpd5sg4wMtVJqWHYn0ktnjFRRnFgjcYkr3mZQ6cmIV5tmyg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112648_84_2416_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.517Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Iit0NkR6NTVweVdodUYvV0FCMk1SRTlUZXpqRTlWbHBIaVZ4d25xeGkrRzNMa2NXd1hpK09aVjBTQmFkREUveXFNV1c0dGdKcW92eUk1N254SkdVYlJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDhfODRfMjQxNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDc2ZTZmYzA5OTM2ZjA2NDM3YjAzZWIyY2YwNTk0ZmY3MjcxZThjYTBiNGIzYjNmNjZhYTNhNTgzYmI4ZGEzYWFjMWVkYWYxNDUzMDMwY2UzYmIzZDI5ODJhOTQ2ODJmMDQ2Nzk4MDBlMWRmYmNkZDhkNjZiYzBhY2M3YjI3NTQ1NDU5ZDE3YmRjN2RmNTdmZmUwZjcxMDMyNjQwOTI0MDI1MmMwYzQzYzk4MzQ5N2JmMmI4MmYxNjFiN2M5NjRjYjU3MzI3NjgyMGVlYzVkMGI4Y2YzZjJhMDk5OTJkMWQwZjMxMjIyZDZmOTE2YmI1MDZmYjJmZGU4ZDRjNjFiYWZkMWMyNTRjMGViNmU0NDFhYTJkMWEwOGEyMjFjMzQ3YzU4NWYwMzQ4YTEzMmZkNGY4ZGQ0MzA4N2NjYTcyZWJlZjdlN2FkNzcxZTZiMWQ2ZWMyODc5MDI0ODg0MmY0NTBiYmEzMjJhM2MyOTA3MmUzYjhkOGZkNjU1NGIzN2ZhNmI4NGRhZmQ5ZWFmOGZkNjhmYmY4ODIxMWQ3OWY1YTdiMzhhZGU2MGM5MzFhNzJhMzgxMzMwMTczYmVkZmM1ZGU0ZTIzOTY4ODUxNWUwYTFiNGZhMDQ3OTM4MGRhMmFjMDAwYTViZDgyNDliZjllMjg3NDJhMDc3MTczMDE2NTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2_pDxXNQYz_Ho9iM_6skAme1Ogx0_WlbYyj-de_-OGHKvRFPTRlwYYYzQh-CAiQFBysxlbDEFoYi-GBv8eqMDA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112648_84_2416_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.520Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImcvNkt6Vmh3MTJDV0ZiYjdSZUg2bUFvUFhWNmx0UVpzVDFXdG51Sm5qMENBVVp1Mi9kV3JQMGhjd2JiQnJ1WmFqRS9nT0Y5UkwzTFlHTnBRd0pnemFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDhfODRfMjQxNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTE1YTU5ZDJhYzFmYjlkNjJmOTg4MjhkNTcwMWNkOGQ4YzAzNGM2MjUwMGNkMDRjZjBlZDk1YjE3OTRhODI2MWJkYzBiMmViZTViN2ExOWU1NWY0NmUwMGFlM2ZiNGU5YWY4MTE4YWRmOGMzZDBiYzc0ZDRlNWU0MzIwNGU5YmY2ZjE2NmU4ZTIxODA3ZGZlOGY4Zjc2ZTUxOGM0ZTYyOWUzNjRiOTMzNjMxYjJkNzdlMmMwYzljODViYmNjZDVkMzRiZGUzYTA2ODkwM2I1M2VjNzM0NzIxNjJiNjk1MjIyOTBjMmViNWY5MjM0MzRmMjNlZDdkMzExMTYwMWQ3OGQ0N2Q2NjlkYTMzMzA4ZWI0NzFiNzZmNjc4MTVlNTQxZmYzYmVkYTNhYzEwNTlhMWNlZTQyZmM3ZWE0M2VjZjlkYmIyNTg4ZTA3YzUwZWRkOWU0ZDFiN2E0YTUyYjViYWM1NzE5MTU5ZjJlNGM5NWQ0YTY1ZDdiM2FhY2U1YzdhNzAwY2RlNjkzZDdjNjE5ZWI4YmQxMmUyMWRjZDFjY2FlODQ4NDNiMzIzZTExYTE5M2ZiMzI0NjIzOWFjZGZkZTMzZTBkZjA0OTI4MWUzNzJlNGE5MDI2ZWQ1ZDBiY2Q4MmI0ZDEwNmU2ZmMxYWNhZDdiZmUxZjNiN2I3OTMwMWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QA6Ukvq5a3ihfUaQ8VxsGmrC0NPyaOA9tzbhFiVHdnIrtawRiVwYzD-gl4GFkKgccxMOKV78k-wtGCQWETrYpA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112648_84_2416_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.523Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBRYnp4SFprQk5rWU56R1pjRzRNdlhvaUJQREhuUVpQM3oralRhRWQ2amZRS1NWUWNMS1RmelpqUVVnYVduVHVVL0ZKdHp5bHJkM3JQdlUra1JYWG9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMTI2NDhfODRfMjQxNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWIwODgzYjA3ZWIyNTA3OTc0ZmU2YWU0NTFiMDY2YzkzOTVmOWJjNmY0ZjEwNjRmMWNmZjJlYjI0MmZjZDY2MTYyZDA1YzhkN2EwNmYzNTRjYzk1NGQ2YzdmNDU3NDY2N2M0NTNlMmU3ZmI4NTEyY2M2OWUyYTQ5ZGExODRlYzk4OGRhY2Y3NzI5NDJmZjkwMTI5MjU4YWUwOWI5ZWViNTljYjM4YWMzZTc3ZjFlOWVhZGY1NDJkZmQ5NDI4MTkyZTcyNGQ0MmIxMDQ3YjQ4N2RjNzg2M2VkMjZkM2M2OGQwMmIwNjg5ODcxOWVhMDAxYjgxOTNkZTY5MDdmOTExN2ZiZDY3MDU1NjE2NGYwZGQxMjYxOTE0OGIxNDE1ODA4ZDM0M2Y3OWM5MDRlNTljYWQ1MWE3ZTNjZTgzZjM2ZThmM2E5NGVkODkzNmJkMDU3Y2U3MjJjMWEyOGQyNmE5ZGI4NDRlNzJlZjM1MDZhYWU1NDk1NjcwM2FkYTNlMDU0Yzg0MTY2MWVmNjU3MzNkNDRmYmEwNzk5ZDYwMDE1NzlhMzUwYTc4MGVmNmY1NjgxZWRmYmRmOWEyZTYzMmZlOWZiNzI3MDk4MWMyMmYzOGUzMjc1MTc3ZTZiMTgwZjZiODgwNzFhNTMxNGIzZWM1NDBhYWI2MDEwNzZjOGJjNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SH-ciG7lr9BDfsdyTS-wfWLtK-7g6njChaxz6Yrty12F8LRmx91ElIfUTM7IUEC4aYoNsaGvN08rr6ih4hhbpA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_112648_84_2416_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.526Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IitPL0EwcDdZYkx0S3N0OG44RFlUbGlHdDFPbHFyUE5LRjgwVFdYcVUwRVgxYVBOemY1WnBvWHg2d3FQYzZSMlNJMDZuSkY3S3k3RHFUcDQxNzN0OTRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTlfMjhfMjQ1M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGEzNzAzOTlmZGJhODBjZjBiMmJmMzNhYTk5ODRkNGQ0Y2NlMGFhZWVjM2M4MTMyMDNiMmE3ZjlmNTUyNzhlMGViOGU4ZjY2Y2RkYzdiMDIyZDEyMzFmZThmNmVjZjkzYjUwZmQ1Zjg0NjgxZDMxYmUyODI1YzRiOGRkYzk1M2IzYjAyYzhiODg4YzhlNjFkYzZhYjc4NTIyYjYzNDM1YWY5NjM5ZmM4NjIxYTQzZDVmZTY0Y2FlYmJkMDU0YjFjODIwMDhlMDVhYjIwOTA4NTE2NmYzODFiMjQxMjcyNzMyZjAzNjk4YjJiYmZiMDY3ODE0ZmExMjc0MjdkZWM3ZjUwZmRhMjg4MjExYjdiN2ZkN2U4NjNlYTk1YjNjNzdiZGZhNTUxMTZkNDI0MjczNGRiNDg3MWQwNzg1NzUyN2M2NTRlMzZkN2JmOTZhY2NiMjY2MTFhMDI1Mjg3NTVjZjllN2EwYjYxNWE3NmFkZmJjYWU3MmI4ODFiODRiN2ViMjBkZDRmYTkwMWYxYzdkZTNhZjRlY2M4ODZiNTdlMTQyMjBhY2E5ODhlZmNjNDc5NzYwN2M2ZmU0Y2I5ZTJlMDg1YTM2N2JhYjI5NGExMTE2NzgyMGU4N2UzY2Q3NmExZTkzZDViMGJmM2RiOGRlNjM5ZmZiNzJkNjhiMjhhMGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xv_rgObsySFgMlW9JfT4r_wcUFOC-wjZqFk_FaifvPtmZnEz8C77ntM3BfeyRHr2QX7Lxf2b4PrdoLlyPwFqKw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103659_28_2453_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.529Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InBwWjgzNkFaTUp0SndLdUZpUWQzd3lnT1VwZnhpN2RGaGRnSUZVRmhkS0FqalpVK3BCaFJxYjJsNzRMMjJDMHVIVkFGRFFLK05XYVBRcnFPSzRPUVRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTlfMjhfMjQ1M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmVkYmZkYzk4NTQwYjExZDdiN2UyNmRmM2VmZWRjMTRjOTkwMTcwODYzYmQyYTgyMjQ2MmFhNDBjYjA0Y2UyNDA0ODlmZmIyZjgzYWU1NmU4MWMxZDgxOTNiMGQyZGE2ZWNhNTZkNzlkZTlkMDYyMTMzMWU2NmJmZDE4MDBiNTU5NmI2MDM3ZGNlNWFlZjY3ZWFmNDE1ZGI1ODNjYzlhMzk3ZWNiNjcyMGJjODkxZDEzMGEwMjUyNGZmOTQ2MDUwYjk1ZDk0NDJjOGMzNjE0MzU0NzA3NTNhMjJjZmVhY2ZmYjE0MTgxZTI4ZjUxMDZmY2NlMTkwNmYxOWQ2MWFhZjgwNGM3MmMxYjE2NzU3ZDAwNzM5ZWJhMmZjODRmMDg3NmIyYmYyMjRhMTVhMGM0ZWFlOThmNDYwYTM3N2E5YzdjYzQ2OTM1M2E3ZTI1NWM1OGVjNDBjZmUyNmY1YmNmYjI0OTU0OWQxNTkzM2Q3N2ZmY2FmY2U3N2NkNTg0MjNmZmRiZDRmNjdiODZiZGU0MTFiY2RmNjE0NWQxNzE2OWMyN2QxNWVmNzk4ODU3MjM1ZTM5ZmI1MWZlZWU5Njc3MzE0ZDBjNDY3MGQxMGIxNThlNWVjMTA2MzBhOWYxMGMwYTQ2MGU1YTcxZmIzOGNmNTgwN2M4MGZhMDliMzBkNGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.x68KA9Wg3umXXJF-xnGPBkrEXfcG5-vYfRZeq33GzbiY31zKGWVR-0tSDUy3M7Vo9i3Tgiamek3vbFnrB2-rIw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103659_28_2453_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.532Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRCSkl4ZHNSYkZ4YkdtMENNOG1ZRk9SOWszRDRaeXo1dVFEQktKekZaQXptTy9DOFJTa1prNE51Kzd2NUFiSzNRZ2hBVjZyMFVFcHZaemRhTFo5TmdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTlfMjhfMjQ1M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTNjYzUxOTgwYmJhNWM2NmU5NDQ4MDIwYjUxM2NjYmY0MGFlYTY1MjFjOTk4YzRkNDI0MThmNTI5YjMzNTQwYmUyZTJjY2I4NGFmNjMyNjE2ZTFhNWI4NTc4MTU3N2E4MGY0NDRlNmU0ZDNjMjhiYjg5MGM3MGU2YTg3ZDUwYjVkNDY3MmIwODE0ZTJjZjEzMzQ0N2QwNDE5NWNjNjY3ZWJkN2I0YmRiZTM1NzU3MjZhN2EzN2Y2YWNiYjQyODdkZmFiNzhhNmQ2YWMxZWU3ZTI4ZTIyYjcyMDFhZDVjYjg0ZGQ4ZWNhNDA0MmQ5MzY4OTkxM2YwY2U3ZTNmMjRlNjZmZDVkYzM0MTFiMzE5OTMwOTdjZGZlYjZkZWFjYTMyMTg0NGZmZDI3M2Y2NjAwYTg4YTNhNzc2MmY5YWM3Yzc5Mzg4ZmJjOTYyY2E3NTg2NDNjZGVhZjZlMDFiM2UyNTljOTdiNDU1NDBkNjIyNGZiZWQzYWY0NmM0MWQ2ZDJkNjFlZjQ2ODQwZjZiNTI1MjFiMzE2ODZkNGE1OWI3MDUxYjU1MDM1ODkyYjUwMzAzOWNlMzc5NDE1MGQ0ZDY4OTIwNGVlMjdmZmNmMjBiYzg2N2Q4NzFhMDQ5MWY0YjQ3ZDI0M2I1OTg2OWY3YWYwNjNkNjZjMWU3ZjM4ZjlmNDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.k_8Xqs0-ta8pR_4_sQb-K5ZOL307Hd3xKiTWIilyYLLwtsQIDbwPRLjuHHcO4jzjgPJgWed19I6KcGnggs0JVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103659_28_2453_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.535Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlB4ZTFEbzkwRTR5WEVzWUlvTk1RL0FGUjdqWmlzYUt1anVZTXRYUjhzRnIwUTdWVERlL2FtUm9zVUlPMVRPQ08xdHJ3TGR6OUI0SStlcWRqYkdyZDB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMF8xMDM2NTlfMjhfMjQ1M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDY0YTQ4MzQ3MjYzY2MxODRiMzc0YzMxNDU4NDg0ZDc4NTkxZDJkMjc3MWY3OGUwMDE3ZWE4MjI4M2NjMDA3ZGVjZGU2NjgwM2EyZmJhYzhlOTA3YzgyZGZhM2NmMzlkZDUwZjFlZWFiODBiMjc1NjliYWUxNjgyNjc2MGYzNTc0ZjAxZDZmZmUwMDIwYmUzM2M3M2QwODAyZWY2ODNjMjg3NDBiMWE1YjgxOTVhOTYzZWZjZDAyZWI1YmVlNDBiZjVmNWMzNjhjZmY3OTI4YjI2N2I0NDhhOWJmYjg3MTZkZGJmN2M0N2VmNGQwYzllMDhmNTVjNzhlZmZiYjVlNWU0NWJjMGUyOWIyNGIwZDU5NjQzZmYwYjg4ODc4NDEyZjQ3ZDUxZDgzYTRhNDY5YzI5OTA0ZDhlYzc0MmJjZWE3ZWRhZTE3OTA1NWM4OGZhNDI1Y2QwYzdlYWIyNWQ2ZTYzY2U3NmVjYmJmOWFmZDY1ZGUwYzFlMzg3ZTQ2YjFkYTY4Zjg1MmQ3MjEzOWUyODY4NTE3ZjBjODkzNzUxMDQwMzViNWIwNzRkZTI5OWI0NjFlNTZiOGVmNjRkMTEwNjRmOTIxZjJkMDZlNGQwMDFhZjE1NjZlZjAzNWY1ZWM4ODdmMWY4ODkzYWNlYWY0YjM3Njk5NmIyYmM3MzEzYTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.U0W42-kYINfuBlxXeQkfIipYKvVxGurmVxtlnkpo0L9BVS-rxCjh7wRV4ZkwvyfK9Pq-xSIAIettlJxT2hKEmg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210530_103659_28_2453_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.538Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjgrRzZRaTA0SEFDTlVEeHhsbjZueFRXdHovSGRyMGhNSXVUd1dmeTlBdjhlTXZsRG1qNHdiOGF1bkVERDd4d0M0NFRHYmpSai8wMHRYRUdXRGZOWXV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYxOF8xMDU4MThfNDRfMjQ5YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDA0NGVkMDAxNDYwNTljODE1ZDI5ZTZmZWY2OWIzNWJiZjNlMjY4MmY1Y2NjYzZmYzQyNjliOTM0MWEzN2EwNzdiYmI5MGQ3NmFmNTgwYmEzYWMyNTI2OWFiYWI2MGVkN2RjMjE1YWM4ZTIxYTA2ZmE4NDRlMTc3YmRjMWUyMDY4OTEwN2ZiMjJjYjI2ZjU5YTFjMTgxZDRjMGJkMWNkNDBiZjdlOGNlMDYxNmQ2YTM0OGNlYWMzMDJmOGFjODlmMjlmMjgwMmMyMjgyZjU3ZWJmNDM5OTc1ZTVlZjAxMjk0NzZiMmI5MzhiMDUyNjA2Mzc2ZWYwZWI0OTFkNGMzMTA5N2UzMzFhNTA1YWU3YWY5OGVmNDcxMDE3NWY5ZGE1ZTA4MjZlYTBmY2Q0MGE1NDkyOTkyMmI1ZDMxMjcxZDVjY2M2NmZlYzgwM2UzYmE4YmJiODE0M2Y0YjZlYTI1MWQ0YzJjOTZmZTU2NzUxMjg5M2MzMGIzMzMyYzdhZWMyYWViOTVhOTVkMGE2ZGM2Y2YwZjM2ODE4ZGMxMWY4MTUwMzY2NzY2OGY2YWYxOGFkYmJiZDBiNWYzYTk1OGYxODYxY2VlNzVmMjAwYzdmN2FhMWM0ZGEzY2JjMmUzZTM2M2RkMjNlOTdjOGQyMDNhODIyMjE2NjBjZTI1YjMwOGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WdTkrjcV8oinMw0ZHuLsVp11mso0hFrp2uKbrNilez118U9CL10MD-35pEofN8pPTYb13aDROIGTT5T_MX8SLA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220618_105818_44_249a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.543Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ii9ETFFXZW1pRE5Yc2pxTnh2ai9zZnB4TENCVEtON2lHMDFKSTNxN25TdWFxWVRYRkNUWE95eFNRZkd4K2I4YTZweFY0Y2JUSUhTWFFCWkplU3ZmeHRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYxOF8xMDU4MThfNDRfMjQ5YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTI2M2ZiZjEyMjgyZmM3NjAwNDk3MDgwNGQ3ZjU3ODhiODMwM2E2NWEwN2YyNWJkZmE5YmNiZTZmMDhjZTUzOGEyZTk2YmFmZGU3N2YxNmQ5ODA5ZjM2YWJmODY2ODM1MWU5ZjJjNWU1YTk1NGFhZGYyNjNkMjE2M2E4MzUzYTY5YTE5NzgyMmNlNWZhYzE5NjBjYjc1Njk5MDZkNjdiNzMxNTQ0MjQyODMwNmRmMjQzYzk0NTBiOWZjOWI5YThkMDFkYjY1MDkwZmNjNTMzZjBiOWE5MzM3ZjY0NTlhMTA5ZTNjMDE3NWU1ZTAyZDQxZDAyYzFiNzNlNTJmZGE2ZjkzYTZlZWNkMzg0ZDUyMTNhMGM0MzhhZDEzYTllMmM2ZGY3NTgzZWEyODc0NDNjNWFhZTcyZjIzYjgwZDQyOTM4MjJkMTE3YjYzODRmZDZiZmE2YTQ5ZmZhNDVkZGI1ODY1YTYzOGY2YTMyMWFmZTQwMTQ3ODQ3M2IwMjEwN2MzZmQ1NWI2YzRmZTU2YzhhODMyNGE0MTA3MGQyZjY1Mjg5Yjg5ZGUyNmM3NWU0YjFlZWU3NDk1MGVjZTY2MWNhMmJjZmY0MWQ2NDg0ZmJiNTJjY2U1YTQxOTA2ZTA1NDA4NzQ3ZWY1MWYxNzdlMDJjODNkYWI1YjQ5YWViYjQ1ZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gjQIbC0c_SDWQjkvNWeNt9_I0uwl1e11uFvDn8z5GNTgTsUZI9R0BG51kNMGxeX-LVJJQxYusWfjRy6M4sDDOQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220618_105818_44_249a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.546Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVPNzZ1SjlFSTFzaHEybUVldC9MNStFVFVEemx3ektqdEdVWkJjWXZtblUycFJhK21hejd4QzRxR0RnWVU2N2wxbDd6RTlISkIwa1ZNb0J0cWUxWGNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYxOF8xMDU4MThfNDRfMjQ5YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDJlYWRkYjAzMTAyNWVjOTM0NzBiNmM0MGExZTUzODMxZDI1NzY3M2UwMzYxMDFmMzdlYTU4NzFkZDRhMTcxZjNiYTJkZDA3ZjhjOGQzMzY4ZWYyYTMzZWQ5N2MyMDE5MjBhNGI5ZDg3NzIzNDBmZDJiYTk0ZTI0ODUzNzg3MGU1NDVjOTlkNDFhNDZlN2RkNTRkOTEwZDhiYWQyODkyYjNiYTBlNTViNDIwZThiYjljOTY0NjcxZmI5YzMxNjQ4N2IyYmRkYmVhNWQzZTYwMzlkN2I5ZTQxMWY1YjY0MDNhNWZiZjc2ZGRkNjQ3OGZmZGRhN2IwOGRhMjgyOGUwYzU4ZWZiOGY1OWFkY2IwYTFkYjdmNTczYjYxYmExNDNiNDhmMmM5ZjQwNTQ4NTFkMDRlNjk2MmExYjc1MjNlYTdiZDc3Y2VjNzU2NzZlZGEzNmRmOGNmZWVmZGUzOTljMmQ0M2E2NzdjNWYxNTc4ODQ5ZjA4MTAwYzczN2EyNDZmYjU2YWE1NGZkOTc5NmExZTM4OWIzZGU0Y2EyZTAxMzlkYTIxNzYzN2MwY2E1MDhkNWI1YTMwMTg3MzhiYTRhMmRjMzYyN2Y1ODNmN2RkZDU0NGE5NjZiMGQxNWRhM2JjN2I0ZTJiZGQ5YmE5OTIxMGM4ZjJjMjBkZWUwYjQ2ZjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yZWTp586NMiGkDLrFGU4v_ToR1xSIspZtj9URUFpwtZ9PmsWEFsVVXaBlC6LYkK-taZU8fIKavifnSBkPwVOtw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220618_105818_44_249a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.550Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InhQWUJRbkJrL21HWm5IaFBUbzVjWGJob3d4MVc4cE5qdW9ra2xFVndhN201VHJJbzFFQU1WTW92ZXFJNkN4QTlSMER1cEw2bmxTTzhaYmRNa2hmNk1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYxOF8xMDU4MThfNDRfMjQ5YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTE2ZDYxZTIyNmMwZTFhMDRlOTIzZjc5MWQzZmU4NzVjZjNmNjU3MzExMmFlNDE0YzRiN2E0MWZmZGJkMTFiZDI5OGViYThjYmE2MmFkMzc4N2EyYmRhMTJiNmFiNTc0OTgzNGI2ZDEzMTYyZDkzYzBjYWJlOGQ2MjIyNmVlY2U1ZTU5OTM3N2ZiYWEyNzM2YzczZjAxOWIwYTVhYzkzZGQ5YzdkNDRkZGZiNTlmYjQxNjEzMmNlZDllYjg0NDllOThkYTVhMzg0Y2FmOTNlZGJiNDZiMDQzNWFkODIzMTMzYzliOWI1ODI5MTI5NWE1MzdkM2YwODNlY2FmYTVhMGM1ZmQ2MDNiZjZjOWFkZjg0NTBhNzBjYjU1YTIyNDJhYjczNDIyZWU2YmVhYTRkZDUzODcyMGExZDJlZTVkNzM5MWJjYTU3OWQ4ZjAxN2MyOGI5MTc2MWM1ZWQ3NWNkYjJhMDc2MDZiM2Q2MTRlMmU0MjllMTUzZmNiZGE3MGQwZmY3OWQxY2EzZmVlZTM0NDBlNTEyMzI1MDBmYjRkYWYxOTc5ZmI3OGZmYjZmYTZjN2U0ZDZkYmU1M2E1ZGQ0ZGE1ODMxODQ4NmRiYjYyMDA0NDgzYjdkYWJhM2QyOWE1NDg5NTUzZTU5ZmRmNGRmNjUzNzlmNjFiNzhmY2U4OWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7Im7W-Uafppm5UIBaThFSWU5uf7EWrHRoNVVTbFcyDHQ4lGWT9ObXtaUNjdvwSxVu6tBfcT3choPxYlX9pvIBA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220618_105818_44_249a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.553Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhEY3Y5elhPdEt5U21IUzlsYmFicGZSUmNXM2xpN25PcUlMQkRuZVJ5Ym1qSDRqTWZvVS8rS1BEMVJPOVhMOXlNaTcrUnRKd1JaRVl2SGwzcThSaGRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzdfOTdfMjQ4MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjcyNTJmMTIxY2ViMDI4ZmM5NmU4MDdlNTk3YTAxZWM3NDE0YzM2YmRmZGE4M2M1YzljMWQ1ZGM1MjU1ZjhmMWNhOTQ3ZjFkYzVhNmQxY2RjZjM0MzAwYTMzNzBlZWI3ZDYxMjRmNWUyOWQwMjUyZTZmMDI3MDg4OGNhZGE4NjNkODJkMmU5ZDU1NGNkZDdiZjhhNTU5N2RhNGNlNWFhNmE3ZDNiNDEzN2FhYTQxMTFlYTE5NjE4OTJmODUwODExYjNhNWQ3OThiNGE4ZDYxZjM0OTdkY2ExZTExNGU5OWMzNmYyZjQ0NGFkMjRlZjE3NDM0YzVmMGM5N2IzMzMxZmRhNzQ5MTQ1N2JhMjA1ODlkYjBmMTMzNWM4ZTdkMjUwZGE4MTJkMTZjYThkNDQ0NjFiM2NjNzFlM2M1ZTE3YTFjNDE0MWFjYTJiYTM1NmY1Nzg3NDk0ODNlOTA3ZjhlOGQ2ZWJjM2Q0NzY5ZWEwMWY3ZWM5OTY0YzhjNjhmZTJjNTU3NTRjMDJhMDE0ZjQ2NzM0NjQyN2E5ODMwMWUxNGU1MzFlNGJhOTEzODc5MDdjYzAwYWQxNTZiZjlkMDVmNThmNTkyZTZlNjliYzAyZWQ3YTQ4NDEyM2UxOWM0MWMwMmM1OGIwNDA3NGQ1YWRlNmFiNDBkM2FlMDU1MWM3ZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.0RgJSf7YYZ1F5Vz4ll4RA0scYWSe2kdxf_5o5gnok62imot-roer4cmZ81cbtYUCtSp7P5q3KJc2h3dQbNCqaA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110437_97_2481_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.556Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZ4WHQvSndsUm1OV1BHSlZzWGN0eXJoSHB5OE1pNDZYVzdLOG0wVEc3dW1mdXpEL1MzMFFHYUdkL3RObEhoT25IdkoxZHgrdzRvUDUrQjZ1RkNHcjdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzdfOTdfMjQ4MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmIyMDE1ZDdjYzAzOWIxY2E1OTIzY2RiYzVlYjQyYjczMmM3ZGYwZjNkNDk1NWRmN2MwZmNmZTY5ZTFiNGE1YzY1YzcxM2I0MmI5YjczZDJlMDMzZTE4NTIxY2VjMTAxNDQzNWQzNDExZjNiYzAwNzAwMjJjMWYzZDE1YzI1ZDExYTMxZGM3NjI4YmM3NTllYjliM2JiZjU0ZTRhNGQxYWI1N2M3Njk5MjFiNDA3MmMwNmJlNzJmZmNlZGQwZmY1Y2YzMDViZGVhYzY5NThkOTY4OWNmZTMwZjE5M2Y4ZmI3MTBjZmI1MTQxZjAyNWFjYWE4NzgxMWVmMWRmYmJjZTUwZDhlNGY5YWY1MGE3MzkzYmIxMWEzYmRjZGIxNzZmOGFlMThjNDNlYmVkMTdhNDAzMzRmOTUzNjBiNDY1ZmQyNWJiNGJhZTY3MDM0MzcwMzIyNjcyZGU3Nzk0ZWVjZjM1MzZjYWRmMWYxNGVlMjMxN2U1ZDg1OTUzMDg4YmUyYWFlZmQ2NzhjZDdlZmY4OWFmNDUzYzdiNDYxZjlhMjI1YzdiNDkwYTNjZGQwZWEyOWI5Nzg3MmNlZjcyNzcxYWE2NWIyMTA1MjE2MGM3YjViM2RiNGZlNjlkYzQxNmRjMDNiYTBiZGQzMzU4OWQ0ZDg4MzBmNjk1MTViNjY2YThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zy4wtaKGlohjhhQ6_7YlbttEv5R9ZixTkj7VxEUFUgknt6d0pMSaFIHGJaYKgIMfzimUX5T6_DPf1ItBxF107Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110437_97_2481_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.564Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjFlTUhnQWhQWUFXSDZ5UnZLSWVHVmlIbE5HWXY0SVNXZkRsdmtTdDV4bmZnNE45NU80blN2cG8rcmtLbzRRYWhZdFhFb0pmUklicjU3aVVJckdOUnVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzdfOTdfMjQ4MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTNlMTZjZDUzMzU1OWVhNDQzMmZmNjBlMTdlZTM5ODUwN2FmN2Y5YTI1MTgyNmM1YzczM2QzODI5MWE2MTM1NzIyN2EyMTdkZTFlZTQ5Njc5MzhhYjI5ZjhjNzZiZjY3ZGJhODBkOWYxYmNiZWJmNGU0NmFiNmM2YmJjODVmZTVkN2FjNDNjNTdmZjBkOGY4Y2VjZTU3OTM3NzZmYWVkODVjMTgyMTg5NjAxNmYwMDgwZWI5YTE2MjI5ODZmYzg2NTQ5YzA2MTQyMThiMzI4NTRhZTRlMDllYzY5MjU2OGQ5MjA2NDk5ZmQ1MDYwNmQ3ZDc3MTMzOWRmMmZkY2M2ZGFkMmZmNjY3NDRkNGVlOGNkOGZkMGUxNTk1MGNjNmRkNTk1OWZhMmQwMTgyZDAyYjY2M2RhNzE1NzFkMWIzZGYyNGI2OTk1ODk3ZDNhOWYwNzc2MzljODYwODI4NTExOWExMzdiOTQyYTBjOGQ3YmVlZGNmYjg0NmU3MjljOWY0NzA1ODBiNzIwNTg2ZThiN2Y2NDA0MmY2ODA2NmE0ODUxZDQ1OTJlNWU0ZDJhYWVmNmZjODczNDgxOTdkZWYwMDk2ZjRlYzZmZDE3NWE3NTU4MWNkNzQ4MzAzY2Q4ODIxNmM3ZjI1MzQxMTZkZTlhZWNiYWYzZmE5ZWRmMzk3ODhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RYpTFh3u9lNKe5uAab47KuB06EJRDhPGeG66UR-TbsSENPm8jm4asz6zVjXrMnJtCa9vQaaChyIwZi-HXzWHuQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110437_97_2481_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.567Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNPc0xLMnlieW9NRHgrZXR0dGt4K2Y0TUNnUTNIR0hUenZ6c0FIRXNRZTY0VEtMZStidnM4TDV6OUo3WXJtTVp0dDR5M25QblJwV2dSR2FUWGtwYWx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMTA0MzdfOTdfMjQ4MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWUyOTJlYzE4MzA1MDFkNjk2Y2I3Yjk4ZGE4MTY4MWFhODhjYzZhM2UzOTNhOGMyZWU4ZDNjZWM4MmY3MGZlYWU3ZDQwNzZiZDhiMDdmYmY1YTcwMmIzNjQwNzM5ODkwYTNkOTUyMGJkZGRkODUxZGJhZGJmNzkxYTIzZjIzMDhjNDU4MDEwMGEzODQ2OTkzYzRhMGVlNzQ4OTRlZTVjZGFjMGIzYTkwZWEwZTM4MWNjMWQ3ZjAwZjQ3YjYyMGU3OTk4MjlkZDk5MzM1ODE1ZDA3YzgwMGFmM2Q0ODhjZGE5ODIyMjcwZWNjMjRjMGM3NGUxZjgzZjUxODI2OTNjNjNmZDI5MzliN2RmMzEyNmM1NmYyMGNlZjFmNjQ3NWZjOTY4YTQ2NDg4Y2RmYjRiMzUzMGI4Njk3ZjQ4MzFjNDI4NWM3YTE3NTAwZmRlNWE3NzhkZWE1NDNlNTQxYzdmMjVkOTNhMjA3N2M5ZjVhYTE5NjUzZjU5ZmY3NDI2N2VhM2I3N2Y5NDVkMjlmNjYyM2ZhMzY3NmFkYWQwNzExNGEyNzJkYjk5YWJhNzcyOGFhOWFmOTVjZTdhZjQzNGVmZTdhYzFhMGUyYjUzYmY2MDEzM2M4MTlmNTI0MzhiYTAxZDVjYjU0ZDQ4NWJlYmE2MmNkOTU4ZGYwZDI1ZmRlYjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IxjMZKSjFb428qdAHfws9mCbyqFVVtz6QMY4179G-6cUali36PJlg30DQwMhGNDZWQWkwCVOrxxUDghzrxqnmg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_110437_97_2481_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.572Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ikx4ZG9BZ2FOUC9LYnpQb2NPWUs3Y3hyR1p3OEpqTCtINUwxaWZXdkpCYXdjdVVpNjNDeUJwQWc5Rnp2Y2pFY1A5U1R4NEVPWks5Q3ROSTZ0a1J5TnpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMDI0MDBfMjRfMjQ1NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2JiNGM0MzY0MDhiNTU3NWY1ZjY2Y2EzNDBmMmE1NWZhYTI3MjY1MGUyNGE4MTlhOTc2MmY4ODA4OTRmM2I2ZjA5NTBmMDFiOWQ3MDZmZmY0NWJkNTYyNTE1OWM2YjNlNDEzNDY2ODExYTQ3N2M1ZDA2YzBhOWRiMzRkODA3OGE3NGI2ODZhZjJjNjEzOTRiOGFmOTI2NTRjYmQwMDkzY2EyYWIwYmI0MTdhZGE5YmQ1NThkZmJhNzhkNWViOWQ0Y2E1OTE0YjM4OTM0MWEyNTA0N2I1ZGJmMzljNmRhOTFiODE0ZWY3ODNjMjliMDQzYWU5YTA5ZDhhNGMwYWRkNjJjNjI2MzAzZWM5OThjYzYyYjBkNDAxNmM5YzI3MzlkYmJjMDU3Y2ZjNjg0MmY4MzQzMGRhMjdhZDVjNGEzMTA5ODM0NTc2ODgyZDM1ZGM5ZTU0ZDE0OTVjNzI0YTMyYTg3NjA2ODczYjkyZTQwMGVhZmRiZDdlNTU5ODIyNDdmNTIzMzU1MWY1MzdhOTAwZjU0NThkYjk3YjNlMmNiZjhkYmViMzA4NTBlZjBlOGQ1ZjZiY2NhYzViOGIzNTI1YjdkY2E3M2NmZjc2OTA0MmFmYjYwMTM5ZmQzODVjMDRhZTYwZTJjZjIwYTRkNjU5OWU3ZjgzOWJjYTJkMDE3ODhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ml68E9BDKrbju_FEYLTt-VkBeuabAzpPcMnQ91FEIgNOHSxQjnEYtinwFTJbmPZ1PbSXOxHQwrSkgdXKcQQLcg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_102400_24_2455_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.575Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkY5eDloVFY4SUw5aWNxeldrSGdwaWpMWFljM2NxVFNBejJnY2pGY1BDVmRDckhvRGNLa0FrSHgrZDNqTXE0RjRNVDhXbXB0U1FvZnBJZG81amNJdldRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMDI0MDBfMjRfMjQ1NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQ4MDhlNjlkZDEzYzdiMTJiYzQzNjE2NTM2YjQyNGVjNTU5ZDgwZWNiODdmZmVjOGViNTQ0ZTJiOTg4OWQ0NzllZTZjMmU0YTk3ZTRjZTI3MGNjMzJjYzU2MjAzOGM0ZWVlNmIyNWIyMzM2MmQ4ZjU0ZDdmYmJmODNjMGE0ZTIzMjQyOTgwMTRhNGY1MWRiZjU0YjhjY2E3ODE5NWQ0NmNiZTEwNTU0MzllYjViZDhiMjA4OGNmN2UzN2EyMjBkZGIyZmQxZGI2M2IwOThmYmU0Y2M4YTRiNDdjMzE5YTQwMjI4NTc5M2Y0N2U1NDA5YTAwNjM2YTI2OWYzZDA2NzMwMzdmNjhiZTgzMmUzODhiYmIxYzc4YzJmNjNhMWQ5ZjZlN2ExNjQ4YjJhNmRiYjFlOWU2YjczOGUyOWUyYjg4NDdlYzhiOTQ3NWZkODk5NGE1N2QzYjAzZjVkNmNiODcwNzJiYjYwNTg2YzdhY2FiOWU5ZDg0ZmRmYjk1M2U1ZDY3MWE4ZGFlYTk0NmE1YWY2Yzc1OWQ1ZWYwM2VlZDVkZmVkYWVjNGIwM2NjYzM4NjE5OTgwZGNhODAwZTEzYzM2ZjkwMzI4NWJmNzllZjhhNGU3NmMxOTYxMjE2NDNlNzIxZDhmODNlN2Y0MzMzYTI2ZGY0ODBlNmQ5ZDIyYmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.y7iKN2zkFDZiLJVUaSdP4NXhKjHpxm7yt7vbmJ6eR2Mg31rTq8mu72Gw-I6f1ONqe3stWXhefCZ6b-PyPvKvDQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_102400_24_2455_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.578Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJXeW8rdlhZZENDenJ0SFcweGN2VUpDbVdxamxsRUVpZFR4UGU2ZEtPcGZqSnJHMi9XM2dXeTRJTFZvUlhBWGtaY2hIbmxxSEs2QVVWNFNzaE02ZElBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMDI0MDBfMjRfMjQ1NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzcxMWMzNTkzMmQxMmExNDE0Zjg3OGYxOGVkMjVmNThkN2QzMTNlNGI2YjJkMjAyNjkwZmI1ZmU3MTZlZWVlZTJkYjdlMjJkNTU2MWRkZGIzMTZjZWE3ZDcwYThjMTU2NzUzNzI4YmY5MGZhYTBhOGUwMzY3YWI0MmQwYjRlYjgyNzZlZTEwMDE4MWJhYmE1NWFmMDNiNzEyMjE5NjlhNWY3OTI2YmMxYjlmNDY0NDNlOGZiNjY2YzdjM2I0NmI0MjI4ZDk4MzYyYTI3ZjcwY2I2ZDhhZWE3OGMyOWZlZDVmNTYzY2IzZDJkYjFmN2RlYjI2YmNiMjlhYzM0ZTVjOTIwMmIyMDRhNDk4MDU5NjY2NjU4NzdjNzRmNGJhMzRlZTA0MGU3ZGYwMGViYmJiMjdkNzliNDY3YzkwMjU0NjMwZjFkZDQ3YjE1ODkxYjBlNDIzYWY4YTMwY2JiNWIxOTRkMDJmZTIwMGVhMTc2Yjg1OTFlMGYzNmI1YmVhMjAwOGM4Mjk0YWNmYTViN2U4OTVhOGViY2RjN2QxYmM4NWY4YWUwZDcyMGE3NzRhM2UzMjhmOTQxNjUyN2I3NTNiNjIwYzUwMTYzN2Q0NjAyNTgzMWU0NDEyNzVkMjliNzRkYzYxOWU1N2Q5YTE5NzczZjY2ZTJlMTM0YTc3YzQxZTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZIK37v5OgxviEhaTxEir-4chIS_86ufzJBMCLEL495QjAKh4njflTD1biyHg3GBkafoK26ac23vtt2A2UHLEjA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_102400_24_2455_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.581Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJzbEg3Wmt5RWo1eGJxVUFlQnN2RUpCU0cvNTEzMHpRK3FvQWdMb29Kb002TzhWb3o3UHc4TEY4YkIzaDE5WEY3WkNBSW5VZmo4UG0xUE05VDNSdndnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwOF8xMDI0MDBfMjRfMjQ1NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2Q5ZDllNWNjMjBjYTM5MjM0MjhmMWE1ODZmZTdlOWY3ZmM3ODM2ZGJkZmZjYjczZjc3ZWRmMDNlMGJjM2Q2NWQ5MDRmYzc1YmYzMTIxZDRiYTA2Y2Y0MGJiZGRlODhjZmU1ODdiNTcwMDg4NDVkNzA0NDMxYjZlM2RkMTI3ZGJmZTQ0YTM1ZmVmNDcyNWIwN2JhZWI4YzkxNzM0MDgyY2RkZjYyM2E3ZDM2YjlhMzUwZjI4YzdmMDg3MmY4MDg4ZDEyMTA5YmEzYmRlMjlkMjNmZTNjMzc4OWZkNThlNTcwNzE3ZjliNTBlMmIxYzczOThlOTRjODkyNTFkYTFkZTIxOTE5ZWJmMTM0OWQ5ZjI2YmNlNjE4ZGE1N2EyMjgyMjRhYmNmZjExZTZjNTNkNzBlMzZiNWUzODc0YjQ5ZTQ3NGM5OTczYWQwNWJlZjcyZTFhMzdkNGVjMDM1NjYwMjAwYmI3MmJmZDkxYmU2MTQ2YTI2YjZiZDM0NWQ5OTkyODQ1MDFhNDRkNWIxMDE5NTc2ZWQxMzM1MWQ1ODdlNjJkNGM1YjRkNTE4MzRkMmVkZjE4ODlmNjViYTQ5Y2E5OGZkZmMyM2ViZmU4ZWRjNDhiZTVmNzI1ODg1ZGRkMjU3NzUyOWI5YTA5M2ZjNmYzYWVlZTgwNWY0MjI2NTI1ZjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YY8hJB5u7rskPhL0h2EG5Em6f9dgkLRLJ9_1uwbSe_IKrlVCp7StCxxplZAlAw1yOcCr9YE6E9TsnWBAtF9Gqw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230308_102400_24_2455_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.585Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ilk0OFozSncvRitvZ3hqU2dyMERMR2dRV0s4Vi84bjB5a3NLSjFaN2FhY3hOTFM1U3htNnYxRlJmYmp4SWVDWERweStuNFE4bUZpMGo4QXR0bm43R3dnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExNDBfMDRfMjQ4Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE4YjRkOTljZjgwM2M5ZjM5MzgxOGEwNDQwNjMzNWJmZGMxMTZjOTQyMDJkMjc0YmM4Mjk1NDI1NzhiNGYwMGQyZWZmODAxNWFkZDNjZDBmZDc2NzkwZTc3Njk2NzQ3NmY4N2Q5MmIyMGZmNWM2MzU3YmQ2ODNhYWYyOTUxMGM2ZTQwMzIwZDYyMWY1ODJiZjczZDQ5YTU1NDFhNGI0NTkzOTE2MGYwYThhOTUwZGY3YTg1NGY5OTQ5NDQxZTA2MWFiMjU5MTQ5YjU5NmVmNWY5ZGQ1OThlNzFiOTVmYmRkY2YzNmE4ODFhZTI1YzQyZmJhOGZhZjBiNjkxM2I2OTE3NzM2ZjYzZjM4YzIzYTc1MTI3NTRlMmVkZWU0Y2YxMGMxZTEwNWJjZTM2Nzg0MjQ3YjRhNWY0Nzg1MDQyM2EyMjAxOTcxMzE4OWZjNjczN2ExYzk5ODFjMTFhZTNiYmMwZTUxYTQ0ZjNlNzNhNzJkODkxZmUzYTc4N2E5M2IxNTM0MWZmYmUzN2UwYWE1MGU2YzY5ZWM0YzZhNmI1MGIwMzZlMmYzNmVjODQzZWUxNDJhNWQ1ZDA1MWE3ODEzOTNhMWVkMDZiYjhlZmJkODYzMWQ0NzNmYzlmYzg3NzM1Y2ExODAwMjExNjEyNDk2YmQxYTgyNjBjOGMyODlkYmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9peRUd1zJKcrpd3fz433nBbO4ig6jKLigqEDSlSwVGH1O-7y0MVos_lSCuPcpxX2ZyrlBSy3HlpO4ApMtovJRQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111140_04_248f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.589Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJZaVNTbktHUUVmNldwSExlR2JmL3B6SnZlNlU0ckdqZ3AzK3hTcksrT3RLSzV5T21VM0Rma0lIN2haSnMyQ1cxSHFHTzQ0cEUvZ29PZHRRUDgxUkN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExNDBfMDRfMjQ4Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWUyMTY0OGEwY2YzMzAwYzk0MmE1OTY3ODk4Mjc2NmFlYjhlZDdlNjViZDQ5ZjBiYzlhNTA5YWU3MjY2NzAzM2NhMzk3ZDk5NDNjZmI1YjY2ZmY2OTQ2NTRmM2U5MTdmMTkzODUxNDJiYTQ0YjlhYmM5YWU4MDIzOWQwZmZkOWUxNTE0YTZhMGNlNDU0MDA1Nzg2NGZjOWIyYTc4ZGM5MTJlODhmMTA4NmZjMzU0MTMzNzliMmM2ODMyODY4NDY1YWNiOGFlOTViZTA1MDZhNDc2ZGEwOTI4MjAzYTgxNDg0ZjQ4NDFhYjIzM2ZkZDhkNGQ2NjNhYWUzOTQ1Y2RlMTc0OTk0MWRiYTFmYjIzYTgwNThlNTg3Mjc1YTE3NDg4YjJiM2ZhYWQzZmMyZTY4N2JkZTI5ZTg3MTQ4NmEwOWU0NWNjMGE5OTI2OWQ0MTYxM2NmNTc0NmIyNzVmYWUxZjIyYzNjY2ZmZWJlOTYzMWI1YmJkMTgwZTMzMzM0ODQ4ZTY0ZTk3NzlkMWZkOTZmMTQyZjEwMjFiMWRmMDU0MjlkZDQ5ZTc3Mzg2MDAzMGEyNzc3MDRkNjVjYTBjOWNkOTYyYzJjYTFkMzFiNDE4NWE0ODA2NGJhNTdhMTg1OWIzNTg3N2ZhMGQ5Y2M2YWQ1YWRiNDQzZmI5YzY3ODBmNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sR8kxufzjEOi8YXR-7KLh5ou2bN7Rxq-GCRKc0szxn2lVlHZkztFa2kG9q25f-dGCBFdMWnpRQsBKqVyxRJXxA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111140_04_248f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.592Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJ1SjN4TnBCVVdlcmZHTUtWVXVVRXdXRStxcm41MkxXSTBFWFloYWh2UHhRMnQreWo1QngyTERjUVVwUWsrL1FhMkE3NjFZalZOcTJFd2YwT2p5T0ZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExNDBfMDRfMjQ4Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmFjYzg4YWY3YTc0OWMyM2VhMmE0MGUxNTIzZjFmNjJhY2I0ODJlZTM3M2EwMmQ5YmJkOWU3NTBlYTAyMWE3OTU0MzJmZWRiYjZkOWUxYWRlOThkMDYyMmEyMDVlYjgxNWU4ZTkzMjQwZWRjYWM5NDk0YWU0M2EwZjIyOGI2ZDg3YTZkM2M2NzIwYWU0OGIxMjFkZjY1NzI0ODIzYzIxYjk5MmYxMTI1ZDRhNjc0MzI5OWQ5NmQxMWMxNzQyYjExYThhYTBhYmMzYzU4NjQ4Zjg5ZWI3MmE5NWQxYWY4MjdkYzk2ZmIzNzI5ZmViYTBhN2IwN2FkOTFiN2IyMzJlNzg4NGU2ZmRmZGIzZjIzMGM1ZDI2ZGQzM2JjNDkzMmM3MDYwYmJkMzdjMDgwYWFjMTQ1OWQwM2MwODEwZDhhZTNhMDRkYTU5YmU5NDg4NzUxMGRjZWE5MmYyNzdjOThkNTk5MmRkZTU5ZDNkNDJiOWRjOWFjOTdmNjI1M2VjMjY2Y2M5ZTliZWU5YjFlNGYzZjM0ZDA3YWQ4NmE2YzZlMjlmNDk3NjRhM2EzNTcyY2EzZTQ0MTRlODlhYWJhOWJhM2FiYzU2OGM5Mzk2NzY1MmM3NzBkYmY1ZDQ3OTg0NzBiM2MzNTJmMTFhNzRjOGRhZWExOGU5NjI2NDhmNDM2OTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UdeUzg1G8080aZwWvMumMwwTjFBJH1gv8jXgt1dt6PnXMwfE7F51Mj9QmPkgltFoYmT9LQ_9d3bpTNzjC6Wx9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111140_04_248f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.595Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im1HVSt1YjdleXhKSjU3YVYxQUthQzNBZi9BcDBCazNXbEo0NUJHRlpEZHFvR3kyUE01Tjk4UDE1T0s4K2dFWnA3c215ZGwzOTA5a0J3RHBDWnBmenJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgzMV8xMTExNDBfMDRfMjQ4Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGU0YTUxM2MyMjE4OGMwZDM0NDg2NTAyODNkZjU0YmQ0YmM3NjM0YjM1NTgzNzYzYTMzOGFiYmEzZTc4OGQ5NTlkMDZhYzA1NjA0NDUwZDE0ZDkzNGQzNDllMDZkMmIyMzlkNmYxMjM0ODRkZWUyMGVmN2M1NzhhNDQ2ZmViNTFkMGYwZGVhNzlkODAwMzIwZTVlNDE2NDE2ZTQxODY1NmFlMWM4MTYyOGM4M2FhZGZhYjg0MjNmNWU3ZGNhNTc2ZTNlMDEwZGQzM2E5ZWY5YTA1ODU4NzU4NzNjZTBjZDA2NDFlNDBhMDAwMjBiYjEwYzhjMmVlYWZmYWMwMzZmY2ZmOTEwZDcyODcwNDUwMTdiNGExOTZmZTMzY2VhZTU1ZTEyNGJhODk4NDEzNzliNmQ0YzJiYTQwYmM0ZTk3MTRmMDMwNDk1YmQ4NmU5YmFhZDQ0NmE2NTliMDUwODNmMzlhYjcxMzUxZjE1MDMxYjczMGQ1YTRhMTNkNThmZmY2MmU2MzkwZjk4ZjliZWI2N2JkNjY2OTc1OTE5MTU0ZGIyODAyMzQ0NWY3MDUwOThhMTQ5NmI4YjgxYjIyMzYxZGU0ZGNhNDU2MzkxNGUyZGVhY2U2YWUyNmI4ODM3YWE4OWMxNmU4NDJiOTczYzM3ZmFiOGIxZjg3NjBlMGU3MjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.L6uKR_wutZu61U1u-Y4lIKIwowUcQfQzVNt5f1uoJhy3QdmZt11g59VNF75RWSuSFYbgqXu-RodD7v63IZTBtg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230831_111140_04_248f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.599Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjNnTzNpN1RTbHRmcnE5dFhsSXFtSUl1ei96WVIyT2lhUndQcndVUmNURGhGcWlNSE9wdkhLY2xna1NVSnQ3MzJsc0p0eUhKZTZ1bW1TRFE1dDEzbzNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDNfODhfMjQ2NF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDFhOTgyODI3YjY4N2Q2ZmEzM2M3MTliMGZiNGFhODk3MTdjMWJmY2I4ZTlmN2I5NDhjMTNkNmIzNTUyNjYyYjdiOTgyYTljY2EyMDU2N2EyZjQ1Yjk5Nzk5YWEzYzFiNWI4MmQxNDA2MDQ2MWY3OTNmZGNkMDc0NjczNTRhNzczM2EwM2M4ZmI1MTA1MDBlZDAzZjg2YTVjYmI0ZTYyZTQxMmJjZDg1MzM1ZjdlYWIxMzU5YWE1NmJiY2YwYzBmOGFkM2U5M2I2MmI1ZjU3ZDM1NjEzMGZiNzEwMDQzYjJjMGFmM2FjMGNlMzNkNDU3ZGE2YmNlZGZlOGE2NWZjYTIwNDg2NGViZmYxYjkyMDc2OTNkMzIzOWU4YWVlYjQ1ZTg4NGRiNzYxZDgwYzU2Zjc1Mzk2M2NkYzQ3MTEwNTM2M2IzMzljNWU0M2Y2NjRmZjIwZDAzODk0ZTIzOTA5ZjRiOTQ2NTBhNTgyZjUzOTVlNzNiYTk1ODk5Y2NkZDVkZDA4N2FhMDMyOTBlNDRjNjVlM2QyNzg0YjkyM2M5MTM2OGU3MmVmZDljMTgyZmM5ZTQ4MmMwYjFhNmM3YTcxN2I5Y2E3NjVkMTNjM2RkMTQ5M2IxYjA2MzdjZmE5ODBlMmVkNzU5MDg1MDI5Y2M5ODIwNjE5MDQzZDk5N2JjN2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.j9vEEkUYO6Z2k3LlsIvpi6AYqdUVGR_nG_pWR9GI58mTw9MvtqmuEZKSF42aNA_DtJHD_mK3z3wuv76wqBiJkw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102803_88_2464_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.602Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InJac3BCTUdWWnhyc0pQTVZNNG5wTUhpTVlORVBZOWJ4eGpjTUFyK0hhdjNHb2wrM2h3SnpqdFJ5Q0xOTUlaY2tIdVBjVk9qdTIyVEx3ZGpPdzBRVmh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDNfODhfMjQ2NF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzJjYzRlNmI2OTVjY2Y1MWVlYTZkYzkwMTA1M2U1MjQyZGZkNzM0NTY1YTM5OGI3MDA2YjFjZTk3MjFkOGQ5YzAyNTU3YjZjOWIzYjA2NWFiMmRlY2ZhZDIxZTQxMzk2ZWZhZDdkZDI4NmJlN2I2OGM4OTEyZGFhOGJkODliNDVhMjFmMThkYjJlMGMwODMxZDc4YTM1ZjI4OGVlOWMxY2E5MWM2NzQzNTM5NDgzYjRiYzM0MjY5YzljZjRlNWM2YjkzMzlkMDYzYTg1MWYwOTRkNTRkNDY5ZmE3YTUzZjBhZWMxNjRjN2U2MjM1YmNhNGI2YTNlYzczZTllZmI2ZWRhM2IyYzIwMmViOGI5ZmIxOWQ3ZGY0NDIwZjIzOWQ4YTljYzFlZDIwNDdjNThiMjM5ZmM5ZWE3MmQ4OTVlMjhmNDc0MmIyMmVjMjg4YTVlOTFiZThjZjYwZWQ4ZGZkMjE5NjhkODEwYmUxZGRiYzc4ZTQ4MTFiN2ZlOTg0YTRlYzNjMGQ4MmQ0NTc5NTkxMTRmNDY0ZWIwMWEyMmFjMGQxYTI1OWNmNzZjMzRjNWY2Yjc5NDZkYjZmMWI3NThkZDJmOGJjYjNiNjJhZTExZjIyOTA0MGNiNTM1YmMyNjkzY2RjNzY1YmVlYTk2MjkyY2JlNzQzNGU3YmJhMjliMWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pYuNbtyRR3_DRK5z7P_cIDymk6yt1b7tOLokjStxzLkvXV1cYESIc08_uILH2d6uZfRqbntNucMRTNzOq4gXZQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102803_88_2464_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.605Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImYvdHduTG1OK0FGNGhJM2t6TzFjQWFJR0RnRENsbkJhM0h6SWJSOTUzalFJRzR6R0p4dHo0Q1pwaFVnMGUvWFM1WmJCYVA0VU1jRDl0WERHemtVYUhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDNfODhfMjQ2NF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODc4ZTJmMjJmMWY2MGE2N2NlZjg4MjhiZmY3ZGVkYWVlMmZiY2E2MTQ3MmQ5ODVmZDMzMjdlZTVhOGFmOTU5ZjQ2ZWMxZjU4OTM1MTZkZThiMDIzOTNiNGEzZjcyZmRmMjczMjM2Y2JiN2ZmOGY5YTA3MDQyNDk1NTk4NjNkZDIwMzczM2NiZTg1N2Q3YTliMDM0ZTI5YzFiMGY1YzYxMmZjYTA3NjM4NzVmMmVhNzZkMDMzMWNkNTdlZmZlMWIxMjQwMGQ5ZGQ0YWYwMTE2N2UxOTgwMzAyMjEzZDY3NWVlYjkyMzYwNjE3ODJhZTBiNWYxYmMyYmM0ZGM4ZWJmYjY4OWU2YzgzYmZiOGI0MGI0Mjk4YzAyZGJjMDQzMmRjZGU0Y2UxYzg2YzY1OGYxN2Y3OTEzYjljNjcwYmJhNzRhZjcwZjJkYjc4NjNkMGE1MTVjMzczZmFlNzUzOTdmYzg1MDE1NWEzNWQzZDZjN2FhOTkzNGM0ZDYwNDhmMjBkMzdmOWRkZGI2MThmM2U4MWY2YjhhYmY5MmFmMDIyM2I1Yzk5YmRjYzhjOGY4MTI0YzMwYmNhYzFmOGU2NWRmNDY3NTZmZTQxMjNmYTA4MDI0MmQzODBmMDU0MjU1MWYyOWNjOGYyMjczNmYwOGRjMDkyODYzOWZjOGJlNjQxMDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._vFkN54217uK8gQfbNofSmSuALWFUCd4thyHtDcHpZO6oSXO7eRUWKeNjEuuLUSDUJ5ulEzPgYKFmcDCR5YGZw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102803_88_2464_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.609Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpXTXRON2dzRStMUWUvMkNJZ3ZwWit5YzFsNWJaWVc2K3V3VFd5dzFxRlpBY1p3R2tIWXBFVUEvb3drczA5bmt2SE42a0Z2YWZUemtBVk5MYjFJSlVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyOF8xMDI4MDNfODhfMjQ2NF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzhiY2FmZTMzMjA1OWM4MmM3NmU1MmM5N2JiMzE2YmRlNTZiMTU2NjIwNTRjODc2NGY5NjQ3ZTQ5NGZiMDU5YjFmYjlmNDgzYzQ2YWQ2ZjMwYmRkNmRhMGVkZGNjZjdmMmNiNmFlMTlkZjI0MGEzNzYxNjk1MDY0M2E2ODMyYmJhMTA2MjUyYjQxNjcxNmE5NDY2ODY3YTM1ZjllNTcxMjI1NmQ1MjQzMTNkNTE3YzFiYzMwMmM1OWJjNzVhYzQ2NTdkYTYxNTc5ODZkMGNmYzQ5YTU0YWI2ZWY5NTUzZjU2N2I3MTBkZmU3ZTc4N2Q1MDAwMmVkMjFiMTdlNzJiNmQ5ZDg4MWZkMzkzMGVmNTAyNDFlNGNhOTdmYjVjMzYzOTc4MWQ1M2Y5NDY5ZTc4Y2Y5N2FkMjZmNWZiZDUxODdhODhiMGE5MGYyOThmNjc4YzI0NzQxM2EwMmNhYzQ1M2Q2Nzk2OTdiNTAyOTg3MmQxYWM1NTlkNjBkMTM1MTcwODNkMjk1YjY0MmQyYjgyMGQ0ODY0MmQxZDA5NTQxNTdkMmMyOTMyM2FkOTNkMTYwNWM1NTc2YTQ2NzZmYTUxMDk0OTQ5YmQyYWNjMTYxMmYyNTQ5OWZkMzQwYTI0YjhjZmE1YmQ3MzhkNDY0YzJlODllMTk5YjQ0ZTk4ODA1ZTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.c4PPiRhvSgScv_PVDWtZM_I4T3v3V1RzZ4ngCMFvwkSP_N9aRNNuP1BKJ_W2XGeAlezkgP5SysSemaGEmK7v9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230128_102803_88_2464_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.612Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNXRHIvaEZPbGlveldHZjRNNHhVSVNEa1k4OUVMN21sYi9aSE1hMVJYV1pBMHpQdFdROTF4UldqdGpQWVlxeE1OMlN6UkRWaDdTQmwzUEVndWd4QUJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDZfNThfMjQ3Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWZjOTVhNzJkNWYyM2M0NDExZWY0N2M1ZjI1MDY5OGNjNzBlMzA1ZWU1ZmRkZDU3NWQxNmZlOTQyZTY2YmQ1NjVjNWRjZmE1NWMyMmE2NWQ2NDUyNDMwNGMzZjI1N2Y0YjkxZjY5ZTM4YjQ5YmJhZThlOGZlYzZhMDEzZmE1MWQ3MzViNGRjYjYxMDg4MjY2ZjNjYTI5MGQxOGM0M2NlM2JmMjlkNmE0MDdiNzRmNmZjNjhlMWJlOTQ2NmU3YzBlMmZlZjZhOGY3ZDljYjQwODAwOTg0MTBlMGFmODc2YjNlODM1NTAxZGI4MGNiMDRiMmM5Yzc3YTVlM2EyODUxMDBmOWZmYTU0YTBlMjJjMWE4MjIyMDY4YWZhYzRhZWFiMjE0OWY4NTU4MDA0OTg1NTU3ZmRhYmM5NzRhNTdhYjVjYzRiZmRhMTdhZjI0NGFiNzE0OWM0MzhjZjdiNDg3ZDJiN2U0YjJlNzFkM2ZhNGYzMmMwODNkMWQ3ZTgxOGFmZDU5NWIxZjc5Mjc5MGQxNmM1Y2RiNGFiMzNkYmVmMTkyMmFjNTU2YThlMGI3YzZhYmU5ZGRjZDY3ZGQ4ZTRjN2VlOTkyYWFhZjkwOTk1ZDAwMjhhODY0OGU2MDgxZDI4YTU5MzNjNWNkYjZiOTlmNTdiNjhhNTQ1YzY5ZDRlNjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hs-k1WZokBv3bhlhMPLHhbw6rqKeBx_ZExhcsNpLyv90FEC2EhlTqakmaAD-q5n0xXWpX5DBE_W2Iy-jLS693Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111146_58_247f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.617Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IldOSDhpdFllV2Y5MWR2OWk4aVBDTGp4WGk1NjFkNXRON1RSNDBtcG5iRzdFMm1zSTZhcFRZaUl3bXYyVm4xT0N2c0ZIM09lL2Yxa2hpaWJGbkVoRUFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDZfNThfMjQ3Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWJjMmEwY2I5OTM0NTBlYWRlZTVmNWVkNTBhODkxZTlkM2NjZGY2OTEwMThjMTA1MWI1Njk4MjJmZmU0YzgzOGNiNDQ2ZDA0YjZjODFlNDI2NzRkZTA5Y2M0ZmI2YjBiYmQyZGVlNjRhY2QzMzcwNzg2MTg3MWFjNThhNjUwNjgxMDlmYTgwNThhMzBlYzZjNDZhYTdlMzUzM2U2ZGJmMDJjYWZlM2JiYTRhNWVmMmI1NWU5NzM0MDk5YjNjYzdhOWE1Y2E1Mjc4MTVmYjE2Y2VkMGZkNzM0ZjUxZmY1MGVkODRlNTU4ODY4ZDdjMWY5NDBlYzk1ZWQ4MDA2NjJjNjFhMzAxNjljNzA1ZDk3Y2RiYTlhYzZiMzViZTE5MWU0ZTU2ZDRmZTBkNjllNTE4NDlmNWUzZDQ2MmVlZDdjZjQ1YjlkYmYzZTg0MGEzMWVhNTM3YjlkMWI5ODY1OTFmYzNmNTRlYWQ1ZGUwOTYzZWY5ZjNlMDIzMTc1OWJkMWFkYzMwOWViZjUxYjVlOGZiNWViZTU0NjgwZGI2NjRhZTYzZmUzZWViYjRjOGUyMWQ2ZDE0MjNjMzc0NDM4N2NiZDEzMWEzN2NlZDE2YTk4YzEzY2YzZGZkYTA2OTFjNTI0MzU0N2QyOTk0ZmQ2OWI3NmFhOTU5YTg1ZDU3YmNlMmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hA7ArUhp4EFNi3qz2LqY3Bs6dLea-kY7t5i5YZMw6nUEiDEBMDp9t_jIHylcJ-86LwbPKPsmKM_xjP6hqUJeCQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111146_58_247f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.620Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImZlNjVwdVBJQ2Ricm10YlMreFVvdy9NWnZiTFM2WTZSUXpsU2s2M21ZTlhRZ00zSG4vdWNaSWNLV0toaE5KQXFUQytRNFduR1k5RHJMUkV4ZStDWTF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDZfNThfMjQ3Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2EyZTY1MDczNzQwYjRiNjU3YTVhZTIyZmQ3NmQwNmFkMzg2M2EzNGJhNTFhNWVkMGQ5NDk3ODdhNTg4YTkyMzc0N2Q4N2U2NzRmMGExNWEzMDMwMjZmMzIyMTdjNGIyNGQzNDViMGExNmVkNTcxYjJjNTI1MmY0ZjllODcyYWRhYWI4OWEyOTg0NmJkYzI2NzY4NjAxMDk0MWIyZjU1NDc3ZmVlOWIzMmUxMjliOTA2ZWJmYjQ3NDIyNjM1ODc2YmNiNTA5Y2Y5MTkwMTI5ZjEzZmQ4NDRkMTVjNGZmNmMzNzZlNzM2YzRhNzgwNjZmMWE2Y2YxZmMxMTk1YWI1MzY1ZTBlYTJiMjBkZTkzY2NhNDc5ZmM1MjViOTU5YzI3ZmU3OGVmNzkwNDUwYmZlNWQ3N2Q0MDUzNDQyNjMyN2NhZGI1ZjNiNmY0YmVjZjU0ZjdkYTIyMDM2MWZiZTQ0N2Q5ZGQwNWNiMGFhMGUwYTI2N2VjYWU1ZTUxMDliYmEzMDIxZTAyNjNjYjJjOGQ4ODRlNWM0Njg5ZWQ0MjE1ZmU3NWE0Njg1ODQyNTIxNzhmNDJlYzNkNDliMjYwYmY0OGU3ODU0NTlmYmFjZDI0MTdjYTNkN2I0NGM2YWEwYjg0OGViMTZkOWZjNTAxYTE4ZWRhYzJhZjRkODZhYWFlOGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lOpftxYhiX1W3K13NzhmOblnxAu_0--dbrMvS9Nq0pi6t-tlBCSdYiNVJIjhAOzGrYGt_Zzkz6fapX66gJQqZg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111146_58_247f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.624Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImpLblJFS2R0N0p2T3ZaWjg0WnBzaHdiSUt4Ymh1ZjhmLys4TXkvUUtqcWYwTmlwVFZwK210WkhEdExlYlo4YXNXbGcxUGM2alNhUjZoYkNjTUdpVGNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwN18xMTExNDZfNThfMjQ3Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmQ5MzE1ZjMxNjUyZGRjOWY1NzY4Yjg5ODAzZmZkZTA4NDRjOTA3ODZkNmJjNDU4YjI0NjM0NThmODQyNmU4MDdlMjBiNzdiZGE1ZWI2MWFmNjM0OTg3M2JhMDFiNWQyZjdmMzFjYWNkNzFlY2UyNTc0ZmJjNTllMjhiYWNlODVjNzc4OGZiMGNhY2QxNGEyYTI0OTM2ZDBlMDczMTliZDliNDVhNzFmOTA5MWZlNGVkNDBlN2FmMmMzYjc5MDBmNDlmNzEyZWRjNzdmZjI4YjNlNTEyYTcwZWI0MDg0ZWIxNGU1MTM0Y2U3MDE5NzNkZmJiODYxZWVlOGU5OTJjMWVlMzVkMWYwMzM0MGViOTAxOGZmYzQyMjhiMGNmMGY3ZDg2YTk2Yjk3NGY5NDQzMTA3ZWQ0MDI5MGQ0MTI1MDJlYjUzMDRlYTNhZDdjODQ2MTllOTM1OWZlNDY3MWFmZmYxNjI2NmQ0ZTE0NTgzODNkMWMyOWQ4MGYxYmQyOWM4ZTkxNWE0MGEyNjcyZGIzNmRhMzY1NmEyZWI1NjUwODg2YTM2MjYxNjJmYTM5MTE5NGM1N2IzMTJiNWUxODViMmI1ZjIxYTEwY2MxZDA0NmJlYjEyZTlmYjhiOTE2YWU3MzRkZGYyYzU2YTVhYTg4MzNjZDY1Mzc1YTFlZGYxOWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.W_pEungvX16g9mKGeUDPtYUtG9vWSZIYINODKvoh9fEKfZl5TZMgIRFW6oCSHpYfyXe6OtiE3u4aSHB5l9OkKA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230907_111146_58_247f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.628Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im85TzRQYlpDVkZ2VDVBMlRRcXorTmI3UlhIdFp3TGZOVUt6ekYxUlc0bzNIRnJzMEQrN1pGVjh3SnFISmpVaDNsWFhjcGdtd284T0ZybHovaDdJVHhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzdfNThfMjQxNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDBlZWFiNjQxODgzN2IyYTQ4NWQ1ZTIzZmNiZmIyMDg0NDZiZTRlY2U3MGM4NjA4MTIzY2VkZmFkMWFkMjViYWJjYTk0MjJkNTE2MDZmMWQ2ZGVmNmUzZTM2NzRhMmE0YmI3NDFlNGIwYjc1NTMyZDBmOTAwNThiMDA2OTM2NzJmNTUzNTUzMDBkZDBjZmVkZTc3MGRmYmJiMDA2NzlmMTEyZjkxMDIzNjdjMGVjNjMxNmZkNTcyNGRiZDc0NjFhNDZlNmU3YTY4NzJiY2M4ZThlZTQ5NTRjOGFhNzM2MjZiMDM4OTNmMTczYzdjMzllOGYwYjY1NWJjZDVlZjA5MmExMzZjNzczOTAyMjY0MTAyZTFlZGU3NmU3ZjcwN2RjZDk4MDg5MWM2NWM3ODY2OWMyZTkzYTdmM2NkYTEzMzllMDdhYmFhMmUyODkyMmE0MGNkZTkzZjBkNTAxYTA1MTk5MzBiZDQzMmIwZjlhM2Y2Yzk3NDY0YWU5MmFhNzFkZjM3MjcxMjZlNTYxZDNjMmY0MzJiZTQzOWRmZTdhNTMyMTQ4OWJhOTliMDMyODcwNjNmNDUwMjkyN2NmZWRlY2E2ZWUzNjQxODZjMDJlNDA4NjE0YWNmZmM5ZTYxMDJhMmIxMDc3NmJhZmVmZjJiMWM5MmFhOTZlYzY3YjgxYjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._wO_1TlvApoZ19fWbtfByBi_vsXAQH9Jxx5LVhx5JpbxHVq7TmyutCb8DCy0r1a5KPohvzWJycLySOqoGTdyrw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112737_58_2414_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.631Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IldKMFRPY2N6byt3WWlOVFU2MlJrZ3l3MVBOWG90NWlsb0tUQWtycUZ0ckhrOHltaEtLd2RnVlNDaGZvREE3L1RzN1NrcWgxd2lyeUVoaGo3SnFUdkR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzdfNThfMjQxNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGExM2RkM2Y5NGE5YTBlMjljM2I4ZjIxMmYxZjI4NTU3MzYwZTA0YWM4YWMwOTQzZDcwMjUxYTc4ZjQ2OWUyZWFhODM0YzI3ZmM2Mjg0NjkyYTE5ZThmNDAwYWY4ZWU2ZTQyYTg5MzcwNTRjZTk1YmQyYzJhNzA4MjgzMGU1MmIwMDdjOTU1YjAxOGVkNDhkYzI5OGY5NGI2M2E0YjdjODdhOTQ4Zjk1NTllNjk4OGYzN2MzZWY4NmIxMDY4NTlkNWU2MTMxMTAxMGJjODAxNTUxZWQ1ZjNhZDA5NGYwNmM2ZTRkZWIzMzdkMDMyMTQ2ZTYyYTg0YWQ5MmU2N2RkZDAwZTJjNDY5NTg2YzI5MzZiNGQxZjllZDhmNzkzYTdjMTI5ZTQwZmEzY2U1YmQxZDY1ZWNkZTNiNTk1MDNkNzY2NDA5ZDNlYmI2NWNjMzE1OTMyMjg5MmRjYzVkYjE5NDM5Y2JlNjkwNTk5NzlhNjI1ODdlODRlOTFjMGVkZmI1ZjExYjAzZjJiMzJkY2MxMjEyM2NlM2QwZGEyMmU0MTMwZTMyYjU5MGJkOWJkNDI3N2Y1MmE3NGE1MzJhYjI1NjUzM2QyYjVlMTFhZTkxMmZmMzFmYWE1YWI0MGNkYzE3N2ZlM2VlMWZlNWI3ZWRiNTljYzdiYTM3YjhiYmRmYjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ofDYUxrOezbqiicpNhSD8jOiAoYvqAhr_GE6lJNVd4CIMWhhZF9ZU4XqD2hFTrl8xjipjjmqYKZdcNW74kecBA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112737_58_2414_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.634Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imc1dGkzNkFEL28zNUhmejB0SlBtbjBkSmhVOVUrZFJ4Nkw5cTh6ekJKUzAxMUhPWFdnRm0vK2syZWxkOS9nQ1Jid1UzVzdYMnJDamVUUkcwcnZ3OU9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzdfNThfMjQxNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGZkMTkwMzY0Y2JlMTYzZDE1ZDAwMWVmMDZkYmYzOGUzMTZlNTgzMzI5ZGQ2ZGQ5YWNhYmM3Mjg3ZTMzNWU2NmY0MmQxZDkyMDlkNjQwZGE3ZjdkMzZkYjJmYzQ0NTUyODA2ODlmYTdiNDI3ZTMyNDk1NDkxZjVlMDg1NzA2MWMwYmYzOGZiMjFhYzQ4N2RlMjhlZWI4NzdjNjgzNDRhYjM5NzYzYjYyMzQ5NDQxMTE0ZjY1M2JmNGJhMjkzYjE1NTRmZTYyNmNiODQ3NmVhZWRmYzA1ZmFkYjBhOGZlYzg0OWZhNzY3YjFlZTk0YjAyNzk5YTI3M2FlYjlhZTJkMDZmNmIxNDY5YTg1Yzk2OWM0NGY5YTk4ZWFhYzdkYTM0M2FhNGRkMjNmMTk0MTU2YTdiZTdiODM2ZjViYWNmN2RhZWEzNGY4YTFhYTI4OGU1YzBmMzhkZjA5NTNjNjdkMDhhY2U5ZTc4YTBmZTc3YjlkZjY4N2Q0ODAyOTQyYjAzY2UyZDc2NDg5MmIwMDQwNTc5NmY5NjliOWZmZjIxYzVlYzRmODAzNjNiMjg4ZGViNTQ5NDgxMTU3ZDk5NmU4NzgzNzhmODEyOTU0NDllZTE4M2Y2ZmUyYmU5MWY1ODQ4MjZhMTI5MDVhNDEyNWIyYmRiNzMxNDM3ODA5ZjUxOTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9kAkyprpz0fZhHJE0OiNFpH_4EBeyToLiQqCljGrrwzGupF_DlbxCOXnv7ewHuBbzqOgCMMoUE3kkqa26RAb6w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112737_58_2414_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.637Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImpZTjRuamNjRnorc0xLdFhlYXczcjF4TmwyYzdoNjdDNW9Dc1Jta2xTeCt6Mmk4bmlWWVExU1Ztb2sxZm5MMUtDOW13aVNjWVlvdUIvN1Zic1Z3MGhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMTI3MzdfNThfMjQxNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGFiZTI4Yjc4ODZhMDJiZTE4ZDg0YmRkZTE5OGJlNzM3NTZlNjc5NGIxYWI0NjYzNTQzNmNmZmU1NmY1MGY5MGJkNzM4NDAyYWU0MmJmYTI0YTIyYmMxNDViMDZjMGIxOWE1MjExNGZmNDY1NjdmYjk4MmVhMmEzMDA0ZTRkNjkzYTZhNGY4MjA4YmVkZGExYTFjOTNiZGIzMTVhZjhiYzM3OTkwYjkzMjUwMjhjYmQ2OGMzOWFiZGJmNDY5NjkwY2JkMTlkMzcwNWU0NzZiZDhlMWRhY2I1MWEzNjNkNzU0MDc3ZDE5NjBlNTIzOTg5OTE5NGYyYTRhZGE5MjMxMDI1ZGUwYWE3YjM4YTYwOTFhMmZmOTgwMjMxNDg4YWQxYjdmNmJiZjM3ZDExMWFmNjkwYWQyZjNlMjhmMDEzYTdmYmVlY2FkMGRiM2E4ZGE1NDJkMDkzNmMwNGVkMzhlY2Q5ZWI5NmExMTQ1MGU4MWJiNTdjODBlZjJhNWIwOGE3ODhmNjNjN2M2MjU5MWYxNDEzMmFiNjEwMmQ5MjY0OTVkNGYyMDJjNDhmOTE0MzMyY2IwMWQzMDM1OWQwYzViNDc0ZDZhY2Q2MTlkZDMwZGE3NjE1NzM0MDNkMDNlYWJhNjZjZDQ0MGM0OTQ2MDY0MmQzMWNjZWMzOTVhNDg1YzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.M65WqPfSRMuqVoVDTFSP7ByZ32RZDpWsEt9xWfJmeTRVApM8Iedachtw83iwtB9MvF_7hMVJ6goYSvIDlGJZIg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_112737_58_2414_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.640Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Imd3MXlJT0c0U04ybTA0aVV0Ym1Qd2ZuVDBWMkY0NlgxS0lKRHU2MUwzSGdNZi8xSEJwcm5hbG1VWEdaRjd0QlhWdHVsTERHNG9LZ0FhbllsUFBLUWhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMDQwNTdfMDNfMjQxNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2M3OTk1MGRiMTBjN2M3NDhkYjM5N2E4YTU0ODM0MDA4ZTdlYjdkYTg3MmRmZmM2NTM4MGQ5NGIwODIxZmI4ZmY3ZGMzOTcyNjBiMGY1MDAxMmY1NGUyYjA1MWE2NzllYzQ0ZGEzYmE1ZmJlNjZjYTI0YjYzYzkyZjI1MjdkYjE0ZmI4MWVmNDQ0Y2YwYzJmZDRkYjEzMWMyYjBhNDZmYzRjZmYwOTU5OTY4MjY1NjYzZjAwNjRiOTFlYjgwMjVlMTg5ZTU5ZDE5ODQyODkwZWY4Mjk2NGFlNzk0ZDBkMmUxMmIzNDU5M2Q3MzBkMjk1NDVmZTM4ODI3YTRkYzgzMjFiNzA2NTBiMzIxZjU2NGVlNGNiMTM0ZDVmZTNmZjYzNGY1MDM4YzI4NDMyMzM4ZTM1NmEyMWJmZTc2MjkwMzhkNDMwMmIyNDNmYmJlMjAwZDEwNzk4Y2Q1ODQ0Zjg0MzA5ODAxMGM3ODhiNzZlNmQxM2QwOTMwZGE0MzdhNWU2OGRmNWIxOTRkOWEzYzA1ZTJkYTIxMjYzNjkzM2FiZTk1MGZlODg3ZTRmNjdmODRiYWVlNzFmYTMyNjAxMmNkZjY3OGY1MmY2ZDg4NmRmMzZhNTMwNDg5ZGFkNDk4NDdlMzFkNmIxN2E1MGQ1NWJhNzAxMjZhMGU1ZTU1NDAxODJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5cD785F4u1_Bbrr2X681BKcHCn4yhsnvD33Z_fvZOI1ghNcRRz86fU_tTJYopY7z8TIlXwxNVw7qH1kEIDFpGA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_104057_03_2415_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.644Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IktBYTErdUc2NVhoSHN5ZnJDYlBzc0FGQzZJOXpnODluVm0zR0Y1dkpmVjZuUjdrLzZBZ2tEVWg4OGdPOHlEUG1PSGIvTmNDd2wrRVFNdjJ6TUVYd3ZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMDQwNTdfMDNfMjQxNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzI0ZjkyYmJjNTFlZGY1N2E5NzFjNDAwZDM1OTM1YTI0MWNlOGY4MTM1ZTI5MDZhMDdiZDNjZThlOTk4MWFkOGZiNGU2ODQ1ZDBlM2Q1MmY1MjE5YmQzYjU1NDQyOWZiYzY4MzIzZGMxNTFhZDI4OWJmZDljNTM0MjgzZTA1ZjJkZTI1MTY3YWI4OGZiMmFmYTFhOTVhZDJmMjQ4NWM2MDFiYjU0M2JmZDNhYzQyYWViMGJjMzM0ODE1ZTAwZDI5NGU4MjJjNjFlNWExYWU4ODFlNDQ4M2E1ZGM5ZTc5Mzc3ZWZjMWEyNzAxY2NhNzY3ZGUzOWM4NzNhMDg3NWUzZjhhZjQ2YTI0YzFhMTgyYzJjY2NmMTBhNzdkMmQ3YmI1NDdkMTMyMTcxOGI3ZjU2NmRhMmRkY2MxZDA3YzI5NzFlMDcwODY0ZTM0NDEyOThiMTBkYmE0ZjQ3OTgxMmQzZjM3YzcwZTRlZGM1YzNmZGRmMzcyNTFlNDExMDI4YzY1OTllNjhlZDEzMTgyN2MwYmYwZTM2N2VmNGM2YzA0MjYxMjFhZDBmMDc3YjVjMzUzZjA5Y2FiZjg4ZjNmNGVmYjI4N2Y4Nzc3MGMyMTQ3MjlmODY5NTNlYWQ4NGI2Njc2MjVmYzczYzBhODU4MjQyZTFjZWQxMDZmNzM1MTg3NDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hicqav6euf8HHFNHlyZ7mFOK9ue_JCfIukzSNpLQ7iEHl2kvK5xQ3As-xBnSX2qFKXPXfxThx6bFPqCUCkV8gg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_104057_03_2415_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.647Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkxkQk1jd2dGWkwyd0t6cHhvTXBNQ1F4aTNNZVFHck93K0pwbk1VNGxkeEY3cmFQcStyYzFRaEhSbmI3SUMwYzF3SHlJSGlBMEtob0lXQ3BsQlAzQzhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMDQwNTdfMDNfMjQxNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTlmZjBmYjU4NzdlM2I1YzhiZGUwZjVkOTEyN2JiYzhjODEzYTYwZTUxZDYzMDYwYmQxYzY2ZjgzZjI0NzQ0MTBlNjFlMGMyMTNkYTgzNzhkMTI4ZTZjYjMzOWExNTJlMTY0ZWFjM2MxNWIwY2M4ZTQ1OWIyZGE5OWEwMDIzMzA0OWZmMzcwNmUxMWEwZjhmN2ViYTc4MTE5ODM1NmI2YzA1M2UwYjIzNGRkMzEzYWY1OTQ4OWVlZjY0NDgyNzYzZjcwZjc1Y2E4NDU5OWY2NjBiYWI1NWYyMGEzZTIxZjkxYWQ1ZDY4MjJmMGE1NjEzY2VkYzM0MmFiZWE2YzU2OWJmZmU5NmM1NDc4MjRhNTIxZDkwMThiNGI0NWI2MDZmNzM1M2RmNGY2YzU1NDBjOTY4N2U1NzA3ZDk5NWEwYmFjMDNkNmI2MDI2YjNlZTkzMTYwOWQzMTQwZWM1ZjBlM2MxMWRlNzY1Zjc3MzJjZmIxZTNiOWMyZWQ1YzEwMzM2MjU1NTgxOWY1YTJlZDdiNGFmZmIzMGE3YWE1NmJkNzg0OWIwMDI4NDg1NzVmZmQzMjk0OGZiNmNlYTc2MmZmMzk5MDQ1ZDkyZjAwNmFhZjViZTFhMTU0M2Q3OThmOTMzNDJiNjI0OGE3ZDliM2U0ZjY0NWMzNGU2NTQ2ZTNlMDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ONWcwC5cQ4_PoFw2hk2BYHCnrJKpoA4I90JwLxQnDMtkPblQV6Hmwoh4W6R1y3n_QlintV_AL83MFnYUmlOO2w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_104057_03_2415_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.650Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ijc4T1hrZmFkZG9vQUl2aXJlNkc4eGpWWXlOV1NZNzJLUWFHbER4MVU1aDQyNytic3hQcCt0NU1jYm04T0NUYjBya3hqYkRWckFZbjhzOTRYRVZvVHhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMDQwNTdfMDNfMjQxNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTU3ZTI5NmUwN2Q0NGEyMGIzMWY1ZjZhYzA3N2RjNWI2ZWYxNWFhNDRlNzhiZDI3ODc0NWY2ODVlZWRkM2Y2ZmNhYzEwZDQ4Y2M4YTRhZDY2ZDg5ZTY1ZjJiYTJkZDY5YWIwMzhlY2JkYjRjYzM1YTgyYTdiY2I2ZWU5MTJmZjBlMThiZTA2MmFjZWEyNmViMjQ2ZjU0ZThkOWU1NzY3N2UxYmYzMTk0ZmQ3NTY5YjFiZDYxMDM0YzVjZDgwNjJiZTAyODQzMDFlMDI5YjcxMGVmNTQzNWIxODQ4YzUyMDc5NjY0OTkxMDYzMGM3Y2RmNjI0NDMxZjQ1YWM2MDQ5YTk5MzhiNzZlZmM0MWMxZjJjODRlOTE2OTc2NjM2NjFmZWJlYjI0ZjQ5ZDU2M2RkM2Y1NmY0MzZiNjMyMmZmMmIwMTI0YTBlOTczMjljZmZmMjU5Zjc2MDA5Y2JjZTkwZjMwODMyN2I2NmRkZGI1NDQyMGEzOGJhNTc1MmQ2MGQwYWJhZWU5M2VmZDA3OGI0OTczYTFkZjMxZWM1ODA5ZTdmM2ViNGM1MTBhZmZhZWM5MGY0ZDAxNzFmZmVjMTc5NDNmNThhNDNiN2VlOTIwNjA2M2RlYzNjZDk0YTlhZTk1N2VhZDcwNGNjOGZkYTgwNDY5ODg0YzQ3OTRiY2QyZDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lhWF8BIuvUdpHomxyblj-0Ir8GlmMQsz8Z1uXYIqMaCVVX3ScgzxqW7d9XN1z31Kcz7SPVjwprEzQlPUVBKh3A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_104057_03_2415_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.653Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBjR3ZpRGh4eHVSZWNXNlFJbVNaclh0RlNVSE9TQnhWQTVTTE1jMnZOd3dDVGVFOE9vdU5ITkpGcmYwbVRRdWlrQkd4RDJFbnhHckkyTkh5T0tlQVdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI4NThfMjNfMjQwY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODIwNjkyNTI2NjFhOTkwNTE0YTBhYzhlMmFmYThlMzYyZTM2NjJhMmI0YTI3NDdlZTRjNTMxYjA0ZjNjY2FiOTQ4NzMxN2U2NGYwNTliODhjNDZjYWQwYzY4MTZiYmY2MTk0ZDJkMWIxMTc0OWYyY2EyOTlkNjgzYWY0MzUyMjNkNTU2MTNkZjYxZTVhMDU1NzE5MzI2MDEwOWQ4ZjliMWE1NjYzMzFjZDYzOTUyNmU4NWY3YjFjNmE0ZjMwY2RmN2YxMjgxYTEwZTdmZGUxMzA4NTQ5ZWQwODA5YmY0NDU4ODYyODc2YjNmNjM5ZjU3YmZjZDA0M2RlMzdkMjY2NTM4MGExMjU5Nzg2NjNiZTg3NDhlM2NiYmMzMTAxYzc3NThlOWEyYjZjMTA0OTU5NTA1YTFiNGEwM2MwNmVkYjRiMTBmNmMxMTk4YmFmZWJlNjQ0ZTljN2VjYWMyZjE3MTdjZWNhZGYzNDZmOGE2NGI3MGIxODViOTEzN2I2MzRlODdjYTU0NDRkODZlMmIzMDM5YTZmNTdkYjU3ZTg4NzYyZDNhZDYyMTIzM2YwZTczN2JiYTllZmY4M2U2NzQ0ODQ4YjljNjQxNjBjOGIzNDUzNmZmZTExZTZhNjI1MzNiMjgwYmMyYjUyOTQxNGUwZTMwMzgyNmUyMTlkOGYwM2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GwpzGf-3uqDvMFcSJ1mo4CNEPLT3MaN3eySNfE3nsp4UYsWz1qTeEkE2eStIoov27fzcfPFrq6HnsX7WspoMIQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112858_23_240c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.656Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlkR1ZXZE1hMXFYYzk2cVl1cHRDNEV4TnNleDZKOS9VTDF2Q3ZLZ1dXdXhLcEtmSzUvZjE5MzlSb2NhQXZiSzBBTE1WWU1qcEhvaGVxSDkvcHZ1WU13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI4NThfMjNfMjQwY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTFmY2Q5YzgwYzE1Yjc0OWJhMjBiNDZjZWRjODcwNjk3NzQxZjNiOTdkN2IyMjNkMTQ1YzI1MDUzYjEwMWZkNjdiNDY0NDc4ZDY3YzU3OTA2ZTI5Yzc4OWFhZmE4ZGRjOGMyOTliNDMyYWEyYzVlMzMyMGFmYjgxZmRmNTQ4YTZjNzc4YzEyOGIwYmM5OWY3MTNiYzRmMDQwM2Q3OTNmNjMzMGUzYWQ4ODQ4Nzk5MDRjYmYyMDI5YWJmMGZhOWVlYjBlMjcyZWYxMWU4YTkwYzFkN2Y3MGU1YmM5NWUyM2ZkZmIyZWE2NWJkNGY0MGJjMzRjZjk2MjNhNzY2ODQ0MmI3NDE1NzFlMmM1MTNkZjZmN2E3YzI2YWRlZjYyYzJhODJiYTAxZTQ3ZDMxYzkxMDAyMzYwNmYxMjY0MGMwMzJmYjQ3NmQyNDkzYzA5MWE0YzQ3MzMzNGEyNWUyOGZhNjY5NmU5ZTgxYjQ5ODcxZjIwZTU3ZjM0NmQ4ODRkNjlhMTI5NDJhMDViMDgwNWM2ZDRkN2VjYmNhNDNjZjAyZTgyM2UyYzNkMjFjMmMwODU3MGIzN2UyNThjMzNiNDE2NWIwNWYzYjE1NWQwY2E4MmE0MWQ1NjNjODVhZGJjOTQ0YmJjZDA4Mzg5Y2M3OWY0Y2M2NDJlM2U5ZDM2MTY1NTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xRpJr1sME2usfz2I81jmqL-KMT0lEiliPicj2804AJ4y6zgyuRR1cpCXuQWDtZ4Nmgk5abmjkRGRg5SZPGSblA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112858_23_240c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.660Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlozbVdWSWJmcUR0cGNQaE1lWU1lM29mQVpudDhWYSs0cGtmVmxONEsvTnVycXJDUkIydDVtL2pyUndSbjJNU0RFN1JSTkpHRFpqVkZzU2xVck1MRzdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI4NThfMjNfMjQwY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTUxNzU5ZjMxZDI3Yzg4MDE2Njg4ZWQwYTZjNTEyNTI4ZjFiYzM2NTBkYzgxYzZiYWQ4N2IyZTBjZmMzZTc5MjViZmM2ODUxNDM5N2RlYzI4YjIyYTBjN2YxM2QyZjU5MjE0OTk4NmJmOTBkZTg1MGI1OGYyOTk5NzRjM2EzMDE1ZGExNDg3NWI3MDZiMzE2YmRmOTlmZTg2N2Q2MDg3OGFjNWRkNTc4MWZmNWQ0NTQ4YTlkNWE1ZGMzZWNkNTVlMDgyNDY4OWQ5YWQ2NzNiOTU5YzU5MjU1OGFjNDgyNGVmNTlkYjc1Zjk4M2Y4Nzk4MjcyYTZkNzIwNTNlZjYxMjI1OWE1NDk1YjBkNWJjNzdjYmNiOWRjODFkMzQxOGI0ZGFlZWRmMGU5YTU0OThmNWMwNDJlNjAwMzU5NmIwODYwN2MyMWMyZWZhOWE3N2M4ZmI2YTdiNTdlZGU0OTc3Nzk0ZGQxZTczNjFmYTdiYmViZjYzNmM5Yzc1ZDFmZmEwNjgwNDEwM2Y3OGNjNTJjNWIxZDU3OTQ2MmE3Mjg2OGM4NWY4Nzg1NWI4YzgzODkxZTVjYWMxYjAwMTMxODIxZTE5M2UzNzk1NzgyMGI5YzA3MTBkNjM3NzE5NjNhMWY4ZDZlMTQ3NDM5NzRlMTU4ZDAyYzY0ZDFhOWFlZjcxODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.S5awG1WAldWcFPMTfiniZKMz2sM97yMrHvDXPBqG_2phWF31yf4GrWb3xTTijLL6QhZUTEv61MaQEne9AUf6lw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112858_23_240c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.664Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkE0TzJtVzhlS2ZCU2lUdURNL0d2NU1wejAvUC9kcmgrMUFBTEpiRVJLUmdHOUdwck9OQ0tzdzg4cDRrdy9FYWY1dlBscmdCcnN4TGdTRTJHTm01TUJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEwNl8xMTI4NThfMjNfMjQwY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTk3MDE2OTk2YTk0MjUxY2Y4ZGNkOThlOGMwM2FmYjRlYWQ4NDg4YjhjODA5OGRhYTZlOTE0ZmE1MWVjODNhMmNlYjIwY2VmMTU5ZTFjZmQwZGEwOWI2ZjQ2NTk2MzE5MDM5MTc1YjA2MmVhNmY4ZWQ1YjQwYmVlNjgwNjM2ZjM1MWJkOTU5NTNkNTJhMWJhYmRhY2ZkNzVjNGVjNzViZTQxY2I3ZTA0NjdmZTQzNGZhMzhhZDIwZGE0NzVkZTVjZGExNjVlNzNlYzExYzJmNTcyODZjMTViYTQ2MGQ4NjE3ZjE4NTQ3MjAyMWQwN2MzODUyYTM0Y2Q4MWJkNTBjODRhMGJhOWI2NDE4Mzc4ZjQ2ZDE3MWIwYzY0NTk1MzdhM2RmMTAwMzZhNmQ1ZjdlZDcyYjM5OThmYWJiMDMxYzRlMTkxMGVmYzMxM2YzZmQ4ZTIzN2I3MzE2OTUxMjQ0YTY4ODZlYWM2NjBkYTJhMDY2ZTdiOTU3YTg2Njc5NzZlMzY5MWFiZDcwMWZkODdmNTc1MjdiZDgzN2YzYzRiZmUyZThjZmU3YTI4M2U2MDRkZjk5Y2VmYzk5YTExMjg4ODI4MzZkOWFjYzlhYjdhZDJlYjYwNzVkNzNkMjdjYzMwYjQyNWM3MDUwMzEwZTBjZGJmYjc4NzYwYTNkZWU1NGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ezVlipszkrQ1FXtchHbayLFwLT7A90rYMvLTCGPagAQep4QHbXyQVgcxYZnxDLAjMJpGCxuFv9b1SA_pGehxZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210106_112858_23_240c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.666Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkVkVkVqVGovTWd1Nm9Kb1RRSFVRblU4NXd2RFIzejFndGExdHh2ZGpRM3U3dlM4NnRFNkMxakcvTHQ5R09NOW00S1J0Q205bHphUHhWT3BTUHJTUkNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMDIzMzZfNDBfMjQzNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGYzMTg5NGRmMjMyNjYyM2I3ODZmNWY3ZDEwMDU0NGUwZjEyZmQzMjAyOGU0MDBmN2UyZTM0MDM1MTc3YWY5ZWFlODk2ODhhNTEyZWY0NzhkMWYxNmZiN2Y3Yzk5NTEzOTE1NmEwMjI2MWJmODRlY2RjZGNiODNmNzViNTUzNzAyZjIzZWQzNmU4MWZkMzJiZmFiYTUxMDg0MGFlMjAwZDc3MzIyMWViNjUyMzZhYjBkYWI1MjE3MjBjY2I0MTBjODc1YTRkNDM0Zjc5YjZlOWJjNjc0NjM0OWM3ZTVmNTc4NjUxMzIzODE3YTgxYzU5ZjkyZDRmNTk1N2ZkYWZkNjdlYzZlOGFlZTI3ZTdmNDRlZWQwZmM5ZjhlNTIxYjkzOGIzNjQwNzMxMzg2ZDAzMDQ2Y2JkNWZhNGQ4YWFmMjZhYzE2YmM0MTA0MDQ0MzJlNTMxYTExNjFmNzY5NWNlZmEyOTFiY2EwZmU2Y2MxNzU0ODg1OWY4MDlmYTFjOWE3MWEzZDg5OGU4OGM5NmIyNDZlNmU2OTYxNzBiY2U5YTkwMDRjMTBhNTE5ZTkxMzYzYmNkYjc2MjA2Y2E3OTI0M2JiNGE3OTc3Yjk2OTkxOTNkNzZmMGFiYzFlZjkzYmEzMjJjOGFjMmM0YzZiMGQ0MDU2NGI4ZDdiM2U5MTc5ZDRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JJqWcE2s43fouMzRqfTKilYbXA-i8gdOKSngh4vuitUH1DbYJ42B5aHz0C2d723aK7HllfsZl2T8qwXsVBqsgQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_102336_40_2436_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.669Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik9LTzg5NFIxMFpxUklOK3d4cWNWbHJqdWJEMXBCd25DUSs0Yno5ck1hcHVLOFRnbU9ESWVmZjc0M1R3S0I1RVBBYVVCTUJZQ1VIUmwrb3FYWG1sejZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMDIzMzZfNDBfMjQzNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmYyYTI3N2YxMTNmZTNlNzY1MzFhZmY2YzJjODE5MzVlYmZkMTljN2Q2OWI3ZDg4YjAzNTExODFmODZiMGIxYjlmMjljMmFmYzNlYmExM2QxMzhmMDc0MGU5NDAyM2UzZmMzNTllYTRlY2NkNTZlNjJkNTFhYWIwM2M0NzBmNGUwZmNiZTM0M2M2NzdhYjI2NWQ2NmYxZWY0NGE4NjBjNjFkOTI0OWFkODNiZWNmMjU5OTA3ODU1NTY3OTI0ZmVkZjM1OGY2MjdhYThiOGM5NDc3MTE5OGRhMzcwNDJkNWM2NGQ2MmUzM2FmM2YyZjdkNTFlOTZmMGNmYzE5YmY3NTkzMDBkNzFlZTlkMjBiNzljNmI3YTU2MWZhNDgwZDcyOTc0MjQ5NjU2NGM4MTc0ZDFjMDExNWM1ZjE0MjgwOTdmMGI3NDMyZjY2NjMwZGRkNTU5YzJmNGQ1ZWU1OThjMDY4MDliOTNhNzU3ZGE4MjYzODdjMDE3OGZmM2FkNWZlODU0YmY4ZTZkOGY3MmQ4YWZmYmE1ZmU1NjA0ODQ4YTRjZWJiY2Q4ZWFkZWEzNGYyNTk2ZWRlMDY0YTQ2ZWQ2NjE5NmJiYjZkNDJiNDAzMWVlNDY4NjZlZDZmMmYxOWZlMGI3MTRmNTFkNjgyN2RkYmMyNDM2NzA3YjQyOTM3NTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FOmChRpo1mTTA8ZWQIy5ELTY_Tnl5p2_eDIjeAtqnz6FfRfux54xPp4KByi7loZgn3wPBcUWAyZa5OIpzHj1xg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_102336_40_2436_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.672Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZreXdzVlMxVXBxd0tUUzJEU21PVEZlRkt5b21tWFQ1MGVXRE9yZE8zbDR6eDUxb2RnYTFYMEx5T1lDQUJIYjluQXY1dWE1UEdTcU9YR3NnSS9yc0x3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMDIzMzZfNDBfMjQzNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTlhNWZjYjQwYTY4NmJiODM5NDMxNjBiNTIzNjA5NGEyYjRhYzY1ZDUyOWQ2YjlkZDM3YTA1ZWZmZGM2MzIxNGVhYmU4NjAxZjI0MGViZTIxOWM2YzJiOThkY2FlMzMyMDNkMzhmNjYwMDBhZjFkYmQyOTM2ODkzMmE3ZDhhOGRkNTJlMDEzMzZhN2UxYTk2MmUwNDM0NzYxMDYzMzM0NzY3NWU2MzIxZWFjN2YzNmY2NTU0MzZkZWQ1NWVjOTVhNjBmMzA1NzI2MWYxZDVlYjdlNzdkN2QzYmEzOTQwMDkxOTIwZjA0NDQxMWIzNjg3NzNkZjdmZTViMGYxNzAzMzY5NDdmMWEyOGIyMDI1YmI2OGJlNmQ4OGQyN2E4MDIwNmQ0YTJmOTMxZDcxM2E3NDNhNzc1ODUwYmM3ZTAzNjQyMjM3ZTM5MTllOGNjNGRhZTEzOTQzNzY5NmY2Y2Y4YzRjOWQ2NTA5NDUxNzg1ZmY5ZDc5YzllYjgwYTYwZWQxZGU4ZDlkNzJjNDY1MTA4MWY1YzM3ZjJmNTQzMjliOTUxMWMwY2UyNmYwYmQ5MzUzNzVlYmVlODJmYzJmNGRhN2QxYTczY2U5MTBhYzhhNWQ0MGEzNDRmYzU0M2Y1YTZlODI4NDhhMTBhYmQwMjkxYmQ1OGNkYWU4YTNmZGVlODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dibQKGQWmcT9iPWf-cv9o-rsO8ILT1WC7Cc010ttYOykFYaO5APnejdTHColaoP3wnCWdyoMQh1IuZJipGzlew", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_102336_40_2436_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.675Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVxTW92aUgwRFdOekx2SWtYYjBySGJvZFk3cGk1VGVBM2tTVUNOQlhCcmtaR2pJQjFwQlRRbWdCNzFHQVZOVGduSDVGSWFWTEtOMkVTM2g3MUtsVTFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDYwNF8xMDIzMzZfNDBfMjQzNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGQ3MzQ3YmRhNGNiNzMzZDNmZmZhNTRiZTdiZTM4YWYzYzZkZDkwMDExOTg1NjZlNmM3YWFlOTg5NTBmZTk4ZjdkMjg4ZGJmNzA5NzhiODBjY2U5ODA3M2Q4OTBjODM1YzI2NGNhZDQ0OTEzNzIzYmI2NzY5NTg1M2IzMjEyY2I2MmJkNjk2MmMyN2QzYmIwOTY5NmVlMmM5ZDU2MDY5NzNjMzk0NzYyNzM5NmYzYWQ0MTAyNmY2ZjhmMjIzMjU3OTQzYjExYWMzY2I0MDQ3YTk4ZGU2ZTc0MTBlMzkyMWYyZjdiNGFiYjVjODU3ZDc4ZjQ0NTI5MGZlNmE4MTc3OWFiZTJjMmQ0MDA2MDk2MWEwZDg0YmY2NDg2NmUwM2FkOTYzNGQwNTVlYTZiZjMzMjk3NDEwNzE2NmQ1ODQ5NWZhZjZmNGQ5ZjJkNTcwYjMxOTc0NmQxMTk2ZWQ1MjdjODU4MzZkNTBmNTM1NmZhZmZmZjAzZWVjODliODhkZjc4YmY0ZjdlMWY1MDAwMGE0ZjExMDg1MmFmYTgyYjA1ZTQ4MjRjMjIzMTJhNzgxMzYxMjhhYWExNjA5ODhhZGEyYjgzNzBhMGVkOGRkZjk2ZGJmMWVhMDVjYTViYjg2NzA1ZmYzMjk2NTY1ODI3Y2M5NjU5OWUyOTFlNzQ0YTMxMDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LP-Kd-PmD3wokAlZbQCxvpI77lCOv6Ma9Cv005PwaBm6ctQ_FbG0OKLPodU6lKzX4wraa8sWozFUZRDLKlpe9g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220604_102336_40_2436_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.678Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjFORXRFcjk0Q0tBbW1NdlJ2VDJiU0dwTWtCOThUMEtzSHZaYkRMTW9JVXUxSGc4aGFtZEorTjBQMVJpZFVMdVMvajQ1dWdFRW9NUmoyRjlMRStDeTNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTVfMDRfMjQ4Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWNjZGVjODI2MjBkOGM0MWIzNjJiMzJmYjFlMDQzMGNhY2JhYWQxZjQxNzQ2MzZjNDA1NWY0OTdiZmI2MzUxMTkxZDMzOTZlMDNmNmY5NDhjZGEyNjM1MGIzZDY2OGE4N2FlZGE4MjZkMzQxMjM0NjQ5NjE3Y2Y0YzJkN2IyNTE3ZjBkNzM1MzNlZDQxYjc4MjdkOGZhOTY5NDYyNWY2MTViZjBhYWM0NTFhZDE2ZjhmNzIyZDQxYjViMjJmNDg3YmJkNTEzNDBjMzE4ZWQwYzdkMmNmNDJjMmE3YTNmNjJjNTM5MmQ4ZTc2M2ZmMTAwN2UzZWQ3OThlNzdlZTdjZWM2MGM2MTdmNDJhMzkzMjQyOWIzZTUwZDA5NDE5ZGIyMzAzZThlYjk3M2YxYjJiN2U3MTMxYjk4NjI3MzQ5YTNkMWRlMjQxNDQ2MjNlZGI3NThmNDg0N2M3Mzc0NmFlMDEyNjk0ZDljMWVjNTU3MGM3MzBlMjE4MWQ0ZDUyZGJlZjhmYTE1NzA2ZGRjNWFmMzIzMDM0NWU3NDBjMzc0ZDIzNzcxNTgyOGVlZTEzY2U2MGZiMmNjODkzYzhlMmI2NWY5NDdkODlhY2MwMDVjZGM0M2YzMTJiYWE4YzcxYjVkZTc2ODE2MzM5NTcyMGE0MDQ2NDVjYTU1NTQ3MDc4MGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Frl4gzUP0Uty_WcTIcgMp9hIayeuRiAuYNgZCyB3ufjK3UYpmt75D73qqUHGSRKbzJbc7CvNz6bQiynZUnlgPg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111015_04_248c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.680Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InhTNEUyTERFQ25kY2hQV2N5ZTF0M2l2VjhtZHd2VDE2VzNGVjlNaEJQNjZveTEzUHR5Um1vWENGU1g1SjY2L2RQaTVZV3haRzBBN2NCYkFVL2FxdnN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTVfMDRfMjQ4Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM1MDM5NmFhYTVmNDA2ZWE1ZGFlMWRjOGYzOTYxYjVhNzY2NzE3ZTZlZjVmYjM1OWZlYzZhZGU3ZTQ5YmI1YmM4NDBhNTZhZDdhZTMxODc0NjY4YmU4M2U2NThjODY4YWQyMmZhOWJiNzlhYjVjMDY3ZGU3OGJmMTIwMmEyZDExMmIzYTFkMGM0YjIwYjkzZmIzZTgyM2VlZmY4OWMwODFiY2YxMGFhMzY5ZmNhOTk5NTNhYzk0NjBiNzc5ODMwMDQ3MTI1MWZlYWQ4MmUwNjYzM2FjN2U1MmUzZjFiYWIxMzYyYzNkYzFjNWQ3N2VkNjU0MGIxYWNkNmNjYjMwYjhlYTM3MTJkYTc5NDQyZWRhMDFmNmJkYjhiMzBkZjZmYmQ0OTQ5N2IzOTdkNzU4YjMwZmE5N2I0YjQ4MDgwN2Y0YjhhYjA3YTYyMmIwNWVjNmZhZTU2OTUxOWY3YmY3YzA1NWYwNzU3ZjM2OTY4YTQ3OTgwYWFiYWVmNGIwOGNjNjFlZmMxNmM1ZmE5ZTI4ZmM2ZjMzNzU1ZTc1ODJhMzQ2NGU4MGFhMDdkNjJhMmQ0OWUwOWQ2NGJjNTc4OGJkYjA0OWFmNDA2MGM0YjExZTgzZTUwOGI2ZDUwNjkwYTU4NTRmZjhjNDc2OTZjODExNjg3OTQ3MmU5NmE0MGQ3NGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OrmAKpgwmd5R6LkkCGiuTlvU50Iz4QXJOHm-Hom7b_IsCGBb2HOCyk61Xjy15wLFl7p8XC-FsiCw-TA5Ixzhow", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111015_04_248c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.688Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjdBZ1Frbm43Mk5MT0pSN1hXQ3AweDFQd3JMWXlNbzM5cnJCU2xUNVFBNnE2amRydlNRemROUWwyNWhEc2RvWmF6dGpkc2xSMEx3S0xneGU0R1JFekFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTVfMDRfMjQ4Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWM3N2NhNTQ3ODg0MzBjOTNlYjcyYjgyMGQ2NDhjMDRhZWY4OTBmZGY4NDVkMmQ3MjVlNjQ1MzYxZjM0MjQ1ZTU3Yzg3YjQ4YmI5NjlmYzVjYWE0OTRiZDhkOWY1NmY1NDA2Mjk4NTc2Yjc3MjU0MmI5MzIxNmM3NGM5MTU3ZmVmYzUwMDExZjE3M2NhNTc5MzNiOWJjZTVkM2ZmZTgyZDVmNDM1MDBhOGU1YzI0YTIyNzhhZjE3YzYzYWFkNDRlY2M2NmY5ZTIwZDVjZTc3NzMzNmNhZmIxYWIyOTNmNzU1NWM2ZTdhMWQzNjI1ZDU5YTc0NzkzZmJmZGY0MGU1Nzk2ZDc3MjQwMGExNzUyOWZjZGFkZDE0OGZlNjg4NmM4Zjc4YWU2MjFmMzQ5NTViMzcyMTA3ZDkzN2Y0NTIyYjFiMjRmN2YzMjZlOTQ0YTFiMzUxZGQ5MTFkMDY1ZmZhZDczODNhNTBkOTQ4YjUxZjY1NjQwMWYyMjcyODBlMDFhNTVkZmNjYTVkYmVjNjk5NjhiOTBjNTg5YTMxMmUyMGVhNDc5OTNmYjJiOWRkMzg0ZGViYzljZjAyNDQwOTI3ZTllY2Y0ZWNkNzU1NDY0MTg2NzRlMzllN2MxNGUxMjZmMDJhMWI2YTAzZjBkMDI2MDAyMDUxMzBiZDZjOTBmNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.R5ShpKcBJKONtsSDEvrSO4LYEl6lFG__VfJFf8t8CafJHqAhk7xfAOeP9jH3ZxzpAXrzZxwJgyq93RU_V4Mm5g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111015_04_248c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.694Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjhzSVZkTXdUTklCWkpLaWFZTnNKamdQUmFkcDFsSTVqYXMvV3RXUlRBMG45VFc1Q1F0YjhrdkFJU3dHMFA0eVd4dm0xRGR1bTZ5bDI4b3RyNkN5S2hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcwMV8xMTEwMTVfMDRfMjQ4Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTZjMGZlZThjOWZkOWE3ODE1MzA4Y2UwZDBlNjM4YTk1OWZhMDQ2NTZkMDY3OGYxODZmOTY5OTc0NTI5YmUyMjgxODI5ODcyZThmZDY2YWNmNDdkM2FiNDY0M2FiYzU0ZTQ3MmZlYWYzMGQzZjJjYjM3YWM2Njg2NmM3NDYxNTJlZTBmNDIwYzEwMzcxN2I4ZmRiM2FlZDE0NDRjZjM2ZjEzZmUxMmY5ZmU0Y2JiMGFjNWNkNDUyYTlhNjlmMWU3MTYwMWQ4MjllOTI1OGE1ZjkyYjI4NjliMGFlZWZkYzEzYTVlMTIxMjZhMjEyMDVmODBkYTMxZDA0YThlZDk1NTI2YzdiNTI2YTlmZTEwZWIwZjBiZWZiZmRiYmIyZjY0Y2EzOTAxZTA0Nzc4OTk1NGVhZDU4NmMwMjcxZWExNTQzYjY2ZDRjZTk5ZjNlZDcxYWE4NTY4YTlmNThkOTIwYTE0MGRjYzQ2OWI3NjdjYjlmYTdlODE2ZDhlMWY5YmIzNzJmNmI5ZmQ5YWU4ZDg0Y2M5MTViMTU3NDM0YWU4ZGEzM2QzMzI5ODlkMThjYTc5YjdkMDM5MjcxMzhiYTkyOGNlMTcyYTU2NWQ4MjZmZTM3ZjM4ODhiMWJiNGZjZDI4OTMwNGE0NTVlZDlmNTZlYzVjZGVkMmE3Njg3OTdkNTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.43Ht6LTiuJAMouHiPe1Ook9VnT06mxU7Drzrh9qgLHsgG6uO22_OMuYP_GBMOO169NyulHRKmHtVI2qGWW6FfQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230701_111015_04_248c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.696Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkFLRloyei9qb0ZUbU5FNjE0T09qNDd5Q1M2VWpXVTZnUzRJV2NPc1o2cEMzazBWaWpITzhmNFY3eTlwdnhEWHN0QTE5ZE5mT0JDQTFBYk5TbGhKcTV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMDMxNDdfNzRfMjRiZl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Nzk4ODgyYzk1NzczNGNlZTkwYjc1OGZlNjI1MTVmODQ0YTQ0YmI0YTkyNDA4MzY5MTFiZjBjMDE2YjRlYjc4ZmM4ODBiZjJjNDg2ZjNiYzBlMjdlOGM4YmYwZTI5MWM3ODBlMjMyZDU2YWMwZWU1MzY1MTQ2MjkwZjZmZDA1ZTRjNTliZTJmZmY0ZmRjMzBjNzFmMTVlMTE2NmVkZTI0MmZhNjUwYmFkZWJmYzExOTk1NDI2YWRjODEwYzQxZWNkYjFlMDBlYTdkMTgzYmQ5Y2QwOTI5OGU4MDVjY2UzNzg0NTMzYWNlOGM5ZjU0NDY2NTI2ZGY5ODBjZmZkOWZmZDE1MzdjY2M0NDZiZDQzZmFjNzI2YTU0YjI0MTY0YmE5ODFmODdhZjk3ZmEyM2RkODJhMjgxOTkzMzYwYzA3NjMxOTk5Yzk5OWU3M2MxNTE5MzEzMmQ2YmY3YmRiYTFhZmFmYzNlNTY3YzFlZmVhZDU0OWE3MzJmMDg3NjhkYzA1ZGRiZjk3ZGIwZGExNzVlNTkzMTYwNzUzNTU5MTdhZGQ4YmE1ODkyYzM4NTIyNTEzODU1YmJiNmVhMjE3NGVlMmZhYzhiMDNlMDdhYjhkZGUzZmU5YWQ1Yjk0YzZjZDhlMzYzNzA2OWY3OWM1NDExMjJlY2M1NGJhYmUzMDhmOGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.p6HTWrS_ry9ydlJaIXTGuGVZ7qAcRfVDI1KFyIFKasCcd6zpQuqzy8UxSWlNR72sQyk_4l0Os9Dkp57iRoXrXA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_103147_74_24bf_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.700Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjRwV0lxeVpnYmlZMEhuMnVhZGFjZ1FYZ203Y2gwYTFDejVHYjE3N1Y3OFgwVDdQN3QybnpJbnFpaUpIU1hURFpBNmg0Z3E1YnNwK0hBSXJnaC9WVDZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMDMxNDdfNzRfMjRiZl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzRjYmQ0NDM1MDJjODAwNmI1N2I5YjE4YWI2MTA0YWRhNGJlMDQ1NTI2ODkwYmI5NTNjNjkzNzAxZTE4MDZiNzIwNGQ5NjUwMDM1ZDU4MGFhNGFjNzkyYWY0MDE4YTJmZWI2YmM0Yzg1MmJkZDFkZmFhYWRlMGM5NWNjMjlhMTg0NzZjMDczOTg1OWYwMTFlZTIxNTJjNzRiMDRkYmFlZjBhYjkwMjAzNGUzYzkxMjJkMDc3YTZlYmE1NWUxMGU5ZThkZmQ2ZGJmZDc0ODVkZjVkYzMzNWM1ZmRjY2M0ZmViZTEwYjljNzM0ZGQ2Y2MwOGIxZGQ5N2M4OWIxNDMwYWE2OWMwNWI3M2ZlMjU5N2Q1MDY2Mzg2MTU1OTlmMmUzZmQ1ODE5ZDIzNzk2ZTc3MmIyZWJmNTJlYTIyY2JkMjY0YjkwY2NhMGQ0NWJiMmE5NjYwMmE2ZDJiOTQ5MDRhN2NjOWI3ZmVkMzk5MjkyZDg1ZTBjM2JiOWFhZDc5ZDlhZDVjYjJjMWY0ZTVmNDA5Mzk5MjNiYWNmMmY5ZWZhNmE5MDZhMzI5NDZhNDZjODVlNmYwZGNlM2UyYjQ5YTI1NWY4MWZmMDkzMWI4YzFiMDRmM2M2ZDQ5OWQ3ZDE1YTVhN2E4MGQ3MDViYTkxMDQ4NTc1ZmU4YTFjNWM5MjBmYzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.REn5X5fyc1bH6RRiV6GlJ4qGWh09kUlIMKIlRRcNQsMNGuYwJDBvE8R7xVP0nimW3pl7GXbsvfWBXmoYXlovcg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_103147_74_24bf_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.704Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImFSTTA1bXZNRUZGbTJTUGFZY2ZHdHZQZmZvakxOd1JHN0lPVjZ1RGZ2VG9tRGIyU3QyZjNrUU9ZMTJaMHBEOXVndmtDL3F2TFNrUFVVR2FQdG9Va0xBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMDMxNDdfNzRfMjRiZl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjMyOWY5ZWJkMGI2MzQzN2EyYzcwNjJmMzhlYjQxNDY2YTA4ZjJkZDUyYWRiYTNmMjNlZDBlMDA3MjIwODI4ZGY4MTVlZTE2NjAyN2ZkOTI0OTdlZmQ0ZDZmY2YwODBkMjg4OTNlZGRkNjJiNDA1ODQxN2M3MjNjM2E3OWIxZjg0MzZiODJkZjc1ZjkwODkxYTY0MjE5NmViYTJmMWU3MDYzMjkwYTc5OTFkMzZmZGM5M2EzOTRlNGJiZDBkZGE1M2NmMTUyMTc0ZmM0OTMxZTFjMGM0NDQxNDg2ODc0YTc1NTU0YmE4MTA3MmZhOTBiMzY5YTMxMzc1ZGEzYmI2YzVmNjU3ZmM3YzM4M2RiZTZkODM4MTE3MGIwNDk0NTNiMmE1Mjk4ZmQxMWZjOTM3ZWFjMjU5YTAxNmJjMDFkODM2OGMxYmEzN2YwZDZhMWE0NWQ0M2E4N2RjMmU0ZjE3ZGNjNTAwNDUxNzc1YWQ2OWMyNmI0YTY1Zjg5Mzc0Y2E3NzZkM2QwNGJlMDZkZWY1YTdlN2VjODZhNmYxMWVjZGMxNGQ5MTU2YTdlNzBhZTFhNDllZjViYThmZmJlYWQ5NDU2YjIzMzI0ZWE1ZWFlM2RlZThkMjg1NWVkYjUwNzk2MmZjZjdhYzJhYmQ3YmY0YzUxZDE4ZWRmNzRiY2Q5M2JcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Yz7a4XNbHDliJkDRd-NrbXLzztrhtlkP88nzAXgvUInW7Gb75LfiEfJ3uBdYcye-rGTE73nrOjXZlaGC_5ZQEQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_103147_74_24bf_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.708Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkdFc0d2SWRXSGpyQTBVbVZYeVBtR1Vic3o3VGpXWFlPWSswR3ZoYzlHaTMyNFNuOVNwbVQvZi8vcVJid2JsUU5YWkRTcVRXQnA3ZFZYWSs3WWdva3p3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcxMl8xMDMxNDdfNzRfMjRiZl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2FlMTIxNzFmZDNkMjYzMGRlYzQ5NTdkNTJkNTEwNDNlYmNkNTE0MDZhNWM4NjExNGE1YThhNjg5ZTgwYjYzNDVkNDdlYmI2MjE4OGEzNThkOGYwZDk1ODdlOTc3YjM4ODU1MzVjMTI4MWQ2ODYyOTdkMmVkNGY4NzUxZTY2YWQ4MjdlZjdmZDNiZjZhN2U1NDYyZjZlN2EwODcyNzU1ZDNiNGVhYTMyZWZiZDZjMDEyMTI3OTdhNzU1MWNlYThhNDZkZmEzNjJhZTczMDdmZjlkZmNlYTg4ZGViOWZlN2Y1NGI5ODdkNjc3NTE2OGQzNTFlOTZiZjY0MjJiMTc1ZjVhOTBiOTk5ZjdlMjJkODEzNzc0MTBmYjU2OTVmNjdiZTZjZDM3MTIwMTYzMzY3ZDJiMTJhYzRiYzA2YjljM2E0MGYxZmFjNDgyNTU1YWU2YTgyOTYzOTc4YTU1ZThhNmExZGU0OTk5MTBmMjk4YzM3OTc5YzQ0OWEwMjMyN2EzNmFlNTBiZTU4MThlM2FhYzE0MWE0NGVhNjViYmM5NTNlZGIzN2Y1MGE3NTE5OTY5MmMzOTdjODMwNzY5NjFjZTJlOTM4MzJlOTRjZTBlNzgwNGFmMDYzNzUwM2M4YWEzOGVlYWIyOGRlNTNmMGRmNzBiNGFlMTQxNzgzYjI0NDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IIHO-10ZaotRyWTZwWE1DR5YeTqcytXpycMbNHFFg5-EOoeG37KixFXU2HLxubXTyiV0Ho_6TSWe6K5DY_UWNQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230712_103147_74_24bf_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.711Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9ObG4wYnlYbGFXa0hzeWowdVBXVDJjemhIN2Vka0J4d3FQN081UlJLNStrWVp0ZTkzL0RIaHJxdElUWForbTdxM0U3R1lkVmttZFZVZkgwM0RJQlRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjdfMjlfMjQ0NV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjY5ODQ0NDMyZWU3NGU4NzEwNTQwNmUxNzM3MmYyYTA4ZmE0NDU2NmI1YWJjY2U0MTAyOTk2ZTA1ZjFjZGVmZjYyZjAyZjgxYjJhZDBkYzYzZjUwYTAyODUyMGY5N2M5MmQ0N2Q5Y2YyYjZjNzRiMDcxZWFlMWMxMzgxN2EwZDYwMDMyNmNjMDgyYmQ1Y2ViNWE3YmM5MjFmNTQyMTM0NTA3Mjg2NjdlNTI1OTA2NDAxZDY0MTRjYjA1ZTEyNzYwNmY2NjEwNTUxY2QzNDFhN2EzZTRkMDNhNDVkMDk5N2E1NjdjYTMwOGU1NzU5OTFlOWM3ZjNkOTc4OGVkOGMwYjA0MGY4MzgxOTMzZDQyZTM5NjhlN2ZiMTQ2NjA0MDI2YzIzYWUzYTI0MDQwNDk2NTdhMWZiZTc1Mjk5ZmJlN2M5M2Q1NjBkN2FhMWM3NDEwY2FiYTg4NWJiMjFkMGQ5YmRiMDc3YzMyNjFlNTM3ZjFmZTNhMmU2MjkxN2Q5ZWVjYmY0NWFkMDg4ZjllNDI5OTAwZDlhNDdkNTVmYjkwYzVkZjBmMGVhM2M0ZDdjMDg1NTQwM2QzZjdlZjE3Njc2YTcwMjAyOGZiM2ZlOGJmMzhiYTk1Njk1OTUzZjkzMzhlNTliYzUxM2IxNTQwNGEzMjc5MDNmZDFmMjFiNzFkOGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.j4x_yzb57SQm2I9fNa0EbALclVAL9pPmShrM0L_qmxF9m1wb-Rp2YMzf8VDgs-lwZZLVKR4kCezQn_Gq59G1ug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101427_29_2445_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.714Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik8rcmZ3QWtwZTJhcjRXUC84MGFmYThCeFRXZTRWYzdEWmlSQWpLN29FYk1sMHVYL1N5QVRZU1BLdHVqS1dzZnpZRDFvVSsyQWRvYkFKU3FhTm9LeGlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjdfMjlfMjQ0NV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjQzNjlkZmIyN2ZjNDQ1YTMxYjY4YWZkNzRlMjNkMTk0MzU4N2NlYzkwYTIxMzYxY2ZmYTFiOWFlZTlhMTMxODYxOWI0ODgxODlmY2RiNGE2M2MxMzFiODI3ZTFlOWNjN2Q3ZTUxNmMyZWI0NzgxY2I5ODFkMDZiNDAwODVkZGM3M2M1YzYzMmVjMDA3NDUwZmI2NGJmYzgxOTVlNDkyNzI3YzYzYTAzNzY1MDg0MjEwOTc3MTJiZmI0ZTJlYTU0YjZkNzhkOTQ0MjM3MDNiYzQxZTlmMTBhYTE3M2VjNTg1MzYyZjI2YTUyNzgzYTQ5NzZlZjhjZDJkM2YxZDg4NzkzMzkzZmFjZGIwYmI0NjAxNjU5ODA5OWU0NmU4NTY4MWNjYzE0YTFiNTQxYjdiMDdjY2ViNWJmZDMxNzM4NmE4MmMxYWExZGM0OTgzYjEyYmYxMGYyYzliMjM3YTdlNjI5YmVlNDQ3MWM5Y2MxMTM0Y2EyZjliY2MzYzA3NGE1MWRlMzEyMjkzODhjOWFlYTQzYzViMzAyN2M5NmMzMjg0ZDM1ZmNjZTk1MjY5ZGNmNWIyMzQ1YzI4NjBiZTkyZGQ3NzVmNzZmYjc0NjA3YmUzODEyY2UzOTY4NWE3ZTk5M2Q5ZTBjODc2N2VmYmU1YzcxMmU4MzJiMGQ5YzhlMmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.m3ZChBKuMSRfYDi8foCFIx1V2MqDZAf83-Y_UA_PDAsJ7l9eh1REJQAkO3UkCScOSG1RH16yArc0DptEAL37QQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101427_29_2445_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.718Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJhUDErUFpMZ3dQWWN1RTN0UTQyVzM1VEZ4ck5mVzBTZ01GcDY2Z0JJdWk2LzBHZ1J0MmQ1SjhQK2hNQzhkZ0xOZldnblp5UXV4ZDNKRlV4emhKc0pRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjdfMjlfMjQ0NV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTljYzY2NDU5N2FhMTNmNDczYWRlOGI2MWFhZWJjMjZhNGY5NTE1OWJlNTk2YTYyZGE4MThkMTA5ZGI5ZjJmYmQ4NTg1M2U3OGU1NTMzMmI2NTg4Y2I5NjdkNDYwMzY4OWM4Y2I2OGZkZWJmNjRiZmU1MTg2NDk5MWMyODhkMjYyMDk5NjAyOGYzNzgyYTY4YzQ2MTJkMWQ4NjAyOWI1Nzg3YWU4NmI4YjgyZjZlYjRjMTNkMDg0ZDIwYmEwNmRkMGYyN2M1YTdhYTdhNGFmZjBlMDc2YzE4YzA0MGQyOThjNzFiMGU5MGUxMzNkNGUxODQ2OWZlNjY2OGU0NzQ3ZWI3MWM2YTA4Mzg1NWVlNmIxZTlkMzZmYzhjZDlhNjQxMzhjMDJiNTMzYWNmMGNlNWUxY2U5MDYxODQ2MTgwMjMzZTRkOTFhODYwZTJkNzJiNTgwM2RkMTRjYWVjNDgxODlmNTBkZjQ3NWRhNDMxNzU5M2Y3YTI4ZmE5OWY0MTM3MzQ5OTI0OGUyNmQzZGYwZDI1OThkNWY3Yzk1OWRjYzA1MTg5NjRkMjJjNmU5MmM3MDU3ODNlMGQwMmQwNzliMDdkZDZhZDBkMDg3NTNlOWJiZWZhOWI1MWYyMWNhMDljMjQ5Nzk0ZjI5NDE3YzMwMzJhYmU3ZGQ0MTc2ZDM0OGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xpNPYkzwzwwnYCHE47pxclc74pZ78paMavMC_ItaC-SGxDpfpW1yyXoW0jD8V3QmSkGAxxbBvQKGurQj89ejBg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101427_29_2445_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.721Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9iNUN2aWI5R1E1WHFVZHZKUENvUUU2alhySkpZVEJYaGQweXY2bGZxTi95dDl4UnVZQlZZRGV6MHFNbUQ5U1p4SFFTaDhmQzhrUTA4Vmt6d29ySEZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEzMF8xMDE0MjdfMjlfMjQ0NV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWIwNzY0YjdmM2I2ZGQzM2I4YTI1YjI5ZTFjNmY0ZDM0N2EwN2E4MTllYzFiNWYwODNlZmU1NGIyNmQ4YTA0OTUyMWNmNGIyYWE5NjAxNWVjMTdkODUxMWIxOTY0OWU5OGIwMWNmODRlMTE5NjNlMWU2ZGIzM2IzN2JjYTA2MGQzN2YwYzcwYzY0ZmZkY2M0ZTdkMTdhYzM5M2ZlOTU5ZjE4YjRjZTNiMWM5ZjkyMTAyMjkyOWI2YTUyNjNhYTAxMGE3YTc3ZDBlNjEwODVjYmNkMGUwMDg3MDRiNjVmZTgyNmEzODlkYTg1ZWIyZTJlZDZhMTVmNTNiZGMzMWFiYzJjZmU4MjYwNDFiNTEzMDc1ZWU2MThjMDM2ODVkYWJlNjBhMjdkNzU4N2IxZGQwOTRmNmE5ZDYwY2UzNTMzZTRlOTIxZDZkNjUwNzhiZGNlMTkxMDkwMzU3YzZlZDY0YjAyNWI2MDU3MGU1N2Y4ZjJjYzJlYzE1ZThmM2ZjNzY2ZjUzYjA1OTYzYmZkNmM2MTkwODE0YWI0ZGIxOTUyODc2OWUxMDg0OWE4NjRmZDgxY2I2ZTRkZTIzMzNjMTgyMzI5NDI1Mzk0MzJkMWI4ZGUzYjEyOTgxMzFhZWE2YTg2Y2UxOWFlOWQ1N2I2MDFmNjgwZTI2ZGE1MzY1M2MwZGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mqtDZ4yAUUYGAZgtmUGjeFNtOyD4FsyN1z6lRaOPGCm9HNTQCuc0rvECqDB7lglqwaj7wriKYRm-h-i9qGJPvA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230130_101427_29_2445_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.726Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im5EZStnTzIwL0U4bjd0ZG5SN29sbHQ0YzVyK1BpSHBQdXRoOXNFeUZpcmFjakxkZ2ZpcE9BSGdOYm94a3pUZXhkVTU2aUoxVE53cjlNT0JqU1lsL0hRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MTFfNjRfMjRiY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjM0ODE4YzFjMDMyMmIxZjk1YTkwNzEyNWRiNGExNTUwOTMwZjBkZmU5YmVlOWQ5OGVmNTkwYjczZjM3ZjNkOTlhODFjZDAzOTY1NTgzZGZhN2E2NGNiZjI3ZDIzMTgwNTQwNjAwOGYxZWQ2YzdiMDY1M2QwMDZkM2M1OGY2NGY1YzA2MzI0OTNjYjM1NmRhNmNkMjJhY2FjNGI3MjI3YmMwYzk0N2Y3ZmYzMGY2YTMwMDExMTQzYjgyNTczY2RhMjc4MWRhOWYyM2YwZTMwODcwOTNhOTkzZDQ3ZTQwZmU2NTQ2MWNhZGM1MjRkNWU1YzIxMWVhNTQxYjg0NGUzMzlhZTZlZjZjZDViOGJjZGNhMzA0M2FlY2RkNWJkNjE4ZDQzNDBjMmM0YjRjN2Q3MjM2NmM1NjIzOTlkODVmYzdmNzVjOTk1ZjQ3NmFkZGM1NjJkZTBiYjI5NTZkOTk0N2Y3N2Q4ZDI5MTEzN2ZmMzIxNmQ1YzEzZDNmNWVkZDA3YTA3NTE3MzQwY2Q5MjZkODcyZWMyMDhiYWFiMWI0NTgzNWE1ZmU4YjFlYzZhNmQ1Y2ZlYzJjOTQ2ZDFjOWE4MzNlYWZhN2ViMzcxNDIyYzE5MDFjZGYzNDNhOWE4NDYwNzA0Yzk3ZGZmMGQ3MGU2MDUzYmNlMmMwMzk1YTA4ZTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FkTDl3DQsk1X_YnTycQsJQFBeAJJ85UkPheFN2VWaVG9qCsdCzXozzqk2mFUSEyEttDgQEC88EJX6sUHjwHuQA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103911_64_24bc_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.731Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ilp3bm1kTkEyL08yUU9rS3JFLzU0QXAvVHlHUGc0blVvOXFHY1F2UmpGamkrdzBPSTVjcUhpVEVBakJSVklabllLLzNLUnJVQ09DdVRHTmlrL09RbUh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MTFfNjRfMjRiY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2NkYmM0MDg4ODUyMzc1M2I3YWFhZTI5ZTcyNTkxNDM2ZmJlZDVmNGJmNDY0MTc4ZGJhYzIyM2RmNWQyMDBmZGU1ZTJlMmNlY2E5MTg2NWU4NDNmMjg5ZTEzNTkxYjVjNTA5ODgxZDg2NWY3M2ZhOWQyNmYyMzI3NTQ2MWNiODQ0N2RkNTJjOWIwMjIyOWRiODJlNTgzMjIyZDQ4MGJiYmNkOTZjZjgzZDAxOWYwZTE2M2NiYTRiNDQwZjU1ZTQyZTc3ZTA4OWUwYTZiZmZlMDYwZjdjNjNlMGYwMWZjZTRkNzk1MjBiYjlkMDg3YzRkNzVjZTIyOWE3NjRkODQ2NDY0ZWY1MjQ1OGJhZmFkZWRjYTZhM2I1N2I3NzU0MDQyYTk2YTkwMDlkMGY1MjJhYmEzYWMxYzE5MGFjMGQ2OWJlNmQ2ZmY5YTgxNzBiYjZmNmM0MjYwYzc4Yjk5ZjY2ZWNmNTRhNmU4NGM2N2I3M2NhYmEyY2IxMTE2NTI4ODA2YjIxOTQ4ZGIzZjNlZDZhZTk0ZGVjZWQ3MTY5MjIxMTY4OTEyZjk1N2I0ZjQ0NjcwOTBiNTczMmUwMWVkYmVkYmY3N2Q1ZDc4NzcyMDM1MjZlMWMxODQ3NTJmMzNkNGM1MTkzNjNhMGE1YjVhNzdlYWZkMTM3ZmM0YzE0NGNjOTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.T0GekJgpHF33ljpuAywg5IlW_IBJju6ba2d-KArKGR4SbjOP7jDDQAGIOzWGUBAHVUyTZbW4_aWDu79No0TwKA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103911_64_24bc_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.736Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9ZalZMcUlSalowMG52eVVZeVB6RjFqWjdmRkFoUDg5Mzd0c2VCRmRSWnlyRlo4VjZ0OWc2Y25iRHIxTTYxRzVZSHBha213TWVYWHRPY2hQdU9oeXl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MTFfNjRfMjRiY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODUxZjU0MGZhODkyZjMxZjJjMTgyZjA2NGZkYmViYmFmOGFlODdmOWVlMGEyMzFkMDU5YzIyMGQ5ZjQ4ZDQ5M2VmYTNiOTY1YWY3ODI3YTFjZjFiODkwMDI2MmQ5MGNmODA5NTM2OTU0Y2JkOTQ3NWQxNzdmNjY4Y2VmYWQzNjliY2Y1ZGI3NTRkMWQ0YjRjMGZhNWFmY2Y4MzJjMzkxNWUwMmFjMzRjNjZjN2RiZTY5OWU1MzcwMGJhNDgwNjRiOTI4N2RhOGQyYjhkOTk1ZjJmNGJmMzdkNzQ0NTFjMzkwZmE0MTRhNTI0NmFhNWU5Yzc3YjY2ZjVhNTU3MTdkZGFkZGEzMTFjMmE5YzgwZmNkZWY3Y2ZhYTdhNmMzNjlhZDIxNDUxNWU4NWQ3YjlhYWU5YWFlOGJhMGQ1NTEyZGVhN2VmNjVmOTlhOGQ0NmJmNmY1NzM5ODE2OTM2Nzg4ZWI3MzkwNzBjOThkMTY4NDNjMzk2NWRjYzM4ODI3ZjU5ODFkY2Q4M2QxNWQ0ZTIyZmU3NmRhN2Q4ZTY3MzdlN2ExMDAxOTdlY2RjYmViMTA3ZmZkMDY4OGU2NzhiNWJjYjQ4ZGY4M2IzZWI4Y2E1OTJiNGE4NDFhMzc4N2E5Mjg0M2FmY2Q3ZDRmZmE2NDlkMGFiY2ViYzhlMjRiNTAzZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cR_lrc-6ueGi_-bIsVy5wEDN3ShHpqJtFNyFJooLUv6CJj-a-LnZJCZZXfPOHHyJiLlcaCpbwhvcmG6no7msjw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103911_64_24bc_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.740Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ink1OWIrejBCMDI2eW93ZTdHK2NoVzJhb1NOZTdrZTl0U2FsT3pwWUNSNHZiNHJPVVBiQ0h1cVovNWFtTGxNdFlTZkFQZ2V3aWl0WWJOREhLOGI0akZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDM5MTFfNjRfMjRiY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTgzMTkzOGVmOGM1MGMzYTM2ZjI5ZmEyMWUzNmFmZTVhMGE3ZjhkMjMyMTE2M2M0NWVlODg4OWFiMWQ1Y2U3ZjAzMDRjM2E2ZWIxZjg1YmFmNGQ2MTFmZGQwOGZlYjY3NzA3NTZmOTYyMGM1ZDdlMGMyOWU3OTBiMzMyMjVkNjJjMGEwMjZjM2Y4NTVmZmRmYmFlZGY5ZTE1YzQ3OWUwYzNjZjgzODk4MzJhMWMyM2Q1MmNlYTcyMzA1MGMzOTc3YmJkYTUxOGRjYTRhZjY0MjM1Mjk3OTg4NDM0MzlmODZiZGE3N2M2MmM0NmVjMjY3MzQzMjA5YjA1OGI0YzU3M2I5ZTk0NWUxZjU4OGE4OTE5ZWQ2NjRhOTdiZGZhMDk0ZjU4MjYxMDlmYWFjODQxMGY2NGMyZjQ1NjU5Y2RhYWI3OTkwOWNlM2U5NTM0ZTEwNmQ1MzRiOTRhYjkxZTI3ZTY5NzZkNTc4ZDIyMGMxYTk3NGRjNDgwMDI5ODIwYTVjZjIwNTA0MjNmZjkxNTI4MDYyMWQ4ZjI3Mzk5YzI2ZDkwZTg4YWM1YWVlZjEzY2M3OGE0NzgwZmY0MmE1M2FkMTU4ZDI5MDUyODNhYjQ5YzY2ZDYxMDM1ZmRiMzk5ZmY3Y2ZjYmQwOGQxODUxMzM1NDk3MzYzNWYxZmJkMjE5OTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jMdalRiWwAMaRuF62_n3zWaZy4bHlbHJmpIU9RN8NHySnWfFD2XVuRYnD_R3KRRyGo8ZcmjwdBKPWYDLehvJ_A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103911_64_24bc_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.745Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImRWc3ZRaWtqWDF2Z3c3RERSNmxNd1M4K2ZtSElHT3RJMUkzbXM5Y2psK0I1c3p1WWhUTngzMXNaazJXdjIvenQ1SzFRUVZRYjJJNllFTnQrOVFWZWV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDM2MzFfNDhfMjRiMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njg2ZTg3MDE2NDgzMWU1YjJiY2M5YzAyMzY1OTU4ZTM4ZWM5NGRiNjMyNTA0ZDI0MjI3MjNiN2NiNjY0YjgyODY1ZDEyODg0YzdiNDkwOGY2NjZmNWMwY2U0ZGNiYjMyMGVlZDc3ZjRiZjlmZDBhZDhlMjY3M2E3MmYxOTc3NDgxYTMyZGZiMzZiMGI3OWJkNzViYjcxMTRkZjFjNjgwZTA1NmFjNzQyMTFlNTFjNGEzMGRkM2IzYzYxODYyYTllY2RhNTM0ZGI5ZGQ2NjMyNDk3NTgyNjA2MWY1MGQzNDUzNzczZjYyMmU1OWUxMTljZDI2ZDFlMjY5ZTNmM2M3ZTE1Nzg3YTFhMzAwODc1Mjk1YjU3ZDdkMjNjNTExZTFhYWUyYWI5NGNjMWExZjU2OThjY2NhNmY2NDgwMTM1NjU4YzY1YTQyNjY0Zjg2YWE0OWZjOTY4ZTNjNzMwZTNjMWM4NTkyYTRjNjA1OGMzMWFmNDgyMGJhZWU1MzM5ODU5Y2NjZmZkODcxNmY0YTkyN2M5NzY5NGNhNTk1ZTE4NjFmZjg2ZDc1OWVmNWViZmUzZTU4YzA3MTEwNzM4YWJmMTY4NjIxZjcyYzE3ZTMxOGVmYTgxOWExYWM0OTc1NTI4MzNjNzdjYjJkOTQ5NmMzMDI3MGJjOGY1ZjI4NmNhNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9tWHbBj1BwslgqRINmrE1IwCBU1Ey7lTlUXWXYvg6Dfp-LIrJtXhwVmlKSv4nVYJgAubG3TJd_iWN2mRMew7bQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.749Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZET1kvUU1TenV2QU1VQWd2Rlg5M2Z6cjQrcGlNMEEvclhkRWhMQjB2b25QbnpwNzB4bEt4TEVFakJic3FBZE5DSUxPQ3g5VVhFQVN6MEtaOXBXdHp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDM2MzFfNDhfMjRiMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTZlNjg3OGM2NGJlZjZiYzg0YjA1ZDRlOTA5ZjUzNTA1ZDUzMzU1NDJiYzkwMTAyZGJlMmMxNDBjOGU0NDExMjQ0Yzg3ZjUwMWYxN2QzZDIxYjczNjg3ZjAzZmMwMzA4MWRhYWY3MzcwNmI3NDhiMThkM2U3N2RjYjZmYTkzNWYyMTU2ODBjYWRlM2U5Njg2N2U5NWE1OWYwNDNkOThkNTRjMDYxOWI3YWIzOGM4ZGJmODQwNTAwNTY4NWZhMmVlYzUxZGE1OWI3YzdmZTNkNjg1MmIwZWY5YTk2YjUyMTE5YjI4MmNiNjU1ZTNmYTkxYTE0ODk3NzRlYjk1OWMzNWQ3NGQzMThkZDc3NGQwOTM2ZGUxNGJhZTZkNWUwNjBkYmQ2ZmU5NTgzOWY3NmUyM2ExN2FjOTBkMzM4MWYyOWQ2ODU5ZTE2ZWVmNzE4YjQ1OGFkYTExZDIxODgyZWZkZmI0NmM0Y2U4MjE2YzgyOGYwZjU1ODliNzM0MTIzNzZmZDRhMmJlZGFjZTRhOTJmNjZhYWRiMDcwYmZhNTllNzVjNjVjOWQ3YjBiOTYzNGEwZWYxMjU1OGE1Nzc5NmUxYTU2YjJlNTY4ODI5YWUwNWM4MWVlYjBiNzFlNDU3NjgyMzRhOTJmMTFjODQ5ZDFjM2FiZDZjMTdjZTQ2NmNkMzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lxglxtsaX25I9f7TydSxTUEhQWTgFYwb3O-xSX2bjpLKSCvZl_LKzA7PozKfU9LpRXwUhFQJbES7Nb4iiPOPeA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.753Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImNucnNQa1VCU3liZXZsVWo3OW9IejNleGVWNGxFUmhGYkJhWHlDWlNYVGpwNXZwUWM3RHBmVXhGQkJsWml2SG9pQngrZCtQeCs2dVJuMzhnWVN4R2F3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDM2MzFfNDhfMjRiMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmZmMjU4ODdlZTE2OThkM2RhODEwYzFiNjMyZjA0YTIyODE2ODBmNTJmMWVhZWIwZjQwZWQxYWE3NTY3ZWMzMTllMmMwMmI5NDMzZWUzYjk0NjZmYTUyMjA0YmM3MmIyODJhZWRiZWRhMGU3OTNmYzIwMzY2ZTc1MGQ4MDg3MDI3ZjA5MTZiNjM1ZjBkZWY3OWY1MjI3Mzc5MjA0NTQ2M2IxNWEwZjY0MjBkZjYyOGY2ZmY2M2U0ZjYzNjMzZWY4NzE4YTM3NjBiNzA5ODQxZjhjMGFmNzEwZTcwZmQxMTQ3ODVmNDE3YmY5ZWQ0OGQ1ZGZjZTFhNzE0MmQzNjJlODFlMDg1OTJiMjhjMGYzMjIzNTAwODdlMzU3NGZlMzE5MjI0Mzk2YjExMTg0ZDZjYzA3M2I1YmVhYzQ2Y2U0ZTEyZmJhODhmNDY2ZDdkZjI2MDc2MGQyMmU5Yjg4YTRmNzk2N2E1ZGFlYWE3ODZiN2NiODA5ZjA1MzNkMTAxNTNiZGIxMzQ4YWVmM2RiMGIyZjg4MDA2NjY2MWEwYzRmNDZjMTgzZjMxN2E1YWU0YmUwOGQ1YzQ1YTQ2ZTU1ZmZkNTRhMDRkN2JmYTNkZTFjY2RkMWRkNWQwYjI3MWM3N2FmM2IyNWNhYzVjY2UwMWE0MzVhN2E5OTJkZTk2MmExZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uu7lGesn0ztNKdgBIvguG5T0LEr4YvCozXZyB3GD6NLaUSPt8zh308fDnp_GexD2yQulwW2jCZHHDaqgwLdeuA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.756Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InN4aldSVGMrODZCNnMyekRwRmZUYmxpTDk5Z3JKMG1idFNXQlljWjRHMFRwNSs0STAzR3ovNU1yNkluMXpiS01qWFVEMzIyQTFmNkYrc2t1dzBOWUZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDM2MzFfNDhfMjRiMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTUxODk1MmEyZDFmMDBiMjgxZTQyODgwNzkyYjU5MDQ4ODdmM2JhZThiOTRkYmU5MWFlNWVhOTEzMTA4MzA5MGE2YjA2MTM2MzM5MzkwMGRkMWJkYTUzZWUwNDg2NTRlZmEyMjcyOWYzYzQ0YTUwMGExMTQ4MDFkYWVlNjY0YTAyYTAwNDdiYWY2MDE5MDdkYzZmYTQ0MTU5NzZkMDQ1OThlZmUzYmY2MzgzYWJmMzI5NmFkNTg3ODkzNTIyYmUzODg1NjdhNDNiYTQzMTdhOGEwZDJmYzcwNjMyMDM1ZWMzMTZkM2RmZTc4M2EyOTI0N2I4NDBmZDMwYzRhZWZhMzFiZjNlOTM4ZTRiNGYxMGZhOTY1MzFiMWQ4MWYzMWU1NDM2MTViYzQwMzJjZDc5NmZiZmE4YTQwNDQ1NDRhMTFlMzhjZjlmMzQyMzMwMjNmYWE0MTQ4ZGRlY2U0NTNlZjZiN2MwMDk5YWU2MDI0NGZiZDViNDJkNDRjMjIxYTcwMWUwMDJjYzZlNjdiMTVmYTQyZmY1MWZiMDkyODA5NGQ4NWRjNzU4MzE0NDBhMzNhYWU2MDBiY2VjN2IyYWFlMjQzODVhMTI0OGFlYWYzODdiYTdlZTgzNWMzNzI5ZmEyNjhjYWUyYjJhOGE2N2I5MTE0ZTkyZWE1MDEzYzM1ODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1x_kfUO9t6OqYK5p0kI37BOYCYxY3MtPiY7ASCIwsfeZU_cHZY5wOG9udVs0KY2zPAyrq4vEfFyNgTDDgpp8gg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.759Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkNjTFJkK1BQZUdNMUlrdEhxT3FQNkhLMS9sUVBNT3BQalV0dTl5djRsRjVSTENXa2lWbmVybDduWFNRM0pZQmNEZmlzWlgwQVIyT2J1Q1hNYXdybUhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxOV8xMTI4MzBfMzFfMjQyNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTZmODQ3ZTZmMDI1MTQ3MmNjMWQ5M2QyMDdmOWVlYWVjMTQyZTE4Y2U5NzZlM2Y2ZWRhMjY2MDliZmUyZTY3NDI0ZjY3ODgxNWIwNDBiOGZkYzhmNjRkN2E3ZDc3MzdlNTgyYzI1Yzg2NzE2NjE4OGY4N2FhOTIzMWNlNmZkOTRiYTI4ODhlNzgzNzMxMTlkMTczYTUyMGMyZWVlZDAwNWQyODljYTQ0NDBhN2ZhNzBlN2M5OGQwYTZlZGZjZmU2NGJjODg4Y2VjOTAyMWVjYmIwZDJjNWM3NGUyYjBhODkxNmI2YmVjMmRlM2YzNmU4OTQwNjJkNTRkMDFjZDk2ODFjNDdmY2YwYmRjOGZlMDVmYWQ1Yjg5NWQ1NGQzYTlkOWYyZmU0ODFmZjkwNDRjYTA3NjFhYTI0NmUwMTRmODM4YmQzNjgxNzkyNDkyZTE0NzNiNjJiZWE1OTgxZjAyNzFhMzc0MTdlYTkzZWJlYzlhYWY1ZjZkMmJkMTRiZjdiNTk1MGUyYTg3ZjRmOTI3MWExYzM2OTYwNDBlMmU0OWUwNjhkNDVkMTZmYzJlMWI2OGVjMDUxOGViMmM4YTg4ODVjNmQ0MWQ0MGViMWZhYjk0NjdiNGI0OTU0YTkyZmMzNTRiYzNmZDA0YjZjNzY3MGE4ZWMwYWJjYmQ4Y2ExNTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ogDT_bWkDDOr52fBUyTq2MM8LyVTJn9mWmVZK_1DnxecfWnHr0dXh72WIKOqaLI75ZUWci2ija6URgETZXRR8Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210419_112830_31_2426_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.762Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IklQVWY2RlpKdWJzZ3V5cnJsbzJyd21QQWlaNUpaRmw5MTVzd2RiNmdobmJuRDdjcnExRi9rQ3JPSUVQUDB6MTQxZGJXTUkyeFkxUHVGUkhkYlhSUitnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxOV8xMTI4MzBfMzFfMjQyNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODVmYjRiZTk1MzMyY2ViZmQwODA4NzVlM2JiOTQyMzU1ZWY1MjIzYzc2MDFkMGJlZmMzZWZlNWMwY2I4MTE3YjkxZDUwZjNhNzA3NTdhYzhmYTEyZjcxYzYzMWY0MDAyYTI4NWRiODFhNTE3OTU3NjYzYTRkMDY3ZTE4MGYyZTEyOGMzZDUzNzkwMGZhOWEyYWYxYjNmZjIxOTZlNGVhMTJjNjAxYjcwMzg3ODU2NmIxODRjMzY0MThmMzBiNTU3ZmYyNWExNTYyNzQxZGJhN2QxYTg0OWM5NWFjMWVkZmE4ZGFjZjczYmI2MzAyMDVmZGQ1Mzk1Zjc1YTNhMThhOWNlNTM3MDcwZWY0ODcwOWI2YmNjOGM5YWFmNDc5NDlmNTg1NWIzYTk2YmM5OWMyMDI0ZDI4NzdkMDY4NmIxZTY1YWI3NmM3MzFkZjljMDRmODIyYTk1MTBhYTE1ZDdkMzQ3ZDQ0MmIyMjUzZGZhOTA3MDAzOGUxZTAwMmQxM2NlNGFmMGVkN2ZkMjgzZDdmZDQ2NzQ1ZDBkOTQ1Mjg5YzVmMzYwYzQwYzQwODlmMTE2MDFjNDE4Y2MwZTQ0ZDczNjUzZWRjMTZiZjdjNTllNDFjNTU4ODRmNzYwZGE2ZjFhNTEwNWJjMDFmMzljODRlNjMzNzNiYTI1NDI3MDI2NDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SB15_f-4IrQHqH6aM9E7iyaP-m1F1hkqKkbT9IwTGj8nolgb4YU4OBW-X52XBG560rO_PPYEEz3gfw7PhqEG7w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210419_112830_31_2426_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.765Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im5FemExOGVndkxwSTdzazBsVmFrN2Q2a3NmeVlHbTNrWVpkU3l6U0FRWHI1NlBXL3JDcGU4K2k0RDBZbHhPRTJIalBFSlBob0N1ZDZxT0c0bFVRd2xBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxOV8xMTI4MzBfMzFfMjQyNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzI5OWU0MTg1MGE0ZThkYWU5M2Q2YmRiOTgyMWU0NjY3OWFjNDQ3NzQ1NDdmMGJkNjFiZDJjMmIzN2MzOTc1MzA1ZTk5YmE0ZTMzOGQwMjhhYmZkYzkwOWQyZThlZGVlYmMzY2U4ZThhYjU0YmI2M2EwYTc3YmU3OTY0YjY3NTU1OTJhNTllYzcyMmVhNzhjN2E0ZTAzYmE5OGQ5ODIwZGYwZmM2ZmFkMmFmNzMwMThmYzk2ZjQxNjVhNThmYzU1MmU4MDA3ZTUxMWE4YjM2MjkzZjZjYzcwYTQ1NjFiNzdjZDM1OTBhZWE2NTFhNDllMzcwNGY5NzI0NmNjM2VlZjlhMWU4ZDU3MWM2YjU3ZTdmZGFkN2Q4M2FlZGYzZDI2YjUzNDgzOTE2Y2RjODFiZjJjNmIwZWE5MjA5YWZlNWJjNmI5ZDE5N2I4NGZjNzM4NzU2MzgyNzM1YWRjMjAxODJlMzg4NmI2NjNjMDQ2YjczZGIyNDFhMGMwMjc5ODM5NDI0ZTA4ZTFkMDhhNDFmMzQzMmQ3NzViNTRiNzE5NzZkODg2MWQxODU2YTFiNzFmYjkxYWJlNjBkYzQ5YThhMWIyODlkOTYzZjA1ZmM3NjY3OTQ2MDI1MWJjYjk1ZTRjYmU4ODE0ZjlkYzM2OTVhY2M3ZTY2MjY1OTVmYTNlMzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.77HZH45sal7xaOLtFhvLROp16Ffpk53wY1xHVOqp7KaByl5PCCmarORXV8UQmGT6YtEUEz9o0X1KSEJ6AM5Opg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210419_112830_31_2426_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.767Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjFmK2h6RHlRd05PNFQ5ejM4TTBremlXcG0wbG4rU3RxMEVRY2wrUWduZ0xGUEd4SmR2OGZwdktjOS96cVFtaW52eVZLZmRQdS9DNnJENmNhMEtsMThRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQxOV8xMTI4MzBfMzFfMjQyNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDk5Zjg3YjM1ZDc0M2NmZDgzMjBhYWRjYzFhMzUxMDJjYjMyNmEzMDkwMDlmMWJkMWMzMDI5NThhMjVmMTk1NDIxOTYzMzFiMTk2NDQ3MDE1Y2JiZmUyNWY2OGY2NTg2M2FjNmYwYWJlMjk0MTZhYjcyNzc3MzAxOGZhYjk4MmM1ZTY4MzhmMTY1MmYwODViZWFmMmFkMDcxZjJlYzZhYzc4NmVjMjQyZDE4YjVlOWRmMjBmMzg0YjU1NzYzNjc0NGE0MTk3ZGRlYTA2MmY5N2JhY2ExMzY0ZGE4ODU5MTFjNDU5NzNhNmE3M2QxZDBkOTU3NDc4NDMwOTczODMxOWEyODg1NWRjN2VjZDI1OTE5ZjI4OTZmZjU3YjIxZTZhNjRjMjE5YWM1MDdlZmRiNjg4ZDMwMmYzZmEyYzkwYTRhYzgzYWE0YTllZTcwNWRhODc3NmVmNDY4YTYyOWJhM2M1NzAyYzgyZGFiODM4YTEyNjgyNTVhYzI1NmZkNzE5MTc0YTk5ZGEzNTQ2YjM3NWVkNTYwZGYxMDdkMjg5ZDFiYjZkNTZkYzdjZjI5NTI2ZDY1ODU0NjMzZmU0NzI0ZjVmNjhlOTcxZDQyZjFmZmFkYWUxNGQzMDhlOTk5OTAwOTYyZTMwNDJkMDNiM2Y5OWI1MDEwNTEzMGJiMjk2M2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mZZkFjOynmewaSNh-6ckzVLtHyHftTtYL57CjXMuTJiSL_CwX6fmQ9JuO-j0zYyoK044Tc02R3c5_ldfBUjMzA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210419_112830_31_2426_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.770Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImFzSnVhTWxGdGU5UXhHcngyN2RQNnhoeXhRcURUSjU4czN4SVdqQXBCZW1oMU1xa2JHb29nUWhnSDFaNDFMZFNKM3J0aitoWFdNNjRwU0JVRlFuUkF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA2NTNfMzVfMjQ4Nl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDlkYzVhMWZkZjVkZmZiZGU2NGY1ZDA4MzE1ZWZmZGU0NmRmYmZhMTcwNDE5Y2RlMzU3NmMwNTcyZThiMzU2MTE4Mjk5ZGI2YjI1YzQ1MjdiZjk3ZmRjMDIyZGE1Yzk1MGZiMWE4NGQ5NjFlZGNiNjE1ZmM1NDVkNzIxMmJjYTVlYTJjMjg5MjY5M2EzYjI5OGM0NDg3ZTNlMjM2MTYyNzAyZmM3N2Q3MWI1Nzk1ZjI5ZTMwNjk3Y2U2MWVlZTg5ZGEyYTEwMTlkNzU5Mzg4MzRmZDY0OTAzZWI5NTAyM2RkNDg0Y2I4YTBjODcwNDBmODNkNGMwMTViMjNmMjg1YzcyNmEyNTdjOGYyOTRjYmU3ZmUzNDQ1NGUyNmVhYmJhZTMyYzM4M2Q3OTg3MDMxMjZjZWY0OGRkM2I5YWI1ZDZhNzQxNTliODFhMTFlNDYzNDU0NTJmNTAxZTBmZjRiYzUzNjI3YmI5NWQ3MTc0MTAzMDE3ZGM2YmM1Mzc1ODI1NmMxODhiOGJhYWJmYTM2NWRhNDdlMGMyNDdkMjc5OTkyOWMzZmM2NjlmMjM1NzIyOTQzMjZhYjFlMTVjZmMyYTk2ZDFiMzhmODdkOWI1Njg4ZTkwZmE4ZmEzZTI3YzQxNWZjYWRjZDkzODI4MTdiOWM0NGVlM2NiZjU5OWJjOWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.twN1te9mL_NUaZoq7d2J7YQ1tj60oqd5udXOelN6wd0LfVO9zwv3E4EILIF0ScZtaPt1DN0IAkaE9Eh6i1rOCw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110653_35_2486_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.773Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IitvdVozY3N2ZXM3RTBrcjdOVzQyaXV5NHdBaEdidHE4WnBuTm5XZURWeUxUazE4SE95UEM2WlY5eSt4anlJMVA1TlBIQ1hDY1hCRE0zYWZqb0pCTEVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA2NTNfMzVfMjQ4Nl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmVhMmNhZTY3NTc3ZGQ5MWQ0MDhkMWNmNWM2YTJlZDg3ZDc3OWUxMDFkOWYwMWUzMTAxNTRmODIwYzMyZGYxNGMwZWUwNzcwN2JmNzBkNjM5ZGJhYTMzNjRmMWY0MjQxOGM5OTNjMzJmNDc3ZmU1YmQ5NTI5NTIyYTEwMTk5MWU1ZDE3YzY4NzA4MWVmNDJmZDUwZjg1Y2ZhODVjZjFkYmExZWM1M2U4N2UzM2VlYjE2NWU2NzMxZDczMzJjNjA3ZDkxNTY2YzYwNDdmN2Q2ZDJhYTJkMjg2MTdmNjY0Njg4ODcyY2RiOWM3ZDM0MWQzZDBiMzk3Y2RmYjA1NjM2ZjRhNzEzZGFmMTJmY2RiNTI5MTE3MzNkYjViOTY4ZmM2Yzc4Y2RmM2E1NGVmOTAwMDJhNDIxZDEwYmMwOWRkYTcyMzgwZTc3Mzk0ZjAzOGM5MTEwODVkYzc1ZDhjNDM3MjljNGVhMTRiMTY5YzZmMTYxOGE4YjBkNTE1M2U1YzU5NWViYjFiZDE5ZjkyNDE4YTU5YmY2ZTdkMmM5MGU0YmJiMDMzNmFjOWFlMjBiN2M1NGRlM2UyODViNjQxMWVmYWI4ZjJiMDFkNzkzYTYwYWExZDI1NmQxOWFhYjdmNmRmNmUyNGUwMWE2ZTUxNGIyMzMzYjZlOTc2YjFjNGQ5YTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OhixKAEfAHDQLF6deyzeJtLNVbL7OMZDsQllnwXquVJ8cGN0COcf61pkPhD-K3N4NiyIJBSoSicdmf35UpHA_Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110653_35_2486_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.776Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ing5bXZFQ3g3VXVLT0l6a0xvZ2xaalhFdlRLZm1MWE9RM09aRUhBRmk4aTVVTW1UMWMwdW9nb0lWc3dRV05GZ0krTFdCV2ZuL1JnSDVINFdDY244QmZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA2NTNfMzVfMjQ4Nl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQxMWUzYTY0ZDM4YzM5ZWYzYTFiYjQ3OWY0NzEwMzBiNWE1Y2FjZGEzYjA5NmZmOTI1ZGRjNTg0MTEwZDY2MjMzY2FmNGFmMTVjODgxMDA4Yzk1ZjYyYTRjMDFjZjgxODJmZjJhMWE2MDQ0OTdmNjQ2MGFiNDM2N2UyZTY4NGY2ZjJjZDQ2NTFjOTk2ZTE0ZmE0MjY1NjE2YWIxMjEyZTYxMmQzYmM5NDM5MjgxYWQyNWIyMTQ2YjdlMzhmYmVkZDJkNWE2ZTkxYjRiZTlkZDA3NzYxYzc5MjAzOGZjYjcyOTA2NmZjN2M4YzNhYzBkOThjYzU2YmQxODg5OGUyYjYxZjU4OGNmYWQ1NzQ4MDExM2VhOTI1MzBlZjY2MTVjZjg5MzRlNDUwYzdlZjNkZjM0MzQ2NjdjNzE2NDhlNDhlM2M0ZjVmNzlkYmI4ZmQ4ZmNhOGY5Y2RjZjlmYWUwNTM0M2Y1MjNjN2VkZGU0ZWE0NjgwZjU5NGQxZTQ3N2MxZWRmZTJmMzVjZjA0M2VjYTVhNGYxMjliNGEwMTIyOThlOGNiNmIxMTZjYmY1MWFlMWRjOWU2NTM0YWI5MWQ2MzI1NWRhMjljYTljNzYzMTIxOTU2ZjgxZDEzZjg2NjJlZWY0OTdjNjEwZDg1ZjY2ZTg0ZjZiOTg5NGI1YjhkNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.N0sHWODNCj3fK7x8Yb0E0mJMGcpS_yV6YlrX45vQhEt-_ea4-rikuSfMXbP5wdVBsiVjp0f3IFOygXeDFlaqDg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110653_35_2486_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.781Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik81ZkhIS2FBc2tsMmp0QVJMNUpSamk1WEpNb1Qyd3hLelRXMnRkbzdudkdRZTBLNXJnV1Qrd2N1SXprS1ZuMUFWQWgrZGVYRnVjWUxrM2ZNZDN1dmZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxOF8xMTA2NTNfMzVfMjQ4Nl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGU4NDA2ZGQ2MDhjNTM2ZWYzZDIyNmFlMDhmNzI2YjQ5NTgwN2FkZWNkYmY4MDY4MzUyMmU4ZjlmMzMwYzZkMDljZDViZmI1ZmZkZjY3Y2FhZTk5YzM0MDYyOGM4OGQzMjRkNDk3YzQyZjQ5ZDA3YTVlNzM0MzU3M2Y5YzU5MjZiMmY0NzQzNGMwNGI1MzM1ZTljMTBiMDIxMmRkZGI5N2JjYjhkNzNhZjUwNmE1MDVmMjBkYjk3ZDkzN2I5YTc0MTg0MzBmODQ5NWQyOTc2OTk2YWI5NzlkZjJhOTQzYmRiYzIwMTg2ZDIyZmIyMWQwZmQ3YWY3ODI0ZDBiOTlhNDY1YjA5ZWU1Nzk2ODM2ZWQ3OTkzNjZlODM4MGNiNThlMDIzZWRmMTljODdmMWFhNjM2MmI0ZjI1YmRhYmQ5NWFkYjViYzkxZjBiMDhjNjg4YWJlZjRkZWFiOTJiOWM0YjViMDhlZjA4YmEyMjY5ZDcyZTk4ZGY5OGMzN2EwZTgxNzE1OThiMWQ3NzIwMmZhY2VkODI5OGVlYjk3OWVjMDg5NWQxNTY3Mzc2YTAwYjA3Zjk4ZmRhYTRiODNmYjc1YzE3OGJhNTc4ZmViNjk1M2MyMzJkZDVhMDdmZjhiYWMyZGIxOWMzZTlhZTRlYTdiNTEyNTM4YTAxODRmMDYzNWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._5ObPmYC6LGpWLxDieRLVvQC9hP_GZMRdwRUXBBU611utTTsNK5CNqhnEj50LJr27VOVKesbdg2ghPYuHymRJw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230418_110653_35_2486_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.787Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlIZ29hdmtZT3MxS2tRakxLMEpIOUhJeHNaSnRnVjc4eXFlNzZENFFxNldSdXBLR1ptWnJZdFlHc3lLeGFjSi9iTy9KMUtNcEVteEovT2pNNFhKZkJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzVfODJfMjQxM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2VjYmRkYjYxMGFkZTY1NzA4ZmM4NjFiYjZiYTNkYWRjYzc2NmY4MTE3YTYyZDliZWM5MDNhMzBjMDRhZTFjZjZmNzhjZjdjMDBiNjYyMGVjMmFlNmJjNjg3MzdmZWM4OWJkOWZkNzRmZGRlOGE4OTVmMDllYmU3MGE4NTUxMzNmNDExYTBiMDZkMzJhYjZjODkyOTY5MWExOWYwOTMzYmRjNTVjNzM0ODU2MzM0Y2VkYjIxMDg5MjBiMzFhMDVjN2UyZTNiYTQxMWY3YWY3N2I2Mzg5YjMxYTU5YmQxNjg4YWVmZWM4NjQ0MmQyNTk4ZTFlNDMwMjZkNGU3NWFiNjRlNzU4N2I3MmRlOTAxOTUzOGViODY5YzNiZWYwMDhiOTg2MDAyNDFlZWIwNGI1YTRmMTM1YTlkNTFlNTlhODJmZTVlMDA1OGZhMjc3NjM5ZTI5YjgxOGIyNDM0NzA5NmNiNWE2YTgyMzEyZmMzNmEzYjNjNmY3M2RhZGIyZTE1ZjZhYTVmYzFlODU1ZGJkZjI4NzRjMmJhZmNjMmIwODNkNGJiZThiNWZiYzM4ZGIzYmMwM2QzMDMxMTdlODkzMTA2Y2E2NWI2MGU1MmFlOTg0NWU1ZmEzMzM5NjRkNTg5MzM2YjY0YTc2ZjU0YjhkNjkxMTkyN2YxMTdmY2U5OWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qKFV8oDNfXyeJMkh07Yfnz1yogdnyH4jiC7i81lQhuG7H0EydWYIvh38iivVANm_QWeB5BnTRjSyQP2n7fGJbw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112635_82_2413_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.791Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InovUlg4U2NqYkdzT0hpbDMwb2RMR2x2cmtpSFJaSDZmeXdxRDZ4VTB6TjBXeDRFSi9MRE5udzAwK1lJcmRJeXNXa0xUUDd1K2VJdlRKcWI4QmdiWWxBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzVfODJfMjQxM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzRkOGEyMWU2OWU2NGE4MDJiZmVkY2QzMjFjMGY2MzViNDRkYzQ1NmMyMjQ4YzVmMWU0MDg0ZWIxMzdjZDRmZDkwYjhlODYxZWYzNGI4OTUwN2I2ZmI4ZjU3YTM2YmVjMTViYjc4OWIzYWNlODVkM2U0OTY2ZjBkYzBjY2IwMmUyNGMxNmNhZWYzZTI5ZDdlMzY5M2ZmNWMyYWMwNjVmMDM1Y2M0MmQ3MmM2NGY2MzlmNDMxYTRhOGRlZDJhNTZmYmZiNTc5ZjQ2MzViY2VlMjNlZThiOTk1ZTJiZTcwZTJmODg2MzJkYWIwMmFhMWUyMDUyOGFjMjAzZGMzOGE3NmE5OGQyNDU0MzYyMzNhNmQyZTI3NzlkZTQyYTNjNDA0MjM0MjJjN2M0MTUzNDJkNmVjMDA5NTdkNmViMzNmYzg0ZDFkODc0ZGZiYmY5Zjg5MGY5ZmNiZmE3OWY3NGFhZDgwODllNjI5NWIwMjdiMjY5YmVjOTliYTEzM2MxMWFlNWE5MDU0ZGU4MGUxY2NkYTJjYmUwNTRlMWJmNzVjYjgzNmEyNTdhNjYyOTQ0MmU3MmUwZDZjMmYzMWRhOGI2ZTk5Y2JlMWI4NjUzM2U2MGRmMDE4Zjg0NWIyMjY2MmE5NzEwYWRiMTJmNDU2NjllZTgxZjg4N2FlZTRkOTg5YjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YlWkqkKxez7FpcX2lfhtdo1NBEaYBia0HotsfbHXELRwohciKdbpuoeSijBvxyvgDayL2m3jBMYqTM3WScAeZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112635_82_2413_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.793Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxrVmp5YmhQQWs5M2xnaUJ5cWg5cXBZdVJhT204R1p0czBKQlFJVkxieDJ1UGhmUlVMdlFxakc2RERRODRCTmpwY05zSmdKT0E1SW1WTjNlYVpGbmJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzVfODJfMjQxM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWY0MWQ0YmRhYmY2YjZlNWVlZmJhOWFmZGMxOTgyMWVjMjliYTdmYzZkZGI0OGRlNTQ0NjA3MWNiNjgyMjJiYjU0Zjk2YWY3Yjk5YjE4ZTJhMzAwN2E5ZjRiYzdiNWUwNjQyZDkxMjk2MDM0ZThhZTI2NTM0ZjE0ZDA3NTI4ZmRjOWEzOTc3ZDI1OTZkMjNhMGMzMWUyNjE3ZGIxY2QwYTZkNmQ2MGYwNzBjNzkxOWViOGMwZjJlOTU5NzExMWNlOTJiYjgxNTUzZGQ1Yzc5YmVhMjczOGI0MzU3ZGNiMTExNGRkNWVkNzczZGYwYWE1NTAxMTRmNzZjMjEwNTMzMzgzN2ZlYjRkNjk1OTQxZDE2OTM2ZjFkYTNlNGMwNTFhNWU2ZWU4ZTY1ZjNjZWJjMDgxNjg1NTM0ZjI0MWQ2NGY1ZjMxNDhjODc5M2E2NGM3YmNiYzgxYTQ4Njg5ZDIzYzcyMTZhNzA4ZjIxZTE3MmQ3NTQ5NGE5ZmI1NDUyODY4MTE3YTA1ZDE1ZjQ5ZjhmODQ0ZWVlMGQ4OWZmN2Q4ZDM4M2I5MjJhOGU4ODNkMGU0MTc1OTU4OTllNjExZTk3YzNhZmU5OGE0ODk5NWFjMTExYzEzOTQ5ZmQ5NzI2YmEyMDUzNjkwMTI5MTk0NWE3YWFlNjY4MDk5NGRiYzA5MjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ImAaTnFk85_FdMpNBQv9S4ZUYP4Ch-ppaRvbfZBWfDPUMN3F1E8rT7n8fRq-JvSW_c_ZeldoJ5Ej1kSqvkbgwQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112635_82_2413_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.797Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ik1FTHlXV25MdzNaNkFsT01uTERKeXRtTFg5U2JXOUJ1QlpCS3B0VEdlSDJ3RWpDcThsa2h3Z0tQdEg2bVFnUW5OWUlKZS9IMlhSOFVNVDdlWXYrUXJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMwMV8xMTI2MzVfODJfMjQxM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTA5MWEyMmFiMDQzMmQ2MjhlZjkzMGZiNTY5MGQ1MDI0NDg0OGMzMWRkZTM3YTU4ZjViZDMyYWZkNmEzMWY0MzE5Yjc3N2UxYWU4NzM3ODYxMWY5YWNhMzYwNGIzNjQ3NmI4MGEwYTdmMjkwN2U1OTllOTcwOTcyMTA5NzgxMWZjMGZkZTY4MjExZmU1NTU2NzAwNWFlMWRkYTBhZGJiNWU4ZTk3ZjIyZGZiY2ZlYTQ1NzdhZTRmMDliMjc3ZmRlMjJlZTFiZDk0MjU2ZWI0ZjgxYTlmYmEzNDkzNmNiMGQ2MjQ5NjU0YTZlYmYyMjBlMGJmMzZhZjI0NjliNTg2MTVjOGU3MjNmM2E0Nzk3OTA1MjQ5Yjg3NGU5ZmViODY1ZTYwZTExYjZkYWQ4MWZiODY4MmI4NzJhNzFkZjdhMWViYTQ3NzIyMDA1ZWI1MmY0MzY2NGNlZjg3M2M0OGExMTU4ODg2YmMzYWNiNWVkY2U4NGIwMmRkNzQ2ODI0NzZiZGU3YjM0MjllNGYxODM3MGExZDFhNjBhYWVjMzFkYjY5MTI3YTZlN2JlZDc2YWVmOWM0ODU5OGEwNDNlNGMzODRhZjdmNWRlYzQwZjgwZDc2NjU4N2MyZjUxYjc0YWE0NTE3OWUyYjc2MjliNmNkY2Q4ODRmYzJjZTFhODRmZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IpS1RZTGNRvmlgYTz1Sy_gcdeIbDkzvyFIDkuUr7MpTESxjIyIKVRbtbqALcVQ00OGmwVonPHTYkLaX6b2rFbQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210301_112635_82_2413_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.800Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpsYXoyUEtxNUIzeWpIS3lyNEFrb3Q1RXRvTk9tVWJ6UUkwWlJnbjhQYXl1NmVvUTdld2hTdCsyV2RqUEZZYnFLcEhrQkhFbGRkR1F0RDFKYXJEcFZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDdfNDBfMjQ3N18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2IyZmJhZDQwMTU4OGRkYmM0YTY5Y2Y0MGMzNTg5OWQ2Yzc0ZjUxNjdhODkzODE2ZWI5YTljNjhlMjdmNjg0Yzg3NGU3Y2Q5ZTdkYTkzMmY4YzM2YzIyMmMzNjhkODUzNWI5YjY3NzYyNDMyMzUwZmFmZjJhNzk0NjRkMzdjNWNjMDg4YTlkNTViZjYzY2IyZTVjOTY5ZWVmZWQ3ZjU3YWYyZTkyMDBhYmMwMDc2ZjlkOTAyOGNjNTAyNWM4ZDE4MTg0NTcyYjI2MTg4ZWZlNDNjN2QzNWFiZDI4ZjViNGZkMjcyYmU4ZGZiNzgzMjBiYmJlOTg5NDBlNzZhZTAwMWM1MWY1ODVjOGI4YzZhOGEyY2ZmZmE1MjlkZTk4MjBlY2Y5N2M3ZTJhMDhjYjk1ZTA4YmY0MTM0YTQ2MDgyM2FkZDQ5ZGU4NmQyNTcxODkwNjllZWM3ZDc0NWNhYjUxOGY1YTlmMzk0M2ZiMTI4Yjg5Y2U1MzQ3MjA5NWMxM2YyN2Q5NWY3MzJiZDYyZGEyNDExZmU5YzQwNDM2YzI1ODc1ZjRjYzBkYjNkOTExMzUzZjgxMjlkNTBhM2FjM2I5OGY1ZjQ4YzZjN2Y0Y2Q3YjU5MzJlOWUxYTg1M2I1ODM2NDFmNDE5YWUwOTJjZGM5NmUzNGU0YTI2ODhlODM5NjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yxmyq3CZykvruu0fPSTAFey2vqJDmQec_itzBu76JZX-m3SN7i0MBwlTsAu76nEC8klwGsDW7d4X4iKWcodO-A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110647_40_2477_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.804Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IktHYXlLdE4wUUVaOXlJalp4RW5Zb01xVWMyaEdwdjBhTVFiNmsyYlhqL29CZ0VHZTdRb3l0c3g5cGczWC96eSt4Ynk4U09LMG10SFBoNnlqN28zNHhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDdfNDBfMjQ3N18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDg0NTI4OWM1N2JmMDdkZjJmMmMzMmVkZDAzMGVhYjRhZGJmMTc5ODhlNWY0NmY1ZTUyZGVhM2NiMWFmZDJlYTlkYmQwY2JiZGQ0MTMzNWFjMTkyZDQyZDVhZDgyYmVmMTJjN2I3M2I5NmQ3N2I2YzE1Zjk3OGUzODAyYjNkNTU5MTljYTZiYmFiZWE4MDRkYTFhMTM4OGQzZTJmZjNlNjNkZmM1ZmE3YzAyOTIwZDYyM2U4YWI1YzY0YTA1Nzk4ZTVjN2FlYjM2MTI0ZWVlNTdlOGUzMzMxMTA0MWVjMDczYjU0MWU1NGU1ZTA2M2RhYmNmY2FiYzM1NzdjNjM2Y2Q5ZTVjMTQxYTUwYzBjZWJiMzQwOGIzNDVlOGRmMzE0M2M3NmViYjQ0YTdkMDM0NWVkN2U2Yjk0YTlhYzM4MzQwNWUzZGRlZWY3MjY4MjM1NDUzYTY2YjQ2MDljNjI2Y2Q0NmI1OTIyZDMzNDg0M2ZlNTcxN2U4YjMyNDM4ZjM4NGQ2YTc1MjNiNWI1N2Y4ZDAyZDAwMTFkYzg1ZDMxYzM5MDU1MTIxMzUwM2M5NTVlYWU4NmU0OTM3YzU0M2JkMDdkOTI2NGNkZmE3N2Q0N2JmOWRkMTMwMTBjMTg1Nzg4YmFkNTA2ZjM3ZGUzOWJmZWZiOTRlNGYzNjA1Y2Q5NzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ag99YBcvy1Sc0oGut5BY9npO-Xe-Nb1f0oeJhyPXyR82a_OLofnDyUCgd_8X7-sxnJi2HIxEnOM7W0RXUM0RYw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110647_40_2477_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.807Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Illzc1ZoMkIvNnM3TWZuS1JvMWptcGF6TmRsbEtjcjZISCtiSWdVSEFwNUFYN24yRHlVbDQycHBBMGZXdXA1YUo5Qks4OUZTYXFqNG9Pa0ErMURTVWtBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDdfNDBfMjQ3N19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGI1YjY0MjgzNzA1ZWQyNmIzOWYyMDg1NmIxMGVlNDY4ZTc3ZDI1YjI0MDNlNTFlZjMyMDM1MzdkNjdmMTllOTQxOTdjZDZlZWUxMTQ5YTc5MWRjMTJiZGJhOTBmNTBjN2ZjODg2ZGM3ZjIyYTJjMTRiY2NjMWM4NzgyYTIzOGQ5M2U0MWM4NDgzZmFlNjUzMzAwMDExYjgxM2VhMWM0YmMwY2RkMGRkN2M4Y2IxMzhkNDVmZDk4ZGMwZTA3OTJhMGFhOWIyMThjOWM4ZWFiODdlZTdiOTVjZDRjODgxMmZiMjQwMzAxNjViNzYzMzRjOWM2OTZiZTc1NGVhYjc5OGU2MWVlMmMwYTU1M2NhZjQ1MjBiNWE4OTE4ZTU1OGJhN2VkNmM2ZTJjNjRkNjBmMTQ5N2M0OTYxNTJkNzI3NTE5YmYxYzI2YTkzOTAwNzNkNGU1YThjMWQ3Y2FiZDIxZWNjY2RjY2RmMWU4ZGVlMjViNTQ5NzAzMTNmMjI4NTJjOGYwNTAxYTM3YzA4YTA5NDQwM2UwNjEzZGY5NTNhOGExYTRlOTY1Y2UwNDNkMTJjNTE4ZWY4YWZkMWY4OTAwNGZhODU2MGRiNjU2NzUwOWJlODg2MTg4OTYzOWE0MjNlYjdmMTRhZWJmNTgxZjVmYmY1MjE0YzY0MWRiZjkyZjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OiNEMV_Upkj5ADE3oTmplrpe_XUtxKf_l_3P_AKvrTp61J1YTXooHoaJk9T9Rk_VMdYdZoRouYwtmSMFlmpRpg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110647_40_2477_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.810Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InFibUtUU1R4ZGlES1lxeW9yT2lOQUlzMHNEUzhQSHc0ZkZiV2lOcWtuVG45Z0x4SHMrbVBjUXFCUDJDbHo1MlkrYzZSb1hTVkR6Uk1GWTh3T3RQUEhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDgyNl8xMTA2NDdfNDBfMjQ3N18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTUyNDBiN2JlMDk5ZTJmZTg0ODUwODZjYzQ2OTg1NWQ4ZTUxNjI4ODQwMDVlNmNkOGU5YjM3ZjZiNzY2OTkyMzNmM2Y0NzUyY2FiYjVlNjU3NDAzNDQ3OTI5NDJjYzA5ZGQyM2I3YTQyMjdmNGVlNjliMzExYjAwMjNiNmExY2ZlMzMzN2NiZDc2MTIxZTIxNzY1MjhhNjM1OWMzNjA3YTczMTRjOTE1ZWM1ZGQwM2IzZjA5MTcwMjA5ZTZmNjY5ZjE4NzdiNjU0Y2NiYzEzOTJiNTFiZGE5ZDFmYTNlYjhhZTIyNTNkODNlYzIwNjc0ODgwZTYyYzRkZDE3YTNhY2IxYjk2ODhiNmQyZWMwNzYyYTA0NGFiNzRhN2RmYjBhZjBjZDdjZGJkMGRlMDc3YTE0MjlhOWQ0MGYxMTNhM2Y0ZWM0MDk1YTQzYTRhMzA5MjBkNmNlOTU1YWY2MDdjYWRlZDE5ZTcyYWU5ZjZmNjM4NzIxNzMwNzYzYzhiNTk0YmJkNGVjMzE0ZDFlZGFiM2JhYmJjMTM3ZWQzOWQwMGY4NzU4N2ZlOWFhNDBlZmYzZDBmYzI0NDUwOTE3MzY5YWYyZTEyZDI1Y2ZkNDIzMTlhMDA1NzY2ZDZmMTQ5MGJiYTkyNWY3NGNiYTBlYjJlNWFhNDRhYzQyMjcyOTFlODRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MnWfTOnCZql7L-RVNbWdGiC4Zwd4dYxWRIUlLjPTVUF8AlFKcgE72vuLQRgOmlKRW4SChzDXAtKBe05L-sRRVg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230826_110647_40_2477_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.812Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9XdVUyeVZiMktsOXVKWUZLTjdJUC82MTU2SVVlTmRMRFBrTklIVDFiRkczdllLZThRVFpGN1hSa09OR3FNbDE3Z2w2dGFVRkRjNDlXTHB6bzRLUzhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDdfNTFfMjRjYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWJhZWQ0YTcyYTk0NmQ4ZTUzODAxMDdlMzBmNTFiN2RlYTI4YjYyOWRkMGQyM2Y3NWE1MmZmZDI0ODAwNjFkMGI4MGRhODg0NTlkNDBkNjM3ZjIwZDcxMjlkN2M0MTZkMGUzZDdhYmNlY2I1ZTI5MWViNTg5NDAyZDE5NDY3ODU4MzM5NjM5MDM0ZDJhMjA4NDNkMjc1YWM2Y2UxZjJlMmZlZDNjNDFiNzI1YTk3OWM2OThhNjlkMDZiNmVhODdkNDM1M2NiODFmNDhkY2U2N2U2OWQxYTM4MGVlNTVjOWM2NmJmMWVjMTJjMGU0MDhiZGMyMThiNWY1OTZkNmM4ZWJhNWM2MTM1ZjEwMDkxOGNlNjRkZjBjZDNkZjYzNzRiMDhlODYzNDEzZWIwOGY2ZmJjNGY1YjczNzRmYjgzOTkxMGMwMjM3MDc3NGFmZjFhZTk0NGI1MzM5MmFmZWYxNGE2NDMyMDYxMGIwYjYxN2NlZTExMGVmMjVhZGIyZTQ2ZjEyMGI0YmJmZGNlZjE5MzYyMmFiNGY5NTljNDVjZTE1YzJiOGRkODYzODMxNTUzNWI3MzM4ZjViMTg4MDc5ZjVkNDk3ZDFkOGVhMzhkNWE4OWU4MzY0ZWM3NzY4MjRjOWQzYjU3NWRjZGI2MTFmYzJhZThmNTU2MTE0ZDE3YmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.l5OMjn9sKT2hU5Elfq1h1JNYoV3C2ZxddhS_ck5IAdM3aGCdnGb2cCiijY2VCNUs59dBCaevu-UsH_OH4LtbBQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103247_51_24ca_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.815Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjlsZk1pZWo4ZWw0WEo0THFkUTRiOHdpaVozaG0vUWhEQjJBUzdNV2x1bmgzZEZWSXVIMVJBSGJOOGxKWXBDdXhmN3ExVDMyY2NBNUp1WThCSE5yWXpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDdfNTFfMjRjYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTRlYjc2NTA2NjRkZDZlYmYxOTE1NWI3ODJjNzhlODU3N2IwM2NlYjc4YzEwZTcwNTY4YjkwMDVlMTY4ZDNhMDIxNDY4NzcxZmYzYmYwZWY1OGM0MTM2MmQyMWMxNDIxMzQwYzA2NGM0YjhiY2U1MmYxZjM0ZmNhZWI4MTRiZjc4YzI1NzliMjM3OTZhMmRlNTg1ODFhYmFkZTEzMDRlZTczNWRkMjg0NzMzNjBlMzA1YzFkZDQwZTg3NmJmYTUzNTAyMGUyYTk3NTVkYmNiMTYzMjhkNTMwODJkMzVkNGI0N2RiYTBhM2MwOWUzOGE4ZTA4ZTA2MzlmNzI3YzdiMGY4Mzk5MDYxZjZmOTE3MDNhYTI3NjBiYmY2MDc3YzczODZhNzI4Y2Y4NjUxZmQyOGYwZGVhMGJhNmYxYzY1ZGUxZTEzZTA1MDUwZWFmMTRiZWRiMjBkYzk1NGJlNDM3ZDY4MDQyZjM4YTFkOTdlMDc0MjkzMDZlY2YwMDM5NjQ1NjlhZDMyMzAyYjA5OWFiMzZhMjI1ODgxMjhkZTJhOGE5ZDdiYTgyMjVlOGJjOWM2YTUwZDc5MjljNTMwZmViN2VkNzE2ZGY1MjliMDViMDhiZjQ4NGQ0ZmI4OTliYzAyNmFkM2Q5NTgzMmMzODE4ODg3YjQ4YjIwOTMxNjAzMTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Tx3DrshuQhGX-OdD3wb4S7xiWDyMR2kHDBsF8xCL43Uc-8ynJLxthOMSICrCpKODGEctYzC0CgjhbPVYjTa-OA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103247_51_24ca_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.818Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImxFNG1QaWJueGd4c3pYeFBPSVkyVU0rUGFBaUI4ckpDWTdxZ3N2eGIzLy9wZVFQQllPdmQ4Z01TdWFRcjZreGNGb0lFZ2hBRCtHSnR6cHc0cVRTcVJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDdfNTFfMjRjYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQzNTVmMjQ0MDZlZDFmZDkxNmU3ZTZmMzIxNjRmZThhNjdlNThhYWFhZDFmZTkwNWNlNjg1YzNjMmU3MTk0ZGIxYTU4MjVmY2Y0ZjQ1NjdjZDA4MjM4OTM0ZDAwM2EyYWNjOWFjNjAzMTZlODUxNGE3NzUyODdjN2MxODZjMDY4NmU1Y2Q3Y2I0MjE5MDI5MmU4MjZiMjZmZTNjNzJmMjk3ODhlY2ZmNmRhNWFhYTk3NTYwMGJmZDc2ODQ0OTgxMTE1YmY1YzgzZjExMDYzNWJlNDE4MzlhYzI1NTVmZGI2N2IwZmI0ZGFmNTU2Y2JjMzJiNGE5ZDEwNWQ2ODVkMTM3YWRhMTk1ZTAxYjRkMjhhODNhODI1ZGY5MGM5OTk1YTExNjJjZWYxNjEyNWRlMzVhZTMwNDFjNzc0MmIyZTMyZTAzMTE2N2ZkNmNhYzNlOGEwOWRkNzQ3MTczNjNkYWUzZGM2ZWZiZWExYzlhZWZmOWM0MWQwZDA0MGQ1YmQ4MTUyMjI3ZjgxMzY4MzRkNzY5N2UyY2M4ZjgxMGE4NTM2YzIxMzFlYmMzOTQ3ZjE2Yjg0ZGY1YjEzODA0MjU0NWNlNTY3MWUwYjdhNmY0NDQ2NjViOTA3OGQ2M2EwZGU2MWU0MWUxZTkwZTY1NGU3NzdlMTdmNGJiZjEyN2U1ZDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.x4JkPtr2mCIAO48ywYgav1t05CXB2oXPliyi7e9BLMquMrgyG_eSdqdeZiVbXxMNZZDTMFz-poEWws-__VjA4A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103247_51_24ca_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.822Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImppMDFEbFdhajRUS002ZS85NlpoWEJHbW1vbGJ5QUdXN09CSEVFNTczSCtHSHNGaWxsNnhsZm1kdllRdjFEeTZBM1cxY2hmQm5aaUdHMEZuVkQ4cEJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEyNF8xMDMyNDdfNTFfMjRjYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTcyNGQzNTI4NmQ3Mjc1YmQ5YzYzYzY5YzJlMDM4NGFmM2I1OTY5NjE0YWQ1YTFhYjY0Mzc4NTc2MWZkYjVjNDZlNmNhYmFlZjkyNTI4N2Q5MWRjYzFmOWRkMTZiNjVjNjY1YzQ1NDIxYTBmY2RhNTQ0YjczZGMwYmQ5MTI0MjAxZmEzM2FlOTc3MjcxNzcyYWM0OGNiZDVkNzI0ZjU4ZDgxYjEzYTRhNzNhYzkyYmUxZTgwOTAyNDQxYTc2OGYyNTRkZGU5ZDcwYTIyYWI4ZWExYWRiNTY5OWRjNjIwNjdhZTkyNWU5ZTk4ZGRkZTY5NDQ5MjM3YjA2MGRmOWYxOTdjMjM2ZWUxYTA4NzVkNWUzZWY4ZWZiMzNiYzI2YjczNTY4ZWFlYWM3M2ZiZWQ5Njk4NjVmZTZkMzY3MWQ5MzU3YTI1ZjRiOGU5MWNmMzkyMWQxYmQ4MzJlOGIxMjdlZDM0ODNiMDgzN2ExNTRkODczMTI5ZGI1NTEzNWZkMDU0MDU5NjZhZTIyZDkzMTZhOWFkNmRkNzY5YzQ3OWVkMDM4M2U3NTY2YzI5YTg3OTk0MjhhNDUwZjExOTMyNzc5MDcyN2ZkMTgzOWZlY2I2MTdiNjBmM2ZhZDU3YzVkMGZiZGYxZGY1Yjc2NWFhNzA2MjgzYWE4Y2Y1ZWJmNzAxZmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pLyfTgMFQIK1CkW4Bhk8AYA44031GHOKea_AWRHhyZgXQBRIbpALahtxD1JBIF1jlAyZLCA-ktpHzvHU5NDtUw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231124_103247_51_24ca_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.825Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjBSTjgyVEt3dmlURGtRSzhHZDByZ2d1bS9teU1FMjBFMHh1NVRrZExhbjRGNEpOMHQrdlZDNmNvanhqMHlSZzZlUWpMTjhCTEVTOEZiYTBOcXBaTEt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMl8xMTAzMjFfNTlfMjQ5Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTVkOTRlYmJmY2E4MmIwZGUyMjZlN2E4NGI1ZjczYTk3MWEzODY4NWI0ODRkN2IxYjNiZTVjYWY4MTFkODdlNjJhOGZhNWEyOWYwM2Y3NzkxZTY4NjMyYTQxMjVjYjYxOTc3MzcyNWE1MmMzZDk5YWI5ZDE3MTcxYmU5MDYxNjRkZjlkODA4NmUwYjc5MWMyOTI0MThiOTE0ZGM2OTIzMDlmZjEyZmQzNmZkOTA1NGMxZWIwYjk0ZjljMGU0ZjllYTNjOTMzN2UyNWFiOTNmZDMzOTI5ODU3YTQ5MDgyYTE2MDRhODRjMDUxMzFmYTk4MTYwYWNkMDJhNDdiMjU3YmNmMjBmOTM5Njk1YTgzYzY4OTEzNGRlN2FiYjcwODFiNWNjZGQ4ZjRjMzMzMTIyOTc0NzU3OWIzYzEwMjg2NjcxMjAxOTQ2MTU0NDk2ZDc0YTRjMzgyMTFhM2FmZmVjOTdmMzhlYTQ4NzUxYTIxYzZiZTNkMDc0NzZiNGI2NzUyY2U0NjEyNWUxNjZmMmE2NjNjZmM2MzY3YTI3MmIwNjIwNWJmYWYzNDBjNDAwODIzYWEzMDc4MTQ0MDcyZGI2ZWRjMTI0YTZhYTQ4OGM1M2FhNGEyY2EzYThhMjA4NjY2YzI2NGYyMDA5ZGI0N2RkMGM0YWI4Y2M1Yjk1OWM1ZWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.T4kZrnG5vOzkAyMw194-4D52oHqkOkNpaX_ewYflRbtD6Vk642I5tn9PsqudyeGQpJYH_WGruIL3JNw9mhcJVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230422_110321_59_249c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.828Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InRwaFhIWENlRUVyalBTZCtuL3oxc3VnUmdNS3JOZEpHK3NkaFBzcHM1L01WWDRvcnNnRkhha05QVFVscjNVdGNDdEk0OWhiSkNDWHYvYmhLcGV0bnRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMl8xMTAzMjFfNTlfMjQ5Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQ4OTExNDUwZDk2MWM4NTBlNTZiZGRiYjlmNTA4NTNmMmEzNTEwNWU5MTZiYzUyOTNjMzllYzcyOTc4MGY0NThhODBjNzM5NzA2ZDY2MDZhNjgxZGZjN2Q5MDJjZjU1ODUxZTE2Nzc1ZTczNDVhNjgyNjUwZTY5YmI1ZDQ4NTBlNjQwZDBlMWIxZDEzNmZhYWY3ZTU0MzE2ZjQ5NWM0ODdkYzg2Y2FjYzE3NDE5YmEwYzdmODY2ZjA5MzhjYTZjMTU5N2ZmYzRkZDFmZWRmOTFjMTM4M2Y0YzRjMjUwYTcxN2E2ZTdiZThlYmE4YzQwY2NlMDNiYjlhZGI5MDI4MmZmYjdiMmU5N2Y4MmM2NWQxZWY3YTk4OWI2YjcwOTY1MDA5ZWJhZDFlZmQ2Y2VjNDc5ZTc2NzIwNGY5MGEzZDAwYTg4ZGMwYzkzNDliYTUzMjljNTZlZTQxMGY3YTJiZjNjZmRjODkxMWUyMDFhMTM5ZGRiM2IyZjEyYjEwYmQ3OTdiMzgwMmJhNWZiZDE0MDIyZjExMTRiNzA3NjEzMWNiNmY1YzhhMDljMjk0MjM5YzJjODg0OGJiZTdiYzVmYTg1NjkyNDE2NDJjNjFlNDVjNzg3NDU3YzE3ZTAxZWI5ODgzM2NiNGVhNzI1Yjk3Mjc2MDQ0ZjI5ZjkzMTg3N2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.n2JQTNNuAZpZTQx8eGPdyriRkVF3hdgIhWeBGEVfDd3Bd9gq-kDi_tJ6sSN4F3e9QalWf4O2j2MzFcm_sp6NlQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230422_110321_59_249c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.831Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNKMkNBR2ZUTWRkNXhZTHBTZnBYSitOeTlIRmozMEY4Q0o5Q1VpelRHNkRJcitkQ0ZGRjJkMGlhN1ltbWVLTTlETGR0dlNSdzkxK3FVVnJrTFpidUdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMl8xMTAzMjFfNTlfMjQ5Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQ4MjAzNWFmMDQ3ZGVmZGZlZmZjOTBmMmM0NjY3Y2NlM2Q0NWRlYzNiMWMwZjI4MDk4OGJhZDQwMmQ1NWQ1NmMzMjk5ODBlMTEwYzUyM2ZhZmY2NDAwOGQ3MDI1MmM5MDYyYjVmNDBlNTM1ZDdmYWVkMWE4OTA4ZDUyMzZmZDM4ZTllMjk1YjcyYjIyYTY0M2EyYmE2YTBiN2I4NWE3MjlhMjMwZDUyNTQxYThiNjcwYTA2ZDgzM2Y4Njg4OGViNmVhMWQ5Yjg4YTM4YTI5MWQ4YTM1ZGE4MGEwZmQxYjk0MzM5MmU2NWRiNGY2M2M4YWRmZTcwOGM2NjlhZjY0N2MwMTExMDJmOGMxOTAyNjAzYzYyMzA3NTBjZjQ2NWI4OWFjMzM0NTc0YTE4NTJkZmMwMTkzZGNkMGYyYjdiNTkyZTFmMGE4OWEyY2QwZmE4OWZjNzE2OTBlN2M4NDE0ZTI3YzJhODRhN2I4NjM0Mjg1MmUxNjVjZDQyNjBkN2NlYWNlNDBlYWExNzk4YjljZTNiMmI0OWNkYmIxMGEwNDIwZjFhNmIwMmE2ODBhZGZhOGUzMjZkYzkwYjBkNDQ4MTM0YjNiMTZjNzYyYWUwODM1NWJhMDA2YWJiNTI5Zjk2MDc2MTQ1MDE2ZDE5OTIyMGUwNDQ1OGU0YWJhNzQ0NzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gu-vYp0647AnQKwqe1lEzX1YBlDZKqRBcnBR8f4II4tLoiShHO54LMWeBDDPpOPNZKPsi5FFWj4oFlOUkkB5Eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230422_110321_59_249c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.833Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Ijl4NzN5a2N4SDRNYWx0Q0pDQVY1Z1dKOTNWMVpPM2JwcVJBUDNGR2RWNnE5ak1DMVAzc21IU1hrZkRLVzZDWkpXcUgvaDZBOWhEU1lMeVhkMG1EQUlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMl8xMTAzMjFfNTlfMjQ5Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDYxYmJkZDlmNGVhYTk4OTFiZDQ3YzQ0ZDVkZTNjYmQ4OTg0ODY0MGM0OWQ2Y2IwM2VhNjRkM2Q1OTNiNmVhYWNlMmMyN2U5NGY1YzEwYTNlY2MxMWYxYzQ0YmM4YTk0ZTY1N2E4MTMyMzkzNThhMmU2NmEwMmVjZGEwZjNhNTQzYjhhZmY3MTUwYzEzYzVhNGNhN2FjZGQ2ODRmZDNmZWE3OWQ2YTI1ODJkZTA3MzBjNDlmZGRiZDhmYzFhZmEwYTlhYTEwYThhNjYyNGMwN2I1MjdlYWE2ZGUzYTE3ZGY5MWY3NGI0YmRhMTc2OGU2NDJiYWEwMGIwZGFmZWZjMGYzYjMyZjMzMzAwNGUxODMzNzZmZDUzODM3OTI1MTc4M2M0NjJhYTczZmMzODBmOTlkMTkxNjYzNmI5NDk3Y2EwMjNhNjcyZDMzZDgzYmE4ODYzZGQzOGY5YmI4NTcwYTg5ZTRkYmZlN2JmNTVlNjZkOTYxZjczZWE0ODlkYzE0NzI1YjE2N2U3NDYzYjgxNDg1MDM0ZGJhZDVjMDUwZDA0NzFmZWI0YjM4YmU3OWZkNjM4MjQxN2QxM2Y2Y2ViNDVmNjdlYjUyYzQxOTRlOTZiMzI2NWFjYTE5NmE3NDJiZDQ4YmVjZGRjOTk0OWMwNTUyNzEzZWRiYzI1N2ExYmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZWHBljH7ujNcjH2W6LsvaNhdnVKubTr53GdQv4Yl4bMCVNNuuiVjx6X7qOAL_RXUrm7VJm6RfZbu3nzAejqPXw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230422_110321_59_249c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.836Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImRzUUlyWVducjU2Q0RGbzg5bnZHVWM4dUV1eG1GMkRzRDBhWGdxUVoyanA0YTB3akNwS3FhOFExUURHa1YzeHJMNnh1RUE4c0grZ3g1eHVQdDhRYkx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyM18xMDM0NDNfMzlfMjQ1ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmEzZmViYjZiNDVhMWJiOGQyYmQwMDU2ZmMwMGE2NjYxY2NhZWM0OTI4N2VhN2Q1YmVhNGM4MThhZDk4YTgxMTgxOGFhMGJiZmIyMDY2YjQ0M2VmYjI2OTE0MmVmNjFlOGIyZjRmYzM5ZTc3YWJiOTVlMzMwOWJlYzY3OTg4MzM5YzZlMzg5Y2ViYjc4OWU1ZTBkZDc2YjFkY2Y1ZTYxYzg1NDg3OTFjZWEzYjRjODQ4MDMzOWIxZGE0YWIwZWNhYjBhNTQ0Y2Q1ZDMxOTcyMDRjYWM5OTQ4YjM5ZDlmOTAxZjNiZTljNmFlOGUxZGE2ZmM0MDE3ODFjODgxOGZlMzg0NGUwOTIzN2Y2Mzc3ZDY3MjYzNTFkMTBlMzc0YzE0NTFiNzNhOGJlMmEzMDcxMDc0NmU1MGJmNmFlYmJlZjIwMTNmMmZiNmFhNjcxMDYxMDk2MDFiYmZhODllNTM2YTE4ZmExNTc0ODY2MjYzZjg0ZjEwYmNlMDRkNDdiZDRhNWE1ZDEyMjM4YjE5YWQwNWY4NWY3NjE0NjhlYWFiNTU2ZTdjYWU1ZmQ4NWRlZjZhNTQ0YjY3YjdkZTkzMzUzMGVkOTBmZTQ1NmY3MGI0ZmNkZGMwYjI5NTZiMjQzYjU0OGJkNjc1YTJiZDY5YzBmOWEzMDYyNTZjNWQyM2Q0MGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6F9VrADLut7-oOFUjFXJSiamAOreQQ39xtokALPd6fZ-r0T8kBStj1yAvalOTBMhIXr-qrUGhAp1VOfnTsRvRQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210423_103443_39_245e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.839Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjMvMjl3N016bzk5VmE4OEtDaDZwcnRqT0E5QXlvUEI0MGlweUdTZytJY2hOR25zNzBOcGZEdjNhazBCS1VVakYxdG5NcGJFcmpjYUlHRnZsSGgyYjB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyM18xMDM0NDNfMzlfMjQ1ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmM5MDhkY2EyOTFhMzA2ODQwMDVjMzc3MjJjNTkyOTZkYmYwNDZmNjU3MDFmODliNGRmNjljNzhjZmZlZDJjODM4MDllMGFkMTg5Yzk3MTEwOTg1ZWQ2ZmQ5MTlkODNmYWQ5MjRkOThkNjgyNzg1MWRlMDU3NDM3Yzk2N2NiZTBjODE1M2IxMTdjMjEyOTUyYzhiZDEwZjdiZmViY2IyMjE3NjM5NTM5YmI4YWUxN2RmM2VjMDY4NGQ5MTQ2M2VkZjUxN2JjNzM0M2UyZTQ5ZDYzYmY2NmNmZTRhODNhZDYwNWU2YTg4M2YwMGQ0M2E3MjZhMjI3NDI4OGY1MzI3YTY4M2RlODUyMWE5MjI0MTZkMzZlODYyYWYwMDQwZDVhYmYzMzkxZDc1OTAyNTkwZDkzY2E3OWZlNDgxNjFjNGFhODBiMmQxODVhMTJjOGVjMDFkMzhlNGRiODFhODc3ODJmZDY5MzdlOGIwM2E1ODBlYzEwN2VmMTc4M2Q4Zjk3YWM2OTI3NTA5YTRhZGVhZTJiODBhZmY4YzI5NzJjNzAwOGVmN2FkM2JiYjU0MmIzYjU2MWJmZGRkODFlMTZjMzRkZWQ5NTUxYzI0YjU4MjVmOTRlNmVkNmFjYzFhZjM4MDIyOGEwM2YzZGRhZDA5YWIzZmZlMTYzNTBlZDZkYmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9Ec39W3L3nkram27MC1EKEAIGAoVviMZJqceM49mmAB3SmPSs2XTMgo22spXblssst3cfLPWZpjmiYlvT138Ww", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210423_103443_39_245e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.843Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InRPc1Vkbm9lajlqbzB3dEpjZUxZS3h1TlAvT3JhTFUvNjBzb1J3czF5NEx6U2Nqa2FkSEYyZjQ3aDBTQWFsUDlDYStQYVppS05LaFI1dTM2VHppR3dRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyM18xMDM0NDNfMzlfMjQ1ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2Y3NDkzYjJiMDhjNGI3MDllOTlmMDc3NDQwMThhNTg1OTU5OTYyNGYxY2E5YmY5MTMyN2MyMzc5ZDY3NWRiOTI4ZjJlNmQ2N2UxMzM5NTYxNzUzN2Q0MGZiYjMxYjYxOTYwMmU5NDAxZDdiMDM5ZDUxODkyNjU4NjIzMGE5NWFhYTE1ZjcyODgwMjA1ZDFkYzQ3MjExNTVhZTg3ZmRkNjY3NGM2ZTJlOWEwZjY4ZjkxZmNmMjBmYzA3ZDQ3OTZmN2IxNTk1ZTFiZDgyMDEwNzA1YzhhMWNjMWUyMjg4ODUzZTE1YzA2ZDYwOGMzMDJhMTUyOTgwZGE4NDZhYzgxYTM1MTg5M2JiYjk3ZmY3YWJiZWRiMDM4ZmUwYWE2MWU2NDE0OGIzNWRiZTE0ZTQ5NTc1YjNhNTk1NWU5ZTQzZWNmMGZiMzk4MGE0ZWEyOWU4Mjg0OTQ0YmE4NGEyYzY0ZmFhYjY5OTA4ZmVjODkwMmJmZjI5NDg3MzAxMDQ2Yjc1YWU4Yzc4MDQwZDdjYWFmNTAxNTY4NzAzMzZmNzg4YTNlNzg5ZjQzMWY3N2RhY2IwYjlmNjczMzVmM2Y0YTgxNjUzNDUxNmFmYjIyNjdjZTY0OWFmNGVjY2FmMmE5NTlmYThmMTkyYzI0NDk5ZGZiYmFkNGQ3YjM2ZTQ3ZWM1ZGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6fj6B4kB5j8k4XNQCXEyHg9P8-XRK1P1f2QZO-bSd_5_qRu5bnt1w2HCtLmkux0te8j7VlQzSmCe5S2ZJkRp0g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210423_103443_39_245e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.846Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkVHNHZKdi9sUHFVWmFUY29XbzJ4emhwZjhqakU4cDVaNzdhbHA5d0VRdktXZHl2OVRId3Z2VjRTalNyTWVEdkdMTmZhUlJ0RVZkWHBLT2g4YVo5RlBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyM18xMDM0NDNfMzlfMjQ1ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTIzZDA5OWRkNTc1YzYzMWU3Nzg0ZTgwYzE3MjFjZmRhMGZkNTBiMGFmODM3MGQ1YzU5YzkxNDcyMDlmNTRhYWY4YzM5YjhkMjc1NmM5MzIyOWE5YWJjOGY4ZmYwOTY2OWZiYTM1MTM2NWU4MTY1Mjg0ZWIzMjJhZDYwOWM2NTg4YTE2NGIwN2VhZmZiOTk5MzZmNGZiZmQ1MGM0YTZmOWYyZWEzZjQ4NWQwOGViOWI5NGJhOWU1ZWJmOWU4NDRkY2UwMmFkYzU0MzczZmQ2ZjgwNDIxMTQ5MzdmNmI0MWUwYTY4YmJlOTY4YTU5ODBkMGYzM2IwMDc2Y2JhNWQ2Yjc5ZTMxNGEzOGUzZmZlYThjNTA3MTYzZmU0NDA2NGUzMTNmYjBkYzEwZWFiOTExY2VhM2Q0YTJjN2NlNWM1M2JhNmM2OWY3ZTQwMzA5YzcxZjc2ZTZkNTA4ODMwYzM2ZDhiNDA0NDU3MTVmNWM0NTAxNGYzNGJkMGRiMzg1OTUwM2RjMDViZTQ5M2IyY2IxMGI5ODFiZTQwMmZhYjA0NzIyNjhjZGY1MDYwMzUwN2VjOGM5ZmM3NjYzOTFmMDlmYzQzM2U4Yjc2ZjU2Y2U5YjY1MTdiZmY0ZDFkMDJlYzYwMDY1ZGM5ZmUwNzMwODlhOTA2MjY1ZDg5MTkxMGUzZmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6JInSGZoTHbrQ0GU2GnbXgWNMknrNrQ6-FyMkpY60yn3HzwyA86lVDtr8rUd-0Nz7YWN_usI6sD5cBLg_Z1hcw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210423_103443_39_245e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.849Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkgrYVVYempFT05yZ0lZZ3p0N1hKdm9Ha25HOUFUY3EyNFZCaC9ZMG5EQW5HeUxYR2lpNGZUbGkyTUJWcDZPcytUWk1TN2R3a0RsdEQ1blI1VXU1K2lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUwN18xMTA1MDNfMDhfMjQ3YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzY3NmVkNjEwN2U4NTM2NjkyMmQwZDM4MzY1M2M2NmU2ZTZlNWJkMTM0ZWEyMTY3Y2Q5NDAzYzJjMzg1YjcxZTcwY2NkYzg2MmU1MzRlMjViZDc3ZTVmMDNjNzQxOGVjNTllMDdhOTk5NjI2OTEwMGFmODFiZmRhNTk1NmQ4YTc1OGI4ZTFiNGI2ZjU3ZjcwZmYwYWY0MzBmZGI2NTkzMjcxMjAyYzc3ZTVmNTIxYzczMDVkMDYwN2MwZTExNmFlZjU1YWI1ZDhjNGJiYmVhNmI3NWJlZGI4NzU4YWE3ZTc4MzU2M2RlMDZmN2MzOGRkNDA1ZWJmMjIyOGI1MDI1ZTQ3MTQ1OTUxNTAyMzIwMjAyYjEyZGY2OTZiMDExYWViYmJmN2Q2OTY3NTZiYzM1MWM4YzBmZjRkNTU1OTkxYjUxOGI5ZjY0NDcyYTgyZjRmNmI2NDQ0ODM0ZjJmYTZiMWFhY2MwZjRkZDM1MDdmNjI5NDZlYzM0YjMxOGRjYjJlMDMyNzJlZTA2YmFiNjA0N2ZkZjA5YmEyZjJhNDIzZWUxZjliZTkxMWNkZDUyMDc2NjhkMDAwZWU2NTU0YzJmNDkxMmQ3NTFkYzkxMzk3NWJlYjA4NTkwNTQ0MjIwOGFhMDAyYzA1NzdmOTIyYzlmYzFkMTk4YWU4Mzc1N2QzZTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.r8IDS_nfIwfHGFfhdk9R7vqwyP8CaLKCSF33GuIPzCqtxa-_lcCTQKMPuub0GOmjviXNfaJWdLRlvdahJPLwLA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230507_110503_08_247a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.852Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjdnUUhZQmJRTnRVNWllS3lGTEc2VTRteUJzaW8xRVU4RDF6SzNQK1NjRUh0VFNlM3c1ZEZFTTNnNGF6YUJSV0QwcVl5MW5yalVON2pHR0ZqWGIySTlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUwN18xMTA1MDNfMDhfMjQ3YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTViMmU4OTcxYzNlMzQ2OWI2YmViOWRhNjFmNmYzY2ViMTg5NmE5MjRkZmJlNTdlZmZlZGFkZTA4ZGE2MTVmY2YxZjU4NjNhYTBiNDAxNDg1M2U5MDliNDMzMjFlZTVhYmY2MDVhN2EzNTg5NmUwNDY2YjczODc3NmYwZTBjZTEyMjVhZGQyYTMyZWIxYTZhYmQ1MjAzYzYzOGU1MzhmOGVkNTk4YmUwNTdmMDI1YzlhMWExNjRiNjIzMjgxOGUzMmE5MDBmMGY1MTA3OGVkNGQwYzZiMjM2YWM0MTMyNmMxZTUxYjAyNWEzODU4N2Q0ZDc2ODYyNTAzYWY0ZTg4MzE1MjgxODE1YWNkMTBiMzA0ZmNhZTg1MmVjOTY0NzFkMWRmYmZmZGVhNDBkNzc4ODY3ZjJkODkzMDE3OWEzNTBmYTcwNDM5ZWI5NmYyMDJhODYxMjZjYWE4MGVmMTQyODVlMmQ5MWQzMGRhZGNmYTYwMzBjMGVhMGU4NmYzZmE1ZDNhMGU3MTMwOTRhY2VlMjY0MzNjOGNlODJhMDE5M2M2N2VkNDYwOWU2NjgyNjc0ZThkMjdhNGQwZjRmYjYzYmMxYmM0NDI3NWVhNjhjOTEyYzQzODdkMTFjNjU5NWE4NmVkYzM5ODM1ZTJkZDI2ZTk2ZTc4ODJiMTdhOWVlN2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oAbeDZDD7ebETXGKPUFri-3d0ndhq9ageRUybNno3hWxKZEkH7q3I9V-sT6h2uKYEinwE_Yth8J9D3jwBT93Ng", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230507_110503_08_247a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.855Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkFSRmRuc205a3FwWk5VZWh5blMwb0VpamwwV0xMWEF0SWdoNE04eHFDek9wa2h4UEYwSGJJOXkxY0Jxd0I5aENIUGNtbTRtcjZUWVFBbzZKK2ExeFh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUwN18xMTA1MDNfMDhfMjQ3YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTQyNWMwNGIyYzY1Y2I0ZTY0Mjg5MmViMWYxOGEzYmQ3ODRiNDg4MDYwZDVhMzE3YzUyMWQxNTlkZWQ3Yzg5ZTk1M2QxZmYwMzY2NjBmYzc5NTkyNTk1MjViMGRjNjYzNzNmZDM5NGQ5NTJiYzE4MGZlMzlkNDVkZDMzMGU4MTlhMTUxYjI2OWJiNDE2ZDliNmY1NDZjMzE5MWQ1YTEyOTA5YzQ5MGY5Y2QxMTU5ODQ2MDIyNzgzMmVhODFiZGJkZWZiMTliODUzODgyYmI4NDlmNDIwZTMzMjM1OGNlMDc4Y2Y5ZjlhYjNjYTM1ODQ2NmEwNGFjNzk1OGM2ZmExNWM0ZGU2MDk0NTdlZDIyMzY3MTliOWQxNDYxMmNjNjJlMThmN2Y1ZmY0YTZlZjRkZTE0ZTlhZDc0ZGUwYzJhYWM5OTRmODAwNjhmMmY5ZTY3OTI0MDgzNWFiMmUzMzA2YjQ4NGM3ZTRhZjFhOGIyYzZhNTliZDcyMTJmN2MwYzdhMThjOWVkZGM4ZWFkMDA0OGZiYjc1NGNkMjM2Njg4MThlMDYwYzNhY2FmNDQ1NGZmM2Q0YzRlYWY3MmVhMmI5ODI2NTk5ODE5NGJiYzEyMjVkYzJjYjI2NWQ4NzVlMjI0NDFmYWExZWVhOGI0MDkxNDE3ODAyYTkwMmViNjY4MjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EIA_5QfG7AJhC1wpJuxnD7YS3WYvVu24ucF-NPOX0xwDVNRfGRuHE6yuP4sbuT8OFmRurbAO6BPOv4Is6uJsXA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230507_110503_08_247a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.858Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJOQjdnejRPQ1RMYWJaSU5RRTVSeXlRSmZpRlB5dUpqWGdSUTJpQUNFamVlQlRiM0JjOTVCNU9yemFMUWg5THZDODVNOUtENHNESWlWSlhsVnBZN2Z3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUwN18xMTA1MDNfMDhfMjQ3YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQ0Y2Y4ZWMxNmQyNDFkYjdjZDdhMTdlNTZhNzMwZDQ4NDY0ZTc3MWI0ZjdkMGJjMDg1OWFjM2ViYjljYTIxZTc4NTM1ZDM2YmIzYWVhZmVmYTFiZjgzZWEyMmQ3NGVjYWEwZTQwNWRiNzg1Y2VlMDMxMDA3NjVmNjE5MjVkZDg1Zjg4ZjZjNjdkY2U0NDdjZDgxZWRkMDcwOGJmOTYwY2JkZGMzNGMxNjQ3NzkyNmZmMzg3OTdkNWI1MDcyZDE1OWM1YWM2MzczZWQxNWVmZmYzZjVmY2NmNmNhYjZkMzllMzU2MzE5NmMzNWY3MDJjZTFkMThmYzJiMDY3MzVmNTdjN2M4ZTJiNDkzNjJjYzFiMzJjZjc0MmE0NGZmYWViY2I5MzU2YWI2Mjk0MGI4YzJmMzFhOTlhNzAyNzhmNzI0YjdlMmZlYjg1NWVmZTY4MDBhNjBjMjBkM2Y2ZjlmZDdiNmMwNzNmMGRhYWZhODI1NmRhMTc3MDMzNTkzZDY1ZmY5MDE0MGNkYmI1MTJlNmY0ZDFmMmM0MjUwNzYwODY5ZjRjM2ZlMDEwNmQwYmEwZmJiYjIzMzViMGFlM2M3MDE2NzQ1OGZhZmRjZjk4MDEyOTQ1YzUzMzczOThmNmFiMjExNzczNTE1YWRmYWNlODY2Yzk1OWJjM2E1N2FiNWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DaN7yjPo1nEkdA6jdW1tCZd-HyOlGkS8iKsd0ar60zAN0UKV9RjktJ7um-hm5Y3JMTD4zsrWCt7EZkWS9zjhuA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230507_110503_08_247a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.863Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InZRWjFiMFoyditWUkFrY2FJd1FQUkJTd1ZZSlZsUUtWK21Sem1IRHd6Z29odms3MTA2MlFIbW5meGZUZ0F2K0VIbkRhbWJBQ3Z3aSs4d1BFRkUySmNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwN18xMDM5MzhfMzlfMjRiMF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWRkMzNlZTQyODI5Y2VhMmJlYWY0MjJiM2YwMmQ1ZTg0MGVhM2VhYWM3YTY1ZDU4MTJhZmFjNWExOWNiNDQyMGQyMzk3MjhhOTQ5YzI2NGQ4ZDYxMmNkZjc1NWU4ZDM4ZWRhOGY4ZTJlNWMwYzgwYjMzYWNkZGZkNmMxNWNkYzM1MzA1YTQxNTZlNTI4NWZmMGU4NDQ4NDJhNDc5MWYwM2Y2YThiY2Q1YjE1YzM0YzZlMzMxMGNlMWEyNDczNTVmYjNmNzgwZjk2NGNiMTA0NWI4YzkyNTE5MDNkMDc0MzgwMjhhNTYxYjBhYTBhMWRhNTM0ODFkN2JhNWVhYmU4ODEwMjIwZmY4MmI0YTkwYWI4ZGJhOTU3MDg1ZWFjNzExNmM0ZmZlZGY5ZjdkY2RmZjQzZjZlMzFmNGIwODQzMjM0MmNiNjk1YjRmZDdlODZkY2NhYWIwMmM3MzU2MzQzMzMzYjc5NzEzMzllNWNkZjI3NDBkNzRlMjc2MjUzYjM1NmZiODRlOWMwMWJkZTQyZThlNWIyMzY3YTRlNzIwNzlkODdlMjBmNjU2ZGRmY2Y3YjVmNzlhZWIxMzgzMmI0YjMwYjNiMDE0MDgzNjU0YzdmYTY1NTcwZDAwMjVhMDg0ODU1N2U1NDQ2NDYzODE2MWZkMzI0MDg0OTcwNTkyNzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Vjmc9WjgE5QkgkrG53t0D7MYXamUbHegFjAYIoOqI5elWb18MScKojMMz8hNkVtenL4guNvxUtm0d_9a95J61g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240207_103938_39_24b0_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.865Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlhxaTNOWElCUnlDVU9EeXZKTkxzdGs1Q0h2RU5uaUJkT0cyUGJJazB0YUZWMDl4YnpWS2p6b3QzVVV0NnNLdFpRL2pjWnhHWldOaEhUdm9QREtQUCtnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwN18xMDM5MzhfMzlfMjRiMF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTczYjhmYWM4MmI1Y2ZhODRkZDI4N2U5NTJlYzc2MmYwZDhiZWFiNGU5NmQ4MTYzN2NjY2Y3YWRmNDZjM2E0M2Q4ZWNlYmQ1ODM2MDJjYmRiNDI3ODk2Yjk5M2MxMTY3NDkwYmE1ZTAzZmQ4NzFlYWJhYWViNGRiNTEyMWY5Y2JhYzdiNTc4ZDBlMjdiYzI4NTdmY2UxZTc1NWRmYWFhYmU3YzBlZjliNTVkZTE2MDk5OTI3YmYyNTVkZWVmNjk2YTc5Y2QxMjY3NTMwMjczNWI5MWUwYjZmOGQwMTYyM2U3MmMxMTIxN2I5YTViMTRkMGY0NzdiMGZjMGVmM2Q4ZjYxZTY1ODhmOWE2ZTE4NTFlZDllNGZkNjRkMWI4ZmE3Mzg0NDE4ZGFmNjU5OWZmZjJkNmE4YzdjZjBjNzQ5NzFjMzEwYjdmZDA2ZTJkOGU0ZjI2OGYxZGVjYzUyM2NjOTQ1MWI5YzExZDk4NGI3MDM4NTZjMzc5NjA2ZGFlNDI2MmUxZTgyNThiNDliZmZiMzlkYTE4N2FlNmE2NWE4NWUyMTA3Y2ZjZmVjNGZiMTNhNzlmODYwMTlhOTFiNzNiZjY2MmZlZDliZWIxOTQ4M2EwMjE1OWY5OGNjN2M1ZjA1N2I5MjkwYTQ1ZjI5MGFmZjU0OTYxMDU0MDQ4MzAwM2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.k-zbNFVVkbM5xWxyW1saPMUeHx1jbbQi9cKZaBxFlebAPUm7S2lUHirPnUq_7BUR8YCCXBsI8Jy7lvzrANucVQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240207_103938_39_24b0_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.868Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im9TSVZBSXlibjByNFpxT01oanhBZUFMNzhWMUx6c0c1VzBsdSs3YzJwRVd0dndUd0RMUEVVSkZKMjFSdW9TMjNFRWFZNS9ETFg5MjJOSWNBUDI1NjBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwN18xMDM5MzhfMzlfMjRiMF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGNkMWFjODJjNjAyZmIzMTA5NjhhNTE1YmI3YzdkZmNjMWFiYjk5NjA4N2UyMjcyYTRjZmU0ZGIwNDVhZDc4YTUwZjIxNDM3NGIxZTUxZDgxYzdjZGNjNWE0NWU4ODJiMjdlZjBlZjY4NTgzMGZkYTRhOGRiMjZmODI4ZjA0MTA1ZTYzY2E2NmJhMzRjNTFkMjcxZjdkMzA5ZGU3MzRhNTdmMWM3ZWVmZmVkNDZhMmQ1MDZhZDgwYWVjMmRhZDIzYTE0M2YwNjNlZGIwMTYzMWZiMjNlOWEzOWIyYWQ5NzIyZTEzOGFiYTE4M2NlNzhkZmE4N2I3NWYyNjk2MDZiYzM1MDkzNGJiNTNjMWUyYjAyNzFkOGRiMWE4MzJhYzkxNWY5NjEyYmI4OGZhNjNkMjY3Y2FmODBjNjdmYWViZTVhMWYwMWM1Yjk0ZjdlMjBiODNiZmQ1ZjI0NzJkMDNiNThhNDZkZTUwMTYzZjRlNzllZTI3NjY3NGM0MDkzNmIxYTg1MDQ3MWNmYjFkMjRmZjg5NzVkOTUwMTdkNTg5MjhjZTZlZjZkMzUxMGEwOTg4MjExZmY0MzdlYzYwMTc4Y2YyNjNhNDYzMzVhMTkzNmUzMDliODEwMzk5MzY4M2YyZjkxYTRlYTliOTAzM2Y3YTQ4NWM1NjhmYjhhNTY3MTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qrjENFyDVumfauCb9hUzw-9k78rVXlQXMvBMRhJanO8ZGmHcG7VdnvsPiEz_8ObScbi7tVVosUsYPAz0l04RwA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240207_103938_39_24b0_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.872Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVNaTczQ3ZHYmxMdHJHb29hU3kyNmZaTXhpUDhzWHprQ0lra1lWdC8rSlVlVHdNK2ZzbUZFWXFXUGFDOVc1SnZPd3dhdmZCNEJWWU1GTm9kV2FhTnBRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIwN18xMDM5MzhfMzlfMjRiMF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTgwOGVjZjYzNGFmMjU4MTBiNGMxYWJlYTg0NTEzMTM5OWM5YzlhZWFkODU3MGE5NGY5NzYyYzgwNDg2YmVkZTZjYTQzOWJhMjg0YTgxYTU0ZjdkZWRlZGU3NWFhOTZmOWRjY2EwNGNkN2Q3YmY0ZThhNDMwN2I4YjI1NzE2Y2U4ODA3ZjllZmM3OWVkNzkxYjFmZWU4YTBjNTFmOTUyMzZjOWFjM2VlOWRmNjE2N2Y4OTYwZmVjNDVmMTljODgzNzhmMWRjYjdkYzU5MzhlM2JiZTIyNzY4ZjA1N2UxZmY1ZmE3ZjlmODkyNzlkMmJjZWE3NmRlMDk3YWIzYTJmYTM0OWNjNTMzM2MwOWUzMjk2NzdhNDRkY2Q2MzhmOWMxOTQ1OWQ5Njg0MDExY2M3NmU3MDk1ZGM0OGU1NzQ1MzEyODRjMzkwOWJlNmExYjI3MzIyNGFiMzRkNjFlZjQ2ODlmODE0ZGU5NmJjNWQ4NzM5ZmExOTNhODQ4NGE3ZDRhMzlmZWE5OWY4MWNhMDc0MDBkZjMwMTMyNzhkMDE1NDdhYzkyYWVjYjhkZGYyN2M3NDg0NmMzNmZlYjUxMmM0NzU5ZjczYWIwNWJmNTExOGI0YzQzZjNiNjRmYzg5ODQyNWFmZDc4OWUwZjI3NmMxZGZhYTU4OTRlOWRmOGYyM2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xY1yRmw2TTPwPvO8D0vGzIUx3rWLgzfNRARtY8JCAyevbrwUQUfcj-H4OQPc3GJ1nHluM3wqYFjm_UT4UPIllw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240207_103938_39_24b0_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.874Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InpTblExcGF1N3liYWZOdmFQMkVnODdEUWU2V3ZNVjhKMDBSMlNXb3RORFFNMWJpREY5SE1xdzYxNTVvK2UvZE9sZXdHdmlTVml5Z25oQlpMeEExNFpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDdfMzZfMjQ4Zl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2QyNWMyNGQ4ODIzMzllY2ZmMjJkY2M5ZGRkYTA0M2FiZDc0OTZkMGQxMjljNTMzMmVmZTM4ZWQyZjI3OThiN2EzYzU4ZmZkNDNlZWRlZTU1NzBhOWQ3ZWQzNTkzYjY0NjY2ZTNkY2FlODc3NTE3OTRkY2FmNzRjMjg5M2YwZTIwYzdmNDBiMTVhZTdlMWJhYWRjNTNhMmIzYTFkY2QyYzAwODAzMDczYWUxOWU5YjljYWYyNTgwZTJiZDI3MTE0OTQyYWE4NmY2NjcyNWFlMzIzYzQ1MzAwNjE0NDI3ZWUzNjFlNzhhMjcyOTlkNzg4NTliY2U5MzZjNDY1YWQ2ZmE4OWFlODgwZTVkMWUwNzNkYzc4YjYyZDZhYzAzYTY4MDIxNjY0YWNlOTgwNjlmMDA2YjAyMzg1NWY4ODZkMTJiNThkY2U3MzlkNTZlYzdhMTM3YmQ4YTkzMGUxMjIyYzg4ZDA4ODE2Y2FmMGM4YzE5NWZiM2M1Mzc3ZWMyMzVkYWI3ZDQ2YmE2ZTU5Nzk0ZDYxZTk2YzI3MzgyNzk5ZTU5NGM2MTQyNjE3OTc0Njk0YmQ0Y2FmZThlZDFiZGM4MGJlNTYyZDU5N2VhOWM4MWUwMWZjOGZhZWFkNDhiMzk4MDRmNjEyYzdhNmZlZTQzOTM3ZjI2MzYxZjk0ZGZhZTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7GNpqD4GxamLrJzKZsbxo-FgQJrK09jJOKu529comZ9dDZzfDvB6EPfat76jHeKsG0FpLLu5UUs2jhNnAJC8lQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110507_36_248f_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.877Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlNGQ2lNVUw5aW1pa09kbmRwWEd6Mzl6UERlR3l6cDI1RXJoeGowQ2FSZGoxT01zYTRhNWtDcmplc3A4Y2d3akkzNnp0U3l0K2p4RUtIaW54SnhHSkhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDdfMzZfMjQ4Zl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTY1ZjUxNDlhMTFlMTMwZjYzZmE0MzgxZmU2NzQ4MmQ3MWZlNGNjYTIwZTExOWNmYzYyMTg2ZTQyOTA2NzllN2E2NDAyMDQ4NDZhOWEwMGE0OGNlYjI5OGIwOWJjMzlhNjk3MGFmMDlmMzYwZWExNTBjODAzNWQ1MWMwOTE3ZDYxNGFmZTIyODM0ZmY5YzQ1YzI1ZTgzNDViNjJiNDE1ZjBmMTFjM2MyYTk4ZmQ5YTZjNmMzODBiMDMyYzI4ZTVjMTgyMTVkODY0NTk5NmMxYzY1MmIwMTAxOWVkOTNhYWNmODAwYzNmZjdmYjJlMTA5YzdlMjIwMzFmYzg1ZGRhNTIyZDhjNDc0NmU3ZTM3YWE5MzA1Nzc0MzUzOTkzM2UwMjQ0MjhkOGZhNjYxNzkyOGIzMzMxNzMyMjBlMjMwNTY2MWU2YmRlNGQ2YWUyNDA0MGFkMjcwMzcyZDZiNjliMWY4YzZjMzQzNTdiMmM0MDk5MDUwN2UzNzE5Y2EwYjEzZWMzMzk3Y2I4MWVlZDEyNzJhODU0ZDc1OTkzZmVmMWQ4YTQ4NjY3ZTI1ZmFlYzQ1NWZlNWFhNTQwYjExMDk1MDkzNTBhOWJmZjVlY2RhNWRhNjc5YWZlNDdkMzA1N2IwYThkYzUxY2QxNDIxZGYyNGZlYzEwMTYzZGM0MTFjODdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.sPGbbXLcRidhF4RbwhQTpCY1SkUsWjs-0u-OTuMFTNF2aDtjjlk61-jMS-8cDbF03cQgW3hqzcnpM0SgZZOQ4Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110507_36_248f_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.881Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkdHUTB4YVVYWmRIYVVIdTYvSHp1aDNTSkp2VW13K1piTm1kS1FyL1YveTM1MkF0N2o5L1UzeTQ1UkEzRFUrSkZlOHpSdGk0Q3ZWaFlpRWFCL1dnZ1FRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDdfMzZfMjQ4Zl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDA0YmNkMzAxYTIzMzE4MTVlMmQ5MjRlNjcwMWQzZjhkYzY4OTI1OTU4M2FjZTA4NzlmMTYwYWY2NjFmZDBjNTU3YmIzYjZlYjI3Njg2M2YyZWY0OThkNGQwOTU3NmYyMTRhMjU3NTczZmEzM2VmOTE5YmNmZjQyMWQ5M2MxNDk5YmRjMTUzYzE3MTE5OTc2NWRiNDRkZDY2NzVjNzdjODA4ZWI1ZDdmYTVjYTcwMzk0YzI4OTgwYTllZWYzOGY5ZWY2N2M4ZGIyYTBjMjZjYTMzMjcxZjlhY2Y0ZjM5MmZhMzJlZWRlYTg0NjlhNzMyMWE0NjNmMzZiOTM0MTAyM2MyMTEyZjFiN2U0NTg1MDRkYmViMWI0YmQ4YjY0NGUzMzg1YWFhNWNhY2ZiNWY3ZjlmZjc4OGYzZjczYjBmZDA4OGM3OTQ5MThjN2ViZjc0MzljMzhhMmZjNjJlNTIzYWJlOGMyMDJmNTYyMzRkN2M4MWMwMWNhY2QyNDdmMTZmNWJkMDhjMDcyZDY4OWFlZWNjMGVhNDZlYzdhZTBhMzg0MDg1M2U0MTM0MmYyNjYxZjc3ZTViNDI3NGI5OTBmNDc1ZDRiNDgwZWEzZDM1ZjQxMjZlZDUxMTJlZTZhMTA4ZDYyNDY3ZWNjYzY2OWQ1Yjc2MDNiZmYzOGI5MmE0MmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ow_EcTXzdh3fi5LKY7hg3gWq51zc-H47sE-LOk4MAGbOl4XFk073okNcGPCCVSlh79aGCOseAVVRdQH4Yblyvg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110507_36_248f_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.883Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkgveHduTHg1WFBLaWxhUmVWb3hIaXRnR2JsTlkzNkd2MDVBaGhXRjBvOC93QUk0cVJjMnM2N3B3MXZFRVdNbyt4aFdmZ2dxTXp1aEZleG1Rd05jZzhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMwN18xMTA1MDdfMzZfMjQ4Zl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzkwZTJmOTc1NzJiY2ZjNmRhYWE4MDlkYmU1OGM1NTE1OTc3MmM3ZWRkMTUyMzk2ZmZiYTkzODdkNTIzMmY0ZjliZTczZDczMDJmNDRjMDE5MmIzMzk2MDNiMWI1OTFhZmIwM2U4NjM1ZDMzNDA1NmU2MGIyNmNhNDFkNzdlZDg3MmVlNGVmZmM5OWNmY2M1YmFjNWE2Y2E4MWViZjhjNTdmNzcwOTc4ZTAxODJhODA4ODdlZTk5NjM3NzUzNzZjYjQ2Y2ZiZTdiODMyMmYxMzFiY2MzY2I3NzU2ZGIyMDUxODIyZTIzNTA3Mzk5YjJiNTBiYWMwZWVhNzFhMzE4Mzg3ODdiMWViNjU4NDMwYzI0YzgyMDc5NGJhYTYzN2RmYzQxODk1NTg5OTc4NmRmMWEyMjIyZWNlNDA4ZjdmZjVkMmU0NDdlNGY1N2FjMGRlZGVlN2VlYjlhZGM0N2VjMTYwNTk2MDI4MWJkOGNjMjFlZTE1NGFlYWRlMjY3OGUzZjA0OTViMDUwOWU2YmY5MGM1MjI0YzhjNmY5ZWFiYzc1MDFkOGJhNmU1MDVhNjM3ODM5ZDI2NGFkMjQ2NGE3Mjg5MWI5YWIyZDRkNTdjMWZjMWIzZDk3MjEyMzNjNmQxMGE0NzliZjI4NWI2NjVkMjMxNDVlYjM3YjlhMzRlYWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8bCQEsAA_u6rZp-Zd5gevOsGL5PM4_r89VPQeldtT2Y7yerS0fLKsS8NGymVaYFE3Em0Npz-fF0SZg1LGcxhPA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230307_110507_36_248f_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.886Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlvRnU5R01CamhzZXpwU1IwMDlGKzZnalFFRWFJSHloVXBtNEZiVXlYbCs4QWxlZmpraWJ2NldZUHdwVHEybTAvZHoxUkUrSWx2bm5iemNSUCtSaFRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDMzMThfMzlfMjQ0Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTgxOGY5N2QyODQ0NGQ0NTAwZTkyYjBjN2M4OWUwODVmMzM4ZmQ0ZDQ1Nzg3Y2IyYWI5ZDQxMDc5YTQ0YzU3NjFhMWNmYjIwYWRhOTgwODhiM2FjZDU0ZWQzYzk3OTJkYjU2NmEyNmY4MWFhMmRkZGFlNjAxMDdlOTg2N2FlNWMwNTcyZDY3MDJiOWFjYTZjOTFiZjkwM2NmNWY3NDI0MzYzNTViMGQzNTJlZTE4MzBkODYxYTI1ZWJjMjkyMjBlNGZjZDI1YWIzYzc1MzUwN2QzNTRhZjE2ZmVmNzkwOWFmZGU1YzcyMTc1NDNkMWJjYjQ5YjdmYjYyNjcxZmY2YWI4MGRjZWFmZmNmNGUwNTRlNjU0ZjE1NzhjYThlMDNiZmQ4MzUyMTBlYzEwMjc0NmZhMDc4ZjQ1MTVjZWZkNWM1ODViYjExMTJhZmRlZDRkMzllNDk5NmM3NWFjNTgwMTY4MzcxZmYzZTZjZjg1ZWY3ZWJiNWIxZGM5ZWZkMDllZmY3OWEzOWY5NGNjYzY5YTU3MTM5MDA1MzhhMzFmZDcyYmZiNzYyZDNjMDYyNTc1Y2IyOTU0NzgxN2Y2Zjg4YjAwMzQxMjdjODIxZDM4MzY0ZmE4ZGJmOTRhNDk2ZDczMTEyZWJlM2E4ZDcwNWNiOTg0OWU0MTU0MGU2MDhmNjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NWms-uhjZhRkniks3KNct2Q62_sfSheH4Dyiokwm6GsMgG8v3Z3Yc06yuGNCddJmiFrbMtT0pXduKjOE49-97Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103318_39_2442_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.889Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkJjZHhwS0xWSGtCWjhwRXNRRlNKWEN5NjA5ZWU0dmM3UnBnUjl5NjZ4UTBLMFhvdFZIL2E3eG5jWEZZTXJBN1Y4WWtNSS9LMkhhVkxTOVVDNnVMYjlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDMzMThfMzlfMjQ0Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2NiN2E0M2Y5MTFhZDkyNGI4ZjUwNjlhZjI1MWU2ZTkwNTYyZjA2OTU2Mzc0MzQ5MWIwYzM0NmI2MTdlNjU3NmE4MmRiNjkzYTI3Y2NiY2ViMjk2YjE4N2I0ZDgyNzE4YjEwYTJjZDUwYjhmYWM4ZWQ1NmM2MGNlYzU0N2EwMjI3MzAxYjJlYTAxOWZjMjViODdmMzZkZDE4NzVkMTAwODA0NjAyZTgxMTJhYTA3NDJjNDAwMDg1YzI5OTY1YTA3YTZiMGJiZDU5ZDljZmZlNDUzZjFkOTEzZDk4ZmRhMmEwM2Q0OWJmYjAxNDE2MTFlYTcyYTRmYWVkNmQ3ZGMyMGUwOWI4NjEwNWNmM2UwZjNiNmRjNDBiMzljNWIwNDQ2ZTY1NTI1N2VhZmI0ZTVhYWQ0YzFhYzM4ZWM0NGY4ZTVlOGY4ZWQyOGNhZWFiMDA5ZWJhZGM5YzI0MmQyM2UyMWUwZjViNTA4ZTQ1NjY1MmNiMTkwZDQ4MGI5Zjg1YTNjZWVmOWY1YTI1NzMwMzM2ODFjMDBhODlkZWExNDk0NjAzYmQ1YjA4NjYzZGQxM2Q1MzkxNmZlZDMwZjgwMjRhYTA3MWQ4ZjA2YWMzMWI0MDY1MmRiMGI0MDEyZDIwMzM4ZTQxNzE5NGNkYTc3MzUyNDg2ZDNhYzE3MDAzMDJiYTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gSdfgM-AuiW0zl7MjJtI9j_o6Sm6fXR2YLjzqayckrT1FqvjJOmBzKKm7YE-cSsFjtxwa00ylcSUyLGeg_Nkug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103318_39_2442_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.891Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IklwRWl5N0JVLy92ekdkeWpqR0VvbXNzdHk4Wk8wMVJQamxaUTR3NHUrVDBnVDVrRlRGNVdrTmpMMUN1TXZZcE9jcWlFcUREVitpQkhUQ25ld2xJYXVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDMzMThfMzlfMjQ0Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTM3Njk4Y2U1ZjkxMTk3NGIxZjQxYTk5MjJiNDIzZGMyZjk0ODVmYWZkYzhmN2ExNjFmNzcwMWIzODIzNDNjMmE5NTI4MGJiMjZlYjg1Njc4N2RhYzk0YTM0M2M1ZTc1MmQ0YmQzMmRmNmVhYzM4MWIxYWMzYjZiOTNlNzY2Nzc1OTY2MWUyZGE0NGJkYTU4YTZkNTlhZWRhNDQwMzM1Y2QwOWRjZDg1ZWU5YmIyZTRiMDJlZGNjYWVhY2Y1OGM5Nzc5M2VkZDU4YjYwZmY0M2M2NzFjZWIyOTk1MmFhZjlkYjZlNTMxOTM5YzJlMTFkMTAxYzFjMTU4ZmNjNGVjNmQyYjYyMDE5MjkwOGFiYmI3MTNmNTRjM2FjNDdlYzUzYWFmNzUxOTcxOTdhNGM3MDkwOWVlNGRmNzM2NTU2ZjIyNjNmNzAxZTI0OGJhMjliYTE1ZGU5MTNjN2U2NDRlYjAwZWQwMjI4YTc1OTc1MjgwMjdmMmU0ZmQ2Y2U2Zjc1ZTYyZjZlMDZlNjllN2VkNWM4NTg5YzU5ZGE2Nzg4ZmNkNDNmZjE3NTcwNmY0ODNkMWIxZTcxZTA2ZGQzZGYwNDg4Y2JjMDE4MjBjOTAxZjFmNzExZjVlNzYzMmMxOWVjYTc3ZGYxOTJiZTNmNDIxYjU1ZTY2MmRhMzMwNTIzOGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Kib-mYVd0k2KdcJ2wa_JZpy22RjAZ77p6bCHqxGvfpx3XwjBJIqfdH90J6J_SeXHyxToe2IZhNEafYvbljYyZQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103318_39_2442_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.894Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InVIcG13SnR5M2FDQkU4OVY0YllXM3dhaE9CYUN6ZG1JOUdZVVdWYWRQcm0zc0RMZFV2ckJlZ2NkcTl6amhVNzBMUzIxblNidlcyckJOS1VFQ3RWS1N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyMl8xMDMzMThfMzlfMjQ0Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2ZhOWQ1NmNjZjQ2YjgzYjllZDY0ZWIxZTFlODhmODczM2JlOWY5Y2JlZDZhNDExYmUzMGVlMWE5N2JlYjc2YmQ4Y2YzMTQzOWE3YWM0N2UyMzg3YzJiM2YzYTk1Njg3ZDI0YmExMzgzYzEzYThjNWQzMzgzYjNhMzUzYWVkYzlhZjEzMWM0ODIzZjI2N2RkMTk4MjZmNjU2OTgyZjc3YTVmZjNlYTBiYjZmZWEwYjRmMDZmZjI3MDYzZDQwZTliNmRmMTFjNWIxZGQyZWQ1NzE2MzFjNDdhYzYzZWQ4YTMzYTE4OTBiNjA3MGUyY2MzZGNkMGZmNzk0ODdlMDRjYWNlYjYwMTRhZGYzZjIwOTQ2N2Y5NzU1M2FjMDE4YmE3MmI0MjBiMWRkYmM4YzM2NmI2MmVhMzlhM2E2MWQ2ZTlkZjA2YmU2ZTgxNjlmZWQyOTI2Y2MyMzViMzFmN2FhYTE2NDdkZGNhNmVhOTM4MzMxNTZiYTY4MTk3NGM2Y2RjNmEzN2YzOGM1ZGFmZTg4ZGUxMzg5YzA3ODA4OTZkYTcxZGM5MTBmZTE2NGUwMjZiYmRlYjU4NDBmMjQ2OWY0ZTIwNWZkMTYxYTBkMTEwMjE0ZmZhN2U4ODgxZWU5NmE3ODQ1NTRhZWRjZDI2ZWM2MzdkODczYmJkZTUyY2U0MzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.d2TozGDcwJUa8CFRqX8Vfd4w1JZ_tWE2z81su7Pp1Gdxlez1p2ysRRlP0wlwSby_7OxsIKyPQCNNeespfKMzvg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103318_39_2442_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.897Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InBDZGV6WXdqK1VmYWFTZGdCMGFrN3RNSEJQUFhRaEo4ZmJBb1N1RDFEOWRPQ0VybERmVGp2YUZ6Vmp5NnBYcngwVnVNczBiWU8ycFVzQUZnSUU1SXZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDQyMDFfNjZfMjQ0MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjJiZTQzM2EzN2RmODZmOTY1YmIwZGE4NjJhNTJlZTRhYTVjYWE4MjY4OWQ5NzM0NDcxMGZmZWZiNWVmMjBiOGJhOTdmM2U3MDA2OWJiMTc3N2UxNDNkNDRiMTBkMGFlMDRkOGRiOTZjNmIyN2I4ZWQ4OTdiYzc5MjJlMDc4ZDMxZjAyMjQwNzc3NzM3YjVhODhiY2ExZGMzNjQyNmQyYTQxMTE4MmQyMGMzZjlkYTQwMmNkYTM3Yjc0MWRhZTdhZjlhNDZhYTJjZDRiNzZmZmQ5ZGU2NzcxZmViZmQ5ZGM3YThhOWY0YzQ0MzFlZmVlMjNkMjZiMGY2OGIyZTJiMDY1MTY5OGQwN2NiYjE3MjY3NTMwOGZmNDVhNWFiNTE0ZDc4YThlMGRiOGI0ZWY5MjA3ZTM4ODQ4NzlkOWU1ZTE1ZDM0ZDhlOTE3NTY5YzBiZDZhOThhNGNhNDY4NGJiZGM0YjgxNjk1NTJiYmQwNDlmNDYyYzdlNDgzYTYyZTRiMWYyZTQ0OWNiMjQzODczYjk4NWQ0Mzc5ZmYyZGYzMzQwYTZjNGZlNWY2ZjY3ZjIyNTJkY2JiYjkzMDc3MGMyNjNhZTkxOGI3MTAzYTYzYjQ4MTY5MmVmMmJhMGJhYmEzZGY3ZmM3MWMwYWE3MTRjNWMxZDdkYTM4NWVhZjkyZWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jVcbDsEhvUOeyai5Uy_xMnyJ91HOH8eQCMXTDMjFETDr_vVBEvLL6TwGOmS4wh9yV5DPsv5F3O8--Yp6tmrxZg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_104201_66_2441_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.900Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZJT1Z2eFVXaklLK2hsV280S2pieUxLV2tpRHlnSjRaOW96Z3ZmNXkzcW9sZWE5R3NwaWVaSFVreG1yWEQvZ01JVWNYazcxNGtnKzErUEQyWkZ4NmlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDQyMDFfNjZfMjQ0MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjViOTVjYmY3Y2EwMzVhYTlmOWE4OGUwMGQwNjEwZjU5MGU3NTMyMDFjZjJjN2MxYmJmN2JkMjBjZDRhZWIzMzhhZGM5YzBjNDcyNTM5MDMxYzVkM2NhZmQ5MjYyNjIzYzViMGVhNzJmNzIxYWE3ZWFiNGExNmM2MzYyMjMzOWNhMmM2N2QzNDQ0Yjc2MTlmZGYyNjY0OGFmOGY4YzJiYmI3NDMwODM5ZTVhY2NiMGQ5MzY0NTdlNjY5ZTY2ODE4NGU1NzMyNTNlYzczNmIzYmE0MjBmMTdjMDdhZjg3ZWZjZWZjNDFlYWQ3MjlkMzc3ZjQ0OTcyNjdmNTlkOGQ5ZjMyMmM5OWFlZGM5ODFmMDE0ODNhOTgyMWM5MTZmOTQ2NmFlMDFhMDkyZmE4ZTgxZjRhMzdkY2I0YTk4MTY4Y2E5MjY1MGM0OTczMmYwODE1MDNjY2U0ZGQxOTAyMjk1NWMwOGVkN2M3MDJmZGRjMGQ5NjFjOWNkNzY0Y2U3ZDc0MDM3NDgxYWZjYThlMGFmYTA2Y2JmOTUxMGQ1M2YwOGNkMDFiMGFhMDMyMTI2MmE5YjgyYjMwOTJlMmUwNzYwMGUzY2E0MThjZWU2YjQ2NTlkZGQyYTk1YjA3MDlkNWNmNmRlNzRhYzRhOTI4NDgzNzdlZjBhYmMzYTVlNDdjZmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MrX_aA2b-pg7VnFiwPKGNofP3TXICXzXsBmm65gvlrSpgJGLz7xDCTCcD5BCcTlBCIzO6KnGZV998YtchpwqnA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_104201_66_2441_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.903Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjJpUm1uUDBnTEsvRXd0R3BPWUttL0hxQlRmcHZIbjYwa1g2YUExRDk2QzNGbU1CZHAvd291REhRSG1EYjZnVFRtaitOd2pyRS94ZG1RT0c4Y0NYRHh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDQyMDFfNjZfMjQ0MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmUxZDk3YTc2ZjQ4YzEwNTYyMmM2NTkzODFjMmUzMGU2MDk1N2Y0OGQ5OGM3MzYxZjMyYTQ5YTc4Mjg2OWFmMDJjM2U5ZGFmZDZiMDA0MmRlOWU5NmMwMTI1MDc1NjYwOGI4MzAwOTQ2ZTM2MTY3NzUyMjExOWVjZjg3YTYwMjE1Nzc5ODcxMWZjMjJjZmMyZjY3YmZkYWI2MGFjMWU0ZWNiMzdkYjFlOTczYjg0NDMzOTlmN2E2M2I1MmQyZjc3ZjUzNTBlODkwOGUwZjlmMzAxMjdmZjM2NWU2MDI3N2MyYzVjY2YyYjA2NjdkZTExNWQyMjRiZjUxOTdkNjE1MWNhYmJlOWMzOGZhMDA2MGU1Y2JmNjVmYWFjNzhmMmI2MzU0NDM4OGRjMTU2MTgwY2VlMWIxYmZkMzUyM2U5NzU0MzA0YjIzNDRiZjRmZDU0ZTI2ZjUwZjdiOGUwOTMyZmY0OGEwYmFhZGMwMDk2NWQ0YTI3NTZlMDUxZmNiNTc2NWVlZTU2Yzk1MzY4NmNiYmY0ZTkxZjdkNTdmODVhZTNmNjg3ODBjYTAyYWM4MGMyMGI1YjBmMmE2YWRlZjQzMWNjNTFiYTVjODdhYzM1ZmI3ZWIyZDcyNGFjYzlkNjlmNDZlNzllZWM3YWRmMzNmNDIzMGY2MjhmNGFiM2UxYjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.frWbsO4ZhWlp5840qBVFOc9U6QzSrSnwRmZ5tg2BlV-1wRzVGfXSnI_HZ0BFXJej271JCtKyP87PeKb37ZdcSA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_104201_66_2441_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.906Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InNUWExuYWVoYzIrVlZGU3RsVFdXY2xOdmxYOXdrT00rV0pSZFJnWHVLTVBHRytKcEZMSDg4VkZKRzg2clBTa3lGZUI2V0UwWlUzK1lwYjdxM1BZOCtBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyNV8xMDQyMDFfNjZfMjQ0MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWI1NzJjMTg4MzUwYjM1OTM1OWQ3ZTExOGE2MzU3NTdlZTQ3MDkzYTdkY2Q1ODYxM2M3OGUzYmNkMWRhZmVmZjFkMThjZTM3ZjcxOGQ5ZDgyY2ZmYmFiNDM1ODg5NjYyYmIwZjUzZThiNmZkNjBkOGVlYTg2NDU3OTcyOTgyMjFhOWQ3ZDU5MjRhZTBjZGViYTk2NjI0YzI4MzkzMTU1OGJmOTNmOWE2NWMzZjEzMTBiZDBjNWRkOWRjZjFmYjU5NDk4NWFhY2FhOWVkZjc0NTRlYjQ1YzdmM2JlZDdkODFiYTFkNzVjNDkwMDRmYmNjMDk3N2I0NTA1ZWUwMDQ3OWU1OGZkM2I3MDBhM2NkMmM3YjJjZTM4N2YxOGQ0MTQ5MGE0NDlmNzhhN2VhNzkyNzcwYjA0MTM5ZDIwMzk1ODRhY2UzOTQwOGYyODRlOWVjZDFhNjNiNjk2YTY3NDIxOWVlYzEwOGM2OGQ1ODM1ZTU4ZGM5NDRjOGZhYjZmZTNlMzkxZjk5OGM5MDFjMDEyYzI1OTVlNGM2MDYwNDg1NjdlOGUwNzNjYmJhMGMyMDI4YTg2ODU4MGI4NmRkNTZlYzdmZTY3MmIyMDNmYWRmYWJmNTliNDhmMDcxNmM3ZWI5ZmJhYzQ2MjNjNjYyYTQ0YmUxZjJlNzk5NmYwZTY1M2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LCb-L_f8KAXmS8jAxJLbA-LgtI3rMSMcVchix4-OlLRWrmnNKflOdG5ClrPprHPUfQctEzMiElzbYuPmgfgijA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210425_104201_66_2441_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.909Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InN2aEJabjFHZ1JsWk5lVGRUeTF0R1gxK2oxazRRaW9WNk5sZXd0a3MwK2lnSFNvbUpHRUpheUsvQnJCRGhSbk9zZ0ZNWmFFRWt1NlN0Nysrbjc2VTJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTNfMTlfMjQ2MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTVkMTQzYjVhNzg0NDI5Y2Q1ZmM4MzQzMmRkNDMxZTNhYjEzZWJiNjMxZjZmMmIyYjE3ZDM1ZWNkMTJjOTEzZTA3NzUxM2M4MGI5ZWQ2ODg0MjVmNTc3OGY2OTJlMzgzMGMwYmIyMzcyYTAzMDFhMTJiYWRjODIxNjE3YmMxZTBjOWQ0YjBjYTg1YzU0NDIyODBmZTI3ZWNiMThlZTkzYzBlNmQwMjYwYWE2YjVmNDZiNTg2NTBlZTUwMTkyYmM1MmNhMjQ3OTc0M2VjZDk3YzUxMDIzNzA4ODA0MjlkNWE2MTZlNzQ0OTIyNDczZDA5MGIwYzAzMzk5ZGY4NGIzYzNlMjIwZjRmOGNkM2Q1Y2RjNGM4NGU0NzUxMWEyYTg1MDQ4NTY3MDhmYmU1NjdmNDVkNGIzMGQ1ZjU5OWRiNzliMGY4OGFhNjVjYjI3OTE4ZTRlYmM5ODBiOGUxMDM2Y2M1NDU0OTc0MzA1YjlkZTNiMzg0YmQwMjJhNDUyNGVhNDBhMGIxMDNkNDI3NTYwNzJiYzkyMjg0ZGEzZWFjMmExMGQ0YjVlNWI1MTNlM2E3NjljYTk3ZjczNTUxZjYzZWY4NzM2ZTIzY2YwZDQzOTU2Mzk2MjIwNzBjNTExMzBlYTNiNjIwYmVjZTRlNTMwODQ0MWNkOGIxZjU5OGFmODZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OkiUiJ2OgP1us-XivAq3V0cCfim2kL5K4PV-I4dbqWZ9N5DdQ_9RB1a7fkNnqfZkaWUx583C67d3j4Uq3nWYaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102453_19_2460_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.911Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlQvYlpHN3IrVkdWdjVQY0M2blBzN0R4WDNYbnVFL2RheVhjMnJDUHY4aTlRaXp0Q3dFMHhpVnhsaHpCc2lRcjBNU1RmYjNwNDd4RExVRVo5aW5GUUNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTNfMTlfMjQ2MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTUzNjMyMzFlMzFmZWVjNzFkY2ViMWFmYzNhNjJiZDBlMDQzN2RiMDMxMzM2Yzk0M2Q5Nzg0ZGIxNzQ2NmQ4NDczNjgxN2QxMWMzNDJkMTg2OTNlMjc3ZWEzNjBkOGI2MDUwN2UxZWU2MGEwNGFiOWVkNDA3ODNiZjA4YmIwNGNhYmUxZjE1NWE0MjAzNDM2NDU1OTdjYzE5MTIxOTA2YWM4ZGFhOTdhMzA5MWMwMTBkMmI2Y2M4OTZhYzVlZWQ4MmQzOWU5YzhhZmRlYjFkNWY0NDVkZDU2ZTZiNmM4MmYxYzE4M2E2MzhjZDNiMWQ1NGYzNDkyMjJkNGFmYzIxNzAzZDE4MzE1ZmJjYzEzMDcyNzRlNGRlZDE5NmM0NzFiYWVkNWZkZTI1MWI4M2NmNjM3YTZlODg2NTFjMzk0NzY1MDNjNjM3Yzk5Y2RlNzFiZTc2ODUwOTJjYmE0YjlkNWUyZTE1NDAyYmZmYzEyM2Q2MzZmNTkyZDU0N2I1YzNhOTE3N2MwMGZiMjZmZmE0MzhiNDNiNmEwOGU0MjRlNjBmZTJjMWI0MjJiYjM1NzI2MWQ3MmI2MjhlMDE5ZmJkNThmNTY0NzRhZGQ3YWYwMDk4MmE0ODRhMDM4MWU2MmU0YTMzNTFjZjI1YWE1MDMzNjE5NGI4ODdjODM3OTMwODBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6bIZFWIjeHLiPXOOVRVzaC30stab4e2pPGJ0fdch_l77SDDvU2PWZpn5XB7bKn8-bW_UbS1AbyYIa6LUxFSnzg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102453_19_2460_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.913Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im42alBNSmxLVFhxSnRsZFBYRlpuZmtOYjU5ZmU5OEpzZmFUNndkWHFKMFVpNmZvcEZSb3orQ1hLNk5KbTgvQ0FxZmhXWFVuM2dqc0d5eWpZMVZLdGFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTNfMTlfMjQ2MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzBjNTY5ZGYyOTYwZTJjOTQ4ZjIzMTdkYzIyZjE1NDBjZWU1NWMzMzlhYzg4MzFjNzVmMTkzYjViYWRlN2UyMDc2NTI3NTgxMzJjNzFkYmE5ZTM1NzJiNzRhYjVkY2Y2M2Q2YzQyNWUwNGY3NjJkNWQzNWNkN2ZmYzZlNzhlZWE3NTg0YWMxYzg1YzgwY2QwYTU3MDE0MDVmZTE3ZDBlZTQyNGMxNzBjYTMyYzIxZGNmZTgxODI0NzBjMThjNzJhZjAwMGZkM2QzZjczYzMyNzBiY2VhMTJjOGMzYjJkYzI3N2M5YjI0ZWU1YTc0MDllZDI5NzJkNWIyMjdhNGQxNzJiMmY3Yjc5ZjgzNzAxOTI5ZjYzZDkwZTNhYTFhZTMyODg4NTYxNzdkZWQwMzAyMjE4ZThhOWFmMzBmNmE3NmY1NjVjNWFkMzA2ODA2ZTliZThmYTJiMGEwZTA0NTIyMGEzNmJiYTlkNmU3YzcwMTI0NmUyYzI1ZGJmYWYxM2ZkNzRhYTNjZTNmOWNiYjhkYmQ5MjQxMTNlOGZlYjQ0NmUyNDhjMjBmZDMyYTM0ZDI4OGY4ZTI3ODNhNzAxMmRlM2E5YTNlMzdmNmRhYjMwNTg2NjhlMzM4ZTIwMDVkMTViMTNhNzgzZDdiNjU4ZTdhOWNmZTExNDA1ZTQ1YzBiYjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IG0exEj4RNZiXQc7jOiWBDrnNAQJzx6nyXTz-Rzu01TLkBkNTweGiVvmr3g4dzK0WbjYNuHex-kW_i2wccsdFw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102453_19_2460_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.917Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjhiN1FXWWNJdVZyMndEMm9iK09LY1p2a3RkWmRuUUlWUmFzTjRxSFJXUEFsWXIrL0pEelFGbVJ4cHZYMHZUcmtzcktrbjFkZHhOYTlkdGZMcnFlVkZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTNfMTlfMjQ2MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTYwZjNkN2U2NzI2ODE0NDJkYmU4YjM0NTVlYjQ3ODQ3NWQwYzkyNGNjNTQ4YTliODg2YjE3NjlmZDRmZGU1OGFjZjU0YWYxYjVjNDE3ZmQ1NzExZGY2ZDYxODczNjIwOTViNWU2ZDgwNzZiMzU2NTIzZDZlZDAyMDMyNGMyMGY1OWU4NmFjNDUyMzlkMzk0N2VjNmQ5OTFkZGZmNDA2NTE5MDg1YWVhYWY3NDZmODBmOGNlODJkYThiY2IzZTljNWRjYTM4MzJkMDYyOGQ3ODY5YWUyMWY2ODVhMjg4NjkyZTY0ZGE0NDQxNDcwNTk0ZmU3OTQxY2QxYzhlMzVmNWZhMTQ4MTYzOWZjNmJmOTMzNjBiMjFjNTliNGM5MDgzZGNkOThkMWYxZDQ1MTVmYTI2NjA2NTNiOTZjN2EwM2QzYzRiZDMyZDljOTA5ODIwYWE4ODY1ZWNhZmI0M2Y5NmMyNTlhNWYzOTVhZmM4MjQ5MWU2OTZjODNjYmEzYjMwOTkzZjBhNjJjYTdjMjcyZmI0MjljZDk4YmY2YTAwODM0Njc5M2Q2N2Q0YzYwZTlhZTMxZjkzNDFiNTEzNDhjOGUxZGMxNDc5ZjUxODIzMjYxZDU3Mzk1YWFiZTg4MjgzODBjMjhhYmMzMWU3Y2VjMDYzNWE3ZmY5OTRmZjM1MTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BINA5hdqjFhkcTnOpqmovbm9A5zGstVMd0o4U11xhI5Pc6lPPToHKCHuRPgeeTvICgyjhgyIAPUSyBVQOGC-5g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102453_19_2460_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.920Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IllrNmZDVXVhelpaY3hCSWtUYjJUNkMwMlcrMWNUWWFVMFl4VlI2RldDalFseVRSd3BzV3M0RnYyanN3Rk54cnFyck5xbitKRTlTdGV2YWpRNHdwcklnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTVfMzhfMjQ2MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2Q0MGY1ZTBjMjUxYWQyYWU5YjQwN2Y0NDM2ZTU4ZTEwNThiNjVhMjk4MDlmODMwZTE0ZmZlNzI0MDc4YjlmMTViNWJmMTZhYzhmNTIyMTJiZWQwOGFkMThmNzNjYmI5MjRhODg0N2I4NWQ2MDU0YjE3MzQ4M2MwOTdmYWQ2MDhhMTM3NjUzYmIzYjA0ZDIxMmI3NjYwZjllZmY4YjgwN2Y4MWQ0MzI5MjU0NzI5MTAzMjUzOTQzZWUxNjkwOWFhZmUwZjY5ZmZjYTdhY2RhOWFkOWIzMjRlM2NhMDE5NjU1Y2QyODdiOGM4MjRhODczNWNjMGZmYWFkOTVjYjAxZWE3NWNlNDExZGFjZDk1NjI2MDFlZTY4YTMyOTQ0OTRhOTM4NTM1YzcyYmFhMTg5MDE2MmM2YmJhNzNmZWI4MTAzZDk4Y2YwYmYxMmZiNTA3NjFmMDY2MjAwYWZjYWFmMmIwZWYxNTYyNmFmM2Q0MjAwMTdiYmJkZDQ3MzhmMzVlYzc0NjY4OTFlZTg2YjAxNTU2YWIzNGQyNTZjMWQ0NzU3NjFmYjY2MjJlOTUyMGVlMDBhMzU0OWEyNjJlNjUyNmUxNmQ0ZDg1NWE2MjNmZDgxNWZjZmQyYjVlZDEyZDlhNTg2YTg3ZmQ5NDIxYjM4ODBhN2QwNDFmN2Y1YmYxMWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.01tPfpUtho_uHag3qfT0P7bmpgPCpmlIlgF4o3kHahJ_zEfDj6INohCCuQP2DgzqA_zXAij4aYZiCYEYQ9tgMw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102455_38_2460_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.924Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InB4OGxQNDVaR3lEMmJZbk40SlJpZVVNOXVXWk13c01MUmFmV3VmakszZnpra1phTzNXZ0VZV2lGU0FNY0dtWld0QkpVVnFCOXFmNkI3VitsRUlQVHhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTVfMzhfMjQ2MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTY5YTY0YTI5MGQ1NGI1YWRhZjYzZmJkOGE5YzJiNmQ5ZjMxY2FiNzlkMDkwZmExMzAzYzYxMmYzMWM4NGQ0NDFlMjEzNWEzOGVmYjAxNGNkNDc2ZWVkYTg1MWNmNmY1ZDA3N2RhYmVkNmMwZjJiNDQ3NTVlZTAyOWQzN2I5YThjNmM3Nzc1YmU3MGY3YzUxZTc5OGMwNTJkYzU4MmFiNDhlMmFlMmJlMWY4MzY5MzUyMjg0MGRiM2JhMmYwNTQ1NjVjODExMDM3MGU4NWE4YmIxMGFiMWQ4M2ZhODA5ZjRmYWM2YmZjZTA1YmQzMWI1NmFkNDAyZjc3NjVlMGM2ZTU5ZThiMGQxNmM0ZWY2NzNjNTA3YjQ3Yzk4MmM0OWRkNWI2MmU1NDQ3M2RmZWJhZGE0MjhlNjU3NjM5M2EwZTBkM2Q1Yjc5NGY1ODQ2ZDcyNzI1MDQzZjBjZThlZmNjNDQwNmI0YjNiMTU5YzU4OTQzODM3MjdiNzhkMDUzYjE4YTRhNmJiOWUxOGY3ZDk1YmZjOGUyMDFiNGFkZmRlNmNjNDk0OTNiOWM2MjQ1MGE2Njc0ZmJiN2RkOTE1OTE3ZjMzMGVkZWY2ZmE2ZGJjZGUwZDRlMzk5Y2Q3NWQ1NjgxNDFiNzZhOTE0ZjA1YmUwNGE2YmYxMGMzOWQ3OTdmYjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cxLM1OuFEP1vyaLMoWDG03Rad-4jgbty94hGWEb4i14_S0sNIVGeu_Piwa1kMLlQLlAxAPxOXF4LocK4dF6m-w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102455_38_2460_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.927Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBXQWRqOWdyNWdTZWFEWEFRSjFYS0trZm9Xc0k0V2gyY3VIMUkxSUpoTmVGdEZSZ2l6VkpFMTdzVSs3YThPYkd4MkpnLzFJakhaQ3hORjJ2QldSOWJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTVfMzhfMjQ2MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODIzMzcyZmIzMmIxOTE0ZjJkMDM4ZGU0ODJkNjk1MmE0NjljYWMwMzQ2NzI4N2UxYTM5Yjc5M2FiN2E2Mjg5ODkzZGM5MzdjOTE4MmVmODZlMzZiZTZjMmM2YzVhZDIzNzVlN2JlNmZiNzY1MTk3YTE2MjMxNzRhODRkMmZjM2MwODc4ZmVjMmYyNDI4Yzg2NTNkOGNjOWQ1MGMwMmM4NTU0ZDJjMWEwYmJlYWU4MGE0Yzg1ZDllODBjNjk3NWYxYzc3ZjA3ZDAwMzNmY2FlY2NlMTM3YjQ4NTc4MjNhZmExYjI3ZDgwZjQzODMwODdlMmNmZGJlNDU1YWQzNWNmOGFmMzc5YWMwMWI4ODQ0N2IyMzczNjA4ZjA3ZDU5NmYzMWU3Mzg4OThlNzlmMjI2YzdiZTNhYTkyODVhMDczNzhhMWQ5ZjFmM2Y2ZjkxYzZlMjJmOTI5ODY3Nzg3YTRiYWE3OWYxOTZkNTE4NjUyYjcxYTA4ZTE3ODQ5MDU5YzY0ZDdjNjk5NGU0MDBlMjRmMGU3NjUzZTA1ZGY1NzM3NTA1NTc3YWFhYjIzMjk4NDIxZGI4Y2RiYWU5MGUzOWJjY2UwZTI5YjAxNmVkMzc3OGExMWRmMjg0NjBmMTVhZmQwMmNkODM1NmIwNjk3MzM1YTk3ZjUxZmEyZWQyZWNiNmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AULD2HyeNrz3ANWu-jXGixfknJmALNfROFvvzVXuVL6x3kwRtHX8THCL-a12sip1D3rUJRV7Lm0ZbZsuTwxRHw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102455_38_2460_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.929Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjA3SElsL1ZYdHdIc3VDSTArWkxLNS9rcEdTcmlHYUthbHVoY0M3aW13aXAwR3I2aWQwTzdlRVdoeVhyWHMvb042MjRFMStyMkFydW1xL1YzQ0xPUVRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMF8xMDI0NTVfMzhfMjQ2MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGFjMjNjZTY0YzhmMTEyNTkyZGJmNzU2ZGIzMWM1MzNmYWEyYjYzM2JkNTJhNDRhMjUxMmY2NDgwN2VhMTBlNWZiYjU2ODFkZmM2NzkxMThkNGY5ZTQxNTEzNDNjM2M4N2U1YjhiOTg4YTg1ZGFjZWM3MWRjNDk1MmZmYmRhODkyYzhhNzc1MGI2NjIyMzg0MGJkYzFiYjE1YTg2ODQ4OTMxNWRiMWIxYjA5MzYzNTgzYTM1YjczNGUyYmZlZjMwMGI4OTgzMWZiZDhlNjY2ZDRhZjNlMWI0MGNlMWI0MjM5ODlmYTY3ZTRlN2YzZmIyMjIzNjU5YWI4YjRlNjI2ODkzMmFmNTAzZTFhMDljODFlMzNhNDU4ZTliODNmOWM2OThhZGU5YmZlMmM5MTg5N2I2OWQ4NGNlY2NkOTFjMDVjYTRjMDRkODRkNjg2NzkxY2FmNGMwNTIyZjRmMDc5NWFmYjk4YjcyOTQxN2JlYWM2MDFiYjIyYTA3ZGUyODYyYWUyY2U1ZWRiNmU0NmVkYmU0NDNjNGVlN2JlYzI3MmRjYmQ5OThhMWM0YTUyN2FjMzAzNDcwMjBiZDI4YjA0YzQwMDMxNWNlY2I1MzA0OTU5ODBjNDcxNDgyMzAzNTZmZjYyZTk3NmNhY2ZmZmU4MGUyNmZhNTNhZmZlYTlmZjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.e6zSsm0EfhzwUoakJGVrl3EMbdyde0_oTiLrM3yl-z_i5F0HPIZcbYgPUeInn7rDpOEMBaCaXfpar9ALVex63g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230420_102455_38_2460_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.932Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkhrWGJwR04zUUhWOFhYeWZCVldvbG0rOWpaVk1TMGtjZTZwcU5CWGc5Q2NLQXJqMmdkS1BIbGd2c1Q2SHRNU1V2aVhCVWpRQXRMOWlLWWZlVTgvZlB3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNF8xMTI0NDVfMzVfMjQ5Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmVjOTQzYjQ4OWFkOTllZTk4YzdiYWM2MWFiZTE4NTNmNjg3YzY2ZGQwMWNlOTUwYTlhYjMzNjNiNWJlOTBlMzgwOWM5MjcxZjc3MjI5MDkxYTEzZGYzOTQ0YzUyMGMxZGQzN2QxMmQzMWMzMzQwOGYzMWM1OTM4MWMxYThjNDYyZjVmZTlhYTZkYTFkNjBjMmFiMjJkZTNkZmIwYzUwZjAxM2I3ZDNlOWZkYjI4MTFlN2M4ODYzODdjNTY0MjI0YzY5ZTUyMjlmN2Q3YTA5NjVmMDRkODIwYWRiYWM5NGFhNGVmMmUwZTIyNTI2OWZkYzY5MDM4MWRiMWY0NjAyYTcyZTMwMWZhMGE1MTQ2ZjM2NWZhMWZjN2Y1N2I5ZTNmOWM2ZTU1NmUxMzBlMmU1MWQ3Yzg1MWQ0MmY5NzRiNjJkNTRlN2NhNmFiYjFmN2UwMDk2ZWE3OTg5OTY4OTM0NzIzMjc3YTIwZmZmYTRjMWQ2YWU0YWI4ZDYxYzQwNWE1NjQzMTFjNzJhMTA5OGY2YTQyMTQ5ODVlOWM1NjE1MjZiNTFmYzA2MzBmMzdkNjA3N2IzMTFiZThhNTFiOTEwZDcyZmYxNTY5N2FjNjMxYjRiOTQ0YzUwOTc0ODBmMjA0MWJkZjI4MGU2YTI0MzRlMzEzNWUwMmIwZGFmZjYzNjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TxkuRu4JSEzZjFKnjeZDmqn76VijP4Op-VFJS0EXs1CZX8LwbT-a1i4Y3M_DdP9m3BmzKz9L9JH6BaPlOkci3A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240224_112445_35_249c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.934Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Inp4WmJQbDlCYkxmS0VVTnhaM1VNeWtOSFoyVE04cWczd0w5NDdlMTBYa2l2WnJEbGdVaE0vNEx0WUxZb2c2TzQxOEdMQVRzS2k4NFRMcDR6RnVnbmFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNF8xMTI0NDVfMzVfMjQ5Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjIzMzZhYWVjZmVkMTk2ZWFmOGIxNzUzODgzYjA0MzU3NDYyNmJjMDVhZDQ2MWMyY2QzNTQ5NjdjOWM1ZjA0M2JhNjVhNTllNTlhMTdhYzU4YjhhMTk5NjJjMWNmZGFkODkxNWE3NTFkZGYzMWJmMjhjZjA5YjQ2NDhhMTY2ZjdjNjUxNjM2MTc3NjkwMjM1ZDcxMGM5YTlmYjMyMjMyZDY4NTA1ODAzNDM0MGM1NzdiODA4ZGM2NGJmODE2OTNlNTU5NWVmNzgzNzU0NzhmYWRmZmI1NzRhMGE4OGFjYjcxNDdiZThlMmU2M2I4MTE0ODQ3YzYxZDIxMjg3ODc2ZjYwM2IwNmJkZTg3MDc4YTY3N2I3NTMwYzkxOWU2MDU4MTBiMTRhNWVjNjM1ZjVlYzhkOGE2YmJlMWI5MDUwMzQwNDc5NTgzZjE5OGQ0MDc2NGM2MWE4NzE1MjFiMTdmZWQzMWYxNzM5NmQ3YzJkMDI0NTkwZTBlYTA2MWFkNjVlZWYxMzEzOWEzNTM1YTI5OTE2M2EyNTYxMTc0OGI2MWM1MGFkMmU4MmMzZmQzOTYwYzMwODQ0NjE1M2JkMWUxNGRkMWViM2ZkNjBkYWYyMmE4Y2JmYzJjMTZmM2M0MjYxZGM2ODY1NDA0NzIzZjY0MjBjYWIzODI3YjgwZjA3OGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lF-O0S8RRYMe2ZkoEkcFbi8At1xQjZCbK7ZijOXgYTVnIybMrmQ6TmNFVETd6wGB67KY-8TEU13VNZeZlK1_pA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240224_112445_35_249c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.937Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Inl4TTdxdnlWUDZoSzk2WURIMW5pSWFQRGpEaE5pY0NQNElJTTZyYXZIZndlYXF1UThPcG9uTHpKTUlmNFNvNWhkeUdMdk96ZkRqRm9ocFAzUDRmSnNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNF8xMTI0NDVfMzVfMjQ5Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmE4ZmY4ZjQ4NzY4ZTE4NDdiYjkwYmM0MzliYWNjOGE5ZTQ1N2EzZGU3NzA0OGRmNTVlZGY3ZDgyZmIzNTc3Y2E2ODhjNTQ0ZjZhMTdiNDU2ZWJhZTI3MzlhYjdiMmJiMGM5Nzk2ZWEyZGE2NWRlZWI5NWIxYzg0YTNkZWMzNjZkODE3OGMzMjQ2MTkwZDcxMWJlMWFlMGUzOGRiOGQ4N2Q3M2E1NjM3ZjI0NmZkYTk2ZmU3YTdiMTc0YzlmNjgwZTY5ZjEyYTUxMzhmYzY5YjAwNmU4MDAxOWZkZTc5Y2Q0Njc1MTcyYmFmMDM1ZTA3YjkwZjU4Yzg5NWZhOTQwMzlmMzRmZjM3ZTQxYzhlYjhlYzdlYWM5NmFlMTI4NGJhNjUwMWM1NWQzYjVhNWFjZTVjY2E5MDNjYzk1ZTVlMGE2ZWY2Mzk1MTkxYzFkNDFjNDEwMWY1Yjg2YjBhM2IwYTdlOTU4ZWU5ODljM2QzYjhiNWM0ZTRmNzc3ZTQyZDAyY2VlZWY4NWE4NGFjZjMwM2M5Nzk2ODcyYTE3NWRiZWI2N2UyYzg1YzYyNjZiYmI0ZDRhNzZmMmQxNWU3NGVmNmZmMWE3MmUyYmJlYjFkOWMxMmUzNWI5ZWE5YzI0YjNiN2ZiZGNkM2MwODAxNTdmNjJiYTE5NWVkZmVmNWUzM2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ieJYi5JEkHmwFzsGpGP8OgUB5DMNaGKpqe4IfMAodp4RBo6F8LJAy7QmEDhGL7TvRLyf8KARrwMGRzBuKrFg7Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240224_112445_35_249c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.944Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImRTRHlSdHZsS0R6MHl0MkZHZC8yeFZnaUJGall5ckxKRkV6MDQyNXovMS8yakRsSms2WjB5bXZNdXFvZkgyZjREUzV6cGN3bTRvQkpPZ1kyYS91aUNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNF8xMTI0NDVfMzVfMjQ5Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVlZGY0NGIwMTBlMGMzYzg1Yzg2NmI2ZWE4NGI5ZDZlMzY0N2FiMGU2MDhkMmEzZGJhMjJlYTg3YjlhMzhhN2Q3NzUyZjQ5NmI2OTZlNWJkYmIxNmZlMWIyYmVkOWYyZDcxNTZmMWJlMTA3YWZjYTQ0MTg3YmY0MmRlNGQ0ZTYyNmU5NmQ0NDc4Njk2N2E2YzUyZTkyMmU3YzZiMThkMWQ4MTk1MjVjY2EyNTVkMmY0NzViZTY3OWI2OWJlNDM3OTQ4MmIyYTE3ZDdiZjgxYTRlYmEzMTJjYzk5YWVlOTU2MDY3MmU1ZmI5YjU3OGJmNTVhODUxYzRmMDVmYzkzOGZmNzEwMjYzY2ZlMGIxYTQ0YmUwMDZkZGJhY2Q2M2Q0MmMzMjdkOTY0NWVjYTM3ZDc4OTdlYTA4ZjA3NDY5ZmI2OTNmMjc4N2IyNmVjODgwYmYwN2Y3ZjRlYzY2NGE4NDhlNjYxZDJjY2I2MmIwYzA4ZTgxMzk3YjJkYzdmMWZhMTc3YzcwY2RjNTQwZGJkMWJjODEyMTI2ZjRlMjZhMTlkNzE4MmZlM2E3Mjc4YjIwYmE1ZjZlMmRmMzZmNDRjOGRhMWE3MWViMzUxMzQwMWY4YmExMjkyYmUyNjhjYTk4ZWFhYjJiYzlhOTAzYzBmYTNjZWYzMDhiYWZlY2E2MGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.kdGYuhjGeLCW6p5ZERRqxWNuKkl2St9DCn_SPnLCXY1qh7DS6Q8dEwEGuKlXiTBc6lnBtlnqm1zsHjye7Nif0A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240224_112445_35_249c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.948Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkxSSnFFVENpamdCckFRYlQ5OVRZOGxUWUJiYnpUU0xGNjk1VzFEci9HNEVDbnVMV0hJajBYRnV6amE3TW1hQ0UyQkZkZ2FBV1lFZU1tN00vdlEybEFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxMV8xMDMxMjNfOTZfMjQ1OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzU0M2VhMWUyYTcyNDcyNTU4ZWU3NTJkZWM2OTIxOWEwMGQ4OGE2ZjNkNDM4NTQ4YzBhM2I4Yjg0NmY4MDE5OWFiYThjMTQ2NDg5YzExZWMwZmEwY2U0ZmY4ZGI4OTE0YmMxYTRjMzQ5NzNjNWM0OWQzNTEwZGIyZTNlMTIyNTM4NjcxY2RhZDhiYTI1MjExMTY0OTQ2YTUyMTg0ZTZiZDg4ODFiOTgzZjQ3MmY0ZjgwYzhlYzBiNWFkYzYzM2UwZTcwNGZhZTY5ZmU1ODY0NjkzOGFiZGUxOTRhNGY0ZDU2OWViOGQwMTk0NjZiZWQyNDZkNzUxNjBhZjUyZTliMzY4NTY0MjRhMTI4N2RiNGQ5YmNlODQ3OGZiZjM5ZjIyODQ5ODA2Y2NiZjZkMjA4MTcyNjM3OGJmMTIwNDkxOWZjMTcwY2EzZjI0NzUxMDI0YzlmZWZhYTA5MzRmODJkZDliYzBiYmY0ZmM0MWU4ZGMzNTljOTRkNWUyMTgyYjMzMWM5YjdhNmEwMTI5NzM1ODY1NWRhOWUyMGFlMzFlMjdjM2MwMzNhYTAwYmNkMGQ2NzE1YWE4ODgwMDE5NzYxYzZjMzQ5MWMxMjEyZjRlMTRjMzg3YjliNzAwODgyNjUxZmEyMDVmM2UxNjY5OWFjYjJiMWUxZjMwZTRhMGQxMmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Mi9nea4Se7caF0WjQyhCwjTT0xnei0RqeVFmo6erdznxGd9w6WR-PeLyn13QE5bNk_SMgh0RekJBoJY3wzjJmg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231011_103123_96_2459_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.952Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkxRWnJGM1V6OEtHSWxWT0N1UTM3R0daWUxaZEdSbGJiL2JuN3BYNXJEdXRDOFFLSXlISm1ZWld5bXpOVmRQNXJPQmJWaStuZ01zckNRa3dCaXRhQ0RBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxMV8xMDMxMjNfOTZfMjQ1OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQwMDc1OTZkMjI0M2RmNmY0YjJmNjUwMWVmMGIxNmM5NGQ1YjI5ZmFlZDY1MThjOThjMmRhYmE0ODMxNjdlOWZmZjQzMmZiMjcwZGI3ZDI4ZTAwNjkwMjQxNGViYTllMDdmNjQ0MGM2ZmE0ZmVjYjY5YzA2Y2ZlYmQwOGQyOGE3OWYwNGUxNDYxOTA1ZWM1Y2IwOTM3Y2NlYmIxZjE5MTExZWQwNTdiODQzNjkyNTI0ZTZiMGNhMThlMmE4ZDgzOTFlOGE4MDBkZDliNWY4Nzg4NjJjZWQ3NWUxY2UzNWYyZjMzOTAwYWNkYzY2Yzc5YmQyMDRhYWNiNjc0ZjA2NzZkNDk4NTIyYjEyYTI1NTdiNjA3OTI4ZDc5NzFjN2M1ZGIwMDQxY2ZkZmVjMjA4MjVhZWQ1NTAyNTZhZjRhYTdmMTQ3OWQyYTA2YjFkNWVjMDVlZDI3OTAwZjljZTYxNzg4MzBhNTAzYmEzNjY5OWE0NDM1ODcxMjNmY2IzMDM2YTc4ODY4YzA1MTI3NGIxZjUxMGRhZDU4MWM4MzlkYTcxOWExMGZlY2U0YzYzNThlZTQ4ZDgwMDdmNjc3YmIwYjNkODFlNDA4ZjQwNmZiYmU0OWFhMjRiMDFmMjgyOGUxMjIyYjc5MjkyYTJkYWJiNGY0MjZmODQ0Zjg5OTAxNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.F0yNIKd4h4g7655WcGs5v1sw-8RxOfr2qWv5Sj5G9KQva1PLRCfc6cwzVNdba8PKh69y4nMgKGS5C3eRm_S9YA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231011_103123_96_2459_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.956Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6InhKMDBTbDZQWkEzRndqek4zWFpMNkZrdTdmYWxlRW5Ia1hzV202Q1dNZHJzQmg5cWF4OGxyRE1INitoUXJPb0ZIWGlVVVF1WERDdDAwbGg0c3pXUG1BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxMV8xMDMxMjNfOTZfMjQ1OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjRjMmQ0Mjg5MzhlMzZlYTdiNGZkMWRkZjQ1NTU1ODM2YjEzZTg5MTUxYjA1ZTEyNGFkYzVkMTllMGJiMGExZTFjMTRmZWU4ZmVjZTBiYmRhNjgyYjYwMWY3NGM5NzdmMzJhMGZjNGIwZmVmMjNiNjkxNGJmMmU5NzY0OTM2NzI1YzVlYjI5M2E2YzA0ZTZjNDZhMzA2ZjEyZDcyMGE2MTNjYWRhOWRkZGFhMzBjZmMxMTk3ZmU5YzlkYjgyZmFhNWRlMWVlODNmMTQ0ODQ5YTMwMTViNDU0ZmJkMmIxOTVmOGQxYWNiYjRlZGI3NmVlODNkNjcxZDE5NmY5YWZlYjhmOTU2MmYzZjkzMmM1YWI5OTJhYmQ3NGZhNjViMGQ0MGM3MmVhYWQwYTlhZWU5YjJlNDkxMGJiYjI2Njc4NmY3OTYxOTAzOTlkZjIzMWY4NWE4ZDI1ZTNhZTI5NWNkZWExMjg1ZTRlNzMzZjExMTk3YjIwOTQ4ZGIxZmJkY2VlNGI5ZDJhOTQ2MDQ4MDlhNTZlNzE1MmMwMWEyYmQyODliOTE3YWZjYTFlYzdkY2RjYzFlN2U1NTU1Y2NiOTM4NGY4NWYyZmJjZWY5YmVlNTU2Y2RlNDkwM2M4OWY3YTk2YjhjOWMzNjQ2NWVmMjBjYThjOTJiNzIwYjFlNDU4NzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GJz_IcpJL-PQ0fow9pdOCEJjveJiDQbH7Q_ii5Llo2DtjxEMpuvKPy7jMFcmky4cMx3C1hZAI6ulvkp0nNaA4A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231011_103123_96_2459_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.962Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im16S1JSWndiVjJPQ1cyaWc3UXZibFRwL1NqT056dnJtWkJvZVdkN2Z2MFk3UFFLSEhNUGhPVWV4VVlJU2tuYWdtdlFFL2FFVDJ0R1E4RCtjK3RCN0JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAxMV8xMDMxMjNfOTZfMjQ1OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDEzMjkyYTkxZjA1MTA1YzNhZDkxMWRlMzBhNDM1ZjljN2E3OTEwNmRkMDk0MjE4MzA1ZGJiZWQ4YTEyMjA3NWQ3Y2NiZGUzMTRjOWRmNDA2ZGU0ZjE1OTc4MTQ4Mzk0MjI0MGIyYzRjZGExNjIyZTNjOTFmZTc0OWVlYzIzZWMyZTRhZjQwY2VhNWY1Mzg2MGZjMWJhMzg5OWJmNDM3YzY5N2FiODcxYjMwNzg5YzMxYzA1YWJmYjc0M2RlNDcyMmU4MzJlMDM1ZDM5ODk4NWM5YTgyNTI1MjY2NWVkNjljZGQyMjY2NWY0YTZmMzYyM2E1Y2Q5MDUzM2NmNTQ3ODIxOGIzMmI4NDYzZjk0M2FjOTAzOTBhMGM2MWFmNTk4OTRlZTg2N2I4ZDUzOTljODhkODgzYWZhN2ExMmI4ODlkNmE2NTQ1YTVlZWRmNDg3NWEzNWQwYjIyMDI2Y2IzZmRhNTMxOTllZWQxMjM3YjIyZTgyYmM1Mjk4NjBmNTlkMTk3ZTdlNWQxNmVlZThjNTMwMzU2YjY0ZTM5YjJjYmQ5NTQyMjVlOWViMmQ1NWZlYzMxZmQ5OTZkMzBmYjU3ZjQ5NWQ0ZThiN2MzYWM1NzY3YjJlZDdiODM1ZjhjMGExMzgxYzc4MGViZDczMmQ3MzU3MDgxMTJmMWM0NGUxZThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RD1Nm2rSLEPLZLtYFJroGMbY2rjUS52HaV3D7hwidT_qmVkCkiG-8canE5HiJrf-aR9rCpf60vPK78pFLdPmiw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231011_103123_96_2459_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.965Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImJ3ckMydC9QSm5PTUM3UFZUdXFXTEJIWkZ4RldURXphbUJLVGhOU1M5bzR3dUxpaXordFd3Z240RWhQOHlERHlIaG5yUDZLZG8vVE5HNElLZHJSa3JBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMThfNTBfMjQ1OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTkxNmQ0NDJkZjFkM2U5MDY4MTMxMzc4NWMwZjE2YTg0Y2ExZGMyYjMwZTlkM2U2ZDNlM2QxMWNiZjAwOWYwMmY4MzI0NDAzZWM5M2Y1MTcyODdlOGQwM2Y1ZTdkYzM4MDNhNjFhMjIyMTZhMzg2ZTRhYzRlMDg0YjMzZjFlMmQwMDNmNzFjOTk2NjliYzZmYWM4ZDMxODc1ZWJiY2FkNmYxNDQ4MTVmNWUwZTIyYTgwNmZhYjM0ZjU5NzRmZTdhYjBkOWQ3NTE4ZWVhZjJkN2E4MjRiNGM2ZWFiNGJhMWNmZTc2NDE1MzUyYjQ4OTFjYTUxYzZiMjlhYzQ2ZDVkMmI4NWUyZTcxZmU4NTE2MTdhZWI0NWNlNzA2MGYwZDk3YWVlYzI1MzA0MzA5OGEyZWJmMTQ1ZDA5YjM5M2IyNTU2ZWFiODk2NWVlY2Y3ZWNhYWM2YjFmODM3ZGFiYTI3OTEyMGE4NjAxODAyNjcyMmJmMzAxZGZkOGY2MGRhNjljODgxYjE1N2E2NDAxZjgwOTJhYjE3ZjcyZDY0YmU4MGM0ZjZhZTE5MTU2YTY2NTdiYjVkMDJjZjY2OGE5M2ZlYTcxYzU0YTViMmM5MWZmYzQ0NDI5ODg2NzBlZjdiMTRlY2QyZmU0MTA4NjI0MGZjZDU0MWI5MzYyYzE1ZTcyZWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hcopucNOAh5uHXv3fCkwZkSyssDlJSvELi9IeNsfQgvMJ8Q6G7mFjmGiMmfOq5YHvP6JJcDrReNhrqumcLM5eg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.967Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkNaUXp0WWxKSGdtZlRvMEMvdWlSVUNLQW5BN0lpRUt3RXBvdThFdDdyQVBwNW1xN0ZKQWVicXNVekw2TVNLYkZJcmxVSitpUnIrRGxWUEFjcDZ6RnNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMThfNTBfMjQ1OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjllNGJhYzc4Y2EzODYzZThlOWJhMjYyYjlkMjcxNjk5MDk2MzNiMDU0YWY3OTk3YThhMDAxMjE2MmU0YmM3Y2ZiNjg0NjA1NDQ0YmUzYWIxYWU5NGM4Mjg3Y2E3NTg2OWI0MjczNTU5NzhkZDI4OGQ3MTBjZDExMWUwZmU5ODkyZGI3YTk2YzViMjhjZWJjYWY1ZmE4ZDgxZTA1NWYyODdhMzY1ZjQ1NTVlMzBmNWUyZWVlYTVjMjIzZTkyYmU1M2U5OTBlMDVhMWUwY2FlNWYxM2JjZGVhNzgyYjFlMDE3ZGNmY2YzNzllYTkyYTdjODVjZTQ0MWQwYzJlNDhiMWQ4MWE3NjNhNTE5YWY4ODIxMTg4NTY0N2UwZTVkNDAxYmJiYzc3ZmJlNTY3ODc1Yzg5NzlkODZlMjBhMDRlMWQzODZlOTQ0MDM2NDM0NWJhMjA3YTBjNmRiYTk2NzhmY2I4MjgwOGVkZmEwOGIzMjEwNmMyZTBjNGE1YzVjMTU2YTE5N2U2MWJiZWMwNTQ0YjU1Njg5MjE3ODMwOWYwYTc2NDdmYjczYjhiNmZkZjE2MjdkMDViNjczZDJkZjBiYTYwNzE1NTY3YTE5NTM5ZjBmNmI1YjQxMDJmY2M2YmJhZmViNjA4ZTNjYjg1ODA0OWFmODQwNDgxZjI1YzUxMmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7H5epGG6DGkb3dqrpgjC2ROzsEsFd_frILnO5zjkiW1giskh2Rtcxh2Pq90tt05GpRiGK3C3bBaMVPt3odKqwA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.972Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6Im45RXRFVmphRkg5Ti9lbHZub1VLMDRJNHlqTWZpMnFtMFZFNmtGQXkyV0pLNHVrM3RNZDFvZktVeVZ5Tmp2TmFtS2FEblAvanVQRkIzbmNsa2J6bzZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMThfNTBfMjQ1OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTVhYmYzMjMyNDE0NjY5OTJmYzMwMWJhNzc2Mjg1MjYwNmY1NWM1NjBmMTA0ZDQyZjNjYzUyYTAyZDhmMTNhZjRmNjEwMDlmYmNhNGY2MDkyNjAyMWFlZDM3OTk0Zjc3MTVkYzRiMjc2YTJlMjA5OGI4N2NhY2NiYzRkZGU1ODE3ZjJkYjE0ZWE3MDk0NGM2ZWE4NzYwOTQ1ZmUyNzUwZmY5NDRmMTIwMzQxZTMyZDgwMGNlZDUzNGFiZjk0YTc1OWIyYTk3NzhiNzZmZjZmODYyNjQ2MzFkYTJiNThjOWFjNDk1YjE2OGE2OTc5Yjg2ODQyMjM5ZTk0NGM5NzA0N2NmYTk5NDZiYTJjOTIzOWI1NTQyMWQ3ZmQwYWIxMzYwNTAwNGY5NWU5NWFiOTYxOTA1MDg5OTkyNDU5YTA4YjYyOTliN2M3MTg0MzQ3NzRmNjMyY2VjZDgxMzQ5ODc0M2NjMzQyYjU0N2M1Yzg2NTUyMjgzN2FjMDE3NTViMzExNzZkNWUwMWI3MDc4NzA2OWVhYmQ0ZDY5MGQ2YTg0MDIyZjQ1Njk5ZmNiYzNkYjI5NDk1MTEwZmVlYjZiNGUzOWQ3YjhiODU0MjhiNDdkYWRlYjIwYjNjYWI4YWYyOTllNmMyN2JkMDJjYzljYjgxZGQwMGExM2MwZWY3ZGEzMmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ._m7Msdru1mnVBgSLVuOau8LInWQJ_sdVPWE2XIfcR1JG1Ni4LEPhiMaoBjFy7PLIWoTMM-xBG1wD4XyszW0wEA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.976Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IkFHd1diY0pweEJrOXBHSXlaS0IvZlprbkJhT1R0VFd1bjRWN3pnTXlTVFI0UnRoMjVCU0ppb0tSRGw2QVRWZk4wa2wrdzB0STRFcUFQYU5SaHZWclhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUzMF8xMDIwMThfNTBfMjQ1OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjlhNTc2MDYxMTFiOGNiYjFkMzcwMzAzOWQ4NzQzMDI2MzZjYmI5MjVjZGJlNmY5MGIxZTIzMTQwZWQ0MTdiN2Y5MjdlYzliMGRmMWE4N2UxNGQ5NDM2NmE0NWRjMjFhN2U1MzQ0M2FiNmRiYmY2MzYzNDc2MDE4ODAyMGJjYmVlMzhmMDMzY2NjYjRiMjBmYWZhOTExMmY2NjZhMDk0OWNkMjY2ZmFiNzc4NmM4OTAzNDY1ZDY4MDFkZWE1ZDIwODVjNjdjMTY2NTYxZTI3ZDI4ZTM0ZDRmZjY4M2ZiNzZkZDFkYWIzMTUzNTQ1ODc1ZGMzNzAxODJlNTNmNDNhZGViNDQ0ZGZjN2FiMDg1NWQxMmZmN2Y5M2JhMTllOTlmNTUxZGMzN2IzMTEyZDNjMTI1ZDg5NWNlZDdkMGQ2YWU5ODFkMzRjMDhiMDhiNDFmOGIwMjVkMjdmODg5MTYzNTdkNTlkMzY3MmEyY2NkZTVlYTY4YTdkNmEyODlkZjRhYWM1YWJlMWFkMGUzOGZhODY4YmU0OTg5ZmE1NDNlNDcxYjIwNzc3ZGMzYmVlNWE3MTVkOTdjNDFhMzI1NGVhZjViNmJhNjMwOGQ0ZDIxZDIxYjJkYTk1NDgzYjY5M2MyNDU3ZjExZjFiZTUwODdkNWU0MWM4ZWRhYWVmOGNkNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.E7NSJo3bVCTY6hh_mF3T3-09ZcwobIxDssLR0nbvaj6i7Eybrkz_rI75kD0d_5ka_ns-R3eRvsH4J3P1Yru1wg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.981Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImlJOFVpMENrWnBUMVJmWkM1eHVlQkhjWmZ2ekdkVCtHZUEyZlpTd2JqNzNMWjF1WFRjdTVYMjhhVllFYU1Za2cxeWxLeUx4dW1YakJOSVltWVJocW9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTlfNTZfMjQ4MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2RiYjBjNGNmNDMyOGMyMzAzZjFjODQ0ZWFlZDg0ZWM4ZjRkMzUwNWQzMWM1Nzk2NWUxYzQxYjFhNDRiOTliZjAzZmZiZGFiZDZlNjkxZWZhMTAzYjA4ODI0MjdmZmVlMmI2NmE5NDVmZDlmNDZhNWRiZWVlNjlkNDVhZmZmMmQ0NzJmMzI0MjQ0ZWU3MDc1NTQ4OWFkZjNmZjgzNjYzMzgzZDI0MDNlNDM1YTQ4ZTUyZjk3Y2Q2NmI1ZjRjNjJjZTdjM2Q3ZTcxOWViZGM3YzNiZGNjZDNmZWQ5OTc1ZjNkZjA4MTZkMjExZmI1MjZkYmQxYzkzMGFkYWMyZGQ4MmU4MjUxMTA2MGIwM2Q4NGI0Y2I3MjA4NTkwOGY3YjFhZjIyOTEwYzJiM2Q4ZTMzZTRmNzIzOGZmOGM1MjNkYTYxZTM4NWY0MDFkMGU1MTE5Yzc1MTk2NTQ3NDc5ZWE0OGNkMzIyN2E5NGFhNGFiMjcxZDE4NzUzMDNkNmFkN2MyYzlkNGY1OTM4OGQ5ZTRmYzBkYTg2NTg1MTcwOTNjN2U1NTIyNGFkYzNhNTQ5NDY0YWY2MTI0YzViMzFhMmMyNjBiNzc2Y2UyZjIxMzYxNjQ2MTgyZDY1ZmJlNDVlY2M1NjNhN2E0NWFlYWViYWJmODU5MGMxZTNhYjMyODQ2ODdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zW-J7jFWVN-yb7_HZhIOkg3MDNnZQc7_HwOaCOKfKRphx11VRulWvFptDAwd59nOXTU3XlrWp0T8_5mBN-0Emw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.984Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlVJeHBBdzhYamQ0dkxLdzF1S2t4dGdoRG5ZT0R2c3pHRzFQbVZ4K3orQnZzN1RlSnFoeE9JZFRKcUdwT2s1OHZhQWdNRUZOU1JOY1FWbkw0Ry9ibnRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTlfNTZfMjQ4MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWY1MjY4YWY3ZTE5NTQ4NzgyM2I1NzQ1Y2M2ZGI5NmRkMzgwMDM3MTFkNzMxNDkyZjYzY2IwOTkxZTI5ZTljYWJkMjE0ZmY1NTNmODllZTQyYjJlOWYzYjU2NDhhNTc2ZDQ4NGJiN2VkNWU5OGFmOTkwN2JmYjJkNzdjNTQzZmUwOTBiYTI2MzhjZjNjZjc1NmY5ODM5NDcwMTkwMmQ1YjEzMmM1YmI2YjIwZGYzN2RiY2NmYTViZWRlNDRlY2YxNWY4ZWZkYWY0NjE5ZWU2NDEzMzYyMWNkMDM2ODJiYWM0Mjk5NTA0NTE5NzIwYjVhYjc1ZTMzY2Q4YmE2ZDRkOGY5NjFjMWUxYmEzZjlhYzhjMWM5M2U4MGE1YzMzMDVlZWZjZWNjMjBkMmNiODIyYmE5ZTI5Mjg2M2MzZWIyYjg2NmYzNTIzMjBiZGYxMjgwYzRkN2VlYjM5MmM4NGZkYTNhYzQwMjQyYzliZTMyOWU4ZDEyY2U1OWQ3NjVhNzBiYWFiODUwZWNjZTNjZDI4M2NkNDk3MDY0NjljZTUyMjlkNzg4MmIyOWRkZWQ3ZDE5ZmFjYTU4N2FlODdiZGIyMmJhOTkyMzQzM2NjYmM5NjVmYTJjMzJjNTY3MzExMmY2ZGVhZWMzMWY1M2ZlYzg3NTk0N2UzZmNmMjMzODRmZDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JvZQ4iFMOnPHjh8N-sE87idxHzWJbo_Okak6spfkpK3zl41NCT4oJCiZEfinccgjss_KB6ExecpnbaoR6VbX2w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.986Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjZxQVN3QlMvTmM0NWxrd0tRR0hmZ3BsVHVIcGMyK29zZi9HRER2Y05HMGJtYmJtaWRBd2Y1NGZEWGtGTFd3SWV6eTZGYzczM3hFM1VuWnBMZG5ZTTV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTlfNTZfMjQ4MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2JkYTc1NWFiMjk3NjQ2YjlkMzI1YzU0MDVmZWQ2NTExZjAyZGMwZGY1OWZlNzUyZDc5NTlkN2YyOGE1YTE1YmI0ZmIwMGRhZTkxNzNjMzVhNjAxMGU3ZGYwNzA0NmE0MGM4NjVkZDk2N2RmODU3NjVjYzdlYmU4ZDY2YWVjYzgxZTJiYmViZGMxNjUyYmExYTdhMGY5YTRjMGQ4M2QxNGIxZmNhZGI5MjVjZmEzODkwZjhiMzk2ZDBjZjk0Y2M4NzE4ZDIzNzdlMTUxY2MxOWNkOGY1ZGUyMzZkMmRhYzlkMWE2YWNmMTlmYzAwYTkxMWYyNGRjNmYwZDljMDA1YjM0OTg2OGI4NGZkMmQzYTc2ZmJhNjU1OTdlNDcyZWZkMzZjMGY3MGQ0ZWNmMDQ3MjQ5OTMwOGYxOWZjZGUxMDJmMTJjNjgxNTZmOGU3MmE3YTU5NjI3ZGRlNjY4MDFjNDZkN2MyNjJlNDVjNGJjY2QzYmNkNGE3N2QyNDg5ZTdhYzgzMTVmYmI0OWJmNDY0OTgwNTVlZDBiZDY4ZjU1YWM1ZmI5NmY4ZjkyN2U4ZjU2MDI4YWZiNDE1N2JhY2U4MDZhYTZiMmMwMWQ3MzRlYTk0ZWI4ZWIyOWRjMzFlNTFjYjk3YjRmM2ZhMjVlZjQ1ZDY1NzFhZTRhODVhMjkzMDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DAAAKx69qyzKynN-qlG2SuzeSILYwFUjUa0Sa6QbdvK2_kAVsmX8Ih2Wl39oDgEySywyZzHVPXSm7prCS7EsbA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.989Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlBTOWpyTHBSd0crZkFPV0grSHFET2VmN09xcnBNcXc4eTkyWjd4L3ErNytETnhldEMzQW5lMk92YkFwR0JHTEUyc0x1cmowYjhPWDVseUJ5Uk9TL0ZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcwN18xMTA0MTlfNTZfMjQ4MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGQ2NzU3YjI0ODM4MWY2YjNjNTdkZjMzZDZjOGY1MzhmNDJlMWU5MTY1NzI4Y2FhNmVhMGMwZGU1MDk0OGQ5NTY4ZjAzZmQwYzc0NTdkMzNhNjUxNzRkMjQ2ZDFhM2Q1NWI3MzYyYzJhNTcxNGJmZDlkMTJlYmUwMzA2NmIyMTc3ZjVhZDYyYTI4MjllMjY0MDkzMzI3NjViNTA2ZDk0ZDYyMmRlNjkwZDk1NjFiYWFkY2JlYzAwODdiZjY3NzY0MjNiOTFmN2E0OGZjYWEzZTg5MzcwNmUxMTQ1M2FmZjI5OTQ5YWZjNDAyYzA5OGE3ZjU3ZmM5MTY5NTc5ZGE3MWY5MTg0MjczMjUyMmU0Mzk2ZjMzOGQ0ZmIxYTNlYWQxYWNjZmM5YWNkMzUwYzg3ZTc3NjBlMjYwNjM0Mjg2NzQ3NGZiMTllY2ZiZDQzYjc5YzUxMjZiNGQwMzBmMzQyYTE5ZWE2NGQ4YjQyODNmN2VlYWU3NWYxZWIwN2YxN2MzMGU4YTZkYzQ3ZDM4ZWIwNWY2NWQ5MTQxYjk2YmEyMzNiMmY2NjFlN2U4MzM4ZjhhYjU3MDc5ZjJiMGFkYWRiZDU1OTI4MzA5ZmM0ZmRlZWZiMjQ1NzFjNWJhMmYxZmQwMGExM2E3NzMxNDc1MjY3YTVjZTlkODQ0MTZmMDZmYWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1Ov7mXdLkvpO8ZpIgqKSJZ_DSkGtWitssfgFxR3Hb7L2W0WiIyADygjBysN9MZUPpB0RZ5XtFkUFQfeDB-1OaA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.992Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6ImRISUFkRTNkeGZZeWE4WDJVaWNDNHZkZ0ZkaUJjZnNmblo2aTFETy9zdzgxRWsxSThQUlpXai9UY2IxMWpwdlc0ZDNPS21tNndydUdmNmZsdmQxanl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzNfNTZfMjQ0MV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTJjNzc0YzZmMGQxMGI5MzE0OGJiN2QzZGQ0NDFkZWViNWZiZjRhZmUyZDY0Y2EyNzY2OGYwYjdkYzFlMTdmZTBmMjA3YWU0NGYzMTU0YzFiZDMzYzU2NTIyNmE3NDRmYWY1NjU1ZTdhYjJlM2UxYmI2MmQ2MzY3ZjYwYjBkNGI2NzhkZGY0YWE5NTc0ODNiYmM3ZjhjZTI5YjQ5NTI5YmYyYWRjOGFhOTMwY2QwYzllMDVhMzhmNWQwZTc0ZDc1ZjM4OThlOWU3YWU3OGJhMTVhMmY0ZWQxYmFhZDJhMjM0NWU0Yjk1N2QxYjA5NTlkNjE4YzE0MjBkNzlmZTExNTFlYzNlMjYxNmVjODRhMDAyZjYyNWE4ZjFkNjUzZDBlMzdmNmVhYTFmMmRlOTVkZjlmZjVlZmYwZDEzYTdmN2RhMjE4MjE1ZWNiNzVhMzE0NDhmNTQ0OTE3YjhiYmRlZTEwYzM3ZWJhMjkwMTFjYWMzNGU4NDU5MWJiYmRhZDVjOWY1NjNkNTQ5MGFhMzZiMWJiM2QyNWMwOTU5NGQ5OTMxMzQxOWIyYTBiMjI1ODhmYTFkN2M0Yjg1ZmYwMDk0NGI0MjJiOWM0Y2YxZGJiYWUyNjNmZmNkZDMwMTYyOTJlY2U0ZTMzMmVjNjk0NjRkZjRkMWZhNWIyOGUyN2JkMGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.A_YEibuEeczMGt50Gn43xIA_CGeI1KNfT75CLMS2q8FcuPV5mPPOaseJmxJfeT1BhDSbn6ip8VJi19MKjSKCFA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104633_56_2441_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.995Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IlRhL1ZJZWdobFRwTTlUSDJ0OGZXOXZVRmFGbHA2dUlDR0U5WC9LbUFmeTZyVG5KZldoNDV0aGQ2dmFlNDRQUUYrYXAzY0tOZjhZNGllLzlDSjBqejRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzNfNTZfMjQ0MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2E1OTcyYzVhYjliNTBkNjYzODk2M2Q0MDIyZGNiYzk5ODZiZjk0Y2E1ZGE4NzVlNDdkOTUzZjMyNzEzMGM1M2ZiZmIyODVhYzUzYTUwNjFmMzY3ODAzZGVjZDE2ODVjOTJjZTIwMGU0NWU0ODc0OTEzYmY3ZTAzMDBiYmZlNDY2YzM2ZTM3NjI1NTM0ODA4NDczYTI2NDJmNDliYmM0ZGVlMWI5OTQyZDViYjM2YjE5YWI1ZWNmMzM0MTI4YzA4NmJhNDkyMjhkMDU4MmExNzlkMjY3NDcxYzliZjBhZjIwOTcxYzg2NzNjZTY5OTQ4NzE5OWJjMzUwODUxOTllMWUzNWViN2Q3MmUzNzc3ODhjMTEzODgwZWY4ZGIyN2VmMjVjY2E5ZGQ3M2JkZmY4ZjMxYTdlNWY5Yjk5MDk5ODdjYWVjNjZmZjNkYWQwZDg3NTgxZTljODQzOTRjMzgyNTUyM2UwNWVjODJmNmU5ZmYxNGM4ZWVjZGU1ZTliNDA5ZGEzZTc5NTNmNjA0Mzk3YjI4M2ZmMmIwZDNkNWQ3MzI0YzNjNjFjZDIzZGE3MzAyYTEzYjU4OGIxMDU1YjI3OTg2YWE5ZTI1MTFkZDdkYmQzM2I3MzYyMTkwMTBjMmYzMzM4NDk2YWVlZDM3OTZiZGNiZTkzYjM2YzZjOTlmZDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7qZ8g-iWWfxptWyzaw4_GJeqDJbKMxbRl9z37M-zYlrz3XkyEkKOoXj3Cm0uk45AGXPNrDRrMq6OGdWqzGnKnQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104633_56_2441_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:33.998Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTMsInN1YiI6IjcxOVhVODRyeU52VlovazFIVzlMcVQ5cGlhR2xIZkdKd2RJSCs2SndzLytWa3R2b056eWZiRUFMK09aN2dRZy96TVpKTTBrV1MzdTlBWGR4b3Qzd2RBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzNfNTZfMjQ0MV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2EyN2EwMjY0ZGMzZTBkNDQwYzNiMmQ0ZTFlZDBjN2E3YWQ1ZDRmN2JmNzVmY2UyNmViYmVmYjdiMGNhNDgxYjA4ZjI5OTRhNmJiZTgzOGY1MzA4ZGM5NjUyMTM2YTJlODZmYTdmZjA4ZGE4YWQ3NzJlYWQzMTdiYjU4ZDYxMjk0MWY5MWM1NGQwMzk2NzYwYzNhMzcwNzE2MTM5NDIwNjhiMDRiZGIzMjliYTc2OTY2ZmRhNWM0NjAzMWE2ODJkNjk5ODI0NjE2YTMyYWQ2N2IzOTI1NGRkNGM5MDkyMzFkNjA1MDMyNDZjMjE2YTViMTMxMzQ2MGQ1OWJhMjQxOWUzOWM1NDQ2Y2IwNGM1ODQ2NWY2ZmE1ZGI4MzRmNzQ2ZmZlNTNmOWU4YTA5ZDc2ZjVjNzE5ZjUyYzUyMjk5MDhiMjkzMmI3YmU3MjIyYzgyNTA0MjhlMzg0MTdmZmQ3ZDhlZTAxN2MwZDk3MDEyN2RjMDQ1MDgzYzUwMzFlMjdmOTAxN2VmYWUxNTFkMDdhMzE2YThhMjQ0OGU3ODdjODI2ODc3ZGUzMzU3ZGI0OTk0OTVhNzY4N2M0MTBhZjFmMWQ2MWZlMjBiMzQxM2U4NjFjYzc5NGU1NDU3YjlkMjY3ZmM1MzY2NGNjMzM4NmFjMmJmMjM3NTU5Y2I0MTg4MTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rRGmQxAbJuFTZdHDsGardhJ4toVi02o-NEuXyw5Bwga2zf0y_hV5srZK8rhHFLooulOyrWTe8XgqX1OgPuQ44g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104633_56_2441_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.001Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imx0QjJuSTEvbDVTcFQ2SnAxZjVwaUJJekV4a0lVa2ZLcTlkWXVoNDAxYzUzRHpaREtRcG5YWHdtdzNMdEx1ZTVwQ0JyU1Q3OTY3Zkp3TEdVVUlJZFNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDEyNV8xMDQ2MzNfNTZfMjQ0MV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzM1pcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWRiZGZkNDI4MTI3MGVhOGY2ZjNmM2QzMmNmZGEyNmQ1YzUwNThjYmZjMmU1M2I3ODMzMmMxYTUwZDQzYjdlZTk5Y2EwNDYwZjI0ZWQ4YzMyNzk2MThlMzUzYzUwNzZjYTJjY2Y5YjJkNTc0N2NjOGFmYjUwZGJiMmRiZmY2ZWI4YjM0YTkwMzMwYzcyOGQ5NWMxMzlkYjVjMzA0NzgyYjIyOGUxYjliYjFlMGMzODM2MzE0ZWU4MDgwNTUyYmZjNjUwMGRhMjliOWI2ODJlZGNhMDhjNTczODNlMTdmMmU5NmYzNjdjOGQwMzUxMmRkNWY2MzFhNGMwYmU2MGE3Y2JhZGY4MzU3ZWQ3NzE5OGEwNjhkYTRlYzYzMjQyNDRjMzc4NWM5ZDlhNGU5ZTYxMGIwMWM1YmFjZWFkMDY1MTlhNzRmNjBmZmM4MDM2NmY2Mjc1OGVkOGM1NjI1Mzg3MTViY2VhNzZmODEyYmYzYTNlMTk3M2VmZWFkZDhiNTk0OTNlZTBhMzcwNDk0MjMxNjJiYTczOTc0N2Q2YTI4MWRjYzA0YjVjNjYxN2Y5MzY1ZDBkMDQ1Y2I2MTcxNGY1ZDllYjZiMmZmNWFiNWU0NTA3NjQ3ZTI1NmJiMThhNGRmZThhNDE4MDhiOTRjNGNmNmIxOGU4YTUxNWFiYmY1NDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8Ir2i3Lkzozm1c8nD0KV-PSp3K7vvh60yEQsF61TqlSsjxp5zdrQxJJ6livXemaAzs02FNcDKoAinB9DtmtEkw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_104633_56_2441_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.004Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InM3TWEvZC9GZkhIVEU2cnNkdmdHYW9sSmVLcGJZeUNyMmpnTTdWZzBWRHlncWhZZXhkL0ZOS2pHYVV4dGlJeVRPaDYrZDNaVnpFcHRUbzBadHF5dmJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwOF8xMTE0NTRfNjRfMjQ3Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWIzZjUzMjQ0ZmRmMGQ2ZTI2MzE4NDQ1MGQyODQ3N2FmNDBhMDU0YjdmYWQ0YjhkNjEzZTNjYzJlYzViNTZkM2VkYjBjZTQxOWM1YzIwYzAyY2FkNjFlM2I4YjlkYTU4YjlmNTAxMGY4ZWExNGQzNDdhMzA5MzhmY2E3MWJhNjM3MDJiOTViMDJjN2MwYmYwOGJmYTM1ZjAzMTlmMWFkZGNlZDEyNTc4YjE0ZDAzZDU3YTNkOGRkNGUyY2IzZGE1MWZhMmViODRkYTNkMTA5NGRmMGI3NDIxM2EzODBiNjY1NzdlZGM4ZTUwMDM4MjM4MzExNmI5NzNjZTE4ZGNmYzdjMGIzNzg0YTY1NDg2MjliM2YzYmNkMWZmZWVhMjkyMjJhZWMxODU5ZmU4Njg3ZTc2NDI0MTg5NjVjZDk2OWE3YmM2YzczMDNkMjdmZmM1MDI5OTMyMWRmNjliY2EzMjIzMDI0MjhhY2UyZTIzNTI2YzFlYWJlY2UzOTVlMTMxYzg1ZWZlMjA2NmQwNGYyMDJmMzUxYjdlZGYxNTJmMTk2NTc5MTUwY2ZkNmMyZmM5NDk0OTZiNzMzM2NkZjRmMzc5OWZmMzJjZDNiNGI4YzdjMTljYTIyOWZhYTNkZGEyMDNkMjJhNTE0N2IyOGEyYzQ3OWQ3NDRjNjM2ZTM5ZDJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wFDnnVaJvRJRKK1Wg7hvaVaHyPZNSGr08rEKj2PxRa6pKnwW5WFAFe242BWC5oga8RZuF5qrvTpifhqqWXXuYQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231108_111454_64_247b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.007Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjRWVDVZMWwwZUdkbTNhczMvTzZMSmNmQXBkUXQ0bWljNG9TdFd2S2J1c2RjN2w4MUZqbGs2bGU4cWkyZjZjQks2RnBoSDVEbkFLZDlyR084NGxROENBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwOF8xMTE0NTRfNjRfMjQ3Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTM5MTk4MWMwNjQ4M2IxNGFjMGI4N2Q1ODJmZjQ5ZmM2ZjgwY2I2MTBkMTg1ZTNhM2I1ZTgzNGMyMWJiNjcyNTI5ZWE1NWJiZDkxOGZmYWI3YzJmNTNhNDZjNTliMDlhNGEzZDUxNzQ5MzQwMDljMjRlYTM4Yzg1ODVlZmQwYmY2ZGJiYzUyNzMxNzYwNGIwYjI5OGEzZDc3ZGViOGM4OGM5N2ZjMjJjODAxMjQwZDgzYjllMzRmMWUyODMwMTlhZTg0Y2I1YzMxYzhkMWIwZTg3YmQ5NTU2M2RkNDVmNzcyNmEwNTA5OWQzMGUyZTNlMTRhMzJlYTg0ZTJjOTQ0MWJmM2Q0OTc4ZmVlZTc1YTdkZDZkMDJhNmEzODE5YTQ4NWRkODBkNzU5YTYyNTI3YzZhNmEzNzU3MjFjZDEwNmQ1MTk2ZmRiMWZmYTZjNzRkZmU1N2Q2ZDUwOTI3MDA4Nzg5NTUxMzcyOWM4YWQ1MWIyMGIzMWU4ZWJmZTJiMDIzODVlYzAyMTZjZWI3MmU1ZTk3OTg1NjkwNDdiMTZmN2UzYzMzMjU2ODA4OWU3MjVkYTc1NTc5ZjA5YTU1M2FiOThmOGFjMjhlZDU2YWNiNDM1ZWZkMjljZWYwMDUwMGMwODIxNWQyY2YxMDhhMDczNDBiZjJmY2IwODU3Zjk3ZmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.-LEGjASBQmnos70uURo7LkvmWYTohOe2LYYy9ruTYtR_bnkBj2gyvNBR7R6qCMncypHyKIXW1YvtEEj5x0PpEQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231108_111454_64_247b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.010Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjUwTUtyY3EwN0s3N0Z4Y3VmVzlTZEs0M1BRVDUwVFdFR0wvY0ovdG9KckpPSW5rS2Q4dklsS0NzdmRJd2hvdy80dXFyTlVodlp1S0ZWekYrSWlQWFhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwOF8xMTE0NTRfNjRfMjQ3Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTUxOTgxZmM5MWNjNzk0MjMwNzkwYjQ5YWQyYzI2NTVlN2QxZDRiYjNmOTFjMGU4ODNjOWY0NTI5YzQ4YzEyMjY4MWFjYWFlZDYxZjcxMDM2ZDQxZDU2NWEwMjUwNDUyNmIyZTVjY2U5MjllOTQ5YjVkNTBhMjM1M2VmNzgwYjA1ZDJjYjgwOGEyMjVhZDc5MTUxN2MxZDBhNGU0NjgwMmUzZjIzMjg3MDE5ODhkODAzNjJlYjBkMTE0NjQzNzY2ODkzZTMwNGZmMTM1ZjEwMWVlZjg1ZDRlYmFkMzhjNjQ3Mzg1NzEwZjU1MWI3MjJiOThmNWQxYjEzYzQ0OWRiNGU2MDU1MmIxNjM4YjQyNjlmMjBjYWMwZjQ5MDVlMTQ2ZWJkMTgxMjM0YjNjYTI0N2U2MmU2M2QwZDQ5YzgyNTMwZTNiNjkyYTBjN2JhMzljYmE1YzViMmQyZjUyOGVhYTVhMWE4YzhlNzQ3NWJmZjU4ODlkNGM1MDMxYzg2YjBkNDI5ZjY4OGFjNzE4N2E5ZjBkNWE3Y2E4MTIyNTJlNTU1YzY5OGExY2NmOWY2N2U4N2JlZWJlMWQyMmUwNTcwMzdlNWMwMzM4Y2QwYmEyZmQ1ZTQyYjgwNTM4YjA3NzMxNDM1ODE0ZjI4YTYyNWI2YWEyNGNlNmE2M2QxNWM5YjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2PmDmTfw26PRkFmuQChzmVJoCNtOwaf87HnkxskwA3Uuwj1QG_BAF2VagKOe6p8wXme-kIusuw5zP8WIlNmLFQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231108_111454_64_247b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.013Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtkKzZ4Ymt3RGhlcTBCaXhERFY4QVpXR3k2L3BURkY2Uk94ZElWOWc4d0I3ajdJQ3Z0SnBRQWJDR0hleThxQ1VBZ1hvOG1aa2RiUDF1SWdjcGgwYzNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwOF8xMTE0NTRfNjRfMjQ3Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2NiNDFkYWU3MjQxMThiMWQ2YjQ1NzFiNGIwNWM5NTRiNjU5ZTg2ZDI2MmYzYmExMzM2MGVmMGYxMGJlMmFjZDkwNGEwYjQxNjI1YzQ0YTRhY2U5MzY0ZjM2Y2I4MDg2ZTk3YmRkOTQzNjNiMTk2ZjY0MGM1YWNhYTFjMmNhNDE2YzBiNzBkNTA2MDU5MjdmMzczMDNkNjNkYjcxYmRhMDQ0YTY0OGIxODQwNTAxYjk3NjVhYjlmYWZiYjQwYTkxMmUzNjUxNjhhYmE2MzZiYzAwZDhlMTdmZThjMWQ2NGMxZTdmZDc2NTY4MjUyOGUzYjYyMGZmMzg1M2I1MjlkMmEzOWEwN2ZlY2U3MGFlNDJjOTRjYTUyOWQ5MTAzYmQ2ODk1ZTg4MzQzYzg0NDdlNTU3MzhjOGYwYzg2NTMzNzMxMDI5Y2IzZDIwMjM1YTk4ODFmOWMzNGZmZWJlNmZlOWMxMGFlNTE2ZGI5ZDI2OGEyNWRmYWZmNzYyNzUwZjViN2EyNmU2YzdlNGM2MDQ5MDg5YTUyMThkMjk2YjEwOTkyNzU5ZTY0ZmIxZmVmMmM2NzdhMDA0YWY2YjAyMWRmMWYzNTJiOWM1MTVlZTMwMDdhYWRkY2Y0NTQxMjljNWQzNTM4ODI0OTFiM2EwYmVjYWFmMTllNDVmNDk3ODg3YmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.InkOiCvsNs4M60KqZnuk1rq4HrFk1xJqGJFOON7ePxHfGHf-Ga-uXIyEgo1ktTgXLZxVT8LcJEWMUIqwAFu5CQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231108_111454_64_247b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.019Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImFEemxNUEw5NW90RVk2ZTd3LzYxZ0puMUt4L1FKODd5S3BlNm5jSk9pcllrNzdZYjBMYnlCQTEyOUNrcVhYMXNjOXpZQVJJU21wTTlJdUw4U0tsTm5RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxMF8xMDMxMTZfMjJfMjRjOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDM1NzkwODY5M2EzMTNlZjFjM2E3MDNlMzNhY2MzODdiZjc5Njc3ZTAzY2Y1ZGU5ZmQzYjc5MTFlNDU1NjlkNjhiYmFjYzYyYzNiMWIxZDljYzU2YTlhODcyMjZjYmIzNGM3YTY3YjRkMzYwOTY1NTY4OWNiNWI4NTkyOTYyOWNkODZkZTBmOTQ5MjhhZWRhYTFhMDAwNWJhYmI4Y2MxYjM3Zjg0MjZhY2NiZDZhZjQ3NDEyODdkODNhMTQ0ODAyYjRkNWVhMzdjODAzNzYwMTNiYzczMTJiZmJmMDdkMjE1NjMzYjU1MDBmZDBkNWI4MDk5MzA2M2YzNmM1YzllMjI2NTMxNDQzNmFjNGVmNTZiZWI0ZTljZDExY2MyNjA5YjMwNDZlOTVlMjI1MjE5ODA5YTFlYjg0NDdjYTcwNmZkODI1M2E3MmFiY2MyZjU1NjA0OWY4NDVkOTIxYzRhZTcwMGM1ZWM5NmRiNGUyOTU5MTUwYTE4YWJiYzFkMGQ0ZTMwNjlkM2JhOTNmMjNmMjZjYjI5M2EyZDAyNGY5YmVlOTk2NDQ1YzEzNGNkN2MyNzFiMzlmMzQwYTc4NjNkNmU4OWI2ZWQxNWQzMThhNmMzZDc3ZTcyYTVmZTc1ZTdjYzAxNDc4NDM1Y2Y4ZGExYjA2YTcyYWZlODU4YTFkOTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.L1I_K95JKKFDND1Occ3Qh7pkdzBw1-Pvf-LNguPVtPx8tUJVyk8IBkROiEOabWgf19KeJbDKhPaX1zJZbJFNZg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230610_103116_22_24c8_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.023Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImgzYm1XbDR0alBZV3prdFJOb21kemdyUFBDQUdtNjNicTUra1FGcVVpRXFrblowUzA1N3l0SUVuTXdScXVMdE00Q0NiRzBQZFlJQXFOQUFhK3pxeGpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxMF8xMDMxMTZfMjJfMjRjOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmY3YmRlZjk2NTUzNjRiMjgyOTE1MGEyM2YyNzQzOGIyYWQwNmE1MjA5ODUxNjAyM2U1ZWRiNWFlMjVmNGYxMzFmZjc4ZDlkY2Q4ZDUzMWE5MTNmYzg1N2EyOTkwMDI2ZTY2ZDJjMDNkNDQyOThlNGM2MmEzNzliMjc2YmI2OWVkNDQyMTU2OTVhNTg3NjA3YTNlNDIzZjBhZTVkOGY3MDljODI1MDhmNWNlNmVjMjE3NzI2N2ZmMjU2ODJlMzUzZWYxMzU2YjE1ZmI4MTBjZTNiZGRjNzc4NmUxZGJlNzZjNGFiMjg0ZTIwMGU0MjgyNzMwYWRhNDI2NjczNGEyMGVhMDY2OTUyZTFiYjA5NDA0MzBkOWQyN2Q2YzZkOTdjMmU4MTdjNGFhYWMwMDRkMjlmYWFlZWZjYjNiMzU0ZGU0ZDE4NTdhYTI5N2YzNzcxMmVlNDU5YWViNDE4OWM5NDZmNTI3ZTk0YThjOTk5ZmNmNjk2NmRhNzhmM2I4ZWEzNjA2NTJhOTg2MGZhMDMwYmM5NmNiMTMxZDJiNjUwNWRjNjdhMTU1OGFhYWUzNjcyMDU5OGViOGZhY2EzMDNjZjI4Yjg3ZmJmYWU5ZTU3MjU4MjUzNjEwNTI1ZTA1M2EwZjEwYjQ3M2U2YzVlYWYyNDZhMGI5MDA3NTJjMmU0YTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3StfGsaIuD0vuShz-DmDGW8Zd4VprnYxF8EyFiRgIv7aYmuImZi2rhlIWzIgkoGIY5PuUabQ0jVNpq9kWZ8gHA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230610_103116_22_24c8_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.027Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjNaaTJrUk93ZHEyakFyMDRiaWFvYmlNYVh3bE5oVzN4NUE4VFRKU0U0N0E2Qk9NK2RLR3JpRGJqbTZxZWNpVDV6TG1jYmVKYVRDQUhpY3U0RDB2TzJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxMF8xMDMxMTZfMjJfMjRjOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQwZmEwOWVlYTZhNDBmMzBhNzA3NTUyYWQ2MjJkNmViNjU0YjhjMjVlZjBjMDQyMjA2MmY5MDcxOTc0ODc1ZjU5OTY5ZGFmMzQyNjBjZWUwZWE3MjczNzdjN2MyM2Q4NzBjYzhmYTAzN2JhMTI2YjM2NmRhZDRhODQyMDJjZDMzOWJiNWY2MTQ4ZTE4YTk1NzM3OWFjYWI5MjMwMTJjMTE1OTk3YTBmNjA3ZjViYTEyMDczYjY0YTc3NDdkNjg5ZTMwNjU2YmIyNTUyNDdjY2FiNDYwOWM3ZWRkMTI5ZTA3NjE0YzNmMTRiZjMyNDYyMGIyMTk2OGViMDQ5YzAxNDEyNGVjMjE4ZjJiMDRmNjUwMDI1MDFjZDUxY2RlZmIyYmU1ODUyNTE4Mjk2NmE1MjFjYWUwMzNmNzMwMDU4NTkxN2Y4ZGYyZGE4MTRmNjhhZmRlZWVmOGNjYTg2ODc2ZTE1ZmVkMTgxNTJkNDQ3OThhYjE0MmYxOGNiM2JmZjBmMTllMWRlMjQ5Yzg2OTI2YTgxYjQ2NjNiZGJlYjM4NWZhODk3Y2VlY2I4OTkwOWVkNTNjY2E5OTkxY2I5ZGFiYTkzNTQzNWM0ZGIwODdmNTc2M2ZhYTNkOTIzMDE1ZDJhMWZiMGNkMjAxZDYyZDJhNmEyMjk0NGJlYmY3ZDkwYzNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.DIc5zlqFjQ_7rAupqZUV_H-pDjnRri5pT23wcb7-oYfAVjHnD1Pdfnr0qA8YWJ6Q2szzqs5CtoKK5TKo8OYC5g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230610_103116_22_24c8_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.030Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik0wY1plRkJsNHlhTFlZVThYL3UvTjU0Nkl5L3piVmZWWFpaazBFTmJYSkEyak5qUGQwSG50N1R6dzgwbWVCNGxxWXc1MWs1aTdPR2tuYWg4Rzl1VG5nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxMF8xMDMxMTZfMjJfMjRjOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWUxMmNhMWRjNTM5Mjk3YjZmYmQ1ZTUzNTUzZTE2N2E3ZjY5NDM4ZjVhODZlODljMjhlODVjOTMyYTUzNTNiMWYxZTk5Y2ZkNTRhM2Q2YTQxZDBhMjdiZDAwYTM4MzIwZGVmODlhOGJkMWI0N2M0ZWY4NGZkNzRhN2I0MGFmMjBmYThjMzE1ZTEyMWEwOGIyNjhmNWVlZTNmNTkzNmZkMWVjMmM0YjJjZjQ1NDY2MGRjYWY3YTE3M2IxOTkwODk2YzBhOGU0NzkxZjhiMTljYzM1ZjdkZGUwY2I4MTZlODNlOGVkNjE0ZmY1ZTY0OTUxNzQwYzIyNzE1Y2EzOTE3Yzg5M2M1YzgzNjI4NzczMzM5YTg3MGI3Yzg5MzRjZjAyNzAzNmY2NWU4MWZjYjQyYTY4OWU0ODI0MTcyYzJjZjMwNDZlMGQ1ODI0YzBkODA0NzdjOTZlODMyNjE5OThkODkyNDVhNDc0YzU4ZGNiNGYzNjBhYzMzYjc4OWFkYzMxMzFmYTJjZWJlZGMxZDQ4YmI1YTU0ZDYwNThhNzFkMWI4NWUyZmYzZWJiOTZlMDU3ZmU5ZDJkZGMxZmY0ZmZiYjJiMDQzNjdkNWIzNTU1OGNkNzliYTAwYWNhNzU4N2YzNThlYzBhMGY4MWY1NmQwMjZkOThkNDllMWE4YmYwYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3QZ4BLUXf6O69zk4E3-TKMPDMv_1mVafKYmGF-CcHjuZrv6Rr-NLvjfzIdU9Pz3ixxxRCSiVDRf-6W5tecLY7Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230610_103116_22_24c8_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.033Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii9NakhrU09tbUVTQzUya3Z1YU5CbGdpVEJrSW1JWjdrV09zeC9PMDdrUExkaGZnSGhHUjlvRUYyOTdza1BzWTh2dW5IMllJT3diaHQrb0RaVzJSWHh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDZfMDhfMjQwMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjA0OTEyOGY2ODIyNjJiNmYzZDJiNDA0Yjg5NGE4NzdhMWQ1MzY4ZTQxYTFlYWZkN2MwNjJmMGYyNjdhMjQ0MzhjY2U2MmYyNDA1MmRlMTM2YmU0YmU2NjgxMzExNTIzZDg4OTM1ZjVlNGYwMTBlZGNkOTJiZWIzMGU3MzA2YmRlMzNkODhmYmNiODk5MzJmODdkY2U5MTNmNjg0YjhmMzYzZmJiMjA0ZWIxYzQxZDlmNDUzM2YyOGYzY2VmMDRlNzY3ZWFmYWFiNThkNTA3NGU5ZTk0YzAwN2Y0ODE4YzBmNTBlODQ4MGFmZDdmZWM4ZTJmNTMyMmIzOGMwZjIxM2FmM2VmNWRjOTVlYjg0MThkMjNhYzBjMTZmYTZjNTdiYWEyODk5OWUwODYwZWNiNzQ4ZGM5ZWI1NGE5YmEyMTIyMTQ4NjdmMDJjOGRiMzJjZTgzMWE1MzkwZGQ4MGYwMTJiNDEyYTBhODE2MDRiZDZmNGFhZjViMzQ2OTFmMzM2ZGMwNjI1NGI1ZGFiOGYzYzQ1MzE4MGRiZDU4ODIwYTg0ZTVmNzA0NmI0NmZjZWY5NTBlYmM2NjAwNDJhODkyMDFkNjJkMTU2Y2M4NTlmOTc4ZGM4YzI2MDM3Y2ExOGEyYTQyODg1YWNiOGJmYmFjMmNkOGY3ZTJiMjU5YzZmZDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lfiwM7nMQ59dVJQPDkPf92xeBKqBYai8GpB-zGo-Zcmkdsaed_E2t1hRXsLdh0lZtLgxLUF_jiATJfAo3l5DjQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112106_08_2402_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.038Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImE5ZU5HU3hCdjVPRFpGZ0owRTZBQ003Tis4cHdBSDJJckpHazRLYnk5cEVVZFIxNE9hUVNBNFFwalRxZ1hUM1JOenJVQ0oxUU1xOWh2VWVUT2JFNGVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDZfMDhfMjQwMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDM1NGU2ODFiMDcyZWU4M2I2NTUxZDI3MjNjYmQ4NjljYzE1MTBjODcyMTlkM2ZjNzI1ZWFiNjMzNTFmMTE3MzBlOTc2YmNhMmVkYjVkYTk1YzJhNTk5ZDk0Y2UwYjRiMWEzMjQ4YzZlYzRkZTFlMjVjOTZmNjczZjkxYTIyMDdiMDAxYmEyODRiOWYzZTJjMTkwODQzNmUxNjUyMWI2MWFjNzRkNDc3NTMxNmE2NDM3YWYxNjBhMzgxNDA1M2IxZWI3MmYyN2E0Nzc5ZTk3YTJlMzdjZDM3ZGU0ZTdjMTQ1NGU4ZDAyYzMxNmE3ZjAzZjFhNThiYjQ0NTA4NjgxYmZmZTU1OWY2NmQzYzA1ZmQxYmNkYTRmYmJmMGEwMGYxYjNmNDc2ZWU4ZTM2NWQ4ODJjNjMxYmVkODZjOGU0Y2UxYWM3Y2VmMTEyYjZlYzNiMzk2NmNjZGI3OTQ0MTkyN2IwODQxYzU3MjY4YjYxYTYxM2Q5MmY4YjE3ZDk2NGM4YWEwZjQ5OTY0ZmU5ZWU0OTFiMzY0NjIxZTc5MjdiYjIxNjhmNThmMzI0YmZjYTk2YmRjOTZkODM4YjFjZDc5ZjI0MWMzZGY3Mzk0Zjk1NDA5YTkwOWU1NWMyOTEzYmMzNTMxYmQxZDg0NmY1MjBlNGIxYzQwZGViZWRkMDY5OTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Zyxg-rbOtwVaeEVHWaOtiTodlhgVmtCQXRMsjsPVaHVpnhyc4TMIya2ZSmV3oxTL8ryilLTMYCC9rPDNwBzO5A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112106_08_2402_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.045Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Iis5UEVLd0Vvd1dzY29LRlFreHJrZlp6N2xTamc1S3RVNEtqdU13aVVEL0lRZEhqNEI1Vnk4WlR4b2RyNHFCUFgrRWtwRFZnN1VyQVcweER5OFRCNVZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDZfMDhfMjQwMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjgxZWRkZWQ4ZjIwMWNlNjU1NGVmOWQ1ZTBjYjI3YmY3MDg3Yjg3Zjk5NTA3NGU1ZTQ5NWU5YmIzNDZjYjcxM2E5ZWM3NjhiODNkNjEwYjY3NmI3ZDQ5ZTZkMGM3MmViZDRiZmVlNGUyYTBhNzY4NjIxMDcyYzg2ZWM1NDhhNjFjY2U4OTBjMTNhNDAxMWNhNDBlMDk2MDc3Yjg2YTE2NDVkYzg5MTdiZWM1Yjc5NmI0YmIxZTE2MjQ2Njg4ZjhkYTg2NGRmYTk2MTkwYmMwMTc0Nzg3ZmEzYWRhYTg3YTY3Y2E1ZWIyMDljYWUyZWUwYjY2YmI2YjQ3ZjJjZmI4MGIyODQ3YTlhNmU3MWUxNzcwNmExYTY5OGMxNjY0ODJmYTljOThiNTU3YmJmMzk0YWE3ZGNhOTY1MGI5ZWI5MTU4ZjJlNTA5Y2M1MTI5YWUyZDYzY2Y3MDg0YzA0YTc0Mzc1MjA2YWQ4ODA0M2ZkZTEwOGVkYTc3MGQxZmU4NzUyNWEyYWM3MTNlOWMyNmExODVkZWU0MGQyY2ZmYWFjMGM5ZDEwMzZkMWIxYzU3MzAwYzI3MTI2YzUyZGEwMjNlMDM2MjkzOGVmM2U5ZDlmZGRhNTBlZmYwOWY0YjFhNDNmMzZkM2E0ZDA5YTQwZDliZjM2NGU4Nzc4NGQ0NGRkMjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UGnM61mBlaBhpYJ7LwdijTkNmBmhFtq4BsbFc9lyF4GYhQukrpbqhous5XTNNqHG5tllr99kekmY4_mupdtarQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112106_08_2402_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.048Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlhtVTZQeGl1UUlLZXRQMVgzTFdRTGt4ekRsU2gxSHJNNDZ3Y0JqRHRGOGZFNW5EK2RmakNDK2E4OUNzb2dKMmVmOEpJRDE3aHZrbm5aSVpTTFM3QVFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDUyNF8xMTIxMDZfMDhfMjQwMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjU3OTcwZGVjOTczMmFmYTg4NDIyODBiOTM4YWE5NWVhODI0NjdiOGM0MDExYzRjNmU0ZDQxYmNmOTIzMjRhZTVkODgzYzI0YjUzMWNiZTNlYzAyM2ZlMmViNGE5MGRhNjE0ZWUzMDI1YTI0OTJkMmM4MzVhZDMxMDAxMzc1MjFlZmI2YmY2YTgyNmYwZGM2ODlkMzJkZGJjN2I1MGViODQ3ZGIzNGZlNDY4OTE1NWIzMTA5YzVjMjFiNTVkYTFiYjliMDExM2UzZTdhOWFiNGFmOTM2ZjhjYzk3YmRkNjU3NTJiYTFhZDRiMTc2MzlmZmQzZDY3MDQ2NTE4YmEzZjZmY2FmMWZhOWEyZjUwMDhhYTAwZTc4Mzg2ZDIyYzdjOTNlOTUwODFiNThmMjBlNDY1NGQxMzZmYmNjMzQ1MTMwODQyMGMyMjdiNWI0YWYxOWZlZDRiMmU2NTQyZTM5Njg3ZDE2NDAwZGQzNjYwNzk3ZTcxZTViNmI1ZjFjNDQ4MjY3NmRiZWFiZGNmOWE0MDRjMDNhN2FkOTE2MzgzNGQ5Nzg2ODFhMjA0YjA5ZWVlZjk0ZGI1NzllNjgyNWU1NTM2NDFiNDQ2ZmJiY2VlOTMxMmM2YTIwNjQzYjgwNzgyY2I4MzUwZjU1N2FjMGNiZDA4Yjg0ODQxNTZjOTg1OGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.scX41250cwDahm-gTjRv34YJ0t9YLf6Py4L0J10mEQ6RYJXC7Ms47tvG_qrtvK1dYcV-EFb2ue0EOSJSgj6A7A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220524_112106_08_2402_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.051Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlBiVVVKelpKek5STVU0SDJ3dzFCRFpoYk9FRytURWJBSXYrSEpEMWFxYjhIMVFsODlQcXNIa29yNGxzY2k4VUZsV2F3QlhLK2xnZzJyUERGNkN3OWZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDBfNjlfMjI3Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTgxMWExMzNmNmJkZWZiMWIwOGM4MTZmN2Y5MzUyYWY2NzljYWIyYzJiODY1Mjk3NmViOGQ1YWY5ZjJkMjU2MWRmODRhODdjMzNhYzExYzBlOWIzNjUzMWI4NDQ5NDc4YzQ4OWMwZWYyNWZmYTNiMzRiOTJhNzQyYzQ5ZjViYjgwYjE5ZTJjMjNlZTZlYzYwYTFkYTFlOGE3YzFkODVhMTY0M2VkNWI5ZDFlYTg2OWVhZjY2ZmZhYWRhMGU0YTQ1Y2VkNjJmZTg3ZTllNTE4MDhhNmYzNGRhNmUxOTc4ZTg1YTBmNDFkMDY2OWQ1NWRiZGNmYjQ1M2U5ZTE2Y2I1YTkzNzcyMzJhZDU0MDFiNWFlYjU3ZmIyMTM4ZTJjNDBjM2ZhMmNlYzI4OWRkNGM4MzUwYWMyMDMwY2JhMTEyZGMwYmFmYTBlODBiZmI0MGUxYzEyYjFhYzljY2Q4Y2I5MGUxYTQyY2QyMjA2MDYzNWU4YzZkMjlhZmI0OGE0NWVhN2JjNmMwYmUyOWRiMWNkYzlkYzU2MDAyM2M1ODJmMDQyMGI4OWQ0ZTM4NjQ5ZTY0MjI4NTA3NDA2ZDk4NTM1N2U0ODI4NmY3ZGE1MWM3NmU4YzM0NGY0ODYwMjYwOTc4MDkzY2JlYzVjMWQ3M2IxOTlmZDVkYWNkN2QzM2JjYTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3-c0-HEZMW1BhZoiyGfhxwBF7TBgCpYpzCB17bAqAZszfJ3QJ6ubdyfUqKUdRG6LjL8pDSn-uUmmQ3plAoR0ow", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113140_69_227b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.053Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpJNk5acENrdjlXRnlQNEw3VzRuZm1Bd1Jjc0k0UHRrbFFqb0FsY25FWWk5eHdVQW9pM01XV1M5czZ1WTdoLzV4MC9SVWN3RUhjeWFRMjA2QStSaStnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDBfNjlfMjI3Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTRkMGVkNDIwZjViNmMxMzdkZjExNWUwZjdhNzBhMjcxZWY5MTdjY2JlMzA0OTA5NWE5ZmI0ODZlN2E0MTFhZWY3MTc3YjJkOGZmMGFkZTJjMDAwY2JmNjdhZTdkNmUyYmE4MDZjZjQ2YTJjMjRmOGMyNDU3N2I3NmJiN2RiNGJlNWRlMzliYjRjODEwNjM1OWQ0MDUyMGZkMTMyMjQzNjY2OTdlMTgyNjdjOGI0NDk0OWZlOGIxNzBmZDVhY2FiYTc3MTJiMGQ0NGFiMzdhZGMyNzhhODZlOGZlMWI4NzljYTJiNTBkMmU1NjIwNmZhNDM5M2YyNzE4OWU1MjgxMWE1MjMxMzAyNmMwZTNlNzhkMjBjNWU1ZWVkMDUzNzg1ZTdlYzM1NGI3Mjc3ODYwMTJmODg5NjExZTBiNTNlZTc1MzcxMDg0NmQzMTEyODEyNTg4OTc3NDM1NGViZjZiZGVjMzRiNzAwZTI5NjRlMWE0ZTZmMWZkZWVmZmUxMzAyYmY4NjYzM2JhNmMyOTE0N2ViYjlmMWY2MWQ5ZTAwODE5MDk0ODVhNWY0NDJhMDMzNjQxZjY4NTljNDNkZGE3M2E3OWQxYTAyMzgzZGMzYzBlYTNkMzg4ZjVhYTdiMmU1OTUyNjI3ZWI2NjVlNmNmZjFjMTJiMDJkZWRlYWRhNGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.A4aVuofFtptwRNF4g6YsJmNIxkP917nKW22fOGA12bSeTNstKco_zfL34Yp2ugL3m1DyYmW0tZfu_qxgl82w9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113140_69_227b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.057Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InRhOHk3ditJVEptaVBCSmxCQW1pa291LzRyZXhzTUZUamlGZWJvMFJMSGRpeTlwRzdkVnVacGZxTGVoNURDNWo4R3ZhWlNjN01UUnMvazlVNzQ5K2lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDBfNjlfMjI3Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmJlZTBhOWVlY2Q4ZGU2YjA1ZjRlYTEwMTRkMDMxMTdhMDZhMWJkMzZmYzM4M2I5MjJiZTczNGE5Y2UxZmE4MDAwNjM1NmU4NTE1ZjU3ZDExMzIxOGIzYmRmNGQ0NmIwNzM5NDIzNDAxYzdjZjA2M2NhMGYzOWVhOGNhMGExNTk2NDE5NTlkNDU0ZDVhMGEyMGJjMTIzOGZhZTNiMGNjZDhiODQ1ZmYzN2U3ZWI5MmY3NTU1NTVhN2IyZmNiMWQ2NzRjODNjYzljZjllYjIwNDJlMDYwZGU4NWQ0MDJiMTM1OTllN2VhMWRmZTY2MDQ4NGQ5NjY4YjUwYjQ2NGYwZWMxNDA2OTU3NWVhNGEwMmVhOTdlZjBiYjYyOWE1MTRlNGVhMGZhODEwNDhiM2E2YTE4NjE0OTg3ZmFhOTVlMmFkOTJlZTc4MjBjMWQ2ZGM1YWFkYjg4MmRmYmZmZjZmNjQyMzMzZjI1N2JiODRkMDIwZjUwNWNhZTcyMjFlY2U3NDVmMjk2ZmZjZjc0ODViNWM1ODgyNjRmZjBiZTdhZTBiOTkwMzgwNTFkMzU0NTk1MzcyZDA1ZWY4ZTQ0MjQ5MDk2MjY3MjYzMmRkMjZlMDliNGU3OWNiNTM5OTljMjBhNjIyMDc2Mzc1ZmMwNmI0MGVkYzUyY2IyZmU4YjAzMWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8GwVUj3Pgi9_VovYSN71Eb-Amv5xLxNbI8ofru0PO_BQ9Or1KTSHo0lL_T4kRFUaHFzohqN_1smWpYZiEocMUw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113140_69_227b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.060Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkdtenNnRDhPbUh3VE1SWXN3K21vdlVFdFRuUTk0RlRhZlNJMDZBb1NHaENOOEdzWUthVnlnYXArWktwOWFGNkZrMGdCNTBQbXNiTG1rQXRuSERsMjJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIwMTIwM18xMTMxNDBfNjlfMjI3Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTlmNTMyMTJiZDYwMzM3ZDA5YTgyM2IxODAwMGI5YmRiYTI2MzA2YzYxMjgxYmU2ODIzYTg2ZGEzNzY4ZGNiZjJmMjQ0NmNiZWQ4M2Q4MTlhYTIwMjg4OWE5OTU5ODg2ODkyNzBjZGZlYjJkOTgxMGYwZjJhZmM5NGQyZTZhN2JkZmY1YzY2MjA0OGQzMjYxN2U1ODU3MjBiZTZiN2ZhNDRjODFlYmNjMTI1Nzk1YzZkZjJiMDFjYzEzN2MzMjlmN2Y0M2U1OTlkYzVkMTY5NTkzYmZlY2IzMTQzMWYxZDY5YjkwMzg3MjJiNTRjZjFkNmRhZWE0YzI1Zjg2OWVlZWViMTFkM2EyMWI4MzExNTliNzE1OTczOTQxYTdjODRmYzgyNDRmOTU4ZWUzMzRhNjY0MmZjMjVmM2Y5N2I4MzgwNTkwMmY2MDYxYzNiNmU4ZWYwZmJjYWQ2NGY5NTYyMzVhNGU0NTA5MDU0ZWQxYWM1MjI3MmZjZTE1MzRjZDdhZTYyNDc4YzQ0NTQxMDcwZDFkNDJjNjhkYzc1ODRlNzU0MTMwM2MzMGFkYjYzNzMwN2RjZmEyZTBjM2YxMTU5NjRhYmJkZWMzOWQ3NzFhYjc5YjYxYjY4MzI3NDRhMjA5MjQ3ZTBiODZiNjgxZjU3ODQ3YzgyOTBiZmJjYWUxZTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.mLkrCRsX8jLzEfYXA9XHV70yu7CKk7V3f_LCVVAMo5YhcQLCC6mq4UISuFky5-SkNEX4hjFIgv2HLRj0fduU9w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20201203_113140_69_227b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.065Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjFMNit4dklMZm91RHVSZXBJeHVKOHFsSmFNWTNNMEJpV2xTeVh5WFR0dGNVd3NNU3ZYWkpBZE5mL2RnTHo5ZmVKTDlVQmk1dVp3Nmk5MDhpQlZGQ2Z3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxOV8xMDMzMzVfMjVfMjQ1YV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjY1NjUxZTE1NjMxMzU4OTA5ZDY1NTc0Yjk0MDVlYzQ5OGNlYWQ4YmE5ODJkMTliYjZlMWI3ZGE2NmI3YzYwNTk3M2RiODQ4Y2IwY2FmZmQ4ODJkYzExNmJkMjhmZGNhNzUwN2MzN2IwYjFiNGZmZjQ4NmU5NzQyMGE3ZDEzZDI5MzkyNzZlMjQwMzI2YTlhNTlhN2M4MjA1NTM0MmMwZmU2ZWI3YmVjNzI2ZTYxMDcyZGRjM2JhNGQyOGRiZmQ4NjBiNzM5OTM3YWIwN2JlMTc4YWI3M2ExYzBiNzI3YWRlYzhiMjQ1M2NlODAwNDc4ZjU2MzBmN2M4NzJkYWM1YWU1YWEzZTEwZWU2YmMyZWYzNjhiOTdmNjM5MzMyOWRiYWYzZmM4ZGZmZmQ4MGJmYjU3YmI5OTY4Y2MyMWZjMmFmNDBkMjhlNWZmZTU3N2QxZjcxMjJiZjFiYWEzMjg3NjZmNjVlYTdiMDA1NmQ1NGQwNjkyNTJmMDllNWZlZjU1NWM2NmU2NTdiMmNlMTMzYmM3NDc2NzljOGZkOWIyYzUyNzZkNDgxOTZmMTZlMzJlNTNkYTA1YjRiZDNiZjI1MWFlNGI0OTEyZTM0YzU4YjI4ZGI0ODljZTZmOTBlMTVlZmJjMWJiNzAwNWNjMzM4ZDAyYWU3ZTAyYzg0OWRhYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UTzBZ2cuEzmr-xtZUpYXOgL7dZ0ss3yvmn6zZ1x0V4Y2mUaFVP-9TSE60gkfDg0wz3ORocHpHqHVmLOjCs1oRg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210319_103335_25_245a_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.068Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlNxNnd1UDczZm9aZG5QWWhmZU1henpEZVprT3lIOEIyR0xOMFk4RVdtMGFXQ3lITEcxck9WLzV6MDdLMHQ5NVlaemRsSTNiaEZiM2VkQlpWUW11eXhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxOV8xMDMzMzVfMjVfMjQ1YV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2E2OGI3MTdlNDY5MzViNWQwYTA0ZjllNTkxZjc1MmI5ZTRjY2FkNThlZDY0ZWEzNGUzNTExMTRkODE0YTAwMjFiMTMxMzQ5NDZiZjlmNWM2M2JjZWY5MGIyYjQ0OTQwZWU0ZjExYWNlNmE0MGY5OWM4NTNmYTNjOWMzNmExMTI5NzdiNjc1ODNkMmQzNDNmMWMxNTIyYjRlN2Y3NTQ2YmY5NTUzYTVlYzFhZjQ3MmRlMmUxMzA5YmRlNjA1NGFjNWNmN2FmMDA0Y2Y2NDRhZjg1YThiZDU4NmI4ZTEyMmQ2ZTI3ZTI1YzBmNGJlYWY0NWU1MjQzY2VhZWFhZDRkNDc4ZDgwMTQ4YjYxYThmZjFhMzc5M2E5ZDU5ZWY2ZTE2YjMzODM1NjQzMDZiNTE4MTcwZGE1NThlN2RhY2E3YjI3MjkwODY2OGM2ZGJkMTEyMjc3NmIxMzhjMjJiY2Y1MzdmN2FlMGNiNDQyNjRjOTFjOWE2M2I5MjYxZmNmMWE4OWNmZTljZWNlODcwN2E0NTNmY2QwMDBlNjI5NTE3MTBkZDdmY2Q1MWQ3MjEzMTYyMTUyODM0ZGQ2MGY0YjI0MTBjOWM3YTgwZDcxNDBhNDhiYTg2NjM2NTcwM2YzMTY1MTU5MWVmMGYwMGUxYmQ4YWYwOWY1OGI1NWMxNGEzMmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QIZ8TrCWvyq8C-TCe7qcqVyaLXM2K-FGYLV7L1MYBxXam4cOTgFM35i23EYqPKknaUjgVjoSHNQJctrbyEv5zA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210319_103335_25_245a_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.071Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InNDNVRSWVNTeEwxOW8yQ1NGVlZYeTBaQXNuZHpMT29qUVNqdE1haVZVRk5VVXEwYjFuZWVxUkNraWJtekV2Mm5Rb2NwcE1ESjJHREQrcnk1Y0hUNVlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxOV8xMDMzMzVfMjVfMjQ1YV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWMxMzU3Yjc4MTgyOWNkZmYwZGNjZTZhZmJmNzQ2NmZlYjYwOGUyOTI4MmQ4MTc0NWEzM2EzZDc2MTNmZjcwYzkzOGJlZjFlYzcyZmU3ZWEyMThmZmEyNzVmMmQyODJiOGQxYWZiMjEzZDc0Zjk5YTNkN2E5ZGU3ZTQ3ZDUwZmJjZDllMzllYWExZmQ4MzdhODEzMTM1ZTdiYzc3YTk4NTM1ODhhMjIzOGY3MzY2MzlkNmNlODhhYzhhMDI3YzBlZTYxMTZlZTc0MDdlYmQxMjNjOGExMGI5NDZjMzkyYzE4MjMwMDE4MjFiZGQ0NjczMzYxYmQ0YWY4OTk4Y2M0MzljMWE5YWRkMDMzNzI0MDA1MzUxMTE0ZGIwMjhjNWJhMzIzMTc0MTU1NmI2ZDFjYzU5M2Q3NTNkNGZmMWYxNTIzZDExZTE0ZjQwMGE4Zjc5OTY3MDFjZGE4NTQzNDRlODhiOTM5YzNmMWFjMThjZWE5NGY3MGU4ZTQ0ZmZmMGQ5YjVhM2Y1ZWJkZDZkMTA2YzE3YmQ5NzE4NzI1NmIyMDU2M2E2MDI4YTliNjdhNDA4ZDQ3ZDhjYzEwMjFlNzk2ODVlMDgxMWVkNTUxY2JkZWY0ZDdlZGEzYWQ3MzYyNTNmOGI2NzdlZmNjY2M0YmM0N2MxZmY2MmIwMjg2YzUxYjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wDFVkN-sqin0enqAhbUYPEPeH1vQHoGhKHuxkVzxc1v3ygqxLK317v9CVbGKztv2YorEAjgRIxtRgHnwBUa7CQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210319_103335_25_245a_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.074Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjZjZElpSmtEYmtHUG5sQ1NxRkJBdUFRNU5JNjU0dWNqWXpqam5JdkpKNFpQamxNLzFRZFVkdlF1Qm9RYUNGc0luU2FnY2RsdXpjY054WTAyNjFKN1FRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDMxOV8xMDMzMzVfMjVfMjQ1YV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTkxZjE5MWU0N2JhZTAxMDQ3ZjJjZDIzNTVmY2ZiZGY4OTljNmY2N2ExNzkwMWEzMGQ0MTEzNGE5MmY1NWI3N2Y2NTQwNDE0NDI0YTU4ZTFmYmIxNmI5N2JkMWRiYzY5OGNkY2MxMzg2MmY1NTU5ZDk2ZDQxMjVlZjc4ZDdlYzgyMjE2ZmMzOGJlY2Y4NjZiOWUzOGFkMTg0MTRhMDAwOTU4NTQ0MjI2MTAyZDI2NGFmNmY1NmM3ZTEzYWIwNzQyNDBmNTliMTdhNzk5MTc5YjA1ODhlNjMzYmEwMjhiZTQ0ZWFlOWNiYWEyZTM5NjQzNTcyNDFjMDYzZDk3NjJlNThhNDVjZGVmMjI1ODY2MTcwZjNlN2M2Yjk1ZGI4ZjRmM2Q2NjZkMzYzNWFlMTY1MDExNmNkNmFhZTVjYjUxYjIxN2NkMzA5NjQwMjBlMWQ0ZDA1OWUxZDg5MDdhNzIwNWQ2M2JiYTUyZmQ0OGQyZjU2YWVhMzg3YmE2MjllMjU0NjQ1NGMzOTYwOWViZThlOWFhODE4MjcxODAxNzBiM2I1NTg5ODVjZWNjMGY3NmU0OGQ0NjVmOGNiY2YxN2ZjOGVjY2VlOWFkZjc3ODRlNDNiOWE2ZjRiODg5M2RkOGEzMjI1Y2IyOTMyY2MxOWM1Yzk5YmMzOTY1ZjU2N2QzYWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.e5E8nJdN9rEx0iPUfrfqC3ikJ2f5NegiDajPpx4SKzT0QenHiXzwAjKOqpV4PYv5JI-XCuUu2qyjWeGkAXohmg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210319_103335_25_245a_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.077Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZVdm8rOG4rMDdLODBNSWdRQ2J2TVRieThkWS9VdkxJaG9zZlZ4VHl4YXhDOG1EZWw4WE9qR0tHM2tvUnIvY00vbXVHVWt5c2M0cklHSG40SU5Qc3hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxM18xMDI1MzRfMDVfMjQxZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGU4YTdmNDQwZjNjMjc1NjcxMjlkMmQ1YTU5ZmM0ZDY0NDZiOGFiMTJhZmVlZDE5N2RjOGZjN2ExNzQwZjFjZDI0M2M0YTAwMjZlOTM4MjVkYTFlOWNiZWEyZTZjMTJmZTZjMTNkY2NjYjJiMzY5NTBlMjQ3N2ZiMTJmOWQzN2VmNDcwNzI3MjE5NzMyYTA2NTgxNjQzMDBmYjA4MTYyNjYyYmJhNWFkMzMyNmQ5ZjAwMzM0ZTdiYWU3MGJlMzU4YzU1NjM5ZGYwMjhjMDBlNTU2YmI3ZmVhYzBiOGNkZjQ5ZGQ3ODNkM2M4YzIwZmRmZjgxMDI5YzFiZGVkNGRlMGUyODVhMGI5MzhmYjdiMDEzZDJmYjE4ZjFhNTdjZWFhOWQ0YmViZDMxOGFjNWQwNWI0YjhlOTRjNDE2OTRlYzZjZGVkODJkZTk0MTBhNzg0ZTk1ZmJjMTZkNTcwMzZjYzZkN2UxYWVkOGRiZWRlYjhiYTU3YjBkOThiMDlhMTMxMzk1NTdkOWU0ZGUyYWFlM2NjOTk4ODFjM2I0YWNkMzFlYmVlMTJjMGUxMDI4YzZjN2RiN2NjZjY5ZDVkMzIxOTA3OTQwYmVjMTdkZWE0MzczNjg1ZjcwNTFiYzgyYTdhMTU1NzExNmVmNjM0MjBhYThkMDk1Yzg3NDlhYjdhMjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LX-cF06QL8pKcRpS_hQNU0plAHUh5wJjaCqFAeSch9-xMY1ukttNxIZdI2_UZAqwOVKV66OJEIBLbNstKf_c7Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220913_102534_05_241d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.080Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjRjV3owOE1hMkRDU0lkb0xFamUvQVl0ckxZS09BZWNBbThkOUdwSE4wVTNWOTcvQnZmR3hkT0NycTZiL005R0hSU3lZRjRUQStpeVJ1eUV4c0JsRjRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxM18xMDI1MzRfMDVfMjQxZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDZiNzM4ZTM5NzFkZTA3NzM2YmQ5YmYxZjcyOWFjZTk0MGZkOGY0OGI3NDcwMjIzYWM1MWIyNDIyNzRjZTUwZWFjMjcxNjlhMThlMDQ3Yjg4NjVjMWFkMDMzYzM1MTJmMTk0MGIxYzRjMmVmNjkyN2EwNjZiYjQ0NmJmNWZhYTUyYTc0ODZmMDFlYTJhMzYyOGFlNDA3MTY4MjgxOTg3ZGRjYjQ3ZmJiYzk3NzJlODI0NTBlMDYwZjdjYzdiNjNlMTg2NDQyZjViYmMxNDFmM2UzYmQyNDczZmEyYzZmZDZjNmM1Y2YwMDUyYzNjYjM5NDMwYWM3MmI1YjExYWFmZWEzNTJlOGY2YmQzM2Q4ZDNmODFmYTBiZTk1YzQ4N2Y2Y2JhODZkNmJlYTM0YjQ4OTcyOGI4NDkxZWFlNmRmM2Y0NzUxYzQxM2I4OTQ3NTVkODQ2OWU4MjdiOTA1MDhlYTc0NTczZDFkOGJhMzE0NTM5NjczNzkwNDY5YWIxMWMyODU0MjE4Mjg0MTUxMGI2ODliM2RiYWVjZTViZGI0ODJhMWY0NzFkY2MwZWVmNDgyYzRmZmM4OGI3NGE5YTVkOTAzYTZjM2YxMjIwNTNkMGIzYTZkNjQ1MzE5MGJjN2QyYThkMjEwMDJhZDZkN2FiOWZkN2M4ZjBlNDJjODFmNmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zXH9WrSAckF1-yrwgCP4_cKAi0Q_3T76QEj4llcM02-9EydcwGS0OvCXY3Ik1rY6NdMsan58oiCQPKAH7vCxHg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220913_102534_05_241d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.083Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImJKWm1BWUQ1cVJJa0hUb0xYOUxrTXNLUFdIUkxZSmN4dVMyK1BHbkh5aVVCeS95T2JCY3FaR05XeDRDZjh0Ykkya3I0Qnl2cTFTNjZWUHFDY0ZLb1NRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxM18xMDI1MzRfMDVfMjQxZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWIyY2E2NGFlYzUxZjg0Yzg5ZmJmYTNlYzBkM2QzOWI2YjJkMzgyOGRhYjQ2OGQxNjhjY2E0YmI4ZTdlMDY5NTRmMzRjNjhjZDUzZDYwMjllZWU5ZjQxZGI4MWQ1Y2U1ZTcxMmRlNDBjZGEwZGUyOTgzZDVjODI0OTYxODU3ZjE3MTMzNTcwM2EzZTI1NWIwMzc2NjYwY2NkMGM3N2M4ZTgyZmRhZjQ4MDU1MjYyM2FkMzYzMTdiNDM4MzIyZDQzNjNlODVjY2E3ZTdhNzdlMjc5NTU5N2JhODVmNGQ1MGRlOTUxZGM1YjI4YWFlMjE4YmJmOWJkNjk0YTQ1ZGJkMjM4MWVhYWYxNzlhMmQxZWEzZTU0OTJkNmYwZDM4YWI3ZjdmYTdjOWVmZGU3NmE5ODBlYzlmMGZiMWFiZmYxZTAzNjIwOTA0NDVmNzYxZTY0NTY5YjI0Y2RlNzE5YTFkMmFkMGE5YjlmNWUzOGRjYzBlY2E4MjQ3NzhjNTE1MTY1ZGQ0NzY5ODVmZjU4MGEzOGNiZTQ0MTM3MmM4YzM0YzkzODE4YzJlYTM1MTIxOWJlNDZmY2ViZmNmYzc0ODQwM2E3NzBmYzE0YTZhN2E5ZTNkYzE1ZjI2MmExMjA3MzdjZWQ0NjBmNGYxMmE5YTc3NDYwYTg1ZDlhZjE2ZjUxNWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bdvqmZv02qD0LAAtpJOLb0vmPI2dtB9QfL8L8lZPUDvRYEsszDzrWmlSQUJzY3D-7YecDlOwcDax1alBudnLIA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220913_102534_05_241d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.086Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InlrVUJabUpFaEJpVHJEVDJOV3MvZFpmNGpTcGpEZnViTWpITStIQU5yMDZOWFg1K2ovVmhQRGU5RVJzYjhvLzRpU1lPTkEzOFAyanlmTCszMGdES2lnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxM18xMDI1MzRfMDVfMjQxZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzVkMjZjYTEzYzE2Mjg1ZDFmOGZhNmI2MDM1YTU1YTk1ZDAzYzg4ZjhhMzQ1OGJiYjA4N2I4YjA5ZjNmNWVjOTlmOTQ4OWI0OGJlYjdkNTI0NjQ0YzU2N2UyMWMzYTM1NDJkZTVlMDQzNGI4MDJjNTAzZDhkZjE1MThjMjZhZWMxZjIzMTI0M2I4ODk5ZjI0MjViMjUzZDM5OTdmZjMwOTgyMjBlZjZiMDA2YzE3NzJjMjhkYmJhMzJjNTMyMDk1Y2RiNzliMmQwZjkyOWMxMDg2YmYxOTQ2NTFlMjMwMzM5NTExZmM2ZmQ1ZDRlZWE5NGNkNmUwYTg3NzdlNTRmOTAyMmFjMDNiNTgwNDUzZWUyYzUxNjE3MTE2ZDVmZDljNTViNjAyOTgzOGYwOTMyNGE5ODI4ZDBmOWVkNGZjMmYzZTkyYzQzOTQ0NzBkZmFiNDc0ZmRlOTFlMzQ1MjVkYzIxYWQ3MTU5ODVjYWVkNDJlZTczZjllYTNiM2NhZTkxY2ZkOTQzMDRjY2Q5YWM1N2JiOWQ0ZDQxOTkzZDllOTgwMTNmNWU0NjZkNDQ0NjIzYzE0ZjgyM2FkODBkMjhmMzM3NDk5Y2IwN2ZmYjhhMzI1NTc5ZWUxODA0ZjM1ZTdiNTUxNzM1OTM4YWIzMTgzNWVmOTk2NmRhODc1NmFjZWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.u4sSaA-6_bMyZ5j4ggxlWqYXC5si7yG42syS5LMPeJVX6M-iU9zIEacsOu0QG1FSDW-CB9JKbxhc7gzzFroOUw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220913_102534_05_241d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.089Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImdiTGdrMjhBaklKWnpHa1hvTEo4bXdNRFRjYlFEcnA0bmFRRVJyeFpSNFZVZDdSS3BlTFYyQ0dMbVc3aSt4engzd3h1aFJ3TGZsSDNSb2UrZGtwY29RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM4NDFfMTNfMjRhMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzJlYjBjOWFkZjE0NTIwYWEzY2RlOGNjNWI5OTVhNTk3Nzk4OTVmMzE3YmRlMzk1OTI2YjEwNTBhNjU2MTI4MjVjZDU5NmJmYWJmMTZhNDRmYjkwMmFkZTJkNzE2MGRlYzRhZTEzNzc3OTVmNzYxMGVhMDc3YTEzZWRhY2Y4OTBiMjEyZmQ5NmYyM2Y5ZjY4ZjkyMWU4OTNjZGFiODZkNWNmYWQ1OTVlZTYyZDA0MDhjNGJmNjBhZWZjZGJjNWM4MmZlNDEwMzk0MGViNzY1M2I1ZWM4YWYyYmYzY2Q1NWEwZTNlOTQ1MDIzMWFhYmRhOTcwYmYxNWFjMWYwNmZjM2VkZjgzNzNiOGExZDVjYzI4MDUzZTU2NjQ4NzczMzhmYjUzODFkM2RjMDNjNzRmOWM1ZTY0M2NhODQ0ZjE0OGZlNDU4MDM5YmQ3NDlhZmYzNTdjZWZmYzAyZWUxNTQ5YTcwOWFkMjQ5OWZlNDJiNzRjMDIyY2ZjOGRkZDlmOWU1ZWUwNjY3MTMxMjVhYTFmZjM5MjBiZGIyOTIzOTJlMGQ5ZmZhYmZmMTFjN2FlYTQ3Njk3ODkwOGNlY2JiY2NjYWFkMDk1N2RkYTY0NTFmYTc2NTg0NGNjNzUxY2EyYTIwMzc5OTA2ZjVlYWQ4ZWUyOWUyMzFlY2VkYTJkZjZlN2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qlCjYib_5sSqk84sbc45F4VSCTBa5_OFidQpS2AnMMdnUtsRtbR2qJ_nc-NR44qzvOmAVOW8TsS-wWkVCaj5Jw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103841_13_24a1_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.094Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik1uYWZxWUIzZlgvZXRWVi9NMnNrc0p6a1NjcFExblhSWUNKQzlPSlJlVUlrdHJmMytNamV3TGxMcGdtZHFWTmhXNUROc0J2UklJa2V5YTh1NzcwYmVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM4NDFfMTNfMjRhMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODY0ZTY1MzA2NzA1Y2E2OWE0YTYyZTkzZTk4MmJkYWVkNDg5OWU4NWY3NTM4Yzk1YTAyZDRhY2FhOTY5ZDk0NTdjYTRmMWExMGI4NjVjMDUwYTY3NzU2ODFjMjU2ZGNjNjJiMTg2MDViOTk1OTEwZGM0YjZhNzhlOWZmYTM1OGRiNTFhMjA5NDgyMjcwN2U3YjhlYzAzNzc0NDQ0MjJkZmQ4ZjNhOWU2ZWI2NzQzMTAzNmMxNzdlNGUwOWU2NGRhMmUwZDZhZDVmZDJhMGYwNjkwZjEzZGU1MmY1MDdiNDQ5ZTM1MTMxYzkwMDVhN2UwNzNjYTcyZDg3OTAyMjNiZjc2MjQzMDVmNzFmMGJmNWFjM2Q1YTZhYmM0NjdjMWQwZWQ1Y2UyNTYzMjQ0MTQ3MDIxODM5MTdlM2JlY2M5NzQxOWYyNDJiM2U0ZDhhZWQwODdmMmZhMTdhMjllNDRjZDFiOWVjN2QzMjM4NzBhN2M2NDQ2NmJkMjU4OWNiYTNlMzY1YzhiYWZjYjhlNDc3NDhjM2RlZmNhZDNhM2Q2OThiNDkwMjg0MTY2OTgxM2YwZjdkN2Y1NTdiZjg3OGFlOTAyZDQ3ODFjY2E0MGUwODk4Nzc1MjczY2NiOGI5NTBiNzg1ZjY3ZTJlNjZmM2FmNjhjNWY0MmY5OWQwMjQ4YTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.W6PSgbQrD2O0ODg-7zHS5MBO6qrEPnQq8I3I0YFmm491IOLweC1dq_-IKm1LzoIongKa7iszwdjYCfDfgj_W_Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103841_13_24a1_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.101Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImpjdTRTVTVnVktqYUxQc2Jxb1BnL1lPVHQ5NVRwUGl6emJNSlUyR1RzRFV5c1hzVlNMMTVRRWdBbFBCL2t0K2hPRVdZcHk3ME15cEF2NGdNeGNJYkZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM4NDFfMTNfMjRhMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmMxNTg3MGZkY2JkMWE4ODE3NWVhNmI2NDU5YTc5MjAzNzMzMDQ5NjY1MjhiNjc0YjU5ZjAxNWY4ZGUwMjQzNmUzN2ZlZmM0N2IyNmFiZTU5YWVmYmRjYTIxYzc2ZTFiZmExYWEwOGVjZGE3NDZkOTk5NTE1MzY5MjY4MzMxOTIxMjcxZTg1MGY4MTQ5N2U5Mjk0YjkyMWY5NjYwZGRlOTA0MzdlYjEzZTQwMzlhMzJhNmRjNDkyYjEzOTFhYzViZjNiYmFmMmVhMDhkZTE3ODFlOGRkZWU1MDAzNmM0Y2UyMGVlODJjYjI2MzNmNmQxYTcwYTQ1YTQyYWI2ZTZkYWNiZGZmYjZlNjUyNjg5MzhjNDNlODc3YmQ5MzlmODlkODQxYWI4ZmRkYzFmNmY0NmZjNzkzNzY5ODRjZjBlYzE4MWQzYzNhNDM0ZTAwYzZkMDQ2NDY0NTUzNjI2M2VhNWM0ZTBmYzVjZDRhNDhhOTM2NDk5NDEwMzZjMTQ1ZDhiMTc2NWI2NzM4ZjUwZTdkNTM2NDRjMTVlZGI4N2MyNDU2MjczODk2OGUyMDlmNjRiMzExNWZlZjg2OTZjODVmZTQ4MTlmNmYwYmUxNzc2YjQ0MjE1MzdkMDMxYzMxMGI0Yjc5MDFjNzU5OTA5YzQyN2JjMzhhZjNhYjlmMTY3OWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cpL1vLE1a2Z9E4uGUTJdgvxbHXhfNQHjFw-Z5EQtMUBebE7RCCG7ajjxn_2w9yuwXqnH2REpMjJsh0J1PX0sSw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103841_13_24a1_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.105Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjBUZWk4bVo0TVJjSmR5b3RxLzlxZEdNdzNNaWF0K2Z2YThHR2Zndit5WHB0M1JkZ1REQ05wNVpPYjZsT1lKbUd2MmZQcFBxVXN2YmNJWWkxb29VdmZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyMl8xMDM4NDFfMTNfMjRhMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDg5NmExMTdlYjc4NTY4MDcxOTU2YjEzMDkzOWM2MjExMTU2NGYwNGE4NTQ4YzdmYTdhNjM5YWIzMWEwNWZjMDNhNTJhNjc0YTJmYWFlNWZlNjE4N2NmMTAxOWY1ZWFmMmVkODVjNDFhZDFhOWU5MzA1Nzk5OGU1NTZkOTJlNDMwZTJiOGRmMzVjNzc3M2U0ZjUwODFmMjE2NmFhN2I4YTgxMjlhYTk3MDk1YWE3NjdjOTQ0NDE1YTIxYTYxOGFiZWFiMGY3ZTNiYTMwNjFlYzMwZGE0Nzk4MTg5ZDU1YTAzNWYwMmM1ZDdlYzJkYmZiMTBkMmQyODczNDE3ZWExOTQyZjIxZmZhNThlMzBiMzlmYWM5MjA1YjcxOGMyYTJkY2I3M2QwNDdiYTI1ZGI1M2M5YTk3NDgwYzk5N2UxNzQ3NjNmMDU2NzM3YzFhOGVjMTlmMDBmYzg4NzFjMmYwYzhmM2ZmMTY2MTg0YTM5Y2EzMWRjYWJiMjc3ZTQ2MzNkNzY4NDgxNzE5YTU1ZTIzYzIzMzQ4NDk5MWE0NjhmMjZlNjUyZjk1YTMxMDc0NjcwYjEwYzM2YTA3YmFlNTUyODk2MDU4N2JlZTg1OWU1ODY1MjQ2N2RmM2UwY2Y2ZTYzMDIyZDhiZGE0MmJmMjdkMmI5MDgwNGY1ZDljNzhiMTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qIbber8o8ZSVrKkhK5-oZrFShaOEik2awEPLq55tfp5tjJUIxVr4Qo3c2iv-9NL2Q8B9Mc8aqPUOZLFJWOAezQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240222_103841_13_24a1_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.108Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ijhocms0d3g3RGlzMTh0dWgyQVBlQWdPZzlzaFB3OWdpdENmVkpLeGJrTFlxTDdjdFVaek9DNDlUOU1jU293bmd1Wm1rb1BPOWZoLzFBZFYzUGJNK3B3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjRfMjZfMjI3Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2VhMzA0OTJhOTk0MTgxNjk1Njg0M2Y4MTdhZTkxOTZiN2Q0MGVjOTZmNTRiYzhlOWM0Y2FjYjQ4YzY2MDVlZTk2Y2EyYTJlZDNiYjU1ZTI5YmY4YjkzZGZjZTE0MWUxZTk1NzQxMjIzYWMzNmI3ZWI5MjUyNzBiZGQzODQ4NjM4ZDg5MGMzNWE2OTBhMDY0YTcyNGJhMWFjYWNlOTY1MjBhNmFlYzQ2NjRlZTFjYWQxNzgwZjc5OTQzOGI1NmYwY2ZlNTNlMGU4YmQ4OTRiNjM5MWIxMmQ2ZWI1NTM2ODQ5YjEwM2ZlYzBhZDUxNjQyY2VmN2FmNTRkMTBmMTQ4ZTg5NmU4M2RjNTc0NjhiZWQxOTkyMTA3NDNiNmE0ODgxYzYzMmMyMmU0YmJjZDBlYTgwNWVkYWM0NTcyNTNmNzkwNDZkYmQ0ZjY0Y2IyZDNhMTJiOTMwMjAxOWNhOGFmYjlkZjcwNzdmYzAwMjc2OThjM2IxYTYyMTNiNmZjYzExNzAyNGY3Y2EyMDVjNTU5MWQ3ODExMzRiMDk3MDZjYTJmMGYyNjdlODg3MGNmMjFjNDAzYTQ3NjYyZWY4ODQ5MDBiZWY0MWU1ODljMDI1NGZlZTcxMGQzMGEzZTE4Y2JmYWU0YzNmNDJjZWFjN2E0MmMwOGI4ZjBlZjkxMDMyYTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7PiRoI6oyU2LwvlKBJaE-AebKJ1_1VKdWIXukEb4Q-AncNtar7xbDyEN_hpv-ylYgL5kQWNc5xMWw8gT_OVufw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111224_26_227b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.110Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBJSmVNY0Mxb3ZOTDdGaWxOd3VlbGxXM3l2REltaGhTbWRlM0RiMEJoTWJhNWNGWlJOQVl4TWN6WnhFMjdIcUczbWNTLzZZcXI1Rmh3MjdsMzg1cUtnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjRfMjZfMjI3Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGIwMDUwZTU1NjdjMmY1NGUzYTVlZjc2ODRjNjUwNDVmMTMyZmNhYTIxZmExZDg5MzZkNGZjYTc0ODNkMDczMzE0Zjc5NjU1NDIxNjhjZmRmNzBlYjE4YjA4ZjQ3NDY3YTU3MmZiNTgyZjQ2OGZmZTIzMzVhZTkyNzAwNjA1MzY0YThmYmJhOGRhMGVlYTM0ZGI5NTMxMGZhYTBkZDZkOGY2MDU1MGZhMjJkZTQ4YTUyODY4YTdhYmE1ZWQxYWVkY2E5MmNjY2ZmZTk2MDFkNDk4MDkwMmIwOTZkZmY3NzVlODZiZTQzM2FiNDcwZDEyZTgzOGU0MTNmMjliNzZhMTJmN2Q5ZWM1YjA0ODI4ZjEyMjVhYmZmOGM1ZTNiZjk1MGY5ZjY4ZjM2YWRjMzQ1MjIwOGZkZGY4MzUzNGI0MGMzY2Y0Mzg3ODRmMDdlZWI5MGNhODhiMmU3Y2MzNmNkM2RmMWRkY2U0MmU2ZjBhYTYwMWRkOWQ1MGZjNmNhMjY0MTAzMzllNWY0N2IzYTU4YmM5ZTU5MDUxNGNlZjI0MDk4ZGZlYmVjNzJlMDhkNzM4NGRlMGQ0MWM3Y2M4MGMyNDZjMTJhMWJkNGJkMzM5NWRjOThlMDYxM2VlZDBjMjcxMjVjYjJmNTQ0M2U1M2JlNWMyMDMyMGE0MTE4YjdjNWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8eGQR6-JnfqcKcSuffJfQOC-PBAQLbjaf4ggtcC7El3fDsvsXig0vIWtBA4hHNlqB-hPL3zQ3EWmBeqaw9QxYA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111224_26_227b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.113Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxpWGJzU3c4TkFsN3VZaVI2Uko4NmVtUzVQZ2dFajRsSDJKd1U5cWJyU1BQVTJvaTZPdmsyRTJnUFBacFVaY2E3MjR2OXg1Q21IbytFTVJKdjJMaHR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjRfMjZfMjI3Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDc0YjJiZWE5NjFlY2U0YmIzZWJiOGEzNTA1OGE5N2QxMjg1Y2VhOThjM2Y4NTZiZGVjZTBiMjBiODkwM2FjY2I4OWFjNmM4M2M0NGIxNjM1NjNhZGI2MTc0NTY0NDg3MjdjNDgwZDdlMTA2ODIzZjI5YTQ4MmU4NjE3NWQyNzIzMjI4MjkyOWE1MDMyMTA2YTFlZGU1MzEyYmFhYjU2NzM5NjYxMWJhMDg0ODg4ZTE3YTI5NjcxMTBmZTUyZjM5N2NmMTZlMGZmZDYxNzAwM2NmODZmOWZkYzZjNzM3NDJlOTYwZDMzYzc4NmUxNWI2ZTBhMmI5Mzk2ZDljYjM5NzNjMzQ0MWY2NDk4MDk1MGM5ZDFkZTc0MmQ2NjY0Y2FlMTEzYzc1NWFiYzBmMTRkY2E5NzEwOGI2NDE1MDU2ZTFlODA2NmExNjE0YzhlODJiZGQwYjE1Njc4NjU3NTBmZjFkMGZmYmE0MTZlMDI1NWNkNzAyMzA1ZDg5YmEzYzA1NGY4YmM4ODI3ODRiYWY2ZTM5YTZlMDA5Mzc5MjI2ZGZiMjI3OWRjOWY5MzA3ZjY5OTRiNGUwYzE4OGIwMjhmZTdkN2Y0M2FjZmU3YzdjZjk2YWViZTA3ZjY4NjM3OGFkYTlhYmM0OGE4NTVlYTE0NjhlMzA0OWFmMmY0MmEwNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.veqYlAfhhOVs6Paqw1U7hv2b2lQ9VfgxLd8GnlutjBWOiRpUDZdyInzdBIzCkLbY5tAT1G9-oZvkVhoBbY2PoA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111224_26_227b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.117Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBxN2lxY0ZjbEl1TDV1QmFIUERhTjU0MVFUK3FlempCTi85cjhCNkxaa2JNQTN6MFBMR1F6K0l2cTYrVjljSXgvY0VuN2lIdkFpV3hZVUl4VjRrOVdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTIwNl8xMTEyMjRfMjZfMjI3Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWI2MDQ1NDIyZWUwNDFlYjg1MzIzYzBjZjI2MWFmM2U2N2IzYWY4MGNkMGNlNmFlYmM5MzIxY2U1NzU2ODUxMGI2ZDg0YTcxMWRmMGY3NjE5YTE1ZWVjMzJiY2Q4MTVmMTcwNzIzZGJkY2JkZWIyY2Y0Nzk0NjU1NGEyZDQzOWE1MTZmMTMyOWNjMWYyMjJjNzFmMGZkNGU5YjI3MDVhODE4MjRmZjdlZDJhYzA3NzM3MzJkM2RkZjlkNTlmMGZhNTEwNmNkMTE5OWNhZDIxMTMzODBhYmRlMzg3N2ZlYWU4NTY3YTYzYWE3MzE0Njk2YWQxNmJjYmJlOTQ4YmE4NmUyMGZhNDEwY2ZlMjE1ZGE4YWQzNmRjZWQzY2Q2M2UzMWM4MjNkMTViZjNjNmEwMzI0NzEwYmViYjRmY2ExZWY4YzdlNGIzODVmZDQxMjE5ZjU4OTJhZmJhNTE0MDY2OTJmMWRkOWFiYzg0MWQ2ZWUyNTZmN2JjMzU0YWRmMjI3ZTRmZGZjMDExZjk1NGFhNzRmMDk0OGYzN2QyNjAxODFkODQ5NzlhODU0YjIxYzIyNmZiZWExYTU1YWRhNTAwZDA0NWJhODI2ZmQxN2E3NTAwODk3MWU2NTg3ZWQ1NjhiZTM0YzBmYjU1YjI3ZGRjNGZmYzg5ODU4NGI5ZTAyMjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cVQFzKL-uW-dK9dWF060JH_SvBORGvfCdEnVlx72aWb5XjuMwYttx2WByIjnywYvPcIuhCLNgxcJtjLvQ2mBAw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111224_26_227b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.122Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkJhcnpoSVd1QTRWWGRGYStUTVZZWkhjWDdIb2F0MG05Zlg4djk0dEZHejhsWFUzQ1ZPcng4TEtQZGhEandrRjNmUU5iazMxdFNOSThOcXFDSXYwd3BRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyMl8xMTAyMTRfMTdfMjQzOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTk3YzFmYzM1OWY1MTAwZjE4NjQyYjczM2Y5NWNlNTVkNjAyZTkxZDRhZTJiNDg3YjM2ZGNkY2E1OWRkMWRjZTU4MGMwMjU3MzFhOWNlNzQ1YmU1MDM5ODFhNmQyNDdjODQ3ODJkZWZiMjFjN2FjOTI5MjhlN2E4ZTBjZmVhMDQ0MDI2NGI5YWNhNzljOWRjY2E2NmI1OTc5ZTczMDM3NmQ2NjI4MmI5ZGZhNzQwNmE5ZDVmMmI1N2Q2NGJlYzgzZGIwNDRlZjgyMjk2NWZjMzcwOTZhNDU2ZjJkYmI1NGJhOTAzYTIxYzFlMTQzNmM4ZWYxNmI2MmE3ZjBlZjczZTQ1MGRkNGVkZWIyNDdmODk5ZTkxOGRmMmNlMzVmNDdlN2MzOTk4YmY0MTNhNjhmNzdjZjc4ZTg5YjMxY2NhNTU2YjE2NGEzZjBjODk3NWZjYTg5NDQ5ZTUxMDk4NWI5YjRmMmY1MmQ1Y2YxYjRjOWUyYTJkMzNlMTkxMGE5YzdiYTA4MWIyOGYzZDZhMzkwMjI3ZTU4NzUxYWE4OGQ2YzhmMmNjNWFiYWE0MDgzY2Q3N2I0YTkyMjZiYWZlNTA4NzNkMzQxNDIwMDc0YjcxOTA5OTUwYjUwZDY2NjBiYjE1MTY3MmFmMGM4MWU0MDE2NGQ0Y2I4MTdhOTY5YjEwZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.6_z4QfXHA4Eadg15wFKEZdEJN3f9C9TaM6glSfvzdkWt5A-gx1qKFrS2NQVANcPRBNx2OvzuAYQ2fHmTwciHiQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230322_110214_17_2438_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.126Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjFGUnFzeWJnUDlaa0U4Q2RETUpLT0ZwOVg4UzdXSzk4SnNJZ3NIMkxlOFdKVnJSamR3SGI0cmxIeS9ma2dzU1hJUEMxbXVQaHVnVVByWFd5akhvdC9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyMl8xMTAyMTRfMTdfMjQzOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmUyZTExMmM1OTRhMDMxNTk1YmRkNTY3MjMxOWE3ZGYyZmJlZDU4NDZjZjc2YmY2ODFjMmViZGQ0NWE4MGFiODQzNTA5ZjkyMmRlMzM3NjkzMDkxOWQwNmVlY2Y3M2EzNzg2NDFmZmRlODg1YzBiMWFhNzdiMTRkNTYxNWU4NThjNmM5ZWM2ZWI5NDZhY2YzOGM0MTgzYzFkMjliZjY1OTkzOTY4NzFhMjAzZmEzNjY4NGQ4MjQzYjRiZmIwYTExZDYyMTAxYzU2ZmUyOTdjZTliNjg0MmFkMTE5MWE3MjdmYmZhMmVjOTQwYWY2ZWVhZjI5OGFlOTZkZjAxZTBhYzliYWZlMjAyOTlhMmEyZjVhOWRlMTZjYzE2NjNiODg1NDc4MjFlM2JjYjZhNDA2ZjAxN2ZjYTRlMzA5YjliMDFlYTIyOTA4Zjc1OTZlZjlkYzJjZDE4ODY3MWVjOTQ2NTM5OWE3YmQ3MDZmMmFiODZlNWUwMzkzZWJlZTg4YWM4YmRlMWQwZWQzZDZmOTRlOGNkOWYxNzFmYmVlNzZhOTVlNmMyYTAwNzYyODk1NDI4MGNkZjdiOTlkNDVlNTY1ZDdkY2IwODkzOTZmZTkyMDFkM2UyMzNkMzNmN2VmZWUzODQzZTA1ZDUxOTdiN2IxMTQyZGFkMzdkYjk4N2M2OTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3SBzepck7f0ExAuS60zCKAz3tSthTxoaJ-c7QJWd4ob2D_44HiJDBL16ejPTGTHMWNikumPQNDyKTx8CJCl3XA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230322_110214_17_2438_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.128Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjVFOElnVlFPYmRPUklsRkVYNnhUcTU5bGVXbGRVcitibTIra2w5OTlET1JvN3B0V0oxR1dVdnFldXl5eTZqYVNSb3FqYzJTVEJWalREOEoydzZ0TUdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyMl8xMTAyMTRfMTdfMjQzOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM3NzBjZjNjYWQ4Y2JmYTU3OTk5YzNmYmRmNzFjNDcyZjAxMjEzODhlNzA1ZDU1YTA5ZWMyNjVjOGE0YWI1MmE5NzM4YWMzNWJjNjQ1N2Y4YTdjNDE1YTEzOTUyMmU1MTk3MjU5OWViOWZlNTI5YzNjZDhkZmUzNTllZTlhMTVhZDUzN2QyNDZkOWMwMzYxZjRiNzY4ZWZiZTY4NWY4NzEwOTQ4N2ZlMzU0ZWQ0MTc1ZTdiNjFlZTM2YmQ5YzQ4ZGNjMGQ5YTgwODYyYzYwM2Y2NzEwZDdlMTI0ZmY3ZDM5ZTBhY2FlYTk5ZjBiYzY2YjdjNDVlNzIzNGQ3NDVkOGE3ODkxMzk1OGI4OWRhMzQzOWU4ZmE3ZjY5NWE4ZGQzOTg2NDIzZWY2OWQ3MmVhYzZkNjg5ZGY3MzBmZWRhNDJkYTI3NWIwOGIzZjQ2OTQ5NjRiZGU2ZDljODk0MTczMzYwZjE3ZjM1MjhiNmJlZjc5ODY4MDM0OTc3ZGFhMjA5OTQxNGYwMDQzZjE0NGNlN2I4ODNkMmQwMTQ5ODIzMjk3NzcwYTMyNzYwMTY3OTk1NmUzYWYxNGY5Y2E2MzI2N2FiYmI4YzRlNDllNjNjMTI0ZmQ5OGZmYWI1ODJjN2Y2YzRhNzdmZmYwZDMyNDhiYTBiZDk0ZTAwZDYyNjQ5M2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.p2tJJBQcXXUSyY6HCglOd7cNWc_f1L-WVu_wcn4taaW5QyBoyerN9bke_z4yFHeXtw2diqjklcoEMSUuwAOSJg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230322_110214_17_2438_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.131Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlIxWWQ2Y2llRzdyOEd0aE95ZHkvNmhkanY1Z0V5K2NDM2xyWTF0UGEraERlbk4zNk1JemF6YlBiV2lZV1VTckh1bWpFQisrMzI3ME9uZVUvMW1LZThBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMyMl8xMTAyMTRfMTdfMjQzOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzM0MmE0ODcxMGVlZTkxZDA4MmViN2Y0NjI0YjRlZjJmZTgyZGZlNjA1YzViNDZjMGRiM2I1NzMwYTAyMDg5ZTA3NTEwNmMyOWE5N2Q3MDY2YjA2N2MxMWZhYzg4ZGE4YjFmMDViMTMzYWI3NjRkZWE4NzgzMTc3NjExYTRlMzQxNGE5N2NmMDY5ZDZkZTZhNGU4MTQzZDMyZTY4ZTE2MjUyY2I4ZTdkYTEyODNkYTIyMzEwOWFiYTllZDM2NmUxNzMyOGExNjIwODE3NGNjZGJhNjYwYmMwNTA2MDEwNGY3MmYyNGM0MDY4ODRlYTIxZjYzOWU5M2EzN2IyNGY0MmU1M2ZjNTAzMWEwOWM3YzY3NmMyNzZlZGRmMGUyOWZkYzgwMjZmYzUxOTMzZThlMjBhMjUyNzEyMTg5NDA4YTE1NTdlNGZhNWI3NzMzYjQ4MDljZjE4NDg3NjM5NDRmNmRkZjg0NGMxZmJiMTdjYTkyNGQ3ZGU4N2QwMTQwNmJmMzRmOWQ1NDZiNWI1ZTVkN2JhNzQ4OTI2MDk2ZDlkYjViOTI2Zjc1MGIyNDg3MjdmYjc4YWVhOGY4MGYwZGRkZjIwOTQ4YzVkNWU1MzNjNDFiMzFlN2JmNTNmMjkyYzFjMWE4OTcyMjM5MjVjZTJhYWQzMDlkYzRhYTY3YTg5M2RcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XgWUKWIdmUXvvOAJmYYjQEBlkg4d5y-C8PG-uhpKIUD3v-m5trQ5aRqf2fo4DkA8f-X46UfQQgtWCuneVHTVKQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230322_110214_17_2438_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.136Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Inp2VjlpeGo3RkJzVHVIbW9VbUIrTmxrSlYrcCtJc21SQ3RuNkhVWW40TzRIdWNRekc5d3NpQTZuRWFVcVZHb01xVVRXOTNzZ3BNUE5WR2VyMW9HNWRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMDM0MTNfMTRfMjIxMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDYwODZkY2VkNDM2M2Q3MDFkZjRkYjczOGM2MGJjMmI0MzZjOTBjMjM3ZDg1YjA4MWE1M2ZjNmJiNDBmZDYxNzE1M2QwYWM2MDU1NWY0MzVkZjQxYmMxZWE4YTQzZjhhNzNjYTE1NmJmZGQ0NTI3MGZlMzVhZWNmNWViOGI2YTk4NzE1NTY4YzQ4YjA1Yjk0ZWJiYjhkYzg0MmNjNmIxOTE0ZTEwZTczMjlmNDNhNTk4NjFlZTYxZTJkMDUwNGQzNjc0M2I5YTFmNWMxNjQzOGIwYzZmODI2ZTY2OWM4YjY1YTNiMWM0N2I2YThkMDkzMDVkZGVhYmRjNzZiMzI3MDhmOTQ0OThkYTU2NDUwYTk2Y2ZlYzQyZWRjMGMxZDhhYWFhMjkzMTc5NDRlY2E3MjFlNTMzY2IwNzFjNjU0MjYwYzU1YTFjMGJjMjJkZDEyYjhmZjYwMTQyM2ZhZjQyNWExNjA2NDVjMjgxYzljZGQzOTY4NDQ5YzU0OWU2MjRkN2YyYTc4ZTdiZDA3NGYzOTU3YzU5MTBmNWQ1NTU1MDRhZWUzNTY5MzM4YWE0MDUwZDM2ZWQ4ZTY4MTQ2NDY4NDcyZjU3ZGZhZjNiNDg0YmYxYmMzMzA0M2RjYjY1YWQ2ZjQwZDMyMTQxNjVjOWU3M2E2Nzk4MDcwOTZhNmRlZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rSZibgQwhVKUNTn3NRDYkINRDkrHD9uaXwT0ewrMZC0zKsXrxHsqUJOAIK9bmoA8kZGu105me_TEjh4pPgw-FA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_103413_14_2212_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.144Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlVwVnphTWR4d2JkRVRYVzBseXpVb2NaZzVFemljVTZBRlNtUzJWeWs3S2Nkc2hHN3dBN2VFUTg2MGZsMFh6cVZWODdKdGJLdHR3ZXc2dC9pSjFZTnpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMDM0MTNfMTRfMjIxMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTgzZmY3MzZlZmRkMzE4NjU3YTExMjc4NDA5ZjBmNGNlMjExMGUzOTg2N2E0NzQ1NjFjNzZiOTM5MzI0ZjExNzk4MjY4NzY4Y2YzMzNlYjYzZmMyZjAzZjRiZWYwYTM0YjllMDIwZGYxYWI2NzhjNjk4Y2FhM2ZiOGIyMzEwYjE5OWNlYWU2N2E0OWM2MzhhYTA5ZTk1YTVmN2Q3YjBhMmVhN2E1MzBmZWYxMWY0MjZiMmQzYzI1MmQ1OWNiNGNlZTVhZDdjYTNhNmUzM2U0MGVmMjAwMDZiNTEyMjg2YzlhY2E1MTQ4OWU0MzMzMzJhNWEwMjNmMzg1OGMwNWY1NzY5OGRhYWRhMzY2MzEyNmFmMTZkY2IyODcyMDU4YWVjNDcwNjZhNjAwMDRmNzEwMTY2OTk5OTQ0YmU4YjJjYTNhNzhhYWRmNWZkODQzOTYwMWFmMjE2ZDMzZjQyNjM2NTllNGNmODU1M2EwNWY0MGNmNzkyMWY1YmRmNjk2YzY0Nzg0MGViYTUwMzM2ZmQ2MmQ5ZjQwMzY3NjZiNWM1NmNhYjU5NjFmNWJhYTE4ZmY3YTNhZjU0YzQzYjRkYmZlZDU5OTU4NjUyYTY1ODMwZTY1NmYxM2Q1YTc2MjBiODVkZThkNGIwZjlhZDAzMTA1Mjc2MTIyM2YwNGYzYjhiZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.b4cp7ZeQKv_RvAuo1E67cjA0Vi4eUInxsE6QRARFhy33BvIytLM6R-3V_eWnpQSvo3mY9-0Uah-kNRPaPTcLKw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_103413_14_2212_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.148Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InpIaFg1UnpmcG9VVGNiT3prOTJYTHdxL3VsSXM0MExpMXhYVkxyUHA5VWt2QTdGZTNGZzZWMTRuOTZMNXNPcnlRUm1ZSnFtNTB2WFhqTFNlTjVRVzlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMDM0MTNfMTRfMjIxMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTEzZmJjNTgyYWY1NzI3ZGQ3MjA1NzEyNThhYjI0MGZkODY0MWUyNTIwMDdiZmNiOTkyNTMyNTZiNTFiZmRjYTE3YzgxYzBjNWIwOTc4MDQ3YWI3N2Q2ODM0YzA3ZjU1M2MyYWMwYmRhYjAzNmJiYTIxNjRmZGZmYzMzMjk3YTU3NGY2MzA2ZWI3MTZmM2VjYTQ4Njg4Y2Q5ZTM4NWM3YjdlZjUxNDU3ZDQxY2Q5MTcxOTdjZWE1NWM2NzZiNjNiZGE4OGNkYzllMGNlZDZjMWEzZjIzZGQxOWU0NTZjYjIwMjdmMzQ1NWQ5MjA1MDE5YTU4YzdiYTZmM2MzYmZhMGZlMzcyZjE0NWI2OWI4OTFlM2M5YjFmNjA0MTEzZjIwZjQwMjc3MzM1ODQ1ZDc3YzhlYjM3NzUwMDVjZDYyNWNiNmQyYjA5OWI3Y2ZmMTM5OGZhNDExYTk2ZGNjMTdjOGE4MTJhMjJiM2Y0ZWIzMWNjOTA0ZTA0NDEzMmZkMmUxOGJkZTEzMjYxZTJkOWFhNjE3NzE4NTAwYWIwOWZmNTAxYjlmZDFmZjhhNThhYWIwMmI3NjRjM2NmODcwNzllYWU2ODYwZmYxMzEwNzU0MjRhMzZkMGRhMjNiM2IwNTIwY2JjNjM4NmQ3ZWNlMWU0YTAzNTkwMjU2NDA0ZTU5YWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NXycLaJ3Fd8LT7U-IunKzOilTljMYBXabpcHCgcYbyUX7me-SR4Pf8lnswHr_XYt7B0KhlMrXRrO6F7wsqxpAA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_103413_14_2212_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.152Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFXUUlLT3RVa1VVNFM1RG9QMi9PUldZamVlaXArNW5ZdzZtWWdLd3g4TGcrTEhzR2loRlgwTlNyQUIyVlN3MGs5SGJnZ1pQcEliYTlLdDZSZUUvMHp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDQyMl8xMDM0MTNfMTRfMjIxMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Nzk1NTlmNjQyZmY1ZWE0MDViNmUzMzBhMzkyZTRjMTk0NmY2NjA0N2Q0M2IzNjQ5MmRkZDBkOTU4MWZiN2Y4NGQzOGVmYTdkZTFiZDY1Y2VmM2UwNjNhNDk3YzAzYjJmOGU2MzdlNTgzNzk1MzY2OTgzNzRiZDcwZTgzNWQ1MjNkNzQwZjMzYzFhNThlM2NhNTcyM2Y1MDE5OTlmMmYyOTJkYTE5ZWI1M2EyZDczNTQwZTU3NTM3NjliYjljNTU1ZDcyNWEzNWY3NDY5N2IyNDI1NTQ4OWJlOWE0ZWEzZDczYzI5NWQyZWI5OTdkMDAzZGI3YTViMWE2NzM3MGNkZmM2ZDZmNDI5YzNmMjEwYjI2ZGRhNjAxMjI2MzQ0YWVlOGE4ZjZkODFhMTFiMTVkZmRjNjU1MzYzNDM5Yzg0NTkyMTk1ZmQwMDY5YWU2NDUxMjBhOGM4OTk4OWM3YWQxNDlhNjlmYmY5NWQwYTFlYjNiNDM5ZmE5YTE0NzAwOGM1MzVjMTQ1ZmMxYWI3OGEyNTE4OGExMDA2M2FmOThmMmVjZTQ1N2JkOWNlNjY3MGRmNmU0ZGVkMWVjMzcyNDk1MjRmOWMyNjk5MWIzMDRkYjcyOTAwOGEzMmNlNzQ0MDQ1MmNkNGIzMjI0Nzc1Y2RlMTllNTQ3OGFlZDE4YmQxODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Md6_1iRT6-RC5e623Xmnz3ZQL9mZNH-dT9KwV3hS6h_xE0NSnYSDEneR-XtF4GbmokKALSOOkq3Mrrv0jTtO-w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210422_103413_14_2212_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.155Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im1RVkI0Ujd3YTd5R1pHb3JwSTRuamZMRDU4ZHZIUmd5VkJoZ245R2NkY1dnTkJ3QU9WRFFYWjhkU2dOcGVzdkE3QTdBclRpd0g0L3p3T3RGNzFzODFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMTAxNDdfODVfMjQ4Yl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGRjNTQyMzEyYzJkZDlhOTdiMzU4NWU4NWFjMWMwMzBkMjNmMTdlZjRiMDhjMzI2OWZjZWIxZmUxOTJiYzAyYjYzMWNmYTc2ZDVjMjEyOTY4MWM4NTM5YTlmZDlmODRhNDQ3MThhYWFjNzVhYmY0OWVlZjhiZWM4M2Y0NzZjZWJiOWYzZGRlNTFjZTA3M2RiMjRmOThkM2EyYzRhYmZhZGY4OTEyYzg5YTQ0YmM2ZmNhYTIyNzBkODM0YzY3MTliNDY1N2ViNGU0MGU3NzNjMzkxYTZhNGQ5NWEwMzM4OGY0MzAzMDNjMGFlMThkNTk4YTUzYjk2ZTI0MDZkM2VmMjA0YTk0ZDRmNTMyNDIwN2Q1OTc3ODE4ZDU4ZjllYTYyNDdlNzVhNGVlNjYwYjBiZGRjZmNkOTVkOGQ4YjQwOTlkZmJjM2Y4OTRkZDM1MzBmNWU3NDY2YzU3ZDQ0YWY5MjBlODQzZTUyYzZkYThhNWU0ZTZjNzc2OWE2ZmU5ZjI2NGU1OTM5YmNhNjNkMGZmYjZjYjRmY2U3NDFhNjlhM2QwZTY4YTlhMDY3OGM5MWZjYTE1NTVlZTdhYTUwZTY4MWZmNmVjMGQ3MjlkMWIyMDBhNGMxMGE0MjI5OTM3NWY2ODU5ZTQzNGM5YWQxODlkYWEyMGJmODM0ZWYyMjZhNmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MBGiheFmzV63zwpODn3H5C8eL89LVXuAwEW0yJirFd0IZOPYI_5Gg0LmudYgCUMKAAt1yTpIhmAWAuriS1zLaQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_110147_85_248b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.158Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjhRUDNOTlB3R3VENzVLUW9SeHNMUFZ4bFpLWHFmS1E2c2YxQ3dMdEZ2eFd1L1orWjNvNzF4WXovVDBHbnFDb2xCNnhrcWVsWTR1QWFDcERRcHI4VVVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMTAxNDdfODVfMjQ4Yl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmFlYWI3ZTM1ZDBlMzllODkwMjRkZGJmNmE5ZmMyNWQ2NGY4OGE0OWVkZGUwOTUyZmYxYTAwMzZjZDQ0YmE0MGI4MGViOWMyODdmNDAyNWU0NDY2YjUxMjhlMzczNDk0YjlkYTFmMDM3MWE5Mzk3YzdjZDgwYjYyZmY2OWVhMjNkZmUxMWMzZjhlNDA3MjIyZjIzNWMyMDMyOGY0MGNhZDBiZmM1ODA2Y2RiZmQxZDM5MzYxOGMwODk4YTMxZTliMTI5NzM1YjI1Y2I5ZDMyNTU0MDAxMjYxMjJiYzJiMzhjMmEwZWM2MWZhMWY2MmVmZDQxMzI1MjU0MmExZWY3M2ZmN2QxZmFiZDUwN2IwMTc2MjM4MjhmNzYzMmMxYWNlNjVmNDIzMzk2NDYxNzgzMWY0MjliMzBlMzM3NTk4NDBiZjJiZjE3NWVkODY0NzA4MjQ4ODFlYzFmODgxZDkwYjJiMjdmMmU2MDUxMTVjYTRhNTgzOTZlYjhlZDZhZGYyMjE1NDIwNmUyODMwYWExOGZhNTEwM2ViYTg0NjgxZjQ2NTUyMzRlNjVmNmU0ZDFkYzJjMDI1NjUyNzU5Zjk1Y2Y0MzU3ZjcxZmM5ZTc5MjdlM2EwZjA4NTFiZGUzODI1ZDc4YzYwOTZhZDI0NGYwNzE1NjZlMDFiMDBjNDg1YmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.R_Qh_PY5W2ffalBrSCK3yCdZmAhee2RS_lTBotRCTvm3NSYzIPT-sSXz9u4xiGWRvGBQlIlbnsWYjc8V0FgR8Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_110147_85_248b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.161Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii9MSXYxTWpyQTYvMU95aDVwbUNKb0wxVkhUU2w2OE0xRHFOZVhRV3U3aTdQYi9VaVA0WFhqdkhDL3BISjhJTzRUSmJodUd3R3JaMW9GUUpUUFNBNWdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMTAxNDdfODVfMjQ4Yl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmQ5NTFkMjYwMjI1ZmUxOTM2MjMwNTZmMTM4M2Q1OTQ4NzZlYzBmMWFjMzJhYWYxNTYyNWFjM2IwMWZkNzFjZDdlOGQ4ZDU3MTViNmNmYjk4ZTgxZmYyOGIwNGIwZjk2M2FiNTM2NzliNmRlZjYzNDJmMmQ4MDhhYjZmN2ExOWY4MmYzMTM2ZmViMWU5ZDhlYjAwYmYzYmEzNjg1MTc2ODNjNTFkZjBmYmY1ODY4ZGJiYjdjZmQ0MTQyZjY4ODg4YWU5MTU1OGJiMmJkYjMyYjc5ZWQ5NDkzZWZlYzcwM2FkZjVjODNiNjJkYTFkYWQ2ZDA2NDZjNTE3MDNiNWU5OTgxYmY5YmVlYWQzMWMwNjA2OGQyZmY0NGRiN2EyY2Q0ODU2NjM1MWRmMTRmZThjN2UwYmRjMDFhNWQyM2Q1MjJlMGI1ZjgyM2FhYjNjNDc1NzgzYzI4OWFlY2I1MWEyMjk5YjZkNTM5NWY2MDg4ZThhMWIwODI0YTRmODc3YWM5YjVmNzk4N2QxYTJiNWU3OWZlZmU5M2I1YTgwOWMzM2FkOTBjOTMzNTkyMGIyM2E2OGFkMDY4ZDY1MzAwOTQxN2Y4YjMyY2E2MjYwZjIxNTdhN2NiNmQ5NzZjYWJjYTYzMDhjMDA0OGFkODUxMWYwYjY3NGNkMThjNzc4MGEyMTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.R5t_ijtnfQMSgfQsPKqJt94RH3k4nbLlpDJnQpkEuK5ZydEtP68ea9xPMldaSfzahb3VHwtn2h66nOlGI4SBcw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_110147_85_248b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.164Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikg2Wk9JejVNcFBoMVphZkYzWGtRdy9GWFlRMkhqZFFhRk83WElPVG5pN2ZibXcvamM1ZTZGakkxNFYrOXcwYXJPRCtLQTZFY3F0dXR0UHVYWnF1V0hnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNF8xMTAxNDdfODVfMjQ4Yl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjJiMTMxMTdmNGRmZjAwZGIxYzNkYjM5NGI3NzlkN2NiOGIxNzBjMmM1YWE5OWRkZWQ0NWY0YjM2YWY2ODA0ZDdjMGNjNDI1MGIxZjM4YTc0ODliMzU0ODU2ZDM3YWMxZDczNzM5NGY1NTM2NzFmZTQzNDlhY2RhY2FkYzEyMDA1MGRkZTliZDMwNWNhMWQ2ZjYzYWQ1YmQ1NWRmNTk1NmYxNTQ1MTNhZGE2MTAzZGJhYjhmN2Q0NmQ3MmIyNWU0YTYxODNjZTkxOGI2ZDlhYjlkOGI2OGVhNDgyZGJlZDNhYTJjNDZlOGYzOWY4ZjliZjM1MTg3MzQzZDQ4YTkyOTVkYTcwNmFmNTQ4MzI2NjM1YmFlODg5ODdiMTEzNWFkNmYyYTVhYzdjMGYzM2Q3N2ZkMGJiMmFiMzFlMzg0OTA3MzMyMWFlODJhN2FkOTMwNTVmYzI0ZGQ1NzMzOTY0OWIzZmFjM2I1ZjExMWE3MTBjMDFlYmQzNmNjODU2NTZhYzk3NWRhNDA1NGMzOTMyZDQ3YjhhMjQ1MjM2NWQ5NzkyNGUwMzg2ZTk0NzhkMWZkYTJiZWQyMmQwNWZmMzEwNTdmYzcyMjY4ZTIxOGI0ZjA3ZWM4OTA4YTcyZWVkODMzNzllY2VlYTZmYjg4MGQxYmRmYzBlNTY0ZTllZWI0ZDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MAxio7f42q5pSR9Lb-yTXNiHgSMEZfXsiYs-WgvwZe9v2pz5I50p9FVp6AZFhjBjbC30iYxYnn6Vj0ZeRJc0CQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230414_110147_85_248b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.167Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjBQalh5V0p1by9pWTh5NTROcGcva0ZPaWRCUUZmWGdicFlqa3hGUzJXbVl5WW83WnQ5dVVIQ2ZwalpHalNzcGJjTmJkcDk1VUxsa0JmRWI5VmQ0eHl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMDMwMzRfMDhfMjRiY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWEwOWYyM2JjZTUwMjdhZDNjNzUxMjgzYzMxZjU5MWQ5MWRhYjUxYTQzOWM4MWFlMWUzYTNkNzRlYzI3NWVhYzMxODU0ZjdiODg0MjRlNDExMDVhMTJjNDViNjYzM2E1ZTk0NTg0MGEyN2Q3MjljNTQ5Yjk4NWE0MTQ4OGM5ZWUwNWUzY2YzZGZhNmE5ODA1MWFjYzM0ODlhNTY5YzliZWU0NWMyZDE4MTAyOGUzYzBjNzQwNzA0NGY4ZWFiNzk1MWE2MTI3MzM1OGU4MzI3NWZkZmVkZWUzYzMyZGQyNTRhZjQ2MmU0YzZjOTQwYTMzOWM1NDk0MmFkZTUyNjgxMWFiMTA3MGQ0ZGY0YTA2YmYzMzg3MjA4Y2I2MjA4Y2UyNzE5YTFjNzRlYzk1OGFmODVkY2M0YmYxZmIyNDdiNjU1ZGM5MDYxYjFlNjlhNjM1NjlhYjcyZjFjYzMxZmI1MjIzZjE5MGU1YWM2ZWI4Mjk4ZDM1YjM4MTExYjE1NmJkOGUwODJjMGU5NjQ2YjAxMWMxNjA0NjJiZmQyZDc3MjIwMGVkZDY1Nzk1OGVjMDdjOTc2ZjAxMjRjYjMyN2QxZDNlOTVhNTAyOGVmMjVhNjlkN2E4MjgzMDcxYjc4NjM0N2QyMzNjNTA2OTY1ZGU1OTkyZmEwMzMyMDRiYzc1ZGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8tmbVqGcF29uUMqZVrCNgcqt5DY3pqkMUyH8iI9ZceNR9w4Eyl10JPDhqpAuBtFkd1_-EtAOXogDF8peuuhudw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_103034_08_24bc_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.171Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFyTnNaVU51RTNuM05NS2cxWTZ3dWZxc1VCaHNsclgyMGxCb0xnakN1c3o0OExrWEV1azdrNWd6SDk0allaaFJyMkI4WG8rMmpFb0NiVWN2LzRLM1ZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMDMwMzRfMDhfMjRiY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGNkZjYzZDQzYmM3ZGNmMWM3YjQ1ZDVmMDM3MTYxZmEzMDAyMmRlYmVhY2I3NWFmYzI1ZmJmYmMyYzI5N2U5YTA4Yzc1ZmYwZDBkZGUyM2I0NzgyMTdkZGI4OGI4ZTVkYTI5YWM0NDJkNTBkZWE1YzM0YmRjOWFlOThmY2NlYTYzZGM0YTMxOTAzOTdlZjY1NmMwMWRkMjUwNzcyODg4NDc1NTkyNDQ5YmQyOTEyY2I4M2FkYWMxMGFiODYzZWY4Y2NmYWM4MGI3OWYyZDY5NTljYjc4YzRjOGUyNTM2YmIyYTA1ZWFmZTRmODAwY2UyYzNmZTlkMGVhN2Q5ZDUyMTMzOWY1YmFlOTc1MWQ2OTQxMjFiY2I3MjgxNjk0YWZiMzdmODBlZjRhNjY4NGJlZjc2NTY1ODY2OWVjNzA1YzE2MTlmZTllZjJmNWQ2MTcwZDMyZmY5NTBlYWMwMDNjNDY5NjdjZDY2NDNmYzFlNjJhZDc4YTVjZGI5ZjBkZmRlMDY0NmY5MTVmYjBkMTNjMmJkMjYzNDg5ZDdiODA5ZTRiNmYxNmYxNTA5OGI5NjNlZjUzMTQ1NWY4ZjhiYWMwZmMzZTZkZDNiZjMzODQyNWZiYjRiNjdlMTJhMzcxYjViMmUxNGUxNDQ4NzQ0NDk0NzY5MzJlYzY5OTA1NzgwNDlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8Jjgnu6rCVasYlkV2OwC5Zj6Xon9h94nmZnmzhgWcZAv-XoL5kktHFBoa2n7imoyATC0y9sKkHpgai9s1kK54g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_103034_08_24bc_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.174Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ink4UnZ4OEoyUE9XNWF4cEp4bkU5anRCTnhrblNJR3R3UXliQ1VlaFA0UUkxUTArQ3dQOU5raHlJRG02elE0Zmo4NzBYbmFOcDlRUUViYUhTbnlhMEJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMDMwMzRfMDhfMjRiY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzNmMmY4YzlhYmUyOWYyMTgwNzZjNThmYjA4MzkwZGIwZDA2NWZjMjAzZjRkOWIyYjg1NWQyMjU5YTMyNTU5NDc2MzJmNzAyNTBiZmU3N2I3YTU0YTkxNGFlNjQwNmY0ODNkNDIyZTYwNDU5Njc4ZmYyMzVkMGEwZTczN2VhYmRlYjM3YzcwNzA1OTZmNmRhYjlmOTUxMDQ5ODZlNTMyMWRmZDg5NDcxODA2YTUzYTcyMDZkMmQ4YmNiNDVkYzEyZTc4NTQxYWM1YTZiMWE2YWUzNzBiM2ZiOTIxMWZjZTE0NmJlNGMxZDAxM2RjNzhlMzZlNTZjMDYyZWEwNTU1MzI1ZWQ5NTQ4MzQ3NTFjYzNiNWU0YjM0MDdhMGVhY2U4YTM3MzAyNWY1NzA1ODU1ZGJiMTc3NzQ4MDc1Njk4MjJmNjk3OTJjZjdmOTUxYzMxYmM0YTQ1ZTRlY2Y4NDRhNGE0YzZkODRjYWNlOTM5OWVlMWM3YWM4NTZkYTBkNzg0NGY5MTU1ZjYyMWNkNzMzYWM2ZjMyOTVhNzdjZGMzZWM1NmVmYzg5NmMyYjZmOTZhYmY0MjM1ZjNhMzllNzQyMGE2NWYwNzNiNjkxNDJmMzY5NzU3YzhlMzMyMTRlYmI3NTM4ZDM1MjAwNjc5ZDFiMGM2YTIwMjZmNWMxMDZmN2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ACsxH9XBH_EhjuX3PSf5qVcMAxDnmwtWncv6r9rvz1udRKXpZQ6Vwl2KXYQr8CkFTlIlJ3Gyu3VXIJqWKFa_GQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_103034_08_24bc_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.178Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkZUQXdmckpCYmhrNXhsTnJUalY1L0gvRDZJNmNlQjhpZHlhUU9lYWZISDgrb1pvNkx5TlVObXZHRVVhandocmdEMDJTQmlJSmY1Njd2K2VETWZUbFdnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYyMl8xMDMwMzRfMDhfMjRiY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTQ4NDEzOTkzODJlYzNkYTZmMTQ0MWU0NDIwMWU1NWU2MjNhMTBlMTlhZDI2ODg3Njc1NjVhODE1MzQ0YzlkZjQzYTVmN2RiZjQwZGEwMzMyM2Y4NDk3ZmMxZDNiMTZjY2M4ZGY2MTUzNDI0MTBkMTNhZTRkMzA0NzE2MzVlNDIzMmViMmNiMzEyODY4NzQwOTZiMzY1YWRiNzY4Y2EyYzUxZmJjOWU1MmU3NTVkMDJiYjQwYjUwNTA1MDg2YmYwMWQ0M2U1ZThhMWU5NDgyM2I3ODkzNDRkNDZmOWY4N2MzNTI3ZDBjNjUxYzYxN2M1YmJlNGNjMTEyMjgwMjliMDlmYTQwODQzNDFkOTI4OWIyMmE4YzU1MjAyOWI4NWEwOThiZTkxMDkzMzAyOTk1YWUyOTU0ZjIyNzA0NmRhN2ZkNWRhN2EwMGYyYTBjZjMyYjQ3ODc4NGY2ZGE0MzZiZjU5MGRiZTI1YTQxZTNjMWI4NTc3NDE1NDM1YTM5ZDA1ZDFlZmQ2N2ExNzRiYmE4MjQzN2Y5YjgzNjU4NWMwZjNjYTE3MmM5OTEzNGY5YTcxYzkyZWRiMmFjYjU0ZDRjNGE5MDUzMTA2ODNlM2FmMzEyNTU5YzVmNWZjYzJiOGQwYjQ2MTZkZGUyMDgzNjAyNTZiNjMwZjU2YWYzMWM5ZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9XG2K5hMKD6chCZAvltGDmExht9PhedUTL8ltjerxx3yRq4_zEz0C3G4JOhpG6w0kODg3PSz6EhCa3tA4bWfFw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230622_103034_08_24bc_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.181Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkE2dHgyWFBBanZUemxPUzllR1haN2F1czhWUk1tY3NhdzVyb0FDUFZNcFRiVGpGL1RxQW4vbUlPTHJjSXlJK1pEWEdTUkl4dzhNU1dtS09LUkxDNUVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDhfNjNfMjQzOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjYxYzIxMGQ4MGY1YjNkY2QyNDVkZGZkNjc4OTVlODg5M2E1Yjc0MzM0YWM0Njc5ZGU4NTk3NTBjZWRiYTNmNTRkZGZlYWFmNGE5MWJmZGFmNzJhY2QyYzE5OGNmZDZmNzk0N2YzNGRiYWIwMGNjMDU4YzYwNTFhMDM1OGExM2EzZDgwYmE3MmExYmNhMTYwOGQ2MTA1ZTJlZTJkZjI1MTc1NjE2MDBhYTRlMjZmMzNlNjY3NjE1ZTU0NDE2ZGEyOGRmNDM2ZWY4MWUzZjY5NGUyYmM1ZWJhZGQ2YTEzOTcyMGE3ODk5ZWU0ZmUzZTJlOGQyN2M4ZjRhNTMwZDVlOWQyNzM0ODRmODIwOGIwMmMzZTYwYmQwYTg1NDgxYzllMTg2YzlmMWM1ODA4ZDlkZThkMmJlYzYwMjk3YWExNzYyOGZlMDhmZjQ1ZTA0NjkyZDM3ZDI0M2RkMDJiNGM4OTRjYjc1M2NkZGIwYTNlNzg1NjA2ZDQzYjE0ZjNkZTJlYTNiMDg2NWQ5MTJlYWEwZGM0ZWNkMDkwZjU1ZmUxOTY3ZjE1OTQ2MTk3OTViZGVjMWNkMTljNjkwNDc4NjY5MTkyMGEyZWZiOGY2YzZjYTljOGVmYjAwMDgyM2NhMzkxOGU0ODRmYTg5N2EyODcyZmFkZTMxNjI3YTgyZmQ2MTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.lS4Wv8gO4DaA31HbFGodImT_diEhMuRMCXmwE9IfTlwPmBN3V3Wo5VcMeDfKAemAtsa0hYlAlrKBZm3FfgUriQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111008_63_2438_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.186Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikh6dThIY2NwY2YzVEJEM0xDbUpKNzQ5djIzclJSTnZlWmxFMCtkYTdKQ0lURzhDTE00a1RHNGYyTmhsMlg4dVBtcjVuMUdOaHJ0UFpEcDE0Y0tQWVN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDhfNjNfMjQzOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQ0Mjk3ZDE2ZWViNTNhZmQ2NjI4Y2I2NGE1MDBjNWY3MzhiZjg0YmViZjExNWQxMTM1MTFjM2QwN2M0ZTg0ZDU3ZDM5OGQ1NmJiZWFkY2ZhM2Q0ZGNiNWY1ZTQ1MDk1ZWZiNDM3Y2Q0NDk2ZTFlY2U1NWNjM2QyZjY0MDZmNDkyOGE3YzYzYjFmOTlkMDM0YzNkODM0ZDMzN2M1MTIwYzlhMWMzMDc5ZjQ2ODNmMWIzMmQxOWFhYmE3MzU3OGNhNmM5MjMxODJlZGM5YTM5OGM0NmI4MTMyZjU0NDg1MDJjYzljNzE5NzI4YTYwODk5MWNlZjc4MTMyNWZiNGIwZDljMjk4NWNiMDg4OWZiZGRlMGQ1MDMzY2NiZmI5YjFiNWFjZmVkNjVkYzFiMWE0YWE5NDIzYjU3Yjc4MDAxMzNkNDViN2M4ZmViNWVlNzhlOTkxZWI0NTAzNzEzYjM0MjA3NWUyZjE2MTk0OTFjMWQzOGYxZTNkNDAxODVjNjFjZmM4MDAxNjhlYjgxOGExYWE4ODg2NDk5NTBiMTIxNDk2NzM2ZjFhNTdmMDBhNDU5ZGZmZGIyNGY1MmU5NDAxOTkxNjk3NGU5MTJkNWQ0Mzg2NzQyZDZlNjFlYzEzNDkwNTlmYTJiY2IxZTIxM2MzZTlmNWNjYzY1NmRlYThjNzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FMz3b-OlkoiseAW0LcY7DoeMjlj4_y_5ryRWc4JElj2rFCw-BDRGd7AqypahnHzP_pQ5E9DZiE59E41IzOm0cw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111008_63_2438_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.189Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImJKTmFvTGJlcHI3K0tUVzBNVzJXaFE3Uk1sUXl5Vll4Q3N1QnhwMjg2cFZRRk8wUlZUVUVQTkVqRmdSWllFNjVlZnJDYU1lZy9nSzZ0b1k3L0ppL1J3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDhfNjNfMjQzOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmQyNTQzMmQ3YWQ2ZmJiMTg0OGMzOWMzZDkzZWJiZTk5ZDgwMzhiODM2MzVlY2E5M2U1YWEzZDIzMDYzY2RkODI0MmNhM2Q2N2RjZGFjNjhhYjNjMTdjNDkyMjE2NDY1NWFlNTJhMDlmZDVhMWRjNzAxOGIyODYzNGM0ZjJmMTlmNGM3NWVkMWE3MzMxNWY0ZjZmODM4MzExM2Y5YzI3ZWM1NWE3MTMzOTlmYjM4ZDhiY2Q2NjBhYzg3M2IwOWE1MzViZWI0M2M0YWMzNjU2ZDJkMGQxMDQzZDJjNzgxNTIyYTJmNjRmZjY1ODgzZDRmZWMzNGQwYjA3NmI3OTQwMzhkZTMyNjBiOTNiNjVhMWNhYTRkNWVhZDI2YWVlYjRkYzFlODg1MGRjYmE3YjY3YzRhNmE3MGZhNDA3YWJkMmM4ODNmNzFhNTY3M2ZjZDA3ZGIzOTE0NTM3MmFjZTU0N2MxMjQ0NWEyY2ZhMjU2NWZkMjJiM2QzNmJmOTA5ZWJiZGIyOTI0NmFlNWQ0NzMzZWQ4ODU0N2ZiOTJhMDE0NTk2OTQwNmRmOWFlYzlmNWMwZDJjN2U4ODI0MjAyOGUwZTY3MzE5MTU5YjI2ZGNkZDlkNzc1MjUwNmQ3OTY0MmIyMDRhMDYwYTAxZDJhOTg5NTA1MTI5MWY0Mjk5MzcwMTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qfss9hjaXdNQHCoKtxgULoOgDNFlI3XS95NX8UrYILUOkJiJqC4ZMG4Alfi9-bJC0vWNpggjUJBQYg3lTHS3BQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111008_63_2438_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.192Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Iis3VS9WTzY3TWpzNWdZdkN6Z0drR0VlSmhtSWE3RWZMazhVQ2VWT0pVMUZtWEtqTzNGL2ZEcHFhYnhhYmZEaGpwWlBHZTc5dmF3TXFJYWF4V2UyeDlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwOF8xMTEwMDhfNjNfMjQzOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmUyZjA4YjVlNDIwYWJhNzQ4NTBiZjJmMzEzYTcyN2Q0ZWMxY2QwNjA4NWJmNzUzYzRiMTZiYzM2Mzk3N2IzYjFmZjE2MWUwNjg0NTJhMTBiOWQ2MTUzOTEyY2YyNThhODUzODYzZDIxMjkzMDRjYjI3MmRmZTQ2NmE4Y2NkMmVkNzgwNTQ0YTQyYzAwYjgwNDYxMDgzY2ZjYTVhMjBjMDM0ZDAxNTBjNGIwYjY2NTNiNTk4MWZmZGYxZjRlNGJjZjc2OGRiNzFiYzUyZWJkMjk1MWNiMzY5OThhYTU3MjQ3NGJkZjY2YmVmZjM4ZWE0YzMxNTQxMThiNTAzN2Q1YWIwMjBhMTFlZWMzZGQ3NTczZWM3MGMwYjc3OWUzZjczNzUyZDE1OTYxOGJjODcxMzE0MDMyN2IwNDM0OTk3YzYzYWEwZTllMzRhMDhjNWQ5MzQwYjYxODk0MTE0MTFkYmM1MzU2NGI1MzdmZGI1Y2M4MzU2OTg1YTk5Njk4ZTg0ZjMxY2ExODk5ZWVjNzkzN2M0ZDQ4MjNkNGM2NzRmOWEwYjcyOTE1MDMwMDRmMGE0NWJlNzllNjRlZTQ0ZWIwZGQzMTZmYzNkMmE0OTA0ZTk4ZGJlNDJhMWZlMDk4ZDE3ZWRkOGU0ODc3MGI5N2M5ZjMxMGVjZjVkZmY3N2ViZDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.o8MEuFzWx_NP7eKI9QaS2gzwErzhcp4eFtCU_bmJGzgSfZxpvVeY0MtLTrcxK9xV-B666phvCMceUuEr5tphrw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230908_111008_63_2438_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.195Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImQ0MlVoL1REVkJhTWhMOEh3K0h0US94MjJyQjhmS0JReS9Sd0ZwY0l3bGJWcER0NnQ1bEh2MHh3OWpIb2g5QjBxTkUydVZMaytkeUhUSkowVGlSQ3lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NThfODFfMjQ0MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQyYTRjM2EzMDUyYzJhMDAxNDUzODcyOGZmNmJhMTU0MDgwNmVhYmNhYTNjMzg5NGRkZjVmOTJmNWM5MWFiYTY3OTBhNTk5ODczM2NhMzEyMzAwMjRkNzNlM2U4OTJhNThhMDQzMzBmZDBhZTVkOTcwMjJkNDU5NzhiYjA1YTQ2MjkxODI3MzI0YzFjNWY1MDgyYzgwNmJmZWY3YTY2Nzc1NDk3NGYwOTFjMjBhYzA4OTU5MzQ2ZjU2YjNiMTIyMDI3Njk4Y2RlNTE3ZmQxY2IwYjUzMTc5Yzg4NzA3OTZlOTBjOTE5ZmFmYWE2OTRkNTBjZmRmNzRkYmNkMDZiYTFkNTcwNGE0NGU2NjkzNDFkODRjNzZlOGZhMDMzNTlkZGRhNjViY2RkMjkxOTkxYjM2ZTZlMDM3OWE0MTU4M2NkOGI2ZDNlOTIwMzAyYzllNmJkYTFlMjhjYjI2NmQzMzVmZTFlODU5OWU4Mzc4ZTEzNmIzYWNkMmQ4YjMzMzZmMmIxNjhkMzJkNDFkZjU1MWZlZGE2NDIxY2FmZDJmMzBkNDdjMDkzNTUxNjk5OWY1OTA0Yjg5MmY2YjRmN2IxOTVlYjNkZjNjY2JlNTI4Y2Y2NTY1YzZhNDNlOTczMjU1MGEzMzZhNmUwZGRjNzM0MTRhNWI4YjJiNmQ0Y2E4N2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EFAkFlDYrAF1cZ8gCL4qfFZA9fPrnVaLKWm_EhuOyPAZbZc48h2QgqHpXhmPwVen_OPEKUeamSjm_cYGFXwoSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102658_81_2440_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.198Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik0raE42T0ZOUFliMzVQZzRtd0dJWVcyVHpzNGY2cDVKMWowSlFocDhUZS84T3hYMXZGWHZZRjF2Z01sRWxYMW41YmlLeWNVZUpQNW92TVAzWDRWa1FnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NThfODFfMjQ0MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDNlM2UyNDgzNDhkODZkMzRkZTJhYzkwZGQ1YTAwYWUwZjQ0NDRjYzFlNmFjMmIzMGQyZDA1MGJlNWExNTc0MmViMGI2NTNlYmQwZDBjYmZhNzkwNDhlMGYxMWE3OGI1ZWY4YjgxOTgwYjNhNGJjYjViYzVmOTk0NjM5OGMyNGFlZjc4NDdmNjdhMTE5MzZlNzBlNzU4YTgxOWYwODgyZDU1ODU4NTg4YmRmOTY4NGRiM2ViMDljZjMxMzI1NmE1OTFjNjdlOGQ2NjVmNjkxZjc3ZWVlZTY2ZTBiY2FiY2ZiNmIwZjZjYzdiNTkyZGFkMDk0ZGU4ZjY0YjcwODlhZjk2ZGMzY2MxYTBlZDY1ZmU0NjkxMmNiNGQzYjYxZWE2OTVlM2UxODQ1NGNjNjVmMmFmZDNiMTRhMTdmOGU3NTJjMzEzNjI0M2M1MGQxNzg1ODJmYjcxNDBjZjExYWI1YzQ0OGIwNTk2YWQ4ODc1ZjA3ZGNiNGIwMzZjMmFjZGY3ZjgyZDdlMjU1YWYwMGE5MDAzNTJhY2Q5ZjhiMTRlOGE4ZTIwN2EyZmFjMTdiMmYzNTBmYTgyMjRiODMzNDEyYjc2ODAwMDE5NGVhNTBiOGJiMjNmNjFmMTg3OGRlMmNlOTRhMWY5ODZkYzUxMmIxNmE4Zjg2ZDRmMTAxY2FjNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eMPfQMj0Wv1qV-5G1DiM6wqEghy5f8eMiXrBNihbLUyH_qNdnllfxluqEntaEkmLD0yrx2wn9GNjDzDZcyNCXw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102658_81_2440_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.201Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjV6M2xvN003SGd0Q3ZuU0RGOGxDYjQ3OVB0SUhVWlF6N0dtYkczOThHbVgyWERjeElOZ3huTGxzYVM2anptNTlPa2FsdS9jQmFPZW9SMEcvY2c5VXhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NThfODFfMjQ0MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGZkZjlhMGQ3NDI4MGNiMzNjZmViZmNmODY0ZTk5OTVjNzUyM2QyZWQ3MTcwZGU2ZTcxZWZkMWUxMjA3MjQyNzk4NzhlMjAzNTYxMTMwZWY4N2E4ZDUzMzNjYWJmMmY4Y2ZlZjZmNTJkZDRhMWJlYWZiMmI2M2EwZGRkYTI5MTc2Njg5NjkzMmZjOTY0NzJkNTIxMTU5NDU1NzFjYzJkOTUwNzhhZWE0NjRjNzBlM2UyMDVjZWVmYjRlM2ExYzBhZGE1YjQ4ZmE0ZmQzNjE2MjM1OWQxODc2NzlmMTc3YmEwZTAwNDUyYjE3NTM1ZWIyZGJkOTU0YjlkZTc1MGE5ZjAzNjRlMTU3ODlkZWM3NzQ2ZTc5NzUxYjU4MDNlYTI1ZTM4NGE4YWZkNGFiZTQ1ZjE1MDhlMzUxM2U5ZDY2NmNlZGQxYzE4OWY4YmU2NGRkYWY3MjM4ZGJkZWQyODA0MWUyNzczMGM1OTU1OTFiYzgyZDM4NTJiOTViYWVlMDA5ZDkwZGJmYWRkNTY2YWEyYWU3NTM5NmMzODRkNjZhMzBmNWVhNDYwZDkyMzU4Njk2MmQyZWRlMGMwMzFjYjk1ZTUzMThkMDMyOGVhMDE0NmFiNzE1MWFmYWZjZjI3MjZmMGZjN2UwZTQ0YTVmMmNjNThmNTRjMTk2YzcxODhlNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.npcA8HU33dA6Pte93v4sBUDHWr6OivD4OoIL0ANoN4kuDKs8I0tGfGdzJvTyPQbcceAVBuvIeDTQA_LFMrjbFw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102658_81_2440_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.204Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjJLQ3lLZkUyeE95NHQrbU1kM0VVSkNjTkdZYXlSYldxUERDUURMV04xeGEzbWhZZDJJSnplU1lSLzhqV3lwNDkwTFpHd3k0R2NhZk1UYlY1OXlpQVl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYwNF8xMDI2NThfODFfMjQ0MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTNhMGUxNzkzOGUwNDFjMWU3MTA5MjEyYjNjZjg1NjJjZTc0NmViNGQ0NTVkNjY2MTQ2MzJjYTYyZTNiYjczY2UxNGRkM2U3OGE5MjNlMDVmYzM1ZGVlMTczZTk0ZTM4MGI1NzdkM2RkYzQwYjE4YTljYzU5NjRkMDBmOGEyNmE1NDI4NDdiMGYxNDZjYzIwYTZiZmUxNGEyYmI2NWU5MzRiMGRiMWMyNzFiYzgxZjY0MzZkOGE3NWZiZWMxYTQ0ZGNjZTVjMGFlZDM0ODI0YzI5MDliMGU4YjAyYWM2OWRmNzJlNjg4MzY4YTQ5YzI0ZGY1YTYwMTM3OTFiMmViNWE1YmFmZGMxMDU5ZWNmMjg5YjViMGYxMWFmMjJmZTgyMGRhNDJmNmVmODQ4ZDlhNTFhYWYzNTdhNzM4M2U2OTc2NjRiNjM3MTM1YTlmNTlhYmMyNTIzZjQyNjIwODU5YzYzNWJlMGE1NWNhZTQ0NjY2ZGFmZGEyMjY4NjgzYjE4NTc2MDM0ZGE3NTU3OGY3YTIxOGNhZDZjN2Q3ZGY2OWJiYWE4ZWQwMTUzOGI1OTk0MDlhYmRkYmQ1OTY4NDhlZWU5OTNjZjgyMzRjMmI2OGE5Mjk2ZDVjM2Y1M2Q1ODQ0ZTY4M2ZhOGIwNDI3N2MxZTI4NDUyODA4MWUxN2IzMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JpYUnEJaHkv54skoND3UhE6DBoIeL7QI9A2Ed1yvsxpDoxIf5PShTHaYTQkURb8b0tOhxu_49H-EQHsBLcHZyQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230604_102658_81_2440_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.206Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlpFS3A2aGIxMTNNS2R4QkhFbWdBbkFmTFZQTzNNVFRXWUlTRDYwazMrTHlZWnl3SGlNaDJDN0VQeEhyNmlZWTBDQ1hjQzJRVWxpczFYdWZCYmJzNTFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzVfMzFfMjQ4Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTExYzdhNjJiMGVlZmRiOTQ4MzEwN2M2OTA0OGJjZjQ3OTA0ODA1MGQxOTlhNWZmZjVjYmJlNDljMTA1ODMyMjcyZDE3NjU0ZjZiNTczYjFkYTZhNDYyNDA5ZmI1Y2MyZTVkZTZkZGIzODJmZDllNmQ1ZTY4MWYwODI1ZDE0NWEyOTdlOTRjNTEwYTQ2NTFlOTdmNDRlMjQ4ZWYwYTlmYjk0NjYwMDY4NDQ1MjE5MGNiZGEyYWNhMGQ0NjA3ZmY0OTg5Y2Y4ZGQ1OWIxZmY2MjRiNjVjMDY4MDc5ZjYzMjNiNDg3NjU2YTI4NzUwOTE2NWExOTYzMWU2YTk3MDE3NTNhMTk1YmY3N2UyMTUwOTY2ZGUxNjgxZTFjNDdjOWQ2NjA5OGUwYjBjODQyNGYyZDkzMWUxMmUyMjQ2MDlhZTkzYTVlOWU5ZWI2MGRmMjEwZGM5ZTcxZmJkZjViMDIzMDQzMWJlODlmNTc5MzA4NzViOWIyNWViMTI5NzE0Y2NjM2IwMDUwNDk2NzlkY2MzZDllMmJkOTNkNGI5MTdkZDkzNjVlYTUxNGIwZGUzZGMwNzRmODJjOGNkZTcxZjA1OGU0MTIwYzNlNmE2ZmI5NmY2YjdmYjI4ZDQ3MzA5M2I2M2JiOWIxY2M1MDFkOWQ2NmZhMzUzN2JiN2MyMDlhMDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NZaXMeZJiIJcAXR8cEhvdBcoru-4HJsRdVzvTO6tiVwHCCGXh_mTYul0pJi0z7x2aey_tKsqIQHbvG5VATODbA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110135_31_248c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.209Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImlvOVVZZjZEdDdGMFFTUnp3SWY5b2ptSzRjQytvQjR6TzdrZHl5WTRNclJqa0E1QkRtSDJab2JjS1c4b0czOWsvZDhjbUxuTkNmVmlhanlrVHFRUFhRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzVfMzFfMjQ4Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGMzMTA1MWMxOWNhMWRjYjQ0NGYyMTgwZTYzMmRlMzEyNGE3YTVhNjFkZDQ1YTBkMGQ4ZjcxMDg0OGIxMjRhZGZhODRlN2U0ZTJjMGM3NjA2MTMyMjQzNTE4MDZkNDNmZjQ0ZjdmODlmYTBhZjAyMGQzOGVkZDM3ZGQ2YmQ2YjRhZTUxNjVkOTMzNWE5ZmY4YjYxZjQwNzdlZGJkZTFhYzVkMWIwZGJmMjQ4ODBmYTEyZGFmN2RmMTU5MTdkNWY4ZWZkY2MyODJlODVjNzU2YzY0OGFlZjkwMzY4MGFmMTI2MWM5NWUzOTkwZjA1ZGEzNDdjMjI1OTk2MGU1Yzg4NTU3NWJhNDUzNGJlZWU1ZjNhZDcyMTY1MGU0MWFiZjY2MTRkMDdkNGE3Njc3OTM0ZDk1Mjc4Yzc0MTQ5NTNjMDBjYjdlMWVlOWU2YjBjNTdmZDM4MDBkOGQ5NWYwN2E0YjJjNmQ1OTdhOTUzY2ZmZTkxN2E4ZWQ1YmM5NzQwYzc4YmI3M2EyYzE0OTE2ZjRiZTU2NzdjNzU2N2U0NjY1N2ZiOGQxMzI5MGRhMDBkYzNlMzE3ZDY1MzVmODYyNmI5ZjYzZGM3N2UxNmMwNWY3YTQ1M2M3NGUwN2FkMmJlZjM4YjVkNjdhMTlmYjM2YWVjNmI3NmU0NzEwYWNlYTJhOWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.47-CnbppMOFDS1jXm1C7Jm3tMG3qDVTolXDEJ2X5SXOi7xYgrsRCRkPxFPJeVEPZCF6Zm2g3lRwPPP9arvOQWg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110135_31_248c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.213Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im1OUWV3UFJJbytaTUdNZDVaMlhoSWxEcm90L1k1N3k3cWJIL080L25sMFA4SXNJYXNSdERiaUxXRjVKNENLUmtUdXUyNk5oUDg4TUszQkp1ZUR6azJRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzVfMzFfMjQ4Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjNkNGI0MTc5MzlhZDQ1NTg1Y2JiNTY2ZjIzNTRiNzZmYzA3YmJiN2ZjYWRjMTQ3YTQ2OWM2NjFjMDU5NjgzMTg2YjhhYjJhZGE1MDVmY2JhNThkYTMyODA3OGJhNWRlNmNlZmVkMWQ0YmVlZmJjNTVjOWUyOTEyOGY3NzZhYWFlZWJmMDAyM2RkOThhYTVmYzFmODVhMzk3NjExNTJkY2JjMjZhNjAzYWJkYTIzMzk1NTQ5MzAzY2E5OTkwZmQ1MDcyYTY4NTE3NjkzNDU3YzMxNTYwZDJmY2RjNGRiOGI5NTU1ODNkN2JhMGE4MjQ4ZTAyZmI2ZjgxMDE4ZTExYzk2MTFmMDI0YzIzMzkzMmY4NTdiZWE2YWE0Y2NkNGQyODNkYmFkMDBkNDhkZjQ0MTNlYTYxMjUwYWM0ODgzODBlZmU4N2QxZjdkN2M5MWQ2NzAxMGJiMmQ3ODhkNTQ2YTUzNDk0ODhkY2I1MjdkYzMyNWVhZWE3ODcxZWY0ZjZiZGU3NTQzNzdmMzRkMjQ1M2YzOWE0M2U2ZWUwNzM3ZjdkMmY1Y2FkMWU4ZjRiNzdjNzQxNzlmNTAzY2UxODI4MDA1NWE2ZmMwYjY1NDZlZjljOTUxODE4Njc5YTJlNGZkYjk3MGEzYzgxOGYyODMyMDg1Zjg4MWVkMzhkZjUwMGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jgYLLfXMlMEVft3XMfXpqbpiV5bwlBJj3m2Oj23oCFEr7r3ohsoUpbLNunrKF9QYTOZCGICGocMYOIgslCFwBw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110135_31_248c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.216Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjRmRVdhN1J1QXhaU3FGV1g4cVJPQzN2WkNFVy84aFh0Nys0OG1FL0JIb3d0VHZPb2F6NldpUWN5M3F3Wk9NcUdmZktvYUl2dENicFVvS0FwVzEyU2JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQyMV8xMTAxMzVfMzFfMjQ4Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzM4ZGE0Nzk4NmE5NGZhZTkwNmM2NTc1N2I3NmNhMmViZjc5OThmMTZjMWYwNDIyNWFmN2EzN2RhZTVjNDE1OGExZmNlNjllMzExMjcxYzEwNDZhOGI3ZDY2MGI1NjYxOTI4Y2U0MGM3NGQ4ZmYzNTE5YzM4NWU2ZWRiM2I1OTcwNTk4NjRhMGVmY2EyNWEwZjNiODQwZjQ3MmU0ZmQ5NWNhYjY0MjUyYTU2MmUyOWI3ZmYyN2ZlZTUzNmFmNmExOGQ4Y2UzNGNlOTM3Mzk4OGNlOGI3OTBlZDBjZGZkMzVkODE5MzRkMjI5NmRkZjA0OWUyZDIyMmU3ZGI2MjY0MDg3MmUxMmY0OGFkNzk3OGY0MDg2MjEzZTliMDVhZmI0ZmE5NDQ2YzEzODhlYjAyNDBlMDhjZDMzZWRiMGY3YTgzZmFjYTkyMzg2Y2E3NjE3OGVjYjIxODk0OTNiMzMyYjY1MTI5ZDlhYzIwM2FjZDFkNzAzYTUyNjgwNjRiMjQ1MjcxODQwZTg1OTViNjA4MDUwOTY3MTAzZTQyMDRmNjU5N2NjYzZjNDZiZTZmMTE5MjY3ODAyNzlhYzNiYzQwZmZjN2QxNmEyMTJjODk1YmYwZWI3ZWZiZTQ0YjlmY2I5OTgyNzQ5MjlhODA2ZDE4ZDU0MWIzYzQ2ZDY4ODk2ZGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EBn3nDx305P0e6pBSceI1Jwev7nQDqxrBsdiRggLrmNxZVDV_yuSFryHY9XUzlpK7iszv3yQGT1661C9-c91rg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110135_31_248c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.219Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InJscEZsM0l2WlNwV3pYRDNoTTkyZDB1SDcwUzhGaXJKUXh3RXFXYkJ4TDVoeGFZU1BUd2hyUEI3bWpGZUl0dkZSOFFOUnA0R0JKRjdsNGEwTUR1dnpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MzBfNzdfMjQwY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTg2YzNlYjZiMTFlZWJlODBjOWQ1YmMyMDRjNTFmMzM2MjYzMDI3ZWIyZDJjOWY2NDMzNzdhOWIzOWQ4NTFjMWI0YWJiZjFlNTgzMjI0ZGVkNGNkNjI1ODFiNDgxZjM5MjcyN2VhYzE0YjA2NDJmZGM2YWE4MTA3MDMxODc4ZTM1OTE1NmEzNWM4NmZjOWRkNDRmNjEzYjE5ZTFjMmZiZDVlYWM0YjBhZWVmZDBiZjU0ZjliODE3YzdkYjAwYWY4ZGY5NGRlYzlhZTA3NzFiOTQ3MTU3MDE0ODdjNTc0ZjRhZWE4YmVhNjNjZjZmNmJjMzZlOTQ5ZDBkNjFjZjY2MDA1NDVhZjA4NjcxNDMyMjg0OTM5NWY2MzBhMjJiMWE5NGUzYjEzNWViOTAyM2FlMzc5M2U3YzJjMzVmY2Q1NDgyZDc5OTkxNjVkZmY3NmY0NjUxNTJmMGUwM2ExZGZjZGNjMzNmMjkzYTVjOWU1YzUyNmNjZDU3MTZlMjc0M2EyNWQ5ZGNjYzg1YTdhNjYzMGUyODdhNDgyZDA1YjkxMDE2ZDE3ZGE3MGVkY2YyZjg5MGVhYzA0OWEwYzFhMWZjOGZlMjA3ODY1MWJiOTgwYzQyYjkwNWRmMDM2ZTViOWU2NDBkOTY5NDM1YWQwYzVkZTRkOWNlYTM4OTRmNzc4ZjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zSr-VpiGkfrPf2VaK_0up7EdHr152wXKOCZDYbhYd_Su0jcpyCmsGNul6QPC_6UusV8iUlhkHVGIvzNr5oYYiw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.222Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InNLelo1cTloNHN0NVU5Mk1wM0tTNVR6ZzNiaWpzM3lMUnVQeXEvRlNEdGxoUStCWHYrcDlqMU9Pek41SnF5SExYOUp0R3I1WmF6d0hvQlFYNHBUL0tnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MzBfNzdfMjQwY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDcyZGE0MzYwMjczMjU3YTMzOTc1Y2Y2YTBlNTJjZWE2NzJlYzlmYjMyMzUwYmFkMWNmYTRlYzMyMzgzYjdiNGMwNmVjMmJjOWU0ZDc5OGY0YmE3NmFlYTI5OGQ2YTE4OTE2MDVlNDg3NjNiOTNhMTk2OWI4MGNjMDhmZjZhNjIxYjc5NzljZjcwMzI4OWI0MjFjNjExNmVhMzFiZjM3Y2M1OTQzZDVjODYwNDBkOWY4MGZlOTU5NzI5OTlmMDAwNWFjYTg5OGI2MTJkYWU1ZjI5YmZiYTBhZDE1ZjI2NDdhYTIwMjk2NzM4ZjI4ZjY0ZDYxNTBjNTQ3OGU3NTdhODJlOTdjZmU0NmU0MzA0ZTAyYjRjNmE1ZjA3ZTg1ZjZmMjkxYjI0M2VjOTQzYzNhYTg5YzhmZjQ1NTZhMTQ2MDZhZTg3Nzk1OThkNmFiYWRjYzAzY2VhMTkzNGYzODczYzgwMGI3NjU2YmIyMTE2YzAxOTI2ODQwNDA2ZTUyMGFjODg3ZDRhZjEzMzVkYzkwOTRlZTdjMmM2ZDdlYTJjZWE2NDQ3ZTJkMzNmZDgwN2Y5NDNlMmExMDM2ZDA5MjExNzk5YjljYzI2MzQ4NzMxMzRlMzA2NDRjZDM4YWI4MmRiOTZlODMxYzgzNTU1NzYxMWY3NmVmZWUwYzRhOTM5NjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.BEqCOxV_ZRKpDZ5txR6vpQw0Tw8HA7EXT1z_SfKBpTaPn-6wP7MABzjfeXX3BWuC9HaPrNtrLgb8DUj5fuf3TQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.224Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImtMYm5jMTkwQnNOZDdHck9GNHEwWXJYKzE2eFRRVElzNmEwNW9HUVEwQmdnWlRKdnlYSisxenJTUCtoSmhMREtCMkwwZVcyQzBlMjF1MmVJaDZGcThRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MzBfNzdfMjQwY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGUyZWNmMTZkMjkxMTI1OWE1ZWVkY2Q5OWU4NDVkZmRmNDkyMTE2ODFhOTFmNjBlMzJmOTBlNTFhYTQyYzA3YzhhZGI3YmUyZTkyYTg3ODIzNDJiODc5Mzg5M2JjZTM0NjYyNTM4MjU5ZDNmYTIyOTIwMWFkMmI5ODJkZWFkMzYyNGY0MTU4ZmM3NmIyMTUwM2FiOGQyNGJmYjYxMDE3Nzk4ZDdmOWNhMGZmNjc3NDFkYmY3YWYzOWM5YzlkOTAzYjExYTM4NGU1OGI4YjM3NzQ3ZjIzZjAzMDBlZmQxZTQ3NjFkYzUxNWQ3YmI1YTMzNzA4NzdiOTVjOGRjODU1NTdiZjgxZWE4NjAzZjhiMmUxMDE4MjAwYWVhODMwZDgxMzQ0YmNlMTBmOWZjMmYxNDhiNTUwMGYxNmFkNzMyYjk5NDlhNGRkY2I1NDgwMGNjYmFlNDk5MTM5NWM0NThjOTU3NDBiNzc0Yjc5NWI5ZTMyNWUwNTJhM2ZlMDM0ZjRiMzNhMjJkNTYyYTY3YzU3ZDE4YTVlYWQ4NDVlNDg0ZDQwODYyOTZlODYzMGY5ZDVhN2UyOTkxNWE0YjZlYWU0MmM4ZjY4MDcxYzViMjE2NmY1ZjlmZWJjYzM4ZjU3NTJmMGE1MjJjM2ZlOWFjMTkxZTA1Yzk5ODJjM2RkNzM5MmNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AS5GLXECceoVh7uEWuKDz2BnkWb5gr0jeISfPT8ajqn0XZ8qjKvJavJjqwMe8wSycxlDmZqpgAda1G0h51Xgtg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.227Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpTaG9VemJlWnVsZS9rOStSSTdUV0lXRzM1QlBXSTNiY3Q0VEZqdHdOWjZQLzNRWXNoRUlKZEtiMzRMTTEwMWNOWUVPcnNHT1J1SkFrQWxuZk01Ym5RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDIwOV8xMTE2MzBfNzdfMjQwY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDZhZTQzZTY4MzZlODVlNzVhNWNmZTRkNzU1NTdmNzVjOWFhZWMyODgxZWM5YmUyYzRhYzA4MjY4MTA0OGIxNWU0MTc0ODcxOTVmM2QzZDQ1N2FmYzYwM2Q1MDE3N2YwNWUyNjZkOWVkYzQzNjU2Y2NmMjllMjkzMTBlNDNkZDBkZjFiNTVkMDYyMzdkYzM3Zjc2ODcxNDZkNGQ1NDIzMjUyZTgyMjcyYjQ5ZGUxMmVlY2M1YThiNDEzYjZlY2M5ZWIyZjBiOWFhZmEwMmZiNTc1MjMwNjk3MGVkOTg5MDUwYzcwMTgwMDU1NDRmMjNmZmE5Mzc1ZTZhOTQzMzQ2NjNjOTM0YjVkNDk4ZGM0NmVjN2VkMDBmYmE1MTVmYmU5MTZjN2Q3NGM5OWRhMDkyZGMyMjRjZTJkZDkwOGExNWRkYTU3NTE2MDVkNGU5MTk1ZWE2MzdjNGNhZGQyMjQ3NWNhZDRkZWNmODYyMzNkYTBkMGM5ZTgxODA0MDIyMWVmOWIwNjA2NTAyNWM2MjE1ZDAwZDFiMzQ2NjMyY2YxNmY1NjUwZGMwNDM3Nzc1YWM1NWI5ZTc4ZmQ4YTQwZDI5N2Q5NjE2OTBhNGI1OWU3YjE0MTI0NjU2MzM1MDdmODg4ODE2MjJjMDFkMDE2NGU0MWYyY2IyZTQ5NjgwNWIyM2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Sjpm2yx4MVyp3nE55rTz4C7MPkHBUg4GFeIdlBOT92KoiHa6SVdiXafMOt0yTgl57i0USnK6qTiPF84euiNbCA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.231Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik5qU2N0V2Z0RUFseGxZZzRQWlZPRkFQYWwrandkQXNiODA4VGxPNmdCRXEvYzFPRjZ4d0R5dVdNdFBEVG9iNnAzUEdNZW81MDlxS0Q2Um5mbitSSVpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTlfNjdfMjRjNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2I2NmRhNDhkOTdhZWM0MTgyOWIzZDI0N2NhMTNhYjI3NWEzNWRhOTkxYTIzNDZhNjg2ODcwYWVlYWJmZDUzNjk3Nzk4NjJmNjhlMjM0YzcxOTAxM2U5NjA4ZWJjNmFhNTg4NmI0YzFiN2U1Y2ExNWJjZjg2OGQ4MGIwNDU1M2Y2ZWEzNTdkYTZiZTdmYzU2MDcyY2ZjY2VkYTRjNzMxMThiOTFmYjIzYzQ0YThiNmU3N2YyYTZkMDQ2MmQxMjQ4NDZkNWZmYzMwZjVhN2UwMWVkNjllYjkzOGZkMTU3YzFiMmU3ODI5M2ExZjk3MmJhNTgzY2IyMjkyMjk1ZGQ2NDRjZTcwYWJmMDM2YzRlOTI4YTUyMDRjMWY1NDlkOGRjMGFkOTcxNTExYzIwN2MxOWUyNDMwZDU4MjJlMDA2OTQ1ODYwNDA3ZDhmODM0Njk1NDI3MjE5ZjQzNGNjYjA0NDMyNjUzNzlkMThkMTkyMDZmYTRmMzg5NmVjYmQ4MjhhMmZlOWFhYjQ1ZDRiYzhkOWFlZTY2MTEwOWM3MzJkNDNhMjcxNjUwNDA5ZTdkMzRlYWUwMTI3NTA1ZGY4NTVmMzE5MGQwNGI1ZmRmYmNhYTk3OTE4MDZjNjNjYTFkOTE1MjE5ZDMzYTY4MzJmY2RkNWViYWJiNWRiNTljYzcxYWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qWEAoAR-tBoa-ORaS5F-SYf_6xwlAKYSsd8htr1O01LLeNlvJ02jB9j-4h3MylX38JNQ5Tk5fWBE0q8TID2BNQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112959_67_24c6_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.233Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlArN1dWMXJhYnhPLzZVUHNSdG4ya1liNFV1aDF2Nk5RQWszTVR4L0RJOTNVeHZaQXQ1RHFhM0t3WWxOQWkyYVBDQ3R1d2RML0IrWGNkWTNUV2pCdHR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTlfNjdfMjRjNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGNmYWQ1MzA1YzRmMmZjNWQxZTc4OTk4MWIyZDNhMjY0MjU1MzFiY2Y2MzUwZWY1MjMyMzUwZWUzMzcyOGVlZjI0ZTBlMjIwMzg1YjFjMWYwNWY5MGE1ODNhNDE1MjdiNzJiZThmNThiODRkZGM1NWRlNjFjYjRmNTgwNDU2ZWY3OThmZjczYzU4MjMwZGRhZDYyNWNiYWE5YmMxMDRlZjIxMzBkYjA0YTU1YTI3MTg0OWNhNDc0NzBjZTdmZDBlOWEwMjQzOGZjNGVhZDM3MTc1YjNkNjk1YzIyYzI3NjYyMjZmODljMWYwYmZjNGU0MmViOTFjZjMzZmY0N2IzMTQ3MGU1ZGUyMDIwNDM2NDI5ZDQ1MDQyYzY1OTg2ZTA0MDhjM2FmY2MzMWE1NGVmZjhjMWVmNTQ4YmM4YmZlMDM2YzVmODE2ZTEyMTc4MDE4ZmI3MDFiYTc5MDYxYmVhNDFhYjA1N2FmMTk3ZWQ5MzU5M2Q0ZWM1NGUyZGE4YmVhZWZiNmY4MDNhODVlYTZmMjVkYThjM2Q5MTFhOGQ0MTViZGFlZWZlZjUwYjk5MTFjNjM5YzZmMTNjNDljOWQyNTRlNDFhNjQ1NGJiMTU2NTcyYzUwMDZiNjg1ZTYzYzczMjI4MDdiMmQ2N2MzOThkZjYxNDZhZTdlYTkwN2UwMGFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Gwd1VMWrV4AIvkBrNtEth9qS2-0DlpwfjMkXKu61ishQewKvWX9cT4Tmd-uV2_n1Bpuw4zDw0XSuEJnwb8OC9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112959_67_24c6_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.236Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Iks1ekliZ2FKL3ZyM0s5eHJodE5KWVNlcGtueTNodk9OWVVMdlhtZlowcm9RbVpkUCtSK2xHWW42RFlFYTBMNG9GdFVuZ3FxMGJLd3pWM0JLK3hyV1pBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTlfNjdfMjRjNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDJhODRlODk5YTMxOThkYjFlMTMzZDY5NDNlMTI3M2I2YWNjNTdiMWU5YTcxNDhhYTFmMTc5ZjhmODY2Y2VjNzgwNzAzYjhhZjM3M2Y0MGQ1MWU0MjNiY2VkNjNhZTAyNWQ1NTljNjU3M2QwOWNhZmU0MGE5M2RkYzczNWM1MjQ4NDA0YjhkYWVmYzExZjU2MjM1NzgxZTdmODdkYmE2NTEzYzdiMGM5OTMyY2QxMDU4NGE4ZTJlOThhMzYxYzBmNzJkNDk4ODVmM2UxOTg4ZGNhYzJlODgxMjVkZWIzYzIzNmYyYWFmMzA1Y2RiNzA0ZWI5M2E2MWIxY2I1YTkzZDIyM2E5MTBjMmE4NzA5MTlkNmNhOTRhODY0MTk5NTQ4N2UzYWY0YjJjNDNjNDZhZWQwYWQzNTQzZTEyOTJjYzBiNGZlNGEzN2YwM2FlOWIyMTQ1YjdkODlmYjFkNjEzZDMxN2ZhYTE2ZWUwZDg3OThjYWVmNTNjZDRiNjhkMDA3YzE0ZGQzNjA1Y2QzYmI5N2RlYWMwNmIyMTUyODcyY2FmMTVkNDk2YmRhZTdiZmUwMGQ4NTE1ZDZjZTY3YTQ1Mzc4YTA5ODY5NDY4MGZjNzE0ZWI5ZWZmNjRkYzUwYmM5YjcwYjRmMWNlMDY5ODY2MzAxZGE1ZTkxY2ZmODJlYTJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AOu7RK-ARNoYz6VggOeealf62LEpX1Ei89slauwmlQQD_u9nUUC6MJ6FQj6eN3baGrAwoOmOeQypKlLOhK9nUg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112959_67_24c6_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.239Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InZmL2NoMzJzMFZtTmJySkxWbWRwRC9FMFMrRVN1T2lOYXlZS2swemdHQ3MxVVBxVGxTQXJVWTBVZWZaREMrWTZSWEc1STFnRU1mbHI2ODc3OGMwZmlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTlfNjdfMjRjNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQ0MzY1ODMwMWE0MzNlNjJlZTNkZWVhMDJmZWM5ZTAwN2Y3ODI5MmNlOWM5NjRlNTYzODQ3Y2MyYjE2OTJlNjQ0YjUyY2E0NTMwOGZiYjBiMGQ2ZDk1MDQyMTRmMGQ0NjU3YzI2NTk1MDEzZjFjNjNmZjk0Y2I0MWI5YWEzMjU0YWEzYzkwMTcwMzhjZTQyN2Y3M2Q4YzIyYThmYWZjMzc1NDQwZjhhZmI1NDc3OWViMmM4M2UzMDllZDJiODc5ZWE5N2RhMmVmYzEwYTYxYTE5ZjAxMmNhMmNkMmFlMjU0NWFmMTlkZGJhYzVkOGMyYzZkMzgzZjhhYWM3Mzg3N2QwMWMwOGIyN2FiMDliZWJkZjllNDNjMTdiNjUwOTIwYWRhOTg1ZmI5Y2MxNzUwM2Q0NTYzY2NhZGUwYmUzYzI1NmQ5MDQwMDIyOTcxNmYzNDliNDNlZjJlOGMxZjU5MDJkMDRiNjY2N2U5N2UzOWMzZDk3ZGE5MDQ0N2E0YjAwMjIyNjJhOGJiMDJlNzU4MDBiNGI5YjI2NjdlOGVhOGVkMWQ0ZmM4NGQ2NmY2ZmY1NzFkNThiMDc0YjExOTRmMDY0NzY1OTc1ZDhmNGIyOTdhZDg5OGNmMjAwMzdiYjMzMDBkMGY1ZmEwZTYxMjA1ZTU2Y2U5OGYzZDliNTM2MmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.tGzTaP1mI4g1lB2vaije4FQEiOodAm9LUxQ34qnFyr6KO-qLW1Pzex8thzmobeeSiJiDTjdXiJ7z0Syk4uXhNA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112959_67_24c6_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.242Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik1XTXFpSlZXd0poR25pTGNpZVhxSUs2eThEK2pVemVRU1NxVmZBWUx5WmFWR0hNUmE3RWUvelVyeEdmM0lHRExkdHh3cExLWldJdzkxTU5KeWZxQUVBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI5MjdfODBfMjRiMF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTNjNDgxMTU3NDM5NTFkMDBiZmY0ZDY5NzZiMTZhZDc5MDdiYTVjZmI4YjkxYmJmZDRjZjJiM2UzNTNlZDUxYTdjZDQzNDZhODM2NzI1YWM3YTg4YmZmMzBkOTU3YjRhMGUwZWFhY2VjYmU5NzdjNjI3NzQyZTNmNGUxODEzNmIxNjVmODE0ODMzMWU2MTg2NmUxZWY4MTZiOTU2ZjMzMWE2ODgzNTZlNTFkYjFhMjMwY2IwNTE2OTFiZTMyYjI5ZmEwYzhiMDczZWFhMDAwYTNlZTc5MGViZGM2MDMyMTFkYjRkOWYzYTkwMWY1NjRlYjM1MTdhN2JlNTNjY2YxYjVmMWYwN2IyYTBjOWIzZWNmODlhOTEyMTZlYTNkYzEwZTlmN2MyODI5NzgwYTUyNWJiYTFjMjk2ZGY0NjgwODMyZWQxMjY0ZTgzMTJiOWU1Yjc0YzgwMTIzOTI2MmJhMDU2Mzg2ZTY4NzcyNDk5ZjQxYTYwYjQ4OGFiZTY4NGRkNjJhMWZmMTE2OWQzZjNlYjBkMjZjM2E3ZDljOGZlMDlkMzgyZTlhYThmOGMwMjNkMzlhMWVmMWE2ZGQxOWJkODUxNWIwZTU5Yzk1OGU0MThlMDYwNmI1ZDE3NGMxMzM5YzhhYmE0Yjk4NjMxZjVhYmJiMzcyMjVjYThmZGI2MzVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UjKqvbnpHYIUFyLkDIMFEKawm8RGjvlteVdV3DhjxuHx2JlB_wTDggrYNTWCnyzu71LGbbdeoo7Owps6Bij0Vg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102927_80_24b0_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.245Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik9EM3F0S05MeThnenJMcXlHNVp1ZnJqSFhORUQxSUQzT3JoSnlqcXdHaG5rK0tybkRMWUgxK3Ezb1lqVkRqS1haQ0JoY3lsZzhydUpyYmlKTVlCblpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI5MjdfODBfMjRiMF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzM3N2NmYzBhN2UxOWQ3ZWIxZjEwYzAzNmRlZTE2YTY1MmE4ZTc0M2E3YzhkZWVlMWZkMmU4NjMzMzdiYjQyZjMwMmMxZTA3MDgyZjU5MjM0MDViYTVjZjZhMzExNzNmOTIzYTcyMTViNzM3NzdjYjU5NmM4YjU3MjdkZDk4NDU2MmRmZWFlOTg1YjE5MzhmNjA3YzA4NzI0OGZkOGQ1NDJjMWEwZjI2ZmE2MWQ5MDFjODllMzczNjNlYzA3NzY2YTNmY2I0YTlmNmNmYTY1Njk4NDg5MWUzZDNlMjBhNDgwMWFjMjE2YWNmZTRlZGFmMGIxYzdhZGVhMmExN2JhYmQxYzE0ZTkyZjhiMjgzYjhkMTI0ZTEwMjQxZjRmNzNiY2EyMjA1MDAzNGNjYzMwZGFlMTUzNzM0YzY0MDIzNGY1YjcyOGEyNmEwZmIzNTA0ODc2M2RjOTUwMDJhMTM4YzAwZGRhOThkYWE3Y2U0OGEwYWFjYjMyY2MzY2ZiZWU5NzZiNjUwOTAzZmQwNzdkZWE1MGYwZmQ1YThkMjhmNTc2NmY2MGQ5YzRiZWY0NGUwYTlhMjUyNzczNmMzN2U4NzcyYjg3YTZlM2FmYmZmZmEzNDJiOWVjNjE2YTIzMzJkOTkzNTdhNzVlZDlkZDk0YTAwMzFjZTlhOGU5MjllNzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.naBxtOOJS2uI1zqMDVMyXj279vdlmIlN1ohDQakALOUds9YeTrQ-AvbMSxGXHR8qP2EsrJ9R0--mA1AYzhtlnw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102927_80_24b0_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.248Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Inlpa3lzWmVNL1dpTko5VmRxaW1IcmFRbTZLUHBoSzdBRk1iVC94cUpPbGtoaGJ2dkNEUktIbXFtaWthdmdDSEE3UUluWXd4S3hYbUlUNS9wTGR1ZVZ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI5MjdfODBfMjRiMF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjFjNGI4YTdiYmM1Njk0YTY1MTZmM2MyNDdkYWNkOGVkNjViOTQ2YWJlODE3Y2U1YjAyMGJmNmFkNjE2MWI2MTMwNzkxMmYxNTNjMTg5NWNhNjI2OTUyNmY1NDg4NGNmOGY2MTAwZWJiNjEzYTNiZGVhOTQ2MTg2NDNhOTY0ZWI3OTVkZDdmM2Y5N2QyMzFiZjMwYzVlYTM5MGU1OWJjMGU1NjU0NjVkZjMyN2I5MGE0OTczMzMwY2Y1OWZiOTBhZDNjZTYzN2RkYWY3ZGU0Y2Y2M2E1YTkxZTk5ZWE4ODMyZjIzY2Y0YzIzM2NiMDJkN2Q4MjZhZmJjODM4YWQyOTczYTFjYTQyYWUxMmE4OGRiMGZlOTMyY2YyMmIzZWI0NjJhOGVhYjE1ZjBjZTkyOTRmYWM1MjkxOThmNDhkYThhZDFkZjVhYzlmZDFhNzE2Mzk4MjAxZmUzMTM4MzBhNTg2Njc4NTZkMTRjN2FiNDcxYmU2MzFkNTI0MDk3MzU0Y2Y4M2QxNjE1MjdiNDUyMWU1YWM4MjkzYTIwNmJlODA0OWFhYWQwNjAyYTEyNjBhYzVhOTA2MTdiNGRhY2I0NDgyYzc5MWQxMTI0MmUyNmI3ZjhkOTg5MjU3MDIxMzM2OTJhYjY0ODZkNzljMGQwNzY0YmZiMzZhY2QzYjQ5YWJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WnBRpDktaYEQ5cMl07ISPB5SNhNK9Llc5t496-qmZehrni1LEqy2j7RdzPOTvR9q_V7zZlCBLmjKnhGR0W0fgA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102927_80_24b0_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.252Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImgvQ2NaSlVJQ3ZETThvRWtmR2t5U1VyYnEzZXlzU1FuVFM4Zkt5Snc4cWs1R3hpTzNqckV1NHdUcFp1MUVjWFdzdTNncHZUMWRZeUY4NWJWbEFhUWd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQxNV8xMDI5MjdfODBfMjRiMF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmE0NDkxMzIyZjQ2MmM0NDg2NDllMTM1Y2NlY2YyMTM5OWVjZDBkMDdiZjVjZjAxMjE5YzM1NzE1MGU4NTgyN2YyOWI0NWY4OTBmNGU4NjllYTA5MGRiNzRmYzRhZWFmNDNiNTBiYWMzOTk0YmRiNGMzMDk4MjQzNWFlMzFjMTFiZmRmZmVlMGMwY2U2YTY0ODFjNmFlNjg4MWMxOGQ2NDc0M2ZlYTFiYTFmYWE2ZDE2MWUwZTZkMjNhODIxZGE2NWFkZDMxYzNjMDdhODYxMWY5NjEwNTRiZmZhNThkYzI5NTE0NjBmZTlhYzQzZTdmZGI3MmJkOWYwMGM4NjEyMzAxZjk2ZjI4MjhiMmQ4MzQ4NTU5MGZkOTNmMzAzOTE5YTQ0OTI5Mzk0ZDM1ZjlkNGQxNjk5NWI5NGMzMjBmMWM2YjQ4Y2NmZWViMmM5MWJhYzM3YTk3ZTU5YmM2NDU0ODljNjljNDY5NTExZTUzN2E4ODE4N2I2Y2E2M2E3NTVlMzFlNzE3MDExNmI2OTViNGE3YTlhZjg4ZDlhMDY3NjkwOGRhNmU2YTk2N2Y5MTM2ZDliOGVjOWRiZmY5YmU5NDlmMmVmYzVjMDUxYzZmODkxYzQ1NDUxZTVlN2U0Mjc2NDYzNjM1YTYwZDI2NTc0MzI1YTUwZjIyN2JlNmY1ZjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.eBmZGFnwYo_I9ADbpmMkqxMtF66vwMX5yq_IbG8yqDcE6QMHL9TjzOh0_IoKbHzG_OXCqkZ5NgWygUC7NSq2hw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230415_102927_80_24b0_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.255Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikx1WGJSVjV4bHFnK2ErQm5za2NWOVpYc0Mrc0M4YWZFVVdJcnN5YVdGaEtTdzRxOHFQTFZpVE5wMGkzbTNkc2x6dm1mWnhXdk9ERFRkZ2loY3Q4em1nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjVfNDFfMjQyZF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTE5N2UxYTM4Y2YyNDg4YzVkMTA1ZWI4YmRiYTBmMTczYzljMTVjOWZkYTkzMTFkZGE1OTYwNjY0NmIyMmE5MDA2NGQ5NjA4ZjkzNDE4MDU3MjYxOWVmMDVjNWUyNjFmNDQ4OTUwOGMzMmI2MjE2ZmE1ZjY5ZWU2YjNmYjg2NmU2MzgzNWRjYzM1M2Q4MDg5MGNiY2MwNTdlYzIxNGFmZjdkMjkwMmUzNDViZTk1OTExMTZhOGI4Y2Y3YjJjYTlmZjNmYmVlMzZmMjA4MWE0OGM0MDJmNWYyMTNmY2I5OTQ3NzI2MDZiNDQxZGU3N2I0MTgyZmZiYTY3YzNlM2Q1Mzg4MjA1YzMxMjg1ZTY0MGVkOTIwYWVlZGY3MWU3NTZmZDZlZDliZDU5Y2UwZjBlMjRhZjU1NWE1NDYyMzdhYjU5YmJhZDVlMzg0MDVkN2UzNTdhNjkxODYwZWM3MTAxNDBhNGRlNDU5YzBhOTI0MGE1OTMxNzNiYjM0YzZmZGE2OGU4MDg2OTcxYjJhOGFmNDY1YWVlNWNhZjAyZTUyMjE5M2U2ZjUyNTM4MjIxZTcwN2JmOGFjY2UwOWY5MWI1MjU2NjZiODYwYjZmMjQ1MDdiZWQ3YTgxOWRlOGQ0ZmI1NzMwYWM1OTQ1ZmJhYTlmZjA3MDIwNjhkOTQyZTZkZTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Fy9pwnShm8YciZbZIpozbpz-E7UQNecFjVey6rrLWhK3h0IApwrc3iORrbLwrbMjGCu39XflU61MHbBGhkISMg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102725_41_242d_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.258Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjU3QmxWYXgxMUxHQythMmlUSGRjY1hzN3hCL1p0VW1salVnVEhMTGdibzlmZUJhc0h5MG9iNlpTOTZ0ZGI1M1JlamxpR1FBUWd2V0wyUDhkcU1SZC93PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjVfNDFfMjQyZF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjlkZGNmYjEyNjcwN2IzOWJjNDRkNWQ0Mjc1ODJlN2IyYzc1NmE4NWEwYjAxZDQ3YmY1MTg5ZjkzMDkxZmM3Y2MyNWY0NGIxMGJjMmE0NzZlZjAxYmI5NTFjYjkxNGZlOGRlMTI4OGQwNzIwYzM0NGJiNjM0ZDkzNjFkMTc0Njc0ODUyNWI3NWFmZjUzNzMxOTUxZWZiMjVkODU0Nzg2YTVmMTdmMjYyMDJhZWQ3Yjg5ZGRkMDMzOTUyZmIyZmE2MDhhZTJkMWUxNDVlYzU5MTUxMDU1ZDJhMzIzNDEyMjIxY2U2YTIxN2I5YjYzY2YzNDYzNmM0ZTZlZjk4NmE3NWU5ODQwZGUzYzlmNDc4ZWM1NjUyYzYzMTQzOWZmNGI2MjAwNzFlNTYxYzExYjU3MTZjMGFlMDc2MDEyMzc0MTliZmEyYjM4ZGQ0M2I2MTkzZmU2YjE3YTBlYTIxZTMyY2I5NGIzN2E1YTdkMGExMmE3ZmM5NTg1OGY1MTNjNjU0NzFkZmFkY2E2MDBjNWQ0NzI0YjQ5YWRmODllZTdiNzgyOTRjOTdiZTIzMjY1YWQwM2Q2OTJjN2JjNjBhYjljZGRlOTBiYWY3YzRlM2JjZTU0OWQ4MmM1MGMzMTljOTNiN2RlNDFmOTAzZGRlOGU3YWIzYjdiMjJjNjYxNTUzMDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.asD8oW6Mzy0EiKL0tlIXRhlfl5oDwbTELecKwWhHlyxrA84rynwmuDMf5Z4t34-Y4r465-mu6dhAv906pPwiPQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102725_41_242d_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.261Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImRPVzZwdW5oRGNhWHVVWGRzMGZIUUpqS2dhUlUvQzV5Z1Rzdno4eitOQ2xNMVVERlkzOE85K1BsalQ0UllNVHhrVTVncnBIK3U3d0NHT1YxQXBLYy9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjVfNDFfMjQyZF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGRiMDUwMDVkYjY4MDA4NzhjZjNkZTE2YThlYWFiZThkYzgwOTkwODE3ZGY2YjhjN2M1ZjI0NzQ1ZTYxZWU2MDIxYWM1MDJiNzRjYWJhYWQyYzI2MjI5NzBmMDNiZWU2YjdmYWMyZWNjZTJjZjAzNzZiZDYyMTIyOTYwNDM1NzA0NGEzZWQ2YzdlODhlZmJkM2ExZDU0NjBmMjQ3ZGE3YjEzOWJmOGRjMzljZDk3ZTBiZGQ1YmFhMDIzYzcyNjQ4NDUwNjNlODA5MTQ3MTA0MTZlODEwNzk3MmM1MGYyNWE3YzQwMjhjNzc0ZWVkODc5MDIwM2U0NWMwNmY2MzYxNzExNTVmNzViYmM4MWIwNzNiZTE4ZmI1ZjE1OTM3ZjBhMjYyZDNkOTg3YzkwZWQ4YjM5MDg1MTRjOWZlM2E1M2IzMzk4ZjBiZmZlYjg3YmUyOTVhOTVhMmE4MjY3YjIxZjZmNDI2ZDAxM2MxNmM2ZDgxMzRiN2Y2ZjMxOGNjZjQxMjcwZjg5MDBmMmFhODlkOGY2ZTgyN2E5NGY4MzA1ZGYxZjQzYjE2ZjhkYzg0MmNlMDk2OGZhOWM2YThmNWNjZmRkNGJiMTBjYjc0NzU2MTI0NzgxMzA4NWYzOTJjMDFmYTgwZTBhNDYxYzMyMzliNThlYTU3NTM1MzUwZWIxODZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.azdDW1jhn3Bc5tf850y95QLGjnUL2yZ66p8OTnr-MRAYsajgqLzj7-Xgi5y6pTnGHRMMywW1VT2UUNK-_C_sAA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102725_41_242d_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.264Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjFZSUptcVZLVFZlSzdjL2g3cDdiMmJTQnJzd09tdWlub245YnNGNk5jRlNrS3FYOHdOQ0ZXK2VUZnZlR2NQYnRRRStic1hHZlZNVFUzZE90RjdIQkJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDMxMF8xMDI3MjVfNDFfMjQyZF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTBiZmM5NzQ2YzQyN2I2YTMwNGM1YTA2MjhlYTAzZWEyZWI4YjMxY2NjMzViMzVkMWJkNDYyMmFiZGIxMDdmZjI1ZDQyOWIyYjJjMjdmMDRiMmUwODc3MWJlYjkzN2VlMWZiNmNhMmI1MTIyMTM3N2ZhN2FiYzVlMjdjNTUwNDRjOTBiMjNkYmM2NGE4OGRkZGNjZWEzYjA5ZjA3NGIwNGM4Zjg0YmI2MDdmMzhmMjFmNTMyOGI2MjBmMWFkZTllMzQzNGRlNWFmNzY4NmY4NTY2YjlhNDMxOTI0ZGNiODhiM2QzYWRlNWJhNjZkOWY3YzVjYjIwODQ4YzM1OGU3YzAwN2I0MDk3YjU1ZThhYTY0MTU2ZmViYjA3ZTJlNTVhYjgwNmM1NDYxMzBlNTY5YzFhNGUzZjc2ZjFlNDMyMzE3ZjcyNDE3Y2Q1ZDc2YWM3ZDBlY2JlODA3ZTU5NmE3Yzc2MjU1ZjZmMDgzNTlkOTA1NWYxZDIwYTUzYzJmYTkzZTA0M2I4M2ExYjA4ZTVjZjk4NmZlMjdkMGM3MTVjNTEwMmY3MDY3MjAyMGQwYjE0ODA4NWIxODI4YzI3ODA0NDMwMDQ1Yjc1ZDA2NGY4MjI4M2QzMGEyNTQ0N2EyNzM5NTViMjg4N2ZlMjMxN2YwMDkwNjZmMmIxYTNlYjdlYjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OKFatPDlvn6BeOv32mZdyKzxH-S1zxIxzrmuttATkld5c_R9DgdtDK6q3N6LIR00oybMYJ0rwSJ_DQw-I8021Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230310_102725_41_242d_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.266Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlNPbW9PaVlocVZGSGluWG9RYjE4bTNDK2taS29ISWxNYlRrcEpPOU9ZWjFpdVkxeXRJOWJlcUFEQTgyemtXQjNPdVZNTjNGbEg4VGhQRHhnL0VlQ3h3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDFfMDFfMjQzOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODUxMTkxN2RmNTc1YjQ0ZTZlYzQ3OTk5MDA4MWNjZDljOWNmNGVmNDBlZDFjM2U2ZmUxNGMwOTc2NmY0M2Q2ZDdlZmFlMzhlMmZkNzc0YTRjMDIwYzA4ZjliM2M2YTAxNTU0NTUxMGRhYWZmYzIxZDNiNDVjNzBjYTY0ZjI1N2IzYWJiYTVmYjA0NWM1NDBmMDBiMTBhZmRiZGQ0ZmRmYTFkZDJhNWRmZDFiZTNlZDFiZTE2MmQyZWQ0OWVkOWIwM2Q0NTI3NWU1OWRjYTMzNGMxOTU5MjAyODcwMGQ3MDUwNzM2MjNlZjhiMmIzNWRhOTlhNmRlNjk3M2YzYThjMDk5NzhkMzk2NTZlMTQ1Y2Y5MGFmZTkyMWU4MzNmZjI0YTJiMDg4MjY4MmIxZGE0NTAyZTkxNmFmN2RhOGYzYTUyY2JmODI0YTYyYzhhYjRlY2Q1NDk5ODdhNWRmMzNkZGIyMGRjYWJiNzI2ZmIwY2ZkODYxZTRhMzY1OTMwN2FlNGRmMTk5NjdmYzAxZjM3YTgxZDM2MzFmZjM5NjU2ZTE1MzAyMDIzNjc0YjQ4NDMxNDE2N2UwZDcwYTBiZGMxOGY1OTgyOTkzN2Y1NzI4NGQ0MzY1NzIxZGQ2NWE2OGQ1N2JiOWZkM2Y5MzEzOWE5YzdiYmFiMGZjMzg1MzVkZmZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JKMGgLcoWdyrbLH1KzhTnoSU-Sc4L679_c8HaaH3VTskiPkeCEHZh-S03a1xSZzkH3UeOI7jakIJmC7YDPVFjw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103341_01_2439_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.269Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFNRGZwdHVodC9lRzUyTGEyYzJxOWc1NytPZTNPdjA2eTlndTh1ZjJZdE1DYmJVMVMrWVVHNytsL2dnT0owTjVpOUU2ei9mZVlCNms3bnFJS2tiOHlnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDFfMDFfMjQzOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGUyZTE3YmE1NTU0OGFmNGM5NWE0YjAzMzNhOGQzYmFlMmQ2ODZjOTRmMTA2YjFkNjJjMGQ1ZjMyM2M1ZWZlYjYyMmJkMjVlNTE2MzE5YmIxOGQ5ZjI0OGQ2ZWFmM2UyYzFmY2U2NTlkNjg5NzdkNDlkNGQxM2NmYThkNThmMWI0OTExOWQ3YTc1YjQ1NTkxODk2MGQxYTZkMDU2ZmJhNTcyNDg1N2I5ZDMyZThlOGI4NjU3Y2Y2M2FjYzhmNDkzMWNjMzA4ODgwNzc0NzViNTcxMTFkYTE2ZmRjZTI2MDY0OTI4M2YyZDhhM2Q4Y2ViNmY4YmJmMzE3NDVmNTVjZDQ4YjJjM2NlZDJkM2Q4N2RhYTBmYjJmMmQyMzQ3YmRkMDQ2ZjBhNTQ4ZWRjOTZjNGFjMjE0MWFjNmVkYTI0OTczZmFkMGYzY2I1M2M2YTZhZDExZmIzYTM4MDZjMTExNDQwZmYzYThiY2JjYTQ2N2M0NjUwNTEzOTg5ZGYxODFmOTRhNmE0YzYxMTI1ZjJlZjdmNGIyMzNjNjFhOGNhYjhkM2QzYjAxOTZmMzdhYjRjYjE3Y2E1MTExNThhNTI2MDk5NWQyY2Y3ZTQ2Nzk5YzMxMmNiNTE5YWVmNGE5NDFkNzE4ZGI0ZmQyN2QxYTljNmViYjQxMDI5YjE4Y2U2MzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hdmsC9vRiChEeJdzqdhg_jjF74Hap5ZdhZ5M0lb5cF4WJ0-FfJg6ObxSjgakumDwbje6bDncZRhstf1lPsQSIA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103341_01_2439_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.272Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkUzdWxwUEx3cFh2Z2ZoYWpuRVF6L3JFa1N4dm0wY28vYXFVNXNsTWJQWjFDZkw3Q0k5ajVLNE94YVAyTkYycDRQUmFGM3VKdmkrUXB1alowZkNyd05BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDFfMDFfMjQzOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjA1MWQ3NjRiMWZkY2UyMDExM2JhM2QyZmJmM2FhNGZjZmY4MzI0OTQ5ODVjN2JiOGMwMmQ3MDQ2MzU1ZDljNTc2Mjc4NjM2M2Y3MGVkYTllMmU5MTQ1ZDRkOGUwYTJkNTk5NGVkYzM5NGM3MmI4Y2Y1ZTg4NjYzZjEzYzJlYzE0M2RmMzQzN2Y0YTUxZThhZDNkM2ZiY2NjMzE1ZDhlM2RiOGYyN2Q4ZmJlZTM2Y2ExNWQ2OTMyY2RmYjUwZGMwNjM2OWE3MWM2Njc2MzJhZjY1ZWFhMGNmZmViNTVkMWRjNTViMTc3YTVkNjE3YjNlOTEzMGQyZGI5Y2FkN2Y2NjZhMDA3OTFjNTZiYzhlZTc0MWY1YjIyMmM3NjBiZjEwYWY1ZTgyZmYxMmRmMmFjYTgxNWRlNjRhMmZjYzBiNGFmMTgwYmY5NGM5ZmI0MjVmYzQyNTM5MzY4ZjdiYWZjN2M4YTQzN2I2YjEyNTUyNjYxNmRjOTg0NWE0YjJiNDRjMDlhNTQ2MWYxYWVlM2JmZDc0MzZmZDZkZWFmOTA1YWRjNjdkYTU3NjgyMmQ5OGMzNGE1YTgxMTExODhlMDc0MDQ2ODY5MDZmMDc3YjEyOGFmYzgzMTJhYzBhZTlkY2ExYjE2NWNjOTE1OThhZmZkNDYyMGFmYWE2ZWZlZTg2ODJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NXWxUgdKuLaLvQtIh279YA6OiVjpq5yaH9j5w8s2qGSmrEW6T2sr2fDM2ZEW8AEoi2uD-4bVkMFLVNXhB45_hQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103341_01_2439_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.275Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlVCdWlpSlBSSVNVN096bDlwVTlzRTM2WmdHNWxnRGtWUnp6dEdUdlZZOTA1Ung4QWVyOEg5SDYvNHdRaDJyZU9FWUJMbjJhblJ5YlhBV3VtSW5XMm5nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTEwNV8xMDMzNDFfMDFfMjQzOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDhmODk0ZDYzNDEyZDFjYmE5MGU5ZmQ1ZmE0YTViNDJkZTIwMThlMjMwMWQ0MGMzYmU3NDk0ZGFmODA3OGRhMTQ3ODhiNjQ5MzNiOWU0NmNhMDI0NThkOGJjZWY1ZmVhZWQ4ZDE4NzQ2Y2NhYjg1MGE1OTMzMWYwMzQzMWNkOTZhM2JjY2E0NmU2MWIzZjIxNDJjYjgxY2MzYTdhZWRjYmNjNGQxMWU2ODYyMjc1ZTVmZDk3Zjc2ZTlhNDUzMDdmODAxZGM3OTg5MTIyNzVmNjg3YjRiYWZiOTAxMDZjOWMyMzYzNTg5ZWM3ZjU1YWI5NzdiOWQ3ZTM3NjE5OTc5NTk4ZmYzMTdmNTQzOTNjNGIwMTFiMjFiNzI4ZGZhYjIxYmE0OTc2OGNmNDI4MWJkMjdjZDk5YzNlNmZkY2UzM2IwYzIwMWU3ZGQxMzdhOGU2OGJjNmNmYjk3OTJlOWQyZGYyZWZhMDk5ODM5MTkyMTkwN2RjNDQ5OTc1ZjhmNTMxZDg5MWE1ZDQwYTVlZTFjM2Y3MDA4OGFlNTZlNTRjNGIyYTc4M2YzOGFiYWU4YmU1ZTkyNDBmYjA4ODhmOTc1YTEyMTU3Y2M0NjdlNDU4YWNlYjBmYzhjZjI1YzNiOGNkZjlkOTZiYTU4MmRiNDg4ZDlmNDZjMzFiZWVkMTQyMzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WLcNKFtzSi8_5o9gRklx10T5_3eaBSAddrh4UAYWhEKnGwbnHdrAQ2rEehKi9VwrXoCcOqmTe5yQKfC2Fk9Daw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231105_103341_01_2439_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.278Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImlwdmNhTmhkcUtHeXBJSnNCcWFCM0Y3WFV4UjBMcjdjeDdvV3lWTUdrUTRNQkFFMjdYNHpqYmRyY04wR1NyTnpreG45VTcrMjJOdllvKzkydC9yMXdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTdfMDNfMjQ1M19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjBjOWMzYWZhOTgwNDFiODUzODQyZTVmNTA0N2VkNGUyZWM4ZTIxMjJlNGVkMmMzMmVkNDAxODQ4NzZiYWI4MzA3MDZhOTE1MDMzMmMwYTUzYTYyYmFlNDRiMmY3MzZjNzEwOGEyZDRmNDE1MDc3ODZjOGMyOTkyMThlZmVkYzdhNTg4ZTEzNWUzNjBmYjcwNWJmOTgwNjkxYWUxMjIwNDM1Yzk1YzBhYTZiYzM2NzdmMThmN2VjMzAzMjY3YjJkZGFlZjRkMGY2MmQwOTk1ODNkOWFkMmYwMGY3MWQ2MjNhMGNhMzI3YTVlMDMyZjA4NDY3ZWEwMzBiOWVlOTlhOGNmOThmNGQ3OWJkYWZjYjNkZTU1MWZjYzMyNDk1NGNmMDQxYzJjZjhkYjE0NzJhZjFlODQyODRmZGRiOTgzODBkOGRlN2NiMWExMWU0ZjlhZjYwMzhjZmM4ZmI2NWUxZTc0N2Y4Y2ZlZGZkOWM3ZTI3ZTE4YzA1OWNlY2U1ZGFlYjJmOTZhZWVlYTE4NjlmYjNjODM1MTVkYTYxZmFjZGViNGZlYTcyYjNmZGRkYjczMDhkMDZiODYzZTA4YWU1MzBiMjEzZWQyM2U3OTY5YzA2YTNkNzFkMTZiMmU1YjhmNjliOTVkZWJkYWViZTc4MjQ0ZWIwY2M4ZjA3YmViODRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.wMOfIRRCQCW8YWjOALAl7j-s-AA6oItdDYXfxrzVLOifHylNCqM0E-lwP4tobOq5EJsRhUPQv6qV6PggfsTs4A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102657_03_2453_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.281Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjI2L3hqYzZOL1pGb2VJZEtQR295a2Q0c3ZsK3dLK0lySG9JVUNobEJ5SmMxR0VaQkxleDdTNm15OEF6R2FnS2FJdnozVHRhcmVURG9wQ3N3YW1kakNRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTdfMDNfMjQ1M18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTFmZjM5M2Q5YzFhZDZhM2U3MDQ5YjBhNWJmNmQzODEwMWEzNDI1YjZkMTgwMjY3MWMyZWRlZGI3Nzc5MDBlMDI0ZTNmMmQ3YWMxZjQxMmI1OTdjOTE2NDc3OTQ0Y2UxM2RlODE5YjRkMjRlYmU3YmQ1Yjg3Y2I3YTMzNzI4YzI5YzA4NmYyYzdhYTcyOWM0MzJjYTVjYjI1MjlmZjQ3ZmI1N2Q0MmFlMzNhZjM3YjQ3ODI2Y2M4N2MzMWE1NzY5YzFhODhlM2ZjZDdjNmE0NjA3ZjI1OTgzODBjMTBiZTk4YmNjYTE5YzI4OTg0YzBhOTEyMzAyYTNiYTI2MTM1NWMwYzIyOWIzNDBkNjAyMjViYjRjYWNhOTlmZTU3YTA1YTk2YTNhZmM5ZTQ0ODJkMWIyZmZiMWI1YzVhMjFlNTJiNTU2NWE0OGU4MDdiMTRjNDI3ZDNiZDkxOWY2NGU1NGM4NjM0NTgyZjU1OTNkNmE3YTUxODAyNjBhNWFkNDJiM2JmNmUwZmE4NWYxMzNlMDJkODg1ZGRiNmM3MzFkNTNiODVlY2Y3MjZmZWE1YjA0NWEyOGY5OWQ5NTEyM2RhMjZjM2JlMWY1YWRkZTFmNGJiZmJiNzlmMDMzYzdmZDM3NWU0YzYwODU5OTQ2YTNhMGVjZWRjODdjYzgwMzQ3MWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TC09RYpE78QrP5DPXZQl-3QEXzDvXjNgvJWLJ19V1HBrjWd_ygKD3l9JPk9rshvtJTGI43OrYuuM47_BKBkiXw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102657_03_2453_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.283Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRSQlJNMTJlTkVxdGxHc2NjNTFTNEp0SG1ITzNrNUR6UWt6ZEJ1VWIrbElNN1ZsZDJZb3MySXFyd205STdEQklVT2pEY2VMYVRsMFVSR2NFSFVYdnl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTdfMDNfMjQ1M18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjRhZjAxZjk1NTg0NWQ4YjVjZGVkYmEzNDlkYzI0YTM2YWE1NWIyYzY2Yjg3YzM3YWM4Nzg0NTQ1OGJhN2NlOWY5YThkMjBmYjQzZTgyYThmOTMyNjY4ZmVjMzgyZWMwZTNjYWRiNGZjNTJkMjU4ZTg5MjNjZjA5MjkxZDAwNDE4ZWU0MTZkMjAyMmNhMTM5N2ZhMDViYmZiYmQ3MDQ1Nzk4YmFhZjI5ZTFjOGY0ZWQyNGQ4ZjNmM2IzNjJhN2VhMGNjMjUxOTJkNDQ5MGM1YjQ1YjA0OGJiNWIxYmFmY2NmYTEzN2Y3Yjc2NjdkYTQyMzRjNjQyMjdmYWM1OTVhNGRjMWY5ZTVjNDFhZDE4ZWYyMDU0YjA2MmY5ODJjZjQwNDcxMTRjZWFkYThjZDIyNTE5YWE2N2M0MWQ2MjExODYyNDMyYjZhZmVkYTcyNTk4ODQ1YTZkMzhmZjZhN2ExN2VkZDZiNzU3MWFlYTZhYTY3M2VlNjA1NjI4NDVlNTdiMjRjN2U0M2RmMzc5MWRkY2Q1NGNkNTg1ZmU0MTI2YWNmNGQ1NjE1M2UyZDViNGIwOTFlZGExNTg1N2M5NTkwZDQ3ZjMyZmIwMDQ2YWQ0NjVjMWM4ZDcyMmUzOGI4YjllNGZkY2U0MmIyZjYxN2E2NGE3OTdhYzU4M2M3MDliZTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7o_bZzdeA8OAXeDukTuEWqEfUOxerJjT_acRYpI93g_hGaCdvVqj8YL4bQv7vPDu0kI5NO6w3Y6EWzQ93GrHfQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102657_03_2453_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.286Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImV6T3dXZ0NYZVhheVYrb1JreG8yYyt1K2ZkdzZZMGtpUmVPMk5vU0lpNFMxeHRWRjNJUVBkdW9USUcwbmM2bi9MbmVGRk41c3NRMC8wRW1hbjhBT3dRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQyM18xMDI2NTdfMDNfMjQ1M18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTQ3OTIwMTAzODQwNWY4N2Q4MGI0OWQ5Yjg2MTM4NTc0NWE5YmVjODRjMjBiN2JkZTFmOTE1MzgzODUyZDk5ODY1NWIwNGY2MTBkY2NmY2IzZDM4ZGVkNGQyNWY0OTU4NmU2ZGZmOGE2YjYxYTFhMzdjZjA0ZWIzOGFhNjRjNzVmMmQxYmNiMWZkZGM5MDQyYmNjYzJiNGRjYTU2ZWFlYjZkYjk1ZDZmOTFmODY0NGViNDVmYzRiZjgyOTIxMjU3MWQ4NjNiZmVlYTBhZDQ4OWY1YTdhMDYzODZjMDVlZDhmZmFkMDJhMjY1ZTg1OGY3ODVlMjgwOGI5YjQ2M2E4OTUyZDFlMDk1ZTZlZmUxYjRhMDg3Y2E2ZTdlMGU4OGRjNjZlYWE2MGQ0OTExODE5MmFjYjI1NWJmNDJiYTAwNDgxN2M2MjJlNWFiMzU3YjRiNzU4M2M2NzExOTZhOGUxNzkwODE1OTA4ZDliYjc1OTQwOTcwMzJmZjM0NDE4ZDc4ZjQyOTEyMDBkNDQ2NjA2OTk1ZjhkOTJiNmFhMDdiZDA2ZjA2ZWEzYTRhNmFlZGNlYjg5OTllN2QxN2RkOWM0OGM5Zjk2NzYwOWMxMjYxYzIyNjYxZmNkODUxNDRiYTE1ZTE2ZWU2YjYzZTAwYjNhOGNhYjFkNzA4M2Y4ODdhOGVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CI9YGiOy8NcO9VBpMZDkpHB10tEE8-TxymeU_7hEPlbfzfpbfs95XHhOl6z7zCZ5YQI9xin6ocCsg4UwaZV0Fg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220423_102657_03_2453_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.289Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImNrdXVuc21CNlRqSDNtbGFCYnRqelk0R08wUEtUNkRVWHBtN25nWC9aNldrZXhBRzc2TUZNanJHeWovclZ4NWU2MXlROEFscjZQdm5aaFRGWjNLSGlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTFfNDhfMjI3Nl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGNmYzU1ZDUxZTUyYWM3Mzk3OTQ1NjFkZDA1N2JkZmNlMjc5YzdkMDhlMWIyMjY3NTBlMDFlZWZkNjAyMDdiODZmZmUwZGQyZmIyOGUxZmNjMDYxOGQ5OWFkYTQ5ZWFjODlhYmRjN2RmNTI0YTdhOWY4MmU5ZDNkYTYxYzg1NTY5Njc4NjJjMzdkYTliYmE0MDNiNjYxODMyNjlhYTE0ZDBlYzBlODU5YWJmN2VlYzk0NTk1MmViNTcxZTgwMjUxYjg5MDYyNjcyMDgyMTBhY2Y4MzIyZTljMGM0MGFkY2ZkZTA5ZDg1NTQ5Y2UxYThhZjVjODQ3ZDJkZjY3NzQxZjZlMmY2Yjk3NzgxYzliY2RhOTNlNmFjY2IwNDlhODQyZDNhZDA2ZDJjMzg1ZmU5MzQ3ODQwMDc4ZTkxM2ZjMGRjOWFjNzkwODA0MDAxYWRjNTdiNjA2OTc1MjIzNzA3MzU0ZDQwOGI4NTQ1NzNlNGY2YmUyYWU4OTM0OGY1OTc2MGM3MzdlNzY5OTk1YThkYThjMWEwZWZlYzE5ZTM3MjhhMTM4ZDAzMjYxMWFkMWEwYjlhNDc1NTExNDIxZGQxZDkwMzgyNWYzN2JmNzY3ZTQyMWQyMjRhNzMxNTNhOTRhNjllMDY5Y2IwNzc5MTRmMGViMzcxYjk1MjM0NWVhMjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ey9GmKEF_KJjwhOs0Qrw3bp1L6QRJdj5WzP1e3K1Vurij7O5_QoWA8HLpZ4Xy9tKxSyxieZEJFGSlFi-_RtLeA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104711_48_2276_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.291Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im5jRE9tZ2FDaGlSMU9MNlN2b0xCWmVnVXRlbUpMbFppdnIxekw2VE81bGlQMTZwRGFwaEpZb3FYVE5oeXdWNjYyL0V3OElvZUpVYm53cmNJYTlWZU9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTFfNDhfMjI3Nl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjBiNTU0YzUxN2VkMDM5MjgwY2QyODQ0ZDY1ZDNjMDFmN2JlMTY4MWI3OTI3ZTYwYzdmMzBlMDAxYzEwNzcyMTQ2ZjUzOWJmM2U5MWNkOTk3NjMyNWQ0ZDUzNGM5NmRmOGEyOTg4NWY0MzI5YTY4YWZlNWE0MmYxZTZlNWU4MjJkYTM3ZmZhYTRmMWE5NWJhOGY1MzFkYWJjMjY2OTI3YzZjZjlkYjE1NTgwYTVjYTI3MTdmZDM0ZTZkNjk0ZjQxZTc5NjE0ZmFlMTgzZWRkZjMyOGFhZDZiNDM4ZDFhNmVmNzZjMzUxNmU4MDlmMDVmOTIwOGZjMTAzYTI1Y2FhNmIxMWU5ZTM3OWE2ZDAxNTI2YzY1NzY5MjZkNTQzMTc0NWNkZjAxY2VlMzRlYWFiOTUxNzBjODA4OWM2MmE4NjljNTgzNjc3ODk2Zjc2MDliYTI4OWZjNGJhMzU1NDFiNDJkNTBlM2YyYTE5YzU0MzUyNTE0NmMwN2Q2MjkxN2I1YmY5MDQyMGFmZTI5YWFhYzU4ZTJlYmU1MzA5YjY4MzAxZDRkMzhjOTBiZWFiZGZlYTFmZGY0ZTRhYjBiODJlMzQzMDU3MmFmM2NjMjgwOGJhOWNlOTk3ZTVmNmQ1N2FjMTk2YWM1MzU1MmE5YjllNjc1NzFiZTNiYTVmZTYxOTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HsMNBRFSplogOnvSFgxKqJQMqwH5WKYa7OP6pptfOEv67B-kYOS41ePKsDAm7E_v7Sez11dI69gCbLM8a8FB-w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104711_48_2276_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.294Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkxySFVZYTJrYUlENkpQc0ZqS2xwTlY2K3JlLzZnNldYTWZTVUVSSC8wZDVISmlQV25CNGdMdndrKytQYlNzZlhlejFvRklNN29Bdm41THBhOWt1WFh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTFfNDhfMjI3Nl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTU1OTRmMjMwNzdlYTlhYTMyM2U2MDJiZWYwOGNhMDBkM2YxYTlhNDIwMWIyMDZmNGViOGJlZjUyYjliMmNmMzYxMTM0N2QyOGQyOTE5NTA0NWI4MjI1MmQ1ZTZlMTA0OWRkZDM1ZjY0OTExYmZjOTZmMGI2ZmJjYjRmMGZlNzE5MDcyZTg1ZjM3ZGMwNjhkNDg5YWQ2MzhlNTI2NTRkN2UzMTIxMDUxZmRkYTRkNzMwNTg5MDY1ZDM5NjU3YTY2MGIxNGQwM2ZlMDVlMzU2YzRjZjRiZTA3ZThiYjJkYjE2MzA0ZGQ4MjUwNzFkZWNjNDk3YjE0ZmRhNWZkNTVmZjBiOWM5MTY5NTMxMWZlMDJkMzBhZGU3ODlmYzljYWY4NTE2MjBmZTMzMjFjOTJkZjA0Y2FkZWE4MzgzMDVmMDY5NjgyMWY2OWQ4ZmRhMmFiYzMzZWE2MTUzMTYyNTQ2MGZmYWQ2M2ZhNTBlMGJmNDhkNzYzNDE4M2Y4OTlmMzc2OThhODUzMjkwNTE4ZTM2YTY3OTE3NjAzNGZhNWQ0YzIzNjI0YjFiODhkMGRlMTA1NjUxOTA1ZTNlNzcyYjBmYjI0MTdjMDNkZDEyMjNjOWRlZjk1NWI2NGUzOTc5NDdlMGRhOWJkM2JiYWZhYzZkMDQ4OTAzOTY1MDFmMDhiNmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EaGTKIe5uwlk8jYLH03wbd7srJpPigcUuDJZ15nTrDI-6WYyBfrUY1b9G8eaUlKrLaAemo9p4KPbfQsusmaVRQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104711_48_2276_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.297Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRQSm0yKzBiYnZTMkZLNkRRR0xiWjAxOXpRTUQzQ3RZQ29jazRNaXBCSVJ4ZFVHcVlaYjdORHhaaVRLTVhuWjZZTit1MjR0R2MrNk1wUXRWWlNrRFBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDQxOV8xMDQ3MTFfNDhfMjI3Nl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjIyMDg0NjczNDRhZjMwODE2MmJiYWZiODJmZmY5MGQ0YmU3MmVkNWJiMzdiMzcwMjBiN2EzNDk4YmFiOTRjOTc0YTkzZGFkNjQxZDhhMDJkNTllZTkyNDY4NzhjYTU2ZDJhNjgxNjY1MzIwNWFkMTcwNTI3MTFlNjA4MTM5N2M5ODMxMjM0N2NkYmFmODhmMTRhZWQxNDQwZTc2ODE0MmQyMjM1MmVkNWVkYTlmZTkyYjFjMGQxZGZjZDQxOWI5ZDAwYTQyYTg2Y2MyMmRhNzQzZjliZGFlM2RiMjkxMDkwM2RlNzE5MWE0M2UxZGMxMWE1ZGU2ODI1MDU1Yjc2NTJhZjFlZWI5MTQ5ZDgzNjdkZDlkYWM2NjlhZGEyODViM2NiOTAzMGQwNjUyYTdjZDAzNzM1N2E2NDZmYWY4NjhjMzJkNDBjMmIwYTIyZjUzM2Q2MzM5YjQ4MDhhMzJmM2ZmYjZmMWMzZWM5ZWFiYzJhMGI4MzM0M2ZmNzM3ODE2Mzk2MzBiZGI4Y2UyMWM5NWU3ZTE0MDI3YTQyZGNjZWViNjljODUyZTkwMjRjOTM3NWU0ZGIyZjMyYTg1NmUzMzgwYTA2MjVkZWE4NmQzZGQ3ZTgyZDhjMDk5Mzc4ODAyZjczZDhiNmVmYmMzNjg0MzRkNWExNGY2MDcyOTg4MmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nXK2BSFsAx9-vPS_oIOgBnVuO8DtROLNFPnwXig-AhR3l3OBniX3jwGHgaO7sP23SwYGpdHMCxpZd6S_vEuVqQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220419_104711_48_2276_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.300Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik91cnNnVEc2VDZWTUlqWjZ6RytKdm0rZUJyU2RkQzh1dmdzejZVVFVXZFM2NmRFbUhWbGdudWgrK3VYUGZtZEE1bEFnY3V1b2dXZ3VjZjgwNXBhQUd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjNfNTZfMjQyN19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjA0ZjEwY2NkNmNjOWQ5NjRjNWUwYzYwMjlmMWQ2NWJkNGYzNWU4MjljZmQ0NWM0OTBmYjQxNTI2ZDcwYmNmNDQyMjU3YThhYWI5YTdmYmNmNGQ3MTg4OGRhNzNiMGZlY2M4NGJiNDIyYzFkNDc3MjAzMGNjZTBlMmJjNzQ4NjRlOGZhODhiM2ViNGUzNGUyMjJkMjdjOTBlMmM3OWZjOTQ3NzExOGM0MTA5N2FkMWYzZjBiZjdhMjdjOWFjNDk3YTFmNjYwODk1NTRjOTg5MjU5ZGUxOTBkMzk0NmNmMmIxNTc3MDBiOTAyYzA5NTA0NDE1MzUyOTZjODVlN2RjOGExMDE5NjNiYTkwM2I5ZTE1MTgyODIzMzIyMjQ3OGNjMWEzOTdlZGRmNmNiNWQwYWE2OGY4ZmI5NGNlZWU2ZTk4YjY1YWM1ZGMxMjljODFkYmMxOGEwM2RkMzMyNzcxM2JiNmVlYjBkZjZiZjliZmJkNjY1ZGIwNjZkMzAyNWMyZWQyNTM3NjE3NDBjYTFjZDJlMjhlYWQ1Yzc4ZDE5MGJmZThiMmZhM2E2OGExYzgxNjJkMGViYmJmZGFkMDliOTMwNWJhMDY2YWE3NzA2NDZkZjBmNDA3NzlkOGM4ZTI5ZGQ4ZDhkZDVmOTkzN2RiZjAwYzVjMjZkMzRmNTJhNTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.9QIyQow8xFP2q7vrnGLkrV8vAfzjUvKXqptbr660ySjPpQsoCBVv5AvG7upG78nKivAvysvL-dgKAq9XKDajdQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102823_56_2427_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.303Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImdDL2hMdWQ0WnZQbWc3dlJqRkRxQXpUckJvaTl4ekNtZFc3bDRLNmxxSFE5bFNkZGJSQVpGaE81MVNhOXFrQjdrYWJSNHA1ajIwSm5HQjBWSVFYdE9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjNfNTZfMjQyN18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDUzNmRmMjJiM2NjMjEwZGIwYzU3MTg4YjgzNDc0MjA2N2E4M2UzYjQ0YTYwNDdkYzBkNGNlODA2YzY5MDBkZDBhMGJiOTcwN2QwOTRkM2ZjYzEzMWM0N2Q5MjcwYjc1OTVmY2MxMjZkZGNkZjhhMjcyNjlmODU1MGE4MTFjMWZhOTY3MDdmZmNiNTlhZmU3ZWQwOTMwMzY2MGI2YTVhODExYmQ5YmZkYTY3ZjgwN2U5MDQ1NjcxZGY0MGMzMDcwOWVkZjQzNTU2NjllM2YyNGI1MWFkNmRjODY1YzZmNWY5NmU0ZDllNDlhYzUwNTcxNTcxZDVkZjVmMmJmMDg1ZTBhZjdiODFkMWQ0Y2IxYjBiNTVhM2YzMGZjOWY4YzkzNTc4YjZmY2RjMTNhNTMwZDg2Y2MxNzg2YzI2N2UwMTZmYzI1MDYzMjFlZmVmN2MxYmU2OGYwN2IxYzAxMjgzNTcyM2ExYTQ3YWUzMmU2OWJlOWM2NjViYmVhMmI2YjM3ODQ3ZGZlMzY4Y2Q2MTE5OGU1MmY5ZjBmMWY1NzgyMDM5YTk1MTNhMWJiNTVkMzg0NmI1YmJmZTYyZmNiYmYwMmIyNWM5NzhmMmMxNjA3N2ZlNWVlY2QwMGI0M2NlYjQzMjAwMDQ3ZGM1MGExMDJhNzVlZTg2M2FiZTBiNGE4MzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3DEEtV3ZHWSWxY1sEe1ZKQV91ZiXMdY3avhmHx13mSAYO-M9iSJozB992ED-qmooTzhlkCCL39wzW0gwn4pi_w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102823_56_2427_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.307Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjVlNlVjcTcxc25nd3pTTHNTQTNLQzRwQmxGUU1VTXE2NjJKaytTeVBxQlJxamNWWjVDT3BpUWoxVy8rYmxTUGRsdERGYzJFNkk4NElON1BlcjQ4aGZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjNfNTZfMjQyN18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODZjNWMyMWM3MjAzYjZjMTFhZjUzZmQyNDQzMDkxZGExNGMxMDY5NWRmNTBjMWExYTc1Mjk3ZGY4ZjMwZGU3NzZmZTEzODUzNzA2NzdmMDA5M2I5N2E0MmFiODExZGUwNDBkOTI3OTgzNjNlNGVlNmMzZWJlMmQ2YTZhOTVkOWQyMDIxOTQ2OWI0ZThiY2E1OTJhZGQ4ZGIwNzI4NzUxNDI3ZWEzMDgyZmQxMDZkNDlhM2Y3ODhhNzU1ZDBmNzJlMWY2YzE1MTc4NDg4ZDE1OTcwODJjMzYzY2YwMTkyYzA4ZDM1ZjBiZGUyZTA2NDM0MzQ4ZTExYmJmMWE5ODc0ZTBkNTI1MWQ0MzRjOGMyNjg5NmMzMjllZmRiYjEyOWY0MmNkMTY1NTcyMzdmN2IyNTZjODY5ZGZkYmQyZjkxNDFjMDczZjZmNzk2NDQ5Yzg4MWI5YjNlYzBiZjIxM2ExZGM3OGIyOWQ3OGUwOTRhYTRmY2YwN2U4YmVkMjc2MzlhMjI2YjRlMTk0Y2NiNDg5N2Q1MWEyOTMzZmM3NzMyMzdlYjlmYmNlMzAyYzBjYjNjY2UxZjIzODhlOGE2NDcxNGIzNTRkNTcwMWU1M2RhZDBkNjM1NzM1ZGJkNjg5ZTE5YTNlODQ2YWQ4ZWQ0NDUwMjgzYzgyOWViZGMyOWU5NTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cdjYWQGVLweo5EPN_1p_Ds76WTP1KK-9TFzWyyetd9aaHXo9GaUfy-0gFcxiMnresowv3cBt9troQViQ2sI_sQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102823_56_2427_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.309Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlY1SFgwUnBXRWRXUlFzRzBaYXhQT1AyTWVoSWtmOXlBNnZMend1bGN4VXBTYVR3NHRHelpib3JRaWJYRkhmVmV3UThQZzN4QnVpUk1wREh5OTZHSmNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMV8xMDI4MjNfNTZfMjQyN18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjE1NDgxZmI0Zjc3M2ViM2YyZDA0MWM2ZGFjZmUzMzcyZmUwZTg4YWI1NWY4YTdmNTFiOTQxOTYwZDk5NjA4N2IwZjY4NmJjNWJlMDliZmEzNzU0YzczOTc1ODM0MTM1YmE1MzhjODQ3NzU4Y2RkNmY3ODY2Yjc0ZTY5ZWFmNmYzZGUxNzViZmEyMmEyZjc0NGMyY2U5OWQ3OTM0MmI3MzBkMDRiMDk1NDM3MjdlMmNjYzI5YThjYmIxYjVlZDI0YmUzNzcyNGFmOWI4MTM1MDU5ZjRhNWUyOWNiMzI1NjU0OGRkNTc1Yzk5OWY4NTI4ZWM0NTNmNjU1YWMyODQ3MjAxY2FkOGMyNGVhMDRiZGE4MGRmN2ExYjhjMGUzZWQyNGIyY2E3YjU2NDg1YjA4YTBhNGM3ODJhNTNkZjBkY2NmZjkwMGUxZjE4N2JmMWUxMDU4OTM4YzZlMDJmMmQ1NTQyMTg3M2VmYWFiYzE3YjRhOTlmYTRjNDkwMjBmNTQ2MGIxOTYyYzdkZDJmMjJlNDMxY2Y3NmNjNWZmYmRkNDc1YzZlN2M0ZjA3NTFmOGFlMTZjNWY3NTEwNmY0MzZlMWYxYTVlMzg4MWIwOTkyYmI4OWJhMmZkOTQyODNhOTJhZDkzNjZkNWQ3NDhjYmY2YWI0OWU4Y2RmMTc1YzNmYTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.pUGewyWoU2ngalHwpNGw6z--o0JWin3USpkKGS1lAnsZ-jtNpxReLJoG9y8TGJhiXYrSwRbnpJqqHqdzO7BuOg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220811_102823_56_2427_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.312Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlBTU041dEZIK1BmWThrWVdxLzZ3cWdyMlZtdWNxWnpNRmtwV1hPTVZzVndXbnVreXg3Tzl0NmUrT053NUJudUZ1aktqY2pwaDNwOGhSWGpuazJEckVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMF8xMDI3MDZfMTdfMjQzMl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzdlMTk4Nzg1YWY4MDlmMmM4ZGNhMjdiYWExNmI3Njc4MDkwYzQxOGM2ZDY2NGQxN2UyZDdlOGMwNDAwZjI1YzllODNjYjA2MTIwN2U3ZmZiYWVjNGEyNWJmNTcxZGRmNWZlZTMzYWFkOWM5OGI0ZjU1NjJiMmUwMzRiNGQxNGNiZTI4Yjg3ZDkwYTYyZjY0NGQ3MjRmMzdjNDg2NTZlZTI5ZWM0ZWJjMmZkNGVmN2YyYjliOWZmMDc4YWFlYjI0MzU2NjYwYzE0MmZlY2RlNmViODhkYWE2M2E0OGFlZmRkZTJiZmQyYjNiNTE4YjE5ZjJjZTEwYmY0MzFjYzhlM2RkMzBiYjU3MDQ3NGJiYWRkMGIzM2YwYTk4ODZjYmYwMTUxNThlZGE4OGFmMGQ5OWZmMjI2MzJhMmU0ODI1ZGViZjA5OGM0N2RmMTA0N2MzODU4MTIwOGQxYzUyYTEyYWU1N2EwNGJjMTk3OWI0OWJlYWNlZGE1NjJkYTQ1NWE0ZmZmNDBmZmRmZTdmMzA3NGY4YjAxMmNjMWI3ZTlkODRiOTdiNTUxMDM2MTcwNThiMzk4YzQ3OTY2OWUyYjAwYmE0MDgyYjE3MWYwM2YzNThhMTE3MzgwNzEyOTUyNmM5YjM1NjM1Yjc2NGJkNjljOGJlZDRiOTcwMGE1ZDVmNDVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.EOCWVGUEqJQCrS-GADGv2e9u_QpNTfbKP6tWUMdD5Dcv57YKruALgtEPKmSarGOCl8a4aE0NNVvxJWKfFe0fyA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220810_102706_17_2432_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.315Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlF4NE8vWFBnRnZscFp0dXloSmQ2RC9ZZlpMMDFzL2NXcTQrSkVvRUlRa2srbTlkUGh2dkVybGNuUWU3OTh6b2JrMUEvN2htZWtYK0p6NG1JaitJL1FBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMF8xMDI3MDZfMTdfMjQzMl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmU1ZTUyMTVkYjkwM2RjMzQzYmNhNGEwOWRkZGVhMWQ5YzVjMTRkZmUzNmFhYjdmZmY2NjM3Njg0NTQ4NzMwMzIwMzYzOTc3ZDk2ZmEwMGUwYTZhODUyNGNmNzg2ODAwNTBhYjNlZWIzNTY0ZjdlYzJjMTJmNWJhZDljNDg4NDcxMzQ3ZGUyN2Y5YzMxOWYxZDYwZmFlZTE1NjI0MjVkMzIzMjkwY2RiZDg3ODNjNzM0OTk4MzMwYTJiYjk1Y2IyNDA2NmZjNjFjZTI1ZGZiNjk5N2JlMTE4NDEwOWIyYjNhYjc4OTQxYzU1NTZlY2RjZTI2OWQyNTBiY2ZlNDhmZjFjZWU0YWJiZjc1NGU0MWQzZWRiZjllZTAxMmZhODZjMzU4ZTE1NjY5YjljZjJiNDhjNTRiOTAyMDU3OTJiYjQ4MGM4OGY4NmRjZjU1NGMxMzY0NzU0ODFhNGRiZmRhOGFlYjMzZDUwZWEwZWY0Mjc5NjAzNjg5NDFmMGUxODQ0MmE1YzA1ZTJhNTkzMWU5NDAxM2JmNzE1NTY1NGM0Yjc4NGFhMjg4NDViZmEzYmI3MjNlYmZhZTY2MDU3ZjE0NGU0MWVkODZmYzU3MzRlMWVkZDI3NzI0NTdmYjI3YjQyYmQ0ZmFiN2FkMDdlYzYwY2EyNTdiOGQ3MWI0YzIxOTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2ORkfnncGYL7QZ0fNFI7y6Tjg-8AYxj-LsaZLWfVgfWN9B0zChjsOZHC0jNH9jDofA_0ur48TJGneWoZdSb3ig", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220810_102706_17_2432_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.318Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImUzcWJaa3M2TjVtdENyQ2xwUnpYcTVLWmo5SUY3NzFPa3BoNy8rellzTWF1SVk3cURuY3J2ZXZqREpHekFOU0d3ak5lNFJrckpUZ1NIME5QTDVpV09RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMF8xMDI3MDZfMTdfMjQzMl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTZmOGM3OTg4YWFlMWYwOWRkNzg4OWQ3ZTFiNmMzZTdhMjg2MzRlMzI1ODIxMjdjYzc0YmQ4MGYxNjY5MzVlNzUzYTU3MmFiNDU3NzBhNjE2YTcwNzJkMzNlMGUxZmJiNjhiNjIzZDczMzQyODZmY2NhZGUyNTkwYTIyMzUzOTdhNjNmNjQ3N2Y0MWY4ZDE3YjI5ZGU1ZjZlZGIyNzYxMTM2M2RkMGMyZWYxNzNiYjZlNDk4ZjVlZDUxMTE2NGIyZTM1OGVlMzljZDVjYzk1NTQ2YzQzMzU4YTBkNDU2OWZlMTA2NWMzMzk1NTQ1MDQ0YTAwMTE3MzBhMmJjYTU0NTdlMGFkYTRmMWViOTNjZTNiNTM2ZTdjMGFhOGE5NjQ2ZTg5NWVhNTFjZjE1NTJlMjYwYWUzYWQ0Mzc0ODFkYzAzZmI2ODQ5Yjc4YTUxOThmNWRjOTA4ZWViMGM3NDAwN2NjMmU0MTZhMjQxMmRjYjhjNzkyNDQzMmMzZTBkNDZhZjY0YjM3ZDFiZTMwYTg2Y2E1M2E1NTc2NzY3YzYyOTgwNGU5OTVmMmI0ZjA5NDc1M2I4ZWQ5NTJkNDZiMjI4MzNjZTI2MGFjYzcyMDk1MzFlNTAwNGNmOGZhMzY3OGFmODdmMzcwZDFlYmU0YzcyY2NiMzRiMDRlMjRiZGIzOGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8ZFMMqpUxJlnSb3UxCV7QKOkafdIqekmo1uJdwrrADvYZr-7kgl8kGQy0ZqxxzTN54vBiUb6_Q4cxPEEHxWN_g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220810_102706_17_2432_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.320Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlYrajBCZlZoL0doLzd1T1kxUjRnaVJseFlWKzlMRWdvbkYzZWsvVmJYeUhIeENyUlhJMlVXL1FUd2pwV0Fxa3B5cHFsQ3dPaHY3RTZqUzJiR2VKajBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxMF8xMDI3MDZfMTdfMjQzMl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmMzOTQyYjNjN2EzZjAyZDQxMGQ3NzIxZmY5ZTE2YmU0NDM5YmI4MWMwNDE0ZTE3ZmY3MzY2M2UzZGU1MmQ4ODU0OTI0YWMxNTk1ZjAyYjBkY2M0ZmIwYzNkNzdjMmMyM2JkNDliZTM3Y2Q5MThmOTRiYjNhMGFlYzQ2YmQ3NThlN2YyMWIzNTA2YWZjNTE0ODNkYjdmNzJkNWVkMzFjODBlMThhZTk5OWJlNWJkMjM0M2U3ZGE0NjM2YzAwYmQ2MTJhMDljYjUzYmIyOGQyYTg2NDMxZTljNGViODU1M2ZjYTdmZDVkMjVlYTM3YmY0Nzg2MTBjYzhiZjNhMzM0OGEyNjM5ODQxNzk0MWRkZTg4YzZjNzZlNDNhMzE3YzU4MTI4ZGU0ZTRkZTdmZjM1ZjlkMDBiZDkyZTlkZTA4YWUyMjk0MmY4NTQwZjEzMWZhZGNiNWU4Yzk0NmZhZjQ5MzA0YTA4NDZjOTFhZDkyYzE5ODdlMTRlZGNmYTYyYTkzMGM2MzhkYWU0YTY3MTc3ZWNjNTI2ZjgzYjdiOTRjNzY4ZTZlMGVjMTcwNTQwZDkzM2YzM2I1ZWYwOTA2MTc3YTdmYzYzMjAxZDM3OGUzODU3MDlkNzQ5ZjljNzU3YWExNWU3M2Y2MTU5MmE5MTJkMjYxNjA5YjhiOGZhMWJhNjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.y6QhXk1EJn5pSixrpKuA7wYjkJrLGe3zcQKyWDoeknigf_CWTa2I6mD8H5PPSuEU_3TiHtfcizxQVZICarmQMQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220810_102706_17_2432_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.323Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InViZXdjajNWSE5rTUVDa2IzdVQvRkxCQzJpdjlrRklXU2kyVnRpUDlLNGxBc1FTY0dnbUtrL0ZuWmVLSEZHTloveGhDR0VQbC91VkxPaXpUWUtZT0V3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI2MDRfMzNfMjQxZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTQwMzg3N2QzMGZhM2ZjMzhiNGI0YzViZTE1ZWExYjMzMTEyNGQxYmQ1MTY1NzlhYjQ0ZTI3MzNlNGFmYWIyMmQ1OGFmNDYwY2Q1NmE4ODAxOWU5ZmQwZWYzNmY0NWU4MjUwNjM3NTg1MDc5NTBlYmIyNDZjMzk3ZDdmOGE2Y2JjYzY4OTE4Yjc2MTg5M2MyZDdlODRmOTNiYTRiMWM5YmJhYTg3M2U0MDAzMzkxNzA0YjliN2JlNWMzZWVmZTIzNjJiNTk5OTAyZGY0MWQ4MTQ3NWQ4YjlhZDg2ZmMxZmRhMWY1Y2QxYjNmN2ZjN2U3ZTViZDA3OTdlNGRlZjg5ZmY3NTRiOTkwMTIxNGUxZGIwNDY3YzZkZjQwZGQ4MzdjYjY3ZDU1YmFkMDYxNDljYjMxNTk1YjE5MzMxYmEzYTY4MzFiYWUzZTVhOGQ0MmUyNGJiMTI1NTBkNDg1M2M0MzlmN2E5MWEzZDUzODk5MjNiNDM4Zjg2Njc2ZmE4ZjYyODc5M2E0NjhlOTI2Y2E1MTU2Y2Y4ZmU5YzRhNDE0YmFhMmM0ZDFmMmEwNTZmNzRiNzg2YzExOTEyMmRlODllYzUyZWYzM2ZjNDJlMzU1OWU4MDA0OGMwZTA1YzhhNzgxMmY4YzdkYWNkNDQ4NWE5NmMyMzJmMDQ3NDE5MTRkNThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ghuHgmxTXgZ7mXsZVGdHe4L3kh8jvt4wQHAKLhP7bHjRqX1gAXePDWZzU0v8w6tNMND7ANd8SadX_7Vj6sa0Yw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102604_33_241e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.326Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InM2Rml0NjVXeDlZS2FJWHQ3ZnlCRm9BRUdmSVA3QmZoTUtIOGhmYUZnRzBhOUN3WUIzU2ZSWWVPTWg5RkhXY0hGcWR4aHJQVHN0clZVY1FmYjF4Nzh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI2MDRfMzNfMjQxZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTI0N2YxZmM5ZTJmMjkxMWM1OGY0ODM5NWI1ZGZhODI1Y2IxODUzOThlMGQ3ZWNkZGYyZmYwMjhlNzA2YWQzOWZjM2Y1M2U3YmEwOTZiYjYxZjg5ZDFlYTlmZWI0YjliN2YyNDcxNDU4OTY4Mjc4NDBmOTdjYjdlMjYyNzAyZGIyM2Q3MmE4ZDhiMDQzMmIwNjlkNzA5YTYyMGMyZDJjZjdmMjcyZTljYzM3ZThhOGEyOGU5NGE0MDczMTQ4MzRjNzVlY2E5YzIzZThkNGY3YmMxYjdmNzFlNGMyM2JlZGUzMDhkOTdhNzNjMTM4MzFmZTk1MGI3NzkzMTIzZmE5NzkwOWRmNzQ3OGFhOWUwYWUyOTVmNmZhOTQ1YTQwYzUzNmM4ZDY5NjQ4ZWMyZWM4YmMzMDAyMzM4Y2Q3MzZjOWM1NzE5NWJhN2IxYmZmNjg4ZTdjMTZhNzk3MmM1MGE2MjNiZTc4ZmVhYmZiNTQ0YTY3OThkNGUxOTU0M2JmZTZkZDRkM2QwM2M3Zjg5MTg4YjI3MGU3NDdkNTQ1MDZkNjEwM2VhNjE1NGZkNWFkNGQyYzRjMjMwYmZlYjQ4OGNiZGE2YmFiNTIxZGI1MTQ4MzY2NGVjMWFmNDE0OGM4MDFkMDM0Nzg5NjcyODgyOTQzODM0ZTQ5YmVhYzY1MGNmYzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QNiF3ZHyNwVm31Ng0pWwvw2iZXAaM2_hmkIPDUON7LeAYwg51FEcsRUtvKcrgsW6_jv4IgFves2HfnUZrgGmKg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102604_33_241e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.328Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImQ3ZytrUmt4dHdSYkdISWRzRWpmZnZXdVAwb1pRRTlYeGZIekFDdy9takxmaE50OEFrN0hzaGRhN0dacERJMEc1OVhBZlNMazM1TGZGWXQzVjJEMHBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI2MDRfMzNfMjQxZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI2MmZlODgyNGE2ZDkxZjA3MzJlZmU4ZDA0NzU4MTA3YzA1NmVmYzY4MTc5ZjYxMzNlY2JjMzBlY2MwYWZjM2Y2MTcwMzAwYmU4OGQ5YWI2MmQ1OTZlYjcwMzI5NmFkODNjZjdlMjFhZmZlNGQxNDYwYTQ5NGZmZTliODAxZTExYTY3ZDhjMTZkZjRmNGMwYzM4OGMzOGZjMjBhOTdjNmExODU0YjM0YzFkZGZkNjAyYWU4NjM3MGQyOTY1Yjc0YTU5NjFkZWJlNGE2YzhiODQ3ZWY0MDAzYTgzNTNhZWQ3M2NiNGNiYzMwNDdjZjIxNjJjMGVjNTU0ZDgyNWRlYjA4MTNmMWVmOTU1ODE3YTFmMGZmNmM3Yjk4MGY5OWY2ZGI0YzhhYjA0YzQ2NjRhNzU2MDRkNWI0NzNlZWFhMjU3MDZlNmZkYTBlYTY3ZTZjMDczYjBkYzc3Y2VhNmYwMmUyMDFkNGQ1YjFiMWU2ZmU0NTA0ZDg4ZGNjM2U5YWQ1Y2E5YmM5MzFjNTIwMDgxNzdmMTJiMDUxNWU4Zjg5MzVlYTFlZjkzODNjOWUxYjMzYzIxMWFjZmViZWMzNDZmNjAwYTViNmQ4YTAyMjE2MDZmMTUyNWJjYzU0YTZhN2FiNzI1OTYzMzhmMmJmY2QzOTQxMmI4YzM1NjM3YmI1ZWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Fufau--KewlFDv_luiUH7Dv1Sb2OgmiTCteePVL1sCZeIpSbfi3C2XLcM0RcAndc1LCTs631c92gGVUyAXE72Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102604_33_241e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.331Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii9NOTF2SlhIOHhESTlkbEg5eTBuMEJmeHdNZUVlc1o1NzhvcURMbnI2ZVNGR3JicEg4WFA1eGpsU1BNU1lXMEJubW9wSC82RSt4b01SSm80SlkzdU53PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDYxM18xMDI2MDRfMzNfMjQxZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTlhNzQyZDU1MzljODkyODhhMWEzOWM1MzRmOTFiOWQzYThiYjg1MzM1NjBlMzVmY2RhZmQ0Y2MwNDk3NDg3MGYwODE2NThkN2VhYzAzOWQ3YzE0OTc1YjczM2ZjNzA3Nzc3ZTc3N2U5MjdmZDlhMzEwZGNlNjhhNDYyYjNmZTQ4M2NmMzcyNDZmMzBhNWZhZjM3ZmI0YTA0MTUxYzQ1MDNiNDhmZWFhZTE4NjI5MDBjMDJmMWVjMmRiMGEwZGUwZDQ4NmJhNWRlYmEyYTE5YzZmNTY2OWYwODgyOTZjY2ZmOWEyNDA2MWU3YTQxOTYzNDRmNjA1MWExMjVhZDQ0MWI5ODUyYTQ5N2ZhNzRhZjg5ODc5MjgyNGM5OTk4YmFiOTUyYTQ2YjUzMTQ3ZTVkZGY1YzBlMTc3YjE2NTUzZDc2ZjJmMzhkYTEwZjczNTlhMjA1MDc4NDFiNGFlMDViZDFjOGE2MDI1NDhjM2Y0NGU0MDliNGE0NTkzNDI5NjEwZDc4NzkxMWQ0NDM4ZWZjYTU1MWU3YjFmOTdkMGI1YTkzOGEwNDc5Y2VkOTJmY2JmYjc3ODgwYWMwNGM4N2NkZDY0YjIwYmY1NWU5NWE1MWFmMmMzMDUxNWMxYjNmODY3ZjgwNGI3N2QwZjRjMjY3OTdhMDI1OGE4MGFlMWNjNjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Fset3FRr3afUsYi-JE86QcRy57u2YrqBkZlFpDV7-uBslayncTGlx7C94nZlSfES71Hcnd2-_62V56zFfSvMew", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102604_33_241e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.334Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkEyNTI5TnhFOU40djVvOUxyeWxrNjBUWmRsS2VtNDNLSXhtZTAvZ2FVaExJODNJMHFlOXlLQk0vL2hyQ3V2NnJ2Ym91WWwvU0tYNmh5Vkg4WWxQaHN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEyNl8xMDM5MTdfNjhfMjRjNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjBkZDk4OGZmZjMwM2MwZTQ1OThjODU0ODJlMThhMzcxNjZmZjU0MWI1NGQzNmIwNmQxZTg2ZGRhZWNhOThiNGIxMmMxZDhmYzRkMjkyNTYyZjk0MWVjMDM2YWFkODZjMDZlMmNhZTllOTdhYTJkMmY0ZTJmMTMxMDk3OGQ4NDFhYzM3NjQ0OWU5ZmRmYTAyODIwOWI0YmIzYTFiZDc4MWUwYzdmNTJlY2Q2ZDYzMTMwZjNkNDBjODI5ZDBiNjcyMzM3YjM3M2ZlYTU2NGIyMDM2MjAwMTBkMjY2OWI1OTIyMWIxMDc4YjM1NTcxMDQ0ZWYzYjAwNDIxYjExZWFiYjFkMGM3NjM2ZjBjYzRhYjE0MGIxODU5MTA5YjZhYTgwZTljNjMwOGEwZmFjN2RjMWRiN2RhYzEzMWQ1Nzc3NmY1MWViYzU3YzFhMWFmODBmODlkYzgzMDA4ZTQyYzgxYTk3MTE1ODk0MTBhN2E0YmYwZjhhNTVhNTViYTA5MGUyNWNmMmYxYzY0NDFhNjllN2E1ZDk4MDIwODkyMzkxMWE2YzJkZjg2MWJiZTIzNThkODEyOGIyMGJjMTlmYjAwZDc3N2M4ZTNhMWZiZWY1N2M3MjI3MTEzNGJkYmM5NDdiNzBkOWQ5MWVlM2RmYWNiNTU2NGMxYzAwMjRiOWJhNzdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Be-InQoyTwiQfxVxmKv1vDgm5FesWyMP7iJi9pUsNaMFB_KUCzKFtppfg8Zx4FRgjQ-hZocYW97Svhw5_SkMFQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240126_103917_68_24c5_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.337Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpZVy9Db2JIc1FTOW14emxDdnlpNDVENHMrTVE0ZWg3NFA1SWx0QkpaRFZMa0R0VEs1czlkZEl5YVNEQmV0K3pRL09SN01DMlVhVm1ETEt2VHU0dCt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEyNl8xMDM5MTdfNjhfMjRjNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODhkMDI1ZWMwNDdjYjNkZWFjNDg0N2Y3M2JjYWJmYmU4MDA1ODgyM2QxODYzYWJjNzllMmUzMjdiNmM5MDU3MDU0OGQ1NzNhYWQ4YWY3NjVhNDZjZWRmMDczMzhiY2JlMDNhNWM1OTUyMzk4YzNjNjE4NmM1MTg0ODAxOGExYzZmNWJlNmIzMzQ4NGJlZWU2ZWFlNzY1Yjk2NWY1MjQyMjUxYmE4OWViOTcwYjFkMTNmNzJmMTdjM2I3MzNiYjVmNjZjMDQzZDdkYzczNWI5NDRjMDIzZTJkYjU3M2JlNTQyNDg2MmFjMmIzZWU3YTZiMGI1OWVjOTYwMTNkYzA4YmMyZWNkMjc0OTlkOTJkNDRlYTJmMTVmZWFjNGZlMTM0YjlhZDJmZjkyZWNhYWE2ZjJkOTE1YzFiZWVhM2JjNTA3MmE4ZjI2Zjk0YzUyYTE4ZjAwZTA2NTY4ZWIxZTE1MjMxZmNkZTFhNmRhZGE3MDlhOGJhOTg0M2E3MTI1NjFiNmYwZTYyZWY3OWZhZjQwN2NlYjc0MTdlMzRlMzZlM2NkNzg2ZmQzMzY0NjE5Yjk3Yjc2Y2M4NzljYWVhNzA5NzQ3YTlhYjU2ZGJlNzhmMmM2NzNjNjc2M2E0ODZiZjk1MzY0OTgyNTQxY2NmMGRjODkxNzE4YjMyNjQzNTgzYzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8VmJk_kpQ5z3xIPzwHPMsPVQSvjCVf-HmzylqEPqwa3EakMzkzYqu3jUFv3nm1ENp3HKPci3gKHcNChuqyep-A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240126_103917_68_24c5_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.340Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imk3dFN5bmlPVkFXVzU3Zmg1dUo1cndTYVRrSXJsS2tWbWFUbWlDcEh2TXR0VklxYTdXLzVPV1o1MXBURGxlTGpWNVdtYkluQ3lLQnBIUzJVV2E1Mk9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEyNl8xMDM5MTdfNjhfMjRjNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI1MjQwYzgyYjE1M2Q0MDNiZDI5MTdjY2RhMWRhN2U1MGFjMWU1OTcxOGM2YzdlOGQ4ZWVlMWE5ZGRkZTNlMzVmNTgyYTA0YTgyZjI1MWY5ZmIxNzhmYzE1NmEzMWY5MzA5MjE0YmYyNzMyNGFiZGE3MjViNDU1OWJhNDM3ZjM3Mjc2YTA2OWZhMTUzNDVlMTdiZjE5YmIxZDJlNzg0NTFkYWQ2ZGQ2NzBlYTcxOWFiYjlhNDg4Y2Q5NTA1ODc2NGQxZGM4YzgyZmNlMmE1MDc0Y2UzNmY2OWJiZTk2NTJhYTdmNGNmMDNmMzhlOWU2NzExOGQ0ZWE1NWJiMDE0OGRkYWQ0OTZiYTlkMzc2N2M4ZWE2NjA2MzNjZmYzMzU0Y2Y1NjZkNzA0NjEzNGMxYjhmMWYxZTM5OGU3NzNlZWJkY2FmNmRmYjNjN2UwNjUzNzg2NDBmZjBmNGM4MmU4Y2YwYTZhMGExYzI2MmVlNWM1MjhhZjBjMTk2MTIwYmU0MWVjMTE4YTBhYTYyZjZmMDdkOTk5NGFjMGFhNjkwYTE5YTI3MzZjNmE1YzUyNzJiOTYzMjI0YWU4NjY1OTk1OWFkMzRiZGY0ZmZhYmM3ZDI0NzUwOTIxNWUzMTUxMTJjY2E5ZDk2OTQyZDc2MmI1OGJhYTkwMjBiYjcwNmYwYThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.AjYAFxh44w-4CATs6tpygbe97UX-TUifzluoB1cg4QZD9ZEx4T4fRyoN5BElAlLr5OJS878I0SvYjfnfwZadWA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240126_103917_68_24c5_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.342Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpkNmVUOExvQ0g0eTA5MndCdXBpWGtYY2ZlUzF2MEY0YndqbHVpc25pTkVIU1dDN2kzTmxDOGQ0SXFBbTcwd2FQREdJeFpPUVN6SFhMWTFJSGRUa1N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDEyNl8xMDM5MTdfNjhfMjRjNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTdhYmRjYjJkMjQ1Nzc3N2NiOTU5NzQzNDA0M2Y2ZWEzMzczODVjYjhhNGRlY2UyYzhhNDJkMzI0MmM1YzY0NWI2MzQ3NzZhZDU3YzQ4ZjE3NjUyM2FhNTlmMjlmOGQwNjJhZjljMTE2ZTM4NzQ4YjJjN2UxNmZiYzQ5ZTBjNTUxMWQ0ZmRkZWQ1YWI0ZmJjNWUyOGMzZWUxZjE5ZGY4YzMzMmRiMTZjYzJhYjQ4OWMzMmFkYmQzZGI0NWJjYTAwOTczNGMxYzg0NmUzNjkwNzg4YjFjYzJmZWZkZTY2MjllYjQwY2M0NTcwMjNlNGM4NTdiOTMxMGI4NzU0MzRhY2E4NjZlNGJmZTJmOTU5NDk4ZmMyYjc5ODU2MjYyMDAyMDZkZDM0OGY4ZTJkMzg4ZWE5MDg2ZTJiZTUyYjExNDdiZjViNjljNjkzOWEyYjg5YWNiYmJlZDk4OWIzZGMzMjg3ODkxM2IyZjMwYjYwZjhmYTljZDA2ZGRjMGRlOTI4NDc4Mjk0N2JkYTQ3MjA0NDdiNTRkZDc1ZmRlZDVkNjM0Yzc1OTVkZGI4YTQ2OTI2MGI4MDAxNjFjNmM4NTRjOGJiNWY4NGFkYTM2OTYwZTFjMDQ1YmEyY2NmM2QxMjcyMzczZDc1YzYxNTc2YzVmZTU0ZmIwOTRjM2Y1NjQ0ZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gcC2corsmMgRkqI97Je4FYKac9q5BBhqlsQ4DvWugFKxB41qYnReL420wUn4jXnxpPv_Ln0NSFPgrCPkB_biSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240126_103917_68_24c5_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.346Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjJEQlVqSzRmV29sQzJpSUZGd2x5L2kyNFhjZGhEcVd6d25hdzNoNjBFU21xTmNKRE1aT3VTWjc3MWFrZndGS2N1cGZjU3dFMER5WmQyaVdBWWRLaWtRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTFfODhfMjRhNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmVhZjk0MDczNjBmMjA5OTc4NmRjNzhjZjYxNGFkZmFlNDNjMzU1MzBmMmI2Zjg3NjRjMWY0MzBmNWExMDhmYWYwM2Y3YmJmM2U0N2YyNWI3Y2Y3Mzg0ZTg5YjYyNDExNGQ3YmFjYmY5NTU0NjVmZjVkNmNiYzY1Y2UxYjE2MTBjMzJmYTYwNGU0MWM5MjhjMTExODE0YjZmNmEyMDc0MjJmMGFmNjIxNDE2OGU1YmJiOWNiZTkxOGI3YmZjNzgwNjYwZTY1MzFmYTYwMTVmODhiNzllNmNkNDZlNTgxMDBkYzg0ODA5MDlkMDlkNTNhOTJjODZlNTJmODg0YzkxM2U4MTVhZTE5NDliZDM1ZDRjZTVhMGRkYTM3OWY3NjAwMWRlNTVjYjBmODg1MTNhZjI1N2M5N2RhNWU3ZDk1M2Q4ZmU5NDRjMDIzMWE0NWFmNGYwZjI5ZjI2YjQ2ZTU3Y2I3NWFlMjk0Zjk5OTYwYTExMWEzNTExZDZhM2VlNzBjOTEzMzhkZjAxNDVmNWEzMmQ4ZjlmMWRmYjVjYjE2MGRjOTY5ODYzODRhYjQwN2JmNWM4MWFmYjQ3OTY2ZGFhZmJmMDMyZWNlYWI0M2E0MmZhMmJlYTQ2NGVlNzQ5MDMzNTRjYmVjMmVhOWQ1NmMxNDRjZDdjY2FkYjJkZDBkYjRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5Cee9_VVi3ncvdym2sVNtjHtT5XFZ1NNmkgxxdxC_dCsIY-IYQCf09viWc9iiNOE-vlcNMQq3sP1RUCxh8AHYg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112211_88_24a4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.349Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InNtZHBWMFkzeXh4am5nTTJDTnAxYkdDL0NsSmZQYXpWRFg3eFhHS21ja1ZHNDlveWhLVEx2NVVFb2ZYOHJVeXM0Sk5hYko5ZWVlcW5WSDNXdkpyaDRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTFfODhfMjRhNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODU1MDU4YjhiMWZmYjNkM2NmNzJkNjY5YWM4YTZlNjc4NTNkMDcwNDllMTE1N2Y2MzcwOWQxOTAyNzg2YTkyODczMDIyY2JlZmY0NjI4MGYyM2I5ZGFiZDE5MjZkNDMyZDcwZTJmYzUwOTA1ZTY0NzU0YjdmOWRhMmI0ZDNlZGEyMjIxOWE1Nzg3YmYwZWJlMWMyMGYzYjY1NjE1Mzc5Yzc3ZmMzOWNmODNkNDI4OGNlOGI5MTNhMzNkNTViYmU3OWRlOTBjM2UwNjQxYTVkMTg3ZmEyOGJiMDYzYmUyNTVlNzkxYjhlY2QzMmRmODg5ZjI5ZGMzMjUxNmQyYzU1MjNjNTkwMjkxODVjOGNhMDUxZTJhYmQ4NmEyMjczYjQ4NTM5ZWYzNGY5ZTFmN2QwY2FiMzA1ZTgyZTBkNDRmOTM0ZjRhNmExMzRkYmMzMDM1ZmU3ZmNiZTVkOTkyYzdjMTcyNzY1ODE3ZTkxOGU3NzNlMjgwYzYzZDYyNTYxZTQ5YzhkNDg1OWQ0M2NhNjFlNDg2OTFmYjY3ZTAxMjE1NDhiNWM4ZmE2OTg4MTU3NTQ4OGUyODQwNGJlNmJiMWJlYzRjOGIyZTgwNzBkYzlkMDJiOTg0NWNmMDM3MTBhODNhZjc3Yjk2ZWQzYzQwYjJlMmVkMjUzN2U3YTRkNDBjNjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.SQT0VhFzgbALNdTZSHgp7WLUbDxhmqDVdisJcoFliOUAfFNgwf7AZaJxUZ6yHElWVH7TxhQblPKCxZWk60mVfA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112211_88_24a4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.353Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFmOWJDVVdLbXdvUnEwTWMxOE4xRDNWT2xNZzVUV1l5b0NQVk5SSzZPVk5oenB1ZC80eVhFM2tKazlpdGpvZHZHODZqdGptanNsM2hDOUxhTUZKV2ZBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTFfODhfMjRhNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTNlZTA4YjQ4NTYxNTI5ZjMzNTRhYmY5MTVhMTAzZGJkM2I1ZGEyYzRjOWVmOGY0ZDc0NWYwZGFlMWE5Mzc4YjkyN2ViNzMzYzMxYmFlYjAzZDIxNzc3Y2MzYWMyZGFlYWJiZWQ1MjE4ODBjMmQ4OWRmNzdiNjNiOGQwZGNhMGUyOTliYmEwMDQ0OWI2NjIxZjc4OWY4MTE0MGM1NWVkY2VkYTViOTczZjNiZWE0YjgyNzZlNTI3N2E4MmE1YjdmMTA2NTJhMjMwMzJkZmE1ZjM1NDM2Mjc2YzBkMmUxMzU1NWQ0ZTZlMDUyNzlmODZiZmU0MGJjMzc1YTUyY2I2NmE3NDIzMTQ1MWY5YTAwMTJlYzZjNTdkYjFiMmRhZmM4ZDk5ZTdkMGVlMTEwN2MxMTY0YmFjMWQ2ZTc1YzNmYmIzN2E2ZWEzZGU3NjQ3MTFkNjg1ZWM2MDhhOWJhMzc3YTAyZWNkYmM1MzQ2NDEwMTY2ODVhNjE1YmUwZTI5OWQyOTA1MjhiYmMxZTMwMzBiZGQxMmYxMTI1NTAzZmI2YTgzYTQwNTliN2Y0NzcyZjA3ZTNiYjI0NGVhNTFlNTc4YzNiMTEzMTY3NzBiYzY4MjNjN2ZmZTAzNjA2MWRhYTY4NDk3ZjkzOTNiMTM5ZDJmN2E3NzIzMDc2NWI5YTc0NDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IcGoI5yFcP9rIqxtunA1dYB_gi8cG8_CEtB4f9GWpsQgGxVEoGLeNaE_Yz1UHQtBxxgUOCyUqukbRlXNpoQfTA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112211_88_24a4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.356Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxJb0JKcEwrUW5WcFlBV2dWalVYak94blh5OGFsOFA1T2xIZG9vY1JNSms5eXBZRWNIMERobCtMSzR0L01MRW5OU3NVRCtkR1BCcjhqQmNvMll2N1dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExOF8xMTIyMTFfODhfMjRhNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjNkODk1YjNlMWEzMGNiZmE4MmY0NTBmZWI0ZTFhNGFhZTA4MDZhZmQyZTJkZGI0MDlkNWVkODc0NDkzNTE4YzE3N2Y5Yjk4MzdhNTYxN2RjOTU5NDRiMzdkOWU2YjE3NmYxMzJmYzJlM2Y3ZmYwMTBlNTIzMTBlNDdiMGY1NDJiNzFmZmI3YzJmNWZkODZlMzFmOTE3ZWNhZmRmOTRhNDMzYmUzZDgwY2RiNjI1NjlkMjI2Y2EzM2QyOTMyMjI0ZTc0Y2M3ZjE5MTM4ZjQ1NTY3OWQ0ZmU0OWUxYjgxZWQ4MDE3MWIxYmE5YTBiOWUyNGE5ZWZjZGZkODgwYzc3MzczNmIwNjNmOGQ1YzA1YzFmMDUzNjY5ZTdjNmMxODNhMjM3YzhiMzNkNGQ0YmMxZjhjNmM2MDVkYmM3NGY0ZjFmODcwYTIwYzI2MTlhN2EyNTNkZDA1MjIwZTAyMjJjN2FkNTY3N2FiZjIyOGRjZjBlMzcxNThhZjFiMWY3ZmFhZTI1NTYyZjgxMjEzNTU0NWNiMGY2M2IxNTU5MDg2MzZiNjdmZjAzOTM4ZTkwODY5Mjg4ZmU3ZWM0MDFjMmMzZmM3Mzk5MzYwNjk5OTEyNjg1ZTcwNzhmOGVhODUxYjMzMjFmNjBmZDMxZDk3MTJmZGFiOGNlYzkzYTYyZGU3ZGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LG1gkrYNQnQzzGOaTa9lvOKTMPVpqECcXO1MRl91GXyEPkaYKSYyOPDZFwuLQczKvJQjzZgdg_TXJf71QJFGug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240118_112211_88_24a4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.359Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkxXd2c1alpIR0ZYeVY0Z2dnWW9XaFlGa3RWK3VzTHpJVGV0R2kzNkYzeDYwV3Qrb0N4bDdPLzBtMmtsM2tWYklqeFg3MEI1YzU4THVXSDNiKytDMk5RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDMyNDBfMDdfMjQyOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmM1ZDM3YTAxMWE2ODk2ZGIwMTIzYzIyOTI4N2ExNDY3NDY0Nzc5MDM3MjU5NjI0MTQ4OTM1NDJhODAyNDQzMjMwMTYxOWE1YWFiZmQzMTM1MTgwNWMzNmY0MmE0Zjc5ZWZiMGJlYTgyNTM2ZThjYjcyY2M2MTk4YmZhNzE0MTk3ZWRhOTllMzlhYjEwZmFkNmJkZTM2ZDBiMGZmYTIwYjdkMWU4NWQ2Yjk0MGU1YzdjOThmMTIyZTljNDFhNTIyZTAxYTkxN2YyNzViZGFjOWQzZGI2N2Y5MTcyODY1MjNmNmJjYWNlMjBhMzlkYTJhNDI0NjRjMjBjYmFhN2VmMjE5ZjZkMWE4NjA5ODZlNTQzOWExOTU1NjkyMTMxM2UzYTQwMWE4YTM0M2UyMDExNDU5YjNmYWMzY2U1YjRjZWJkZDg3Njg0YmZkNjlmYTJhOWU3NzI4MDRiMTdkYzE0NWM2OThhMGYyMWY4M2RhNzUzN2RmYTQxZWI5ODY1OTE3YWYyNmZjNWE1ZmQyOTE1YmI1NDQxOWQzMmQzOTZjOTY3YTc4ZjVhYzY2ZTFiZGRkZGNiMjlmMmQ5YTRjNTk2NjZmYTAzOGJiZDQ5ZGIwYTAwMmM5MGY1ZDkzMjIzMzJkNzIyOTIxMGFkYzFjZWM1YjQyNzA1MTU4ZjBmNTQ5ODNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.IZpY5DIEVX0Nk7_xHrjEuZN5Ex9fdRkqK_0gna7zz7wVF2ie5Ke-Zq10qFD015ozkEF72cYTulnpuB5mIZYONg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103240_07_2429_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.362Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InIybDErMU5aTlVmd1VSZE03dE5KUVd0QTU0Q0FZelN0WHJ5WmNoQ3Z1aXlDTERkeTZCMWY5cUZMTkxkWWhHajA5VHhBL2pxWm8waU94U0p0SU96c213PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDMyNDBfMDdfMjQyOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjlhZjRlM2M1NGQwYWE2MWYzNDQxZjNkMTNmZWY2NTY0Zjc5ODdhNDZmNDBkZDBmOWQ0NjUyZGI4NzlkNWQzN2E5OTJlOWM3ZTJlZTg5NzY3YzI0ZGM5YmFmMmM0OTJkZWE2ZmUzZTY1ZjI3MjQ1ZDg3YjgxYTBiNTE0M2E5NzkxNDYxYzdjOWRlMTA0NzMwMTRiNDVjZGRhNzE4Y2ZiNDBiOGNhMmM2M2Q4MmY1OTEyMTU1NTAwODFhYzViYzVlZTBhNjkwOTE1ZmFmMjQ0MDkxYzQ2NWNmZGJiYzhjMTE2NzE0MDhlY2I0Y2ZhYWUwMzU5MDQ5NTk3ZTExMDRhNDRiZDVjYTVjMzM0ZThiZTdlMjliMTlmMTAwNzA4MTAwZmNkNDI4YTk0YjhkZTIyZGUzNGE0YjE0YTg5ZWNmYzE4ZjRkNzI4MTY0OGQzMGMzY2RiNDgyYjc1YTkwMWUxZTljZDViNDRlNWYxMjJlMjkzMGNlMGJmNTVhZTUzZGM4OTY4ODFhODMwNjFmYjA2YmMzZjQ3YzgxOWVkMTEyODI2ZGViMTNjZmI1ZWYxOTJhNjY2NzcyY2QyMjNhYjEyMzIzMzdiNDk1NWYyNmIxYzk4MmUxZmQ2ZjQ1NmQwYTc2YzNmZjAwOTY5OGQ1NjI4YzYxMjAwOTQxNjUwMDJjMzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.G2D-bzcwMyH4faUH3w2YP3nxcLxVB3C1jNTzi6Y6JwekgW3ZV-vm_3JLuqNWOrLxmm7LUS2qfEooS_dob1xfVA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103240_07_2429_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.365Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik9vT1loSE1oTU1mampWUEIyYVlKbzlVN0w5RHpRdHVUb2FhTWErMnhYZWptc1lkUi81WnVJT1RNdWVNTjhKUzl6Q2VxVUE0ZnA3Y0syYXk5SVNYREd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDMyNDBfMDdfMjQyOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzZlMmU4NDQ3NDVjZGE1NDA5YmVmMjZhOWQwYzQ2ODlhOTk0ZTUyZmQ3MjhlZTJmMmFkMmE3YjgyNmZiNWRmYjk1Y2JlNDU4ZTMxYzRhYTg2MTdmZGEyMTlhMmE5ZTY2ZWIxZGY4YjY3NjFjODc1ZWIwYzgxNTdhYTU2MWFkM2I3ZWJhMjg3ZTZmMzRhYjQxMDkwMjlkNjNmZTBiYzU4OGE2OGQxZGIzZmUzNDMwZDQ3OTBkZjU4ZGQxYzE2NGUwYTQ5MGYxOWQ5MGJiMWEwYjQxMTYyOGU1OTU1MzUzOGZkYWUwMTcwYjQzMjNjOGYzOWE0YWNlNzZkY2JiYmYyOGFjMTY0Y2NiYjNjMDk2MWQ1ZWRjNjc5ZmMxM2E2MDc4ZGZiM2YxYjc0MmNkNWRjNmJkMjY1ZmFiMTllODkyYzJhMWRjN2ZjNzkxZjgyZGNmNjFkYjQ1ZWE1Mzc0OTU3ZDdkOGY1ZGUwYjA4OGRhMmMwYTY0OTllYzQxMzhlYWYxYWRiYzI2YjI0M2FjNGMyNDAwMTU4MGEwMDZjMTY4NDg3ODAwYTMwY2E1YjViYjVlMGNhM2JmYTdkNzU1ZjA1OTY2NzMzODFjMjg5MjhjNTlhNjQzNjFhNTFjMDRlMzE4YWEyOGJjMDBjZDAzYmFmMGUzNTgwYjM5N2EwMWE1MzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Qmiy7r6NB_S6Ln_s04JqQOwWKe0Yxb3QYiBl-Ity9IxcAYqOSs9ERpz7ZJhLELzI_XIu6duVFAu1QoofGqyGHQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103240_07_2429_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.368Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImwwT3o3ZTVWaldseVNaWFVJK25CcFZHSVVWT0NWQUlwSUFJbzYrYVZHNitoeW5FcUF3b285Q2FSR0FOTC9Ta0MzcG5KOTJUKzQ5S3l4S2s3TEM1RXFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDMyNDBfMDdfMjQyOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTg0OWEyMmE4YjAyOTkwNjY3MGU1YTZjZWVmZWExNTIwNjk5NGRlZTRlNTczYzkyMzllYThjNTk3ZTgxMTZjNmRjMDBjMjM5MWFmODZlNzM0MzI3M2RmZWQyYmQzZDFlMmNkZDBmMTliMTU2OTk4ZDYyZTE4ZWIwMmJiYjQ1NmEwZDQ4NTQ5M2RmZWFjMzRjZDhlMTZiODhmMzg3ZWYzNzFmMDQwYTMzZjhiMzNkZDI1MDg1NTg3MTU1ODYyNDBhM2QzYmIwYzg2NmY3NGM4YmUwODljNWEzNTlkZjliYzNlNGRlOTA0MWI5ZjI4YjViMDIyMDc3ZTA1NDNhZGE5MWI1Zjc3ODNiZjBlZjVkZTA4YWUyODY4NjQwMzgwM2MwMDBiOWMyNGNjMjgzYzE0MzFlMmVmYTEwNDAzZjliOWZkYTFjOWY1MzhkODRmOTY5OGJkNDZjMTk0ZDRlZTM3NDI2ZGI5MThmMzkwOWEzYjc0MDYwOTU4OWNhNmMxZDAwODU5NzQ5ZTZmY2VmYWExOGViNzFhNTllZGZlZjFjOGUyZDEyZDE1YWFmYzc2YzhkNGRlZDM2YjViYzRlOGQzNmVjOGE4MWVhNjYzY2NmMjA2ZWFjZmQxYmJmOGU4YTFhNDUwZGQwYWY1MTg5MzhkOGJiZmQ5MDVjMDhjMzNlMTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4ikJKzzefofWFVDf_PzoWaOOrHFMOOgKawau2JdXi1kxIpmNPOfTlbG9GPXb-HAfD0_CDDfzXdsEGhtNFTMi1w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103240_07_2429_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.370Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImRoTHA2R2t6QWpvQ3BXNEVzTGVLU1h1OHNsYm54byt1NmtoL2o4R1RtTk01UUpITmVBM0ZJZDgrb3F6WWJFRVdvLzNiU090MDRLQmRGajBDblJycDRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNThfMjZfMjQ4Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGY0Yzk4ZGU3NjE0MTQ2M2U4YWYyNDc4Y2Y3M2Y4MDZiZTI1YjhmYTNiZGFmNWVhM2NhNmUyODkwMDU5NWFlYjkzYzk0MThmNDM3YzEyNjk3NWM3YTFjZWNiNjM0ZjgyMGM4ZDczZjU3ZWExZjZlNTM5NTUxNjU1NjViY2FlYzE5MTNkM2E0ZjZkZGE5ZWUyMzAyMGFkNmQ4ZTkyMWRiMGU1NDcxNDUxYmZlMDFkNDVjNThkM2FjYjUyNWE4YmY5ZWVmYjM3ODQxNmQ0ZTU5OTA2NTFiZGE4ZGFlNzk5YjJmNmM3NGM1OTRjZjViMzI5Y2I3ZDY5NzEyYzlmOWJiYmE5YmZlN2QxMzBhOTc4YTJjNDk0Y2RmMWJkNDRmMWJlZTlkZTI5NjU0OWQyNzA2NjI3MDQ2NjRkZDI5ODk4OGIzYTIxNDNiMGE4ZjdmMzFkMWQ5ZWY5NzQ3ZWM4YWNkNTQwMWY0MTRkMzBlM2U4NDlmYWUxMTBlMzkyMWE5MzNmODIwZGMxMDQ0N2I2ZDIzMjExZDIxNjQ3NzY0NjU1NmFhMTdjMzNjNzZmOTQ2ZWYwOTAyYjdjMjAxNmRiMzIzMTQwMzJlOTZmYzE2MjcyOTA5MGQwMzQxMmFjZWEzZDY5MTFiYWE3OTUxMmI2OTE2NDAyYTliM2ZlZDYyZTI0YjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WauFYcXpA5bgGzfVKuKtAXDfGIztN3hg8bVvOy7wUVDxXLzZf0jmzUJ7mwwezXhhEreBJlp8WGWqrCyOlmMv4g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110358_26_2482_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.373Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkNpMVVmZ1EwNHFSQjdFSFI2cm9Ec3hNTTdQYTJNVVVOczZ4akt5RVNoOVd0MlM4Y1hGaU5jUkZBOE1UZkE3SFJ0SlMzM25XZ2RWZ1lqdzM2UVE5bzhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNThfMjZfMjQ4Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGE1NzAwZDk3Zjg4NTY1ZWYwM2Q0MTNkMjNmZDg2ODQwOGU5ZGVmMjZiOGNiNDMzN2JmNjBhZjY1OWY4OTgzM2ZkMjZiYWExYjYyMWJlNzg2ZWI1ZTQ0MDdiMDMxYjc0ODEyNzRjZjU0NDc4ZTFiNWUzYjkzOWQ4MmEyNzFjZjAyZDAyNjFiMWU2NDliMDNmZjU5ZTQ2M2IyMzcwMDk2ZmFhMjRkZWE1ZGQ3ODQ5M2EwYmQwZjM4MTQxZjlkNjQzOTkzMjg0MmRhYTI3NmQxOTRjMjVlYmIxOWU3NTkyNmE1YTQ2ZDMwMGEyZjU2MDRiM2RiNThjMjk5YjA2NTI1N2Y3YWE0ODBkODVmOWRhY2I5ODYzMmI5YzZiOTNjYjkyMzE4ZDUyZTJjMThhODM1M2I3OWU1ODM5MTNiZTM3NDJkMjJkY2RkNzhlZDcwMzJkOTJkNGNjMGFlOWExZjI1YzgzZTYxYWE1MmFhZGVmMGVjOTVmY2RmNDYyM2ZjY2YyNGJjOTg2NzZhN2ZjNGRkMWEwMjFlNzQ4NTJhZjZjOWM0ZmVkNTFmNzczMzY3MTUyNmJhNzY1NWM4YjU4ZTVlNmQzYzA2ODE4MzY5MjM3MmQ2Y2YyYWNjNTE3NTUzYmU4NzVkN2FkM2Q1ZGVjNGUzOTBhNzI0MWQ2Njc0MzUwYmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.5dE9sbwCqOtoofTaUM0thvh93pXsqR_k7pvWVBGICMylu9T5uO-WWAtACRanfZwqYeuSN7joReSXQtt2mt3Vtw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110358_26_2482_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.376Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjkrbE1DRnVIRjlpa3AzdlBTbk9tWlc1eS9PYU5QaDlXdUozeTFOalpRQ09OL1pEK2xoa0hHY1BDaWFNVWpnQ09hT1U0bXVIVEZoM0krK2NDSmJQY1JnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNThfMjZfMjQ4Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTUyYWNiMjMxZTZlYTVhYWFmYjkxMjczODk4YWMyYzRmMjZlNWFhY2M0NmQ2ZjRlZWU0ODViYzhjNzYyYWQ1MGY5YTQ0NjVlMWMwMWVkYjM5NGIzNzNkOWE2NDNjZmU1NDg4NjU0ZjE5MDIxYzdlNzRhYzA5OWQ2YjY1OTc2ZWYwMWE5MzVjZTYwMmVjY2IwNzYzZTc2MDNmNWExZDZlZGY0ZDAyNTM1MTMzM2MyNmY4NThlMGU4M2VmZjUyOTIyMjhmNTBjZDYwZGFiMjBkZDU2ZDc3ZWVlZDMzMTI4ZTdmMzM1MWIwNTMwZjNkOGUxMDM2ZjY1M2NlZmI3MmZjNzE3ZmFmOTY0ZmY0Y2M3ZTFlOGEyZWNmNWUzZDVjM2Q1ZmY4ZTU4Mzk0NzFmZDUyODgzM2FhNmQwNDcyMjlkMjUzZGZmYzllMjM2Y2VkMDg3NjIxMjFiZmJhMjQ0MGQ3ZTRjOGVjODgwZjg1NzBhNTg3ODA1N2U2N2E3ZjBhZjQ4MDE3MmY5NDE5OGViNjczODdlNzk1N2NjYWIyNmMzOTRiYmZhYTY3N2ZmMzI2ZmI3ODg1YmNlMjYxZmQ2NDgwMGRiYTI3ZjNjYWNkZDExNzQ0OTdmNDc5NTFhOGUyMjg5ZmMyNTFhYjdkOTViMmFiOTgwZGJiNmZjNDM0NDAyYzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.C8OsJkkGSM4INiaR2YoScp49EomunywBWpNssdrAX18dnd3jJuxGl0_oOAATE3lX1qZvvRWh8tLpCPDOxesFdA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110358_26_2482_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.379Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InJ3SGhDMlgxeldPWGpGM1VHZGxrOGNyVkhQVmJ5eW85V1c4SytPYnE3Y1VwbWV0TnhkY1NvNzRGcDFQN21WR2FycHN1WUJ2OTVNbkxiWGMreTF3Vi9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgxN18xMTAzNThfMjZfMjQ4Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzFhMzYzYmIwMTE4MWI0NmU2MDZjNjhmOTE4OGU3ZjU1MzhmMjRmZGY3Njk0YzBkZTE1ZTJlMzRmNmU2N2YzMDI0ZmQ0MzA2MzBmOWU1NGE0NTg5MWJmMTI4MTM1MDg0ZmVkMzFjOGNlYjBhYzUyOWYwYTBhOWJkOWFhZDBiOGM2MzRiODBmYTJmMWZhNjY5NDA2NGYxOWRmZWRjM2Q4MTdkYWE0M2YwZTg0MzlhMjYwMTNhOWU3NDUxNjhlNjAyOTg5YTE0ZTVlOGVmMTczYjFhNGNkODVjOTNlMmZjYzRmMzUyZTgwZDA5NjVjMTQxYWQ5N2RjMDRhMGZkMzQzNGZjYTVhYzhjNjdjNDMxOTI0NDM2ZTc0ZjA0NWI1ZmQyYjg0ZGQxZDAyNzBmMmYxM2VjNzE1MDEwMWM1ZTgyN2ViZThhOGU3MmIwYTU2ZDVlMTFhYTM4OWY1OTczNmI2YTRmMDgxZmU4NmY3NWU2ZWMzY2Q0YTk0ZWVmNjVjMDAxN2MzY2VkNDQ3ZDE2ZWRkZDZmMzk5YTFjZTNkOTI4M2Q1MjM5YzdjYzBhNDRkODAzMzM5Zjk0YTdjMTcyYTUwY2M2NWVkZGZkMDdkZjFlNTVhMTJjNGJmZjUwYmNjMmRjN2U5ZTBlMDNhZDdhNzQxZDVmYTU2NjNiNjQwNDQyNzRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4p4vwQ8TPr7AEY5hHN7NPWmAZAvZQXywRk1okoRZ4DoK8Y0re2lec70cBA11cjskHqr6p7exr5s86HaUVZX9TQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220817_110358_26_2482_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.382Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjcyZGhzOFBGWkMxYkpwREpkRndVR2F4K0hhSDVIdVFUd21NbWpaNTdaM1Rob1JhUDBxdURnR0lpRFNZNHlWK2hicTVjdkF6NWpjWnZSL1FnY0RreGdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjJfMzlfMjRjY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjNiNDQzYTZjZDNmMjRmMDc4OWFjMjUxZDVmM2UyNTU3N2RlZTFhZWE0N2VjY2RkNWUwNjgxOTlmYjZkMGViZGY0ZTc2NGY1MWUzY2JiZGNkOTAxOTk0YTk5ODI5NzFiZTdmMTBiODE5OGUxZWE4NzcwOTYzMjM4MTE3ZDJlZjU2Y2QxNzA2MTNiNDAyYTVkM2Y1YTVjOTZhOTljN2YyYzAzODI2MzAzN2U3MjI1MWYwMmE2Y2U3NmY1ZTM3YWRhZjdiZTU5ZmU5MTZlYzJhMzEyNmZkZjM1NTFiOTY1ZGFhOWQ4N2NhNjBjMThlZWYwZDgwMDNjMmJmNjkzMzBiZTFmYzdiNWZjZmIyNmI5ZDZiNmQ2MWQwMTU3Nzk3NmQ4MDMwMzRiY2JlYTQ0MWUxMmJjYzAwMmQzZmFhOTA5MjBlZjk1MzRmYzgzZTIyNDI0Y2U5NWI0Nzg2YTBlOTY5YmNhNDQwYjE2MTUzMzdlZGQxNmM0MWVmOGEzZTExNDdlZmY4ODhhM2QzZjQ4MjZmNGRmYzA5MzRkNDAyNGEzODliN2NmYjViZGNmNTk4YzBkZmVkNTZlMGQwYjRlZTA5OTE0MjI1NjFhZDU0MmJiYzQ4MjI5NTcxNjg3MDUxNWMwZDdhYTRiMzAyZWM1MDkyNjgwMDIzYjg0ZWM4OWU0NDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dz-ykc1cLhIXizJJ6zICMlehtv1_IQ4Je0cmABMJkLoO1-rR2-JRZ1kazljsQim5d8d1mhbMAGHP57XEyKbM0Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103622_39_24cc_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.385Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InNBMjlrTXdnVUkrdFVacC9CWjhiYnJvYS9zRmlxb1hham5RL0lrd2kzdW9NYkNzZ0Z2cmlKRldOajRzQWdkWUg5N09PVjZOWEJXdTNLcEZYOUsydG9nPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjJfMzlfMjRjY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDY3OWZjYTEzM2VhZWZjMDAyZDQ2NDc2NzE0ZjRhNGQwNjViMDc5NGNiYTNlMWEzNTM2OTk0ZjJkZDEzYmUxYTNhYjQxNGYwY2QyYjY5MmNkYmE4MWVmZDk4MzdjNDA1ZWM4ZjMwNDU3NjliMGE4ZmFjMWM2N2FmNDkzZTBmM2NkMGMyMTQ0MmJiMDlkYWVkNGEyYmI1ZWYxMDllMzU4Mzc4YTc0NDUxZWVkNGI2Y2ViMDM4ZjUzMTE4OTBjZWY4OTIxODJiYjIxNTgzOTcwYWVkZGVmZTgxMjQyODY2NzU3Yjg5YjczZmY0ZGRkM2NhYjZiNzllZjQxNzVhNjM3OWJhMTA1NjIzNDM1ODViZTA1M2U5ZDE2OWM3ZjA1ODNiYjMxMTg2MTg5OWZkZTcwZGEyMjgxNDllNDIzOTk5YzA3NjI0MzMyMzNjZGEwOTAwZWU4NTQ0MDVkZjAxMGViYWI1NGUxYzY0M2NmNDZlNDRmNTEyZmQ2OWYyNmQ0MTE0MDE2MGU0NDdlODI4ZTQ5ZDRkZjJmYTkyNzMwNmQzMDVkZDQzOWViYTQ5YTM2NDkyYzI3NTMwMzI3NGNmYzBmN2RhOTM2MWFjZTg5ODM5MTNiODI3ZGJmODBjNDhmZGQ3ZWY1NzNjOTIzNmRkNTM4YTRkYzc4ZjllYzE1YTliYWNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MLdMy6oF1Nngpck6Ukaqr6fLlQ5u3KlSIQg11GEPYyN33yD5AkK2I9mQSHcS1LMoXBlHEGLOT7R8w_t0f0pa1g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103622_39_24cc_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.387Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImY2d0dRTlRpanNiQlhpUjRlREhKZEZPTGhCaXNQRGw0NlRwTmNsNjhWOFNVbytiNzJqaUVCd3dSeWFVNnFmajlja0dTSk0rM2FkTVArTzAwZ1h0YjBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjJfMzlfMjRjY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDQ0YWNjMzBjZjNmNDJiZDhiMDdiNDA1OTNmNjBjNTYyOWQ2ZjNiOWMwNjczOGM4MTQ2ODllZjk5ZmYyYjc3OGJmYmRkNzdkMmFiM2U5ZmE3MjQwOGRmMzQ1MmY5ZmFjY2Q2N2Y0NDQ2YTc4YjU4ZjU3ZDdiZTc2NGZhODYwNmY3ZGE0YzVjZTA5YjY1OTU0OTA5Mzk2NGEyZjFmOGMyZWY1YzNhZjFlZWViMDY0ZTgwYjg5NTBiMTZkMTllOGEyNDFjNmU1ZmU1NDRhNjQ2NTdlYzUwZDYxMWEyYTBjN2FjZDg5MTdlMmViMzE4MDc2MGMxNGE2MzAyNzk5NDYzYzI4ZjQyOWYwMGY1OGVkYzBhZTUwNGM5MjJlMWJhMDI1NjcxYTY1YmEzNTdkOGQzYmMzZTNkNTIzNWY0NGFmYzllNTBkZjk3MDZkOWQ1N2NmOTk5NDI0NjhjYTliYzE1YmE3NWUxMDk3YjFhNmI3NjcwZmMxNTc0YTMwYjcwMDVjNzg1MWZmNzJjOGJmMjU0ZjRiZjI4ZjZjYWUwMmIzMzBiYzQwMWJjMTI2ZjdjMGM3MTMzNGRiOGQxNjgwZDNkMjJjMDI1OWRhOGJhNWQzMmZmNTdiMzU3ZTQyNDE5Yjg0NGZkOTU1ZmFjOWJhN2NlMzJlMDVlNzIyMzFjNzVmZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.B-arlej0SCZtbvRnFDBzlzuqbqtH7Dv-UE07CmOZaHjQ9_4tlOsrZ_7UgfrxEhPTmKtijKFUz0WiA6O__CuMug", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103622_39_24cc_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.390Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ilg3blhUQlg2RFJkWlJ5T241aWJzQmZWZzZteFpaTllJVm0xTVlVNFdmd096c3p4UjZhRERIUm12S05UMjlwUmFRTUdsWDFZWDZtVVVhR2dzRmtSMURRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjJfMzlfMjRjY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzA1Y2JjZmYzYzZiNjM0ZmVlODE1NzZhOTVmMDUwNjAyZTBlNzI0MDE0ZGYyZTBkMTcyNWE0N2M3YjJjMTJlN2FmYTE0NDUzMjM0NjhjYjk2YTRhZDg3ZTk2YmQyNGE2ZjA1Mjk4MTNjOWFlMGFhOTlmYTljOGM1MjM4NjRkNTdiOGI2ZDk3OGRiMTgxZjc3OTgyODM3N2MyNTBmMTk5OWYwMjY2ZjBhMWQ0ZjZkNDI4Nzc1MDA3NzllYzI5OTQzNjI5ZmVhNjViMzUwZWIyYTJlNDBjY2YyZjgxYmEwYjFhNTEzZDI4ZGFiZjVkZDA3YzFhZTUyNTIzYWI0M2Y1Yjg4YmM0NjFiMDI1OTJjMWE3ZjFiZDEzY2QwNmI4ZDU1ZGM1NmU0YThhZWVkOGMyNzUxY2QyOGVhODZmYzViOTMwOWE1ZWM4NWIyZGZmY2MzYTM4ZjFjZDk2YTU0NDM5ZDk0OGQyMTkwZDk0NDg0NjU5YThlYzU0ODY3OTVkZDI5MDRhNjQwMzEzODllYjUzZTgwNDQzZjczODZhNzk5ZTczMmUwNTdiOTk4Zjk3NDdhMTAwMWFlZGQzMWQ5OTZiMTkxNThiZTAxYTY2ODYxYWVhYmRkZTkyOTBlZGQ1MzAwOGE5NjAyZTIwNGNmMGNjOTNjYzcwN2ZiNzZjMmQ1YTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HcyrVJr0mrO_6QAvMADJo-avBP7wU0ULOh0M0tSGZYZ-EYWIRr7MfPXKb2Mzdc4k30w_Jq-uRnuKPVTrqIcbcA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103622_39_24cc_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.393Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBHWmFZc2RGSlNpQ2V3bU1TdjUzRk8vY1NNQytrZmRva2VHVEtqMDFiNTFQVE9seXpyY2NNRUxOejVrTm90bmZmTVZEc2ZjRkFIVFdXVWdodEpXMWNBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExN18xMTIwNDVfODNfMjQ5Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjM3NTZmMzk0OGYwY2I5NDU1ODE2YzI2Mjc2YTE0YWVhNTlmM2JlZWJjMjE1MDRiZjA2NjdmYWVkY2FmZDNiZDc1MzMwNTU1Y2MyNTU4YjcyMmEwMjkyOTRmN2JiY2JhMDRjNTUwYjdhZThjMTE1YTgxZTMzZWEwNDc0NWQ0OWZmODJlMmJiMWJkYzllZTM4MjdiZWY2Y2JiNmQ3ODA1OGRjNDUwODhmMzIxYTVkZjBjZjJjMjhhM2M4Y2U2YzFkNWYzOTE0ODRhM2RhMmY0MGI1ZDZhMjc3Yzc2NzAyYWY1MWQ2ZDE1NzRmOGEwZjdjOTZhYWE2NWQ5NzE3ODU4NzI4YzJhZDhkYjI5ZDU2OTQzY2M4MjVkZTA1OWEwNjAwYzE3ZDMxZWFiMmM2NTQ2NzFjYjBiNGE5YjM5NjVmZWVkOTg2YzU5ZjMyNmU1NzliNTM2N2MxMWE1ZmIxNzQwY2Y3MTRlNDJmNGJhNGY4MTAxYmYwZTQyZDY5Y2YwYmI3YzhkNTZmZmFiN2E4NDdiYjdmYjM0NjRhNTMzZjczOTMyMmU5OTdiOGY5MGU4NTE4N2ExNjViYjVlNTM4Njk4YTRlMmY1YmI5YzE2NWJlYjk3YmRkMmQ3OWY1N2E3NDkyM2FmYjAwNDA3N2VkYTkzOWE2ZjA0YWY2ZmY1NzQ4ZjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.xGsEXQ_262ZyD74I1YAVYA0eoZ4-xsQb2VI4Q7jqP5ASPQLadNWcRfV8g4_2wJXcaDQVO0dUVdtET0zh7Phl4Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240117_112045_83_249c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.396Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjRIRUV6RGZuWkUzRVg2SzltbzJocmF5Z21wK2xkdG1tbVJOV05iYW9QVEVpZEZqd2piWW9yNHVROVRHS0tXZnpvQWQyNEtoVmVNUFg4eUhSZWR3MHBnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExN18xMTIwNDVfODNfMjQ5Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzNjOWViYWZjODE2YjgwN2U2ZDY5NTY3OTEzOTZmOWIzNzM3OWJmMDEzNDFhYzc4ZTQ4M2JmMmJhNGFkZTE1NTQ1ZWRlMDI1M2E3NTJlZjk1NjM1ZDU0ZGQ1OTJkMGJlMDQ2Nzg0N2E1YjgzOTI1NzExYTMyYzk0NDY5ZTNkOTA1N2IzYmM5ZDVhNjNlM2FjM2E4YjVhMjcwZjJhYzJjZmZlZTUwYWFjM2ZhMmFkNjUyODEzMDdlY2EwNzE5Y2JiMTNhODhkMmU4NTM3M2UzNTEzNzAyZjU1NzYxODE5NzUxMDdlNTMwOTZlNDI5MDM3MTI1YWU0ZjkxOWZmYzliZjYzNDAyZTJiMDU4NzFkYzNhMWI5MjA0NDk0NjFiMGFmNzFlMzA1MGE3YTg2ODBkNzE5NzhjOGQzNmMxNjQyZDlkNTVlOGQ1NzAyNGQ3NGU3MzUzNWYwZDBiZjUzNzcwNDE3OGEzNjhiNDM1ZjU5NmRjYzNkZjZkYzRlZTkyMmVhYWExZWY3NjJiZTFkYTI2YjUyNmM4ZjhkNzE2MDFiYjkxYzUzNmRkNTViN2M1MzA5NWQ5ZGFlYTJhOTBlYzExNmJmZDU4MmY5NWRkMjcxNTNjY2I0NmE5ZGIxYmFlZDQ5Zjc1ODYwNjA1MWUxM2Y3MWE2NDY1OTg3OTM1YThkODBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.QFybD9_60jXVFUBMTS63jCGd_yiFgdCHonRPASOLNi9cHH1v4ZSKUpL2Bl74NRboOl1yDr23KTBTszhhAdQIeA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240117_112045_83_249c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.399Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjViSzRuaEEvTnd5aVViTjNhdHI1MW9DaE95ZmNCdjFLSDZVTEhMWXBMMURrV0xmdTJ0bGZiZWVRWkN3UzZsLzdVazFtMitCNnRZclNzdTVSQlB3NStnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExN18xMTIwNDVfODNfMjQ5Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjUxMGM3NmMzYjMyYzA3NmRlMGUxNWRiNGEwZjQ4OTdmNmM1NDMyZjQ2M2YyNWY0OGQ5MTJiMDkxZDUxY2Y2NDQyYzQ5NTI0OWQwMTUwZGM5Y2Q4YzhlYTdmNDhkNmRiNDJiZjkxZjJhMWVkNDBiOWY1NWQzYjQ4OGYyYmViYTRhYzJlMWZiZTMyZWZlZWNmNjEwMzAzZDE5ZjczNzg4YWMxMjgyZmRlMmZmNmI4ODMwY2ZlZjI5YjUxMmQ4NDhhOWJmMjZmODM0YzJlODY5MjNlN2UyYzUwZWI1Y2QyNzQyZjA3MjU3YjhhNDA2NDk3NTc1NGRjMThhMjJjYmI0MGJhYzczZTEyNzIwMTQyZjdlMzBlYWRlZGJlYjJkOTUwYmM0MDdjMTMxNWM2N2Y3ZTYwN2FjNTNiZjViMWVmYmU5NzE5ODcxZDFiYzM5MmUwZThhYzE4ZjE0NWMyNWI0OTdmMWUwZjYzZDRlNjg1MTFlZjUyYTkxNDhlNzRiN2E5YzA0ODIyZDkwZGU5YmZjNjFmNTVjNWI1Zjc0OTZjMTRjMjEzOTEzNTU2MWU1YWZiY2NlMTY1ZmQ4MGYwZGYyNzI2NWM1YTQyMGIwOTI2ZWJjZjdjZTJjMTk0NDI5YWE5ODFlMDBkNmFmYjc5MGYyZjA4NzdiYjYxZGJjN2I2MDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XTrUNUmVnT61KRfjYHSmTl4_8McEYFZfiVjm24JgvbFXQCX5eVD8b5A_lZ4LOiwpPRde_Qo6Q_wpvTCVoNWStw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240117_112045_83_249c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.402Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImordzFlSEI4TUhSUnUyeDVKNkdFU2FiVXhmSm5yWFBFdG9jY1FHNTU0KzZKSno1QWRCU0dqV0JIWTRteEpHdnpFREdOWXYzRkNHR2QyU2tEelFEUkt3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExN18xMTIwNDVfODNfMjQ5Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDM1YmYxNzA0YTczZTdhNmNjYzczMjgyODcyZWZjYmUwNGVjMGJhMTI4NjhhYTQ4ZGE4YWJhMjhkMGRjMDVlZTFmZDU1MmM5ZWI5NjRkMWFhMGQyYzRlMmMwMDc2MDkzMDZjNWE1NjI1ZmU4NGNiMjE4ZmY4ZmYwMzkyNzAwYWQzZWU3NzQyYzNjYjVkM2Y1MWQwZjlhOTg1ZjljYzZhMjM4ZTEyMzBlNWVjMWI5YWI2NWZjOTQ5Y2RhOTM0MDYyOTQwNzQ3YTg3ZDQwNTVjM2EyZjEzMzg3NDFhMzgxZGViMDBhNmU2NTcwZjg2MDgyMDhjNzc1OTUzODQ4NGVhMGFkYzBlYTk4NmU5NTQ4MmFkOThlZWVhMjM4MmEyNjQxNDU1NzkyNDNkMWM4MjlmYWE5YjA4ODFiYjM0OThlNTU1Njc2ZDgzMDAxNDJhOTgyZTViY2E1M2NlNmE1NDc1ODU1ZmQzYWZmYTkzZDg0MTc4ODkzMmVmOGZhMjc1NmVkODk0MTliYTJkNWFiNmZlZWM0YmUxYTZlMzFmZWY3NzkxYmZkNDViNzIwMDQyODNhOGQ0YjlmMDY2YzMzNzdkOTlkYzNmMDc1YzIyNjRiMTgyOThjNTU1MGUyZjViMzlmNmFlZWUyYjRkY2RhYzcwZDZjZjBlM2I3NjA1MjZiMzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cFG-ZCIQjdI9B56j2t8ApWuKtYiV7DZVnixwYDReQ7-d_K4hn6R6WyDQH9bMQwbvBAqg90N4HrIiMkLxjh7b_w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240117_112045_83_249c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.405Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ijdya05pV2d4MnNIYzFtVDFWRXIzNzZ3SUJwRUtvQkNlNUt4MmRlS3lkY0JiRjd2NXI5Z05NYnJRSzFLVUcxU09lNUNyNTYvdUVad1RrOC9UK2MvWHRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM1NTlfOTdfMjQ2MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWNlZmIyNThiNDAyOTMyYTkzODQxNzdiYzkxYTM5ZDIzMDcxMWFiNzM5MDIyNTExZDk4ODkyODc4YzJmN2NmN2Q2OTU1ZjY0MTc0ZmM1MDAxYjVlZmUzNjdkOWQwZTVjM2I1MTNkMzE1ZmFkZjRlODNmNjllOTA3YzM5NGM0ODYxZGE4NGIzY2RlYzVhZmJiMzkxZTc0ZGViYjY1M2RmZGYzZGMwOGExYmMyZTc4ZTFkYmExZDc1MTAxNDI0NDg0Zjc4ZmQ2N2E2NTk1OTMxNzkwYTk0YTM3MWFjMDEyY2UxYTY3M2FjMDBjNjdiNzkwMWNmMjY4MDE1ZGIzNGI5M2RlNmMyZjdjOGJhZDM0ZDQyYTM2NjdiYTVjYTBiODEwN2E5ZTAxNGQ3YTFmNGMxMzMxOTNhMzMwMjAwYWQyNzM4NDBmNTM2ZjQ2NjYwMDBmMGNjMzlhNzk1MGZlNmMwN2Y4YWJjYWIzZDgyOTY4ZGRjZTQxMDIzNmM5ZTI1MDIxMGEwMTc4ZTE1MGVlODY1YjBlZjQ0NWY1YWY3Y2M3MzYzZTc2ZmEwOWZlODNmNGM5ZWQyYWJiMTc1ZWZiYzY3OTQwZjA2ODQzY2IxOWI3MWNmNWQyMDUxMjhiNzFlNTg5YzVlODJmZDliMDY3MWVmY2Q0YjVjODIyN2IxNmNhOGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MOgIAIpR2rzdE2Twctx5xbnOdn-SaDi2FJWqj58OY5yW97_28r0rqiOhpfFZ0mfmyRI7nqIvRWRyHhxEwWylNQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103559_97_2460_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.408Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImlrdHBuaEhIdmtCbUV2VnBlbmFGTHI0QmZJY0JFV3NRYklqZmJ4TE8xNGtqY203U2RpWkRpSFZnUzhndFZ2NmFGU1ZIb0VDZHdacHQ5Y1ZJSlFJd0NBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM1NTlfOTdfMjQ2MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2NjMzU5NjlhYWM4Y2E4ODZkYWFmMmU3ZjU2ZTFmOGI4NWVhZjYxZWYxYjZlZjUyZDQ2OTRlZjM0YzAyMTM3NjdmNWYxYThlMDdjNTU4NTBmMWU1ODk5ZjFhYzA0YmQ2MjZlZjBiNDNkYjRhMDhjMTQzOGJjMmQyMDVhMGE5MTdkNDJkMzNkOTQwMjY3NmFjNTg2YjAwZmZmMTQyY2VmOTE4N2RmMTUwOTFiY2FlYmM0MDE2NmM4NzM4YzAzOTIxZTUwYmY1NWQ2MjQ0MmJiMDdjMjMzMDdjMDZiZjA5MjRiMDgyYzQxNDllYWFkMmM1Yjg5Yzc5MWY2NmZmMjlkNTUxZWJiN2Q4YzUxYmQ5OGQxOGNmYWU3NDk4ZDQ4MGVlZDBhMTI2MzhjZDllYjc3MGM2ZTU2NzA1NzI4YWZkNmE3OGQ1ZDdjMGY5ODhkMjNlYTRhZjg0ZWFmOWMyZWE4Y2QwYjcwMDgxOTM5ZjY1M2UzMGQ1N2E2M2Y2MGYyNDBhNzBlOWQ5MTUyZDA5NmY5YTJkYTM4OTllMzFhYjg0YTdhODIxOGI0ZGFkMDA3NTAxOTRmZWUyODRkZTA5YmY2ODM1NDcwZDI5N2VhNTA2N2ZmNDUwMGFjNTU3ZmI1MTU1MzIzNDBhYjVlMzI0NDQ2OWU0YWJhM2EyMmNjNTI2OTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TxHLgrOOzVAg23hcA-UBv3ucmPGd54SREneN0qkRqXRVp2iMb_5ERKwDsh6c0_obUdcC4oFuQt8G_dbzjqYVnA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103559_97_2460_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.411Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik1jWWUxTTNPemtuNTJRZUYybVZRdGwrdm5ST0NRRkZSOW4yZEhSL2s1Um1LR3E0WjA2aGhrSGVQOUQrMXhLVWdBdmxXTGcxVTBTM0k1N2cxNnVmYTJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM1NTlfOTdfMjQ2MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTA1NGMyZGFkYTZhYjM2ZmYzMmUwYmM2Yjk3MDg3N2E3ODYzZmIxYWI0NjdiNGFjNmUwOGNlYzdlN2RiY2I5ZWU3NDA0ZjMwZDIyZDk1OTg3NzMyOTE4MjQ0NTc3ZjJlODllMGMzMzljMTBiYTRiY2I3ODIyOWQyMjk4NmRiYTFjZTg1NDQ2Y2VmOTMwMDhlMWNhYTI2MTlkOGYzYzhhZTMyMDcxMWU5YWQxYTM4NDQwZDQ4ZmQwZTRhZmEyY2U3OWUzYmNiNzllNWIwYjY0M2Q5Yzk5MTk5ZDUwODY2NjY0ZTg0YjAxMTJkNmY5Mjc3YjVlNTEwZGEzZDE3N2U5NmVmOGIyZDEzOWRhYjU3NWRmYzRlM2IzMjQ0ZjRiNTJlYjhiNjgxNzY1YzFhYTcwYjgwNzY2YmFmZTU2NTY0MmUxNGZlMWE3NzY0OTdmZjUxOTcxMmM0OWM2YTExOThkMDY2ZTQ3Zjc1YjkzMTY2ZGZjMmQ2Y2I1MDMwMmE0NzMxNTIzNDkzMDc1N2I2ZTU4YmNmMjQ5MDkzNjQ3ZWM3NTA2MDEyYjZmZWM0MWQxYzZlNWVjM2ZlN2E1NjQ4ZTA0MDM5MmRiNGMxYjQ3YmFlNzNhZTI3OTg4MjM3YTdiNWIyY2ZiM2FmNDIzM2E0N2IwMGQ5Yjg0OWYyMWVlMWVkYWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.8_A5RoAQKj7vtR6dXnSUnd6tQWepCreJ18wJc_72oKwZy_5mXnyqGkfeAii1Rzif0s_PLmE6bEQBZpj_x-hcGQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103559_97_2460_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.413Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImdJb0FHaVUwMG5lTlNWRGc0MlBZeWViZGJLNzVxTUFJU2hBRmQ0bXB4ZGx6YWUxdU5KZVJBbGRjZmQyOXZ3NW9JTW4zWGZkVDhJQ0hOa2g0REtPNzVRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDkwMV8xMDM1NTlfOTdfMjQ2MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjNhMWE5ZTNhNGEwNGI0ZDk1YzViMWJkOTk3MzMwMzliMjRkYjQ2YjAyMDUxZDQ3Zjk0OTdjNWQyMmUwYzA0NmE2ZmVlZTUxOWZiY2VhOWM5NmYxZDEyMTU2YzJlYTE0MDI2OTY4ZDVmOGM1OWIxYTU0MDFkOTVhNzA4Mzg2ZTFmOGNjODU5ZWI1NDVlOGVkMTU0NjE4YTFjODY4OWY3M2ZmY2M3YzM4OGIxMmU3YzdlOTYxY2FkZDdlMTM4Mzg1MzFlNzQ1NmVmYmVlMjQxZTM4ZjczYzU1N2MyY2IyZGNmMTJmNGFjYmUxZGE4ZGE0MDYzZGRkODhlOTU2ZDc1ODc2MTYxMTFiODllZjZjZDcwYzdjZjRlM2Y1MmQ5YjVlN2JiNDhjOTU2N2RlOTI3ZTliMjdiMDEwOTlmMjMzMTQxZTZkYzkxODIxNDRjMTAyOGYzYTIwYTkzYzRhM2UzODI1NThjNzFhMjU2MjUyZGNiNjY2ZGUyY2QwYWU4MzdiYjRjNjlhYjNjMTQ1MTAzNTQ4YjgyMGVjNDVmMzM1YTBkOTU4MmM1MTEyZDVkYTIzOWE1ZWU4Y2JjNzUzZjYzNmYwOTJlMDViZDNiMjM2OWU5NjVjMTNkMTM4MTU3YjgyYTI5MDQwNmVlZjhjZWQ4YjQ3ZDRkM2RjYmJlOGZhNWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RjRw62QctOEfc-WedIDQPnEI19XLBKGCQmpd6KGDCHtW373t53f9b7_0C2JbM3JLnCnjDOetFZc1T1hvQqBgPg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210901_103559_97_2460_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.416Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkxWMEhRdjhxU3dPd3pBYzVzbDV2UnN0aGE3TWx0aUh0eWs3bFI2ZmhqNU5VcnZmbUtuOHlmeUhkVi8zckU2SVpqaFdJT0dnYWxsTmVYSWswdGcydVlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTIyMTdfOTBfMjQ4Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTdmNTdkYzU2ZDdhMGNlNDgxZDE2YTYzNmEyMTg0ZjRmZWIzY2RmZWMzNjgwMGViOWY4NzE4NTUzNTIxNTVlY2ZlM2JlZjc4NmZlNzkxOWQ0MWNmODRkMTEyMGNhNDFjZjgxN2NhY2E1NDBhOTc0YWM5NmEyNTE2MmQwYmUzZDg5YTQ2N2IwOGRiMjY4OGY2YWY4MmIyMDQ5Nzc3NTBkMGFjNzBmYWI4ZWE2ZTE4ODQ2ZmIwMDAwYTIyY2ZiY2M1MjBhNjljZDgxMGNjN2MzNDRmNTlmMjY5ZmU2OTE5Yjc1NmNhYTMxZGY3ODYwMmNmMmZhNWRmMGI2MTYzNjlhYWY2YjIxNmQyYzY2ZDIxN2EwZmY1ZTI4ZDhkMWYwYzI4MDAzOTg5YTFkM2Y1NjBjMDcxMTBlODc1N2U5NDQwYWNjMWY4ODc3ZGQ0NjgxMWFkYjM0ZGI4NDBhY2M1NzNlNTgwNGE4OGMwYjA1ODY5ODAyYjYwZjM0YzFhNWQ2NTYyOWZjNDU0MWQ0NjFlMjc2NTI4N2YxOTE0OTM2YmUyYjkzNmRiMGZhMjlkMzI3M2U0Y2JlNGRjOTVkM2E0MTY0NTMwOGU2MjRjYjc4MGVmN2UxYjI5OWU1N2NiYzhkNjVlMmEyYTU4ZTIyNGI0NzE1NTdlNmYzZTIyMTViYmMzZTlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.vMGF2bqDLYt99qOoZi67J91HfGRlDdQxSy1q0EnHMHXmjif4HqnbS0KK02QME7D8bqIoTlyvwvB6MWw07fBSGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.418Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ilg2REdRVVp0N1c3eEV4bWR5YWtQRFZDaDdqRHB2U0cxS1ExeWJQOWt6cVZGY0Zkd3hLRXlwTi9BZXlnZTV6MGdkSkpFZUhJOXo2Y2p1dVNZZTcwOUtRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTIyMTdfOTBfMjQ4Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDliNWRmYWFiMjJkODYwNjVkNjAxZDI3MjUwNTFiY2EwZDc4Zjc3YjA5NjdhNzE5MTBjZWI3ZDhlYWQ5OTZiNjEwYmMyMzZlYWY5MTJkM2FiZTcxNGFiODQyMDlkMWFjZTVmZmUyNjgzYTQ5OGE4YjQ1NjViM2Y3MTZlZGVlOGI4ZjI4OTUyMzlkNjRmNjI5ODU4OTNiYWY2ZTJlYzY2ZWE1YzYwOWQwMDQyYmM3YWUxNzk0MDFlYzVkNTMxNDRhYzBiMWMwYWE3YmFhYTljYWFjNjFjYjE3MzllMzhjNzllOGE4YzJkYjQ1OWJiOWIzODZlYTgwODgyMThkMTM4MjdiNTMwYTk3NzhhOTk1MTU2ODQxNzM0Nzg1Zjk1Mjc3M2YwYjcwMGYyZDhhYzVhOGE4NzJiNzJiNDZhYzRkODRkMWM3ZjFmYWI2Yjg3NTU3MTY1ZGI1M2Y3N2VlNjdkMTk2ZDJjNDcxN2RiYzllODI5MjQ4MTBiMTI0NWM1YzM0YTEyMWIyZTE4MzY1YzlmZTU0NjU3YTczYTQ0NmRjZThiMDc5MjFhN2JlMGVlNzQ1ZmYwODllYjZlNzRiMTJkZDEwYzYzNTUzZDI2ZDM2ZGFlZmFhMGYzMWFlNzgxOTJmZWFiZWZkMGE3N2IzMmI5MjgwZDUzODAyYzE1ZjIzZDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CXwio3FpbwLXikMuBhat7s2uQSAEdi6dgQWIsGjga2RxIsDZc-ebKkID87NCqZU-WYPt6_NyKHmWTl2ERy5m9Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.421Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik4xODkyd0psR0dQU1Q1czFLRldBZXdkVzhrbFZLc0laWXowd1BRTG9ZN0tHOTdFaGMwV1BVY1dKTDh6WGpWZFBNV0llTDY2TWtXSk9uekQrbE5qK0Z3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTIyMTdfOTBfMjQ4Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTA0Y2ZhNzNlMzAwMGUzNGFlYjgwZGFiZjUwYmE0MTBkYjdjYzNiNGM2M2Q0OTQyMmI2OTVlNmYyMmU0ZDgxNTk2NGVjNmI0ZGQzNWQzZDk3NTNiYmIwZjFjYTc0NGY2NzExZGNkOTgzOWVjNjRlMTQ4ZWY4MTYxNWJhMWY2Y2M4MGUyNDk1Y2U1N2MyNGYyZDY5N2E2ODg5OTQ4Yjc1ZTI1MDQ3Nzk4YWUzYjRkZDUyMTcyNzU0NGEyMDc1MzRlYjUzYmUyNzc0NWVmNTk4NjYyYTg0ZmE5ODcwYWRmMDA5ZDVkMzY1MTBjODYxNzIxMDU5ZGE5YjFjNGI3OTlhNDExZTMyMTZkNWY4NGY2YzgxNDI3ZTZiMTA1YWFjMjEyNzcyMjM3MDcxZGNhYmIwNGJlNGEwNzVjNmIyY2I5NzA1NGRlMzdmY2U2OTlmOGM5NDQ0N2UyMzNiODlmOGY3NjFhY2Q5ODY3MGQ2NTYxNWMyNGVhNjk4M2U0MjUyMWFmNDkzNzE2ZTA4ZDQ4ZWMxNzE4MGViODc2ZmI2ODlkNDIwM2U0NTYzYTQzMDA3NWMxMDUxNjZhYTZjMWFmOWMyMDhjOTdmMTJjMGIxNWQ4ZjE1NzQwNjYzNmUyZTk1MmZhMWJhZGI2YTY2MzMxZjVlOTYwMTRiNjJlM2IyMDcwNTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Yn2ewLBF5eQ7XU6WmV8-WK3i5cAVQCWa_lmtdrQ35pEfL60csQHjshkd9Ku2IOtFW-lF0e7dDBOYhM_Cs3BRXQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.424Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlVmTVViSktOYkx4dXlxMzFMMC8ycmpXd2VFY0M5NTJQNWNFWTcvbGZJQ0hMeUIxamx1S2ZPc0pUNGplTXZrK0laYlJhSjFxQmlRZ3RyNkxRVlVucTBBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTIyMTdfOTBfMjQ4Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVhNmM1MzM1NzE5OGE3YWNjODNkMzZkNGRhNzk1YzU3MGIyYjBiZjNhZWU1YmE0ZTIzOGQwYmUzODJkNDJkNjNmOGY3ZDM1YjZlMDA0Y2JlMDc4NzQ0Mzk3MjBmNGMwNWJmOWIyMTYzOTg0MjE4MDA0MjQwYjllMzljNThkOTU5ZmQ1ZTE4OTA3YzM4ODU4ZDgyYTZlNWNjMzEwYWRmYzIxOGNjZGM4ZjJlYzA2M2IxZmM4NjNkNDQ4ODMyMDgzNmUzYzhhOTNhNjBkYzFjYzZjZTY5MTBmZmUxYjFmNTAzZWU2ZTM5YzQ1YzkxMDU3MmU1YTQzOTMxMzdiNDE2NDcxYTU0NWIyZTczZDg1ZWMyYjFkZDc0MTcxZDNmNWZlN2I5OGFlMWM3NzM0M2E0ZDYyOTA0YzAyMDEzYjQ2Mjk1ODM1MzljOGRlMDQ4ZDNiZTJlM2Q3OWQxMDlkNDEwNTBmNjY3N2M1YTUwY2I5YmI4YzE4NjE2NWJhOGE2NTUxYWFhMjc5ODc1NmUwZjAwMzAyYmI2YmIyMzZmZTQ2YmU4YmQyMTM5ZDdhYzdlYjM5Nzk5ZWY2NTQ0ODc0NWFlMWZjNGVjODA4NzRhMDE4NWYxOGRhMjk0YjNkM2QyMDI2ZTFmMzEzNmIzNmExZTJiNTg1NmZmYzUxNjI4Mzk3NjZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.uL_MheTdV1-VTGJSk46pLMrtNeZwsfCKNxQntAQgB-Aqx1oRPRbgdYlZGTTmKliiX9tfRzjLZrcoFFExB3VwsA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.426Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IllqUjVsODRpdWlOemJNVjBpRjZIUVJrbWtxWWRkYjc4T0o1ci9sbVk4S2QvTmp3dW9xczhGSURCc3JIODkrMVdVUHJmOG9DdDRRNWtJVTUveUo5RXRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjdfMzNfMjRjNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTY0MGU1YTJhYjFjNjhlM2NkNDExNmU2MWNmZGNmMjFmZDQzMTI1ODQ3NmMyOWU0MWU3ZWQ5N2I1YjAwMzE0NWIwNjFlZWVkYTcyMmViNDU5ODg4YzUyOTI2NzIzZDZiNmM5ZTNmNThiZjM4NDBiMmNmNmE4ZGRiMzkxMDk3NWU1NGQ5OWRiZjE1OTc1NDRjODBjODUxNTYxNDJhOTExZjM1ZGU4YTI4NjViOTdjMzQ1NGIwN2M4NTllZjhmYzZmOTlkZGNkNTVkNmMzMDM3Yjk2YzllZmRlZjUxNTMyYzkyMWViZWUzMWQ1NmU4NGJmYmM4ODkwZjU3Njk2M2UzZGY3ODk2MWFkZGFhY2I2ZWVlY2ZjOWE5MzY1M2JlMzc4M2Y4NDQ1ZGU0ZDg2ZGU5NTRlYjQ1OTUyNGJlOTY4YjI0MTJlYjk4YTYwNzI5ZTljMzE1M2RlNWNjOTdiMDA0Njg3MGVhN2Y3MzY1M2Q2NjAwMTc4YTlkZWQ2ODAxZjRiMjZmMWEyNzYyNjU1MGIwNjIxMTNkNjgyNjNkZDBkMGYzMTNlMTM4MDJiNzM2NzBkZDM3OTc4NGQ2YTM0Nzc5MGU4NGU3YTM0NzkzNDdkZDVkMWQyZDRmZWJiY2JhNDYwNjAwMzBhNjg5MzI4NDdhNjZjOTg5ODRjZGU0ZTQ2MjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.m9YaYg0Qy3xDH0q7mBCHrNsKj0U6eNcvct47UKNWw43XftBphpE7Y5RaFVz0wysitvf5k2Wc1rFfuwiaH66kKA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.429Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlQ1dm50SG9WZlZ5UGlJbHlSOXFyMjFkdVNmZlorMHVmV0c1d1FmTkNiSlJpTGdCVWwzTnB6K1BFanpVRXZ1ajJxVm9vV3dmYmpBWkt1emtDRUhsSVRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjdfMzNfMjRjNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWVjNzUzM2RmZjNkMGI4MmI0MDcxMWFiM2Y5MzEzYWM1NzdlNmI4Mzg3OTMwNDc2OTE3NjJkYmM2NjVhYzAzZTIxNjM2MzJlOWU1MzNjYjFiY2EwN2RmNTY4ZGE5NzVjMjVkN2NhNmViMGFlMDg5MGMyY2M1NjY0NTc2OGQwZDZiNTRjNTlhN2FkYzBiZWQ4OWI0ZjdjMjU1ODgyOGI0OTE0Mzk4NjZlYzU2OWE4NjU4ZmI0YWVlMWZjZjM0Y2ZiMjBkZmFjNjhmNDhlMzI0MzE0ZTc4MDJlODEyMzYyYTY5ZDVmYzY3ZWIyZjljYTVlMzg1MTA2NjEyZGE2MzI4MjJlMjZiMjk3ZGVjNTFjM2JjNDQyMTI2MjlmMWVjYjNkMDkyZDY0Y2I0YzM4MmNkMDVmNGY5MzZlMDJlMjRjOTNmN2RjZDIwMjE2NGYyZWE4MDFjYmMzOTQ2ZDc5NGUzNTcyNTc4OTdlZWU2MDFiNzFkZDRjZjFjMTZkOTI3YTg4Y2QwOGQ3NjU1NmY0OGNmMjFkMjljMWY1ZDNlMjY5N2NjMjY4Yzc3ZGY1NjM2MWUzOWQ2YzIyYmEwOGU5YmMzMmVkZDdmNjdmYjRkYzM2YjhlNDVmOGM0MmNmNjIwMmZmM2I5ODgxYzI4YzAxNjUyYWZiZTIzZDVlMTdiMzAzYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LGYmJZqAh4vME3rOvV10EUEr3Ji8mH_NncDCl713pX5yN3oY2if8PWbuJZiQj5hTpRFg2AOP0xOLZ5KPaCCQhQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.433Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlpIS1dzcTNaZnY2UTIyUU1xTm5zdEJjSlkrOEtHK0hlWHRYMzFlNXFRN3BtODgzN0hlZXZlSktrNGFxZUpEQ1crUmtaT0t0YUxUNUdLblV5SWx2SUlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjdfMzNfMjRjNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTc5YTEyMmIzNTI0ZDg0OTJlZGE4ZmY4ZDUxNTlkZTNiMWZkMTJjYjMwNGI2Yzg1MzA3ZWQ0ZTRjZmM1NDU4NmY5ODZmYzBiYzRlYjM3ZDVhZTQ1MTlhZGUxNzkxM2RhYTMxMDIyODhmZmU1M2UwNjJkNDU0NWRkZTRmMTljNjc5Njg1OTdjM2E0YzJhZjI0Zjk0MWY3YzMyNDdjYWRkYjQyMjNmNGUyNDJjNjFmMTYyOTk2MGVjZmRiZTEyYzg5NjY0MDA1MzZmZTczOWQwMzcxYThmMWRmZWJlODdiYjY2MWNjYjE0ZGFlY2UzN2U4OWRmOWQ3NDhjYjZlMzk3NTMzYzQ1ZjIwYTY0NDEyOWNlZWIzMmU5MGI1YjM2Nzc4MjQxYWY0NjI5OTIzNzMzYmM3MmQ5ZmZhMWNlN2FiZDAzOTE3ZTliNWYxOWVjMTczZDJjOTU5OThhYzQyZmI2NDhiNDZmMDQ3NjJiMTkzOGM0OGI3MjVhZWY2NzAwZDI1NmVhNjhmOTAyNzUxNDUwODk1MmVlMWUwZWY0NGE0NWViODY3ZjdhYjU2ZWEzMDEzODg2YjNmOGMzZDEyOTU4OGJmZWYzNjYzMTVjMjRkN2U1ZmQ1Y2JjN2E3NmI1YTYwZDg3Y2ZmMjY3MTdkYmM5NjFkNGEyNGQ0ZTcxNzczYzZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.t-J2hhDuTl3yitzgtb8ZyKffAqr3ZTTOWuHCFLF5wXbWez4gwHslVceREKQiT0EqsA4rnKctWSyUs5PObAvjig", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.435Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImdQZVlaREdCUDV4Y1F0UU1FSjgwSnRNTEcvYU8wMG1DSHhzdjN4NG5yOTRjZFd0YmhDMys0ejJCZHhVNjNxSzRIOVN5WW93RDY2OExUV1IzRXY2cHpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwN18xMDI5MjdfMzNfMjRjNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTZhNTk3Y2Y0Y2FlY2Y2ZTk2ZTYwMTczNjk2ZjMxY2I1YzM2OWZjZGIyNmY3ZWQ4MjMyNjU4ZGNkZWM0ZjhlOTFhZWViZTA0ZGYxMjdlYjhmMWViZDU2MTNmNzU4ZmI4ZDY2OTgzMGQxNTBmODY0ZGQyNzdlZjdlNTdlOTFlMmEzMzBjM2Q5YzY1NDY5M2YxNmU2OGNmYjUzZDZjOGVkM2Y5NTNiYWEwY2E0YTMwNjQyN2MyZGI1OGJlNDgzYjA1MzRjYjBiYzNhNzRhMjgyYjMwYWQzMTdjMjhhYmNkZjdhYTNiOWJkN2FlMDkyYTJhOTdhNGNkYzYwZjcwYmNhYWJkZjZmMmI3MjNlYjhmZWZiZGU4MTQ1YzZjNWEzNzJhODljZjY5NjlhNGI0ZmYzZjBjYzBlNzhhNDVlNWQ3NjI0ZWQzODU5NzkzZDYzN2JkODdjOWM1NjUwOWEyMjRlZDYyMDM0ZTdkMzgzYzIzOWJlOTVlMzNlMTVjZjZmY2RlYTBlNDJhMjk2NThiODI5YzllMzM2MDRmMzRiZDA0Mzc5MjliYjQ5MGYxNzJlOThhYjc2ZTkzZjY2NzdjYWFlYWNkOGJmZDkyZjVlYzE2ZDEzYWI3ODYyZTAwYzMzYmVmNGQwYTkzZGZiZmNiZDAxNTAyNWU5ZWM3MjQyMWNhOTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.amBgb5wb5zmPW7Fml1w4f8A_OtjQ2z1fTsYH8YarcGbrDrU1s8WnHZ3aamaQxSo3hJgYIscjocM4WiNg5eI6ZA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.438Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjMwa292bFQ1RFRBQVM2SkN6cUxhS3RuVnlDRzlnVTdRUHlKN3VTU3VYNTY3U3U0UGlVeFdlZWpEeVgzQjg0VkVTRHBibjVIUGNSUUNsT2M5c1NHWi9RPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjJfNzBfMjQ0OV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDVlNDZjMDcwMWYyNzM1ZTNjZjg3MmQzYzQxYjlhYTZhNTdiY2ZjYTIwZThjZmEzMzA3NGQ4NDQzNWE2ODBmZTNlYTliYTNiNmVjMjYwM2UyZmFlNmRiNWY0MzE0MmM0ZTFiNzUyZWE2NGJhMTI4Y2ExYmJjMjc1MTBjMDZhMzdhMjM1NDY3ZTNmNzU5MGEyOTY5ZmVjMzU2NmRjM2IzOGI4YjA0NTE5MzBjNjFmNjQyZGMzZTRkMGVhMmEyMmYzMTQ5YmRmNDNiOWNkMjRmZTQ5ZGJlODIxYmQzMWM0YzlmZmM1N2I2OWU4ZWU4ZjAyNTY4NjJkNjU1MGI1NDI3OWMyMTk2MmFjM2E3NWU5MWY5Nzk2ODVlM2EwNWRiMmJjOTFlODQyZTFjOWJkNjJkYTEwNWExNDA5MTFhNDRkOTM2NjYzNjRiYjI0ZDkzZTQyNGJhYzcwM2M2NjZjM2E4NmE1NTQwMDdjMmQzN2NjZWFjYzc0NTFkMzg1ZDZhNTFhOWIwMjY2MTY0Njk2NzE3OGMwMTg5YTQ1NDNkNGVjMzdlMzc0MjA4ZWIyOTA3MTAyZWMxNzRjZDdiZDFiZDNiOWZkNDc3ZTU5ZGQ5MmE3Njk5YzlhZjUzMmJkNjlhOGU4NmNmMjQ4ODBhNzQ3NDBhNTgzMzc4MzMzZGY0YmQ4ZDBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FZ6XY9JQQDUZuWMw_yUd4eW3szvM-lb7pZk9QjwjUYa0IgX7Xpc7wqQ_WEM1_kUQf3T5uUSLd6lM1uuc19pLkA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103422_70_2449_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.440Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im4wT1dUR1hZQlBBQ24ySE05ZEdTOWJWUkNiNFU4cS8rVjVFMWVnWkhIbFpBeW5QQWJwbWtodjFmOG5yUHlYMSs2MFpaWW5kZTFSOVdzNjJBWWx1bzlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjJfNzBfMjQ0OV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE4NTUwOGU3YTlhY2ExOTFiOGM2MDcyMjQ0Yzg0MDEzNmVjYTUxYmY0Zjg5MzllNzUxMTVmYzM4MDJkZTFlNDY2ZDU1MjFkNmY5MjQwYWUzNTEwZjNiYzhiY2VmNDMyMzdkNDM1MWRjOWQ3MTU0N2M1NjM5ZTQ5MjdmMGJlYzNjMmQ3YzM1OWY5Y2JkNjcwOGUxMWY2OTY5YmJlYjNiMGY1YmVmZDM0MGU4MTBkYmI0YjdhOWE1MGE5NzBjMDFlOTFjMTU0NjNhOTMzMTg4OTBmYWVhMWM5OGQ2NWUwYmE5MjY0MGE2OWRiOTMwMjE0YjA5NTkzNzk3MmU4NWQ2M2VjOGI4OWQxZWY2NmVmZjU1NzdmYTdjNzE2N2E2NTJlM2ZjNzcwYzZhYWRjZjY0ZmVlZTNmYjZlYWZmOTIwMzdmZDc0MTg0NzA5ZWIxM2UxNzI4OTlhM2QwZGMxZDcxM2YxY2Y4NjkzZTg4YmQwNmFhNGMxZjRkYTE4YjM5ZTU2MjEzYWFjY2UzYTg5ZDIwYTZkNTA1MThhOGNkYjBhZTRkYmQ1ODFlNDU5ZmQzYmU2MjVlNmFlOTMxZWE1YTY5N2U1OTkwZjZhMzkyNGQ0YzgzYmI1MjA1NmQxMTdhOGJkNTdhYjI0YmU2OWQyMzQ1YTY1ZDE4ZmRkNjgwMmVmMDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JKjMjhAsVhLnOwxvjNgEaqYil0yYKcneS5oFD-0QEbwKq2GRQ7mVrHJPYSbgu788ZKL3ZtWrFfGL-uqjTOzCCA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103422_70_2449_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.443Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFSZEFTSkx2SFlKYlJmd3VzOXFTNy85L3ozSUkzRkFjeUE1U1ZHODlSelFSbzRkZmxTd0VORzlGUUQyZUlQN1NIdnl0MjNLazlVZFhmb2xucmtWQkh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjJfNzBfMjQ0OV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OWJiODRkMzM5YmU1Y2U1NDI4MjlhYzNkYzE2MWMxOGJlYTc4ZTNhZDQ2NzNiNmM5ZGUxMTk5MjIyYzhkODEwOWI1Mzk2MzhhNDU3YmQzN2Q2YTViYjEwYTEyODM3OWIxYzc1Y2I1YWU1MDcxZjdjYmI4NjUwMmQyYmQ5NjA5NTQ2OTNmNGM1OWFhNjdlN2Y1NjEzMThkOTAxNzMzYzJhZjM2ZjYyMjQ1MDY0MjdlM2NlYjliYjFlYzc0NTJhODkwMDIwMGRkY2I4Mzg4NzIxMmUzMmU2ODM5YTEwYTQwYTVhMDBmM2Y4MGVkNWE4ZmI4OTg4Y2RlMjM4MGIzYzI4NmEyMGYwMTk4OTIwZmIxOTJhYThjNmY0NDNlZWQ4ZmUxYzEzZDBhNmI3ZjZjNjNmMDVjMDY2N2M2OGZhZjQ2NWM3ODg1ZDgyNjg4ZjgxZjM2ZmY4MTIwMDE0MGZkNjNmYzQyM2U4ZjA5MTZjY2U3MTAzMzNiZjM3YWRkN2U5NTdiMDI5YzMwYjdhOTA0NjcyZDE1ZDcyNjQ5YzExOWRjOTcwNTE1ZjA1NDdlZjRlM2IxMTk0MzBmYWJlN2UyNTZlZTBiZGI0ZGE3N2E0NmJkNTcxYjFjYThhYzllM2E2OTViYzlkOTNhNjY1NWFlZTkyZDRkOGU2MWFkNjMxMTU0MDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3zqhYl6Iw089tFA4Qj1jOZvqPyqcr230SrZqUpMfqtHsmR8cwSc9Xn9eVWy-YfIMwFg9ZtFv45aRg4FzhrqMVg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103422_70_2449_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.446Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkFJZFJJMnczR1QrWmpnS1lYd3N3TmJRZ3pRb3FGSlVNTVozVng2dDdWanJUcEJaSUl1UEVmR0szaEdnVEJWbWM3MHhKdkQvN2VxQStrTWx5eHRwNUp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDcxNV8xMDM0MjJfNzBfMjQ0OV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzE4NGVkYTYwMjNhMGU1MjUyNTg4OTA3MDJkNjhmMGQwODk5NmYyMDE2YmIyMDRhYjZmYTcwMTRkMDczMmI0YmUzMWMxN2NmNDA0MDg5OGZhNDkyZjM4OTZlMWE0YTM0NWI5ZTA0MzJhMTg0Njg1NDBmYzY2YjI4Y2ZlMmE5NTk5ZDVmZWI5MTMwMzZiNzljMzFmNjE3YjVhZGNlMGZjZDBlOGJmZGVlMmE5OWI2YmZhMDYzMTkwMTBjM2Y3MDZhOTM5OWFkZWM1NTMzMWY2YzNmMGEyMmY2MjY0NzNmM2U4MzljMzI0NzgxYWViNDgzOWMwZGJkMjU5M2ZhMTY3ZDcxNmRiMjNmYmFlMmNiNzY1ODk5MTNjZjgxNGRmZjIzMjkzOGY2NTU2ODRlZmVkZGNmZWZkMjRiMjU5OTdjNjM3NTkyMmNiY2FiZTlhMGM5ZWI3ODY2MDU3OWU4ZTY4NDVlOWRkZmUxZjFiNGU2YWNkYzAwOGExZGFhMzI0NDE1YmRmNjkzNzBjOTcwZjE1YWE1ZWQ4MmQ4ZmMyNDg4MTA0NzIyN2RhMWI1ZWYzYzhiYjQ4MjllZDc3NTUyZGRkNmU4Zjk0ZmFlZGE2NzAzMjdmZDZkZGFjODYwMGJlZGU3NTMzYzUyNzgwNTgzNzJlMmVhYmY1N2E1NTVjZTVkY2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.YSrLU9BRxOHwXqIXUegM7-tT9xxqlpVFDbQv7T6IvwEu6I1o9eIqlwZijYrZuJS2illCNEWnA0cBo2UyuVRHaw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_103422_70_2449_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.449Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFYY3BFc1I5ZWhNUGxjY0NiRUxGTENTL1N0Q3VxREk3WkswMnB3RHRwZTE4VDdvNHEvZXpjWVd4d0dRbk9RbmFVVjZDSUdmRlNpSERsNVMwYUtOM2lRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwM18xMDM2MzNfMjRfMjRhOF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzgzMWZhYzViNTVhZjBmMzc4NzM3MzZiZmUwZmU2NTVkMzM1NWRlYWQzNWNkMTRjM2FhNzI5MDIzNzRjNjg5N2EwMTEzMGQxNWUxYWMwYTQ2YzA2MDJhOTJhZmVhZmI0NzA0ZTYzYjkxMDVhZWU0ODQxOTcyY2I1MDVmNTE3MDQ3ZDFjODhmZGVjZDcxODhhZGZlMTM0N2M4N2Q0NjE2OTQ5ZGQ2NGExMmIwYmYxZmQyZmUyYWI4NDkwZDllNTQ4MzBmYjM2ZTlhMTNjYzQ2YmI1NDJhYzhkMDA0ODY3ZjY4ZWY0NTI2MDZlODZlNzU1MWZlYTI4ODI2OTBiNmYxM2U2ZGFiYTc0YTI3ZmVjMDA3NjEyZWExODMwZTQ5ODYxOTE2NzYwOTI1MGU4YWI5OTc0ZWU4ZGQ1YmI3YTk3ZjkwYzI3MmNhZjNmM2RiYTcxOGE2MzY1YTg3NWRiNDZlNWEyZjc4OWRkZDM1Nzk3MjgxYjQ4ZTYyZGY2NTllNTJjOGRkOGJmNDQzYmM1ZGNmOTM1OTU0ZjY4MDljYTEwYjg1MzllOGZhNTE0M2M4MzZkZWFkZTc2ZjZkYmYxMzBhYzFhZjlhMjllMGY2Y2RmZjg2YzdmOWMwZDNhZTljOWM0NDc5YjI3MjRjODg3ZmQ2ZjRmYmFkNWUwMDE1ZmYzZWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rmJMSkDnp7QMgC9R8OKWn8JOTB_XNaoky1HiKoGPx77I3psOXZAjGis6U-BoWtGt3LXgO9zTe9hvde7CiO4nKA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230903_103633_24_24a8_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.452Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkcxRXVqOVFpMUIyR2ZlVVNDSzVYKzlrMUZRSHRKMTFMQVVsRGdwT1FMaXJBYmovcDJ4bzlZZHlZK1k0bFcySWswR2RaSVdpaUNCSGdkdUFNam1GbUZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwM18xMDM2MzNfMjRfMjRhOF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2VjZmExOTZiMjIxZWI3NTAyYWIzZTc0M2RjZTFmZjEwMGVlN2JkYjJmYmIxMmYzZDE1NDUwYjk2YmVjYjg4YWFkOWMwMzI4MTUyMmUzNmU3MjdmYjYxNDM2N2IzN2ZkNzQ5OGExN2UwNjAxYjJhZDdjZjM4YzBjYzg5YzczZWQzNzhkMDQ0ODQxOTI1OWZkM2YyMDk4YzdmODc4YWI5NzNiY2M0MTFiODJmNGEwZGFkMzkxNTVlNWU3ODNkN2I4MmJhNTUyYWY2YmVjZTI1MTNiY2Y3MWIzMGFjMTNmNDk1YTg0NTMyOGJmMDg5OGU2MmQ2NmQ5NmM3YWUwMTIxOGUyYWU1NzgwYWUxYWUwMDUzZDNhNjM5MTE2NDQyMzVmMTdmN2UyNjdkNmY0NDc3YTAzZDZiMzExODY2NDAxZDIzMjM3NjYwNTQ4YmZlYmI2MTk1MzNhYjE5MDVlMDk2YTNlMjZiNWUyMTVlNDgzMTA4MGI3YWE1MGJhZjUwNDk3NmE2ZDcyYTI3NTc3NWM5YmI1ODZjNGZiZDE1ZTAxYTlhOTc2YjMwYjYzZDBjYTUyMzI0ZjliZjdhNjJiOTNhMmJmODQ5ZjY0NjdkNjQ4MmU0ZDJlYmVjZWVjODQyYzY1ZmQwNzZhNTQ3YWEzMjU4MTNiYWMwODUxMmFjOGNkYjFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.2thhGJvZq0eMBRrWTH3gboL-bY0LIgZNyhnEySA47ciepPaeVG9h8mhGJX5UemhxKCG9YbJz-XaEEnTEZwUMfQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230903_103633_24_24a8_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.454Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFIYmVod3JKemtQcWo3N1dRRWV4NlVoQStMR0p5aUszMGE1TVh0NjFHaDR6eE1jZGg5cGNHM0w4RTNCcldwS1Q3YnpwNjJlV2ZDNitIOVNOWVczVGpRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwM18xMDM2MzNfMjRfMjRhOF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWZiNTU1NjZjYTFiOTk3ZDc1NmE2NTYxNGUxY2ZmMDBmOGI0NTk3ZTA4MjZjNThlOGZhM2ZhNTM5NTIzMzdlMzBiMTZlOWQyMTk0ZmQ0YWVhNjE1MjY0MzNkZTU2MDU5YmQyZWM1OGRmOWFhZjBhOTM4MzViODViNGUyN2FiOGM2OWU2OWE3MjM4MGY5NTUyZDNiZjA3ZmY5YzEwNzkyMGMzM2YzNjgyMzExYTFlNzM1MjBiZDZlOGFjZTcxYjgxMWNjNGIyYjc3YTIyYTljZmEzM2VhOWIwMWM2OGNmMGU3MmZhMDU3OWM2ZDgyYjQ0Yjc5ZDJjOGQ5ZTI2M2RiZWY4MjQzZmRhMWU2MDEzZjJmY2Q3ZGRjYmEyZDYyMzQwZWRmNzM4ZTc3M2NlMWU4OThmNmUwMTZiMmVkMDkyNjg0NTgyY2YyNTVlODlhYjFiZDQ5MDgxYzQ0MjE1MzdiNzM4ZDViODliNzdkMzljOGRjOWFmNTQ0MTY4NjgwNTMyNmRlMGIxN2E5ZTk2MjEzNGUyYzJkZDU4NjI3ZWUzYWRhY2NjYmIyMjFiY2MzOTlkMTE2MGMxODFiZDVmYjEwNmRmYmY2YTFiYTk3YjI3YTg3NWI1NGYzOWRkNWIzY2UyNjU0NjYwNzMwZTJhODQyODNjYTRlNmJmNGQyZDA0ZDNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HEoN212YL5o4j4C12OFSxbtL2D9PUL_7Oa30DRMt_0HgsXsmulzHz_nhScNozbmOkcSncLy12i6a30S9UuWt4w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230903_103633_24_24a8_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.457Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRZUm1BaXZBOTNpQ21KOXVGeStvT0N0NGpac0puczVGcy8zRUhXNldzOGxWUmVCTHdQOElYcHorM2oyY0VCcFAzTHl5S0IvcnIraFJ1a2V1U3VTZDZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDkwM18xMDM2MzNfMjRfMjRhOF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTQyMDQyYWMzZWYyYmY4NDBmYjhhMzQ0Yzc3NWRiNTVhNTFjOTg2NTM2ZmUxMGQwMTFkMzljNDhhMzA1ZTk3ZTBhNTIyMDUxMzg1NzcwNjI5Y2IxYTIzZWFhZTMzZWFlZDE3ZTA3YWZkODkyNDZhNTI0NTZhNjY0YmEzZmJlMTliNTM4NGI0YjM5ZGQzNWIyYmZhODM0OWQzNGEyOWE4MjExYzk1NjY2NDkwN2MzZTA5NGYxYzZkNmQ2NWQ5YTJlZDM1MzU2MTIwYzE4ZTk0OTVlMGQ1MTdiY2IyM2UzNTE5MTk5YTM2YmIyMjAxNWYwNjdkNjY0NGUyY2ZiOGFjNTBjMGQ5YWExOGExMmQ4YWNiODlhYzJkNzJlZTIzZTQ2ZjNiYTgxYWRhYjY5NzE2MjEzMzJiYmJiYmJhYTc5YzcyNmNhNTE1MDUwMDYyYWEyY2ZhZGYwYTNhZmMwNDMxZDZlNTgzOWQ1MWRmYWQ0MjJiODFlYTVjNTlhMjhiYWNiNjUwZmMxMjIxZDdiNGM5MzI1Y2U3ZmJjMjg0NTlkYzBlNjFkM2VhNGEzMGI2ZjNkN2RmMzQ3NjYzNDUwNGY4YjQ5MzEzN2ZmNjdiNjJiMGQ4MjZiZGJhOTUyMmUwMTE3NWUzYWU3MDgxOTFiYmE1NTMzMjBmNDdhNzQ4N2ZhZmJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.p_CIcL_S9fddq2kGJzYr5EdoJ-wyyq3tTdxYf0RxF2lbXRb35DvNq0nrjFmO-brcsJAvpsE-j0wfPbctSOIPjA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230903_103633_24_24a8_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.459Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InV5bHpqa1ROeXFVRkdFMXNtNFRzR05mL3JSaFduRENtQURtU1BwRmlRQ3hLalh0NUZJVUFRVGFjTWx5UGZHcy9Hd2RPZnRzODlFMWI4WHhBRFZXL0dnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjRfNjRfMjRjY19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDM3NWFkZTViODdiOGJkMzA5OTZhNTM0NzM4MmJjZTgxOTc1ODE4MTBhNjVhYTU0ZGU2YzQ1YTRlZTk5Y2FmZjJjYjQwNzI2OGZlYWQ3NTI0YjMzM2E2MjFkM2ZhYTZkZDIwM2Y1YjM4ODVhYmZmMjg5OGEzM2JiMjNlMWE0NzZkNDk4OTdjY2ViMmYyZTg3ZjEwNDc1YTExZDAyYjY2YjU0MDRhZTc1ZDUzMjRlNGJhYzI5ZmYxZjgxMzFkNmRkZWFmY2YyMGE2MTVmZmVmZDVmODI4OWNkMDcyNWFkMzNiZTkwZWIxNzhiNTUwZDk4NDgzMjI1NWIzODA5ZjU1N2YyMDA5YmQxNTcwOTgxYmU1YjQ2ODU1OWUxNmRhZTFhZmNmMzNlOGQ1ZWVhMTBiZjY2NzNlNGMzNzRmNGM1ZDEzMjk5ZDA5Mjg5MjljMWZhYzdmMzVmZTMzOTBiMmM1ZTkyOTMxMmQ0YzZkMDQ3YWY3YjhiM2RhZmEzOGE3NGZlMWFlZDBlNTI3MTJjMTcxNDk4ZmJiNjk5MzYzNTI0NWVhNGYwN2QxOTJjMzlmZjY1N2U4YmI4MDA4Mzg5NjFlZmVkY2Q5OTcxNmYzNjg4ODkyNWMyNmUxY2M1MDVmYjlmZTE5N2RiNTlhOWQ2N2ZjYzZlNDE4NzAzNjJiMGE0ZjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.3O0MiUfCVJ1OFpy4I08qzsJ5TVKEe6r73k1l7f3Y_RHiKnP9dNVWpCmFSVwXQszd_lUMxvYzBtESDV64qljtPA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103624_64_24cc_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.462Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InErSkVSTUNQeTFHZ0NOakJONGUzUkRhWWZ1QTEyejVPcHUvY3VadkorU2JMK0syc0lwOXBJZ2N6RmJUM0paSXJoZVV6Q2UyZ2ZHNllyS0o5VkxZR3JRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjRfNjRfMjRjY18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzYwZDFlYjg1MzcyYmJjMTc1M2I2Njk2Y2RkMmM3M2ZmNTVlZjdiMDAzODRhNGRjMTliZDQ4Y2NhZjFhNDVhZmYyZmExMWMxMTA4NWExM2VkNWNiZmYwOTllY2VlMjQxOGIyOGRkYTRlYmZjNzg3YmZjM2QxNTNlYWQxN2M5MjFkNWM5ZWE3Nzk1NGQ2ZDI5ZjdkZDAyMzE2OWQyODgyNDZjNjQ0MGQyYTRhYjRkMjg5NGRhYjQ5MzI2NmMzNTAwZDQxNDk4ZWM2NjdlZDNmNzUxZDU3NWFmMmZkNmQyNDc2ZTI2ODVlNWNkMDZhMDdjNjA2NWJmM2ZkNTE2NDc4ZmUzMzBiMmRlNGQzNTgyYWRjYjA1YmM5ZjZiNDM3OTA1MzhmOWI0YTZlYzFlNzEzNmExZWRiNTFlZDA4MzAzMTcyMzJiYmVhMjU4NGM2MzU2ODQ1YTZmMzJmYTI3NDgxNWIwMTJlOWQ2NzU3ZTkzYmQxMzY2MDc0YTVjMDM4NzliMzZlYzZlNmY4NWVjMzExNGJjMzhmMGUyNzlkZDg4ODZhNzk3NTI5Nzk5ODdkNDAyMTUwNWVhNDNmZDIwNzBlZjZiMTZlMjZhZGUyMmEyZmUyYzQzNzAxOTYzMmY4YzNiZWZlNzMwZWFhY2RmZTBiNmVlNTc4NDFmYjEwNzI3ZGRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.GAUnvULDJsl7DKcfg3PcNGwDOVyBvfROWIfb5y9XgGX83y3j1GBdZu7KUXrNk4aJI4_mIqUAgWZ6HzyEWkk5vw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103624_64_24cc_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.465Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjgyOXdhaFdWQ0tvc245NVpQUlBvL2V2czFyK2NhL2FCUytNbzNZbzVueVNrOHJyN0FrVUxMMTVVVmZ5b05UOGx3MUVDaFFpcGQraG1FVFJrRmR4b3NRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjRfNjRfMjRjY18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDIyZTk5NzVkNTVhMGYyMWNiNDkzMDVlYzU1OGM0NjU3ZmUwMTU0ZTJkOGQ2MzdjNDUzMDI2NWEzZDI1YmRlNGVhZDM5YTc3NDI3MjAyOWMyMWRhMjc4MjBjYzNkNWVlNDhhMWIzOTM4MzRkZGJmYzIwNDU4NDlkNjJiYjIwMDY0ODU4YWY4NzgxZWEyMDNhZDYyMDNiYjRlMWZlZGE3NzhhNzViYmIxNmJiNWJiMDc1NzY4N2M1MTE2ODQ1ZjdkN2UyMmFmMDgyZTU5NDJmMGNkMDQ1NWQ2NTAxOGY5OWRhOWI1ZjA3Y2RiZmVmY2M4YmNiODI2MTE1Y2YyYWIwNTJkZjQzZjhjNDg4MmE0MWMyOGExMjZhNzdiY2E3ZWIzMzBlMjA1MWQyMGU5OGMwMjllMWI1ZDVjYmRkOTc4NTU2ZWJjYzJlNjNlZjNlNWNmMzMwZmNkNTJkMjY1M2FiYzBiYWYyMWEwNmJkNzNiYTY0Yjg4MjgxNjJiYmViYWQ1MjBhMGExNDIzOTU1YTViNmUyYzE5MDBjYTcyOGY2N2YwYzJhNTFkMzRjZTQ2NGE3ODM5MWVmOGE0NDQ0Yjc1YWNjZjcyMzM1YTg5YzEwZDhhMzkwNDlhMDZkNzAxMjAyYjc1NGY2NTQ2ZTQ0ZjgzNDA3MjQ3OTUzMmRiNjU4MzBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HOaHRZXsY4zKu4mOPAOuP0pbqgdhieFJINxx0X_LUKKwg2OdGGZ-rqBzV0BeJXfyobWp4VQo9CknsRhCQf2cvw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103624_64_24cc_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.468Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im9ONWVwSDNDMFMwU04xVkhObitaYmZJdnlDZGR1cEpZdFdEYXhDbVpoZjRwMDd1dDRkbFdVNWJSZnRtN2Z1ekphbE5NUEVaQTQ1OWQ3bzNjbFUrODR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMTAyM18xMDM2MjRfNjRfMjRjY18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTY0YjU5ODM1MmQ2NjAwNjQ4MzZhMGY1YjRkZDMwMjdiYzQ3Yjg4ODI2ZGIwYzkwY2RhNDMwNjIyNzdlMDQzN2Q0MGE0Nzg2ZjUwNzA4NDY0Mjg0NjZmMGExMzZlNjYyNGVkODQ2ZWY2NWU3MWUwYmZjYTk4MWI0ODZmZjI4MTIyMzY5YzdhZDdhZmVlNTk0NGIzZGU5NTg1ODkxN2EwNzNmZGMyMDI3ODZhOWRhYTUwZWZkYTI5ZTBkOGI2M2RmNjU3MzRiN2NkNzgwNGZmNjUyYzU2NzY3ZjljZDgzMTM4ZDA0Y2ViM2RhM2E2YTg2MDlmYmVjY2VlM2VkZDVkZjhkNTM5NzM2NTRiMzIwZjQ1MjZjYjIwZGRlZjU4YmU2N2U0NGE2ODQwMDQ5ZDM2Yjg4NjQwOTJjNzIzMDlkZGE5Yjk1MTY1YjBkMzAzNDFlZmIxYjA0OWZiNDg5MmNjYjVjNGNjYzJlY2ViZGU0NzE0Yzc3ZTRiMjhiYWI0ODYzMzk1NWNhMTg4ZDAyMTNmMTI4NTkwNjdkMGI1ZjdiYjYyYTNkMGNjZjAyNDMzYWI4YjhiZGY1NTA4ZDdhYjYwODg4NjcwZDljY2Y3MGExYjBiNjkzYjcwYjExZTBlNTc0ZDE2ZWU4ZWU4NzhjMmYzYzk2Y2U1MjIwZWU0MWQ5NDhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WM06PfNXNE0onudPSEqP6L2bchCxSL8KHKZkQIoNbznee_E9FPsUALZPiZSq5D4Ug9bsWcIhnVO6DauN_hjG8w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231023_103624_64_24cc_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.471Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpJNEozRUExZmtTblVhbXI1d1htcUozZXc2OHlPRGYrVGtMaVA3ZHExNkxjR3U3S1duOUNMbFBGaE45YzhzQVkzYUw5VUl1OGRudW1WY2Y1MHhxZVNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMDMxMThfOTRfMjQ2MF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWU3OTcxODg4MzJkODkxZWZiOTA0YzRmYzE2MDgyNjg0ZTZlNjVlOWRlYmRhOTU5NTVmNjliMGU2OWVjNjQ3ZWMxZTM5YjQ3MmRkODhlY2E0MGFlOTI1MGEwZmNkNGMwNzRjNTlkM2VkMzQyMmVjNGFhYTA3MWVkY2JmODIxMjdhMzRkODRjNzlhYmUzYzM1NWIxMGQ0OTM2NTEyMzQ0YTZiMDRmMzljNGVkY2QzNGNkMzZhZGJmYjY3Nzk2NjViYTQ5NTczZDdiMTA2ZWE2NmUxZmE0NjNiZTU4Zjc3MDBmZDViOTEwZGZlOGJlY2ZiMGI4MzViY2E2ZjY2YzE5M2I5YWIzZDIyNjZhODA3Mzk3ZGU5YTI5YjEyZDQ4ZTllYWNjMWRjN2U1MWE5YmU1YWM1YmNiNmM1YWNkY2I5YThlYjhkMTA0ZjU4NjA4NGI2M2MyNzI1OTUzNGIwYmViN2ZhMDBhYTIyN2M0YTc4NWM4MmIxZjhlNTY2MjZmZjAzNGZiY2JjYjUzOGI3M2IzYTZlZGIxNzgxZmYyYzYxMjk5ZjgwZGQ1MGVjYWM3N2VlMzM3ZjYzNWE0ZDdmZDY2NzFmMmM2ZmQ4MjY4M2M0ZDc4YzIxZjlkMzdlMTljMzAwODI4YTViNWU3YTFkMzk0YzRjMGFkOTM2MmQ3N2QwNjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ke7XEV2iH5ZZZDiEs5T8ehwPRwK43l2xZgw0KKX8cIglE7OYDc3B1XNb8A02S746KELfMXkYE-HKZZzT1Uh4bg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_103118_94_2460_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.473Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjJKWFFpQW1qRjJqbGprTHJDc3haSURlNWJ0VVZXYUo1RCtxZmdtdGxuL21scTFlU3hSQS9mOHhYd0p2Y0p3SkpRaUxqNkZpckpwTGNyV243WW41Y2lnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMDMxMThfOTRfMjQ2MF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzYzYWU1OWJhZTg3ZTNmNzU1ZTc2OWMxYzE0ZWQ3NjgxODY0OTEwNzI1Njc2M2RmNWY2ZDhhNjZjYTU0ZTJhMGQ5M2VjZGVlNzEyZjVmM2M3ZTcxNmZiYjE0MDJiNDA0NDU2Y2NiOGU0NjA4ZmVmOTkxMTY5ZWI4Nzk1NzE2YmMxNTQ5ZTZlMDNhZDhkYzIzZjU5OTY2YjY3NjI1YzhkNWI3ZDMxZWY4NmYzN2VlZjU3YWJlYjI1MGZlZjVmMzRkNzg2NmUyZmNiZmE2MDUxMDM4MzYyN2RlYTM5YWJkN2YxMTEyYjEwNTdmYjZkOTNmOTYzMGZiNjY0ZDg5MmRhZTg4Zjc5NTRjMDNiYWM5ZjRhMDQxYmY5NTIxZmM4MzQ1MjZlYTEyNGUxZjU2OTY1YmU2NGE3NDFlNGQ5Yzk5MTQ4ZWRkYTU5NjBhMzM1OWM3ZjIxZjcyMzI0ZDFjMThmYmJjN2NmOWYzOTAzNjY3YTcyYzEzODZkNDJhZTIzNjhkOWYzNGFkMzBjOTc4MzBiNmQ3OGQ0OGZjZGYyMTEwY2Y4ZTQ3NjMzOTZjOTc3ZWIxYjYyYWUzZmQ1YWVhYTczOWI2NjUxMmI1YzJjMjE3YTk3NDdkMDFkZDM5ODEyOGU5NDgzMTUxYzE1NWFhMDc1MjFmODY3OThhMjM3NDE3OTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rUmWnXmvPF1jj_HBQQUjDXYTNAxSqFwGqxxfPCuF1As4k0XZIJByuh1oTNE9vwfltI-qOd8p8uZjegZwYCr0Ig", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_103118_94_2460_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.476Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjVkZUxscGN4dXcwZ1hVQndaNmJDWEVnRWZPQ28rSy9MV056WkpqKy9EMnRQejRPbHJSOEpER0x4bWNJUU9sbTB1c24zbmhkZURqYktQL1l0ZXNIcFRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMDMxMThfOTRfMjQ2MF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGI5MzI1OTQ1MzZhZjU5MmM0ZTk3NTE2YWI4ZjJiY2JmMDc4NzBjZTRkZWY5ZTcxODQ1ZjgyZWQ3ZTg2ZGI5YzY4YTQyODBkYmI5YTk3NDhhYjU3MzM4ZGZmM2E1MWFkN2IyYTFhMjZkYWY0ZTFmMzdjZjg4OWQ0YjRkMWIyYzEyZmFmMDc3OTY2YWMwOTA1NzkwYmYyZWVhMWZmYjVmYWZiNGUwYmM2YjVkMWYzYzNkYjMyZGRkMGUxZmFiN2E1YWQ4N2MyOTQ1NjNlOGRiNzNiNGEzY2MyNDFjMzRkNzNjMGE2ZGNlOTM1NjQ4YjNmYWRmMDhlMmQ2OGM5YmY5YThjMzg1OWM1Y2FjYTAyNGMxOWI2YzAyOTBjZTI3ODg5ZGQ2MGU2YjUzMmNmZTFjMGNiYzlmMTUyN2VkODcyZjAzZmFjNjA2MTBlMTg5MTVhNzg0ZmI0MzVlYWEwYzc2OWFiMTMyZTZjMjM4OWYyNDA3NGJjMDY1OWQ4ZmQ0YWNlZmZkOGEyMjc1YjhiOTFlM2NkYTI1OGZhM2FhNDQxZDAzZDBmYTJiMGIyMTE1OGYxNWFkNmMyNmUwNmE4YTRkMTM3MmM4NWY5ZjAyNGIxY2RmOTJiZWUxZDNjNzRhOTNmNWQwYzM3YjI3NGY4YzU3YzVlN2Q2MjRmMzAxZTc0MzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.UWzBCamwhDuk2d4gLVb5A6xlfZerb64zzLHHpEXiXr9APYoHoDRwLYQQmmgfV3VJN-Nfyul7UdVcSAgIffEtig", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_103118_94_2460_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.479Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjlyVXhKS1ZjOVVwQXJ1SkI4aGhZb2M1d2NKcEJxQk94RURHY2NaMEVrT0JXVmJlYitmbGhrYjEzZnEvOGp3M2VKZE96VEsvVityZTNzMEtMUXVzd0R3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDEyN18xMDMxMThfOTRfMjQ2MF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmQzOWY1ZTZmMjQ4YTY0MDE1NjYyMjA1NDA4ZGU2MjFiZTdiYTI1YjQ0ODk1ZDA2NzNlOWViZDAxOTE1NDgwNDY2NTExN2MyNGM2N2IwNjhiYTgwMDIzM2NiZWUyNzc5M2I5ZGVhN2U5ZjY0ZjI5MjI3YzY0YzVlMmQ1ZjAwZTRjNWEyMWYxMTNjOTczMzk2Yzc5ZTE2NWYyMWM3MmIwNTBlYjBlMDU0OTdkOGUyODc0MGQ4MDNkZGY5NmUzYmU3MmRlOTEzZDllNDc1MDE0YTI3NjJkOTM4MzIwNTA3ODY1ZTJiNzIwY2Q4OTkyNDAwZmE5ODU4ZTI0MzBmZGIxMzMyY2JiOTNmYzA0MThlM2VmNGIwNWE5Nzk2MzNmYWU4MGQ4MWU5ZTVlOGEwN2UwN2FhMDA5ZTQ0ODAzNTg3NjAzODZiYmU1ZWQyMjBiYjViZmQ5ODA0ODYwMzgxZjNiNzYxMTJhMGI2MmUwYjVjYjQ2NmFiZThkNWQ0YmJkNjhhMGY3MjE4NjcxMGFlYzA2OTVmZWZhYWFmZDFmMjQyN2FkMTZjM2QxNDJhYTYwMGI3ZDg5ZWQ4Y2U1MjU1OTdlZjE0YTJjNThlMTI4ZjczYzMzMTYwOTFjMDkwMTg3NDkwODM4MDg1MzU1NzMyMTQ3OTc3MWI1OGQ5YWQ4NTFkNjhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.L6UUUpEr0R3MNysQxdWBJkpKw_qyh6Ei1GYqE4Ny1FoSAarYZfI5s8KUAJm7xbkgQlB0X1BZ7RkLTgeyAZwMWw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220127_103118_94_2460_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.482Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlBadk5QZ2cvRzBKZndMdzZ1K2JDQXROdm5CZ0hVV0ZvNWNQVURtRmQraUNna1dSNWtGTUJoSlZzaVo0MHJraXdXTGY2VmhEYTJDTVdsRHlkVHJRdWpBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyMl8xMTA0NTRfMTdfMjQ4ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGIwMGE5YmM3MTRiNmVmMjNmMDQzMzZkNzAwMjllZWI5MjBiYjBjYWEyYTI2ZGMyOTA5OTNmYTUxZjNkNjIyY2U4ZjYwZGJmZjRjNmQ5NTM5MTJmYzkxNmI5MTNjNmYxODQ3ZDg2ZTA1MDExNDM3OWIwYjI0YTRmNzc3NTE2ZDNiYTc2YmI1ZDQyOThlN2U1NzRjMzAwNTIxYWNhMjA2ZWNlNjgxODg2NzE5ZDAyNzA2NWQ0Y2UzOTdlNzkzMzYwYjVkNTMzOTQ4YWE3ODNhOGY5OTJlMjIwODdiZWQ4YWMyNTEyNzUyZDQ2ZDUyODdjZjlkMThjN2YxYjg0ZjZiNjMzNjNkNzVjNWViMjRkOWVhYTVjZTI4NmUzYjNlYzQyZDIwYzQ3YmY2YTUxNWVjOGE4ZDhjZjgyZWM4YmM4NGFhNGI2NjlmYjAwZDgxN2Q1ZTM2NDk5ZDkxZTBmYWQ5OTM1ZTIyOGNiNTRmZjU3OTE2ZDk1Yzk1NTE2OGNlOTdmYjk1ODlmY2Q0MDFhNjFkNGZiZjVlN2MyOTNkMTE4ZTU5OWMzNWEyMWZkNWUzYzYzMGRiZmI5ZTBkZTgwZTA1YWM0MGQ1NDM1OTlhNTM3N2NjZGVhMDViY2E4NTFjMzIxMGNkZWE5OTU2Mjg3YTNkMGI2YTc4NjFlYzU3Njc2OWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ZjSoKCRshncVgRhKHpXYk_8jW0rxlm9HHXWSet6n-AnQUNOQi3NL87cG1Wdq-A1QmgcYB4zeX6fbStNm0otl9g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230522_110454_17_248e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.484Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImlrQVVvM0ZvMEhYdFdsdVRvYnF4VXBlSEhRTS9MRUJiUllTN1liR0RJWTY0aThjNVEvak9aVGJNdy9WbkwrekNDRFVYbjFmbDh5c1VDTk9vWkduVmh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyMl8xMTA0NTRfMTdfMjQ4ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OGRjOTQ2NTBmOWMxYzEwYmU4MGRjZjM4MmNhNzU4Y2MyNzliNzg4OGY5MjJmY2Q0N2QxNDA3MGVkYjE2NzdhZDIwYzQ1OWJmMjYyYmMyN2NmYTI1MjQzNzhjZDI5NjE4MzJmNmZlMjdhMWFkZGI5OTJlYjlkYmMzODAyOWU5YWZhODlmNTYyODAxMzgwZGRjOTU1NDM1YmRkM2ZiMjY2ZDc1ZjJkNmRmZjcxOTc4NjZhMDkzNDc2OTFkNDU4ZjJkMzM5ZjZjZThmNDVkYmFiZDA1MjIwYWZlNWMxYTVmNGYxYzBlMTYwNTU2Nzc5MzI5MDM4NjFjYzgyOTE1NzhmZWIyYzZhOWI3YzJiZTgyYmFlYmI5YjRhOGNkNzM2MjlhZGY3ZTFlNjA1N2Y2Y2QyNjAwMTk5MGM2ZThkMDBjNGY3MzdlOTMxY2YwNmY2MDVkNGU2NGIwMDM1ZmE4NmFhZjhkMjA3MDQ2YzExZTdlNTdhNmI0MzUwNWE0ZjE2YWNlMzg4YzgwZmNjMzQzZDMzNTlkOWQ3NDBhNDYwMGFkYTMyNTJhMmZiOTk4ZTVkMDNkMzM3ZjBhNzVmMDg3MDNiNTMxNzg4ZDIxYmViMmQyZDVjZmIwN2ZiMWNiMzAyNzRlMWFhOTIwZTliOTY0ZDQzYjAxY2M4MTYzZDMzNDAxYjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.p0u2bSOIRLGWzo4hSdKEO-2A3H7Cz005QC2Yyc_bTm2akWF2oUbhYGcglVYoluUxBTEi63BDVYCKArAeVkGlcg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230522_110454_17_248e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.487Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ii9hQ24zeEh0ZnhEWW9IenZlK0Y4d2VrbmFFTkVPdW96ZUtMZEttRjNzd08xZUgyUEtSRGJQbUx4c2QyclhUQk9uRGh0WlE5RE5uYUp4VmljTlFsWW13PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyMl8xMTA0NTRfMTdfMjQ4ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Mjg5NjU1YzdmYWZkYTNhNzAwYjVkNmNiN2Y1YTExOTUzYTM2ZWQyNmQxNTMwYzU4MzNkYTBiNDU3YjlkYWUyNmMxMzY3NTMzMDJiYjJhNmU0MmRhZDVmZDczMzg3Y2E1NDU4YThhM2RmMDRjOTA4OTIxN2UyNWMyNTcwNzlhNTczNGRjOWFlNmU0ODM4NmE0MjA1ZTFiZWZlYWRkMTYwZjVkOGY2MmQ4NmIyNTJkNDEwNGM4MmE1NGY3Yjk0NWU4MWQ1MzlhOTQ5NjA0YTdhOGRjMTdmMDA1NWI3YjU4Yzk2OTBjMzIwOTk1ZDYxYzA1YzE3NjEyZDQyZWZhODYxYmQzMDhhYjg0MGU4NDM5NjU0Y2NkYjMzZWUyZDQ1NmNhMDA0NWFjYmNmY2ZkYmI4MDc0MWQzNTI2ZDhmNzYyNjI3MGFlNjhiZjYwMTI5NzliNjQyYWIxYzU1NDRiYTI0MDgxODBlMjkyMDNjMDMwM2YwYzc0NjMwZjU4OTI4NmFiM2U1ZWY5MDg5ZWI2NDg1ZjcwYzhkMWNmN2Y1MWFjYzdiODM4NDdiMjUxNDBjMmVhMTAwNjc1ZjllNWMyNjhiYjRmNmUxY2ZjNGJmYzA2Mzk0ZmFmYTQzZTA2ZGIyZjM1ZGY1YmUwMjczNWMxNzAwODE1NjRkZWQwOWFmMDgxODZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.KyAn_hMKjJIDzUS0QrhpFE_AtpBC8tzBpJ9NNyfnyS0Gei8OH6e6K3ieVIxouCmDu1f79uKFSch1rDma-ZVmjA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230522_110454_17_248e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.490Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InJDNmlZL1lpYzRoWkdta3pFSDJsVnQwY0NPaUpXYis3aU9jd2w3MkFlVWNrT2NXemtkcW1ObmpVVUJ6ZDBTS3Y4ZmZxd1lYN0pyM0gvTEdFWlFvSU5BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyMl8xMTA0NTRfMTdfMjQ4ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODU2MmRjOWYwZThkZTFlYWM5ZWFkYzZhZWJlMDE5NDgxYTZlNGE2MTZiNWE3NjUzZGQ1ZTljOWMyM2FlZWE4NjcwMTU0MGFiODMwNDUwM2MzMTE0MmRmMDU1YWM1ZGM0MTc1N2UxMjA2NTA3NGVhYjZlYTMxMTg3MjZmODUwMjQxMDRhMzJmNTFkZmUwY2UyNjMwMzBmZjY4MDdmMTdlZmI0MGM4MzRkOTZiNmRjODc0MDlhNDE2NDJhM2Q2OTJhNTZkYzA3YTEwNzRkOWI4N2VjNDRmOTY2ZTc1NmI2YzBkNThmNmI5ZTExN2FjMDA1YTFkYmY4OGE0ZGVjM2JkOGJmMjIzOWM4ZmJkYmNjMGQ0YTJmNWU4Njc3MDZiNmZhNzYxZmE4ZDQxZTMzOTg1YzEyOTE2MWE5MmMyMTA4YzdjODI5NzhkNWNjODQ2NTgxYjk4Y2Y2NDJlMWFkMmRhY2MxZjc1NDFiM2JkMzRkYzU5OTljNzFhNzQ3MmVkZTVkYjFjZjFjY2RkNjRkMmUxNzUyNzk4MDdjOGI3M2I5NTdiMTlkYTgwMWQyMzFiMTEzNzJiODYxYzQ4NjA2MzBlZWJmNTkxYTEwOWI0OTcxMGM5MTUwZWZhODEwZDA1MjYyNmE3NWJlZWYyYWFlNzRiZmNlMjQzMzZlMjUzNzhiYWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.m1I7AHUQdsIIMa19CKdBy3xv_aDclkXcnEsX6DLuxRpVcJUMEAus5eaR2EX6WsHmVEz-UJqePzMbYICSvG140Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230522_110454_17_248e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.493Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImxFR1VlcU5iSXNNUDJ4MXBGYW11Mm9LTXA2U2pvbW9QZW9Fd1c0ajFNem0ySmJCb0wzZ29qZnVIbXIyOXlweGtnN3h5a3E2K2Rla1JjK012RUFKVWRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTdfNDNfMjRjNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDk1MWVmMTliOGUyOGMzOTc3Y2YzODNkNjAyYWMwZjIxYjlhOTg0YjlkZThjOTc3NWYwYTQ2MWZiOTZhMmEzZGFlN2M1NzAwMWI2M2JkMjdkMzk3ZDc2ZjQxNjFkNTcxOTNiYTk0MDhkNzgxN2Q0MDM1Y2VjNmUwMTJmNzk1NzBiZTA5NzZjYThlYmZlNTZlOWE0MTk0NTFiYTA5NDY1YmNlZmY1MmJkZjI1ZGZiZjZiY2ExYzU0MmI4YzIzNDc2ZGUzM2ViNzgyZmNiYjJkNzQ3NjY4ZDk4M2YwYmNkMGNmNDJmMTg0Y2FiMjIyNzQxOTU3MDgxZjlhZGZjZGYzOTUwMjI0N2Q3NTJkMzgyZjFkMmY4NTQ2ZTNmOTNkMzBiNDhlYjZkY2Q2ZGIxZDQwNjY5MTFiYjY4NjliNjMyYWNjYmE5MTJmNGVhYmQyYmY0MDdlZGRhMWY2MmVmNDY3YmNiMzkzN2U0NmIwMGM2NGFjNjBmNjI3MmY5NjM2NGZhZjZjMzcyNTFkY2MwNTk4MjJlMDZkYTM3MDVmNmE0MjhmMzIxNDQ5ODdkZTM2ZmYwODU3NGQyNWFiMjYyNTJmYzEwMzhkZTgxY2VmOTZmMzM5ZDMxNDZiNWNhMzZlNTQ2ZTJiOTAzYzhjNGI0ZTM3MWEzMzdkNGQxNGJlMzNlODJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.g2156FnVnz2RSKFCnEufv0tXCKSLJksgDdsdT8dfHrcDgEXhVC9eKVO3fDv9jpWlKo0jDJeASj4Y3q7YfpsW1Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112957_43_24c6_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.496Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imt5SU5XL1FjbW5Fb29KMXEwcHZLSlYxdk5nNkk3V1ZZREVMeUswTEFuUFJoRmZoVGZteE9sUEdrL3ExQ0pmMVFhVFgxN3ZEQ2JOejVGMTc1SU9KM2dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTdfNDNfMjRjNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzEzN2RjMGNjMWY0MmI4ZTMyMDU0MDdkZWM4Mjk4YjgzMGVjNGQ1MDYyODk5NzI0NTc0Y2MyYTMxNWJiN2QxZTViYjVlOGQ4OWQ2M2FmNzA1MjY0YWNhODQyOTVkMDkxZDRlMzI1OWE2NzRlZTkxNjM5OWYxMDRmZmI3NzA5MmVhZGMxYWU0ZjcwNDg2MjRkMzllY2QzNDgzMzMyMjEwMTM5NDJhNzEwMjYwNTMxNzJmYmRjYTIxZDc3M2FkNGQzYTc3NDM5ZjA1YzIzMTc1N2M4ZWJiZjNiN2RkNjY1NGIxNmNjYjM2ZTg0NWY1ZDViOTQyMTUzY2RhMWM4MTRkMDFkNmZiZGUzYjJjZjA4MTcyYjJjM2U0NGIzMjJiODJiZmExNjc2OGZlZTIwZmZkNGE2NzcxMzJiOWZjZDU5M2M1ZWY0MTZiZmE3ZjgwNzRkMGViYmFjYjNhZDUyNzIyNGUyNGUwODZlMTRjYjFkNmIyOTU2OWFlMWU5MjZhNTY1MzNhYzgwMzg1YzcwZmY2YmE5NzIxMDFmOTQ2ZmMwNzdmOGFhMjMzZDJlZmQ1NGU3NmFjYzlmNTMxN2U0NTc4NDI4ZjM3MzM0MzEwZGY2NmYwNWY1ZmNkYjEyODJiODEwZmFiZjk2ZmExNTgyNTE0NTkxOWYzYzVhMDdmNWRlZWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hufc4kw1XurwBMBQwQKLIGwUnpbOqlY0Nf_0LEwVGTrtd3GJ8I7zgC3I2jhfrSSImhtDWLg4PBswvIOAhZW60Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112957_43_24c6_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.499Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlcxRDNQOVJ1RnlkNVprZ09JY2JVeE1BdTQ4TU5oNlBPZ2hDUzE1NUZSR00zK3dYa2cxK1F4a0hEREc2S0NuOFBsbDhQa0JCZDE1VnZMRjBhd0JaeFRRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTdfNDNfMjRjNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjQ4MWM5MjZhOGMyNTgwYmFhODRlYTBlNmE1Zjg1ZTU4Mzk2ODUyNWFhNzdkMWI2NTJlZjY2NWQwYmMyYTJjNDg1ZTQ0ZmU4ZDE4YzU1M2M3MWFlYzk5NGNmOGUyZmYxMjU3OGZiNmM0OGNjODUzMTUwNzE0NWY5NTliOGU1YmExMWJkZDZjMjJmMzZkMzNlMzc3NTAzYTQyMDI4ZDFmODE3N2Y0ZGM0OGYxMmNhMmJmNTYwYzU2NDkwNzQxMDI1YjhmZDMyMzQzMTEzYmI0YTI3YzNlYzkyOTEwZDhkZmRlODFhMGE2MDUxOTkxYzEwZTg5ZmFmMDg5ZmZlMDk0OWRmYjA0NDZmYWJhZTIwYzcxNTVlYzZiNTQ3ODRmZDM0OTA5ODM0ZThmOGUxNzc5OWI0ZTAyNmZlZWNjZDA1NzZhN2I0MTkzZGJiN2ZlNzM0YTQwYTIyN2M4NGI3Y2MzZGU3MTQ1Y2ZmYTAwMGIxYTc1YmMxMDVjN2Y4OTVjOTlhYWNhMWIzODQ3ZDRmM2UyNmU5YzgzMjJjMzY5MDRkYzM3OTAwYTE5YTFkOTA5NTQwMDI3NTk2MjZhZDQ3Y2NhYjU1YTFkNGViNDQ5ZWRhMmE4NDI4NGU0ZGNhY2I0OTJjYmViZWFhNjljYWRlMWI0ZDg2NTZjZDRmZTQzNDFkZWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.oq6vmS4F_vVALGofurF_dlun-CypqOPDTiJWvn-1L94UaQuNFEL_PjdIxStfap0gYLlvKlbAajzcZq3PAHUtWg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112957_43_24c6_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.502Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ikw3RDVqYjFYeVdCZkt1YzZpQXIwcGtmbHZvdVJZYWl3enFQNjRjMWRLNWs0WDlIZjFucjFZcjIyNXYrMnhyYk9odEsrYWx3aTIxcUFSRjd3bGZwSGZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMzMV8xMTI5NTdfNDNfMjRjNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzAyZmI5ZGZkYTUzZWU2ODllMTA2NTE5NTU0ZGQ5ZTZmNzc1NzQ0ZTZkNjJmMmQ3MmM3Nzc4NzYxMThhOGMzOTNkNDRkMGYwOWI2ZGZjN2IyODAzNDVhNWQ0OTNiYTE5MzRkYTE5NTczMThmOTMxYzY0NjMxNTJkNTU3YWUxMGE2MDY4MmIyZDc3ZDUzNjg4NWZiZjk3MGM4YmFiYmRhOWY3YzEyZmMwOGIzOWFkMzY3MDlmZDMzZmJjYTkwNWJlZjQ5MGVhOWMxMWU5Y2Q4MGY0YTE5ZTY0ODRiNjMxMjBiMmIyZmU5ZDkzMzlmZWQ0YjlmZTI4Y2FkZWI0YjE5NjFiZDc0YTUzMmJjMTFkNDM0MmNmZjc2OWNhZTVlYTlmNmFiYTU2NTcwZDBmODMyNDY0NTNiMWQ0YmY2YWU3ZThlMmQ1M2RkMDc1YWY5NDRlMWQwYWQwODAzYmEyYTFlMjZjOTczMDBiZmUyYjEyMDc2ODFlN2FmNDY3NDNlNWY5YmI0ZjkxNmU3YThlMWQwNzYzN2U1ZDFiMjIxOTIzNzhjN2JjODcxZGIwYTNjZTlhMzJhNmM5MjhlZGIwZjZjOGFkNTJlYmRkOWNmNzY1MWM1MjBkYTg4MzI3NmYxNDU3Y2ViOTU2ZjU4NjExZmNhMjRmOGUwZDliMjdjMjNmODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.krPqf0pz-QUH8K-WPnQ0XzVrr89s9AyXcfIRKF60Ws51V1DW8B0mAEEjAKupHfrr8pt-b4SoRxwhgrDP7jiVLw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112957_43_24c6_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.504Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjV3ZURyRjhjRVorYnFRTzhNUndBYUkrM1pFNGlDVm1KeUc2QitFR0lVaC9Pc3JwMTE1MFVPdWw5R2RleWVlTFhjTkc2d296MGVoM09FNEE3MXlKVi9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTRfMDZfMjRmOV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTIxNDE3ZTg3ZjcwYmRjZTliODFjNTYwNGQ1ZTgyZDcyZmYzMTFkYzUxYTE4OGZkNDdmMDRkMTcxYzQ5Yjg5MjM5ZWU2N2FlNjU4NWFkODY4ZjVlZDA3NWY5YTAxMGZkOTM3YTYxMDliNzU5NmIzY2M0YzZmYmEzOGIwOGM3OTJkM2E2NTcxNGEzNTdhMTdmNjBkNjZhMDZhMDZhMmY2Yjc3ODAzMjY1Y2YzMDVhZjZlNThmMjk4NzRjNzBjYjE2ZjYyOGRkMjQwNzBjZDE2YmYzZDY0MzEzNTM5NDY2MjI4MGUwZGY4YzI1YTNjZmQ0NzFmOGIxMGFhM2ZlNGE2NDZiMGU1ZjNjMmIyMTA3ZGQzODUwNjU0YjgwNjdlN2E1NDU3NGEwODg4MDlmMWFkNjJkMzljNDgzYmY0NWMyMjczZTEzMjZlYjY1NTYwNGVmZTc3OWJlMjcyMjJmMzM4MDJlMGM3YWY2YzQ2MDUyMTg5NzA2YmQ3ZjUxMjAxYjU3MzBkNjBiMzM2N2YzMDQwY2IxMGIxODY3OTNlNTg1Yjc4YzVmMjc2ZTk4ZGNkNjRmOThlNzgzZWZiYTYzNDBmZDRjZWFkZGRkMTAwYmZjYjlhNjIxZDUzZDg0ZWNlMTY0MWQ0ZTcwMTJhNzYyN2ZmM2JjNTEyZTM0ZGNhNzFiYWFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.NPz3EN3hHoTpzr7-eqNd0GrbTY827kLjQTv6YFnYQVJQyhYFzZmxNQvAZ4kt6dRObR0kt-rUQoDNVOUXQ0rGIw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113314_06_24f9_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.507Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBkY0l3c0dMVXpvZWhjVEQ2SzlSckNhb1hjSnlHOVd2aFp0aHNkT2J3Mi8rSE02NGJTa1g1WXhjTHlMYm83V2lRVS9TRUtDdHhqNURsMmhmT3pzQVFRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTRfMDZfMjRmOV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDI0NDFlZmNjMjRlMTBiYTJiNTU5MDExODM5YTQ2MzhiZGNjNGJjYTE0OGNlM2ZkOTY4MjM5Y2I4NDE1MjRlNGU0YTZjY2RjMzk2MTBhYTczYzYwYWYxOGFmNGE5YThmMzI4NWQ0ZDE3OTQ3MjY3NzMzYzU1NTJmOTg2ODIxNjRkMmExNDk3NGU5ZGIxMDRlNzE1MWE1Zjk4MzVmY2Q2MzM4NTlmMTUxMTdmMWU2ZjI3Mjg3NmFjMzZhNzQyMGMwNjMzYzdlOTI1MmM4YTI1NjlhZDk2MmMwZWZmZTEwZDJkY2M4YzNiYjY4ZTQ2MTVkYzRlMjIwNjQ3NDJkYjFmZTdjOTU3ZTNjNzYxMmNmYTk2OTEzMWQ0ZWM0YWY4OWU5NTA2MDU1ZjBlNDg1ZDUwNDRjNzdhNzE0ODVlN2YxNTJiMDBiNDY3NDE4ZDQ0MjBmZGMyNjNlZWFkZjY5MDRlZGY4MDNlYzQxNzU2ZDMzZDZmOWI0NTA3YjA1NWFjYmMwODY4MmRiY2NjNDhiODBjM2I3MGJkZmJiM2VlOWMwMjJjYmQzMzQwMjM5ZDk3ODQ0MjY4MTc5ODZmMDJkODU3MzUwNzkzN2QyMDNmMzQ4MjVkODVjYzAxNWY1OWMzMzIzZDAxZWIxNWI1M2MyMmM4OGI0OGYzZTAwZWU0NDYzNmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ER5lV4x_J3XILSvbCT3vWO-ayT1pdTQzbi2at_VLwECBXM76yM0Hq-HYCa3ZxkdcD1wwMZb0-wmjdyu884_4HA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113314_06_24f9_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.510Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFPcXBXdlU0TXZMdEpSVEUvTEZ0TEpwY0l6VlIxRjBjeFNpNDh5SEFyc2czWWFVSXZaYWdZVGFOV0wvVVc4UHJEY2w5SVFpa2FGK3VzWm9iTlRjbEFnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTRfMDZfMjRmOV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODEwYjkyMjczZjIyNzZmNTM5NjVjNjFlYjFkMTRmYWJlMTZlNDUwY2E3MmM0OWJlNDg3MDQ1OGI4ZTEwODA4ZDFkMWU1YzRkYmM4YjZlYTZhZjRkYjY3Y2UyMWQxNDI4NWE2NWY3NTIxMDg3MmM4MjdjODVjZDE4NTI1ZDYzY2EyZmM5N2FkMDA5MGY2N2UyOGI4ODY0NDQ5YjgxODQ4YmJlY2Y5YTRlYzY1MzI1ZWUzZTM5MDgzM2E5MWY1YzkxODk0MzM4YWM3YzVmNDlmYzRjY2E4ZmY3YjVjZmJiMDk2NzUxOWE0MzNlN2MyYWJjOWIyMTExM2JiNjkzYjYwYzU2ZDRmNGJhMWIyMWVlZDJhNGYwOWFlMThlZTYwYzM0ZWE4YWFlYzZiOGI2Y2EzMTQzZTVmNTVlZDVjZjhlZDBhMzE5ZjI5OWIyOWVlYmE0NGJiNzlhN2VkZTg2MDM1ZDg1YTg0Y2VlNTU4M2IyNmYzY2UwYTEzMzAwYzY5MTA1NGFjNzBjMzkyOTk0ZDE1NzcxNDEwZjI5ZWE2NjA1M2M4MGZiMzNiYzMwYmYzZTQ5ZmM4NmU1MjlhMWUwODMyNzVmYzM2YWMxZmQwYzk0OTkyNDhkYTNlZjczYjcwOTdiOTAxN2NiZGYwOWI1NGY2NjZjODYyOTMzYmUwZTgxYTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dfrr9TYNmuvV2q-qQ_D2unjxnHzO09QppWKrJVykBeYbmORUUfqLzwslTLDtYk4DyU27gqiA9gCjWG0wO_WKyA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113314_06_24f9_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.513Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjYwTGRXZzFtY3pSVWw2TEwzUmFNbTBmN0lnT3BraDhDN0FCQjJRcEZObnRkRUZkVzdMOUVTOHBZZ2FLK25FbmViSHdqTTNKWXRITVRkK2daTTZLdHRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDQwOF8xMTMzMTRfMDZfMjRmOV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MzY4YzllZjdiYjZmMGU1MThjMDc1YWYzMzYyNGNkZDQ2N2RlMzY0MzEzNTBkODkzMzgyZGZjZmVkZGUwY2Y0ZGQ5NzMzZGMzZmNmZjcwNGViYWViNzI3ZWE2YWM1MTVkM2RhZTE5ZjIxYzY1ZDJkZDdlNjAwMDc0OWVkZTNiN2JmZDZmYzVmYjhhZDQ3OGE5NWEyZjYyMGQxYzBlM2FlMzVhY2M4OGYxZWQyYjNlMDcwNTZmOThjMDc5ZjUxYWM3YmM4NTU1ZTJiYmU4YzIxMTYwODNjMDM2M2Q3MDRkNDA5ZDg3YzViOTg2ZjMyN2QwMTA1ZTNkMDMxODJhZWNjY2U0YjljMjRkZmZlOTJiMmE2ZGNmZGZlMDJiNDIxMDI3OGYwOGQ5Y2ExMDNkMmZkOTllNDEwYjliZjYzODQxZjg4MDUyYWQ2Yzc1NWQyODUyZDI5MDI4MDExZGMyNjYzMTA5ZGY4M2RjOTc4ODc1ZGU0ZGRlODVkOTU2OTVjM2NjZWQ1OTYxODAzNzQ2ZmI3YWQzMzg4YTc1MzY3YjFiMzQyOTJlZjM5YmMzNTJhODAxZTk1NmMzYTRiOWMxNDRlYWExY2NiYmM1NmI2ZGI3OGVjYTA5YjE5NWU2Yzc0NDkwOGI3NDQzNTRmNDJiYjYwZDllMTdiZmVkMzhkNDVjYjNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.XpGqTmxaNMgnaxkQ98ap4qvjSpQu1OQlNXGMIza287zVsR4oi6kybaXOUjZur72bL-BQUZfpgQBmyGaV2-MCmw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240408_113314_06_24f9_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.515Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFxRE8xWjhsQ29zcEpXejdVQTFycWg4SVk3b2xuWHV0bkRRUlk0cmVEeDJEZExxTHhWd1ZXM2pjNkpyNlpTbHFLUHFSVXFITWhJN3BCaExiMmxsUTdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI4MDJfMTlfMjI3Y18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWU2OTQwYWY5MTA4MTdiYjFhZjAwNGU3ZmZiMjFkMmM4NGJhZWRhMTBmODJmODQyYTI3YWYzNDFiZGFiNGI2YjMxODk3M2Y5MTU1ZmMzNDY0N2Q1M2JhNDAyZmRmOGJhYzBlYTFhNmFhMjRlZGZiNDZmYzhkMTg2OTZjNWJiYTRkZGU1NjUxYWJmNzA1NWRlZDUyYjIxZGM3ZTNhODgyZjJlY2UyZDM1NmRhOGM0NzhjZjliN2IxZDgzMDhlNzM4YmE1NWIzM2QwYTFmNzU4NWExZTNhOWMwYzMwMmFiZjYwOWMwZGRhYzIwMGVhYzkzZWUyMjU3NzA2NTE1MjllZTAxZDZmOGE5OTcxYjIxNDBmOWU1MzRjYzMyNzRhY2Q2MDBkMWM1OTc0OTQwZDY0YmM4ZTEyZmM4YzFiOTJlYTZkZGEyZWNlYTQxYjA5YTU2Y2I1YmY0ZTM1Zjg5MWJjMjdhMTI3Yzk4ZDk0MzJmNzNhYmM4ZGMxNzJjNjFjN2JkMTczMjBkOWMzNGM4ODhhYWIzMjg4ZWUxNTgzZDlmZjEwMzI5NDFhZmUyOTQyY2Y3ZWMwZTBhMWNhMzY1OWE1NWVlYTdkZGZiNjk5ZGEzODQ1NTdmYTVmMTE1NDdiNGY2NmJkODdjZWNiZmIzZDExMjI3MmExZDU3NGM2ZGJhY2FcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.FQJdC_16raUXXlhnE7dw86BW5kp_O-M-X9hZgkcQ_Truik0mziHDtlPyAZj5zaVVREcCLW5fuJMqYT81FQhIHg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112802_19_227c_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.518Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlZGYStTMFhlU3EvcW0zYW9Ld2RlMXoyaU5IaUhUMlQvbzJ4STRjaEVnTkx3QUdPa0VXbnpaVEtsaTRZSWNtWnVicEZDTXJGemZxRzQ1ZjMrMnBwQ1R3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI4MDJfMTlfMjI3Y18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDY3MTJkNGI1OGQ1MWRjODQ4N2I0NTFlMDYyMDg3MmE0Yjc0OWRiZjBhMWUyMDZiMDM1MzZlMzg0NjQ0ZTIwNTJiZjYxYTk2NTIzZWIyM2E2NWE2YTRmZmMzOTk0MWEwYjYwNzg2ODgzNDkzYTQ5OTQwMGMwY2QzNmRjMmFiYjhhMTgyYmUyMTU2YWIyM2JkMmNkOGJmMzg3YTJlMTMyOTY5ZTY0NzEzNmYwYjMzMDcwMzI2YWRkOWMyYzNjMTg2ODE3NWYzMTY5NWY4OGVjZTQ4YzFjNTZlZmNiODY2MTU0OTcwZGNlYzg3YTU3MDhkZWY0MGY2NDNjZDE5YjlmODRkZTQ3ZGY5NDZmOWYxYTlmOGQ2ZmU0NDVlMzRlMGJkODVmMTNmOWVkODU0MTllNTQ1NjRjOWU3NDNiNmRlNjdiMWIzM2Y0YzQ5MjNiOTYzOGYyOTE3ZGRlODUyOThlYTE0NDVhMTRiYmQ4OTg1MTU2ODFkOGQ3ZmFmOTdlNzUyNTBhOGVkN2NkZTAzZDgzOWVjMjU5MThkZjcxNmRhMWM5ZTNjYTE4OWQzOTZlYTZjZDcyNDhiYzhjOGYwNTE3ZjFiYjhhMjBjYjUwODhmODM1NDcxODQ3OGQwN2JkN2QzMWYxMmVlNDlkODAzYmQ4ZjY1ZWRjM2E5NGI1MWJmNWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rzIZEj9qRD-fuI8Efrcs7_5o8ZuOr_-FZo1OEV-WbCwNrIfHmtdwZ7B6Qnh2KKC5L4Poq9EN5pUQ3E-JksYPqg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112802_19_227c_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.521Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImV1UDNGQktROGJmNVZXMWlyZ2NTUnR4ZG5RWGwwdVF4cUJNclA5MUxxYlN6ZmUyNEpZQ3paUS8wblkrVUNSOHV2Mjhqd3dnUHRkYm5laXRMTTNBaHJnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI4MDJfMTlfMjI3Y19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmVlMzVhNDE1NjI1MjZmNjYyOGViOTViZDNhMmUwOTIzMDMzNDZiZTJlOGY5NDhkMWU0ZjI5MGUwN2QzNTk0ZTg2YmUzZjkyNzVlODhmOGY5ODczMmVlYTk0MGEzOWE3ODk3OTI0YTEyNWJkNGFmMTkxMGVkNTNkZDRhODljODljMWY3NjEwYmRhZjhhYzE4NjdhYjEwZjI0Zjc5N2Y2Njc5Y2Q4NDJmZjdiMDFkMDExMWVlZTVjODhlYzI5M2E4ODI2ZDIzYWY4MDM0NzAyZmVhNzNkYzM2NmUwNTBkZGQwMjQxY2Y1MmUyZGQ5ZDFhMWE0MjU1NGM1NGNhYTI5OThjMDE3ZDRiZjJhMDIyYjdkMmU2OGY3YTg2MmVmOGQ2NWMxMWM1NGMxNWJiZjEwMTA5MzE1NTdiOTcwMTM2YWZlNGE3OGQxYWUzOWY1YThiM2Y4NmVkMWM3MjQ3YjYyMzk2MjRkZGM5MDM4ZDAzNWM5OTYyNWM2ZTYxNWM0MjMzZDk5ODFiZDY3NDU5M2RiNTQ0ZWE3Y2I2OWFmMjI4OGU1NmQzZGJmMGMwNzJmZTE4NWU3ZGVkMTA3MzRjNTNjN2FlOGFhM2RhNDc1MWU0NmY0MTEyZjU5NGUzZjk4ZTIxZjFiZmJlZWNmYjY2NDE4NzJmYjIyMDA4M2IzY2FmMGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.zgo5APqGpp95jve_qbffUCOBqWrpgVhHFtE1RIqdfgMRk_d4rKW6TPjaZizDLX6Nkf9V2mb1Xk-Ga58tSb1JIw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112802_19_227c_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.524Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im01Zk5wMTN0dTd6MEVRaW4rcFZhT2hSeWdSZllCOExZMlpwNmdDblNieDVnbS9SSEdSZ1NaNklxQmoxSHpsSzJKVUs5N2ZxSlk0clFQQUpRVHlhOWdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIxMDUzMV8xMTI4MDJfMTlfMjI3Y18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTg1ZjZiZWJhNTU1NzUwZTQ0YzA1NmViZjkwNGUwYzUxZTNjM2Y1Y2U3MTQyMWNjOWMyMTgwMmVhMzQ3MjNjN2ZkZGI2NDEwYWQyODZiMmVmZTM2MmYzYjk0OTc5OGIxMTI3ZTJkMDkyMjRiMzgxZTVmMDBmM2NiNGIwOTk0NTY4M2M2OWYyYzNkYWUxYjRmM2Y4ODdkOTQ2Y2QzMzNkOGIzMGEyMzQzMDE5MjJjZDcyNTAyOTI1ODRjZTI0N2Q1N2FlNTA1NTA4MTlhMTc0MWE2MTk1MjY4NWZkODQ2NmZkYjRkYWJmZTE0ZWJmMmI5OWZmMjZmZmQ4YzFlZTAyYzhiODU0MDkyM2ViZDc0OGUzNzA0NzVlMWFhMjA1ZjkyNDg0YjdkYjMzYTUwOTk1N2UwYjg2ZTBhZmUwMmYxYmMyOTM3Mzg3Y2UwMzk5NTUxNmQyODNhZmMyMjY1OWJkYWYxOTcwMTBkOGM3MjM4NDZkM2NiZTU3ZWI4ZDE4Y2ZkM2MzNTAzMmJjNjEwZWE1MTNmMTBmNWIwZWMwZGZlNmViNWMzZTU3MGM0ZTA0OGE4Yzg5NjYwMTliNWRkOWE1YjQ3NTkyZTQ3NWJkYTlhOWQ3NjI0ZmRiZmFlNWVjMWY5MTY3OTE3ZmY3OWVhOGM3MGQ5NjI4Njg5NDZhNWFkNWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WgACPz2h4_KwAFBZ8zjKsHbMKvVwNhpiiYkQTTn7sXChXyyP132PkNiUiQXmSmX66-IV93KACCdRfBQi3Paw9g", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210531_112802_19_227c_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.527Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkNMSTM2MzlCNFI0U2NDVDFEVFlvQVlmV2F6dTYxZW05b3lZUzNjMEtyckkzOEdsemcwbTdHTm90cEpkSUNzQkczdEFDYTN6VXJmMGpWTDFZcTZYenJBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM1NDZfMjFfMjRjNF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzQ0ZGZjN2FmODg0ZTUxYTAwZGEwZWMzNTY3ODMwZGMwNDlmYTAwMGY2MjQ1NjJmYTY4NDQ0YWVmNTk0ZTY3ZDQ4YmUwMzliMDNhMDRjZGM3MWVhYTFkYjQ3ODY1NmQxMWUwMjcyZDAwYWM5Njk2ZWU4NmNkYjQ4NzRmOTE0ZWVhZmE3ODM1YTI1NzhlZDRmYzMxOTYyMjY3NDI3OTBlZmEwMGYzMTJkNTBlNDk0OTBkZGNlZDFjNzE2MTg0ZTk2MTI2Yzc1NjI0Y2E0ZjVmOWQ0NDU2ZWNiOGIxN2JiYzdiNDI2MGZlNDAzN2Q2MjE5ZDU1NTRiNmFiYmRlOWFjOWRmYjI4ZTkzYzY0OWE5ZTRlYjk4NzBkMTkxMWU2NzdjNTZkYTkzM2I3NmE5ODkzMTc1MDc5MGQ2YWY5YTQ3MGRjZDkxYjllNjg5OGVlZTZiMGVjZjFiZTNjMzllYzdmZjY0ZTI3MjA4NDRhOGI0ZjliMjI5ZmE2MGExNzJiMGQyYjZmNTQxOGJlZDg5ZWNhOGIwODQ2ZmJiNjVjYTI4MDk5Y2I4MjJkOWU5NTBjNWNmODM5MmUxYmVlODlkZjdiMzRjMzE5ZGJmZjZkMjczMDYzNzU1ZmJjMzIwYzBjN2RhYmQ3N2FiNGFlMGIwM2FiMjljZjAzMzk1YmRlZDg1MTRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.OptoYUNLaMJ9d9XwwxJnU3SoTUju2ehtH0mX5f6V3mHzv4TfuVhGrTJ3pGeLix_O8mry2KqroOG6fCzWk0K6Qw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103546_21_24c4_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.530Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlJPTGFWcDBhQU9CdGZTL3ljRk53c3F2RjNPL2ZSTE42bXZHSVVDYjZwVklOM0Q4TStxejQ2YUtrNVlFNlJkMUxlU0ZleTdlMnRwU3ZaT0p0Rnh1dkR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM1NDZfMjFfMjRjNF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmM2NDY1MmEzMzc3ZmM2ZDA4NDA1OGYwNWY2MGE1NmNmMDA2Mjc2NDE3ZjlmMzUyZjY1ZDE3NDhkOGE0MzNmZGU0OTU0MzVlMzk2YWIyNGNmMjU1MmYwZjc3ODVlNTFiMTI4MGVjMmU1MzNiNDE0OTllN2FiNzYyY2I2ZjYzYWI4N2ZmZDliNDI1YTNmODllOTZiZGY0NDE3ZTU4MTIzMTA5YzU0ZmZmZjJmNzdmNTVlODA3YTg4OTFlNDNlMzFiYmJhMDRhNWRjZjAwMjAyZWI1ZjIwNTRmMWZmZjhjOTE5M2I2YThjODVhMjFmMGIwOGJhNGYyOWVkMmY3MmUwOWFiNWM4YmQ4YzFhMTU2MDY5NDQzZjUyOTU1MDI2ZGNiMGEzY2FjNGQ5OTgxNWYzNmMwMGQ4YzMzOTFiYjVhMTY2MmM5ZmVjYTNjOTFlNjE5YTdjNDEyZTc4NzQzM2Y5MTgzNGEwMzMwOTRmOTVmYTg3NmRjNGM5ZDYxNjE3YmUwYTk1NjhlYjcyYWRhYzMyYWZkYzNiNmM1YWMzZDVkM2ZmZjkxYjQyNmYyNWY2Yjg2YzUwZTliMzVjOTEwMzU4YTQ3YmMxOWM0ZTJiNzE3MDkyNDhjOTYzMDA2ZjViZTljYjMwM2YyMGZhMjJkMTZjMzVlNTQyOTUwMGEwYTJjNGZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gVE6yjnUoUxz8RL-9bGi0NvEgI7pv5KHEH3_aEB8_PaDbTNz3pEDT1RcDNNDSW-i4A5KEvGfzCSR_yAqaEYELg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103546_21_24c4_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.533Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBxTGF1WHY3Nm1UUnpDUFZ4WFgyNXZOWmZBMzcyNHpBNWJMRndtcnpXUHo2KytDWFBQMmZYQ29DV29CRk4wQnByajBJUHNaNmxEdFNTZ2xlODZRRjRBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM1NDZfMjFfMjRjNF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTZjZDhjZDYzM2FiZTQyODk3ODJhYTQ2ZjM1OWEwMDU0NjBjMGRkM2Q5ODA1YjA1ZTllNGY4YzMzM2JlZWNlOTBiMzVhZTZlNGRjZmY5ZjZkMzkzNjVlMzMxZGRlNTY4NDE4MzI1ZDNmODhjNDIyMzBlNzI4YjYwMjMwNTIyM2E5ZWNkMzNlZGFlMTliZTUxZGVmOTk1MDE5ZjZkNGNkNmFjMGQ5NmFjNDgzMGQwODNkNDI5NDg1YWRmOTE4ODNhOWQxZGNjNzMzMzg0OGFmYjZiYTBmZmIwNzBlMzQ3ZjIxNzI1ZmEzOTkxMjY2ZGI1YThiYjA2Y2VlYzVkYzdjZWRkYTBlOGNmNDJiNzlhMzU3NTZhNDQ4MjI1ZmVjN2ZhZmM0NGI1MjNiYWQzNGUzYjhjNTM3OGIwMmE1OTQwNGI3NTQ2ZGEyN2RmZDk0MzJhZTFiZTdhMTNlNmI1YTVlNDM5ZjhmMzg2ZTU2NDQ0MGMyM2MwMDg1NzEzNjMwOWM5MzdlMjIzOGY3YTljOGM4NTQzNGQ5NjIyZmRkYWU5YWE5YmMzZTRiYmIzODA3N2Q4MWU5ODY5ODViNDhiZWE0YmUyZGRlYjM0Yjg2OWQyNDBiOWFhNTY1ZDM1ODczMjk1NTNlYmVkNTU2ZDJmNTM1MzA4ZDk5MmJkZTc3MGQ1ODNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ccbhEMygGxZJkmOxQeqWPO4qQME2GlqtaC6biAO7ge4vpOsigaVwSGChpofoh0FdO3MMe89EjL1hfzshNBs4YQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103546_21_24c4_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.536Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im00MmF0TndPWlJvK2R5clhRbFBmVVh6dlhLd1FmMDYzZDJpR0lEYTdVNkxrdjlDS3FmQTQ5MzlBb3RhdXNJajRZZWxWTmI0eEpaTWc4L25ET3VBdThRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDcyMV8xMDM1NDZfMjFfMjRjNF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDc1YzdmODgwNjkxNjlhYmE0OTZjYzlmM2MwMzNmNTFhYmFlZWUwM2UwOTE2NzZlYTkwYmMwOTJmY2EyYjg4NWU2MWY0OGRmNmI4MmRkNGExMDU1ZGY0ZTA4YTg0ZjRhYWJiZGNkZTcwMjczMTk4NmY2YzNlMjYwY2NlODM1YWFiNTBiYWJmZmNhNjhmZGMzYjA1N2UxNzhiYWRhMzlkMmQwOWUxODk0YjVjODI4Y2EwNTYyN2E0Y2RhMTMyYjViYTNhNDRkZjdjYzkzN2NlOGJkNTEzZDY4Y2NjNzQ1MDk5MjkyMDRhYWUzOWE3NzA4Y2MyNGMxNzcyMjVjODE0OTVkMTZiYTk0NTNhM2U4M2FhN2I3NmJlODE1OTljMWFkNzI0MWQxYjhjODIwNjJlNzM1NGQ2Mzg4NjczM2M2YWE4MTVmOGQyNTZlOGViMDQxYmJlNGI2ZjgyODgyMjk1ZjUzNDE0M2Y3YjA0ZjkxMjQzNmEzMjNiM2M3MGY1ODc4ZjBhOTk2NTI1NWU2YmU2MjczNzkyMmFmZGM4MzA0NDRhMzg2M2IzMjUzZGJhMzAyNWY3YzdmNGEwMjZlNzhmZGFmMGM2NmJkYzkyYjNlOTZhMzM5MjI4Nzc3MTM1YTM1ZGYyZTJkMzJiMjUyMjk4MTM5YTNiNzAxYjQzYzc5ZjlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.4eGWaQQeYskbeenKTYoKni91k2msw6zH-Wt8jJM6_03u2zCtszeCkDOFGHKiVq73sQF6vRu3nNJMCwswKBClUg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230721_103546_21_24c4_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.539Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im93emZGY2JJV0s2akJKZnZYY2pLVndjMkdzbHQ3NUtUa1dreEo3L2VNaHZvM1oyNGR3ZVNhTjRpYUJDMFhWN2Znc3UrV2dnbUg2MjB4Zkxoc0VGYUl3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcxMF8xMTAxNTJfMTRfMjQ3ZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjYwM2JhNWFiNDViMjBhMjc4NmY2YmQ0YzBhOGJhNDZhZTdhMDgzM2FjY2E2M2NiOWI1ZmUzZDlhZGQwNGVjNDU0ZWQ2Y2YyZWU2YjdkN2RjMjBkMmM1YTA3YmZlYmE5ZTU5ODZmZDZjYWJlMzIwNjY2MjVmMTAzOTMyMzMyZDRmNDE4NjY3YThlOTNlMmI3ZTZiYTg1YzZhODZhYTFjYmNiZDZlNWQ5MTM5NGQwMDg3NjE4MDhhYWZhNzcyNDU0MDEyYmUyZWU3NmVlZmNmMzQ5MzdmZDUyNWU2YThkOTAwYTY2OTZhYWI4MGExYzkwMWEzMzU4ZWQ0YTg2NDFhNjM2ZWFkNGNiNGUyNzJlYjdkNDE4N2NiZDViNTc1NDI3NDczMmI0ODExMjExYTYzMDY4YzZjYzJmODFiYzZiODQ5YmJmN2E5ZTlkYTgzOTc2ZjBmNTNkZTk5MjAzYWZmOTM4YWRkODk5OTBlZTZlNGNmYWYxMTcwMjI1ODJmMGM3NDBmMTZhNWZiMDJiZWQ0Yzk2Y2FmYjRjY2NiOWU3ZWM5MzUwNzM1ODVkZDI0ZTFmMDM0OTExMTM2NzRjYTQyMDA2NjJhNDNhOGQxZjQzZTIyOGM1NDcwODQxNjI1YjEzNzdiMjg3NjBlNmQ5N2RkNDZmNWNiOGQ0MWI1ZWE4Y2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.iaXZWnfjtFwOltCs8hO4DLp-01BGmZGXRf51wmYML6GqP88Ay3bTneOUHtD7K5ouCQqu_mO3V_naLW42BPleLQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220710_110152_14_247e_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.542Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik12TndXaVUwWUhpQnRwdjBHaitiU3NJR09Wc0VwYkpURmkyVXFLQjJMaEtRUm1YZ3NBcENOWUlnQTluUnlESW0wQmZ2dHhRU1Z2aWJIbkF0bjAwbGdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcxMF8xMTAxNTJfMTRfMjQ3ZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmNhNTk5Y2U2MmZiM2VlOWIzY2E1OTcyOWFmNzk5MjkyMTNhMTA4ODliZWEzNDIxZmUxOGRkMzdhOWM3MGI2NmIyZmQ5OTQxZjhlNTNlZWExY2Q3Nzk2MWM3MWEyYTExZjEwMTAyM2QyZmZkYTBkODU2OGM0NzVlNmY4NTczMWMzYTJmNmE0NGJiN2E1NTRiOTQ2ZTgyMjE3ZWI4NjBhY2QwYmZkYjVkNTRmYjgzYTkxNjY4ZTdjMWJhYjJmY2U3MWFmMjMzMGFkMzQ1YWRlNjgyMjJjZTY3OTI5MjNmNmExMzYzMWY0ODM0YTRmMjFmNzUyZTA5YjgwYzcyODgxZmI0NjczZDliYzVmZjYxMzk2YzE1OTBlNzE5NjVhOGU5MTg3OTVjNGZhYmZkODgwZmNlOTJkY2FkOTQxZTE4OTliNjg0ZTExNDk3YjlmMzNkZGYyNGM5MjIyYjhkNjI2ZTZmNDJmOTNkYTY3N2VlOTI0YjZmZDA0YjgyOWQxMWVhY2Y0NmRhYjUwZTQ1NjdhNzAzOGZhYmZkZDNhOTQ5MDZjNWU0ZmZhYjE4YjcxZjdmMGYzODI0MmY1ZDQ3ZjljMzc4MGFjZDU3MTZjZTU4YWI3NTcyNGFiYzM3MDgzMmYzM2NhMDIwOTgzMzc0NDJkZTA1Njk4ZjEyMjAzNDQ3MWRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.TzQ3zwcjdnXJBAbk7mzeWl2RS0Y-toW8Q2zD-PQIEAOo7DaOmvTW24VITStq1gf8FlXOoW9TUYPyroz1xFMMUA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220710_110152_14_247e_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.545Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjF3ZVJmNDUxYllqL3ZiUFRVeTZJK2N2V3MrTGlHbEpHdUZVdkJIQkhQQUYwZzZObHZ3cm1VYVBkN2NCUGR1SkF1UUlEVnZjSUtHVXNGbHJVVkJxMXhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcxMF8xMTAxNTJfMTRfMjQ3ZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODAwZWZmMTFkYmI3OGEyNWZhMDA5YmRiYmM1NTVjYTg3OWVkMzE2ZjI2MTZmNTRmNGMwMmJlODY3ODdmNmY3ZWE3MTVhNjUwNWZkNzU1MTE4YTFkYWY2MTc3NTE3ZDBlMmIyODJmODBkOWM5ZTVmMzY2ZTU2MWZmODAxMTRmM2Q2NDc4Yjk2OTg1ZmU3MzgzNzUxODg3OGZhYTE2MmUyZmQzMjYwMGZmNzc4ZjAyZmRlZmYzNGNhNTY5MzRkZjIwY2VmMzk2N2FkNGRlOTY5YjA2ZmZmMTQ0YWMxZThkNzAyNTg3MDgxYzhlODcxMzA2MzAyYzZmMWY1MDA4ZjVjYWI5M2FkMDZlMWY3N2E4OTM1YmNhYTM0OGYwODJiYTk3YWQyZjBjOWE3MmNlNzZmOTY0ZWY2NDViNDZhYzI2MjdjNGY0YjU4N2U5NDc2MjM1MGM1NzVjN2Q3YjE2MWJmZWIwZTdmNzljNWMzYmM5YmM5MWEzMDQxOTNjNjY4MTM2YzNmNDQwMmQ2ODdiYTUzY2VhMGYwM2Q5NWNhYWFkNzQxZjczM2U0ZDEzNzUzNDg2YjYwZjZmMjlhODZkNTdmMTNkZDQ3MTg4NzQzNGU5NzBlMjY1MjMxYjRlNGFiNTg1MTUyZWViNDI1NDgwZmQzZWRkMjYwYTUzN2IzZDZjNjBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.1IzxOm4mx42zqSlpaeeDrKFbswlOTwRH2nTdaz86ruIT47KZ1EGvaU5inBBw4D6A-Vr1oYes10cmwW_yzlomVQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220710_110152_14_247e_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.548Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjVhUHdrK0d6VnVPYjRVeENaMU1veFpZZWFQL1BQeXJOTWljZjVQcVByQzkwc2VZbm1YZFM4VGN4bnNHcE9CN08zbUlISllVM0QxalFybXRZdDlvMFVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDcxMF8xMTAxNTJfMTRfMjQ3ZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2FlMjIxMGU5OGNiOGUzY2FhOGIyMGExOGRlMTdiZmM4YzQ5ODE4YzE0ODBlMTJkOGVhMTQ2MjcwZmRlMjgwZmNkMmM0ZTA2OTMxNDU1YmU1YWE2MTMzODMwZGYzODBiYTVkZGQxOTYwOGJiYjc4ZWU2YjdhMDkwNzMyNDFlMTVhMzRlMTAxZjFmNDMwMjE3OTU1YjI3ODYwZThiMzk0NmE0NzE1MWVkNmVkODE5ZTAyNjk5N2YyOGI3NmY3Y2RiMTQ4MmM3YzYwMGMwOWY5YTcxYWQwZTgzMTQ5NmQzNGVjNzFlYWM0ZjBmMjAwMzhlZGIyNzI1NzY1ZTdkYjg2MjQ4ZTVlNDkwNDQ1ZjFhYzdlMDBjNjAxZWNiZmJhYzg5ZjVkMDg2Y2ZjNDc4ZjJiZWRhNmU4YWNiYjMyNDk1YWRlZTM3MTk3MzQxNDVkOWE4ZDZjOGM3M2EwZjc3NjdjZjkwMzk2OTIxNjkzMjYwOTA3ZGVjNmFjMjY5NGI2NTMxODU0YWRlMGRlMWQxNmVmYzljNTNhMWQ4YzcxMWI5MTk2MGJiZDA3ZjU4OGIzNDY5NzEzZDc0MjVmMzgyYmU5NjU4OTQ3YmQ1ZjU2OGU4MTg4MWRhODY3Y2M1MzMzMjI0YzU4Yjc2Y2E1Mzk1ZDQ2ZjBiNjBjMThlYTM3Mjc1NmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.RnWq3QFfGGHG1zKbD-IohJhFeNWkRa5QbAqBj_Stg52loBCNCTmtO2hTTkO8mmeYTZCr160IP0-aDdIYD2EljQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220710_110152_14_247e_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.550Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InBVUC9UeGlpUS9kMk16Q2xoUGIrcVhUZWpBMTVaV2x6VGJkdTFiV0dzbVc2L2xJUXNzdXpQVm44Q3JVNXNzcjhXdFF5dmRBV2RhSncreUIwR1RJYjlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTMzMzhfMjFfMjRlMV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTU5MGFhZDdhZGU1ODI0MzIyZDU5OWRmMDM0Njk2YzFhZDg3MTE2OWMyMjc3OTk5MzBkZGY0ZmY0YzMwYzIxYWZkNDY2MGYyMjc1OWVmZjliM2ZkNGYyY2VlY2Y3ZWFhYjJmYjg1YWJlZWE5N2EwYTA3MDY0ZjgyNmFjMTU1MDVlNDBhZTQxNmVhN2QxYzU1ZjQ1NDFjMTM1ZTk3NzNlMWRjMjIyNjVlZDQzOGY1MWJjODdjNDIwZWM2MThlNzFiNjMzMGU2ZmVlYTdlZThjZjViOGU4YjQxNTI1ZWQyOWE4OTg1NmE2YzdhMTMzNTdhYzJkZGU0M2I3MTk1MzI2ZWYwOGQxY2YxODA5OWRlMzBmYjkwMjk5OGYwOGMwMGJiY2IzMDNhZGU3ZWViNTBlNDE0OGIxODNkMTMyNTIyMWZiYmIwNDNhYmEyMDEwNTk5YWMyMWEzYThhMjE4YmZjMzAzYzQwMGVmZDljZGQxMTRlMjVlOWFmYzIyZTYwY2I4MTk0MDVhMjBkNDk3YTQzM2MyZDgxOTQxMjBiYWJmYzdjMDM2ZjFlMjFmNmMwYmZhZDM1OTA3ODRjMGQ5YmYxNjllNGJhNGVkMTg2NDg4YzM3M2Q4MWIwYjk0ZTYwM2Q2ZmMxOTQ5ZGJmMDk2MjBiODUxZmNkZjg5OWY2ODQ2YWVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.N_Ns0kIKJUTlYuNwiRNvc6i7Nol-DOyEo3JmwMhUMkuuC_5pa-GNt1nYCH_iVWinTDmiOs3c7n5Be474n9eRqg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113338_21_24e1_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.554Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjZSTnlhWkMvRHo4S3RHSFVVWDRjczA2WXFKblNoQlI2bEpkQWg4ek5IRWo1S2tmZ2t0MzFCUS9LdjZpTS8za1RrWGJNNC9MN1p6aXd4ZG10WWJ6MFFBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTMzMzhfMjFfMjRlMV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MmUyNGY0N2M5NjI3ZTEzN2I5NjVjYzdmMmYxNjE4MzI4NDlhY2RmNmQ5ZWI4YTYzMTc1NmU2NjE4MjRhNmU1YTU0ZjExNWNlNzllZTkxZWViZTNkYzVkZmZhNGQxNjQzNjQ0MTRlZThkMWUxZDk2ZDNkNjViYTE3OGZkYTNjYzkwMWI1N2M4NDkyN2M3Y2I2NmM4NWJhYjQwZTZmMGRjZjJjMDlmZGZmMDA1M2RkYjY4ZTE2OTQzNDUwN2RlZGMyNTg1NjE2M2VhMzFlNjczMDBmMDM4ZmFlZjUxMjY1ZThiZTgxNDIwNGZmYWIxMWZiM2E4NjA1MThhNjNlNTY1ODcyZjBkZTUwM2VhMjVhYmFlOWM3NDIyMjkxNDAwYzQzMjU4NjAxNDkyMDFlNWMzYTJiZjQ0YmU4N2MzZTE1MTlhMWI5OTA2OGM1NTAzZjIyODA4NmJmMGNhMmQxZjRhYTQ2YjEzNWVjY2Q5MWJiOGQxZWRjMDEyYTZiOTY1NzdlMTc1ZWQ5ZDAyMzNkYWM2ZmFiNjEzZjliMGY1MzBlZWEyNmJmYjI5OTMxMzIzNTM2NjRlYWQwZWI3OWRmNjZmZDFiZTliMjg1NzFmZTlkMWVlYmViNGRkOGMyYzYxZTQyMzcwM2JjYTgyYzFjZmEzNjFlNTEzZjRjNTgyZmYyMzJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.HuAVQgLBRWKYfn_V7ktcwAqpvg6vPDGkmvPPkX0mruwH_RRwIvy2cOHb61s5XJqoFGM9qHd5MUwF0c_gwIqd5Q", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113338_21_24e1_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.556Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjFvd0NZakloMGF6Sm1uVGY1N2VvbVBnNmY4eFJHOGRoL2hlMWpZTHhkZ2wvcmRwekVsY2RxUWJjZDNyNXB3bk1JSVFvUVRWVjVsY2o5RkdaMjZvMnlRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTMzMzhfMjFfMjRlMV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODE1OGRiZDkwZWVlZDJhZWI5MzVkNWFhNWU1MTJhMTFiYjQ1NWM2MGQxZjYzZTVjOWZjZjg0Yjk1ZDBjNmIwYTNhMjZhYzQzYmNmZmQwZWM5ZWYxMTA4NjllNGRjNjUzNmQyNzllNmE3OTFiNDI4OGM2NWZiYTE2YzA3MDZhZTg4ODFjYzQ3YTI1ODA3MjYxZTk5M2JjN2VjNzdkZjQ4ZDM3ZGU0MDU2ZjYyNjU1ZDliNDFkOThmNjkwYzE3NTg1NWE3ZDU0ZmRiNmFiMmE1MzY5ZjViM2I4YmVkNmQ1YmQ3ZjQ4ZTc4ZmI1ZGZhZGQ2NGVjOGMwNjdmZjY1YzQ4YWZiODM4ZWVmYjViOGVjYjQxOTMyZmY0YjVlZDgyNGFlZjNjYTA5ZDg3NzhmNWY2NzE5ZGYxMjg3M2E1Mjg3MzNjYTZlMzE4ZmQ4ZmQzYjkxOGFhZDc4MTVjOGY0NWU2YmFmODA2MTZhNzNhY2QwOTVlZjljN2MxMmE2NGMzNTdjODczYmM0ODU0OTJlMzA4NWZkMzlhNTk4M2UzNzI0YzdmNzVhNThmZGNiYzQ1YmJhODBlMTBjNDFjNDAzNWY5N2I5N2E2OWEwOGFiOWU1ZTlhMDg4NjI3NjQ0ZjhlMjE5YTA2OGI0YWY2ZGI5NTMxYjVjNGNlZjNjMzVkMzYxNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.INdc1Sv6hYtn0NMPY_GaLu_k4KBFz5a8T0XMvtvx5-Kh5Nac2shdo6KuIp1ANc4V3NGCMYhpnRUJjoG06bEcbQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113338_21_24e1_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.559Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImZ3V0JrSitrTm1qSE5BSGUyZFdjNlJJMkNSc3BlZVZEMytTSnlnVGU3eE9VTVBuZWF3VEswZElUZ2QwSUg5NlFUVlpDNnFRTlRFV0Y2M3ljTHB5UU9BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMxNl8xMTMzMzhfMjFfMjRlMV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2E2YmE0OWUyNWZjODhiNzBlODQwOWMwYzFjNzZkNjBkNDVjNzY1NzMxNmZlM2JjMGUzNmRmZDM0NDg5YWU0ZGVjMGJkOWZhMDY4NWVlN2VlZjM2ZWMyYzRhNzRiMzBiYTQ1YzI4MDYzNDZkYjZhNDdlMjlmNWZkZjVmMzM0MGM0MWM4NTMzYmU5ZTAyNzlmNjM4NmFhMjAwMjJiYWQyN2ZmODc5OTc5NmYzZTYzODUwMGU0Y2JjNDM4ZDVjMmFhZTA1MWE5NDBiNDMyNGUyYmQzZDE2NTM1YTc1NzQ4ODEyZjA3OGFjZjE2ZDJmOTUzM2I1ZmExZDgzOWRmZDMxY2MwNWMyYjE3NTgwMjMzYzkwZDY5NTgyNjAxOThhN2I0NTVhMjE3MjhiMzNiYjA0NGRkODNiMGNkMWFiZTJhY2M0NmQ3NjliYTQzZTg1YmEyOWIyY2JkNDFiOWI3YjczNTFkNWM5YmUwZDU3YmYyYjFiN2Y2MzBmNmIyZTE5NmM4YzUxYTg5MTc5NzM3NDYyMzE3NTI0NzY5ZmM5YTI1ZThlZjFlMDAwMmE5NjI4ZDEyNmQ1NGQ5MDBhZDY4NTBkMDc2MzM3ZjZmN2I5OGVlZGI0MDYzMTQ2MjY5ZmQ1ODNjNjk3ZTI0NGM4NTEyMmRlMWUwMjAwYzE0ZjgzNjcxNzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MRsRdZ3sySo1CusZTIiuyZu_-IHF6TqXI6VBrakPCtN7-asizNDovOQpLburg9DTxmL1oOqs83XmulCZD4IPdQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240316_113338_21_24e1_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.564Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IktveVFZOHU0RXgyK21BcG9zMGhxQzV2N0tJbnRCODNodkM4YUJVZHFLZnJzUEVRREJqL3cxcTFmZi9QQUhBRldxLy9KSDFiaEhUUlVEZ2lqRVhlNWdRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDQwMjFfMjFfMjRjN19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzA0N2MwYjY3ODk2NDQxNGVkMjA0OWYwMTVlZTUxYTExNTcxMDFkNDJjZTU5ZDk0YzRkZWIxYWQ0ZGMyYmVkOGVmNDQwOTQ0ZWI3MjRmYzQyYmQyYWVmY2QzNTQ3YzZlYjc3ZTY5ZjJlMTNjOTExNTQxOWFlMmFjN2NjZmJmYTdhYjg1OTVmZjZmYWY4NGNlOGM1NjQ3MWQ4Nzc1ZTNmNWVhZWM4YTQ1MGJhOWVmYjA1Mjk0YzkwNjE4MzJiM2ViMmE3ZDUyOWEzNjA1MjgwMWY5MjQ4YzU0MjRlNmRkOGIyZTg3MjI4M2E5MmNlYzlkODgzMjcyOGQ0NTQ4ZmYzOWE2ZDliY2RmYmNmZWI0YjA3NWY1YjIwNzM1ODQwYjdiN2ExNWE0NWM3NTIwOTNlZmNmNzQ0MWMwMTllMzlmNDJjZTJmNWExNTNmMDY3NDczMDc2YzlhNWE4NDZjMzg1NzdiZGMzOTI2ZDRlZjI5NTNhY2RjNjI0ODMzNDY2YTJjZDI2ZTAxMzczNzVjNDY3NWU3NjBlOGY2ZjdkOTI5Yjk2ZDFlMjc3NTkzMGVkZThmNmQ1Y2QyNjY5YWM1YjgyMzgzZGE0YTQ3ZWFjNmI1YTAzNWJiNjkzN2EzZDUzMjViMGIwYmZiNTcxM2Q1YjFhNjViYTI4YTlkOGIyZjMxMGNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Cll8CdcImX0bmfCwWwH1XzIiOI2cVeBBkq57payTOxC_bnvcezS0ek9o2nqT3DAhOS1qtsJyxXjI-AcBiFWxwg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_104021_21_24c7_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.566Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImV4L0RhYTg1UXNNN3p2Q094R1pHOVBQRWhkYzBIQndHclpVTU55M2pLNnZKdTVXV2laYlp1Y1dNNTNxUWpJbFJpSitaaVJic1pleFl4MVcrZVhTVGR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDQwMjFfMjFfMjRjN18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NDhlNzMwNjRjYWY1NmI4ZTUzMmU3YThhYjg1MjRiMmMwODg5M2RiMjRmMmM0NGQ3MTBkMjY5ZGM3OWMzYzc3MGE3OWM2M2E5ODJmYzI2NTJmNjNmYTM5YjNlYmNkNmUyNzI4NmU0YmJiYzgzZGZjZmI3ZWQ3ZmJiMTRkYTcwZWI5MzZlZDZhY2I3Y2U1YzEzYjA1OWFjZjBhZTJkZWNiNGY3NDM3NjNkZDIwY2ZlODQxMjk2MTIyMDY5Nzk2ODU2YzYzYWVkNjdhY2ZjODZmZTg2YWIxNThiMDc3ODIyNTJmZTY3YTFkYjMyOGQxOTU4YWM4MTcxMDc1NzQwOWIxYTlhMmFhZDRlNDA2Nzg4OGYxM2EyYTM3NGRhZDRkZGJkOWZlODNlZmQwOWY5MjU1Y2U2Yzk4N2RhNjBhYTFkNjZhZTA0NjAwOWM4MzYxZDE0ZGIwMGNmYjQwYmViZGY5ZjY4ZTE3ZGFhMGZkNDJhZGQ0MTUyZDQzNzA5OGNiZjgyNmE2ZWYyNDQwZjQ3NTRiMmVlYTBmNDA0MWY5YzJmMWRjNTVjZDZlNzMyOTE2YTEzYzdjZDM4MTFmNmIyYzNkNmJjZTliNTE4MmQwMmIxZGM2MjQzOTVkY2QzZmQ0MWZjMWQ2OWIwYTQzZGE1ZmEzYzEwNjVjNDA3NWU1Y2I0ZGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Fo40TkwOH6cd8QL5yGAYNo6wh4PMPyvMnve_wo1SqbXN_zUGcsiKHPOszYlj8gpsmhU9rDHG8N8I4jNaH9QMHw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_104021_21_24c7_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.570Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6ImhOcEpuVm9EcHo2a3NFOERVaXl2UXdUK0grZHMxQkVSVzJqMXQ5L3JtVzk1anE0YXBxSmFUekwxUU1PTXdpY3lSVFM0a1hhWEJCODFIdlczbTlidVF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDQwMjFfMjFfMjRjN18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjU0ZGU2MDU1NzhlNTFhMWFkZjVjOTIwMmRlMThjNDkwZjg0YzZmZDIxMGRhMmM4OWI1MTlmZmZiMDhkZjk3NTc3NWY0ZjlmZjAyM2EyZTEyZWRlNTA3NzQwNGY5YjViNzc3NmJjZGM1YTRlZjhlZTExMTE5ODRjYjgwMzlmNmU5NTNmY2NmMTY1NTFlMDlmZDVmZWE5MGVlMjM0ODY0MjRmZWIyYWUxN2Q1ZDEwYjIyZWNmZDk2MmMwYmNjOTNiMmIxNjhkNDg4NjhhNGExMDRiNzJjMDU0ZmQ3Zjg2MzllZTRiNWNhM2FjOTQ4M2Y2NDlhNDQzYjNlYWY3M2NhMTgxMWI2MzAyZjEzYWQ5ZjZlOTRlOTQ2OTJlYmQwZWMwNWNmNmVlNGE5MzY2ZmQ1MzZjYzc4YmZjOGU2OTg3NTcyNzBkMzNkOTA0OGIzMDU2MzFjZmM0M2Q3NWQyZTg5MGZlZTM3YTQ4NmIzNjc4NDUzMGQ2NmUwMDJhN2I5MDExMDYwNDBkNThjMjdiZWU1YjcyYmJjMzM5YmUxYjA5Mzc5MmRkMTRjYTJkZTFmNmQ5Mzk3M2JiZWVhNWQxMGYyNjYxODJjMDk2ZDUyMDM5NTk0YjM2MzQ1MjIzYWZkNjU5M2M2ZTc2YTg4MGFhYjI3MDIxNWVjNjdhNmU3YmViZTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.hxVMrKEozTXJHe_ljliW5wygnUbMpBvxnqtwTjVa8wS97-X1qKfkhsloTljGWxeMAd4ExlMLRhEq1ErjpCVmTQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_104021_21_24c7_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.574Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InhoSjl3Wm5BTHlJaEFoRGFveW5qSG9wZjk4RXUvT29SbWpLR3k2M2hWWHBteVd0Q3U0YjhpWFQyNWhnUzBRSWZRUVB6NkhoOXAxdnBRcEE1NGw4Uzh3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDIyNl8xMDQwMjFfMjFfMjRjN18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGQyODRhM2NmMGFkMjk4NzBlYjBlZWMwMTU0YzljODAyMTYxNDY4OWI0ZTRkZTA1MmM4ZWFjODQ2YzU3ZTY1ODFmZjVlZGU4YjhmOWEzNzNlNzEyNjRmOWI5Yzk3Mjc0ZTE2NmM2N2RlODQ1YzUzNzA1ZmU4Zjc2ZDhhMmUxYzQyMWM5ZTU5OGJjZmU1NmFmNTUyNDU0ZTA3NDJiODZkZTc4YjEyM2QyNDZkODNhMDZhZDIzOTg0OGU1YzQ4YTNlNDRlOTE5ODRhZGEyYjk5MDE4ZWU4ODQyYzQxZmRlNWUyOGEzYzJkZjk5MzI3OWM5MTEyYzFmZWY3NmZhMWQzYWQ4MTdiNWI4YmM2NDA3YmEyMzVhNjIxODNlNDRiOTMyZTZhNDVlNTI1ZDMyYmE4Yjk2YzI2MjlhMzI2MGI1NDBmOGZiNGVlYjg2ZjQ0NjgyNGEyN2ZjNWFhMjRhN2Y2Y2NiNzkwMjk5M2QwMzAyNjI5ZTRjN2IwODFjYzc3NWQ0NWRhYjAzOWYwZGU1YzBiY2M0M2Q0Mzg1MzRkYjYyMjE5YjI4ODI3MTZlNGNkMTczYzQxZmNkNjVhMjI2NjYzOGRmOTU4MzRhMTYwY2Y2OWU0N2UwYTM4NDUxYjNiZjliODRjNTQ4N2FhNzUzNWRjMTFkNmNiYmY1YjhhMzg0ZmRcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ali7hKEh3BIWxkHt1twBkqkb5KJ2siyHyyrlVvtahUo2y2IwQsQ6TZCIXkUI3eDV8HE9ClD77octgL-qNcv63w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_104021_21_24c7_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.578Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Ik00RTEyOVY0Y1RkVXRRZnV5eGJ1OFFZRmxJS0dQdHMzRU12c2NBQ2J4WG1LdnZuQlVKSDNITWJwL3JmQkI2cGFMNGp3WU1mZkRZM05nbmxSc04rMlRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDdfNTJfMjRhYV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODgxNWI0ZmY1YTJiZmFmN2VhNjlhZWE0YzJiMDMzZjM1M2RhNjYyOGQzNmQyYmI3OGVlM2Q4ZDhjMTM1ZWVjMzU0MzM4MzQ5MjhjZTY4OTlmODY0MzViYjEwYThmMGRiM2MzNmZhNTIzMDZiZGRlZDQ5MzNkODNmMzdlYzIzNDQxN2Y2OTUzMDZjMWNjMTdjNTMwYmExZTY2MjNkNzRkZWY4ZmM1MDIwNDc1ZWViYTYxNjJmMzY0MTc3NGFjODhlOWE4YTJmYTk5OWQwNGQwZjI0ZGY3ZDdhMjUyMTEzNDY0YTY4OWQzYmJkN2IzZjk4ODFhN2UxYTEwY2NjY2MxMDE0MTI1NmNkYmVkYTgwNzBiNDg2ZjU0N2U2ZmE2MmJhMDgxYjEwOTRjZTdjNTcyZGZmMWZmMjI5MmJkYjdhMTFlNTk3M2U2MDFjOGFhNzY4ZTcwNDJjOWY1NjQ1OWMxODdiOGE5NWE2YWY4ZTg4ZWQxYWRkMGJiNWNiZDFjY2IxODM5ZDE1Yzc1YjY3MGQ1OTIwNzdjMDUwOGMxZTU3YWM3YjU0NDFlODEzNmQ4Mzc4YTc4N2IwOTIwZDhhMjU3OTM1OWFkMzc3OTRiZmJiOTVkMzgyYjk1YjBiYzI5NDhlNjdlOWQ2YmFlYmU4ZWNiYzI5YmQ2NGNiMTVjMmMyY2VcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.dTicDO7UuyYjBrd9oHAOjyI22oJwS7xCXVQPxlnbhlyb579NQsOIsBI160bndUdGbPmwm9AxuJijv1YCI2FlNg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113147_52_24aa_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.582Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkVXMWZ1RHVqRGhYRUxFbkc4TythRVd5dU12Ym96T3VROUorR3lXc1JwWUNOZ0h1VHVxdHFTQjFVd1ZGRXR1WGJsTzVaMVBTcUlTaUEvSkVZM2doV09BPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDdfNTJfMjRhYV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjhiNzJmMTNlZGEyMjQ3YzNlMmM0MGNlOTZlNGNiYzI5ODZlNDQ1NDkwYzM3NGYzN2VmNGMxM2NhN2FmNzRhOTVjOTMzMjg1MzM5ODYwZTgzYjQ2YTJhMjJiNWRjYThkM2U5YzBiMTM2MzIwZTcyOTk4ZTZmZDUyNjUxOWE1NmUzMWMwYTg5NGMwZmQzNTcxYzY2YTkxODY4NjIwYmViZDEwODU1N2YwYWYyZDkyMjVjYWMzYWFkMjkzYjkzNGMwYzczYWExMGJiOGE1M2E4Y2YzNTlhNzg0ZWFiZmIzZjgyNWRhNWM5NGEyNjRmYTM1Y2FlN2FlZGE1ZDIyYzBkZTE4ZWRjYmU1MThmYWFhYTE0NjBiMDFjMGVjOTI5YjQwMjgyZjg1NGE5ZmM0ZTQ1MjE5NWVhZjlkYzliZGRmZjViNGEyM2NjODA0ODkwMTIyNjMwZDAxMzcwNGJlOGMxN2E5NjljY2MzZDU1NmViMmFhZmEzZjJjNWZhZjI5ZWE4OTkzN2MyNTVmNDBmMWM0NDBiMzczMWYwMzY4YTZhZGQzYmI1Y2IwNjQ1NzVkZGI4OTRkOTlhYjJjMmQwZjNmZTQ2Y2UzY2VlYjNkYzA5YjNlYzRjZDRhZDYxZjg5NjllOWJkODg3NDE2MTM0M2Y1NGE2OWY1MmQxOTA3ZmVhODBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rHlOro_Ku3clFhV3O_cXtDPMIMjnEohg4N1abdoESHcTnbL27gSf2jSQee4aO3YxzZE1yZh5zaOWt4OXsSbDZg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113147_52_24aa_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.584Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlpmMkRMNHpCZ2xIcnFIRG5Qak12Nk50SkdsTzl3eWVuV3BvWFhOZkN1UG1RTUlVd2ZpMDBmalNJNi93YmRSVUtDMmJJWXFCNkp4YmdtTGlUblRoS2ZnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDdfNTJfMjRhYV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NWJhNjFjMzZiZjkwN2MwNjAxMmZjMzdjYzQ0ZTZkYTZlYTUzNTFlZDBlMWZkMzEzNDBiN2RiNjc0YjNkMGJjZjIzMDM3NmNjNmFiZTQzMGFmZTVhYTJjODkzODM4NGRmMGVhMGU5YzEwZmQwNzJkNDYzZTNjYzU5YzM5ZTBkOWMzMjdhMjBkMmNjZjBhMDI0ZTJhYzdkMWQ1OGEzOTA3MjgyNmMxZDMxYThlNDNiMTFlZWZlNjU0YzRhYmJlMTM4NGYzYzMxMTFmMGQwYTQ5ZDczYTdmYTM5NDk0YmU2NTI0ZjVjN2NlZTg1NGQ3Y2VhOGM2NjUwZTg1NDA4YmM5ZWUyMTYwM2Q5YjlkNjliODEyOTBhYmI2ZTBkOGE4YTZlMzVjYzgwNDFlZjQ5ZDk3YzFjYTRlYWFkN2QzYTY1ZThlOTliZTNkZmQzNzI1YzI1MWY0YzcwMjYxZmEyYzBkZTE4MmUxZWY1ZDdjNjY2NzRhNDY4NzM3M2Y2MGQwM2E0YjlmZmFkNThiM2Y0MjJlNjk5NWQ2YzU2NTA1ZDExMTFlMTNmNGFkOTBlYmIzNGVkMjY3OWE1NmFlZDk2NmJhNDUwZDg4NTNiZmFmOTkyODk1MTcyOTg3NzU0NGJlNzJmMWRmZWYxZTkzZTlhNmNhNWZiOTBkNTk5MmM0ZDM3MmFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gvcbqh-LCN_g3LGvtum4HaoOVSU3UrzFtJtVZtKb-MTVT7ZWaDKDNpjJAI0-RvaQkrmsO5lhTTi86q-7DxTbBw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113147_52_24aa_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.587Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Iit2cDREVVAydDVSRjZodm1VTWJDZGVkRzltSzJuYUowQ01zMyt6K2xCTjYvQ1dCM3ZSc3V3RmJPUEx4OXBQSUtvT2VIUUdCbUxLOGI4Z2tNTG1WcmpnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDExM18xMTMxNDdfNTJfMjRhYV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MGFkYWUwNWVlMzVhNjAyNjZiMzkyMDBlYmM3M2Y4MzU3ZTc3NGRmNWFlNzc0ZWQ3NWJmZDY1NGZhZmFmNGIwMjdiNTA2NWNiYmYzMDhkOTg4MzY5OGU4YTc5NjgwMTZlMmIwZmU1ZGUzZmNmZWI0YTdhMDk3YWQ0NzQ3MThjMGE0NDc0YTVlOTVlNTY1MjRlY2MyY2EwNWE5ZDY5OGY1MTExYTYwYTdiYWI2ZGM5NGUyM2VjMTY1ODMyMDcwZDNjMzIxNjQxMmM1NzJjYjIwMDY3YWZiNmQzODA0ZmU5NjA1NWIwNGY3ZTE4YzY0YmFhYjdiNjQ5Njg2OWU4NzRlMTNjMDEzZGFlNjlmMDIxMjFhOTQyMmFmZjYxMjZlODQwMTczYWRjZjdhNWRkNGEwOTkxOTMyZmVhMDlhNzJkYzJhODY0ODZiYWMwODljYzI2NjRiZjRmZGEzOWU4ZGNmNDU2OTVkMGZiMDg1ZjBiMjdlOGVmNzM0OGE4ZGM2YzI5YjFkZjYwNzcyOTRlNjkxOTVjZGYxMmIxNmVkOTU5MTk1OTAzZTdjYTg5MzY4YzE0YzlmNTk1NDI4YmI1NGM2NGNmMzlmNWYzNjhlZTVjOGM5MzBjODVjYjFjY2FmYzYyOWE2MWU3ZTc3ZDg5ODQ2Zjg1YmQ3MDBkMGJkOTdmYTVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.B2GV4E-ZOcNSiZZ3kKql903AcG_TjUMCkfXkvMLvjjKQ3A-_AzCoR7Uv74Brx203IIQshtCsmmWABKMu1ZfwfQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_113147_52_24aa_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.590Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IldDODhxT3lCOHhwRmVyZ1hiNWE1T0Y5VEdVaXBtWWJudFM3TGFoQUhFaWlLMWRRbDZFOW1KNUxQOWxLUXp2QU1iQTBld3RDaVBmalVvRDNPWXRCd3dnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzJfODJfMjI2Ml9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjEwOGNmNzkxYmFhMDI4NzlkZjlhY2E0MGMwYTJkMDVhNWIzYzY0OWI1MWQ5ZDU1NzFjZWQxZDU0NGZkZGIzNTEzYzY1ZmIxYWQ2YTJkZDMwYzBlNDY0NzAyN2RmZDhlMDMyZjA2YTYxMjg0MDY2NTc2ZjI2OGI3Njg4ZWM2ZGYyMTZjZjk3YmU3YmY2ZGY1NDY4NzY4OTkyM2RhMjg5NzA2YjIwMjkyNjg1MDRiZTdlZTg2ZDI3OWMwYzkzNmE3YjBhODY4ODQ0MjQ1OWUxNmE3MGVlNzY2ZGFkYmI0ODI5NjBhYzU4Zjk3MjNkM2M2ZjU4ZjQwZDk5Y2UwNzRlNjVkY2NiMjdmYjBmNWNhNjI0NDI0OWU1ZmI1NTgwMDhiMDFiMzQxZDk5NWRlNTJiNGYzOTIyZjMwNzljYzE1ODliZWMyMzA0Y2VkYzE5OTEzMTAwYjIzN2NhNjI4MjM1MzdkYTNhMzFmZGVlODQzYjU3YTkwM2YwY2YzM2YwNDJlYzg5MjRkYjAxNDUyM2E4NWY1MzgyODhkNTEwN2FjYWExMWVjMTg0NTI0ZGIxZjllMzQwNzU0ODM5ZTAwYmE4N2Y4MWM2MDMxODNlZmQ1NDU0ODkwYTMxZjdhMTlmYjY3YmRkNzk1M2RlZGYwNjZjZmFlMzJiMmYxYzNlZDBkODdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Czd0B2nXd_MZuzAvGcasCCdCzat_UUiPX5skYCPNB8NcbD-X877iLxh6tWTFki8y-knuPjidiNwhAUmxJzFqew", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105032_82_2262_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.594Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFDcDdTTzlkY0g1MnhvdjdmeHdhelJtRjNsMWxSZjUwbzZMVEhNRGpqSmI2VUVrRkhpaWNBQ2duVEduYTlEOUpkeFJwdDdQanBFZ1dVYWd1bzEyRkZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzJfODJfMjI2Ml8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTE1M2I4MzE0NTE0ODA1MmNhMTE1ODUyYzQ1MGQ0YjlhNDFlZDk2Y2NkZjZhMTFhNTFiNjg2ODE3MWFiZjA1ZmQzNDUzYjg5NmRkMjFmNmQxOWJmYjJkMzBmNjA4MjY4OTI5Mzg5YWJmOGNlMThiY2JjMWU0NzJhZTkyN2VkNTA4MzFhY2VmZDYxNjM1ZGFlNTMzYjJiYWYzZWYxZDk3NWViYWNlYjRlYzVmMGI4ZWVjZmRkMzA4ZmU4NmE4OTQ3MjYxZTJjMGM5MWZlNjYxODFiNTY1YTVhOGUwOTYyZTNmYTY2YzdhMmZmMWQ0MTBjMzkyNWY4MTIwNzNjOTFmMmM4YTM5NjYxZTE0MzllODNjZjUxZWQ4NjM0ZDEzMDA2MDVjOWFkZjViMzEwNjlkY2MwZDU1ZDQ0M2JhNTMzODgzMWIyY2YzMmQ4MjMzZmE2ZjgwYmM5MzM4YmQyZDQxMTlmZWNhODVjYTk5ZDk3MWY1MzFjNzE2OGZjOWY0MTNkZTYwODEyZThiMTIyMjBlYmJkM2YzYmRjNjk3MmYwZjQyOTFjODE2ZWE4NWE2ZDhiOTgxMDMzMDIyOTRiMDNjMGJiZTE0NDUyY2JmNTYzNTRmMzRiMmViZGIyNWE4OTZlMzI3ZWZlOWJiMmQ3YmRlYWFiNDQ2OTIxOTI3OTljMjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.LUVJwhZYXv3YUiu1EIi3WheFOAafirHwrmdY2WhKOxnbqcJSo2LzpJjcfXsX3bD5-VaQBAzLRQ7bzHX7vrUTVw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105032_82_2262_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.597Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpCME9EVlF5VE05RmpUbnVRWlB4aEtpOUl6TFVFZGZ0dlRVZ1dmMVlHc1Y2Znl6VVZscnE4YXI2dE82T1ZpYzI4WWl5VGM3MDV1aDd5bFY3TGdyWGV3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzJfODJfMjI2Ml8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Nzc1ZDQ0NmJlZjY0NmVlMGVkODI3YWU4NDk3NWZmY2U1YzQ3ODE3NGU0NDI5MzFiNzc1YjEzOWFjNjE5NzNlNGQ5OGI1OThkMjBmMDdiMzhjNmUzZDhjZGQzODM5ZjhkYWI2YWRkNjlkZGM5MTg5YTM5MTM0YzdjZTE1ODJlY2E2Yzg0MjhmZGE3N2QxYjRjZGQ5YmZjNzZjNjE1MDg0ZGMzYTU3NGI4Y2EwMzllYTZlOTgxODc2MjhmNjhjYjFlMTFkMDExOGI1NWMwNjkyNzI3YTY1MjA2NjFlYzQzNWViZDVhMGYyZGQyMzM1Y2EzMmRkYmFkM2I0ODc2NDc2ZTY4M2E0ZjM4OWZmY2Q2NmYyN2RlM2Q2MjVmOTUxNWZhN2EwYzhiODdmZjQ2ZjI1ODY5MDc0MDgwOWQ5ZTAxZjhkYzQ2MzkyMGMyNWJjMDZhZWZlNzgyZTc0ZGIyNjEwY2MyM2NjN2Q5ZmFjOGQyMDQ3ZjNkZmIwNjMzY2I0ZDhiNzIwNmQyZjFlMTQ5M2U2ZDM4YmUxN2U2ZDZkZDQzNDNhZTc3ODI5MjM5NmE1MWQwYTEyNTBkMzU2NTNmYWYyMGM0ZWIwOGRiYzdjYjFmZGNhZTg2NDIyOTc4OTI1YTZhYzg4ODI1ZTI5ODQ3YmI3Njg3OWFkNDdhOWUwNTdhMzFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cCKHFcLB5APbaFOGoKMwnabwBGd6rozuwTrEGTwWfQurxIWGITJqgFfc8jFuYLw5uHIUpLcaNLj-u4iK6TDQhQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105032_82_2262_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.600Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IllIMURWWGF1L2xBZzd3R3JQeDc3QllKT0ZZUGNkVGsrVXZhQ0lyNlhhYXhRSld5ejd6eHBpMUU1VlhjRWFITkYvd3Qxd1RnUHpoMXhOSytCNEFabmR3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDQwM18xMDUwMzJfODJfMjI2Ml8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NmYxZTdkZDk4MDAxNTUxYmMyZDQ4YzNjOWNiNDQ4MmYzYThiZDMxYWViZTY4NDZiNzkzMWQxYzkxZGY2NGVjZTA5ZmQ5NmU2YzZhYjk1NzA1NGE3MjUzZjBlYjI2YTc4MDVhMTZlY2JjMjA3ZGI2MmI0OTI3ZTEwMGVkMDA2YTFlZWMxMjRiMDFlODY2MjgyODNjMzQ1NTgyZmJhOTMxZDVjMWRlOTgxOTZhMjg2MmRmMTE1M2Y3ZWQ5YmMwNTJlMjgzNWYyNGNhNGNhNWJhMzZmYzUwNzkxZmZlNzRiZTE5ODdjOTY4MDdhYjRlMzA1ODZkODEyMjQyYjg5Y2JiZTUxOGY5N2YyZDFjYjFlYTQ1OWIzZGQxYjFlMGRjNTM0Nzk2MGRmYTBmNjBjMGJhZWJjZWNiMDJlOThkZjFhNTNjYzNlZDYxNzZkMDU1ZmQwNGY5YWU0N2I5ODRhZjAwMmJjYjBlYTQyMDIyNDZhMGU1MGM2MzMwYjE5OWFjZmQwY2MwZGE5ZGFkMzk4NTYzNzIwZGUxZjg5ODM3YTY2NjhhMWY1YzZhN2JhZGE1MDM0ZjcyZTc2ZWNlZmIyYzFiOWM1ZmYxNGQxNjQ3ZTFhMWVjODZhNjBjYzFhNjY5MDE3OTZkODUyOTA5YTNjYmIwOGNjOTcyMGVjMGQwY2JkZTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ccu-2dgriE50TAARKXnceFc53JYjnJcfOVCHbSRcqwM6x9kM4fyRG8jWyBwpn018wGRh8hOBmMkYXB4YbeX8VQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230403_105032_82_2262_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.603Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IldzY1M5MytzYlhqS25uUG9ONkhrR2FNUDEyYmFYc1lxaWkwNG1LUStZLzNnczQ2RnpKNTZLNU5xS2lIa25EZ3FSWFJwc3l3SVBXd0Yrd0tzRENua3lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMjFfMThfMjQxM19tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODYwNzEyZGZkNmRmYmI4ZDBkNmFiYTdiNmUwNzJjN2YyN2E1ZWZlM2JmZDc2Y2VkZmU5OGRmODdjMDcxYWExNjFjYTE3MzFjOThmMGIzMTA0OThhM2Q4YjkxN2FkOTZhMzI4YWE0ZDhmYTQ3Yjg4NDUxMDVkZDQyNzc5NDI0ZWQ4OWMxOGU4NDg2OWYxYmFkMGQyYzM4YTZiYTdhZGUwYjE3MmRlZjZhNmU4YjQ4MTYyMDNhMTUyZjdlNWFjYzY0NmIyZGU3ZjQwMWQ5N2VhOThhNWNiNTEwNTIyYzhjNzBkNzIxODk2YmEzMGMzODc2YWE0NDA0YmUwNDY5YmVkOWZkYmZmOTY4MjBiYzE2YjJhZTc0NTRhNWNkNjM2M2FlNDY0NWViNzliOWU1YTZlNTI3OTAxODNiNzQ5MzRlMjUwZmU1YTg4Mjc0OGY1MjNhMjYzM2ZlODU2ZTc2YWEwNmQzYjUyYzgzOWQwNDc3NDhiNTczZTEwNGRiZWU2OWMzNGZmNGZmZjdkZjJmZGQzYWIyZmI2MGQ1OWU2OTVjZGYzMThmY2Y3YWY5ZWQ4NzJkMzAzNjI3MGFhM2NjM2I3ZTI1M2E2NzAwNWZkNjJjZDE0OGRmMTVjMTczZTY0YWY2ZTRkMDQyYzY2NWM3YmI3OTFiYzU1ODQ5ZTYyMmQ1YjJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.CKAN_g7ZrMSk6Q9qHGrn_dFB3CC-SdyPu0LOhCGA_QOTBqGcfyNT3oM0QPZjwhBh1kX2BInqk2BtTW7WrJ7MQg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111321_18_2413_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.606Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlJsUmtjV2IyTHlpT0ZYazJGME11OEVFcG00aFUwVXdyYVlPOWg3MlJXMDVLblltUGZDcjFFUDZuczRNSDFmK085OGhaWmc5YnlockJVdGsydGJkYzhBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMjFfMThfMjQxM18zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTRmYzU0YTkzODY3ZmU3OWFkMmE0ZTRmYWZhMTkxYmFkZjY3NjkzMDUzYWQ2NDJlYWFjNTg5NzBmZDRmYmZjOTdlYzkxNzJjZDA1NDNmMDFkNzE0OGZjNmEyMDBlNWY5YzUyNmQzOTlkZTdjY2U5ZWQ0OGUwYzUxOWRiYzYzZjQwMjZkZDBjNTNlZDU5ODkxOTFkYWU3ZmRhODgxMjM2MTYxMzk4NmJjYmI4NDliMTFhNzQwYWEyNDBkMGNhYmUxMDFjNmZmZGE3NWNhNTE0ZTA1ZjAxY2EwNmYxYzUyYWI5NzMyY2M5NzUwOWZjMDg5ZmE2OWE5MTY2YzUwYzc1YjQwYzcwYjQ4YThmY2FjMWVmMjYwNTI5MTZhNDJiZjQxOTU0ZGRlODkzNDM0OTYwZDk0M2Y5MTExYmY5NWM5MmRlZTRhMTEyMmIxNTY0NGIxOWY5MjlmZTkwYTQ4ZTU0YzU1ZmZlYTRkOGUwYzJlZmQzNGI4ZDRiODkyZWEyOGQ2NmUzZGMwOGNlOTg0ZTk2MDBhZGZlZTM1MjQxMDBkY2RmYWEwOGM0ZmMzYTYwZmM1YzY0MjM4NGY0N2EyODdiNTE3MzNhYWVhNGFmNWVmYzA3ZjY2MDE0Yzg5ZDNkZjZiYzA5OTAxZjNhMzc1ODU3MGY0MDliYjFjMjNjNGU4MThcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.es7qmkxtyTTui4wHKu0ubmHmQ2p0cvz8cAGFj-58y6WltnGeG4wDlwTojLxgQwh-cc-aO5w3bEPhSB_aENouhA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111321_18_2413_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.609Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFVR1dhL1g0d3Z3WUtaRS9rTGx1YlMrNTVHMkZQdWs2c2JET3Z2TDBEbEVabjZXcUM5WDNXTXRzMGV0dUgrU0ZlaE9HWlpKY0RBT2Y3RG8wNjNoNUp3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMjFfMThfMjQxM18zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2ZjNmUxMDgyNjMxYWU0NmQzZjcwMmVlN2VkODBjNDI2OTI0OTE5ZDFiMmY1ZTZjZjRmMzY1NmJlNzU5MWM2MGZjZTE3MGVjYzRjYTE2OTk4NmMyNmQ2Y2VhZTIyZjM5ZDMyZDI5MGJkNzRlODZkYzVkYjM5YTViY2Y5Yjg2ODBjMDcyZDYzM2M5OTMwOTFkYjQwMGIxNGU3NzkxZjRiNjgwNWI5MjAzMjhlYWU5MTVmYmVmNWVhZmVlMDNhMTVhYTJiYWFmNjBlZjcxNWI4NjFkNTNiZTM1ZDAxYzYzMWNmMjNkY2RkMTAyNDljNWM5MGIzYWRiODJlYTNlNjhiMmRmYjhjYmNkMGZlMDk3YmU0ZWVlMDgwOWZiOWNkMDA0ZjlkYTM0ZGZhMDk4OTdiMWM1OWU1YWQwZjhjNDQyOTFiMWNmZjE3MTY0NmM0MjJhZTUwYjdkN2FlN2UxMzljY2I3ZTNiZTRjMjVjZmY4NDg0NzY5ZGIyYTFjZmQwMDI1ZGQwZGUxNThlM2Q5MTEwOTk2NzhlYjNlNTFlNWM1NmQ1ZDA4NmJjZTdkZWEyYmU4NDU4ZGJmYWY4NWVjNzNkNjE1ZjFkMTE1YjU1MWJjNzYxYjEzNmJmZTU4MWU5ZGJmZjQxNzhjMGRhOGI4NDU1MDAxNGYwOTUwNTk2MGNkNmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.jV0c9BqeZXSCY4QC76PHPH08ohtZjS4OhN-KKRN8Jsm9nV1DcdsEgrnBLl6lUIgUevpGKeCPpF7lKf4OFhiP7A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111321_18_2413_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.612Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjlqZUxVQWcranYvKzNOS0NHamxyUGY5UmRIMXNaVktBN0k4bjgvSnpSQ2ZUTXE3U0Y1NlUxTVhhUDV6OW04YW5OWTZrNDQrdHZsOW1TTG9RVmcrSnJ3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMTEyNV8xMTEzMjFfMThfMjQxM18zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NTcwMzdmMzA4NTljNGRjNjhjMjc1MWQxMmQ3YWUyYTUwMjkzMmM1YWFjNmUzZDJkNzEwOGIyODFmNjJjYmVjYzQ3ZDFmYmY5ZmQzMDdhZDAwOTZjMDQ4ZTkyZTljYTM3ZmRlNzAzYjkyYzc4Yzk0ZmI1NDJkZGZjYTdiNzdjZDYyZTU2ZGFjMjM0OGRmMmY4Nzk5NjQ0YWQwZTljNGM4MWYwMDczNzM2MGJiYmJjMjY5NDJhNTE3NGZmYWU0MTk4NDA3YjFhNzgzMWYyMzU1NTQxNjUzMWJmZTY1YTBlMDZhNmQzMGNiNTE5YWU5YmZhZTFhMGEwNjMxZDQ4ODFlMTUwMWY3NThiMDg5MTZjOWE0OTU5OWEyNzdkNWJmYjIzOTQwZjg1NjI3MWZhOGU4ZDI2ZDQzNmQzNjAxZTZhYjE0MjdiYjE4YTk0OGY5ZGVlYjk0ZTFjYjY3OWU3MmYxZWNiODAyMjU1MWZlMzM4ZmEyYmU0MDM2NjFlNTY1NWMzOThiOTQ2N2FlZWUyNzU2YmIwNGFlMmI0Yjg5NGE1NDM0YzkyYzA3NTVhMDY2ODBiMmRjYjcyMjE5NDcyNDhiZmU2YTdiNjJhZjBiYjViOGNmMTlkZmFiNDE5OGFmYzJjMDU0YjdmYTc3NGE1MDM1ZWJlOWQzNGZlZWJhOGE1MTNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.ruxi_MlCd8hd-FS7bS8_hTCJIyFmEiRrgElga6fTBf-CLb-9KAUNxCS1MdPNMnIQE2iJfGWszUkNxT45kaaB8w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221125_111321_18_2413_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.615Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InIrMGFkN21kZ3FPSnlxYU9iWmVjWXQzUzNoM1BPOGlmN1ExaFhibjhoc1RLay9sdDhlanFaV3JTb21OVUNLT0ZuWWJDQk9NcW5GN0R5QURYOVY2MDVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNV8xMTAzMDRfNDlfMjRhNV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9N2FlMDcyOGMyZDQ4Zjg2OWFmZmQ0ZDA3YjZlMmY0YWUxNTk2Y2ZjYTBiODM4ODVmZTJiY2MyNDIyNTA2MzZiMDdlNmExMzNjMWI1Y2ZhYjI5OWE2NjVlM2JiOWVmMzEwZDU0MzY4MTFiZTNhN2U2NzAwNjQyY2Q0ZjQ4ODAyOWQzNTJmYzg4NzJjZWMyMDY3ODEwMTkzZmQwNjNiOThhY2Q5NDc3NjE1MDg4NjljMDJkZjBjOWI1MWU1YjIwZTU3MmU1Mzc0MDAxMjgwYmQ1YTNmZmZiYjViYTMzM2E3ZTdiNjc0OGJkZTQ0ZTM2ZjkzYjU5ODIyYWU3ZWRkNTEwNjQ3ZmRiNTgzZWU0MzJmZGZkNmRkNDE1YzBjNTgyMzQyNTk3NDUzNmFlMWEyNGFkOWI4OTZmNGQ4YWM5MDgwYWRhNmYyMzU5ZTk5MzI3NzA0OTg1YzA2NjUyMGExOThmMDU1YzNkNjNlNGIyNzIzYzlkZGRiNjMwMGMyY2U1MWE2Mzg0MjNlNjAxMWM3OWU4OTI1Zjg3YjQ3MmY0NTZlNGExMjM0YTQzNWRmYTlmMDVjNWMxZWZkNmI1OTMwNDc1MGY4NDljZWI1Y2M3OGZkMGQyMDQwMTYxZmExZjVhYmRlMjUwNGE4MmYyYzkyMTk2ZGVmY2NjMjkwOGYwMzFiNTZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.b1yW-BKoBTLb6JGmY5quxOSzP3tCNrd9AJhEbVeBt2wJTW3KV8Urp4P3L615uH-lVpjIqAj7JKB8lZ9wPprCVg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220915_110304_49_24a5_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.617Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InZzWEh4TWZLQjl3QWNuS0dnWVd6M3dtbDkza0hkSlV5dHlGTU1oUWswWHJvRjhlOVBFR1RCS3J6ZEsxNkdiYmpsUCtyUWFpUFVlREFDOHNaWXFRYmRnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNV8xMTAzMDRfNDlfMjRhNV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDYwNWVkNjc1ZDA2MGQ2MDIwMWU1YzZiZTIxZGM4MWU5MDA5NDFmNzc1Mjk3NzFiYzBmMzZhNjJhYWZjOGRhZWJjN2FiYWM4MTVmMzcxMGE2OWVlMjU1NzhmOGJlYWExZWQ1MDBjMWM5MTA2OGU2Yjg0MzVmYTczNTE0ODBjZDYwMmFmZGViYTcyYmM2NmQ0ODg1YWFlODA4NDllNGI4Yjk4ZjQyNGU5ZmRiODZmMTM0MjExMGQyYTMxYzExMmE2ZGFjYmYwYmUyMWI3NzMzMzBhZTAwZGJlNTYzMzg1Y2NmNTZmYmRhZjM4ZDYyMDM3MjQ5YzgzMTZhYjcyMzFkOTc1ZDU1MzQ2MjQzYjgzNTgzNjdiYmI1YzQzZjU1MDFkYTIxNGQxYTM2ZjE5YzI4NGFhMjAwNDdkNjNkNGFiNDIwY2ExYzBkZTJlOWYzMzBhNzVjNzhhZTRkNTdiYzU5N2Q2MGQ2ZWI1YTM4ZWZiZjgzNDc5ZGNmYTdiOThhMjlkMzdkOGY2ZmE4ZGZhOTJiNGJlMDZkZTIyNDEwZWU1YzJlYWM2YWE1ZjdkYWE2MjViNTk3ZTA2ZjQ5NTc4ZTIyZTNmZTk0NjA4OWFmM2FlY2IwY2ZjMWUzZDQ0MjJjOTVjNjdjMGE5OWMzMjhjZjFjNTk0ZWE0YWEyOGUyYzgwY2NcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.COwoUvlZjCTEwYvAYsCUPZQQZtGUT6qAxPAqEBt4U8g4ARlvUkPH9ExvUVSfFSVYAAGx071HvYzSzCqxXmT2bQ", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220915_110304_49_24a5_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.621Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InVBMitraVZieTVqN1BDVEJzbjRUdTNSdEZqQ3FCZ0JXRFlqbjI3YnlueHZzd0NibzJFU0dkODdaS3Z3cnpWSHlMMXI3VXU1eCsrcHNZcjQ2ZUlwL0dBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNV8xMTAzMDRfNDlfMjRhNV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGE5OGE4MGNjMWZlNTZmNDYzMzZlZDUwMmMwZTY3YWUwMjI3Zjk0ZWM0OGE2ZDgwYTU3ZDlhNmZiNzFiNWI1YmIyNmU3MTM4MDcwZTAyNzNmZTkyYzQ5ZDQ5MjUwZTFjNTQ0MjUwMWI3NjQ1MmI0N2NkYmZkOTQ1MWJjYThlNDM0Nzc4NjY1OGI1OWJkMmMwZmM2MTYyOWU2MDAzNzA3YzE0ODY1ZjNkYmVhYTZlZmMwOTBhZWZjNGFkMTVhY2U3ZDg0YTg5N2Q1ODhmOGFkOGMwM2ZkNGI5MzEyYzNjOTJlMjFkZWU5MDM1ZjFmYzQwOTIyYjQ2ZTAxNDE3MzE5NGMyMDljMTk1N2FmNDdlNzM4ZWRjZjA2NGViOTQxMTFlNmRlNmRiOWZhYjRmYWI4ZTcyM2IyODYxODZiMDBmYmVmZjAwZDBjMjZkMWYyNmFlZjRiOTZjMGM1ZWUxYTUxNjVmZDlkY2FlMDNjNjZiODNlMjkzMjRjMjUzNTU1ZmI5MTQ0MDM3YjlkOGEyMzQ2MDI3YmJkYTliNzY0NTM1YWNiNTZkZDA5NjJjYThjNTQ5MjY3ZjExZjAxOTQ5OTY3ZDgzNmE5YWFmMWM0ZjQzMDVhNmM2YWM1NmVmMzIwMGZmZGUyMWVlYWYzZmRiMmM2YjgzMTcxYWJkMGU4YTliYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.qNUc1ktG0GjXazjI6BQJExbuH9oL9ros2-SreUl18yXnpmK-r46AoJm45H8fJoPuSWvIIH17XY5xWC_SFO9TGA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220915_110304_49_24a5_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.624Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlZzVzdMSVFGV3ZZeE9HaW93cXlpYlZ1V1FLVVhWOTUyUXZVK2JYVmhDbTBrcEFjbnFtSHA2Ny92TUJQU2xjTm4rbG9XZEpyMy9FalhaeTJnSlIzMlVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkxNV8xMTAzMDRfNDlfMjRhNV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9M2Q2MDk4ZGUxMjQ4MDE1NjZlMzcwOTFjZjVlNTc1MDA4MmMyYTY2NzdjMDY3OTA1NmU0MDRhM2EzOTEyMDQxOTJkOTAwY2M5NjIyZmZiZWQyZmRkMDRkZWI4ZWQ0NjQ4MTQ3NzY5MGQ5Y2Q2MTVlYTUxNmJjOGI5M2IwYjZjYzVmNTA1NmEzYjRkZjE5ZjhjZWZjNTczZjY5NzUzZTllZjBkMWM0ZjYwYzJmMjc1ODQ5YTdlMTdlMDc0MWVmOTdkZmQwOTE2MDc2ZTg1Y2ZlODcxOWM0OTc1M2NmYzRhYzNjYTgyMjk4M2U4NzBlNDk2OTQ5OWY3ODY2M2Y0Yzc0MzY3M2ZmODJmODA1Y2JiYzQzNzNlOGZkNDg1NzEzNDU5OTUzYmQ4YzJjY2IzMDVlZGVmNzNiNjYwM2Q4NDY1NDhlNTU4N2NhZGYzMzNlNDk1ZjFmNjQ1NzdjYzk0ZTg2ZGRjOWU4ZTgxOGY0OWQ4Yjk1N2ZmNDUxOWQ2YTAwMTE4OWQ4OGFhM2FmOTU3NjEyNWMyNDVkOTVjYmY5MzczYTUzYzk2OWI0OWUzYjBiZjM1NmU0Mzk2ZWY4OTlkYjNjYTA1Zjk3OGIzM2E3NWUxZDEzOThiYjQ2MTljNTc0OWZlMTgxN2M0MDQzODc4NzBmNmUwYTI2MGQ3YWI4MDUyMTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.trspQfqHdY8rICdOHMhzLjSJBbKhF-avpAUz8o5NM5ExuOuczhAn4vPwdB-6rnCq4evYKe6wuUPW97T4J89Gdg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220915_110304_49_24a5_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.628Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlN0YTNodXVvYkpxOE9RMkhDRG1LZE1yU2lhY2FCa21kelBPVFBaU2pTN0N4aFNNelFSZ1JmUW1PZkkybG9ONC9UMW8zbGM5N1QzNmZ2dWlZa3ZXYTlBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDZfMzRfMjQyYl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTBhNzViOWJhNDdjOTJkNGZkOTY1MDU0NDI5NDcxYmRjMTgxOGM3MTAyY2I0NDVmYTYyNDFhZjY1NDBhOTQyMDUxODFlZGFiNjAzY2NmZGNmZjg1Njc0YjI3NWJlYzc0ZjUxNTZmZDI2NzhkYTA4NmJmNmMwMzg3YzNlZGFkNzBhYmU5M2MzNmY5ZTI2MTU3MmVkNjlhMWE3NDI4ZjUxOWY3MDUyMTAzNmZlOGY0ZThiNTM5ZTY3OTU3ZjczYzkwZWYwMWZlZDEyYmIxODNkMzUwNWQ1MmNhYTQ5MWJhMDlhY2M2NTgyMjExNTJjNDZhNWI5YzViMTg5NTFlMTNhMzdiOWRiNzYwZThlNTU2OWQ0YmU2ZTNjNWMzNjI1M2Q1ZDBhMjZlYjY1OTdjMDAwNWNjODIzMTVlNzQ2NDc4MTkzZmM2MTdkZDEzYTA0ZmEzZWFlNjliZTYyY2VmMjhjYzZjMGMwNjg3Nzk3YjRmOTY0N2E1ZWVkMTM4NDhiOTRlNGU4ZmIzMzU3YmU2YzAxMzk0ODJkMDlmYjNiN2ZjMzQwZGVjOWExYmMxYTQ1YTIzMzcwY2JjODk0NWQzM2UwNjY3ODZkYWM3M2E0NjBlYmU4YzZmMjVhZjUzNTljZWRmNDE1MzdlNGM5YmVmNTNmZTAzZWYzYzc3YTI4OWU3OTdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.atzvVj0LcOZ-znApm-9yR0xZB4Bf_ReP8lyXYOv6Ka3Gv1jeSSQy6pGGXp5wQxwt2864KKEDCdepV2g2okz7ww", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.631Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InFTQXlnVmdGSUcwaGlSR1pvcGYwTzUwNk8zOGxia2dHMXFUV21BUEdjSjFHRnZBMFUzcm9zUmxIeXowK3Btek1UWGl1ZjNJMVVFVVdRa3VlRUhuZnhnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDZfMzRfMjQyYl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTdmYTIyM2VhZDIyYmI1OWRlNzA1NTI5MWMzMTAxZGYxNWRkZmI3ZmZlYjNjNjFiNjk2MzI1NTMyYTgwMjA1YzgyMTRmYjhjNjJkNzhmYTBjY2NmMGZkNzc4Y2ZmOTRjNjNlMzQ5YjUyZjk3M2QxNjIzY2NkYWQ5ZGQwNDNjZWM2NzkwNzY2YWI2YTA4NDdkMjkyZjFmZmJiMDQ0ZTZlYTlkMjdjMzQyY2Y1Y2Q1OTAzYWRlMmZkMDFkNDM1NDUxYmFiNTMyNGE2MWQ2MjE3ZjdiNzYwMWQ4MjNiOTE2YmM0ZDU2MTE2Y2VkYTk1ZWQxOGFlYjhjNWYxOWQ2NDAwYmUyZGFjZGQ1YzZmY2M5OTk4OGYxZDU3OTQ2YzJkYWRmZDVkMzdkMjk1NGU2MThkYjQ0MDI1ZTlkMmVhNWMwY2E1MDJlNzVkMzUwM2QzM2E4NDg3YTc5ZTVhZTAwNTM0ZTVmNmRjZTU2MjYxZTA4M2FmOGFmMDVkYWIxZDNmODIyMGZlOTEzOWVkNjYyNGU2ODIyYTAyMjczZDMwYmI0YmRmYzNmY2M5MzY5NzI4YmQwMDA4NTQxZDdjNDFiMjgwMGQ2NjExYTljMzU2NjVkNDI1NjY1OWUyOTk5MDQ5ZDFhMWQ1NjhmYTlhNjM1NjVlNTEwMDU5NWJjNTI5YTNmZDdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.MpFQE6z6dIT02ENf9oMZ1T1CP_a8Pwde50pMoy2ghUTckKqxfwjXZuINyacHQsaomRnPG9Tq87YWTrB1ofE9cw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.634Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkEvMjRTQ1kyNDBxY0hHSlRXNFlab0RyRFU4Mkh5emxpVE5XSDVFWUIrc0djQWNYdWtXSUhaY2t2c1YvakFFK3U1ZzdVSjBQRndkZi9iQTNMU3JmVjd3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDZfMzRfMjQyYl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODJiYTg1Y2JhMDVhNDE3NWNjNjJiYWE5OTYyYzJkOWJiNDhhMjQ4MDEzNmE4NzExZWNkNDJhYjJhNWIyYjIyNTFiMWJiZjljM2YzMTk5NmI3OWNmOWVhODZkZjcwMTIyNWQyODEwMGFlNDViYTY1NjIyYTIyYTlkZDAzMjcxM2EyNjFhYmQ5MjhjZmM5NWI2ZmE0NzgwNzM1NTQ1YjI1ODBjNjIzZmM5MGU0ZWYzNzBkYmU0YTI2N2U2ZWRhYzYwMjAyOTM2OGJkZTI2ZTcwZTEzZjIyYzE2NGFkZTEzYjNiMmJlYWVkMDE5YmI0YTk1MzQwMDllZGRlYmIyNzExMzczYWY2YWE5MzZiZDI0YTEwZjIxYTQwMjU5NDcyYTVhNWMyYTI1ZTMwYzFkOWUxMjRlZDAyNTRjNWY1YzhmZDg1NDhlZjdlNzRiMmMyZTJmMTVhYTQ5ZDUwNDg5NDQ1OWI4ZDRjMjFhNmQ3MDNjMjhlMjVmODBjN2I3YzJlZjU4OWU1YjQ0YzJmYzgzMzY5NzExZGJlNDljMzQ3ZTRjODIwMjlkODkzYjE0M2IxZTI2MTc2MzhhOGMxN2Q0NTBmNGIxMGNiNDExYjkzYzY5NDIyMGU5NzVhYmE5Y2QyYzdkYWE1YzI1NTkwM2JmYTEyZmY0MmMyODY3YWYxZWY4YmVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.yGxl8Ww5GyGPORQrr_68LDtKawuhzVjtov6TDqlOGF90BtWGjEcGw9LY_Z077W_Cebyx3JIuycMcieT_O3quXg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.637Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im44MFhhb3NTeDdWZjhSMkhkL0VXc25XSHRId3pNWGhVN2c0ejNFQnhaWkl3b0xHM2hsMml1QkpheGdUU1pHNWZlRC9YcnQyMTZpRTlla0JjWTJvRkN3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDI0MDMwMV8xMDQwNDZfMzRfMjQyYl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NGE0ZjlmYjY1M2E4NmQyN2ZjM2Y1MTc5N2M5NGEwNzA0MjY0YzFhYjJlNGE1YWUyMzRiMzQzNDRmYTFjMzdmMjVkYTEyMjc2YmJjNzU4YWQ4MjE0ZTE3ZGQwMjAyODA2Y2MzMTY4M2RkNzUxZjE5NmUwNzEzN2RmMzg2YzE5ZDk4MDhhZGRkNWQ4MDkzOWY1Mjc4OGU0ODFjZDMyMmQ2ZGQ0ODVhZjAyOGRlZTMzZTUwNGVlMTBhY2M2MzQzNmM3NWIyYWRhMzRjODExYWE1NDgxYjAyNTUxYTkyODlkMGI3M2U0NWE5MzA0MTgwOGYwMzU1NmEyMjhjMDEwZGEwYjNhMjNkYWUzODQ5ZDdiMjMzZTVlODU4NWIxZGExNmQ4ZTQzYzYyMmEwYWJhZjkwNTNkMDcxYjNjOWEyZGQ4YTBmMjIwYmEwYTQxNjE1NTM5MTE3NTQ2NjY0YjViZjc4NDFlYjU5NTA0MzJkYjUxMzE3OWU5M2MwOTg5ZjA2MWIyNjMzNzQxYzdhMDE4NzBmNTQ2YmM5MWRiNDM5YmI1OTc3YWZkNTMyOTM3ZWY1YzU1NWVjMzU3ZDE2YzAyYTNjMWM0MmU0OGYyOWM3NTBmY2Q2MjljYThlOWVhYTNiYTFjZmQxY2JiOWIwNDEwM2JmMzVkZjQyZjY2OTUzZmY5YjVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.W-G43U7bLTaqqwLcs-9KjKFfqJW--yvzgyl4O5-jWUa422MS7K3xhcvAwDadcGoKGglMmdOX9RZgtiJ0jana3w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.640Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlFjajd5K2FaQ2J0UVZhZnRtMGlHb2J4TXQwZU85WTRaejFKN1dpNDBDek5rZ0ZXK0ZpTmR3RVdmTFF5bGtuMXQxVmxnR2o5Q2FCMmF6T29ZbytIeWx3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDMwNTVfMThfMjRiZV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODQ3Njc0OTQ2ODdjZDI1ZTVjMDJlMGVlMzFiNTY1M2YzY2FjZTVlNGIyODk2OWZkYjU5ODRmZGVmYzAzYjFlZTViMGNlZjdhZDdiNWEzNGU3MGFjNjc5YzBjOGJmNzZiNWRkNmMyY2JlNzM5ODY1NTMwZDU1MTVkYjM4NjRkMmNkOTE4YTg0NzMzYzM0OTk4YTBhZmY0ZjhhOGU3YjFiMDA0YmQzMDMzMWIwYmVlZmVkOTBmYmU5MTQ2NzlkYjgwMjU2MTQzMTIxY2YwOGE4M2MzYTU0N2IwNDE0M2FkYzM1MTkyNzcwNzhiZTZhNTdhNTVjMGM5YjFjOTdiZjI2ZTg3MzdlYzQ0MTI2N2Y3YzBmMjBkODU1ZjgxY2ZkMTZlYzYwM2QyMmFlNWExMDYwNTczZWI3MmFlYjUxNDJmM2E0MTFhN2NkNmZlNWRhMzNkZTNiNzY2MzBjMDZiNzFhYjFmMDVjMmJiOTViMjUyZGM4NDExZDVhNzE3MmFiM2E2MTEyMTVhNWM4OTljZTgyMjRlZTVlYTFhZGRhODJlYTExOTdiZWJlYWU5MGZhOGY2MDNkY2M5ZGJkN2IwNTRmN2FlMzY3YzdiNWNhZGQ0NWViYmJjYjc3ZjVhN2ExOWQyMDZjMGE2MzYwZWMxYmFkNWI2YjlhZTQ2ZjA1NTFjNzhcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.x-tmNDckqOl79_NU_422WC2ioHbZ0wbmpdbx_TN1-GykEw3OXi7wdNDmwgzM42P0Ae1eb46zag1JZPvi7svztw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_103055_18_24be_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.643Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IklPZXg1bitPbTJnVFN2YlQvb241a0txRzdlYk1UZ3M3cnVMVzFlbkFNQjlRM0hWTkF2d3VGSXlFT3h5UzROZ2NWSjFNSStDNzhGdFNWdGF5T0lya2xBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDMwNTVfMThfMjRiZV8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MWE0MDc2ODU5NDM1MWI2Mzk4ZDc5ZTQ0OGZhOGMxZDY4ZGQ5YTZkYmU1ODk0NTFmNGJiNDFkMmRjNDBhZWUxN2UzMGM5YTI3OGVhY2RjYWU2MjhmZTU1MDczODY5NzdiMGYwZjJlYjg2MWVjNzFkNTEyMWZlNjA0MDM0YjEzZGU4NTQ5YmU0MjNlNmQ4YTBmNWM0NjYyZTY4YzdiNDk1N2E5MDFlMGI2Mzg4YmVkZjc2MWZmZmE4NWI3NzU3ODVmNzg1YjVmYzk5MmMwMTA0ZGIwZmE1OTgzYmRiNTNlZDg4MDc1MTlmYmViNjQ0NzkyZWQyOGI3MDU4ZjVhNDMyNDgxNmJiOTczZjBmZjc1OTFiMTcwOTMzYzUyZjI1NDBjYmI2ZWUwZjVmZmRlOTY1Njc0NGVjN2IwNTk1MDdhNWZmNDAyZjVkMmFjYzBkZjVhMGYzMmUzZDcwNGU1YWI0YWM5YjVkMjZjMzRiYTFjODg4YmU2Njk4NTE1MzI5YjYwNjExYWNlZTBmMTVhMjBiODBkMTc0Njg0ODc2NTliZTE3ZmZhZDhkYTBkM2JmYjc0YzdiNjU4MjdiM2ViZGVmOGQ2MWQxM2ExZTBiMmE1MzAxYTM0MmZiZjNkOGNkY2UwZjZlMTc3NDJiMTA4ZDZkNDgzZmJhN2IyNDQyMDg2NTFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.cBU8-Z51j1OhHzzEpo4nkYKDA6Fhjel_COs_p6TDJJ9tjjdt5MbEqDzSFv76VEO7O4bZPNhZSWcDbtTke6pNww", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_103055_18_24be_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.646Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IlhqcDFSakszSUp1cmhDODBTVHM4OHFhNWhBUHhPK2syclNSc2lYeU00bjdCdkZ0VFA2M2JuekZHRXBUb2M4UXFTRXhob0VOYjkzZFNzazZEUUxzRWdBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDMwNTVfMThfMjRiZV8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9Njk5Y2U2ZTRmZmU0NjlmZDdiZWZiMzBhOGI0ZTliZTBmMjAyODA2MTkxNGUwMzUzZGM2MmE2NjIzZjg2NThjOGMwYzcxOGVjNmIxYmFhNjJiZDA1ZDU0ZmZjMTVmZGZmYzc5NDcxZTlkNTVkZjQzMGFhYWIwMGUzMGFkNmViOTIwYzE2NmRkNTZmY2ZmMzRjOWFjMTJjNjI0OTg1NGU0YzgwZTk3OGM4YjU0MWRlNDRhZDE0NzExMjgxNjk1YjI5Nzk2ZGI3MDUzMjkyZDc0ODZmNzI5ZDY3ZmU0NzFlMGJlNGY4ZDZhOGUzZDAyODViNGY1MjllNmVhZWZhNmU0MmMxYjVmOWE4YzAwOTk3OWNjMzRlYzc5Zjc1ZDEwNWJkODRhOGFjNDJhYzYyZmRiNWFjMDE2NGQxZGFjYWFkZGJkNzNhNmFmYTY1MDFjYjlkM2I3N2MyNjE2ZDk2NGQwMDQ2OTg4MGE5ZWY2NmQ0MDYyMzM3NjIxN2EwZGI5Mzg3N2Q5NjU4YzA0ZDM4N2I4OThjZjEwMzllN2NlY2VjZmFlNGUzNzQ2NjQ3NDQ1OGU0NzE0YTdmY2MxNTgwN2JiMWRiOWNlM2JmN2U1ZjVlYjdlZTRhYWQ4NmMxOGFjNTI0YzM0YzE1YzY1MThiNmJlODdmYzQ0YTA5YzEzYTIwODVcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.gk323oixUwsBO88hPntj2vtOUDLnYGUCy_0p5C3Zxtnt8vaza_kuEiDdJ-4JIGOprBv5v6_btJoLsD_xYP_rGg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_103055_18_24be_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.649Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InlZdkd6c3p0bzh1WW5iclFac1ZkM1NpN3RmUUIxNGpxTHFod2w5eDNSQVlaVlBhMXRKSVJhRjhIeHc0TjU3K1BmS3RtLytRWkkrT0JHT1BPUEFyQ2ZRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDEyNV8xMDMwNTVfMThfMjRiZV8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9YTg3MzA3NjQ0NmQyM2Y4ZDYzOGZkNmVjMzNhZDk4MDBiMDkyNjMxODgyZmY4MGU0MTA5YWVmYWY1YjAyODRjNzFjZWM2Yjk2OTUwMDE2ODBjMjFlMGZhMzljNDZhNDVjYWI3MDdhOTlkNDlkNmFhMzNiMjAxNDg2NDg1MDJkY2JhNmIzZTViMDY0OTZmNjJkMDE4ZmMwM2RhZDg1ZjUyNjM5OThhYzY1M2NmMDM3Y2Q2YWQwZWFiOTg0ZTNlYzIwNTQyNDNmYWZjYmYxN2NiYTBjNGY0ZjkzZTRlZWVlNmQ5ZWNjYjBhNWE3NWJjNmU2ZTZjOTBmNWZiODc3NDVmNDExY2NiNWQ1YzUyNzhkNDFiY2FhZmIxNzkwYjdjMTViYzc5NzZlYTA0M2JiYmYxYWE3ZTNiY2JhNDFlZTU1NjIzMGE0YzQwZTdlMjdjMDdkYzQ5NGZjZjFjNTZhNWQxMGU3ZjFkMWJmYTI0NTBiMzE3NjYwOGM4N2RkNmQ3NzY2Y2U1Zjk4ODg1ODkxYTlhODQ4Mzk1MTFiNDhhODM4YmFjZjMxMDgyNTM4MTMxNzE4NTI2YmIxY2MyNjFkZWY5N2UzZWFiNGViNjRmNjIyNjViOTgyZDkwNGFjNDNiYzJhNTliNDI3MDk2Mzg0MTZlOTlmNmUxYmFjMDE0NmYzMDZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.bcOArELgfOERpuj4si0qV3zAn4psWlh2VazL-ACxiV3GgsUxSmaPa8FiGvIf4Vz3EXXpU4OYWeoPcmYiu2wHVg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_103055_18_24be_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.652Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkRvWERXN0pYUU4zNG8wSC9CT0hWeUFBRjVLay9lUnJxbVJBT256N0hmeHFPZ2RuNEs3K0ZkM2RSNTNmSFFTUnlYbG90QU00ckpMc3I1UW9mTU54c0p3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNV8xMDI5NTRfNDRfMjRkMF9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9ODU3OGU4YWFiYzI0MjczMTA3YTAzZjZlZDFjOWIwMTlkNjRjYzE5NzExNThkNmU5YzA0NjJmNzZlMTVjYmI2NmIwOWZjMDA4MTA1NmNlOWQ3NjYzNWQwMjdiYTEyMGQ1OWJmNDg2ZTFhYjNlNWU4Y2ZlOWFiM2E4MWI4Nzg0NTZmZDc4NjU0MDVjZjRjOWE0NGM0NzI0YzFmMjU5NGQ3ZDRhNjljZGNiYjU2OGI1ZGY4M2VmM2IyN2QzYjkzMzVmNDE1NjhkZTJlMGNlOTUyNmY3OGUzNjFkYmU1YjdmYzZhNDFkNzM2YjY3NDljMGE2ZjY1MDBiNzgzZjRiNzBiM2MwYmNjYjE4MzU1NjFlMTMxYmZlZWRlZWE1MGM4ZTY4ZTNjYmUzZTczM2YyYjFlZjk4NDY1MWJmZTBjMDAzYjZkYTYyNGQyZjgwZTVkNTRmOWQyOGZiMmRiMTM2NTRkMjdhNmE1NDRiMjAxNzQ2MzRmNTY2ODcwNmQ0ZmIxN2FjMGU4NzFjNDJjYjk4MjJjMTEzNzVhYmQwYjMzNmJhNzQ0OTQ1ZWYzNzFiNjg0OGRhYTZlY2VkOGRhZjBhMjVlZTA2N2RkMTc0YmU0YTgzNjNiYTYwNzMzOGFlNDgwZGUzNDdjNzg4YzdjM2VlNTQzMTQyYmJkZWYyN2ZjMzQ0MGJcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.nLVQHkXuT-hQ1xCWSctRTxfHpQnEWJL7i9umllP64n6vJRybLS4IBxHxHTn_wkV9Xe5gOSkYzS2tw3E8DAOfRw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230525_102954_44_24d0_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.655Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IjJLNzhtaVhKWVZXY1g4Y1k0LzE0aTlJZUtWZmNDRkVZempCOWhDcGtVTEE5Q3MrU25VdGREY3JVRFR1THppckM3d1pIRXgvZkxuclNYRDhaeURCVkF3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNV8xMDI5NTRfNDRfMjRkMF8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjAzMmQ1ZDQyM2QwNzlkMTEzZWU3YWFmZTk1MTZkNThkYTYyMTBiMWMxMTMyOWQzM2IxOWNjOGM4MDE4Njc0MTkwOTgzYTM1MmZmODBhMDcyOGQ1NDViNWVkY2NiMjQ3ZGI3MzcyMmNmODZmYTNhMDFiYjZmY2MwMjRlMzZlNDkyZGQwMzI3ODlmMDE5YTI5MTkwZTQ5NmY4NzVhY2E2MzIzZjJkZWVlNmM5NWVhNDdhNjUxNTUwMmM4YTU4MjRhMDVmZDJjYzY2ZmYxZDhiNzE5NGFiYzNmYjY0YTlmNzI2NjQ1NzViY2UwOGQ2ZTk5M2I0OGNmNDBmNjQ2ZjhjMzA5MjNjMTc0ZTNlMjEwM2ZhMDJhYTU4OTRlY2RhMjc2MmEyNjgzZjk3M2E2YzcyNDM0MGEwZTEwZGE4NjlhN2U1ZWUzZTM2OTBkMTIwZTQyZjE3YWEzMmNmMTAzNzEzMjA1M2QzMzU0YWJkMDZmNDIwN2RiMTFmYmNkZmI2YWIyMjBlMzkzN2Y3ZGFmNTZkNmYyZTdkOWIzYTI1ODhjZWRhODU3NDY3MDcyM2IwMGU2ZTQwMzAxZjA4MjkxZGQ0MGRjNjMwZDIwZjQyYTliZGU5NGJlZWZkNjYxMDllMjA0YWUzMmNjNWMwYzQ1MGUxMWY4NzIxYmQwNjIxNWIyN2ZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.JXLF3bKfpwrYh9CuSotBW3gX-I_a5eeNur7F5jr0Ii_sb_G2tWTv7Gj5eMtnSYC4Z2XzWMdPHzQfkJR80G1c3A", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230525_102954_44_24d0_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.658Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkJNclN6U3pJbTRQU3NUOHlqc0tEQ1NiTjZCc1EveFlnNXVWN200cmZTa2V0QVBiUTVPa210cXJtNFlFZzFNdHNhTHg4RlNDTjcvNUJNUHN5TlV4THNnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNV8xMDI5NTRfNDRfMjRkMF8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTBlYWQxYjBhN2FhZjdhMjlkZmI0OGZmMWI3NWNlZWVmY2Y2NjlmY2YyNGE0ZjM4YTM2NGQ4ZjUyZWEwZjI1OWZlMTc0N2EwMTZkYzYyZGFiMzUyZGVmNDhhYTJhM2JjOGQ2YTY0ZmQ1MGIzNDQwODE3NGRmYmU1YjZmYTkxNzA5MTAxZjlmMjYyNDQzYTg1ZDFiYzViZWNlYmQzM2ZjZmUzOTNkYTM4MmMxOGJkZjBlMjg4ZTViOTdkZDJkZDI3YmRiMDFhMDNjYmE1Nzk5ZGMxY2NiYmY5Y2FmYWU1MzMwYmI4MWMzZWEyYjkzZmQ5MjI2MmUyN2Q3MTA4NDY1YTIzOWVjNDkyYjc4ZGJmOGIwMWEzYTM3NWIyNzM0ZTBlODNjODAxOTMyNGFiMGI1MWU4N2Y1NjBhZTI3OWUxMmY0OTRmMGIzMTRjNTI3OTg3ZjQxZTliZmU3MzRjNDg5YmJiNGU1MDM5ZTNiNWM0YWU5OTMwM2FhMGZkMTljNDBjM2Q5Y2M0MTk4YjRiNDNiYzZhZDhkYjVhMzE4YTUwYmVhNzZjZTkyNDE2MDMwZWQ2YWQ2ODc1OTEwMjA3YzZmMDc4MTEyMGI1MzI1YzlmY2M3NTY2MzI4ODYxZTFmNjBhMWI5MjQxMDExYjdiODc4ZDMzOTZjYWI4YjkxMTM0YzlcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.T1BRiznYDkGplDQOxu4fxkiEnsN6WGwFoOvocN3Pgw7gnDyBMlug9aX_kkk8GdbCILquNj_uETOFADGcm5-klg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230525_102954_44_24d0_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.661Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Im9Cc1BxbU1SN3prUEp4K2x0Qmxja0VQVkRFY1MzVWowQUpCa01XR0FTQWpKODgrM1MzTHVBUzlWZXVlaUlnbnEwN1phTFRpYVlaSnJrTHAyeTNnRERnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIzMDUyNV8xMDI5NTRfNDRfMjRkMF8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NjM2MWFmNTBiYTQ0YzVmNzEyMTJhNTBlZTY4MGIxNTg2ZjRkNGMzNzQwMDFjYTRmMDI2OWYzZWZjMWQ2ZjE3OGIzZjgwN2M5YzY1OWQ1ZDkxMjU3ZWQ3YWY4ZDZmMzE2MTAyMjc3NTEyMTI0ZDEyMGEwNTJjZWJkMGMzNzcwZDZmYWVjMjAyMDY2MmI2ZWVjOWVlZGIzODM3YzRjNjk3YzA5ZTFjZmFlYTk1MzMzYzY1NDRlZjU0MDBiNjRjYWNiMTk5MzgxNjc2YmJlMDk1OWNlNmI3YzViNzk3NTk4NjkzMDcyZDYzYzU1MzRkNWUxNDRiODM5ZTM3NjA1MmE2OTYxMmE1YzMxNGY4MGIzOGNmZjY5NTgxM2E1ZjgwM2ExMjAxNDkzYWE3ZmQzMmZjZGE5NWJiMzE3MDFlNTQ1YzRlZGM4Y2UxNWJhODFkYmI1MjMwNjc4YWM4ZDg4YmNmNDhiZWU2NzMwZWE0ZDMzZjdiNDIzMjFiN2ZiNDgzZjQ3ODZhNDFiMGFiMzE4MzU4Yjc2ZDE3MDllYmM0Mzc0ZDM1MzA3NzVhMTU2ZDliYTc4MGM5NzczOTFlNjFlMWEzODRlOGUxMzA1YTgyODEwYTFjYWFkZWE5MmM3YTQzMGI0ZTdlMWJjMDMyMDIwM2Q4ODdiM2YzZWNmOGZkNjRiMWZcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.rh7RFdxw8eKGX3vTDU4GiJPJDCkrY0gbCa00JjlYtsAo1HLSPFlqUJq8BBY__OORnPFy6crqXJDqcP4Um4nAqg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230525_102954_44_24d0_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.663Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6Imk5ZjBEU2hpU1VaQm81anpTOTZzT1dOd1RTdUViQ3MrMHJRZFBwUFZGSVhTY0hFVXRNdWRQTGNjdEtBeSt5Z01qVXFoQ3ZwTDRiTFhGenpGK1FiRWxnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyMF8xMDI2NDdfNTBfMjQzNl8zQl9BbmFseXRpY01TX1NSXzhiX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9OTgyN2YwYjNhZjk5ODZmMTlhNzdmMGQyNmVhY2VhODRlYjNkODU2MWFkNWE5NzRiY2QwNDI1N2NjY2YyMTU4Zjk1MDM3YTc0NTYwZWMwYTQ3ODUyY2Y2Y2M4ZDU0YjdmMDQxZTIzN2M5NDUzYjk1NGI3Y2E5OGY2NDNhYzVmZDRjZGU1ODAzMmJhZmQ5MDJkNjhiZjQyNzgyNTVlY2JmMDdkMDU5MmY4M2I1Nzg5NGMwZWZiNjM5ZGJjYWUxNjQwMjRkMjY5YzYyODJiNTRmMDUzMWZjMjIzNGNmODNmNzc3ODJiMDJmNzgyMzg0NzM1ZGRkMmIwOTg2N2MzMDliNGM2YzE0MDM3NmY2NGYzNjZmZTc5ZGFlOTA2NGMxZWM3N2RkMWZiODI5ZDdkN2ViZWIwYjQ0ODA3ZjY2YzNlNTE5ZThjMDUzYmMxMzk5MjRlYjRlN2ZkZjFkN2Q5NjJkMDFiYWZiYWQyNjIxZTlhMmRiOGIzZjQ1ODQyMTQ2ODgwYzM2YjRjNjZmNDU5NzFhYzY3ODI1YjgxYjkyY2Q4M2Y0YzRhNzQ2YjE2ZWU4NGI4MTNhNTIyOTE0MWRjNDE2Yzc1NWQ2YzlkNDkzNTJhZDc3MTJkZjUxZjBiNzk3Y2YwMTdjZWRiODQ4MjMyMzY2YzQ5OGI5YzZhMDgyNjdmZjdcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.WwYv3K3Lt66btxxnCYL_8NR9sDhPJV7uGn6EbUowo-iL2YaufJnYFlwBfbkMT122Qzwm-NrjcPE6H-jybvkXpA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220820_102647_50_2436_3B_AnalyticMS_SR_8b_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.666Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkpDVUVRR3RmdFNXT2FmUzEvUE5OakNhcGxMREVQVXF2QVZEcStUUEVxWDE0cmVuMHlvQngzZmlqQ3pPU3pxZDQweU52dTZreFpvMm1mK2Q1OFd4V0tnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyMF8xMDI2NDdfNTBfMjQzNl8zQl91ZG0yX2NsaXAudGlmP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9NzQ3MzZlZDRhMDFjZTcwYjdkNmNmM2FmZDM3YjA3NDJhYjllYzViYTliZTE4N2I5YTNiMDVlNmI0NGZmNDVjYTc0ZWU4N2M1NTcxZThkZjFlYWE5MmEzNjM4MzkyMWQ1MDRlNzhhMDQ5N2ViMDFhZTM0ODE0Yjg4ODFkZTViZGE5OGY1ZGE2YjE5ZmI5NmZiYTM0OTc5Y2E5ZjcyYTk0NjQ0NGJlYzgzMzNmY2UxOWQwMDQxYWE5MGZhNTdlNjZlOTdkMjQxZTNiZmU0MmI5ZDkwMGUzNjEzNjNjOWViNTM4N2Q3ZTNkNmY2YTFmYmU1MmMyOWZiZGIzZjY0MGUzZGNiNDAxZWRhNjQ1MzU4M2M2MDhmNzY5OTFhZTRjNGQxNzQzNmE4MjNkZDZjNDNmYjFmYTViZGJmM2E5YmQxMGZiM2VkNDZjNTQ1ZTQxOTMzMjcxMTljNmJjNzFlMzZhNzNjOTg3NGRkYzE5MDlkYTVlMzI3NmY5MGFiZDE1YjBiNDI0YzlmZTk2NDRmMTU0NDEwZDI4NjM2Njg4MGY5Zjc2NzdhMjIyOGQxZGZkOTAxZjYyZWI1Mzk0YjczYjgxOGZhOGEyMjhlYTI5OTUxN2NiMGRkYTI3OWU0NTM0MmNlMzNiMzYzYjQ2ODU1NTgwNTRhMzdiNmJlMTI2YjViODNcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.PxhZFQJykJbYAn_AiV5wpH-h6np1pz6oBC9uf6dHH0oFX5MMwP9nPC_gRnyrWJ7aSsFzpg15YMSGkW4qEYlj8w", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220820_102647_50_2436_3B_udm2_clip.tif"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.669Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkY2a3VlVGNENFV0SmlRMkxNcFcvS3NSWWt2Y2duNGhpZjFpVVpSY1ZkanVXZ1VqbFBadnpqV2RQb1dMK3oyWUU1RWd2UUdtUmc0SzlMQzFDaFZHS2lBPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyMF8xMDI2NDdfNTBfMjQzNl9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MDJhM2IzNDBjMmY2Yzk5NzRhZDBjODRhNzM3ZWEzNGMxMTZiNGE0ODAyNGFmN2MzNmY4OWE2NDJiNWVlMjUxNWZiYzZhMmExMDY5MWIwNjNiMWY0OTRlODRkNTNkYTY1ZWNhODU4Zjk2OTM5YmIwNjZjNGEwZTM0MGZmZjFlYTU4YjYzYTZhZmYyMThkNjBkNWFkZTI3MWQ2ZTE1NWY0MzNhNjI3ZjNiMTllMjllMjVlMGZkNzQ3MmJiMDMyYjY3MDgyMjU1NjM2NDZiNjE5MTJjNzY3OTNlOTA4ZjE5MzM3ZDlkNWQ4YTE5MGE4ZDA3N2VkMTU3NWI2MTUzNmYxMWJjMWZjYzY0YmZlZjgxNDFmOTdiYTE0ZDE5M2RmYTU4NmUzYWMzYmQyZTQ1NDA5NjM2YzdmMGJhYzFmZTM1ZjJiM2QwNjdmNDlmN2FkNTk4YjhhNTgwYTU4MWRlODkzZjdkZTA0ZTI0ZjA5ODRlYzc5YmY2ZTJhMGFhYjBjY2MzN2IzZjJmYzBmMzZiZWU0OTNmNDc1OGQ0Nzg2OWYyODM5MGExMTFiNzFjMmI2ZTQyNGQ2OGE5YmNkZjJiZDFjNDBlYTg2OWQzODE4ZWFjYTk4NTg1ZTMyZWVlMzdkZjgyOWQ5Mjg2YjFkODliZjU3MDg5OWQyNmI2MjMxYzI5MDFcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.7Z9RsYx8k5x7cqHK4nxBZ-OtznOH0g-jcDocg-Ifu3XnUgsII2RxJ08EvHRav1a_l-VVro7rjx903PSLRwWFzw", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220820_102647_50_2436_metadata.json"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.672Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6IkFNTlk0ZUlGRjl2WVRHQlJ1dDFQS0dFSGpURThPb1g4L1ZmdU9nOVlPSHo0Y2xDREdSd3ZPK2J2ZGwya0VYU21UUERvZVlhNUFXUFJIQUVGTU5Sc2N3PT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDgyMF8xMDI2NDdfNTBfMjQzNl8zQl9BbmFseXRpY01TXzhiX21ldGFkYXRhX2NsaXAueG1sP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzNFpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MTU3NDU0OTFkZTU1NWNjMjgwMWNiNGExZjU1MzE2OGNjZjJhOTFiMWJkMWM1NGMzMjc1YTUyY2Y4MjQxYzE5OTMwODYyMTIyOTY5OTBlYWI1ZWY2NWY1NDBiYjA3Y2RiY2YyN2ZiMzc5NThlZjUyMTRhNzgzYjg3OGFjNjkyZDM5ZDBhM2E0NmMzYzRjZDhkZjMzNGQ5ODc4YmY3OWU0ZTNiNWQzZmViMDg5NWQyMTMzYTQ5OTA0YTYxOGY1MWVlMjk0ZTdjZjdjMTljZmRkODFkZWU5M2Q1Y2NjN2M2MTMyMTJmMDNjYjA5OWFkYWM5MjdkYTE0NGYxNTk4NGJiN2E4NTJlMTRlYjE4ZDM5ODU1MjJhYTNmZDc1ZGY3M2UxNjhlZDk5OTc5ZTBlMmY5ZTZjYTI2YTUzMTczNzMxYmM2NTQ1NTU3ODA0ODE2MzEwZTNlZjk3MDE3MDFjODZiMzM0OWY3NDk0NDliNTczYzkxNjFjNDRjNmIxODM5NThmMzMyMDgxZTgzMjU5NmE1OThiMGVhYmU0MmJkYTZjM2Q3YTZjY2FiZjBlMjQyNmNkYTliZDU2YWQ0MTVjMGRiMGRjZGE5OTJmZjdhZmNkMTg0MTNlNDAyNjdlMGJkYjE3ZjlhMjQzMDAzMjAxYWU2ZTFjZmU5ZjRiZTljMmVmYTBcdTAwMjZYLUdvb2ctU2lnbmVkSGVhZGVycz1ob3N0Iiwic291cmNlIjoiT3JkZXJzIFNlcnZpY2UifQ.Ci9Jl8TTj-0yKMmMhi8h_UMMrpyydd_qYhB9hLd6lbr3hyIpKHJ6r2YJcqLJTgASzlVL-py_4y0Uv2GclF-hSg", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220820_102647_50_2436_3B_AnalyticMS_8b_metadata_clip.xml"}, 
{"delivery":"success", 
"expires_at":"2024-05-31T14:36:34.675Z", 
"location":"https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTQsInN1YiI6InpzSjFaNUYxSzRZVkpDZDFSS2JXOGVMRHJkd2hpb0RPYkwzVHU3dHVEeVhkYXhTZUhqb1VUaWJMRG8zMlQyLzdQZ21NL2kyTDZ5TVNFMUJFMTlHYXVnPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvbWFuaWZlc3QuanNvbj9YLUdvb2ctQWxnb3JpdGhtPUdPT0c0LVJTQS1TSEEyNTZcdTAwMjZYLUdvb2ctQ3JlZGVudGlhbD1jb21wdXRlLWdjcy1zdmNhY2MlNDBwbGFuZXQtY29tcHV0ZS1wcm9kLmlhbS5nc2VydmljZWFjY291bnQuY29tJTJGMjAyNDA1MzAlMkZhdXRvJTJGc3RvcmFnZSUyRmdvb2c0X3JlcXVlc3RcdTAwMjZYLUdvb2ctRGF0ZT0yMDI0MDUzMFQxNDM2MzRaXHUwMDI2WC1Hb29nLUV4cGlyZXM9ODYzOTlcdTAwMjZYLUdvb2ctU2lnbmF0dXJlPTRhZjg4MDE1Y2YwYWQ2NDY0NzVjYTliNzQzY2E1NWMwYTRkOTBmNTdjYjY1ZTc5MDdmY2FiYTEwODljOGY5ZTdmNWNhMmJkOTkyNWEwMzI1YTIxODVjNzhhZmFiMjQxYTEzNWIwZTI1ODE5NDlhOTFjYWI1OWY5ZWI4ODRhNzkwNWE2MzA4Nzk0MDlkNjYyODRkNTQxNDI2ZDhjNzU2MTUyZmI3ZjQ1YWZlN2MxYzNiYTcwNmY2NGU3NzlkYTM2YTlkNTBkNmQ3MTY5ZmE0MTA4YTcxODM3OTkwMjRiNDAzZDRkYmNjZGMyMjM2NWY4NjJhN2UwNTRiMTU3ZmEzNWE5ZGNmZWQ4ZmQ2YmMyMWZlNDI0NGYxNWYyYzQ3NGY4NGJiMzkwYTMwMGYxZDU0MWU5ZjdjYjNiNDVlZTc2MDJlZTMzNTkwOTkzYTUzYTYzZGE5NjExYjY1ZTdkZjk5MjE3OGQ3YzRiYzEyZWY1N2EzNjMyNzhlNWJkMTMzYjRlN2E4MjM1OGYzNWY0YzJkMjUxNjQ3YmNmMTI1OWNmMWY2OTRhZjVhYzdhMDZkZWFiMTIwZjU3NjQ0YjU5ZTc0NjYzNDE5MzljMTAyZjEyZDg5ZGY5ZmViZDk5ZWM4NTRkYTRlZjk4NzdhOTkxYjc5OGU3MWQyZGI4ZjEzZWJlYjljXHUwMDI2WC1Hb29nLVNpZ25lZEhlYWRlcnM9aG9zdCIsInNvdXJjZSI6Ik9yZGVycyBTZXJ2aWNlIn0.VDD2Ghl3MN9nOcPtrpL3PwmLSwU_ELOMqLFScpNjnjdHA041PjwXXlNWxpygCv-WlJesvIqGNPo5SH8kNMoAUA", 
"name":"9c146ad9-20eb-4407-83b5-7ad963549f7c/manifest.json"}]}, 
"created_on":"2024-05-30T11:55:21.873Z", 
"error_hints":[], 
"id":"9c146ad9-20eb-4407-83b5-7ad963549f7c", 
"last_message":"Manifest delivery completed", 
"last_modified":"2024-05-30T12:32:56.018Z", 
"name":"full_airth_SR", 
"products":[{"item_ids":["20221125_111318_95_2413", 
"20230209_111628_64_240c", 
"20220305_105837_49_248e", 
"20221128_110618_08_227a", 
"20220920_105809_51_2495", 
"20220215_102554_23_241b", 
"20220321_102740_31_242b", 
"20230126_110351_81_2485", 
"20220227_111348_12_2274", 
"20220227_103256_21_2430", 
"20221205_110047_16_2484", 
"20230125_104939_27_2276", 
"20230126_110127_66_24a5", 
"20220910_110421_57_249c", 
"20230128_110445_35_2480", 
"20221016_110338_23_2461", 
"20220327_111309_86_2416", 
"20220924_104649_13_2251", 
"20221206_111226_60_227b", 
"20230126_110354_09_2485", 
"20220324_111806_41_2403", 
"20230224_102308_97_241d", 
"20220331_102509_46_2451", 
"20230128_102801_66_2464", 
"20230117_105702_44_2475", 
"20220308_103324_53_241e", 
"20220127_110501_81_2475", 
"20230201_110354_54_2475", 
"20221018_102515_23_2451", 
"20221002_105839_86_2482", 
"20220217_102736_74_2428", 
"20220208_111800_46_2254", 
"20230130_101429_64_2445", 
"20221119_110126_09_249d", 
"20220920_110455_06_247b", 
"20220914_102314_67_2448", 
"20220331_102511_75_2451", 
"20220131_103330_66_2447", 
"20220324_111808_70_2403", 
"20221125_111321_18_2413", 
"20221119_110123_73_249d", 
"20220924_102733_83_2429", 
"20230125_103055_18_24be", 
"20220120_112258_12_2413", 
"20220910_110423_85_249c", 
"20221121_105824_65_2489", 
"20221116_110123_19_247d", 
"20221121_110816_23_2416", 
"20220320_111106_52_2424", 
"20220327_105920_66_2438", 
"20220219_102611_79_242b", 
"20220305_105839_85_248e", 
"20220914_102316_96_2448", 
"20230130_101427_29_2445", 
"20220121_103503_67_245c", 
"20230120_110046_97_249a", 
"20221104_110226_41_2485", 
"20220912_110206_64_247f", 
"20220120_111559_26_241c", 
"20230225_103319_69_24ab", 
"20230130_102227_00_242b", 
"20220211_112004_51_2416", 
"20221104_110224_05_2485", 
"20220121_103501_36_245c", 
"20220227_111350_42_2274", 
"20221206_111224_26_227b", 
"20220322_102819_57_2427", 
"20220912_110204_35_247f", 
"20220131_103332_96_2447", 
"20220301_110456_52_2473", 
"20230209_111630_77_240c", 
"20230128_102803_88_2464", 
"20220127_110459_50_2475", 
"20220127_112308_86_227c", 
"20220211_112002_21_2416", 
"20230114_110751_23_227a", 
"20220211_112033_92_2403", 
"20220127_103118_94_2460", 
"20230116_110258_70_2499", 
"20220301_110243_91_249e", 
"20220920_105811_81_2495", 
"20220328_111832_58_2424", 
"20220913_102534_05_241d", 
"20220327_105948_19_2477", 
"20221128_105914_48_249b", 
"20221121_105822_31_2489", 
"20221128_110212_03_2475", 
"20221001_103006_87_2439", 
"20221128_105912_33_249b", 
"20230912_103201_88_2415", 
"20230826_110649_57_2477", 
"20230907_111146_58_247f", 
"20230901_102501_45_2451", 
"20230708_102951_29_242d", 
"20230721_103437_99_24b3", 
"20230908_111006_51_2438", 
"20231011_103123_96_2459", 
"20230831_111137_87_248f", 
"20230712_103147_74_24bf", 
"20230712_110526_63_248c", 
"20230831_111140_04_248f", 
"20230721_103546_21_24c4", 
"20230911_103046_44_24b2", 
"20230721_103440_28_24b3", 
"20230701_111012_94_248c", 
"20230907_111144_43_247f", 
"20230903_103633_24_24a8", 
"20230907_103039_40_2448", 
"20230912_111116_87_2475", 
"20230622_103034_08_24bc", 
"20230901_102503_55_2451", 
"20230908_111008_63_2438", 
"20230701_111015_04_248c", 
"20230826_110647_40_2477", 
"20231001_111051_14_2483", 
"20230622_110743_09_2488", 
"20230824_102843_49_2455", 
"20210125_104635_84_2441", 
"20210315_103452_81_2421", 
"20210423_103443_39_245e", 
"20210211_113113_80_2403", 
"20211115_103642_53_2458", 
"20210222_112512_89_2408", 
"20210315_103455_16_2421", 
"20210715_104635_59_222f", 
"20201119_103352_21_2235", 
"20210505_103609_03_2445", 
"20210530_103838_88_242a", 
"20201203_113140_69_227b", 
"20210715_103425_00_2449", 
"20211121_103423_95_2449", 
"20210715_103422_70_2449", 
"20200926_113127_32_2412", 
"20210321_112622_77_2401", 
"20210531_112802_19_227c", 
"20210125_112737_58_2414", 
"20210531_104849_25_2259", 
"20211021_112311_88_227a", 
"20210301_112635_82_2413", 
"20210106_112446_22_2416", 
"20210416_103945_82_2465", 
"20210327_103854_29_2463", 
"20210818_103121_67_241b", 
"20210112_112441_70_2414", 
"20210125_104633_56_2441", 
"20210130_112944_25_241c", 
"20211022_112631_93_2406", 
"20200407_103546_31_2275", 
"20210225_104655_34_2264", 
"20210422_112648_84_2416", 
"20210208_112811_36_2424", 
"20210422_103413_14_2212", 
"20210627_103640_07_2449", 
"20210810_103630_73_220b", 
"20210403_112730_08_2412", 
"20210405_103433_50_245e", 
"20211126_112138_63_2413", 
"20210531_112759_70_227c", 
"20210901_103602_27_2460", 
"20200926_113129_67_2412", 
"20200918_104047_72_2223", 
"20210701_112856_26_2405", 
"20211121_104533_42_225a", 
"20210122_113113_32_240a", 
"20210208_112813_70_2424", 
"20210122_113115_67_240a", 
"20210327_103856_65_2463", 
"20210530_103659_28_2453", 
"20210422_112646_49_2416", 
"20210726_103643_58_245d", 
"20211115_104512_70_2233", 
"20210208_113054_06_2407", 
"20210321_112620_51_2401", 
"20210301_112633_55_2413", 
"20210425_104201_66_2441", 
"20210530_103836_57_242a", 
"20210530_103656_98_2453", 
"20210627_103252_52_245a", 
"20200918_104045_51_2223", 
"20210818_104349_95_227e", 
"20210319_103335_25_245a", 
"20210715_104637_80_222f", 
"20210412_104349_05_2259", 
"20210425_103321_12_2456", 
"20210818_104347_66_227e", 
"20210901_103559_97_2460", 
"20211121_103426_25_2449", 
"20210125_112739_86_2414", 
"20210419_112830_31_2426", 
"20211126_112140_93_2413", 
"20201203_113143_06_227b", 
"20201125_104259_22_2278", 
"20201125_104257_01_2278", 
"20210818_103124_15_241b", 
"20211128_112201_58_2274", 
"20211115_103640_06_2458", 
"20210106_112858_23_240c", 
"20230309_105942_18_2495", 
"20240201_103853_41_24c3", 
"20231128_111441_82_2482", 
"20230310_102727_65_242d", 
"20230310_102725_41_242d", 
"20230308_110437_97_2481", 
"20240331_112959_67_24c6", 
"20230307_110505_13_248f", 
"20230309_103005_10_2439", 
"20230322_110214_17_2438", 
"20230309_105944_41_2495", 
"20240111_113446_95_24e1", 
"20231023_103624_64_24cc", 
"20240322_104625_77_2465", 
"20231105_103343_06_2439", 
"20231108_111454_64_247b", 
"20231128_103832_88_2439", 
"20231105_103341_01_2439", 
"20231124_103247_51_24ca", 
"20240322_104623_87_2465", 
"20230311_110349_22_2490", 
"20240118_112213_86_24a4", 
"20240226_103909_50_24bc", 
"20240331_112957_43_24c6", 
"20231127_103351_89_24c4", 
"20240113_113147_52_24aa", 
"20240316_113338_21_24e1", 
"20240207_103938_39_24b0", 
"20240226_103911_64_24bc", 
"20230307_110507_36_248f", 
"20240316_113433_62_24d2", 
"20231111_111212_97_2473", 
"20231107_111233_30_247a", 
"20231023_103622_39_24cc", 
"20240222_103613_74_24bc", 
"20240222_103841_13_24a1", 
"20240117_112045_83_249c", 
"20240226_104021_21_24c7", 
"20240408_113311_82_24f9", 
"20240201_104151_48_2458", 
"20240301_104044_41_242b", 
"20240113_113149_86_24aa", 
"20240106_113530_64_24f5", 
"20240408_113314_06_24f9", 
"20231022_103318_39_2442", 
"20231107_111235_39_247a", 
"20230327_110515_77_2489", 
"20240224_112445_35_249c", 
"20230308_110435_78_2481", 
"20231124_103249_69_24ca", 
"20240106_113532_97_24f5", 
"20240126_103917_68_24c5", 
"20240111_113212_49_24f1", 
"20231111_103148_65_24b2", 
"20240203_103426_29_24c4", 
"20240111_113444_62_24e1", 
"20231030_103621_68_2439", 
"20230307_110231_89_249e", 
"20231127_103354_05_24c4", 
"20240301_104046_34_242b", 
"20240304_103919_69_24b5", 
"20231014_111055_07_241c", 
"20231023_103240_07_2429", 
"20231123_103650_81_24af", 
"20231022_103631_48_24b2", 
"20240118_112211_88_24a4", 
"20240113_112217_90_2482", 
"20230308_102400_24_2455", 
"20230311_110351_44_2490", 
"20240408_104057_03_2415", 
"20220704_111657_22_2402", 
"20220618_105818_44_249a", 
"20220604_110430_92_2446", 
"20220915_110304_49_24a5", 
"20220707_110419_56_2481", 
"20220620_102924_31_2423", 
"20220424_111656_16_2407", 
"20220419_104713_76_2276", 
"20220810_102706_17_2432", 
"20220604_102336_40_2436", 
"20220817_110356_00_2482", 
"20220524_112108_37_2402", 
"20220811_102823_56_2427", 
"20220620_102922_02_2423", 
"20220710_110152_14_247e", 
"20220605_111555_14_2274", 
"20220524_112106_08_2402", 
"20220811_102821_27_2427", 
"20220820_102647_50_2436", 
"20220707_110417_27_2481", 
"20220817_110358_26_2482", 
"20220423_102657_03_2453", 
"20220429_111304_58_2416", 
"20220813_111341_99_227a", 
"20220419_104711_48_2276", 
"20220827_110001_35_2499", 
"20220423_102659_32_2453", 
"20230407_102929_63_24c4", 
"20230526_101929_20_2460", 
"20230616_102635_77_2459", 
"20230507_110503_08_247a", 
"20230530_110447_41_2486", 
"20230604_102555_27_2420", 
"20230525_102954_44_24d0", 
"20230530_102018_50_2459", 
"20230610_103116_22_24c8", 
"20230529_102147_63_242b", 
"20230516_103033_96_24b4", 
"20230530_102020_67_2459", 
"20230526_103007_64_24ca", 
"20230616_102924_87_24b9", 
"20230403_105030_78_2262", 
"20230603_102952_98_24c3", 
"20230422_110321_59_249c", 
"20230421_110133_15_248c", 
"20230415_102433_11_2465", 
"20230407_102927_33_24c4", 
"20230420_102455_38_2460", 
"20230420_102453_19_2460", 
"20230418_110451_89_249c", 
"20230421_102328_33_2451", 
"20230613_102533_50_2451", 
"20230414_102822_97_2439", 
"20230603_102950_65_24c3", 
"20230421_110135_31_248c", 
"20230614_112214_26_2402", 
"20230415_102927_80_24b0", 
"20230418_110653_35_2486", 
"20230602_102855_95_24c9", 
"20230613_102604_33_241e", 
"20230609_110806_88_2470", 
"20230418_110449_73_249c", 
"20230419_110631_28_247f", 
"20230604_102658_81_2440", 
"20230604_102656_74_2440", 
"20230414_110147_85_248b", 
"20230408_110119_93_24a4", 
"20230522_110454_17_248e", 
"20230403_105032_82_2262", 
"20230415_102430_94_2465"], 
"item_type":"PSScene", 
"product_bundle":"analytic_8b_sr_udm2"}], 
"state":"success", 
"tools":[{"clip":{"aoi":{"coordinates":[[[-3.925805, 
56.143895], 
[-3.911592, 
56.143895], 
[-3.911592, 
56.149272], 
[-3.925805, 
56.149272], 
[-3.925805, 
56.143895]]], 
"type":"Polygon"}}}]}


In [49]:
import requests
r = requests.get(order_dict['_links']['results'][0]['location'])
r.json()


{'id': '20220924_104649_13_2251',
 'type': 'Feature',
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-3.925805, 56.143895],
    [-3.925805, 56.149272],
    [-3.911592, 56.149272],
    [-3.911592, 56.143895],
    [-3.925805, 56.143895]]]},
 'properties': {'acquired': '2022-09-24T10:46:49.139042Z',
  'anomalous_pixels': 0,
  'clear_confidence_percent': 100,
  'clear_percent': 100,
  'cloud_cover': 0.0,
  'cloud_percent': 0,
  'ground_control': True,
  'gsd': 3.8,
  'heavy_haze_percent': 0,
  'instrument': 'PSB.SD',
  'item_type': 'PSScene',
  'light_haze_percent': 0,
  'pixel_resolution': 3,
  'provider': 'planetscope',
  'published': '2022-09-24T13:36:50Z',
  'publishing_stage': 'finalized',
  'quality_category': 'standard',
  'satellite_azimuth': 281.7,
  'satellite_id': '2251',
  'shadow_percent': 0,
  'snow_ice_percent': 0,
  'strip_id': '5951754',
  'sun_azimuth': 155.9,
  'sun_elevation': 30.9,
  'updated': '2022-09-25T05:50:04Z',
  'view_angle': 3.9,
  'visible_confidence_pe

In [59]:
for result in order_dict['_links']['results']:
    url = result['location']
    name = result['name']
    response = requests.get(url)
    # print(response.json())
    if response.status_code == 200:
        with open("full_SR/" + name, "wb") as file:
            file.write(response.content)
            print(name + " downloaded successfully!")
    else:
        print("Failed to download the file " + name)


9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220924_104649_13_2251_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20221206_111226_60_227b_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220121_1035

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104347_66_227e_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210818_104347_66_227e_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230901_102501_45_2451_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103640_07_2449_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103640_07_2449_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210627_103640_07_2449_3B_Anal

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230609_110806_88_2470_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231030_103621_68_2439_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230126_1101

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113532_97_24f5_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240106_113532_97_24f5_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240201_103853_41_24c3_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102533_50_2451_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_102533_50_2451_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230613_1025

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220208_111800_46_2254_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220914_102314_67_2448_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220321_1027

...

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103911_64_24bc_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240226_103911_64_24bc_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20231022_103631_48_24b2_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210419_112830_31_2426_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210419_112830_31_2426_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210419_1128

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230530_102018_50_2459_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220707_110419_56_2481_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210125_1046

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110135_31_248c_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230421_110135_31_248c_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230209_111630_77_240c_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112959_67_24c6_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_112959_67_24c6_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240331_1129

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240113_112217_90_2482_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230407_102927_33_24c4_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20210715_1034

9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220915_110304_49_24a5_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20220915_110304_49_24a5_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_3B_AnalyticMS_SR_8b_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20240301_104046_34_242b_3B_AnalyticMS_8b_metadata_clip.xml downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_103055_18_24be_metadata.json downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_103055_18_24be_3B_udm2_clip.tif downloaded successfully!
9c146ad9-20eb-4407-83b5-7ad963549f7c/PSScene/20230125_1030

In [56]:
order_dict['_links']['results'][0]

{'delivery': 'success',
 'expires_at': '2024-05-31T14:36:32.367Z',
 'location': 'https://api.planet.com/compute/ops/download/?token=eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJleHAiOjE3MTcxNjYxOTIsInN1YiI6ImN4YjcxTjRwc3R1OUtsaHprMlA2R281VWt1a2FKMlQ5UGhxTUN2QmlsUWlQOFplRXUvSnB0cGlIK0hRcEtKYk9Ya055dGdJMVAvMHoyZUZKTGhiVllRPT0iLCJ0b2tlbl90eXBlIjoiZG93bmxvYWQtYXNzZXQtc3RhY2siLCJhb2kiOiIiLCJhc3NldHMiOlt7Iml0ZW1fdHlwZSI6IiIsImFzc2V0X3R5cGUiOiIiLCJpdGVtX2lkIjoiIn1dLCJ1cmwiOiJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vY29tcHV0ZS1vcmRlcnMtbGl2ZS85YzE0NmFkOS0yMGViLTQ0MDctODNiNS03YWQ5NjM1NDlmN2MvUFNTY2VuZS8yMDIyMDkyNF8xMDQ2NDlfMTNfMjI1MV9tZXRhZGF0YS5qc29uP1gtR29vZy1BbGdvcml0aG09R09PRzQtUlNBLVNIQTI1Nlx1MDAyNlgtR29vZy1DcmVkZW50aWFsPWNvbXB1dGUtZ2NzLXN2Y2FjYyU0MHBsYW5ldC1jb21wdXRlLXByb2QuaWFtLmdzZXJ2aWNlYWNjb3VudC5jb20lMkYyMDI0MDUzMCUyRmF1dG8lMkZzdG9yYWdlJTJGZ29vZzRfcmVxdWVzdFx1MDAyNlgtR29vZy1EYXRlPTIwMjQwNTMwVDE0MzYzMlpcdTAwMjZYLUdvb2ctRXhwaXJlcz04NjM5OVx1MDAyNlgtR29vZy1TaWduYXR1cmU9MjQ4OWEyOTYxMDNkOTE2Y2V

In [35]:
async with Session() as sess:
    cl = OrdersClient(sess)

    filenames = await cl.download_order(order['id'])

In [ ]:
show_rgb(img_file)
show_rgb(clip_img_file)

Alright! Clipping worked. Beautiful! This saves a lot of download bandwidth and processing on our end.

### Band Math
<a id='bandmath'></a>

To demonstrate band math we will order an NDVI image.

In [ ]:
bandmath = {
  "bandmath": {
    "pixel_type": "32R",
    "b1": "(b4 - b3) / (b4 + b3)"
  }
}

In [ ]:
bandmath_request = {
  "name": "band math",
  "products": single_product,
  "tools": [bandmath]
}
bandmath_request

In [ ]:
run_bandmath = True

# allow for caching so we don't always run clip
bandmath_img_file = 'data/06a733ff-3034-4ffe-b1c4-8f5d004feaff/1/files/20151119_025740_0c74_3B_AnalyticMS_bandmath.tif'
if os.path.isfile(bandmath_img_file): run_bandmath = False

In [ ]:
if run_bandmath:
    async with Session() as sess:
        cl = OrdersClient(sess)
        order = await cl.create_order(bandmath_request)
        download = await poll_and_download(order)
    bandmath_img_file = next(downloaded_bandmath_files[d] for d in downloaded_bandmath_files
                             if d.endswith('_bandmath.tif'))
bandmath_img_file

In [ ]:
show_rgb(img_file)
show_gray(bandmath_img_file)

This is a beautiful NDVI image!

### TOAR
<a id='toar'></a>

The `toar` tool converts imagery to Top of Atmosphere Reflectance.

In [ ]:
toar = {
      "toar": {
        "scale_factor": 10000
      }
}
toar_request = {
  "name": "toar",
  "products": single_product,
  "tools": [toar]
}
toar_request

In [ ]:
run_toar = True

# allow for caching so we don't always run clip
toar_img_file = 'data/d22684ab-49b5-4696-ab9c-d0a893e5c193/1/files/20151119_025740_0c74_3B_AnalyticMS_toar.tif'
if os.path.isfile(toar_img_file): run_toar = False

In [ ]:
if run_toar:
    async with Session() as sess:
        cl = OrdersClient(sess)
        order = await cl.create_order(toar_request)
        download = await poll_and_download(order)
    toar_img_file = next(downloaded_toar_files[d] for d in downloaded_toar_files
                     if d.endswith('_toar.tif'))
toar_img_file

In [ ]:
show_rgb(img_file)
show_rgb(toar_img_file)

The effect of conversion to reflectance isn't really apparent in one image. But when comparing two images taken at different times of the year, the conversion to reflectance will remove effects of the sun's location and make the images more similar.

### Composite
<a id='composite'></a>

The composite tool combines multiple images into one image, similar to mosaicing. The input images must have the same band configuration, and that band configuration will be propagated to the output image.

In [ ]:
composite = {  
   "composite":{  
   }
}

composite_request = {
  "name": "composite",
  "products": same_src_products,
  "tools": [composite]
}

composite_request

In [ ]:
run_composite = True

composite_file = 'data/be20feb1-b084-43b9-9c1c-b1d4a1ffb9fc/1/files/composite.tif'
if os.path.isfile(composite_file): run_composite = False

In [ ]:
if run_composite:
    async with Session() as sess:
        cl = OrdersClient(sess)
        order = await cl.create_order(composite_request)
        download = await poll_and_download(order)
    composite_file = next(downloaded_composite_files[d] for d in downloaded_composite_files
                          if d.endswith('composite.tif'))
composite_file

In [ ]:
show_rgb(composite_file)

Beautiful! The images have been combined into one seamless image.

### Combined Tools - Clip and Band Math
<a id='clip_bandmath'></a>

This toolchain demonstrates how we can combine the clipping tool with the NDVI band math tool to only process and download the NDVI values for pixels we are interested in. Combining tools is as simple as combining the tool definitions in a list in the order request.

In [ ]:
clip_bandmath_request = {
  "name": "clip and bandmath",
  "products": single_product,
  "tools": [clip, bandmath]
}
clip_bandmath_request

In [ ]:
run_clip_bandmath = True

clip_bandmath_file = 'data/d1a94528-c1c3-4fe2-b3e5-6ff0c340a1d6/1/files/20151119_025740_0c74_3B_AnalyticMS_clip_bandmath.tif'
if os.path.isfile(clip_bandmath_file): run_clip_bandmath = False

In [ ]:
if run_clip_bandmath:
    async with Session() as sess:
        cl = OrdersClient(sess)
        order = await cl.create_order(clip_bandmath_request)
        download = await poll_and_download(order)
    clip_bandmath_file = next(downloaded_clip_bandmath_files[d] for d in downloaded_clip_bandmath_files
                     if d.endswith('_clip_bandmath.tif'))
clip_bandmath_file

In [ ]:
show_gray(clip_bandmath_file)

It is easy to add tools into a toolchain and saves a 'ton' of unnecessary pixel downloads and processing on our end. Big win!

## TOAR, Reproject, Tile
<a id='toar_reproject_tile'></a>

For a more complicated example, we will convert the pixels to reflectance, project them to WGS84, and then tile them.

In [ ]:
reproject =     {
    "reproject": {
        "projection": "WGS84",
        "kernel": "cubic"
    }
}

tile = {
  "tile": {
    "tile_size": 1232,
    "origin_x": -180,
    "origin_y": -90,
    "pixel_size": 0.000027056277056,
    "name_template": "C1232_30_30_{tilex:04d}_{tiley:04d}"
  }
}

In [ ]:
trt_request = {
  "name": "toar reproject tile",
  "products": single_product,
  "tools": [toar, reproject, tile]
}
trt_request

In [ ]:
run_trt = True

tile_files = [
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8243_3175.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8242_3175.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8247_3178.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8243_3174.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8244_3174.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8244_3177.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8243_3176.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8247_3177.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8242_3176.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8244_3175.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8246_3177.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8245_3175.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8244_3176.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8245_3177.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8246_3178.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8245_3176.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8245_3178.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8246_3176.tif',
    'data/8482721f-7711-49f6-8e12-6874c6decb38/1/files/C1232_30_30_8246_3179.tif'
]
if os.path.isfile(tile_files[0]): run_trt = False

In [ ]:
if run_trt:
    async with Session() as sess:
        cl = OrdersClient(sess)
        order = await cl.create_order(trt_request)
        download = await poll_and_download(order)
    tile_files = list(d for d in downloaded_trt_files.values()
                     if d.name.startswith('C1232_30_30_'))
    
for f in tile_files:
    print(f)

In [ ]:
for f in tile_files[:4]:
    show_rgb(f)

In [ ]:
test_file = tile_files[0]

In [ ]:
!gdalinfo $test_file

The files have indeed been reprojected to WSG 84 and tiled. It is that easy!